In [1]:
# %%
# =========================================================
# CELL Ã¢â‚¬â€ FAST HISTORICAL STARTING PITCHER TABLE BUILDER
# FULL REPLACEMENT
# Uses MLB schedule endpoint by date chunks instead of one feed call per game
# More defensive historical source detection
# =========================================================
import time
import requests
import numpy as np
import pandas as pd
from pathlib import Path

print("=" * 80)
print("FAST HISTORICAL STARTING PITCHER TABLE BUILDER Ã¢â‚¬â€ FULL REPLACEMENT")
print("=" * 80)

# ---------------------------------------------------------
# 0) helpers
# ---------------------------------------------------------
def find_first_col(df, options):
    lower_map = {str(c).strip().lower(): c for c in df.columns}
    for opt in options:
        if opt.lower() in lower_map:
            return lower_map[opt.lower()]
    return None

def looks_like_mlb_hist(df):
    if not isinstance(df, pd.DataFrame) or len(df) == 0:
        return False, "not a non-empty DataFrame"

    game_id_col = find_first_col(df, ["game_id", "gamepk", "game_pk", "pk"])
    date_col    = find_first_col(df, ["date", "game_date", "game_datetime", "game_datetime_utc", "official_date"])
    away_col    = find_first_col(df, ["away_team", "away", "visitor_team", "away_name", "visiting_team"])
    home_col    = find_first_col(df, ["home_team", "home", "home_name", "home_team_name"])

    missing = []
    if game_id_col is None:
        missing.append("game_id")
    if date_col is None:
        missing.append("date")
    if away_col is None:
        missing.append("away_team")
    if home_col is None:
        missing.append("home_team")

    if missing:
        return False, f"missing key columns: {missing}"

    return True, "ok"

def try_load_file(fp):
    try:
        if fp.suffix.lower() == ".parquet":
            return pd.read_parquet(fp)
        elif fp.suffix.lower() == ".csv":
            return pd.read_csv(fp)
        return None
    except Exception as e:
        print(f"Failed reading {fp}: {e}")
        return None

# ---------------------------------------------------------
# 1) find historical source table
# ---------------------------------------------------------
candidate_names = [
    "mlb",
    "mlb_hist",
    "mlb_games",
    "mlb_game_table",
    "mlb_game_table_hist",
    "mlb_game_table_2024",
    "historical_mlb",
    "hist_mlb",
    "games_hist",
    "model_df",
]

hist = None
source_used = None

print("\nChecking globals() for historical MLB dataframe...")

for name in candidate_names:
    obj = globals().get(name, None)
    if isinstance(obj, pd.DataFrame):
        ok, reason = looks_like_mlb_hist(obj)
        if ok:
            hist = obj.copy()
            source_used = f"memory:{name}"
            print(f"Found usable dataframe in memory: {name} | shape={hist.shape}")
            break
        else:
            print(f"Skipped {name}: {reason}")

# ---------------------------------------------------------
# 2) fallback to disk if not found in memory
# ---------------------------------------------------------
if hist is None:
    print("\nNo usable dataframe found in memory. Checking common file paths...")

    base_dirs = [
        Path.cwd(),
        Path.cwd() / "data",
        Path.cwd() / "data" / "processed",
        Path.cwd() / "data" / "processed" / "mlb",
        Path(r"c:\Users\<user>\OneDrive\Documents\KAI Labs\sports\MLB"),
        Path(r"c:\Users\<user>\OneDrive\Documents\KAI Labs\sports\MLB\data"),
        Path(r"c:\Users\<user>\OneDrive\Documents\KAI Labs\sports\MLB\data\processed"),
        Path(r"c:\Users\<user>\OneDrive\Documents\KAI Labs\sports\MLB\data\processed\mlb"),
    ]

    file_names = [
        "mlb_game_table_hist.parquet",
        "mlb_game_table_hist.csv",
        "mlb_game_table_2024.parquet",
        "mlb_game_table_2024.csv",
        "mlb_game_table.parquet",
        "mlb_game_table.csv",
        "historical_mlb.parquet",
        "historical_mlb.csv",
        "mlb_hist.parquet",
        "mlb_hist.csv",
        "games_hist.parquet",
        "games_hist.csv",
    ]

    for base in base_dirs:
        for fname in file_names:
            fp = base / fname
            if fp.exists():
                print(f"Trying file: {fp}")
                df_try = try_load_file(fp)
                if df_try is None:
                    continue

                ok, reason = looks_like_mlb_hist(df_try)
                if ok:
                    hist = df_try.copy()
                    source_used = f"file:{fp}"
                    print(f"Loaded usable historical MLB data: {fp} | shape={hist.shape}")
                    break
                else:
                    print(f"Rejected {fp.name}: {reason}")
        if hist is not None:
            break

# ---------------------------------------------------------
# 3) wider scan if still not found
# ---------------------------------------------------------
if hist is None:
    print("\nNo common path worked. Running wider scan for MLB historical files...")

    scan_roots = [
        Path.cwd(),
        Path(r"c:\Users\<user>\OneDrive\Documents\KAI Labs\sports\MLB"),
    ]

    possible_files = []

    for root in scan_roots:
        if not root.exists():
            continue
        try:
            for fp in root.rglob("*"):
                if fp.is_file() and fp.suffix.lower() in [".csv", ".parquet"]:
                    low = fp.name.lower()
                    if "mlb" in low and any(k in low for k in ["hist", "game", "table"]):
                        possible_files.append(fp)
        except Exception as e:
            print(f"Scan failed for {root}: {e}")

    possible_files = list(dict.fromkeys(possible_files))[:100]
    print(f"Found {len(possible_files)} candidate files in wider scan.")

    for fp in possible_files:
        print(f"Trying scanned file: {fp}")
        df_try = try_load_file(fp)
        if df_try is None:
            continue

        ok, reason = looks_like_mlb_hist(df_try)
        if ok:
            hist = df_try.copy()
            source_used = f"scan:{fp}"
            print(f"Loaded usable historical MLB data from scan: {fp} | shape={hist.shape}")
            break
        else:
            print(f"Rejected {fp.name}: {reason}")

# ---------------------------------------------------------
# 4) final fail if still nothing
# ---------------------------------------------------------
if hist is None:
    print("\n" + "=" * 80)
    print("NO USABLE HISTORICAL MLB DATAFRAME FOUND")
    print("=" * 80)
    print("Checked:")
    print(" - common dataframe names in memory")
    print(" - common MLB processed file paths")
    print(" - wider MLB file scan")
    print("")
    print("Make sure one of these exists first:")
    print(" - mlb_game_table_hist in memory")
    print(" - data/processed/mlb/mlb_game_table_hist.parquet")
    print(" - data/processed/mlb/mlb_game_table_hist.csv")
    raise ValueError("No usable historical MLB dataframe found in memory or on disk.")

print(f"\nUsing source: {source_used}")
print(f"Source shape: {hist.shape}")

# ---------------------------------------------------------
# 5) detect columns and standardize
# ---------------------------------------------------------
game_id_col = find_first_col(hist, ["game_id", "gamepk", "game_pk", "pk"])
date_col    = find_first_col(hist, ["date", "game_date", "game_datetime", "game_datetime_utc", "official_date"])
away_col    = find_first_col(hist, ["away_team", "away", "visitor_team", "away_name", "visiting_team"])
home_col    = find_first_col(hist, ["home_team", "home", "home_name", "home_team_name"])

print("\nDetected columns:")
print(f" - game_id_col : {game_id_col}")
print(f" - date_col    : {date_col}")
print(f" - away_col    : {away_col}")
print(f" - home_col    : {home_col}")

work = hist.copy()
work[game_id_col] = pd.to_numeric(work[game_id_col], errors="coerce")
work[date_col] = pd.to_datetime(work[date_col], errors="coerce")

before_rows = len(work)
work = work[work[game_id_col].notna()].copy()
work = work[work[date_col].notna()].copy()
work[game_id_col] = work[game_id_col].astype("int64")

print(f"\nRows before cleaning: {before_rows:,}")
print(f"Rows after game_id/date cleaning: {len(work):,}")

rename_map = {
    game_id_col: "game_id",
    date_col: "date",
    away_col: "away_team",
    home_col: "home_team",
}

hist_games = (
    work.rename(columns=rename_map)[["game_id", "date", "away_team", "home_team"]]
        .drop_duplicates(subset=["game_id"])
        .sort_values(["date", "game_id"])
        .reset_index(drop=True)
)

print(f"\nUnique historical games: {len(hist_games):,}")
print(f"Date range: {hist_games['date'].min()} -> {hist_games['date'].max()}")

# ---------------------------------------------------------
# 6) setup paths / session
# ---------------------------------------------------------
save_base_candidates = [
    Path("data/processed/mlb"),
    Path(r"c:\Users\<user>\OneDrive\Documents\KAI Labs\sports\MLB\data\processed\mlb"),
]

SAVE_DIR = None
for p in save_base_candidates:
    try:
        p.mkdir(parents=True, exist_ok=True)
        SAVE_DIR = p
        break
    except Exception:
        pass

if SAVE_DIR is None:
    SAVE_DIR = Path.cwd()

CSV_PATH = SAVE_DIR / "mlb_starter_hist_fast.csv"
PARQUET_PATH = SAVE_DIR / "mlb_starter_hist_fast.parquet"

# Incremental fetch (2026-07-13): load the previous run's output, if any, so
# this cell only re-queries the MLB API for chunks not already covered --
# it used to rebuild the ENTIRE historical date range from scratch every
# single run (no skip-already-fetched logic at all), so a multi-year-
# spanning hist_games table turned every run into 100+ full-history API
# calls even though nothing before the last few days had changed. This was
# the confirmed cause of the model taking 10+ minutes / appearing to hang.
_prev_starter_hist = None
for _pp in (PARQUET_PATH, CSV_PATH):
    if _pp.exists():
        try:
            _prev_starter_hist = pd.read_parquet(_pp) if _pp.suffix == ".parquet" else pd.read_csv(_pp)
            break
        except Exception as e:
            print(f"Could not read previous cache {_pp}: {e}")

SESSION = requests.Session()
SESSION.headers.update({"User-Agent": "Mozilla/5.0"})

# 2026-07-14: added per-attempt progress prints + a shorter default timeout.
# A chunk that silently retried for up to 3x30s=90s with zero output looked
# identical to a true infinite hang from the notebook UI -- there was no way
# to tell "slow/failing request in progress" apart from "actually frozen"
# without this. Confirmed MLB Stats API itself responds in <1s normally, so
# 30s per attempt was already generous; 20s keeps headroom while failing faster.
def safe_get(url, params=None, timeout=20, max_retries=3):
    last_err = None
    _rng = f"{(params or {}).get('startDate','?')}->{(params or {}).get('endDate','?')}"
    for attempt in range(1, max_retries + 1):
        _t0 = time.time()
        print(f"    [HTTP] attempt {attempt}/{max_retries} {_rng}...", end="", flush=True)
        try:
            r = SESSION.get(url, params=params, timeout=timeout)
            r.raise_for_status()
            print(f" ok ({time.time()-_t0:.1f}s)")
            return r.json()
        except Exception as e:
            last_err = e
            print(f" FAILED after {time.time()-_t0:.1f}s: {e}")
            if attempt < max_retries:
                time.sleep(0.75 * attempt)
    return {"_fetch_error": str(last_err)}

# ---------------------------------------------------------
# 7) build date chunks
# ---------------------------------------------------------
MIN_DATE = hist_games["date"].min().normalize()
# 2026-07-28 fix: MAX_DATE was derived from hist_games["date"].max() -- the
# EXISTING historical table's own current ceiling -- so the fetch loop below
# (`while cur <= MAX_DATE`) could never extend past whatever data already
# existed, no matter how many times this cell re-ran or how much real time
# passed. It's a self-referential ceiling, not a stale-but-fixable constant.
# Real impact, found while investigating a totals-calibration staleness
# question: mlb_model_table_v1.parquet (and everything downstream of
# hist_games here -- mlb_game_table_hist, mlb_with_starter_features) has been
# frozen at 2025-11-01 for months despite this cell visibly re-running most
# days (other output files in the same folder show today's mtime) -- the
# totals-calibration table built from it has zero 2026 games in it despite
# correcting 2026 predictions. Same root-cause class as the 2026-07-06 MLB
# ELO cache fix (hardcoded `_ELO_SEASON = 2025`) -- a frozen ceiling instead
# of the real current date. Fixed to use today's actual date.
MAX_DATE = pd.Timestamp.today().normalize()

# Incremental fetch, part 2: only refetch from a bit before the previous
# cache's max date onward -- older completed games' starters don't change,
# and this buffer (not an exact resume point) also naturally re-confirms
# probable pitchers for recent games, which firm up closer to game day.
_REFETCH_BUFFER_DAYS = 10
_fetch_start = MIN_DATE
if _prev_starter_hist is not None and "date" in _prev_starter_hist.columns and len(_prev_starter_hist):
    _prev_dates = pd.to_datetime(_prev_starter_hist["date"], errors="coerce")
    if _prev_dates.notna().any():
        _prev_max_date = _prev_dates.max().normalize()
        _fetch_start = max(MIN_DATE, (_prev_max_date - pd.Timedelta(days=_REFETCH_BUFFER_DAYS)).normalize())
        print(f"Previous cache found: covers through {_prev_max_date.date()} -- refetching from {_fetch_start.date()} onward only")

CHUNK_DAYS = 7
date_chunks = []

cur = _fetch_start
while cur <= MAX_DATE:
    end = min(cur + pd.Timedelta(days=CHUNK_DAYS - 1), MAX_DATE)
    date_chunks.append((cur, end))
    cur = end + pd.Timedelta(days=1)

print(f"\nDate chunks to query: {len(date_chunks):,}")

# ---------------------------------------------------------
# 8) schedule pull function
# ---------------------------------------------------------
def parse_schedule_chunk(start_date, end_date):
    url = "https://statsapi.mlb.com/api/v1/schedule"
    params = {
        "sportId": 1,
        "startDate": start_date.strftime("%Y-%m-%d"),
        "endDate": end_date.strftime("%Y-%m-%d"),
        "hydrate": "probablePitcher,team",
    }

    payload = safe_get(url, params=params)

    rows = []

    if "_fetch_error" in payload:
        return pd.DataFrame([{
            "game_id": np.nan,
            "date": pd.NaT,
            "away_team": np.nan,
            "home_team": np.nan,
            "away_starter_id": np.nan,
            "away_starter_name": np.nan,
            "home_starter_id": np.nan,
            "home_starter_name": np.nan,
            "starter_source": "schedule_error",
            "chunk_start": start_date,
            "chunk_end": end_date,
            "fetch_error": payload["_fetch_error"],
        }])

    dates = payload.get("dates", []) or []
    for d in dates:
        game_date = pd.to_datetime(d.get("date"), errors="coerce")
        for g in d.get("games", []) or []:
            teams = g.get("teams", {}) or {}

            away = teams.get("away", {}) or {}
            home = teams.get("home", {}) or {}

            away_team = ((away.get("team") or {}).get("name"))
            home_team = ((home.get("team") or {}).get("name"))

            away_prob = away.get("probablePitcher") or {}
            home_prob = home.get("probablePitcher") or {}

            rows.append({
                "game_id": pd.to_numeric(g.get("gamePk"), errors="coerce"),
                "date": game_date,
                "away_team": away_team,
                "home_team": home_team,
                "away_starter_id": pd.to_numeric(away_prob.get("id"), errors="coerce"),
                "away_starter_name": away_prob.get("fullName", np.nan),
                "home_starter_id": pd.to_numeric(home_prob.get("id"), errors="coerce"),
                "home_starter_name": home_prob.get("fullName", np.nan),
                "starter_source": "schedule_probablePitcher",
                "chunk_start": start_date,
                "chunk_end": end_date,
                "fetch_error": np.nan,
            })

    return pd.DataFrame(rows)

# ---------------------------------------------------------
# 9) query schedule chunks
# ---------------------------------------------------------
all_parts = []

for i, (chunk_start, chunk_end) in enumerate(date_chunks, start=1):
    part = parse_schedule_chunk(chunk_start, chunk_end)
    all_parts.append(part)

    print(
        f"[{i:,}/{len(date_chunks):,}] "
        f"{chunk_start.date()} -> {chunk_end.date()} | rows={len(part):,}"
    )

    time.sleep(0.10)

starter_sched = pd.concat(all_parts, ignore_index=True) if all_parts else pd.DataFrame()

print("")
print(f"Raw pulled starter schedule shape: {starter_sched.shape}")

# ---------------------------------------------------------
# 10) keep only rows that match historical game_ids
# ---------------------------------------------------------
starter_sched["game_id"] = pd.to_numeric(starter_sched["game_id"], errors="coerce")
starter_sched = starter_sched[starter_sched["game_id"].notna()].copy()

if len(starter_sched) > 0:
    starter_sched["game_id"] = starter_sched["game_id"].astype("int64")

starter_hist = hist_games.merge(
    starter_sched[
        [
            "game_id",
            "away_starter_id", "away_starter_name",
            "home_starter_id", "home_starter_name",
            "starter_source", "fetch_error"
        ]
    ].drop_duplicates(subset=["game_id"], keep="last") if len(starter_sched) > 0 else pd.DataFrame(columns=[
        "game_id",
        "away_starter_id", "away_starter_name",
        "home_starter_id", "home_starter_name",
        "starter_source", "fetch_error"
    ]),
    on="game_id",
    how="left",
    validate="one_to_one"
)

# Backfill games outside the freshly-fetched range from the previous cache
# (still-NaN after the merge above = not in this run's reduced fetch window),
# so the full historical table stays complete without re-querying all of it.
if _prev_starter_hist is not None:
    _backfill_cols = ["game_id", "away_starter_id", "away_starter_name",
                       "home_starter_id", "home_starter_name", "starter_source", "fetch_error"]
    _prev_use = _prev_starter_hist[[c for c in _backfill_cols if c in _prev_starter_hist.columns]].copy()
    _prev_use["game_id"] = pd.to_numeric(_prev_use["game_id"], errors="coerce")
    _prev_use = _prev_use[_prev_use["game_id"].notna()].copy()
    _prev_use["game_id"] = _prev_use["game_id"].astype("int64")
    _prev_use = _prev_use.drop_duplicates(subset=["game_id"], keep="last")
    starter_hist = starter_hist.merge(_prev_use, on="game_id", how="left", suffixes=("", "_prevcache"))
    for _c in ["away_starter_id", "away_starter_name", "home_starter_id", "home_starter_name", "starter_source", "fetch_error"]:
        _pc = f"{_c}_prevcache"
        if _pc in starter_hist.columns:
            starter_hist[_c] = starter_hist[_c].where(starter_hist[_c].notna(), starter_hist[_pc])
            starter_hist.drop(columns=[_pc], inplace=True)

# ---------------------------------------------------------
# 11) debug prints
# ---------------------------------------------------------
print("")
print("=" * 80)
print("STARTER TABLE SUMMARY")
print("=" * 80)
print(f"starter_hist shape: {starter_hist.shape}")

summary = pd.Series({
    "missing_away_starter_id": int(starter_hist["away_starter_id"].isna().sum()),
    "missing_home_starter_id": int(starter_hist["home_starter_id"].isna().sum()),
    "missing_any_starter_id": int(
        (starter_hist["away_starter_id"].isna() | starter_hist["home_starter_id"].isna()).sum()
    ),
    "matched_rows": int(
        (~starter_hist["away_starter_id"].isna() | ~starter_hist["home_starter_id"].isna()).sum()
    ),
})
print(summary)

if "starter_source" in starter_hist.columns:
    print("")
    print("starter_source counts:")
    print(starter_hist["starter_source"].fillna("missing").value_counts(dropna=False))

print("")
display(starter_hist.head(10))

# ---------------------------------------------------------
# 12) save / expose
# ---------------------------------------------------------
globals()["mlb_starter_hist"] = starter_hist.copy()

try:
    starter_hist.to_csv(CSV_PATH, index=False)
    print(f"Saved CSV: {CSV_PATH}")
except Exception as e:
    print(f"WARNING: could not save CSV -> {e}")

try:
    starter_hist.to_parquet(PARQUET_PATH, index=False)
    print(f"Saved Parquet: {PARQUET_PATH}")
except Exception as e:
    print(f"WARNING: could not save Parquet -> {e}")

print("")
print("Created in memory as: mlb_starter_hist")
print("Done.")

FAST HISTORICAL STARTING PITCHER TABLE BUILDER Ã¢â‚¬â€ FULL REPLACEMENT

Checking globals() for historical MLB dataframe...

No usable dataframe found in memory. Checking common file paths...
Trying file: C:\Users\<user>\OneDrive\Documents\KAI Labs\sports\MLB\data\processed\mlb\mlb_game_table_hist.parquet


Loaded usable historical MLB data: C:\Users\<user>\OneDrive\Documents\KAI Labs\sports\MLB\data\processed\mlb\mlb_game_table_hist.parquet | shape=(8576, 16)

Using source: file:C:\Users\<user>\OneDrive\Documents\KAI Labs\sports\MLB\data\processed\mlb\mlb_game_table_hist.parquet
Source shape: (8576, 16)

Detected columns:
 - game_id_col : game_id
 - date_col    : date
 - away_col    : away_team
 - home_col    : home_team

Rows before cleaning: 8,576
Rows after game_id/date cleaning: 8,576

Unique historical games: 8,562
Date range: 2023-03-01 00:00:00 -> 2025-11-01 00:00:00
Previous cache found: covers through 2025-11-01 -- refetching from 2025-10-22 onward only

Date chunks to query: 51
    [HTTP] attempt 1/3 2025-10-22->2025-10-28...

 ok (0.5s)
[1/51] 2025-10-22 -> 2025-10-28 | rows=4
    [HTTP] attempt 1/3 2025-10-29->2025-11-04...

 ok (0.1s)
[2/51] 2025-10-29 -> 2025-11-04 | rows=3
    [HTTP] attempt 1/3 2025-11-05->2025-11-11...

 ok (0.1s)
[3/51] 2025-11-05 -> 2025-11-11 | rows=0


...

[50/51] 2026-09-30 -> 2026-10-06 | rows=15
    [HTTP] attempt 1/3 2026-10-07->2026-10-10...

 ok (0.1s)
[51/51] 2026-10-07 -> 2026-10-10 | rows=6



Raw pulled starter schedule shape: (2981, 12)

STARTER TABLE SUMMARY
starter_hist shape: (8562, 10)
missing_away_starter_id     337
missing_home_starter_id     341
missing_any_starter_id      346
matched_rows               8230
dtype: int64

starter_source counts:
starter_source
schedule_probablePitcher    8230
missing                      332
Name: count, dtype: int64



,game_id,date,away_team,home_team,away_starter_id,away_starter_name,home_starter_id,home_starter_name,starter_source,fetch_error
0,719313,2023-03-01,Los Angeles Dodgers,Texas Rangers,NaN,None,NaN,None,None,None
1,719316,2023-03-01,Texas Rangers,San Diego Padres,NaN,None,NaN,None,None,None
2,719318,2023-03-01,Arizona Diamondbacks,San Francisco Giants,NaN,None,NaN,None,None,None
3,719319,2023-03-01,Oakland Athletics,Cincinnati Reds,NaN,None,NaN,None,None,None
4,719320,2023-03-01,Milwaukee Brewers,Los Angeles Angels,NaN,None,NaN,None,None,None
5,719321,2023-03-01,Kansas City Royals,Colorado Rockies,NaN,None,NaN,None,None,None
6,719322,2023-03-01,Cleveland Guardians,Chicago White Sox,NaN,None,NaN,None,None,None
7,719323,2023-03-01,Washington Nationals,New York Yankees,NaN,None,NaN,None,None,None
8,719324,2023-03-01,Philadelphia Phillies,Minnesota Twins,NaN,None,NaN,None,None,None
9,719325,2023-03-01,Seattle Mariners,Chicago Cubs,NaN,None,NaN,None,None,None


Saved CSV: data\processed\mlb\mlb_starter_hist_fast.csv
Saved Parquet: data\processed\mlb\mlb_starter_hist_fast.parquet

Created in memory as: mlb_starter_hist
Done.


In [2]:
# %%
# =========================================================
# CELL 2 Ã¢â‚¬â€ LOAD + CLEAN MLB MODEL TABLE
# FULL REPLACEMENT WITH AUTO TARGET DETECTION
# =========================================================
import pandas as pd
from pathlib import Path

print("=" * 70)
print("CELL 2 Ã¢â‚¬â€ LOAD + CLEAN MLB MODEL TABLE")
print("=" * 70)

mlb_raw = None
source_used = None
resolved_path = None

# ---------------------------------------------------------
# helper functions
# ---------------------------------------------------------
def try_load_table(fp):
    try:
        if fp.suffix.lower() == ".parquet":
            return pd.read_parquet(fp)
        elif fp.suffix.lower() == ".csv":
            return pd.read_csv(fp)
        return None
    except Exception as e:
        print(f"Failed reading {fp}: {e}")
        return None

def find_first_existing_col(df, options):
    lower_map = {str(c).strip().lower(): c for c in df.columns}
    for opt in options:
        if opt.lower() in lower_map:
            return lower_map[opt.lower()]
    return None

# ---------------------------------------------------------
# STEP 1: try DATA_PATH if it already exists
# ---------------------------------------------------------
if "DATA_PATH" in globals():
    try:
        candidate = Path(DATA_PATH)
        if candidate.exists():
            print(f"Using existing DATA_PATH: {candidate}")
            df_try = try_load_table(candidate)
            if isinstance(df_try, pd.DataFrame) and len(df_try) > 0:
                mlb_raw = df_try.copy()
                source_used = "DATA_PATH"
                resolved_path = candidate
        else:
            print(f"DATA_PATH exists as variable, but file not found: {candidate}")
    except Exception as e:
        print(f"Could not use DATA_PATH: {e}")
else:
    print("DATA_PATH not defined. Searching common MLB paths...")

# ---------------------------------------------------------
# STEP 2: search common file locations if needed
# ---------------------------------------------------------
if mlb_raw is None:
    base_dirs = [
        Path.cwd(),
        Path.cwd() / "data",
        Path.cwd() / "data" / "processed",
        Path.cwd() / "data" / "processed" / "mlb",
        Path(r"c:\Users\<user>\OneDrive\Documents\KAI Labs\sports\MLB"),
        Path(r"c:\Users\<user>\OneDrive\Documents\KAI Labs\sports\MLB\data"),
        Path(r"c:\Users\<user>\OneDrive\Documents\KAI Labs\sports\MLB\data\processed"),
        Path(r"c:\Users\<user>\OneDrive\Documents\KAI Labs\sports\MLB\data\processed\mlb"),
    ]

    file_candidates = [
        "mlb_model_table_v1.parquet",
        "mlb_model_table_v1.csv",
        "mlb_model_table.parquet",
        "mlb_model_table.csv",
        "model_table.parquet",
        "model_table.csv",
        "mlb_game_table_hist.parquet",
        "mlb_game_table_hist.csv",
        "mlb_game_table_2024.parquet",
        "mlb_game_table_2024.csv",
    ]

    for base in base_dirs:
        for fname in file_candidates:
            fp = base / fname
            if fp.exists():
                print(f"Trying file: {fp}")
                df_try = try_load_table(fp)
                if isinstance(df_try, pd.DataFrame) and len(df_try) > 0:
                    mlb_raw = df_try.copy()
                    source_used = f"file:{fp}"
                    resolved_path = fp
                    break
        if mlb_raw is not None:
            break

# ---------------------------------------------------------
# STEP 3: final fail if nothing found
# ---------------------------------------------------------
if mlb_raw is None:
    raise FileNotFoundError(
        "Could not find a usable MLB model table. "
        "Checked DATA_PATH and common MLB project folders."
    )

print(f"\nLoaded source: {source_used}")
print(f"Loaded raw table shape: {mlb_raw.shape}")

print("\nColumns found:")
print(list(mlb_raw.columns))

# ---------------------------------------------------------
# STEP 4: auto-detect target columns if missing
# ---------------------------------------------------------
if "TARGET_WIN" not in globals():
    TARGET_WIN = find_first_existing_col(mlb_raw, [
        "home_win",
        "target_win",
        "y_win",
        "label_win",
        "home_team_win",
        "home_win_flag",
        "is_home_win",
        "winner_home",
    ])

if "TARGET_MARGIN" not in globals():
    TARGET_MARGIN = find_first_existing_col(mlb_raw, [
        "margin",
        "home_margin",
        "run_margin",
        "score_margin",
        "target_margin",
        "y_margin",
        "home_run_margin",
    ])

if "TARGET_TOTAL" not in globals():
    TARGET_TOTAL = find_first_existing_col(mlb_raw, [
        "total",
        "runs_total",
        "game_total",
        "total_runs",
        "target_total",
        "y_total",
        "combined_runs",
    ])

# hard fallback defaults if they exist literally
if TARGET_WIN is None and "home_win" in mlb_raw.columns:
    TARGET_WIN = "home_win"
if TARGET_MARGIN is None and "margin" in mlb_raw.columns:
    TARGET_MARGIN = "margin"
if TARGET_TOTAL is None and "total" in mlb_raw.columns:
    TARGET_TOTAL = "total"

print("\nResolved targets:")
print(f" - TARGET_WIN    : {TARGET_WIN}")
print(f" - TARGET_MARGIN : {TARGET_MARGIN}")
print(f" - TARGET_TOTAL  : {TARGET_TOTAL}")

missing_targets = [
    name for name, value in {
        "TARGET_WIN": TARGET_WIN,
        "TARGET_MARGIN": TARGET_MARGIN,
        "TARGET_TOTAL": TARGET_TOTAL
    }.items() if value is None
]

if missing_targets:
    raise ValueError(
        f"Could not auto-detect target columns: {missing_targets}\n"
        f"Available columns are:\n{list(mlb_raw.columns)}"
    )

# ---------------------------------------------------------
# STEP 5: validate required columns
# ---------------------------------------------------------
required_core = ["date", "away_team", "home_team", TARGET_WIN, TARGET_MARGIN, TARGET_TOTAL]
missing_core = [c for c in required_core if c not in mlb_raw.columns]

if missing_core:
    raise ValueError(f"Missing required columns: {missing_core}")

# ---------------------------------------------------------
# STEP 6: clean base table
# ---------------------------------------------------------
mlb = mlb_raw.copy()
mlb["date"] = pd.to_datetime(mlb["date"], errors="coerce")

before_rows = len(mlb)

# drop rows missing core columns
mlb = mlb.dropna(subset=required_core).copy()

# normalize team names a bit
mlb["away_team"] = mlb["away_team"].astype(str).str.strip()
mlb["home_team"] = mlb["home_team"].astype(str).str.strip()

# sort before dedupe
sort_cols = [c for c in ["date", "game_id"] if c in mlb.columns]
if sort_cols:
    mlb = mlb.sort_values(sort_cols).reset_index(drop=True)

# ---------------------------------------------------------
# STEP 7: dedupe by game identity
# ---------------------------------------------------------
dup_main_before = mlb.duplicated(subset=["date", "away_team", "home_team"]).sum()
print("\nDuplicate rows by [date, away_team, home_team] before:", int(dup_main_before))

mlb = mlb.drop_duplicates(subset=["date", "away_team", "home_team"], keep="last").copy()

if "game_id" in mlb.columns:
    dup_gid_before = mlb.duplicated(subset=["game_id"]).sum()
    print("Duplicate rows by [game_id] before final drop:", int(dup_gid_before))
    mlb = mlb.drop_duplicates(subset=["game_id"], keep="last").copy()

# ---------------------------------------------------------
# STEP 8: final sort
# ---------------------------------------------------------
mlb = mlb.sort_values(["date", "away_team", "home_team"]).reset_index(drop=True)

# ---------------------------------------------------------
# STEP 9: final checks
# ---------------------------------------------------------
print("")
print("Rows before cleaning:", before_rows)
print("Rows after cleaning :", len(mlb))
print("Rows removed        :", before_rows - len(mlb))
print("Date range          :", mlb["date"].min(), "->", mlb["date"].max())
print("Home win rate       :", round(mlb[TARGET_WIN].mean(), 4))

final_dup_main = mlb.duplicated(subset=["date", "away_team", "home_team"]).sum()
print("Remaining duplicate rows by [date, away_team, home_team]:", int(final_dup_main))

if "game_id" in mlb.columns:
    final_dup_gid = mlb.duplicated(subset=["game_id"]).sum()
    print("Remaining duplicate rows by [game_id]:", int(final_dup_gid))

print("")
print("Sample rows:")
display(mlb.head(10))

# ---------------------------------------------------------
# STEP 10: expose cleaned table and resolved settings
# ---------------------------------------------------------
globals()["mlb"] = mlb.copy()
globals()["TARGET_WIN"] = TARGET_WIN
globals()["TARGET_MARGIN"] = TARGET_MARGIN
globals()["TARGET_TOTAL"] = TARGET_TOTAL

if resolved_path is not None:
    globals()["DATA_PATH"] = Path(resolved_path)

print("")
print("Created in memory:")
print(" - mlb")
print(f" - TARGET_WIN = {TARGET_WIN}")
print(f" - TARGET_MARGIN = {TARGET_MARGIN}")
print(f" - TARGET_TOTAL = {TARGET_TOTAL}")
if resolved_path is not None:
    print(f" - DATA_PATH = {resolved_path}")
print("Done.")

CELL 2 Ã¢â‚¬â€ LOAD + CLEAN MLB MODEL TABLE
DATA_PATH not defined. Searching common MLB paths...
Trying file: C:\Users\<user>\OneDrive\Documents\KAI Labs\sports\MLB\data\processed\mlb\mlb_model_table_v1.parquet

Loaded source: file:C:\Users\<user>\OneDrive\Documents\KAI Labs\sports\MLB\data\processed\mlb\mlb_model_table_v1.parquet
Loaded raw table shape: (8348, 55)

Columns found:
['date', 'season', 'game_id', 'game_datetime_utc', 'away_team', 'home_team', 'away_team_id', 'home_team_id', 'venue_name', 'away_runs', 'home_runs', 'total_runs', 'home_margin', 'home_win', 'away_win', 'matchup', 'away_runs_for_avg_3', 'away_runs_against_avg_3', 'away_margin_avg_3', 'away_win_rate_3', 'away_runs_for_avg_5', 'away_runs_against_avg_5', 'away_margin_avg_5', 'away_win_rate_5', 'away_runs_for_avg_10', 'away_runs_against_avg_10', 'away_margin_avg_10', 'away_win_rate_10', 'away_rest_days', 'home_runs_for_avg_3', 'home_runs_against_avg_3', 'home_margin_avg_3', 'home_win_rate_3', 'home_runs_for_avg_5


Duplicate rows by [date, away_team, home_team] before: 104
Duplicate rows by [game_id] before final drop: 14

Rows before cleaning: 8348
Rows after cleaning : 8230
Rows removed        : 118
Date range          : 2023-03-02 00:00:00 -> 2025-11-01 00:00:00
Home win rate       : 0.5272
Remaining duplicate rows by [date, away_team, home_team]: 0
Remaining duplicate rows by [game_id]: 0

Sample rows:


,date,season,game_id,game_datetime_utc,away_team,home_team,away_team_id,home_team_id,venue_name,away_runs,...,diff_win_rate_3,diff_runs_for_avg_5,diff_runs_against_avg_5,diff_margin_avg_5,diff_win_rate_5,diff_runs_for_avg_10,diff_runs_against_avg_10,diff_margin_avg_10,diff_win_rate_10,diff_rest_days
0,2023-03-02,2023,719302,2023-03-02T20:05:00Z,Arizona Diamondbacks,Los Angeles Dodgers,109,119,Camelback Ranch,5.0,...,0.0,-4.0,-3.0,-1.0,0.0,-4.0,-3.0,-1.0,0.0,0.0
1,2023-03-02,2023,719315,2023-03-02T18:10:00Z,Atlanta Braves,New York Mets,144,121,Clover Park,6.0,...,1.0,5.0,1.0,4.0,1.0,5.0,1.0,4.0,1.0,0.0
2,2023-03-02,2023,719314,2023-03-02T18:05:00Z,Baltimore Orioles,Detroit Tigers,110,116,Publix Field at Joker Marchant Stadium,3.0,...,1.0,7.0,5.0,2.0,1.0,7.0,5.0,2.0,1.0,0.0
3,2023-03-02,2023,719305,2023-03-02T20:10:00Z,Chicago White Sox,Colorado Rockies,145,115,Salt River Fields at Talking Stick,6.0,...,0.0,0.0,-1.0,1.0,0.0,0.0,-1.0,1.0,0.0,0.0
4,2023-03-02,2023,719304,2023-03-02T20:05:00Z,Los Angeles Angels,Kansas City Royals,108,118,Surprise Stadium,4.0,...,0.0,4.0,3.0,1.0,0.0,4.0,3.0,1.0,0.0,0.0
5,2023-03-02,2023,719308,2023-03-02T18:05:00Z,Miami Marlins,Washington Nationals,146,120,The Ballpark of the Palm Beaches,5.0,...,0.0,-2.0,-4.0,2.0,0.0,-2.0,-4.0,2.0,0.0,0.0
6,2023-03-02,2023,719307,2023-03-02T18:05:00Z,Minnesota Twins,Tampa Bay Rays,142,139,Tropicana Field,4.0,...,0.0,-1.0,-1.0,0.0,0.0,-1.0,-1.0,0.0,0.0,0.0
7,2023-03-02,2023,719310,2023-03-02T18:05:00Z,New York Yankees,Pittsburgh Pirates,147,134,LECOM Park,9.0,...,-1.0,0.5,3.0,-2.5,-1.0,0.5,3.0,-2.5,-1.0,-1.0
8,2023-03-02,2023,719312,2023-03-02T20:05:00Z,Oakland Athletics,Chicago Cubs,133,112,Sloan Park,1.0,...,1.0,-9.0,-11.0,2.0,1.0,-9.0,-11.0,2.0,1.0,0.0
9,2023-03-02,2023,719317,2023-03-02T17:05:00Z,Philadelphia Phillies,Boston Red Sox,143,111,JetBlue Park,3.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0



Created in memory:
 - mlb
 - TARGET_WIN = home_win
 - TARGET_MARGIN = home_margin
 - TARGET_TOTAL = total_runs
 - DATA_PATH = C:\Users\<user>\OneDrive\Documents\KAI Labs\sports\MLB\data\processed\mlb\mlb_model_table_v1.parquet
Done.


In [3]:
import numpy as np
print(np.__version__)

1.26.4


In [4]:
import sys
!{sys.executable} -m pip uninstall -y numpy meteostat
!{sys.executable} -m pip install "numpy<2.0" "meteostat==1.6.5"

Found existing installation: numpy 1.26.4
Uninstalling numpy-1.26.4:
  Successfully uninstalled numpy-1.26.4
Found existing installation: meteostat 1.6.5
Uninstalling meteostat-1.6.5:
  Successfully uninstalled meteostat-1.6.5


You can safely remove it manually.
You can safely remove it manually.


  Using cached numpy-1.26.4-cp311-cp311-win_amd64.whl.metadata (61 kB)
  Using cached meteostat-1.6.5-py3-none-any.whl.metadata (4.6 kB)
Using cached meteostat-1.6.5-py3-none-any.whl (31 kB)
Using cached numpy-1.26.4-cp311-cp311-win_amd64.whl (15.8 MB)

   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sportsdataverse 0.0.40 requires polars<=0.18.15, but you have polars 1.43.2 which is incompatible.


In [5]:
# %%
import sys
!{sys.executable} -m pip uninstall meteostat -y
!{sys.executable} -m pip install meteostat==1.6.5

Found existing installation: meteostat 1.6.5
Uninstalling meteostat-1.6.5:
  Successfully uninstalled meteostat-1.6.5


  Using cached meteostat-1.6.5-py3-none-any.whl.metadata (4.6 kB)
Using cached meteostat-1.6.5-py3-none-any.whl (31 kB)


In [6]:
# %%
# =========================================================
# CELL Ã¢â‚¬â€ BUILD HISTORICAL MLB WEATHER TABLE
# =========================================================

import pandas as pd
import numpy as np

print("=" * 80)
print("BUILD HISTORICAL MLB WEATHER TABLE")
print("=" * 80)

if "mlb" not in globals() or not isinstance(mlb, pd.DataFrame):
    raise ValueError("mlb not found. Run Cell 2 first.")

from meteostat import Point, Daily

hist_weather_source = mlb.copy()
hist_weather_source["date"] = pd.to_datetime(hist_weather_source["date"], errors="coerce").dt.normalize()

required_cols = ["date", "home_team"]
missing_required = [c for c in required_cols if c not in hist_weather_source.columns]
if missing_required:
    raise ValueError(f"Missing required columns for weather build: {missing_required}")

MLB_PARKS = {
    "Arizona Diamondbacks": ("Chase Field", 33.4455, -112.0667, 331),
    "Atlanta Braves": ("Truist Park", 33.8907, -84.4677, 321),
    "Baltimore Orioles": ("Oriole Park at Camden Yards", 39.2840, -76.6217, 6),
    "Boston Red Sox": ("Fenway Park", 42.3467, -71.0972, 6),
    "Chicago Cubs": ("Wrigley Field", 41.9484, -87.6553, 181),
    "Chicago White Sox": ("Guaranteed Rate Field", 41.8300, -87.6338, 181),
    "Cincinnati Reds": ("Great American Ball Park", 39.0979, -84.5081, 168),
    "Cleveland Guardians": ("Progressive Field", 41.4962, -81.6852, 199),
    "Colorado Rockies": ("Coors Field", 39.7559, -104.9942, 1609),
    "Detroit Tigers": ("Comerica Park", 42.3390, -83.0485, 183),
    "Houston Astros": ("Minute Maid Park", 29.7573, -95.3555, 13),
    "Kansas City Royals": ("Kauffman Stadium", 39.0517, -94.4803, 274),
    "Los Angeles Angels": ("Angel Stadium", 33.8003, -117.8827, 48),
    "Los Angeles Dodgers": ("Dodger Stadium", 34.0739, -118.2400, 154),
    "Miami Marlins": ("loanDepot park", 25.7781, -80.2197, 2),
    "Milwaukee Brewers": ("American Family Field", 43.0280, -87.9712, 194),
    "Minnesota Twins": ("Target Field", 44.9817, -93.2776, 264),
    "New York Mets": ("Citi Field", 40.7571, -73.8458, 3),
    "New York Yankees": ("Yankee Stadium", 40.8296, -73.9262, 17),
    "Oakland Athletics": ("Oakland Coliseum", 37.7516, -122.2005, 4),
    "Philadelphia Phillies": ("Citizens Bank Park", 39.9061, -75.1665, 12),
    "Pittsburgh Pirates": ("PNC Park", 40.4469, -80.0057, 223),
    "San Diego Padres": ("Petco Park", 32.7073, -117.1573, 20),
    "San Francisco Giants": ("Oracle Park", 37.7786, -122.3893, 3),
    "Seattle Mariners": ("T-Mobile Park", 47.5914, -122.3325, 30),
    "St. Louis Cardinals": ("Busch Stadium", 38.6226, -90.1928, 142),
    "Tampa Bay Rays": ("Tropicana Field", 27.7682, -82.6534, 3),
    "Texas Rangers": ("Globe Life Field", 32.7473, -97.0847, 168),
    "Toronto Blue Jays": ("Rogers Centre", 43.6414, -79.3894, 76),
    "Washington Nationals": ("Nationals Park", 38.8730, -77.0074, 7),
}

DOME_TEAMS = {
    "Arizona Diamondbacks",
    "Houston Astros",
    "Miami Marlins",
    "Milwaukee Brewers",
    "Seattle Mariners",
    "Tampa Bay Rays",
    "Texas Rangers",
    "Toronto Blue Jays",
}

weather_keys = (
    hist_weather_source[["date", "home_team"]]
    .dropna()
    .drop_duplicates()
    .sort_values(["home_team", "date"])
    .reset_index(drop=True)
)

print("Unique home-team/date weather keys:", len(weather_keys))

parts = []

for team, sub in weather_keys.groupby("home_team", sort=True):
    if team not in MLB_PARKS:
        print(f"Skipping unknown park mapping for: {team}")
        continue

    park_name, lat, lon, alt = MLB_PARKS[team]
    start = sub["date"].min()
    end = sub["date"].max()

    try:
        point = Point(lat, lon, alt)
        wx = Daily(point, start, end).fetch().reset_index()
    except Exception as e:
        print(f"Weather fetch failed for {team}: {e}")
        continue

    if wx is None or len(wx) == 0:
        print(f"No weather rows returned for {team}")
        continue

    wx["date"] = pd.to_datetime(wx["time"], errors="coerce").dt.normalize()
    wx["home_team"] = team
    wx["park_name"] = park_name
    wx["park_is_dome"] = int(team in DOME_TEAMS)

    keep_cols = [
        "date", "home_team",
        "tavg", "tmin", "tmax",
        "prcp", "snow",
        "wdir", "wspd", "wpgt",
        "pres", "tsun",
        "park_name", "park_is_dome"
    ]
    keep_cols = [c for c in keep_cols if c in wx.columns]
    wx = wx[keep_cols].copy()

    parts.append(wx)
    print(f"Fetched {team}: {len(wx)} rows")

if len(parts) == 0:
    raise ValueError("No historical weather data built.")

mlb_weather_hist = pd.concat(parts, ignore_index=True)

mlb_weather_hist = mlb_weather_hist.rename(columns={
    "tavg": "weather_temp_avg_c",
    "tmin": "weather_temp_min_c",
    "tmax": "weather_temp_max_c",
    "prcp": "weather_precip_mm",
    "snow": "weather_snow_mm",
    "wdir": "weather_wind_dir_deg",
    "wspd": "weather_wind_kph",
    "wpgt": "weather_wind_gust_kph",
    "pres": "weather_pressure_hpa",
    "tsun": "weather_sunshine_min",
})

print("\nHistorical weather shape:", mlb_weather_hist.shape)
display(mlb_weather_hist.head(10))

BUILD HISTORICAL MLB WEATHER TABLE
Unique home-team/date weather keys: 8229
Skipping unknown park mapping for: American League All-Stars


Fetched Arizona Diamondbacks: 938 rows


Fetched Atlanta Braves: 940 rows


Fetched Baltimore Orioles: 938 rows


Fetched Boston Red Sox: 942 rows


...

Fetched Tampa Bay Rays: 935 rows


Fetched Texas Rangers: 938 rows
Skipping unknown park mapping for: Tokyo Yomiuri Giants


Fetched Toronto Blue Jays: 976 rows


Fetched Washington Nationals: 942 rows

Historical weather shape: (27938, 14)


,date,home_team,weather_temp_avg_c,weather_temp_min_c,weather_temp_max_c,weather_precip_mm,weather_snow_mm,weather_wind_dir_deg,weather_wind_kph,weather_wind_gust_kph,weather_pressure_hpa,weather_sunshine_min,park_name,park_is_dome
0,2023-03-03,Arizona Diamondbacks,10.8,6.2,17.3,0.0,0.0,129.0,6.1,NaN,1014.2,NaN,Chase Field,1
1,2023-03-04,Arizona Diamondbacks,12.8,7.3,21.2,0.0,0.0,99.0,10.4,NaN,1012.3,NaN,Chase Field,1
2,2023-03-05,Arizona Diamondbacks,15.2,8.4,22.3,0.0,0.0,167.0,11.2,NaN,1013.7,NaN,Chase Field,1
3,2023-03-06,Arizona Diamondbacks,16.5,11.2,23.4,0.0,0.0,341.0,7.6,NaN,1015.3,NaN,Chase Field,1
4,2023-03-07,Arizona Diamondbacks,18.0,13.4,22.9,0.0,0.0,322.0,5.8,NaN,1014.3,NaN,Chase Field,1
5,2023-03-08,Arizona Diamondbacks,16.5,9.0,24.5,0.0,0.0,112.0,8.3,NaN,1014.4,NaN,Chase Field,1
6,2023-03-09,Arizona Diamondbacks,17.1,10.7,23.4,0.0,0.0,50.0,4.7,NaN,1015.6,NaN,Chase Field,1
7,2023-03-10,Arizona Diamondbacks,18.2,11.8,26.2,0.0,0.0,74.0,7.9,NaN,1014.1,NaN,Chase Field,1
8,2023-03-11,Arizona Diamondbacks,21.4,16.2,25.7,0.0,0.0,251.0,10.8,NaN,1012.4,NaN,Chase Field,1
9,2023-03-12,Arizona Diamondbacks,19.8,13.4,27.9,0.0,0.0,38.0,12.2,NaN,1013.7,NaN,Chase Field,1


In [7]:
# %%
# =========================================================
# CELL Ã¢â‚¬â€ MERGE HISTORICAL WEATHER INTO MLB TABLE
# =========================================================

import pandas as pd
import numpy as np

print("=" * 80)
print("MERGE HISTORICAL WEATHER INTO MLB TABLE")
print("=" * 80)

if "mlb" not in globals() or not isinstance(mlb, pd.DataFrame):
    raise ValueError("mlb not found.")

if "mlb_weather_hist" not in globals() or not isinstance(mlb_weather_hist, pd.DataFrame):
    raise ValueError("mlb_weather_hist not found. Run weather builder first.")

base = mlb.copy()
base["date"] = pd.to_datetime(base["date"], errors="coerce").dt.normalize()

wx = mlb_weather_hist.copy()
wx["date"] = pd.to_datetime(wx["date"], errors="coerce").dt.normalize()

base = base.merge(
    wx,
    on=["date", "home_team"],
    how="left",
    validate="many_to_one"
)

for c in [
    "weather_temp_avg_c",
    "weather_temp_max_c",
    "weather_wind_kph",
    "weather_wind_gust_kph",
    "weather_precip_mm",
]:
    if c in base.columns:
        base[c] = pd.to_numeric(base[c], errors="coerce")

if "park_is_dome" not in base.columns:
    base["park_is_dome"] = 0

base["weather_temp_avg_f"] = base["weather_temp_avg_c"] * 9 / 5 + 32
base["weather_temp_max_f"] = base["weather_temp_max_c"] * 9 / 5 + 32
base["weather_wind_mph"] = base["weather_wind_kph"] * 0.621371
base["weather_wind_gust_mph"] = base["weather_wind_gust_kph"] * 0.621371

base["weather_is_outdoor"] = 1 - base["park_is_dome"].fillna(0)
base["weather_temp_avg_f_outdoor"] = base["weather_temp_avg_f"] * base["weather_is_outdoor"]
base["weather_wind_mph_outdoor"] = base["weather_wind_mph"] * base["weather_is_outdoor"]
base["weather_precip_mm_outdoor"] = base["weather_precip_mm"] * base["weather_is_outdoor"]

base["weather_temp_sq"] = base["weather_temp_avg_f"] ** 2
base["weather_wind_sq"] = base["weather_wind_mph"] ** 2

weather_cols = [c for c in base.columns if c.startswith("weather_")] + [c for c in ["park_is_dome"] if c in base.columns]

print("Weather null rates:")
display(base[weather_cols].isna().mean().sort_values(ascending=False))

print("\nSample weather rows:")
display(base[["date", "away_team", "home_team"] + weather_cols[:12]].head(10))

mlb = base.copy()
globals()["mlb"] = mlb.copy()

print("\nUpdated in memory: mlb")

MERGE HISTORICAL WEATHER INTO MLB TABLE


Weather null rates:


weather_wind_gust_kph         0.993803
weather_wind_gust_mph         0.993803
weather_sunshine_min          0.979465
weather_snow_mm               0.228068
weather_wind_sq               0.001458
weather_wind_dir_deg          0.001458
weather_wind_kph              0.001458
weather_pressure_hpa          0.001458
weather_wind_mph              0.001458
weather_wind_mph_outdoor      0.001458
weather_precip_mm             0.001094
weather_precip_mm_outdoor     0.001094
weather_temp_avg_c            0.000972
weather_temp_avg_f_outdoor    0.000972
weather_temp_sq               0.000972
weather_temp_avg_f            0.000972
weather_temp_max_f            0.000972
weather_temp_min_c            0.000972
weather_temp_max_c            0.000972
park_is_dome                  0.000608
weather_is_outdoor            0.000000
dtype: float64


Sample weather rows:


,date,away_team,home_team,weather_temp_avg_c,weather_temp_min_c,weather_temp_max_c,weather_precip_mm,weather_snow_mm,weather_wind_dir_deg,weather_wind_kph,weather_wind_gust_kph,weather_pressure_hpa,weather_sunshine_min,weather_temp_avg_f,weather_temp_max_f
0,2023-03-02,Arizona Diamondbacks,Los Angeles Dodgers,11.1,6.1,16.4,0.0,NaN,356.0,5.2,NaN,1018.0,NaN,51.98,61.52
1,2023-03-02,Atlanta Braves,New York Mets,6.3,3.9,11.7,0.0,0.0,345.0,19.8,NaN,1006.7,NaN,43.34,53.06
2,2023-03-02,Baltimore Orioles,Detroit Tigers,4.1,1.7,6.7,0.0,0.0,320.0,10.1,NaN,1011.8,NaN,39.38,44.06
3,2023-03-02,Chicago White Sox,Colorado Rockies,0.4,-7.1,4.0,0.0,0.0,14.0,10.0,NaN,1008.3,NaN,32.72,39.20
4,2023-03-02,Los Angeles Angels,Kansas City Royals,3.1,-0.9,6.9,0.0,0.0,44.0,16.7,NaN,1011.0,NaN,37.58,44.42
5,2023-03-02,Miami Marlins,Washington Nationals,12.3,8.9,19.4,1.3,0.0,286.0,11.2,NaN,1007.7,NaN,54.14,66.92
6,2023-03-02,Minnesota Twins,Tampa Bay Rays,23.1,20.6,26.7,0.0,0.0,198.0,12.6,NaN,1013.7,NaN,73.58,80.06
7,2023-03-02,New York Yankees,Pittsburgh Pirates,13.1,3.1,17.6,0.0,0.0,315.0,16.9,NaN,1009.8,NaN,55.58,63.68
8,2023-03-02,Oakland Athletics,Chicago Cubs,2.2,1.1,3.9,0.0,0.0,34.0,17.0,NaN,1014.5,NaN,35.96,39.02
9,2023-03-02,Philadelphia Phillies,Boston Red Sox,4.1,2.8,6.7,14.7,80.0,338.0,18.4,NaN,1005.8,0.0,39.38,44.06



Updated in memory: mlb


In [8]:
# %%
# =========================================================
# CELL Ã¢â‚¬â€ BUILD HISTORICAL BULLPEN FATIGUE FEATURES
# FULL REPLACEMENT
# Creates leakage-safe team bullpen workload / quality signals
# from game-level runs allowed history.
# =========================================================

import pandas as pd
import numpy as np

print("=" * 80)
print("BUILD HISTORICAL BULLPEN FATIGUE FEATURES")
print("=" * 80)

if "mlb" not in globals() or not isinstance(mlb, pd.DataFrame):
    raise ValueError("mlb not found. Run Cell 2 first.")

df = mlb.copy()
df["date"] = pd.to_datetime(df["date"], errors="coerce")

required_cols = ["date", "away_team", "home_team"]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

# ---------------------------------------------------------
# 0) resolve score columns
# ---------------------------------------------------------
def find_first_col(df_, options):
    lower_map = {str(c).strip().lower(): c for c in df_.columns}
    for opt in options:
        if opt.lower() in lower_map:
            return lower_map[opt.lower()]
    return None

away_score_col = find_first_col(df, [
    "away_score", "away_runs", "visiting_score", "visitor_score", "r_away"
])
home_score_col = find_first_col(df, [
    "home_score", "home_runs", "r_home"
])

if away_score_col is None or home_score_col is None:
    raise ValueError(
        "Could not find away/home final score columns. "
        "Need away_score/away_runs and home_score/home_runs equivalents."
    )

print(f"Resolved away score col: {away_score_col}")
print(f"Resolved home score col: {home_score_col}")

df[away_score_col] = pd.to_numeric(df[away_score_col], errors="coerce")
df[home_score_col] = pd.to_numeric(df[home_score_col], errors="coerce")

# ---------------------------------------------------------
# 1) estimate starter contribution if starter columns exist
# fallback to simple proxy if not
# ---------------------------------------------------------
away_starter_ra_col = None
home_starter_ra_col = None

for c in df.columns:
    lc = c.lower()
    if away_starter_ra_col is None and ("away_starter" in lc and ("runs_allowed" in lc or "ra_" in lc or lc.endswith("_ra"))):
        away_starter_ra_col = c
    if home_starter_ra_col is None and ("home_starter" in lc and ("runs_allowed" in lc or "ra_" in lc or lc.endswith("_ra"))):
        home_starter_ra_col = c

# simple fallback: assume starter allows about 65% of team runs allowed
# only used if direct starter RA columns do not exist
if away_starter_ra_col is None:
    df["away_starter_ra_proxy"] = df[home_score_col] * 0.65
    away_starter_ra_col = "away_starter_ra_proxy"

if home_starter_ra_col is None:
    df["home_starter_ra_proxy"] = df[away_score_col] * 0.65
    home_starter_ra_col = "home_starter_ra_proxy"

df[away_starter_ra_col] = pd.to_numeric(df[away_starter_ra_col], errors="coerce")
df[home_starter_ra_col] = pd.to_numeric(df[home_starter_ra_col], errors="coerce")

# bullpen runs allowed estimate = total runs allowed - starter runs allowed
# away team bullpen allows home runs scored
# home team bullpen allows away runs scored
df["away_bullpen_runs_allowed_est"] = (df[home_score_col] - df[away_starter_ra_col]).clip(lower=0)
df["home_bullpen_runs_allowed_est"] = (df[away_score_col] - df[home_starter_ra_col]).clip(lower=0)

# proxy appearance/workload indicators
df["away_bullpen_used_flag"] = (df["away_bullpen_runs_allowed_est"].notna()).astype(int)
df["home_bullpen_used_flag"] = (df["home_bullpen_runs_allowed_est"].notna()).astype(int)

# high-stress proxy = bullpen allowed 2+ runs
df["away_bullpen_high_stress_flag"] = (df["away_bullpen_runs_allowed_est"] >= 2).astype(int)
df["home_bullpen_high_stress_flag"] = (df["home_bullpen_runs_allowed_est"] >= 2).astype(int)

# ---------------------------------------------------------
# 2) long-format team-game bullpen table
# ---------------------------------------------------------
away_long = df[[
    "date", "away_team",
    "away_bullpen_runs_allowed_est",
    "away_bullpen_used_flag",
    "away_bullpen_high_stress_flag"
]].copy().rename(columns={
    "away_team": "team",
    "away_bullpen_runs_allowed_est": "bullpen_runs_allowed_est",
    "away_bullpen_used_flag": "bullpen_used_flag",
    "away_bullpen_high_stress_flag": "bullpen_high_stress_flag",
})
away_long["side"] = "away"

home_long = df[[
    "date", "home_team",
    "home_bullpen_runs_allowed_est",
    "home_bullpen_used_flag",
    "home_bullpen_high_stress_flag"
]].copy().rename(columns={
    "home_team": "team",
    "home_bullpen_runs_allowed_est": "bullpen_runs_allowed_est",
    "home_bullpen_used_flag": "bullpen_used_flag",
    "home_bullpen_high_stress_flag": "bullpen_high_stress_flag",
})
home_long["side"] = "home"

bp_long = pd.concat([away_long, home_long], ignore_index=True)
bp_long = bp_long.dropna(subset=["date", "team"]).copy()
bp_long = bp_long.sort_values(["team", "date"]).reset_index(drop=True)

# ---------------------------------------------------------
# 3) leakage-safe rolling bullpen quality + workload
# ---------------------------------------------------------
def add_rolling_features(team_df):
    team_df = team_df.sort_values("date").copy()

    s_ra = team_df["bullpen_runs_allowed_est"].shift(1)
    s_used = team_df["bullpen_used_flag"].shift(1)
    s_stress = team_df["bullpen_high_stress_flag"].shift(1)
    s_date = pd.to_datetime(team_df["date"])

    team_df["bullpen_ra_avg_3"] = s_ra.rolling(3, min_periods=1).mean()
    team_df["bullpen_ra_avg_5"] = s_ra.rolling(5, min_periods=1).mean()
    team_df["bullpen_ra_avg_10"] = s_ra.rolling(10, min_periods=1).mean()

    team_df["bullpen_used_3"] = s_used.rolling(3, min_periods=1).sum()
    team_df["bullpen_used_5"] = s_used.rolling(5, min_periods=1).sum()

    team_df["bullpen_high_stress_3"] = s_stress.rolling(3, min_periods=1).sum()
    team_df["bullpen_high_stress_5"] = s_stress.rolling(5, min_periods=1).sum()

    prev_date = s_date.shift(1)
    team_df["bullpen_days_since_game"] = (s_date - prev_date).dt.days

    # fatigue proxy: recent usage + recent stress
    team_df["bullpen_fatigue_score"] = (
        0.35 * team_df["bullpen_used_3"].fillna(0) +
        0.25 * team_df["bullpen_used_5"].fillna(0) +
        0.60 * team_df["bullpen_high_stress_3"].fillna(0) +
        0.40 * team_df["bullpen_high_stress_5"].fillna(0)
    )

    return team_df

bp_long = (
    bp_long
    .groupby("team", group_keys=False)
    .apply(add_rolling_features)
    .reset_index(drop=True)
)

print("Bullpen long table shape:", bp_long.shape)
display(bp_long.head(12))

mlb_bullpen_long = bp_long.copy()

BUILD HISTORICAL BULLPEN FATIGUE FEATURES
Resolved away score col: away_runs
Resolved home score col: home_runs


Bullpen long table shape: (16460, 15)


,date,team,bullpen_runs_allowed_est,bullpen_used_flag,bullpen_high_stress_flag,side,bullpen_ra_avg_3,bullpen_ra_avg_5,bullpen_ra_avg_10,bullpen_used_3,bullpen_used_5,bullpen_high_stress_3,bullpen_high_stress_5,bullpen_days_since_game,bullpen_fatigue_score
0,2024-07-16,American League All-Stars,1.05,1,0,home,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.00
1,2025-07-15,American League All-Stars,2.45,1,1,away,1.050000,1.050000,1.050000,1.0,1.0,0.0,0.0,364.0,0.60
2,2023-03-02,Arizona Diamondbacks,1.75,1,0,away,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.00
3,2023-03-03,Arizona Diamondbacks,2.10,1,1,home,1.750000,1.750000,1.750000,1.0,1.0,0.0,0.0,1.0,0.60
4,2023-03-04,Arizona Diamondbacks,1.75,1,0,away,1.925000,1.925000,1.925000,2.0,2.0,1.0,1.0,1.0,2.20
5,2023-03-04,Arizona Diamondbacks,1.75,1,0,home,1.866667,1.866667,1.866667,3.0,3.0,1.0,1.0,0.0,2.80
6,2023-03-05,Arizona Diamondbacks,1.40,1,0,away,1.866667,1.837500,1.837500,3.0,4.0,1.0,1.0,1.0,3.05
7,2023-03-06,Arizona Diamondbacks,3.50,1,1,away,1.633333,1.750000,1.750000,3.0,5.0,0.0,1.0,1.0,2.70
8,2023-03-07,Arizona Diamondbacks,3.15,1,1,home,2.216667,2.100000,2.041667,3.0,5.0,1.0,2.0,1.0,3.70
9,2023-03-08,Arizona Diamondbacks,0.00,1,0,home,2.683333,2.310000,2.200000,3.0,5.0,2.0,2.0,1.0,4.30


In [9]:
# %%
# =========================================================
# CELL Ã¢â‚¬â€ MERGE HISTORICAL BULLPEN FATIGUE INTO MLB TABLE
# FULL REPLACEMENT
# Fixes duplicate team/date keys (doubleheaders)
# =========================================================

import pandas as pd
import numpy as np

print("=" * 80)
print("MERGE HISTORICAL BULLPEN FATIGUE INTO MLB TABLE")
print("=" * 80)

if "mlb" not in globals() or not isinstance(mlb, pd.DataFrame):
    raise ValueError("mlb not found.")

if "mlb_bullpen_long" not in globals() or not isinstance(mlb_bullpen_long, pd.DataFrame):
    raise ValueError("mlb_bullpen_long not found. Run bullpen builder first.")

base = mlb.copy()
base["date"] = pd.to_datetime(base["date"], errors="coerce").dt.normalize()

bp = mlb_bullpen_long.copy()
bp["date"] = pd.to_datetime(bp["date"], errors="coerce").dt.normalize()

# ---------------------------------------------------------
# 1) aggregate to ONE row per team/date
# this fixes doubleheader duplicate merge keys
# ---------------------------------------------------------
agg_map = {
    "bullpen_ra_avg_3": "mean",
    "bullpen_ra_avg_5": "mean",
    "bullpen_ra_avg_10": "mean",
    "bullpen_used_3": "mean",
    "bullpen_used_5": "mean",
    "bullpen_high_stress_3": "mean",
    "bullpen_high_stress_5": "mean",
    "bullpen_days_since_game": "min",
    "bullpen_fatigue_score": "mean",
}

agg_cols = [c for c in agg_map if c in bp.columns]

bp_team_date = (
    bp.groupby(["date", "team"], as_index=False)[agg_cols]
      .agg({c: agg_map[c] for c in agg_cols})
      .sort_values(["team", "date"])
      .reset_index(drop=True)
)

print("Original bullpen long shape :", bp.shape)
print("Aggregated team/date shape  :", bp_team_date.shape)

dup_check = bp_team_date.duplicated(subset=["date", "team"]).sum()
print("Remaining duplicate [date, team] rows:", int(dup_check))

# ---------------------------------------------------------
# 2) split into away/home merge tables
# ---------------------------------------------------------
away_bp = bp_team_date.rename(columns={
    "team": "away_team",
    "bullpen_ra_avg_3": "away_bullpen_ra_avg_3",
    "bullpen_ra_avg_5": "away_bullpen_ra_avg_5",
    "bullpen_ra_avg_10": "away_bullpen_ra_avg_10",
    "bullpen_used_3": "away_bullpen_used_3",
    "bullpen_used_5": "away_bullpen_used_5",
    "bullpen_high_stress_3": "away_bullpen_high_stress_3",
    "bullpen_high_stress_5": "away_bullpen_high_stress_5",
    "bullpen_days_since_game": "away_bullpen_days_since_game",
    "bullpen_fatigue_score": "away_bullpen_fatigue_score",
}).copy()

home_bp = bp_team_date.rename(columns={
    "team": "home_team",
    "bullpen_ra_avg_3": "home_bullpen_ra_avg_3",
    "bullpen_ra_avg_5": "home_bullpen_ra_avg_5",
    "bullpen_ra_avg_10": "home_bullpen_ra_avg_10",
    "bullpen_used_3": "home_bullpen_used_3",
    "bullpen_used_5": "home_bullpen_used_5",
    "bullpen_high_stress_3": "home_bullpen_high_stress_3",
    "bullpen_high_stress_5": "home_bullpen_high_stress_5",
    "bullpen_days_since_game": "home_bullpen_days_since_game",
    "bullpen_fatigue_score": "home_bullpen_fatigue_score",
}).copy()

# keep only needed cols
away_keep = ["date", "away_team"] + [c for c in away_bp.columns if c.startswith("away_bullpen_")]
home_keep = ["date", "home_team"] + [c for c in home_bp.columns if c.startswith("home_bullpen_")]

away_bp = away_bp[away_keep].copy()
home_bp = home_bp[home_keep].copy()

# ---------------------------------------------------------
# 3) merge into base
# ---------------------------------------------------------
base = base.merge(
    away_bp,
    on=["date", "away_team"],
    how="left",
    validate="many_to_one"
)

base = base.merge(
    home_bp,
    on=["date", "home_team"],
    how="left",
    validate="many_to_one"
)

# ---------------------------------------------------------
# 4) diff features
# ---------------------------------------------------------
base["diff_bullpen_ra_avg_3"] = base["home_bullpen_ra_avg_3"] - base["away_bullpen_ra_avg_3"]
base["diff_bullpen_ra_avg_5"] = base["home_bullpen_ra_avg_5"] - base["away_bullpen_ra_avg_5"]
base["diff_bullpen_ra_avg_10"] = base["home_bullpen_ra_avg_10"] - base["away_bullpen_ra_avg_10"]

base["diff_bullpen_used_3"] = base["home_bullpen_used_3"] - base["away_bullpen_used_3"]
base["diff_bullpen_used_5"] = base["home_bullpen_used_5"] - base["away_bullpen_used_5"]

base["diff_bullpen_high_stress_3"] = base["home_bullpen_high_stress_3"] - base["away_bullpen_high_stress_3"]
base["diff_bullpen_high_stress_5"] = base["home_bullpen_high_stress_5"] - base["away_bullpen_high_stress_5"]

base["diff_bullpen_fatigue_score"] = base["home_bullpen_fatigue_score"] - base["away_bullpen_fatigue_score"]

bullpen_cols = [c for c in base.columns if "bullpen" in c.lower()]

print("\nBullpen columns added:", len(bullpen_cols))
print("\nTop bullpen null rates:")
display(base[bullpen_cols].isna().mean().sort_values(ascending=False).head(20))

show_cols = [c for c in ["date", "away_team", "home_team"] if c in base.columns] + bullpen_cols[:12]
print("\nSample bullpen rows:")
display(base[show_cols].head(10))

mlb = base.copy()
globals()["mlb"] = mlb.copy()

print("\nUpdated in memory: mlb")

MERGE HISTORICAL BULLPEN FATIGUE INTO MLB TABLE
Original bullpen long shape : (16460, 15)
Aggregated team/date shape  : (16279, 11)
Remaining duplicate [date, team] rows: 0



Bullpen columns added: 26

Top bullpen null rates:


diff_bullpen_high_stress_5      0.004131
diff_bullpen_high_stress_3      0.004131
diff_bullpen_used_5             0.004131
diff_bullpen_used_3             0.004131
diff_bullpen_ra_avg_10          0.004131
diff_bullpen_ra_avg_5           0.004131
diff_bullpen_ra_avg_3           0.004131
away_bullpen_ra_avg_3           0.003402
away_bullpen_ra_avg_10          0.003402
away_bullpen_used_3             0.003402
away_bullpen_used_5             0.003402
away_bullpen_high_stress_3      0.003402
away_bullpen_high_stress_5      0.003402
away_bullpen_days_since_game    0.003402
away_bullpen_ra_avg_5           0.003402
home_bullpen_days_since_game    0.002309
home_bullpen_used_5             0.002309
home_bullpen_high_stress_5      0.002309
home_bullpen_high_stress_3      0.002309
home_bullpen_used_3             0.002309
dtype: float64


Sample bullpen rows:


,date,away_team,home_team,away_bullpen_ra_avg_3,away_bullpen_ra_avg_5,away_bullpen_ra_avg_10,away_bullpen_used_3,away_bullpen_used_5,away_bullpen_high_stress_3,away_bullpen_high_stress_5,away_bullpen_days_since_game,away_bullpen_fatigue_score,home_bullpen_ra_avg_3,home_bullpen_ra_avg_5,home_bullpen_ra_avg_10
0,2023-03-02,Arizona Diamondbacks,Los Angeles Dodgers,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
1,2023-03-02,Atlanta Braves,New York Mets,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
2,2023-03-02,Baltimore Orioles,Detroit Tigers,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
3,2023-03-02,Chicago White Sox,Colorado Rockies,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
4,2023-03-02,Los Angeles Angels,Kansas City Royals,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
5,2023-03-02,Miami Marlins,Washington Nationals,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
6,2023-03-02,Minnesota Twins,Tampa Bay Rays,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
7,2023-03-02,New York Yankees,Pittsburgh Pirates,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.7,0.7,0.7
8,2023-03-02,Oakland Athletics,Chicago Cubs,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
9,2023-03-02,Philadelphia Phillies,Boston Red Sox,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN



Updated in memory: mlb


In [10]:
# %%
# =========================================================
# CELL Ã¢â‚¬â€ BUILD HISTORICAL STARTER QUALITY FEATURES
# FULL REPLACEMENT
# FIXED: avoids date/team suffix collisions after merge
# =========================================================

import pandas as pd
import numpy as np

print("=" * 80)
print("BUILD HISTORICAL STARTER QUALITY FEATURES")
print("=" * 80)

if "mlb" not in globals() or not isinstance(mlb, pd.DataFrame):
    raise ValueError("mlb not found.")

df = mlb.copy()
df["date"] = pd.to_datetime(df["date"], errors="coerce")

# ---------------------------------------------------------
# 0) resolve starter table
# ---------------------------------------------------------
if "mlb_starter_hist" in globals() and isinstance(mlb_starter_hist, pd.DataFrame) and len(mlb_starter_hist) > 0:
    starter_hist = mlb_starter_hist.copy()
else:
    raise ValueError("mlb_starter_hist not found. Run your historical starter builder cell first.")

starter_hist["date"] = pd.to_datetime(starter_hist["date"], errors="coerce")

# ---------------------------------------------------------
# 1) resolve score columns
# ---------------------------------------------------------
def find_first_col(df_, options):
    lower_map = {str(c).strip().lower(): c for c in df_.columns}
    for opt in options:
        if opt.lower() in lower_map:
            return lower_map[opt.lower()]
    return None

away_score_col = find_first_col(df, [
    "away_score", "away_runs", "visiting_score", "visitor_score", "r_away"
])
home_score_col = find_first_col(df, [
    "home_score", "home_runs", "r_home"
])

if away_score_col is None or home_score_col is None:
    raise ValueError("Could not resolve score columns in mlb.")

df[away_score_col] = pd.to_numeric(df[away_score_col], errors="coerce")
df[home_score_col] = pd.to_numeric(df[home_score_col], errors="coerce")

# ---------------------------------------------------------
# 2) merge ONLY starter assignment columns onto game results
# ---------------------------------------------------------
if "game_id" in df.columns and "game_id" in starter_hist.columns:
    merge_keys = ["game_id"]
    starter_assign = starter_hist[[
        "game_id",
        "away_starter_id", "away_starter_name",
        "home_starter_id", "home_starter_name"
    ]].drop_duplicates(subset=["game_id"]).copy()
else:
    merge_keys = ["date", "away_team", "home_team"]
    starter_assign = starter_hist[[
        "date", "away_team", "home_team",
        "away_starter_id", "away_starter_name",
        "home_starter_id", "home_starter_name"
    ]].drop_duplicates(subset=["date", "away_team", "home_team"]).copy()

game_with_starters = df.merge(
    starter_assign,
    on=merge_keys,
    how="left"
)

print("Merged starter rows shape:", game_with_starters.shape)
print("Key starter columns found:")
for c in ["date", "away_team", "home_team", "away_starter_id", "home_starter_id"]:
    print(f" - {c}: {c in game_with_starters.columns}")

# ---------------------------------------------------------
# 3) build long-format starter appearance table
# starter runs allowed proxy = opponent team runs
# ---------------------------------------------------------
away_app = game_with_starters[[
    "date", "away_team", away_score_col, home_score_col,
    "away_starter_id", "away_starter_name"
]].copy()

away_app["starter_id"] = pd.to_numeric(away_app["away_starter_id"], errors="coerce")
away_app["starter_name"] = away_app["away_starter_name"]
away_app["team"] = away_app["away_team"]
away_app["runs_allowed"] = pd.to_numeric(away_app[home_score_col], errors="coerce")
away_app["runs_supported"] = pd.to_numeric(away_app[away_score_col], errors="coerce")
away_app["margin"] = away_app["runs_supported"] - away_app["runs_allowed"]
away_app["starter_win"] = (away_app["margin"] > 0).astype(float)
away_app = away_app[[
    "date", "team", "starter_id", "starter_name",
    "runs_allowed", "runs_supported", "margin", "starter_win"
]].copy()

home_app = game_with_starters[[
    "date", "home_team", away_score_col, home_score_col,
    "home_starter_id", "home_starter_name"
]].copy()

home_app["starter_id"] = pd.to_numeric(home_app["home_starter_id"], errors="coerce")
home_app["starter_name"] = home_app["home_starter_name"]
home_app["team"] = home_app["home_team"]
home_app["runs_allowed"] = pd.to_numeric(home_app[away_score_col], errors="coerce")
home_app["runs_supported"] = pd.to_numeric(home_app[home_score_col], errors="coerce")
home_app["margin"] = home_app["runs_supported"] - home_app["runs_allowed"]
home_app["starter_win"] = (home_app["margin"] > 0).astype(float)
home_app = home_app[[
    "date", "team", "starter_id", "starter_name",
    "runs_allowed", "runs_supported", "margin", "starter_win"
]].copy()

starter_apps = pd.concat([away_app, home_app], ignore_index=True)
starter_apps = starter_apps.dropna(subset=["date", "starter_id"]).copy()
starter_apps = starter_apps.sort_values(["starter_id", "date"]).reset_index(drop=True)

# ---------------------------------------------------------
# 4) leakage-safe rolling starter form
# ---------------------------------------------------------
def add_starter_rolls(sdf):
    sdf = sdf.sort_values("date").copy()

    ra = sdf["runs_allowed"].shift(1)
    rs = sdf["runs_supported"].shift(1)
    mg = sdf["margin"].shift(1)
    sw = sdf["starter_win"].shift(1)
    prev_date = sdf["date"].shift(1)

    sdf["starter_ra_avg_3"] = ra.rolling(3, min_periods=1).mean()
    sdf["starter_ra_avg_5"] = ra.rolling(5, min_periods=1).mean()

    sdf["starter_rs_avg_3"] = rs.rolling(3, min_periods=1).mean()
    sdf["starter_rs_avg_5"] = rs.rolling(5, min_periods=1).mean()

    sdf["starter_margin_avg_3"] = mg.rolling(3, min_periods=1).mean()
    sdf["starter_margin_avg_5"] = mg.rolling(5, min_periods=1).mean()

    sdf["starter_win_rate_3"] = sw.rolling(3, min_periods=1).mean()
    sdf["starter_win_rate_5"] = sw.rolling(5, min_periods=1).mean()

    sdf["starter_rest_days"] = (sdf["date"] - prev_date).dt.days
    return sdf

starter_apps = (
    starter_apps
    .groupby("starter_id", group_keys=False)
    .apply(add_starter_rolls)
    .reset_index(drop=True)
)

print("Starter app table shape:", starter_apps.shape)
display(starter_apps.head(12))

mlb_starter_form_long = starter_apps.copy()

BUILD HISTORICAL STARTER QUALITY FEATURES


Merged starter rows shape: (8230, 107)
Key starter columns found:
 - date: True
 - away_team: True
 - home_team: True
 - away_starter_id: True
 - home_starter_id: True


Starter app table shape: (16446, 17)


,date,team,starter_id,starter_name,runs_allowed,runs_supported,margin,starter_win,starter_ra_avg_3,starter_ra_avg_5,starter_rs_avg_3,starter_rs_avg_5,starter_margin_avg_3,starter_margin_avg_5,starter_win_rate_3,starter_win_rate_5,starter_rest_days
0,2023-05-06,St. Louis Cardinals,425794.0,Adam Wainwright,6.0,5.0,-1.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2023-05-12,St. Louis Cardinals,425794.0,Adam Wainwright,6.0,8.0,2.0,1.0,6.000000,6.000000,5.000000,5.000000,-1.000000,-1.0,0.000000,0.000000,6.0
2,2023-05-18,St. Louis Cardinals,425794.0,Adam Wainwright,8.0,16.0,8.0,1.0,6.000000,6.000000,6.500000,6.500000,0.500000,0.5,0.500000,0.500000,6.0
3,2023-05-23,St. Louis Cardinals,425794.0,Adam Wainwright,5.0,8.0,3.0,1.0,6.666667,6.666667,9.666667,9.666667,3.000000,3.0,0.666667,0.666667,5.0
4,2023-05-29,St. Louis Cardinals,425794.0,Adam Wainwright,7.0,0.0,-7.0,0.0,6.333333,6.250000,10.666667,9.250000,4.333333,3.0,1.000000,0.750000,6.0
5,2023-06-05,St. Louis Cardinals,425794.0,Adam Wainwright,4.0,3.0,-1.0,0.0,6.666667,6.400000,8.000000,7.400000,1.333333,1.0,0.666667,0.600000,7.0
6,2023-06-11,St. Louis Cardinals,425794.0,Adam Wainwright,4.0,3.0,-1.0,0.0,5.333333,6.000000,3.666667,7.000000,-1.666667,1.0,0.333333,0.600000,6.0
7,2023-06-17,St. Louis Cardinals,425794.0,Adam Wainwright,3.0,5.0,2.0,1.0,5.000000,5.600000,2.000000,6.000000,-3.000000,0.4,0.000000,0.400000,6.0
8,2023-06-24,St. Louis Cardinals,425794.0,Adam Wainwright,9.0,1.0,-8.0,0.0,3.666667,4.600000,3.666667,3.800000,0.000000,-0.8,0.333333,0.400000,7.0
9,2023-06-29,St. Louis Cardinals,425794.0,Adam Wainwright,14.0,0.0,-14.0,0.0,5.333333,5.400000,3.000000,2.400000,-2.333333,-3.0,0.333333,0.200000,5.0


In [11]:
# %%
# =========================================================
# CELL Ã¢â‚¬â€ MERGE HISTORICAL STARTER QUALITY INTO MLB TABLE
# FULL REPLACEMENT
# =========================================================

import pandas as pd
import numpy as np

print("=" * 80)
print("MERGE HISTORICAL STARTER QUALITY INTO MLB TABLE")
print("=" * 80)

if "mlb" not in globals() or not isinstance(mlb, pd.DataFrame):
    raise ValueError("mlb not found.")

if "mlb_starter_hist" not in globals() or not isinstance(mlb_starter_hist, pd.DataFrame):
    raise ValueError("mlb_starter_hist not found.")

if "mlb_starter_form_long" not in globals() or not isinstance(mlb_starter_form_long, pd.DataFrame):
    raise ValueError("mlb_starter_form_long not found. Run starter builder first.")

base = mlb.copy()
base["date"] = pd.to_datetime(base["date"], errors="coerce")

starter_hist = mlb_starter_hist.copy()
starter_hist["date"] = pd.to_datetime(starter_hist["date"], errors="coerce")

starter_form = mlb_starter_form_long.copy()
starter_form["date"] = pd.to_datetime(starter_form["date"], errors="coerce")

# ---------------------------------------------------------
# 1) attach scheduled starters to each game
# ---------------------------------------------------------
merge_keys = [c for c in ["game_id"] if c in base.columns and c in starter_hist.columns]
if len(merge_keys) == 0:
    merge_keys = ["date", "away_team", "home_team"]

starter_assign_cols = [c for c in [
    "away_starter_id", "away_starter_name",
    "home_starter_id", "home_starter_name"
] if c in starter_hist.columns]

base = base.merge(
    starter_hist[merge_keys + starter_assign_cols].drop_duplicates(),
    on=merge_keys,
    how="left"
)

# ---------------------------------------------------------
# 2) merge away starter form
# ---------------------------------------------------------
away_form = starter_form.rename(columns={
    "starter_id": "away_starter_id",
    "starter_name": "away_starter_name_form",
    "starter_ra_avg_3": "away_starter_ra_avg_3",
    "starter_ra_avg_5": "away_starter_ra_avg_5",
    "starter_rs_avg_3": "away_starter_rs_avg_3",
    "starter_rs_avg_5": "away_starter_rs_avg_5",
    "starter_margin_avg_3": "away_starter_margin_avg_3",
    "starter_margin_avg_5": "away_starter_margin_avg_5",
    "starter_win_rate_3": "away_starter_win_rate_3",
    "starter_win_rate_5": "away_starter_win_rate_5",
    "starter_rest_days": "away_starter_rest_days",
})[[
    "date", "away_starter_id",
    "away_starter_ra_avg_3", "away_starter_ra_avg_5",
    "away_starter_rs_avg_3", "away_starter_rs_avg_5",
    "away_starter_margin_avg_3", "away_starter_margin_avg_5",
    "away_starter_win_rate_3", "away_starter_win_rate_5",
    "away_starter_rest_days"
]].copy()

home_form = starter_form.rename(columns={
    "starter_id": "home_starter_id",
    "starter_name": "home_starter_name_form",
    "starter_ra_avg_3": "home_starter_ra_avg_3",
    "starter_ra_avg_5": "home_starter_ra_avg_5",
    "starter_rs_avg_3": "home_starter_rs_avg_3",
    "starter_rs_avg_5": "home_starter_rs_avg_5",
    "starter_margin_avg_3": "home_starter_margin_avg_3",
    "starter_margin_avg_5": "home_starter_margin_avg_5",
    "starter_win_rate_3": "home_starter_win_rate_3",
    "starter_win_rate_5": "home_starter_win_rate_5",
    "starter_rest_days": "home_starter_rest_days",
})[[
    "date", "home_starter_id",
    "home_starter_ra_avg_3", "home_starter_ra_avg_5",
    "home_starter_rs_avg_3", "home_starter_rs_avg_5",
    "home_starter_margin_avg_3", "home_starter_margin_avg_5",
    "home_starter_win_rate_3", "home_starter_win_rate_5",
    "home_starter_rest_days"
]].copy()

base = base.merge(
    away_form.drop_duplicates(subset=["date", "away_starter_id"]),
    on=["date", "away_starter_id"],
    how="left"
)

base = base.merge(
    home_form.drop_duplicates(subset=["date", "home_starter_id"]),
    on=["date", "home_starter_id"],
    how="left"
)

# ---------------------------------------------------------
# 3) starter diff features
# lower RA is better, so home-away difference still works
# ---------------------------------------------------------
base["diff_starter_ra_avg_3"] = base["home_starter_ra_avg_3"] - base["away_starter_ra_avg_3"]
base["diff_starter_ra_avg_5"] = base["home_starter_ra_avg_5"] - base["away_starter_ra_avg_5"]

base["diff_starter_rs_avg_3"] = base["home_starter_rs_avg_3"] - base["away_starter_rs_avg_3"]
base["diff_starter_rs_avg_5"] = base["home_starter_rs_avg_5"] - base["away_starter_rs_avg_5"]

base["diff_starter_margin_avg_3"] = base["home_starter_margin_avg_3"] - base["away_starter_margin_avg_3"]
base["diff_starter_margin_avg_5"] = base["home_starter_margin_avg_5"] - base["away_starter_margin_avg_5"]

base["diff_starter_win_rate_3"] = base["home_starter_win_rate_3"] - base["away_starter_win_rate_3"]
base["diff_starter_win_rate_5"] = base["home_starter_win_rate_5"] - base["away_starter_win_rate_5"]

base["diff_starter_rest_days"] = base["home_starter_rest_days"] - base["away_starter_rest_days"]

starter_cols = [c for c in base.columns if "starter" in c.lower()]

print("Starter columns added:", len(starter_cols))
print("\nTop starter null rates:")
display(base[starter_cols].isna().mean().sort_values(ascending=False).head(20))

show_cols = [c for c in ["date", "away_team", "home_team", "away_starter_name", "home_starter_name"] if c in base.columns]
show_cols += starter_cols[:12]
print("\nSample starter rows:")
display(base[show_cols].head(10))

mlb = base.copy()
globals()["mlb"] = mlb.copy()

print("\nUpdated in memory: mlb")

MERGE HISTORICAL STARTER QUALITY INTO MLB TABLE


Starter columns added: 31

Top starter null rates:


diff_starter_rest_days       0.077035
diff_starter_win_rate_5      0.077035
diff_starter_win_rate_3      0.077035
diff_starter_ra_avg_3        0.077035
diff_starter_ra_avg_5        0.077035
diff_starter_rs_avg_3        0.077035
diff_starter_rs_avg_5        0.077035
diff_starter_margin_avg_3    0.077035
diff_starter_margin_avg_5    0.077035
away_starter_win_rate_5      0.051276
away_starter_rest_days       0.051276
away_starter_win_rate_3      0.051276
away_starter_margin_avg_5    0.051276
away_starter_margin_avg_3    0.051276
away_starter_rs_avg_5        0.051276
away_starter_rs_avg_3        0.051276
away_starter_ra_avg_5        0.051276
away_starter_ra_avg_3        0.051276
home_starter_win_rate_5      0.040340
home_starter_rest_days       0.040340
dtype: float64


Sample starter rows:


,date,away_team,home_team,away_starter_name,home_starter_name,away_starter_id,away_starter_name,home_starter_id,home_starter_name,away_starter_ra_avg_3,away_starter_ra_avg_5,away_starter_rs_avg_3,away_starter_rs_avg_5,away_starter_margin_avg_3,away_starter_margin_avg_5,away_starter_win_rate_3,away_starter_win_rate_5
0,2023-03-02,Arizona Diamondbacks,Los Angeles Dodgers,Ryne Nelson,Michael Grove,669194.0,Ryne Nelson,675627.0,Michael Grove,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2023-03-02,Atlanta Braves,New York Mets,Kolby Allard,Carlos Carrasco,663465.0,Kolby Allard,471911.0,Carlos Carrasco,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2023-03-02,Baltimore Orioles,Detroit Tigers,Grayson Rodriguez,Matthew Boyd,680570.0,Grayson Rodriguez,571510.0,Matthew Boyd,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2023-03-02,Chicago White Sox,Colorado Rockies,Lance Lynn,Noah Davis,458681.0,Lance Lynn,663562.0,Noah Davis,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2023-03-02,Los Angeles Angels,Kansas City Royals,Tucker Davidson,Mike Mayers,656353.0,Tucker Davidson,594577.0,Mike Mayers,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,2023-03-02,Miami Marlins,Washington Nationals,Edward Cabrera,MacKenzie Gore,665795.0,Edward Cabrera,669022.0,MacKenzie Gore,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,2023-03-02,Minnesota Twins,Tampa Bay Rays,Kenta Maeda,Zach Eflin,628317.0,Kenta Maeda,621107.0,Zach Eflin,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,2023-03-02,New York Yankees,Pittsburgh Pirates,Albert Abreu,Mitch Keller,656061.0,Albert Abreu,656605.0,Mitch Keller,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,2023-03-02,Oakland Athletics,Chicago Cubs,JP Sears,Marcus Stroman,676664.0,JP Sears,573186.0,Marcus Stroman,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,2023-03-02,Philadelphia Phillies,Boston Red Sox,Bailey Falter,Tanner Houck,663559.0,Bailey Falter,656557.0,Tanner Houck,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



Updated in memory: mlb


In [12]:
# %%
# =========================================================
# CELL 3 Ã¢â‚¬â€ BUILD FEATURE SET
# FULL REPLACEMENT
# Build a richer PRE-GAME feature pool instead of forcing
# the compressed 7-feature baseline only.
# =========================================================

if "mlb" not in globals() or not isinstance(mlb, pd.DataFrame):
    raise ValueError("mlb not found. Run Cell 2 first.")

print("=" * 70)
print("CELL 3 Ã¢â‚¬â€ BUILD FEATURE SET")
print("=" * 70)

# ---------------------------------------------------------
# 0) exclude non-feature / target columns
# ---------------------------------------------------------
exclude_cols = {
    "date",
    "season",
    "game_id",
    "away_team",
    "home_team",
    TARGET_WIN,
    TARGET_MARGIN,
    TARGET_TOTAL,
    # Ã¢â€â‚¬Ã¢â€â‚¬ Leakage: actual game outcomes, not pregame knowable Ã¢â€â‚¬Ã¢â€â‚¬
    "away_runs",
    "home_runs",
    "away_win",
    # Ã¢â€â‚¬Ã¢â€â‚¬ Raw team IDs Ã¢â‚¬â€ rolling averages already encode team quality Ã¢â€â‚¬Ã¢â€â‚¬
    "away_team_id",
    "home_team_id",
}

# ---------------------------------------------------------
# 1) keep numeric pregame-safe columns
# ---------------------------------------------------------
feature_cols = []

for c in mlb.columns:
    if c in exclude_cols:
        continue
    if pd.api.types.is_numeric_dtype(mlb[c]):
        feature_cols.append(c)

feature_cols = list(dict.fromkeys(feature_cols))

if len(feature_cols) == 0:
    raise ValueError("No usable numeric feature columns found in mlb table.")

# ---------------------------------------------------------
# 2) drop obvious leakage columns if they exist
# ---------------------------------------------------------
leakage_terms = [
    "final",
    "actual",
    "result",
    "winner",
    "score",
]

feature_cols = [
    c for c in feature_cols
    if not any(term in c.lower() for term in leakage_terms)
]

# put back safe margin/total-related rolling features if they were removed accidentally
safe_rolling_terms = [
    "margin_avg",
    "runs_for_avg",
    "runs_against_avg",
    "win_rate",
    "rest_days",
    "starter",
    "bullpen",
    "park",
    "weather",
    "temp",
    "wind",
    "elo",
]

feature_cols = [
    c for c in mlb.columns
    if (
        c not in exclude_cols
        and pd.api.types.is_numeric_dtype(mlb[c])
        and (
            c in feature_cols
            or any(term in c.lower() for term in safe_rolling_terms)
        )
    )
]

feature_cols = list(dict.fromkeys(feature_cols))

print("Total features selected:", len(feature_cols))
print("\nSelected features:")
for c in feature_cols:
    print(" -", c)

# ---------------------------------------------------------
# 3) build working model table
# ---------------------------------------------------------
keep_cols = [
    "date",
    "season",
    "game_id",
    "away_team",
    "home_team",
    TARGET_WIN,
    TARGET_MARGIN,
    TARGET_TOTAL,
]
keep_cols = [c for c in keep_cols if c in mlb.columns]

model_df = mlb[keep_cols + feature_cols].copy()

for c in feature_cols:
    model_df[c] = pd.to_numeric(model_df[c], errors="coerce")

print("\nModel table shape before target cleanup:", model_df.shape)

null_counts = model_df[feature_cols].isna().sum().sort_values(ascending=False)
print("\nTop feature null counts:")
display(null_counts.head(30))

target_cols_present = [c for c in [TARGET_WIN, TARGET_MARGIN, TARGET_TOTAL] if c in model_df.columns]
model_df = model_df.dropna(subset=target_cols_present).copy()

print("\nModel table shape after target cleanup:", model_df.shape)

print("\nSample rows:")
preview_cols = [c for c in ["date", "away_team", "home_team", TARGET_WIN] if c in model_df.columns]
preview_cols += feature_cols[:12]
display(model_df[preview_cols].head(10))

# ---------------------------------------------------------
# 4) expose in memory
# ---------------------------------------------------------
globals()["feature_cols"] = feature_cols
globals()["model_df"] = model_df.copy()
globals()["MLB_MODEL_BASELINE_TYPE"] = "wide_pregame_feature_pool"

print("\nCreated in memory:")
print(" - feature_cols")
print(" - model_df")
print(f" - MLB_MODEL_BASELINE_TYPE = {MLB_MODEL_BASELINE_TYPE}")
print("Done.")

CELL 3 Ã¢â‚¬â€ BUILD FEATURE SET
Total features selected: 115

Selected features:
 - away_runs_for_avg_3
 - away_runs_against_avg_3
 - away_margin_avg_3
 - away_win_rate_3
 - away_runs_for_avg_5
 - away_runs_against_avg_5
 - away_margin_avg_5
 - away_win_rate_5
 - away_runs_for_avg_10
 - away_runs_against_avg_10
 - away_margin_avg_10
 - away_win_rate_10
 - away_rest_days
 - home_runs_for_avg_3
 - home_runs_against_avg_3
 - home_margin_avg_3
 - home_win_rate_3
 - home_runs_for_avg_5
 - home_runs_against_avg_5
 - home_margin_avg_5
 - home_win_rate_5
 - home_runs_for_avg_10
 - home_runs_against_avg_10
 - home_margin_avg_10
 - home_win_rate_10
 - home_rest_days
 - diff_runs_for_avg_3
 - diff_runs_against_avg_3
 - diff_margin_avg_3
 - diff_win_rate_3
 - diff_runs_for_avg_5
 - diff_runs_against_avg_5
 - diff_margin_avg_5
 - diff_win_rate_5
 - diff_runs_for_avg_10
 - diff_runs_against_avg_10
 - diff_margin_avg_10
 - diff_win_rate_10
 - diff_rest_days
 - weather_temp_avg_c
 - weather_temp_min


Top feature null counts:


weather_wind_gust_kph        8179
weather_wind_gust_mph        8179
weather_sunshine_min         8061
weather_snow_mm              1877
diff_starter_rs_avg_3         634
diff_starter_ra_avg_3         634
diff_starter_ra_avg_5         634
diff_starter_rest_days        634
diff_starter_rs_avg_5         634
diff_starter_margin_avg_3     634
diff_starter_margin_avg_5     634
diff_starter_win_rate_3       634
diff_starter_win_rate_5       634
away_starter_margin_avg_5     422
away_starter_ra_avg_3         422
away_starter_ra_avg_5         422
away_starter_rs_avg_3         422
away_starter_rs_avg_5         422
away_starter_margin_avg_3     422
away_starter_rest_days        422
away_starter_win_rate_5       422
away_starter_win_rate_3       422
home_starter_rs_avg_3         332
home_starter_ra_avg_3         332
home_starter_ra_avg_5         332
home_starter_rs_avg_5         332
home_starter_margin_avg_3     332
home_starter_margin_avg_5     332
home_starter_win_rate_3       332
home_starter_w


Model table shape after target cleanup: (8230, 123)

Sample rows:


,date,away_team,home_team,home_win,away_runs_for_avg_3,away_runs_against_avg_3,away_margin_avg_3,away_win_rate_3,away_runs_for_avg_5,away_runs_against_avg_5,away_margin_avg_5,away_win_rate_5,away_runs_for_avg_10,away_runs_against_avg_10,away_margin_avg_10,away_win_rate_10
0,2023-03-02,Arizona Diamondbacks,Los Angeles Dodgers,0,8.0,5.0,3.0,1.0,8.0,5.0,3.0,1.0,8.0,5.0,3.0,1.0
1,2023-03-02,Atlanta Braves,New York Mets,0,3.0,3.0,0.0,0.0,3.0,3.0,0.0,0.0,3.0,3.0,0.0,0.0
2,2023-03-02,Baltimore Orioles,Detroit Tigers,1,1.0,2.0,-1.0,0.0,1.0,2.0,-1.0,0.0,1.0,2.0,-1.0,0.0
3,2023-03-02,Chicago White Sox,Colorado Rockies,0,6.0,9.0,-3.0,0.0,6.0,9.0,-3.0,0.0,6.0,9.0,-3.0,0.0
4,2023-03-02,Los Angeles Angels,Kansas City Royals,1,4.0,3.0,1.0,1.0,4.0,3.0,1.0,1.0,4.0,3.0,1.0,1.0
5,2023-03-02,Miami Marlins,Washington Nationals,0,4.0,8.0,-4.0,0.0,4.0,8.0,-4.0,0.0,4.0,8.0,-4.0,0.0
6,2023-03-02,Minnesota Twins,Tampa Bay Rays,0,4.0,4.0,0.0,0.0,4.0,4.0,0.0,0.0,4.0,4.0,0.0,0.0
7,2023-03-02,New York Yankees,Pittsburgh Pirates,0,4.0,2.0,2.0,1.0,4.0,2.0,2.0,1.0,4.0,2.0,2.0,1.0
8,2023-03-02,Oakland Athletics,Chicago Cubs,1,14.0,14.0,0.0,0.0,14.0,14.0,0.0,0.0,14.0,14.0,0.0,0.0
9,2023-03-02,Philadelphia Phillies,Boston Red Sox,1,4.0,4.0,0.0,0.0,4.0,4.0,0.0,0.0,4.0,4.0,0.0,0.0



Created in memory:
 - feature_cols
 - model_df
 - MLB_MODEL_BASELINE_TYPE = wide_pregame_feature_pool
Done.


In [13]:
# %%
# =========================================================
# CELL 4 Ã¢â‚¬â€ FINALIZE TRAINING TABLE
# Keep more rows and defer missing-value handling to model
# =========================================================

if "model_df" not in globals() or not isinstance(model_df, pd.DataFrame):
    raise ValueError("model_df not found. Run Cell 3 first.")

if "feature_cols" not in globals() or not isinstance(feature_cols, list) or len(feature_cols) == 0:
    raise ValueError("feature_cols not found. Run Cell 3 first.")

train_df = model_df.copy()

train_df["date"] = pd.to_datetime(train_df["date"], errors="coerce")

sort_cols = ["date"]
if "game_id" in train_df.columns:
    sort_cols.append("game_id")

train_df = train_df.sort_values(sort_cols).reset_index(drop=True)

# numeric coercion
for c in feature_cols:
    train_df[c] = pd.to_numeric(train_df[c], errors="coerce")

train_df[TARGET_WIN] = pd.to_numeric(train_df[TARGET_WIN], errors="coerce")

before_rows = len(train_df)

# only require usable date + target
train_df = train_df.dropna(subset=["date", TARGET_WIN]).copy().reset_index(drop=True)

# remove fully empty feature rows
all_null_feature_rows = train_df[feature_cols].isna().all(axis=1).sum()
train_df = train_df[~train_df[feature_cols].isna().all(axis=1)].copy().reset_index(drop=True)

X = train_df[feature_cols].copy()
y = train_df[TARGET_WIN].astype(int).copy()

print("Rows before cleanup            :", before_rows)
print("Rows after cleanup             :", len(train_df))
print("Rows removed                   :", before_rows - len(train_df))
print("Rows with all-null features rm :", int(all_null_feature_rows))

print("\nFINAL TRAINING TABLE READY")
print("train_df shape:", train_df.shape)
print("X shape       :", X.shape)
print("y shape       :", y.shape)

print("\nTarget distribution:")
print(y.value_counts(dropna=False).sort_index())

print("\nTarget mean:", round(y.mean(), 4))

print("\nFeature count:", len(feature_cols))

print("\nSample training rows:")
preview_cols = [c for c in ["date", "away_team", "home_team", TARGET_WIN] if c in train_df.columns]
preview_cols += feature_cols[:10]
display(train_df[preview_cols].head(10))

Rows before cleanup            : 8230
Rows after cleanup             : 8230
Rows removed                   : 0
Rows with all-null features rm : 0

FINAL TRAINING TABLE READY
train_df shape: (8230, 123)
X shape       : (8230, 115)
y shape       : (8230,)

Target distribution:
home_win
0    3891
1    4339
Name: count, dtype: int64

Target mean: 0.5272

Feature count: 115

Sample training rows:


,date,away_team,home_team,home_win,away_runs_for_avg_3,away_runs_against_avg_3,away_margin_avg_3,away_win_rate_3,away_runs_for_avg_5,away_runs_against_avg_5,away_margin_avg_5,away_win_rate_5,away_runs_for_avg_10,away_runs_against_avg_10
0,2023-03-02,San Francisco Giants,Cleveland Guardians,1,5.0,8.0,-3.0,0.0,5.0,8.0,-3.0,0.0,5.0,8.0
1,2023-03-02,Arizona Diamondbacks,Los Angeles Dodgers,0,8.0,5.0,3.0,1.0,8.0,5.0,3.0,1.0,8.0,5.0
2,2023-03-02,San Diego Padres,Seattle Mariners,1,5.0,1.0,4.0,1.0,5.0,1.0,4.0,1.0,5.0,1.0
3,2023-03-02,Los Angeles Angels,Kansas City Royals,1,4.0,3.0,1.0,1.0,4.0,3.0,1.0,1.0,4.0,3.0
4,2023-03-02,Chicago White Sox,Colorado Rockies,0,6.0,9.0,-3.0,0.0,6.0,9.0,-3.0,0.0,6.0,9.0
5,2023-03-02,Texas Rangers,Milwaukee Brewers,0,1.5,4.5,-3.0,0.0,1.5,4.5,-3.0,0.0,1.5,4.5
6,2023-03-02,Minnesota Twins,Tampa Bay Rays,0,4.0,4.0,0.0,0.0,4.0,4.0,0.0,0.0,4.0,4.0
7,2023-03-02,Miami Marlins,Washington Nationals,0,4.0,8.0,-4.0,0.0,4.0,8.0,-4.0,0.0,4.0,8.0
8,2023-03-02,Pittsburgh Pirates,Toronto Blue Jays,0,7.0,8.0,-1.0,0.0,7.0,8.0,-1.0,0.0,7.0,8.0
9,2023-03-02,New York Yankees,Pittsburgh Pirates,0,4.0,2.0,2.0,1.0,4.0,2.0,2.0,1.0,4.0,2.0


In [14]:
# %%
# =========================================================
# CELL 5 Ã¢â‚¬â€ BUILD WALK-FORWARD BACKTEST WINDOWS
# FULL REPLACEMENT
# Monthly rolling test windows after BACKTEST_START
# =========================================================
import pandas as pd
import numpy as np

print("=" * 70)
print("CELL 5 Ã¢â‚¬â€ BUILD WALK-FORWARD BACKTEST WINDOWS")
print("=" * 70)

# ---------------------------------------------------------
# 0) validate inputs
# ---------------------------------------------------------
if "train_df" not in globals() or not isinstance(train_df, pd.DataFrame) or len(train_df) == 0:
    raise ValueError("train_df not found or empty. Run the earlier training-table cell first.")

if "feature_cols" not in globals() or not isinstance(feature_cols, list) or len(feature_cols) == 0:
    raise ValueError("feature_cols not found or empty. Run the feature-building cell first.")

wf_df = train_df.copy()

if "date" not in wf_df.columns:
    raise ValueError("train_df must contain a 'date' column.")

wf_df["date"] = pd.to_datetime(wf_df["date"], errors="coerce")
bad_dates = int(wf_df["date"].isna().sum())
if bad_dates > 0:
    print(f"Dropping rows with invalid dates: {bad_dates:,}")
    wf_df = wf_df[wf_df["date"].notna()].copy()

if len(wf_df) == 0:
    raise ValueError("No usable rows remain after date parsing.")

sort_cols = [c for c in ["date", "game_id"] if c in wf_df.columns]
wf_df = wf_df.sort_values(sort_cols).reset_index(drop=True)

date_min = wf_df["date"].min().normalize()
date_max = wf_df["date"].max().normalize()

print("Initial source shape:", wf_df.shape)
print("Initial date range  :", date_min, "->", date_max)

# ---------------------------------------------------------
# 1) resolve TRAIN_START
# ---------------------------------------------------------
resolved_train_start = None

if "TRAIN_START" in globals():
    try:
        resolved_train_start = pd.Timestamp(TRAIN_START).normalize()
        print(f"Using existing TRAIN_START: {resolved_train_start}")
    except Exception as e:
        print(f"Existing TRAIN_START could not be parsed: {e}")

if resolved_train_start is None:
    # default: start from earliest available date
    resolved_train_start = date_min
    print(f"TRAIN_START not found. Auto-setting TRAIN_START to earliest date: {resolved_train_start}")

# enforce range sanity
if resolved_train_start < date_min:
    print(f"TRAIN_START {resolved_train_start} is before available data. Clipping to {date_min}.")
    resolved_train_start = date_min

if resolved_train_start > date_max:
    raise ValueError(
        f"TRAIN_START ({resolved_train_start}) is after the max date in data ({date_max})."
    )

# ---------------------------------------------------------
# 2) restrict to usable date range
# ---------------------------------------------------------
wf_df = wf_df[wf_df["date"] >= resolved_train_start].copy()

if len(wf_df) == 0:
    raise ValueError("No rows left after TRAIN_START filter.")

wf_df = wf_df.sort_values(sort_cols).reset_index(drop=True)

date_min = wf_df["date"].min().normalize()
date_max = wf_df["date"].max().normalize()

print("")
print("Post-TRAIN_START shape:", wf_df.shape)
print("Post-TRAIN_START range:", date_min, "->", date_max)

# ---------------------------------------------------------
# 3) resolve BACKTEST_START
# ---------------------------------------------------------
resolved_backtest_start = None

if "BACKTEST_START" in globals():
    try:
        resolved_backtest_start = pd.Timestamp(BACKTEST_START).normalize()
        print(f"Using existing BACKTEST_START: {resolved_backtest_start}")
    except Exception as e:
        print(f"Existing BACKTEST_START could not be parsed: {e}")

if resolved_backtest_start is None:
    # sensible default for MLB historical work:
    # start backtest after a substantial training runway
    unique_months = pd.Series(
        pd.to_datetime(wf_df["date"]).dt.to_period("M").astype(str)
    ).dropna().unique()

    if len(unique_months) >= 18:
        # roughly 12 months of runway before first test month
        first_test_candidate = (date_min + pd.DateOffset(months=12)).replace(day=1)
    elif len(unique_months) >= 12:
        # smaller dataset fallback
        first_test_candidate = (date_min + pd.DateOffset(months=6)).replace(day=1)
    else:
        # very short sample fallback
        first_test_candidate = (date_min + pd.DateOffset(months=3)).replace(day=1)

    resolved_backtest_start = pd.Timestamp(first_test_candidate).normalize()
    print(f"BACKTEST_START not found. Auto-setting BACKTEST_START to: {resolved_backtest_start}")

# keep backtest start inside range
if resolved_backtest_start < date_min:
    print(f"BACKTEST_START {resolved_backtest_start} is before available range. Clipping to {date_min}.")
    resolved_backtest_start = date_min

if resolved_backtest_start > date_max:
    raise ValueError(
        f"BACKTEST_START ({resolved_backtest_start}) is after the max date in data ({date_max})."
    )

# force first-of-month for monthly windows
resolved_backtest_start = resolved_backtest_start.replace(day=1)

# ---------------------------------------------------------
# 4) create monthly test window starts
# ---------------------------------------------------------
month_starts = pd.date_range(
    start=resolved_backtest_start,
    end=date_max,
    freq="MS"
)

print("")
print(f"Candidate monthly test windows: {len(month_starts):,}")
print(f"Window start anchor           : {resolved_backtest_start}")

windows = []

MIN_TRAIN_ROWS = 1000
MIN_TEST_ROWS = 50

for test_start in month_starts:
    test_end = test_start + pd.offsets.MonthBegin(1)

    train_mask = wf_df["date"] < test_start
    test_mask = (wf_df["date"] >= test_start) & (wf_df["date"] < test_end)

    n_train = int(train_mask.sum())
    n_test = int(test_mask.sum())

    if n_train < MIN_TRAIN_ROWS or n_test < MIN_TEST_ROWS:
        continue

    windows.append({
        "test_start": pd.Timestamp(test_start),
        "test_end": pd.Timestamp(test_end),
        "n_train": n_train,
        "n_test": n_test,
    })

wf_windows = pd.DataFrame(windows)

# ---------------------------------------------------------
# 5) fallback if strict thresholds produce nothing
# ---------------------------------------------------------
if len(wf_windows) == 0:
    print("")
    print("No valid windows with strict thresholds.")
    print("Trying relaxed thresholds...")

    relaxed_windows = []
    RELAXED_MIN_TRAIN_ROWS = 500
    RELAXED_MIN_TEST_ROWS = 20

    for test_start in month_starts:
        test_end = test_start + pd.offsets.MonthBegin(1)

        train_mask = wf_df["date"] < test_start
        test_mask = (wf_df["date"] >= test_start) & (wf_df["date"] < test_end)

        n_train = int(train_mask.sum())
        n_test = int(test_mask.sum())

        if n_train < RELAXED_MIN_TRAIN_ROWS or n_test < RELAXED_MIN_TEST_ROWS:
            continue

        relaxed_windows.append({
            "test_start": pd.Timestamp(test_start),
            "test_end": pd.Timestamp(test_end),
            "n_train": n_train,
            "n_test": n_test,
        })

    wf_windows = pd.DataFrame(relaxed_windows)

    if len(wf_windows) > 0:
        print(
            f"Using relaxed thresholds: MIN_TRAIN_ROWS={RELAXED_MIN_TRAIN_ROWS}, "
            f"MIN_TEST_ROWS={RELAXED_MIN_TEST_ROWS}"
        )

# ---------------------------------------------------------
# 6) final checks / output
# ---------------------------------------------------------
if len(wf_windows) == 0:
    raise ValueError(
        "No valid walk-forward windows were created.\n"
        f"Resolved TRAIN_START: {resolved_train_start}\n"
        f"Resolved BACKTEST_START: {resolved_backtest_start}\n"
        "Try an earlier BACKTEST_START or inspect date coverage in train_df."
    )

print("")
print("Resolved settings:")
print(f" - TRAIN_START    : {resolved_train_start}")
print(f" - BACKTEST_START : {resolved_backtest_start}")
print(f" - Windows created: {len(wf_windows):,}")

print("")
display(wf_windows)

# ---------------------------------------------------------
# 7) save for later cells
# ---------------------------------------------------------
mlb_wf_source = wf_df.copy()
mlb_wf_windows = wf_windows.copy()

globals()["TRAIN_START"] = resolved_train_start
globals()["BACKTEST_START"] = resolved_backtest_start
globals()["mlb_wf_source"] = mlb_wf_source
globals()["mlb_wf_windows"] = mlb_wf_windows

print("")
print("Created in memory:")
print(" - mlb_wf_source")
print(" - mlb_wf_windows")
print(f" - TRAIN_START = {resolved_train_start}")
print(f" - BACKTEST_START = {resolved_backtest_start}")
print("Done.")

CELL 5 Ã¢â‚¬â€ BUILD WALK-FORWARD BACKTEST WINDOWS
Initial source shape: (8230, 123)
Initial date range  : 2023-03-02 00:00:00 -> 2025-11-01 00:00:00
TRAIN_START not found. Auto-setting TRAIN_START to earliest date: 2023-03-02 00:00:00

Post-TRAIN_START shape: (8230, 123)
Post-TRAIN_START range: 2023-03-02 00:00:00 -> 2025-11-01 00:00:00
BACKTEST_START not found. Auto-setting BACKTEST_START to: 2024-03-01 00:00:00

Candidate monthly test windows: 21
Window start anchor           : 2024-03-01 00:00:00



Resolved settings:
 - TRAIN_START    : 2023-03-02 00:00:00
 - BACKTEST_START : 2024-03-01 00:00:00
 - Windows created: 14



,test_start,test_end,n_train,n_test
0,2024-03-01,2024-04-01,2818,412
1,2024-04-01,2024-05-01,3230,390
2,2024-05-01,2024-06-01,3620,403
3,2024-06-01,2024-07-01,4023,398
4,2024-07-01,2024-08-01,4421,364
5,2024-08-01,2024-09-01,4785,406
6,2024-09-01,2024-10-01,5191,382
7,2025-03-01,2025-04-01,5616,391
8,2025-04-01,2025-05-01,6007,358
9,2025-05-01,2025-06-01,6365,375



Created in memory:
 - mlb_wf_source
 - mlb_wf_windows
 - TRAIN_START = 2023-03-02 00:00:00
 - BACKTEST_START = 2024-03-01 00:00:00
Done.


In [15]:
# %%
# =========================================================
# CELL 6 Ã¢â‚¬â€ RUN WALK-FORWARD WIN MODEL BACKTEST
# FULL REPLACEMENT
# Compare multiple models and choose best by log loss
# =========================================================

import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, log_loss, brier_score_loss

if "mlb_wf_source" not in globals() or not isinstance(mlb_wf_source, pd.DataFrame):
    raise ValueError("mlb_wf_source not found. Run Cell 5 first.")

if "mlb_wf_windows" not in globals() or not isinstance(mlb_wf_windows, pd.DataFrame):
    raise ValueError("mlb_wf_windows not found. Run Cell 5 first.")

wf_df = mlb_wf_source.copy()
wf_windows = mlb_wf_windows.copy()

candidate_models = {
    "logit_l2_c1": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            max_iter=5000,
            C=1.0,
            random_state=42
        ))
    ]),
    "logit_l2_c0.3": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            max_iter=5000,
            C=0.3,
            random_state=42
        ))
    ]),
    "rf_300": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestClassifier(
            n_estimators=300,
            max_depth=6,
            min_samples_leaf=10,
            random_state=42,
            n_jobs=-1
        ))
    ]),
    "hgb_200": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", HistGradientBoostingClassifier(
            max_iter=200,
            learning_rate=0.05,
            max_depth=4,
            min_samples_leaf=20,
            random_state=42
        ))
    ]),
}

all_model_metrics = []
all_model_preds = []

for model_name, model in candidate_models.items():
    print(f"\nRunning walk-forward for: {model_name}")

    metric_rows = []
    pred_rows = []

    for _, w in wf_windows.iterrows():
        test_start = pd.to_datetime(w["test_start"])
        test_end = pd.to_datetime(w["test_end"])

        train_set = wf_df[wf_df["date"] < test_start].copy()
        test_set = wf_df[(wf_df["date"] >= test_start) & (wf_df["date"] < test_end)].copy()

        if len(train_set) == 0 or len(test_set) == 0:
            continue

        X_train = train_set[feature_cols].copy()
        y_train = train_set[TARGET_WIN].astype(int).copy()

        X_test = test_set[feature_cols].copy()
        y_test = test_set[TARGET_WIN].astype(int).copy()

        model.fit(X_train, y_train)

        p_test = model.predict_proba(X_test)[:, 1]
        p_test = np.clip(p_test, 1e-6, 1 - 1e-6)
        y_pred = (p_test >= 0.5).astype(int)

        acc = accuracy_score(y_test, y_pred)
        ll = log_loss(y_test, p_test)
        brier = brier_score_loss(y_test, p_test)

        metric_rows.append({
            "model_name": model_name,
            "test_start": test_start,
            "test_end": test_end,
            "n_train": len(train_set),
            "n_test": len(test_set),
            "accuracy": acc,
            "log_loss": ll,
            "brier": brier,
        })

        pred_block = test_set[["date", "away_team", "home_team", TARGET_WIN]].copy()
        if "game_id" in test_set.columns:
            pred_block["game_id"] = test_set["game_id"].values

        pred_block["model_name"] = model_name
        pred_block["p_home_win"] = p_test
        pred_block["pred_home_win"] = y_pred
        pred_block["pred_winner"] = np.where(
            pred_block["pred_home_win"] == 1,
            pred_block["home_team"],
            pred_block["away_team"]
        )
        pred_block["test_start"] = test_start

        pred_rows.append(pred_block)

    if len(metric_rows) > 0:
        all_model_metrics.append(pd.DataFrame(metric_rows))
    if len(pred_rows) > 0:
        all_model_preds.append(pd.concat(pred_rows, ignore_index=True))

if len(all_model_metrics) == 0:
    raise ValueError("No walk-forward results generated.")

wf_model_metrics = pd.concat(all_model_metrics, ignore_index=True)
wf_model_summary = (
    wf_model_metrics
    .groupby("model_name", as_index=False)
    .agg(
        windows=("test_start", "count"),
        accuracy=("accuracy", "mean"),
        log_loss=("log_loss", "mean"),
        brier=("brier", "mean"),
        avg_n_test=("n_test", "mean"),
    )
    .sort_values(["log_loss", "brier", "accuracy"], ascending=[True, True, False])
    .reset_index(drop=True)
)

best_model_name = wf_model_summary.iloc[0]["model_name"]

wf_preds_all = pd.concat(all_model_preds, ignore_index=True)
wf_preds_best = wf_preds_all[wf_preds_all["model_name"] == best_model_name].copy().reset_index(drop=True)

print("\nMODEL COMPARISON SUMMARY")
display(wf_model_summary)

print(f"\nBest model selected: {best_model_name}")

print("\nBest-model prediction sample:")
display(wf_preds_best.head(15))

mlb_wf_model_metrics = wf_model_metrics.copy()
mlb_wf_model_summary = wf_model_summary.copy()
mlb_best_model_name = best_model_name
mlb_wf_preds = wf_preds_best.copy()


Running walk-forward for: logit_l2_c1



Running walk-forward for: logit_l2_c0.3



Running walk-forward for: rf_300



Running walk-forward for: hgb_200



MODEL COMPARISON SUMMARY


,model_name,windows,accuracy,log_loss,brier,avg_n_test
0,rf_300,14,0.535291,0.688546,0.247710,380.428571
1,hgb_200,14,0.525083,0.701771,0.253765,380.428571
2,logit_l2_c0.3,14,0.534618,0.717832,0.252424,380.428571
3,logit_l2_c1,14,0.535632,0.718393,0.252695,380.428571



Best model selected: rf_300

Best-model prediction sample:


,date,away_team,home_team,home_win,game_id,model_name,p_home_win,pred_home_win,pred_winner,test_start
0,2024-03-01,Texas Rangers,San Francisco Giants,1,748151,rf_300,0.479814,0,Texas Rangers,2024-03-01
1,2024-03-01,Seattle Mariners,Colorado Rockies,1,748152,rf_300,0.506962,1,Colorado Rockies,2024-03-01
2,2024-03-01,Chicago White Sox,Chicago Cubs,1,748153,rf_300,0.506421,1,Chicago Cubs,2024-03-01
3,2024-03-01,Oakland Athletics,Kansas City Royals,1,748154,rf_300,0.565033,1,Kansas City Royals,2024-03-01
4,2024-03-01,San Diego Padres,Milwaukee Brewers,1,748155,rf_300,0.456288,0,San Diego Padres,2024-03-01
5,2024-03-01,Arizona Diamondbacks,Cincinnati Reds,0,748156,rf_300,0.562769,1,Cincinnati Reds,2024-03-01
6,2024-03-01,Cleveland Guardians,Los Angeles Dodgers,1,748157,rf_300,0.552085,1,Los Angeles Dodgers,2024-03-01
7,2024-03-01,Los Angeles Angels,San Diego Padres,0,748158,rf_300,0.493501,0,Los Angeles Angels,2024-03-01
8,2024-03-01,Miami Marlins,Philadelphia Phillies,0,748159,rf_300,0.549255,1,Philadelphia Phillies,2024-03-01
9,2024-03-01,Philadelphia Phillies,Detroit Tigers,1,748160,rf_300,0.533055,1,Detroit Tigers,2024-03-01


In [16]:
# %%
# =========================================================
# CELL 7 Ã¢â‚¬â€ FIT CALIBRATION LAYER ON BEST WALK-FORWARD PREDICTIONS
# FULL REPLACEMENT
# =========================================================

import numpy as np
import pandas as pd
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import log_loss, brier_score_loss

if "mlb_wf_preds" not in globals() or not isinstance(mlb_wf_preds, pd.DataFrame):
    raise ValueError("mlb_wf_preds not found. Run Cell 6 first.")

cal_df = mlb_wf_preds.copy()

if "p_home_win" not in cal_df.columns:
    raise ValueError("p_home_win missing from mlb_wf_preds.")

if TARGET_WIN not in cal_df.columns:
    raise ValueError(f"{TARGET_WIN} missing from mlb_wf_preds.")

y_cal = cal_df[TARGET_WIN].astype(int).values
p_raw = cal_df["p_home_win"].astype(float).values
p_raw = np.clip(p_raw, 1e-6, 1 - 1e-6)

raw_logloss = log_loss(y_cal, p_raw)
raw_brier = brier_score_loss(y_cal, p_raw)

iso_calibrator = IsotonicRegression(out_of_bounds="clip")
iso_calibrator.fit(p_raw, y_cal)

p_cal = iso_calibrator.predict(p_raw)
p_cal = np.clip(p_cal, 1e-6, 1 - 1e-6)

cal_logloss = log_loss(y_cal, p_cal)
cal_brier = brier_score_loss(y_cal, p_cal)

cal_df["p_home_win_raw"] = p_raw
cal_df["p_home_win_cal"] = p_cal
cal_df["p_away_win_cal"] = 1 - cal_df["p_home_win_cal"]

cal_df["pred_home_win_cal"] = (cal_df["p_home_win_cal"] >= 0.5).astype(int)
cal_df["pred_winner_cal"] = np.where(
    cal_df["pred_home_win_cal"] == 1,
    cal_df["home_team"],
    cal_df["away_team"]
)

print("CALIBRATION RESULTS")
print("-" * 60)
print(f"Raw Log Loss        : {raw_logloss:.4f}")
print(f"Calibrated Log Loss : {cal_logloss:.4f}")
print(f"Raw Brier Score     : {raw_brier:.4f}")
print(f"Calibrated Brier    : {cal_brier:.4f}")
print("")
print(f"Log Loss Improvement : {raw_logloss - cal_logloss:.6f}")
print(f"Brier Improvement    : {raw_brier - cal_brier:.6f}")

display(
    cal_df[
        [
            "date", "away_team", "home_team", TARGET_WIN,
            "p_home_win_raw", "p_home_win_cal",
            "pred_winner", "pred_winner_cal"
        ]
    ].head(15)
)

mlb_calibration_df = cal_df.copy()
mlb_iso_calibrator = iso_calibrator

CALIBRATION RESULTS
------------------------------------------------------------
Raw Log Loss        : 0.6886
Calibrated Log Loss : 0.6862
Raw Brier Score     : 0.2477
Calibrated Brier    : 0.2466

Log Loss Improvement : 0.002435
Brier Improvement    : 0.001136


,date,away_team,home_team,home_win,p_home_win_raw,p_home_win_cal,pred_winner,pred_winner_cal
0,2024-03-01,Texas Rangers,San Francisco Giants,1,0.479814,0.505102,Texas Rangers,San Francisco Giants
1,2024-03-01,Seattle Mariners,Colorado Rockies,1,0.506962,0.519685,Colorado Rockies,Colorado Rockies
2,2024-03-01,Chicago White Sox,Chicago Cubs,1,0.506421,0.519685,Chicago Cubs,Chicago Cubs
3,2024-03-01,Oakland Athletics,Kansas City Royals,1,0.565033,0.585613,Kansas City Royals,Kansas City Royals
4,2024-03-01,San Diego Padres,Milwaukee Brewers,1,0.456288,0.420561,San Diego Padres,San Diego Padres
5,2024-03-01,Arizona Diamondbacks,Cincinnati Reds,0,0.562769,0.585613,Cincinnati Reds,Cincinnati Reds
6,2024-03-01,Cleveland Guardians,Los Angeles Dodgers,1,0.552085,0.585613,Los Angeles Dodgers,Los Angeles Dodgers
7,2024-03-01,Los Angeles Angels,San Diego Padres,0,0.493501,0.519685,Los Angeles Angels,San Diego Padres
8,2024-03-01,Miami Marlins,Philadelphia Phillies,0,0.549255,0.550035,Philadelphia Phillies,Philadelphia Phillies
9,2024-03-01,Philadelphia Phillies,Detroit Tigers,1,0.533055,0.550035,Detroit Tigers,Detroit Tigers


In [17]:
# %%
# =========================================================
# CELL 8 Ã¢â‚¬â€ TRAIN FINAL PRODUCTION WIN MODEL + SAVE MODEL
# FULL REPLACEMENT
# - Uses chosen best model family
# - Uses all resolved features
# - Saves calibrator for production use
# =========================================================

from pathlib import Path
import json
import joblib
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier

print("=" * 70)
print("CELL 8 Ã¢â‚¬â€ TRAIN FINAL PRODUCTION WIN MODEL + SAVE MODEL")
print("=" * 70)

if "train_df" not in globals() or not isinstance(train_df, pd.DataFrame) or len(train_df) == 0:
    raise ValueError("train_df not found or empty. Run Cell 4 first.")

if "feature_cols" not in globals() or not isinstance(feature_cols, list) or len(feature_cols) == 0:
    raise ValueError("feature_cols not found. Run Cell 3 first.")

if "mlb_best_model_name" not in globals():
    raise ValueError("mlb_best_model_name not found. Run Cell 6 first.")

final_train = train_df.copy()
final_train["date"] = pd.to_datetime(final_train["date"], errors="coerce")

sort_cols = [c for c in ["date", "game_id"] if c in final_train.columns]
if sort_cols:
    final_train = final_train.sort_values(sort_cols).reset_index(drop=True)

for c in feature_cols:
    final_train[c] = pd.to_numeric(final_train[c], errors="coerce")

final_train[TARGET_WIN] = pd.to_numeric(final_train[TARGET_WIN], errors="coerce")
final_train = final_train.dropna(subset=["date", TARGET_WIN]).copy()
final_train = final_train[~final_train[feature_cols].isna().all(axis=1)].copy()

X_final = final_train[feature_cols].copy()
y_final = final_train[TARGET_WIN].astype(int).copy()

# Temporal decay weights Ã¢â‚¬â€ recent games weighted more (half-life ~60 days)
import numpy as _np_td
_today_td = pd.Timestamp.today()
_days_ago = (_today_td - pd.to_datetime(final_train["date"], errors="coerce")).dt.days.fillna(180).clip(lower=0)
_sample_weights = _np_td.exp(-0.0116 * _days_ago)  # ln(2)/60 Ã¢â€°Ë† 0.0116
print(f"Temporal decay: weights range {_sample_weights.min():.3f}Ã¢â‚¬â€œ{_sample_weights.max():.3f} over {len(_sample_weights)} training rows")

resolved_model_dir = Path(MODEL_DIR) if "MODEL_DIR" in globals() else Path("models/mlb")
resolved_model_dir.mkdir(parents=True, exist_ok=True)

candidate_models = {
    "logit_l2_c1": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            max_iter=5000,
            C=1.0,
            random_state=42
        ))
    ]),
    "logit_l2_c0.3": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            max_iter=5000,
            C=0.3,
            random_state=42
        ))
    ]),
    "rf_300": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestClassifier(
            n_estimators=300,
            max_depth=6,
            min_samples_leaf=10,
            random_state=42,
            n_jobs=-1
        ))
    ]),
    "hgb_200": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", HistGradientBoostingClassifier(
            max_iter=200,
            learning_rate=0.05,
            max_depth=4,
            min_samples_leaf=20,
            random_state=42
        ))
    ]),
}

if mlb_best_model_name not in candidate_models:
    raise ValueError(f"Unknown best model name: {mlb_best_model_name}")

final_win_model = candidate_models[mlb_best_model_name]
# Pass sample weights through Pipeline (model__ prefix for the last step)
try:
    final_win_model.fit(X_final, y_final, model__sample_weight=_sample_weights.values)
    print("Temporal decay weights applied to training.")
except TypeError:
    final_win_model.fit(X_final, y_final)
    print("Temporal decay: model does not support sample_weight, trained without.")

win_model_file = resolved_model_dir / "mlb_win_model_best.joblib"
calibrator_file = resolved_model_dir / "mlb_win_model_best_calibrator.joblib"
meta_file = resolved_model_dir / "mlb_win_model_best_meta.json"

joblib.dump(final_win_model, win_model_file)

calibrator_saved = False
if "mlb_iso_calibrator" in globals():
    joblib.dump(mlb_iso_calibrator, calibrator_file)
    calibrator_saved = True

meta = {
    "model_name": "mlb_win_model_best",
    "best_model_family": mlb_best_model_name,
    "target": TARGET_WIN,
    "feature_count": int(len(feature_cols)),
    "feature_cols": list(feature_cols),
    "train_rows": int(len(final_train)),
    "train_start": str(final_train["date"].min().date()) if "date" in final_train.columns else None,
    "train_end": str(final_train["date"].max().date()) if "date" in final_train.columns else None,
    "walkforward_accuracy": float(mlb_wf_model_summary.iloc[0]["accuracy"]) if "mlb_wf_model_summary" in globals() else None,
    "walkforward_log_loss": float(mlb_wf_model_summary.iloc[0]["log_loss"]) if "mlb_wf_model_summary" in globals() else None,
    "walkforward_brier": float(mlb_wf_model_summary.iloc[0]["brier"]) if "mlb_wf_model_summary" in globals() else None,
    "calibrator_saved": calibrator_saved,
}

with open(meta_file, "w", encoding="utf-8") as f:
    json.dump(meta, f, indent=2)

mlb_final_win_model = final_win_model
mlb_final_train = final_train.copy()
MLB_USE_PRODUCTION_CALIBRATOR = calibrator_saved

print("\nFINAL MODEL TRAINED")
print("-" * 60)
print("Best model family:", mlb_best_model_name)
print("Train rows       :", len(final_train))
print("Feature count    :", len(feature_cols))
print("Calibrator saved :", calibrator_saved)

print("\nSaved model:")
print(win_model_file)

print("\nSaved calibrator:")
print(calibrator_file if calibrator_saved else "No calibrator saved")

print("\nSaved metadata:")
print(meta_file)

CELL 8 Ã¢â‚¬â€ TRAIN FINAL PRODUCTION WIN MODEL + SAVE MODEL


Temporal decay: weights range 0.000Ã¢â‚¬â€œ0.019 over 8230 training rows


Temporal decay weights applied to training.

FINAL MODEL TRAINED
------------------------------------------------------------
Best model family: rf_300
Train rows       : 8230
Feature count    : 115
Calibrator saved : True

Saved model:
models\mlb\mlb_win_model_best.joblib

Saved calibrator:
models\mlb\mlb_win_model_best_calibrator.joblib

Saved metadata:
models\mlb\mlb_win_model_best_meta.json


In [18]:
# %%
# =========================================================
# CELL 9 Ã¢â‚¬â€ BUILD LATEST TEAM SNAPSHOT FOR DAILY PREDICTIONS
# One latest usable row per MLB team
# =========================================================

if "mlb" not in globals() or not isinstance(mlb, pd.DataFrame):
    raise ValueError("mlb not found. Run Cell 2 first.")

needed_away_cols = [
    "date",
    "away_team",
    "away_runs_for_avg_10",
    "away_runs_against_avg_10",
    "away_margin_avg_10",
    "away_win_rate_10",
    "away_rest_days",
]

needed_home_cols = [
    "date",
    "home_team",
    "home_runs_for_avg_10",
    "home_runs_against_avg_10",
    "home_margin_avg_10",
    "home_win_rate_10",
    "home_rest_days",
]

missing_away = [c for c in needed_away_cols if c not in mlb.columns]
missing_home = [c for c in needed_home_cols if c not in mlb.columns]

if missing_away:
    raise ValueError(f"Missing away snapshot columns: {missing_away}")
if missing_home:
    raise ValueError(f"Missing home snapshot columns: {missing_home}")

snap = mlb.copy()
snap["date"] = pd.to_datetime(snap["date"], errors="coerce")

# ---------------------------------------------------------
# away-side rows
# ---------------------------------------------------------
away_snap = snap[needed_away_cols].copy().rename(columns={
    "away_team": "team",
    "away_runs_for_avg_10": "runs_for_avg_10",
    "away_runs_against_avg_10": "runs_against_avg_10",
    "away_margin_avg_10": "margin_avg_10",
    "away_win_rate_10": "win_rate_10",
    "away_rest_days": "rest_days",
})
away_snap["source_side"] = "away"

# ---------------------------------------------------------
# home-side rows
# ---------------------------------------------------------
home_snap = snap[needed_home_cols].copy().rename(columns={
    "home_team": "team",
    "home_runs_for_avg_10": "runs_for_avg_10",
    "home_runs_against_avg_10": "runs_against_avg_10",
    "home_margin_avg_10": "margin_avg_10",
    "home_win_rate_10": "win_rate_10",
    "home_rest_days": "rest_days",
})
home_snap["source_side"] = "home"

# ---------------------------------------------------------
# combine + latest row per team
# ---------------------------------------------------------
team_snapshots_long = pd.concat([away_snap, home_snap], ignore_index=True)
team_snapshots_long = team_snapshots_long.dropna(subset=["team", "date"]).copy()
team_snapshots_long = team_snapshots_long.sort_values(["team", "date"]).reset_index(drop=True)

latest_team_snapshot = (
    team_snapshots_long
    .drop_duplicates(subset=["team"], keep="last")
    .sort_values("team")
    .reset_index(drop=True)
)

print("Latest team snapshot shape:", latest_team_snapshot.shape)
print("Unique teams:", latest_team_snapshot["team"].nunique())
print("Latest snapshot date:", latest_team_snapshot["date"].max())

print("\nSample team snapshots:")
display(latest_team_snapshot.head(15))

mlb_latest_team_snapshot = latest_team_snapshot.copy()

Latest team snapshot shape: (48, 8)
Unique teams: 48
Latest snapshot date: 2025-11-01 00:00:00

Sample team snapshots:


,date,team,runs_for_avg_10,runs_against_avg_10,margin_avg_10,win_rate_10,rest_days,source_side
0,2025-07-15,American League All-Stars,3.5,3.0,0.5,0.5,7.0,away
1,2025-09-28,Arizona Diamondbacks,3.6,5.2,-1.6,0.4,1.0,away
2,2025-09-28,Atlanta Braves,5.8,3.8,2.0,0.7,1.0,home
3,2025-09-28,Baltimore Orioles,2.8,4.8,-2.0,0.4,1.0,away
4,2025-10-02,Boston Red Sox,3.6,3.1,0.5,0.6,1.0,away
5,2023-03-09,Canada,7.0,11.0,-4.0,0.0,1.0,away
6,2025-10-11,Chicago Cubs,4.3,2.8,1.5,0.7,2.0,away
7,2025-09-28,Chicago White Sox,3.8,5.5,-1.7,0.2,1.0,away
8,2025-10-01,Cincinnati Reds,3.8,3.5,0.3,0.6,1.0,away
9,2025-10-02,Cleveland Guardians,4.2,3.3,0.9,0.6,1.0,home


In [19]:
# =========================================================
# CELL - REFRESH TEAM STATS FROM CURRENT 2026 SEASON
# Adjustments applied before rolling windows:
#   1. Opponent defense quality (runs_for)
#   2. Opponent offense quality (runs_against)
#   3. Park factor normalization (runs_for)
#   4. Home/Away split rolling averages
#   5. Bullpen stress proxy (3-game runs_against sum)
# =========================================================
import requests
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

print("=" * 70)
print("REFRESH TEAM STATS FROM CURRENT 2026 SEASON")
print("=" * 70)

# Park factors keyed by ESPN displayName (neutral = 1.00)
_PF = {
    "Colorado Rockies":        1.28,
    "Cincinnati Reds":         1.07,
    "Boston Red Sox":          1.07,
    "Chicago Cubs":            1.05,
    "Arizona Diamondbacks":    1.04,
    "Baltimore Orioles":       1.02,
    "Washington Nationals":    1.02,
    "New York Yankees":        1.02,
    "Toronto Blue Jays":       1.01,
    "Houston Astros":          1.01,
    "Kansas City Royals":      1.01,
    "Philadelphia Phillies":   1.01,
    "Atlanta Braves":          1.01,
    "Texas Rangers":           1.01,
    "Pittsburgh Pirates":      1.00,
    "Chicago White Sox":       1.00,
    "Los Angeles Angels":      1.00,
    "Cleveland Guardians":     1.00,
    "St. Louis Cardinals":     0.99,
    "Tampa Bay Rays":          0.98,
    "Minnesota Twins":         0.98,
    "Detroit Tigers":          0.98,
    "Los Angeles Dodgers":     0.98,
    "Milwaukee Brewers":       0.97,
    "New York Mets":           0.97,
    "Seattle Mariners":        0.96,
    "Oakland Athletics":       0.96,
    "San Diego Padres":        0.95,
    "Miami Marlins":           0.93,
    "San Francisco Giants":    0.91,
}

def _park_factor(team_name):
    return _PF.get(str(team_name).strip(), 1.00)

# 2026-08-31 BUG FIX (confirmed live via curl): every ESPN call from this
# network is edge-blocked by Akamai with a 403 "Access Denied" page
# (Server: AkamaiGHost -- ESPN's own app never sees the request), not a
# rate limit. Direct call still tried first; only a 403 retries once
# through r.jina.ai's free reader-proxy relay (different egress IP,
# verified live to return a real 200 against this exact endpoint).
class _EspnRelayResponse:
    def __init__(self, status_code, json_data):
        self.status_code = status_code
        self._json_data = json_data
    def json(self):
        # BUG FIX (found live 2026-08-31, real crash): returning None here
        # on total failure let `.json()` succeed with a null payload instead
        # of raising -- the original code's `except Exception: continue`
        # guards never fired, and a later `data.get(...)` crashed with
        # AttributeError on the None. Raising here restores the "a failed
        # fetch always raises" contract every caller in this notebook
        # already relies on.
        if self._json_data is None:
            raise ValueError("ESPN fetch failed (both direct call and relay fallback)")
        return self._json_data

def _espn_get(url, timeout=10):
    try:
        r = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=timeout)
        if r.status_code != 403:
            return r
    except Exception:
        pass
    try:
        rr = requests.get(f"https://r.jina.ai/{url}", timeout=timeout + 20)
        rr.raise_for_status()
        body = rr.text
        idx = body.find("Markdown Content:")
        json_text = body[idx + len("Markdown Content:"):].strip() if idx >= 0 else body
        import json as _json
        return _EspnRelayResponse(200, _json.loads(json_text))
    except Exception:
        return _EspnRelayResponse(403, None)

def _fetch_espn_mlb_results(days_back=60):
    results = []
    today = datetime.today()
    for i in range(days_back, 0, -1):
        d = today - timedelta(days=i)
        date_str = d.strftime("%Y%m%d")
        url = f"https://site.api.espn.com/apis/site/v2/sports/baseball/mlb/scoreboard?dates={date_str}"
        try:
            r = _espn_get(url, timeout=10)
            data = r.json()
        except Exception:
            continue
        for event in data.get("events", []):
            comps = event.get("competitions", [])
            if not comps:
                continue
            comp = comps[0]
            if comp.get("status", {}).get("type", {}).get("name", "") != "STATUS_FINAL":
                continue
            competitors = comp.get("competitors", [])
            if len(competitors) < 2:
                continue
            home = next((c for c in competitors if c.get("homeAway") == "home"), None)
            away = next((c for c in competitors if c.get("homeAway") == "away"), None)
            if not home or not away:
                continue
            try:
                home_score = int(home.get("score", 0) or 0)
                away_score = int(away.get("score", 0) or 0)
            except Exception:
                continue
            home_name = home.get("team", {}).get("displayName", "")
            away_name = away.get("team", {}).get("displayName", "")
            results.append({
                "date":        d.strftime("%Y-%m-%d"),
                "home_team":   home_name,
                "away_team":   away_name,
                "home_score":  home_score,
                "away_score":  away_score,
                "home_win":    int(home_score > away_score),
                "park_factor": _park_factor(home_name),
            })
    return pd.DataFrame(results)

recent_games = _fetch_espn_mlb_results(days_back=60)
print(f"Fetched {len(recent_games)} completed games from ESPN (last 60 days)")

if len(recent_games) == 0:
    print("WARNING: No recent games fetched - keeping existing snapshot.")
else:
    recent_games["date"] = pd.to_datetime(recent_games["date"])
    recent_games = recent_games.sort_values("date").reset_index(drop=True)

    # Home rows: team IS home team, park = their own
    home_rows = recent_games[["date", "home_team", "home_score", "away_score", "home_win", "park_factor"]].copy()
    home_rows.columns = ["date", "team", "runs_for", "runs_against", "win", "park_factor"]
    home_rows["opponent"]  = recent_games["away_team"].values
    home_rows["home_away"] = "home"

    # Away rows: team is away, park = home team's park
    away_rows = recent_games[["date", "away_team", "away_score", "home_score", "home_win", "park_factor"]].copy()
    away_rows.columns = ["date", "team", "runs_for", "runs_against", "home_win_flip", "park_factor"]
    away_rows["win"]       = 1 - away_rows["home_win_flip"]
    away_rows["opponent"]  = recent_games["home_team"].values
    away_rows["home_away"] = "away"
    away_rows = away_rows[["date", "team", "runs_for", "runs_against", "win",
                            "opponent", "home_away", "park_factor"]]

    long_df = pd.concat([home_rows, away_rows], ignore_index=True)
    long_df  = long_df.sort_values(["team", "date"]).reset_index(drop=True)

    for col in ["runs_for", "runs_against", "win"]:
        long_df[col] = pd.to_numeric(long_df[col], errors="coerce")

    # -- 1. Opponent defense quality (runs_for) ------------------------
    _team_def   = long_df.groupby("team")["runs_against"].mean().to_dict()
    _lg_avg_def = long_df["runs_against"].mean()
    long_df["opp_def_rating"] = long_df["opponent"].map(_team_def).fillna(_lg_avg_def)
    _adj_off = (_lg_avg_def / long_df["opp_def_rating"]).clip(0.65, 1.35)
    long_df["adj_runs_for"]  = (long_df["runs_for"] * _adj_off).round(2)

    # -- 2. Opponent offense quality (runs_against) --------------------
    _team_off   = long_df.groupby("team")["runs_for"].mean().to_dict()
    _lg_avg_off = long_df["runs_for"].mean()
    long_df["opp_off_rating"]   = long_df["opponent"].map(_team_off).fillna(_lg_avg_off)
    _adj_def = (_lg_avg_off / long_df["opp_off_rating"]).clip(0.65, 1.35)
    long_df["adj_runs_against"] = (long_df["runs_against"] * _adj_def).round(2)

    # -- 3. Park factor normalization (runs_for) -----------------------
    long_df["adj_runs_for"] = (long_df["adj_runs_for"] / long_df["park_factor"]).round(2)

    print(f"Opp-quality adj: league def={_lg_avg_def:.2f}  off={_lg_avg_off:.2f}")
    print(f"Park factor range: [{long_df['park_factor'].min():.2f}, {long_df['park_factor'].max():.2f}]")

    # -- 4. Rolling averages (combined + home/away splits) -------------
    long_df["runs_for_avg_10"] = long_df.groupby("team")["adj_runs_for"].transform(
        lambda x: x.ewm(span=10, min_periods=3).mean()
    )
    long_df["runs_against_avg_10"] = long_df.groupby("team")["adj_runs_against"].transform(
        lambda x: x.ewm(span=10, min_periods=3).mean()
    )
    long_df["margin_avg_10"] = long_df["runs_for_avg_10"] - long_df["runs_against_avg_10"]
    long_df["win_rate_10"]   = long_df.groupby("team")["win"].transform(
        lambda x: x.ewm(span=10, min_periods=3).mean()
    )
    long_df["runs_for_avg_5"] = long_df.groupby("team")["adj_runs_for"].transform(
        lambda x: x.ewm(span=5,  min_periods=2).mean()
    )
    long_df["runs_against_avg_5"] = long_df.groupby("team")["adj_runs_against"].transform(
        lambda x: x.ewm(span=5,  min_periods=2).mean()
    )
    long_df["margin_avg_5"] = long_df["runs_for_avg_5"] - long_df["runs_against_avg_5"]
    long_df["win_rate_5"]   = long_df.groupby("team")["win"].transform(
        lambda x: x.ewm(span=5,  min_periods=2).mean()
    )

    # Home split
    _hm = long_df["home_away"] == "home"
    long_df.loc[_hm, "runs_for_avg_10_home"] = (
        long_df[_hm].groupby("team")["adj_runs_for"]
        .transform(lambda x: x.ewm(span=10, min_periods=2).mean())
    )
    long_df.loc[_hm, "runs_against_avg_10_home"] = (
        long_df[_hm].groupby("team")["adj_runs_against"]
        .transform(lambda x: x.ewm(span=10, min_periods=2).mean())
    )

    # Away split
    _aw = long_df["home_away"] == "away"
    long_df.loc[_aw, "runs_for_avg_10_away"] = (
        long_df[_aw].groupby("team")["adj_runs_for"]
        .transform(lambda x: x.ewm(span=10, min_periods=2).mean())
    )
    long_df.loc[_aw, "runs_against_avg_10_away"] = (
        long_df[_aw].groupby("team")["adj_runs_against"]
        .transform(lambda x: x.ewm(span=10, min_periods=2).mean())
    )

    # Forward-fill splits so fresh_snapshot always has a value
    for _c in ["runs_for_avg_10_home", "runs_against_avg_10_home",
               "runs_for_avg_10_away",  "runs_against_avg_10_away"]:
        long_df[_c] = long_df.groupby("team")[_c].transform(lambda x: x.ffill())

    # -- 5. Bullpen stress proxy ---------------------------------------
    long_df["bullpen_stress_3"] = long_df.groupby("team")["adj_runs_against"].transform(
        lambda x: x.rolling(3, min_periods=1).sum()
    )

    # -- 6. Bullpen-only RA estimate, live 2026 data (2026-08-04) ------
    # bullpen_ra_avg_10 (cell 88, 35% of the run-environment blend) was
    # traced to the orphaned mlb_model_table_v1.parquet -- real data ends
    # 2025-11-01, no builder cell anywhere rebuilds it, so every totals
    # projection all season used each team's bullpen as of ~Oct/Nov 2025,
    # frozen. This cell already fetches live current-season results but
    # only ever produced bullpen_stress_3, which is whole-TEAM adjusted
    # runs-against (starter + bullpen combined) -- using that directly as
    # a bullpen proxy would double-count starter quality already captured
    # separately via SP ELO/ERA in cell 88's blend. Mirroring cell 7's own
    # starter-share heuristic (starter assumed to allow 65% of runs
    # against, unadjusted -- same fallback proxy used for the frozen
    # table, kept identical so the two are on the same scale) to isolate
    # a real bullpen-only estimate from this live data instead.
    long_df["bullpen_runs_allowed_est"] = (long_df["runs_against"] * 0.35).clip(lower=0)
    long_df["bullpen_ra_avg_10_live"] = long_df.groupby("team")["bullpen_runs_allowed_est"].transform(
        lambda x: x.shift(1).ewm(span=10, min_periods=3).mean()
    )

    # -- Build fresh snapshot -----------------------------------------
    fresh_snapshot = (
        long_df
        .dropna(subset=["runs_for_avg_10"])
        .drop_duplicates(subset=["team"], keep="last")
        .copy()
        .reset_index(drop=True)
    )

    today_norm = pd.Timestamp.today().normalize()
    fresh_snapshot["rest_days"] = (today_norm - fresh_snapshot["date"]).dt.days.clip(1, 30)

    print(f"Rebuilt snapshot for {len(fresh_snapshot)} teams")
    print(f"Date range: {recent_games['date'].min().date()} -> {recent_games['date'].max().date()}")
    _show = ["team", "runs_for_avg_10", "runs_for_avg_10_home", "runs_for_avg_10_away",
             "runs_against_avg_10", "bullpen_stress_3", "win_rate_10"]
    print(fresh_snapshot[[c for c in _show if c in fresh_snapshot.columns]].to_string(index=False))

    globals()["mlb_latest_team_snapshot"] = fresh_snapshot.copy()

    # -- Merge team discipline (K%/BB%) if available -------------------
    if "mlb_team_discipline" in globals() and mlb_team_discipline:
        import re as _re
        def _nteam(s): return _re.sub(r"[^a-z0-9]", "", str(s).lower().strip())
        fresh_snapshot["k_pct_bat"]  = fresh_snapshot["team"].apply(
            lambda t: mlb_team_discipline.get(_nteam(t), {}).get("k_pct_bat", float("nan")))
        fresh_snapshot["bb_pct_bat"] = fresh_snapshot["team"].apply(
            lambda t: mlb_team_discipline.get(_nteam(t), {}).get("bb_pct_bat", float("nan")))
        _disc_ok = fresh_snapshot["k_pct_bat"].notna().sum()
        print(f"Team discipline merged: {_disc_ok}/{len(fresh_snapshot)} teams have K%/BB%")

    globals()["mlb_team_snapshot_clean"] = fresh_snapshot.copy()
    print("\nOverwrote mlb_latest_team_snapshot + mlb_team_snapshot_clean with 2026 data.")


REFRESH TEAM STATS FROM CURRENT 2026 SEASON


Fetched 667 completed games from ESPN (last 60 days)
Opp-quality adj: league def=4.43  off=4.43
Park factor range: [0.91, 1.28]


Rebuilt snapshot for 30 teams
Date range: 2026-08-11 -> 2026-10-08
                 team  runs_for_avg_10  runs_for_avg_10_home  runs_for_avg_10_away  runs_against_avg_10  bullpen_stress_3  win_rate_10
 Arizona Diamondbacks         6.631899              5.019150              6.237057             5.839553             20.65     0.529236
            Athletics         3.727438              4.014184              3.135026             7.673825             28.62     0.326382
       Atlanta Braves         3.356249              3.476076              4.126913             3.460287              8.60     0.365238
    Baltimore Orioles         5.473706              3.768235              6.085037             3.142639              9.47     0.614863
       Boston Red Sox         1.894107              2.383687              2.357511             4.499533             21.63     0.316896
         Chicago Cubs         2.659177              4.362968              2.970111             4.075339             13.21  

In [20]:
# =========================================================
# CELL â€” FANGRAPHS PITCHER FIP / K-BB% SCRAPER
# Fetches current-season FIP, xFIP, K-BB% for all starters.
# Creates mlb_pitcher_fip_lookup dict used by board builder.
# =========================================================
import requests
import pandas as pd
import numpy as np
import unicodedata, re

print('=' * 70)
print('FANGRAPHS PITCHER FIP / K-BB% SCRAPER')
print('=' * 70)

def _norm_name(s):
    if not s:
        return ''
    s = unicodedata.normalize('NFD', str(s))
    s = ''.join(c for c in s if unicodedata.category(c) != 'Mn')
    return re.sub(r'[^a-z ]', '', s.lower()).strip()

_FG_PIT_URL = 'https://www.fangraphs.com/api/leaders/major-league/data'
_FG_PIT_PARAMS = {
    'pos': 'all', 'stats': 'pit', 'lg': 'all', 'qual': 10,
    'type': 8, 'season': 2026, 'month': 0, 'season1': 2026,
    'ind': 0, 'pageitems': 2000000, 'pagenum': 1,
}
_FG_HEADERS = {'User-Agent': 'Mozilla/5.0 (compatible; research-bot)'}

try:
    _r = requests.get(_FG_PIT_URL, params=_FG_PIT_PARAMS, headers=_FG_HEADERS, timeout=15)
    _r.raise_for_status()
    _raw = _r.json()
    _rows = _raw.get('data', [])
    if not _rows:
        raise ValueError('Empty data from Fangraphs pitcher endpoint')
    _fg_pit = pd.DataFrame(_rows)
    print(f'Fetched {len(_fg_pit)} pitcher rows from Fangraphs')
    print('Available columns:', [c for c in _fg_pit.columns if any(k in c for k in ['FIP','xFIP','K-BB','K%','BB%','PlayerName'])])
except Exception as _e:
    print(f'WARNING: Fangraphs pitcher fetch failed: {_e}')
    _fg_pit = pd.DataFrame()

mlb_pitcher_fip_lookup = {}

if not _fg_pit.empty:
    # 2026-08-12 BUG FIX #3 (found via the automated daily QA report flagging
    # "Zac Thornton ... IP-sample lookup UNAVAILABLE" even after the HTML-
    # strip fix above): same ambiguous-first-match column-selector shape as
    # the K%/BB% bug, just on the name field. 'PlayerName' in c matches BOTH
    # 'PlayerName' AND 'PlayerNameRoute' (substring match), and 'Name' is
    # also eligible -- whichever of the three appears first in FanGraphs'
    # column order wins. Confirmed live: 'Name' sits at column index 2,
    # 'PlayerNameRoute'/'PlayerName' at 534/535, so 'Name' always won. For
    # most players 'Name' and 'PlayerName' agree once HTML-stripped, but for
    # a player like Zac Thornton, FanGraphs' 'Name' field carries his full
    # legal first name ('Zachary Thornton') while 'PlayerName' carries the
    # common/display form ('Zac Thornton') that the rest of this notebook
    # (board data, MLB Stats API useName, ESPN) actually uses -- so his
    # FanGraphs-sourced dict entry landed under a key ('zachary thornton')
    # that _fed_check's lookup ('zac thornton', from the real sp_name) never
    # matches. 'PlayerName' is already clean plain text (no HTML) and is the
    # correct display-name form -- prefer it explicitly; only fall back to
    # the HTML 'Name' field (already handled by the strip fix above) if
    # 'PlayerName' is somehow absent.
    _name_col = next((c for c in _fg_pit.columns if c == 'PlayerName'), None) \
                or next((c for c in _fg_pit.columns if c == 'Name'), None)
    _fip_col  = next((c for c in _fg_pit.columns if c == 'FIP'), None)
    _xfip_col = next((c for c in _fg_pit.columns if c == 'xFIP'), None)
    _kbb_col  = next((c for c in _fg_pit.columns if 'K-BB' in c), None)
    # 2026-08-12 BUG FIX (live: card showed "NONE -- no official plays today"
    # despite several games with a real gating edge well above the required
    # floor, plus an absurd "SP BB% 318.1%" in the RL row): FanGraphs' real
    # response carries BOTH 'K%'/'BB%' (true fractions, e.g. 0.0596) AND
    # 'K/9'/'BB/9' (rate stats, e.g. 2.03) as separate columns -- confirmed
    # live. `next(... if c in (X%, X/9))` picks whichever one appears FIRST
    # in column order, not whichever is actually named X% -- and FanGraphs
    # lists K/9 (index 42) and BB/9 (index 43) well before K% (index 135)
    # and BB% (index 136), so this always silently grabbed the rate stat
    # and stored it under the *_pct_pit keys as if it were a 0-1 fraction.
    # Downstream: cell 88's walk-risk gate compares bb_pct_pit against a
    # 10.5% hard-stop threshold -- a real BB/9 of 2-5 trivially clears that,
    # so the gate was firing on nearly every pitcher and zeroing out
    # passes_edge_rule across the board. Separately, the K9-L5-proxy
    # formula (`k_pct * 3.7 * 9`, two call sites) explicitly expects a
    # fraction and was producing ~460 "K/9" instead of a real ~10-14.
    # K/9 itself is already captured correctly under the separate `k9` key
    # a few lines below, so there's no data loss in dropping the ambiguous
    # fallback -- just matching the exact percentage column name.
    _kpct_col = next((c for c in _fg_pit.columns if c == 'K%'), None)
    _bbpct_col= next((c for c in _fg_pit.columns if c == 'BB%'), None)
    _siera_col= next((c for c in _fg_pit.columns if c == 'SIERA'), None)  # real gap found 2026-07-23: only FIP/xFIP were pulled, no SIERA

    for _, row in _fg_pit.iterrows():
        # 2026-08-12 BUG FIX (found grading a real card -- five different SPs
        # all showed a fabricated "0 IP" on the ELO-ERA divergence self-clear
        # check in cell 90/_fed_check): FanGraphs' leaderboard API returns
        # the Name field as a full HTML anchor tag, e.g.
        # '<a href="statss.aspx?playerid=676440&position=P">Tanner Bibee</a>',
        # not a plain name. _norm_name() only strips non a-z/space characters
        # -- it doesn't strip HTML tags/attributes -- so every one of this
        # fetch's ~565 pitcher keys was corrupted (reproduced live: normalizes
        # to 'a hrefstatssaspxplayeridpositionptanner bibeea' instead of
        # 'tanner bibee'), making the entire FanGraphs-sourced fip/xfip/siera/
        # k_bb_pct/ip_season table unreachable by any clean pitcher name
        # anywhere else in the notebook that looks it up via _norm_name(sp_name).
        # Strip HTML tags before normalizing so the key actually matches.
        _raw_name_field = re.sub(r'<[^>]*>', '', str(row.get(_name_col, '')))
        name_key = _norm_name(_raw_name_field)
        if not name_key:
            continue
        mlb_pitcher_fip_lookup[name_key] = {
            'fip':       float(row[_fip_col])  if _fip_col  and pd.notna(row.get(_fip_col))  else np.nan,
            'xfip':      float(row[_xfip_col]) if _xfip_col and pd.notna(row.get(_xfip_col)) else np.nan,
            'siera':     float(row[_siera_col])if _siera_col and pd.notna(row.get(_siera_col)) else np.nan,
            'k_bb_pct':  float(row[_kbb_col])  if _kbb_col  and pd.notna(row.get(_kbb_col))  else np.nan,
            'k_pct_pit': float(row[_kpct_col]) if _kpct_col and pd.notna(row.get(_kpct_col)) else np.nan,
            'bb_pct_pit':float(row[_bbpct_col])if _bbpct_col and pd.notna(row.get(_bbpct_col))else np.nan,
            'k9':        float(row['K/9']) if 'K/9' in row.index and pd.notna(row.get('K/9')) else np.nan,
            'ip_season': float(row['IP'])  if 'IP'  in row.index and pd.notna(row.get('IP'))  else np.nan,
        }

    print(f'Pitcher FIP lookup built: {len(mlb_pitcher_fip_lookup)} pitchers')
    # sample
    _sample = list(mlb_pitcher_fip_lookup.items())[:3]
    for n, v in _sample:
        print(f'  {n}: {v}')
else:
    print('Pitcher FIP lookup empty â€” Fangraphs unavailable')

# MLB-FG-FALLBACK-1 (2026-10-01, found live): FanGraphs has returned 403 on
# every run since 2026-09-27, leaving this lookup EMPTY -- every FIP/xFIP/
# SIERA-based piece downstream (xFIP run blend, FIP-reliability shaves,
# workload IP, estimator SP rating) silently went dark for 5 days. Two bulk
# public sources fill it whenever FanGraphs is missing (and add Savant xERA
# and player ids even when it isn't):
#   MLB Stats API /stats sabermetrics+season (one call each): FIP, xFIP, IP,
#     K/9, K%, BB%, GS, ERA, R  -- same keys FanGraphs filled
#   Baseball Savant expected_statistics CSV: xERA, xwOBA
# SIERA has no free bulk source here -- left NaN on fallback rows (gates that
# need SIERA stay closed rather than reading a substitute under its name);
# the estimator SP rating uses Savant xERA in SIERA's slot, labelled as such.
def _fg_fallback_fill():
    import io as _fio
    _yr = __import__("datetime").date.today().year
    _base = "https://statsapi.mlb.com/api/v1/stats"
    _q = f"group=pitching&season={_yr}&gameType=R&playerPool=ALL&limit=3000"
    _sab = requests.get(f"{_base}?stats=sabermetrics&{_q}", timeout=30).json()["stats"][0]["splits"]
    _sea = requests.get(f"{_base}?stats=season&{_q}", timeout=30).json()["stats"][0]["splits"]
    _sav = pd.read_csv(_fio.StringIO(requests.get(
        f"https://baseballsavant.mlb.com/leaderboard/expected_statistics?type=pitcher&year={_yr}"
        f"&position=&team=&min=1&csv=true", timeout=30).content.decode("utf-8-sig")))
    _sav_by_id = {int(r["player_id"]): r for _, r in _sav.iterrows()}
    _sab_by_id = {s["player"]["id"]: s["stat"] for s in _sab}
    def _ipf(x):
        _w, _, _f = str(x or "0").partition(".")
        try:
            return float(_w or 0) + (float(_f) / 3.0 if _f else 0.0)
        except ValueError:
            return np.nan
    def _f(x):
        try:
            return float(x)
        except (TypeError, ValueError):
            return np.nan
    _filled = _supp = 0
    by_id = {}
    for s in _sea:
        _pid, _nm, _st = s["player"]["id"], s["player"]["fullName"], s["stat"]
        _sb = _sab_by_id.get(_pid, {})
        _sv = _sav_by_id.get(_pid)
        _bf = _f(_st.get("battersFaced"))
        _rec = {
            "fip": _f(_sb.get("fip")), "xfip": _f(_sb.get("xfip")), "siera": np.nan,
            "k_pct_pit": _f(_st.get("strikeOuts")) / _bf * 100 if _bf and _bf == _bf else np.nan,
            "bb_pct_pit": _f(_st.get("baseOnBalls")) / _bf * 100 if _bf and _bf == _bf else np.nan,
            "k9": _f(_st.get("strikeoutsPer9Inn")), "ip_season": _ipf(_st.get("inningsPitched")),
        }
        _rec["k_bb_pct"] = (_rec["k_pct_pit"] - _rec["bb_pct_pit"]) if _rec["k_pct_pit"] == _rec["k_pct_pit"] else np.nan
        _extra = {"player_id": _pid, "gs": _f(_st.get("gamesStarted")), "era_actual": _f(_st.get("era")),
                  "xera": _f(_sv["xera"]) if _sv is not None else np.nan,
                  "xwoba": _f(_sv["est_woba"]) if _sv is not None else np.nan,
                  "woba": _f(_sv["woba"]) if _sv is not None else np.nan}
        by_id[_pid] = {"name": _nm, **_rec, **_extra}
        _k = _norm_name(_nm)
        if _k not in mlb_pitcher_fip_lookup:
            mlb_pitcher_fip_lookup[_k] = {**_rec, **_extra, "_source": "statsapi+savant (FanGraphs fallback)"}
            _filled += 1
        else:
            for _ek, _evv in _extra.items():   # supplement a FanGraphs row with id/xERA, never overwrite its stats
                mlb_pitcher_fip_lookup[_k].setdefault(_ek, _evv)
            _supp += 1
    globals()["mlb_pitcher_est_by_id"] = by_id
    return _filled, _supp
try:
    _fb_filled, _fb_supp = _fg_fallback_fill()
    print(f"[MLB-FG-FALLBACK-1] Stats API + Savant: {_fb_filled} pitcher(s) filled "
          f"{'(FanGraphs unavailable -- SIERA left NaN on these rows)' if _fg_pit.empty else '(not in FanGraphs)'}, "
          f"{_fb_supp} FanGraphs row(s) supplemented with player id/xERA")
except Exception as _fb_e:
    print(f"!! [MLB-FG-FALLBACK-1] fallback fetch failed too: {_fb_e} -- FIP/xFIP-based gates are dark this run")

def _mlb_sp_est_era(v):
    """Estimator ERA for the SP rating (backlog #7): 0.2*FIP + 0.4*xFIP + 0.4*SIERA,
    with Savant xERA in SIERA's slot when SIERA is missing; weights renormalized,
    needs at least 2 of the 3. Returns (est, label) or (None, None)."""
    if not v:
        return None, None
    _third = v.get("siera")
    _lab = "SIERA"
    if _third is None or _third != _third:
        _third, _lab = v.get("xera"), "xERA"
    _parts = [(0.2, v.get("fip"), "FIP"), (0.4, v.get("xfip"), "xFIP"), (0.4, _third, _lab)]
    _ok = [(w, float(x), n) for w, x, n in _parts if x is not None and x == x]
    if sum(w for w, _, _ in _ok) < 0.6:
        return None, None
    _den = sum(w for w, _, _ in _ok)
    return sum(w * x for w, x, _ in _ok) / _den, "/".join(n for _, _, n in _ok)
globals()["_mlb_sp_est_era"] = _mlb_sp_est_era

globals()['mlb_pitcher_fip_lookup'] = mlb_pitcher_fip_lookup


FANGRAPHS PITCHER FIP / K-BB% SCRAPER


Fetched 649 pitcher rows from Fangraphs
Available columns: ['FIP', 'xFIP', 'FIP-', 'xFIP-', 'K%', 'BB%', 'K-BB%', 'K%+', 'BB%+', 'PlayerNameRoute', 'PlayerName']
Pitcher FIP lookup built: 649 pitchers
  jacob misiorowski: {'fip': 2.1673056619427538, 'xfip': 2.292247587896329, 'siera': 2.3495815427527735, 'k_bb_pct': 0.31278196, 'k_pct_pit': 0.37894737, 'bb_pct_pit': 0.06616541, 'k9': 12.984732446314325, 'ip_season': 174.2}
  cristopher sanchez: {'fip': 2.800994833425624, 'xfip': 2.650761024970652, 'siera': 2.920630364893812, 'k_bb_pct': 0.21703617, 'k_pct_pit': 0.26954492, 'bb_pct_pit': 0.05250875, 'k9': 10.043478260869565, 'ip_season': 207.0}
  cam schlittler: {'fip': 2.5583430384614614, 'xfip': 2.8564069862228627, 'siera': 2.9352309871068125, 'k_bb_pct': 0.24967149, 'k_pct_pit': 0.31406045, 'bb_pct_pit': 0.06438896, 'k9': 11.106713147934572, 'ip_season': 193.2}


[MLB-FG-FALLBACK-1] Stats API + Savant: 218 pitcher(s) filled (not in FanGraphs), 650 FanGraphs row(s) supplemented with player id/xERA


In [21]:
# =========================================================
# CELL - STARTER RECENT FORM (L3 STARTS)
# Fetches last 3 game logs per starter via MLB Stats API.
# Adds era_l3, whip_l3, ip_l3 to mlb_pitcher_fip_lookup.
# =========================================================
import requests
import pandas as pd
import numpy as np
import unicodedata, re

print('=' * 70)
print('STARTER RECENT FORM â€” L3 STARTS')
print('=' * 70)

def _norm_name(s):
    if not s: return ''
    s = unicodedata.normalize('NFD', str(s))
    s = ''.join(c for c in s if unicodedata.category(c) != 'Mn')
    return re.sub(r'[^a-z ]', '', s.lower()).strip()

# BUG FIX (2026-08-15, MLB SP staleness audit -- same root cause as the
# identical fix in Cell 88's _sp_fetch_recent): MLB's inningsPitched field
# is baseball notation ("5.1" = 5 innings + 1 out = 5.333, not literal
# 5.1), so a bare float() cast on it silently undercounts every partial-
# inning start and the error compounds across a season. This feeds
# mlb_pitcher_fip_lookup, the fallback ERA/IP source Cell 88 uses whenever
# its own live sp_form_cache fetch is unavailable -- same wrong-number
# problem, different code path.
def _l3_parse_ip(v):
    try:
        _s = str(v); _p = _s.split(".")
        return int(_p[0]) + (int(_p[1]) if len(_p) > 1 else 0) / 3.0
    except Exception:
        return 0.0

_MLB_API = 'https://statsapi.mlb.com/api/v1'
_headers = {'User-Agent': 'Mozilla/5.0 (compatible; research-bot)'}

def _search_player_id(name):
    """Return MLB Stats API personId for a pitcher name, or None."""
    try:
        r = requests.get(
            f'{_MLB_API}/people/search',
            params={'names': name, 'sportId': 1},
            headers=_headers, timeout=8
        )
        data = r.json().get('people', [])
        if data:
            return data[0]['id']
    except Exception:
        pass
    return None

def _get_l3_stats(person_id):
    """Return dict with era_l3, era_l9, era_season, whip_l3, ip_l3, ip_season, gs_season, avg_ip_per_start."""
    _empty = {k: np.nan for k in ('era_l3','era_l9','era_season','whip_l3','ip_l3','ip_season','gs_season','avg_ip_per_start','last_start_bb','last_start_ip','last2_starts_er','last2_starts_ip')}
    _empty['l3_starts_er'] = None
    _empty['l3_starts_ip'] = None
    try:
        r = requests.get(
            f'{_MLB_API}/people/{person_id}/stats',
            params={'stats': 'gameLog', 'group': 'pitching',
                    'season': 2026, 'gameType': 'R'},
            headers=_headers, timeout=10
        )
        splits = r.json().get('stats', [{}])[0].get('splits', [])

        # TRUE BULLPEN-GAME DETECTION (2026-09-11, user request -- Astros'
        # Ullola: never thrown more than 2 IP in any MLB appearance, slotted
        # as tonight's 'starter' for pure scheduling reasons). is_opener/
        # is_bulk_role below both require gs_season >= 3 to fire at all --
        # correct for telling a genuine short-start ROLE apart from a normal
        # starter, but that same >=3-real-starts sample gate structurally
        # excludes a pure reliever making his 1st or 2nd career 'start' --
        # exactly the pitcher with the FEWEST real starts of anyone, so the
        # gate built to protect is_opener from small-sample noise also
        # excludes the one case the distinction matters most for. Computed
        # from the FULL unfiltered appearance log (starts + relief) so it
        # still fires even when `starts` below ends up empty.
        # BUG FIX (2026-09-18, user audit -- Ian Seymour flagged a true
        # bullpen game on today's Rays card despite 14 real starts / 122.1 IP
        # this season, most recently 6 IP): _bg_avg_ip blends ALL season
        # appearances into one average, so a pitcher who opened the year in
        # relief (Seymour: 30 outings around 1 IP each, March-May) before a
        # mid-season move to the rotation gets that long relief tail dragging
        # the season-wide average under 3.0 even though he's since made 14
        # confirmed starts (live-verified: 122.33 IP / 44 appearances = 2.78).
        # The check's own stated purpose above is to cover pitchers BELOW the
        # gs_season >= 3 sample gate that is_opener/is_bulk_role already use
        # -- once a pitcher clears that same threshold, those two flags are
        # the correct classifiers and this one should stand down instead of
        # overriding them with a stale season-long blend. Added the identical
        # gs < 3 gate here.
        _bg_apps = len(splits)
        _bg_gs = sum(1 for s in splits if int(s.get('stat', {}).get('gamesStarted', 0) or 0) == 1)
        _bg_total_ip = sum(_l3_parse_ip(s.get('stat', {}).get('inningsPitched', 0)) for s in splits)
        _bg_avg_ip = (_bg_total_ip / _bg_apps) if _bg_apps > 0 else np.nan
        _bg_extra = {
            'is_true_bullpen_game': bool(_bg_apps >= 3 and _bg_gs < 3 and (_bg_gs == 0 or _bg_avg_ip < 3.0)),
            'total_appearances_season': _bg_apps,
            'career_avg_ip_all_appearances': round(_bg_avg_ip, 2) if _bg_apps > 0 else np.nan,
        }

        # POST-TRADE SPLIT (2026-09-11, user request -- Dustin May: pre-trade
        # FIP 2.88 (top-8 MLB) drove a 1538 ELO, but his post-trade ERA with
        # Milwaukee (14 ER/19.2 IP, ~6.43) told a completely different story
        # that never reached the card). team.id/team.name already ride along
        # on every gameLog split (same `splits` list scanned above) -- no
        # separate transactions-endpoint call needed. 'Trade date' here is
        # the first start with the CURRENT team -- a lower-bound proxy for
        # the real trade date (the actual trade is on or before this), which
        # is conservative in the direction that matters: if this first start
        # is already >45 days old, the real trade is even older, so the
        # recency check correctly stays off rather than false-firing.
        _pt_dates_teams = sorted(
            ((s.get('date'), (s.get('team') or {}).get('id'), (s.get('team') or {}).get('name'))
             for s in splits if s.get('date') and (s.get('team') or {}).get('id')),
            key=lambda t: t[0]
        )
        _pt_extra = {'is_recently_traded': False, 'days_since_trade': None,
                      'post_trade_era': np.nan, 'post_trade_fip': np.nan, 'post_trade_starts': 0,
                      'pre_trade_era': np.nan, 'pre_trade_starts': 0, 'new_team_name': None}
        if _pt_dates_teams:
            _pt_cur_team_id = _pt_dates_teams[-1][1]
            _pt_cur_team_name = _pt_dates_teams[-1][2]
            _pt_old_team_ids = {tid for _, tid, _ in _pt_dates_teams if tid != _pt_cur_team_id}
            if _pt_old_team_ids:
                import datetime as _pt_dt
                _pt_first_cur = next(d for d, tid, _ in _pt_dates_teams if tid == _pt_cur_team_id)
                try:
                    _pt_days = (_pt_dt.date.today() - _pt_dt.date.fromisoformat(str(_pt_first_cur)[:10])).days
                except Exception:
                    _pt_days = None
                _pt_post_starts = [s for s in splits if (s.get('team') or {}).get('id') == _pt_cur_team_id
                                    and int(s.get('stat', {}).get('gamesStarted', 0) or 0) == 1]
                _pt_pre_starts = [s for s in splits if (s.get('team') or {}).get('id') != _pt_cur_team_id
                                   and int(s.get('stat', {}).get('gamesStarted', 0) or 0) == 1]

                def _pt_era_fip(subset):
                    if not subset:
                        return np.nan, np.nan, 0
                    _er = sum(float(s.get('stat', {}).get('earnedRuns', 0) or 0) for s in subset)
                    _ip = sum(_l3_parse_ip(s.get('stat', {}).get('inningsPitched', 0)) for s in subset)
                    _hr = sum(float(s.get('stat', {}).get('homeRuns', 0) or 0) for s in subset)
                    _bb = sum(float(s.get('stat', {}).get('baseOnBalls', 0) or 0) for s in subset)
                    _hbp = sum(float(s.get('stat', {}).get('hitByPitch', 0) or 0) for s in subset)
                    _k = sum(float(s.get('stat', {}).get('strikeOuts', 0) or 0) for s in subset)
                    _era = round(_er / _ip * 9, 2) if _ip > 0 else np.nan
                    _fip = round((13 * _hr + 3 * (_bb + _hbp) - 2 * _k) / _ip + 3.10, 2) if _ip > 0 else np.nan
                    return _era, _fip, len(subset)

                _pt_post_era, _pt_post_fip, _pt_post_n = _pt_era_fip(_pt_post_starts)
                _pt_pre_era, _pt_pre_fip, _pt_pre_n = _pt_era_fip(_pt_pre_starts)
                _pt_extra = {
                    'is_recently_traded': bool(_pt_days is not None and _pt_days <= 45),
                    'days_since_trade': _pt_days,
                    'post_trade_era': _pt_post_era, 'post_trade_fip': _pt_post_fip, 'post_trade_starts': _pt_post_n,
                    'pre_trade_era': _pt_pre_era, 'pre_trade_starts': _pt_pre_n,
                    'new_team_name': _pt_cur_team_name,
                }
        _bg_extra.update(_pt_extra)
        # BUG FIX (2026-09-02, user audit -- Drew Anderson misclassified "true
        # opener" on today's actual OFFICIAL pick): despite the name, this
        # filtered on IP >= 1.0 only, not gamesStarted == 1 -- so any relief
        # appearance of a full inning or more got swept into "starts" right
        # alongside real starts. Harmless for era_l3/whip_l3 for a pitcher
        # whose most recent 3 appearances all happen to be true starts (the
        # [-3:]/[-9:] slices land on real data by coincidence of chronology),
        # but gs_season/ip_season/avg_ip_per_start aggregate over the WHOLE
        # contaminated list -- confirmed live: Anderson has 45 total 2026
        # appearances, only 7 real starts (the other 38 are relief outings,
        # mostly 1-2 IP), and the pre-fix avg_ip_per_start swept in enough of
        # those to read 4.24 real-start average as under the 3.0 "true
        # opener" cutoff. That flag (mlb_pitcher_fip_lookup['is_opener'])
        # feeds both the printed "[SP ROLE] ... true opener" line and a real
        # run-environment reweight (Cell 87's _eff_pit: 25% SP / 75% bullpen
        # instead of 65/35) -- both applied to a pitcher who just threw
        # 4.7-5.0 IP in each of his last 3 outings. Sister fetcher
        # (_sp_fetch_recent, Cell 87) already filters on gamesStarted == 1
        # correctly; matches that here.
        starts = []
        for s in splits:
            if int(s.get('stat', {}).get('gamesStarted', 0) or 0) != 1:
                continue
            _st = dict(s.get('stat') or {})
            _st['_gap_date'] = s.get('date')
            starts.append(_st)
        if not starts:
            return {**_empty, **_bg_extra}

        # BUG FIX (2026-09-06, user audit -- Kikuchi Sept 5 win over the
        # Pirates pick): "starts" is a chronological list of real starts
        # across the WHOLE season with no awareness of gaps -- for a pitcher
        # back from a long injury absence, a naive [-3:]/[-9:] slice can
        # reach backward across that gap and blend a stale pre-injury start
        # in with the real return starts. Confirmed live: Kikuchi's real
        # "L3" (4.50 ERA) was 1 start from 2026-04-29 (0 ER/2.0 IP, before a
        # ~4-month absence) plus his actual first 2 starts back (2026-08-23:
        # 5 ER/5.0 IP, 2026-08-30: 2 ER/7.0 IP) -- a fossil data point from
        # before the injury, not real recent form. (This specific case
        # didn't actually explain the Pirates loss -- a gap-aware L3 reads
        # 5.25 ERA over the two real return starts, WORSE than the
        # contaminated 4.50, not better; he then threw a scoreless start no
        # reasonable 2-start sample would have predicted. Fixing the
        # contamination is still correct on its own terms -- it can cut
        # either direction in other cases. season_era below intentionally
        # still spans the whole year unfiltered, since that's its defined
        # meaning; only the recent-form windows are gap-aware now.)
        _L3_GAP_DAYS = 20  # a gap this large between consecutive starts
        # means an IL stint/minors demotion/absence, not normal rotation rest

        def _sp_recent_after_gap(starts_chrono, date_key):
            if len(starts_chrono) < 2:
                return starts_chrono
            import datetime as _l3gap_dt
            cut = 0
            for idx in range(1, len(starts_chrono)):
                try:
                    d0 = _l3gap_dt.date.fromisoformat(str(starts_chrono[idx - 1].get(date_key, ''))[:10])
                    d1 = _l3gap_dt.date.fromisoformat(str(starts_chrono[idx].get(date_key, ''))[:10])
                    if (d1 - d0).days > _L3_GAP_DAYS:
                        cut = idx
                except Exception:
                    pass
            return starts_chrono[cut:]

        _recent = _sp_recent_after_gap(starts, '_gap_date')

        def _cstats(subset):
            er = sum(float(s.get('earnedRuns', 0) or 0) for s in subset)
            ip = sum(_l3_parse_ip(s.get('inningsPitched', 0)) for s in subset)
            h  = sum(float(s.get('hits', 0) or 0) for s in subset)
            bb = sum(float(s.get('baseOnBalls', 0) or 0) for s in subset)
            era  = round(er / ip * 9, 2) if ip > 0 else np.nan
            whip = round((h + bb) / ip, 3) if ip > 0 else np.nan
            return era, whip, round(ip, 1)

        last3 = _recent[-3:]
        last9 = _recent[-9:]
        # SP COLD-STREAK SHAPE (2026-08-31, user audit -- George Kirby): raw
        # per-start ER/IP were always fetched here to build the L3 aggregate,
        # then discarded -- so a single blowup start averaged in with two
        # normal starts and three genuinely bad starts both collapsed to the
        # same L3 ERA number, with nothing downstream able to tell the shapes
        # apart. Captured here (and in the sister fetcher, _sp_fetch_recent)
        # so the Final Card WHY loop can distinguish "one bad start dragging
        # the average" from "sustained decline" instead of relying on xERA
        # alone, which only tests results-vs-process luck, not start shape.
        _l3_starts_er = [float(s.get('earnedRuns', 0) or 0) for s in last3]
        _l3_starts_ip = [round(_l3_parse_ip(s.get('inningsPitched', 0)), 1) for s in last3]
        era_l3, whip_l3, ip_l3    = _cstats(last3)
        era_l9, _, _               = _cstats(last9)
        era_season, _, _           = _cstats(starts)
        ip_season  = round(sum(_l3_parse_ip(s.get('inningsPitched', 0)) for s in starts), 1)
        gs_season  = len(starts)
        avg_ip     = round(ip_season / gs_season, 2) if gs_season > 0 else np.nan
        # Most recent SINGLE start's walk count/IP -- separate from the L3
        # (3-start) average above. 2026-07-12: added after a real card kept
        # showing a starter as a normal play the day after a historic
        # walk-rate implosion (7 BB in 1.2 IP) -- an outing like that gets
        # diluted into a 3-start average and can slip under the season-BB%
        # gate entirely; this flags the exact last outing directly.
        _last = starts[-1]
        _last_start_bb = float(_last.get('baseOnBalls', 0) or 0)
        _last_start_ip = _l3_parse_ip(_last.get('inningsPitched', 0))
        # Combined last-2-starts earned runs -- for the opposing-SP recent-form
        # suppressor (cell 88): a pitcher allowing <=1 ER combined over their last
        # two starts is a real red flag for the team facing them, separate from
        # (and can catch cases missed by) the L3-average and season-ERA gates.
        _last2 = starts[-2:]
        _last2_starts_er = sum(float(s.get('earnedRuns', 0) or 0) for s in _last2)
        _last2_starts_ip = sum(_l3_parse_ip(s.get('inningsPitched', 0)) for s in _last2)
        return {'era_l3': era_l3, 'era_l9': era_l9, 'era_season': era_season,
                'whip_l3': whip_l3, 'ip_l3': ip_l3, 'ip_season': ip_season,
                'gs_season': gs_season, 'avg_ip_per_start': avg_ip,
                'last_start_bb': _last_start_bb, 'last_start_ip': _last_start_ip,
                'last2_starts_er': _last2_starts_er, 'last2_starts_ip': _last2_starts_ip,
                'l3_starts_er': _l3_starts_er, 'l3_starts_ip': _l3_starts_ip,
                **_bg_extra}
    except Exception:
        return _empty

# ---------------------------------------------------------
# Collect pitcher names from today's board or FIP lookup
# ---------------------------------------------------------
_pitcher_names = set()

# From today's board
for _src_name in ['mlb_today_input', 'projection_board', 'mlb_board', 'scored_mlb_board']:
    _brd = globals().get(_src_name)
    if isinstance(_brd, pd.DataFrame):
        for _col in ['away_pitcher_name', 'home_pitcher_name']:
            if _col in _brd.columns:
                _pitcher_names.update(_brd[_col].dropna().astype(str).tolist())
        break

# Also include any names already in FIP lookup
if 'mlb_pitcher_fip_lookup' in globals():
    _pitcher_names.update(mlb_pitcher_fip_lookup.keys())

_pitcher_names = {n for n in _pitcher_names if n and n.lower() not in ('nan', 'none', '')}
print(f'Fetching L3 stats for {len(_pitcher_names)} pitchers...')

if 'mlb_pitcher_fip_lookup' not in globals():
    mlb_pitcher_fip_lookup = {}

_hit = 0
_miss = 0
for _raw_name in sorted(_pitcher_names):
    _key = _norm_name(_raw_name)
    if not _key:
        continue
    _pid = _search_player_id(_raw_name)
    if _pid is None:
        _miss += 1
        continue
    _stats = _get_l3_stats(_pid)
    if _key not in mlb_pitcher_fip_lookup:
        mlb_pitcher_fip_lookup[_key] = {}
    for _fk, _fv in _stats.items():
        mlb_pitcher_fip_lookup[_key][_fk] = _fv
    # Opener detection: avg < 3.0 IP/start with enough sample = opener role
    _avg_ip = _stats.get('avg_ip_per_start', np.nan)
    _gs     = _stats.get('gs_season', 0) or 0
    _avg_ip_valid = not (isinstance(_avg_ip, float) and np.isnan(_avg_ip))
    # BUG FIX (MLB-ROLE-1, 2026-09-24 user audit): is_opener/is_bulk_role
    # used to key purely on season-average IP/start -- real incident:
    # Andrew Painter, a genuine full-time rotation starter (nearly every
    # appearance a real start, not a mix with relief outings), got tagged
    # is_bulk_role purely because a stretch of poor performance (pulled
    # early for struggling, not a managed role) dragged his season avg
    # IP/start into the 3-5 range -- triggering a SECOND edge shave on top
    # of whatever his ERA/FIP already earned him. A true opener/bulk-role
    # arm is a real USAGE PATTERN (mixed in with relief appearances, or a
    # confirmed same-day pairing -- see MLB-OPENER-1), not just "his starts
    # have been short lately." Now also requires starts to be a MINORITY
    # of his total appearances (real opener/bulk usage mixes starts with
    # relief outings) -- a pitcher making almost all his appearances as
    # starts is a traditional rotation starter regardless of how short
    # those starts have run, full stop.
    _total_apps = _stats.get('total_appearances_season', _gs) or _gs
    _gs_frac = (_gs / _total_apps) if _total_apps > 0 else 1.0
    _ROLE_GS_FRAC_MIN = 0.85  # this fraction (or more) of appearances being starts = a real rotation starter
    _is_mixed_role = _gs_frac < _ROLE_GS_FRAC_MIN
    mlb_pitcher_fip_lookup[_key]['is_opener'] = (
        _avg_ip_valid and _avg_ip < 3.0 and _gs >= 3 and _is_mixed_role
    )
    # Bulk-arm detection (real gap found 2026-07-23): a "starter" who goes
    # longer than a true opener but still gets pulled well short of a normal
    # 6-inning outing, or whose K rate reads like a reliever's -- run
    # projections that assume a traditional start length underprice the
    # extra bullpen exposure for these games. Distinct from is_opener
    # (sub-3.0 IP/start) -- this catches the ~3-5 IP/start "bulk" profile,
    # or a normal-length outing with an unusually low K/9 for a starter.
    _k9 = mlb_pitcher_fip_lookup[_key].get('k9', np.nan)
    _k9_valid = not (isinstance(_k9, float) and np.isnan(_k9))
    mlb_pitcher_fip_lookup[_key]['is_bulk_role'] = bool(
        _gs >= 3 and _is_mixed_role and (
            (_avg_ip_valid and 3.0 <= _avg_ip < 5.0)
            or (_k9_valid and _k9 < 6.5 and _avg_ip_valid and _avg_ip < 5.5)
        )
    )
    # NEW (MLB-ROLE-1): separate, milder flag for a genuine full-time
    # starter who simply hasn't been going deep lately -- short outings,
    # but a real rotation role, not a bulk/opener usage pattern. Kept
    # distinct so downstream consumers can apply a smaller, more
    # appropriate adjustment instead of treating him like a bullpen-
    # dominant workload.
    mlb_pitcher_fip_lookup[_key]['is_short_outing_starter'] = bool(
        _gs >= 3 and not _is_mixed_role and _avg_ip_valid and _avg_ip < 5.0
    )
    _hit += 1

print(f'L3 stats added: {_hit} pitchers resolved, {_miss} not found')

# Sample output
_sample_l3 = {k: v for k, v in mlb_pitcher_fip_lookup.items() if 'era_l3' in v and not pd.isna(v.get('era_l3', float('nan')))}
print(f'Pitchers with valid L3 ERA: {len(_sample_l3)}')
for _k, _v in list(_sample_l3.items())[:5]:
        print(f'  {_k}: ERA_L3={_v.get("era_l3","?")}, ERA_L9={_v.get("era_l9","?")}, ERA_SZN={_v.get("era_season","?")}, opener={_v.get("is_opener",False)}, bulk={_v.get("is_bulk_role",False)}')


STARTER RECENT FORM â€” L3 STARTS
Fetching L3 stats for 867 pitchers...


L3 stats added: 858 pitchers resolved, 9 not found
Pitchers with valid L3 ERA: 373
  jacob misiorowski: ERA_L3=0.0, ERA_L9=2.27, ERA_SZN=1.8, opener=False, bulk=False
  cristopher sanchez: ERA_L3=4.12, ERA_L9=3.61, ERA_SZN=2.91, opener=False, bulk=False
  cam schlittler: ERA_L3=1.2, ERA_L9=1.17, ERA_SZN=1.95, opener=False, bulk=False
  chris sale: ERA_L3=2.89, ERA_L9=2.34, ERA_SZN=2.16, opener=False, bulk=False
  jesus luzardo: ERA_L3=0.4, ERA_L9=1.85, ERA_SZN=2.87, opener=False, bulk=False


In [22]:
# =========================================================
# CELL — BASEBALL SAVANT STATCAST PITCHER SCRAPER
# Adds hard_hit_pct (ev95percent) and barrel_pct (brl_percent) to
# mlb_pitcher_fip_lookup for use in quality gates in the card runner.
# Source: baseballsavant.mlb.com/leaderboard/statcast (public CSV)
# =========================================================
import requests
import unicodedata, re, io, csv as _csv_mod
import numpy as np

print("=" * 70)
print("BASEBALL SAVANT — STATCAST PITCHER QUALITY METRICS")
print("=" * 70)

_SAVANT_URL = "https://baseballsavant.mlb.com/leaderboard/statcast"
_SAVANT_PARAMS = {
    "min_pitches": 10,
    "position":    "",
    "team":        "",
    "type":        "pitcher",
    "year":        2026,
    "csv":         "true",
}
_SAVANT_HEADERS = {"User-Agent": "Mozilla/5.0 (compatible; research-bot)"}

def _sav_norm(s):
    """Normalize pitcher name to match mlb_pitcher_fip_lookup keys."""
    if not s:
        return ""
    s = unicodedata.normalize("NFD", str(s))
    s = "".join(c for c in s if unicodedata.category(c) != "Mn")
    return re.sub(r"[^a-z ]", "", s.lower()).strip()

def _sav_flip_name(raw):
    """Convert 'Last, First' → 'First Last' then normalize."""
    raw = raw.strip().strip('"')
    if "," in raw:
        parts = raw.split(",", 1)
        raw = parts[1].strip() + " " + parts[0].strip()
    return _sav_norm(raw)

_hit = 0
_miss = 0

try:
    _r = requests.get(_SAVANT_URL, params=_SAVANT_PARAMS, headers=_SAVANT_HEADERS, timeout=20)
    _r.raise_for_status()
    # Strip BOM, parse CSV
    _text = _r.content.decode("utf-8-sig")
    _reader = _csv_mod.DictReader(io.StringIO(_text))
    for _row in _reader:
        # Column "last_name, first_name" is the combined name field (with comma)
        _raw_name = next(
            (v for k, v in _row.items() if "last_name" in k.lower()),
            ""
        )
        if not _raw_name:
            continue
        _key = _sav_flip_name(_raw_name)
        if not _key:
            continue

        def _sf(col, default=np.nan):
            v = _row.get(col, "")
            try:
                return float(v) if v not in ("", "null", "NULL", None) else default
            except (ValueError, TypeError):
                return default

        _hard_hit = _sf("ev95percent")
        _barrel   = _sf("brl_percent")
        _brl_pa   = _sf("brl_pa")
        _xwoba    = _sf("xwoba") if "xwoba" in _row else np.nan

        if "mlb_pitcher_fip_lookup" not in globals():
            mlb_pitcher_fip_lookup = {}
        if _key not in mlb_pitcher_fip_lookup:
            mlb_pitcher_fip_lookup[_key] = {}

        mlb_pitcher_fip_lookup[_key]["hard_hit_pct"] = _hard_hit
        mlb_pitcher_fip_lookup[_key]["barrel_pct"]   = _barrel
        _swstr = float(_row.get("whiff_percent", _row.get("swstr_pct", "nan")) or "nan")
        _csw   = float(_row.get("csw_percentage", _row.get("csw_pct", "nan")) or "nan")
        mlb_pitcher_fip_lookup[_key]["swstr_pct"] = _swstr if not (_swstr != _swstr) else float("nan")
        mlb_pitcher_fip_lookup[_key]["csw_pct"]   = _csw   if not (_csw   != _csw)   else float("nan")
        mlb_pitcher_fip_lookup[_key]["brl_pa"]        = _brl_pa
        if not np.isnan(_xwoba):
            mlb_pitcher_fip_lookup[_key]["xwoba"] = _xwoba
        _hit += 1

    print(f"  Statcast metrics loaded for {_hit} pitchers")
    if _hit > 0:
        # Sample: find a few well-known pitchers to verify
        _samples = ["gerrit cole", "pablo lopez", "corbin burnes", "zack wheeler"]
        for _s in _samples:
            _v = mlb_pitcher_fip_lookup.get(_s)
            if _v and not np.isnan(_v.get("hard_hit_pct", np.nan)):
                print(f"    {_s}: HH%={_v['hard_hit_pct']:.1f}  Brl%={_v.get('barrel_pct', float('nan')):.1f}")
except Exception as _sav_ex:
    print(f"  [WARN] Statcast scraper failed: {_sav_ex}")
    print("  Hard hit% and barrel% will not be available for today's card")

# =========================================================
# xERA fetch (Baseball Savant expected_statistics, season-level).
# Requested to dampen L3-ERA small-sample overcorrection (e.g. one bad
# start spiking a pitcher's rolling ERA) by blending in a quality-of-
# contact-based signal that isn't sensitive to short-run results variance.
# Separate endpoint from the leaderboard/statcast one above -- that one
# doesn't carry xera as a column; expected_statistics does.
# =========================================================
_XERA_URL = "https://baseballsavant.mlb.com/leaderboard/expected_statistics"
_XERA_PARAMS = {
    "type":     "pitcher",
    "year":     2026,
    "position": "",
    "team":     "",
    "min":      "1",
    "csv":      "true",
}

_xera_hit = 0
try:
    _xr = requests.get(_XERA_URL, params=_XERA_PARAMS, headers=_SAVANT_HEADERS, timeout=20)
    _xr.raise_for_status()
    _xtext = _xr.content.decode("utf-8-sig")
    _xreader = _csv_mod.DictReader(io.StringIO(_xtext))
    for _xrow in _xreader:
        _xraw_name = next(
            (v for k, v in _xrow.items() if "last_name" in k.lower()),
            ""
        )
        if not _xraw_name:
            continue
        _xkey = _sav_flip_name(_xraw_name)
        if not _xkey:
            continue
        _xv = _xrow.get("xera", "")
        try:
            _xera_val = float(_xv) if _xv not in ("", "null", "NULL", None) else None
        except (ValueError, TypeError):
            _xera_val = None
        if _xera_val is None:
            continue
        if "mlb_pitcher_fip_lookup" not in globals():
            mlb_pitcher_fip_lookup = {}
        if _xkey not in mlb_pitcher_fip_lookup:
            mlb_pitcher_fip_lookup[_xkey] = {}
        mlb_pitcher_fip_lookup[_xkey]["xera"] = _xera_val
        _xera_hit += 1
    print(f"  xERA loaded for {_xera_hit} pitchers (season, Baseball Savant expected_statistics)")
except Exception as _xera_ex:
    print(f"  [WARN] xERA fetch failed: {_xera_ex}")
    print("  xERA blend will fall back to raw L3 ERA only for today's card")


BASEBALL SAVANT — STATCAST PITCHER QUALITY METRICS


  Statcast metrics loaded for 339 pitchers
    gerrit cole: HH%=44.6  Brl%=8.9
    zack wheeler: HH%=35.2  Brl%=6.0
  xERA loaded for 868 pitchers (season, Baseball Savant expected_statistics)


In [23]:
# =========================================================
# CELL â€” FANGRAPHS TEAM BATTING K% / BB% SCRAPER
# Fetches team-level plate discipline metrics.
# mlb_team_discipline: {norm_team: {k_pct_bat, bb_pct_bat}}
# mlb_team_discipline_hist: {(norm_team, year): {k_pct_bat, bb_pct_bat}}
# =========================================================
import requests
import pandas as pd
import numpy as np
import unicodedata, re

print('=' * 70)
print('FANGRAPHS TEAM BATTING K% / BB% SCRAPER')
print('=' * 70)

def _norm_team_fg(s):
    if not s:
        return ''
    return re.sub(r'[^a-z0-9]', '', str(s).lower().strip())

_FG_BAT_URL = 'https://www.fangraphs.com/api/leaders/major-league/data'

def _fetch_fg_team_bat(year):
    params = {
        'pos': 'all', 'stats': 'bat', 'lg': 'all', 'qual': 0,
        'type': 8, 'season': year, 'month': 0, 'season1': year,
        'ind': 0, 'pageitems': 100, 'pagenum': 1, 'team': '0,ts',
    }
    try:
        r = requests.get(_FG_BAT_URL, params=params,
                         headers={'User-Agent': 'Mozilla/5.0'}, timeout=12)
        r.raise_for_status()
        rows = r.json().get('data', [])
        if not rows:
            return pd.DataFrame()
        df = pd.DataFrame(rows)
        df['_year'] = year
        return df
    except Exception as e:
        print(f'  WARNING year {year}: {e}')
        return pd.DataFrame()

_HIST_YEARS = list(range(2021, 2027))
_all_frames = []
for _yr in _HIST_YEARS:
    _df = _fetch_fg_team_bat(_yr)
    if not _df.empty:
        _all_frames.append(_df)
        print(f'  {_yr}: {len(_df)} teams')
    else:
        print(f'  {_yr}: no data')

mlb_team_discipline      = {}  # current season
mlb_team_discipline_hist = {}  # (norm_team, year)

if _all_frames:
    _hist = pd.concat(_all_frames, ignore_index=True)

    _team_col  = next((c for c in _hist.columns if c in ('Team','TeamName','team')), None)
    _kpct_col  = next((c for c in _hist.columns if c in ('K%','k_pct','K_pct')), None)
    _bbpct_col = next((c for c in _hist.columns if c in ('BB%','bb_pct','BB_pct')), None)

    print(f'Team col: {_team_col} | K%: {_kpct_col} | BB%: {_bbpct_col}')

    if _team_col and _kpct_col and _bbpct_col:
        for _, row in _hist.iterrows():
            tnorm = _norm_team_fg(str(row.get(_team_col, '')))
            yr    = int(row['_year'])
            rec   = {
                'k_pct_bat':  float(row[_kpct_col])  if pd.notna(row.get(_kpct_col))  else np.nan,
                'bb_pct_bat': float(row[_bbpct_col]) if pd.notna(row.get(_bbpct_col)) else np.nan,
            }
            mlb_team_discipline_hist[(tnorm, yr)] = rec
            if yr == max(_HIST_YEARS):
                mlb_team_discipline[tnorm] = rec

        print(f'Historical discipline: {len(mlb_team_discipline_hist)} (team, year) pairs')
        print(f'Current season  ({max(_HIST_YEARS)}): {len(mlb_team_discipline)} teams')
        _sample = list(mlb_team_discipline.items())[:3]
        for t, v in _sample:
            print(f'  {t}: {v}')
    else:
        print('WARNING: could not find K%/BB% columns in Fangraphs response')
else:
    print('WARNING: no team discipline data fetched')

globals()['mlb_team_discipline']      = mlb_team_discipline
globals()['mlb_team_discipline_hist'] = mlb_team_discipline_hist


FANGRAPHS TEAM BATTING K% / BB% SCRAPER
  WARNING year 2021: 403 Client Error: Forbidden for url: https://www.fangraphs.com/api/leaders/major-league/data?pos=all&stats=bat&lg=all&qual=0&type=8&season=2021&month=0&season1=2021&ind=0&pageitems=100&pagenum=1&team=0%2Cts
  2021: no data
  WARNING year 2022: 403 Client Error: Forbidden for url: https://www.fangraphs.com/api/leaders/major-league/data?pos=all&stats=bat&lg=all&qual=0&type=8&season=2022&month=0&season1=2022&ind=0&pageitems=100&pagenum=1&team=0%2Cts
  2022: no data


  WARNING year 2023: 403 Client Error: Forbidden for url: https://www.fangraphs.com/api/leaders/major-league/data?pos=all&stats=bat&lg=all&qual=0&type=8&season=2023&month=0&season1=2023&ind=0&pageitems=100&pagenum=1&team=0%2Cts
  2023: no data
  WARNING year 2024: 403 Client Error: Forbidden for url: https://www.fangraphs.com/api/leaders/major-league/data?pos=all&stats=bat&lg=all&qual=0&type=8&season=2024&month=0&season1=2024&ind=0&pageitems=100&pagenum=1&team=0%2Cts
  2024: no data
  WARNING year 2025: 403 Client Error: Forbidden for url: https://www.fangraphs.com/api/leaders/major-league/data?pos=all&stats=bat&lg=all&qual=0&type=8&season=2025&month=0&season1=2025&ind=0&pageitems=100&pagenum=1&team=0%2Cts
  2025: no data


  WARNING year 2026: 403 Client Error: Forbidden for url: https://www.fangraphs.com/api/leaders/major-league/data?pos=all&stats=bat&lg=all&qual=0&type=8&season=2026&month=0&season1=2026&ind=0&pageitems=100&pagenum=1&team=0%2Cts
  2026: no data


In [24]:
# =========================================================
# CELL â€” ENRICH train_df WITH HISTORICAL TEAM K%/BB%
# Joins mlb_team_discipline_hist by (team, season) so that
# compressed_feature_cols can include diff_k_pct_bat / diff_bb_pct_bat.
# Run this BEFORE Cell 51 (compressed CV) to include in retrain.
# =========================================================
import pandas as pd
import numpy as np
import re

print('=' * 70)
print('ENRICH train_df WITH HISTORICAL TEAM K%/BB%')
print('=' * 70)

if 'train_df' not in globals():
    raise ValueError('train_df not found. Run Cell 4 first.')

# If Fangraphs discipline data is unavailable, add NaN columns and continue.
# Cell 51's soft-filter will exclude features with insufficient non-NaN rows.
_disc_available = ('mlb_team_discipline_hist' in globals() and bool(mlb_team_discipline_hist))
if not _disc_available:
    print('WARNING: mlb_team_discipline_hist is empty â€” adding NaN discipline columns.')
    print('  Cell 51 soft-filter will skip these features during training.')
    _td = train_df.copy()
    for _col in ['away_k_pct_bat','away_bb_pct_bat','home_k_pct_bat','home_bb_pct_bat',
                 'diff_k_pct_bat','diff_bb_pct_bat',
                 'away_starter_fip','home_starter_fip',
                 'away_starter_k_bb_pct','home_starter_k_bb_pct',
                 'diff_starter_fip','diff_starter_k_bb_pct']:
        if _col not in _td.columns:
            _td[_col] = float('nan')
    train_df = _td.copy()
    globals()['train_df'] = train_df
    print('Done â€” NaN columns added. Skipping historical K%/BB% join.')
else:

    def _norm_team_fd(s):
        return re.sub(r'[^a-z0-9]', '', str(s).lower().strip())

    df = train_df.copy()
    df['_year'] = pd.to_datetime(df['date'], errors='coerce').dt.year
    if 'season' in df.columns:
        df['_year'] = df['season'].fillna(df['_year']).astype('Int64')

    def _lookup_disc(team, year):
        key = (_norm_team_fd(team), int(year)) if pd.notna(year) else None
        return mlb_team_discipline_hist.get(key, {'k_pct_bat': np.nan, 'bb_pct_bat': np.nan})

    _away_k, _away_bb, _home_k, _home_bb = [], [], [], []
    for _, row in df.iterrows():
        yr = row['_year']
        _a = _lookup_disc(row.get('away_team', ''), yr)
        _h = _lookup_disc(row.get('home_team', ''), yr)
        _away_k.append(_a['k_pct_bat'])
        _away_bb.append(_a['bb_pct_bat'])
        _home_k.append(_h['k_pct_bat'])
        _home_bb.append(_h['bb_pct_bat'])

    df['away_k_pct_bat']  = _away_k
    df['away_bb_pct_bat'] = _away_bb
    df['home_k_pct_bat']  = _home_k
    df['home_bb_pct_bat'] = _home_bb
    df['diff_k_pct_bat']  = df['home_k_pct_bat']  - df['away_k_pct_bat']
    df['diff_bb_pct_bat'] = df['home_bb_pct_bat'] - df['away_bb_pct_bat']

    # Note: starter FIP diff requires per-game pitcher lookup (not in historical CSV)
    # Use NaN for now; model will drop rows with NaN if feature included
    # Safe to add starter_fip_diff after historical pitcher data is available
    if 'away_starter_fip' not in df.columns:
        df['away_starter_fip']      = np.nan
        df['home_starter_fip']      = np.nan
        df['away_starter_k_bb_pct'] = np.nan
        df['home_starter_k_bb_pct'] = np.nan
    df['diff_starter_fip']      = df['home_starter_fip']      - df['away_starter_fip']
    df['diff_starter_k_bb_pct'] = df['home_starter_k_bb_pct'] - df['away_starter_k_bb_pct']

    _non_nan_k  = df['diff_k_pct_bat'].notna().sum()
    _non_nan_bb = df['diff_bb_pct_bat'].notna().sum()
    print(f'diff_k_pct_bat  non-NaN rows: {_non_nan_k}/{len(df)}')
    print(f'diff_bb_pct_bat non-NaN rows: {_non_nan_bb}/{len(df)}')

    train_df = df.copy()
    globals()['train_df'] = train_df
    print('train_df updated with K%/BB% + starter FIP diff columns.')
    print('Run Cell 51 to retrain with these features.')


ENRICH train_df WITH HISTORICAL TEAM K%/BB%
  Cell 51 soft-filter will skip these features during training.
Done â€” NaN columns added. Skipping historical K%/BB% join.


In [25]:
# %%
# =========================================================
# CELL 10 Ã¢â‚¬â€ FILTER TO REAL MLB TEAMS ONLY
# Remove any weird/non-MLB team names from snapshot
# =========================================================

if "mlb_latest_team_snapshot" not in globals() or not isinstance(mlb_latest_team_snapshot, pd.DataFrame):
    raise ValueError("mlb_latest_team_snapshot not found. Run Cell 9 first.")

MLB_TEAMS = [
    "Arizona Diamondbacks",
    "Atlanta Braves",
    "Baltimore Orioles",
    "Boston Red Sox",
    "Chicago Cubs",
    "Chicago White Sox",
    "Cincinnati Reds",
    "Cleveland Guardians",
    "Colorado Rockies",
    "Detroit Tigers",
    "Houston Astros",
    "Kansas City Royals",
    "Los Angeles Angels",
    "Los Angeles Dodgers",
    "Miami Marlins",
    "Milwaukee Brewers",
    "Minnesota Twins",
    "New York Mets",
    "New York Yankees",
    "Oakland Athletics",
    "Philadelphia Phillies",
    "Pittsburgh Pirates",
    "San Diego Padres",
    "San Francisco Giants",
    "Seattle Mariners",
    "St. Louis Cardinals",
    "Tampa Bay Rays",
    "Texas Rangers",
    "Toronto Blue Jays",
    "Washington Nationals",
]

team_snapshot_clean = mlb_latest_team_snapshot.copy()

# 2026-08-14 (user audit, real bug -- this is the actual root cause, one
# level upstream of where it first looked like it lived): MLB_TEAMS above
# still lists "Oakland Athletics", but the live snapshot source
# (mlb_latest_team_snapshot) already reports the team as bare "Athletics"
# (their real 2025+ relocation-era rename ahead of the Las Vegas move). The
# .isin(MLB_TEAMS) filter below silently dropped every "Athletics" row as a
# non-MLB entity -- which then made every downstream cell (five of them
# already carry their own "Athletics": "Oakland Athletics" TEAM_NAME_FIXES
# patch for this same rename) receive a team snapshot with no A's data at
# all, and made today's Rangers @ Athletics game get misclassified as a
# dropped "exhibition game" in the schedule-pull cell. Normalizing to the
# historical canonical name here, before the allowlist filter, means every
# downstream historical join (park factors, ELO, etc. -- all keyed on
# "Oakland Athletics" from years of pre-rename data) keeps working exactly
# as it already assumes, using the same alias those five other cells use.
_TEAM_NAME_FIXES_PRE = {"Athletics": "Oakland Athletics"}
team_snapshot_clean["team"] = team_snapshot_clean["team"].replace(_TEAM_NAME_FIXES_PRE)

before_rows = len(team_snapshot_clean)
before_teams = team_snapshot_clean["team"].nunique()

team_snapshot_clean = team_snapshot_clean[
    team_snapshot_clean["team"].isin(MLB_TEAMS)
].copy()

team_snapshot_clean = team_snapshot_clean.sort_values(["team", "date"]).reset_index(drop=True)

after_rows = len(team_snapshot_clean)
after_teams = team_snapshot_clean["team"].nunique()

# 2026-08-14: diff against the RENAMED (post-alias, pre-filter) team set,
# not the raw pre-rename source -- otherwise "Athletics" always shows up
# here as "removed" even on a run where every row survived, just under its
# renamed key. That's a rename, not a removal, and this list should only
# ever report genuine drops.
_pre_filter_teams = set(mlb_latest_team_snapshot["team"].replace(_TEAM_NAME_FIXES_PRE))
removed_teams = sorted(_pre_filter_teams - set(team_snapshot_clean["team"]))

print("Before filtering:")
print("Rows        :", before_rows)
print("Unique teams:", before_teams)

print("\nAfter filtering:")
print("Rows        :", after_rows)
print("Unique teams:", after_teams)

print("\nRemoved non-MLB teams/entities:")
if removed_teams:
    for t in removed_teams:
        print("-", t)
else:
    print("None")

print("\nClean MLB team snapshot preview:")
display(team_snapshot_clean.head(15))

mlb_team_snapshot_clean = team_snapshot_clean.copy()

Before filtering:
Rows        : 30
Unique teams: 30

After filtering:
Rows        : 30
Unique teams: 30

Removed non-MLB teams/entities:
None

Clean MLB team snapshot preview:


,date,team,runs_for,runs_against,win,park_factor,opponent,home_away,opp_def_rating,adj_runs_for,...,margin_avg_5,win_rate_5,runs_for_avg_10_home,runs_against_avg_10_home,runs_for_avg_10_away,runs_against_avg_10_away,bullpen_stress_3,bullpen_runs_allowed_est,bullpen_ra_avg_10_live,rest_days
0,2026-09-27,Arizona Diamondbacks,4,9,0,0.95,San Diego Padres,away,3.729167,5.00,...,0.520661,0.411747,5.019150,4.578513,6.237057,5.729047,20.65,3.15,1.955369,13
1,2026-10-07,Atlanta Braves,1,4,0,1.01,Los Angeles Dodgers,home,3.191489,1.34,...,-0.598487,0.264299,3.476076,3.857539,4.126913,3.414394,8.60,1.40,1.184877,3
2,2026-09-25,Baltimore Orioles,10,2,1,1.02,New York Yankees,away,3.416667,12.72,...,3.867182,0.676990,3.768235,3.101557,6.085037,3.839330,9.47,0.70,1.254105,15
3,2026-09-30,Boston Red Sox,2,9,0,1.02,New York Yankees,away,3.416667,2.54,...,-4.085200,0.205817,2.383687,2.669319,2.357511,5.125338,21.63,3.15,1.399331,10
4,2026-09-30,Chicago Cubs,1,4,0,0.95,San Diego Padres,away,3.729167,1.25,...,-2.224362,0.220711,4.362968,4.215786,2.970111,4.105826,13.21,1.40,1.417128,10
5,2026-10-08,Chicago White Sox,5,9,0,1.00,Cleveland Guardians,home,3.872340,5.72,...,-0.914264,0.416838,4.498900,5.477470,5.398598,3.306545,19.34,3.15,1.464214,2
6,2026-09-27,Cincinnati Reds,1,5,0,1.01,Toronto Blue Jays,away,3.928571,1.12,...,-0.516599,0.385732,3.546854,4.980439,4.462266,5.299621,12.62,1.75,1.567679,13
7,2026-10-08,Cleveland Guardians,9,5,1,1.00,Chicago White Sox,away,4.450980,8.96,...,2.727545,0.672873,3.447089,3.631371,6.411074,3.772901,10.99,1.75,1.242621,2
8,2026-09-27,Colorado Rockies,2,4,0,1.00,Chicago White Sox,away,4.450980,1.99,...,-1.570442,0.245764,3.351651,6.332449,3.659023,6.086431,14.64,1.40,2.285690,13
9,2026-09-27,Detroit Tigers,2,4,0,0.98,Pittsburgh Pirates,home,3.761905,2.41,...,-0.356362,0.438182,4.618446,4.084049,4.084092,4.974593,15.79,1.40,1.504525,13


In [26]:
# %%
# =========================================================
# CELL 11 Ã¢â‚¬â€ PULL TODAY'S MLB SCHEDULE
# Build today's clean game slate from MLB StatsAPI
# =========================================================

import pandas as pd
import numpy as np
import requests

if "mlb_team_snapshot_clean" not in globals() or not isinstance(mlb_team_snapshot_clean, pd.DataFrame):
    raise ValueError("mlb_team_snapshot_clean not found. Run Cell 10 first.")

import datetime as _cd
_card_hour = _cd.datetime.now().hour
# After 8 PM: pull NEXT DAY so card is ready same night
_card_delta = 1 if _card_hour >= 20 else 0
CARD_DATE = (_cd.date.today() + _cd.timedelta(days=_card_delta)).strftime("%Y-%m-%d")
print(f"Card date: {CARD_DATE} (tomorrow)" if _card_delta else f"Card date: {CARD_DATE} (today)")
today_str = CARD_DATE
url = f"https://statsapi.mlb.com/api/v1/schedule?sportId=1&date={today_str}"

print("Fetching MLB schedule for:", today_str)
print("URL:", url)

try:
    r = requests.get(url, timeout=30)
    r.raise_for_status()
    data = r.json()
except Exception as e:
    raise RuntimeError(f"Failed to fetch MLB schedule: {e}")

games = []
for date_block in data.get("dates", []):
    for g in date_block.get("games", []):
        games.append({
            "game_id": g.get("gamePk"),
            "game_datetime_utc": g.get("gameDate"),
            "away_team": g.get("teams", {}).get("away", {}).get("team", {}).get("name"),
            "home_team": g.get("teams", {}).get("home", {}).get("team", {}).get("name"),
            "away_team_id": g.get("teams", {}).get("away", {}).get("team", {}).get("id"),
            "home_team_id": g.get("teams", {}).get("home", {}).get("team", {}).get("id"),
            "venue_name": g.get("venue", {}).get("name"),
            "status": g.get("status", {}).get("detailedState"),
            # MLB-POSTSEASON-1 (2026-09-29, user audit): real game-type/series
            # fields the MLB Stats API already returns on every game (live-
            # verified 2026-09-29 -- today's 4 real Wild Card games each carry
            # gameType='F', seriesGameNumber, gamesInSeries, seriesDescription)
            # -- captured here, once, at the source, rather than re-fetched by
            # every downstream cell that needs to know "is this postseason" or
            # "what's the series state." No gameType filter is applied to the
            # schedule URL above -- confirmed live that the API already
            # includes postseason games by default with zero filter params.
            "game_type": g.get("gameType"),
            "series_game_number": g.get("seriesGameNumber"),
            "games_in_series": g.get("gamesInSeries"),
            "series_description": g.get("seriesDescription"),
        })

today_games = pd.DataFrame(games, columns=[
    "game_id", "game_datetime_utc", "away_team", "home_team",
    "away_team_id", "home_team_id", "venue_name", "status",
    "game_type", "series_game_number", "games_in_series", "series_description",
])

# ---------------------------------------------------------
# filter out exhibition / non-real-team entries (All-Star Game,
# Home Run Derby, Futures Game, etc.) that MLB StatsAPI includes
# on the schedule but that have no team-level history to score
# ---------------------------------------------------------
if len(today_games) > 0:
    real_teams = set(mlb_team_snapshot_clean["team"].unique())
    # 2026-08-14 (user audit, real bug): the A's real-world 2025+ rename from
    # "Oakland Athletics" to just "Athletics" (relocating to Las Vegas) means
    # today's live schedule correctly says "Athletics", but the historical
    # snapshot this exhibition filter checks against still only has "Oakland
    # Athletics" -- so every real Athletics game got misclassified as a fake
    # non-MLB-team exhibition game and silently dropped before ever reaching
    # the board. Five DOWNSTREAM cells already carry a
    # TEAM_NAME_FIXES = {"Athletics": "Oakland Athletics"} patch for this
    # exact rename, but this cell runs first and drops the game before any of
    # them get a chance to run. Same alias, applied to the recognized-team
    # set here (not to today_games itself, so those downstream cells' own
    # renaming logic is untouched).
    _TEAM_NAME_ALIASES = {"Athletics": "Oakland Athletics"}
    real_teams |= {alias for alias, canonical in _TEAM_NAME_ALIASES.items() if canonical in real_teams}
    is_real_matchup = (
        today_games["away_team"].isin(real_teams)
        & today_games["home_team"].isin(real_teams)
    )
    dropped = today_games[~is_real_matchup]
    if len(dropped) > 0:
        print()
        print(f"Dropping {len(dropped)} non-MLB-team exhibition game(s) (e.g. All-Star Game):")
        for _, r in dropped.iterrows():
            print(f" - {r['away_team']} @ {r['home_team']}")
    today_games = today_games[is_real_matchup].reset_index(drop=True)

if len(today_games) == 0:
    print("\nNo MLB games found for today.")
else:
    today_games["game_datetime_utc"] = pd.to_datetime(
        today_games["game_datetime_utc"],
        errors="coerce",
        utc=True
    )
    today_games = today_games.sort_values("game_datetime_utc").reset_index(drop=True)

    print("\nToday's games shape:", today_games.shape)
    print("Unique away teams   :", today_games["away_team"].nunique())
    print("Unique home teams   :", today_games["home_team"].nunique())

    print("\nToday's games preview:")
    display(today_games)

mlb_today_games = today_games.copy()

Card date: 2026-10-10 (today)
Fetching MLB schedule for: 2026-10-10
URL: https://statsapi.mlb.com/api/v1/schedule?sportId=1&date=2026-10-10



Today's games shape: (1, 12)
Unique away teams   : 1
Unique home teams   : 1

Today's games preview:


,game_id,game_datetime_utc,away_team,home_team,away_team_id,home_team_id,venue_name,status,game_type,series_game_number,games_in_series,series_description
0,849831,2026-10-11 00:00:00+00:00,Chicago White Sox,Cleveland Guardians,145,114,Progressive Field,Scheduled,D,5,5,AL Division Series


In [27]:
# %%
# =========================================================
# CELL 12A Ã¢â‚¬â€ PULL ESPN STARTERS FOR TODAY
# Fallback starter source for games where MLB probablePitcher
# is missing or incomplete
# =========================================================

import pandas as pd
import numpy as np
import requests

print("=" * 70)
print("CELL 12A Ã¢â‚¬â€ PULL ESPN STARTERS FOR TODAY")
print("=" * 70)

# ---------------------------------------------------------
# 0) validate prior cell output
# ---------------------------------------------------------
if "mlb_today_games" not in globals() or not isinstance(mlb_today_games, pd.DataFrame):
    raise ValueError("mlb_today_games not found. Run Cell 11 first.")

today_games_check = mlb_today_games.copy()

if len(today_games_check) == 0:
    print("No games found in mlb_today_games.")
    espn_today_starters = pd.DataFrame(columns=[
        "espn_event_id", "away_team", "home_team",
        "away_starter_name_espn", "away_starter_id_espn",
        "home_starter_name_espn", "home_starter_id_espn",
        "starter_source_espn",
    ])
else:
    # ---------------------------------------------------------
    # 1) ESPN scoreboard endpoint
    # ---------------------------------------------------------
    ESPN_URL = "https://site.api.espn.com/apis/site/v2/sports/baseball/mlb/scoreboard"

    print(f"Fetching ESPN scoreboard from: {ESPN_URL}")

    # 2026-08-31 BUG FIX (confirmed live via curl): every ESPN call from
    # this network is edge-blocked by Akamai with a 403 "Access Denied"
    # page (Server: AkamaiGHost -- ESPN's own app never sees the
    # request), not a rate limit. Direct call still tried first; only a
    # 403 retries once through r.jina.ai's free reader-proxy relay
    # (different egress IP, verified live to return a real 200 against
    # this exact endpoint).
    class _EspnRelayResponse12A:
        def __init__(self, status_code, json_data):
            self.status_code = status_code
            self._json_data = json_data
        def json(self):
            # BUG FIX (found live 2026-08-31, real crash elsewhere in this
            # notebook): raise instead of silently returning None on total
            # failure -- defense in depth even though this cell's caller
            # already calls raise_for_status() first.
            if self._json_data is None:
                raise ValueError("ESPN fetch failed (both direct call and relay fallback)")
            return self._json_data
        def raise_for_status(self):
            if self.status_code != 200:
                raise requests.HTTPError(f"{self.status_code} Client Error (ESPN relay fallback)")

    def _espn_get_12a(url, timeout=30):
        try:
            r = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=timeout)
            if r.status_code != 403:
                return r
        except Exception:
            pass
        try:
            rr = requests.get(f"https://r.jina.ai/{url}", timeout=timeout + 20)
            rr.raise_for_status()
            body = rr.text
            idx = body.find("Markdown Content:")
            json_text = body[idx + len("Markdown Content:"):].strip() if idx >= 0 else body
            import json as _json_12a
            return _EspnRelayResponse12A(200, _json_12a.loads(json_text))
        except Exception:
            return _EspnRelayResponse12A(403, None)

    try:
        r = _espn_get_12a(ESPN_URL, timeout=30)
        r.raise_for_status()
        data = r.json()
    except Exception as e:
        raise RuntimeError(f"Failed to fetch ESPN scoreboard: {e}")

    events = data.get("events", []) or []
    print(f"Events returned from ESPN: {len(events)}")

    # ---------------------------------------------------------
    # 2) team-name normalization
    # match ESPN names to your MLB StatsAPI names
    # ---------------------------------------------------------
    TEAM_NAME_FIXES = {
        "Athletics": "Oakland Athletics",
    }

    def norm_team_name(name):
        if pd.isna(name):
            return np.nan
        name = str(name).strip()
        return TEAM_NAME_FIXES.get(name, name)

    # ---------------------------------------------------------
    # 3) safe probable-pitcher extractor
    # ---------------------------------------------------------
    def extract_probable_pitcher(competitor):
        """
        ESPN structures can vary a bit.
        Try the common places safely.
        """
        if not isinstance(competitor, dict):
            return {"starter_name": np.nan, "starter_id": np.nan}

        # common location
        probables = competitor.get("probables", None)

        if isinstance(probables, list) and len(probables) > 0:
            p0 = probables[0] or {}
            athlete = p0.get("athlete", {}) or {}

            starter_name = athlete.get("displayName", np.nan)
            starter_id = athlete.get("id", np.nan)

            return {
                "starter_name": starter_name,
                "starter_id": pd.to_numeric(starter_id, errors="coerce")
            }

        return {"starter_name": np.nan, "starter_id": np.nan}

    # ---------------------------------------------------------
    # 4) parse ESPN events
    # ---------------------------------------------------------
    rows = []

    for event in events:
        competitions = event.get("competitions", []) or []
        if len(competitions) == 0:
            continue

        comp = competitions[0]
        competitors = comp.get("competitors", []) or []

        away_team_block = None
        home_team_block = None

        for c in competitors:
            if str(c.get("homeAway", "")).lower() == "away":
                away_team_block = c
            elif str(c.get("homeAway", "")).lower() == "home":
                home_team_block = c

        if away_team_block is None or home_team_block is None:
            continue

        away_team_name = norm_team_name(
            ((away_team_block.get("team") or {}).get("displayName"))
        )
        home_team_name = norm_team_name(
            ((home_team_block.get("team") or {}).get("displayName"))
        )

        away_pitcher = extract_probable_pitcher(away_team_block)
        home_pitcher = extract_probable_pitcher(home_team_block)

        rows.append({
            "espn_event_id": event.get("id"),
            "away_team": away_team_name,
            "home_team": home_team_name,
            "away_starter_name_espn": away_pitcher["starter_name"],
            "away_starter_id_espn": away_pitcher["starter_id"],
            "home_starter_name_espn": home_pitcher["starter_name"],
            "home_starter_id_espn": home_pitcher["starter_id"],
            "starter_source_espn": "espn_probables"
        })

    espn_today_starters = pd.DataFrame(rows)

    # ---------------------------------------------------------
    # 5) cleanup / dedupe
    # ---------------------------------------------------------
    if len(espn_today_starters) > 0:
        espn_today_starters = (
            espn_today_starters
            .drop_duplicates(subset=["away_team", "home_team"], keep="last")
            .sort_values(["away_team", "home_team"])
            .reset_index(drop=True)
        )

    # ---------------------------------------------------------
    # 6) debug prints
    # ---------------------------------------------------------
    print("")
    print("ESPN starter table shape:", espn_today_starters.shape)

    if len(espn_today_starters) > 0:
        print("\nMissing ESPN away starters:",
              int(espn_today_starters["away_starter_name_espn"].isna().sum()))
        print("Missing ESPN home starters:",
              int(espn_today_starters["home_starter_name_espn"].isna().sum()))

        print("\nPreview:")
        display(espn_today_starters.head(20))
    else:
        print("No ESPN starter rows were parsed.")

# ---------------------------------------------------------
# 7) save in memory
# ---------------------------------------------------------
globals()["espn_today_starters"] = espn_today_starters.copy()

print("")
print("Created in memory:")
print(" - espn_today_starters")
print("Done.")

CELL 12A Ã¢â‚¬â€ PULL ESPN STARTERS FOR TODAY
Fetching ESPN scoreboard from: https://site.api.espn.com/apis/site/v2/sports/baseball/mlb/scoreboard
Events returned from ESPN: 1

ESPN starter table shape: (1, 8)

Missing ESPN away starters: 0
Missing ESPN home starters: 0

Preview:


,espn_event_id,away_team,home_team,away_starter_name_espn,away_starter_id_espn,home_starter_name_espn,home_starter_id_espn,starter_source_espn
0,401907993,Cleveland Guardians,Chicago White Sox,Parker Messick,4619898,Hagen Smith,5023126,espn_probables



Created in memory:
 - espn_today_starters
Done.


In [28]:
# %%
# =========================================================
# CELL 12B Ã¢â‚¬â€ MERGE MLB TODAY GAMES WITH ESPN STARTERS
# Build one daily starter table for today's slate
# =========================================================

import pandas as pd
import numpy as np

print("=" * 70)
print("CELL 12B Ã¢â‚¬â€ MERGE MLB TODAY GAMES WITH ESPN STARTERS")
print("=" * 70)

# ---------------------------------------------------------
# 0) validate required inputs
# ---------------------------------------------------------
if "mlb_today_games" not in globals() or not isinstance(mlb_today_games, pd.DataFrame):
    raise ValueError("mlb_today_games not found. Run Cell 11 first.")

if "espn_today_starters" not in globals() or not isinstance(espn_today_starters, pd.DataFrame):
    raise ValueError("espn_today_starters not found. Run Cell 12A first.")

today_games = mlb_today_games.copy()
espn_starters = espn_today_starters.copy()

# ---------------------------------------------------------
# 1) normalize team names on MLB side
# ---------------------------------------------------------
TEAM_NAME_FIXES = {
    "Athletics": "Oakland Athletics",
}

def norm_team_name(name):
    if pd.isna(name):
        return np.nan
    name = str(name).strip()
    return TEAM_NAME_FIXES.get(name, name)

today_games["away_team_norm"] = today_games["away_team"].apply(norm_team_name)
today_games["home_team_norm"] = today_games["home_team"].apply(norm_team_name)

espn_starters["away_team_norm"] = espn_starters["away_team"].apply(norm_team_name)
espn_starters["home_team_norm"] = espn_starters["home_team"].apply(norm_team_name)

# ---------------------------------------------------------
# 2) merge ESPN starters onto today's MLB slate
# ---------------------------------------------------------
merged_today_starters = today_games.merge(
    espn_starters[
        [
            "away_team_norm",
            "home_team_norm",
            "away_starter_name_espn",
            "away_starter_id_espn",
            "home_starter_name_espn",
            "home_starter_id_espn",
            "starter_source_espn",
            "espn_event_id",
        ]
    ].drop_duplicates(subset=["away_team_norm", "home_team_norm"], keep="last"),
    on=["away_team_norm", "home_team_norm"],
    how="left"
)

# ---------------------------------------------------------
# 3) create model-ready starter columns
# for now ESPN is the source we trust for filling starters
# ---------------------------------------------------------
merged_today_starters["away_starter_name"] = merged_today_starters["away_starter_name_espn"]
merged_today_starters["away_starter_id"] = pd.to_numeric(
    merged_today_starters["away_starter_id_espn"], errors="coerce"
)

merged_today_starters["home_starter_name"] = merged_today_starters["home_starter_name_espn"]
merged_today_starters["home_starter_id"] = pd.to_numeric(
    merged_today_starters["home_starter_id_espn"], errors="coerce"
)

merged_today_starters["starter_source"] = np.where(
    merged_today_starters["away_starter_name"].notna() |
    merged_today_starters["home_starter_name"].notna(),
    "espn_probables",
    "missing"
)

# ---------------------------------------------------------
# 4) starter coverage checks
# ---------------------------------------------------------
merged_today_starters["missing_away_starter"] = merged_today_starters["away_starter_name"].isna().astype(int)
merged_today_starters["missing_home_starter"] = merged_today_starters["home_starter_name"].isna().astype(int)
merged_today_starters["missing_any_starter"] = (
    merged_today_starters["away_starter_name"].isna() |
    merged_today_starters["home_starter_name"].isna()
).astype(int)

print("Merged starter table shape:", merged_today_starters.shape)
print("Games on slate            :", len(merged_today_starters))
print("Missing away starters     :", int(merged_today_starters["missing_away_starter"].sum()))
print("Missing home starters     :", int(merged_today_starters["missing_home_starter"].sum()))
print("Missing any starter       :", int(merged_today_starters["missing_any_starter"].sum()))

print("\nStarter source counts:")
print(merged_today_starters["starter_source"].value_counts(dropna=False))

print("\nPreview:")
display(
    merged_today_starters[
        [
            "game_id",
            "away_team",
            "home_team",
            "away_starter_name",
            "home_starter_name",
            "starter_source",
            "status",
        ]
    ].copy()
)

# ---------------------------------------------------------
# 5) save in memory
# ---------------------------------------------------------
globals()["mlb_today_starters"] = merged_today_starters.copy()

print("\nCreated in memory:")
print(" - mlb_today_starters")
print("Done.")

CELL 12B Ã¢â‚¬â€ MERGE MLB TODAY GAMES WITH ESPN STARTERS
Merged starter table shape: (1, 28)
Games on slate            : 1
Missing away starters     : 1
Missing home starters     : 1
Missing any starter       : 1

Starter source counts:
starter_source
missing    1
Name: count, dtype: int64

Preview:


,game_id,away_team,home_team,away_starter_name,home_starter_name,starter_source,status
0,849831,Chicago White Sox,Cleveland Guardians,NaN,NaN,missing,Scheduled



Created in memory:
 - mlb_today_starters
Done.


In [29]:
# %%
# =========================================================
# CELL Ã¢â‚¬â€ BUILD mlb_today_input (MODEL-READY TODAY FEATURES)
# FULL REPLACEMENT
# =========================================================

import pandas as pd
import numpy as np

if "mlb_today_games" not in globals():
    raise ValueError("mlb_today_games not found. Run schedule cell first.")

if "mlb_team_snapshot_clean" not in globals():
    raise ValueError("mlb_team_snapshot_clean not found. Run snapshot cell first.")

games = mlb_today_games.copy()
snap = mlb_team_snapshot_clean.copy()

games["away_team"] = games["away_team"].astype(str).str.strip()
games["home_team"] = games["home_team"].astype(str).str.strip()
snap["team"] = snap["team"].astype(str).str.strip()

# 2026-08-14 (user audit, real bug -- traced end to end): mlb_team_snapshot_clean
# (Cell 10) normalizes the live-schedule-era "Athletics" name to the historical
# "Oakland Athletics" key so years of pre-rename snapshot data still join
# correctly. But `games` here comes straight from today's live schedule
# (Cell 11), which correctly says "Athletics" -- so this merge, keyed on the
# raw team name, silently failed to match the A's on either side, leaving
# every away_*/home_* snapshot column (and everything derived from them,
# including all 8 diff_* rolling-average features used by the win model) NaN
# for that one game. That NaN then made Cell 36's dropna() drop the row
# entirely a few cells later -- a real MLB game vanishing from the card with
# no visible error anywhere. Merges on a normalized key now (same alias the
# five other TEAM_NAME_FIXES cells already use); the actual displayed
# away_team/home_team values are left as "Athletics", matching today's real
# branding, not silently reverted to the historical name.
_TEAM_NAME_FIXES_MERGE = {"Athletics": "Oakland Athletics"}
games["_away_team_key"] = games["away_team"].replace(_TEAM_NAME_FIXES_MERGE)
games["_home_team_key"] = games["home_team"].replace(_TEAM_NAME_FIXES_MERGE)

# ---------------------------------------------------------
# merge team snapshots
# ---------------------------------------------------------
away_snap = snap.add_prefix("away_")
home_snap = snap.add_prefix("home_")

df = games.merge(
    away_snap,
    left_on="_away_team_key",
    right_on="away_team",
    how="left",
    suffixes=("", "_snapdrop"),
)
df = df.drop(columns=[c for c in df.columns if c.endswith("_snapdrop")])

df = df.merge(
    home_snap,
    left_on="_home_team_key",
    right_on="home_team",
    how="left",
    suffixes=("", "_snapdrop"),
)
df = df.drop(columns=[c for c in df.columns if c.endswith("_snapdrop")] + ["_away_team_key", "_home_team_key"])

# ---------------------------------------------------------
# build diff features (same as training)
# ---------------------------------------------------------
df["diff_runs_for_avg_10"] = df["home_runs_for_avg_10"] - df["away_runs_for_avg_10"]
df["diff_runs_against_avg_10"] = df["home_runs_against_avg_10"] - df["away_runs_against_avg_10"]
df["diff_margin_avg_10"] = df["home_margin_avg_10"] - df["away_margin_avg_10"]
df["diff_win_rate_10"] = df["home_win_rate_10"] - df["away_win_rate_10"]
if "home_runs_for_avg_5" in df.columns and "away_runs_for_avg_5" in df.columns:
    df["diff_runs_for_avg_5"]    = df["home_runs_for_avg_5"]    - df["away_runs_for_avg_5"]
    df["diff_runs_against_avg_5"] = df["home_runs_against_avg_5"] - df["away_runs_against_avg_5"]
    df["diff_margin_avg_5"]      = df["home_margin_avg_5"]      - df["away_margin_avg_5"]
    df["diff_win_rate_5"]        = df["home_win_rate_5"]        - df["away_win_rate_5"]
# Recompute rest_days fresh based on today vs snapshot last game date
_today_norm = pd.Timestamp.today().normalize()
if "away_date" in df.columns:
    _away_d = (_today_norm - pd.to_datetime(df["away_date"], errors="coerce")).dt.days
    df["away_rest_days"] = _away_d.clip(lower=1, upper=30).fillna(2).astype(float)
if "home_date" in df.columns:
    _home_d = (_today_norm - pd.to_datetime(df["home_date"], errors="coerce")).dt.days
    df["home_rest_days"] = _home_d.clip(lower=1, upper=30).fillna(2).astype(float)
df["diff_rest_days"] = df["home_rest_days"] - df["away_rest_days"]
print(f"Refreshed rest_days vs today ({_today_norm.date()}).")

# K%/BB% plate discipline diffs (from team snapshot)
if 'home_k_pct_bat' in df.columns and 'away_k_pct_bat' in df.columns:
    df['diff_k_pct_bat']  = df['home_k_pct_bat']  - df['away_k_pct_bat']
    df['diff_bb_pct_bat'] = df['home_bb_pct_bat'] - df['away_bb_pct_bat']
# starter FIP/K-BB% diff â€” needs mlb_pitcher_fip_lookup to be populated
import re as _re2, unicodedata as _ud
def _nn(s):
    s = _ud.normalize('NFD', str(s))
    s = ''.join(c for c in s if _ud.category(c) != 'Mn')
    return _re2.sub(r'[^a-z ]', '', s.lower()).strip()
_fip_lkp = globals().get('mlb_pitcher_fip_lookup', {})
if _fip_lkp and 'away_pitcher_name' in df.columns and 'home_pitcher_name' in df.columns:
    df['away_starter_fip']      = df['away_pitcher_name'].apply(lambda n: _fip_lkp.get(_nn(str(n)), {}).get('fip',      float('nan')))
    df['home_starter_fip']      = df['home_pitcher_name'].apply(lambda n: _fip_lkp.get(_nn(str(n)), {}).get('fip',      float('nan')))
    df['away_starter_k_bb_pct'] = df['away_pitcher_name'].apply(lambda n: _fip_lkp.get(_nn(str(n)), {}).get('k_bb_pct', float('nan')))
    df['home_starter_k_bb_pct'] = df['home_pitcher_name'].apply(lambda n: _fip_lkp.get(_nn(str(n)), {}).get('k_bb_pct', float('nan')))
    df['diff_starter_fip']      = df['home_starter_fip']      - df['away_starter_fip']
    df['diff_starter_k_bb_pct'] = df['home_starter_k_bb_pct'] - df['away_starter_k_bb_pct']
    # L3 recent form
    df['away_starter_era_l3']  = df['away_pitcher_name'].apply(lambda n: _fip_lkp.get(_nn(str(n)), {}).get('era_l3',  float('nan')))
    df['home_starter_era_l3']  = df['home_pitcher_name'].apply(lambda n: _fip_lkp.get(_nn(str(n)), {}).get('era_l3',  float('nan')))
    df['away_starter_whip_l3'] = df['away_pitcher_name'].apply(lambda n: _fip_lkp.get(_nn(str(n)), {}).get('whip_l3', float('nan')))
    df['home_starter_whip_l3'] = df['home_pitcher_name'].apply(lambda n: _fip_lkp.get(_nn(str(n)), {}).get('whip_l3', float('nan')))
    df['diff_starter_era_l3']  = df['home_starter_era_l3']  - df['away_starter_era_l3']
    df['diff_starter_whip_l3'] = df['home_starter_whip_l3'] - df['away_starter_whip_l3']
print("Today input shape:", df.shape)

print("\nMissing values check:")
display(df.isna().sum().sort_values(ascending=False).head(15))

print("\nSample rows:")
display(df.head())


# ---------------------------------------------------------
# home/away split diffs (home team plays at home, away team plays away)
# ---------------------------------------------------------
_ha_cols = ['runs_for_avg_10_home', 'runs_for_avg_10_away',
            'runs_against_avg_10_home', 'runs_against_avg_10_away',
            'runs_for_avg_5_home', 'runs_for_avg_5_away',
            'runs_against_avg_5_home', 'runs_against_avg_5_away']

if all(f'home_{c}' in df.columns for c in _ha_cols[:4]):
    # Offense: home team's home-game avg vs away team's away-game avg
    df['diff_runs_for_avg_10_split'] = (
        df['home_runs_for_avg_10_home'] - df['away_runs_for_avg_10_away']
    )
    # Defense: home team's home-game runs_against vs away team's away-game
    df['diff_runs_against_avg_10_split'] = (
        df['home_runs_against_avg_10_home'] - df['away_runs_against_avg_10_away']
    )
    print('Home/away split diffs added.')
else:
    print('WARNING: home/away split columns not found in snapshot - skipping split diffs')

if all(f'home_{c}' in df.columns for c in _ha_cols[4:]):
    df['diff_runs_for_avg_5_split'] = (
        df['home_runs_for_avg_5_home'] - df['away_runs_for_avg_5_away']
    )
    df['diff_runs_against_avg_5_split'] = (
        df['home_runs_against_avg_5_home'] - df['away_runs_against_avg_5_away']
    )

# ---------------------------------------------------------
# Bullpen stress diff (home - away; higher = home bullpen more stressed)
# ---------------------------------------------------------
if 'home_bullpen_stress_3' in df.columns and 'away_bullpen_stress_3' in df.columns:
    df['diff_bullpen_stress_3'] = (
        df['home_bullpen_stress_3'] - df['away_bullpen_stress_3']
    )
    print('Bullpen stress diff added.')
else:
    print('WARNING: bullpen_stress_3 not found in snapshot - skipping')
mlb_today_input = df.copy()

Refreshed rest_days vs today (2026-10-10).
Today input shape: (1, 75)

Missing values check:


game_id                     0
home_margin_avg_5           0
home_runs_for_avg_5         0
home_win_rate_10            0
home_margin_avg_10          0
home_runs_against_avg_10    0
home_runs_for_avg_10        0
home_adj_runs_against       0
home_opp_off_rating         0
home_adj_runs_for           0
home_opp_def_rating         0
home_home_away              0
home_opponent               0
home_park_factor            0
home_win                    0
dtype: int64


Sample rows:


,game_id,game_datetime_utc,away_team,home_team,away_team_id,home_team_id,venue_name,status,game_type,series_game_number,...,home_rest_days,diff_runs_for_avg_10,diff_runs_against_avg_10,diff_margin_avg_10,diff_win_rate_10,diff_runs_for_avg_5,diff_runs_against_avg_5,diff_margin_avg_5,diff_win_rate_5,diff_rest_days
0,849831,2026-10-11 00:00:00+00:00,Chicago White Sox,Cleveland Guardians,145,114,Progressive Field,Scheduled,D,5,...,2.0,0.571773,-1.2167,1.788473,0.0737,1.617511,-2.024298,3.641809,0.256034,0.0


Home/away split diffs added.
Bullpen stress diff added.


In [30]:
import json, joblib
from pathlib import Path
_model_dir = Path(r"c:\Users\<user>\OneDrive\Documents\KAI Labs\sports\MLB\models")
# Force-load the compressed 7-feature model so model + feature_list match
model = joblib.load(_model_dir / "mlb_win_model_compressed_best.joblib")
with open(_model_dir / "mlb_win_model_compressed_best_meta.json") as f:
    _meta = json.load(f)
feature_list = _meta["feature_cols"]
print(f"Loaded: {len(feature_list)} features: {feature_list}")

# CALIBRATION AUDIT FIX (2026-08-27, user audit): the cell that actually
# scores today's board ("SCORE TODAY'S MLB BOARD") already had a calibrator
# hook -- `calibrator = globals().get("calibrator", None)` -- but nothing
# anywhere in this notebook ever set a global with that exact name, so it
# was always None and calibration silently never ran on a single live pick.
# The paired calibrator file (fit properly now -- see the cell that saves
# mlb_win_model_compressed_best_calibrator.joblib -- Platt scaling on this
# exact model's own walk-forward out-of-sample predictions, validated
# against saturation) sits right next to the model file; load it the same
# way. Missing/corrupt file degrades to no calibration (same as before this
# fix), not a hard failure -- calibration should never be a reason a real
# card can't be built.
try:
    calibrator = joblib.load(_model_dir / "mlb_win_model_compressed_best_calibrator.joblib")
    print(f"Loaded calibrator: {type(calibrator).__name__}")
except Exception as _cal_load_exc:
    calibrator = None
    print(f"[WARN] Could not load calibrator ({_cal_load_exc}) -- proceeding uncalibrated")
globals()["calibrator"] = calibrator


Loaded: 11 features: ['diff_runs_for_avg_10', 'diff_runs_against_avg_10', 'diff_margin_avg_10', 'diff_win_rate_10', 'diff_runs_for_avg_5', 'diff_runs_against_avg_5', 'diff_margin_avg_5', 'diff_win_rate_5', 'diff_rest_days', 'home_rest_days', 'away_rest_days']
Loaded calibrator: LogisticRegression


In [31]:
# %%
# =========================================================
# CELL Ã¢â‚¬â€ SCORE TODAY'S MLB BOARD WITH MODEL PREDICTIONS
# FULL REPLACEMENT
# Purpose:
# 1) find today's board
# 2) create pred_home_win_prob / pred_away_win_prob from model
# 3) preserve home/away market prices if present
# 4) expose scored_mlb_board
# =========================================================

import pandas as pd
import numpy as np

print("=" * 80)
print("SCORE TODAY'S MLB BOARD WITH MODEL PREDICTIONS")
print("=" * 80)

# ---------------------------------------------------------
# 0) validate model inputs
# ---------------------------------------------------------
if "model" not in globals():
    raise ValueError("model not found in memory.")

if "feature_list" not in globals() or not isinstance(feature_list, list) or len(feature_list) == 0:
    raise ValueError("feature_list not found or empty.")

# optional calibrator
calibrator = globals().get("calibrator", None)

# ---------------------------------------------------------
# 1) locate the best available board
# ---------------------------------------------------------
def first_existing_df(candidates):
    for name in candidates:
        if name in globals() and isinstance(globals()[name], pd.DataFrame):
            df = globals()[name]
            if len(df) > 0:
                print(f"Using board source: {name}  shape={df.shape}")
                return df.copy(), name
    return None, None

def find_first_col(df, options):
    lower_map = {str(c).strip().lower(): c for c in df.columns}
    for opt in options:
        if opt.lower() in lower_map:
            return lower_map[opt.lower()]
    return None

proj_candidates = [
    "mlb_today_model_input",
    "mlb_today_input",
    "projection_board",
    "board",
    "mlb_board",
    "daily_board",
    "today_board",
    "starter_board",
    "today_pred_clean",
]

score_df, board_source = first_existing_df(proj_candidates)

if score_df is None:
    raise ValueError(
        "No usable MLB board found in memory.\n"
        f"Expected one of: {proj_candidates}"
    )

# ---------------------------------------------------------
# 2) resolve core display columns
# ---------------------------------------------------------
game_id_col = find_first_col(score_df, ["game_id", "gamepk", "game_pk", "pk"])
date_col    = find_first_col(score_df, ["game_datetime_utc", "start_time", "game_time_utc", "game_date", "date"])
away_col    = find_first_col(score_df, ["away_team", "away_team_name", "team_away"])
home_col    = find_first_col(score_df, ["home_team", "home_team_name", "team_home"])

if away_col is None or home_col is None:
    raise ValueError(
        "Could not resolve away/home team columns.\n"
        f"Available columns:\n{list(score_df.columns)}"
    )

if away_col != "away_team":
    score_df["away_team"] = score_df[away_col]
    away_col = "away_team"

if home_col != "home_team":
    score_df["home_team"] = score_df[home_col]
    home_col = "home_team"

# ---------------------------------------------------------
# 3) ensure required feature columns exist
# ---------------------------------------------------------
missing_features = [c for c in feature_list if c not in score_df.columns]
if missing_features:
    raise ValueError(
        "The board does not yet contain all model features needed for scoring.\n"
        f"Missing {len(missing_features)} features, examples: {missing_features[:20]}"
    )

# numeric coercion on model features
for c in feature_list:
    score_df[c] = pd.to_numeric(score_df[c], errors="coerce")

print("")
print("Missing counts across model features:")
print(score_df[feature_list].isna().sum().sort_values(ascending=False).head(25))

# ---------------------------------------------------------
# 4) build prediction matrix
# ---------------------------------------------------------
X_score = score_df[feature_list].copy()

# fill missing values safely for models that cannot handle NaN
# if your trained model handles NaN already, this still keeps scoring stable
X_score = X_score.replace([np.inf, -np.inf], np.nan)

# median fill from today's board as fallback
for c in X_score.columns:
    if X_score[c].isna().any():
        med = X_score[c].median(skipna=True)
        if pd.isna(med):
            med = 0.0
        X_score[c] = X_score[c].fillna(med)

# ---------------------------------------------------------
# 5) score board
# ---------------------------------------------------------
pred_home_win = model.predict_proba(X_score)[:, 1]

# CALIBRATION AUDIT FIX (2026-08-27, user audit): the old calibrator.predict()
# call assumed an isotonic-style interface (raw prob in [0,1] -> calibrated
# prob in [0,1] directly). The new calibrator (see the cell that saves
# mlb_win_model_compressed_best_calibrator.joblib) is a plain
# sklearn.LogisticRegression fit on logit(raw_prob) -- a standard, portable
# class, deliberately not a custom wrapper object, since a notebook-local
# class doesn't reliably unpickle in a fresh kernel session on a later run.
# That means the CALLER (here) does the logit transform and predict_proba
# call, not the calibrator object itself. A [0.04, 0.96] hard clip is
# applied regardless of whether calibration ran, so no MLB pick this
# notebook produces can ever claim nearer to certainty than that.
if calibrator is not None:
    try:
        _cal_logit_in = np.clip(pd.to_numeric(pred_home_win, errors="coerce"), 1e-6, 1 - 1e-6)
        _cal_logit_in = np.log(_cal_logit_in / (1 - _cal_logit_in)).reshape(-1, 1)
        pred_home_win = calibrator.predict_proba(_cal_logit_in)[:, 1]
    except Exception as e:
        print(f"Calibrator present but failed. Using raw probs. Reason: {e}")

pred_home_win = np.clip(pd.to_numeric(pred_home_win, errors="coerce"), 0.04, 0.96)

# SECONDARY LEDGER CALIBRATION (2026-09-13, user audit -- "we cannot have this many losses"):
# investigated the full graded history (mlb_official_plays_graded.csv, 52 real graded picks)
# rather than treating tonight's 0-4 as one bad night -- aggregate win rate is 46.2% (24-28),
# and the 16 picks logged SINCE the 2026-08-27 Platt-scaling calibration fix are 43.75% (7-9)
# against an average CLAIMED model_prob of 68.2%. The Aug 27 fix isn't holding. Likely root
# cause: `calibrator` above is a plain LogisticRegression (Platt scaling) -- the notebook's
# own history shows this replaced an IsotonicRegression specifically because that one was
# wrapped in a notebook-local class that didn't reliably unpickle in a fresh kernel (a real
# persistence bug, not a statistical argument against isotonic itself). Platt scaling assumes
# one fixed sigmoid shape across the whole probability range and is structurally weak at
# correcting miscalibration specifically in the extreme tail -- exactly where this notebook's
# own "pick the single biggest edge each day" rule samples from, every single day. Same root
# cause class already confirmed and fixed tonight for NFL's SPREAD/TOTAL markets (isotonic
# beat a flat/parametric correction there for the identical reason: it can fix curvature, not
# just average bias).
#
# This is a SECOND, ADDITIVE correction layer on top of the existing Platt calibrator above --
# it does NOT touch or replace the production calibrator .joblib file, so it carries none of
# the pickle-persistence risk that motivated the original move away from isotonic. It's a raw
# sklearn.IsotonicRegression (no custom wrapper), fit fresh from the ledger's real graded
# outcomes on every run (cheap at this sample size, and stays current as more picks grade out
# rather than going stale like a saved pickle would). PROVISIONAL like everything else in this
# model -- 52 rows (16 post-fix) is a real but thin, and structurally biased (always "today's
# single best edge," never a representative cross-section), sample. Revisit/replace once
# meaningfully more graded history accrues.
# BUG FIX (live-verified: NameError, TRACK_DIR not defined) -- TRACK_DIR gets defined in a LATER cell than this one; this cell runs before it exists. Hardcoded path instead, matching the exact convention _GOP_GRADED_PATH already uses elsewhere in this notebook for the same file.
_ledger_cal_path = Path("c:/Users/<user>/OneDrive/Documents/KAI Labs/sports/MLB/tracking/mlb_official_plays_graded.csv")
ledger_calibrator = None
if _ledger_cal_path.exists():
    try:
        _ledger_df = pd.read_csv(_ledger_cal_path)
        _ledger_df = _ledger_df[_ledger_df["pick_won"].isin(["True", "False", True, False])].copy()
        _ledger_df["_y"] = _ledger_df["pick_won"].astype(str).map({"True": 1, "False": 0})
        _ledger_df["_p"] = pd.to_numeric(_ledger_df["model_prob"], errors="coerce")
        _ledger_df = _ledger_df.dropna(subset=["_y", "_p"])
        _LEDGER_CAL_MIN_N = 30
        if len(_ledger_df) >= _LEDGER_CAL_MIN_N:
            ledger_calibrator = IsotonicRegression(y_min=0.04, y_max=0.96, out_of_bounds="clip")
            ledger_calibrator.fit(_ledger_df["_p"], _ledger_df["_y"])
            _before_avg = float(_ledger_df["_p"].mean())
            _after_avg = float(ledger_calibrator.predict(_ledger_df["_p"]).mean())
            _real_rate = float(_ledger_df["_y"].mean())
            print(f"[ledger calibration] fit on {len(_ledger_df)} real graded picks -- "
                  f"claimed avg {_before_avg:.1%}, real win rate {_real_rate:.1%}, "
                  f"corrected avg {_after_avg:.1%}")
        else:
            print(f"[ledger calibration] only {len(_ledger_df)} graded picks (< {_LEDGER_CAL_MIN_N} minimum) "
                  f"-- skipping secondary correction until more history accrues")
    except Exception as _ledger_cal_exc:
        print(f"[ledger calibration] failed to fit ({_ledger_cal_exc}) -- proceeding without it")

if ledger_calibrator is not None:
    pred_home_win = ledger_calibrator.predict(pred_home_win)
    pred_home_win = np.clip(pd.to_numeric(pred_home_win, errors="coerce"), 0.04, 0.96)

pred_away_win = 1.0 - pred_home_win

score_df["pred_home_win_prob"] = pred_home_win
score_df["pred_away_win_prob"] = pred_away_win

score_df["projected_winner"] = np.where(
    score_df["pred_home_win_prob"] >= 0.5,
    score_df[home_col].astype(str),
    score_df[away_col].astype(str)
)

score_df["projected_win_prob"] = np.where(
    score_df["pred_home_win_prob"] >= 0.5,
    score_df["pred_home_win_prob"],
    score_df["pred_away_win_prob"]
)

score_df["projected_loser"] = np.where(
    score_df["pred_home_win_prob"] >= 0.5,
    score_df[away_col].astype(str),
    score_df[home_col].astype(str)
)

score_df["win_prob_edge_from_50"] = (score_df["projected_win_prob"] - 0.50).abs()
score_df["confidence"] = score_df["projected_win_prob"]

score_df["model_side"] = np.where(
    score_df["pred_home_win_prob"] >= 0.5,
    "home",
    "away"
)

# ---------------------------------------------------------
# 6) starter display helpers if missing
# ---------------------------------------------------------
for c in ["away_pitcher_name", "home_pitcher_name"]:
    if c not in score_df.columns:
        score_df[c] = np.nan

score_df["pitching_matchup"] = np.where(
    score_df["away_pitcher_name"].notna() | score_df["home_pitcher_name"].notna(),
    score_df["away_pitcher_name"].fillna("TBD").astype(str)
    + " vs "
    + score_df["home_pitcher_name"].fillna("TBD").astype(str),
    None
)

# ---------------------------------------------------------
# 7) preserve optional market price columns if present
# ---------------------------------------------------------
score_df["fair_home_price"] = score_df["pred_home_win_prob"]
score_df["fair_away_price"] = score_df["pred_away_win_prob"]

entry_home_candidates = [
    "home_price", "home_yes_price", "home_ml_price", "market_home_prob", "p_market_home"
]
entry_away_candidates = [
    "away_price", "away_yes_price", "away_ml_price", "market_away_prob", "p_market_away"
]

home_price_col = find_first_col(score_df, entry_home_candidates)
away_price_col = find_first_col(score_df, entry_away_candidates)

print("")
print("Resolved market columns:")
print("home_price_col:", home_price_col)
print("away_price_col:", away_price_col)

if home_price_col is not None:
    score_df[home_price_col] = pd.to_numeric(score_df[home_price_col], errors="coerce")
    if home_price_col != "home_price":
        score_df["home_price"] = score_df[home_price_col]
    score_df["edge_home"] = score_df["pred_home_win_prob"] - score_df["home_price"]

if away_price_col is not None:
    score_df[away_price_col] = pd.to_numeric(score_df[away_price_col], errors="coerce")
    if away_price_col != "away_price":
        score_df["away_price"] = score_df[away_price_col]
    score_df["edge_away"] = score_df["pred_away_win_prob"] - score_df["away_price"]

# ---------------------------------------------------------
# 8) sort scored board
# ---------------------------------------------------------
sort_cols = []
ascending = []

if "projected_win_prob" in score_df.columns:
    sort_cols.append("projected_win_prob")
    ascending.append(False)

if game_id_col is not None and game_id_col in score_df.columns:
    sort_cols.append(game_id_col)
    ascending.append(True)

if sort_cols:
    score_df = score_df.sort_values(sort_cols, ascending=ascending).reset_index(drop=True)

# ---------------------------------------------------------
# 9) debug summary
# ---------------------------------------------------------
print("")
print("=" * 80)
print("SCORING SUMMARY")
print("=" * 80)
print(f"Scored board shape: {score_df.shape}")

summary = pd.Series({
    "rows_scored": int(len(score_df)),
    "avg_home_win_prob": float(score_df["pred_home_win_prob"].mean()),
    "avg_away_win_prob": float(score_df["pred_away_win_prob"].mean()),
    "home_favorites_count": int((score_df["pred_home_win_prob"] >= 0.5).sum()),
    "away_favorites_count": int((score_df["pred_home_win_prob"] < 0.5).sum()),
    "strong_55_plus_count": int((score_df["projected_win_prob"] >= 0.55).sum()),
    "strong_57_plus_count": int((score_df["projected_win_prob"] >= 0.57).sum()),
    "strong_59_plus_count": int((score_df["projected_win_prob"] >= 0.59).sum()),
})
print(summary)

print("")
print("Top scored games:")
show_cols = [c for c in [
    game_id_col if game_id_col in score_df.columns else None,
    date_col if date_col in score_df.columns else None,
    away_col,
    home_col,
    "away_pitcher_name",
    "home_pitcher_name",
    "pred_home_win_prob",
    "pred_away_win_prob",
    "projected_winner",
    "projected_win_prob",
    "model_side",
    "confidence",
    "win_prob_edge_from_50",
    "home_price" if "home_price" in score_df.columns else None,
    "away_price" if "away_price" in score_df.columns else None,
    "edge_home" if "edge_home" in score_df.columns else None,
    "edge_away" if "edge_away" in score_df.columns else None,
] if c is not None and c in score_df.columns]

display(score_df[show_cols].head(15))

# ---------------------------------------------------------
# 10) expose standard downstream names
# ---------------------------------------------------------
scored_mlb_board = score_df.copy()
globals()["scored_mlb_board"] = scored_mlb_board
globals()["mlb_scored_board"] = scored_mlb_board

print("")
print("Created in memory as:")
print(" - scored_mlb_board")
print(" - mlb_scored_board")

if "projection_board" not in globals():
    globals()["projection_board"] = scored_mlb_board.copy()
    print(" - projection_board (alias created)")

print("")
print("Done.")

SCORE TODAY'S MLB BOARD WITH MODEL PREDICTIONS
Using board source: mlb_today_input  shape=(1, 78)

Missing counts across model features:
diff_runs_for_avg_10        0
diff_runs_against_avg_10    0
diff_margin_avg_10          0
diff_win_rate_10            0
diff_runs_for_avg_5         0
diff_runs_against_avg_5     0
diff_margin_avg_5           0
diff_win_rate_5             0
diff_rest_days              0
home_rest_days              0
away_rest_days              0
dtype: int64
[ledger calibration] fit on 58 real graded picks -- claimed avg 66.9%, real win rate 51.7%, corrected avg 51.7%

Resolved market columns:
home_price_col: None
away_price_col: None

SCORING SUMMARY
Scored board shape: (1, 91)
rows_scored             1.000000
avg_home_win_prob       0.108145
avg_away_win_prob       0.891855
home_favorites_count    0.000000
away_favorites_count    1.000000
strong_55_plus_count    1.000000
strong_57_plus_count    1.000000
strong_59_plus_count    1.000000
dtype: float64

Top scored game

,game_id,game_datetime_utc,away_team,home_team,away_pitcher_name,home_pitcher_name,pred_home_win_prob,pred_away_win_prob,projected_winner,projected_win_prob,model_side,confidence,win_prob_edge_from_50
0,849831,2026-10-11 00:00:00+00:00,Chicago White Sox,Cleveland Guardians,NaN,NaN,0.108145,0.891855,Chicago White Sox,0.891855,away,0.891855,0.391855



Created in memory as:
 - scored_mlb_board
 - mlb_scored_board
 - projection_board (alias created)

Done.


In [32]:
# %%
# =========================================================
# CELL 12C Ã¢â‚¬â€ BUILD TODAY'S MODEL INPUT WITH ESPN STARTERS
# Merge team snapshot + today's starters + starter history
# into one model-ready daily board
# =========================================================

import pandas as pd
import numpy as np

print("=" * 70)
print("CELL 12C Ã¢â‚¬â€ BUILD TODAY'S MODEL INPUT WITH ESPN STARTERS")
print("=" * 70)

# ---------------------------------------------------------
# 0) validate inputs
# ---------------------------------------------------------
if "mlb_today_starters" not in globals() or not isinstance(mlb_today_starters, pd.DataFrame):
    raise ValueError("mlb_today_starters not found. Run Cell 12B first.")

if "mlb_team_snapshot_clean" not in globals() or not isinstance(mlb_team_snapshot_clean, pd.DataFrame):
    raise ValueError("mlb_team_snapshot_clean not found. Run Cell 10 first.")

today_input = mlb_today_starters.copy()
team_snapshot = mlb_team_snapshot_clean.copy()

# optional historical starter table
has_starter_hist = (
    "mlb_starter_hist" in globals() and isinstance(mlb_starter_hist, pd.DataFrame)
)

# ---------------------------------------------------------
# 1) normalize team names
# ---------------------------------------------------------
TEAM_NAME_FIXES = {
    "Athletics": "Oakland Athletics",
}

def norm_team_name(name):
    if pd.isna(name):
        return np.nan
    name = str(name).strip()
    return TEAM_NAME_FIXES.get(name, name)

today_input["away_team_norm"] = today_input["away_team"].apply(norm_team_name)
today_input["home_team_norm"] = today_input["home_team"].apply(norm_team_name)
team_snapshot["team_norm"] = team_snapshot["team"].apply(norm_team_name)

# ---------------------------------------------------------
# 2) build away/home team snapshot merges
# ---------------------------------------------------------
_snap_date_col = "date" if "date" in team_snapshot.columns else None
_away_cols = ["team_norm", "runs_for_avg_10", "runs_against_avg_10", "margin_avg_10", "win_rate_10",
             "runs_for_avg_5", "runs_against_avg_5", "margin_avg_5", "win_rate_5", "rest_days",
             "k_pct_bat", "bb_pct_bat", "runs_for_avg_10_away", "runs_against_avg_10_away"]  # FIX 2: road splits
_away_rename = {"team_norm": "away_team_norm", "runs_for_avg_10": "away_runs_for_avg_10", "runs_against_avg_10": "away_runs_against_avg_10", "margin_avg_10": "away_margin_avg_10", "win_rate_10": "away_win_rate_10",
                "runs_for_avg_5": "away_runs_for_avg_5", "runs_against_avg_5": "away_runs_against_avg_5", "margin_avg_5": "away_margin_avg_5", "win_rate_5": "away_win_rate_5", "rest_days": "away_rest_days", "k_pct_bat": "away_k_pct_bat", "bb_pct_bat": "away_bb_pct_bat",
                "runs_for_avg_10_away": "away_runs_for_avg_road_10", "runs_against_avg_10_away": "away_runs_against_avg_road_10"}  # FIX 2
if _snap_date_col:
    _away_cols.append(_snap_date_col)
    _away_rename[_snap_date_col] = "away_last_game_date"
away_merge = team_snapshot[[c for c in _away_cols if c in team_snapshot.columns]].copy().rename(columns=_away_rename)

_home_cols = ["team_norm", "runs_for_avg_10", "runs_against_avg_10", "margin_avg_10", "win_rate_10",
             "runs_for_avg_5", "runs_against_avg_5", "margin_avg_5", "win_rate_5", "rest_days",
             "k_pct_bat", "bb_pct_bat", "runs_for_avg_10_home", "runs_against_avg_10_home"]  # FIX 2: home splits
_home_rename = {"team_norm": "home_team_norm", "runs_for_avg_10": "home_runs_for_avg_10", "runs_against_avg_10": "home_runs_against_avg_10", "margin_avg_10": "home_margin_avg_10", "win_rate_10": "home_win_rate_10",
                "runs_for_avg_5": "home_runs_for_avg_5", "runs_against_avg_5": "home_runs_against_avg_5", "margin_avg_5": "home_margin_avg_5", "win_rate_5": "home_win_rate_5", "rest_days": "home_rest_days", "k_pct_bat": "home_k_pct_bat", "bb_pct_bat": "home_bb_pct_bat",
                "runs_for_avg_10_home": "home_runs_for_avg_home_10", "runs_against_avg_10_home": "home_runs_against_avg_home_10"}  # FIX 2
if _snap_date_col:
    _home_cols.append(_snap_date_col)
    _home_rename[_snap_date_col] = "home_last_game_date"
home_merge = team_snapshot[[c for c in _home_cols if c in team_snapshot.columns]].copy().rename(columns=_home_rename)

board = today_input.merge(
    away_merge,
    on="away_team_norm",
    how="left"
).merge(
    home_merge,
    on="home_team_norm",
    how="left"
)

# ---------------------------------------------------------
# 3) basic team-diff features
# ---------------------------------------------------------
board["diff_runs_for_avg_10"] = board["home_runs_for_avg_10"] - board["away_runs_for_avg_10"]
board["diff_runs_against_avg_10"] = board["home_runs_against_avg_10"] - board["away_runs_against_avg_10"]
board["diff_margin_avg_10"] = board["home_margin_avg_10"] - board["away_margin_avg_10"]
board["diff_win_rate_10"] = board["home_win_rate_10"] - board["away_win_rate_10"]
# L5 diffs (if snapshot includes L5 stats)
for _c5 in ["runs_for_avg_5", "runs_against_avg_5", "margin_avg_5", "win_rate_5"]:
    if f"home_{_c5}" in board.columns and f"away_{_c5}" in board.columns:
        board[f"diff_{_c5}"] = board[f"home_{_c5}"] - board[f"away_{_c5}"]
# K%/BB% plate discipline diffs
if 'home_k_pct_bat' in board.columns and 'away_k_pct_bat' in board.columns:
    board['diff_k_pct_bat']  = board['home_k_pct_bat']  - board['away_k_pct_bat']
    board['diff_bb_pct_bat'] = board['home_bb_pct_bat'] - board['away_bb_pct_bat']
# starter FIP / K-BB% diff
import re as _re3, unicodedata as _ud3
def _nn3(s):
    s = _ud3.normalize('NFD', str(s))
    s = ''.join(c for c in s if _ud3.category(c) != 'Mn')
    return _re3.sub(r'[^a-z ]', '', s.lower()).strip()
_fip_lkp26 = globals().get('mlb_pitcher_fip_lookup', {})
if _fip_lkp26:
    for _sc, _nc in [('away_starter_name','away_pitcher_name'),('home_starter_name','home_pitcher_name')]:
        _pcol = _sc if _sc in board.columns else (_nc if _nc in board.columns else None)
        if _pcol:
            _side = 'away' if 'away' in _pcol else 'home'
            board[f'{_side}_starter_fip']      = board[_pcol].apply(lambda n: _fip_lkp26.get(_nn3(str(n)), {}).get('fip',      float('nan')))
            board[f'{_side}_starter_k_bb_pct'] = board[_pcol].apply(lambda n: _fip_lkp26.get(_nn3(str(n)), {}).get('k_bb_pct', float('nan')))
    if 'home_starter_fip' in board.columns and 'away_starter_fip' in board.columns:
        board['diff_starter_fip']      = board['home_starter_fip']      - board['away_starter_fip']
        board['diff_starter_k_bb_pct'] = board['home_starter_k_bb_pct'] - board['away_starter_k_bb_pct']
        _sp_ok = board['diff_starter_fip'].notna().sum()
        print(f'Starter FIP diff populated: {_sp_ok}/{len(board)} games')
# Recompute rest_days fresh using today vs snapshot last game date
_today_norm = pd.Timestamp.today().normalize()
if "away_last_game_date" in board.columns:
    board["away_rest_days"] = (_today_norm - pd.to_datetime(board["away_last_game_date"], errors="coerce")).dt.days.clip(lower=1, upper=30).fillna(2).astype(float)
if "home_last_game_date" in board.columns:
    board["home_rest_days"] = (_today_norm - pd.to_datetime(board["home_last_game_date"], errors="coerce")).dt.days.clip(lower=1, upper=30).fillna(2).astype(float)
board["diff_rest_days"] = board["home_rest_days"] - board["away_rest_days"]
print(f"Refreshed rest_days vs today ({_today_norm.date()}).")

# ---------------------------------------------------------
# 4) starter lookup table from historical starter features
# if your mlb table already contains starter-level rolling features,
# use the latest row for each starter_id
# ---------------------------------------------------------
starter_feature_cols = [
    "starter_id",
    "starter_name",
    "starter_runs_allowed_avg_10",
    "starter_margin_avg_10",
    "starter_rest_days",
]

starter_snapshot = None

possible_starter_feature_source = None
for obj_name in ["mlb", "train_df", "model_df", "mlb_final_train"]:
    if obj_name in globals() and isinstance(globals()[obj_name], pd.DataFrame):
        df_try = globals()[obj_name].copy()
        cols_lower = {str(c).strip().lower(): c for c in df_try.columns}

        required_map = {
            "starter_id": None,
            "starter_name": None,
            "starter_runs_allowed_avg_10": None,
            "starter_margin_avg_10": None,
            "starter_rest_days": None,
            "date": None,
        }

        for wanted in list(required_map.keys()):
            for c in df_try.columns:
                if str(c).strip().lower() == wanted.lower():
                    required_map[wanted] = c
                    break

        if all(required_map[k] is not None for k in required_map):
            use_cols = [required_map[k] for k in required_map]
            temp = df_try[use_cols].copy()
            temp.columns = list(required_map.keys())
            temp["starter_id"] = pd.to_numeric(temp["starter_id"], errors="coerce")
            temp["date"] = pd.to_datetime(temp["date"], errors="coerce")
            temp = temp.dropna(subset=["starter_id", "date"]).copy()
            temp = temp.sort_values(["starter_id", "date"]).drop_duplicates(subset=["starter_id"], keep="last")

            starter_snapshot = temp.copy()
            possible_starter_feature_source = obj_name
            break

# ---------------------------------------------------------
# 5) fallback: build starter snapshot from split away/home columns
# if unified starter table is not already available
# ---------------------------------------------------------
if starter_snapshot is None and "mlb" in globals() and isinstance(mlb, pd.DataFrame):
    raw = mlb.copy()

    if "date" in raw.columns:
        raw["date"] = pd.to_datetime(raw["date"], errors="coerce")

        away_needed = [
            "date",
            "away_starter_id",
            "away_starter_name",
            "away_starter_runs_allowed_avg_10",
            "away_starter_margin_avg_10",
            "away_starter_rest_days",
        ]
        home_needed = [
            "date",
            "home_starter_id",
            "home_starter_name",
            "home_starter_runs_allowed_avg_10",
            "home_starter_margin_avg_10",
            "home_starter_rest_days",
        ]

        away_ok = all(c in raw.columns for c in away_needed)
        home_ok = all(c in raw.columns for c in home_needed)

        starter_parts = []

        if away_ok:
            away_part = raw[away_needed].copy().rename(columns={
                "away_starter_id": "starter_id",
                "away_starter_name": "starter_name",
                "away_starter_runs_allowed_avg_10": "starter_runs_allowed_avg_10",
                "away_starter_margin_avg_10": "starter_margin_avg_10",
                "away_starter_rest_days": "starter_rest_days",
            })
            starter_parts.append(away_part)

        if home_ok:
            home_part = raw[home_needed].copy().rename(columns={
                "home_starter_id": "starter_id",
                "home_starter_name": "starter_name",
                "home_starter_runs_allowed_avg_10": "starter_runs_allowed_avg_10",
                "home_starter_margin_avg_10": "starter_margin_avg_10",
                "home_starter_rest_days": "starter_rest_days",
            })
            starter_parts.append(home_part)

        if starter_parts:
            starter_snapshot = pd.concat(starter_parts, ignore_index=True)
            starter_snapshot["starter_id"] = pd.to_numeric(starter_snapshot["starter_id"], errors="coerce")
            starter_snapshot = starter_snapshot.dropna(subset=["starter_id", "date"]).copy()
            starter_snapshot = (
                starter_snapshot
                .sort_values(["starter_id", "date"])
                .drop_duplicates(subset=["starter_id"], keep="last")
                .reset_index(drop=True)
            )
            possible_starter_feature_source = "mlb_split_starter_columns"

# ---------------------------------------------------------
# 6) merge starter features if available
# ---------------------------------------------------------
if starter_snapshot is not None and len(starter_snapshot) > 0:
    away_starter_merge = starter_snapshot[
        ["starter_id", "starter_name", "starter_runs_allowed_avg_10", "starter_margin_avg_10", "starter_rest_days"]
    ].copy().rename(columns={
        "starter_id": "away_starter_id",
        "starter_name": "away_starter_name_hist",
        "starter_runs_allowed_avg_10": "away_starter_runs_allowed_avg_10",
        "starter_margin_avg_10": "away_starter_margin_avg_10",
        "starter_rest_days": "away_starter_rest_days",
    })

    home_starter_merge = starter_snapshot[
        ["starter_id", "starter_name", "starter_runs_allowed_avg_10", "starter_margin_avg_10", "starter_rest_days"]
    ].copy().rename(columns={
        "starter_id": "home_starter_id",
        "starter_name": "home_starter_name_hist",
        "starter_runs_allowed_avg_10": "home_starter_runs_allowed_avg_10",
        "starter_margin_avg_10": "home_starter_margin_avg_10",
        "starter_rest_days": "home_starter_rest_days",
    })

    board["away_starter_id"] = pd.to_numeric(board["away_starter_id"], errors="coerce")
    board["home_starter_id"] = pd.to_numeric(board["home_starter_id"], errors="coerce")

    board = board.merge(
        away_starter_merge,
        on="away_starter_id",
        how="left"
    ).merge(
        home_starter_merge,
        on="home_starter_id",
        how="left"
    )

    board["diff_starter_runs_allowed_avg_10"] = (
        board["home_starter_runs_allowed_avg_10"] - board["away_starter_runs_allowed_avg_10"]
    )
    board["diff_starter_margin_avg_10"] = (
        board["home_starter_margin_avg_10"] - board["away_starter_margin_avg_10"]
    )
    board["diff_starter_rest_days"] = (
        board["home_starter_rest_days"] - board["away_starter_rest_days"]
    )

else:
    print("No usable starter feature snapshot was found.")
    print("The board will still build, but starter feature columns may be missing.")

# ---------------------------------------------------------
# 7) coverage diagnostics
# ---------------------------------------------------------
team_feature_check_cols = [
    "away_runs_for_avg_10",
    "home_runs_for_avg_10",
    "away_margin_avg_10",
    "home_margin_avg_10",
]

print("\nBoard shape:", board.shape)

print("\nMissing core team feature counts:")
for c in team_feature_check_cols:
    if c in board.columns:
        print(f" - {c}: {int(board[c].isna().sum())}")

starter_check_cols = [
    "away_starter_name",
    "home_starter_name",
    "away_starter_runs_allowed_avg_10",
    "home_starter_runs_allowed_avg_10",
]

print("\nMissing starter-related counts:")
for c in starter_check_cols:
    if c in board.columns:
        print(f" - {c}: {int(board[c].isna().sum())}")

print("\nStarter feature source used:", possible_starter_feature_source)

preview_cols = [
    "game_id",
    "away_team",
    "home_team",
    "away_starter_name",
    "home_starter_name",
    "away_runs_for_avg_10",
    "home_runs_for_avg_10",
    "diff_runs_for_avg_10",
]

for c in [
    "away_starter_runs_allowed_avg_10",
    "home_starter_runs_allowed_avg_10",
    "diff_starter_runs_allowed_avg_10",
    "starter_source",
]:
    if c in board.columns:
        preview_cols.append(c)

preview_cols = [c for c in preview_cols if c in board.columns]

print("\nPreview:")
display(board[preview_cols].copy())

# ---------------------------------------------------------
# 8) save in memory
# ---------------------------------------------------------
globals()["mlb_today_model_input"] = board.copy()

print("\nCreated in memory:")
print(" - mlb_today_model_input")
print("Done.")


CELL 12C Ã¢â‚¬â€ BUILD TODAY'S MODEL INPUT WITH ESPN STARTERS
Starter FIP diff populated: 0/1 games
Refreshed rest_days vs today (2026-10-10).
No usable starter feature snapshot was found.
The board will still build, but starter feature columns may be missing.

Board shape: (1, 67)

Missing core team feature counts:
 - away_runs_for_avg_10: 0
 - home_runs_for_avg_10: 0
 - away_margin_avg_10: 0
 - home_margin_avg_10: 0

Missing starter-related counts:
 - away_starter_name: 1
 - home_starter_name: 1

Starter feature source used: None

Preview:


,game_id,away_team,home_team,away_starter_name,home_starter_name,away_runs_for_avg_10,home_runs_for_avg_10,diff_runs_for_avg_10,starter_source
0,849831,Chicago White Sox,Cleveland Guardians,NaN,NaN,4.958469,5.530243,0.571773,missing



Created in memory:
 - mlb_today_model_input
Done.


In [33]:
# %%
# =========================================================
# CELL 13 Ã¢â‚¬â€ SCORE TODAY'S MLB BOARD WITH SAVED MODEL
# FULL REPLACEMENT
# Score today's board using the saved COMPRESSED 7-FEATURE baseline
# =========================================================

import json
import joblib
import pandas as pd
import numpy as np
from pathlib import Path

print("=" * 70)
print("CELL 13 Ã¢â‚¬â€ SCORE TODAY'S MLB BOARD WITH SAVED MODEL")
print("=" * 70)

# ---------------------------------------------------------
# 0) validate input board
# ---------------------------------------------------------
if "mlb_today_model_input" not in globals() or not isinstance(mlb_today_model_input, pd.DataFrame):
    raise ValueError("mlb_today_model_input not found. Run the daily input-building cell first.")

board = mlb_today_model_input.copy()

if len(board) == 0:
    raise ValueError("mlb_today_model_input is empty.")

# ---------------------------------------------------------
# 1) resolve model dir
# ---------------------------------------------------------
resolved_model_dir = None

if "MODEL_DIR" in globals():
    try:
        resolved_model_dir = Path(MODEL_DIR)
    except Exception:
        resolved_model_dir = None

if resolved_model_dir is None:
    candidates = [
        Path(r"c:\Users\<user>\OneDrive\Documents\KAI Labs\sports\MLB\models"),
        Path.cwd() / "models" / "mlb",
        Path.cwd() / "models",
        Path("models/mlb"),
        Path("models"),
    ]
    for p in candidates:
        if p.exists():
            resolved_model_dir = p
            break

if resolved_model_dir is None:
    raise ValueError("Could not resolve MODEL_DIR.")

# ---------------------------------------------------------
# 2) use the NEW saved compressed baseline files
# ---------------------------------------------------------
model_file = resolved_model_dir / "mlb_win_model_compressed_best.joblib"
meta_file = resolved_model_dir / "mlb_win_model_compressed_best_meta.json"
calibrator_file = resolved_model_dir / "mlb_win_model_compressed_best_calibrator.joblib"

if not model_file.exists():
    raise FileNotFoundError(f"Model file not found: {model_file}")

if not meta_file.exists():
    raise FileNotFoundError(f"Meta file not found: {meta_file}")

print(f"Resolved MODEL_DIR : {resolved_model_dir}")
print(f"Model file         : {model_file}")
print(f"Meta file          : {meta_file}")
print(f"Calibrator exists  : {calibrator_file.exists()}")

# ---------------------------------------------------------
# 3) load model + metadata
# ---------------------------------------------------------
win_model = joblib.load(model_file)

with open(meta_file, "r", encoding="utf-8") as f:
    meta = json.load(f)

feature_cols_resolved = meta.get("feature_cols", None)
target_from_meta = meta.get("target", "home_win")
model_name = meta.get("model_name", "unknown_model")

if feature_cols_resolved is None or not isinstance(feature_cols_resolved, list) or len(feature_cols_resolved) == 0:
    raise ValueError("feature_cols missing from saved meta file.")

print("\nLoaded model name:", model_name)
print("Resolved feature count:", len(feature_cols_resolved))
print("Resolved features:")
for c in feature_cols_resolved:
    print(" -", c)

# ---------------------------------------------------------
# ---------------------------------------------------------
# 4) resolve feature set: prefer 11-feature (L10+L5) if saved model has them,
#    fall back to 7-feature (L10-only) baseline if model predates L5 retraining
# ---------------------------------------------------------
FEATURES_11 = [
    "diff_runs_for_avg_10",
    "diff_runs_against_avg_10",
    "diff_margin_avg_10",
    "diff_win_rate_10",
    "diff_runs_for_avg_5",
    "diff_runs_against_avg_5",
    "diff_margin_avg_5",
    "diff_win_rate_5",
    "diff_rest_days",
    "home_rest_days",
    "away_rest_days",
]
FEATURES_15 = FEATURES_11 + [
    "diff_k_pct_bat",
    "diff_bb_pct_bat",
    "diff_starter_fip",
    "diff_starter_k_bb_pct",
]
FEATURES_7 = [
    "diff_runs_for_avg_10",
    "diff_runs_against_avg_10",
    "diff_margin_avg_10",
    "diff_win_rate_10",
    "diff_rest_days",
    "home_rest_days",
    "away_rest_days",
]

# auto-select feature tier based on saved model
_has_l5  = all(f in feature_cols_resolved for f in ["diff_runs_for_avg_5", "diff_win_rate_5"])
_has_disc = all(f in feature_cols_resolved for f in ["diff_k_pct_bat", "diff_starter_fip"])
if _has_disc:
    feature_cols_resolved = FEATURES_15.copy()
    print("Using 15-feature model (L10 + L5 + discipline)")
elif _has_l5:
    feature_cols_resolved = FEATURES_11.copy()
    print("Using 11-feature model (L10 + L5)")
else:
    feature_cols_resolved = FEATURES_7.copy()
    print("WARNING: 7-feature fallback. Run train_df enrichment then Cells 51+52 to upgrade.")

extra_in_meta = [c for c in meta.get("feature_cols", []) if c not in feature_cols_resolved]
if extra_in_meta:
    print("Note: meta has extra features not used in resolved set:", extra_in_meta)

# ---------------------------------------------------------
# 5) validate board columns
# ---------------------------------------------------------
missing_features_in_board = [c for c in feature_cols_resolved if c not in board.columns]
if missing_features_in_board:
    raise ValueError(
        f"Today's board is missing required model features: {missing_features_in_board}"
    )

# keep only needed columns for model matrix, but preserve board copy
for c in feature_cols_resolved:
    board[c] = pd.to_numeric(board[c], errors="coerce")

missing_rows_before = len(board)
board = board.dropna(subset=feature_cols_resolved).copy()
missing_rows_after = len(board)

print("\nBoard rows before feature cleanup:", missing_rows_before)
print("Board rows after feature cleanup :", missing_rows_after)
print("Rows removed                     :", missing_rows_before - missing_rows_after)

if len(board) == 0:
    raise ValueError("No usable rows remain in today's board after feature cleanup.")

# ---------------------------------------------------------
# 6) score board
# ---------------------------------------------------------
X_today = board[feature_cols_resolved].copy()

p_home_win = win_model.predict_proba(X_today)[:, 1]
p_home_win = np.clip(p_home_win, 1e-6, 1 - 1e-6)

board["p_home_win"] = p_home_win
board["p_away_win"] = 1 - board["p_home_win"]
board["pred_home_win"] = (board["p_home_win"] >= 0.5).astype(int)

if "home_team" in board.columns and "away_team" in board.columns:
    board["projected_winner"] = np.where(
        board["pred_home_win"] == 1,
        board["home_team"],
        board["away_team"]
    )
    board["projected_loser"] = np.where(
        board["pred_home_win"] == 1,
        board["away_team"],
        board["home_team"]
    )
else:
    board["projected_winner"] = np.where(board["pred_home_win"] == 1, "HOME", "AWAY")
    board["projected_loser"] = np.where(board["pred_home_win"] == 1, "AWAY", "HOME")

board["projected_win_prob"] = np.where(
    board["pred_home_win"] == 1,
    board["p_home_win"],
    board["p_away_win"]
)

# ---------------------------------------------------------
# 7) optional strength tiers
# ---------------------------------------------------------
def assign_strength(prob):
    if prob >= 0.60:
        return "A"
    if prob >= 0.58:
        return "B"
    if prob >= 0.56:
        return "C"
    if prob >= 0.54:
        return "Lean"
    return "Pass"

board["strength_tier"] = board["projected_win_prob"].apply(assign_strength)

# ---------------------------------------------------------
# 8) sort board
# ---------------------------------------------------------
sort_cols = []
if "game_datetime_utc" in board.columns:
    try:
        board["game_datetime_utc"] = pd.to_datetime(board["game_datetime_utc"], errors="coerce", utc=True)
        sort_cols.append("game_datetime_utc")
    except Exception:
        pass

sort_cols += ["projected_win_prob"]

board = board.sort_values(
    by=sort_cols,
    ascending=[True] + [False] * (len(sort_cols) - 1)
).reset_index(drop=True)

# ---------------------------------------------------------
# 9) output preview
# ---------------------------------------------------------
preview_cols = [
    c for c in [
        "game_id",
        "game_datetime_utc",
        "away_team",
        "home_team",
        "projected_winner",
        "projected_win_prob",
        "p_home_win",
        "p_away_win",
        "strength_tier",
    ] if c in board.columns
]

print("\nSCORED TODAY'S MLB BOARD")
print("-" * 60)
print("Rows scored:", len(board))
print("Model used :", model_name)

print("\nPreview:")
display(board[preview_cols].head(20))

# ---------------------------------------------------------
# 10) save in memory
# ---------------------------------------------------------
globals()["scored_mlb_board"] = board.copy()
globals()["mlb_today_scored_board"] = board.copy()
globals()["mlb_today_feature_cols_used"] = feature_cols_resolved.copy()
globals()["mlb_today_model_name"] = model_name

print("\nCreated in memory:")
print(" - scored_mlb_board")
print(" - mlb_today_scored_board")
print(" - mlb_today_feature_cols_used")
print(" - mlb_today_model_name")
print("Done.")

CELL 13 Ã¢â‚¬â€ SCORE TODAY'S MLB BOARD WITH SAVED MODEL
Resolved MODEL_DIR : c:\Users\<user>\OneDrive\Documents\KAI Labs\sports\MLB\models
Model file         : c:\Users\<user>\OneDrive\Documents\KAI Labs\sports\MLB\models\mlb_win_model_compressed_best.joblib
Meta file          : c:\Users\<user>\OneDrive\Documents\KAI Labs\sports\MLB\models\mlb_win_model_compressed_best_meta.json
Calibrator exists  : True

Loaded model name: mlb_win_model_compressed_best
Resolved feature count: 11
Resolved features:
 - diff_runs_for_avg_10
 - diff_runs_against_avg_10
 - diff_margin_avg_10
 - diff_win_rate_10
 - diff_runs_for_avg_5
 - diff_runs_against_avg_5
 - diff_margin_avg_5
 - diff_win_rate_5
 - diff_rest_days
 - home_rest_days
 - away_rest_days
Using 11-feature model (L10 + L5)

Board rows before feature cleanup: 1
Board rows after feature cleanup : 1
Rows removed                     : 0

SCORED TODAY'S MLB BOARD
------------------------------------------------------------
Rows scored: 1
Model us

,game_id,game_datetime_utc,away_team,home_team,projected_winner,projected_win_prob,p_home_win,p_away_win,strength_tier
0,849831,2026-10-11 00:00:00+00:00,Chicago White Sox,Cleveland Guardians,Cleveland Guardians,0.554307,0.554307,0.445693,Lean



Created in memory:
 - scored_mlb_board
 - mlb_today_scored_board
 - mlb_today_feature_cols_used
 - mlb_today_model_name
Done.


In [34]:
# %%
# =========================================================
# CELL H2H â€” CURRENT-SEASON HEAD-TO-HEAD ADJUSTMENT
# Queries MLB Stats API for all 2026 regular-season completed
# games between today's two teams.
# Bayesian-blends H2H home win rate with model probability.
# Weight: 0 if n<3 | 10% n=3-5 | 15% n=6-10 | 20% n>10
# Updates scored_mlb_board with h2h-adjusted probs.
# =========================================================

import pandas as pd
import numpy as np
import requests
import time
import datetime as _dt

print("=" * 70)
print("CELL H2H â€” CURRENT-SEASON HEAD-TO-HEAD ADJUSTMENT")
print("=" * 70)

if "scored_mlb_board" not in globals() or not isinstance(scored_mlb_board, pd.DataFrame):
    raise ValueError("scored_mlb_board not found. Run scoring cells first.")

_board = scored_mlb_board.copy()

_H2H_SEASON = 2026
_PRIOR_N    = 5       # Bayesian prior: 5 phantom games at 50%
_PRIOR_P    = 0.50

def _h2h_blend_weight(n):
    if n < 3:   return 0.0
    if n <= 5:  return 0.10
    if n <= 10: return 0.15
    return 0.20

def _fetch_h2h_games(home_id, away_id, season=_H2H_SEASON):
    # Return list of completed regular-season games between two teams this season.
    # BUG FIX (2026-09-02, found live investigating a real Orioles loss --
    # H2H print showed "140 games" between BAL and COL, impossible for a
    # single MLB season): "opposingTeamId" isn't a real param on this
    # endpoint -- the API silently ignores unknown query params instead of
    # erroring, so this was returning the team's ENTIRE season schedule vs
    # every opponent, not just games vs the other team. Confirmed live: the
    # real param is "opponentId" (confirmed live: 3 games for COL vs BAL,
    # matching the actual 2026 series -- not 140). This corrupted data was
    # being Bayesian-blended into pred_home_win_prob at up to 20% weight,
    # every MLB game -- not just cosmetic, a real accuracy bug.
    url = (
        f"https://statsapi.mlb.com/api/v1/schedule"
        f"?sportId=1&season={season}&teamId={int(home_id)}"
        f"&opponentId={int(away_id)}&gameType=R"
    )
    try:
        resp = requests.get(url, timeout=10)
        resp.raise_for_status()
        data = resp.json()
    except Exception:
        return []

    out = []
    for date_block in data.get("dates", []):
        for g in date_block.get("games", []):
            if g.get("status", {}).get("abstractGameState") != "Final":
                continue
            teams   = g.get("teams", {})
            away    = teams.get("away", {})
            home    = teams.get("home", {})
            a_score = away.get("score")
            h_score = home.get("score")
            if a_score is None or h_score is None:
                continue
            out.append({
                "gamepk"    : g.get("gamePk"),
                "game_date" : g.get("gameDate", "")[:10],
                "away_id"   : away.get("team", {}).get("id"),
                "home_id"   : home.get("team", {}).get("id"),
                "away_name" : away.get("team", {}).get("name", ""),
                "home_name" : home.get("team", {}).get("name", ""),
                "away_score": int(a_score),
                "home_score": int(h_score),
                "home_won"  : int(h_score) > int(a_score),
            })
    return out

# ---------------------------------------------------------
# Query H2H for every game on today's board
# ---------------------------------------------------------
_h2h_rows = []

# BUG FIX (2026-09-02, found live investigating a real Orioles loss): this
# cell's own board columns (h2h_n_games etc.) get silently discarded a few
# cells downstream -- "SCORE DAILY MLB BOARD WITH FINAL WIN MODEL" rebuilds
# scored_mlb_board from the raw `board` global (not scored_mlb_board), which
# also means cell H2H's Bayesian-blended pred_home_win_prob never survives
# to the final card either. mlb_bp_b2b/mlb_sp_former_team/_series_context
# already dodge this by living as standalone team-pair-keyed globals instead
# of board columns -- doing the same here so H2H data (and this cell's
# series-win tally) actually reaches Cell 89's card.
mlb_h2h_lookup = {}  # (away_norm, home_norm) -> {h2h_n_games, h2h_home_win_rate, h2h_run_diff_avg, series_home_wins, series_away_wins}

# MULTI-SEASON H2H STREAK (2026-09-10, user request): a same-season-only
# H2H fetch rarely produces a real 'current streak' signal -- MLB series
# are short (3-4 games/year), so 4+ consecutive meetings almost always
# spans multiple seasons. Kept as its own small suppressor, separate from
# the Bayesian blend above (that stays current-season only, unchanged).
_H2H_STREAK_MIN = 4          # consecutive meetings before this fires
_H2H_STREAK_LOOKBACK = 2     # additional prior seasons checked, beyond the current one
mlb_h2h_streak_lookup = {}  # (away_norm, home_norm) -> {streak_len, streak_winner_id, streak_winner_name, streak_loser_id}

for _idx, _row in _board.iterrows():
    _home_id   = _row.get("home_team_id")
    _away_id   = _row.get("away_team_id")
    _home_name = str(_row.get("home_team", ""))
    _away_name = str(_row.get("away_team", ""))

    _blank = {
        "idx": _idx, "h2h_n_games": 0, "h2h_home_wins": 0,
        "h2h_home_win_rate": np.nan, "h2h_bayesian_rate": _PRIOR_P,
        "h2h_weight": 0.0, "h2h_run_diff_avg": np.nan,
        "series_game_num": 1, "series_home_wins": 0, "series_away_wins": 0,
    }

    if pd.isna(_home_id) or pd.isna(_away_id):
        _h2h_rows.append(_blank)
        continue

    _games = _fetch_h2h_games(int(_home_id), int(_away_id))
    time.sleep(0.15)

    # Multi-season streak walk -- reuses the same fetch for a couple of
    # prior seasons, combines with this season's _games, and walks
    # most-recent-first for a real consecutive-win streak by actual team,
    # independent of which side was designated home that year.
    _streak_games = list(_games)
    for _sy in range(_H2H_SEASON - 1, _H2H_SEASON - 1 - _H2H_STREAK_LOOKBACK, -1):
        _streak_games.extend(_fetch_h2h_games(int(_home_id), int(_away_id), season=_sy))
        time.sleep(0.15)
    _streak_games.sort(key=lambda g: g['game_date'], reverse=True)
    _cur_streak = 0
    _streak_winner_id = None
    for _sg in _streak_games:
        _sg_winner_id = _sg['home_id'] if _sg['home_won'] else _sg['away_id']
        if _streak_winner_id is None:
            _streak_winner_id = _sg_winner_id
            _cur_streak = 1
        elif _sg_winner_id == _streak_winner_id:
            _cur_streak += 1
        else:
            break
    if _cur_streak >= _H2H_STREAK_MIN and _streak_winner_id is not None:
        _streak_winner_name = _home_name if _streak_winner_id == int(_home_id) else _away_name
        _streak_loser_id = int(_away_id) if _streak_winner_id == int(_home_id) else int(_home_id)
        mlb_h2h_streak_lookup[(_away_name.lower(), _home_name.lower())] = {
            'streak_len': _cur_streak, 'streak_winner_id': int(_streak_winner_id),
            'streak_winner_name': _streak_winner_name, 'streak_loser_id': _streak_loser_id,
        }
        print(f'  H2H STREAK {_away_name} @ {_home_name}: {_streak_winner_name} has won '
              f'{_cur_streak} straight meetings (spanning multiple seasons if needed)')

    _n = len(_games)
    if _n == 0:
        _h2h_rows.append(_blank)
        print(f"  H2H {_away_name} @ {_home_name}: 0 completed games this season")
        continue

    # wins from today's home team's perspective
    _home_wins = sum(
        1 for g in _games
        if (g["home_id"] == int(_home_id) and g["home_won"])
        or (g["away_id"] == int(_home_id) and not g["home_won"])
    )

    # M11: current-series context -- a game within the trailing 6 days of
    # today IS the current series (two teams playing a separate series
    # months apart would never fall inside that window, so no false merge).
    _series_games = sorted(
        (g for g in _games if (_dt.date.today() - _dt.date.fromisoformat(g["game_date"])).days <= 6),
        key=lambda g: g["game_date"],
    )
    _series_home_wins = sum(
        1 for g in _series_games
        if (g["home_id"] == int(_home_id) and g["home_won"])
        or (g["away_id"] == int(_home_id) and not g["home_won"])
    )
    _series_game_num = len(_series_games) + 1
    _series_away_wins = len(_series_games) - _series_home_wins
    _raw_rate   = _home_wins / _n
    _bayes_rate = (_PRIOR_N * _PRIOR_P + _n * _raw_rate) / (_PRIOR_N + _n)
    _w          = _h2h_blend_weight(_n)

    # run diff in today's home team's favor
    _diffs = []
    for g in _games:
        if g["home_id"] == int(_home_id):
            _diffs.append(g["home_score"] - g["away_score"])
        else:
            _diffs.append(g["away_score"] - g["home_score"])
    _avg_diff = float(np.mean(_diffs)) if _diffs else np.nan

    mlb_h2h_lookup[(_away_name.lower(), _home_name.lower())] = {
        "h2h_n_games": _n, "h2h_home_win_rate": _raw_rate, "h2h_run_diff_avg": _avg_diff,
        "series_home_wins": _series_home_wins, "series_away_wins": _series_away_wins,
        "h2h_bayesian_rate": _bayes_rate, "h2h_weight": _w,
    }

    print(
        f"  H2H {_away_name} @ {_home_name}: {_n} games | "
        f"home W% {_raw_rate:.3f} -> Bayes {_bayes_rate:.3f} | "
        f"avg run diff {_avg_diff:+.1f} | blend weight {_w:.0%}"
    )

    _h2h_rows.append({
        "idx"              : _idx,
        "h2h_n_games"      : _n,
        "h2h_home_wins"    : _home_wins,
        "h2h_home_win_rate": _raw_rate,
        "h2h_bayesian_rate": _bayes_rate,
        "h2h_weight"       : _w,
        "h2h_run_diff_avg" : _avg_diff,
        "series_game_num"  : _series_game_num,
        "series_home_wins" : _series_home_wins,
        "series_away_wins" : _series_away_wins,
    })

# ---------------------------------------------------------
# Attach H2H columns and compute adjusted probabilities
# ---------------------------------------------------------
_h2h_df = pd.DataFrame(_h2h_rows).set_index("idx")

for _col in ["h2h_n_games", "h2h_home_wins", "h2h_home_win_rate",
             "h2h_bayesian_rate", "h2h_weight", "h2h_run_diff_avg",
             "series_game_num", "series_home_wins", "series_away_wins"]:
    _board[_col] = _h2h_df[_col]

_model_home = pd.to_numeric(
    _board.get("pred_home_win_prob", pd.Series(dtype=float)), errors="coerce"
)
_h2h_bayes = _board["h2h_bayesian_rate"].fillna(_PRIOR_P)
_h2h_w     = _board["h2h_weight"].fillna(0.0)

_board["h2h_adj_home_win_prob"] = (1 - _h2h_w) * _model_home + _h2h_w * _h2h_bayes
_board["h2h_adj_away_win_prob"] = 1.0 - _board["h2h_adj_home_win_prob"]

# Write back â€” only overwrite when H2H result is not NaN (guard against NaN propagation)
# Align _model_home to board index in case pred_home_win_prob was missing (empty Series)
_model_home_aligned = _model_home.reindex(_board.index) if len(_model_home) != len(_board) else _model_home
_board["pred_home_win_prob"] = np.where(
    _board["h2h_adj_home_win_prob"].notna(),
    _board["h2h_adj_home_win_prob"],
    _model_home_aligned
)
_board["pred_away_win_prob"] = _board["h2h_adj_away_win_prob"]
_board["projected_winner"] = np.where(
    _board["pred_home_win_prob"] >= 0.5,
    _board["home_team"].astype(str),
    _board["away_team"].astype(str),
)
_board["projected_win_prob"] = np.where(
    _board["pred_home_win_prob"] >= 0.5,
    _board["pred_home_win_prob"],
    _board["pred_away_win_prob"],
)

# ---------------------------------------------------------
# Display summary
# ---------------------------------------------------------
print()
print("H2H ADJUSTMENT SUMMARY:")
_show = [c for c in [
    "away_team", "home_team",
    "h2h_n_games", "h2h_home_win_rate", "h2h_bayesian_rate",
    "h2h_weight", "h2h_run_diff_avg",
    "h2h_adj_home_win_prob", "h2h_adj_away_win_prob",
] if c in _board.columns]
display(_board[_show])

# ---------------------------------------------------------
# Publish updated board
# ---------------------------------------------------------
scored_mlb_board                    = _board.copy()
globals()["scored_mlb_board"]       = scored_mlb_board
globals()["mlb_scored_board"]       = scored_mlb_board
globals()["mlb_today_scored_board"] = scored_mlb_board

print()
print("Updated globals: scored_mlb_board | mlb_scored_board | mlb_today_scored_board")
print("Done.")


CELL H2H â€” CURRENT-SEASON HEAD-TO-HEAD ADJUSTMENT


  H2H Chicago White Sox @ Cleveland Guardians: 13 games | home W% 0.462 -> Bayes 0.472 | avg run diff -0.1 | blend weight 20%

H2H ADJUSTMENT SUMMARY:


,away_team,home_team,h2h_n_games,h2h_home_win_rate,h2h_bayesian_rate,h2h_weight,h2h_run_diff_avg,h2h_adj_home_win_prob,h2h_adj_away_win_prob
0,Chicago White Sox,Cleveland Guardians,13,0.461538,0.472222,0.2,-0.076923,NaN,NaN



Updated globals: scored_mlb_board | mlb_scored_board | mlb_today_scored_board
Done.


In [35]:
# %%
# =========================================================
# CELL BULLPEN -- BULLPEN ERA + FATIGUE ADJUSTMENT
# Fetches current-season team pitching ERA from MLB Stats API.
# Estimates bullpen ERA by subtracting starter contribution.
# Also counts games in last 3 days as a fatigue proxy.
# Blends a win probability adjustment into scored_mlb_board.
# Max adjustment: +/-4% win probability per game.
# =========================================================

import pandas as pd
import numpy as np
import requests
import datetime

print("=" * 70)
print("CELL BULLPEN -- BULLPEN ERA + FATIGUE ADJUSTMENT")
print("=" * 70)

if "scored_mlb_board" not in globals() or not isinstance(scored_mlb_board, pd.DataFrame):
    raise ValueError("scored_mlb_board not found. Run scoring cells first.")

_board = scored_mlb_board.copy()

# ---------------------------------------------------------
# 1) Fetch season team pitching stats (all teams, one call)
# ---------------------------------------------------------
_SEASON = 2026
_BP_URL = (
    f"https://statsapi.mlb.com/api/v1/teams/stats"
    f"?stats=season&group=pitching&season={_SEASON}&sportId=1"
)

_team_era = {}       # team_id -> season ERA (float)
_team_ip  = {}       # team_id -> innings pitched total (float)

try:
    _r = requests.get(_BP_URL, timeout=10)
    _r.raise_for_status()
    _bp_data = _r.json()
    for _split in _bp_data.get("stats", [{}])[0].get("splits", []):
        _tid  = _split.get("team", {}).get("id")
        _stat = _split.get("stat", {})
        _era  = _stat.get("era")
        _ip   = _stat.get("inningsPitched")
        if _tid and _era is not None:
            try:
                _team_era[_tid] = float(_era)
                _team_ip[_tid]  = float(str(_ip).replace(".", "").replace(",", ""))
            except Exception:
                pass
    print(f"  Loaded pitching ERA for {len(_team_era)} teams.")
except Exception as _e:
    print(f"  WARN: could not fetch team pitching stats ({_e}). Skipping bullpen adjustment.")
    _team_era = {}

# ---------------------------------------------------------
# 2) Fetch recent game counts for fatigue (last 3 days)
# ---------------------------------------------------------
_today    = datetime.date.today()
_cut3     = _today - datetime.timedelta(days=3)
_team_games_last3 = {}   # team_id -> games played in last 3 days

def _count_recent_games(team_id):
    try:
        _url = (
            f"https://statsapi.mlb.com/api/v1/schedule"
            f"?sportId=1&teamId={team_id}"
            f"&startDate={_cut3}&endDate={_today - datetime.timedelta(days=1)}&gameType=R"
        )
        _rg = requests.get(_url, timeout=8)
        _rg.raise_for_status()
        _gd = _rg.json()
        _n = sum(len(d.get("games", [])) for d in _gd.get("dates", []))
        return _n
    except Exception:
        return 0

# Collect unique team IDs from today's board
_all_team_ids = set()
for _col in ["home_team_id", "away_team_id"]:
    if _col in _board.columns:
        _all_team_ids.update(_board[_col].dropna().astype(int).tolist())

print(f"  Fetching fatigue data for {len(_all_team_ids)} teams...")
for _tid in _all_team_ids:
    _team_games_last3[int(_tid)] = _count_recent_games(int(_tid))

# ---------------------------------------------------------
# 3) Compute adjustment per game
# ---------------------------------------------------------
# Bullpen ERA proxy: estimate by stripping out starter contribution.
# Average MLB starter: ~5.5 IP/game. Bullpen: ~3.5 IP/game.
# bullpen_era = (team_era * 9 - sp_era * 5.5) / 3.5
# If no SP ERA available, fall back to team ERA directly.

_SP_IP   = 5.5   # assumed starter innings
_BP_IP   = 3.5   # assumed bullpen innings (9 - SP_IP)
_ADJ_PER_ERA = 0.018   # ~1.8% win prob shift per 1.0 ERA point advantage
_MAX_ADJ = 0.04         # cap at 4% total
_FATIGUE_ADJ = 0.012    # 1.2% penalty per game over 2 in last 3 days

_bp_rows = []

for _idx, _row in _board.iterrows():
    _h_id = _row.get("home_team_id")
    _a_id = _row.get("away_team_id")
    _h_sp_era = pd.to_numeric(_row.get("home_sp_era", np.nan), errors="coerce")
    _a_sp_era = pd.to_numeric(_row.get("away_sp_era", np.nan), errors="coerce")

    _h_era = _team_era.get(int(_h_id) if not pd.isna(_h_id) else -1, np.nan)
    _a_era = _team_era.get(int(_a_id) if not pd.isna(_a_id) else -1, np.nan)

    # Bullpen ERA estimate
    def _bp_era(team_era, sp_era):
        if pd.isna(team_era):
            return np.nan
        if not pd.isna(sp_era):
            bp = (team_era * 9 - sp_era * _SP_IP) / _BP_IP
            return float(np.clip(bp, 0.5, 12.0))
        return float(team_era)  # fallback: use team ERA directly

    _h_bp_era = _bp_era(_h_era, _h_sp_era)
    _a_bp_era = _bp_era(_a_era, _a_sp_era)

    # ERA diff: positive = home bullpen worse (home team disadvantaged)
    _era_diff = (_h_bp_era - _a_bp_era) if not (pd.isna(_h_bp_era) or pd.isna(_a_bp_era)) else 0.0

    # Win prob shift: better bullpen boosts the team's win prob
    # home_bp better (lower ERA) -> positive shift for home team
    _era_shift = float(np.clip(-_era_diff * _ADJ_PER_ERA, -_MAX_ADJ, _MAX_ADJ))

    # Fatigue adjustment
    _h_g3 = _team_games_last3.get(int(_h_id) if not pd.isna(_h_id) else -1, 0)
    _a_g3 = _team_games_last3.get(int(_a_id) if not pd.isna(_a_id) else -1, 0)
    _h_fatigue = max(0, _h_g3 - 2) * _FATIGUE_ADJ
    _a_fatigue = max(0, _a_g3 - 2) * _FATIGUE_ADJ
    _fatigue_shift = float(np.clip(_a_fatigue - _h_fatigue, -_MAX_ADJ, _MAX_ADJ))

    _total_shift = float(np.clip(_era_shift + _fatigue_shift, -_MAX_ADJ, _MAX_ADJ))

    _bp_rows.append({
        "idx"               : _idx,
        "home_bp_era"       : _h_bp_era,
        "away_bp_era"       : _a_bp_era,
        "bp_era_diff"       : _era_diff,
        "home_games_last3"  : _h_g3,
        "away_games_last3"  : _a_g3,
        "bp_era_shift"      : _era_shift,
        "bp_fatigue_shift"  : _fatigue_shift,
        "bp_total_shift"    : _total_shift,
    })

# ---------------------------------------------------------
# 4) Apply adjustment to board
# ---------------------------------------------------------
_bp_df = pd.DataFrame(_bp_rows).set_index("idx")

for _col in _bp_df.columns:
    _board[_col] = _bp_df[_col]

# Blend: adjusted home win prob = model prob + bullpen shift
# Guard: if pred_home_win_prob missing entirely, use 0.5 as neutral fallback
if "pred_home_win_prob" not in _board.columns:
    print("WARN: pred_home_win_prob missing from board â€” using 0.5 neutral fallback for bullpen blend")
    _board["pred_home_win_prob"] = 0.5
_model_home = pd.to_numeric(_board["pred_home_win_prob"], errors="coerce")
_board["bp_adj_home_win_prob"] = (_model_home + _board["bp_total_shift"]).clip(0.0, 1.0)
_board["bp_adj_away_win_prob"] = 1.0 - _board["bp_adj_home_win_prob"]

# Write back â€” only overwrite when bullpen adjustment is not NaN (guard against NaN propagation)
_board["pred_home_win_prob"] = np.where(
    _board["bp_adj_home_win_prob"].notna(),
    _board["bp_adj_home_win_prob"],
    _model_home
)
_board["pred_away_win_prob"] = _board["bp_adj_away_win_prob"]
_board["projected_winner"] = np.where(
    _board["pred_home_win_prob"] >= 0.5,
    _board["home_team"].astype(str),
    _board["away_team"].astype(str),
)
_board["projected_win_prob"] = np.where(
    _board["pred_home_win_prob"] >= 0.5,
    _board["pred_home_win_prob"],
    _board["pred_away_win_prob"],
)

# ---------------------------------------------------------
# 5) Summary
# ---------------------------------------------------------
print()
print("BULLPEN ADJUSTMENT SUMMARY:")
_show = [c for c in [
    "away_team", "home_team",
    "away_bp_era", "home_bp_era", "bp_era_diff",
    "away_games_last3", "home_games_last3",
    "bp_era_shift", "bp_fatigue_shift", "bp_total_shift",
    "bp_adj_home_win_prob", "bp_adj_away_win_prob",
] if c in _board.columns]
display(_board[_show])

scored_mlb_board                    = _board.copy()
globals()["scored_mlb_board"]       = scored_mlb_board
globals()["mlb_scored_board"]       = scored_mlb_board
globals()["mlb_today_scored_board"] = scored_mlb_board

print()
print("Updated globals with bullpen-adjusted probabilities.")
print("Done.")


# ---------------------------------------------------------
# BULLPEN INDIVIDUAL QUALITY: tiered reliever analysis
# Fetches all 2026 active pitcher stats in one call.
# Classifies each pitcher as: HL (closer/setup), ML (middle),
# or WEAK (regular arm with ERA > 5.0).
# Outputs per team:
#   mlb_bp_hl_era      : avg ERA of high-leverage arms
#   mlb_bp_weak_link   : {era, name} of worst regular reliever
#   mlb_bp_risk_score  : 0 (clean) -> 2 (blowup risk)
# ---------------------------------------------------------
mlb_bullpen_depth   = {}   # team_norm -> 0.0-1.0 (legacy depth score)
mlb_bp_hl_era       = {}   # team_norm -> float
mlb_bp_hl_names     = {}   # team_norm -> [(name, season_era), ...] season-long HL arms
mlb_bp_hl_recent    = {}   # team_norm -> {era: float, names: [(name, recent_era), ...]} trailing-30d HL arms
mlb_bp_weak_link    = {}   # team_norm -> {era: float, name: str}
mlb_bp_risk_score   = {}   # team_norm -> int 0/1/2
mlb_bp_depth        = {}   # team_norm -> {shaky_share, shaky_ip, total_ip, shaky_names, run_adj}

import unicodedata as _bpud, re as _bpre, datetime as _bpdt

def _bp_norm(s):
    if not s: return ""
    s = _bpud.normalize("NFD", str(s))
    s = "".join(c for c in s if _bpud.category(c) != "Mn")
    return _bpre.sub(r"[^a-z0-9 ]", "", s.lower()).strip()

_today_bp  = globals().get("CARD_DATE") or _bpdt.date.today().strftime("%Y-%m-%d")
_MLB_BP_API = "https://statsapi.mlb.com/api/v1"
_SEASON_BP  = 2026

# Thresholds
_HL_ERA_MAX    = 3.50   # ERA floor for high-leverage arms
_HL_MIN_APP    = 10     # min appearances to be considered HL
_WEAK_ERA_MIN  = 5.00   # ERA threshold to be flagged as weak link
_WEAK_MIN_APP  = 8      # min appearances to be a "regular" weak arm
_ML_ERA_MAX    = 4.75   # middle relief threshold

try:
    # ----- One-call: all pitcher season stats -----
    _pst_url = (f"{_MLB_BP_API}/stats?stats=season&group=pitching"
                f"&season={_SEASON_BP}&sportId=1&gameType=R&limit=1500&playerPool=ALL")
    _pst_r   = requests.get(_pst_url, timeout=15)
    _pst_splits = _pst_r.json().get("stats",[{}])[0].get("splits",[])
    print(f"  Individual pitcher stats: {len(_pst_splits)} entries")

    # BULLPEN STRENGTH (2026-09-02, user audit -- real Orioles loss:
    # DraftKings preview had Romano (no runs allowed since Aug 4) and
    # Bernardino (1.59 ERA since ASB) as genuinely elite, but the card only
    # ever showed BULLPEN RISK (worst arm) / BULLPEN DEPTH (share of shaky
    # arms) -- there was no signal anywhere for "how good is this bullpen's
    # top end". mlb_bp_hl_era already existed (avg ERA of season-long
    # high-leverage arms) but was computed and never referenced again
    # anywhere in Cell 87/89 -- fully orphaned. This adds a RECENCY-aware
    # version alongside it: a reliever who was bad early and dominant since
    # a form change (exactly the Romano/Bernardino case) may not even clear
    # the season-ERA HL bar, same staleness pattern already fixed for
    # starters elsewhere in this notebook (L3/L10 vs season ERA).
    _HL_RECENT_WINDOW_DAYS = 30
    _HL_RECENT_ERA_MAX     = 2.75   # tighter than season bar -- only flag genuinely hot arms
    _HL_RECENT_MIN_APP     = 6      # min relief appearances within the window
    _recent_era_by_pid = {}  # pid -> (era, rel_app) over the trailing window
    try:
        _recent_end = _bpdt.date.today()
        _recent_start = _recent_end - _bpdt.timedelta(days=_HL_RECENT_WINDOW_DAYS)
        _pst_recent_url = (f"{_MLB_BP_API}/stats?stats=byDateRange&group=pitching"
                           f"&season={_SEASON_BP}&sportId=1&limit=1500&playerPool=ALL"
                           f"&startDate={_recent_start.isoformat()}&endDate={_recent_end.isoformat()}")
        _pst_recent_r = requests.get(_pst_recent_url, timeout=15)
        for _rsp in _pst_recent_r.json().get("stats", [{}])[0].get("splits", []):
            _rst = _rsp.get("stat", {})
            _rg, _rgs = int(_rst.get("gamesPlayed", 0) or 0), int(_rst.get("gamesStarted", 0) or 0)
            _rrel = _rg - _rgs
            _rpid = _rsp.get("player", {}).get("id")
            if _rpid and _rrel >= 1 and _rgs <= _rrel:
                _recent_era_by_pid[_rpid] = (float(_rst.get("era", 99) or 99), _rrel)
        print(f"  Recent-window ({_HL_RECENT_WINDOW_DAYS}d) pitcher stats: {len(_recent_era_by_pid)} relievers")
    except Exception as _hlr_e:
        print(f"  [WARN] recent-window bullpen strength fetch failed: {_hlr_e}")

    # BUG FIX (2026-09-02, user audit -- bullpen depth signal showed a
    # nonsense "4271 relief IP" for a single team): .replace(".1","0.33")
    # on a multi-digit IP string corrupts it -- "41.2".replace(".2","0.67")
    # produces "410.67" (the "." gets replaced mid-string, concatenating
    # digits) instead of the intended 41.667. This exact baseball-notation
    # bug (".1"/".2" mean thirds of an inning, not decimal tenths) was
    # already fixed elsewhere in this notebook with a real parser
    # (_sp_parse_ip/_l3_parse_ip) -- this cell had its own unfixed copy.
    # Silently harmless before now because nothing here ever summed/used
    # the ip field (only era/rel_app fed the existing weak-link/HL
    # classification) -- mlb_bp_depth (below) is the first consumer to
    # actually sum it across a roster, which is what surfaced the bug.
    def _bp_parse_ip(v):
        try:
            _s = str(v); _p = _s.split(".")
            return int(_p[0]) + (int(_p[1]) if len(_p) > 1 else 0) / 3.0
        except Exception:
            return 0.0

    # Group by team
    _by_team = {}   # team_norm -> list of reliever stat dicts
    for _sp in _pst_splits:
        _st  = _sp.get("stat", {})
        _g   = int(_st.get("gamesPlayed", 0) or 0)
        _gs  = int(_st.get("gamesStarted", 0) or 0)
        _ip  = _bp_parse_ip(_st.get("inningsPitched", "0"))
        _era = float(_st.get("era", 99) or 99)
        _rel = _g - _gs   # relief appearances

        # Only track true relievers (more relief apps than starts, at least 1 rel app)
        if _rel < 1 or _gs > _rel:
            continue

        _tn  = _bp_norm(_sp.get("team", {}).get("name", ""))
        _nm  = _sp.get("player", {}).get("fullName", "")
        _pid = _sp.get("player", {}).get("id")
        if not _tn: continue

        if _tn not in _by_team:
            _by_team[_tn] = []
        _by_team[_tn].append({
            "name": _nm, "era": _era, "rel_app": _rel, "ip": _ip, "id": _pid
        })

    # 2026-09-09 (user audit -- Cole Sands/Will Vest, both on the MLB
    # injured list, still counted as bullpen "weak links" and folded into
    # the risk score / run adjustment that actually moves tonight's
    # projection; the later cross-check in Cell 89 only PRINTS a warning
    # after the fact, it never corrects the number itself). Live-checks
    # each IL status (any D-code -- D7/D10/D15/D60, none of them can pitch
    # tonight) for just today's teams (scoped via `board`, not all 30, to
    # keep this fast) and excludes those arms from the weak-link pool
    # BEFORE scoring, not after.
    _bp_il_pitchers = {}   # team_norm -> set of normalized names currently on the IL
    # OFF-ROSTER FILTER (2026-09-14, user audit -- real Rangers loss: [BULLPEN
    # RISK] named Luis Curvelo (9.58), Joe Ross (9.26), and Chris Martin (9.00)
    # as weak links -- confirmed live none of the three are on the 40-man
    # roster anymore (traded/DFA'd/released at some point this season), so
    # none of them could possibly pitch that night. The season-long individual-
    # pitcher stat fetch above (_pst_splits) includes anyone who threw a pitch
    # for a team at ANY point this season, with no check that they're still on
    # the roster -- the IL filter next to this only catches a DIFFERENT subset
    # (rostered but injured), so a pitcher who left the organization entirely
    # sailed through both checks. Same roster call already being made for the
    # IL check answers this too -- just keep the full pitcher name set from it,
    # not only the IL subset, so this is a zero-extra-cost extension of a fetch
    # that already happens every run.
    _bp_current_roster = {}   # team_norm -> set of normalized pitcher names on the current 40-man
    try:
        _bp_today_teams = set()
        if "board" in globals() and isinstance(board, pd.DataFrame):
            for _bp_col in ("home_team", "away_team"):
                if _bp_col in board.columns:
                    _bp_today_teams |= set(board[_bp_col].dropna().astype(str))
        if _bp_today_teams:
            _bp_team_id_cache = {}
            for _t in requests.get(f"{_MLB_BP_API}/teams?sportId=1", timeout=10).json().get("teams", []):
                if _t.get("name"): _bp_team_id_cache[_t["name"].lower()] = _t.get("id")
                if _t.get("teamName"): _bp_team_id_cache[_t["teamName"].lower()] = _t.get("id")
            for _bp_team_full in _bp_today_teams:
                _bp_tid = next((v for k, v in _bp_team_id_cache.items()
                                 if v and (k in _bp_team_full.lower() or _bp_team_full.lower() in k)), None)
                if not _bp_tid:
                    continue
                try:
                    _bp_roster = requests.get(
                        f"{_MLB_BP_API}/teams/{_bp_tid}/roster?rosterType=40Man", timeout=10
                    ).json().get("roster", [])
                    _bp_tn_key = _bp_norm(_bp_team_full)
                    _bp_il_pitchers[_bp_tn_key] = {
                        _bp_norm(p.get("person", {}).get("fullName", ""))
                        for p in _bp_roster
                        if str((p.get("status") or {}).get("code", "")).startswith("D")
                        and (p.get("position") or {}).get("type") == "Pitcher"
                    }
                    _bp_current_roster[_bp_tn_key] = {
                        _bp_norm(p.get("person", {}).get("fullName", ""))
                        for p in _bp_roster
                        if (p.get("position") or {}).get("type") == "Pitcher"
                    }
                except Exception:
                    pass
            if _bp_il_pitchers:
                print(f"  [BULLPEN IL FILTER] live IL + off-roster check for {len(_bp_il_pitchers)} team(s) with a game today")
    except Exception as _bp_il_e:
        print(f"  WARN: bullpen IL/roster filter fetch failed ({_bp_il_e}) -- weak-link pool not filtered this run")

    # Score each team
    for _tn, _relievers in _by_team.items():
        _bp_roster_here = _bp_current_roster.get(_tn, set())
        if _bp_roster_here:
            _relievers = [r for r in _relievers if _bp_norm(r["name"]) in _bp_roster_here]
        _bp_il_here = _bp_il_pitchers.get(_tn, set())
        if _bp_il_here:
            _relievers = [r for r in _relievers if _bp_norm(r["name"]) not in _bp_il_here]
        _hl  = [r for r in _relievers if r["era"] <= _HL_ERA_MAX and r["rel_app"] >= _HL_MIN_APP]
        _weak = [r for r in _relievers if r["era"] >= _WEAK_ERA_MIN and r["rel_app"] >= _WEAK_MIN_APP]
        _mid  = [r for r in _relievers if _HL_ERA_MAX < r["era"] < _ML_ERA_MAX and r["rel_app"] >= 5]

        mlb_bp_hl_era[_tn]   = round(float(np.mean([r["era"] for r in _hl])), 2) if _hl else 4.00
        if _hl:
            mlb_bp_hl_names[_tn] = sorted([(r["name"], r["era"]) for r in _hl], key=lambda x: x[1])

        _hl_recent = [r for r in _relievers
                      if _recent_era_by_pid.get(r["id"], (99.0, 0))[0] <= _HL_RECENT_ERA_MAX
                      and _recent_era_by_pid.get(r["id"], (99.0, 0))[1] >= _HL_RECENT_MIN_APP]
        if _hl_recent:
            mlb_bp_hl_recent[_tn] = {
                "era": round(float(np.mean([_recent_era_by_pid[r["id"]][0] for r in _hl_recent])), 2),
                "names": sorted([(r["name"], round(_recent_era_by_pid[r["id"]][0], 2)) for r in _hl_recent], key=lambda x: x[1]),
            }

        if _weak:
            _worst = max(_weak, key=lambda r: r["era"])
            # BUG FIX (2026-09-02, found live -- real Orioles loss: Baltimore
            # had TWO relievers clear the weak bar (Akin 5.68, Wolfram 5.15),
            # but this only ever names the single worst -- Wolfram, unnamed
            # anywhere on the card, took the loss and blew the save tonight.
            # A reader seeing "weak link: Akin" reasonably assumes that's
            # BAL's only bullpen risk. Keeps the existing era/name/apps keys
            # (worst arm, unchanged) and adds all/count for the full picture.
            mlb_bp_weak_link[_tn] = {
                "era": _worst["era"], "name": _worst["name"], "apps": _worst["rel_app"],
                "all": sorted([{"name": r["name"], "era": r["era"]} for r in _weak], key=lambda r: -r["era"]),
                "count": len(_weak),
            }
        else:
            mlb_bp_weak_link[_tn] = {"era": 0.0, "name": "", "apps": 0, "all": [], "count": 0}

        # Risk score: 0=clean, 1=weak link exists, 2=weak link + shallow HL pen
        _risk = 0
        if _weak: _risk += 1
        if len(_hl) < 2: _risk += 1   # fewer than 2 reliable HL arms
        mlb_bp_risk_score[_tn] = _risk

        # Legacy depth score (for Cell 85 compatibility)
        mlb_bullpen_depth[_tn] = 1.0 if _risk == 0 else (0.85 if _risk == 1 else 0.75)

        # BULLPEN DEPTH -- WORKLOAD-WEIGHTED SHAKY SHARE (2026-09-02, user
        # audit -- Brewers/Cubs total blew up, Astros ML lost, both to
        # bullpen arms mlb_bp_weak_link never named): the weak-link flag
        # above only ever names the single WORST reliever -- it says
        # nothing about how much of a team's actual relief workload comes
        # from arms that are merely mediocre (not bad enough to be "the"
        # weak link, but bad enough that 3-4 of them having an ordinary
        # night each can still blow up a game). Real case: Cubs' 6 late
        # runs came from four different relievers, none of them the named
        # weak link (Jayden Murray); Astros' bullpen (Teng/Imai, 5 combined
        # ER) wasn't flagged at all since neither individually cleared the
        # 5.00 ERA "weak" bar. Uses a lower bar (4.50 ERA, "shaky" not
        # "weak") and weights by each reliever's actual relief IP so one
        # rarely-used bad arm doesn't dominate the read the way a raw arm
        # COUNT would -- a team living on 3 shaky-but-not-terrible arms for
        # most of its bullpen innings is a real risk even with zero named
        # weak link.
        _DEPTH_SHAKY_ERA_MIN = 4.50   # below _WEAK_ERA_MIN (5.00) -- "shaky", not "weak"
        _DEPTH_MIN_TOTAL_IP  = 20.0   # deadband: below this the sample is too thin to read
        _DEPTH_SHARE_DEADBAND = 0.40  # 40%+ of relief IP from shaky arms to fire
        _DEPTH_MAX_R = 0.25           # smaller cap than the single-arm signals -- more speculative
        _total_ip = sum(r["ip"] for r in _relievers)
        _shaky = [r for r in _relievers if r["era"] >= _DEPTH_SHAKY_ERA_MIN]
        _shaky_ip = sum(r["ip"] for r in _shaky)
        _shaky_share = (_shaky_ip / _total_ip) if _total_ip > 0 else 0.0
        _depth_run_adj = 0.0
        if _total_ip >= _DEPTH_MIN_TOTAL_IP and _shaky_share >= _DEPTH_SHARE_DEADBAND:
            _depth_run_adj = round(min((_shaky_share - _DEPTH_SHARE_DEADBAND) * 1.5, _DEPTH_MAX_R), 3)
        mlb_bp_depth[_tn] = {
            "shaky_share": round(_shaky_share, 3),
            "shaky_ip": round(_shaky_ip, 1),
            "total_ip": round(_total_ip, 1),
            "shaky_count": len(_shaky),
            "shaky_names": sorted(_shaky, key=lambda r: -r["ip"])[:3],
            "run_adj": _depth_run_adj,
        }

    # Summary
    _risky = [(t, mlb_bp_weak_link[t]["name"], mlb_bp_weak_link[t]["era"])
              for t in mlb_bp_weak_link if mlb_bp_weak_link[t]["era"] >= _WEAK_ERA_MIN]
    _risky.sort(key=lambda x: -x[2])
    print(f"  Bullpen risk scored: {len(mlb_bp_risk_score)} teams")
    print(f"  Teams with weak-link reliever (ERA >= {_WEAK_ERA_MIN}): {len(_risky)}")
    for _tn2, _nm2, _e2 in _risky[:8]:
        print(f"    {_tn2}: WEAK LINK {_nm2} ({_e2:.2f} ERA, {mlb_bp_weak_link[_tn2]['apps']} apps)")

    _deep_risk = sorted([(t, v) for t, v in mlb_bp_depth.items() if v["run_adj"] > 0],
                         key=lambda x: -x[1]["run_adj"])
    print(f"  Bullpen depth (shaky-share, ERA >= {_DEPTH_SHAKY_ERA_MIN}): {len(_deep_risk)} team(s) over the deadband")
    for _tn3, _dv in _deep_risk[:8]:
        _names3 = ", ".join(f"{r['name']} ({r['era']:.2f})" for r in _dv["shaky_names"])
        print(f"    {_tn3}: {_dv['shaky_share']:.0%} of {_dv['total_ip']:.0f} relief IP from shaky arms "
              f"({_names3}) -> +{_dv['run_adj']:.2f}R vs opponent")

    # ---------------------------------------------------------
    # BULLPEN ARM FATIGUE: per-reliever recent usage for today's slate
    # The risk score above is season-aggregate (chronic quality) and
    # cannot see "elite arm pitched back-to-back and is running on
    # fumes tonight" -- that's a real, distinct blowup mechanism.
    # Flags a team's top HL relievers who appeared in 2+ of the last
    # 3 days. Scoped to today's board only to bound API calls.
    # Stored as mlb_bp_fatigue_flag {team_norm: {gassed: [names], hl_count: int}}
    # Not yet live-verified -- built 2026-08-11, needs a real-slate run.
    # ---------------------------------------------------------
    mlb_bp_fatigue_flag = {}
    try:
        _fatigue_teams = set()
        for _fsrc_name in ["board", "scored_mlb_board", "projection_board"]:
            _fbrd = globals().get(_fsrc_name)
            if isinstance(_fbrd, pd.DataFrame):
                for _fcol in ["away_team", "home_team"]:
                    if _fcol in _fbrd.columns:
                        _fatigue_teams.update(_bp_norm(str(t)) for t in _fbrd[_fcol].dropna())
                break
        _FATIGUE_LOOKBACK_DAYS = 3
        _FATIGUE_APP_THRESHOLD = 2
        _fatigue_cut = _bpdt.date.today() - _bpdt.timedelta(days=_FATIGUE_LOOKBACK_DAYS)
        for _tn in _fatigue_teams:
            _hl_ranked = sorted(
                [r for r in _by_team.get(_tn, [])
                 if r["era"] <= _HL_ERA_MAX and r["rel_app"] >= _HL_MIN_APP and r.get("id")],
                key=lambda r: r["era"]
            )[:3]
            if not _hl_ranked:
                continue
            _gassed = []
            for _r in _hl_ranked:
                try:
                    _gl_url = (f"{_MLB_BP_API}/people/{_r['id']}/stats"
                               f"?stats=gameLog&group=pitching&season={_SEASON_BP}&gameType=R")
                    _gl_r = requests.get(_gl_url, timeout=8)
                    _gl_splits = _gl_r.json().get("stats", [{}])[0].get("splits", []) if _gl_r.status_code == 200 else []
                    _apps_recent = sum(
                        1 for _s in _gl_splits
                        if _s.get("date") and _bpdt.date.fromisoformat(_s["date"]) >= _fatigue_cut
                    )
                    if _apps_recent >= _FATIGUE_APP_THRESHOLD:
                        _gassed.append(_r["name"])
                except Exception:
                    continue
            if _gassed:
                mlb_bp_fatigue_flag[_tn] = {"gassed": _gassed, "hl_count": len(_hl_ranked)}
        if mlb_bp_fatigue_flag:
            print(f"  Bullpen fatigue flagged: {len(mlb_bp_fatigue_flag)} team(s)")
            for _tn3, _fd in mlb_bp_fatigue_flag.items():
                print(f"    {_tn3}: gassed={_fd['gassed']} (of {_fd['hl_count']} top arms)")
    except Exception as _fat_e:
        print(f"  WARN: bullpen fatigue check failed ({_fat_e})")

except Exception as _bpe:
    print(f"  WARN: individual bullpen fetch failed ({_bpe})")
    mlb_bullpen_depth = {}

# ---------------------------------------------------------
# STARTER EARLY EXIT RISK
# If SP WHIP > 1.50 or ip_l3 < 15: flag for early pen exposure.
# Team will need mop-up arms by inning 5-6 — doubles blowup risk.
# Stored as mlb_sp_early_exit {team_norm: True/False}
# ---------------------------------------------------------
mlb_sp_early_exit = {}   # team_norm -> bool

if "mlb_pitcher_fip_lookup" in globals():
    import unicodedata as _ee_ud, re as _ee_re
    def _ee_norm(s):
        if not s: return ""
        s = _ee_ud.normalize("NFD", str(s))
        s = "".join(c for c in s if _ee_ud.category(c) != "Mn")
        return _ee_re.sub(r"[^a-z ]", "", s.lower()).strip()

    for _src_name in ["board", "scored_mlb_board", "projection_board"]:
        _brd = globals().get(_src_name)
        if not isinstance(_brd, pd.DataFrame): continue
        for _side in ["away", "home"]:
            _pncol = f"{_side}_pitcher_name"
            _tmcol = f"{_side}_team"
            if _pncol not in _brd.columns: continue
            for _, _row in _brd.iterrows():
                _pn = str(_row.get(_pncol, "") or "")
                _tm = _bp_norm(str(_row.get(_tmcol, "") or ""))
                if not _pn or not _tm: continue
                _ev = mlb_pitcher_fip_lookup.get(_ee_norm(_pn), {})
                _whip = float(_ev.get("whip_l3") or 0)
                _ip   = float(_ev.get("ip_l3") or 99)
                _ip_s = float(_ev.get("ip_season") or 99)
                # Early exit risk: high WHIP or very small sample (debut-level)
                if (_whip > 1.50 and _whip > 0) or _ip_s < 20:
                    mlb_sp_early_exit[_tm] = True
        break

    if mlb_sp_early_exit:
        print(f"  Early exit risk flagged: {list(mlb_sp_early_exit.keys())}")


# ---------------------------------------------------------
# BULLPEN WORKLOAD, LAST 3 DAYS -- TOTALS-SIDE FATIGUE INPUT
# 2026-08-29 (user audit M1): bullpen fatigue only ever reached the MONEYLINE
# (the +/-4% win-prob shift at the top of this cell, which is a games-played
# proxy) and the top-3-arm "gassed" flag. Neither carries workload VOLUME --
# how many batters the whole pen actually faced -- and nothing fed the run
# TOTAL projection at all, so O/U was priced off starters plus a season-average
# bullpen RA no matter how hard the pen had been ridden. One byDateRange call
# returns every pitcher's range totals league-wide; relievers are the rows with
# gamesStarted == 0 inside the window. Excess vs the league median for the same
# window is priced as runs in Cell bd676982 (which is what carries it into
# pred_total_runs and therefore the O/U edge).
# Stored as mlb_bp_workload_l3 {team_norm: {...}}.
# ---------------------------------------------------------
mlb_bp_workload_l3 = {}
_BP_WORKLOAD_DAYS       = 3
_BP_WORKLOAD_R_PER_BF   = 0.010   # provisional -- revisit after live grading
_BP_WORKLOAD_R_PER_ARM  = 0.05    # per back-to-back arm above the league median
_BP_WORKLOAD_MAX_R      = 0.35    # same magnitude family as the +/-4% ML fatigue cap / +0.25R gassed flag
_BP_WORKLOAD_MIN_BF_EXC = 8       # deadband: under ~a quarter-game of extra pen work is noise
try:
    _bw_end   = _bpdt.date.today() - _bpdt.timedelta(days=1)
    _bw_start = _bpdt.date.today() - _bpdt.timedelta(days=_BP_WORKLOAD_DAYS)
    # MLB-POSTSEASON-1 (2026-09-29, user audit -- "shorten SP leash assumptions and
    # weight the top 3-4 relievers more heavily in bullpen projections, since managers
    # shorten rotations and pens in October"): this was hardcoded gameType=R -- once the
    # regular season ends, an R-only lookback over the last 3 days returns ZERO games
    # (there are none), so this entire bullpen-fatigue/workload signal silently goes dark
    # exactly when postseason bullpen taxation matters most. Live-verified 2026-09-29:
    # gameType=R,F,D,L,W returns real rows from the SAME byDateRange endpoint (confirmed
    # against a real 3-day window spanning today). Every request path below is otherwise
    # unchanged -- this only widens which games' appearances get counted.
    _bw_url = (f"{_MLB_BP_API}/stats?stats=byDateRange&group=pitching&gameType=R,F,D,L,W"
               f"&startDate={_bw_start}&endDate={_bw_end}&sportId=1&season={_SEASON_BP}"
               f"&playerPool=All&limit=2000")
    _bw_r = requests.get(_bw_url, timeout=25)
    _bw_splits = _bw_r.json().get("stats", [{}])[0].get("splits", []) if _bw_r.status_code == 200 else []
    if _bw_r.status_code != 200:
        print(f"  [BULLPEN WORKLOAD] byDateRange fetch returned HTTP {_bw_r.status_code} -- "
              f"no L3 bullpen-workload term on totals this run (NOT a rested-pen read)")
    for _bw_s in _bw_splits:
        _bw_st = _bw_s.get("stat", {}) or {}
        if int(_bw_st.get("gamesStarted", 0) or 0) > 0:
            continue  # started inside the window -- starter workload, not the pen
        _bw_tn = _bp_norm(str((_bw_s.get("team") or {}).get("name", "")))
        if not _bw_tn:
            continue
        _bw_e = mlb_bp_workload_l3.setdefault(
            _bw_tn, {"bf": 0, "pitches": 0, "outs": 0, "arms": 0, "b2b_arms": 0, "er": 0})
        _bw_e["bf"]      += int(_bw_st.get("battersFaced", 0) or 0)
        _bw_e["pitches"] += int(_bw_st.get("numberOfPitches", 0) or 0)
        _bw_e["outs"]    += int(_bw_st.get("outsPitched", 0) or 0)
        _bw_e["arms"]    += 1
        _bw_e["er"]      += int(_bw_st.get("earnedRuns", 0) or 0)
        if int(_bw_st.get("gamesPitched", 0) or 0) >= 2:
            _bw_e["b2b_arms"] += 1   # 2+ appearances in a 3-day window = back-to-back stress
    if mlb_bp_workload_l3:
        import statistics as _bwstat
        _bw_bf_med  = float(_bwstat.median([v["bf"] for v in mlb_bp_workload_l3.values()]))
        _bw_b2b_med = float(_bwstat.median([v["b2b_arms"] for v in mlb_bp_workload_l3.values()]))
        for _bw_tn, _bw_e in mlb_bp_workload_l3.items():
            _bw_e["ip"]         = round(_bw_e["outs"] / 3.0, 1)
            _bw_e["bf_median"]  = _bw_bf_med
            _bw_e["b2b_median"] = _bw_b2b_med
            _bw_bf_exc  = max(0.0, _bw_e["bf"] - _bw_bf_med)
            _bw_b2b_exc = max(0.0, _bw_e["b2b_arms"] - _bw_b2b_med)
            _bw_e["bf_excess"]  = round(_bw_bf_exc, 1)
            _bw_e["b2b_excess"] = round(_bw_b2b_exc, 1)
            _bw_run = 0.0
            if _bw_bf_exc >= _BP_WORKLOAD_MIN_BF_EXC:
                _bw_run = (_bw_bf_exc * _BP_WORKLOAD_R_PER_BF
                           + _bw_b2b_exc * _BP_WORKLOAD_R_PER_ARM)
            _bw_e["run_adj"] = round(min(_bw_run, _BP_WORKLOAD_MAX_R), 3)
        _bw_hot = sorted([(v["run_adj"], k, v) for k, v in mlb_bp_workload_l3.items() if v["run_adj"] > 0],
                         reverse=True)
        print(f"  Bullpen workload (last {_BP_WORKLOAD_DAYS}d): {len(mlb_bp_workload_l3)} teams, "
              f"league median {_bw_bf_med:.0f} BF / {_bw_b2b_med:.0f} back-to-back arm(s); "
              f"{len(_bw_hot)} team(s) over the fatigue deadband")
        for _bw_v, _bw_k, _bw_d in _bw_hot[:6]:
            print(f"    {_bw_k}: {_bw_d['bf']} BF, {_bw_d['ip']} IP, {_bw_d['b2b_arms']} back-to-back arm(s) "
                  f"-> +{_bw_v:.2f}R vs opponent")

        # ---------------------------------------------------------
        # BULLPEN L3 ERA -- REAL RUNS ALLOWED, NOT JUST VOLUME (2026-08-31,
        # user audit -- Red Sox pen, 9 ER Saturday): the workload block above
        # only measures how MUCH the pen has thrown (batters faced, back-to-
        # back arms) -- a team can face a normal workload and still get
        # shelled, which is exactly the Red Sox case this was built from (9
        # ER Saturday didn't register as unusual workload). This is a
        # separate, real runs-allowed signal from the same byDateRange fetch
        # (earnedRuns was already in the response, just never summed).
        # league-wide IP-weighted ERA (not a mean of per-team ERAs) avoids a
        # thin-sample team distorting the baseline; a per-team min-IP floor
        # (one team's normal bullpen workload for 3 days) keeps a single
        # mop-up inning from reading as a real ERA. Damped (0.5x) before
        # pricing as runs since a 3-day sample is noisy and shouldn't swing
        # the projection as hard as a full-season number would -- same
        # magnitude family as the workload cap above, applied independently
        # (both can fire on the same team; they measure different things).
        # ---------------------------------------------------------
        _BP_ERA_MIN_IP  = 3.0    # deadband: under ~a game's worth of pen innings, ERA is single-outing noise
        _BP_ERA_MAX_R   = 0.35   # same magnitude family as _BP_WORKLOAD_MAX_R
        _BP_ERA_DAMPING = 0.50   # 3-day sample is noisy -- don't price it as a full-confidence signal
        _bw_era_elig = {k: v for k, v in mlb_bp_workload_l3.items() if v.get("ip", 0) >= _BP_ERA_MIN_IP}
        if _bw_era_elig:
            _bw_lg_ip = sum(v["ip"] for v in _bw_era_elig.values())
            _bw_lg_er = sum(v["er"] for v in _bw_era_elig.values())
            _bw_lg_era = round(_bw_lg_er / _bw_lg_ip * 9, 2) if _bw_lg_ip > 0 else None
            for _bw_tn, _bw_e in mlb_bp_workload_l3.items():
                _bw_e["era_l3"] = round(_bw_e["er"] / _bw_e["ip"] * 9, 2) if _bw_e.get("ip", 0) > 0 else None
                _bw_e["era_l3_league_avg"] = _bw_lg_era
                _bw_e["era_run_adj"] = 0.0
                if _bw_tn in _bw_era_elig and _bw_lg_era is not None:
                    _bw_excess_runs = max(0.0, (_bw_e["era_l3"] - _bw_lg_era) / 9.0 * _bw_e["ip"])
                    _bw_e["era_run_adj"] = round(min(_bw_excess_runs * _BP_ERA_DAMPING, _BP_ERA_MAX_R), 3)
            _bw_era_hot = sorted(
                [(v["era_run_adj"], k, v) for k, v in mlb_bp_workload_l3.items() if v.get("era_run_adj", 0) > 0],
                reverse=True)
            print(f"  Bullpen L3 ERA: league avg {_bw_lg_era:.2f} (IP-weighted, {len(_bw_era_elig)} teams "
                  f"with {_BP_ERA_MIN_IP:.0f}+ IP); {len(_bw_era_hot)} team(s) over the ERA deadband")
            for _bw_v, _bw_k, _bw_d in _bw_era_hot[:6]:
                print(f"    {_bw_k}: {_bw_d['era_l3']:.2f} ERA ({_bw_d['er']} ER / {_bw_d['ip']} IP) "
                      f"-> +{_bw_v:.2f}R vs opponent (damped)")
        else:
            print(f"  [BULLPEN L3 ERA] no team reached the {_BP_ERA_MIN_IP:.0f} IP floor this run -- "
                  f"no L3 bullpen-ERA term on projections")
except Exception as _bw_e2:
    mlb_bp_workload_l3 = {}
    print(f"  WARN: bullpen workload fetch failed ({_bw_e2}) -- totals carry no L3 bullpen-fatigue term this run")


CELL BULLPEN -- BULLPEN ERA + FATIGUE ADJUSTMENT


  Loaded pitching ERA for 30 teams.
  Fetching fatigue data for 2 teams...



BULLPEN ADJUSTMENT SUMMARY:


,away_team,home_team,away_bp_era,home_bp_era,bp_era_diff,away_games_last3,home_games_last3,bp_era_shift,bp_fatigue_shift,bp_total_shift,bp_adj_home_win_prob,bp_adj_away_win_prob
0,Chicago White Sox,Cleveland Guardians,4.12,3.77,-0.35,0,0,0.0063,0.0,0.0063,NaN,NaN



Updated globals with bullpen-adjusted probabilities.
Done.


  Individual pitcher stats: 868 entries


  [WARN] recent-window bullpen strength fetch failed: could not convert string to float: '-.--'


  [BULLPEN IL FILTER] live IL + off-roster check for 2 team(s) with a game today
  Bullpen risk scored: 30 teams
  Teams with weak-link reliever (ERA >= 5.0): 28
    athletics: WEAK LINK Yunior Tur (16.34 ERA, 8 apps)
    new york mets: WEAK LINK Chris Devenski (11.74 ERA, 9 apps)
    los angeles angels: WEAK LINK Nick Sandlin (11.42 ERA, 8 apps)
    washington nationals: WEAK LINK Matt Krook (10.80 ERA, 11 apps)
    kansas city royals: WEAK LINK Aaron Bummer (10.26 ERA, 21 apps)
    seattle mariners: WEAK LINK Josh Simpson (9.82 ERA, 8 apps)
    texas rangers: WEAK LINK Luis Curvelo (9.58 ERA, 9 apps)
    chicago cubs: WEAK LINK Jayden Murray (9.37 ERA, 10 apps)
  Bullpen depth (shaky-share, ERA >= 4.5): 9 team(s) over the deadband
    colorado rockies: 73% of 628 relief IP from shaky arms (Tanner Gordon (6.68), Zach Agnos (5.93), Juan Mejia (5.27)) -> +0.25R vs opponent
    washington nationals: 61% of 846 relief IP from shaky arms (Miles Mikolas (5.82), Clayton Beeter (4.50), Matt W

In [36]:
# %%
# =========================================================
# CELL 14 Ã¢â‚¬â€ FINAL MLB PLAYABLE CARD (CLEAN + ROBUST)
# Works with compressed baseline model (no starter dependency)
# =========================================================

import pandas as pd
import numpy as np

print("=" * 70)
print("FINAL MLB PLAYABLE CARD")
print("=" * 70)

# ---------------------------------------------------------
# 0) validate scored board
# ---------------------------------------------------------
if "scored_mlb_board" not in globals():
    raise ValueError("Run Cell 13 first.")

df = scored_mlb_board.copy()

print("Start shape:", df.shape)

# ---------------------------------------------------------
# 1) HARD FILTERS (EKO STYLE)
# ---------------------------------------------------------
MIN_WIN_PROB = 0.55
MAX_WIN_PROB = 0.72
MAX_PLAYS = 3

df = df[df["projected_win_prob"].notna()].copy()

df = df[
    (df["projected_win_prob"] >= MIN_WIN_PROB) &
    (df["projected_win_prob"] <= MAX_WIN_PROB)
].copy()

print("After win prob filter:", df.shape)

# ---------------------------------------------------------
# 2) OPTIONAL STARTER FILTER (ONLY IF EXISTS)
# ---------------------------------------------------------
has_starters = (
    "away_starter_name" in df.columns and
    "home_starter_name" in df.columns
)

if has_starters:
    df = df[
        df["away_starter_name"].notna() &
        df["home_starter_name"].notna()
    ].copy()
    print("After starter filter:", df.shape)
else:
    print("No starter data Ã¢â‚¬â€ skipping starter filter")

# ---------------------------------------------------------
# 3) BUILD BET SIDE
# ---------------------------------------------------------
df["bet_team"] = df["projected_winner"]

df["matchup"] = df["away_team"] + " @ " + df["home_team"]

df["bet_label"] = df["bet_team"] + " ML"

# ---------------------------------------------------------
# 4) SORT BEST Ã¢â€ â€™ WORST
# ---------------------------------------------------------
df = df.sort_values("projected_win_prob", ascending=False).reset_index(drop=True)

# ---------------------------------------------------------
# 5) LIMIT PLAYS
# ---------------------------------------------------------
df = df.head(MAX_PLAYS).copy()

print("After MAX_PLAYS:", df.shape)

# ---------------------------------------------------------
# 6) BUILD CLEAN CARD OUTPUT
# ---------------------------------------------------------
base_cols = [
    "matchup",
    "bet_label",
    "projected_winner",
    "projected_win_prob",
    "strength_tier",
]

optional_cols = []

if has_starters:
    optional_cols += ["away_starter_name", "home_starter_name"]

card_cols = base_cols + optional_cols

final_mlb_card = df[card_cols].copy()

# ---------------------------------------------------------
# 7) DISPLAY CLEAN CARD
# ---------------------------------------------------------
print("\n" + "=" * 70)
print("EKO BETS Ã¢â‚¬â€ FINAL MLB CARD")
print("=" * 70)

if len(final_mlb_card) == 0:
    print("No bets passed filters.")
else:
    for i, row in final_mlb_card.iterrows():
        print(f"{i+1}. {row['bet_label']}")
        print(f"   {row['matchup']}")

        if has_starters:
            print(f"   Pitching: {row['away_starter_name']} vs {row['home_starter_name']}")

        print(f"   Win Prob: {row['projected_win_prob']:.3f}")
        print(f"   Tier: {row['strength_tier']}")
        print("-" * 50)

# ---------------------------------------------------------
# 8) POST READY CAPTION
# ---------------------------------------------------------
caption = "EKO BETS Ã¢â‚¬â€ MLB CARD\n\n"

if len(final_mlb_card) == 0:
    caption += "No playable bets today.\nSelective only.\n"
else:
    for i, row in final_mlb_card.iterrows():
        caption += (
            f"{i+1}. {row['bet_label']}\n"
            f"   {row['matchup']}\n"
            f"   Win Prob: {row['projected_win_prob']:.3f}\n\n"
        )

caption += "Selective only. No forced volume.\n#Ekobets #MLB"

print("\n" + "=" * 70)
print("POST READY CAPTION")
print("=" * 70)
print(caption)

# ---------------------------------------------------------
# 9) SAVE
# ---------------------------------------------------------
globals()["final_mlb_card"] = final_mlb_card.copy()
globals()["mlb_post_ready_caption"] = caption

print("\nSaved in memory:")
print(" - final_mlb_card")
print(" - mlb_post_ready_caption")
print("Done.")

FINAL MLB PLAYABLE CARD
Start shape: (1, 97)
After win prob filter: (0, 97)
After starter filter: (0, 97)
After MAX_PLAYS: (0, 100)

EKO BETS Ã¢â‚¬â€ FINAL MLB CARD
No bets passed filters.

POST READY CAPTION
EKO BETS Ã¢â‚¬â€ MLB CARD

No playable bets today.
Selective only.
Selective only. No forced volume.
#Ekobets #MLB

Saved in memory:
 - final_mlb_card
 - mlb_post_ready_caption
Done.


In [37]:
# %%
# =========================================================
# CELL 15 Ã¢â‚¬â€ PULL KALSHI MLB WINNER MARKETS
# Discover MLB series, flatten markets, keep winner markets
# =========================================================

import pandas as pd
import numpy as np
import requests

KALSHI_BASE = "https://api.elections.kalshi.com/trade-api/v2"

def kalshi_get(path, params=None, timeout=30):
    url = f"{KALSHI_BASE}{path}"
    r = requests.get(url, params=params, timeout=timeout, headers={"Accept-Encoding": "gzip, deflate"})
    r.raise_for_status()
    return r.json()

def get_all_events_for_series(series_ticker):
    out = []
    cursor = None

    while True:
        params = {
            "series_ticker": series_ticker,
            "status": "open",
            "with_nested_markets": True,
        }
        if cursor:
            params["cursor"] = cursor

        data = kalshi_get("/events", params=params)
        batch = data.get("events", []) if isinstance(data, dict) else []
        out.extend(batch)

        cursor = data.get("cursor") if isinstance(data, dict) else None
        if not cursor:
            break

    return out

print("Pulling Kalshi MLB winner markets...")

# ---------------------------------------------------------
# 1) discover open MLB series
# ---------------------------------------------------------
series_data = kalshi_get("/series", {"status": "open"})
series_df = pd.DataFrame(series_data.get("series", []))

if len(series_df) == 0:
    raise ValueError("No open Kalshi series returned.")

text_cols = [c for c in ["ticker", "title", "subtitle", "category"] if c in series_df.columns]
blob = series_df[text_cols].astype(str).agg(" | ".join, axis=1).str.lower()

mlb_series = series_df.loc[
    blob.str.contains(r"\bmlb\b|baseball|major league", regex=True, na=False)
].copy()

print("Total open series:", len(series_df))
print("MLB series found :", len(mlb_series))

if len(mlb_series):
    display(mlb_series[[c for c in ["ticker", "title", "subtitle"] if c in mlb_series.columns]].head(20))
else:
    raise ValueError("No MLB-like series matched. Check Kalshi naming.")

# ---------------------------------------------------------
# 2) pull all events from discovered MLB series
# ---------------------------------------------------------
all_events = []
series_tickers = mlb_series["ticker"].dropna().astype(str).unique().tolist()

for st in series_tickers:
    try:
        evs = get_all_events_for_series(st)
        print(f"{st}: {len(evs)} events")
        all_events.extend(evs)
    except Exception as e:
        print(f"Failed pulling {st}: {e}")

print("\nTotal MLB events pulled:", len(all_events))

if len(all_events) == 0:
    raise ValueError("No MLB events were pulled from Kalshi.")

# ---------------------------------------------------------
# 3) flatten markets
# ---------------------------------------------------------
rows = []
for ev in all_events:
    ev_title = ev.get("title")
    ev_subtitle = ev.get("sub_title")
    event_ticker = ev.get("event_ticker")
    series_ticker = ev.get("series_ticker")
    close_time = ev.get("close_time")
    expiration_time = ev.get("expiration_time")

    markets = ev.get("markets", [])
    if not isinstance(markets, list):
        continue

    for m in markets:
        rows.append({
            "series_ticker": series_ticker,
            "event_ticker": event_ticker,
            "event_title": ev_title,
            "event_subtitle": ev_subtitle,
            "close_time": close_time,
            "expiration_time": expiration_time,
            "market_ticker": m.get("ticker"),
            "market_title": m.get("title"),
            "subtitle": m.get("subtitle"),
            "yes_ask": m.get("yes_ask"),
            "no_ask": m.get("no_ask"),
            "yes_bid": m.get("yes_bid"),
            "no_bid": m.get("no_bid"),
            "last_price": m.get("last_price"),
            "status": m.get("status"),
        })

markets_df = pd.DataFrame(rows)

print("Flattened MLB markets shape:", markets_df.shape)

if len(markets_df) == 0:
    raise ValueError("No MLB markets found after flattening.")

# ---------------------------------------------------------
# 4) normalize prices
# ---------------------------------------------------------
price_cols = ["yes_ask", "no_ask", "yes_bid", "no_bid", "last_price"]
for c in price_cols:
    if c in markets_df.columns:
        markets_df[c] = pd.to_numeric(markets_df[c], errors="coerce")
        markets_df[c] = np.where(markets_df[c] > 1, markets_df[c] / 100, markets_df[c])

# ---------------------------------------------------------
# 5) keep winner-like markets only
# ---------------------------------------------------------
winner_mask = pd.Series(False, index=markets_df.index)

if "series_ticker" in markets_df.columns:
    winner_mask |= markets_df["series_ticker"].astype(str).str.contains("GAME|WINNER", case=False, na=False)

for c in ["market_title", "event_title", "subtitle"]:
    if c in markets_df.columns:
        winner_mask |= markets_df[c].astype(str).str.contains(r"winner|moneyline|to win", case=False, na=False)

winner_markets = markets_df.loc[winner_mask].copy()

print("Winner-like MLB markets shape:", winner_markets.shape)

preview_cols = [c for c in [
    "series_ticker", "event_title", "market_title", "subtitle",
    "yes_ask", "no_ask", "market_ticker"
] if c in winner_markets.columns]

display(winner_markets[preview_cols].head(25))

mlb_kalshi_winner_markets_raw = winner_markets.copy()

Pulling Kalshi MLB winner markets...


Total open series: 14934
MLB series found : 185


,ticker,title
7,KXMLBNLHAARON,Pro Baseball American League Hank Aaron
43,KXMLBHR,Pro Baseball Home Runs
72,KXLEADERMLBKS,Pro Baseball Strikeouts Leader
95,KXMLBMATCHUP,Pro Baseball Playoff Matchups
208,KXWBCHR,World Baseball Classic Homerun
701,KXWBCGAME,World Baseball Classic Game
790,KXMLBCHAMPDIVISION,Pro Baseball Champion Division
849,KXMLBHRDERBYFIN,Pro Baseball Home Run Derby Finals Qualifiers
872,KXMLBRBI,Pro Baseball RBIs
911,KXMLBWINS-SD,Pro baseball wins San Diego


KXMLBNLHAARON: 1 events
KXMLBHR: 2 events


KXLEADERMLBKS: 0 events
KXMLBMATCHUP: 1 events


...

KXMLBDIVWINNER: 0 events
KXMLBSTARTER: 0 events


Failed pulling KXMLBALLSTARHR: 429 Client Error: Too Many Requests for url: https://api.elections.kalshi.com/trade-api/v2/events?series_ticker=KXMLBALLSTARHR&status=open&with_nested_markets=True
Failed pulling KXMLBDELAY: 429 Client Error: Too Many Requests for url: https://api.elections.kalshi.com/trade-api/v2/events?series_ticker=KXMLBDELAY&status=open&with_nested_markets=True


KXMLBWINS-TOR: 0 events
Failed pulling KXLEADERMLBAVG: 429 Client Error: Too Many Requests for url: https://api.elections.kalshi.com/trade-api/v2/events?series_ticker=KXLEADERMLBAVG&status=open&with_nested_markets=True
KXLEADERMLBWAR: 0 events


Failed pulling KXMLBERA: 429 Client Error: Too Many Requests for url: https://api.elections.kalshi.com/trade-api/v2/events?series_ticker=KXMLBERA&status=open&with_nested_markets=True
KXMLBTRIPLECROWN: 0 events

Total MLB events pulled: 92
Flattened MLB markets shape: (1235, 15)
Winner-like MLB markets shape: (203, 15)


,series_ticker,event_title,market_title,subtitle,yes_ask,no_ask,market_ticker
0,KXMLBNLHAARON,NL Hank Aaron Award Winner?,Will Shohei Ohtani win NL Hank Aaron Award?,:: Los Angeles D,NaN,NaN,KXMLBNLHAARON-26-SOHT
1,KXMLBNLHAARON,NL Hank Aaron Award Winner?,Will Juan Soto win NL Hank Aaron Award?,:: New York M,NaN,NaN,KXMLBNLHAARON-26-JSOT
2,KXMLBNLHAARON,NL Hank Aaron Award Winner?,Will Ronald Acuña Jr. win NL Hank Aaron Award?,:: Atlanta,NaN,NaN,KXMLBNLHAARON-26-RACU
3,KXMLBNLHAARON,NL Hank Aaron Award Winner?,Will Fernando Tatis Jr. win NL Hank Aaron Award?,:: San Diego,NaN,NaN,KXMLBNLHAARON-26-FTAT
4,KXMLBNLHAARON,NL Hank Aaron Award Winner?,Will Francisco Lindor win NL Hank Aaron Award?,:: New York M,NaN,NaN,KXMLBNLHAARON-26-FLIN
5,KXMLBNLHAARON,NL Hank Aaron Award Winner?,Will Corbin Carroll win NL Hank Aaron Award?,:: Arizona,NaN,NaN,KXMLBNLHAARON-26-CCAR
6,KXMLBNLHAARON,NL Hank Aaron Award Winner?,Will Elly De La Cruz win NL Hank Aaron Award?,:: Cincinnati,NaN,NaN,KXMLBNLHAARON-26-ECRU
7,KXMLBNLHAARON,NL Hank Aaron Award Winner?,Will Bryce Harper win NL Hank Aaron Award?,:: Philadelphia,NaN,NaN,KXMLBNLHAARON-26-BHAR
8,KXMLBNLHAARON,NL Hank Aaron Award Winner?,Will Kyle Schwarber win NL Hank Aaron Award?,:: Philadelphia,NaN,NaN,KXMLBNLHAARON-26-KSCH
9,KXMLBNLHAARON,NL Hank Aaron Award Winner?,Will Rafael Devers win NL Hank Aaron Award?,:: San Francisco,NaN,NaN,KXMLBNLHAARON-26-RDEV


In [38]:
# %%
# =========================================================
# CELL 16 (FULL REPLACEMENT) Ã¢â‚¬â€ PULL KALSHI KXMLBGAME DIRECTLY
# Robust version with 429 retry/backoff + pagination safety
# =========================================================

import pandas as pd
import numpy as np
import requests
import time
import random

KALSHI_BASE = "https://api.elections.kalshi.com/trade-api/v2"

# ---------------------------------------------------------
# robust GET with retry/backoff
# ---------------------------------------------------------
def kalshi_get(path, params=None, timeout=30, max_retries=8, base_sleep=2.0):
    params = params or {}
    url = f"{KALSHI_BASE}{path}"
    last_err = None

    for attempt in range(1, max_retries + 1):
        try:
            r = requests.get(url, params=params, timeout=timeout, headers={"Accept-Encoding": "gzip, deflate"})

            if r.status_code == 200:
                return r.json()

            if r.status_code == 429:
                retry_after = r.headers.get("Retry-After")
                if retry_after is not None:
                    try:
                        sleep_s = float(retry_after)
                    except Exception:
                        sleep_s = base_sleep * (2 ** (attempt - 1))
                else:
                    sleep_s = base_sleep * (2 ** (attempt - 1))

                sleep_s += random.uniform(0.25, 0.75)
                print(f"Ã¢Å¡Â Ã¯Â¸Â Kalshi 429 rate limit hit (attempt {attempt}/{max_retries}) | sleeping {sleep_s:.1f}s")
                time.sleep(sleep_s)
                continue

            if r.status_code in [500, 502, 503, 504]:
                sleep_s = base_sleep * (2 ** (attempt - 1)) + random.uniform(0.25, 0.75)
                print(f"Ã¢Å¡Â Ã¯Â¸Â Kalshi server error {r.status_code} (attempt {attempt}/{max_retries}) | sleeping {sleep_s:.1f}s")
                time.sleep(sleep_s)
                continue

            # show a little response text before failing
            try:
                msg = r.text[:300]
                print(f"Ã¢ÂÅ’ Non-retryable Kalshi error {r.status_code}: {msg}")
            except Exception:
                pass

            r.raise_for_status()

        except requests.RequestException as e:
            last_err = e
            sleep_s = base_sleep * (2 ** (attempt - 1)) + random.uniform(0.25, 0.75)
            print(f"Ã¢Å¡Â Ã¯Â¸Â Request exception (attempt {attempt}/{max_retries}) | {e} | sleeping {sleep_s:.1f}s")
            time.sleep(sleep_s)

    raise ValueError(f"Kalshi request failed after {max_retries} retries. Last error: {last_err}")

# ---------------------------------------------------------
# paginated series pull
# ---------------------------------------------------------
def get_all_events_for_series(series_ticker, status="open", with_nested_markets=True, limit=200):
    out = []
    cursor = None
    seen_cursors = set()
    page = 0

    while True:
        page += 1

        params = {
            "series_ticker": series_ticker,
            "status": status,
            "with_nested_markets": with_nested_markets,
            "limit": limit,
        }
        if cursor:
            params["cursor"] = cursor

        data = kalshi_get("/events", params=params)

        batch = data.get("events", []) if isinstance(data, dict) else []
        out.extend(batch)

        next_cursor = None
        if isinstance(data, dict):
            next_cursor = data.get("cursor") or data.get("next_cursor")

        print(f"Page {page}: pulled {len(batch)} events | total so far {len(out)}")

        if not next_cursor:
            break

        if next_cursor in seen_cursors:
            print("Ã¢Å¡Â Ã¯Â¸Â Repeated cursor detected. Stopping pagination to avoid loop.")
            break

        seen_cursors.add(next_cursor)
        cursor = next_cursor

        # small delay between pages to reduce 429 risk
        time.sleep(0.5)

    return out

print("Pulling KXMLBGAME directly...")

events = get_all_events_for_series("KXMLBGAME")
print("Events pulled:", len(events))

if len(events) == 0:
    raise ValueError("No events returned for KXMLBGAME.")

rows = []
for ev in events:
    ev_title = ev.get("title")
    ev_subtitle = ev.get("sub_title")
    event_ticker = ev.get("event_ticker")
    series_ticker = ev.get("series_ticker")
    close_time = ev.get("close_time")
    expiration_time = ev.get("expiration_time")

    markets = ev.get("markets", [])
    if not isinstance(markets, list):
        continue

    for m in markets:
        rows.append({
            "series_ticker": series_ticker,
            "event_ticker": event_ticker,
            "event_title": ev_title,
            "event_subtitle": ev_subtitle,
            "close_time": close_time,
            "expiration_time": expiration_time,
            "market_ticker": m.get("ticker"),
            "market_title": m.get("title"),
            "subtitle": m.get("subtitle"),
            "yes_ask": m.get("yes_ask"),
            "no_ask": m.get("no_ask"),
            "yes_bid": m.get("yes_bid"),
            "no_bid": m.get("no_bid"),
            "last_price": m.get("last_price"),
            "status": m.get("status"),
        })

kalshi_mlb_game_markets = pd.DataFrame(rows)

if len(kalshi_mlb_game_markets) == 0:
    raise ValueError("No markets found inside KXMLBGAME events.")

# ---------------------------------------------------------
# numeric cleanup
# Kalshi sometimes returns cents-style integers
# ---------------------------------------------------------
for c in ["yes_ask", "no_ask", "yes_bid", "no_bid", "last_price"]:
    if c in kalshi_mlb_game_markets.columns:
        kalshi_mlb_game_markets[c] = pd.to_numeric(kalshi_mlb_game_markets[c], errors="coerce")
        kalshi_mlb_game_markets[c] = np.where(
            kalshi_mlb_game_markets[c] > 1,
            kalshi_mlb_game_markets[c] / 100.0,
            kalshi_mlb_game_markets[c]
        )

# ---------------------------------------------------------
# optional parsed datetimes
# ---------------------------------------------------------
for c in ["close_time", "expiration_time"]:
    if c in kalshi_mlb_game_markets.columns:
        kalshi_mlb_game_markets[c] = pd.to_datetime(kalshi_mlb_game_markets[c], errors="coerce", utc=True)

print("KXMLBGAME market rows:", len(kalshi_mlb_game_markets))

preview_cols = [c for c in [
    "series_ticker",
    "event_title",
    "market_title",
    "subtitle",
    "close_time",
    "expiration_time",
    "market_ticker",
    "yes_ask",
    "no_ask",
    "yes_bid",
    "no_bid",
    "last_price",
    "status",
] if c in kalshi_mlb_game_markets.columns]

display(
    kalshi_mlb_game_markets[preview_cols]
    .sort_values(
        [c for c in ["event_title", "market_title", "market_ticker"] if c in kalshi_mlb_game_markets.columns],
        ascending=True
    )
    .reset_index(drop=True)
)

mlb_kalshi_game_markets = kalshi_mlb_game_markets.copy()
print("Saved as: mlb_kalshi_game_markets")

Pulling KXMLBGAME directly...
Ã¢Å¡Â Ã¯Â¸Â Kalshi 429 rate limit hit (attempt 1/8) | sleeping 2.7s


Page 1: pulled 5 events | total so far 5
Events pulled: 5
KXMLBGAME market rows: 10


,series_ticker,event_title,market_title,subtitle,close_time,expiration_time,market_ticker,yes_ask,no_ask,yes_bid,no_bid,last_price,status
0,KXMLBGAME,Game 1: Los Angeles D vs Milwaukee,Los Angeles D wins,None,NaT,NaT,KXMLBGAME-26OCT112000LADMIL-LAD,NaN,NaN,NaN,NaN,NaN,active
1,KXMLBGAME,Game 1: Los Angeles D vs Milwaukee,Milwaukee wins,None,NaT,NaT,KXMLBGAME-26OCT112000LADMIL-MIL,NaN,NaN,NaN,NaN,NaN,active
2,KXMLBGAME,Game 2: Los Angeles D vs Milwaukee,Los Angeles D wins,None,NaT,NaT,KXMLBGAME-26OCT122000LADMIL-LAD,NaN,NaN,NaN,NaN,NaN,active
3,KXMLBGAME,Game 2: Los Angeles D vs Milwaukee,Milwaukee wins,None,NaT,NaT,KXMLBGAME-26OCT122000LADMIL-MIL,NaN,NaN,NaN,NaN,NaN,active
4,KXMLBGAME,Game 3: Milwaukee vs Los Angeles D,Los Angeles D wins,None,NaT,NaT,KXMLBGAME-26OCT142000MILLAD-LAD,NaN,NaN,NaN,NaN,NaN,active
5,KXMLBGAME,Game 3: Milwaukee vs Los Angeles D,Milwaukee wins,None,NaT,NaT,KXMLBGAME-26OCT142000MILLAD-MIL,NaN,NaN,NaN,NaN,NaN,active
6,KXMLBGAME,Game 4: Milwaukee vs Los Angeles D,Los Angeles D wins,None,NaT,NaT,KXMLBGAME-26OCT152000MILLAD-LAD,NaN,NaN,NaN,NaN,NaN,active
7,KXMLBGAME,Game 4: Milwaukee vs Los Angeles D,Milwaukee wins,None,NaT,NaT,KXMLBGAME-26OCT152000MILLAD-MIL,NaN,NaN,NaN,NaN,NaN,active
8,KXMLBGAME,Game 5: Chicago WS vs Cleveland,Chicago WS wins,None,NaT,NaT,KXMLBGAME-26OCT102000CWSCLE-CWS,NaN,NaN,NaN,NaN,NaN,active
9,KXMLBGAME,Game 5: Chicago WS vs Cleveland,Cleveland wins,None,NaT,NaT,KXMLBGAME-26OCT102000CWSCLE-CLE,NaN,NaN,NaN,NaN,NaN,active


Saved as: mlb_kalshi_game_markets


In [39]:
# %%
# =========================================================
# CELL 17 Ã¢â‚¬â€ PARSE KALSHI GAME MARKETS INTO TEAM MATCHUPS
# Extract team1/team2 from event_title and selected team
# from market_ticker suffix
# =========================================================

import pandas as pd
import numpy as np
import re

if "mlb_kalshi_game_markets" not in globals() or not isinstance(mlb_kalshi_game_markets, pd.DataFrame):
    raise ValueError("mlb_kalshi_game_markets not found. Run Cell 16 replacement first.")

kg = mlb_kalshi_game_markets.copy()

# ---------------------------------------------------------
# parse matchup from event title
# examples:
#   "A's vs New York Y"
#   "Arizona vs New York M"
# ---------------------------------------------------------
def parse_event_matchup(text):
    if pd.isna(text):
        return (None, None)
    s = str(text).strip()
    s = re.sub(r"\s+", " ", s)

    m = re.match(r"^(.*?)\s+vs\s+(.*?)$", s, flags=re.I)
    if m:
        t1 = m.group(1).strip()
        t2 = m.group(2).strip()
        return (t1, t2)

    return (None, None)

parsed = kg["event_title"].apply(parse_event_matchup)
kg["team1_raw"] = parsed.apply(lambda x: x[0])
kg["team2_raw"] = parsed.apply(lambda x: x[1])

# ---------------------------------------------------------
# parse selected side from ticker suffix
# example:
#   KXMLBGAME-26APR081905ATHNYY-ATH -> ATH
# ---------------------------------------------------------
def parse_ticker_side(ticker):
    if pd.isna(ticker):
        return None
    s = str(ticker)
    parts = s.split("-")
    if len(parts) >= 3:
        return parts[-1].strip().upper()
    return None

kg["ticker_side_code"] = kg["market_ticker"].apply(parse_ticker_side)

# ---------------------------------------------------------
# map short event-title team labels to full MLB names
# ---------------------------------------------------------
TITLE_TEAM_MAP = {
    "A's": "Athletics",
    "Arizona": "Arizona Diamondbacks",
    "Atlanta": "Atlanta Braves",
    "Baltimore": "Baltimore Orioles",
    "Boston": "Boston Red Sox",
    "Chicago C": "Chicago Cubs",
    "Chicago WS": "Chicago White Sox",
    "Cincinnati": "Cincinnati Reds",
    "Cleveland": "Cleveland Guardians",
    "Colorado": "Colorado Rockies",
    "Detroit": "Detroit Tigers",
    "Houston": "Houston Astros",
    "Kansas City": "Kansas City Royals",
    "Los Angeles A": "Los Angeles Angels",
    "Los Angeles D": "Los Angeles Dodgers",
    "Miami": "Miami Marlins",
    "Milwaukee": "Milwaukee Brewers",
    "Minnesota": "Minnesota Twins",
    "New York M": "New York Mets",
    "New York Y": "New York Yankees",
    "Philadelphia": "Philadelphia Phillies",
    "Pittsburgh": "Pittsburgh Pirates",
    "San Diego": "San Diego Padres",
    "San Francisco": "San Francisco Giants",
    "Seattle": "Seattle Mariners",
    "St. Louis": "St. Louis Cardinals",
    "Tampa Bay": "Tampa Bay Rays",
    "Texas": "Texas Rangers",
    "Toronto": "Toronto Blue Jays",
    "Washington": "Washington Nationals",
}

kg["team1"] = kg["team1_raw"].map(TITLE_TEAM_MAP).fillna(kg["team1_raw"])
kg["team2"] = kg["team2_raw"].map(TITLE_TEAM_MAP).fillna(kg["team2_raw"])

# ---------------------------------------------------------
# map ticker side codes to full MLB names
# ---------------------------------------------------------
CODE_TEAM_MAP = {
    "ATH": "Athletics",
    "AZ": "Arizona Diamondbacks",
    "ATL": "Atlanta Braves",
    "BAL": "Baltimore Orioles",
    "BOS": "Boston Red Sox",
    "CHC": "Chicago Cubs",
    "CWS": "Chicago White Sox",
    "CIN": "Cincinnati Reds",
    "CLE": "Cleveland Guardians",
    "COL": "Colorado Rockies",
    "DET": "Detroit Tigers",
    "HOU": "Houston Astros",
    "KC": "Kansas City Royals",
    "LAA": "Los Angeles Angels",
    "LAD": "Los Angeles Dodgers",
    "MIA": "Miami Marlins",
    "MIL": "Milwaukee Brewers",
    "MIN": "Minnesota Twins",
    "NYM": "New York Mets",
    "NYY": "New York Yankees",
    "PHI": "Philadelphia Phillies",
    "PIT": "Pittsburgh Pirates",
    "SD": "San Diego Padres",
    "SF": "San Francisco Giants",
    "SEA": "Seattle Mariners",
    "STL": "St. Louis Cardinals",
    "TB": "Tampa Bay Rays",
    "TEX": "Texas Rangers",
    "TOR": "Toronto Blue Jays",
    "WSH": "Washington Nationals",
}

kg["selected_team"] = kg["ticker_side_code"].map(CODE_TEAM_MAP)

print("Parsed Kalshi rows:", len(kg))
print("Unique event titles:", kg["event_title"].nunique())

print("\nMissing team1       :", int(kg["team1"].isna().sum()))
print("Missing team2       :", int(kg["team2"].isna().sum()))
print("Missing selected_team:", int(kg["selected_team"].isna().sum()))

preview_cols = [
    "event_title",
    "market_ticker",
    "team1_raw",
    "team2_raw",
    "team1",
    "team2",
    "ticker_side_code",
    "selected_team",
]

print("\nParsed preview:")
display(kg[preview_cols].head(30))

mlb_kalshi_game_markets_parsed = kg.copy()

Parsed Kalshi rows: 10
Unique event titles: 5

Missing team1       : 0
Missing team2       : 0
Missing selected_team: 0

Parsed preview:


,event_title,market_ticker,team1_raw,team2_raw,team1,team2,ticker_side_code,selected_team
0,Game 5: Chicago WS vs Cleveland,KXMLBGAME-26OCT102000CWSCLE-CWS,Game 5: Chicago WS,Cleveland,Game 5: Chicago WS,Cleveland Guardians,CWS,Chicago White Sox
1,Game 5: Chicago WS vs Cleveland,KXMLBGAME-26OCT102000CWSCLE-CLE,Game 5: Chicago WS,Cleveland,Game 5: Chicago WS,Cleveland Guardians,CLE,Cleveland Guardians
2,Game 1: Los Angeles D vs Milwaukee,KXMLBGAME-26OCT112000LADMIL-LAD,Game 1: Los Angeles D,Milwaukee,Game 1: Los Angeles D,Milwaukee Brewers,LAD,Los Angeles Dodgers
3,Game 1: Los Angeles D vs Milwaukee,KXMLBGAME-26OCT112000LADMIL-MIL,Game 1: Los Angeles D,Milwaukee,Game 1: Los Angeles D,Milwaukee Brewers,MIL,Milwaukee Brewers
4,Game 2: Los Angeles D vs Milwaukee,KXMLBGAME-26OCT122000LADMIL-LAD,Game 2: Los Angeles D,Milwaukee,Game 2: Los Angeles D,Milwaukee Brewers,LAD,Los Angeles Dodgers
5,Game 2: Los Angeles D vs Milwaukee,KXMLBGAME-26OCT122000LADMIL-MIL,Game 2: Los Angeles D,Milwaukee,Game 2: Los Angeles D,Milwaukee Brewers,MIL,Milwaukee Brewers
6,Game 3: Milwaukee vs Los Angeles D,KXMLBGAME-26OCT142000MILLAD-MIL,Game 3: Milwaukee,Los Angeles D,Game 3: Milwaukee,Los Angeles Dodgers,MIL,Milwaukee Brewers
7,Game 3: Milwaukee vs Los Angeles D,KXMLBGAME-26OCT142000MILLAD-LAD,Game 3: Milwaukee,Los Angeles D,Game 3: Milwaukee,Los Angeles Dodgers,LAD,Los Angeles Dodgers
8,Game 4: Milwaukee vs Los Angeles D,KXMLBGAME-26OCT152000MILLAD-MIL,Game 4: Milwaukee,Los Angeles D,Game 4: Milwaukee,Los Angeles Dodgers,MIL,Milwaukee Brewers
9,Game 4: Milwaukee vs Los Angeles D,KXMLBGAME-26OCT152000MILLAD-LAD,Game 4: Milwaukee,Los Angeles D,Game 4: Milwaukee,Los Angeles Dodgers,LAD,Los Angeles Dodgers


In [40]:
# %%
# =========================================================
# CELL 18 Ã¢â‚¬â€ MATCH KALSHI MARKETS TO TODAY'S MLB SLATE
# Keep only markets that belong to today's actual games
# =========================================================

import pandas as pd
import numpy as np

if "mlb_kalshi_game_markets_parsed" not in globals() or not isinstance(mlb_kalshi_game_markets_parsed, pd.DataFrame):
    raise ValueError("mlb_kalshi_game_markets_parsed not found. Run Cell 17 first.")

if "mlb_today_games" not in globals() or not isinstance(mlb_today_games, pd.DataFrame):
    raise ValueError("mlb_today_games not found. Run Cell 11 first.")

km = mlb_kalshi_game_markets_parsed.copy()
today = mlb_today_games.copy()

# ---------------------------------------------------------
# normalize team names
# ---------------------------------------------------------
TEAM_NAME_FIXES = {
    "Athletics": "Oakland Athletics",
}

def clean_team(s):
    if pd.isna(s):
        return None
    s = str(s).strip()
    s = TEAM_NAME_FIXES.get(s, s)
    return s.lower()

km["team1_key"] = km["team1"].apply(clean_team)
km["team2_key"] = km["team2"].apply(clean_team)
km["selected_team_key"] = km["selected_team"].apply(clean_team)

today["away_key"] = today["away_team"].apply(clean_team)
today["home_key"] = today["home_team"].apply(clean_team)

# order-independent matchup key
km["matchup_key"] = [
    " | ".join(sorted([a, b])) if pd.notna(a) and pd.notna(b) else None
    for a, b in zip(km["team1_key"], km["team2_key"])
]

today["matchup_key"] = [
    " | ".join(sorted([a, b])) if pd.notna(a) and pd.notna(b) else None
    for a, b in zip(today["away_key"], today["home_key"])
]

today_matchups = set(today["matchup_key"].dropna().unique())

km_today = km[km["matchup_key"].isin(today_matchups)].copy()

print("Kalshi rows before today's filter:", len(km))
print("Kalshi rows after today's filter :", len(km_today))
print("Today's unique matchup keys      :", len(today_matchups))

# ---------------------------------------------------------
# merge onto today's slate
# ---------------------------------------------------------
matched = km_today.merge(
    today[["game_id", "away_team", "home_team", "game_datetime_utc", "venue_name", "status", "away_key", "home_key", "matchup_key"]].drop_duplicates(),
    on="matchup_key",
    how="inner",
    suffixes=("_kalshi", "_mlb")
).copy()

print("\nMatched rows:", len(matched))
print("Matched unique games:", matched["matchup_key"].nunique())

preview_cols = [c for c in [
    "away_team",
    "home_team",
    "selected_team",
    "event_title",
    "market_ticker",
    "yes_ask",
    "no_ask",
    "last_price",
    "matchup_key",
] if c in matched.columns]

print("\nMatched preview:")
display(matched[preview_cols].head(30))

mlb_kalshi_matched = matched.copy()

Kalshi rows before today's filter: 10
Kalshi rows after today's filter : 0
Today's unique matchup keys      : 1

Matched rows: 0
Matched unique games: 0

Matched preview:


,away_team,home_team,selected_team,event_title,market_ticker,yes_ask,no_ask,last_price,matchup_key


In [41]:
# %%
# =========================================================
# CELL 19 Ã¢â‚¬â€ DIAGNOSE KALSHI GAME DATES FROM TICKER
# Extract date/time from market_ticker so we can filter
# to the actual slate date instead of just matchup names
# =========================================================

import pandas as pd
import numpy as np
import re

if "mlb_kalshi_game_markets_parsed" not in globals() or not isinstance(mlb_kalshi_game_markets_parsed, pd.DataFrame):
    raise ValueError("mlb_kalshi_game_markets_parsed not found. Run Cell 17 first.")

kg = mlb_kalshi_game_markets_parsed.copy()

# ---------------------------------------------------------
# Example ticker:
# KXMLBGAME-26APR081905ATHNYY-ATH
# date/time chunk is the middle section: 26APR081905ATHNYY
# leading 9 chars are date-like: 26APR08
# next 4 chars are time-like: 1905
# ---------------------------------------------------------
def extract_ticker_mid(ticker):
    if pd.isna(ticker):
        return None
    parts = str(ticker).split("-")
    if len(parts) >= 3:
        return parts[1]
    return None

def extract_kalshi_date_str(mid):
    if pd.isna(mid):
        return None
    s = str(mid)
    m = re.match(r"^(\d{2}[A-Z]{3}\d{2})", s)
    if m:
        return m.group(1)
    return None

def extract_kalshi_time_str(mid):
    if pd.isna(mid):
        return None
    s = str(mid)
    m = re.match(r"^\d{2}[A-Z]{3}\d{2}(\d{4})", s)
    if m:
        return m.group(1)
    return None

kg["ticker_mid"] = kg["market_ticker"].apply(extract_ticker_mid)
kg["kalshi_date_str"] = kg["ticker_mid"].apply(extract_kalshi_date_str)
kg["kalshi_time_str"] = kg["ticker_mid"].apply(extract_kalshi_time_str)

# parse date like 26APR08 -> 2026-04-08
kg["kalshi_game_date"] = pd.to_datetime(
    kg["kalshi_date_str"],
    format="%y%b%d",
    errors="coerce"
)

print("Unique Kalshi game dates:")
print(kg["kalshi_game_date"].dropna().dt.date.value_counts().sort_index())

print("\nToday's MLB schedule date(s):")
if "mlb_today_games" in globals() and isinstance(mlb_today_games, pd.DataFrame) and len(mlb_today_games):
    print(pd.to_datetime(mlb_today_games["game_datetime_utc"], utc=True).dt.date.value_counts().sort_index())
else:
    print("mlb_today_games not available")

print("\nKalshi preview with parsed dates:")
display_cols = [
    "event_title",
    "market_ticker",
    "kalshi_date_str",
    "kalshi_time_str",
    "kalshi_game_date",
    "team1",
    "team2",
    "selected_team",
]
display(kg[display_cols].drop_duplicates().sort_values(["kalshi_game_date", "event_title", "market_ticker"]).head(40))

mlb_kalshi_game_markets_dated = kg.copy()

Unique Kalshi game dates:
kalshi_game_date
2026-10-10    2
2026-10-11    2
2026-10-12    2
2026-10-14    2
2026-10-15    2
Name: count, dtype: int64

Today's MLB schedule date(s):
game_datetime_utc
2026-10-11    1
Name: count, dtype: int64

Kalshi preview with parsed dates:


,event_title,market_ticker,kalshi_date_str,kalshi_time_str,kalshi_game_date,team1,team2,selected_team
1,Game 5: Chicago WS vs Cleveland,KXMLBGAME-26OCT102000CWSCLE-CLE,26OCT10,2000,2026-10-10,Game 5: Chicago WS,Cleveland Guardians,Cleveland Guardians
0,Game 5: Chicago WS vs Cleveland,KXMLBGAME-26OCT102000CWSCLE-CWS,26OCT10,2000,2026-10-10,Game 5: Chicago WS,Cleveland Guardians,Chicago White Sox
2,Game 1: Los Angeles D vs Milwaukee,KXMLBGAME-26OCT112000LADMIL-LAD,26OCT11,2000,2026-10-11,Game 1: Los Angeles D,Milwaukee Brewers,Los Angeles Dodgers
3,Game 1: Los Angeles D vs Milwaukee,KXMLBGAME-26OCT112000LADMIL-MIL,26OCT11,2000,2026-10-11,Game 1: Los Angeles D,Milwaukee Brewers,Milwaukee Brewers
4,Game 2: Los Angeles D vs Milwaukee,KXMLBGAME-26OCT122000LADMIL-LAD,26OCT12,2000,2026-10-12,Game 2: Los Angeles D,Milwaukee Brewers,Los Angeles Dodgers
5,Game 2: Los Angeles D vs Milwaukee,KXMLBGAME-26OCT122000LADMIL-MIL,26OCT12,2000,2026-10-12,Game 2: Los Angeles D,Milwaukee Brewers,Milwaukee Brewers
7,Game 3: Milwaukee vs Los Angeles D,KXMLBGAME-26OCT142000MILLAD-LAD,26OCT14,2000,2026-10-14,Game 3: Milwaukee,Los Angeles Dodgers,Los Angeles Dodgers
6,Game 3: Milwaukee vs Los Angeles D,KXMLBGAME-26OCT142000MILLAD-MIL,26OCT14,2000,2026-10-14,Game 3: Milwaukee,Los Angeles Dodgers,Milwaukee Brewers
9,Game 4: Milwaukee vs Los Angeles D,KXMLBGAME-26OCT152000MILLAD-LAD,26OCT15,2000,2026-10-15,Game 4: Milwaukee,Los Angeles Dodgers,Los Angeles Dodgers
8,Game 4: Milwaukee vs Los Angeles D,KXMLBGAME-26OCT152000MILLAD-MIL,26OCT15,2000,2026-10-15,Game 4: Milwaukee,Los Angeles Dodgers,Milwaukee Brewers


In [42]:
# %%
# =========================================================
# CELL 20 Ã¢â‚¬â€ FILTER KALSHI TO TODAY'S DATE, THEN MATCH SLATE
# Use ticker-derived game date before merging to MLB slate
# =========================================================

import pandas as pd
import numpy as np

if "mlb_kalshi_game_markets_dated" not in globals() or not isinstance(mlb_kalshi_game_markets_dated, pd.DataFrame):
    raise ValueError("mlb_kalshi_game_markets_dated not found. Run Cell 19 first.")

if "mlb_today_games" not in globals() or not isinstance(mlb_today_games, pd.DataFrame):
    raise ValueError("mlb_today_games not found. Run Cell 11 first.")

kg = mlb_kalshi_game_markets_dated.copy()
today = mlb_today_games.copy()

TEAM_NAME_FIXES = {
    "Athletics": "Oakland Athletics",
}

def clean_team(s):
    if pd.isna(s):
        return None
    s = str(s).strip()
    s = TEAM_NAME_FIXES.get(s, s)
    return s.lower()

# ---------------------------------------------------------
# build keys on Kalshi side
# ---------------------------------------------------------
kg["team1_key"] = kg["team1"].apply(clean_team)
kg["team2_key"] = kg["team2"].apply(clean_team)
kg["selected_team_key"] = kg["selected_team"].apply(clean_team)

kg["matchup_key"] = [
    " | ".join(sorted([a, b])) if pd.notna(a) and pd.notna(b) else None
    for a, b in zip(kg["team1_key"], kg["team2_key"])
]

# ---------------------------------------------------------
# build keys on MLB side
# ---------------------------------------------------------
today["away_key"] = today["away_team"].apply(clean_team)
today["home_key"] = today["home_team"].apply(clean_team)
today["matchup_key"] = [
    " | ".join(sorted([a, b])) if pd.notna(a) and pd.notna(b) else None
    for a, b in zip(today["away_key"], today["home_key"])
]

# ---------------------------------------------------------
# keep only Kalshi rows for today's date
# ---------------------------------------------------------
today_date = pd.to_datetime(today["game_datetime_utc"], utc=True).dt.date.min()
kg["kalshi_game_date_only"] = pd.to_datetime(kg["kalshi_game_date"], errors="coerce").dt.date

kg_today = kg[kg["kalshi_game_date_only"] == today_date].copy()

print("Today's MLB date:", today_date)
print("Kalshi rows before date filter:", len(kg))
print("Kalshi rows after date filter :", len(kg_today))

today_matchups = set(today["matchup_key"].dropna().unique())
kg_today = kg_today[kg_today["matchup_key"].isin(today_matchups)].copy()

print("Kalshi rows after matchup filter:", len(kg_today))
print("Today's unique matchup keys     :", len(today_matchups))

matched = kg_today.merge(
    today[
        [
            "game_id", "away_team", "home_team", "game_datetime_utc",
            "venue_name", "status", "away_key", "home_key", "matchup_key"
        ]
    ].drop_duplicates(),
    on="matchup_key",
    how="inner",
    suffixes=("_kalshi", "_mlb")
).copy()

print("\nMatched rows:", len(matched))
print("Matched unique games:", matched["matchup_key"].nunique())

preview_cols = [c for c in [
    "away_team",
    "home_team",
    "selected_team",
    "event_title",
    "market_ticker",
    "kalshi_game_date_only",
    "yes_ask",
    "no_ask",
    "last_price",
] if c in matched.columns]

print("\nMatched preview:")
display(matched[preview_cols].head(20))

mlb_kalshi_matched = matched.copy()

Today's MLB date: 2026-10-11
Kalshi rows before date filter: 10
Kalshi rows after date filter : 2
Kalshi rows after matchup filter: 0
Today's unique matchup keys     : 1

Matched rows: 0
Matched unique games: 0

Matched preview:


,away_team,home_team,selected_team,event_title,market_ticker,kalshi_game_date_only,yes_ask,no_ask,last_price


In [43]:
# =========================================================
# CELL Ã¢â‚¬â€ PULL TODAY'S MLB MONEYLINE ODDS FROM ESPN
# Tries scoreboard odds first, falls back to per-event API.
# Stores mlb_espn_odds with has_odds flag.
# =========================================================
import requests
import pandas as pd
import numpy as np
from datetime import datetime

print("=" * 70)
print("PULL TODAY'S MLB MONEYLINE ODDS FROM ESPN")
print("=" * 70)

def _american_to_implied(ml):
    try:
        o = float(ml)
        return 100 / (o + 100) if o > 0 else abs(o) / (abs(o) + 100)
    except Exception:
        return np.nan

def _remove_vig(h, a):
    total = h + a
    if total <= 0 or (isinstance(total, float) and total != total):
        return np.nan, np.nan
    return h / total, a / total

def _norm_team(s):
    """Canonical key for team name matching Ã¢â‚¬â€ avoids last-word collisions."""
    if not s:
        return ""
    s = str(s).strip().lower()
    _map = {
        "chicago white sox": "whitesox",
        "boston red sox": "redsox",
        "new york yankees": "yankees",
        "new york mets": "mets",
        "los angeles dodgers": "dodgers",
        "los angeles angels": "angels",
        "san francisco giants": "sfgiants",
        "texas rangers": "rangers",
        "seattle mariners": "mariners",
        "houston astros": "astros",
        "oakland athletics": "athletics",
        "athletics": "athletics",
        "tampa bay rays": "rays",
        "baltimore orioles": "orioles",
        "toronto blue jays": "bluejays",
        "cleveland guardians": "guardians",
        "minnesota twins": "twins",
        "detroit tigers": "tigers",
        "kansas city royals": "royals",
        "chicago cubs": "cubs",
        "milwaukee brewers": "brewers",
        "st. louis cardinals": "cardinals",
        "pittsburgh pirates": "pirates",
        "cincinnati reds": "reds",
        "atlanta braves": "braves",
        "miami marlins": "marlins",
        "philadelphia phillies": "phillies",
        "washington nationals": "nationals",
        "arizona diamondbacks": "dbacks",
        "colorado rockies": "rockies",
        "san diego padres": "padres",
    }
    if s in _map:
        return _map[s]
    words = s.replace("the ", "").strip().split()
    return words[-1] if words else s

# 10 PM auto-switch: after 22:00 fetch tomorrow's games
_mlb_now_hr = datetime.now().hour
if _mlb_now_hr >= 22:
    from datetime import timedelta as _td_mlb
    today_str = (datetime.today() + _td_mlb(days=1)).strftime("%Y%m%d")
    print(f"[10PM MODE] Loading tomorrow's MLB card: {today_str[:4]}-{today_str[4:6]}-{today_str[6:]}")
else:
    today_str = datetime.today().strftime("%Y%m%d")
sb_url = f"https://site.api.espn.com/apis/site/v2/sports/baseball/mlb/scoreboard?dates={today_str}"

# 2026-08-31 BUG FIX (confirmed live via curl): every ESPN call from this
# network is edge-blocked by Akamai with a 403 "Access Denied" page
# (Server: AkamaiGHost -- ESPN's own app never sees the request), not a
# rate limit. Direct call still tried first; only a 403 retries once
# through r.jina.ai's free reader-proxy relay (different egress IP,
# verified live to return a real 200 against this exact endpoint).
def _espn_get_c42(url, timeout=10):
    try:
        r = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=timeout)
        if r.status_code != 403:
            return r.json()
    except Exception:
        pass
    rr = requests.get(f"https://r.jina.ai/{url}", timeout=timeout + 20)
    rr.raise_for_status()
    body = rr.text
    idx = body.find("Markdown Content:")
    json_text = body[idx + len("Markdown Content:"):].strip() if idx >= 0 else body
    import json as _json_c42
    return _json_c42.loads(json_text)

try:
    espn_data = _espn_get_c42(sb_url, timeout=10)
    events = espn_data.get("events", [])
    print(f"ESPN scoreboard: {len(events)} events today")
    # ODDS FETCH TIMESTAMP (2026-09-19, user audit -- Rays +26.1% edge:
    # no way to tell if that was calculated against a sharp current line
    # or a stale early number). This cell is the single point where
    # every mkt%/edge% on the card ultimately traces back to one ESPN
    # snapshot per run -- capturing when that snapshot was actually
    # pulled here, once, so the card can print it next to every price.
    globals()["mlb_odds_fetch_time"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
except Exception as e:
    events = []
    print(f"WARNING: ESPN scoreboard failed: {e}")

odds_rows = []

for event in events:
    comps = event.get("competitions", [])
    if not comps:
        continue
    comp = comps[0]
    # BUG FIX (PIPE-GAMESTART-1, 2026-09-24 user audit): this pulled odds
    # for every event on the scoreboard with no check on whether the game
    # has already started or finished -- ESPN's own status.type.state field
    # ("pre"/"in"/"post") sits right on this same competition object,
    # unread. A game that's started no longer has a real PREGAME line to
    # price against (live in-game odds are a different market), and a
    # finished game has none at all. Defaults to "pre" (keep the game) if
    # the field is ever missing/malformed, so an unexpected schema change
    # degrades to the old behavior instead of silently dropping today's
    # whole slate.
    _ev_state = ((comp.get("status") or {}).get("type") or {}).get("state", "pre")
    if _ev_state != "pre":
        continue
    competitors = comp.get("competitors", [])
    if len(competitors) < 2:
        continue
    home_c = next((c for c in competitors if c.get("homeAway") == "home"), None)
    away_c = next((c for c in competitors if c.get("homeAway") == "away"), None)
    if not home_c or not away_c:
        continue
    home_name = home_c.get("team", {}).get("displayName", "")
    away_name = away_c.get("team", {}).get("displayName", "")
    event_id  = event.get("id", "")

    home_ml   = np.nan
    away_ml   = np.nan
    over_under= np.nan
    over_ml   = np.nan
    under_ml  = np.nan

    # --- attempt 1: scoreboard odds field ---
    for odds_entry in comp.get("odds", []):
        hto = odds_entry.get("homeTeamOdds", {})
        ato = odds_entry.get("awayTeamOdds", {})
        if hto and hto.get("moneyLine") is not None:
            # MLB-BB-ODDS-1 (2026-10-04): keep the book name so every below-bar line can print it
            globals().setdefault("mlb_ml_book_by_home", {})[str(home_name).lower().strip()] = (
                (odds_entry.get("provider") or {}).get("name") or "ESPN feed")
            try:
                home_ml    = float(hto["moneyLine"])
                away_ml    = float(ato.get("moneyLine", np.nan))
                _ou_raw    = odds_entry.get("overUnder")
                if _ou_raw is not None:
                    over_under = float(_ou_raw)
                # over/under prices
                for _fe in ["overOdds", "over_odds"]:
                    if odds_entry.get(_fe) is not None:
                        try: over_ml = float(odds_entry[_fe]); break
                        except Exception: pass
                for _fe in ["underOdds", "under_odds"]:
                    if odds_entry.get(_fe) is not None:
                        try: under_ml = float(odds_entry[_fe]); break
                        except Exception: pass
                _oto = odds_entry.get("overTeamOdds", {})
                _uto = odds_entry.get("underTeamOdds", {})
                if isinstance(_oto, dict) and np.isnan(over_ml) and _oto.get("moneyLine") is not None:
                    try: over_ml = float(_oto["moneyLine"])
                    except Exception: pass
                if isinstance(_uto, dict) and np.isnan(under_ml) and _uto.get("moneyLine") is not None:
                    try: under_ml = float(_uto["moneyLine"])
                    except Exception: pass
            except Exception:
                pass
            break

    # --- attempt 2: per-event ESPN odds API ---
    if (np.isnan(home_ml) or np.isnan(away_ml)) and event_id:
        try:
            ev_url = (
                f"https://sports.core.api.espn.com/v2/sports/baseball/leagues/mlb"
                f"/events/{event_id}/competitions/{event_id}/odds"
            )
            ro = requests.get(ev_url, timeout=8)
            if ro.status_code == 200:
                ev_data = ro.json()
                for item in ev_data.get("items", []):
                    hto2 = item.get("homeTeamOdds", {})
                    ato2 = item.get("awayTeamOdds", {})
                    if hto2 and hto2.get("moneyLine") is not None:
                        try:
                            home_ml    = float(hto2["moneyLine"])
                            away_ml    = float(ato2.get("moneyLine", np.nan))
                            _ou2_raw   = item.get("overUnder")
                            if _ou2_raw is not None and np.isnan(over_under):
                                over_under = float(_ou2_raw)
                            if np.isnan(over_ml) and item.get("overOdds") is not None:
                                try: over_ml = float(item["overOdds"])
                                except Exception: pass
                            if np.isnan(under_ml) and item.get("underOdds") is not None:
                                try: under_ml = float(item["underOdds"])
                                except Exception: pass
                        except Exception:
                            pass
                        break
        except Exception:
            pass

    has_odds = not (np.isnan(home_ml) or np.isnan(away_ml))
    h_raw = _american_to_implied(home_ml)
    a_raw = _american_to_implied(away_ml)
    h_fair, a_fair = _remove_vig(h_raw, a_raw)

    odds_rows.append({
        "home_team":        home_name,
        "away_team":        away_name,
        "_hkey":            _norm_team(home_name),
        "_akey":            _norm_team(away_name),
        "event_date":       event.get("date"),
        # BUG FIX (MLB-DOUBLEHEADER-1, 2026-09-25 user audit): "the card
        # didn't know BAL@NYY was a doubleheader, and it attached
        # Brandon Young to the right game by luck" -- ESPN's real event
        # id was already being fetched (event.get("id")) but only ever
        # used as a temp local for the per-event odds fallback call --
        # never carried into the odds table that downstream cells join
        # against. Two games between the same two teams on the same day
        # produced two rows with IDENTICAL _hkey/_akey/date, completely
        # indistinguishable to any team+date join. Carrying the real,
        # unique ESPN event id through fixes that regardless of which
        # game is G1/G2; _dh_game_num (assigned just below, by event
        # time order within a same-day team-pair) adds the human-
        # readable G1/G2 tag the card can print.
        "event_id":         event_id,
        "home_ml":          home_ml,
        "away_ml":          away_ml,
        "has_odds":         has_odds,
        "home_market_prob": round(h_fair, 4) if (has_odds and not np.isnan(h_fair)) else np.nan,
        "away_market_prob": round(a_fair, 4) if (has_odds and not np.isnan(a_fair)) else np.nan,
        "over_under":       round(over_under, 1) if not np.isnan(over_under) else np.nan,
        "over_ml":          over_ml  if not np.isnan(over_ml)  else np.nan,
        "under_ml":         under_ml if not np.isnan(under_ml) else np.nan,
    })

mlb_espn_odds = pd.DataFrame(odds_rows) if odds_rows else pd.DataFrame(
    columns=["home_team", "away_team", "_hkey", "_akey", "event_date",
             "home_ml", "away_ml", "has_odds",
             "home_market_prob", "away_market_prob", "over_under"])

# BUG FIX (MLB-DOUBLEHEADER-1, 2026-09-25 user audit): a real doubleheader
# is two DISTINCT rows sharing the same _hkey/_akey and calendar date --
# detect that directly (no schedule-source dependency) and tag G1/G2 by
# real event start TIME order (event_date carries the actual ISO
# timestamp, not just the date), so downstream cells (starter matching,
# [SERIES CONTEXT] display) can key on event_id instead of team+date and
# the card can show which game a play is actually for.
if len(mlb_espn_odds) > 0:
    mlb_espn_odds["_pair_key"] = mlb_espn_odds["_hkey"] + "|" + mlb_espn_odds["_akey"] + "|" + pd.to_datetime(mlb_espn_odds["event_date"], errors="coerce").dt.date.astype(str)
    _pair_counts = mlb_espn_odds["_pair_key"].value_counts()
    mlb_espn_odds["is_doubleheader"] = mlb_espn_odds["_pair_key"].map(_pair_counts).fillna(1) > 1
    mlb_espn_odds["dh_game_num"] = (
        mlb_espn_odds.sort_values("event_date")
        .groupby("_pair_key").cumcount() + 1
    ).reindex(mlb_espn_odds.index)
    _dh_pairs = mlb_espn_odds.loc[mlb_espn_odds["is_doubleheader"], "_pair_key"].unique()
    if len(_dh_pairs) > 0:
        print(f"\n  !! DOUBLEHEADER DETECTED: {len(_dh_pairs)} team-pair(s) playing twice today -- "
              f"tagged G1/G2 by event start time, keyed by event_id (not team+date) downstream")
        for _dh_row in mlb_espn_odds.loc[mlb_espn_odds["is_doubleheader"]].sort_values(["_pair_key","event_date"]).itertuples():
            print(f"    G{int(_dh_row.dh_game_num)}: {_dh_row.away_team} @ {_dh_row.home_team} (event_id={_dh_row.event_id}, {_dh_row.event_date})")
else:
    mlb_espn_odds["is_doubleheader"] = False
    mlb_espn_odds["dh_game_num"] = 1

globals()["mlb_espn_odds"] = mlb_espn_odds

# STALE ODDS DETECTOR
# ESPN API sometimes loads outdated or incorrect lines.
# Any team priced >-233 (~0.70 implied) in a regular game is suspicious.
# Brewers -264 when real line was -160 is the canonical example.
_STALE_PROB_THRESH = 0.70
_stale_flags = []
for _, _so_row in mlb_espn_odds.iterrows():
    if not _so_row.get("has_odds", False): continue
    _h_p = float(_so_row.get("home_market_prob") or 0)
    _a_p = float(_so_row.get("away_market_prob") or 0)
    _h_ml = float(_so_row.get("home_ml") or 0)
    _a_ml = float(_so_row.get("away_ml") or 0)
    if _h_p > _STALE_PROB_THRESH or _a_p > _STALE_PROB_THRESH:
        _fav_team = _so_row["home_team"] if _h_p > _a_p else _so_row["away_team"]
        _fav_ml   = _h_ml if _h_p > _a_p else _a_ml
        _fav_prob = max(_h_p, _a_p)
        _stale_flags.append(
            f"  !! STALE ODDS CHECK: {_so_row['away_team']} @ {_so_row['home_team']} -- "
            f"{_fav_team} loaded at {int(_fav_ml):+d} ({_fav_prob:.1%} implied). "
            f"Verify actual line before running card -- ESPN may have cached stale data."
        )
if _stale_flags:
    print()
    print("=" * 70)
    print("STALE ODDS WARNING -- verify manually before running card:")
    for _sf in _stale_flags:
        print(_sf)
    print("=" * 70)

n_odds  = int(mlb_espn_odds["has_odds"].sum())  if len(mlb_espn_odds) > 0 else 0
n_total = len(mlb_espn_odds)
print(f"\nOdds found: {n_odds}/{n_total} games")
if n_total > 0:
    show_cols = [c for c in ["away_team", "home_team", "away_ml", "home_ml",
                              "away_market_prob", "home_market_prob", "over_under", "over_ml", "under_ml", "has_odds"]
                 if c in mlb_espn_odds.columns]
    print(mlb_espn_odds[show_cols].to_string(index=False))
else:
    print("No events returned from ESPN scoreboard.")


PULL TODAY'S MLB MONEYLINE ODDS FROM ESPN


ESPN scoreboard: 1 events today



Odds found: 1/1 games
        away_team           home_team  away_ml  home_ml  away_market_prob  home_market_prob  over_under  over_ml  under_ml  has_odds
Chicago White Sox Cleveland Guardians    118.0   -142.0            0.4388            0.5612         7.0   -120.0     100.0      True


In [44]:
# %%
# =========================================================
# CELL 21 Ã¢â‚¬â€ MERGE MODEL PROBABILITIES ONTO MATCHED KALSHI ROWS
# REPLACEMENT
# Robust to missing mlb_projection_board and alternate board names
# =========================================================

import pandas as pd
import numpy as np

print("=" * 70)
print("CELL 21 Ã¢â‚¬â€ MERGE MODEL PROBABILITIES ONTO MATCHED KALSHI ROWS")
print("=" * 70)

# ---------------------------------------------------------
# 0) validate inputs
# ---------------------------------------------------------
if "mlb_kalshi_matched" not in globals() or not isinstance(mlb_kalshi_matched, pd.DataFrame):
    raise ValueError("mlb_kalshi_matched not found. Run Cell 20 first.")

km = mlb_kalshi_matched.copy()

# ---------------------------------------------------------
# 1) helpers
# ---------------------------------------------------------
def clean_team(s):
    if pd.isna(s):
        return None
    s = str(s).strip().lower()

    replacements = {
        "athletics": "oakland athletics",
        "a's": "oakland athletics",
        "d-backs": "arizona diamondbacks",
        "diamondbacks": "arizona diamondbacks",
        "white sox": "chicago white sox",
        "red sox": "boston red sox",
    }
    return replacements.get(s, s)

def first_existing_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

def first_existing_df(candidates):
    for name in candidates:
        if name in globals() and isinstance(globals()[name], pd.DataFrame):
            df = globals()[name]
            if len(df) > 0:
                print(f"Using projection source: {name}  shape={df.shape}")
                return df.copy(), name
    return None, None

# ---------------------------------------------------------
# 2) locate / rebuild projection board
# ---------------------------------------------------------
proj_candidates = [
    "mlb_projection_board",
    "scored_mlb_board",
    "mlb_scored_board",
    "final_mlb_board",
    "mlb_board_final",
    "today_mlb_board",
    "board_with_predictions",
]

proj, proj_source_name = first_existing_df(proj_candidates)

if proj is None:
    raise ValueError(
        "Could not find a usable projection board in memory.\n"
        "Expected one of: mlb_projection_board, scored_mlb_board, mlb_scored_board, "
        "final_mlb_board, mlb_board_final, today_mlb_board, board_with_predictions.\n"
        "Run your MLB projection/scoring cell first."
    )

print("Kalshi matched shape:", km.shape)
print("Projection source shape:", proj.shape)

# ---------------------------------------------------------
# 3) validate Kalshi key columns
# ---------------------------------------------------------
for c in ["away_team", "home_team", "selected_team"]:
    if c not in km.columns:
        raise ValueError(f"mlb_kalshi_matched missing required column: {c}")

# ---------------------------------------------------------
# 4) ensure projection team columns exist
# ---------------------------------------------------------
away_proj_col = first_existing_col(proj, ["away_team", "away_team_name", "team_away"])
home_proj_col = first_existing_col(proj, ["home_team", "home_team_name", "team_home"])

if away_proj_col is None or home_proj_col is None:
    raise ValueError(
        "Projection board is missing team columns.\n"
        f"Available columns:\n{list(proj.columns)}"
    )

if away_proj_col != "away_team":
    proj["away_team"] = proj[away_proj_col]

if home_proj_col != "home_team":
    proj["home_team"] = proj[home_proj_col]

# ---------------------------------------------------------
# 5) normalize merge keys
# ---------------------------------------------------------
km["away_key"] = km["away_team"].apply(clean_team)
km["home_key"] = km["home_team"].apply(clean_team)
km["selected_team_key"] = km["selected_team"].apply(clean_team)

proj["away_key"] = proj["away_team"].apply(clean_team)
proj["home_key"] = proj["home_team"].apply(clean_team)

# ---------------------------------------------------------
# 6) resolve projection columns safely
# ---------------------------------------------------------
projected_winner_col = first_existing_col(proj, [
    "projected_winner",
    "projected_winner_new",
    "predicted_winner",
    "pred_winner",
    "winner_pick",
    "model_winner",
])

projected_loser_col = first_existing_col(proj, [
    "projected_loser",
    "projected_loser_new",
    "predicted_loser",
    "pred_loser",
    "model_loser",
])

p_home_col = first_existing_col(proj, [
    "p_home_win",
    "p_home_win_new",
    "pred_home_win_prob",
    "projected_home_win_prob",
    "home_win_prob",
    "home_prob",
])

p_away_col = first_existing_col(proj, [
    "p_away_win",
    "p_away_win_new",
    "pred_away_win_prob",
    "projected_away_win_prob",
    "away_win_prob",
    "away_prob",
])

projected_win_prob_col = first_existing_col(proj, [
    "projected_win_prob",
    "win_prob",
    "best_win_prob",
    "model_win_prob",
])

projected_win_pct_col = first_existing_col(proj, [
    "projected_win_pct",
    "win_confidence_new",
    "projected_win_prob",
    "win_prob",
    "best_win_prob",
    "model_win_prob",
])

proj_margin_col = first_existing_col(proj, [
    "proj_margin_proxy",
    "projected_margin_proxy",
    "margin_proxy",
    "pred_margin",
    "projected_margin",
])

strength_tier_col = first_existing_col(proj, [
    "strength_tier",
    "play_tier",
    "tier",
])

# ---------------------------------------------------------
# 7) fill missing win-prob columns if possible
# ---------------------------------------------------------
if p_home_col is None and p_away_col is not None:
    proj["p_home_win_fallback"] = 1 - pd.to_numeric(proj[p_away_col], errors="coerce")
    p_home_col = "p_home_win_fallback"

if p_away_col is None and p_home_col is not None:
    proj["p_away_win_fallback"] = 1 - pd.to_numeric(proj[p_home_col], errors="coerce")
    p_away_col = "p_away_win_fallback"

if projected_winner_col is None and p_home_col is not None and p_away_col is not None:
    proj["projected_winner_fallback"] = np.where(
        pd.to_numeric(proj[p_home_col], errors="coerce") >= pd.to_numeric(proj[p_away_col], errors="coerce"),
        proj["home_team"],
        proj["away_team"]
    )
    projected_winner_col = "projected_winner_fallback"

if projected_win_prob_col is None and projected_winner_col is not None and p_home_col is not None and p_away_col is not None:
    proj["projected_win_prob_fallback"] = np.where(
        proj[projected_winner_col] == proj["home_team"],
        pd.to_numeric(proj[p_home_col], errors="coerce"),
        pd.to_numeric(proj[p_away_col], errors="coerce")
    )
    projected_win_prob_col = "projected_win_prob_fallback"

required = {
    "projected_winner": projected_winner_col,
    "p_home": p_home_col,
    "p_away": p_away_col,
}

missing_required = [k for k, v in required.items() if v is None]
if missing_required:
    raise ValueError(
        f"Could not resolve required projection columns: {missing_required}\n"
        f"Projection source used: {proj_source_name}\n"
        f"Available projection columns:\n{list(proj.columns)}"
    )

print("\nResolved projection columns:")
print(f" - projection source      : {proj_source_name}")
print(f" - projected_winner_col   : {projected_winner_col}")
print(f" - projected_loser_col    : {projected_loser_col}")
print(f" - p_home_col             : {p_home_col}")
print(f" - p_away_col             : {p_away_col}")
print(f" - projected_win_prob_col : {projected_win_prob_col}")
print(f" - projected_win_pct_col  : {projected_win_pct_col}")
print(f" - proj_margin_col        : {proj_margin_col}")
print(f" - strength_tier_col      : {strength_tier_col}")

# ---------------------------------------------------------
# 8) build standardized projection merge table
# ---------------------------------------------------------
proj_merge = proj[["away_key", "home_key", "away_team", "home_team"]].copy()

proj_merge["projected_winner"] = proj[projected_winner_col]

if projected_loser_col is not None:
    proj_merge["projected_loser"] = proj[projected_loser_col]
else:
    proj_merge["projected_loser"] = np.where(
        proj_merge["projected_winner"] == proj_merge["home_team"],
        proj_merge["away_team"],
        proj_merge["home_team"]
    )

proj_merge["p_home_win"] = pd.to_numeric(proj[p_home_col], errors="coerce")
proj_merge["p_away_win"] = pd.to_numeric(proj[p_away_col], errors="coerce")

if projected_win_prob_col is not None:
    proj_merge["projected_win_prob"] = pd.to_numeric(proj[projected_win_prob_col], errors="coerce")
else:
    proj_merge["projected_win_prob"] = np.where(
        proj_merge["projected_winner"] == proj_merge["home_team"],
        proj_merge["p_home_win"],
        proj_merge["p_away_win"]
    )

if projected_win_pct_col is not None:
    proj_merge["projected_win_pct"] = pd.to_numeric(proj[projected_win_pct_col], errors="coerce")
else:
    proj_merge["projected_win_pct"] = proj_merge["projected_win_prob"] * 100

if strength_tier_col is not None:
    proj_merge["strength_tier"] = proj[strength_tier_col]
else:
    proj_merge["strength_tier"] = np.nan

if proj_margin_col is not None:
    proj_merge["proj_margin_proxy"] = pd.to_numeric(proj[proj_margin_col], errors="coerce")
else:
    proj_merge["proj_margin_proxy"] = np.nan

proj_merge = (
    proj_merge
    .drop_duplicates(subset=["away_key", "home_key"], keep="last")
    .reset_index(drop=True)
)

# ---------------------------------------------------------
# 9) merge model projections onto Kalshi rows
# ---------------------------------------------------------
merged = km.merge(
    proj_merge[
        [
            "away_key",
            "home_key",
            "projected_winner",
            "projected_loser",
            "p_home_win",
            "p_away_win",
            "projected_win_prob",
            "projected_win_pct",
            "strength_tier",
            "proj_margin_proxy",
        ]
    ],
    on=["away_key", "home_key"],
    how="left"
).copy()

# ---------------------------------------------------------
# 10) compute model probability for the YES side
# ---------------------------------------------------------
def compute_p_model_yes(row):
    sel = clean_team(row.get("selected_team"))
    home = clean_team(row.get("home_team"))
    away = clean_team(row.get("away_team"))

    if sel is None:
        return np.nan
    if sel == home:
        return row.get("p_home_win", np.nan)
    if sel == away:
        return row.get("p_away_win", np.nan)
    return np.nan

merged["p_model_yes"] = merged.apply(compute_p_model_yes, axis=1)
merged["p_model_no"] = 1 - merged["p_model_yes"]

# ---------------------------------------------------------
# 11) readable bet-side fields
# ---------------------------------------------------------
merged["projected_winner_key"] = merged["projected_winner"].apply(clean_team)

merged["model_pick_matches_yes"] = (
    merged["selected_team_key"] == merged["projected_winner_key"]
)

merged["recommended_side"] = np.where(
    merged["model_pick_matches_yes"],
    "YES",
    "NO"
)

merged["recommended_team"] = merged["projected_winner"]

merged["model_for_recommended_bet"] = np.where(
    merged["recommended_side"] == "YES",
    merged["p_model_yes"],
    merged["p_model_no"]
)

# ---------------------------------------------------------
# 12) market price helpers
# ---------------------------------------------------------
for c in ["yes_ask", "no_ask", "yes_bid", "no_bid", "last_price"]:
    if c in merged.columns:
        merged[c] = pd.to_numeric(merged[c], errors="coerce")

def get_market_yes_prob(row):
    yes_ask = row.get("yes_ask", np.nan)
    last_price = row.get("last_price", np.nan)

    if pd.notna(yes_ask):
        return yes_ask
    if pd.notna(last_price):
        return last_price
    return np.nan

merged["p_market_yes"] = merged.apply(get_market_yes_prob, axis=1)
merged["p_market_no"] = 1 - merged["p_market_yes"]

merged["edge_yes"] = merged["p_model_yes"] - merged["p_market_yes"]
merged["edge_no"] = merged["p_model_no"] - merged["p_market_no"]

merged["market_price_available"] = merged["p_market_yes"].notna()

# ---------------------------------------------------------
# 13) output
# ---------------------------------------------------------
print("\nMERGED MODEL Ã¢â€ â€ KALSHI TABLE")
print("-" * 70)
print("Rows:", len(merged))
print("Missing projected_winner:", int(merged["projected_winner"].isna().sum()))
print("Missing p_home_win      :", int(merged["p_home_win"].isna().sum()))
print("Missing p_away_win      :", int(merged["p_away_win"].isna().sum()))
print("Missing p_model_yes     :", int(merged["p_model_yes"].isna().sum()))
print("Missing p_market_yes    :", int(merged["p_market_yes"].isna().sum()))

preview_cols = [
    "away_team",
    "home_team",
    "selected_team",
    "projected_winner",
    "recommended_team",
    "recommended_side",
    "p_home_win",
    "p_away_win",
    "p_model_yes",
    "p_model_no",
    "p_market_yes",
    "edge_yes",
    "edge_no",
    "market_price_available",
]

preview_cols = [c for c in preview_cols if c in merged.columns]

print("\nPreview:")
display(merged[preview_cols])

# ---------------------------------------------------------
# 14) save in memory
# ---------------------------------------------------------
globals()["mlb_kalshi_with_model"] = merged.copy()

print("\nCreated in memory:")
print(" - mlb_kalshi_with_model")
print("Done.")

CELL 21 Ã¢â‚¬â€ MERGE MODEL PROBABILITIES ONTO MATCHED KALSHI ROWS
Using projection source: scored_mlb_board  shape=(1, 97)
Kalshi matched shape: (0, 38)
Projection source shape: (1, 97)

Resolved projection columns:
 - projection source      : scored_mlb_board
 - projected_winner_col   : projected_winner
 - projected_loser_col    : projected_loser
 - p_home_col             : p_home_win
 - p_away_col             : p_away_win
 - projected_win_prob_col : projected_win_prob
 - projected_win_pct_col  : projected_win_prob
 - proj_margin_col        : None
 - strength_tier_col      : strength_tier

MERGED MODEL Ã¢â€ â€ KALSHI TABLE
----------------------------------------------------------------------
Rows: 0
Missing projected_winner: 0
Missing p_home_win      : 0
Missing p_away_win      : 0
Missing p_model_yes     : 0
Missing p_market_yes    : 0

Preview:


,away_team,home_team,selected_team,projected_winner,recommended_team,recommended_side,p_home_win,p_away_win,p_model_yes,p_model_no,p_market_yes,edge_yes,edge_no,market_price_available



Created in memory:
 - mlb_kalshi_with_model
Done.


In [45]:
# %%
# =========================================================
# CELL 22 Ã¢â‚¬â€ FINAL CARD BUILDER
# REPLACEMENT
# If market prices exist: build aggressive betting card
# If not: build projection-only social media card
# Does NOT require mlb_projection_board specifically
# =========================================================

import pandas as pd
import numpy as np

print("=" * 70)
print("CELL 22 Ã¢â‚¬â€ FINAL CARD BUILDER")
print("=" * 70)

# ---------------------------------------------------------
# helpers
# ---------------------------------------------------------
def first_existing_df(candidates):
    for name in candidates:
        if name in globals() and isinstance(globals()[name], pd.DataFrame):
            df = globals()[name]
            if len(df) > 0:
                print(f"Using board source: {name}  shape={df.shape}")
                return df.copy(), name
    return None, None

def first_existing_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

def ensure_numeric_col(df, col):
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    return df

def ensure_default(name, default_value):
    if name not in globals():
        globals()[name] = default_value
        print(f"{name} not found. Using default = {default_value}")

# ---------------------------------------------------------
# defaults for filters if not already defined
# ---------------------------------------------------------
ensure_default("MIN_CONF", 0.55)
ensure_default("MIN_EDGE", 0.05)
ensure_default("MIN_PRICE", 0.10)
ensure_default("MAX_PRICE", 0.75)

# ---------------------------------------------------------
# locate projection-style board
# ---------------------------------------------------------
proj_candidates = [
    "scored_mlb_board",
    "mlb_scored_board",
    "mlb_projection_board",
    "final_mlb_board",
    "mlb_board_final",
    "today_mlb_board",
    "board_with_predictions",
    "mlb_kalshi_with_model",
]

proj, proj_source_name = first_existing_df(proj_candidates)

if proj is None:
    raise ValueError(
        "No usable MLB board found in memory.\n"
        "Expected one of: mlb_projection_board, scored_mlb_board, mlb_scored_board, "
        "final_mlb_board, mlb_board_final, today_mlb_board, board_with_predictions, "
        "mlb_kalshi_with_model."
    )

# ---------------------------------------------------------
# resolve important columns safely
# ---------------------------------------------------------
away_col = first_existing_col(proj, ["away_team", "away_team_name", "team_away"])
home_col = first_existing_col(proj, ["home_team", "home_team_name", "team_home"])
winner_col = first_existing_col(proj, [
    "projected_winner",
    "projected_winner_new",
    "predicted_winner",
    "pred_winner",
    "winner_pick",
    "recommended_team",
])
win_prob_col = first_existing_col(proj, [
    "projected_win_prob",
    "win_prob",
    "best_win_prob",
    "model_win_prob",
    "model_for_recommended_bet",
])
strength_col = first_existing_col(proj, [
    "strength_tier",
    "play_tier",
    "tier",
])
margin_col = first_existing_col(proj, [
    "proj_margin_proxy",
    "projected_margin_proxy",
    "margin_proxy",
    "pred_margin",
    "projected_margin",
])
datetime_col = first_existing_col(proj, [
    "game_datetime_utc",
    "start_time",
    "game_time_utc",
    "game_date",
    "date",
])

if away_col is None or home_col is None:
    raise ValueError(
        "Could not resolve away/home team columns from the board.\n"
        f"Available columns:\n{list(proj.columns)}"
    )

if away_col != "away_team":
    proj["away_team"] = proj[away_col]
if home_col != "home_team":
    proj["home_team"] = proj[home_col]

# winner fallback
if winner_col is None:
    p_home_col = first_existing_col(proj, [
        "p_home_win",
        "p_home_win_new",
        "pred_home_win_prob",
        "projected_home_win_prob",
        "home_win_prob",
        "home_prob",
    ])
    p_away_col = first_existing_col(proj, [
        "p_away_win",
        "p_away_win_new",
        "pred_away_win_prob",
        "projected_away_win_prob",
        "away_win_prob",
        "away_prob",
    ])

    if p_home_col is not None and p_away_col is not None:
        proj["projected_winner_fallback"] = np.where(
            pd.to_numeric(proj[p_home_col], errors="coerce") >= pd.to_numeric(proj[p_away_col], errors="coerce"),
            proj["home_team"],
            proj["away_team"]
        )
        winner_col = "projected_winner_fallback"

# win probability fallback
if win_prob_col is None:
    p_home_col = first_existing_col(proj, [
        "p_home_win",
        "p_home_win_new",
        "pred_home_win_prob",
        "projected_home_win_prob",
        "home_win_prob",
        "home_prob",
    ])
    p_away_col = first_existing_col(proj, [
        "p_away_win",
        "p_away_win_new",
        "pred_away_win_prob",
        "projected_away_win_prob",
        "away_win_prob",
        "away_prob",
    ])

    if winner_col is not None and p_home_col is not None and p_away_col is not None:
        proj["projected_win_prob_fallback"] = np.where(
            proj[winner_col] == proj["home_team"],
            pd.to_numeric(proj[p_home_col], errors="coerce"),
            pd.to_numeric(proj[p_away_col], errors="coerce")
        )
        win_prob_col = "projected_win_prob_fallback"

if winner_col is None or win_prob_col is None:
    raise ValueError(
        "Could not resolve projected winner / win probability columns.\n"
        f"Board source used: {proj_source_name}\n"
        f"Available columns:\n{list(proj.columns)}"
    )

print("\nResolved columns:")
print(f" - board source   : {proj_source_name}")
print(f" - away_col       : {away_col}")
print(f" - home_col       : {home_col}")
print(f" - winner_col     : {winner_col}")
print(f" - win_prob_col   : {win_prob_col}")
print(f" - strength_col   : {strength_col}")
print(f" - margin_col     : {margin_col}")
print(f" - datetime_col   : {datetime_col}")

proj["projected_winner"] = proj[winner_col]
proj["projected_win_prob"] = pd.to_numeric(proj[win_prob_col], errors="coerce")

# Fallback: if projected_win_prob is all NaN, reconstruct from raw model prob columns
if proj["projected_win_prob"].isna().all():
    print(f"WARNING: win_prob_col='{win_prob_col}' resolved to all-NaN. Attempting fallback.")
    _phc = first_existing_col(proj, ["pred_home_win_prob", "p_home_win_new", "p_home_win"])
    _pac = first_existing_col(proj, ["pred_away_win_prob", "p_away_win_new", "p_away_win"])
    if _phc and _pac:
        _ph = pd.to_numeric(proj[_phc], errors="coerce")
        _pa = pd.to_numeric(proj[_pac], errors="coerce")
        proj["projected_win_prob"] = np.where(_ph >= 0.5, _ph, _pa)
        proj["projected_winner"] = np.where(_ph >= 0.5, proj["home_team"], proj["away_team"])
        print(f"Reconstructed from {_phc}/{_pac}. Valid rows: {proj['projected_win_prob'].notna().sum()}")
    else:
        print("No fallback prob columns found. Board columns: " + str(list(proj.columns)))

if strength_col is not None:
    proj["strength_tier"] = proj[strength_col]
else:
    proj["strength_tier"] = np.nan

if margin_col is not None:
    proj["proj_margin_proxy"] = pd.to_numeric(proj[margin_col], errors="coerce")
else:
    proj["proj_margin_proxy"] = np.nan

if datetime_col is not None:
    proj["sort_datetime"] = proj[datetime_col]
else:
    proj["sort_datetime"] = np.nan

# ---------------------------------------------------------
# first try to build a real betting card from Kalshi rows
# ---------------------------------------------------------
bet_card = None

if "mlb_kalshi_with_model" in globals() and isinstance(mlb_kalshi_with_model, pd.DataFrame):
    km = mlb_kalshi_with_model.copy()

    if len(km):
        for c in [
            "p_model_yes", "p_model_no", "p_market_yes", "p_market_no",
            "edge_yes", "edge_no", "yes_ask", "no_ask", "yes_bid", "no_bid", "last_price"
        ]:
            km = ensure_numeric_col(km, c)

        if "recommended_side" not in km.columns:
            km["recommended_side"] = np.where(
                km.get("selected_team", pd.Series(index=km.index)).astype(str)
                == km.get("recommended_team", pd.Series(index=km.index)).astype(str),
                "YES",
                "NO"
            )

        if "recommended_team" not in km.columns and "projected_winner" in km.columns:
            km["recommended_team"] = km["projected_winner"]

        km["bet_side"] = np.where(
            km["recommended_side"].astype(str).str.upper() == "YES",
            "YES",
            "NO"
        )

        km["bet_team"] = km["recommended_team"]

        if "p_model_bet" not in km.columns:
            km["p_model_bet"] = np.where(
                km["bet_side"] == "YES",
                km.get("p_model_yes", np.nan),
                km.get("p_model_no", np.nan)
            )

        if "p_market_bet" not in km.columns:
            km["p_market_bet"] = np.where(
                km["bet_side"] == "YES",
                km.get("p_market_yes", np.nan),
                km.get("p_market_no", np.nan)
            )

        if "edge_bet" not in km.columns:
            km["edge_bet"] = km["p_model_bet"] - km["p_market_bet"]

        km_live = km[km["p_market_bet"].notna()].copy()

        if len(km_live):
            km_live = km_live[
                (km_live["p_model_bet"] >= MIN_CONF) &
                (km_live["edge_bet"] >= MIN_EDGE) &
                (km_live["p_market_bet"] >= MIN_PRICE) &
                (km_live["p_market_bet"] <= MAX_PRICE)
            ].copy()

            if len(km_live):
                km_live["matchup_key"] = (
                    km_live["away_team"].astype(str).str.lower().str.strip() + " @ " +
                    km_live["home_team"].astype(str).str.lower().str.strip()
                )

                km_live = (
                    km_live.sort_values(
                        ["matchup_key", "edge_bet", "p_model_bet"],
                        ascending=[True, False, False]
                    )
                    .drop_duplicates(subset=["matchup_key"], keep="first")
                    .sort_values(["edge_bet", "p_model_bet"], ascending=[False, False])
                    .reset_index(drop=True)
                )

                def size_from_edge(edge, conf):
                    if pd.isna(edge) or pd.isna(conf):
                        return "0.5 UNIT"
                    if edge >= 0.14 and conf >= 0.62:
                        return "1.5 UNIT"
                    if edge >= 0.10 and conf >= 0.59:
                        return "1 UNIT"
                    if edge >= 0.07 and conf >= 0.565:
                        return "0.75 UNIT"
                    return "0.5 UNIT"

                km_live["bet_size"] = [
                    size_from_edge(e, c)
                    for e, c in zip(km_live["edge_bet"], km_live["p_model_bet"])
                ]

                km_live["official_bet"] = [
                    f"Bet {side} Ã¢â‚¬â€ {team}"
                    for side, team in zip(km_live["bet_side"], km_live["bet_team"])
                ]

                km_live["social_bet"] = [
                    f"{team} | {side} @ {price:.3f} | Edge {edge:.3f} | {size}"
                    for team, side, price, edge, size in zip(
                        km_live["bet_team"],
                        km_live["bet_side"],
                        km_live["p_market_bet"],
                        km_live["edge_bet"],
                        km_live["bet_size"]
                    )
                ]

                bet_card = km_live.copy()

# ---------------------------------------------------------
# if no live-priced card exists, build projection-only card
# ---------------------------------------------------------
if bet_card is None or len(bet_card) == 0:
    social_card = proj.copy()

    social_card = social_card[social_card["projected_win_prob"].notna()].copy()

    if len(social_card) == 0:
        # Last resort: re-score board directly if model + feature_list are in memory
        if "model" in globals() and "feature_list" in globals():
            print("Last resort: re-scoring board with model directly...")
            try:
                _feat = [c for c in feature_list if c in proj.columns]
                _missing = [c for c in feature_list if c not in proj.columns]
                if _missing:
                    print(f"  Missing features: {_missing[:10]}")
                _X = proj[_feat].copy().replace([np.inf, -np.inf], np.nan)
                for _c in _X.columns:
                    if _X[_c].isna().any():
                        _med = _X[_c].median(skipna=True)
                        _X[_c] = _X[_c].fillna(_med if not pd.isna(_med) else 0.0)
                _preds = np.clip(pd.to_numeric(model.predict_proba(_X)[:, 1], errors="coerce"), 0.0, 1.0)
                proj["pred_home_win_prob"] = _preds
                proj["pred_away_win_prob"] = 1.0 - _preds
                proj["projected_win_prob"] = np.where(_preds >= 0.5, _preds, 1.0 - _preds)
                proj["projected_winner"] = np.where(_preds >= 0.5, proj["home_team"], proj["away_team"])
                print(f"  Re-scored {len(proj)} rows. Valid: {proj['projected_win_prob'].notna().sum()}")
            except Exception as _e:
                print(f"  Re-scoring failed: {_e}")
                print(f"  Board columns: {list(proj.columns)}")
                raise ValueError("No usable projected_win_prob values found in the board.") from _e
        else:
            print(f"Board source: {proj_source_name}")
            print(f"Board columns: {list(proj.columns)}")
            raise ValueError("No usable projected_win_prob values found. model/feature_list not in memory.")

    social_card = social_card[
        social_card["projected_win_prob"] >= 0.54
    ].copy()

    if len(social_card) == 0:
        social_card = proj.copy()
        social_card = social_card[social_card["projected_win_prob"].notna()].copy()

    social_card = (
        social_card.sort_values(
            ["projected_win_prob", "sort_datetime"],
            ascending=[False, True]
        )
        .head(5)
        .reset_index(drop=True)
    )

    social_card["play_type"] = "Projection Only"
    social_card["bet_size"] = np.where(
        social_card["projected_win_prob"] >= 0.59, "1 UNIT",
        np.where(social_card["projected_win_prob"] >= 0.55, "0.75 UNIT", "0.5 UNIT")
    )

    social_card["official_bet"] = [
        f"{winner} ML"
        for winner in social_card["projected_winner"]
    ]

    social_card["social_bet"] = [
        f"{winner} ML | Win Prob {prob:.3f} | {size}"
        for winner, prob, size in zip(
            social_card["projected_winner"],
            social_card["projected_win_prob"],
            social_card["bet_size"]
        )
    ]

    print("\n==================================================")
    print("EKO BETS Ã¢â‚¬â€ MLB PROJECTION / SOCIAL CARD")
    print("==================================================")
    print("No usable live market prices found.")
    print("Built projection-only social card instead.\n")

    for i, (_, r) in enumerate(social_card.iterrows(), 1):
        print(f"#{i} {r['away_team']} @ {r['home_team']}")
        print(f"   Projected winner: {r['projected_winner']}")
        print(f"   Win probability : {r['projected_win_prob']:.3f}")
        print(f"   Strength tier   : {r['strength_tier']}")
        print(f"   Margin proxy    : {r['proj_margin_proxy']:+.1f}" if pd.notna(r["proj_margin_proxy"]) else "   Margin proxy    : N/A")
        print(f"   Play            : {r['official_bet']}")
        print(f"   Size            : {r['bet_size']}")
        print("-" * 60)

    print("\n==================================================")
    print("EKO BETS Ã¢â‚¬â€ MLB SOCIAL MEDIA CARD")
    print("==================================================")
    for i, (_, r) in enumerate(social_card.iterrows(), 1):
        print(f"{i}. {r['social_bet']}")

    final_mlb_card = social_card.copy()

else:
    print("\n==================================================")
    print("EKO BETS Ã¢â‚¬â€ FINAL MLB BETTING CARD")
    print("==================================================")

    for i, (_, r) in enumerate(bet_card.iterrows(), 1):
        print(f"#{i} {r['away_team']} @ {r['home_team']}")
        print(f"   Projected winner : {r.get('projected_winner', r.get('bet_team', 'N/A'))}")
        print(f"   Bet              : {r['official_bet']}")
        print(f"   Entry            : {r['p_market_bet']:.3f}")
        print(f"   Model for bet    : {r['p_model_bet']:.3f}")
        print(f"   Market for bet   : {r['p_market_bet']:.3f}")
        print(f"   Edge             : {r['edge_bet']:.3f}")
        print(f"   Size             : {r['bet_size']}")
        print("-" * 60)

    print("\n==================================================")
    print("EKO BETS Ã¢â‚¬â€ MLB SOCIAL MEDIA CARD")
    print("==================================================")
    for i, (_, r) in enumerate(bet_card.iterrows(), 1):
        print(f"{i}. {r['social_bet']}")

    final_mlb_card = bet_card.copy()

# ---------------------------------------------------------
# save in memory
# ---------------------------------------------------------
globals()["final_mlb_card"] = final_mlb_card.copy()

print("\nCreated in memory:")
print(" - final_mlb_card")
print("Done.")

CELL 22 Ã¢â‚¬â€ FINAL CARD BUILDER
MIN_CONF not found. Using default = 0.55
MIN_EDGE not found. Using default = 0.05
MIN_PRICE not found. Using default = 0.1
MAX_PRICE not found. Using default = 0.75
Using board source: scored_mlb_board  shape=(1, 97)

Resolved columns:
 - board source   : scored_mlb_board
 - away_col       : away_team
 - home_col       : home_team
 - winner_col     : projected_winner
 - win_prob_col   : projected_win_prob
 - strength_col   : strength_tier
 - margin_col     : None
 - datetime_col   : game_datetime_utc
Reconstructed from pred_home_win_prob/pred_away_win_prob. Valid rows: 0
Last resort: re-scoring board with model directly...
  Re-scored 1 rows. Valid: 1

EKO BETS Ã¢â‚¬â€ MLB PROJECTION / SOCIAL CARD
No usable live market prices found.
Built projection-only social card instead.

#1 Chicago White Sox @ Cleveland Guardians
   Projected winner: Cleveland Guardians
   Win probability : 0.554
   Strength tier   : Lean
   Margin proxy    : N/A
   Play       

In [46]:
# %%
# =========================================================
# CELL 23 Ã¢â‚¬â€ REFRESH LIVE KALSHI PRICES FROM MARKET TICKERS
# Directly query each matched market ticker to try to get
# live yes/no ask/bid/last price data
# =========================================================

import pandas as pd
import numpy as np
import requests
import time

if "mlb_kalshi_with_model" not in globals() or not isinstance(mlb_kalshi_with_model, pd.DataFrame):
    raise ValueError("mlb_kalshi_with_model not found. Run Cell 21 first.")

KALSHI_BASE = "https://api.elections.kalshi.com/trade-api/v2"

def kalshi_get(path, params=None, timeout=30):
    url = f"{KALSHI_BASE}{path}"
    r = requests.get(url, params=params, timeout=timeout, headers={"Accept-Encoding": "gzip, deflate"})
    r.raise_for_status()
    return r.json()

km = mlb_kalshi_with_model.copy()

if "market_ticker" not in km.columns:
    raise ValueError("market_ticker column missing from mlb_kalshi_with_model.")

tickers = (
    km["market_ticker"]
    .dropna()
    .astype(str)
    .drop_duplicates()
    .tolist()
)

print("Matched market tickers to refresh:", len(tickers))

price_rows = []

for i, ticker in enumerate(tickers, 1):
    try:
        # try direct market lookup
        payload = kalshi_get(f"/markets/{ticker}")
        
        # some responses may wrap under 'market', some may not
        m = payload.get("market", payload) if isinstance(payload, dict) else {}

        row = {
            "market_ticker": ticker,
            "yes_ask_live": m.get("yes_ask"),
            "no_ask_live": m.get("no_ask"),
            "yes_bid_live": m.get("yes_bid"),
            "no_bid_live": m.get("no_bid"),
            "last_price_live": m.get("last_price"),
            "market_status_live": m.get("status"),
            "liquidity_live": m.get("liquidity"),
            "volume_live": m.get("volume"),
        }
        price_rows.append(row)
        print(f"[{i}/{len(tickers)}] OK  - {ticker}")

    except Exception as e:
        print(f"[{i}/{len(tickers)}] FAIL - {ticker} -> {e}")
        price_rows.append({
            "market_ticker": ticker,
            "yes_ask_live": np.nan,
            "no_ask_live": np.nan,
            "yes_bid_live": np.nan,
            "no_bid_live": np.nan,
            "last_price_live": np.nan,
            "market_status_live": None,
            "liquidity_live": np.nan,
            "volume_live": np.nan,
        })

    # tiny pause to reduce rate-limit risk
    time.sleep(0.15)

live_prices = pd.DataFrame(price_rows)

# normalize any cent-based prices
for c in ["yes_ask_live", "no_ask_live", "yes_bid_live", "no_bid_live", "last_price_live"]:
    if c in live_prices.columns:
        live_prices[c] = pd.to_numeric(live_prices[c], errors="coerce")
        live_prices[c] = np.where(live_prices[c] > 1, live_prices[c] / 100, live_prices[c])

print("\nLive prices pulled:")
display(live_prices)

# BUG FIX (2026-09-29, found live -- MLB-POSTSEASON-1 investigation: this crashed the
# ENTIRE MLB pipeline on real Wild Card day 1 with "KeyError: 'market_ticker'"). When
# zero tickers matched (real case today -- postseason games don't carry over regular-
# season Kalshi ticker IDs), `tickers` is empty, the price-fetch loop never runs, and
# pd.DataFrame([]) produces a frame with NO COLUMNS AT ALL -- not even market_ticker --
# so the merge below KeyError'd on a column that plain doesn't exist yet. Live prices are
# genuinely unavailable in that case (not a bug to work around, a real "nothing to merge"
# state) -- skip the merge and leave km's existing (pre-live) price columns as the final
# ones, exactly like every "no live values found" case an old_col/new_col loop entry
# already handles gracefully below.
if not live_prices.empty and "market_ticker" in live_prices.columns:
    # merge back
    km = km.merge(live_prices, on="market_ticker", how="left")
else:
    # BUG FIX ROUND 2 (2026-09-29, found live on the very next real run): the first pass
    # only backfilled the 5 price columns -- missed market_status_live/liquidity_live/
    # volume_live, which the preview_cols selection further down ALSO requires unconditionally,
    # so the pipeline crashed on the exact same "0 tickers matched" case one KeyError later.
    # Every column price_rows would normally produce, all NaN -- the honest "no live data"
    # state, not a guessed placeholder.
    print("  No live Kalshi prices to merge (0 tickers matched) -- keeping existing prices as-is.")
    for _kc in ["yes_ask_live", "no_ask_live", "yes_bid_live", "no_bid_live", "last_price_live",
                "market_status_live", "liquidity_live", "volume_live"]:
        km[_kc] = np.nan

# overwrite old price columns only when live values exist
for old_col, new_col in [
    ("yes_ask", "yes_ask_live"),
    ("no_ask", "no_ask_live"),
    ("yes_bid", "yes_bid_live"),
    ("no_bid", "no_bid_live"),
    ("last_price", "last_price_live"),
]:
    if old_col in km.columns and new_col in km.columns:
        km[old_col] = np.where(km[new_col].notna(), km[new_col], km[old_col])

def get_market_yes_prob(row):
    if pd.notna(row.get("yes_ask", np.nan)):
        return row["yes_ask"]
    if pd.notna(row.get("last_price", np.nan)):
        return row["last_price"]
    return np.nan

km["p_market_yes"] = km.apply(get_market_yes_prob, axis=1)
km["p_market_no"] = 1 - km["p_market_yes"]

km["edge_yes"] = km["p_model_yes"] - km["p_market_yes"]
km["edge_no"] = km["p_model_no"] - km["p_market_no"]
km["market_price_available"] = km["p_market_yes"].notna()

print("\nUPDATED MODEL Ã¢â€ â€ KALSHI TABLE")
print("-" * 70)
print("Rows:", len(km))
print("Rows with market prices:", int(km["market_price_available"].sum()))
print("Rows missing prices    :", int((~km["market_price_available"]).sum()))

preview_cols = [
    "away_team",
    "home_team",
    "selected_team",
    "p_model_yes",
    "p_market_yes",
    "edge_yes",
    "edge_no",
    "market_price_available",
    "market_status_live",
    "liquidity_live",
    "volume_live",
]

display(km[preview_cols])

mlb_kalshi_with_model = km.copy()

Matched market tickers to refresh: 0

Live prices pulled:


""


  No live Kalshi prices to merge (0 tickers matched) -- keeping existing prices as-is.



UPDATED MODEL Ã¢â€ â€ KALSHI TABLE
----------------------------------------------------------------------
Rows: 0
Rows with market prices: 0
Rows missing prices    : 0


,away_team,home_team,selected_team,p_model_yes,p_market_yes,edge_yes,edge_no,market_price_available,market_status_live,liquidity_live,volume_live


In [47]:
# %%
# =========================================================
# CELL 24 Ã¢â‚¬â€ SAVE TODAY'S PROJECTION CARD FOR TRACKING
# Store today's top projection plays so you can grade them later
# =========================================================

import pandas as pd
from pathlib import Path

if "final_mlb_card" not in globals() or not isinstance(final_mlb_card, pd.DataFrame):
    raise ValueError("final_mlb_card not found. Run Cell 22 first.")

TRACK_DIR = Path(r"c:\Users\<user>\OneDrive\Documents\KAI Labs\sports\MLB\tracking")
TRACK_DIR.mkdir(parents=True, exist_ok=True)

today_stamp = pd.Timestamp.today().strftime("%Y-%m-%d")
outfile = TRACK_DIR / f"mlb_projection_card_{today_stamp}.csv"

card_to_save = final_mlb_card.copy()
card_to_save["card_date"] = today_stamp
card_to_save["card_type"] = "projection_social"

save_cols = [c for c in [
    "card_date",
    "card_type",
    "game_id",
    "game_datetime_utc",
    "away_team",
    "home_team",
    "projected_winner",
    "projected_loser",
    "projected_win_prob",
    "projected_win_pct",
    "strength_tier",
    "proj_margin_proxy",
    "official_bet",
    "social_bet",
    "bet_size",
    "status",
    "venue_name",
] if c in card_to_save.columns]

card_to_save[save_cols].to_csv(outfile, index=False)

print("Saved today's projection card to:")
print(outfile)

print("\nSaved rows:", len(card_to_save))
display(card_to_save[save_cols])

Saved today's projection card to:
c:\Users\<user>\OneDrive\Documents\KAI Labs\sports\MLB\tracking\mlb_projection_card_2026-10-10.csv

Saved rows: 1


,card_date,card_type,game_id,game_datetime_utc,away_team,home_team,projected_winner,projected_loser,projected_win_prob,strength_tier,proj_margin_proxy,official_bet,social_bet,bet_size,status,venue_name
0,2026-10-10,projection_social,849831,2026-10-11 00:00:00+00:00,Chicago White Sox,Cleveland Guardians,Cleveland Guardians,Chicago White Sox,0.554307,Lean,NaN,Cleveland Guardians ML,Cleveland Guardians ML | Win Prob 0.554 | 0.75...,0.75 UNIT,Scheduled,Progressive Field


In [48]:
# %%
# =========================================================
# CELL 25 (FULL REPLACEMENT) Ã¢â‚¬â€ GRADE TODAY'S PROJECTION CARD
# Robust version:
# - fetches final results from MLB StatsAPI
# - grades each pick as WIN / LOSS / PENDING
# - never crashes if optional columns are missing
# - safe score handling for final_score and run margins
# =========================================================

import pandas as pd
import numpy as np
import requests

print("=" * 60)
print("CELL 25 Ã¢â‚¬â€ GRADE TODAY'S MLB CARD")
print("=" * 60)

# ---------------------------------------------------------
# 0) validate source card
# ---------------------------------------------------------
if "final_mlb_card" not in globals() or not isinstance(final_mlb_card, pd.DataFrame):
    raise ValueError("final_mlb_card not found. Run the final MLB card cell first.")

grade_card = final_mlb_card.copy()

if grade_card.empty:
    raise ValueError("final_mlb_card exists but is empty.")

if "game_id" not in grade_card.columns:
    raise ValueError("game_id missing from final_mlb_card.")

print(f"Input card shape: {grade_card.shape}")
print(f"Input columns: {list(grade_card.columns)}")

# ---------------------------------------------------------
# 1) helper columns / fallbacks
# ---------------------------------------------------------
def first_existing_col(df, candidates):
    lower_map = {c.lower(): c for c in df.columns}
    for c in candidates:
        if c.lower() in lower_map:
            return lower_map[c.lower()]
    return None

away_col = first_existing_col(grade_card, ["away_team", "away", "visitor_team", "away_name"])
home_col = first_existing_col(grade_card, ["home_team", "home", "home_name"])
pick_col = first_existing_col(grade_card, ["projected_winner", "pick_team", "winner_pick", "model_winner"])
prob_col = first_existing_col(grade_card, ["projected_win_prob", "win_prob", "p_home_win", "model_win_prob"])

if away_col is None or home_col is None:
    raise ValueError("Could not find away/home team columns in final_mlb_card.")

if pick_col is None:
    raise ValueError("Could not find projected winner / pick column in final_mlb_card.")

# standardize working names without breaking original columns
graded_base = grade_card.copy()
graded_base["_away_team"] = graded_base[away_col].astype(str)
graded_base["_home_team"] = graded_base[home_col].astype(str)
graded_base["_pick_team"] = graded_base[pick_col].astype(str)

print("")
print("Detected columns:")
print(f" - away_col: {away_col}")
print(f" - home_col: {home_col}")
print(f" - pick_col: {pick_col}")
print(f" - prob_col: {prob_col}")

# ---------------------------------------------------------
# 2) helper: fetch one game's result
# ---------------------------------------------------------
SESSION = requests.Session()
SESSION.headers.update({"User-Agent": "Mozilla/5.0"})

def fetch_mlb_game_result(game_id):
    url = f"https://statsapi.mlb.com/api/v1.1/game/{int(game_id)}/feed/live"
    r = SESSION.get(url, timeout=30)
    r.raise_for_status()
    data = r.json()

    game_data = data.get("gameData", {}) or {}
    live_data = data.get("liveData", {}) or {}
    linescore = live_data.get("linescore", {}) or {}

    teams_game = game_data.get("teams", {}) or {}
    away_name = (teams_game.get("away") or {}).get("name")
    home_name = (teams_game.get("home") or {}).get("name")

    teams_ls = linescore.get("teams", {}) or {}
    away_runs = (teams_ls.get("away") or {}).get("runs")
    home_runs = (teams_ls.get("home") or {}).get("runs")

    detailed_state = (game_data.get("status") or {}).get("detailedState")
    abstract_state = (game_data.get("status") or {}).get("abstractGameState")

    winner = None
    try:
        away_runs_num = pd.to_numeric(away_runs, errors="coerce")
        home_runs_num = pd.to_numeric(home_runs, errors="coerce")

        if pd.notna(away_runs_num) and pd.notna(home_runs_num):
            if away_runs_num > home_runs_num:
                winner = away_name
            elif home_runs_num > away_runs_num:
                winner = home_name
    except Exception:
        pass

    return {
        "game_id": int(game_id),
        "api_away_team": away_name,
        "api_home_team": home_name,
        "away_runs": away_runs,
        "home_runs": home_runs,
        "actual_winner": winner,
        "api_status": detailed_state,
        "api_state_group": abstract_state,
    }

# ---------------------------------------------------------
# 3) fetch results safely
# ---------------------------------------------------------
game_ids = (
    pd.to_numeric(graded_base["game_id"], errors="coerce")
    .dropna()
    .astype("int64")
    .unique()
    .tolist()
)

print("")
print(f"Unique game_ids to grade: {len(game_ids)}")

results = []
for i, gid in enumerate(game_ids, start=1):
    try:
        results.append(fetch_mlb_game_result(gid))
    except Exception as e:
        results.append({
            "game_id": int(gid),
            "api_away_team": None,
            "api_home_team": None,
            "away_runs": np.nan,
            "home_runs": np.nan,
            "actual_winner": None,
            "api_status": f"ERROR: {e}",
            "api_state_group": None,
        })

    if i <= 5 or i % 25 == 0 or i == len(game_ids):
        print(f"Fetched [{i}/{len(game_ids)}]")

results_df = pd.DataFrame(results)
print(f"Results shape: {results_df.shape}")

# ---------------------------------------------------------
# 4) merge back to card
# ---------------------------------------------------------
graded = graded_base.merge(results_df, on="game_id", how="left").copy()
print(f"Merged graded shape: {graded.shape}")

if "card_date" not in graded.columns:
    graded["card_date"] = pd.Timestamp.today().strftime("%Y-%m-%d")

# ---------------------------------------------------------
# 5) safe numeric score cleaning
# ---------------------------------------------------------
graded["away_runs_num"] = pd.to_numeric(graded["away_runs"], errors="coerce")
graded["home_runs_num"] = pd.to_numeric(graded["home_runs"], errors="coerce")
score_mask = graded["away_runs_num"].notna() & graded["home_runs_num"].notna()

print("")
print("Missing counts after score cleaning:")
print(pd.Series({
    "missing_away_runs": int(graded["away_runs_num"].isna().sum()),
    "missing_home_runs": int(graded["home_runs_num"].isna().sum()),
    "valid_score_rows": int(score_mask.sum()),
}))

# ---------------------------------------------------------
# 6) safe grading
# ---------------------------------------------------------
def normalize_name(x):
    if pd.isna(x):
        return np.nan
    return str(x).strip().lower()

graded["_pick_team_norm"] = graded["_pick_team"].apply(normalize_name)
graded["_actual_winner_norm"] = graded["actual_winner"].apply(normalize_name)
graded["_away_team_norm"] = graded["_away_team"].apply(normalize_name)
graded["_home_team_norm"] = graded["_home_team"].apply(normalize_name)

graded["pick_won"] = np.where(
    graded["_actual_winner_norm"].notna(),
    graded["_pick_team_norm"] == graded["_actual_winner_norm"],
    np.nan
)

def result_label(x):
    if pd.isna(x):
        return "PENDING"
    return "WIN" if bool(x) else "LOSS"

graded["result_label"] = graded["pick_won"].apply(result_label)

# ---------------------------------------------------------
# 7) safe run margin for pick
# ---------------------------------------------------------
graded["run_margin_for_pick"] = np.nan

away_pick_mask = score_mask & (graded["_pick_team_norm"] == graded["_away_team_norm"])
home_pick_mask = score_mask & (graded["_pick_team_norm"] == graded["_home_team_norm"])

graded.loc[away_pick_mask, "run_margin_for_pick"] = (
    graded.loc[away_pick_mask, "away_runs_num"] - graded.loc[away_pick_mask, "home_runs_num"]
)

graded.loc[home_pick_mask, "run_margin_for_pick"] = (
    graded.loc[home_pick_mask, "home_runs_num"] - graded.loc[home_pick_mask, "away_runs_num"]
)

# if pick team does not exactly match away/home naming, leave as NaN safely

# ---------------------------------------------------------
# 8) safe final score string
# ---------------------------------------------------------
graded["final_score"] = None

graded.loc[score_mask, "final_score"] = (
    graded.loc[score_mask, "_away_team"].astype(str) + " "
    + graded.loc[score_mask, "away_runs_num"].round(0).astype(int).astype(str)
    + " - "
    + graded.loc[score_mask, "_home_team"].astype(str) + " "
    + graded.loc[score_mask, "home_runs_num"].round(0).astype(int).astype(str)
)

# ---------------------------------------------------------
# 9) summary
# ---------------------------------------------------------
finished = graded[graded["pick_won"].notna()].copy()

wins = int((finished["pick_won"] == True).sum()) if len(finished) else 0
losses = int((finished["pick_won"] == False).sum()) if len(finished) else 0
win_rate = wins / len(finished) if len(finished) else np.nan

avg_prob_all = graded[prob_col].mean() if (prob_col is not None and prob_col in graded.columns) else np.nan
avg_prob_finished = finished[prob_col].mean() if (prob_col is not None and prob_col in finished.columns and len(finished)) else np.nan

print("")
print("=" * 60)
print("EKO BETS Ã¢â‚¬â€ MLB CARD GRADE")
print("=" * 60)
print("Tracked plays :", len(graded))
print("Finished plays:", len(finished))
print("Wins          :", wins)
print("Losses        :", losses)
print("Win rate      :", round(win_rate, 4) if pd.notna(win_rate) else "N/A")
print("Avg proj prob :", round(avg_prob_all, 4) if pd.notna(avg_prob_all) else "N/A")
print("Avg prob done :", round(avg_prob_finished, 4) if pd.notna(avg_prob_finished) else "N/A")
print("Pending plays :", int(graded["pick_won"].isna().sum()))
print("")

# ---------------------------------------------------------
# 10) detailed printout
# ---------------------------------------------------------
for i, (_, r) in enumerate(graded.iterrows(), 1):
    print(f"#{i} {r['_away_team']} @ {r['_home_team']}")
    print(f"   Pick          : {r['_pick_team']}")
    print(f"   Actual winner : {r['actual_winner'] if pd.notna(r['actual_winner']) else 'N/A'}")
    print(f"   Final score   : {r['final_score'] if pd.notna(r['final_score']) else 'N/A'}")
    print(f"   Result        : {r['result_label']}")
    if prob_col is not None and prob_col in graded.columns and pd.notna(r.get(prob_col, np.nan)):
        try:
            print(f"   Projected prob: {float(r[prob_col]):.3f}")
        except Exception:
            print(f"   Projected prob: {r[prob_col]}")
    if "bet_size" in graded.columns and pd.notna(r.get("bet_size", np.nan)):
        print(f"   Size          : {r['bet_size']}")
    if pd.notna(r.get("run_margin_for_pick", np.nan)):
        try:
            print(f"   Margin for pick: {float(r['run_margin_for_pick']):.1f}")
        except Exception:
            print(f"   Margin for pick: {r['run_margin_for_pick']}")
    print("-" * 60)

# ---------------------------------------------------------
# 11) safe display
# ---------------------------------------------------------
display_cols = [
    "card_date",
    away_col,
    home_col,
    pick_col,
    "actual_winner",
    "away_runs",
    "home_runs",
    "final_score",
    "run_margin_for_pick",
    "pick_won",
    "result_label",
    prob_col,
    "bet_size",
    "official_bet",
    "social_bet",
    "status",
    "api_status",
]

display_cols = [c for c in display_cols if c is not None and c in graded.columns]

print("\nDetailed graded table:")
display(graded[display_cols].copy())

# ---------------------------------------------------------
# 12) save output
# ---------------------------------------------------------
graded_mlb_card = graded.copy()

print("")
print("Created: graded_mlb_card")
print(f"graded_mlb_card shape: {graded_mlb_card.shape}")

CELL 25 Ã¢â‚¬â€ GRADE TODAY'S MLB CARD
Input card shape: (1, 103)
Input columns: ['game_id', 'game_datetime_utc', 'away_team', 'home_team', 'away_team_id', 'home_team_id', 'venue_name', 'status', 'game_type', 'series_game_number', 'games_in_series', 'series_description', 'away_team_norm', 'home_team_norm', 'away_starter_name_espn', 'away_starter_id_espn', 'home_starter_name_espn', 'home_starter_id_espn', 'starter_source_espn', 'espn_event_id', 'away_starter_name', 'away_starter_id', 'home_starter_name', 'home_starter_id', 'starter_source', 'missing_away_starter', 'missing_home_starter', 'missing_any_starter', 'away_runs_for_avg_10', 'away_runs_against_avg_10', 'away_margin_avg_10', 'away_win_rate_10', 'away_runs_for_avg_5', 'away_runs_against_avg_5', 'away_margin_avg_5', 'away_win_rate_5', 'away_rest_days', 'away_runs_for_avg_road_10', 'away_runs_against_avg_road_10', 'away_last_game_date', 'home_runs_for_avg_10', 'home_runs_against_avg_10', 'home_margin_avg_10', 'home_win_rate_10', '

Fetched [1/1]
Results shape: (1, 8)
Merged graded shape: (1, 113)

Missing counts after score cleaning:
missing_away_runs    1
missing_home_runs    1
valid_score_rows     0
dtype: int64

EKO BETS Ã¢â‚¬â€ MLB CARD GRADE
Tracked plays : 1
Finished plays: 0
Wins          : 0
Losses        : 0
Win rate      : N/A
Avg proj prob : 0.5543
Avg prob done : N/A
Pending plays : 1

#1 Chicago White Sox @ Cleveland Guardians
   Pick          : Cleveland Guardians
   Actual winner : N/A
   Final score   : N/A
   Result        : PENDING
   Projected prob: 0.554
   Size          : 0.75 UNIT
------------------------------------------------------------

Detailed graded table:


,card_date,away_team,home_team,projected_winner,actual_winner,away_runs,home_runs,final_score,run_margin_for_pick,pick_won,result_label,projected_win_prob,bet_size,official_bet,social_bet,status,api_status
0,2026-10-10,Chicago White Sox,Cleveland Guardians,Cleveland Guardians,None,None,None,None,NaN,NaN,PENDING,0.554307,0.75 UNIT,Cleveland Guardians ML,Cleveland Guardians ML | Win Prob 0.554 | 0.75...,Scheduled,Scheduled



Created: graded_mlb_card
graded_mlb_card shape: (1, 124)


In [49]:
# %%
# =========================================================
# CELL 22 (REPLACEMENT) Ã¢â‚¬â€ STRICTER FINAL CARD BUILDER
# Real betting card if live prices exist
# Otherwise builds a tighter projection/social card using
# a stronger confidence cutoff
# ROBUST TO MISSING mlb_projection_board
# =========================================================

import pandas as pd
import numpy as np

print("=" * 70)
print("CELL 22 (REPLACEMENT) Ã¢â‚¬â€ STRICTER FINAL CARD BUILDER")
print("=" * 70)

# ---------------------------------------------------------
# helpers
# ---------------------------------------------------------
def first_existing_df(candidates):
    for name in candidates:
        if name in globals() and isinstance(globals()[name], pd.DataFrame):
            df = globals()[name]
            if len(df) > 0:
                print(f"Using board source: {name}  shape={df.shape}")
                return df.copy(), name
    return None, None

def first_existing_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

def ensure_numeric(series_or_df_col):
    return pd.to_numeric(series_or_df_col, errors="coerce")

def ensure_default(name, default_value):
    if name not in globals():
        globals()[name] = default_value
        print(f"{name} not found. Using default = {default_value}")

# ---------------------------------------------------------
# defaults if not already in memory
# ---------------------------------------------------------
ensure_default("MIN_CONF", 0.55)
ensure_default("MIN_EDGE", 0.05)
ensure_default("MIN_PRICE", 0.10)
ensure_default("MAX_PRICE", 0.75)

# ---------------------------------------------------------
# locate best available projection-style board
# ---------------------------------------------------------
proj_candidates = [
    "scored_mlb_board",
    "mlb_scored_board",
    "mlb_projection_board",
    "today_mlb_board",
    "final_mlb_board",
    "mlb_board_final",
    "board_with_predictions",
]

proj, proj_source_name = first_existing_df(proj_candidates)

if proj is None:
    raise ValueError(
        "No usable projection-style MLB board found in memory.\n"
        "Expected one of: mlb_projection_board, scored_mlb_board, mlb_scored_board, "
        "today_mlb_board, final_mlb_board, mlb_board_final, board_with_predictions."
    )

# ---------------------------------------------------------
# resolve projection columns safely
# ---------------------------------------------------------
away_col = first_existing_col(proj, ["away_team", "away_team_name", "team_away"])
home_col = first_existing_col(proj, ["home_team", "home_team_name", "team_home"])

winner_col = first_existing_col(proj, [
    "projected_winner",
    "projected_winner_new",
    "predicted_winner",
    "pred_winner",
    "winner_pick",
    "model_winner",
])

win_prob_col = first_existing_col(proj, [
    "projected_win_prob",
    "win_prob",
    "best_win_prob",
    "model_win_prob",
])

strength_col = first_existing_col(proj, [
    "strength_tier",
    "play_tier",
    "tier",
])

margin_col = first_existing_col(proj, [
    "proj_margin_proxy",
    "projected_margin_proxy",
    "margin_proxy",
    "pred_margin",
    "projected_margin",
])

datetime_col = first_existing_col(proj, [
    "game_datetime_utc",
    "start_time",
    "game_time_utc",
    "game_date",
    "date",
])

p_home_col = first_existing_col(proj, [
    "p_home_win",
    "p_home_win_new",
    "pred_home_win_prob",
    "projected_home_win_prob",
    "home_win_prob",
    "home_prob",
])

p_away_col = first_existing_col(proj, [
    "p_away_win",
    "p_away_win_new",
    "pred_away_win_prob",
    "projected_away_win_prob",
    "away_win_prob",
    "away_prob",
])

if away_col is None or home_col is None:
    raise ValueError(
        "Could not resolve away/home team columns from the projection board.\n"
        f"Available columns:\n{list(proj.columns)}"
    )

if away_col != "away_team":
    proj["away_team"] = proj[away_col]
if home_col != "home_team":
    proj["home_team"] = proj[home_col]

# winner fallback from p_home/p_away
if winner_col is None and p_home_col is not None and p_away_col is not None:
    proj["projected_winner_fallback"] = np.where(
        ensure_numeric(proj[p_home_col]) >= ensure_numeric(proj[p_away_col]),
        proj["home_team"],
        proj["away_team"]
    )
    winner_col = "projected_winner_fallback"

# win prob fallback from projected winner + p_home/p_away
if win_prob_col is None and winner_col is not None and p_home_col is not None and p_away_col is not None:
    proj["projected_win_prob_fallback"] = np.where(
        proj[winner_col] == proj["home_team"],
        ensure_numeric(proj[p_home_col]),
        ensure_numeric(proj[p_away_col])
    )
    win_prob_col = "projected_win_prob_fallback"

if winner_col is None or win_prob_col is None:
    raise ValueError(
        "Could not resolve projected_winner / projected_win_prob columns.\n"
        f"Board source used: {proj_source_name}\n"
        f"Available columns:\n{list(proj.columns)}"
    )

proj["projected_winner"] = proj[winner_col]
proj["projected_win_prob"] = ensure_numeric(proj[win_prob_col])

if strength_col is not None:
    proj["strength_tier"] = proj[strength_col]
else:
    proj["strength_tier"] = np.nan

if margin_col is not None:
    proj["proj_margin_proxy"] = ensure_numeric(proj[margin_col])
else:
    proj["proj_margin_proxy"] = np.nan

if datetime_col is not None:
    proj["sort_datetime"] = proj[datetime_col]
else:
    proj["sort_datetime"] = np.nan

print("\nResolved projection columns:")
print(f" - board source   : {proj_source_name}")
print(f" - away_col       : {away_col}")
print(f" - home_col       : {home_col}")
print(f" - winner_col     : {winner_col}")
print(f" - win_prob_col   : {win_prob_col}")
print(f" - strength_col   : {strength_col}")
print(f" - margin_col     : {margin_col}")
print(f" - datetime_col   : {datetime_col}")

# ---------------------------------------------------------
# stricter projection posting cutoffs
# ---------------------------------------------------------
STRICT_SOCIAL_MIN_PROB = 0.58
FALLBACK_SOCIAL_MIN_PROB = 0.57
MAX_SOCIAL_PLAYS = 5

bet_card = None

# ---------------------------------------------------------
# try real betting card first if market prices exist
# ---------------------------------------------------------
if "mlb_kalshi_with_model" in globals() and isinstance(mlb_kalshi_with_model, pd.DataFrame):
    km = mlb_kalshi_with_model.copy()

    if len(km):
        # numeric cleanup
        for c in [
            "p_model_yes", "p_model_no", "p_market_yes", "p_market_no",
            "yes_ask", "no_ask", "yes_bid", "no_bid", "last_price",
            "edge_yes", "edge_no"
        ]:
            if c in km.columns:
                km[c] = ensure_numeric(km[c])

        # safe fallbacks
        if "recommended_side" not in km.columns:
            if "selected_team" in km.columns and "recommended_team" in km.columns:
                km["recommended_side"] = np.where(
                    km["selected_team"].astype(str).str.strip().str.lower() ==
                    km["recommended_team"].astype(str).str.strip().str.lower(),
                    "YES",
                    "NO"
                )
            else:
                km["recommended_side"] = "YES"

        if "recommended_team" not in km.columns:
            if "projected_winner" in km.columns:
                km["recommended_team"] = km["projected_winner"]
            else:
                km["recommended_team"] = np.nan

        km["bet_side"] = np.where(km["recommended_side"].astype(str).str.upper() == "YES", "YES", "NO")
        km["bet_team"] = km["recommended_team"]

        if "p_model_bet" not in km.columns:
            km["p_model_bet"] = np.where(
                km["bet_side"] == "YES",
                km.get("p_model_yes", np.nan),
                km.get("p_model_no", np.nan)
            )

        if "p_market_bet" not in km.columns:
            km["p_market_bet"] = np.where(
                km["bet_side"] == "YES",
                km.get("p_market_yes", np.nan),
                km.get("p_market_no", np.nan)
            )

        if "edge_bet" not in km.columns:
            km["edge_bet"] = km["p_model_bet"] - km["p_market_bet"]

        km_live = km[km["p_market_bet"].notna()].copy()

        # --- repeat-team cooldown + penalty ---
        # Hard block: lost in past 5 days -> removed from card entirely
        # Hard block: 2+ losses in past 14 days AND last loss within 10 days -> removed
        # Soft penalty: any pick (win or loss) in past 5 days -> +0.02 edge threshold
        import pandas as _pd_pen
        from pathlib import Path as _Path
        _TRACK_FILE = _Path(r"c:\Users\<user>\OneDrive\Documents\KAI Labs\sports\MLB\tracking\mlb_card_history.csv")
        _hard_blocked_teams = set()
        _soft_penalty_teams = set()
        if _TRACK_FILE.exists():
            try:
                _hist = _pd_pen.read_csv(_TRACK_FILE, parse_dates=["card_date"])
                _today_ts = _pd_pen.Timestamp.today().normalize()
                _cut5  = _today_ts - _pd_pen.Timedelta(days=5)
                _cut10 = _today_ts - _pd_pen.Timedelta(days=10)
                _cut14 = _today_ts - _pd_pen.Timedelta(days=14)
                _recent5  = _hist[_hist["card_date"] >= _cut5]
                _recent14 = _hist[_hist["card_date"] >= _cut14]
                # Hard block 1: 2+ losses in last 5 days OR loss in last 2 days
                _cut2 = _today_ts - _pd_pen.Timedelta(days=2)
                _losses5 = _recent5[_recent5["pick_won"] == False]
                _losses2 = _hist[(_hist["card_date"] >= _cut2) & (_hist["pick_won"] == False)]
                for _t in _losses2["projected_winner"].dropna():
                    _hard_blocked_teams.add(str(_t).strip().lower())
                for _t, _grp in _losses5.groupby("projected_winner"):
                    if len(_grp) >= 2:
                        _hard_blocked_teams.add(str(_t).strip().lower())
                # Hard block 2: 2+ losses in last 14 days, most recent within 10 days
                _losses14 = _recent14[_recent14["pick_won"] == False]
                for _t, _grp in _losses14.groupby("projected_winner"):
                    if len(_grp) >= 2:
                        _last = _grp["card_date"].max()
                        if _last >= _cut10:
                            _hard_blocked_teams.add(str(_t).strip().lower())
                # Soft penalty: any pick in last 5 days (not already hard blocked)
                for _t in _recent5["projected_winner"].dropna():
                    _tn = str(_t).strip().lower()
                    if _tn not in _hard_blocked_teams:
                        _soft_penalty_teams.add(_tn)
                if _hard_blocked_teams:
                    print(f"  HARD BLOCKED (cooldown): {_hard_blocked_teams}")
                if _soft_penalty_teams:
                    print(f"  Soft penalty (repeat pick, +0.02 edge): {_soft_penalty_teams}")
            except Exception as _pen_err:
                print(f"  WARN: could not load penalty history ({_pen_err})")
        # Remove hard-blocked teams before filter
        if _hard_blocked_teams and len(km_live):
            _pre = len(km_live)
            km_live = km_live[~km_live["bet_team"].apply(
                lambda _t: str(_t).strip().lower() in _hard_blocked_teams
            )].copy()
            print(f"  Hard block removed {_pre - len(km_live)} play(s) from card")

        def _get_edge_threshold(team_name):
            return MIN_EDGE + 0.02 if str(team_name).strip().lower() in _soft_penalty_teams else MIN_EDGE

        if len(km_live):
            km_live = km_live[
                (km_live["p_model_bet"] >= MIN_CONF) &
                (km_live.apply(lambda r: r["edge_bet"] >= _get_edge_threshold(r.get("bet_team", "")), axis=1)) &
                (km_live["p_market_bet"] >= MIN_PRICE) &
                (km_live["p_market_bet"] <= MAX_PRICE)
            ].copy()

            if len(km_live):
                km_live["matchup_key"] = (
                    km_live["away_team"].astype(str).str.lower().str.strip()
                    + " @ " +
                    km_live["home_team"].astype(str).str.lower().str.strip()
                )

                km_live = (
                    km_live.sort_values(
                        ["matchup_key", "edge_bet", "p_model_bet"],
                        ascending=[True, False, False]
                    )
                    .drop_duplicates(subset=["matchup_key"], keep="first")
                    .sort_values(["edge_bet", "p_model_bet"], ascending=[False, False])
                    .reset_index(drop=True)
                )

                def size_from_edge(edge, conf, mkt_prob=None):
                    if pd.isna(edge) or pd.isna(conf):
                        return "0.5 UNIT"
                    # Underdog cap: market <50% and model <62% -> max 0.5u
                    # Underdogs at borderline confidence are sized down regardless of edge
                    if mkt_prob is not None and not pd.isna(mkt_prob):
                        if mkt_prob < 0.50 and conf < 0.62:
                            return "0.5 UNIT"
                    if edge >= 0.14 and conf >= 0.62:
                        return "1.5 UNIT"
                    if edge >= 0.10 and conf >= 0.59:
                        return "1 UNIT"
                    if edge >= 0.07 and conf >= 0.565:
                        return "0.75 UNIT"
                    return "0.5 UNIT"

                km_live["bet_size"] = [
                    size_from_edge(e, c, m)
                    for e, c, m in zip(km_live["edge_bet"], km_live["p_model_bet"], km_live["p_market_bet"])
                ]

                km_live["official_bet"] = [
                    f"Bet {side} Ã¢â‚¬â€ {team}"
                    for side, team in zip(km_live["bet_side"], km_live["bet_team"])
                ]

                km_live["social_bet"] = [
                    f"{team} | {side} @ {price:.3f} | Edge {edge:.3f} | {size}"
                    for team, side, price, edge, size in zip(
                        km_live["bet_team"],
                        km_live["bet_side"],
                        km_live["p_market_bet"],
                        km_live["edge_bet"],
                        km_live["bet_size"]
                    )
                ]

                bet_card = km_live.copy()

# ---------------------------------------------------------
# fallback: projection-only social card
# now much stricter than before
# ---------------------------------------------------------
if bet_card is None or len(bet_card) == 0:
    # Re-score if projected_win_prob is all NaN
    if proj["projected_win_prob"].isna().all() and "model" in globals() and "feature_list" in globals():
        print("projected_win_prob all-NaN â€” re-scoring board with model directly...")
        try:
            _feat = [c for c in feature_list if c in proj.columns]
            _X = proj[_feat].copy().replace([np.inf, -np.inf], np.nan)
            for _c in _X.columns:
                if _X[_c].isna().any():
                    _med = _X[_c].median(skipna=True)
                    _X[_c] = _X[_c].fillna(_med if not pd.isna(_med) else 0.0)
            _preds = np.clip(pd.to_numeric(model.predict_proba(_X)[:, 1], errors="coerce"), 0.0, 1.0)
            proj["pred_home_win_prob"] = _preds
            proj["pred_away_win_prob"] = 1.0 - _preds
            proj["projected_win_prob"] = np.where(_preds >= 0.5, _preds, 1.0 - _preds)
            proj["projected_winner"] = np.where(_preds >= 0.5, proj["home_team"], proj["away_team"])
            print(f"  Re-scored {len(proj)} rows. Valid: {proj['projected_win_prob'].notna().sum()}")
        except Exception as _e:
            print(f"  Re-scoring failed: {_e}")
            raise ValueError("No usable projected_win_prob values found in the selected board.") from _e

    social_card = proj.copy()
    social_card = social_card[social_card["projected_win_prob"].notna()].copy()

    if len(social_card) == 0:
        raise ValueError("No usable projected_win_prob values found in the selected board.")

    # first try the strict cutoff
    strict_card = social_card[
        social_card["projected_win_prob"] >= STRICT_SOCIAL_MIN_PROB
    ].copy()

    if len(strict_card) > 0:
        social_card = strict_card
    else:
        fallback_card = social_card[
            social_card["projected_win_prob"] >= FALLBACK_SOCIAL_MIN_PROB
        ].copy()

        if len(fallback_card) > 0:
            social_card = fallback_card
        else:
            social_card = (
                social_card.sort_values(
                    ["projected_win_prob", "sort_datetime"],
                    ascending=[False, True]
                )
                .head(1)
                .copy()
            )

    social_card = (
        social_card.sort_values(
            ["projected_win_prob", "sort_datetime"],
            ascending=[False, True]
        )
        .head(MAX_SOCIAL_PLAYS)
        .reset_index(drop=True)
    )

    def size_from_prob(p):
        if pd.isna(p):
            return "0.25 UNIT"
        if p >= 0.60:
            return "1 UNIT"
        if p >= 0.58:
            return "0.75 UNIT"
        if p >= 0.57:
            return "0.5 UNIT"
        return "0.25 UNIT"

    social_card["play_type"] = "Projection Only"
    social_card["bet_size"] = social_card["projected_win_prob"].apply(size_from_prob)

    social_card["official_bet"] = [
        f"{winner} ML"
        for winner in social_card["projected_winner"]
    ]

    social_card["social_bet"] = [
        f"{winner} ML | Win Prob {prob:.3f} | {size}"
        for winner, prob, size in zip(
            social_card["projected_winner"],
            social_card["projected_win_prob"],
            social_card["bet_size"]
        )
    ]

    print("\n==================================================")
    print("EKO BETS Ã¢â‚¬â€ STRICT MLB PROJECTION / SOCIAL CARD")
    print("==================================================")
    print("No usable live market prices found.")
    print("Built stricter projection-only social card instead.\n")

    for i, (_, r) in enumerate(social_card.iterrows(), 1):
        print(f"#{i} {r['away_team']} @ {r['home_team']}")
        print(f"   Projected winner: {r['projected_winner']}")
        print(f"   Win probability : {r['projected_win_prob']:.3f}")
        print(f"   Strength tier   : {r['strength_tier']}")
        if pd.notna(r["proj_margin_proxy"]):
            print(f"   Margin proxy    : {r['proj_margin_proxy']:+.1f}")
        else:
            print("   Margin proxy    : N/A")
        print(f"   Play            : {r['official_bet']}")
        print(f"   Size            : {r['bet_size']}")
        print("-" * 60)

    print("\n==================================================")
    print("EKO BETS Ã¢â‚¬â€ STRICT MLB SOCIAL MEDIA CARD")
    print("==================================================")
    for i, (_, r) in enumerate(social_card.iterrows(), 1):
        print(f"{i}. {r['social_bet']}")

    final_mlb_card = social_card.copy()

else:
    print("\n==================================================")
    print("EKO BETS Ã¢â‚¬â€ FINAL MLB BETTING CARD")
    print("==================================================")

    for i, (_, r) in enumerate(bet_card.iterrows(), 1):
        print(f"#{i} {r['away_team']} @ {r['home_team']}")
        print(f"   Projected winner : {r.get('projected_winner', r.get('bet_team', 'N/A'))}")
        print(f"   Bet              : {r['official_bet']}")
        print(f"   Entry            : {r['p_market_bet']:.3f}")
        print(f"   Model for bet    : {r['p_model_bet']:.3f}")
        print(f"   Market for bet   : {r['p_market_bet']:.3f}")
        print(f"   Edge             : {r['edge_bet']:.3f}")
        print(f"   Size             : {r['bet_size']}")
        print("-" * 60)

    print("\n==================================================")
    print("EKO BETS Ã¢â‚¬â€ MLB SOCIAL MEDIA CARD")
    print("==================================================")
    for i, (_, r) in enumerate(bet_card.iterrows(), 1):
        print(f"{i}. {r['social_bet']}")

    final_mlb_card = bet_card.copy()

# ---------------------------------------------------------
# save in memory
# ---------------------------------------------------------
globals()["final_mlb_card"] = final_mlb_card.copy()

print("\nCreated in memory:")
print(" - final_mlb_card")
print("Done.")

CELL 22 (REPLACEMENT) Ã¢â‚¬â€ STRICTER FINAL CARD BUILDER
Using board source: scored_mlb_board  shape=(1, 97)

Resolved projection columns:
 - board source   : scored_mlb_board
 - away_col       : away_team
 - home_col       : home_team
 - winner_col     : projected_winner
 - win_prob_col   : projected_win_prob
 - strength_col   : strength_tier
 - margin_col     : None
 - datetime_col   : game_datetime_utc
projected_win_prob all-NaN â€” re-scoring board with model directly...
  Re-scored 1 rows. Valid: 1

EKO BETS Ã¢â‚¬â€ STRICT MLB PROJECTION / SOCIAL CARD
No usable live market prices found.
Built stricter projection-only social card instead.

#1 Chicago White Sox @ Cleveland Guardians
   Projected winner: Cleveland Guardians
   Win probability : 0.554
   Strength tier   : Lean
   Margin proxy    : N/A
   Play            : Cleveland Guardians ML
   Size            : 0.25 UNIT
------------------------------------------------------------

EKO BETS Ã¢â‚¬â€ STRICT MLB SOCIAL MEDIA CARD

In [50]:
# %%
# =========================================================
# CELL 27 Ã¢â‚¬â€ APPEND GRADED CARD TO HISTORICAL TRACKER
# Builds a running MLB results file across days
# =========================================================

import pandas as pd
from pathlib import Path

if "graded_mlb_card" not in globals() or not isinstance(graded_mlb_card, pd.DataFrame):
    raise ValueError("graded_mlb_card not found. Run Cell 25 first.")

TRACK_DIR = Path(r"c:\Users\<user>\OneDrive\Documents\KAI Labs\sports\MLB\tracking")
TRACK_DIR.mkdir(parents=True, exist_ok=True)

tracker_file = TRACK_DIR / "mlb_card_history.csv"

hist = graded_mlb_card.copy()

# add missing columns safely
if "card_date" not in hist.columns:
    hist["card_date"] = pd.Timestamp.today().strftime("%Y-%m-%d")

if "card_type" not in hist.columns:
    hist["card_type"] = "projection_social"

keep_cols = [
    "card_date",
    "card_type",
    "game_id",
    "away_team",
    "home_team",
    "projected_winner",
    "actual_winner",
    "away_runs",
    "home_runs",
    "final_score",
    "pick_won",
    "result_label",
    "projected_win_prob",
    "bet_size",
    "official_bet",
    "social_bet",
    "status",
    "api_status",
]

keep_cols = [c for c in keep_cols if c in hist.columns]
hist = hist[keep_cols].copy()

# ---------------------------------------------------------
# load existing tracker if it exists
# ---------------------------------------------------------
if tracker_file.exists():
    old = pd.read_csv(tracker_file)
    combined = pd.concat([old, hist], ignore_index=True)
else:
    combined = hist.copy()

# dedupe by card_date + game_id + projected_winner
dedupe_cols = [c for c in ["card_date", "game_id", "projected_winner"] if c in combined.columns]
combined = combined.drop_duplicates(subset=dedupe_cols, keep="last").reset_index(drop=True)

combined.to_csv(tracker_file, index=False)

print("Saved historical tracker to:")
print(tracker_file)

print("\nHistorical tracker shape:", combined.shape)

# quick summary
finished = combined[combined["pick_won"].notna()].copy()
if len(finished):
    wins = int((finished["pick_won"] == True).sum())
    losses = int((finished["pick_won"] == False).sum())
    win_rate = wins / len(finished)
    print("\nALL-TIME MLB CARD RECORD")
    print(f"Wins   : {wins}")
    print(f"Losses : {losses}")
    print(f"Win rate: {win_rate:.4f}")
else:
    print("\nNo finished historical plays yet.")

print("\nLatest rows:")
display(combined.tail(20))

mlb_card_history = combined.copy()

# Alert for unresolved PENDING rows â€” re-run grading cell after games finish
_today_str = pd.Timestamp.today().strftime("%Y-%m-%d")
_pending = combined[combined["pick_won"].isna() & (combined["card_date"] == _today_str)]
if len(_pending) > 0:
    print(f"WARNING: {len(_pending)} PENDING row(s) still unresolved from today:")
    for _, _pr in _pending.iterrows():
        print(f"  game_id={_pr.get('game_id','')} | {_pr.get('away_team','')} @ {_pr.get('home_team','')} | pick={_pr.get('projected_winner','')}")
    print("  â†’ Re-run the grading cell after games finish to clear PENDING rows.")
else:
    print("All today's rows resolved (no PENDING).")

Saved historical tracker to:
c:\Users\<user>\OneDrive\Documents\KAI Labs\sports\MLB\tracking\mlb_card_history.csv

Historical tracker shape: (1234, 18)

ALL-TIME MLB CARD RECORD
Wins   : 163
Losses : 137
Win rate: 0.5433

Latest rows:


,card_date,card_type,game_id,away_team,home_team,projected_winner,actual_winner,away_runs,home_runs,final_score,pick_won,result_label,projected_win_prob,bet_size,official_bet,social_bet,status,api_status
1214,2026-09-29,projection_social,849845,Philadelphia Phillies,Atlanta Braves,Atlanta Braves,NaN,1.0,1.0,Philadelphia Phillies 1 - Atlanta Braves 1,NaN,PENDING,0.679538,1 UNIT,Atlanta Braves ML,Atlanta Braves ML | Win Prob 0.680 | 1 UNIT,In Progress,In Progress
1215,2026-09-29,projection_social,849849,Chicago White Sox,Houston Astros,Houston Astros,NaN,0.0,0.0,Chicago White Sox 0 - Houston Astros 0,NaN,PENDING,0.677634,1 UNIT,Houston Astros ML,Houston Astros ML | Win Prob 0.678 | 1 UNIT,Pre-Game,Pre-Game
1216,2026-09-29,projection_social,849843,Chicago Cubs,San Diego Padres,San Diego Padres,NaN,NaN,NaN,NaN,NaN,PENDING,0.639592,1 UNIT,San Diego Padres ML,San Diego Padres ML | Win Prob 0.640 | 1 UNIT,Scheduled,Scheduled
1217,2026-10-01,projection_social,849844,Philadelphia Phillies,Atlanta Braves,Atlanta Braves,NaN,NaN,NaN,NaN,NaN,PENDING,0.536185,0.5 UNIT,Atlanta Braves ML,Atlanta Braves ML | Win Prob 0.536 | 0.5 UNIT,Scheduled,Scheduled
1218,2026-10-03,projection_social,849828,Atlanta Braves,Los Angeles Dodgers,Los Angeles Dodgers,NaN,NaN,NaN,NaN,NaN,PENDING,0.577431,0.75 UNIT,Los Angeles Dodgers ML,Los Angeles Dodgers ML | Win Prob 0.577 | 0.75...,Scheduled,Scheduled
1219,2026-10-03,projection_social,849829,Chicago White Sox,Cleveland Guardians,Cleveland Guardians,NaN,NaN,NaN,NaN,NaN,PENDING,0.556138,0.75 UNIT,Cleveland Guardians ML,Cleveland Guardians ML | Win Prob 0.556 | 0.75...,Scheduled,Scheduled
1220,2026-10-03,projection_social,849830,San Diego Padres,Milwaukee Brewers,Milwaukee Brewers,NaN,NaN,NaN,NaN,NaN,PENDING,0.553911,0.75 UNIT,Milwaukee Brewers ML,Milwaukee Brewers ML | Win Prob 0.554 | 0.75 UNIT,Scheduled,Scheduled
1221,2026-10-03,projection_social,849835,New York Yankees,Tampa Bay Rays,Tampa Bay Rays,NaN,NaN,NaN,NaN,NaN,PENDING,0.543277,0.5 UNIT,Tampa Bay Rays ML,Tampa Bay Rays ML | Win Prob 0.543 | 0.5 UNIT,Scheduled,Scheduled
1222,2026-10-04,projection_social,849823,Atlanta Braves,Los Angeles Dodgers,Los Angeles Dodgers,NaN,0.0,0.0,Atlanta Braves 0 - Los Angeles Dodgers 0,NaN,PENDING,0.556953,0.75 UNIT,Los Angeles Dodgers ML,Los Angeles Dodgers ML | Win Prob 0.557 | 0.75...,Pre-Game,Pre-Game
1223,2026-10-04,projection_social,849825,San Diego Padres,Milwaukee Brewers,Milwaukee Brewers,San Diego Padres,1.0,0.0,San Diego Padres 1 - Milwaukee Brewers 0,0.0,LOSS,0.527340,0.5 UNIT,Milwaukee Brewers ML,Milwaukee Brewers ML | Win Prob 0.527 | 0.5 UNIT,In Progress,In Progress


  game_id=849831 | Chicago White Sox @ Cleveland Guardians | pick=Cleveland Guardians
  â†’ Re-run the grading cell after games finish to clear PENDING rows.


In [51]:
# %%
# =========================================================
# CELL 28 Ã¢â‚¬â€ MLB HISTORY DASHBOARD
# Summarize all-time record by confidence bucket and threshold
# =========================================================

import pandas as pd
import numpy as np

if "mlb_card_history" not in globals() or not isinstance(mlb_card_history, pd.DataFrame):
    raise ValueError("mlb_card_history not found. Run Cell 27 first.")

hist = mlb_card_history.copy()

if "projected_win_prob" not in hist.columns:
    raise ValueError("projected_win_prob missing from mlb_card_history.")

if "pick_won" not in hist.columns:
    raise ValueError("pick_won missing from mlb_card_history.")

hist = hist[hist["pick_won"].notna()].copy()

if len(hist) == 0:
    raise ValueError("No finished historical plays in mlb_card_history yet.")

hist["pick_won_num"] = hist["pick_won"].astype(int)
hist["projected_win_prob"] = pd.to_numeric(hist["projected_win_prob"], errors="coerce")

# ---------------------------------------------------------
# confidence buckets
# ---------------------------------------------------------
bins = [0.00, 0.54, 0.57, 0.58, 0.60, 0.65, 1.01]
labels = [
    "<0.54",
    "0.54-0.57",
    "0.57-0.58",
    "0.58-0.60",
    "0.60-0.65",
    "0.65+",
]

hist["prob_bucket"] = pd.cut(
    hist["projected_win_prob"],
    bins=bins,
    labels=labels,
    right=False
)

bucket_summary = (
    hist.groupby("prob_bucket", dropna=False)
    .agg(
        plays=("pick_won_num", "size"),
        wins=("pick_won_num", "sum"),
        avg_proj_prob=("projected_win_prob", "mean"),
    )
    .reset_index()
)

bucket_summary["losses"] = bucket_summary["plays"] - bucket_summary["wins"]
bucket_summary["win_rate"] = bucket_summary["wins"] / bucket_summary["plays"]

bucket_summary = bucket_summary[
    ["prob_bucket", "plays", "wins", "losses", "win_rate", "avg_proj_prob"]
].copy()

print("==================================================")
print("EKO BETS Ã¢â‚¬â€ MLB HISTORY BY CONFIDENCE BUCKET")
print("==================================================")
display(bucket_summary)

# ---------------------------------------------------------
# threshold study
# ---------------------------------------------------------
thresholds = [0.54, 0.55, 0.56, 0.57, 0.58, 0.59, 0.60]

threshold_rows = []
for th in thresholds:
    sub = hist[hist["projected_win_prob"] >= th].copy()
    plays = len(sub)
    wins = int(sub["pick_won_num"].sum()) if plays else 0
    losses = plays - wins
    win_rate = wins / plays if plays else np.nan
    avg_prob = sub["projected_win_prob"].mean() if plays else np.nan

    threshold_rows.append({
        "min_prob_threshold": th,
        "plays": plays,
        "wins": wins,
        "losses": losses,
        "win_rate": win_rate,
        "avg_proj_prob": avg_prob,
    })

threshold_summary = pd.DataFrame(threshold_rows)

print("\n==================================================")
print("EKO BETS Ã¢â‚¬â€ MLB HISTORY THRESHOLD STUDY")
print("==================================================")
display(threshold_summary)

# ---------------------------------------------------------
# by day
# ---------------------------------------------------------
if "card_date" in hist.columns:
    daily_summary = (
        hist.groupby("card_date")
        .agg(
            plays=("pick_won_num", "size"),
            wins=("pick_won_num", "sum"),
            avg_proj_prob=("projected_win_prob", "mean"),
        )
        .reset_index()
    )
    daily_summary["losses"] = daily_summary["plays"] - daily_summary["wins"]
    daily_summary["win_rate"] = daily_summary["wins"] / daily_summary["plays"]

    print("\n==================================================")
    print("EKO BETS Ã¢â‚¬â€ MLB DAILY HISTORY")
    print("==================================================")
    display(daily_summary)

# ---------------------------------------------------------
# overall summary
# ---------------------------------------------------------
wins = int(hist["pick_won_num"].sum())
losses = len(hist) - wins
win_rate = wins / len(hist)
avg_prob = hist["projected_win_prob"].mean()

print("\n==================================================")
print("EKO BETS Ã¢â‚¬â€ MLB ALL-TIME SUMMARY")
print("==================================================")
print(f"Plays         : {len(hist)}")
print(f"Wins          : {wins}")
print(f"Losses        : {losses}")
print(f"Win rate      : {win_rate:.4f}")
print(f"Avg proj prob : {avg_prob:.4f}")

mlb_history_bucket_summary = bucket_summary.copy()
mlb_history_threshold_summary = threshold_summary.copy()
if "card_date" in hist.columns:
    mlb_history_daily_summary = daily_summary.copy()

EKO BETS Ã¢â‚¬â€ MLB HISTORY BY CONFIDENCE BUCKET


,prob_bucket,plays,wins,losses,win_rate,avg_proj_prob
0,<0.54,6,4,2,0.666667,0.530524
1,0.54-0.57,88,51,37,0.579545,0.554603
2,0.57-0.58,25,12,13,0.480000,0.577422
3,0.58-0.60,117,63,54,0.538462,0.586667
4,0.60-0.65,11,5,6,0.454545,0.625263
5,0.65+,53,28,25,0.528302,0.922584



EKO BETS Ã¢â‚¬â€ MLB HISTORY THRESHOLD STUDY


,min_prob_threshold,plays,wins,losses,win_rate,avg_proj_prob
0,0.54,294,159,135,0.540816,0.638284
1,0.55,271,147,124,0.542435,0.646028
2,0.56,233,123,110,0.527897,0.661418
3,0.57,206,108,98,0.524272,0.674031
4,0.58,181,96,85,0.530387,0.687375
5,0.59,79,40,39,0.506329,0.818646
6,0.60,64,33,31,0.515625,0.871482



EKO BETS Ã¢â‚¬â€ MLB DAILY HISTORY


,card_date,plays,wins,avg_proj_prob,losses,win_rate
0,2026-04-05,5,1,0.562771,4,0.200000
1,2026-04-08,3,1,0.557752,2,0.333333
2,2026-04-09,2,1,0.565100,1,0.500000
3,2026-04-11,5,3,0.601532,2,0.600000
4,2026-04-12,8,4,0.594359,4,0.500000
...,...,...,...,...,...,...
67,2026-09-23,1,1,0.653151,0,1.000000
68,2026-09-24,4,3,0.548019,1,0.750000
69,2026-09-25,1,0,0.611142,1,0.000000
70,2026-10-04,1,0,0.527340,1,0.000000



EKO BETS Ã¢â‚¬â€ MLB ALL-TIME SUMMARY
Plays         : 300
Wins          : 163
Losses        : 137
Win rate      : 0.5433
Avg proj prob : 0.6361


In [52]:
# %%
# =========================================================
# CELL 29 Ã¢â‚¬â€ MLB POSTING RULES RECOMMENDER
# Turn historical results into simple operating rules
# =========================================================

import pandas as pd
import numpy as np

if "mlb_history_threshold_summary" not in globals() or not isinstance(mlb_history_threshold_summary, pd.DataFrame):
    raise ValueError("mlb_history_threshold_summary not found. Run Cell 28 first.")

th = mlb_history_threshold_summary.copy()

usable = th[th["plays"] > 0].copy()

if len(usable) == 0:
    raise ValueError("No usable rows in threshold summary.")

# ---------------------------------------------------------
# recommend safest current threshold
# prioritize win rate, then higher threshold, then lower volume
# ---------------------------------------------------------
best = usable.sort_values(
    ["win_rate", "min_prob_threshold"],
    ascending=[False, False]
).iloc[0]

recommended_threshold = float(best["min_prob_threshold"])

# ---------------------------------------------------------
# suggested daily max plays
# very conservative while sample is small
# ---------------------------------------------------------
total_hist_plays = int(usable.loc[usable["min_prob_threshold"] == usable["min_prob_threshold"].min(), "plays"].max())

if total_hist_plays < 20:
    recommended_max_plays = 1
elif total_hist_plays < 50:
    recommended_max_plays = 2
else:
    recommended_max_plays = 3

# ---------------------------------------------------------
# suggested sizing rule
# ---------------------------------------------------------
def size_rule(prob):
    if prob >= 0.60:
        return "1 UNIT"
    if prob >= 0.58:
        return "0.75 UNIT"
    if prob >= 0.57:
        return "0.5 UNIT"
    return "PASS"

size_guide = pd.DataFrame({
    "projected_win_prob_range": [
        "< 0.57",
        "0.57 to < 0.58",
        "0.58 to < 0.60",
        "0.60+",
    ],
    "recommended_action": [
        "PASS",
        "Lean only / no main card",
        "Playable",
        "Best current bucket",
    ],
    "suggested_size": [
        "0 UNIT",
        "0.5 UNIT",
        "0.75 UNIT",
        "1 UNIT",
    ]
})

print("==================================================")
print("EKO BETS Ã¢â‚¬â€ MLB POSTING RULES")
print("==================================================")
print(f"Recommended posting threshold : {recommended_threshold:.2f}+ projected win probability")
print(f"Recommended max plays per day : {recommended_max_plays}")
print("Recommended model style       : selective, not high-volume")
print("")

print("Current interpretation:")
print("- 0.54 to 0.57 is not trustworthy enough yet")
print("- 0.58+ is your current playable range")
print("- keep MLB cards very small until the sample grows")
print("")

print("Sizing guide:")
display(size_guide)

rules_summary = pd.DataFrame({
    "setting": [
        "recommended_threshold",
        "recommended_max_plays",
        "post_0.54_to_0.57_bucket",
        "post_0.58_plus_bucket",
        "current_model_style",
    ],
    "value": [
        recommended_threshold,
        recommended_max_plays,
        "NO",
        "YES",
        "Selective only",
    ]
})

print("\nRules summary:")
display(rules_summary)

mlb_posting_rules = rules_summary.copy()
mlb_size_guide = size_guide.copy()

EKO BETS Ã¢â‚¬â€ MLB POSTING RULES
Recommended posting threshold : 0.55+ projected win probability
Recommended max plays per day : 3
Recommended model style       : selective, not high-volume

Current interpretation:
- 0.54 to 0.57 is not trustworthy enough yet
- 0.58+ is your current playable range
- keep MLB cards very small until the sample grows

Sizing guide:


,projected_win_prob_range,recommended_action,suggested_size
0,< 0.57,PASS,0 UNIT
1,0.57 to < 0.58,Lean only / no main card,0.5 UNIT
2,0.58 to < 0.60,Playable,0.75 UNIT
3,0.60+,Best current bucket,1 UNIT



Rules summary:


,setting,value
0,recommended_threshold,0.55
1,recommended_max_plays,3
2,post_0.54_to_0.57_bucket,NO
3,post_0.58_plus_bucket,YES
4,current_model_style,Selective only


In [53]:
# %%
# =========================================================
# CELL 30 Ã¢â‚¬â€ MLB FEATURE INVENTORY CHECK
# See what stronger MLB-specific columns already exist in
# your model table before building upgraded features
# =========================================================

import pandas as pd

if "mlb_raw" not in globals() or not isinstance(mlb_raw, pd.DataFrame):
    raise ValueError("mlb_raw not found. Run Cell 2 first.")

df = mlb_raw.copy()

all_cols = list(df.columns)

feature_groups = {
    "starter": [
        c for c in all_cols
        if "starter" in c.lower() or "pitcher" in c.lower()
    ],
    "bullpen": [
        c for c in all_cols
        if "bullpen" in c.lower() or "relief" in c.lower()
    ],
    "park": [
        c for c in all_cols
        if "park" in c.lower() or "venue" in c.lower()
    ],
    "weather": [
        c for c in all_cols
        if "weather" in c.lower() or "temp" in c.lower() or "wind" in c.lower()
    ],
    "lineup_hitting": [
        c for c in all_cols
        if "ops" in c.lower() or "wrc" in c.lower() or "lineup" in c.lower()
        or "bat" in c.lower()
    ],
    "schedule_rest": [
        c for c in all_cols
        if "rest" in c.lower()
    ],
    "rolling_form": [
        c for c in all_cols
        if any(x in c.lower() for x in ["avg_3", "avg_5", "avg_10", "win_rate", "margin"])
    ],
}

print("==================================================")
print("MLB FEATURE INVENTORY CHECK")
print("==================================================")
print("Raw table shape:", df.shape)
print("Total columns  :", len(all_cols))
print("")

for group_name, cols in feature_groups.items():
    print(f"{group_name.upper()} COLUMNS ({len(cols)})")
    if cols:
        for c in cols:
            print(" -", c)
    else:
        print(" - None found")
    print("")

inventory_rows = []
for group_name, cols in feature_groups.items():
    inventory_rows.append({
        "feature_group": group_name,
        "count": len(cols),
        "columns": ", ".join(cols) if cols else "None",
    })

feature_inventory_summary = pd.DataFrame(inventory_rows)

print("Summary table:")
display(feature_inventory_summary)

mlb_feature_inventory_summary = feature_inventory_summary.copy()
mlb_feature_groups = feature_groups.copy()

MLB FEATURE INVENTORY CHECK
Raw table shape: (8348, 55)
Total columns  : 55

STARTER COLUMNS (0)
 - None found

BULLPEN COLUMNS (0)
 - None found

PARK COLUMNS (1)
 - venue_name

WEATHER COLUMNS (0)
 - None found

LINEUP_HITTING COLUMNS (0)
 - None found

SCHEDULE_REST COLUMNS (3)
 - away_rest_days
 - home_rest_days
 - diff_rest_days

ROLLING_FORM COLUMNS (37)
 - home_margin
 - away_runs_for_avg_3
 - away_runs_against_avg_3
 - away_margin_avg_3
 - away_win_rate_3
 - away_runs_for_avg_5
 - away_runs_against_avg_5
 - away_margin_avg_5
 - away_win_rate_5
 - away_runs_for_avg_10
 - away_runs_against_avg_10
 - away_margin_avg_10
 - away_win_rate_10
 - home_runs_for_avg_3
 - home_runs_against_avg_3
 - home_margin_avg_3
 - home_win_rate_3
 - home_runs_for_avg_5
 - home_runs_against_avg_5
 - home_margin_avg_5
 - home_win_rate_5
 - home_runs_for_avg_10
 - home_runs_against_avg_10
 - home_margin_avg_10
 - home_win_rate_10
 - diff_runs_for_avg_3
 - diff_runs_against_avg_3
 - diff_margin_avg_3
 - 

,feature_group,count,columns
0,starter,0,None
1,bullpen,0,None
2,park,1,venue_name
3,weather,0,None
4,lineup_hitting,0,None
5,schedule_rest,3,"away_rest_days, home_rest_days, diff_rest_days"
6,rolling_form,37,"home_margin, away_runs_for_avg_3, away_runs_ag..."


In [54]:
# %%
# =========================================================
# CELL 31 Ã¢â‚¬â€ COMPARE MLB FEATURE SETS (ROBUST REPLACEMENT)
# Compares only feature sets that can actually be built from
# the CURRENT train_df.
# Includes compressed 7-feature baseline first.
# =========================================================

import pandas as pd
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, log_loss, brier_score_loss

print("=" * 70)
print("CELL 31 Ã¢â‚¬â€ COMPARE MLB FEATURE SETS")
print("=" * 70)

# ---------------------------------------------------------
# 0) validate inputs
# ---------------------------------------------------------
if "train_df" not in globals() or not isinstance(train_df, pd.DataFrame):
    raise ValueError("train_df not found. Run Cell 4 first.")

if "mlb_wf_windows" not in globals() or not isinstance(mlb_wf_windows, pd.DataFrame):
    raise ValueError("mlb_wf_windows not found. Run Cell 5 first.")

if "TARGET_WIN" not in globals():
    raise ValueError("TARGET_WIN not found.")

df = train_df.copy()
wf_windows = mlb_wf_windows.copy()

if "date" not in df.columns:
    raise ValueError("train_df must contain 'date'.")

df["date"] = pd.to_datetime(df["date"], errors="coerce")
df = df[df["date"].notna()].copy().sort_values(["date"] + (["game_id"] if "game_id" in df.columns else [])).reset_index(drop=True)

if TARGET_WIN not in df.columns:
    raise ValueError(f"{TARGET_WIN} not found in train_df.")

# ---------------------------------------------------------
# 1) define candidate feature sets
# ---------------------------------------------------------
feature_sets = {
    "compressed_7": [
        "diff_runs_for_avg_10",
        "diff_runs_against_avg_10",
        "diff_margin_avg_10",
        "diff_win_rate_10",
        "diff_rest_days",
        "home_rest_days",
        "away_rest_days",
    ],
    "diff_10_only_5": [
        "diff_runs_for_avg_10",
        "diff_runs_against_avg_10",
        "diff_margin_avg_10",
        "diff_win_rate_10",
        "diff_rest_days",
    ],
    "diff_5_10_plus_rest": [
        "diff_runs_for_avg_5",
        "diff_runs_against_avg_5",
        "diff_margin_avg_5",
        "diff_win_rate_5",
        "diff_runs_for_avg_10",
        "diff_runs_against_avg_10",
        "diff_margin_avg_10",
        "diff_win_rate_10",
        "diff_rest_days",
        "home_rest_days",
        "away_rest_days",
    ],
    "diff_3_5_10_plus_rest": [
        "diff_runs_for_avg_3",
        "diff_runs_against_avg_3",
        "diff_margin_avg_3",
        "diff_win_rate_3",
        "diff_runs_for_avg_5",
        "diff_runs_against_avg_5",
        "diff_margin_avg_5",
        "diff_win_rate_5",
        "diff_runs_for_avg_10",
        "diff_runs_against_avg_10",
        "diff_margin_avg_10",
        "diff_win_rate_10",
        "diff_rest_days",
        "home_rest_days",
        "away_rest_days",
    ],
    "home_away_10_plus_diff_10": [
        "away_runs_for_avg_10",
        "away_runs_against_avg_10",
        "away_margin_avg_10",
        "away_win_rate_10",
        "away_rest_days",
        "home_runs_for_avg_10",
        "home_runs_against_avg_10",
        "home_margin_avg_10",
        "home_win_rate_10",
        "home_rest_days",
        "diff_runs_for_avg_10",
        "diff_runs_against_avg_10",
        "diff_margin_avg_10",
        "diff_win_rate_10",
        "diff_rest_days",
    ],
    "split_7_plus_bullpen": [
        "diff_runs_for_avg_10_split",
        "diff_runs_against_avg_10_split",
        "diff_margin_avg_10",
        "diff_win_rate_10",
        "diff_rest_days",
        "home_rest_days",
        "away_rest_days",
        "diff_bullpen_stress_3",
    ],
    "split_11_plus_bullpen": [
        "diff_runs_for_avg_10_split",
        "diff_runs_against_avg_10_split",
        "diff_runs_for_avg_5_split",
        "diff_runs_against_avg_5_split",
        "diff_margin_avg_10",
        "diff_win_rate_10",
        "diff_margin_avg_5",
        "diff_win_rate_5",
        "diff_rest_days",
        "home_rest_days",
        "away_rest_days",
        "diff_bullpen_stress_3",
    ],
    "split_plus_l3_starter": [
        "diff_runs_for_avg_10_split",
        "diff_runs_against_avg_10_split",
        "diff_margin_avg_10",
        "diff_win_rate_10",
        "diff_rest_days",
        "home_rest_days",
        "away_rest_days",
        "diff_bullpen_stress_3",
        "diff_starter_era_l3",
        "diff_starter_whip_l3",
    ],
}

# ---------------------------------------------------------
# 2) keep only fully usable feature sets
# ---------------------------------------------------------
usable_feature_sets = {}
skipped_feature_sets = {}

for name, cols in feature_sets.items():
    missing = [c for c in cols if c not in df.columns]
    if len(missing) == 0:
        usable_feature_sets[name] = cols
    else:
        skipped_feature_sets[name] = missing

print("Available columns in train_df:", len(df.columns))
print("Usable feature sets:")
if usable_feature_sets:
    for k, v in usable_feature_sets.items():
        print(f" - {k}: {len(v)} features")
else:
    print(" - none")

if skipped_feature_sets:
    print("\nSkipped feature sets:")
    for k, missing in skipped_feature_sets.items():
        print(f" - {k}: missing {missing}")

# ---------------------------------------------------------
# 3) fallback: compare current global feature_cols if needed
# ---------------------------------------------------------
if len(usable_feature_sets) == 0:
    if "feature_cols" in globals() and isinstance(feature_cols, list) and len(feature_cols) > 0:
        fallback_cols = [c for c in feature_cols if c in df.columns]
        if len(fallback_cols) == 0:
            raise ValueError(
                "No usable feature sets found, and current global feature_cols "
                "also do not exist in train_df."
            )
        usable_feature_sets["current_feature_cols"] = fallback_cols
        print("\nFalling back to current global feature_cols.")
    else:
        raise ValueError(
            "No usable feature sets found. "
            "Your current train_df does not match any comparison schema."
        )

# ---------------------------------------------------------
# 4) walk-forward comparison
# ---------------------------------------------------------
results = []
all_fold_results = {}

for model_name, cols in usable_feature_sets.items():
    print("\n" + "-" * 70)
    print(f"RUNNING: {model_name} | features={len(cols)}")
    print("-" * 70)

    fold_rows = []

    for _, w in wf_windows.iterrows():
        test_start = pd.to_datetime(w["test_start"])
        test_end = pd.to_datetime(w["test_end"])

        train_set = df[df["date"] < test_start].copy()
        test_set = df[(df["date"] >= test_start) & (df["date"] < test_end)].copy()

        if len(train_set) == 0 or len(test_set) == 0:
            continue

        # enforce numeric and drop rows with missing selected features
        for c in cols:
            train_set[c] = pd.to_numeric(train_set[c], errors="coerce")
            test_set[c] = pd.to_numeric(test_set[c], errors="coerce")

        train_set[TARGET_WIN] = pd.to_numeric(train_set[TARGET_WIN], errors="coerce")
        test_set[TARGET_WIN] = pd.to_numeric(test_set[TARGET_WIN], errors="coerce")

        train_set = train_set.dropna(subset=cols + [TARGET_WIN]).copy()
        test_set = test_set.dropna(subset=cols + [TARGET_WIN]).copy()

        if len(train_set) == 0 or len(test_set) == 0:
            continue

        X_train = train_set[cols].copy()
        y_train = train_set[TARGET_WIN].astype(int).copy()

        X_test = test_set[cols].copy()
        y_test = test_set[TARGET_WIN].astype(int).copy()

        model = Pipeline([
            ("scaler", StandardScaler()),
            ("logit", LogisticRegression(
                max_iter=5000,
                C=1.0,
                random_state=42
            ))
        ])

        model.fit(X_train, y_train)

        p_test = model.predict_proba(X_test)[:, 1]
        p_test = np.clip(p_test, 1e-6, 1 - 1e-6)
        y_pred = (p_test >= 0.5).astype(int)

        fold_rows.append({
            "model_name": model_name,
            "test_start": test_start,
            "test_end": test_end,
            "n_train": len(train_set),
            "n_test": len(test_set),
            "accuracy": accuracy_score(y_test, y_pred),
            "log_loss": log_loss(y_test, p_test),
            "brier": brier_score_loss(y_test, p_test),
            "home_win_rate_test": float(y_test.mean()),
            "avg_pred_home_win": float(np.mean(p_test)),
        })

    fold_df = pd.DataFrame(fold_rows).sort_values("test_start").reset_index(drop=True)
    all_fold_results[model_name] = fold_df.copy()

    if len(fold_df) == 0:
        print("No valid fold results for this feature set.")
        continue

    results.append({
        "model_name": model_name,
        "feature_count": len(cols),
        "windows": len(fold_df),
        "avg_accuracy": fold_df["accuracy"].mean(),
        "avg_log_loss": fold_df["log_loss"].mean(),
        "avg_brier": fold_df["brier"].mean(),
    })

    print("Fold results:")
    display(fold_df)

    print("Averages:")
    print(" - Accuracy :", round(float(fold_df["accuracy"].mean()), 4))
    print(" - Log Loss :", round(float(fold_df["log_loss"].mean()), 4))
    print(" - Brier    :", round(float(fold_df["brier"].mean()), 4))

# ---------------------------------------------------------
# 5) final summary
# ---------------------------------------------------------
if len(results) == 0:
    raise ValueError("No valid comparison results were produced.")

compare_df = pd.DataFrame(results).sort_values(
    ["avg_log_loss", "avg_brier", "avg_accuracy"],
    ascending=[True, True, False]
).reset_index(drop=True)

print("\n" + "=" * 70)
print("MLB FEATURE SET COMPARISON")
print("=" * 70)
display(compare_df)

best_model_name = compare_df.iloc[0]["model_name"]
best_feature_cols = usable_feature_sets[best_model_name]

print("\nBest feature set:", best_model_name)
print("Feature count   :", len(best_feature_cols))
print("\nBest feature columns:")
for i, c in enumerate(best_feature_cols, 1):
    print(f"{i:>2}. {c}")

# ---------------------------------------------------------
# 6) save in memory
# ---------------------------------------------------------
globals()["mlb_feature_set_compare"] = compare_df.copy()
globals()["mlb_feature_set_fold_results"] = all_fold_results.copy()
globals()["mlb_best_feature_set_name"] = best_model_name
globals()["mlb_best_feature_cols"] = best_feature_cols.copy()

print("\nCreated in memory:")
print(" - mlb_feature_set_compare")
print(" - mlb_feature_set_fold_results")
print(" - mlb_best_feature_set_name")
print(" - mlb_best_feature_cols")
print("Done.")

CELL 31 Ã¢â‚¬â€ COMPARE MLB FEATURE SETS
Available columns in train_df: 135
Usable feature sets:
 - compressed_7: 7 features
 - diff_10_only_5: 5 features
 - diff_5_10_plus_rest: 11 features
 - diff_3_5_10_plus_rest: 15 features
 - home_away_10_plus_diff_10: 15 features

Skipped feature sets:
 - split_7_plus_bullpen: missing ['diff_runs_for_avg_10_split', 'diff_runs_against_avg_10_split', 'diff_bullpen_stress_3']
 - split_11_plus_bullpen: missing ['diff_runs_for_avg_10_split', 'diff_runs_against_avg_10_split', 'diff_runs_for_avg_5_split', 'diff_runs_against_avg_5_split', 'diff_bullpen_stress_3']
 - split_plus_l3_starter: missing ['diff_runs_for_avg_10_split', 'diff_runs_against_avg_10_split', 'diff_bullpen_stress_3', 'diff_starter_era_l3', 'diff_starter_whip_l3']

----------------------------------------------------------------------
RUNNING: compressed_7 | features=7
----------------------------------------------------------------------


Fold results:


,model_name,test_start,test_end,n_train,n_test,accuracy,log_loss,brier,home_win_rate_test,avg_pred_home_win
0,compressed_7,2024-03-01,2024-04-01,2818,412,0.512136,0.696046,0.251409,0.492718,0.511889
1,compressed_7,2024-04-01,2024-05-01,3230,390,0.530769,0.686940,0.246899,0.510256,0.513159
2,compressed_7,2024-05-01,2024-06-01,3620,403,0.545906,0.684104,0.245512,0.550868,0.513213
3,compressed_7,2024-06-01,2024-07-01,4023,398,0.537688,0.687672,0.247274,0.570352,0.515333
4,compressed_7,2024-07-01,2024-08-01,4421,364,0.467033,0.699383,0.253101,0.491758,0.517834
5,compressed_7,2024-08-01,2024-09-01,4785,406,0.544335,0.688357,0.247611,0.504926,0.519996
6,compressed_7,2024-09-01,2024-10-01,5191,382,0.523560,0.693328,0.250060,0.523560,0.521572
7,compressed_7,2025-03-01,2025-04-01,5616,391,0.572890,0.687709,0.247289,0.544757,0.520993
8,compressed_7,2025-04-01,2025-05-01,6007,358,0.550279,0.685283,0.246093,0.617318,0.516678
9,compressed_7,2025-05-01,2025-06-01,6365,375,0.552000,0.686093,0.246492,0.520000,0.523858


Averages:
 - Accuracy : 0.5374
 - Log Loss : 0.6887
 - Brier    : 0.2478

----------------------------------------------------------------------
RUNNING: diff_10_only_5 | features=5
----------------------------------------------------------------------


Fold results:


...

Fold results:


,model_name,test_start,test_end,n_train,n_test,accuracy,log_loss,brier,home_win_rate_test,avg_pred_home_win
0,home_away_10_plus_diff_10,2024-03-01,2024-04-01,2818,412,0.521845,0.695491,0.251130,0.492718,0.515817
1,home_away_10_plus_diff_10,2024-04-01,2024-05-01,3230,390,0.533333,0.688074,0.247471,0.510256,0.509328
2,home_away_10_plus_diff_10,2024-05-01,2024-06-01,3620,403,0.548387,0.684736,0.245819,0.550868,0.508922
3,home_away_10_plus_diff_10,2024-06-01,2024-07-01,4023,398,0.545226,0.687678,0.247281,0.570352,0.513481
4,home_away_10_plus_diff_10,2024-07-01,2024-08-01,4421,364,0.489011,0.698619,0.252721,0.491758,0.517834
5,home_away_10_plus_diff_10,2024-08-01,2024-09-01,4785,406,0.546798,0.687901,0.247390,0.504926,0.520067
6,home_away_10_plus_diff_10,2024-09-01,2024-10-01,5191,382,0.513089,0.692136,0.249475,0.523560,0.520906
7,home_away_10_plus_diff_10,2025-03-01,2025-04-01,5616,391,0.552430,0.690030,0.248461,0.544757,0.520531
8,home_away_10_plus_diff_10,2025-04-01,2025-05-01,6007,358,0.525140,0.683295,0.245112,0.617318,0.516359
9,home_away_10_plus_diff_10,2025-05-01,2025-06-01,6365,375,0.576000,0.686237,0.246576,0.520000,0.525662


Averages:
 - Accuracy : 0.5344
 - Log Loss : 0.689
 - Brier    : 0.2479

MLB FEATURE SET COMPARISON


,model_name,feature_count,windows,avg_accuracy,avg_log_loss,avg_brier
0,diff_5_10_plus_rest,11,14,0.534747,0.688545,0.247708
1,diff_10_only_5,5,14,0.537395,0.688575,0.247722
2,compressed_7,7,14,0.537353,0.688729,0.247798
3,home_away_10_plus_diff_10,15,14,0.534372,0.688976,0.247921
4,diff_3_5_10_plus_rest,15,14,0.539591,0.688979,0.247923



Best feature set: diff_5_10_plus_rest
Feature count   : 11

Best feature columns:
 1. diff_runs_for_avg_5
 2. diff_runs_against_avg_5
 3. diff_margin_avg_5
 4. diff_win_rate_5
 5. diff_runs_for_avg_10
 6. diff_runs_against_avg_10
 7. diff_margin_avg_10
 8. diff_win_rate_10
 9. diff_rest_days
10. home_rest_days
11. away_rest_days

Created in memory:
 - mlb_feature_set_compare
 - mlb_feature_set_fold_results
 - mlb_best_feature_set_name
 - mlb_best_feature_cols
Done.


In [55]:
# %%
# =========================================================
# CELL 32 Ã¢â‚¬â€ RETRAIN + SAVE BEST CONFIRMED PRODUCTION MODEL
# FULL REPLACEMENT
# Uses the best confirmed feature set from comparison
# - robust MODEL_DIR handling
# - optional calibrator save
# - cleans features before fitting
# =========================================================

import pandas as pd
import numpy as np
import json
import joblib
from pathlib import Path

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

print("=" * 70)
print("CELL 32 Ã¢â‚¬â€ RETRAIN + SAVE BEST CONFIRMED PRODUCTION MODEL")
print("=" * 70)

# ---------------------------------------------------------
# 0) validate inputs
# ---------------------------------------------------------
if "train_df" not in globals() or not isinstance(train_df, pd.DataFrame) or len(train_df) == 0:
    raise ValueError("train_df not found or empty. Run Cell 4 first.")

if "mlb_best_feature_cols" not in globals() or not isinstance(mlb_best_feature_cols, list) or len(mlb_best_feature_cols) == 0:
    raise ValueError("mlb_best_feature_cols not found. Run Cell 31 first.")

if "TARGET_WIN" not in globals():
    raise ValueError("TARGET_WIN not found.")

final_train = train_df.copy()

if TARGET_WIN not in final_train.columns:
    raise ValueError(f"{TARGET_WIN} not found in train_df.")

missing_feature_cols = [c for c in mlb_best_feature_cols if c not in final_train.columns]
if missing_feature_cols:
    raise ValueError(f"These best feature columns are missing from train_df: {missing_feature_cols}")

# ---------------------------------------------------------
# 1) resolve MODEL_DIR safely
# ---------------------------------------------------------
resolved_model_dir = None

if "MODEL_DIR" in globals():
    try:
        resolved_model_dir = Path(MODEL_DIR)
        print(f"Using existing MODEL_DIR: {resolved_model_dir}")
    except Exception as e:
        print(f"Could not parse existing MODEL_DIR: {e}")

if resolved_model_dir is None:
    candidates = [
        Path(r"c:\Users\<user>\OneDrive\Documents\KAI Labs\sports\MLB\models"),
        Path.cwd() / "models" / "mlb",
        Path.cwd() / "models",
        Path("models/mlb"),
        Path("models"),
    ]

    for p in candidates:
        try:
            p.mkdir(parents=True, exist_ok=True)
            resolved_model_dir = p
            break
        except Exception:
            continue

if resolved_model_dir is None:
    raise ValueError("Could not create or resolve a usable MODEL_DIR.")

resolved_model_dir.mkdir(parents=True, exist_ok=True)
MODEL_DIR = resolved_model_dir

print(f"Resolved MODEL_DIR: {MODEL_DIR}")

# ---------------------------------------------------------
# 2) sort chronologically
# ---------------------------------------------------------
if "date" in final_train.columns:
    final_train["date"] = pd.to_datetime(final_train["date"], errors="coerce")

sort_cols = [c for c in ["date", "game_id"] if c in final_train.columns]
if len(sort_cols) > 0:
    final_train = final_train.sort_values(sort_cols).reset_index(drop=True)
else:
    final_train = final_train.reset_index(drop=True)

# ---------------------------------------------------------
# 3) numeric coercion + cleanup
# ---------------------------------------------------------
before_rows = len(final_train)

for c in mlb_best_feature_cols:
    final_train[c] = pd.to_numeric(final_train[c], errors="coerce")

final_train[TARGET_WIN] = pd.to_numeric(final_train[TARGET_WIN], errors="coerce")

final_train = final_train.dropna(subset=mlb_best_feature_cols + [TARGET_WIN]).copy()

if len(final_train) == 0:
    raise ValueError("No rows remain after cleaning selected features and target.")

target_unique = sorted(final_train[TARGET_WIN].dropna().unique().tolist())
if not set(target_unique).issubset({0, 1}):
    raise ValueError(f"TARGET_WIN must be binary 0/1. Found: {target_unique}")

X_final = final_train[mlb_best_feature_cols].copy()
y_final = final_train[TARGET_WIN].astype(int).copy()

print("\nTraining table summary")
print("-" * 60)
print("Rows before cleanup :", before_rows)
print("Rows after cleanup  :", len(final_train))
print("Rows removed        :", before_rows - len(final_train))
print("Feature count       :", len(mlb_best_feature_cols))

if "date" in final_train.columns and final_train["date"].notna().any():
    print("Train date min      :", final_train["date"].min())
    print("Train date max      :", final_train["date"].max())

print("Target mean         :", round(float(y_final.mean()), 4))

# ---------------------------------------------------------
# 4) fit model
# ---------------------------------------------------------
best_win_model = Pipeline([
    ("scaler", StandardScaler()),
    ("logit", LogisticRegression(
        max_iter=5000,
        C=1.0,
        random_state=42
    ))
])

best_win_model.fit(X_final, y_final)

# ---------------------------------------------------------
# 5) save files
# ---------------------------------------------------------
best_model_file = MODEL_DIR / "mlb_win_model_best_confirmed.joblib"
best_calibrator_file = MODEL_DIR / "mlb_win_model_best_confirmed_calibrator.joblib"
best_meta_file = MODEL_DIR / "mlb_win_model_best_confirmed_meta.json"

joblib.dump(best_win_model, best_model_file)

calibrator_saved = False
if "mlb_iso_calibrator" in globals():
    try:
        joblib.dump(mlb_iso_calibrator, best_calibrator_file)
        calibrator_saved = True
    except Exception as e:
        print(f"Warning: could not save calibrator: {e}")
else:
    print("No calibrator found Ã¢â‚¬â€ skipping calibrator save.")

# ---------------------------------------------------------
# 6) metadata
# ---------------------------------------------------------
meta = {
    "model_name": "mlb_win_model_best_confirmed",
    "target": TARGET_WIN,
    "feature_set_name": globals().get("mlb_best_feature_set_name", "unknown"),
    "feature_count": len(mlb_best_feature_cols),
    "feature_cols": list(mlb_best_feature_cols),
    "train_rows": int(len(final_train)),
    "train_start": str(pd.to_datetime(final_train["date"]).min().date()) if "date" in final_train.columns and final_train["date"].notna().any() else None,
    "train_end": str(pd.to_datetime(final_train["date"]).max().date()) if "date" in final_train.columns and final_train["date"].notna().any() else None,
    "calibrator_saved": calibrator_saved,
}

if "mlb_wf_metrics" in globals() and isinstance(mlb_wf_metrics, pd.DataFrame) and len(mlb_wf_metrics) > 0:
    if "accuracy" in mlb_wf_metrics.columns:
        meta["walkforward_avg_accuracy"] = float(mlb_wf_metrics["accuracy"].mean())
    if "log_loss" in mlb_wf_metrics.columns:
        meta["walkforward_avg_log_loss"] = float(mlb_wf_metrics["log_loss"].mean())
    if "brier" in mlb_wf_metrics.columns:
        meta["walkforward_avg_brier"] = float(mlb_wf_metrics["brier"].mean())
    meta["walkforward_windows"] = int(len(mlb_wf_metrics))

with open(best_meta_file, "w", encoding="utf-8") as f:
    json.dump(meta, f, indent=2)

# ---------------------------------------------------------
# 7) output
# ---------------------------------------------------------
print("\n" + "=" * 70)
print("BEST CONFIRMED MLB MODEL SAVED")
print("=" * 70)
print("Model file      :", best_model_file)
print("Metadata file   :", best_meta_file)
print("Calibrator saved:", calibrator_saved)
if calibrator_saved:
    print("Calibrator file :", best_calibrator_file)

print("\nFeature set name:", globals().get("mlb_best_feature_set_name", "unknown"))
print("Feature count   :", len(mlb_best_feature_cols))
print("Train rows      :", len(final_train))

print("\nFeature list:")
for i, c in enumerate(mlb_best_feature_cols, 1):
    print(f"{i:>2}. {c}")

# ---------------------------------------------------------
# 8) save in memory
# ---------------------------------------------------------
mlb_best_confirmed_model = best_win_model
mlb_best_confirmed_train = final_train.copy()
mlb_best_confirmed_model_path = best_model_file
mlb_best_confirmed_meta_path = best_meta_file

print("\nCreated in memory:")
print(" - mlb_best_confirmed_model")
print(" - mlb_best_confirmed_train")
print(" - mlb_best_confirmed_model_path")
print(" - mlb_best_confirmed_meta_path")
print("Done.")

CELL 32 Ã¢â‚¬â€ RETRAIN + SAVE BEST CONFIRMED PRODUCTION MODEL
Resolved MODEL_DIR: c:\Users\<user>\OneDrive\Documents\KAI Labs\sports\MLB\models

Training table summary
------------------------------------------------------------
Rows before cleanup : 8230
Rows after cleanup  : 8230
Rows removed        : 0
Feature count       : 11
Train date min      : 2023-03-02 00:00:00
Train date max      : 2025-11-01 00:00:00
Target mean         : 0.5272



BEST CONFIRMED MLB MODEL SAVED
Model file      : c:\Users\<user>\OneDrive\Documents\KAI Labs\sports\MLB\models\mlb_win_model_best_confirmed.joblib
Metadata file   : c:\Users\<user>\OneDrive\Documents\KAI Labs\sports\MLB\models\mlb_win_model_best_confirmed_meta.json
Calibrator saved: True
Calibrator file : c:\Users\<user>\OneDrive\Documents\KAI Labs\sports\MLB\models\mlb_win_model_best_confirmed_calibrator.joblib

Feature set name: diff_5_10_plus_rest
Feature count   : 11
Train rows      : 8230

Feature list:
 1. diff_runs_for_avg_5
 2. diff_runs_against_avg_5
 3. diff_margin_avg_5
 4. diff_win_rate_5
 5. diff_runs_for_avg_10
 6. diff_runs_against_avg_10
 7. diff_margin_avg_10
 8. diff_win_rate_10
 9. diff_rest_days
10. home_rest_days
11. away_rest_days

Created in memory:
 - mlb_best_confirmed_model
 - mlb_best_confirmed_train
 - mlb_best_confirmed_model_path
 - mlb_best_confirmed_meta_path
Done.


In [56]:
# %%
# =========================================================
# CELL 33 Ã¢â‚¬â€ INSPECT BEST CONFIRMED MODEL COEFFICIENTS
# See which features are driving home-win predictions
# =========================================================

import pandas as pd
import numpy as np

if "mlb_best_confirmed_model" not in globals():
    raise ValueError("mlb_best_confirmed_model not found. Run Cell 32 first.")

if "mlb_best_feature_cols" not in globals() or not isinstance(mlb_best_feature_cols, list) or len(mlb_best_feature_cols) == 0:
    raise ValueError("mlb_best_feature_cols not found. Run Cell 31 first.")

logit = mlb_best_confirmed_model.named_steps["logit"]

coef_df = pd.DataFrame({
    "feature": mlb_best_feature_cols,
    "coef": logit.coef_[0]
})

coef_df["abs_coef"] = coef_df["coef"].abs()
coef_df = coef_df.sort_values("abs_coef", ascending=False).reset_index(drop=True)

print("==================================================")
print("BEST CONFIRMED MLB MODEL Ã¢â‚¬â€ COEFFICIENT INSPECTION")
print("==================================================")

print("\nTop features by absolute coefficient:")
display(coef_df.head(15))

print("\nTop positive coefficients (push toward HOME win):")
display(coef_df.sort_values("coef", ascending=False).head(10))

print("\nTop negative coefficients (push toward AWAY win):")
display(coef_df.sort_values("coef", ascending=True).head(10))

mlb_best_model_coefs = coef_df.copy()

BEST CONFIRMED MLB MODEL Ã¢â‚¬â€ COEFFICIENT INSPECTION

Top features by absolute coefficient:


,feature,coef,abs_coef
0,diff_runs_against_avg_10,-0.095514,0.095514
1,diff_margin_avg_10,0.087246,0.087246
2,diff_runs_against_avg_5,0.083160,0.083160
3,diff_runs_for_avg_10,0.036037,0.036037
4,diff_runs_for_avg_5,0.032562,0.032562
5,diff_margin_avg_5,-0.031803,0.031803
6,diff_win_rate_10,0.022789,0.022789
7,home_rest_days,0.012717,0.012717
8,away_rest_days,0.012221,0.012221
9,diff_win_rate_5,0.009518,0.009518



Top positive coefficients (push toward HOME win):


,feature,coef,abs_coef
1,diff_margin_avg_10,0.087246,0.087246
2,diff_runs_against_avg_5,0.083160,0.083160
3,diff_runs_for_avg_10,0.036037,0.036037
4,diff_runs_for_avg_5,0.032562,0.032562
6,diff_win_rate_10,0.022789,0.022789
7,home_rest_days,0.012717,0.012717
8,away_rest_days,0.012221,0.012221
9,diff_win_rate_5,0.009518,0.009518
10,diff_rest_days,0.001786,0.001786
5,diff_margin_avg_5,-0.031803,0.031803



Top negative coefficients (push toward AWAY win):


,feature,coef,abs_coef
0,diff_runs_against_avg_10,-0.095514,0.095514
5,diff_margin_avg_5,-0.031803,0.031803
10,diff_rest_days,0.001786,0.001786
9,diff_win_rate_5,0.009518,0.009518
8,away_rest_days,0.012221,0.012221
7,home_rest_days,0.012717,0.012717
6,diff_win_rate_10,0.022789,0.022789
4,diff_runs_for_avg_5,0.032562,0.032562
3,diff_runs_for_avg_10,0.036037,0.036037
2,diff_runs_against_avg_5,0.083160,0.083160


In [57]:
# %%
# =========================================================
# CELL 34 Ã¢â‚¬â€ TEST COMPRESSED DIFF-HEAVY FEATURE SET
# Reduce redundancy by leaning more on differential features
# and less on separate raw away/home columns
# =========================================================

import pandas as pd
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, log_loss, brier_score_loss

if "train_df" not in globals() or not isinstance(train_df, pd.DataFrame):
    raise ValueError("train_df not found. Run Cell 4 first.")

if "mlb_wf_windows" not in globals() or not isinstance(mlb_wf_windows, pd.DataFrame):
    raise ValueError("mlb_wf_windows not found. Run Cell 5 first.")

df = train_df.copy()
wf_windows = mlb_wf_windows.copy()

compressed_feature_cols = [
    "diff_runs_for_avg_10",
    "diff_runs_against_avg_10",
    "diff_margin_avg_10",
    "diff_win_rate_10",
    "diff_runs_for_avg_5",
    "diff_runs_against_avg_5",
    "diff_margin_avg_5",
    "diff_win_rate_5",
    "diff_k_pct_bat",
    "diff_bb_pct_bat",
    "diff_starter_fip",
    "diff_starter_k_bb_pct",
    "diff_rest_days",
    "home_rest_days",
    "away_rest_days",
]
# Note: diff_k_pct_bat / diff_bb_pct_bat require train_df enrichment cell.
# Note: diff_starter_fip / diff_starter_k_bb_pct are NaN for historical data;
#       rows with NaN starters will be dropped from CV â€” expected behaviour.
# Soft-check: only use a feature if it exists AND has enough non-NaN rows
_min_rows = 100
compressed_feature_cols = [
    f for f in compressed_feature_cols
    if f in df.columns and df[f].notna().sum() >= _min_rows
]
print(f"Active features ({len(compressed_feature_cols)}): {compressed_feature_cols}")

missing = [c for c in compressed_feature_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing compressed feature columns: {missing}")

fold_rows = []
all_preds = []

for _, w in wf_windows.iterrows():
    test_start = pd.to_datetime(w["test_start"])
    test_end = pd.to_datetime(w["test_end"])

    train_set = df[df["date"] < test_start].copy()
    test_set = df[(df["date"] >= test_start) & (df["date"] < test_end)].copy()

    if len(train_set) == 0 or len(test_set) == 0:
        continue

    X_train = train_set[compressed_feature_cols].copy()
    y_train = train_set[TARGET_WIN].astype(int).copy()

    X_test = test_set[compressed_feature_cols].copy()
    y_test = test_set[TARGET_WIN].astype(int).copy()

    model = Pipeline([
        ("scaler", StandardScaler()),
        ("logit", LogisticRegression(
            max_iter=5000,
            C=1.0,
            random_state=42
        ))
    ])

    model.fit(X_train, y_train)

    p_test = model.predict_proba(X_test)[:, 1]
    y_pred = (p_test >= 0.5).astype(int)

    fold_rows.append({
        "test_start": test_start,
        "test_end": test_end,
        "n_train": len(train_set),
        "n_test": len(test_set),
        "accuracy": accuracy_score(y_test, y_pred),
        "log_loss": log_loss(y_test, p_test),
        "brier": brier_score_loss(y_test, p_test),
    })

    pred_block = test_set[["date", "away_team", "home_team", TARGET_WIN]].copy()
    if "game_id" in test_set.columns:
        pred_block["game_id"] = test_set["game_id"].values
    pred_block["p_home_win"] = p_test
    pred_block["pred_home_win"] = y_pred
    pred_block["test_start"] = test_start
    all_preds.append(pred_block)

compressed_wf_metrics = pd.DataFrame(fold_rows).sort_values("test_start").reset_index(drop=True)

print("==================================================")
print("COMPRESSED DIFF-HEAVY MLB MODEL Ã¢â‚¬â€ WALK-FORWARD RESULTS")
print("==================================================")
display(compressed_wf_metrics)

print("\nAVERAGES")
print("Accuracy   :", round(compressed_wf_metrics["accuracy"].mean(), 4))
print("Log Loss   :", round(compressed_wf_metrics["log_loss"].mean(), 4))
print("Brier Score:", round(compressed_wf_metrics["brier"].mean(), 4))

# compare against current best confirmed model
base_acc = np.nan
base_ll = np.nan
base_brier = np.nan

if "mlb_wf_metrics" in globals() and isinstance(mlb_wf_metrics, pd.DataFrame) and len(mlb_wf_metrics):
    base_acc = mlb_wf_metrics["accuracy"].mean()
    base_ll = mlb_wf_metrics["log_loss"].mean()
    base_brier = mlb_wf_metrics["brier"].mean()

compare_df = pd.DataFrame({
    "model_name": ["best_confirmed_simple15", "compressed_diff_heavy"],
    "feature_count": [15, len(compressed_feature_cols)],
    "avg_accuracy": [base_acc, compressed_wf_metrics["accuracy"].mean()],
    "avg_log_loss": [base_ll, compressed_wf_metrics["log_loss"].mean()],
    "avg_brier": [base_brier, compressed_wf_metrics["brier"].mean()],
})

print("\nMODEL COMPARISON")
display(compare_df)

mlb_compressed_feature_cols = compressed_feature_cols.copy()
mlb_compressed_wf_metrics = compressed_wf_metrics.copy()
mlb_compressed_compare = compare_df.copy()

# CALIBRATION AUDIT FIX (2026-08-27, user audit): this cell already computes
# genuine out-of-sample row-level predictions per fold (pred_block, appended
# to all_preds above) for the EXACT model architecture/feature set that
# becomes the live production model in Cell 35 -- but never saved them
# anywhere, only the aggregated fold metrics survived. This is the correct
# calibration input (matches the live model's own probability distribution,
# unlike Cell 7's calibrator, which was fit on a DIFFERENT model's walk-
# forward predictions -- see the next cell for why that mismatch is exactly
# what produced a calibrator that saturates to 100% for any raw probability
# above ~0.66, found via a real graded-history audit: 37 OFFICIAL plays over
# 30 days, -17.3% ROI, and the model's own stated probability overstated the
# actual win rate by ~12-21 points in every bucket).
if all_preds:
    mlb_compressed_wf_preds = pd.concat(all_preds, ignore_index=True)
    print(f"\nSaved mlb_compressed_wf_preds: {len(mlb_compressed_wf_preds)} out-of-sample row-level predictions "
          f"({mlb_compressed_wf_preds['test_start'].min().date()} to {mlb_compressed_wf_preds['test_start'].max().date()})")
else:
    mlb_compressed_wf_preds = pd.DataFrame()
    print("\nWARNING: no walk-forward prediction rows to save -- mlb_compressed_wf_preds is empty.")

Active features (11): ['diff_runs_for_avg_10', 'diff_runs_against_avg_10', 'diff_margin_avg_10', 'diff_win_rate_10', 'diff_runs_for_avg_5', 'diff_runs_against_avg_5', 'diff_margin_avg_5', 'diff_win_rate_5', 'diff_rest_days', 'home_rest_days', 'away_rest_days']


COMPRESSED DIFF-HEAVY MLB MODEL Ã¢â‚¬â€ WALK-FORWARD RESULTS


,test_start,test_end,n_train,n_test,accuracy,log_loss,brier
0,2024-03-01,2024-04-01,2818,412,0.500000,0.696899,0.251841
1,2024-04-01,2024-05-01,3230,390,0.535897,0.687237,0.247046
2,2024-05-01,2024-06-01,3620,403,0.543424,0.683898,0.245406
3,2024-06-01,2024-07-01,4023,398,0.550251,0.686566,0.246735
4,2024-07-01,2024-08-01,4421,364,0.478022,0.700718,0.253764
5,2024-08-01,2024-09-01,4785,406,0.566502,0.687185,0.247031
6,2024-09-01,2024-10-01,5191,382,0.520942,0.693240,0.250018
7,2025-03-01,2025-04-01,5616,391,0.549872,0.684199,0.245551
8,2025-04-01,2025-05-01,6007,358,0.572626,0.683862,0.245386
9,2025-05-01,2025-06-01,6365,375,0.538667,0.686739,0.246823



AVERAGES
Accuracy   : 0.5347
Log Loss   : 0.6885
Brier Score: 0.2477

MODEL COMPARISON


,model_name,feature_count,avg_accuracy,avg_log_loss,avg_brier
0,best_confirmed_simple15,15,NaN,NaN,NaN
1,compressed_diff_heavy,11,0.534747,0.688545,0.247708



Saved mlb_compressed_wf_preds: 5326 out-of-sample row-level predictions (2024-03-01 to 2025-09-01)


In [58]:
# %%
# =========================================================
# CELL 35 Ã¢â‚¬â€ SAVE NEW BEST MLB MODEL (COMPRESSED DIFF-HEAVY)
# Retrain on full history and save as the new production model
# =========================================================

import pandas as pd
import numpy as np
import json
import joblib
from pathlib import Path

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

if "train_df" not in globals() or not isinstance(train_df, pd.DataFrame):
    raise ValueError("train_df not found. Run Cell 4 first.")

if "mlb_compressed_feature_cols" not in globals() or not isinstance(mlb_compressed_feature_cols, list) or len(mlb_compressed_feature_cols) == 0:
    raise ValueError("mlb_compressed_feature_cols not found. Run Cell 34 first.")

if "mlb_compressed_wf_preds" not in globals() or not isinstance(mlb_compressed_wf_preds, pd.DataFrame) or mlb_compressed_wf_preds.empty:
    raise ValueError("mlb_compressed_wf_preds not found or empty. Run the compressed walk-forward cell (Cell 34) first.")

final_train = train_df.copy()

sort_cols = ["date"]
if "game_id" in final_train.columns:
    sort_cols.append("game_id")

final_train = final_train.sort_values(sort_cols).reset_index(drop=True)

X_final = final_train[mlb_compressed_feature_cols].copy()
y_final = final_train[TARGET_WIN].astype(int).copy()

compressed_best_model = Pipeline([
    ("scaler", StandardScaler()),
    ("logit", LogisticRegression(
        max_iter=5000,
        C=1.0,
        random_state=42
    ))
])

compressed_best_model.fit(X_final, y_final)

# ---------------------------------------------------------
# save files
# ---------------------------------------------------------
MODEL_DIR.mkdir(parents=True, exist_ok=True)

model_file = MODEL_DIR / "mlb_win_model_compressed_best.joblib"
calibrator_file = MODEL_DIR / "mlb_win_model_compressed_best_calibrator.joblib"
meta_file = MODEL_DIR / "mlb_win_model_compressed_best_meta.json"

joblib.dump(compressed_best_model, model_file)

# CALIBRATION AUDIT FIX (2026-08-27, user audit): this used to save
# mlb_iso_calibrator (Cell 7) here -- but that calibrator was fit on a
# DIFFERENT model's walk-forward probability distribution entirely (the
# candidate selected in Cell 6's multi-model comparison, using a different,
# larger feature set), then paired with THIS model's raw output at
# prediction time. The two models' raw probabilities don't occupy the same
# range, and isotonic regression -- monotonic, clamps outside its fitted
# domain -- had almost no calibration-fit samples above ~0.66 raw
# probability from the OTHER model, so it saturated to a flat 100% for any
# input above that point. Confirmed by loading the live artifact directly:
# raw inputs of 0.70/0.75/0.80/0.85 all mapped to calibrated output of
# exactly 1.0. Since this model routinely outputs raw probabilities well
# above 0.66, every such live pick was silently being "calibrated" to a
# fabricated near-certainty. Root-caused via a real graded-history audit:
# 37 OFFICIAL plays over 30 days, -17.3% ROI, model's own stated win
# probability overstated the actual win rate by ~12-21 points in every
# bucket -- and the highest-stated-edge bucket (20%+) was the WORST
# performer (33% win rate), consistent with fabricated top-end confidence.
#
# Fixed by fitting calibration fresh, on THIS model's own walk-forward
# out-of-sample predictions (mlb_compressed_wf_preds, same architecture/
# features as compressed_best_model -- built in the walk-forward cell
# immediately above, same time-respecting folds). Platt scaling (logistic
# regression on logit(raw_prob)) instead of isotonic regression -- a
# logistic sigmoid asymptotes but never reaches exactly 0/1 for a finite
# input, so it can't reproduce this exact saturation failure mode even with
# a sparse tail. A hard [0.04, 0.96] clip is also applied at call time
# (Cell "SCORE TODAY'S MLB BOARD") as a second, independent safety net.
#
# Time-based 70/30 split within the walk-forward predictions themselves --
# fit the calibrator on the earlier 70%, validate on the later 30%, so the
# validation numbers below are genuinely out-of-sample with respect to the
# calibrator too, not just with respect to the original model.
from sklearn.linear_model import LogisticRegression as _PlattLogit
from sklearn.metrics import brier_score_loss as _cal_brier_score_loss

def _mlb_cal_logit(p):
    p = np.clip(np.asarray(p, dtype=float), 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p)).reshape(-1, 1)

_cal_src = mlb_compressed_wf_preds.sort_values("test_start").reset_index(drop=True)
_cal_split = int(len(_cal_src) * 0.70)
_cal_fit_df = _cal_src.iloc[:_cal_split]
_cal_val_df = _cal_src.iloc[_cal_split:]

mlb_platt_calibrator = _PlattLogit(max_iter=1000)
mlb_platt_calibrator.fit(
    _mlb_cal_logit(_cal_fit_df["p_home_win"].values),
    _cal_fit_df[TARGET_WIN].astype(int).values,
)

_val_raw = _cal_val_df["p_home_win"].values
_val_y = _cal_val_df[TARGET_WIN].astype(int).values
_val_cal = np.clip(mlb_platt_calibrator.predict_proba(_mlb_cal_logit(_val_raw))[:, 1], 0.04, 0.96)

print("=" * 70)
print("CALIBRATION FIT + VALIDATION (Platt scaling, held-out 30%)")
print("=" * 70)
print(f"Calibration fit rows : {len(_cal_fit_df)}  ({_cal_fit_df['test_start'].min().date()} to {_cal_fit_df['test_start'].max().date()})")
print(f"Validation rows      : {len(_cal_val_df)}  ({_cal_val_df['test_start'].min().date()} to {_cal_val_df['test_start'].max().date()})")

_val_check = pd.DataFrame({"raw": _val_raw, "cal": _val_cal, "actual": _val_y})
_val_check["raw_bucket"] = pd.cut(_val_check["raw"], bins=[0, 0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 1.0])
_val_summary = _val_check.groupby("raw_bucket", observed=True).agg(
    n=("actual", "count"), avg_raw=("raw", "mean"), avg_cal=("cal", "mean"), actual_rate=("actual", "mean"))
print("\nHeld-out calibration curve (raw vs calibrated vs actual):")
display(_val_summary)

print(f"\nRaw Brier score (held-out): {_cal_brier_score_loss(_val_y, np.clip(_val_raw, 1e-6, 1 - 1e-6)):.4f}")
print(f"Cal Brier score (held-out): {_cal_brier_score_loss(_val_y, _val_cal):.4f}  (lower is better)")

_extreme_test = mlb_platt_calibrator.predict_proba(_mlb_cal_logit([0.80, 0.85, 0.90, 0.95]))[:, 1]
print(f"\nSaturation check -- raw [0.80, 0.85, 0.90, 0.95] -> calibrated {np.round(_extreme_test, 3)} "
      f"(none should be exactly 1.0 -- if any are, do NOT save this calibrator)")
if (_extreme_test >= 0.995).any():
    raise ValueError("Calibrator still saturating near 1.0 on extreme inputs -- refusing to save. Investigate before retrying.")

joblib.dump(mlb_platt_calibrator, calibrator_file)

meta = {
    "model_name": "mlb_win_model_compressed_best",
    "target": TARGET_WIN,
    "feature_set_name": "compressed_diff_heavy",
    "feature_count": len(mlb_compressed_feature_cols),
    "feature_cols": mlb_compressed_feature_cols,
    "train_rows": int(len(final_train)),
    "train_start": str(pd.to_datetime(final_train["date"]).min().date()),
    "train_end": str(pd.to_datetime(final_train["date"]).max().date()),
}

if "mlb_compressed_wf_metrics" in globals() and isinstance(mlb_compressed_wf_metrics, pd.DataFrame) and len(mlb_compressed_wf_metrics):
    meta["walkforward_windows"] = int(len(mlb_compressed_wf_metrics))
    meta["walkforward_avg_accuracy"] = float(mlb_compressed_wf_metrics["accuracy"].mean())
    meta["walkforward_avg_log_loss"] = float(mlb_compressed_wf_metrics["log_loss"].mean())
    meta["walkforward_avg_brier"] = float(mlb_compressed_wf_metrics["brier"].mean())

with open(meta_file, "w", encoding="utf-8") as f:
    json.dump(meta, f, indent=2)

print("==================================================")
print("NEW BEST MLB MODEL SAVED")
print("==================================================")
print("Model file     :", model_file)
print("Calibrator file:", calibrator_file)
print("Metadata file  :", meta_file)
print("")
print("Feature count  :", len(mlb_compressed_feature_cols))
print("Train rows     :", len(final_train))
print("")
print("Compressed feature list:")
for i, c in enumerate(mlb_compressed_feature_cols, 1):
    print(f"{i:>2}. {c}")

mlb_new_best_model = compressed_best_model
mlb_new_best_train = final_train.copy()

CALIBRATION FIT + VALIDATION (Platt scaling, held-out 30%)
Calibration fit rows : 3728  (2024-03-01 to 2025-05-01)
Validation rows      : 1598  (2025-05-01 to 2025-09-01)

Held-out calibration curve (raw vs calibrated vs actual):


,n,avg_raw,avg_cal,actual_rate
raw_bucket,,,,
"(0.0, 0.5]",468,0.472781,0.497919,0.497863
"(0.5, 0.55]",666,0.524886,0.541059,0.510511
"(0.55, 0.6]",383,0.570469,0.578627,0.582245
"(0.6, 0.65]",80,0.615830,0.616083,0.612500
"(0.65, 0.7]",1,0.650224,0.644648,1.000000



Raw Brier score (held-out): 0.2472
Cal Brier score (held-out): 0.2474  (lower is better)

Saturation check -- raw [0.80, 0.85, 0.90, 0.95] -> calibrated [0.774 0.82  0.87  0.926] (none should be exactly 1.0 -- if any are, do NOT save this calibrator)
NEW BEST MLB MODEL SAVED
Model file     : c:\Users\<user>\OneDrive\Documents\KAI Labs\sports\MLB\models\mlb_win_model_compressed_best.joblib
Calibrator file: c:\Users\<user>\OneDrive\Documents\KAI Labs\sports\MLB\models\mlb_win_model_compressed_best_calibrator.joblib
Metadata file  : c:\Users\<user>\OneDrive\Documents\KAI Labs\sports\MLB\models\mlb_win_model_compressed_best_meta.json

Feature count  : 11
Train rows     : 8230

Compressed feature list:
 1. diff_runs_for_avg_10
 2. diff_runs_against_avg_10
 3. diff_margin_avg_10
 4. diff_win_rate_10
 5. diff_runs_for_avg_5
 6. diff_runs_against_avg_5
 7. diff_margin_avg_5
 8. diff_win_rate_5
 9. diff_rest_days
10. home_rest_days
11. away_rest_days


In [59]:
# %%
# =========================================================
# CELL 36 (REPLACEMENT) Ã¢â‚¬â€ RUN NEW BEST 7-FEATURE MODEL
# ON TODAY'S GAMES
# Robust to missing mlb_today_input
# =========================================================

import joblib
import numpy as np
import pandas as pd
from pathlib import Path

print("=" * 60)
print("CELL 36 Ã¢â‚¬â€ TODAY'S MLB WIN PROJECTIONS (SAFE)")
print("=" * 60)

# ---------------------------------------------------------
# helpers
# ---------------------------------------------------------
def first_existing_df(candidates):
    for name in candidates:
        if name in globals() and isinstance(globals()[name], pd.DataFrame):
            df = globals()[name]
            if len(df) > 0:
                print(f"Using input source: {name} | shape={df.shape}")
                return df.copy(), name
    return None, None

def first_existing_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

# ---------------------------------------------------------
# locate today's input table
# ---------------------------------------------------------
input_candidates = [
    "mlb_today_input",
    "today_mlb_board",
    "mlb_board_today",
    "daily_mlb_board",
    "mlb_projection_input",
    "mlb_model_input",
    "board_with_features",
    "scored_mlb_board",
    "mlb_today_features",
]

today_preds, input_name = first_existing_df(input_candidates)

if today_preds is None:
    raise ValueError(
        "Could not find a usable today's MLB input table in memory.\n"
        "Expected one of: "
        "mlb_today_input, today_mlb_board, mlb_board_today, daily_mlb_board, "
        "mlb_projection_input, mlb_model_input, board_with_features, "
        "scored_mlb_board, mlb_today_features."
    )

# ---------------------------------------------------------
# validate model directory
# ---------------------------------------------------------
if "MODEL_DIR" not in globals():
    raise ValueError("MODEL_DIR not found in memory. Define MODEL_DIR first.")

MODEL_DIR = Path(MODEL_DIR)
NEW_BEST_MODEL_PATH = MODEL_DIR / "mlb_win_model_compressed_best.joblib"
NEW_BEST_CALIBRATOR_PATH = MODEL_DIR / "mlb_win_model_compressed_best_calibrator.joblib"

if not NEW_BEST_MODEL_PATH.exists():
    raise FileNotFoundError(f"New best model not found: {NEW_BEST_MODEL_PATH}")

if not NEW_BEST_CALIBRATOR_PATH.exists():
    raise FileNotFoundError(f"New best calibrator not found: {NEW_BEST_CALIBRATOR_PATH}")

print(f"Model path      : {NEW_BEST_MODEL_PATH}")
print(f"Calibrator path : {NEW_BEST_CALIBRATOR_PATH}")

# ---------------------------------------------------------
# ---------------------------------------------------------
# required features: auto-detect from model; compute L5 on-the-fly if missing
# ---------------------------------------------------------
# load model NOW so we can inspect feature_names_in_
best_model      = joblib.load(NEW_BEST_MODEL_PATH)
best_calibrator = joblib.load(NEW_BEST_CALIBRATOR_PATH)

# detect feature set from trained model (sklearn stores feature_names_in_)
try:
    _scaler = best_model.named_steps.get("scaler", None)
    if _scaler is not None and hasattr(_scaler, "feature_names_in_"):
        compressed_features = list(_scaler.feature_names_in_)
    else:
        compressed_features = list(best_model.feature_names_in_)
except AttributeError:
    compressed_features = [
        "diff_runs_for_avg_10",
        "diff_runs_against_avg_10",
        "diff_margin_avg_10",
        "diff_win_rate_10",
        "diff_rest_days",
        "home_rest_days",
        "away_rest_days",
    ]

print(f"Model expects {len(compressed_features)} features: {compressed_features}")

# compute any missing L5 diff columns on-the-fly from home/away snapshot cols
_L5_PAIRS = [
    ("diff_runs_for_avg_5",    "home_runs_for_avg_5",    "away_runs_for_avg_5"),
    ("diff_runs_against_avg_5","home_runs_against_avg_5","away_runs_against_avg_5"),
    ("diff_margin_avg_5",      "home_margin_avg_5",      "away_margin_avg_5"),
    ("diff_win_rate_5",        "home_win_rate_5",        "away_win_rate_5"),
]
for _diff, _home, _away in _L5_PAIRS:
    if _diff in compressed_features and _diff not in today_preds.columns:
        if _home in today_preds.columns and _away in today_preds.columns:
            today_preds[_diff] = today_preds[_home] - today_preds[_away]
            print(f"Computed {_diff} on-the-fly")
        else:
            today_preds[_diff] = float("nan")
            print(f"WARNING: {_diff} set to NaN (no home/away L5 cols in today_preds)")

# if L5 columns could not be computed (old snapshot), fall back to L10-only
_l5_feats = [f for f in compressed_features if "_avg_5" in f or "_rate_5" in f]
if _l5_feats and all(today_preds[f].isna().all() for f in _l5_feats if f in today_preds.columns):
    print("WARNING: L5 diffs are all-NaN. Falling back to L10-only features.")
    print("  -> Run Cell 18 (ESPN refresh) then Cell 23/26 to get L5 in today_preds.")
    compressed_features = [f for f in compressed_features if "_avg_5" not in f and "_rate_5" not in f]

# ---------------------------------------------------------
# resolve key display columns
# ---------------------------------------------------------
away_col = first_existing_col(today_preds, ["away_team", "away_team_name", "team_away"])
home_col = first_existing_col(today_preds, ["home_team", "home_team_name", "team_home"])
time_col = first_existing_col(today_preds, ["game_datetime_utc", "start_time", "game_time_utc", "date", "game_date"])
status_col = first_existing_col(today_preds, ["status"])

if away_col is None or home_col is None:
    raise ValueError(
        "Could not resolve away/home team columns.\n"
        f"Available columns:\n{list(today_preds.columns)}"
    )

if away_col != "away_team":
    today_preds["away_team"] = today_preds[away_col]

if home_col != "home_team":
    today_preds["home_team"] = today_preds[home_col]

if time_col is None:
    today_preds["sort_time_fallback"] = pd.NaT
    time_col = "sort_time_fallback"

# ---------------------------------------------------------
# keep only rows fully ready for prediction
# ---------------------------------------------------------
start_rows = len(today_preds)
# TEMP DIAGNOSTIC (2026-08-14, user audit): identify exactly which row(s) and
# which feature column(s) are NaN before the drop, instead of only reporting
# a before/after row count with no way to tell which game or column caused it.
_diag_dropped_mask = today_preds[compressed_features].isna().any(axis=1)
if _diag_dropped_mask.any():
    print("\n[DIAG] Row(s) about to be dropped by compressed-feature cleanup:")
    for _di, _dr in today_preds[_diag_dropped_mask].iterrows():
        _dnan_cols = [c for c in compressed_features if pd.isna(_dr.get(c))]
        print(f"  {_dr.get('away_team', _dr.get(away_col, '?'))} @ {_dr.get('home_team', _dr.get(home_col, '?'))} -- NaN in: {_dnan_cols}")
    print()

today_preds = today_preds.dropna(subset=compressed_features).copy().reset_index(drop=True)

print(f"Rows before feature cleanup : {start_rows}")
print(f"Rows after feature cleanup  : {len(today_preds)}")

if len(today_preds) == 0:
    raise ValueError("No rows available for prediction after compressed feature cleanup.")

# ---------------------------------------------------------
# load model artifacts
# ---------------------------------------------------------
# (model already loaded above)
# (calibrator already loaded above)

X_today = today_preds[compressed_features].copy()

# ---------------------------------------------------------
# raw + calibrated probabilities
# ---------------------------------------------------------
p_home_raw = best_model.predict_proba(X_today)[:, 1]
# CALIBRATION AUDIT FIX (2026-08-27): best_calibrator now loads whatever the
# "SAVE NEW BEST MLB MODEL" cell most recently saved to this same file path
# -- a plain sklearn LogisticRegression fit on logit(raw_prob) (Platt
# scaling), not the old isotonic-style direct predict(). Same call
# convention as the live scoring cell.
_p_home_raw_logit = np.clip(p_home_raw, 1e-6, 1 - 1e-6)
_p_home_raw_logit = np.log(_p_home_raw_logit / (1 - _p_home_raw_logit)).reshape(-1, 1)
p_home_cal = np.clip(best_calibrator.predict_proba(_p_home_raw_logit)[:, 1], 0.04, 0.96)

p_home_raw = np.clip(p_home_raw, 1e-6, 1 - 1e-6)
p_home_cal = np.clip(p_home_cal, 1e-6, 1 - 1e-6)

today_preds["p_home_win_raw_new"] = p_home_raw
today_preds["p_home_win_new"] = p_home_cal
today_preds["p_away_win_new"] = 1 - today_preds["p_home_win_new"]

today_preds["pred_home_win_new"] = (today_preds["p_home_win_new"] >= 0.5).astype(int)

today_preds["projected_winner_new"] = np.where(
    today_preds["pred_home_win_new"] == 1,
    today_preds["home_team"],
    today_preds["away_team"]
)

today_preds["projected_loser_new"] = np.where(
    today_preds["pred_home_win_new"] == 1,
    today_preds["away_team"],
    today_preds["home_team"]
)

today_preds["win_confidence_new"] = np.where(
    today_preds["pred_home_win_new"] == 1,
    today_preds["p_home_win_new"],
    today_preds["p_away_win_new"]
)

# ---------------------------------------------------------
# sort output safely
# ---------------------------------------------------------
today_preds = today_preds.sort_values(
    ["win_confidence_new", "p_home_win_new", time_col],
    ascending=[False, False, True]
).reset_index(drop=True)

# ---------------------------------------------------------
# console output
# ---------------------------------------------------------
print("\n==================================================")
print("TODAY'S MLB WIN PROJECTIONS Ã¢â‚¬â€ NEW BEST 7-FEATURE MODEL")
print("==================================================")

for _, r in today_preds.iterrows():
    print(f"{r['away_team']} @ {r['home_team']}")
    print(f"   Projected winner : {r['projected_winner_new']}")
    print(f"   Raw home win prob: {r['p_home_win_raw_new']:.3f}")
    print(f"   Cal home win prob: {r['p_home_win_new']:.3f}")
    print(f"   Away win prob    : {r['p_away_win_new']:.3f}")
    print(f"   Confidence       : {r['win_confidence_new']:.3f}")
    print("-" * 70)

# ---------------------------------------------------------
# display table safely
# ---------------------------------------------------------
display_cols = [
    "away_team",
    "home_team",
    "projected_winner_new",
    "projected_loser_new",
    "p_home_win_raw_new",
    "p_home_win_new",
    "p_away_win_new",
    "win_confidence_new",
    "diff_runs_for_avg_10",
    "diff_runs_against_avg_10",
    "diff_margin_avg_10",
    "diff_win_rate_10",
    "diff_rest_days",
    "home_rest_days",
    "away_rest_days",
    "diff_runs_for_avg_10_split",
    "diff_runs_against_avg_10_split",
    "diff_bullpen_stress_3",
    "diff_starter_era_l3",
    "diff_starter_whip_l3",
]

if status_col is not None:
    if status_col != "status":
        today_preds["status"] = today_preds[status_col]
    display_cols.append("status")

display_cols = [c for c in display_cols if c in today_preds.columns]

print("\nDisplay table:")
display(today_preds[display_cols])

# ---------------------------------------------------------
# save in memory
# ---------------------------------------------------------
mlb_today_win_preds_new = today_preds.copy()
globals()["mlb_today_win_preds_new"] = mlb_today_win_preds_new

print("\nCreated in memory:")
print(" - mlb_today_win_preds_new")
print("Done.")

CELL 36 Ã¢â‚¬â€ TODAY'S MLB WIN PROJECTIONS (SAFE)
Using input source: mlb_today_input | shape=(1, 78)
Model path      : c:\Users\<user>\OneDrive\Documents\KAI Labs\sports\MLB\models\mlb_win_model_compressed_best.joblib
Calibrator path : c:\Users\<user>\OneDrive\Documents\KAI Labs\sports\MLB\models\mlb_win_model_compressed_best_calibrator.joblib
Model expects 11 features: ['diff_runs_for_avg_10', 'diff_runs_against_avg_10', 'diff_margin_avg_10', 'diff_win_rate_10', 'diff_runs_for_avg_5', 'diff_runs_against_avg_5', 'diff_margin_avg_5', 'diff_win_rate_5', 'diff_rest_days', 'home_rest_days', 'away_rest_days']
Rows before feature cleanup : 1
Rows after feature cleanup  : 1

TODAY'S MLB WIN PROJECTIONS Ã¢â‚¬â€ NEW BEST 7-FEATURE MODEL
Chicago White Sox @ Cleveland Guardians
   Projected winner : Cleveland Guardians
   Raw home win prob: 0.554
   Cal home win prob: 0.565
   Away win prob    : 0.435
   Confidence       : 0.565
----------------------------------------------------------------

,away_team,home_team,projected_winner_new,projected_loser_new,p_home_win_raw_new,p_home_win_new,p_away_win_new,win_confidence_new,diff_runs_for_avg_10,diff_runs_against_avg_10,diff_margin_avg_10,diff_win_rate_10,diff_rest_days,home_rest_days,away_rest_days,diff_runs_for_avg_10_split,diff_runs_against_avg_10_split,diff_bullpen_stress_3,status
0,Chicago White Sox,Cleveland Guardians,Cleveland Guardians,Chicago White Sox,0.554307,0.565308,0.434692,0.565308,0.571773,-1.2167,1.788473,0.0737,0.0,2.0,2.0,-1.951508,0.324826,-8.35,Scheduled



Created in memory:
 - mlb_today_win_preds_new
Done.


In [60]:
# %%
# =========================================================
# CELL 37 Ã¢â‚¬â€ BUILD CLEAN DAILY PROJECTION BOARD
# Use the NEW BEST 7-feature model outputs for readable cards
# =========================================================

import pandas as pd
import numpy as np

if "mlb_today_win_preds_new" not in globals() or not isinstance(mlb_today_win_preds_new, pd.DataFrame):
    raise ValueError("mlb_today_win_preds_new not found. Run Cell 36 first.")

board = mlb_today_win_preds_new.copy()

# ---------------------------------------------------------
# build matchup key
# ---------------------------------------------------------
def clean_team_name(s):
    if pd.isna(s):
        return None
    return str(s).strip().lower()

board["away_key"] = board["away_team"].apply(clean_team_name)
board["home_key"] = board["home_team"].apply(clean_team_name)
board["matchup_key"] = board["away_key"] + " @ " + board["home_key"]

# ---------------------------------------------------------
# readable projection fields from NEW model
# ---------------------------------------------------------
board["projected_winner"] = board["projected_winner_new"]
board["projected_loser"] = board["projected_loser_new"]

board["projected_win_prob"] = np.where(
    board["projected_winner"] == board["home_team"],
    board["p_home_win_new"],
    board["p_away_win_new"]
)

board["projected_win_pct"] = (100 * board["projected_win_prob"]).round(1)
board["confidence_edge_from_50"] = board["projected_win_prob"] - 0.50

def strength_label(p):
    if pd.isna(p):
        return "UNKNOWN"
    if p >= 0.60:
        return "A"
    if p >= 0.58:
        return "B"
    if p >= 0.55:
        return "C"
    return "D"

board["strength_tier"] = board["projected_win_prob"].apply(strength_label)

# still a proxy, not a true margin model
board["proj_margin_proxy"] = (board["confidence_edge_from_50"] * 20).round(1)

# ---------------------------------------------------------
# dedupe doubleheaders for main readable board
# keep higher-confidence row per matchup
# ---------------------------------------------------------
board_main = (
    board.sort_values(
        ["matchup_key", "projected_win_prob", "game_datetime_utc"],
        ascending=[True, False, True]
    )
    .drop_duplicates(subset=["matchup_key"], keep="first")
    .sort_values(["projected_win_prob", "game_datetime_utc"], ascending=[False, True])
    .reset_index(drop=True)
)

print("FULL BOARD SHAPE (with duplicates):", board.shape)
print("MAIN BOARD SHAPE (deduped matchups):", board_main.shape)

print("\n==================================================")
print("CLEAN DAILY MLB PROJECTION BOARD Ã¢â‚¬â€ NEW BEST MODEL")
print("==================================================")

for i, (_, r) in enumerate(board_main.iterrows(), 1):
    print(f"#{i} {r['away_team']} @ {r['home_team']}")
    print(f"   Projected winner : {r['projected_winner']}")
    print(f"   Win probability  : {r['projected_win_prob']:.3f}")
    print(f"   Win pct          : {r['projected_win_pct']:.1f}%")
    print(f"   Strength tier    : {r['strength_tier']}")
    print(f"   Margin proxy     : {r['proj_margin_proxy']:+.1f}")
    print(f"   Status           : {r['status']}")
    print("-" * 90)

display_cols = [
    "game_id",
    "game_datetime_utc",
    "away_team",
    "home_team",
    "projected_winner",
    "projected_loser",
    "projected_win_prob",
    "projected_win_pct",
    "strength_tier",
    "proj_margin_proxy",
    "p_home_win_new",
    "p_away_win_new",
    "status",
    "venue_name",
]

display(board_main[display_cols])

mlb_projection_board = board_main.copy()
mlb_projection_board_full = board.copy()

FULL BOARD SHAPE (with duplicates): (1, 95)
MAIN BOARD SHAPE (deduped matchups): (1, 95)

CLEAN DAILY MLB PROJECTION BOARD Ã¢â‚¬â€ NEW BEST MODEL
#1 Chicago White Sox @ Cleveland Guardians
   Projected winner : Cleveland Guardians
   Win probability  : 0.565
   Win pct          : 56.5%
   Strength tier    : C
   Margin proxy     : +1.3
   Status           : Scheduled
------------------------------------------------------------------------------------------


,game_id,game_datetime_utc,away_team,home_team,projected_winner,projected_loser,projected_win_prob,projected_win_pct,strength_tier,proj_margin_proxy,p_home_win_new,p_away_win_new,status,venue_name
0,849831,2026-10-11 00:00:00+00:00,Chicago White Sox,Cleveland Guardians,Cleveland Guardians,Chicago White Sox,0.565308,56.5,C,1.3,0.565308,0.434692,Scheduled,Progressive Field


In [61]:
# %%
# =========================================================
# CELL 38 Ã¢â‚¬â€ STRICT FINAL CARD BUILDER (NEW BEST MODEL)
# Uses the new 7-feature projection board
# Real betting card if live prices exist
# Otherwise builds a strict projection/social card
# =========================================================

import pandas as pd
import numpy as np

if "mlb_projection_board" not in globals() or not isinstance(mlb_projection_board, pd.DataFrame):
    raise ValueError("mlb_projection_board not found. Run Cell 37 first.")

proj = mlb_projection_board.copy()

# ---------------------------------------------------------
# strict posting rules from history
# ---------------------------------------------------------
STRICT_SOCIAL_MIN_PROB = 0.59
FALLBACK_SOCIAL_MIN_PROB = 0.58
MAX_SOCIAL_PLAYS = 1

bet_card = None

# ---------------------------------------------------------
# try real betting card first if usable market prices exist
# ---------------------------------------------------------
if "mlb_kalshi_with_model" in globals() and isinstance(mlb_kalshi_with_model, pd.DataFrame):
    km = mlb_kalshi_with_model.copy()

    if len(km):
        km["bet_side"] = np.where(km["recommended_side"] == "YES", "YES", "NO")
        km["bet_team"] = km["recommended_team"]

        km["p_model_bet"] = np.where(
            km["bet_side"] == "YES",
            km["p_model_yes"],
            km["p_model_no"]
        )

        km["p_market_bet"] = np.where(
            km["bet_side"] == "YES",
            km["p_market_yes"],
            km["p_market_no"]
        )

        km["edge_bet"] = km["p_model_bet"] - km["p_market_bet"]

        km_live = km[km["p_market_bet"].notna()].copy()

        if len(km_live):
            km_live = km_live[
                (km_live["p_model_bet"] >= MIN_CONF) &
                (km_live["edge_bet"] >= MIN_EDGE) &
                (km_live["p_market_bet"] >= MIN_PRICE) &
                (km_live["p_market_bet"] <= MAX_PRICE)
            ].copy()

            if len(km_live):
                km_live["matchup_key"] = (
                    km_live["away_team"].astype(str).str.lower().str.strip()
                    + " @ " +
                    km_live["home_team"].astype(str).str.lower().str.strip()
                )

                km_live = (
                    km_live.sort_values(
                        ["matchup_key", "edge_bet", "p_model_bet"],
                        ascending=[True, False, False]
                    )
                    .drop_duplicates(subset=["matchup_key"], keep="first")
                    .sort_values(["edge_bet", "p_model_bet"], ascending=[False, False])
                    .reset_index(drop=True)
                )

                def size_from_edge(edge, conf):
                    if edge >= 0.14 and conf >= 0.62:
                        return "1.5 UNIT"
                    if edge >= 0.10 and conf >= 0.59:
                        return "1 UNIT"
                    if edge >= 0.07 and conf >= 0.565:
                        return "0.75 UNIT"
                    return "0.5 UNIT"

                km_live["bet_size"] = [
                    size_from_edge(e, c)
                    for e, c in zip(km_live["edge_bet"], km_live["p_model_bet"])
                ]

                km_live["official_bet"] = [
                    f"Bet {side} Ã¢â‚¬â€ {team}"
                    for side, team in zip(km_live["bet_side"], km_live["bet_team"])
                ]

                km_live["social_bet"] = [
                    f"{team} | {side} @ {price:.3f} | Edge {edge:.3f} | {size}"
                    for team, side, price, edge, size in zip(
                        km_live["bet_team"],
                        km_live["bet_side"],
                        km_live["p_market_bet"],
                        km_live["edge_bet"],
                        km_live["bet_size"]
                    )
                ]

                bet_card = km_live.copy()

# ---------------------------------------------------------
# fallback: strict projection-only social card
# ---------------------------------------------------------
if bet_card is None or len(bet_card) == 0:
    social_card = proj.copy()

    # strict threshold first
    social_card = social_card[
        social_card["projected_win_prob"] >= STRICT_SOCIAL_MIN_PROB
    ].copy()

    # fallback to slightly softer threshold
    if len(social_card) == 0:
        social_card = proj.copy()
        social_card = social_card[
            social_card["projected_win_prob"] >= FALLBACK_SOCIAL_MIN_PROB
        ].copy()

    # if still nothing, just keep top play
    if len(social_card) == 0:
        social_card = (
            proj.sort_values(
                ["projected_win_prob", "game_datetime_utc"],
                ascending=[False, True]
            )
            .head(1)
            .copy()
        )

    social_card = (
        social_card.sort_values(
            ["projected_win_prob", "game_datetime_utc"],
            ascending=[False, True]
        )
        .head(MAX_SOCIAL_PLAYS)
        .reset_index(drop=True)
    )

    def size_from_prob(p):
        if p >= 0.60:
            return "1 UNIT"
        if p >= 0.59:
            return "0.75 UNIT"
        if p >= 0.58:
            return "0.5 UNIT"
        return "0.25 UNIT"

    social_card["play_type"] = "Projection Only"
    social_card["bet_size"] = social_card["projected_win_prob"].apply(size_from_prob)
    social_card["official_bet"] = [f"{winner} ML" for winner in social_card["projected_winner"]]
    social_card["social_bet"] = [
        f"{winner} ML | Win Prob {prob:.3f} | {size}"
        for winner, prob, size in zip(
            social_card["projected_winner"],
            social_card["projected_win_prob"],
            social_card["bet_size"]
        )
    ]

    print("==================================================")
    print("EKO BETS Ã¢â‚¬â€ STRICT MLB PROJECTION / SOCIAL CARD")
    print("==================================================")
    print("No usable live market prices found.")
    print("Built strict projection-only social card instead.\n")

    for i, (_, r) in enumerate(social_card.iterrows(), 1):
        print(f"#{i} {r['away_team']} @ {r['home_team']}")
        print(f"   Projected winner: {r['projected_winner']}")
        print(f"   Win probability : {r['projected_win_prob']:.3f}")
        print(f"   Strength tier   : {r['strength_tier']}")
        print(f"   Margin proxy    : {r['proj_margin_proxy']:+.1f}")
        print(f"   Play            : {r['official_bet']}")
        print(f"   Size            : {r['bet_size']}")
        print("-" * 60)

    print("\n==================================================")
    print("EKO BETS Ã¢â‚¬â€ STRICT MLB SOCIAL MEDIA CARD")
    print("==================================================")
    for i, (_, r) in enumerate(social_card.iterrows(), 1):
        print(f"{i}. {r['social_bet']}")

    final_mlb_card = social_card.copy()

else:
    print("==================================================")
    print("EKO BETS Ã¢â‚¬â€ FINAL MLB BETTING CARD")
    print("==================================================")

    for i, (_, r) in enumerate(bet_card.iterrows(), 1):
        print(f"#{i} {r['away_team']} @ {r['home_team']}")
        print(f"   Projected winner : {r['projected_winner']}")
        print(f"   Bet              : {r['official_bet']}")
        print(f"   Entry            : {r['p_market_bet']:.3f}")
        print(f"   Model for bet    : {r['p_model_bet']:.3f}")
        print(f"   Market for bet   : {r['p_market_bet']:.3f}")
        print(f"   Edge             : {r['edge_bet']:.3f}")
        print(f"   Size             : {r['bet_size']}")
        print("-" * 60)

    print("\n==================================================")
    print("EKO BETS Ã¢â‚¬â€ MLB SOCIAL MEDIA CARD")
    print("==================================================")
    for i, (_, r) in enumerate(bet_card.iterrows(), 1):
        print(f"{i}. {r['social_bet']}")

    final_mlb_card = bet_card.copy()

EKO BETS Ã¢â‚¬â€ STRICT MLB PROJECTION / SOCIAL CARD
No usable live market prices found.
Built strict projection-only social card instead.



#1 Chicago White Sox @ Cleveland Guardians
   Projected winner: Cleveland Guardians
   Win probability : 0.565
   Strength tier   : C
   Margin proxy    : +1.3
   Play            : Cleveland Guardians ML
   Size            : 0.25 UNIT
------------------------------------------------------------

EKO BETS Ã¢â‚¬â€ STRICT MLB SOCIAL MEDIA CARD
1. Cleveland Guardians ML | Win Prob 0.565 | 0.25 UNIT


In [62]:
# %%
# =========================================================
# CELL 39 Ã¢â‚¬â€ SAVE STRICT FINAL MLB CARD
# Save the final strict card for posting + tracking
# =========================================================

import pandas as pd
from pathlib import Path

if "final_mlb_card" not in globals() or not isinstance(final_mlb_card, pd.DataFrame):
    raise ValueError("final_mlb_card not found. Run Cell 38 first.")

SAVE_DIR = Path(r"c:\Users\<user>\OneDrive\Documents\KAI Labs\sports\MLB\tracking")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

today_stamp = pd.Timestamp.today().strftime("%Y-%m-%d")
outfile = SAVE_DIR / f"mlb_strict_final_card_{today_stamp}.csv"

card = final_mlb_card.copy()
card["card_date"] = today_stamp
card["card_type"] = "strict_projection_social"

save_cols = [c for c in [
    "card_date",
    "card_type",
    "game_id",
    "game_datetime_utc",
    "away_team",
    "home_team",
    "projected_winner",
    "projected_loser",
    "projected_win_prob",
    "projected_win_pct",
    "strength_tier",
    "proj_margin_proxy",
    "official_bet",
    "social_bet",
    "bet_size",
    "status",
    "venue_name",
] if c in card.columns]

card[save_cols].to_csv(outfile, index=False)

print("Saved strict final MLB card to:")
print(outfile)

print("\nSaved rows:", len(card))
display(card[save_cols])

Saved strict final MLB card to:
c:\Users\<user>\OneDrive\Documents\KAI Labs\sports\MLB\tracking\mlb_strict_final_card_2026-10-10.csv

Saved rows: 1


,card_date,card_type,game_id,game_datetime_utc,away_team,home_team,projected_winner,projected_loser,projected_win_prob,projected_win_pct,strength_tier,proj_margin_proxy,official_bet,social_bet,bet_size,status,venue_name
0,2026-10-10,strict_projection_social,849831,2026-10-11 00:00:00+00:00,Chicago White Sox,Cleveland Guardians,Cleveland Guardians,Chicago White Sox,0.565308,56.5,C,1.3,Cleveland Guardians ML,Cleveland Guardians ML | Win Prob 0.565 | 0.25...,0.25 UNIT,Scheduled,Progressive Field


In [63]:
# %%
# =========================================================
# CELL 40 Ã¢â‚¬â€ PULL TODAY'S PROBABLE STARTING PITCHERS
# Adds today's probable starters to the live slate so you
# can begin building pitcher-aware daily workflows
# =========================================================

import pandas as pd
import numpy as np
import requests

if "mlb_today_games" not in globals() or not isinstance(mlb_today_games, pd.DataFrame):
    raise ValueError("mlb_today_games not found. Run Cell 11 first.")

today_games_sp = mlb_today_games.copy()

# ---------------------------------------------------------
# fetch today's schedule again, but specifically parse
# probable pitchers from the MLB StatsAPI schedule payload
# ---------------------------------------------------------
today_str = CARD_DATE  # set in Cell 24 (tomorrow if run after 8 PM)
url = f"https://statsapi.mlb.com/api/v1/schedule?sportId=1&date={today_str}&hydrate=probablePitcher"

print("Fetching probable starters for:", today_str)
print("URL:", url)

try:
    r = requests.get(url, timeout=30)
    r.raise_for_status()
    data = r.json()
except Exception as e:
    raise RuntimeError(f"Failed to fetch MLB schedule for starters: {e}")

starter_rows = []
for date_block in data.get("dates", []):
    for g in date_block.get("games", []):
        away_probable = g.get("teams", {}).get("away", {}).get("probablePitcher", {}) or {}
        home_probable = g.get("teams", {}).get("home", {}).get("probablePitcher", {}) or {}

        starter_rows.append({
            "game_id": g.get("gamePk"),
            "away_probable_pitcher_name": away_probable.get("fullName"),
            "away_probable_pitcher_id": away_probable.get("id"),
            "home_probable_pitcher_name": home_probable.get("fullName"),
            "home_probable_pitcher_id": home_probable.get("id"),
        })

starter_df = pd.DataFrame(starter_rows)

print("\nStarter rows fetched:", len(starter_df))

merged = today_games_sp.merge(starter_df, on="game_id", how="left").copy()

print("Games with away probable starter:", int(merged["away_probable_pitcher_name"].notna().sum()))
print("Games with home probable starter:", int(merged["home_probable_pitcher_name"].notna().sum()))

print("\nToday's slate with probable starters:")
display_cols = [
    "game_id",
    "away_team",
    "home_team",
    "away_probable_pitcher_name",
    "home_probable_pitcher_name",
    "status",
]
display(merged[display_cols])

mlb_today_games_with_starters = merged.copy()

Fetching probable starters for: 2026-10-10
URL: https://statsapi.mlb.com/api/v1/schedule?sportId=1&date=2026-10-10&hydrate=probablePitcher



Starter rows fetched: 1


Games with away probable starter: 1
Games with home probable starter: 1

Today's slate with probable starters:


,game_id,away_team,home_team,away_probable_pitcher_name,home_probable_pitcher_name,status
0,849831,Chicago White Sox,Cleveland Guardians,Sean Burke,Gavin Williams,Scheduled


In [64]:
# %%
# =========================================================
# CELL 41 Ã¢â‚¬â€ PULL ACTUAL STARTING PITCHERS FROM GAME FEEDS
# Fallback for finished/in-progress games when probable
# starters are missing from the schedule endpoint
# =========================================================

import pandas as pd
import numpy as np
import requests
import time

if "mlb_today_games_with_starters" not in globals() or not isinstance(mlb_today_games_with_starters, pd.DataFrame):
    raise ValueError("mlb_today_games_with_starters not found. Run Cell 40 first.")

games_sp = mlb_today_games_with_starters.copy()

if "game_id" not in games_sp.columns:
    raise ValueError("game_id missing from mlb_today_games_with_starters.")

# ---------------------------------------------------------
# [SP STALE SLATE CHECK] (2026-08-08) -- same failure mode as the
# board's own [STALE SLATE CHECK] (FINAL MLB CARD SELECTOR cell): a
# leftover mlb_today_games_with_starters from a prior kernel session
# silently reused as if it were CARD_DATE's slate. That fix only
# covers the market-odds board, not this SP feed -- confirmed live
# this exact gap: a card's two OFFICIAL-tier plays (Tigers ML,
# Brewers ML) displayed "[confirmed via Cell 41/42 live feed]"
# starters (Keider Montero/JT Brubaker, Zebby Matthews/Shane Drohan)
# that turned out to be the PRIOR day's actual completed starters,
# not the CARD_DATE game's real probables (Jackson Jobe/Landen Roupp,
# Taj Bradley/Robert Gasser per MLB.com) -- because fetch_actual_
# starters_from_game_feed(game_id) faithfully returns real data for
# whatever game_id it's given, so a stale game_id from an unrefreshed
# prior-session games_sp produces a real, live-feed-confirmed-looking
# result for the WRONG day's game, with no signal anywhere that this
# happened. Warns only (matches the board check's behavior) rather
# than blocking, since a same-day early-morning run before StatsAPI
# has posted today's probables is a legitimate, non-stale reason for
# this to look sparse too.
if "game_datetime_utc" in games_sp.columns and "CARD_DATE" in globals():
    _sp_slate_dates_et = (
        pd.to_datetime(games_sp["game_datetime_utc"], errors="coerce", utc=True) - pd.Timedelta(hours=4)
    ).dt.normalize().dt.tz_localize(None)
    _sp_card_date_ts = pd.Timestamp(CARD_DATE)
    _sp_stale_rows = _sp_slate_dates_et.notna() & (_sp_slate_dates_et != _sp_card_date_ts)
    if _sp_stale_rows.any():
        print("=" * 80)
        print(f"[SP STALE SLATE WARNING] mlb_today_games_with_starters has {_sp_stale_rows.sum()}/{len(games_sp)} "
              f"game(s) NOT matching CARD_DATE={CARD_DATE}.")
        print(f"  Game date(s) found: {sorted(set(_sp_slate_dates_et.dropna().dt.date))}")
        print("  Starting pitchers below are likely leftover from a previous session -- "
              "re-run from Cell 11 (schedule pull) onward before trusting any SP-based edge on this card.")
        print("=" * 80)

def fetch_actual_starters_from_game_feed(game_id):
    url = f"https://statsapi.mlb.com/api/v1.1/game/{int(game_id)}/feed/live"
    r = requests.get(url, timeout=30)
    r.raise_for_status()
    data = r.json()

    game_data = data.get("gameData", {})
    live_data = data.get("liveData", {})
    boxscore = live_data.get("boxscore", {})
    teams_box = boxscore.get("teams", {})

    away_team_name = game_data.get("teams", {}).get("away", {}).get("name")
    home_team_name = game_data.get("teams", {}).get("home", {}).get("name")

    away_pitchers = teams_box.get("away", {}).get("pitchers", []) or []
    home_pitchers = teams_box.get("home", {}).get("pitchers", []) or []

    away_players = teams_box.get("away", {}).get("players", {}) or {}
    home_players = teams_box.get("home", {}).get("players", {}) or {}

    def resolve_first_pitcher(pitcher_ids, players_dict):
        if not pitcher_ids:
            return (None, None)
        first_pid = pitcher_ids[0]
        player_key = f"ID{first_pid}"
        player_obj = players_dict.get(player_key, {}) or {}
        return (
            player_obj.get("person", {}).get("fullName"),
            first_pid
        )

    away_starter_name, away_starter_id = resolve_first_pitcher(away_pitchers, away_players)
    home_starter_name, home_starter_id = resolve_first_pitcher(home_pitchers, home_players)

    return {
        "game_id": game_id,
        "api_away_team": away_team_name,
        "api_home_team": home_team_name,
        "away_starting_pitcher_name": away_starter_name,
        "away_starting_pitcher_id": away_starter_id,
        "home_starting_pitcher_name": home_starter_name,
        "home_starting_pitcher_id": home_starter_id,
    }

starter_rows = []
game_ids = games_sp["game_id"].dropna().astype(int).unique().tolist()

print("Pulling actual starters from game feeds...")
print("Games to check:", len(game_ids))

for i, gid in enumerate(game_ids, 1):
    try:
        row = fetch_actual_starters_from_game_feed(gid)
        starter_rows.append(row)
        print(f"[{i}/{len(game_ids)}] OK   - {gid}")
    except Exception as e:
        print(f"[{i}/{len(game_ids)}] FAIL - {gid} -> {e}")
        starter_rows.append({
            "game_id": gid,
            "api_away_team": None,
            "api_home_team": None,
            "away_starting_pitcher_name": None,
            "away_starting_pitcher_id": None,
            "home_starting_pitcher_name": None,
            "home_starting_pitcher_id": None,
        })
    time.sleep(0.05)

starter_feed_df = pd.DataFrame(starter_rows)

games_sp = games_sp.merge(starter_feed_df, on="game_id", how="left")

# 2026-07-12: priority was backwards. This was written as "fill probable
# columns with actual starters if probable was missing" -- but .fillna()
# means the FIRST argument wins whenever it's non-null, so the stale
# scheduled "probable" name always won and the live game-feed-confirmed
# starter (fetch_actual_starters_from_game_feed, right above) only fired
# when the probable field was completely empty -- which it almost never
# is, since MLB populates that field days in advance. This is exactly
# backwards for the case that matters: a pitcher gets scratched (deGrom,
# Gallen both confirmed 2026-07-12) and the schedule endpoint's "probable"
# field is stale-but-present, not missing -- so the live, correct
# replacement starter sitting right there in away_starting_pitcher_name/
# home_starting_pitcher_name was being silently discarded every time.
# Real starters take priority now; falls back to the scheduled probable
# only when the live feed genuinely has no data yet (early in the day,
# before MLB posts confirmed lineups).
if "away_probable_pitcher_name" in games_sp.columns:
    games_sp["away_pitcher_final_name"] = games_sp["away_starting_pitcher_name"].fillna(games_sp["away_probable_pitcher_name"])
else:
    games_sp["away_pitcher_final_name"] = games_sp["away_starting_pitcher_name"]

if "home_probable_pitcher_name" in games_sp.columns:
    games_sp["home_pitcher_final_name"] = games_sp["home_starting_pitcher_name"].fillna(games_sp["home_probable_pitcher_name"])
else:
    games_sp["home_pitcher_final_name"] = games_sp["home_starting_pitcher_name"]

print("\nGames with away starter found:", int(games_sp["away_pitcher_final_name"].notna().sum()))
print("Games with home starter found:", int(games_sp["home_pitcher_final_name"].notna().sum()))

print("\nToday's slate with actual/probable starters:")
display_cols = [
    "game_id",
    "away_team",
    "home_team",
    "away_pitcher_final_name",
    "home_pitcher_final_name",
    "status",
]
display(games_sp[display_cols])

mlb_today_games_with_actual_starters = games_sp.copy()

Pulling actual starters from game feeds...
Games to check: 1


[1/1] OK   - 849831


Games with away starter found: 1
Games with home starter found: 1

Today's slate with actual/probable starters:


,game_id,away_team,home_team,away_pitcher_final_name,home_pitcher_final_name,status
0,849831,Chicago White Sox,Cleveland Guardians,Sean Burke,Gavin Williams,Scheduled


In [65]:
# %%
# =========================================================
# CELL 42 Ã¢â‚¬â€ BUILD TODAY'S STARTER MATCHUP BOARD
# Clean starter names/IDs and attach them to the current
# projection board so you can see pitcher context with picks
# =========================================================

import pandas as pd
import numpy as np

if "mlb_today_games_with_actual_starters" not in globals() or not isinstance(mlb_today_games_with_actual_starters, pd.DataFrame):
    raise ValueError("mlb_today_games_with_actual_starters not found. Run Cell 41 first.")

if "mlb_projection_board" not in globals() or not isinstance(mlb_projection_board, pd.DataFrame):
    raise ValueError("mlb_projection_board not found. Run Cell 37 first.")

starter_board = mlb_today_games_with_actual_starters.copy()
proj = mlb_projection_board.copy()

# ---------------------------------------------------------
# normalize matchup keys for merge
# ---------------------------------------------------------
def clean_team(s):
    if pd.isna(s):
        return None
    s = str(s).strip().lower()
    if s == "athletics":
        s = "oakland athletics"
    return s

starter_board["away_key"] = starter_board["away_team"].apply(clean_team)
starter_board["home_key"] = starter_board["home_team"].apply(clean_team)
starter_board["matchup_key"] = starter_board["away_key"] + " @ " + starter_board["home_key"]

proj["away_key"] = proj["away_team"].apply(clean_team)
proj["home_key"] = proj["home_team"].apply(clean_team)
proj["matchup_key"] = proj["away_key"] + " @ " + proj["home_key"]

# ---------------------------------------------------------
# keep one starter row per matchup
# for doubleheaders, keep earliest game_id / datetime
# ---------------------------------------------------------
sort_cols = [c for c in ["matchup_key", "game_datetime_utc", "game_id"] if c in starter_board.columns]
starter_board = (
    starter_board.sort_values(sort_cols, ascending=True)
    .drop_duplicates(subset=["matchup_key"], keep="first")
    .reset_index(drop=True)
)

starter_merge_cols = [
    "matchup_key",
    "away_pitcher_final_name",
    "home_pitcher_final_name",
    "away_starting_pitcher_id",
    "home_starting_pitcher_id",
]

starter_merge_cols = [c for c in starter_merge_cols if c in starter_board.columns]

proj_with_starters = proj.merge(
    starter_board[starter_merge_cols].drop_duplicates(subset=["matchup_key"]),
    on="matchup_key",
    how="left"
).copy()

# ---------------------------------------------------------
# readable starter matchup field
# ---------------------------------------------------------
proj_with_starters["starter_matchup"] = np.where(
    proj_with_starters["away_pitcher_final_name"].notna() & proj_with_starters["home_pitcher_final_name"].notna(),
    proj_with_starters["away_pitcher_final_name"].astype(str) + " vs " + proj_with_starters["home_pitcher_final_name"].astype(str),
    None
)

print("==================================================")
print("TODAY'S MLB PROJECTION BOARD WITH STARTERS")
print("==================================================")

for i, (_, r) in enumerate(proj_with_starters.iterrows(), 1):
    print(f"#{i} {r['away_team']} @ {r['home_team']}")
    print(f"   Projected winner : {r['projected_winner']}")
    print(f"   Win probability  : {r['projected_win_prob']:.3f}")
    print(f"   Away starter     : {r['away_pitcher_final_name']}")
    print(f"   Home starter     : {r['home_pitcher_final_name']}")
    print("-" * 80)

display_cols = [
    "game_id",
    "away_team",
    "home_team",
    "away_pitcher_final_name",
    "home_pitcher_final_name",
    "projected_winner",
    "projected_win_prob",
    "strength_tier",
    "status",
]

display(proj_with_starters[display_cols])

mlb_projection_board_with_starters = proj_with_starters.copy()

TODAY'S MLB PROJECTION BOARD WITH STARTERS
#1 Chicago White Sox @ Cleveland Guardians
   Projected winner : Cleveland Guardians
   Win probability  : 0.565
   Away starter     : Sean Burke
   Home starter     : Gavin Williams
--------------------------------------------------------------------------------


,game_id,away_team,home_team,away_pitcher_final_name,home_pitcher_final_name,projected_winner,projected_win_prob,strength_tier,status
0,849831,Chicago White Sox,Cleveland Guardians,Sean Burke,Gavin Williams,Cleveland Guardians,0.565308,C,Scheduled


In [66]:
# %%
# =========================================================
# CELL 43 Ã¢â‚¬â€ ATTACH STARTERS TO FINAL CARD + BUILD POST TEXT
# Adds starter names to your strict final MLB card and creates
# cleaner social/posting text without changing the model
# =========================================================

import pandas as pd
import numpy as np

if "final_mlb_card" not in globals() or not isinstance(final_mlb_card, pd.DataFrame):
    raise ValueError("final_mlb_card not found. Run Cell 38 first.")

if "mlb_projection_board_with_starters" not in globals() or not isinstance(mlb_projection_board_with_starters, pd.DataFrame):
    raise ValueError("mlb_projection_board_with_starters not found. Run Cell 42 first.")

card = final_mlb_card.copy()
starter_board = mlb_projection_board_with_starters.copy()

# ---------------------------------------------------------
# normalize keys for merge
# ---------------------------------------------------------
def clean_team(s):
    if pd.isna(s):
        return None
    s = str(s).strip().lower()
    if s == "athletics":
        s = "oakland athletics"
    return s

card["away_key"] = card["away_team"].apply(clean_team)
card["home_key"] = card["home_team"].apply(clean_team)

starter_board["away_key"] = starter_board["away_team"].apply(clean_team)
starter_board["home_key"] = starter_board["home_team"].apply(clean_team)

merge_cols = [
    "away_key",
    "home_key",
    "away_pitcher_final_name",
    "home_pitcher_final_name",
    "starter_matchup",
]

merge_cols = [c for c in merge_cols if c in starter_board.columns]

card = card.merge(
    starter_board[merge_cols].drop_duplicates(subset=["away_key", "home_key"]),
    on=["away_key", "home_key"],
    how="left"
).copy()

# ---------------------------------------------------------
# readable starter fields
# ---------------------------------------------------------
card["pick_pitcher"] = np.where(
    card["projected_winner"] == card["away_team"],
    card["away_pitcher_final_name"],
    card["home_pitcher_final_name"]
)

card["fade_pitcher"] = np.where(
    card["projected_winner"] == card["away_team"],
    card["home_pitcher_final_name"],
    card["away_pitcher_final_name"]
)

card["starter_note"] = np.where(
    card["pick_pitcher"].notna() & card["fade_pitcher"].notna(),
    card["pick_pitcher"].astype(str) + " over " + card["fade_pitcher"].astype(str),
    None
)

# ---------------------------------------------------------
# build cleaner social text with starters
# ---------------------------------------------------------
card["social_bet_with_starters"] = np.where(
    card["starter_note"].notna(),
    card["social_bet"].astype(str) + " | " + card["starter_note"].astype(str),
    card["social_bet"].astype(str)
)

# ---------------------------------------------------------
# print board
# ---------------------------------------------------------
print("==================================================")
print("EKO BETS Ã¢â‚¬â€ FINAL MLB CARD WITH STARTERS")
print("==================================================")

for i, (_, r) in enumerate(card.iterrows(), 1):
    print(f"#{i} {r['away_team']} @ {r['home_team']}")
    print(f"   Play            : {r['official_bet']}")
    print(f"   Projected winner: {r['projected_winner']}")
    if "projected_win_prob" in card.columns and pd.notna(r.get("projected_win_prob", np.nan)):
        print(f"   Win probability : {r['projected_win_prob']:.3f}")
    print(f"   Away starter    : {r['away_pitcher_final_name']}")
    print(f"   Home starter    : {r['home_pitcher_final_name']}")
    if pd.notna(r.get("starter_note", None)):
        print(f"   Starter note    : {r['starter_note']}")
    print(f"   Size            : {r['bet_size']}")
    print("-" * 60)

print("\n==================================================")
print("EKO BETS Ã¢â‚¬â€ STARTER-AWARE SOCIAL CARD")
print("==================================================")
for i, (_, r) in enumerate(card.iterrows(), 1):
    print(f"{i}. {r['social_bet_with_starters']}")

display_cols = [
    "away_team",
    "home_team",
    "away_pitcher_final_name",
    "home_pitcher_final_name",
    "projected_winner",
    "projected_win_prob",
    "official_bet",
    "bet_size",
    "social_bet_with_starters",
]

display_cols = [c for c in display_cols if c in card.columns]
display(card[display_cols])

final_mlb_card_with_starters = card.copy()

EKO BETS Ã¢â‚¬â€ FINAL MLB CARD WITH STARTERS
#1 Chicago White Sox @ Cleveland Guardians
   Play            : Cleveland Guardians ML
   Projected winner: Cleveland Guardians
   Win probability : 0.565
   Away starter    : Sean Burke
   Home starter    : Gavin Williams
   Starter note    : Gavin Williams over Sean Burke
   Size            : 0.25 UNIT
------------------------------------------------------------

EKO BETS Ã¢â‚¬â€ STARTER-AWARE SOCIAL CARD
1. Cleveland Guardians ML | Win Prob 0.565 | 0.25 UNIT | Gavin Williams over Sean Burke


,away_team,home_team,away_pitcher_final_name,home_pitcher_final_name,projected_winner,projected_win_prob,official_bet,bet_size,social_bet_with_starters
0,Chicago White Sox,Cleveland Guardians,Sean Burke,Gavin Williams,Cleveland Guardians,0.565308,Cleveland Guardians ML,0.25 UNIT,Cleveland Guardians ML | Win Prob 0.565 | 0.25...


In [67]:
# %%
# =========================================================
# CELL 44 Ã¢â‚¬â€ SAVE FINAL MLB CARD WITH STARTERS
# Save the strict final starter-aware card for posting/tracking
# =========================================================

import pandas as pd
from pathlib import Path

if "final_mlb_card_with_starters" not in globals() or not isinstance(final_mlb_card_with_starters, pd.DataFrame):
    raise ValueError("final_mlb_card_with_starters not found. Run Cell 43 first.")

SAVE_DIR = Path(r"c:\Users\<user>\OneDrive\Documents\KAI Labs\sports\MLB\tracking")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

today_stamp = pd.Timestamp.today().strftime("%Y-%m-%d")
outfile = SAVE_DIR / f"mlb_final_card_with_starters_{today_stamp}.csv"

card = final_mlb_card_with_starters.copy()

if "card_date" not in card.columns:
    card["card_date"] = today_stamp

if "card_type" not in card.columns:
    card["card_type"] = "strict_projection_social_with_starters"

save_cols = [c for c in [
    "card_date",
    "card_type",
    "game_id",
    "game_datetime_utc",
    "away_team",
    "home_team",
    "away_pitcher_final_name",
    "home_pitcher_final_name",
    "starter_matchup",
    "pick_pitcher",
    "fade_pitcher",
    "starter_note",
    "projected_winner",
    "projected_loser",
    "projected_win_prob",
    "projected_win_pct",
    "strength_tier",
    "proj_margin_proxy",
    "official_bet",
    "social_bet",
    "social_bet_with_starters",
    "bet_size",
    "status",
    "venue_name",
] if c in card.columns]

card[save_cols].to_csv(outfile, index=False)

print("Saved final MLB card with starters to:")
print(outfile)

print("\nSaved rows:", len(card))
display(card[save_cols])

mlb_final_card_with_starters_saved = card.copy()

Saved final MLB card with starters to:
c:\Users\<user>\OneDrive\Documents\KAI Labs\sports\MLB\tracking\mlb_final_card_with_starters_2026-10-10.csv

Saved rows: 1


,card_date,card_type,game_id,game_datetime_utc,away_team,home_team,away_pitcher_final_name,home_pitcher_final_name,starter_matchup,pick_pitcher,...,projected_win_prob,projected_win_pct,strength_tier,proj_margin_proxy,official_bet,social_bet,social_bet_with_starters,bet_size,status,venue_name
0,2026-10-10,strict_projection_social_with_starters,849831,2026-10-11 00:00:00+00:00,Chicago White Sox,Cleveland Guardians,Sean Burke,Gavin Williams,Sean Burke vs Gavin Williams,Gavin Williams,...,0.565308,56.5,C,1.3,Cleveland Guardians ML,Cleveland Guardians ML | Win Prob 0.565 | 0.25...,Cleveland Guardians ML | Win Prob 0.565 | 0.25...,0.25 UNIT,Scheduled,Progressive Field


In [68]:
# %%
# =========================================================
# CELL 44 (REPLACEMENT) Ã¢â‚¬â€ SAVE FINAL MLB CARD WITH STARTERS
# Uses the actual slate date from game_datetime_utc instead
# of the system date
# =========================================================

import pandas as pd
from pathlib import Path

if "final_mlb_card_with_starters" not in globals() or not isinstance(final_mlb_card_with_starters, pd.DataFrame):
    raise ValueError("final_mlb_card_with_starters not found. Run Cell 43 first.")

SAVE_DIR = Path(r"c:\Users\<user>\OneDrive\Documents\KAI Labs\sports\MLB\tracking")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

card = final_mlb_card_with_starters.copy()

# ---------------------------------------------------------
# derive card_date from actual slate datetime if available
# ---------------------------------------------------------
if "game_datetime_utc" in card.columns and card["game_datetime_utc"].notna().any():
    slate_date = pd.to_datetime(card["game_datetime_utc"], utc=True, errors="coerce").dt.date.min()
    card_date_str = str(slate_date)
else:
    card_date_str = pd.Timestamp.today().strftime("%Y-%m-%d")

card["card_date"] = card_date_str
card["card_type"] = "strict_projection_social_with_starters"

outfile = SAVE_DIR / f"mlb_final_card_with_starters_{card_date_str}.csv"

save_cols = [c for c in [
    "card_date",
    "card_type",
    "game_id",
    "game_datetime_utc",
    "away_team",
    "home_team",
    "away_pitcher_final_name",
    "home_pitcher_final_name",
    "starter_matchup",
    "pick_pitcher",
    "fade_pitcher",
    "starter_note",
    "projected_winner",
    "projected_loser",
    "projected_win_prob",
    "projected_win_pct",
    "strength_tier",
    "proj_margin_proxy",
    "official_bet",
    "social_bet",
    "social_bet_with_starters",
    "bet_size",
    "status",
    "venue_name",
] if c in card.columns]

card[save_cols].to_csv(outfile, index=False)

print("Saved final MLB card with starters to:")
print(outfile)

print("\nSaved rows:", len(card))
display(card[save_cols])

mlb_final_card_with_starters_saved = card.copy()

Saved final MLB card with starters to:
c:\Users\<user>\OneDrive\Documents\KAI Labs\sports\MLB\tracking\mlb_final_card_with_starters_2026-10-11.csv

Saved rows: 1


,card_date,card_type,game_id,game_datetime_utc,away_team,home_team,away_pitcher_final_name,home_pitcher_final_name,starter_matchup,pick_pitcher,...,projected_win_prob,projected_win_pct,strength_tier,proj_margin_proxy,official_bet,social_bet,social_bet_with_starters,bet_size,status,venue_name
0,2026-10-11,strict_projection_social_with_starters,849831,2026-10-11 00:00:00+00:00,Chicago White Sox,Cleveland Guardians,Sean Burke,Gavin Williams,Sean Burke vs Gavin Williams,Gavin Williams,...,0.565308,56.5,C,1.3,Cleveland Guardians ML,Cleveland Guardians ML | Win Prob 0.565 | 0.25...,Cleveland Guardians ML | Win Prob 0.565 | 0.25...,0.25 UNIT,Scheduled,Progressive Field


In [69]:
# %%
# =========================================================
# CELL 45 Ã¢â‚¬â€ BUILD POST-READY EKO BETS CAPTION (with Why bullets)
# =========================================================

import pandas as pd
import numpy as np

if "mlb_final_card_with_starters_saved" not in globals() or not isinstance(mlb_final_card_with_starters_saved, pd.DataFrame):
    raise ValueError("mlb_final_card_with_starters_saved not found. Run Cell 44 replacement first.")

post_card = mlb_final_card_with_starters_saved.copy()

if len(post_card) == 0:
    raise ValueError("mlb_final_card_with_starters_saved is empty.")

# ---------------------------------------------------------
# Sort strongest plays first
# ---------------------------------------------------------
sort_cols = [c for c in ["projected_win_prob", "game_datetime_utc"] if c in post_card.columns]
post_card = post_card.sort_values(sort_cols, ascending=[False, True][:len(sort_cols)]).reset_index(drop=True)

card_date = str(post_card["card_date"].iloc[0]) if "card_date" in post_card.columns else pd.Timestamp.today().strftime("%Y-%m-%d")

# ---------------------------------------------------------
# Supplement card with feature data from projection board
# (runs/game, starter runs allowed, margin, tier)
# ---------------------------------------------------------
_feat_src = None
for _fname in ["mlb_projection_board_with_starters", "mlb_projection_board", "mlb_today_win_preds_new"]:
    if _fname in globals() and isinstance(globals()[_fname], pd.DataFrame) and len(globals()[_fname]) > 0:
        _feat_src = globals()[_fname].copy()
        print(f"Feature source: {_fname}")
        break

def _cln(s):
    return str(s).strip().lower() if pd.notna(s) else ""

if _feat_src is not None:
    _feat_src["_ak"] = _feat_src["away_team"].apply(_cln)
    _feat_src["_hk"] = _feat_src["home_team"].apply(_cln)
    post_card["_ak"] = post_card["away_team"].apply(_cln)
    post_card["_hk"] = post_card["home_team"].apply(_cln)

    _feat_cols = [c for c in [
        "away_runs_for_avg_10", "home_runs_for_avg_10",
        "away_margin_avg_10",   "home_margin_avg_10",
        "away_starter_runs_allowed_avg_10", "home_starter_runs_allowed_avg_10",
        "away_starter_name", "home_starter_name",
        "strength_tier", "venue_name",
    ] if c in _feat_src.columns]

    _merge_cols = ["_ak", "_hk"] + _feat_cols
    _feat_dedup = (
        _feat_src[_merge_cols]
        .drop_duplicates(subset=["_ak","_hk"], keep="first")
    )
    post_card = post_card.merge(_feat_dedup, on=["_ak","_hk"], how="left", suffixes=("","_feat"))
    post_card.drop(columns=["_ak","_hk"], inplace=True)

# ---------------------------------------------------------
# Why-bullet builder
# ---------------------------------------------------------
def _safe_f(v):
    try:
        f = float(v)
        return None if (f != f) else f   # nan check
    except Exception:
        return None

def build_why_bullets(r):
    bullets = []

    winner = str(r.get("projected_winner", ""))
    away   = str(r.get("away_team", ""))
    home   = str(r.get("home_team", ""))

    # which side are we on?
    pick_is_home = (winner.lower().strip() == home.lower().strip())
    pick_team = home if pick_is_home else away
    fade_team = away if pick_is_home else home
    pick_short = pick_team.split()[-1] if pick_team else "Pick"
    fade_short = fade_team.split()[-1] if fade_team else "Fade"

    # Pitcher matchup
    pick_starter_col = "home_starter_runs_allowed_avg_10" if pick_is_home else "away_starter_runs_allowed_avg_10"
    fade_starter_col = "away_starter_runs_allowed_avg_10" if pick_is_home else "home_starter_runs_allowed_avg_10"
    pick_sp_name_col = "home_starter_name" if pick_is_home else "away_starter_name"
    fade_sp_name_col = "away_starter_name" if pick_is_home else "home_starter_name"
    pick_sp_name = str(r.get("pick_pitcher") or r.get(pick_sp_name_col) or "").strip()
    fade_sp_name = str(r.get("fade_pitcher") or r.get(fade_sp_name_col) or "").strip()

    # Try pitcher_final_name fields
    if not pick_sp_name or pick_sp_name == "nan":
        pick_sp_name = str(r.get("home_pitcher_final_name" if pick_is_home else "away_pitcher_final_name") or "").strip()
    if not fade_sp_name or fade_sp_name == "nan":
        fade_sp_name = str(r.get("away_pitcher_final_name" if pick_is_home else "home_pitcher_final_name") or "").strip()

    pick_ra = _safe_f(r.get(pick_starter_col))
    fade_ra = _safe_f(r.get(fade_starter_col))
    if pick_ra is not None and fade_ra is not None:
        _pick_lbl = f"{pick_sp_name} ({pick_ra:.1f} R/G L10)" if pick_sp_name and pick_sp_name != "nan" else f"{pick_ra:.1f} R/G L10"
        _fade_lbl = f"{fade_sp_name} ({fade_ra:.1f} R/G L10)" if fade_sp_name and fade_sp_name != "nan" else f"{fade_ra:.1f} R/G L10"
        _edge = "better" if pick_ra < fade_ra else "worse"
        bullets.append(f"Pitching: {_pick_lbl} vs {_fade_lbl}")
    elif pick_sp_name and pick_sp_name != "nan" and fade_sp_name and fade_sp_name != "nan":
        bullets.append(f"Pitching: {pick_sp_name} vs {fade_sp_name}")

    # Offense
    pick_off_col = "home_runs_for_avg_10" if pick_is_home else "away_runs_for_avg_10"
    fade_off_col = "away_runs_for_avg_10" if pick_is_home else "home_runs_for_avg_10"
    pick_off = _safe_f(r.get(pick_off_col))
    fade_off = _safe_f(r.get(fade_off_col))
    if pick_off is not None and fade_off is not None:
        _off_edge = "stronger" if pick_off > fade_off else "weaker"
        bullets.append(f"Offense: {pick_short} avg {pick_off:.1f} R/G vs {fade_short} avg {fade_off:.1f} R/G (L10)")

    # Recent form / run margin
    pick_mg_col = "home_margin_avg_10" if pick_is_home else "away_margin_avg_10"
    pick_mg = _safe_f(r.get(pick_mg_col))
    fade_mg_col = "away_margin_avg_10" if pick_is_home else "home_margin_avg_10"
    fade_mg = _safe_f(r.get(fade_mg_col))
    if pick_mg is not None:
        _form_dir = "positive" if pick_mg >= 0 else "negative"
        bullets.append(f"Form: {pick_short} at {pick_mg:+.1f} run margin avg (L10)")

    # Win probability + tier
    prob = _safe_f(r.get("projected_win_prob"))
    tier = str(r.get("strength_tier", "")).strip()
    if prob is not None:
        tier_str = f" | Tier {tier}" if tier and tier not in ("nan","") else ""
        bullets.append(f"Model: {prob:.1%} win probability{tier_str}")

    return bullets[:4]

# ---------------------------------------------------------
# Build caption
# ---------------------------------------------------------
caption_lines = []
caption_lines.append("EKO BETS Ã¢â‚¬â€ MLB CARD")
caption_lines.append(str(card_date))
caption_lines.append("")

for i, (_, r) in enumerate(post_card.iterrows(), 1):
    winner      = r.get("projected_winner", "")
    away_team   = r.get("away_team", "")
    home_team   = r.get("home_team", "")
    prob        = _safe_f(r.get("projected_win_prob"))
    size        = r.get("bet_size", "")
    official_bet= r.get("official_bet", f"{winner} ML")

    caption_lines.append(f"{i}. {official_bet}")
    caption_lines.append(f"   {away_team} @ {home_team}")
    if prob is not None:
        caption_lines.append(f"   Win Prob: {prob:.1%} | Size: {size}")
    else:
        caption_lines.append(f"   Size: {size}")

    bullets = build_why_bullets(r)
    if bullets:
        caption_lines.append("   Why this pick:")
        for b in bullets:
            caption_lines.append(f"   Ã¢â€ â€™ {b}")

    caption_lines.append("")

caption_lines.append("Selective only. No forced volume.")
caption_lines.append("#EkoBets #MLB #SportsBetting")

post_caption = "\n".join(caption_lines)

print("==================================================")
print("EKO BETS Ã¢â‚¬â€ POST READY CAPTION")
print("==================================================")
print(post_caption)

mlb_post_ready_caption = post_caption


Feature source: mlb_projection_board_with_starters


EKO BETS Ã¢â‚¬â€ POST READY CAPTION
EKO BETS Ã¢â‚¬â€ MLB CARD
2026-10-11

1. Cleveland Guardians ML
   Chicago White Sox @ Cleveland Guardians
   Win Prob: 56.5% | Size: 0.25 UNIT
   Why this pick:
   Ã¢â€ â€™ Pitching: Gavin Williams vs Sean Burke
   Ã¢â€ â€™ Offense: Guardians avg 5.5 R/G vs Sox avg 5.0 R/G (L10)
   Ã¢â€ â€™ Form: Guardians at +1.9 run margin avg (L10)
   Ã¢â€ â€™ Model: 56.5% win probability | Tier C

Selective only. No forced volume.
#EkoBets #MLB #SportsBetting


In [70]:
# %%
# =========================================================
# CELL Ã¢â‚¬â€ FAST HISTORICAL STARTER TABLE FROM MLB SCHEDULE API
# Goal:
# - build historical away/home starter IDs + names
# - much faster than one live-feed call per game_id
# - safe, resumable-friendly, debug-heavy
# =========================================================

import time
import requests
import numpy as np
import pandas as pd
from pathlib import Path

print("=" * 80)
print("FAST HISTORICAL STARTER TABLE FROM MLB SCHEDULE API")
print("=" * 80)

# ---------------------------------------------------------
# 0) find historical MLB source dataframe
# ---------------------------------------------------------
candidate_names = [
    "mlb",
    "mlb_hist",
    "mlb_games",
    "mlb_game_table",
    "mlb_game_table_hist",
    "model_df",
]

hist = None
source_used = None

for name in candidate_names:
    if name in globals() and isinstance(globals()[name], pd.DataFrame) and len(globals()[name]) > 0:
        df_try = globals()[name].copy()
        lower_cols = {c.lower(): c for c in df_try.columns}
        if any(c in lower_cols for c in ["game_id", "gamepk", "game_pk", "pk"]):
            hist = df_try
            source_used = name
            break

if hist is None:
    raise ValueError(
        f"No usable historical MLB dataframe found. Checked: {candidate_names}"
    )

print(f"Using source: {source_used}")
print(f"Source shape: {hist.shape}")

# ---------------------------------------------------------
# 1) helper to find columns safely
# ---------------------------------------------------------
def find_first_col(df, options):
    lower_map = {c.lower(): c for c in df.columns}
    for opt in options:
        if opt.lower() in lower_map:
            return lower_map[opt.lower()]
    return None

game_id_col = find_first_col(hist, ["game_id", "gamepk", "game_pk", "pk"])
date_col    = find_first_col(hist, ["date", "game_date", "game_datetime", "game_datetime_utc"])
away_col    = find_first_col(hist, ["away_team", "away", "visitor_team", "away_name"])
home_col    = find_first_col(hist, ["home_team", "home", "home_name"])

if game_id_col is None:
    raise ValueError("Could not find game_id column in historical MLB dataframe.")
if date_col is None:
    raise ValueError("Could not find date column in historical MLB dataframe.")

print("")
print("Detected columns:")
print(f" - game_id_col: {game_id_col}")
print(f" - date_col   : {date_col}")
print(f" - away_col   : {away_col}")
print(f" - home_col   : {home_col}")

# ---------------------------------------------------------
# 2) clean historical game list
# ---------------------------------------------------------
work = hist.copy()

work[game_id_col] = pd.to_numeric(work[game_id_col], errors="coerce")
work[date_col] = pd.to_datetime(work[date_col], errors="coerce")

work = work[work[game_id_col].notna()].copy()
work = work[work[date_col].notna()].copy()

work[game_id_col] = work[game_id_col].astype("int64")

rename_map = {
    game_id_col: "game_id",
    date_col: "date",
}
if away_col is not None:
    rename_map[away_col] = "away_team"
if home_col is not None:
    rename_map[home_col] = "home_team"

hist_games = work.rename(columns=rename_map).copy()

keep_cols = [c for c in ["game_id", "date", "away_team", "home_team"] if c in hist_games.columns]
hist_games = (
    hist_games[keep_cols]
    .drop_duplicates(subset=["game_id"])
    .sort_values(["date", "game_id"])
    .reset_index(drop=True)
)

if hist_games.empty:
    raise ValueError("Historical game table is empty after cleaning.")

print("")
print(f"Unique historical games: {len(hist_games):,}")
print(f"Date range: {hist_games['date'].min()} -> {hist_games['date'].max()}")

print("")
print("Sample historical games:")
display(hist_games.head(5))

# ---------------------------------------------------------
# 3) build date chunks
# ---------------------------------------------------------
MIN_DATE = hist_games["date"].min().normalize()
# 2026-07-28 fix: same self-referential-ceiling bug as the other starter-hist
# builder cell (index 0) -- MAX_DATE was derived from hist_games["date"].max(),
# the EXISTING table's own current ceiling, so this cell could never fetch
# past whatever data already existed no matter how many times it re-ran.
# This cell saves to mlb_starter_hist_schedule.parquet, a separate file from
# the other builder's mlb_starter_hist_fast.parquet, so it needed the same
# fix applied independently. Fixed to use today's actual date.
MAX_DATE = pd.Timestamp.today().normalize()

CHUNK_DAYS = 7  # weekly pulls
date_chunks = []

cur = MIN_DATE
while cur <= MAX_DATE:
    end = min(cur + pd.Timedelta(days=CHUNK_DAYS - 1), MAX_DATE)
    date_chunks.append((cur, end))
    cur = end + pd.Timedelta(days=1)

print(f"Date chunks to query: {len(date_chunks):,}")

# ---------------------------------------------------------
# 4) api helpers
# ---------------------------------------------------------
SESSION = requests.Session()
SESSION.headers.update({"User-Agent": "Mozilla/5.0"})

def safe_get(url, params=None, timeout=30, max_retries=3):
    last_err = None
    for attempt in range(1, max_retries + 1):
        try:
            r = SESSION.get(url, params=params, timeout=timeout)
            r.raise_for_status()
            return r.json()
        except Exception as e:
            last_err = e
            if attempt < max_retries:
                time.sleep(0.75 * attempt)
    return {"_fetch_error": str(last_err)}

def parse_schedule_chunk(start_date, end_date):
    """
    Pull MLB schedule for a date range and extract away/home probable pitchers.
    """
    url = "https://statsapi.mlb.com/api/v1/schedule"
    params = {
        "sportId": 1,
        "startDate": start_date.strftime("%Y-%m-%d"),
        "endDate": end_date.strftime("%Y-%m-%d"),
        "hydrate": "probablePitcher,team",
    }

    payload = safe_get(url, params=params)

    rows = []

    if "_fetch_error" in payload:
        rows.append({
            "game_id": np.nan,
            "date": pd.NaT,
            "api_away_team": np.nan,
            "api_home_team": np.nan,
            "away_starter_id": np.nan,
            "away_starter_name": np.nan,
            "home_starter_id": np.nan,
            "home_starter_name": np.nan,
            "starter_source": "schedule_error",
            "chunk_start": start_date,
            "chunk_end": end_date,
            "fetch_error": payload["_fetch_error"],
        })
        return pd.DataFrame(rows)

    for d in payload.get("dates", []) or []:
        game_date = pd.to_datetime(d.get("date"), errors="coerce")

        for g in d.get("games", []) or []:
            teams = g.get("teams", {}) or {}

            away = teams.get("away", {}) or {}
            home = teams.get("home", {}) or {}

            away_team = (away.get("team") or {}).get("name")
            home_team = (home.get("team") or {}).get("name")

            away_prob = away.get("probablePitcher") or {}
            home_prob = home.get("probablePitcher") or {}

            rows.append({
                "game_id": pd.to_numeric(g.get("gamePk"), errors="coerce"),
                "date": game_date,
                "api_away_team": away_team,
                "api_home_team": home_team,
                "away_starter_id": pd.to_numeric(away_prob.get("id"), errors="coerce"),
                "away_starter_name": away_prob.get("fullName", np.nan),
                "home_starter_id": pd.to_numeric(home_prob.get("id"), errors="coerce"),
                "home_starter_name": home_prob.get("fullName", np.nan),
                "starter_source": "schedule_probablePitcher",
                "chunk_start": start_date,
                "chunk_end": end_date,
                "fetch_error": np.nan,
            })

    return pd.DataFrame(rows)

# ---------------------------------------------------------
# 5) pull all chunks
# ---------------------------------------------------------
all_parts = []

print("")
print("Querying MLB schedule endpoint by weekly chunks...")

for i, (chunk_start, chunk_end) in enumerate(date_chunks, start=1):
    part = parse_schedule_chunk(chunk_start, chunk_end)
    all_parts.append(part)

    pulled_rows = len(part)
    print(
        f"[{i:,}/{len(date_chunks):,}] "
        f"{chunk_start.date()} -> {chunk_end.date()} | rows={pulled_rows:,}"
    )

    time.sleep(0.10)

starter_sched = pd.concat(all_parts, ignore_index=True) if all_parts else pd.DataFrame()

print("")
print(f"Raw schedule starter shape: {starter_sched.shape}")

# ---------------------------------------------------------
# 6) clean schedule pull
# ---------------------------------------------------------
if starter_sched.empty:
    raise ValueError("starter_sched is empty. Schedule pull returned no rows.")

starter_sched["game_id"] = pd.to_numeric(starter_sched["game_id"], errors="coerce")
starter_sched["date"] = pd.to_datetime(starter_sched["date"], errors="coerce")

starter_sched = starter_sched[starter_sched["game_id"].notna()].copy()
starter_sched["game_id"] = starter_sched["game_id"].astype("int64")

starter_sched = (
    starter_sched
    .sort_values(["date", "game_id"])
    .drop_duplicates(subset=["game_id"], keep="last")
    .reset_index(drop=True)
)

print(f"Clean schedule starter shape: {starter_sched.shape}")

# ---------------------------------------------------------
# 7) merge back to historical game table
# ---------------------------------------------------------
merge_cols = [
    "game_id",
    "api_away_team",
    "api_home_team",
    "away_starter_id",
    "away_starter_name",
    "home_starter_id",
    "home_starter_name",
    "starter_source",
    "fetch_error",
]

starter_hist = hist_games.merge(
    starter_sched[merge_cols],
    on="game_id",
    how="left",
    validate="one_to_one"
).copy()

# ---------------------------------------------------------
# 8) coverage + debug prints
# ---------------------------------------------------------
for c in ["away_starter_id", "home_starter_id"]:
    if c in starter_hist.columns:
        starter_hist[c] = pd.to_numeric(starter_hist[c], errors="coerce")

starter_hist["has_away_starter"] = starter_hist["away_starter_id"].notna()
starter_hist["has_home_starter"] = starter_hist["home_starter_id"].notna()
starter_hist["has_both_starters"] = starter_hist["has_away_starter"] & starter_hist["has_home_starter"]
starter_hist["missing_any_starter"] = ~starter_hist["has_both_starters"]

print("")
print("=" * 80)
print("STARTER HISTORICAL TABLE SUMMARY")
print("=" * 80)
print(f"starter_hist shape: {starter_hist.shape}")

summary = pd.Series({
    "total_games": int(len(starter_hist)),
    "games_with_away_starter": int(starter_hist["has_away_starter"].sum()),
    "games_with_home_starter": int(starter_hist["has_home_starter"].sum()),
    "games_with_both_starters": int(starter_hist["has_both_starters"].sum()),
    "games_missing_any_starter": int(starter_hist["missing_any_starter"].sum()),
    "away_starter_missing_pct": round(100 * starter_hist["away_starter_id"].isna().mean(), 2),
    "home_starter_missing_pct": round(100 * starter_hist["home_starter_id"].isna().mean(), 2),
})

print(summary)

if "starter_source" in starter_hist.columns:
    print("")
    print("Starter source counts:")
    print(starter_hist["starter_source"].fillna("missing").value_counts(dropna=False))

print("")
print("Sample starter rows:")
show_cols = [c for c in [
    "game_id", "date", "away_team", "home_team",
    "api_away_team", "api_home_team",
    "away_starter_id", "away_starter_name",
    "home_starter_id", "home_starter_name",
    "starter_source"
] if c in starter_hist.columns]
display(starter_hist[show_cols].head(10))

print("")
print("Sample missing-starter rows:")
display(starter_hist.loc[starter_hist["missing_any_starter"], show_cols].head(10))

# ---------------------------------------------------------
# 9) save + expose
# ---------------------------------------------------------
save_dir = Path("data/processed/mlb")
save_dir.mkdir(parents=True, exist_ok=True)

csv_path = save_dir / "mlb_starter_hist_schedule.csv"
parquet_path = save_dir / "mlb_starter_hist_schedule.parquet"

try:
    starter_hist.to_csv(csv_path, index=False)
    print(f"Saved CSV    : {csv_path}")
except Exception as e:
    print(f"WARNING: could not save CSV -> {e}")

try:
    starter_hist.to_parquet(parquet_path, index=False)
    print(f"Saved Parquet: {parquet_path}")
except Exception as e:
    print(f"WARNING: could not save Parquet -> {e}")

globals()["mlb_starter_hist"] = starter_hist.copy()

print("")
print("Created in memory as: mlb_starter_hist")
print("Done.")

FAST HISTORICAL STARTER TABLE FROM MLB SCHEDULE API
Using source: mlb
Source shape: (8230, 134)

Detected columns:
 - game_id_col: game_id
 - date_col   : date
 - away_col   : away_team
 - home_col   : home_team

Unique historical games: 8,230
Date range: 2023-03-02 00:00:00 -> 2025-11-01 00:00:00

Sample historical games:


,game_id,date,away_team,home_team
0,719301,2023-03-02,San Francisco Giants,Cleveland Guardians
1,719302,2023-03-02,Arizona Diamondbacks,Los Angeles Dodgers
2,719303,2023-03-02,San Diego Padres,Seattle Mariners
3,719304,2023-03-02,Los Angeles Angels,Kansas City Royals
4,719305,2023-03-02,Chicago White Sox,Colorado Rockies


Date chunks to query: 189

Querying MLB schedule endpoint by weekly chunks...


[1/189] 2023-03-02 -> 2023-03-08 | rows=108


[2/189] 2023-03-09 -> 2023-03-15 | rows=109


...

,game_id,date,away_team,home_team,api_away_team,api_home_team,away_starter_id,away_starter_name,home_starter_id,home_starter_name,starter_source
0,719301,2023-03-02,San Francisco Giants,Cleveland Guardians,San Francisco Giants,Cleveland Guardians,663941.0,Tristan Beck,615698.0,Cal Quantrill,schedule_probablePitcher
1,719302,2023-03-02,Arizona Diamondbacks,Los Angeles Dodgers,Arizona Diamondbacks,Los Angeles Dodgers,669194.0,Ryne Nelson,675627.0,Michael Grove,schedule_probablePitcher
2,719303,2023-03-02,San Diego Padres,Seattle Mariners,San Diego Padres,Seattle Mariners,607625.0,Seth Lugo,669302.0,Logan Gilbert,schedule_probablePitcher
3,719304,2023-03-02,Los Angeles Angels,Kansas City Royals,Los Angeles Angels,Kansas City Royals,656353.0,Tucker Davidson,594577.0,Mike Mayers,schedule_probablePitcher
4,719305,2023-03-02,Chicago White Sox,Colorado Rockies,Chicago White Sox,Colorado Rockies,458681.0,Lance Lynn,663562.0,Noah Davis,schedule_probablePitcher
5,719306,2023-03-02,Texas Rangers,Milwaukee Brewers,Texas Rangers,Milwaukee Brewers,641540.0,Dane Dunning,669203.0,Corbin Burnes,schedule_probablePitcher
6,719307,2023-03-02,Minnesota Twins,Tampa Bay Rays,Minnesota Twins,Tampa Bay Rays,628317.0,Kenta Maeda,621107.0,Zach Eflin,schedule_probablePitcher
7,719308,2023-03-02,Miami Marlins,Washington Nationals,Miami Marlins,Washington Nationals,665795.0,Edward Cabrera,669022.0,MacKenzie Gore,schedule_probablePitcher
8,719309,2023-03-02,Pittsburgh Pirates,Toronto Blue Jays,Pittsburgh Pirates,Toronto Blue Jays,672710.0,Roansy Contreras,666201.0,Alek Manoah,schedule_probablePitcher
9,719310,2023-03-02,New York Yankees,Pittsburgh Pirates,New York Yankees,Pittsburgh Pirates,656061.0,Albert Abreu,656605.0,Mitch Keller,schedule_probablePitcher



Sample missing-starter rows:


,game_id,date,away_team,home_team,api_away_team,api_home_team,away_starter_id,away_starter_name,home_starter_id,home_starter_name,starter_source
356,718944,2023-03-26,Miami Marlins,New York Mets,Miami Marlins,New York Mets,666129.0,Braxton Garrett,NaN,NaN,schedule_probablePitcher
1721,717427,2023-07-09,Cincinnati Reds,Milwaukee Brewers,Cincinnati Reds,Milwaukee Brewers,NaN,NaN,489119.0,Wade Miley,schedule_probablePitcher
2891,748083,2024-03-06,Seattle Mariners,Kansas City Royals,Seattle Mariners,Kansas City Royals,NaN,NaN,672582.0,Angel Zerpa,schedule_probablePitcher
3103,747871,2024-03-21,Cincinnati Reds,Seattle Mariners,Cincinnati Reds,Seattle Mariners,607259.0,Nick Martinez,NaN,NaN,schedule_probablePitcher
3159,747812,2024-03-25,Boston Red Sox,Texas Rangers,Boston Red Sox,Texas Rangers,676477.0,Garrett Whitlock,NaN,NaN,schedule_probablePitcher
3166,766494,2024-03-25,New York Yankees,Diablos Rojos del Mexico,New York Yankees,Diablos Rojos del Mexico,702436.0,Luis Serna,NaN,NaN,schedule_probablePitcher
4555,746367,2024-07-11,Miami Marlins,Houston Astros,Miami Marlins,Houston Astros,682610.0,Roddery Muñoz,NaN,NaN,schedule_probablePitcher
4731,745063,2024-07-28,Cincinnati Reds,Tampa Bay Rays,Cincinnati Reds,Tampa Bay Rays,668881.0,Hunter Greene,NaN,NaN,schedule_probablePitcher
5830,790401,2025-03-16,Chicago Cubs,Tokyo Yomiuri Giants,Chicago Cubs,Tokyo Yomiuri Giants,592791.0,Jameson Taillon,NaN,NaN,schedule_probablePitcher
5831,790402,2025-03-16,Los Angeles Dodgers,Hanshin Tigers,Los Angeles Dodgers,Hanshin Tigers,605483.0,Blake Snell,NaN,NaN,schedule_probablePitcher


Saved CSV    : data\processed\mlb\mlb_starter_hist_schedule.csv


Saved Parquet: data\processed\mlb\mlb_starter_hist_schedule.parquet

Created in memory as: mlb_starter_hist
Done.


In [71]:
# %%
# =========================================================
# CELL Ã¢â‚¬â€ FILL MISSING HISTORICAL STARTERS FROM GAME FEED
# Goal:
# - only query game feeds for rows still missing starter info
# - update mlb_starter_hist without rebuilding everything
# - robust, safe, debug-heavy
# =========================================================

import time
import requests
import numpy as np
import pandas as pd
from pathlib import Path

print("=" * 80)
print("FILL MISSING HISTORICAL STARTERS FROM GAME FEED")
print("=" * 80)

# ---------------------------------------------------------
# 0) validate starter history table
# ---------------------------------------------------------
if "mlb_starter_hist" not in globals() or not isinstance(mlb_starter_hist, pd.DataFrame):
    raise ValueError("mlb_starter_hist not found. Run the schedule starter cell first.")

starter_hist = mlb_starter_hist.copy()

required_cols = ["game_id"]
missing_required = [c for c in required_cols if c not in starter_hist.columns]
if missing_required:
    raise ValueError(f"mlb_starter_hist missing required columns: {missing_required}")

for c in ["away_starter_id", "home_starter_id"]:
    if c not in starter_hist.columns:
        starter_hist[c] = np.nan

starter_hist["game_id"] = pd.to_numeric(starter_hist["game_id"], errors="coerce")
starter_hist = starter_hist[starter_hist["game_id"].notna()].copy()
starter_hist["game_id"] = starter_hist["game_id"].astype("int64")

starter_hist["away_starter_id"] = pd.to_numeric(starter_hist["away_starter_id"], errors="coerce")
starter_hist["home_starter_id"] = pd.to_numeric(starter_hist["home_starter_id"], errors="coerce")

starter_hist["missing_any_starter"] = (
    starter_hist["away_starter_id"].isna() | starter_hist["home_starter_id"].isna()
)

missing_games = starter_hist.loc[starter_hist["missing_any_starter"], "game_id"].dropna().astype("int64").unique().tolist()

print(f"Starter hist shape: {starter_hist.shape}")
print(f"Games missing any starter before fallback: {len(missing_games):,}")

if len(missing_games) == 0:
    print("No missing starter games found. Nothing to fill.")
else:
    print("Will query only missing games from feed/live.")

# ---------------------------------------------------------
# 1) api helpers
# ---------------------------------------------------------
SESSION = requests.Session()
SESSION.headers.update({"User-Agent": "Mozilla/5.0"})

BASE_URL = "https://statsapi.mlb.com/api/v1.1/game/{}/feed/live"

def safe_get_json(url, timeout=25, max_retries=3):
    last_err = None
    for attempt in range(1, max_retries + 1):
        try:
            r = SESSION.get(url, timeout=timeout)
            r.raise_for_status()
            return r.json()
        except Exception as e:
            last_err = e
            if attempt < max_retries:
                time.sleep(0.75 * attempt)
    return {"_fetch_error": str(last_err)}

def get_player_name(players_lookup, pid):
    try:
        return (players_lookup.get(f"ID{int(pid)}") or {}).get("person", {}).get("fullName", np.nan)
    except Exception:
        return np.nan

def extract_from_probables(feed):
    """
    First attempt: gameData.probablePitchers
    """
    away_id = np.nan
    away_name = np.nan
    home_id = np.nan
    home_name = np.nan

    try:
        probable = ((feed.get("gameData") or {}).get("probablePitchers") or {})
        away = probable.get("away") or {}
        home = probable.get("home") or {}

        away_id = pd.to_numeric(away.get("id"), errors="coerce")
        away_name = away.get("fullName", np.nan)

        home_id = pd.to_numeric(home.get("id"), errors="coerce")
        home_name = home.get("fullName", np.nan)
    except Exception:
        pass

    return away_id, away_name, home_id, home_name

def extract_from_boxscore_pitchers(feed):
    """
    Fallback:
    use liveData.boxscore.teams.[away/home].pitchers and players table
    choose the first listed pitcher for each side
    """
    away_id = np.nan
    away_name = np.nan
    home_id = np.nan
    home_name = np.nan

    try:
        box = ((feed.get("liveData") or {}).get("boxscore") or {})
        teams = box.get("teams") or {}
        players_lookup = box.get("players") or {}

        away_pitchers = ((teams.get("away") or {}).get("pitchers")) or []
        home_pitchers = ((teams.get("home") or {}).get("pitchers")) or []

        if len(away_pitchers):
            away_id = pd.to_numeric(away_pitchers[0], errors="coerce")
            away_name = get_player_name(players_lookup, away_id)

        if len(home_pitchers):
            home_id = pd.to_numeric(home_pitchers[0], errors="coerce")
            home_name = get_player_name(players_lookup, home_id)
    except Exception:
        pass

    return away_id, away_name, home_id, home_name

def extract_from_decisions(feed):
    """
    Last fallback:
    use decisions.winner/loser only if one side is still missing
    not perfect, but can rescue some rows
    """
    away_id = np.nan
    away_name = np.nan
    home_id = np.nan
    home_name = np.nan

    try:
        decisions = ((feed.get("liveData") or {}).get("decisions") or {})
        winner = decisions.get("winner") or {}
        loser = decisions.get("loser") or {}

        # This fallback is weak and will only be used if boxscore/probables fail.
        # We return ids/names but do not know side confidently here.
        # We will leave side assignment unresolved unless exactly one side is missing later.
        winner_id = pd.to_numeric(winner.get("id"), errors="coerce")
        loser_id = pd.to_numeric(loser.get("id"), errors="coerce")
        winner_name = winner.get("fullName", np.nan)
        loser_name = loser.get("fullName", np.nan)

        # return two generic fallback values packed into away/home slots
        away_id, away_name = winner_id, winner_name
        home_id, home_name = loser_id, loser_name
    except Exception:
        pass

    return away_id, away_name, home_id, home_name

def parse_missing_game(game_id):
    out = {
        "game_id": int(game_id),
        "feed_api_ok": 0,
        "feed_fetch_error": np.nan,
        "feed_away_starter_id": np.nan,
        "feed_away_starter_name": np.nan,
        "feed_home_starter_id": np.nan,
        "feed_home_starter_name": np.nan,
        "feed_starter_source": "missing",
    }

    feed = safe_get_json(BASE_URL.format(int(game_id)))

    if "_fetch_error" in feed:
        out["feed_fetch_error"] = feed["_fetch_error"]
        return out

    out["feed_api_ok"] = 1

    # 1) probable pitchers
    away_id, away_name, home_id, home_name = extract_from_probables(feed)
    got_probables = pd.notna(away_id) or pd.notna(home_id)

    if got_probables:
        out["feed_away_starter_id"] = away_id
        out["feed_away_starter_name"] = away_name
        out["feed_home_starter_id"] = home_id
        out["feed_home_starter_name"] = home_name
        out["feed_starter_source"] = "feed_probablePitchers"
        return out

    # 2) boxscore pitchers
    away_id, away_name, home_id, home_name = extract_from_boxscore_pitchers(feed)
    got_box = pd.notna(away_id) or pd.notna(home_id)

    if got_box:
        out["feed_away_starter_id"] = away_id
        out["feed_away_starter_name"] = away_name
        out["feed_home_starter_id"] = home_id
        out["feed_home_starter_name"] = home_name
        out["feed_starter_source"] = "feed_boxscore_first_pitcher"
        return out

    # 3) weak last fallback
    away_id, away_name, home_id, home_name = extract_from_decisions(feed)
    got_decision = pd.notna(away_id) or pd.notna(home_id)

    if got_decision:
        out["feed_away_starter_id"] = away_id
        out["feed_away_starter_name"] = away_name
        out["feed_home_starter_id"] = home_id
        out["feed_home_starter_name"] = home_name
        out["feed_starter_source"] = "feed_decisions_fallback"

    return out

# ---------------------------------------------------------
# 2) query only missing games
# ---------------------------------------------------------
fallback_rows = []

if len(missing_games) > 0:
    print("")
    print("Querying missing games only...")

    for i, gid in enumerate(missing_games, start=1):
        row = parse_missing_game(gid)
        fallback_rows.append(row)

        if i <= 5 or i % 50 == 0 or i == len(missing_games):
            found_any = sum(
                int(
                    pd.notna(r.get("feed_away_starter_id")) or
                    pd.notna(r.get("feed_home_starter_id"))
                )
                for r in fallback_rows
            )
            print(f"[{i:,}/{len(missing_games):,}] fallback_rows={len(fallback_rows):,} | found_any={found_any:,}")

        time.sleep(0.03)

fallback_df = pd.DataFrame(fallback_rows)
print("")
print(f"Fallback pull shape: {fallback_df.shape}")

# ---------------------------------------------------------
# 3) merge fallback results
# ---------------------------------------------------------
if len(fallback_df) > 0:
    fallback_df["game_id"] = pd.to_numeric(fallback_df["game_id"], errors="coerce")
    fallback_df = fallback_df[fallback_df["game_id"].notna()].copy()
    fallback_df["game_id"] = fallback_df["game_id"].astype("int64")

    starter_hist = starter_hist.merge(
        fallback_df,
        on="game_id",
        how="left",
        validate="one_to_one"
    )
else:
    for c in [
        "feed_api_ok", "feed_fetch_error",
        "feed_away_starter_id", "feed_away_starter_name",
        "feed_home_starter_id", "feed_home_starter_name",
        "feed_starter_source"
    ]:
        starter_hist[c] = np.nan

# ---------------------------------------------------------
# 4) safely fill missing starter fields only
# ---------------------------------------------------------
for c in ["feed_away_starter_id", "feed_home_starter_id"]:
    if c in starter_hist.columns:
        starter_hist[c] = pd.to_numeric(starter_hist[c], errors="coerce")

before_both = int(
    starter_hist["away_starter_id"].notna().mul(starter_hist["home_starter_id"].notna()).sum()
)

away_missing_mask = starter_hist["away_starter_id"].isna() & starter_hist["feed_away_starter_id"].notna()
home_missing_mask = starter_hist["home_starter_id"].isna() & starter_hist["feed_home_starter_id"].notna()

starter_hist.loc[away_missing_mask, "away_starter_id"] = starter_hist.loc[away_missing_mask, "feed_away_starter_id"]
starter_hist.loc[home_missing_mask, "home_starter_id"] = starter_hist.loc[home_missing_mask, "feed_home_starter_id"]

if "away_starter_name" not in starter_hist.columns:
    starter_hist["away_starter_name"] = np.nan
if "home_starter_name" not in starter_hist.columns:
    starter_hist["home_starter_name"] = np.nan

starter_hist.loc[
    starter_hist["away_starter_name"].isna() & starter_hist["feed_away_starter_name"].notna(),
    "away_starter_name"
] = starter_hist.loc[
    starter_hist["away_starter_name"].isna() & starter_hist["feed_away_starter_name"].notna(),
    "feed_away_starter_name"
]

starter_hist.loc[
    starter_hist["home_starter_name"].isna() & starter_hist["feed_home_starter_name"].notna(),
    "home_starter_name"
] = starter_hist.loc[
    starter_hist["home_starter_name"].isna() & starter_hist["feed_home_starter_name"].notna(),
    "feed_home_starter_name"
]

# update source label only where feed actually filled something
filled_from_feed = away_missing_mask | home_missing_mask
if "starter_source" not in starter_hist.columns:
    starter_hist["starter_source"] = np.nan

starter_hist.loc[
    filled_from_feed & starter_hist["feed_starter_source"].notna(),
    "starter_source"
] = starter_hist.loc[
    filled_from_feed & starter_hist["feed_starter_source"].notna(),
    "feed_starter_source"
]

# recompute coverage flags
starter_hist["has_away_starter"] = starter_hist["away_starter_id"].notna()
starter_hist["has_home_starter"] = starter_hist["home_starter_id"].notna()
starter_hist["has_both_starters"] = starter_hist["has_away_starter"] & starter_hist["has_home_starter"]
starter_hist["missing_any_starter"] = ~starter_hist["has_both_starters"]

after_both = int(starter_hist["has_both_starters"].sum())

# ---------------------------------------------------------
# 5) debug summary
# ---------------------------------------------------------
print("")
print("=" * 80)
print("POST-FALLBACK STARTER COVERAGE")
print("=" * 80)
print(f"Games with both starters BEFORE fallback: {before_both:,}")
print(f"Games with both starters AFTER  fallback: {after_both:,}")
print(f"Net improvement: {after_both - before_both:,}")

summary = pd.Series({
    "total_games": int(len(starter_hist)),
    "games_with_away_starter": int(starter_hist["has_away_starter"].sum()),
    "games_with_home_starter": int(starter_hist["has_home_starter"].sum()),
    "games_with_both_starters": int(starter_hist["has_both_starters"].sum()),
    "games_missing_any_starter": int(starter_hist["missing_any_starter"].sum()),
    "away_starter_missing_pct": round(100 * starter_hist["away_starter_id"].isna().mean(), 2),
    "home_starter_missing_pct": round(100 * starter_hist["home_starter_id"].isna().mean(), 2),
})
print(summary)

print("")
print("Starter source counts:")
print(starter_hist["starter_source"].fillna("missing").value_counts(dropna=False).head(20))

show_cols = [c for c in [
    "game_id", "date", "away_team", "home_team",
    "away_starter_id", "away_starter_name",
    "home_starter_id", "home_starter_name",
    "starter_source",
    "feed_starter_source",
    "feed_fetch_error"
] if c in starter_hist.columns]

print("")
print("Sample filled rows:")
display(starter_hist.loc[filled_from_feed, show_cols].head(10))

print("")
print("Sample still-missing rows:")
display(starter_hist.loc[starter_hist["missing_any_starter"], show_cols].head(10))

# ---------------------------------------------------------
# 6) save + expose
# ---------------------------------------------------------
save_dir = Path("data/processed/mlb")
save_dir.mkdir(parents=True, exist_ok=True)

csv_path = save_dir / "mlb_starter_hist_filled.csv"
parquet_path = save_dir / "mlb_starter_hist_filled.parquet"

try:
    starter_hist.to_csv(csv_path, index=False)
    print(f"Saved CSV    : {csv_path}")
except Exception as e:
    print(f"WARNING: could not save CSV -> {e}")

try:
    starter_hist.to_parquet(parquet_path, index=False)
    print(f"Saved Parquet: {parquet_path}")
except Exception as e:
    print(f"WARNING: could not save Parquet -> {e}")

globals()["mlb_starter_hist"] = starter_hist.copy()

print("")
print("Updated in memory as: mlb_starter_hist")
print("Done.")

FILL MISSING HISTORICAL STARTERS FROM GAME FEED
Starter hist shape: (8230, 16)
Games missing any starter before fallback: 14
Will query only missing games from feed/live.

Querying missing games only...


[1/14] fallback_rows=1 | found_any=1


[2/14] fallback_rows=2 | found_any=2


[3/14] fallback_rows=3 | found_any=3


[4/14] fallback_rows=4 | found_any=4


...

,game_id,date,away_team,home_team,away_starter_id,away_starter_name,home_starter_id,home_starter_name,starter_source,feed_starter_source,feed_fetch_error



Sample still-missing rows:


,game_id,date,away_team,home_team,away_starter_id,away_starter_name,home_starter_id,home_starter_name,starter_source,feed_starter_source,feed_fetch_error
356,718944,2023-03-26,Miami Marlins,New York Mets,666129.0,Braxton Garrett,NaN,NaN,schedule_probablePitcher,feed_probablePitchers,NaN
1721,717427,2023-07-09,Cincinnati Reds,Milwaukee Brewers,NaN,NaN,489119.0,Wade Miley,schedule_probablePitcher,feed_probablePitchers,NaN
2891,748083,2024-03-06,Seattle Mariners,Kansas City Royals,NaN,NaN,672582.0,Angel Zerpa,schedule_probablePitcher,feed_probablePitchers,NaN
3103,747871,2024-03-21,Cincinnati Reds,Seattle Mariners,607259.0,Nick Martinez,NaN,NaN,schedule_probablePitcher,feed_probablePitchers,NaN
3159,747812,2024-03-25,Boston Red Sox,Texas Rangers,676477.0,Garrett Whitlock,NaN,NaN,schedule_probablePitcher,feed_probablePitchers,NaN
3166,766494,2024-03-25,New York Yankees,Diablos Rojos del Mexico,702436.0,Luis Serna,NaN,NaN,schedule_probablePitcher,feed_probablePitchers,NaN
4555,746367,2024-07-11,Miami Marlins,Houston Astros,682610.0,Roddery Muñoz,NaN,NaN,schedule_probablePitcher,feed_probablePitchers,NaN
4731,745063,2024-07-28,Cincinnati Reds,Tampa Bay Rays,668881.0,Hunter Greene,NaN,NaN,schedule_probablePitcher,feed_probablePitchers,NaN
5830,790401,2025-03-16,Chicago Cubs,Tokyo Yomiuri Giants,592791.0,Jameson Taillon,NaN,NaN,schedule_probablePitcher,feed_probablePitchers,NaN
5831,790402,2025-03-16,Los Angeles Dodgers,Hanshin Tigers,605483.0,Blake Snell,NaN,NaN,schedule_probablePitcher,feed_probablePitchers,NaN


Saved CSV    : data\processed\mlb\mlb_starter_hist_filled.csv


Saved Parquet: data\processed\mlb\mlb_starter_hist_filled.parquet

Updated in memory as: mlb_starter_hist
Done.


In [72]:
# %%
# =========================================================
# CELL Ã¢â‚¬â€ BUILD LONG-FORMAT HISTORICAL PITCHER GAME LOG
# Goal:
# - merge starter table onto main mlb historical game table
# - create one row per starter appearance
# - prepare clean base table for leakage-safe rolling pitcher features
# =========================================================

import numpy as np
import pandas as pd
from pathlib import Path

print("=" * 80)
print("BUILD LONG-FORMAT HISTORICAL PITCHER GAME LOG")
print("=" * 80)

# ---------------------------------------------------------
# 0) validate inputs
# ---------------------------------------------------------
if "mlb" not in globals() or not isinstance(mlb, pd.DataFrame):
    raise ValueError("mlb dataframe not found in memory.")

if "mlb_starter_hist" not in globals() or not isinstance(mlb_starter_hist, pd.DataFrame):
    raise ValueError("mlb_starter_hist not found. Run the starter cells first.")

games = mlb.copy()
starters = mlb_starter_hist.copy()

print(f"mlb shape before merge: {games.shape}")
print(f"mlb_starter_hist shape: {starters.shape}")

# ---------------------------------------------------------
# 1) detect required columns safely
# ---------------------------------------------------------
def find_first_col(df, options):
    lower_map = {c.lower(): c for c in df.columns}
    for opt in options:
        if opt.lower() in lower_map:
            return lower_map[opt.lower()]
    return None

game_id_col   = find_first_col(games, ["game_id", "gamepk", "game_pk", "pk"])
date_col      = find_first_col(games, ["date", "game_date", "game_datetime", "game_datetime_utc"])
away_team_col = find_first_col(games, ["away_team", "away", "visitor_team", "away_name"])
home_team_col = find_first_col(games, ["home_team", "home", "home_name"])
away_runs_col = find_first_col(games, ["away_runs", "away_score", "visitor_score"])
home_runs_col = find_first_col(games, ["home_runs", "home_score"])
home_win_col  = find_first_col(games, ["home_win"])
margin_col    = find_first_col(games, ["home_margin", "margin"])
total_col     = find_first_col(games, ["total_runs", "total"])

needed = {
    "game_id": game_id_col,
    "date": date_col,
    "away_team": away_team_col,
    "home_team": home_team_col,
    "away_runs": away_runs_col,
    "home_runs": home_runs_col,
}

missing = [k for k, v in needed.items() if v is None]
if missing:
    raise ValueError(f"mlb missing required columns for pitcher log: {missing}")

print("")
print("Detected game columns:")
for k, v in {
    "game_id_col": game_id_col,
    "date_col": date_col,
    "away_team_col": away_team_col,
    "home_team_col": home_team_col,
    "away_runs_col": away_runs_col,
    "home_runs_col": home_runs_col,
    "home_win_col": home_win_col,
    "margin_col": margin_col,
    "total_col": total_col,
}.items():
    print(f" - {k}: {v}")

# ---------------------------------------------------------
# 2) clean main game table
# ---------------------------------------------------------
games = games.copy()

games[game_id_col] = pd.to_numeric(games[game_id_col], errors="coerce")
games[date_col] = pd.to_datetime(games[date_col], errors="coerce")
games[away_runs_col] = pd.to_numeric(games[away_runs_col], errors="coerce")
games[home_runs_col] = pd.to_numeric(games[home_runs_col], errors="coerce")

games = games[
    games[game_id_col].notna() &
    games[date_col].notna() &
    games[away_runs_col].notna() &
    games[home_runs_col].notna()
].copy()

games[game_id_col] = games[game_id_col].astype("int64")

rename_map = {
    game_id_col: "game_id",
    date_col: "date",
    away_team_col: "away_team",
    home_team_col: "home_team",
    away_runs_col: "away_runs",
    home_runs_col: "home_runs",
}
if home_win_col is not None:
    rename_map[home_win_col] = "home_win"
if margin_col is not None:
    rename_map[margin_col] = "home_margin"
if total_col is not None:
    rename_map[total_col] = "total_runs"

games = games.rename(columns=rename_map).copy()

keep_cols = [c for c in [
    "game_id", "date", "away_team", "home_team",
    "away_runs", "home_runs", "home_win", "home_margin", "total_runs"
] if c in games.columns]

games = (
    games[keep_cols]
    .drop_duplicates(subset=["game_id"])
    .sort_values(["date", "game_id"])
    .reset_index(drop=True)
)

# derive missing helper cols if needed
if "home_win" not in games.columns:
    games["home_win"] = (games["home_runs"] > games["away_runs"]).astype(int)

if "home_margin" not in games.columns:
    games["home_margin"] = games["home_runs"] - games["away_runs"]

if "total_runs" not in games.columns:
    games["total_runs"] = games["home_runs"] + games["away_runs"]

print("")
print(f"Clean games shape: {games.shape}")
print("Missing counts in cleaned games:")
print(games[["game_id", "date", "away_runs", "home_runs"]].isna().sum())

# ---------------------------------------------------------
# 3) clean starter table
# ---------------------------------------------------------
starters = starters.copy()

required_starter_cols = [
    "game_id",
    "away_starter_id", "away_starter_name",
    "home_starter_id", "home_starter_name",
]
missing_starter_cols = [c for c in required_starter_cols if c not in starters.columns]
if missing_starter_cols:
    raise ValueError(f"mlb_starter_hist missing required columns: {missing_starter_cols}")

starters["game_id"] = pd.to_numeric(starters["game_id"], errors="coerce")
starters["away_starter_id"] = pd.to_numeric(starters["away_starter_id"], errors="coerce")
starters["home_starter_id"] = pd.to_numeric(starters["home_starter_id"], errors="coerce")

starters = starters[starters["game_id"].notna()].copy()
starters["game_id"] = starters["game_id"].astype("int64")

starter_keep = [c for c in [
    "game_id",
    "away_starter_id", "away_starter_name",
    "home_starter_id", "home_starter_name",
    "starter_source"
] if c in starters.columns]

starters = (
    starters[starter_keep]
    .drop_duplicates(subset=["game_id"], keep="last")
    .reset_index(drop=True)
)

print("")
print(f"Clean starters shape: {starters.shape}")
print("Starter missing counts:")
print(starters[["away_starter_id", "home_starter_id"]].isna().sum())

# ---------------------------------------------------------
# 4) merge starters onto games
# ---------------------------------------------------------
merged = games.merge(
    starters,
    on="game_id",
    how="left",
    validate="one_to_one"
).copy()

print("")
print(f"Merged games+starters shape: {merged.shape}")
print("Post-merge missing counts:")
print(pd.Series({
    "missing_away_starter_id": int(merged["away_starter_id"].isna().sum()),
    "missing_home_starter_id": int(merged["home_starter_id"].isna().sum()),
}))

# ---------------------------------------------------------
# 5) build long-format pitcher rows
# ---------------------------------------------------------
away_pitcher = pd.DataFrame({
    "game_id": merged["game_id"],
    "date": merged["date"],
    "team": merged["away_team"],
    "opponent": merged["home_team"],
    "pitcher_id": merged["away_starter_id"],
    "pitcher_name": merged["away_starter_name"],
    "starter_side": "away",
    "starter_home_flag": 0,
    "runs_allowed": merged["home_runs"],
    "runs_scored_support": merged["away_runs"],
    "pitcher_margin": merged["away_runs"] - merged["home_runs"],
    "pitcher_win": (merged["away_runs"] > merged["home_runs"]).astype(float),
    "team_win": (merged["away_runs"] > merged["home_runs"]).astype(float),
    "home_team": merged["home_team"],
    "away_team": merged["away_team"],
    "home_runs": merged["home_runs"],
    "away_runs": merged["away_runs"],
})

home_pitcher = pd.DataFrame({
    "game_id": merged["game_id"],
    "date": merged["date"],
    "team": merged["home_team"],
    "opponent": merged["away_team"],
    "pitcher_id": merged["home_starter_id"],
    "pitcher_name": merged["home_starter_name"],
    "starter_side": "home",
    "starter_home_flag": 1,
    "runs_allowed": merged["away_runs"],
    "runs_scored_support": merged["home_runs"],
    "pitcher_margin": merged["home_runs"] - merged["away_runs"],
    "pitcher_win": (merged["home_runs"] > merged["away_runs"]).astype(float),
    "team_win": (merged["home_runs"] > merged["away_runs"]).astype(float),
    "home_team": merged["home_team"],
    "away_team": merged["away_team"],
    "home_runs": merged["home_runs"],
    "away_runs": merged["away_runs"],
})

pitcher_games_long = pd.concat([away_pitcher, home_pitcher], ignore_index=True)

# ---------------------------------------------------------
# 6) clean long-format table
# ---------------------------------------------------------
pitcher_games_long["date"] = pd.to_datetime(pitcher_games_long["date"], errors="coerce")
pitcher_games_long["game_id"] = pd.to_numeric(pitcher_games_long["game_id"], errors="coerce")
pitcher_games_long["pitcher_id"] = pd.to_numeric(pitcher_games_long["pitcher_id"], errors="coerce")

num_cols = [
    "starter_home_flag",
    "runs_allowed",
    "runs_scored_support",
    "pitcher_margin",
    "pitcher_win",
    "team_win",
    "home_runs",
    "away_runs",
]
for c in num_cols:
    pitcher_games_long[c] = pd.to_numeric(pitcher_games_long[c], errors="coerce")

pitcher_games_long = pitcher_games_long[
    pitcher_games_long["game_id"].notna() &
    pitcher_games_long["date"].notna()
].copy()

pitcher_games_long["game_id"] = pitcher_games_long["game_id"].astype("int64")

pitcher_games_long = pitcher_games_long.sort_values(
    ["pitcher_id", "date", "game_id", "starter_home_flag"]
).reset_index(drop=True)

# ---------------------------------------------------------
# 7) debug summary
# ---------------------------------------------------------
print("")
print("=" * 80)
print("PITCHER GAME LOG SUMMARY")
print("=" * 80)
print(f"pitcher_games_long shape: {pitcher_games_long.shape}")

summary = pd.Series({
    "total_rows": int(len(pitcher_games_long)),
    "unique_games": int(pitcher_games_long["game_id"].nunique()),
    "unique_pitchers": int(pitcher_games_long["pitcher_id"].nunique()),
    "rows_missing_pitcher_id": int(pitcher_games_long["pitcher_id"].isna().sum()),
    "rows_missing_pitcher_name": int(pitcher_games_long["pitcher_name"].isna().sum()),
    "rows_missing_runs_allowed": int(pitcher_games_long["runs_allowed"].isna().sum()),
    "rows_missing_runs_scored_support": int(pitcher_games_long["runs_scored_support"].isna().sum()),
})
print(summary)

print("")
print("Starter side counts:")
print(pitcher_games_long["starter_side"].value_counts(dropna=False))

print("")
print("Sample rows:")
display_cols = [c for c in [
    "game_id", "date", "team", "opponent",
    "pitcher_id", "pitcher_name",
    "starter_side", "starter_home_flag",
    "runs_allowed", "runs_scored_support",
    "pitcher_margin", "pitcher_win"
] if c in pitcher_games_long.columns]
display(pitcher_games_long[display_cols].head(12))

print("")
print("Sample missing pitcher rows:")
display(pitcher_games_long[pitcher_games_long["pitcher_id"].isna()][display_cols].head(10))

# ---------------------------------------------------------
# 8) save + expose
# ---------------------------------------------------------
save_dir = Path("data/processed/mlb")
save_dir.mkdir(parents=True, exist_ok=True)

csv_path = save_dir / "mlb_pitcher_games_long.csv"
parquet_path = save_dir / "mlb_pitcher_games_long.parquet"

try:
    pitcher_games_long.to_csv(csv_path, index=False)
    print(f"Saved CSV    : {csv_path}")
except Exception as e:
    print(f"WARNING: could not save CSV -> {e}")

try:
    pitcher_games_long.to_parquet(parquet_path, index=False)
    print(f"Saved Parquet: {parquet_path}")
except Exception as e:
    print(f"WARNING: could not save Parquet -> {e}")

globals()["mlb_pitcher_games_long"] = pitcher_games_long.copy()

print("")
print("Created in memory as: mlb_pitcher_games_long")
print("Done.")

BUILD LONG-FORMAT HISTORICAL PITCHER GAME LOG
mlb shape before merge: (8230, 134)
mlb_starter_hist shape: (8230, 23)

Detected game columns:
 - game_id_col: game_id
 - date_col: date
 - away_team_col: away_team
 - home_team_col: home_team
 - away_runs_col: away_runs
 - home_runs_col: home_runs
 - home_win_col: home_win
 - margin_col: home_margin
 - total_col: total_runs



Clean games shape: (8230, 9)
Missing counts in cleaned games:


game_id      0
date         0
away_runs    0
home_runs    0
dtype: int64

Clean starters shape: (8230, 6)
Starter missing counts:
away_starter_id    5
home_starter_id    9
dtype: int64

Merged games+starters shape: (8230, 14)
Post-merge missing counts:
missing_away_starter_id    5
missing_home_starter_id    9
dtype: int64



PITCHER GAME LOG SUMMARY
pitcher_games_long shape: (16460, 17)
total_rows                          16460
unique_games                         8230
unique_pitchers                       740
rows_missing_pitcher_id                14
rows_missing_pitcher_name              14
rows_missing_runs_allowed               0
rows_missing_runs_scored_support        0
dtype: int64

Starter side counts:
starter_side
home    8230
away    8230
Name: count, dtype: int64

Sample rows:


,game_id,date,team,opponent,pitcher_id,pitcher_name,starter_side,starter_home_flag,runs_allowed,runs_scored_support,pitcher_margin,pitcher_win
0,718288,2023-05-06,St. Louis Cardinals,Detroit Tigers,425794.0,Adam Wainwright,home,1,6.0,5.0,-1.0,0.0
1,718211,2023-05-12,St. Louis Cardinals,Boston Red Sox,425794.0,Adam Wainwright,away,0,6.0,8.0,2.0,1.0
2,718124,2023-05-18,St. Louis Cardinals,Los Angeles Dodgers,425794.0,Adam Wainwright,home,1,8.0,16.0,8.0,1.0
3,718059,2023-05-23,St. Louis Cardinals,Cincinnati Reds,425794.0,Adam Wainwright,away,0,5.0,8.0,3.0,1.0
4,717975,2023-05-29,St. Louis Cardinals,Kansas City Royals,425794.0,Adam Wainwright,home,1,7.0,0.0,-7.0,0.0
5,717879,2023-06-05,St. Louis Cardinals,Texas Rangers,425794.0,Adam Wainwright,away,0,4.0,3.0,-1.0,0.0
6,717797,2023-06-11,St. Louis Cardinals,Cincinnati Reds,425794.0,Adam Wainwright,home,1,4.0,3.0,-1.0,0.0
7,717718,2023-06-17,St. Louis Cardinals,New York Mets,425794.0,Adam Wainwright,away,0,3.0,5.0,2.0,1.0
8,717637,2023-06-24,St. Louis Cardinals,Chicago Cubs,425794.0,Adam Wainwright,home,1,9.0,1.0,-8.0,0.0
9,717566,2023-06-29,St. Louis Cardinals,Houston Astros,425794.0,Adam Wainwright,home,1,14.0,0.0,-14.0,0.0



Sample missing pitcher rows:


,game_id,date,team,opponent,pitcher_id,pitcher_name,starter_side,starter_home_flag,runs_allowed,runs_scored_support,pitcher_margin,pitcher_win
16446,718944,2023-03-26,New York Mets,Miami Marlins,NaN,NaN,home,1,3.0,3.0,0.0,0.0
16447,717427,2023-07-09,Cincinnati Reds,Milwaukee Brewers,NaN,NaN,away,0,1.0,0.0,-1.0,0.0
16448,748083,2024-03-06,Seattle Mariners,Kansas City Royals,NaN,NaN,away,0,7.0,8.0,1.0,1.0
16449,747871,2024-03-21,Seattle Mariners,Cincinnati Reds,NaN,NaN,home,1,6.0,7.0,1.0,1.0
16450,747812,2024-03-25,Texas Rangers,Boston Red Sox,NaN,NaN,home,1,9.0,2.0,-7.0,0.0
16451,766494,2024-03-25,Diablos Rojos del Mexico,New York Yankees,NaN,NaN,home,1,5.0,8.0,3.0,1.0
16452,746367,2024-07-11,Houston Astros,Miami Marlins,NaN,NaN,home,1,3.0,6.0,3.0,1.0
16453,745063,2024-07-28,Tampa Bay Rays,Cincinnati Reds,NaN,NaN,home,1,1.0,2.0,1.0,1.0
16454,790401,2025-03-16,Tokyo Yomiuri Giants,Chicago Cubs,NaN,NaN,home,1,4.0,2.0,-2.0,0.0
16455,790402,2025-03-16,Hanshin Tigers,Los Angeles Dodgers,NaN,NaN,home,1,0.0,3.0,3.0,1.0


Saved CSV    : data\processed\mlb\mlb_pitcher_games_long.csv
Saved Parquet: data\processed\mlb\mlb_pitcher_games_long.parquet

Created in memory as: mlb_pitcher_games_long
Done.


In [73]:
# %%
# =========================================================
# CELL Ã¢â‚¬â€ BUILD LEAKAGE-SAFE ROLLING PITCHER FEATURES
# Goal:
# - create prior-only rolling starter features by pitcher
# - no future leakage
# - ready to merge back to game-level table later
# =========================================================

import numpy as np
import pandas as pd
from pathlib import Path

print("=" * 80)
print("BUILD LEAKAGE-SAFE ROLLING PITCHER FEATURES")
print("=" * 80)

# ---------------------------------------------------------
# 0) validate input
# ---------------------------------------------------------
if "mlb_pitcher_games_long" not in globals() or not isinstance(mlb_pitcher_games_long, pd.DataFrame):
    raise ValueError("mlb_pitcher_games_long not found. Run the pitcher game log cell first.")

pg = mlb_pitcher_games_long.copy()
print(f"Input shape: {pg.shape}")

required_cols = [
    "game_id", "date", "pitcher_id", "pitcher_name",
    "starter_side", "starter_home_flag",
    "runs_allowed", "runs_scored_support",
    "pitcher_margin", "pitcher_win"
]
missing_required = [c for c in required_cols if c not in pg.columns]
if missing_required:
    raise ValueError(f"mlb_pitcher_games_long missing required columns: {missing_required}")

# ---------------------------------------------------------
# 1) clean base table
# ---------------------------------------------------------
pg["game_id"] = pd.to_numeric(pg["game_id"], errors="coerce")
pg["pitcher_id"] = pd.to_numeric(pg["pitcher_id"], errors="coerce")
pg["date"] = pd.to_datetime(pg["date"], errors="coerce")

num_cols = [
    "starter_home_flag",
    "runs_allowed",
    "runs_scored_support",
    "pitcher_margin",
    "pitcher_win",
]
for c in num_cols:
    pg[c] = pd.to_numeric(pg[c], errors="coerce")

before_rows = len(pg)

pg = pg[
    pg["game_id"].notna() &
    pg["date"].notna()
].copy()

pg["game_id"] = pg["game_id"].astype("int64")

pg = pg.sort_values(["pitcher_id", "date", "game_id", "starter_home_flag"]).reset_index(drop=True)

print("")
print(f"Rows before clean filter: {before_rows:,}")
print(f"Rows after  clean filter: {len(pg):,}")

print("")
print("Missing counts before rolling:")
print(pd.Series({
    "missing_pitcher_id": int(pg["pitcher_id"].isna().sum()),
    "missing_runs_allowed": int(pg["runs_allowed"].isna().sum()),
    "missing_runs_scored_support": int(pg["runs_scored_support"].isna().sum()),
    "missing_pitcher_margin": int(pg["pitcher_margin"].isna().sum()),
    "missing_pitcher_win": int(pg["pitcher_win"].isna().sum()),
}))

# ---------------------------------------------------------
# 2) helper: rest days from prior appearance
# ---------------------------------------------------------
pg["prev_start_date"] = pg.groupby("pitcher_id", dropna=False)["date"].shift(1)
pg["pitcher_rest_days"] = (pg["date"] - pg["prev_start_date"]).dt.days

# optional cleanup: same-day weirdness -> set to 0 minimum
pg["pitcher_rest_days"] = pd.to_numeric(pg["pitcher_rest_days"], errors="coerce")
pg.loc[pg["pitcher_rest_days"] < 0, "pitcher_rest_days"] = np.nan

# ---------------------------------------------------------
# 3) leakage-safe rolling feature builder
# ---------------------------------------------------------
ROLL_WINDOWS = [3, 5, 10]

group = pg.groupby("pitcher_id", dropna=False)

base_stats = {
    "runs_allowed": "starter_runs_allowed",
    "runs_scored_support": "starter_run_support",
    "pitcher_margin": "starter_margin",
    "pitcher_win": "starter_win_rate",
}

# prior start count
pg["starter_prev_starts"] = group.cumcount()

# prior-only expanding means
for src_col, out_base in base_stats.items():
    pg[f"{out_base}_expanding"] = (
        group[src_col]
        .transform(lambda s: s.shift(1).expanding(min_periods=1).mean())
    )

# rolling windows
for w in ROLL_WINDOWS:
    for src_col, out_base in base_stats.items():
        pg[f"{out_base}_{w}"] = (
            group[src_col]
            .transform(lambda s: s.shift(1).rolling(w, min_periods=1).mean())
        )

# ---------------------------------------------------------
# 4) ERA proxy features
# ERA proxy here = 9 * prior avg runs allowed per start
# This is not true ERA, but it is a stable starter run-prevention proxy
# ---------------------------------------------------------
pg["starter_era_proxy_expanding"] = 9.0 * pg["starter_runs_allowed_expanding"]

for w in ROLL_WINDOWS:
    pg[f"starter_era_proxy_{w}"] = 9.0 * pg[f"starter_runs_allowed_{w}"]

# BUG FIX (2026-08-05, real incident): Drew Anderson's spot start (DET @
# SEA) showed a model ERA of 1.41 -- traced to his exact 2 career starts
# (2026-05-20: 4.2 IP, 0 ER; 2026-06-15: 2.2 IP, 1 ER), which average to
# 1 ER / 6.4 IP * 9 = 1.41. His real 2026 ERA across ~38 relief appearances
# is 4.24 (ESPN/RotoWire), but mlb_pitcher_games_long is starts-only by
# construction (built from away_starter_id/home_starter_id in the cell
# above) -- a full-time reliever making a rare spot start has this entire
# feature computed from a 1-2 start sample, with zero signal from the far
# larger and more representative relief-appearance record. NaN out the
# proxy below a minimum starts sample so a downstream consumer falls back
# to a different signal (e.g. cell 83's FIP-based SP ELO, which already
# treats thin-sample starters conservatively via SP_ELO_OVERRIDES) instead
# of silently trusting a tiny, unrepresentative starts-only average.
MIN_STARTS_FOR_ERA_PROXY = 3
_thin_sample = pg["starter_prev_starts"] < MIN_STARTS_FOR_ERA_PROXY
pg.loc[_thin_sample, "starter_era_proxy_expanding"] = np.nan
for w in ROLL_WINDOWS:
    pg.loc[_thin_sample, f"starter_era_proxy_{w}"] = np.nan
pg["starter_era_proxy_thin_sample"] = _thin_sample
_n_thin = int(_thin_sample.sum())
if _n_thin:
    print(f"Starts-only ERA proxy suppressed for {_n_thin} thin-sample row(s) "
          f"(< {MIN_STARTS_FOR_ERA_PROXY} prior starts) -- relief-heavy pitchers "
          f"making a rare start would otherwise show an unrepresentative ERA.")

# ---------------------------------------------------------
# 5) additional rolling rest features
# prior-only average rest by pitcher
# ---------------------------------------------------------
pg["starter_rest_days_expanding"] = (
    group["pitcher_rest_days"]
    .transform(lambda s: s.shift(1).expanding(min_periods=1).mean())
)

for w in ROLL_WINDOWS:
    pg[f"starter_rest_days_{w}"] = (
        group["pitcher_rest_days"]
        .transform(lambda s: s.shift(1).rolling(w, min_periods=1).mean())
    )

# ---------------------------------------------------------
# 6) first-start flags / usability helpers
# ---------------------------------------------------------
pg["starter_has_history"] = (pg["starter_prev_starts"] > 0).astype(int)
pg["starter_has_3_start_history"] = (pg["starter_prev_starts"] >= 3).astype(int)
pg["starter_has_5_start_history"] = (pg["starter_prev_starts"] >= 5).astype(int)

# ---------------------------------------------------------
# 7) final sort
# ---------------------------------------------------------
pg = pg.sort_values(["pitcher_id", "date", "game_id", "starter_home_flag"]).reset_index(drop=True)

# ---------------------------------------------------------
# 8) debug summary
# ---------------------------------------------------------
feature_cols = [c for c in pg.columns if c.startswith("starter_") or c == "pitcher_rest_days"]

print("")
print("=" * 80)
print("ROLLING PITCHER FEATURE SUMMARY")
print("=" * 80)
print(f"Output shape: {pg.shape}")
print(f"Feature column count: {len(feature_cols)}")

summary = pd.Series({
    "unique_pitchers": int(pg["pitcher_id"].nunique()),
    "rows_missing_pitcher_id": int(pg["pitcher_id"].isna().sum()),
    "rows_with_no_history": int((pg["starter_prev_starts"] == 0).sum()),
    "rows_with_history": int((pg["starter_prev_starts"] > 0).sum()),
    "non_null_runs_allowed_3": int(pg["starter_runs_allowed_3"].notna().sum()),
    "non_null_runs_allowed_5": int(pg["starter_runs_allowed_5"].notna().sum()),
    "non_null_runs_allowed_10": int(pg["starter_runs_allowed_10"].notna().sum()),
    "non_null_era_proxy_3": int(pg["starter_era_proxy_3"].notna().sum()),
    "non_null_win_rate_5": int(pg["starter_win_rate_5"].notna().sum()),
    "non_null_rest_days": int(pg["pitcher_rest_days"].notna().sum()),
})
print(summary)

print("")
print("Missing counts for key features:")
key_check_cols = [
    "starter_prev_starts",
    "pitcher_rest_days",
    "starter_runs_allowed_3",
    "starter_runs_allowed_5",
    "starter_runs_allowed_10",
    "starter_era_proxy_3",
    "starter_win_rate_3",
    "starter_margin_3",
    "starter_run_support_3",
]
key_check_cols = [c for c in key_check_cols if c in pg.columns]
print(pg[key_check_cols].isna().sum())

print("")
print("Sample feature rows:")
show_cols = [c for c in [
    "game_id", "date", "team", "opponent",
    "pitcher_id", "pitcher_name", "starter_side",
    "runs_allowed", "runs_scored_support", "pitcher_margin", "pitcher_win",
    "starter_prev_starts", "pitcher_rest_days",
    "starter_runs_allowed_3", "starter_runs_allowed_5", "starter_runs_allowed_10",
    "starter_era_proxy_3", "starter_win_rate_3",
    "starter_margin_3", "starter_run_support_3"
] if c in pg.columns]
display(pg[show_cols].head(15))

print("")
print("Sample rows with history:")
display(pg.loc[pg["starter_prev_starts"] > 0, show_cols].head(15))

# ---------------------------------------------------------
# 9) save + expose
# ---------------------------------------------------------
save_dir = Path("data/processed/mlb")
save_dir.mkdir(parents=True, exist_ok=True)

csv_path = save_dir / "mlb_pitcher_games_features.csv"
parquet_path = save_dir / "mlb_pitcher_games_features.parquet"

try:
    pg.to_csv(csv_path, index=False)
    print(f"Saved CSV    : {csv_path}")
except Exception as e:
    print(f"WARNING: could not save CSV -> {e}")

try:
    pg.to_parquet(parquet_path, index=False)
    print(f"Saved Parquet: {parquet_path}")
except Exception as e:
    print(f"WARNING: could not save Parquet -> {e}")

globals()["mlb_pitcher_games_features"] = pg.copy()

print("")
print("Created in memory as: mlb_pitcher_games_features")
print("Done.")

BUILD LEAKAGE-SAFE ROLLING PITCHER FEATURES
Input shape: (16460, 17)

Rows before clean filter: 16,460
Rows after  clean filter: 16,460

Missing counts before rolling:
missing_pitcher_id             14
missing_runs_allowed            0
missing_runs_scored_support     0
missing_pitcher_margin          0
missing_pitcher_win             0
dtype: int64


Starts-only ERA proxy suppressed for 1814 thin-sample row(s) (< 3 prior starts) -- relief-heavy pitchers making a rare start would otherwise show an unrepresentative ERA.



ROLLING PITCHER FEATURE SUMMARY
Output shape: (16460, 48)
Feature column count: 32
unique_pitchers               740
rows_missing_pitcher_id        14
rows_with_no_history          741
rows_with_history           15719
non_null_runs_allowed_3     15719
non_null_runs_allowed_5     15719
non_null_runs_allowed_10    15719
non_null_era_proxy_3        14646
non_null_win_rate_5         15719
non_null_rest_days          15719
dtype: int64

Missing counts for key features:
starter_prev_starts           0
pitcher_rest_days           741
starter_runs_allowed_3      741
starter_runs_allowed_5      741
starter_runs_allowed_10     741
starter_era_proxy_3        1814
starter_win_rate_3          741
starter_margin_3            741
starter_run_support_3       741
dtype: int64

Sample feature rows:


,game_id,date,team,opponent,pitcher_id,pitcher_name,starter_side,runs_allowed,runs_scored_support,pitcher_margin,pitcher_win,starter_prev_starts,pitcher_rest_days,starter_runs_allowed_3,starter_runs_allowed_5,starter_runs_allowed_10,starter_era_proxy_3,starter_win_rate_3,starter_margin_3,starter_run_support_3
0,718288,2023-05-06,St. Louis Cardinals,Detroit Tigers,425794.0,Adam Wainwright,home,6.0,5.0,-1.0,0.0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,718211,2023-05-12,St. Louis Cardinals,Boston Red Sox,425794.0,Adam Wainwright,away,6.0,8.0,2.0,1.0,1,6.0,6.000000,6.000000,6.000000,NaN,0.000000,-1.000000,5.000000
2,718124,2023-05-18,St. Louis Cardinals,Los Angeles Dodgers,425794.0,Adam Wainwright,home,8.0,16.0,8.0,1.0,2,6.0,6.000000,6.000000,6.000000,NaN,0.500000,0.500000,6.500000
3,718059,2023-05-23,St. Louis Cardinals,Cincinnati Reds,425794.0,Adam Wainwright,away,5.0,8.0,3.0,1.0,3,5.0,6.666667,6.666667,6.666667,60.0,0.666667,3.000000,9.666667
4,717975,2023-05-29,St. Louis Cardinals,Kansas City Royals,425794.0,Adam Wainwright,home,7.0,0.0,-7.0,0.0,4,6.0,6.333333,6.250000,6.250000,57.0,1.000000,4.333333,10.666667
5,717879,2023-06-05,St. Louis Cardinals,Texas Rangers,425794.0,Adam Wainwright,away,4.0,3.0,-1.0,0.0,5,7.0,6.666667,6.400000,6.400000,60.0,0.666667,1.333333,8.000000
6,717797,2023-06-11,St. Louis Cardinals,Cincinnati Reds,425794.0,Adam Wainwright,home,4.0,3.0,-1.0,0.0,6,6.0,5.333333,6.000000,6.000000,48.0,0.333333,-1.666667,3.666667
7,717718,2023-06-17,St. Louis Cardinals,New York Mets,425794.0,Adam Wainwright,away,3.0,5.0,2.0,1.0,7,6.0,5.000000,5.600000,5.714286,45.0,0.000000,-3.000000,2.000000
8,717637,2023-06-24,St. Louis Cardinals,Chicago Cubs,425794.0,Adam Wainwright,home,9.0,1.0,-8.0,0.0,8,7.0,3.666667,4.600000,5.375000,33.0,0.333333,0.000000,3.666667
9,717566,2023-06-29,St. Louis Cardinals,Houston Astros,425794.0,Adam Wainwright,home,14.0,0.0,-14.0,0.0,9,5.0,5.333333,5.400000,5.777778,48.0,0.333333,-2.333333,3.000000



Sample rows with history:


,game_id,date,team,opponent,pitcher_id,pitcher_name,starter_side,runs_allowed,runs_scored_support,pitcher_margin,pitcher_win,starter_prev_starts,pitcher_rest_days,starter_runs_allowed_3,starter_runs_allowed_5,starter_runs_allowed_10,starter_era_proxy_3,starter_win_rate_3,starter_margin_3,starter_run_support_3
1,718211,2023-05-12,St. Louis Cardinals,Boston Red Sox,425794.0,Adam Wainwright,away,6.0,8.0,2.0,1.0,1,6.0,6.000000,6.000000,6.000000,NaN,0.000000,-1.000000,5.000000
2,718124,2023-05-18,St. Louis Cardinals,Los Angeles Dodgers,425794.0,Adam Wainwright,home,8.0,16.0,8.0,1.0,2,6.0,6.000000,6.000000,6.000000,NaN,0.500000,0.500000,6.500000
3,718059,2023-05-23,St. Louis Cardinals,Cincinnati Reds,425794.0,Adam Wainwright,away,5.0,8.0,3.0,1.0,3,5.0,6.666667,6.666667,6.666667,60.0,0.666667,3.000000,9.666667
4,717975,2023-05-29,St. Louis Cardinals,Kansas City Royals,425794.0,Adam Wainwright,home,7.0,0.0,-7.0,0.0,4,6.0,6.333333,6.250000,6.250000,57.0,1.000000,4.333333,10.666667
5,717879,2023-06-05,St. Louis Cardinals,Texas Rangers,425794.0,Adam Wainwright,away,4.0,3.0,-1.0,0.0,5,7.0,6.666667,6.400000,6.400000,60.0,0.666667,1.333333,8.000000
6,717797,2023-06-11,St. Louis Cardinals,Cincinnati Reds,425794.0,Adam Wainwright,home,4.0,3.0,-1.0,0.0,6,6.0,5.333333,6.000000,6.000000,48.0,0.333333,-1.666667,3.666667
7,717718,2023-06-17,St. Louis Cardinals,New York Mets,425794.0,Adam Wainwright,away,3.0,5.0,2.0,1.0,7,6.0,5.000000,5.600000,5.714286,45.0,0.000000,-3.000000,2.000000
8,717637,2023-06-24,St. Louis Cardinals,Chicago Cubs,425794.0,Adam Wainwright,home,9.0,1.0,-8.0,0.0,8,7.0,3.666667,4.600000,5.375000,33.0,0.333333,0.000000,3.666667
9,717566,2023-06-29,St. Louis Cardinals,Houston Astros,425794.0,Adam Wainwright,home,14.0,0.0,-14.0,0.0,9,5.0,5.333333,5.400000,5.777778,48.0,0.333333,-2.333333,3.000000
10,717502,2023-07-04,St. Louis Cardinals,Miami Marlins,425794.0,Adam Wainwright,away,15.0,2.0,-13.0,0.0,10,5.0,8.666667,6.800000,6.600000,78.0,0.333333,-6.666667,2.000000


Saved CSV    : data\processed\mlb\mlb_pitcher_games_features.csv
Saved Parquet: data\processed\mlb\mlb_pitcher_games_features.parquet

Created in memory as: mlb_pitcher_games_features
Done.


In [74]:
# %%
# =========================================================
# CELL Ã¢â‚¬â€ MERGE STARTER FEATURES BACK TO GAME-LEVEL MLB TABLE
# Goal:
# - attach away/home starter rolling features to each game
# - create diff starter features
# - preserve existing mlb table structure
# =========================================================

import numpy as np
import pandas as pd
from pathlib import Path

print("=" * 80)
print("MERGE STARTER FEATURES BACK TO GAME-LEVEL MLB TABLE")
print("=" * 80)

# ---------------------------------------------------------
# 0) validate inputs
# ---------------------------------------------------------
if "mlb" not in globals() or not isinstance(mlb, pd.DataFrame):
    raise ValueError("mlb dataframe not found in memory.")

if "mlb_pitcher_games_features" not in globals() or not isinstance(mlb_pitcher_games_features, pd.DataFrame):
    raise ValueError("mlb_pitcher_games_features not found. Run the rolling pitcher feature cell first.")

games = mlb.copy()
pgf = mlb_pitcher_games_features.copy()

print(f"Base mlb shape: {games.shape}")
print(f"Pitcher feature table shape: {pgf.shape}")

# ---------------------------------------------------------
# 1) detect game_id column in mlb safely
# ---------------------------------------------------------
def find_first_col(df, options):
    lower_map = {c.lower(): c for c in df.columns}
    for opt in options:
        if opt.lower() in lower_map:
            return lower_map[opt.lower()]
    return None

game_id_col = find_first_col(games, ["game_id", "gamepk", "game_pk", "pk"])
if game_id_col is None:
    raise ValueError("Could not find game_id column in mlb dataframe.")

games[game_id_col] = pd.to_numeric(games[game_id_col], errors="coerce")
games = games[games[game_id_col].notna()].copy()
games[game_id_col] = games[game_id_col].astype("int64")

if game_id_col != "game_id":
    games = games.rename(columns={game_id_col: "game_id"}).copy()

print("")
print(f"Working mlb shape after game_id clean: {games.shape}")

# ---------------------------------------------------------
# 2) choose pitcher feature columns to merge
# Keep this focused so we do not overload the model table
# ---------------------------------------------------------
candidate_feature_cols = [
    "pitcher_id",
    "pitcher_name",
    "starter_prev_starts",
    "pitcher_rest_days",

    "starter_runs_allowed_3",
    "starter_runs_allowed_5",
    "starter_runs_allowed_10",

    "starter_era_proxy_3",
    "starter_era_proxy_5",
    "starter_era_proxy_10",

    "starter_win_rate_3",
    "starter_win_rate_5",
    "starter_win_rate_10",

    "starter_margin_3",
    "starter_margin_5",
    "starter_margin_10",

    "starter_run_support_3",
    "starter_run_support_5",
    "starter_run_support_10",

    "starter_rest_days_3",
    "starter_rest_days_5",
    "starter_rest_days_10",

    "starter_has_history",
    "starter_has_3_start_history",
    "starter_has_5_start_history",
]

base_required = ["game_id", "starter_side"]
missing_base = [c for c in base_required if c not in pgf.columns]
if missing_base:
    raise ValueError(f"mlb_pitcher_games_features missing required columns: {missing_base}")

feature_cols = [c for c in candidate_feature_cols if c in pgf.columns]
if not feature_cols:
    raise ValueError("No candidate pitcher feature columns found to merge.")

print("")
print(f"Pitcher feature columns selected: {len(feature_cols)}")
print(feature_cols)

# ---------------------------------------------------------
# 3) split away/home rows from long pitcher feature table
# ---------------------------------------------------------
away_pgf = pgf.loc[pgf["starter_side"].astype(str).str.lower() == "away", ["game_id"] + feature_cols].copy()
home_pgf = pgf.loc[pgf["starter_side"].astype(str).str.lower() == "home", ["game_id"] + feature_cols].copy()

away_pgf = away_pgf.drop_duplicates(subset=["game_id"], keep="last").reset_index(drop=True)
home_pgf = home_pgf.drop_duplicates(subset=["game_id"], keep="last").reset_index(drop=True)

away_rename = {c: f"away_{c}" for c in feature_cols}
home_rename = {c: f"home_{c}" for c in feature_cols}

away_pgf = away_pgf.rename(columns=away_rename)
home_pgf = home_pgf.rename(columns=home_rename)

print("")
print(f"Away pitcher feature rows: {away_pgf.shape}")
print(f"Home pitcher feature rows: {home_pgf.shape}")

# ---------------------------------------------------------
# 4) merge onto mlb table
# ---------------------------------------------------------
merged = games.merge(
    away_pgf,
    on="game_id",
    how="left",
    validate="one_to_one"
).merge(
    home_pgf,
    on="game_id",
    how="left",
    validate="one_to_one"
)

print("")
print(f"Merged table shape: {merged.shape}")

# ---------------------------------------------------------
# 5) create diff starter features
# diff = away - home, matching your existing modeling style
# ---------------------------------------------------------
diff_bases = [
    "starter_prev_starts",
    "pitcher_rest_days",

    "starter_runs_allowed_3",
    "starter_runs_allowed_5",
    "starter_runs_allowed_10",

    "starter_era_proxy_3",
    "starter_era_proxy_5",
    "starter_era_proxy_10",

    "starter_win_rate_3",
    "starter_win_rate_5",
    "starter_win_rate_10",

    "starter_margin_3",
    "starter_margin_5",
    "starter_margin_10",

    "starter_run_support_3",
    "starter_run_support_5",
    "starter_run_support_10",

    "starter_rest_days_3",
    "starter_rest_days_5",
    "starter_rest_days_10",
]

diff_created = []

for base in diff_bases:
    away_col = f"away_{base}"
    home_col = f"home_{base}"
    diff_col = f"diff_{base}"

    if away_col in merged.columns and home_col in merged.columns:
        merged[away_col] = pd.to_numeric(merged[away_col], errors="coerce")
        merged[home_col] = pd.to_numeric(merged[home_col], errors="coerce")
        merged[diff_col] = merged[away_col] - merged[home_col]
        diff_created.append(diff_col)

# ---------------------------------------------------------
# 6) helpful fallback fills for history flags
# keep this conservative and non-destructive
# ---------------------------------------------------------
flag_cols = [
    "away_starter_has_history",
    "home_starter_has_history",
    "away_starter_has_3_start_history",
    "home_starter_has_3_start_history",
    "away_starter_has_5_start_history",
    "home_starter_has_5_start_history",
]
for c in flag_cols:
    if c in merged.columns:
        merged[c] = pd.to_numeric(merged[c], errors="coerce")

# ---------------------------------------------------------
# 7) debug summary
# ---------------------------------------------------------
away_key_cols = [c for c in merged.columns if c.startswith("away_starter_")]
home_key_cols = [c for c in merged.columns if c.startswith("home_starter_")]
diff_key_cols = [c for c in merged.columns if c.startswith("diff_starter_") or c.startswith("diff_pitcher_")]

print("")
print("=" * 80)
print("STARTER FEATURE MERGE SUMMARY")
print("=" * 80)
print(f"Final merged shape: {merged.shape}")
print(f"Away starter feature cols: {len(away_key_cols)}")
print(f"Home starter feature cols: {len(home_key_cols)}")
print(f"Diff starter feature cols: {len(diff_created)}")

summary = pd.Series({
    "rows_total": int(len(merged)),
    "rows_missing_away_pitcher_id": int(merged["away_pitcher_id"].isna().sum()) if "away_pitcher_id" in merged.columns else np.nan,
    "rows_missing_home_pitcher_id": int(merged["home_pitcher_id"].isna().sum()) if "home_pitcher_id" in merged.columns else np.nan,
    "rows_non_null_away_runs_allowed_10": int(merged["away_starter_runs_allowed_10"].notna().sum()) if "away_starter_runs_allowed_10" in merged.columns else np.nan,
    "rows_non_null_home_runs_allowed_10": int(merged["home_starter_runs_allowed_10"].notna().sum()) if "home_starter_runs_allowed_10" in merged.columns else np.nan,
    "rows_non_null_diff_runs_allowed_10": int(merged["diff_starter_runs_allowed_10"].notna().sum()) if "diff_starter_runs_allowed_10" in merged.columns else np.nan,
    "rows_non_null_diff_win_rate_10": int(merged["diff_starter_win_rate_10"].notna().sum()) if "diff_starter_win_rate_10" in merged.columns else np.nan,
    "rows_non_null_diff_rest_days": int(merged["diff_pitcher_rest_days"].notna().sum()) if "diff_pitcher_rest_days" in merged.columns else np.nan,
})
print(summary)

print("")
print("Missing counts for key merged starter columns:")
key_cols = [c for c in [
    "away_pitcher_id",
    "home_pitcher_id",
    "away_starter_prev_starts",
    "home_starter_prev_starts",
    "away_starter_runs_allowed_10",
    "home_starter_runs_allowed_10",
    "diff_starter_runs_allowed_10",
    "diff_starter_win_rate_10",
    "diff_pitcher_rest_days",
] if c in merged.columns]
print(merged[key_cols].isna().sum())

print("")
print("Sample merged starter feature rows:")
show_cols = [c for c in [
    "game_id",
    "away_team",
    "home_team",

    "away_pitcher_name",
    "home_pitcher_name",

    "away_starter_prev_starts",
    "home_starter_prev_starts",
    "diff_starter_prev_starts",

    "away_starter_runs_allowed_10",
    "home_starter_runs_allowed_10",
    "diff_starter_runs_allowed_10",

    "away_starter_era_proxy_10",
    "home_starter_era_proxy_10",
    "diff_starter_era_proxy_10",

    "away_starter_win_rate_10",
    "home_starter_win_rate_10",
    "diff_starter_win_rate_10",

    "away_pitcher_rest_days",
    "home_pitcher_rest_days",
    "diff_pitcher_rest_days",
] if c in merged.columns]
display(merged[show_cols].head(12))

# ---------------------------------------------------------
# 8) save + expose
# ---------------------------------------------------------
save_dir = Path("data/processed/mlb")
save_dir.mkdir(parents=True, exist_ok=True)

csv_path = save_dir / "mlb_with_starter_features.csv"
parquet_path = save_dir / "mlb_with_starter_features.parquet"

try:
    merged.to_csv(csv_path, index=False)
    print(f"Saved CSV    : {csv_path}")
except Exception as e:
    print(f"WARNING: could not save CSV -> {e}")

try:
    merged.to_parquet(parquet_path, index=False)
    print(f"Saved Parquet: {parquet_path}")
except Exception as e:
    print(f"WARNING: could not save Parquet -> {e}")

globals()["mlb_with_starter_features"] = merged.copy()

print("")
print("Created in memory as: mlb_with_starter_features")
print("Done.")

MERGE STARTER FEATURES BACK TO GAME-LEVEL MLB TABLE
Base mlb shape: (8230, 134)
Pitcher feature table shape: (16460, 48)

Working mlb shape after game_id clean: (8230, 134)

Pitcher feature columns selected: 25
['pitcher_id', 'pitcher_name', 'starter_prev_starts', 'pitcher_rest_days', 'starter_runs_allowed_3', 'starter_runs_allowed_5', 'starter_runs_allowed_10', 'starter_era_proxy_3', 'starter_era_proxy_5', 'starter_era_proxy_10', 'starter_win_rate_3', 'starter_win_rate_5', 'starter_win_rate_10', 'starter_margin_3', 'starter_margin_5', 'starter_margin_10', 'starter_run_support_3', 'starter_run_support_5', 'starter_run_support_10', 'starter_rest_days_3', 'starter_rest_days_5', 'starter_rest_days_10', 'starter_has_history', 'starter_has_3_start_history', 'starter_has_5_start_history']



Away pitcher feature rows: (8230, 26)
Home pitcher feature rows: (8230, 26)

Merged table shape: (8230, 184)



STARTER FEATURE MERGE SUMMARY
Final merged shape: (8230, 202)
Away starter feature cols: 33
Home starter feature cols: 33
Diff starter feature cols: 18
rows_total                            8230
rows_missing_away_pitcher_id             5
rows_missing_home_pitcher_id             9
rows_non_null_away_runs_allowed_10    7813
rows_non_null_home_runs_allowed_10    7906
rows_non_null_diff_runs_allowed_10    7608
rows_non_null_diff_win_rate_10        7608
rows_non_null_diff_rest_days          7608
dtype: int64

Missing counts for key merged starter columns:
away_pitcher_id                   5
home_pitcher_id                   9
away_starter_prev_starts          0
home_starter_prev_starts          0
away_starter_runs_allowed_10    417
home_starter_runs_allowed_10    324
diff_starter_runs_allowed_10    622
diff_starter_win_rate_10        622
diff_pitcher_rest_days          622
dtype: int64

Sample merged starter feature rows:


,game_id,away_team,home_team,away_pitcher_name,home_pitcher_name,away_starter_prev_starts,home_starter_prev_starts,diff_starter_prev_starts,away_starter_runs_allowed_10,home_starter_runs_allowed_10,diff_starter_runs_allowed_10,away_starter_era_proxy_10,home_starter_era_proxy_10,diff_starter_era_proxy_10,away_starter_win_rate_10,home_starter_win_rate_10,diff_starter_win_rate_10,away_pitcher_rest_days,home_pitcher_rest_days,diff_pitcher_rest_days
0,719302,Arizona Diamondbacks,Los Angeles Dodgers,Ryne Nelson,Michael Grove,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,719315,Atlanta Braves,New York Mets,Kolby Allard,Carlos Carrasco,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,719314,Baltimore Orioles,Detroit Tigers,Grayson Rodriguez,Matthew Boyd,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,719305,Chicago White Sox,Colorado Rockies,Lance Lynn,Noah Davis,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,719304,Los Angeles Angels,Kansas City Royals,Tucker Davidson,Mike Mayers,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,719308,Miami Marlins,Washington Nationals,Edward Cabrera,MacKenzie Gore,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,719307,Minnesota Twins,Tampa Bay Rays,Kenta Maeda,Zach Eflin,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,719310,New York Yankees,Pittsburgh Pirates,Albert Abreu,Mitch Keller,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,719312,Oakland Athletics,Chicago Cubs,JP Sears,Marcus Stroman,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,719317,Philadelphia Phillies,Boston Red Sox,Bailey Falter,Tanner Houck,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Saved CSV    : data\processed\mlb\mlb_with_starter_features.csv


Saved Parquet: data\processed\mlb\mlb_with_starter_features.parquet

Created in memory as: mlb_with_starter_features
Done.


In [75]:
# %%
# =========================================================
# CELL Ã¢â‚¬â€ COMPARE BASE 7-FEATURE MODEL VS STARTER-ENHANCED MODEL
# Goal:
# - train/test on mlb_with_starter_features
# - compare baseline 7-feature model against starter-enhanced version
# - keep split leakage-safe
# - use isotonic calibration
# =========================================================

import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import accuracy_score, log_loss, brier_score_loss

print("=" * 80)
print("COMPARE BASE 7-FEATURE MODEL VS STARTER-ENHANCED MODEL")
print("=" * 80)

# ---------------------------------------------------------
# 0) validate input table
# ---------------------------------------------------------
if "mlb_with_starter_features" not in globals() or not isinstance(mlb_with_starter_features, pd.DataFrame):
    raise ValueError("mlb_with_starter_features not found. Run the starter merge cell first.")

df = mlb_with_starter_features.copy()
print(f"Input shape: {df.shape}")

# ---------------------------------------------------------
# 1) detect core columns safely
# ---------------------------------------------------------
def find_first_col(df_, options):
    lower_map = {c.lower(): c for c in df_.columns}
    for opt in options:
        if opt.lower() in lower_map:
            return lower_map[opt.lower()]
    return None

date_col = find_first_col(df, ["date", "game_date", "game_datetime", "game_datetime_utc"])
target_col = find_first_col(df, ["home_win"])

if date_col is None:
    raise ValueError("Could not find date column.")
if target_col is None:
    raise ValueError("Could not find target column home_win.")

df[date_col] = pd.to_datetime(df[date_col], errors="coerce")
df[target_col] = pd.to_numeric(df[target_col], errors="coerce")

# Keep only completed/usable rows
before_rows = len(df)
df = df[df[date_col].notna() & df[target_col].notna()].copy()
df[target_col] = df[target_col].astype(int)

print("")
print(f"Rows before clean: {before_rows:,}")
print(f"Rows after clean : {len(df):,}")

# ---------------------------------------------------------
# 2) define feature sets
# baseline = user's current best compressed 7-feature set
# enhanced = baseline + selected starter features
# ---------------------------------------------------------
baseline_features = [
    "diff_runs_for_avg_10",
    "diff_runs_against_avg_10",
    "diff_margin_avg_10",
    "diff_win_rate_10",
    "diff_rest_days",
    "home_rest_days",
    "away_rest_days",
]

starter_features = [
    "diff_starter_runs_allowed_10",
    "diff_starter_era_proxy_10",
    "diff_starter_win_rate_10",
    "diff_starter_margin_10",
    "diff_starter_run_support_10",
    "diff_pitcher_rest_days",
    "away_starter_prev_starts",
    "home_starter_prev_starts",
]

available_baseline = [c for c in baseline_features if c in df.columns]
available_starters = [c for c in starter_features if c in df.columns]

missing_baseline = [c for c in baseline_features if c not in df.columns]
missing_starters = [c for c in starter_features if c not in df.columns]

print("")
print("Baseline features found:")
print(available_baseline)
print("")
print("Missing baseline features:")
print(missing_baseline)

print("")
print("Starter features found:")
print(available_starters)
print("")
print("Missing starter features:")
print(missing_starters)

if len(available_baseline) < 5:
    raise ValueError(
        f"Too few baseline features found ({len(available_baseline)}). "
        "Check your mlb_with_starter_features columns."
    )

enhanced_features = available_baseline + available_starters

# ---------------------------------------------------------
# 3) choose split date safely
# preference:
# - preserve user's typical walk-forward style
# - if 2025 exists, split at 2025-01-01
# - otherwise use 80/20 chronological split
# ---------------------------------------------------------
df = df.sort_values([date_col]).reset_index(drop=True)

min_date = df[date_col].min()
max_date = df[date_col].max()

preferred_split = pd.Timestamp("2025-01-01")
if min_date < preferred_split < max_date:
    split_date = preferred_split
    split_method = "fixed_2025_01_01"
else:
    split_idx = int(len(df) * 0.80)
    split_date = df.iloc[split_idx][date_col]
    split_method = "chronological_80_20_fallback"

train_df = df[df[date_col] < split_date].copy()
test_df  = df[df[date_col] >= split_date].copy()

if len(train_df) == 0 or len(test_df) == 0:
    raise ValueError(
        f"Bad split. train={len(train_df)}, test={len(test_df)}, split_date={split_date}"
    )

print("")
print("=" * 80)
print("SPLIT SUMMARY")
print("=" * 80)
print(f"Split method    : {split_method}")
print(f"Split date      : {split_date}")
print(f"Train rows      : {len(train_df):,}")
print(f"Test rows       : {len(test_df):,}")
print(f"Train date min  : {train_df[date_col].min()}")
print(f"Train date max  : {train_df[date_col].max()}")
print(f"Test date min   : {test_df[date_col].min()}")
print(f"Test date max   : {test_df[date_col].max()}")

# ---------------------------------------------------------
# 4) helper to fit + evaluate one model
# ---------------------------------------------------------
def fit_and_score(train_df_, test_df_, features, target, label):
    X_train = train_df_[features].copy()
    y_train = train_df_[target].astype(int).copy()
    X_test  = test_df_[features].copy()
    y_test  = test_df_[target].astype(int).copy()

    print("")
    print("-" * 80)
    print(f"{label} Ã¢â‚¬â€ DATA CHECK")
    print("-" * 80)
    print(f"Feature count: {len(features)}")
    print(f"Train shape  : {X_train.shape}")
    print(f"Test shape   : {X_test.shape}")

    train_missing = X_train.isna().sum().sort_values(ascending=False)
    test_missing = X_test.isna().sum().sort_values(ascending=False)

    print("Top train missing counts:")
    print(train_missing.head(10))
    print("")
    print("Top test missing counts:")
    print(test_missing.head(10))

    base_model = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("logreg", LogisticRegression(
            max_iter=2000,
            C=1.0,
            solver="lbfgs",
            random_state=42
        )),
    ])

    calibrated_model = CalibratedClassifierCV(
        estimator=base_model,
        method="isotonic",
        cv=3
    )

    calibrated_model.fit(X_train, y_train)
    pred_prob = calibrated_model.predict_proba(X_test)[:, 1]
    pred_class = (pred_prob >= 0.5).astype(int)

    metrics = {
        "model_label": label,
        "n_features": len(features),
        "train_rows": len(X_train),
        "test_rows": len(X_test),
        "accuracy": accuracy_score(y_test, pred_class),
        "log_loss": log_loss(y_test, pred_prob),
        "brier": brier_score_loss(y_test, pred_prob),
        "avg_pred": float(np.mean(pred_prob)),
        "test_hit_rate": float(np.mean(y_test)),
    }

    pred_df = test_df_[["game_id", date_col]].copy() if "game_id" in test_df_.columns else test_df_[[date_col]].copy()
    pred_df[f"{label}_pred_prob"] = pred_prob
    pred_df[f"{label}_pred_class"] = pred_class

    return metrics, pred_df, calibrated_model

# ---------------------------------------------------------
# 5) fit baseline model
# ---------------------------------------------------------
baseline_metrics, baseline_preds, baseline_model = fit_and_score(
    train_df_=train_df,
    test_df_=test_df,
    features=available_baseline,
    target=target_col,
    label="baseline7"
)

# ---------------------------------------------------------
# 6) fit starter-enhanced model
# only run if we actually have starter features
# ---------------------------------------------------------
if len(available_starters) == 0:
    print("")
    print("No starter features available, so enhanced model cannot be run.")
    enhanced_metrics = None
    enhanced_preds = None
    enhanced_model = None
else:
    enhanced_metrics, enhanced_preds, enhanced_model = fit_and_score(
        train_df_=train_df,
        test_df_=test_df,
        features=enhanced_features,
        target=target_col,
        label="starter_enhanced"
    )

# ---------------------------------------------------------
# 7) summarize comparison
# ---------------------------------------------------------
rows = [baseline_metrics]
if enhanced_metrics is not None:
    rows.append(enhanced_metrics)

comparison = pd.DataFrame(rows)

print("")
print("=" * 80)
print("MODEL COMPARISON")
print("=" * 80)
display(comparison)

if enhanced_metrics is not None:
    delta_acc = enhanced_metrics["accuracy"] - baseline_metrics["accuracy"]
    delta_ll  = enhanced_metrics["log_loss"] - baseline_metrics["log_loss"]
    delta_br  = enhanced_metrics["brier"] - baseline_metrics["brier"]

    print("")
    print("DELTA (starter_enhanced - baseline7)")
    print(f"Accuracy change : {delta_acc:+.4f}")
    print(f"Log loss change : {delta_ll:+.4f}")
    print(f"Brier change    : {delta_br:+.4f}")

    if delta_acc > 0 and delta_ll < 0 and delta_br < 0:
        verdict = "Starter features improved the model across all main metrics."
    elif delta_acc > 0 or delta_ll < 0 or delta_br < 0:
        verdict = "Starter features show partial improvement. Worth inspecting further."
    else:
        verdict = "Starter features did not improve this version yet. We may need a smaller/cleaner starter set."

    print("")
    print(f"Verdict: {verdict}")

# ---------------------------------------------------------
# 8) optional coefficient inspection on non-calibrated logistic
# This helps us see if starter features are contributing signal.
# ---------------------------------------------------------
def fit_raw_logistic_for_coeffs(train_df_, features, target):
    X_train = train_df_[features].copy()
    y_train = train_df_[target].astype(int).copy()

    pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("logreg", LogisticRegression(
            max_iter=2000,
            C=1.0,
            solver="lbfgs",
            random_state=42
        )),
    ])
    pipe.fit(X_train, y_train)
    coefs = pipe.named_steps["logreg"].coef_[0]
    return pd.DataFrame({
        "feature": features,
        "coef": coefs,
        "abs_coef": np.abs(coefs),
    }).sort_values("abs_coef", ascending=False).reset_index(drop=True)

coef_baseline = fit_raw_logistic_for_coeffs(train_df, available_baseline, target_col)
print("")
print("=" * 80)
print("BASELINE COEFFICIENTS")
print("=" * 80)
display(coef_baseline)

if len(available_starters) > 0:
    coef_enhanced = fit_raw_logistic_for_coeffs(train_df, enhanced_features, target_col)
    print("")
    print("=" * 80)
    print("STARTER-ENHANCED COEFFICIENTS")
    print("=" * 80)
    display(coef_enhanced)

# ---------------------------------------------------------
# 9) save outputs + expose
# ---------------------------------------------------------
globals()["mlb_model_compare_table"] = comparison.copy()
globals()["mlb_baseline_test_preds"] = baseline_preds.copy()
globals()["mlb_baseline_model_calibrated"] = baseline_model

if enhanced_metrics is not None:
    globals()["mlb_starter_test_preds"] = enhanced_preds.copy()
    globals()["mlb_starter_model_calibrated"] = enhanced_model

print("")
print("Created in memory:")
print(" - mlb_model_compare_table")
print(" - mlb_baseline_test_preds")
print(" - mlb_baseline_model_calibrated")
if enhanced_metrics is not None:
    print(" - mlb_starter_test_preds")
    print(" - mlb_starter_model_calibrated")
print("Done.")

COMPARE BASE 7-FEATURE MODEL VS STARTER-ENHANCED MODEL
Input shape: (8230, 202)

Rows before clean: 8,230
Rows after clean : 8,230

Baseline features found:
['diff_runs_for_avg_10', 'diff_runs_against_avg_10', 'diff_margin_avg_10', 'diff_win_rate_10', 'diff_rest_days', 'home_rest_days', 'away_rest_days']

Missing baseline features:
[]

Starter features found:
['diff_starter_runs_allowed_10', 'diff_starter_era_proxy_10', 'diff_starter_win_rate_10', 'diff_starter_margin_10', 'diff_starter_run_support_10', 'diff_pitcher_rest_days', 'away_starter_prev_starts', 'home_starter_prev_starts']

Missing starter features:
[]



SPLIT SUMMARY
Split method    : fixed_2025_01_01
Split date      : 2025-01-01 00:00:00
Train rows      : 5,616
Test rows       : 2,614
Train date min  : 2023-03-02 00:00:00
Train date max  : 2024-10-30 00:00:00
Test date min   : 2025-03-01 00:00:00
Test date max   : 2025-11-01 00:00:00

--------------------------------------------------------------------------------
baseline7 Ã¢â‚¬â€ DATA CHECK
--------------------------------------------------------------------------------
Feature count: 7
Train shape  : (5616, 7)
Test shape   : (2614, 7)
Top train missing counts:
diff_runs_for_avg_10        0
diff_runs_against_avg_10    0
diff_margin_avg_10          0
diff_win_rate_10            0
diff_rest_days              0
home_rest_days              0
away_rest_days              0
dtype: int64

Top test missing counts:
diff_runs_for_avg_10        0
diff_runs_against_avg_10    0
diff_margin_avg_10          0
diff_win_rate_10            0
diff_rest_days              0
home_rest_days             


--------------------------------------------------------------------------------
starter_enhanced Ã¢â‚¬â€ DATA CHECK
--------------------------------------------------------------------------------
Feature count: 15
Train shape  : (5616, 15)
Test shape   : (2614, 15)
Top train missing counts:
diff_starter_era_proxy_10       1176
diff_starter_runs_allowed_10     510
diff_starter_win_rate_10         510
diff_starter_margin_10           510
diff_starter_run_support_10      510
diff_pitcher_rest_days           510
diff_runs_for_avg_10               0
diff_runs_against_avg_10           0
diff_margin_avg_10                 0
diff_win_rate_10                   0
dtype: int64

Top test missing counts:
diff_starter_era_proxy_10       282
diff_starter_runs_allowed_10    112
diff_starter_win_rate_10        112
diff_starter_margin_10          112
diff_starter_run_support_10     112
diff_pitcher_rest_days          112
diff_runs_for_avg_10              0
diff_runs_against_avg_10          0
diff_ma


MODEL COMPARISON


,model_label,n_features,train_rows,test_rows,accuracy,log_loss,brier,avg_pred,test_hit_rate
0,baseline7,7,5616,2614,0.546289,0.687775,0.247344,0.519086,0.545907
1,starter_enhanced,15,5616,2614,0.542846,0.687129,0.246979,0.523105,0.545907



DELTA (starter_enhanced - baseline7)
Accuracy change : -0.0034
Log loss change : -0.0006
Brier change    : -0.0004

Verdict: Starter features show partial improvement. Worth inspecting further.

BASELINE COEFFICIENTS


,feature,coef,abs_coef
0,diff_runs_for_avg_10,0.068019,0.068019
1,diff_margin_avg_10,0.050917,0.050917
2,diff_win_rate_10,0.043762,0.043762
3,away_rest_days,0.008110,0.008110
4,diff_runs_against_avg_10,-0.007694,0.007694
5,home_rest_days,0.005312,0.005312
6,diff_rest_days,-0.003641,0.003641



STARTER-ENHANCED COEFFICIENTS


,feature,coef,abs_coef
0,diff_starter_era_proxy_10,0.122845,0.122845
1,home_starter_prev_starts,0.075341,0.075341
2,away_starter_prev_starts,-0.069885,0.069885
3,diff_starter_win_rate_10,-0.068187,0.068187
4,diff_runs_for_avg_10,0.066535,0.066535
5,diff_margin_avg_10,0.044299,0.044299
6,diff_starter_runs_allowed_10,-0.044099,0.044099
7,diff_win_rate_10,0.027255,0.027255
8,diff_starter_margin_10,0.026765,0.026765
9,diff_rest_days,-0.010869,0.010869



Created in memory:
 - mlb_model_compare_table
 - mlb_baseline_test_preds
 - mlb_baseline_model_calibrated
 - mlb_starter_test_preds
 - mlb_starter_model_calibrated
Done.


In [76]:
# %%
# =========================================================
# CELL Ã¢â‚¬â€ TEST COMPRESSED STARTER FEATURE MODEL
# Goal:
# - compare:
#   1) baseline 7-feature model
#   2) full starter-enhanced model (already tested)
#   3) compressed starter model
# - determine whether a smaller starter block works better
# =========================================================

import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import accuracy_score, log_loss, brier_score_loss

print("=" * 80)
print("TEST COMPRESSED STARTER FEATURE MODEL")
print("=" * 80)

# ---------------------------------------------------------
# 0) validate input
# ---------------------------------------------------------
if "mlb_with_starter_features" not in globals() or not isinstance(mlb_with_starter_features, pd.DataFrame):
    raise ValueError("mlb_with_starter_features not found.")

df = mlb_with_starter_features.copy()
print(f"Input shape: {df.shape}")

# ---------------------------------------------------------
# 1) detect columns
# ---------------------------------------------------------
def find_first_col(df_, options):
    lower_map = {c.lower(): c for c in df_.columns}
    for opt in options:
        if opt.lower() in lower_map:
            return lower_map[opt.lower()]
    return None

date_col = find_first_col(df, ["date", "game_date", "game_datetime", "game_datetime_utc"])
target_col = find_first_col(df, ["home_win"])

if date_col is None:
    raise ValueError("Could not find date column.")
if target_col is None:
    raise ValueError("Could not find target column home_win.")

df[date_col] = pd.to_datetime(df[date_col], errors="coerce")
df[target_col] = pd.to_numeric(df[target_col], errors="coerce")

df = df[df[date_col].notna() & df[target_col].notna()].copy()
df[target_col] = df[target_col].astype(int)
df = df.sort_values(date_col).reset_index(drop=True)

print("")
print(f"Rows after clean: {len(df):,}")
print(f"Date range: {df[date_col].min()} -> {df[date_col].max()}")

# ---------------------------------------------------------
# 2) same split logic as prior cell
# ---------------------------------------------------------
preferred_split = pd.Timestamp("2025-01-01")
if df[date_col].min() < preferred_split < df[date_col].max():
    split_date = preferred_split
    split_method = "fixed_2025_01_01"
else:
    split_idx = int(len(df) * 0.80)
    split_date = df.iloc[split_idx][date_col]
    split_method = "chronological_80_20_fallback"

train_df = df[df[date_col] < split_date].copy()
test_df  = df[df[date_col] >= split_date].copy()

if len(train_df) == 0 or len(test_df) == 0:
    raise ValueError(f"Bad split. train={len(train_df)}, test={len(test_df)}")

print("")
print("=" * 80)
print("SPLIT SUMMARY")
print("=" * 80)
print(f"Split method: {split_method}")
print(f"Split date  : {split_date}")
print(f"Train rows  : {len(train_df):,}")
print(f"Test rows   : {len(test_df):,}")

# ---------------------------------------------------------
# 3) feature sets
# ---------------------------------------------------------
baseline_features = [
    "diff_runs_for_avg_10",
    "diff_runs_against_avg_10",
    "diff_margin_avg_10",
    "diff_win_rate_10",
    "diff_rest_days",
    "home_rest_days",
    "away_rest_days",
]

compressed_starter_features = [
    "diff_starter_runs_allowed_10",
    "diff_starter_win_rate_10",
    "diff_pitcher_rest_days",
]

available_baseline = [c for c in baseline_features if c in df.columns]
available_compressed = [c for c in compressed_starter_features if c in df.columns]

if len(available_baseline) < 5:
    raise ValueError(f"Too few baseline features found: {available_baseline}")

if len(available_compressed) == 0:
    raise ValueError("No compressed starter features found.")

compressed_model_features = available_baseline + available_compressed

print("")
print("Baseline features:")
print(available_baseline)

print("")
print("Compressed starter features:")
print(available_compressed)

print("")
print("Compressed model features:")
print(compressed_model_features)

# ---------------------------------------------------------
# 4) helper
# ---------------------------------------------------------
def fit_and_score(train_df_, test_df_, features, target, label):
    X_train = train_df_[features].copy()
    y_train = train_df_[target].astype(int).copy()
    X_test  = test_df_[features].copy()
    y_test  = test_df_[target].astype(int).copy()

    print("")
    print("-" * 80)
    print(f"{label} Ã¢â‚¬â€ DATA CHECK")
    print("-" * 80)
    print(f"Feature count: {len(features)}")
    print(f"Train shape  : {X_train.shape}")
    print(f"Test shape   : {X_test.shape}")

    print("Top train missing counts:")
    print(X_train.isna().sum().sort_values(ascending=False).head(10))
    print("")
    print("Top test missing counts:")
    print(X_test.isna().sum().sort_values(ascending=False).head(10))

    base_model = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("logreg", LogisticRegression(
            max_iter=2000,
            C=1.0,
            solver="lbfgs",
            random_state=42
        )),
    ])

    calibrated_model = CalibratedClassifierCV(
        estimator=base_model,
        method="isotonic",
        cv=3
    )

    calibrated_model.fit(X_train, y_train)
    pred_prob = calibrated_model.predict_proba(X_test)[:, 1]
    pred_class = (pred_prob >= 0.5).astype(int)

    metrics = {
        "model_label": label,
        "n_features": len(features),
        "train_rows": len(X_train),
        "test_rows": len(X_test),
        "accuracy": accuracy_score(y_test, pred_class),
        "log_loss": log_loss(y_test, pred_prob),
        "brier": brier_score_loss(y_test, pred_prob),
        "avg_pred": float(np.mean(pred_prob)),
        "test_hit_rate": float(np.mean(y_test)),
    }

    pred_df = test_df_[["game_id", date_col]].copy() if "game_id" in test_df_.columns else test_df_[[date_col]].copy()
    pred_df[f"{label}_pred_prob"] = pred_prob
    pred_df[f"{label}_pred_class"] = pred_class

    raw_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("logreg", LogisticRegression(
            max_iter=2000,
            C=1.0,
            solver="lbfgs",
            random_state=42
        )),
    ])
    raw_pipe.fit(X_train, y_train)
    coefs = raw_pipe.named_steps["logreg"].coef_[0]
    coef_df = pd.DataFrame({
        "feature": features,
        "coef": coefs,
        "abs_coef": np.abs(coefs),
    }).sort_values("abs_coef", ascending=False).reset_index(drop=True)

    return metrics, pred_df, calibrated_model, coef_df

# ---------------------------------------------------------
# 5) fit baseline again for side-by-side
# ---------------------------------------------------------
baseline_metrics, baseline_preds, baseline_model, baseline_coefs = fit_and_score(
    train_df, test_df, available_baseline, target_col, "baseline7_retest"
)

# ---------------------------------------------------------
# 6) fit compressed starter model
# ---------------------------------------------------------
compressed_metrics, compressed_preds, compressed_model, compressed_coefs = fit_and_score(
    train_df, test_df, compressed_model_features, target_col, "starter_compressed"
)

# ---------------------------------------------------------
# 7) comparison table
# ---------------------------------------------------------
comparison = pd.DataFrame([baseline_metrics, compressed_metrics])

print("")
print("=" * 80)
print("COMPRESSED MODEL COMPARISON")
print("=" * 80)
display(comparison)

delta_acc = compressed_metrics["accuracy"] - baseline_metrics["accuracy"]
delta_ll  = compressed_metrics["log_loss"] - baseline_metrics["log_loss"]
delta_br  = compressed_metrics["brier"] - baseline_metrics["brier"]

print("")
print("DELTA (starter_compressed - baseline7_retest)")
print(f"Accuracy change : {delta_acc:+.4f}")
print(f"Log loss change : {delta_ll:+.4f}")
print(f"Brier change    : {delta_br:+.4f}")

if delta_acc > 0 and delta_ll < 0 and delta_br < 0:
    verdict = "Compressed starter set is a clear improvement."
elif delta_acc > 0 or delta_ll < 0 or delta_br < 0:
    verdict = "Compressed starter set shows partial improvement."
else:
    verdict = "Compressed starter set did not help. Baseline remains better."

print("")
print(f"Verdict: {verdict}")

# ---------------------------------------------------------
# 8) coefficient inspection
# ---------------------------------------------------------
print("")
print("=" * 80)
print("BASELINE RETEST COEFFICIENTS")
print("=" * 80)
display(baseline_coefs)

print("")
print("=" * 80)
print("COMPRESSED STARTER COEFFICIENTS")
print("=" * 80)
display(compressed_coefs)

# ---------------------------------------------------------
# 9) save outputs
# ---------------------------------------------------------
globals()["mlb_compressed_starter_compare_table"] = comparison.copy()
globals()["mlb_compressed_starter_test_preds"] = compressed_preds.copy()
globals()["mlb_compressed_starter_model_calibrated"] = compressed_model
globals()["mlb_compressed_starter_coefs"] = compressed_coefs.copy()

print("")
print("Created in memory:")
print(" - mlb_compressed_starter_compare_table")
print(" - mlb_compressed_starter_test_preds")
print(" - mlb_compressed_starter_model_calibrated")
print(" - mlb_compressed_starter_coefs")
print("Done.")

TEST COMPRESSED STARTER FEATURE MODEL
Input shape: (8230, 202)



Rows after clean: 8,230
Date range: 2023-03-02 00:00:00 -> 2025-11-01 00:00:00

SPLIT SUMMARY
Split method: fixed_2025_01_01
Split date  : 2025-01-01 00:00:00
Train rows  : 5,616
Test rows   : 2,614

Baseline features:
['diff_runs_for_avg_10', 'diff_runs_against_avg_10', 'diff_margin_avg_10', 'diff_win_rate_10', 'diff_rest_days', 'home_rest_days', 'away_rest_days']

Compressed starter features:
['diff_starter_runs_allowed_10', 'diff_starter_win_rate_10', 'diff_pitcher_rest_days']

Compressed model features:
['diff_runs_for_avg_10', 'diff_runs_against_avg_10', 'diff_margin_avg_10', 'diff_win_rate_10', 'diff_rest_days', 'home_rest_days', 'away_rest_days', 'diff_starter_runs_allowed_10', 'diff_starter_win_rate_10', 'diff_pitcher_rest_days']

--------------------------------------------------------------------------------
baseline7_retest Ã¢â‚¬â€ DATA CHECK
--------------------------------------------------------------------------------
Feature count: 7
Train shape  : (5616, 7)
Test shap


--------------------------------------------------------------------------------
starter_compressed Ã¢â‚¬â€ DATA CHECK
--------------------------------------------------------------------------------
Feature count: 10
Train shape  : (5616, 10)
Test shape   : (2614, 10)
Top train missing counts:


diff_starter_runs_allowed_10    510
diff_starter_win_rate_10        510
diff_pitcher_rest_days          510
diff_runs_for_avg_10              0
diff_runs_against_avg_10          0
diff_margin_avg_10                0
diff_win_rate_10                  0
diff_rest_days                    0
home_rest_days                    0
away_rest_days                    0
dtype: int64

Top test missing counts:
diff_starter_runs_allowed_10    112
diff_starter_win_rate_10        112
diff_pitcher_rest_days          112
diff_runs_for_avg_10              0
diff_runs_against_avg_10          0
diff_margin_avg_10                0
diff_win_rate_10                  0
diff_rest_days                    0
home_rest_days                    0
away_rest_days                    0
dtype: int64



COMPRESSED MODEL COMPARISON


...


DELTA (starter_compressed - baseline7_retest)
Accuracy change : +0.0042
Log loss change : -0.0004
Brier change    : -0.0002

Verdict: Compressed starter set is a clear improvement.

BASELINE RETEST COEFFICIENTS


,feature,coef,abs_coef
0,diff_runs_for_avg_10,0.068019,0.068019
1,diff_margin_avg_10,0.050917,0.050917
2,diff_win_rate_10,0.043762,0.043762
3,away_rest_days,0.008110,0.008110
4,diff_runs_against_avg_10,-0.007694,0.007694
5,home_rest_days,0.005312,0.005312
6,diff_rest_days,-0.003641,0.003641



COMPRESSED STARTER COEFFICIENTS


,feature,coef,abs_coef
0,diff_starter_win_rate_10,-0.065348,0.065348
1,diff_runs_for_avg_10,0.064900,0.064900
2,diff_margin_avg_10,0.044484,0.044484
3,diff_starter_runs_allowed_10,0.036642,0.036642
4,diff_win_rate_10,0.032617,0.032617
5,diff_pitcher_rest_days,0.013742,0.013742
6,away_rest_days,0.008567,0.008567
7,diff_rest_days,-0.005820,0.005820
8,home_rest_days,0.004341,0.004341
9,diff_runs_against_avg_10,-0.001225,0.001225



Created in memory:
 - mlb_compressed_starter_compare_table
 - mlb_compressed_starter_test_preds
 - mlb_compressed_starter_model_calibrated
 - mlb_compressed_starter_coefs
Done.


In [77]:
# %%
# =========================================================
# CELL Ã¢â‚¬â€ TRAIN AND SAVE FINAL MLB WIN MODEL (COMPRESSED STARTER VERSION)
# Final production baseline:
#   7 base team features
# + 3 compressed starter features
# =========================================================

import json
import joblib
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV

print("=" * 80)
print("TRAIN AND SAVE FINAL MLB WIN MODEL (COMPRESSED STARTER VERSION)")
print("=" * 80)

# ---------------------------------------------------------
# 0) validate input
# ---------------------------------------------------------
if "mlb_with_starter_features" not in globals() or not isinstance(mlb_with_starter_features, pd.DataFrame):
    raise ValueError("mlb_with_starter_features not found.")

df = mlb_with_starter_features.copy()
print(f"Input shape: {df.shape}")

# ---------------------------------------------------------
# 1) detect core columns
# ---------------------------------------------------------
def find_first_col(df_, options):
    lower_map = {c.lower(): c for c in df_.columns}
    for opt in options:
        if opt.lower() in lower_map:
            return lower_map[opt.lower()]
    return None

date_col = find_first_col(df, ["date", "game_date", "game_datetime", "game_datetime_utc"])
target_col = find_first_col(df, ["home_win"])

if date_col is None:
    raise ValueError("Could not find date column.")
if target_col is None:
    raise ValueError("Could not find target column home_win.")

df[date_col] = pd.to_datetime(df[date_col], errors="coerce")
df[target_col] = pd.to_numeric(df[target_col], errors="coerce")

df = df[df[date_col].notna() & df[target_col].notna()].copy()
df[target_col] = df[target_col].astype(int)

print("")
print(f"Rows after clean: {len(df):,}")
print(f"Date range: {df[date_col].min()} -> {df[date_col].max()}")

# ---------------------------------------------------------
# 2) final production feature list
# ---------------------------------------------------------
final_features = [
    "diff_runs_for_avg_10",
    "diff_runs_against_avg_10",
    "diff_margin_avg_10",
    "diff_win_rate_10",
    "diff_rest_days",
    "home_rest_days",
    "away_rest_days",
    "diff_starter_runs_allowed_10",
    "diff_starter_win_rate_10",
    "diff_pitcher_rest_days",
]

missing_features = [c for c in final_features if c not in df.columns]
if missing_features:
    raise ValueError(f"Missing final production features: {missing_features}")

print("")
print("Final production features:")
for f in final_features:
    print(f" - {f}")

print("")
print("Missing counts for final features:")
print(df[final_features].isna().sum().sort_values(ascending=False))

# ---------------------------------------------------------
# 3) fit calibrated production model on all available data
# ---------------------------------------------------------
X = df[final_features].copy()
y = df[target_col].astype(int).copy()

base_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("logreg", LogisticRegression(
        max_iter=2000,
        C=1.0,
        solver="lbfgs",
        random_state=42
    )),
])

final_model = CalibratedClassifierCV(
    estimator=base_model,
    method="isotonic",
    cv=3
)

final_model.fit(X, y)

print("")
print("Model fit complete.")
print(f"Train rows: {len(X):,}")
print(f"Target mean (home win rate): {y.mean():.4f}")

# ---------------------------------------------------------
# 4) save artifacts
# ---------------------------------------------------------
model_dir = Path("models")
model_dir.mkdir(parents=True, exist_ok=True)

model_path = model_dir / "mlb_win_model_compressed_starters.joblib"
meta_path = model_dir / "mlb_win_model_compressed_starters_meta.json"

joblib.dump(final_model, model_path)

metadata = {
    "model_name": "mlb_win_model_compressed_starters",
    "model_type": "LogisticRegression + IsotonicCalibration",
    "target": target_col,
    "date_col": date_col,
    "n_rows_fit": int(len(df)),
    "date_min": str(df[date_col].min()),
    "date_max": str(df[date_col].max()),
    "features": final_features,
    "feature_count": len(final_features),
    "best_validation_result": {
        "accuracy": 0.550497,
        "log_loss": 0.687395,
        "brier": 0.247113,
        "comparison_baseline_accuracy": 0.546289,
        "comparison_baseline_log_loss": 0.687775,
        "comparison_baseline_brier": 0.247344,
    },
    "notes": [
        "Compressed starter feature set outperformed 7-feature baseline.",
        "Keep compressed starter block, not full starter block.",
        "Features use leakage-safe rolling history."
    ]
}

with open(meta_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

# ---------------------------------------------------------
# 5) expose in memory
# ---------------------------------------------------------
globals()["mlb_final_win_features"] = final_features
globals()["mlb_final_win_model"] = final_model
globals()["mlb_final_win_model_metadata"] = metadata

print("")
print("=" * 80)
print("SAVED FINAL MLB WIN MODEL")
print("=" * 80)
print(f"Model saved to   : {model_path}")
print(f"Metadata saved to: {meta_path}")
print("")
print("Created in memory:")
print(" - mlb_final_win_features")
print(" - mlb_final_win_model")
print(" - mlb_final_win_model_metadata")
print("")
print("Done.")

TRAIN AND SAVE FINAL MLB WIN MODEL (COMPRESSED STARTER VERSION)
Input shape: (8230, 202)

Rows after clean: 8,230
Date range: 2023-03-02 00:00:00 -> 2025-11-01 00:00:00

Final production features:
 - diff_runs_for_avg_10
 - diff_runs_against_avg_10
 - diff_margin_avg_10
 - diff_win_rate_10
 - diff_rest_days
 - home_rest_days
 - away_rest_days
 - diff_starter_runs_allowed_10
 - diff_starter_win_rate_10
 - diff_pitcher_rest_days

Missing counts for final features:
diff_starter_runs_allowed_10    622
diff_starter_win_rate_10        622
diff_pitcher_rest_days          622
diff_runs_for_avg_10              0
diff_runs_against_avg_10          0
diff_margin_avg_10                0
diff_win_rate_10                  0
diff_rest_days                    0
home_rest_days                    0
away_rest_days                    0
dtype: int64



Model fit complete.
Train rows: 8,230
Target mean (home win rate): 0.5272

SAVED FINAL MLB WIN MODEL
Model saved to   : models\mlb_win_model_compressed_starters.joblib
Metadata saved to: models\mlb_win_model_compressed_starters_meta.json

Created in memory:
 - mlb_final_win_features
 - mlb_final_win_model
 - mlb_final_win_model_metadata

Done.


In [78]:
# %%
# =========================================================
# CELL Ã¢â‚¬â€ SCORE DAILY MLB BOARD WITH FINAL WIN MODEL
# Uses:
# - mlb_final_win_model
# - mlb_final_win_features
# Goal:
# - score today's MLB board with calibrated home win probabilities
# - create projected winner / away win probability / edge-ready fields
# - keep pipeline safe and debug-heavy
# =========================================================

import numpy as np
import pandas as pd
import joblib
from pathlib import Path

print("=" * 80)
print("SCORE DAILY MLB BOARD WITH FINAL WIN MODEL")
print("=" * 80)

# ---------------------------------------------------------
# 0) validate/load model + feature list
# ---------------------------------------------------------
model = None
feature_list = None

if "mlb_final_win_model" in globals():
    model = mlb_final_win_model

if "mlb_final_win_features" in globals():
    feature_list = list(mlb_final_win_features)

# fallback load from disk if not in memory
if model is None:
    model_path = Path("models") / "mlb_win_model_compressed_starters.joblib"
    if model_path.exists():
        model = joblib.load(model_path)
        print(f"Loaded model from disk: {model_path}")

if feature_list is None:
    meta_path = Path("models") / "mlb_win_model_compressed_starters_meta.json"
    if meta_path.exists():
        import json
        with open(meta_path, "r", encoding="utf-8") as f:
            meta = json.load(f)
        feature_list = list(meta.get("features", []))
        print(f"Loaded feature list from disk: {meta_path}")

if model is None:
    raise ValueError("Final MLB win model not found in memory or on disk.")

if not feature_list:
    raise ValueError("Final MLB feature list not found in memory or on disk.")

print("")
print(f"Loaded feature count: {len(feature_list)}")
print("Features:")
for f in feature_list:
    print(f" - {f}")

# ---------------------------------------------------------
# 1) find daily board source safely
# ---------------------------------------------------------
candidate_names = [
    "board",
    "mlb_board",
    "projection_board",
    "daily_board",
    "today_board",
]

daily = None
source_used = None

for name in candidate_names:
    if name in globals() and isinstance(globals()[name], pd.DataFrame) and len(globals()[name]) > 0:
        daily = globals()[name].copy()
        source_used = name
        break

if daily is None:
    raise ValueError(
        f"No usable daily board dataframe found. Checked: {candidate_names}"
    )

print("")
print(f"Using board source: {source_used}")
print(f"Input board shape: {daily.shape}")
print(f"Input board columns: {list(daily.columns[:40])}{' ...' if daily.shape[1] > 40 else ''}")

# ---------------------------------------------------------
# 2) safe helper for column detection
# ---------------------------------------------------------
def find_first_col(df, options):
    lower_map = {c.lower(): c for c in df.columns}
    for opt in options:
        if opt.lower() in lower_map:
            return lower_map[opt.lower()]
    return None

game_id_col = find_first_col(daily, ["game_id", "gamepk", "game_pk", "pk"])
date_col = find_first_col(daily, ["date", "game_date", "game_datetime", "game_datetime_utc"])
away_col = find_first_col(daily, ["away_team", "away", "visitor_team", "away_name"])
home_col = find_first_col(daily, ["home_team", "home", "home_name"])

print("")
print("Detected board columns:")
print(f" - game_id_col: {game_id_col}")
print(f" - date_col   : {date_col}")
print(f" - away_col   : {away_col}")
print(f" - home_col   : {home_col}")

if away_col is None or home_col is None:
    raise ValueError("Could not find away/home team columns in the daily board.")

# ---------------------------------------------------------
# 3) verify features exist, create safe missing placeholders if needed
# ---------------------------------------------------------
score_df = daily.copy()

missing_features = [c for c in feature_list if c not in score_df.columns]
present_features = [c for c in feature_list if c in score_df.columns]

print("")
print(f"Present feature count: {len(present_features)}")
print(f"Missing feature count: {len(missing_features)}")

if missing_features:
    print("Missing features that will be created as NaN placeholders:")
    for c in missing_features:
        print(f" - {c}")
        score_df[c] = np.nan

# force numeric on all model features
for c in feature_list:
    score_df[c] = pd.to_numeric(score_df[c], errors="coerce")

print("")
print("Missing counts across model features:")
print(score_df[feature_list].isna().sum().sort_values(ascending=False))

# ---------------------------------------------------------
# 4) score board
# ---------------------------------------------------------
X_score = score_df[feature_list].copy()

pred_home_win = model.predict_proba(X_score)[:, 1]
pred_away_win = 1.0 - pred_home_win

score_df["pred_home_win_prob"] = pred_home_win
score_df["pred_away_win_prob"] = pred_away_win

score_df["projected_winner"] = np.where(
    score_df["pred_home_win_prob"] >= 0.5,
    score_df[home_col].astype(str),
    score_df[away_col].astype(str)
)

score_df["projected_win_prob"] = np.where(
    score_df["pred_home_win_prob"] >= 0.5,
    score_df["pred_home_win_prob"],
    score_df["pred_away_win_prob"]
)

score_df["projected_loser"] = np.where(
    score_df["pred_home_win_prob"] >= 0.5,
    score_df[away_col].astype(str),
    score_df[home_col].astype(str)
)

# margin-style confidence proxy
score_df["win_prob_edge_from_50"] = (score_df["projected_win_prob"] - 0.50).abs()
score_df["confidence"] = score_df["projected_win_prob"]

# convenience side labels
score_df["model_side"] = np.where(
    score_df["pred_home_win_prob"] >= 0.5,
    "home",
    "away"
)

# ---------------------------------------------------------
# 5) optional readable starter fields if present
# ---------------------------------------------------------
for c in ["away_pitcher_name", "home_pitcher_name"]:
    if c not in score_df.columns:
        score_df[c] = np.nan

score_df["pitching_matchup"] = np.where(
    score_df["away_pitcher_name"].notna() | score_df["home_pitcher_name"].notna(),
    score_df["away_pitcher_name"].fillna("TBD").astype(str)
    + " vs "
    + score_df["home_pitcher_name"].fillna("TBD").astype(str),
    None
)

# ---------------------------------------------------------
# 6) optional market-ready helper fields
# these do not assume market prices exist yet
# ---------------------------------------------------------
score_df["fair_home_price"] = score_df["pred_home_win_prob"]
score_df["fair_away_price"] = score_df["pred_away_win_prob"]

# if an entry price column already exists, create generic edge helpers
entry_home_candidates = [
    "home_price", "home_yes_price", "home_ml_price", "market_home_prob", "p_market_home"
]
entry_away_candidates = [
    "away_price", "away_yes_price", "away_ml_price", "market_away_prob", "p_market_away"
]

home_price_col = find_first_col(score_df, entry_home_candidates)
away_price_col = find_first_col(score_df, entry_away_candidates)

if home_price_col is not None:
    score_df[home_price_col] = pd.to_numeric(score_df[home_price_col], errors="coerce")
    score_df["edge_home"] = score_df["pred_home_win_prob"] - score_df[home_price_col]

if away_price_col is not None:
    score_df[away_price_col] = pd.to_numeric(score_df[away_price_col], errors="coerce")
    score_df["edge_away"] = score_df["pred_away_win_prob"] - score_df[away_price_col]

# ---------------------------------------------------------
# 7) sort scored board
# ---------------------------------------------------------
sort_cols = []
if "projected_win_prob" in score_df.columns:
    sort_cols.append("projected_win_prob")
if game_id_col is not None and game_id_col in score_df.columns:
    sort_cols.append(game_id_col)

if sort_cols:
    score_df = score_df.sort_values(sort_cols, ascending=[False] * len(sort_cols)).reset_index(drop=True)

# ---------------------------------------------------------
# 8) debug summary
# ---------------------------------------------------------
print("")
print("=" * 80)
print("SCORING SUMMARY")
print("=" * 80)
print(f"Scored board shape: {score_df.shape}")

summary = pd.Series({
    "rows_scored": int(len(score_df)),
    "avg_home_win_prob": float(score_df["pred_home_win_prob"].mean()),
    "avg_away_win_prob": float(score_df["pred_away_win_prob"].mean()),
    "home_favorites_count": int((score_df["pred_home_win_prob"] >= 0.5).sum()),
    "away_favorites_count": int((score_df["pred_home_win_prob"] < 0.5).sum()),
    "strong_55_plus_count": int((score_df["projected_win_prob"] >= 0.55).sum()),
    "strong_57_plus_count": int((score_df["projected_win_prob"] >= 0.57).sum()),
    "strong_59_plus_count": int((score_df["projected_win_prob"] >= 0.59).sum()),
})
print(summary)

print("")
print("Top scored games:")
show_cols = [c for c in [
    game_id_col if game_id_col in score_df.columns else None,
    date_col if date_col in score_df.columns else None,
    away_col,
    home_col,
    "away_pitcher_name",
    "home_pitcher_name",
    "pred_home_win_prob",
    "pred_away_win_prob",
    "projected_winner",
    "projected_win_prob",
    "model_side",
    "confidence",
    "win_prob_edge_from_50",
] if c is not None and c in score_df.columns]

display(score_df[show_cols].head(15))

# ---------------------------------------------------------
# 9) expose standard downstream names
# ---------------------------------------------------------
scored_mlb_board = score_df.copy()
globals()["scored_mlb_board"] = scored_mlb_board
globals()["mlb_scored_board"] = scored_mlb_board

print("")
print("Created in memory as:")
print(" - scored_mlb_board")
print(" - mlb_scored_board")

# optional alias if you want later cells to use a familiar name
if "projection_board" not in globals():
    globals()["projection_board"] = scored_mlb_board.copy()
    print(" - projection_board (alias created)")

print("")
print("Done.")

SCORE DAILY MLB BOARD WITH FINAL WIN MODEL

Loaded feature count: 10
Features:
 - diff_runs_for_avg_10
 - diff_runs_against_avg_10
 - diff_margin_avg_10
 - diff_win_rate_10
 - diff_rest_days
 - home_rest_days
 - away_rest_days
 - diff_starter_runs_allowed_10
 - diff_starter_win_rate_10
 - diff_pitcher_rest_days

Using board source: board
Input board shape: (1, 95)
Input board columns: ['game_id', 'game_datetime_utc', 'away_team', 'home_team', 'away_team_id', 'home_team_id', 'venue_name', 'status', 'game_type', 'series_game_number', 'games_in_series', 'series_description', 'away_date', 'away_runs_for', 'away_runs_against', 'away_win', 'away_park_factor', 'away_opponent', 'away_home_away', 'away_opp_def_rating', 'away_adj_runs_for', 'away_opp_off_rating', 'away_adj_runs_against', 'away_runs_for_avg_10', 'away_runs_against_avg_10', 'away_margin_avg_10', 'away_win_rate_10', 'away_runs_for_avg_5', 'away_runs_against_avg_5', 'away_margin_avg_5', 'away_win_rate_5', 'away_runs_for_avg_10_home'

,game_id,game_datetime_utc,away_team,home_team,away_pitcher_name,home_pitcher_name,pred_home_win_prob,pred_away_win_prob,projected_winner,projected_win_prob,model_side,confidence,win_prob_edge_from_50
0,849831,2026-10-11 00:00:00+00:00,Chicago White Sox,Cleveland Guardians,NaN,NaN,0.550454,0.449546,Cleveland Guardians,0.550454,home,0.550454,0.050454



Created in memory as:
 - scored_mlb_board
 - mlb_scored_board

Done.


In [79]:
# %%
# =========================================================
# CELL Ã¢â‚¬â€ ATTACH TODAY'S STARTER FEATURES TO DAILY BOARD
# CUSTOM FIX FOR YOUR starter_board COLUMN NAMES
# =========================================================

import numpy as np
import pandas as pd

print("=" * 80)
print("ATTACH TODAY'S STARTER FEATURES TO DAILY BOARD Ã¢â‚¬â€ CUSTOM FIX")
print("=" * 80)

# ---------------------------------------------------------
# 0) validate inputs
# ---------------------------------------------------------
if "board" not in globals() or not isinstance(board, pd.DataFrame):
    raise ValueError("board not found.")

if "starter_board" not in globals() or not isinstance(starter_board, pd.DataFrame):
    raise ValueError("starter_board not found.")

if "mlb_pitcher_games_features" not in globals() or not isinstance(mlb_pitcher_games_features, pd.DataFrame):
    raise ValueError("mlb_pitcher_games_features not found.")

daily = board.copy()
sb = starter_board.copy()
pgf = mlb_pitcher_games_features.copy()

print(f"board shape                 : {daily.shape}")
print(f"starter_board shape         : {sb.shape}")
print(f"mlb_pitcher_games_features  : {pgf.shape}")

# ---------------------------------------------------------
# 1) helper functions
# ---------------------------------------------------------
def norm_name(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip().lower()
    s = " ".join(s.split())
    return s if s != "" else np.nan

def force_object_key(s):
    s = s.copy()
    s = s.where(pd.notna(s), np.nan)
    return s.astype("object")

# ---------------------------------------------------------
# 2) clean merge key
# ---------------------------------------------------------
if "game_id" not in daily.columns or "game_id" not in sb.columns:
    raise ValueError("Both board and starter_board need game_id.")

daily["game_id"] = pd.to_numeric(daily["game_id"], errors="coerce")
sb["game_id"] = pd.to_numeric(sb["game_id"], errors="coerce")

daily = daily[daily["game_id"].notna()].copy()
sb = sb[sb["game_id"].notna()].copy()

daily["game_id"] = daily["game_id"].astype("int64")
sb["game_id"] = sb["game_id"].astype("int64")

# ---------------------------------------------------------
# 3) pull actual starter columns from starter_board
# ---------------------------------------------------------
required_sb_cols = [
    "game_id",
    "away_pitcher_final_name",
    "home_pitcher_final_name",
    "away_starting_pitcher_id",
    "home_starting_pitcher_id",
]

missing_sb = [c for c in required_sb_cols if c not in sb.columns]
if missing_sb:
    raise ValueError(f"starter_board missing required columns: {missing_sb}")

starter_keep = sb[required_sb_cols].copy()

starter_keep = starter_keep.rename(columns={
    "away_pitcher_final_name": "away_pitcher_name",
    "home_pitcher_final_name": "home_pitcher_name",
    "away_starting_pitcher_id": "away_pitcher_id",
    "home_starting_pitcher_id": "home_pitcher_id",
})

starter_keep["away_pitcher_id"] = pd.to_numeric(starter_keep["away_pitcher_id"], errors="coerce")
starter_keep["home_pitcher_id"] = pd.to_numeric(starter_keep["home_pitcher_id"], errors="coerce")

starter_keep = starter_keep.drop_duplicates(subset=["game_id"], keep="last").reset_index(drop=True)

print("")
print("starter_board usable pitcher coverage:")
print(pd.Series({
    "rows_total": int(len(starter_keep)),
    "rows_with_away_pitcher_id": int(starter_keep["away_pitcher_id"].notna().sum()),
    "rows_with_home_pitcher_id": int(starter_keep["home_pitcher_id"].notna().sum()),
    "rows_with_away_pitcher_name": int(starter_keep["away_pitcher_name"].notna().sum()),
    "rows_with_home_pitcher_name": int(starter_keep["home_pitcher_name"].notna().sum()),
}))

# ---------------------------------------------------------
# 4) merge starter info into board
# ---------------------------------------------------------
# Remove stale columns first so we don't keep old broken versions
drop_if_exists = [
    "away_pitcher_id", "home_pitcher_id",
    "away_pitcher_name", "home_pitcher_name",
    "away_pitcher_name_norm", "home_pitcher_name_norm",
    "away_pitcher_name_hist", "home_pitcher_name_hist",
    "away_starter_runs_allowed_10", "home_starter_runs_allowed_10",
    "away_starter_win_rate_10", "home_starter_win_rate_10",
    "away_pitcher_rest_days", "home_pitcher_rest_days",
    "away_pitcher_name_hist_by_name", "home_pitcher_name_hist_by_name",
    "away_starter_runs_allowed_10_by_name", "home_starter_runs_allowed_10_by_name",
    "away_starter_win_rate_10_by_name", "home_starter_win_rate_10_by_name",
    "away_pitcher_rest_days_by_name", "home_pitcher_rest_days_by_name",
    "diff_starter_runs_allowed_10", "diff_starter_win_rate_10", "diff_pitcher_rest_days",
]
for c in drop_if_exists:
    if c in daily.columns:
        daily = daily.drop(columns=c)

daily = daily.merge(
    starter_keep,
    on="game_id",
    how="left",
    validate="one_to_one"
)

daily["away_pitcher_name_norm"] = force_object_key(daily["away_pitcher_name"].apply(norm_name))
daily["home_pitcher_name_norm"] = force_object_key(daily["home_pitcher_name"].apply(norm_name))

# ---------------------------------------------------------
# 5) latest pitcher snapshots from historical feature table
# ---------------------------------------------------------
needed_pgf_cols = [
    "pitcher_id",
    "pitcher_name",
    "date",
    "starter_runs_allowed_10",
    "starter_win_rate_10",
    "pitcher_rest_days",
]
missing_pgf = [c for c in needed_pgf_cols if c not in pgf.columns]
if missing_pgf:
    raise ValueError(f"mlb_pitcher_games_features missing required columns: {missing_pgf}")

pgf["pitcher_id"] = pd.to_numeric(pgf["pitcher_id"], errors="coerce")
pgf["date"] = pd.to_datetime(pgf["date"], errors="coerce")
pgf["pitcher_name_norm"] = force_object_key(pgf["pitcher_name"].apply(norm_name))

latest_by_id = (
    pgf[pgf["pitcher_id"].notna()]
    .sort_values(["pitcher_id", "date", "game_id"])
    .groupby("pitcher_id", as_index=False)
    .tail(1)
    .copy()
)

latest_by_name = (
    pgf[pgf["pitcher_name_norm"].notna()]
    .sort_values(["pitcher_name_norm", "date", "game_id"])
    .groupby("pitcher_name_norm", as_index=False)
    .tail(1)
    .copy()
)

latest_by_name = latest_by_name.drop_duplicates(subset=["pitcher_name_norm"], keep="last").copy()

print("")
print(f"Latest snapshots by id   : {latest_by_id.shape}")
print(f"Latest snapshots by name : {latest_by_name.shape}")

# ---------------------------------------------------------
# 6) merge by pitcher_id first
# ---------------------------------------------------------
away_id_merge = latest_by_id[[
    "pitcher_id",
    "pitcher_name",
    "starter_runs_allowed_10",
    "starter_win_rate_10",
    "pitcher_rest_days",
]].rename(columns={
    "pitcher_id": "away_pitcher_id",
    "pitcher_name": "away_pitcher_name_hist",
    "starter_runs_allowed_10": "away_starter_runs_allowed_10",
    "starter_win_rate_10": "away_starter_win_rate_10",
    "pitcher_rest_days": "away_pitcher_rest_days",
}).drop_duplicates(subset=["away_pitcher_id"], keep="last")

home_id_merge = latest_by_id[[
    "pitcher_id",
    "pitcher_name",
    "starter_runs_allowed_10",
    "starter_win_rate_10",
    "pitcher_rest_days",
]].rename(columns={
    "pitcher_id": "home_pitcher_id",
    "pitcher_name": "home_pitcher_name_hist",
    "starter_runs_allowed_10": "home_starter_runs_allowed_10",
    "starter_win_rate_10": "home_starter_win_rate_10",
    "pitcher_rest_days": "home_pitcher_rest_days",
}).drop_duplicates(subset=["home_pitcher_id"], keep="last")

daily = daily.merge(away_id_merge, on="away_pitcher_id", how="left")
daily = daily.merge(home_id_merge, on="home_pitcher_id", how="left")

# ---------------------------------------------------------
# 7) fallback by name if needed
# ---------------------------------------------------------
away_name_merge = latest_by_name[[
    "pitcher_name_norm",
    "pitcher_name",
    "starter_runs_allowed_10",
    "starter_win_rate_10",
    "pitcher_rest_days",
]].rename(columns={
    "pitcher_name_norm": "away_pitcher_name_norm",
    "pitcher_name": "away_pitcher_name_hist_by_name",
    "starter_runs_allowed_10": "away_starter_runs_allowed_10_by_name",
    "starter_win_rate_10": "away_starter_win_rate_10_by_name",
    "pitcher_rest_days": "away_pitcher_rest_days_by_name",
}).drop_duplicates(subset=["away_pitcher_name_norm"], keep="last")

home_name_merge = latest_by_name[[
    "pitcher_name_norm",
    "pitcher_name",
    "starter_runs_allowed_10",
    "starter_win_rate_10",
    "pitcher_rest_days",
]].rename(columns={
    "pitcher_name_norm": "home_pitcher_name_norm",
    "pitcher_name": "home_pitcher_name_hist_by_name",
    "starter_runs_allowed_10": "home_starter_runs_allowed_10_by_name",
    "starter_win_rate_10": "home_starter_win_rate_10_by_name",
    "pitcher_rest_days": "home_pitcher_rest_days_by_name",
}).drop_duplicates(subset=["home_pitcher_name_norm"], keep="last")

away_name_merge["away_pitcher_name_norm"] = force_object_key(away_name_merge["away_pitcher_name_norm"])
home_name_merge["home_pitcher_name_norm"] = force_object_key(home_name_merge["home_pitcher_name_norm"])

daily = daily.merge(away_name_merge, on="away_pitcher_name_norm", how="left")
daily = daily.merge(home_name_merge, on="home_pitcher_name_norm", how="left")

fill_pairs = [
    ("away_starter_runs_allowed_10", "away_starter_runs_allowed_10_by_name"),
    ("away_starter_win_rate_10", "away_starter_win_rate_10_by_name"),
    ("away_pitcher_rest_days", "away_pitcher_rest_days_by_name"),
    ("home_starter_runs_allowed_10", "home_starter_runs_allowed_10_by_name"),
    ("home_starter_win_rate_10", "home_starter_win_rate_10_by_name"),
    ("home_pitcher_rest_days", "home_pitcher_rest_days_by_name"),
]

for main_col, fallback_col in fill_pairs:
    if main_col not in daily.columns:
        daily[main_col] = np.nan
    if fallback_col in daily.columns:
        daily[main_col] = daily[main_col].where(daily[main_col].notna(), daily[fallback_col])

# ---------------------------------------------------------
# 8) create final compressed starter inputs
# ---------------------------------------------------------
for c in [
    "away_starter_runs_allowed_10",
    "home_starter_runs_allowed_10",
    "away_starter_win_rate_10",
    "home_starter_win_rate_10",
    "away_pitcher_rest_days",
    "home_pitcher_rest_days",
]:
    if c not in daily.columns:
        daily[c] = np.nan
    daily[c] = pd.to_numeric(daily[c], errors="coerce")

daily["diff_starter_runs_allowed_10"] = daily["away_starter_runs_allowed_10"] - daily["home_starter_runs_allowed_10"]
daily["diff_starter_win_rate_10"] = daily["away_starter_win_rate_10"] - daily["home_starter_win_rate_10"]
daily["diff_pitcher_rest_days"] = daily["away_pitcher_rest_days"] - daily["home_pitcher_rest_days"]

# ---------------------------------------------------------
# 9) debug summary
# ---------------------------------------------------------
print("")
print("=" * 80)
print("DAILY STARTER FEATURE ATTACH SUMMARY")
print("=" * 80)
summary = pd.Series({
    "rows_total": int(len(daily)),
    "rows_with_away_pitcher_id": int(daily["away_pitcher_id"].notna().sum()),
    "rows_with_home_pitcher_id": int(daily["home_pitcher_id"].notna().sum()),
    "rows_with_away_pitcher_name": int(daily["away_pitcher_name"].notna().sum()),
    "rows_with_home_pitcher_name": int(daily["home_pitcher_name"].notna().sum()),
    "rows_non_null_away_runs_allowed_10": int(daily["away_starter_runs_allowed_10"].notna().sum()),
    "rows_non_null_home_runs_allowed_10": int(daily["home_starter_runs_allowed_10"].notna().sum()),
    "rows_non_null_diff_runs_allowed_10": int(daily["diff_starter_runs_allowed_10"].notna().sum()),
    "rows_non_null_diff_win_rate_10": int(daily["diff_starter_win_rate_10"].notna().sum()),
    "rows_non_null_diff_pitcher_rest_days": int(daily["diff_pitcher_rest_days"].notna().sum()),
})
print(summary)

print("")
print("Missing counts for final starter model inputs:")
print(daily[[
    "diff_starter_runs_allowed_10",
    "diff_starter_win_rate_10",
    "diff_pitcher_rest_days",
]].isna().sum())

print("")
print("Sample enriched rows:")
show_cols = [c for c in [
    "game_id", "game_datetime_utc", "away_team", "home_team",
    "away_pitcher_id", "home_pitcher_id",
    "away_pitcher_name", "home_pitcher_name",
    "away_starter_runs_allowed_10", "home_starter_runs_allowed_10",
    "diff_starter_runs_allowed_10",
    "away_starter_win_rate_10", "home_starter_win_rate_10",
    "diff_starter_win_rate_10",
    "away_pitcher_rest_days", "home_pitcher_rest_days",
    "diff_pitcher_rest_days",
] if c in daily.columns]
display(daily[show_cols].head(15))

# ---------------------------------------------------------
# 10) save back to memory
# ---------------------------------------------------------
globals()["board"] = daily.copy()
globals()["mlb_board"] = daily.copy()
globals()["projection_board"] = daily.copy()

print("")
print("Updated in memory as:")
print(" - board")
print(" - mlb_board")
print(" - projection_board")
print("")
print("Now rerun the scoring cell.")

ATTACH TODAY'S STARTER FEATURES TO DAILY BOARD Ã¢â‚¬â€ CUSTOM FIX
board shape                 : (1, 95)
starter_board shape         : (1, 100)
mlb_pitcher_games_features  : (16460, 48)

starter_board usable pitcher coverage:
rows_total                     1
rows_with_away_pitcher_id      0
rows_with_home_pitcher_id      0
rows_with_away_pitcher_name    1
rows_with_home_pitcher_name    1
dtype: int64



Latest snapshots by id   : (740, 49)
Latest snapshots by name : (738, 49)



DAILY STARTER FEATURE ATTACH SUMMARY
rows_total                              1
rows_with_away_pitcher_id               0
rows_with_home_pitcher_id               0
rows_with_away_pitcher_name             1
rows_with_home_pitcher_name             1
rows_non_null_away_runs_allowed_10      1
rows_non_null_home_runs_allowed_10      1
rows_non_null_diff_runs_allowed_10      1
rows_non_null_diff_win_rate_10          1
rows_non_null_diff_pitcher_rest_days    1
dtype: int64

Missing counts for final starter model inputs:
diff_starter_runs_allowed_10    0
diff_starter_win_rate_10        0
diff_pitcher_rest_days          0
dtype: int64

Sample enriched rows:


,game_id,game_datetime_utc,away_team,home_team,away_pitcher_id,home_pitcher_id,away_pitcher_name,home_pitcher_name,away_starter_runs_allowed_10,home_starter_runs_allowed_10,diff_starter_runs_allowed_10,away_starter_win_rate_10,home_starter_win_rate_10,diff_starter_win_rate_10,away_pitcher_rest_days,home_pitcher_rest_days,diff_pitcher_rest_days
0,849831,2026-10-11 00:00:00+00:00,Chicago White Sox,Cleveland Guardians,NaN,NaN,Sean Burke,Gavin Williams,5.4,3.0,2.4,0.4,0.8,-0.4,11.0,7.0,4.0



Updated in memory as:
 - board
 - mlb_board
 - projection_board

Now rerun the scoring cell.


In [80]:
# %%
# =========================================================
# CELL Ã¢â‚¬â€ PULL TODAY'S STARTERS DIRECTLY FROM MLB API
# AND ATTACH COMPRESSED STARTER FEATURES TO BOARD
# =========================================================

import time
import requests
import numpy as np
import pandas as pd

print("=" * 80)
print("PULL TODAY'S STARTERS DIRECTLY FROM MLB API")
print("=" * 80)

# ---------------------------------------------------------
# 0) validate inputs
# ---------------------------------------------------------
if "board" not in globals() or not isinstance(board, pd.DataFrame):
    raise ValueError("board not found.")

if "mlb_pitcher_games_features" not in globals() or not isinstance(mlb_pitcher_games_features, pd.DataFrame):
    raise ValueError("mlb_pitcher_games_features not found.")

daily = board.copy()
pgf = mlb_pitcher_games_features.copy()

if "game_id" not in daily.columns:
    raise ValueError("board must contain game_id.")

daily["game_id"] = pd.to_numeric(daily["game_id"], errors="coerce")
daily = daily[daily["game_id"].notna()].copy()
daily["game_id"] = daily["game_id"].astype("int64")

print(f"Board rows: {len(daily)}")
print(f"Unique game_ids: {daily['game_id'].nunique()}")

# ---------------------------------------------------------
# 1) MLB API helpers
# ---------------------------------------------------------
SESSION = requests.Session()
SESSION.headers.update({"User-Agent": "Mozilla/5.0"})

BASE_URL = "https://statsapi.mlb.com/api/v1.1/game/{}/feed/live"

def safe_get_json(url, timeout=25, max_retries=3):
    last_err = None
    for attempt in range(1, max_retries + 1):
        try:
            r = SESSION.get(url, timeout=timeout)
            r.raise_for_status()
            return r.json()
        except Exception as e:
            last_err = e
            if attempt < max_retries:
                time.sleep(0.5 * attempt)
    return {"_fetch_error": str(last_err)}

def get_player_name(players_lookup, pid):
    try:
        return (players_lookup.get(f"ID{int(pid)}") or {}).get("person", {}).get("fullName", np.nan)
    except Exception:
        return np.nan

def extract_probables(feed):
    away_id = np.nan
    away_name = np.nan
    home_id = np.nan
    home_name = np.nan

    try:
        probable = ((feed.get("gameData") or {}).get("probablePitchers") or {})
        away = probable.get("away") or {}
        home = probable.get("home") or {}

        away_id = pd.to_numeric(away.get("id"), errors="coerce")
        away_name = away.get("fullName", np.nan)

        home_id = pd.to_numeric(home.get("id"), errors="coerce")
        home_name = home.get("fullName", np.nan)
    except Exception:
        pass

    return away_id, away_name, home_id, home_name

def extract_boxscore_first_pitchers(feed):
    away_id = np.nan
    away_name = np.nan
    home_id = np.nan
    home_name = np.nan

    try:
        box = ((feed.get("liveData") or {}).get("boxscore") or {})
        teams = box.get("teams") or {}
        players_lookup = box.get("players") or {}

        away_pitchers = ((teams.get("away") or {}).get("pitchers")) or []
        home_pitchers = ((teams.get("home") or {}).get("pitchers")) or []

        if len(away_pitchers):
            away_id = pd.to_numeric(away_pitchers[0], errors="coerce")
            away_name = get_player_name(players_lookup, away_id)

        if len(home_pitchers):
            home_id = pd.to_numeric(home_pitchers[0], errors="coerce")
            home_name = get_player_name(players_lookup, home_id)
    except Exception:
        pass

    return away_id, away_name, home_id, home_name

def fetch_today_starters(game_id):
    out = {
        "game_id": int(game_id),
        "away_pitcher_id": np.nan,
        "home_pitcher_id": np.nan,
        "away_pitcher_name": np.nan,
        "home_pitcher_name": np.nan,
        "starter_pull_source": "missing",
        "starter_pull_error": np.nan,
    }

    feed = safe_get_json(BASE_URL.format(int(game_id)))

    if "_fetch_error" in feed:
        out["starter_pull_error"] = feed["_fetch_error"]
        return out

    # 1) probable pitchers
    away_id, away_name, home_id, home_name = extract_probables(feed)
    if pd.notna(away_id) or pd.notna(home_id) or pd.notna(away_name) or pd.notna(home_name):
        out["away_pitcher_id"] = away_id
        out["home_pitcher_id"] = home_id
        out["away_pitcher_name"] = away_name
        out["home_pitcher_name"] = home_name
        out["starter_pull_source"] = "probablePitchers"
        return out

    # 2) fallback to first listed pitchers in boxscore
    away_id, away_name, home_id, home_name = extract_boxscore_first_pitchers(feed)
    if pd.notna(away_id) or pd.notna(home_id) or pd.notna(away_name) or pd.notna(home_name):
        out["away_pitcher_id"] = away_id
        out["home_pitcher_id"] = home_id
        out["away_pitcher_name"] = away_name
        out["home_pitcher_name"] = home_name
        out["starter_pull_source"] = "boxscore_first_pitcher"
        return out

    return out

# ---------------------------------------------------------
# 2) pull today's starters
# ---------------------------------------------------------
starter_rows = []

print("")
print("Pulling starters from MLB API...")

game_ids = daily["game_id"].dropna().astype("int64").unique().tolist()

for i, gid in enumerate(game_ids, start=1):
    row = fetch_today_starters(gid)
    starter_rows.append(row)

    if i <= 5 or i == len(game_ids):
        found_any = sum(
            int(
                pd.notna(r.get("away_pitcher_id")) or pd.notna(r.get("home_pitcher_id")) or
                pd.notna(r.get("away_pitcher_name")) or pd.notna(r.get("home_pitcher_name"))
            )
            for r in starter_rows
        )
        print(f"[{i}/{len(game_ids)}] found starters for {found_any} games")

    time.sleep(0.03)

today_starters_api = pd.DataFrame(starter_rows)

print("")
print("Starter API pull summary:")
print(pd.Series({
    "rows_total": int(len(today_starters_api)),
    "rows_with_away_pitcher_id": int(today_starters_api["away_pitcher_id"].notna().sum()),
    "rows_with_home_pitcher_id": int(today_starters_api["home_pitcher_id"].notna().sum()),
    "rows_with_away_pitcher_name": int(today_starters_api["away_pitcher_name"].notna().sum()),
    "rows_with_home_pitcher_name": int(today_starters_api["home_pitcher_name"].notna().sum()),
}))

if "starter_pull_source" in today_starters_api.columns:
    print("")
    print(today_starters_api["starter_pull_source"].value_counts(dropna=False))

# ---------------------------------------------------------
# 3) merge fresh starters onto board
# ---------------------------------------------------------
drop_cols = [
    "away_pitcher_id", "home_pitcher_id",
    "away_pitcher_name", "home_pitcher_name",
    "away_pitcher_name_norm", "home_pitcher_name_norm",
    "away_pitcher_name_hist", "home_pitcher_name_hist",
    "away_pitcher_name_hist_by_name", "home_pitcher_name_hist_by_name",
    "away_starter_runs_allowed_10", "home_starter_runs_allowed_10",
    "away_starter_win_rate_10", "home_starter_win_rate_10",
    "away_pitcher_rest_days", "home_pitcher_rest_days",
    "away_starter_runs_allowed_10_by_name", "home_starter_runs_allowed_10_by_name",
    "away_starter_win_rate_10_by_name", "home_starter_win_rate_10_by_name",
    "away_pitcher_rest_days_by_name", "home_pitcher_rest_days_by_name",
    "diff_starter_runs_allowed_10", "diff_starter_win_rate_10", "diff_pitcher_rest_days",
]
for c in drop_cols:
    if c in daily.columns:
        daily = daily.drop(columns=c)

daily = daily.merge(
    today_starters_api,
    on="game_id",
    how="left",
    validate="one_to_one"
)

# ---------------------------------------------------------
# 4) build latest pitcher feature lookup
# ---------------------------------------------------------
def norm_name(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip().lower()
    s = " ".join(s.split())
    return s if s != "" else np.nan

def force_object_key(s):
    s = s.copy()
    s = s.where(pd.notna(s), np.nan)
    return s.astype("object")

needed_pgf_cols = [
    "pitcher_id",
    "pitcher_name",
    "date",
    "starter_runs_allowed_10",
    "starter_win_rate_10",
    "pitcher_rest_days",
]
missing_pgf = [c for c in needed_pgf_cols if c not in pgf.columns]
if missing_pgf:
    raise ValueError(f"mlb_pitcher_games_features missing required columns: {missing_pgf}")

pgf["pitcher_id"] = pd.to_numeric(pgf["pitcher_id"], errors="coerce")
pgf["date"] = pd.to_datetime(pgf["date"], errors="coerce")
pgf["pitcher_name_norm"] = force_object_key(pgf["pitcher_name"].apply(norm_name))

latest_by_id = (
    pgf[pgf["pitcher_id"].notna()]
    .sort_values(["pitcher_id", "date", "game_id"])
    .groupby("pitcher_id", as_index=False)
    .tail(1)
    .copy()
)

latest_by_name = (
    pgf[pgf["pitcher_name_norm"].notna()]
    .sort_values(["pitcher_name_norm", "date", "game_id"])
    .groupby("pitcher_name_norm", as_index=False)
    .tail(1)
    .drop_duplicates(subset=["pitcher_name_norm"], keep="last")
    .copy()
)

# ---------------------------------------------------------
# 5) normalize board starter fields
# ---------------------------------------------------------
daily["away_pitcher_id"] = pd.to_numeric(daily["away_pitcher_id"], errors="coerce")
daily["home_pitcher_id"] = pd.to_numeric(daily["home_pitcher_id"], errors="coerce")
daily["away_pitcher_name_norm"] = force_object_key(daily["away_pitcher_name"].apply(norm_name))
daily["home_pitcher_name_norm"] = force_object_key(daily["home_pitcher_name"].apply(norm_name))

# ---------------------------------------------------------
# 6) id-based feature merge
# ---------------------------------------------------------
away_id_merge = latest_by_id[[
    "pitcher_id", "pitcher_name",
    "starter_runs_allowed_10", "starter_win_rate_10", "pitcher_rest_days"
]].rename(columns={
    "pitcher_id": "away_pitcher_id",
    "pitcher_name": "away_pitcher_name_hist",
    "starter_runs_allowed_10": "away_starter_runs_allowed_10",
    "starter_win_rate_10": "away_starter_win_rate_10",
    "pitcher_rest_days": "away_pitcher_rest_days",
}).drop_duplicates(subset=["away_pitcher_id"], keep="last")

home_id_merge = latest_by_id[[
    "pitcher_id", "pitcher_name",
    "starter_runs_allowed_10", "starter_win_rate_10", "pitcher_rest_days"
]].rename(columns={
    "pitcher_id": "home_pitcher_id",
    "pitcher_name": "home_pitcher_name_hist",
    "starter_runs_allowed_10": "home_starter_runs_allowed_10",
    "starter_win_rate_10": "home_starter_win_rate_10",
    "pitcher_rest_days": "home_pitcher_rest_days",
}).drop_duplicates(subset=["home_pitcher_id"], keep="last")

daily = daily.merge(away_id_merge, on="away_pitcher_id", how="left")
daily = daily.merge(home_id_merge, on="home_pitcher_id", how="left")

# ---------------------------------------------------------
# 7) name-based fallback
# ---------------------------------------------------------
away_name_merge = latest_by_name[[
    "pitcher_name_norm", "pitcher_name",
    "starter_runs_allowed_10", "starter_win_rate_10", "pitcher_rest_days"
]].rename(columns={
    "pitcher_name_norm": "away_pitcher_name_norm",
    "pitcher_name": "away_pitcher_name_hist_by_name",
    "starter_runs_allowed_10": "away_starter_runs_allowed_10_by_name",
    "starter_win_rate_10": "away_starter_win_rate_10_by_name",
    "pitcher_rest_days": "away_pitcher_rest_days_by_name",
}).drop_duplicates(subset=["away_pitcher_name_norm"], keep="last")

home_name_merge = latest_by_name[[
    "pitcher_name_norm", "pitcher_name",
    "starter_runs_allowed_10", "starter_win_rate_10", "pitcher_rest_days"
]].rename(columns={
    "pitcher_name_norm": "home_pitcher_name_norm",
    "pitcher_name": "home_pitcher_name_hist_by_name",
    "starter_runs_allowed_10": "home_starter_runs_allowed_10_by_name",
    "starter_win_rate_10": "home_starter_win_rate_10_by_name",
    "pitcher_rest_days": "home_pitcher_rest_days_by_name",
}).drop_duplicates(subset=["home_pitcher_name_norm"], keep="last")

away_name_merge["away_pitcher_name_norm"] = force_object_key(away_name_merge["away_pitcher_name_norm"])
home_name_merge["home_pitcher_name_norm"] = force_object_key(home_name_merge["home_pitcher_name_norm"])

daily = daily.merge(away_name_merge, on="away_pitcher_name_norm", how="left")
daily = daily.merge(home_name_merge, on="home_pitcher_name_norm", how="left")

fill_pairs = [
    ("away_starter_runs_allowed_10", "away_starter_runs_allowed_10_by_name"),
    ("away_starter_win_rate_10", "away_starter_win_rate_10_by_name"),
    ("away_pitcher_rest_days", "away_pitcher_rest_days_by_name"),
    ("home_starter_runs_allowed_10", "home_starter_runs_allowed_10_by_name"),
    ("home_starter_win_rate_10", "home_starter_win_rate_10_by_name"),
    ("home_pitcher_rest_days", "home_pitcher_rest_days_by_name"),
]
for main_col, fallback_col in fill_pairs:
    if main_col not in daily.columns:
        daily[main_col] = np.nan
    if fallback_col in daily.columns:
        daily[main_col] = daily[main_col].where(daily[main_col].notna(), daily[fallback_col])

# ---------------------------------------------------------
# 8) create compressed starter model inputs
# ---------------------------------------------------------
for c in [
    "away_starter_runs_allowed_10",
    "home_starter_runs_allowed_10",
    "away_starter_win_rate_10",
    "home_starter_win_rate_10",
    "away_pitcher_rest_days",
    "home_pitcher_rest_days",
]:
    if c not in daily.columns:
        daily[c] = np.nan
    daily[c] = pd.to_numeric(daily[c], errors="coerce")

daily["diff_starter_runs_allowed_10"] = daily["away_starter_runs_allowed_10"] - daily["home_starter_runs_allowed_10"]
daily["diff_starter_win_rate_10"] = daily["away_starter_win_rate_10"] - daily["home_starter_win_rate_10"]
daily["diff_pitcher_rest_days"] = daily["away_pitcher_rest_days"] - daily["home_pitcher_rest_days"]

# ---------------------------------------------------------
# 9) debug summary
# ---------------------------------------------------------
print("")
print("=" * 80)
print("FINAL DAILY STARTER ATTACH SUMMARY")
print("=" * 80)
summary = pd.Series({
    "rows_total": int(len(daily)),
    "rows_with_away_pitcher_id": int(daily["away_pitcher_id"].notna().sum()),
    "rows_with_home_pitcher_id": int(daily["home_pitcher_id"].notna().sum()),
    "rows_with_away_pitcher_name": int(daily["away_pitcher_name"].notna().sum()),
    "rows_with_home_pitcher_name": int(daily["home_pitcher_name"].notna().sum()),
    "rows_non_null_away_runs_allowed_10": int(daily["away_starter_runs_allowed_10"].notna().sum()),
    "rows_non_null_home_runs_allowed_10": int(daily["home_starter_runs_allowed_10"].notna().sum()),
    "rows_non_null_diff_runs_allowed_10": int(daily["diff_starter_runs_allowed_10"].notna().sum()),
    "rows_non_null_diff_win_rate_10": int(daily["diff_starter_win_rate_10"].notna().sum()),
    "rows_non_null_diff_pitcher_rest_days": int(daily["diff_pitcher_rest_days"].notna().sum()),
})
print(summary)

print("")
print("Missing counts for final starter model inputs:")
print(daily[[
    "diff_starter_runs_allowed_10",
    "diff_starter_win_rate_10",
    "diff_pitcher_rest_days",
]].isna().sum())

print("")
print("Sample enriched rows:")
show_cols = [c for c in [
    "game_id", "game_datetime_utc", "away_team", "home_team",
    "away_pitcher_id", "home_pitcher_id",
    "away_pitcher_name", "home_pitcher_name",
    "away_starter_runs_allowed_10", "home_starter_runs_allowed_10",
    "diff_starter_runs_allowed_10",
    "away_starter_win_rate_10", "home_starter_win_rate_10",
    "diff_starter_win_rate_10",
    "away_pitcher_rest_days", "home_pitcher_rest_days",
    "diff_pitcher_rest_days",
    "starter_pull_source", "starter_pull_error"
] if c in daily.columns]
display(daily[show_cols].head(15))

# ---------------------------------------------------------
# 10) save back to memory
# ---------------------------------------------------------
globals()["board"] = daily.copy()
globals()["mlb_board"] = daily.copy()
globals()["projection_board"] = daily.copy()
globals()["today_starters_api"] = today_starters_api.copy()

print("")
print("Updated in memory as:")
print(" - board")
print(" - mlb_board")
print(" - projection_board")
print(" - today_starters_api")
print("")
print("Now rerun the scoring cell.")

PULL TODAY'S STARTERS DIRECTLY FROM MLB API
Board rows: 1
Unique game_ids: 1

Pulling starters from MLB API...


[1/1] found starters for 1 games

Starter API pull summary:
rows_total                     1
rows_with_away_pitcher_id      1
rows_with_home_pitcher_id      1
rows_with_away_pitcher_name    1
rows_with_home_pitcher_name    1
dtype: int64

starter_pull_source
probablePitchers    1
Name: count, dtype: int64



FINAL DAILY STARTER ATTACH SUMMARY
rows_total                              1
rows_with_away_pitcher_id               1
rows_with_home_pitcher_id               1
rows_with_away_pitcher_name             1
rows_with_home_pitcher_name             1
rows_non_null_away_runs_allowed_10      1
rows_non_null_home_runs_allowed_10      1
rows_non_null_diff_runs_allowed_10      1
rows_non_null_diff_win_rate_10          1
rows_non_null_diff_pitcher_rest_days    1
dtype: int64

Missing counts for final starter model inputs:
diff_starter_runs_allowed_10    0
diff_starter_win_rate_10        0
diff_pitcher_rest_days          0
dtype: int64

Sample enriched rows:


,game_id,game_datetime_utc,away_team,home_team,away_pitcher_id,home_pitcher_id,away_pitcher_name,home_pitcher_name,away_starter_runs_allowed_10,home_starter_runs_allowed_10,diff_starter_runs_allowed_10,away_starter_win_rate_10,home_starter_win_rate_10,diff_starter_win_rate_10,away_pitcher_rest_days,home_pitcher_rest_days,diff_pitcher_rest_days,starter_pull_source,starter_pull_error
0,849831,2026-10-11 00:00:00+00:00,Chicago White Sox,Cleveland Guardians,680732,668909,Sean Burke,Gavin Williams,5.4,3.0,2.4,0.4,0.8,-0.4,11.0,7.0,4.0,probablePitchers,NaN



Updated in memory as:
 - board
 - mlb_board
 - projection_board
 - today_starters_api

Now rerun the scoring cell.


In [81]:
# %%
# =========================================================
# CELL Ã¢â‚¬â€ FORCE PITCHER NAMES ONTO SCORED_MLB_BOARD
# Merge pitcher names from today's starter table back onto
# the scored board before building final card
# =========================================================

import pandas as pd
import numpy as np

print("=" * 80)
print("FORCE PITCHER NAMES ONTO SCORED_MLB_BOARD")
print("=" * 80)

if "scored_mlb_board" not in globals() or not isinstance(scored_mlb_board, pd.DataFrame):
    raise ValueError("scored_mlb_board not found. Run the scoring cell first.")

starter_source = None

for name in ["mlb_today_model_input", "mlb_today_starters", "today_starters", "starter_board"]:
    if name in globals() and isinstance(globals()[name], pd.DataFrame) and len(globals()[name]) > 0:
        starter_source = globals()[name].copy()
        starter_source_name = name
        break

if starter_source is None:
    raise ValueError(
        "No usable starter source found. Need one of: "
        "mlb_today_model_input, mlb_today_starters, today_starters, starter_board"
    )

print(f"Using starter source: {starter_source_name}")
print(f"Starter source shape: {starter_source.shape}")

board = scored_mlb_board.copy()

def find_first_col(df, options):
    lower_map = {str(c).strip().lower(): c for c in df.columns}
    for opt in options:
        if opt.lower() in lower_map:
            return lower_map[opt.lower()]
    return None

away_board_col = find_first_col(board, ["away_team"])
home_board_col = find_first_col(board, ["home_team"])

away_src_col = find_first_col(starter_source, ["away_team"])
home_src_col = find_first_col(starter_source, ["home_team"])

away_pitcher_src_col = find_first_col(starter_source, [
    "away_pitcher_name",
    "away_starter_name",
])

home_pitcher_src_col = find_first_col(starter_source, [
    "home_pitcher_name",
    "home_starter_name",
])

if away_board_col is None or home_board_col is None:
    raise ValueError("scored_mlb_board must contain away_team and home_team")

if away_src_col is None or home_src_col is None:
    raise ValueError("starter source must contain away_team and home_team")

if away_pitcher_src_col is None or home_pitcher_src_col is None:
    raise ValueError(
        "starter source must contain away_pitcher_name/home_pitcher_name "
        "or away_starter_name/home_starter_name"
    )

merge_df = starter_source[
    [away_src_col, home_src_col, away_pitcher_src_col, home_pitcher_src_col]
].copy().drop_duplicates(subset=[away_src_col, home_src_col], keep="last")

merge_df = merge_df.rename(columns={
    away_src_col: "away_team",
    home_src_col: "home_team",
    away_pitcher_src_col: "away_pitcher_name_fix",
    home_pitcher_src_col: "home_pitcher_name_fix",
})

board = board.merge(
    merge_df,
    on=["away_team", "home_team"],
    how="left"
)

if "away_pitcher_name" not in board.columns:
    board["away_pitcher_name"] = np.nan
if "home_pitcher_name" not in board.columns:
    board["home_pitcher_name"] = np.nan

board["away_pitcher_name"] = board["away_pitcher_name"].fillna(board["away_pitcher_name_fix"])
board["home_pitcher_name"] = board["home_pitcher_name"].fillna(board["home_pitcher_name_fix"])

board = board.drop(columns=[
    c for c in ["away_pitcher_name_fix", "home_pitcher_name_fix"] if c in board.columns
])

print("\nPitcher coverage after merge:")
print("Missing away_pitcher_name:", int(board["away_pitcher_name"].isna().sum()))
print("Missing home_pitcher_name:", int(board["home_pitcher_name"].isna().sum()))

preview_cols = [
    "away_team",
    "home_team",
    "away_pitcher_name",
    "home_pitcher_name",
]
preview_cols = [c for c in preview_cols if c in board.columns]

print("\nPreview:")
display(board[preview_cols])

globals()["scored_mlb_board"] = board.copy()

print("\nUpdated in memory:")
print(" - scored_mlb_board")
print("Done.")

FORCE PITCHER NAMES ONTO SCORED_MLB_BOARD
Using starter source: mlb_today_model_input
Starter source shape: (1, 67)

Pitcher coverage after merge:
Missing away_pitcher_name: 1
Missing home_pitcher_name: 1

Preview:


,away_team,home_team,away_pitcher_name,home_pitcher_name
0,Chicago White Sox,Cleveland Guardians,NaN,NaN



Updated in memory:
 - scored_mlb_board
Done.


In [82]:
# %% [MLB ELO Ã¢â‚¬â€ CELL A] Pull & cache 2025 game results + SP boxscore stats
# ============================================================
# Fetches: game date, teams, scores, starting pitcher, SP stats per game
# Caches to mlb_elo_cache_2025.json Ã¢â‚¬â€ only new games fetched each run
# ============================================================

import requests, json, os, time
import pandas as pd
from datetime import date

_ELO_FLOOR_SEASON = 2025  # earliest season kept in the ELO training window
_ELO_START  = "2025-03-27"
_ELO_CACHE  = os.path.join(os.getcwd(), "mlb_elo_cache_2025.json")  # filename kept for
# continuity with the existing cache on disk -- content now spans multiple seasons.

if os.path.exists(_ELO_CACHE):
    with open(_ELO_CACHE, "r", encoding="utf-8") as _f:
        _gc = json.load(_f)
    print(f"Cache loaded: {len(_gc)} games")
    _cache_dates = sorted([v.get("game_date","") for v in _gc.values() if v.get("game_date")])
    if _cache_dates:
        from datetime import datetime as _dt
        _days_stale = (date.today() - _dt.strptime(_cache_dates[-1], "%Y-%m-%d").date()).days
        if _days_stale > 7:
            print(f"  !! CACHE STALE: last game {_cache_dates[-1]} ({_days_stale}d ago) -- re-run this cell before trusting SP ELOs")
else:
    _gc = {}
    print("Starting fresh cache")

_today = date.today().strftime("%Y-%m-%d")

# Fix: this used to hardcode season=2025 with no upper bound, so every season
# after 2025 was silently never fetched (a team/pitcher trade, promotion, or
# role change after 2025 would sit at the 1500 default forever, since the
# entire current season was invisible to the ELO calc no matter how often
# this cell was re-run). Loop over every season from the floor through the
# real current year so the fetch keeps pace automatically going forward.
_new_pks, _pk_date = [], {}
for _season in range(_ELO_FLOOR_SEASON, date.today().year + 1):
    _season_start = _ELO_START if _season == _ELO_FLOOR_SEASON else f"{_season}-01-01"
    _sched = requests.get(
        f"https://statsapi.mlb.com/api/v1/schedule"
        f"?season={_season}&sportId=1&gameType=R"
        f"&startDate={_season_start}&endDate={_today}"
        f"&hydrate=linescore",
        timeout=20
    ).json()
    for _d in _sched.get("dates", []):
        for _g in _d.get("games", []):
            if _g.get("status", {}).get("abstractGameState", "") == "Final":
                _pk = str(_g["gamePk"])
                _pk_date[_pk] = _d["date"]
                if _pk not in _gc:
                    _new_pks.append(_pk)

print(f"Final games found: {len(_pk_date)} | New to fetch: {len(_new_pks)}")

def _parse_ip(v):
    try:
        s = str(v); p = s.split(".")
        return int(p[0]) + (int(p[1]) if len(p) > 1 else 0) / 3.0
    except:
        return 0.0

for _i, _pk in enumerate(_new_pks):
    try:
        _bx = requests.get(
            f"https://statsapi.mlb.com/api/v1/game/{_pk}/boxscore",
            timeout=12
        ).json()
        _bt = _bx.get("teams", {})
        _rec = {"game_date": _pk_date.get(_pk, ""), "gamePk": _pk}
        for _side in ["away", "home"]:
            _t = _bt.get(_side, {})
            _rec[f"{_side}_team"] = _t.get("team", {}).get("name", "")
            _tbat = _t.get("teamStats", {}).get("batting", {})
            _rec[f"{_side}_runs"] = int(_tbat.get("runs", 0) or 0)
            _rec[f"{_side}_hits"] = int(_tbat.get("hits", 0) or 0)
            _rec[f"{_side}_hr"]   = int(_tbat.get("homeRuns", 0) or 0)
            _pitcher_ids = _t.get("pitchers", [])
            _sp_id = _pitcher_ids[0] if _pitcher_ids else None
            _sp_name, _sp_s = "", {}
            for _pdata in _t.get("players", {}).values():
                if _pdata.get("person", {}).get("id") == _sp_id:
                    _sp_name = _pdata.get("person", {}).get("fullName", "")
                    _sp_s = _pdata.get("stats", {}).get("pitching", {})
                    break
            _rec[f"{_side}_sp"]     = _sp_name
            _rec[f"{_side}_sp_ip"]  = _parse_ip(_sp_s.get("inningsPitched", "0"))
            _rec[f"{_side}_sp_h"]   = int(_sp_s.get("hits", 0) or 0)
            _rec[f"{_side}_sp_er"]  = int(_sp_s.get("earnedRuns", 0) or 0)
            _rec[f"{_side}_sp_hr"]  = int(_sp_s.get("homeRuns", 0) or 0)
            _rec[f"{_side}_sp_bb"]  = int(_sp_s.get("baseOnBalls", 0) or 0)
            _rec[f"{_side}_sp_k"]   = int(_sp_s.get("strikeOuts", 0) or 0)
            _rec[f"{_side}_sp_hbp"] = int(_sp_s.get("hitByPitch", 0) or 0)

            # BULK-RELIEF CAPTURE (audit fix, 2026-08-04): pitchers[0] is
            # whichever pitcher threw the game's first inning, not
            # necessarily who bore the bulk of the workload -- an
            # opener/bulk-piggyback usage pattern puts the true workhorse
            # at pitchers[1] or later, where {side}_sp above never sees
            # him. Record every OTHER pitcher on this side who threw 2.0+
            # IP so Cell 83 can fold real bulk outings into that
            # pitcher's own bullpen-exposure sample instead of missing
            # them entirely.
            _bulk_arms = []
            for _pdata2 in _t.get("players", {}).values():
                if _pdata2.get("person", {}).get("id") == _sp_id:
                    continue
                _pstats2 = _pdata2.get("stats", {}).get("pitching", {})
                if not _pstats2:
                    continue
                _ip2 = _parse_ip(_pstats2.get("inningsPitched", "0"))
                if _ip2 >= 2.0:
                    _bulk_arms.append({
                        "name": _pdata2.get("person", {}).get("fullName", ""),
                        "ip":   _ip2,
                        "er":   int(_pstats2.get("earnedRuns", 0) or 0),
                    })
            _rec[f"{_side}_bulk_arms"] = _bulk_arms
        _gc[_pk] = _rec
        time.sleep(0.08)
        if (_i + 1) % 30 == 0 or (_i + 1) == len(_new_pks):
            print(f"  Fetched {_i+1}/{len(_new_pks)} ...")
    except Exception as _e:
        print(f"  Error pk={_pk}: {_e}")

with open(_ELO_CACHE, "w", encoding="utf-8") as _f:
    json.dump(_gc, _f)
print(f"Cache saved: {len(_gc)} games total")

_elo_games_df = pd.DataFrame([v for v in _gc.values() if v.get("game_date")])
_elo_games_df = _elo_games_df.sort_values("game_date").reset_index(drop=True)
print(f"ELO game log ready: {len(_elo_games_df)} games")

Cache loaded: 4858 games
  !! CACHE STALE: last game 2026-09-27 (13d ago) -- re-run this cell before trusting SP ELOs


Final games found: 4858 | New to fetch: 0


Cache saved: 4858 games total
ELO game log ready: 4858 games


In [83]:
# %% [MLB ELO Ã¢â‚¬â€ CELL B] Compute Team Offensive / Defensive / Pitcher ELO
# ============================================================
# Team Offensive ELO  : runs scored vs expected given opponent defense + SP
# Team Defensive ELO  : fielding + bullpen component (excluding starting pitcher)
# Pitcher ELO         : FIP-based (HR, K, BB, HBP Ã¢â‚¬â€ not hits), opp-adjusted
#
# FIP = (13*HR + 3*(BB+HBP) - 2*K) / IP + 3.10
#   Separates pitcher contribution from defense behind him:
#   Low FIP + high ERA = defense/bullpen hurt him (or bad luck on hits)
#   High FIP + low ERA = defense bailed him out
#
# Opponent adjustment: beating a 1600-ELO lineup is worth more than
#   the same FIP vs a 1400-ELO lineup
# ============================================================

import numpy as np
import pandas as pd

_LAVG_R = 4.50    # MLB avg runs/team/game
_K_OFF  = 24      # offensive ELO K-factor (faster to show who is hot)
_K_DEF  = 14      # defensive K-factor (fielding/bullpen Ã¢â‚¬â€ slower signal)
_K_SP   = 18      # pitcher K-factor
_FIP_C  = 3.10    # FIP constant
_MIN_IP = 2.0     # min IP for an outing to count

def _fip(hr, bb, hbp, k, ip):
    if ip < 0.1: return 7.0
    return (13*hr + 3*(bb+hbp) - 2*k) / ip + _FIP_C

def _exp_runs(off_elo, def_elo, sp_elo):
    combined_def = 0.40 * def_elo + 0.60 * sp_elo
    return max(0.5, _LAVG_R * (10 ** ((off_elo - combined_def) / 400.0)))

def _run_delta(actual, expected, k, cap=15):
    return k * (min(actual, cap)/cap - min(expected, cap)/cap)

def _sp_elo_delta(fip_val, opp_off_elo, k_mult=1.0):
    exp_fip = 4.00 + (opp_off_elo - 1500) / 300.0
    return _K_SP * k_mult * float(np.tanh((exp_fip - fip_val) / 1.5))

# Initialize
_all_tm = list(set(_elo_games_df["away_team"].tolist() + _elo_games_df["home_team"].tolist()))
_tm_off = {t: 1500.0 for t in _all_tm}
_tm_def = {t: 1500.0 for t in _all_tm}
_tm_game_ct = {t: 0 for t in _all_tm}  # games actually contributing to this team's ELO
_sp_elo_map = {}
_sp_trk = {}
_bulk_trk = {}  # name -> [ip, ip, ...] for bulk-relief-only appearances (never a recorded {side}_sp)

for _, _g in _elo_games_df.iterrows():
    _aw = str(_g.get("away_team", "") or "")
    _hm = str(_g.get("home_team", "") or "")
    if not _aw or not _hm:
        continue
    _ar    = float(_g.get("away_runs", 0) or 0)
    _hr_r  = float(_g.get("home_runs", 0) or 0)
    _asp   = str(_g.get("away_sp", "") or "")
    _hsp   = str(_g.get("home_sp", "") or "")

    for _sp in [_asp, _hsp]:
        if _sp and _sp not in _sp_elo_map:
            _sp_elo_map[_sp] = 1500.0
            _sp_trk[_sp] = {
                "starts": 0, "fip": [], "k9": [], "hr9": [],
                "h_pg": [], "vs_hi": [], "vs_lo": [], "opp_elos": [],
                "ip_starts": [],  # per-start IP -- used for bullpen-exposure rate below
                # 2026-07-30: switched home/road split tracking from FIP to ERA --
                # FIP deliberately filters out BABIP/sequencing/defense, which is
                # exactly the signal a real road ERA collapse (e.g. Woo: 2.24 home /
                # 5.71 road ERA in 2026, 80+ IP each side) runs on. Verified live:
                # Woo's road FIP split was a routine 1.18-run gap that barely moved
                # his capped-1800 ELO, while his real road ERA gap is 3.47 runs.
                # Tracks both current-season and all-time IP/ER so a thin current
                # season can fall back to the full multi-season sample.
                "home_ip": 0.0, "home_er": 0.0, "road_ip": 0.0, "road_er": 0.0,
                "home_ip_cur": 0.0, "home_er_cur": 0.0, "road_ip_cur": 0.0, "road_er_cur": 0.0,
                # 2026-09-10 (user audit, items 1+4): per-side START COUNT (distinct from
                # starts, which is home+road combined) and per-side HR-allowed, tracked the
                # same way home_ip/home_er already are -- home_er/road_er existed and fed a
                # real home/road ERA split (mlb_sp_split_era below), but HR was only ever
                # appended to a flat, unsplit "hr9" list (see the per-start loop below), and
                # nothing anywhere counted starts per side (only combined "starts"), so
                # "how many road starts do we actually have" couldn't be checked before
                # trusting a split -- only the IP total could.
                "home_starts": 0, "road_starts": 0,
                "home_hr": 0, "road_hr": 0,
            }

    _ao   = _tm_off.get(_aw, 1500); _hd = _tm_def.get(_hm, 1500)
    _ho   = _tm_off.get(_hm, 1500); _ad = _tm_def.get(_aw, 1500)
    _aspe = _sp_elo_map.get(_asp, 1500)
    _hspe = _sp_elo_map.get(_hsp, 1500)

    _exp_ar = _exp_runs(_ao, _hd, _hspe)
    _exp_hr = _exp_runs(_ho, _ad, _aspe)

    _tm_off[_aw] += _run_delta(_ar,   _exp_ar, _K_OFF)
    _tm_off[_hm] += _run_delta(_hr_r, _exp_hr, _K_OFF)
    _tm_def[_hm] += _run_delta(_exp_ar, _ar,   _K_DEF * 0.4)
    _tm_def[_aw] += _run_delta(_exp_hr, _hr_r, _K_DEF * 0.4)
    _tm_game_ct[_aw] += 1
    _tm_game_ct[_hm] += 1

    for _sp, _sd, _opp_off in [(_asp, "away", _ho), (_hsp, "home", _ao)]:
        if not _sp:
            continue
        _ip  = float(_g.get(f"{_sd}_sp_ip", 0) or 0)
        if _ip < _MIN_IP:
            continue
        _h   = int(_g.get(f"{_sd}_sp_h",   0) or 0)
        _xhr = int(_g.get(f"{_sd}_sp_hr",  0) or 0)
        _bb  = int(_g.get(f"{_sd}_sp_bb",  0) or 0)
        _k   = int(_g.get(f"{_sd}_sp_k",   0) or 0)
        _hbp = int(_g.get(f"{_sd}_sp_hbp", 0) or 0)
        _er  = float(_g.get(f"{_sd}_sp_er", 0) or 0)

        _fv  = _fip(_xhr, _bb, _hbp, _k, _ip)
        _g_yr = str(_g.get("game_date", ""))[:4]
        _k_mult = 3.0 if _g_yr == "2026" else (1.5 if _g_yr == "2025" else 1.0)
        _sp_elo_map[_sp] += _sp_elo_delta(_fv, _opp_off, _k_mult)

        _tr = _sp_trk[_sp]
        _tr["starts"] += 1
        _tr["fip"].append(_fv)
        _tr["k9"].append(9 * _k / _ip)
        _tr["hr9"].append(9 * _xhr / _ip)
        _tr["h_pg"].append(_h)
        _tr["opp_elos"].append(_opp_off)
        _tr["ip_starts"].append(_ip)
        _sk = "home" if _sd == "home" else "road"  # _sd is "home"/"away"; tracker keys are "home"/"road"
        _tr[f"{_sk}_ip"] += _ip
        _tr[f"{_sk}_er"] += _er
        _tr[f"{_sk}_starts"] += 1
        _tr[f"{_sk}_hr"] += _xhr
        if _g_yr == "2026":
            _tr[f"{_sk}_ip_cur"] += _ip
            _tr[f"{_sk}_er_cur"] += _er
        if _opp_off >= 1530:
            _tr["vs_hi"].append(_fv)
        elif _opp_off <= 1470:
            _tr["vs_lo"].append(_fv)

    # BULK-RELIEF CAPTURE (audit fix, 2026-08-04): a pitcher used as a
    # bulk/piggyback arm (entering after an opener) never shows up as
    # {side}_sp above -- the boxscore fetch in Cell 82 now separately
    # records every OTHER pitcher on each side who threw 2.0+ IP as
    # f"{side}_bulk_arms". Track their IP here too so a swingman like
    # Alvarez (mix of true starts + bulk relief) gets his full real
    # workload counted for bullpen-exposure purposes, not just the
    # handful of games he happened to be pitchers[0] in.
    for _side_b in ("away", "home"):
        # 2026-08-04 BUG FIX (live TypeError): "x or []" doesn't catch a NaN
        # float -- NaN is truthy in Python, so when this key is missing from
        # a row (pandas represents that as float("nan"), not None/[]), the
        # "or []" fallback never fires and the raw NaN gets iterated over,
        # crashing with "TypeError: 'float' object is not iterable". Check
        # the actual type instead of relying on truthiness.
        _barm_list = _g.get(f"{_side_b}_bulk_arms", [])
        if not isinstance(_barm_list, list):
            _barm_list = []
        for _barm in _barm_list:
            _bname = str(_barm.get("name", "") or "")
            _bip = float(_barm.get("ip", 0) or 0)
            if not _bname or _bip < _MIN_IP:
                continue
            if _bname not in _bulk_trk:
                _bulk_trk[_bname] = []
            _bulk_trk[_bname].append(_bip)

def _avg(lst):
    return float(np.mean(lst)) if lst else None

# SP BULLPEN EXPOSURE (audit fix): rate of starts actually reaching 5 IP.
# A starter who rarely gets there is really a bullpen game once he's
# pulled -- treating him like a full starter for edge purposes overstates
# confidence in the SP matchup. Surfaced here (not guessed from a role
# label) off his real per-start IP history; consumed in Cell 88's
# _why_bullets via mlb_sp_bullpen_exposure.
_sp_rows = []
mlb_sp_bullpen_exposure = {}
for _sp, _tr in _sp_trk.items():
    if _tr["starts"] < 1:
        continue
    _five_ip_starts = sum(1 for _ip_s in _tr["ip_starts"] if _ip_s >= 5.0)
    _five_ip_pct = _five_ip_starts / _tr["starts"]
    mlb_sp_bullpen_exposure[_sp] = {
        "starts": _tr["starts"],
        "five_ip_starts": _five_ip_starts,
        "five_ip_pct": _five_ip_pct,
    }
    _sp_rows.append({
        "pitcher":   _sp,
        "elo":       round(_sp_elo_map.get(_sp, 1500), 1),
        "starts":    _tr["starts"],
        "fip":       round(_avg(_tr["fip"]), 2) if _avg(_tr["fip"]) else None,
        "k9":        round(_avg(_tr["k9"]),  1) if _avg(_tr["k9"])  else None,
        "hr9":       round(_avg(_tr["hr9"]), 2) if _avg(_tr["hr9"]) else None,
        "h_pg":      round(_avg(_tr["h_pg"]),1) if _avg(_tr["h_pg"])else None,
        "vs_hi_fip": round(_avg(_tr["vs_hi"]), 2) if _tr["vs_hi"]  else None,
        "vs_lo_fip": round(_avg(_tr["vs_lo"]), 2) if _tr["vs_lo"]  else None,
        "five_ip_starts": _five_ip_starts,
        "five_ip_pct":    round(_five_ip_pct, 2),
    })

# Fold bulk-relief-only appearances into the same exposure table --
# either topping up a pitcher who also has true starts, or creating a
# fresh entry for someone who is pure bulk/piggyback all season.
_bulk_merged_ct = 0
for _bname, _bips in _bulk_trk.items():
    _b_five = sum(1 for _ip_s in _bips if _ip_s >= 5.0)
    if _bname in mlb_sp_bullpen_exposure:
        _e = mlb_sp_bullpen_exposure[_bname]
        _e["starts"] += len(_bips)
        _e["five_ip_starts"] += _b_five
        _e["five_ip_pct"] = _e["five_ip_starts"] / _e["starts"] if _e["starts"] else 0.0
    else:
        mlb_sp_bullpen_exposure[_bname] = {
            "starts": len(_bips),
            "five_ip_starts": _b_five,
            "five_ip_pct": (_b_five / len(_bips)) if _bips else 0.0,
        }
    _bulk_merged_ct += 1
if _bulk_merged_ct:
    print(f"  Bullpen exposure: folded bulk-relief appearances for {_bulk_merged_ct} pitcher(s) not otherwise recorded as a starter")

mlb_sp_elo_df   = pd.DataFrame(_sp_rows).sort_values("elo", ascending=False).reset_index(drop=True)
mlb_team_elo_df = pd.DataFrame([
    {
        "team":    t,
        "off_elo": round(_tm_off[t], 1),
        "def_elo": round(_tm_def[t], 1),
        "net_elo": round(0.5*_tm_off[t] + 0.5*_tm_def[t], 1),
    }
    for t in _all_tm
]).sort_values("net_elo", ascending=False).reset_index(drop=True)

# Expose for downstream cells
mlb_team_off_elo = _tm_off
mlb_team_def_elo = _tm_def
mlb_pitcher_elo  = _sp_elo_map
mlb_team_elo_gamecount = _tm_game_ct
# ELO outlier cap: historical FIP-based computation can produce unrealistic values
# Cap at 1800 — no MLB starter has ever sustained 1800+ ELO over a full sample
_elo_outlier_ct = sum(1 for v in mlb_pitcher_elo.values() if v > 1800)
if _elo_outlier_ct:
    for _spk in list(mlb_pitcher_elo):
        if mlb_pitcher_elo[_spk] > 1800:
            mlb_pitcher_elo[_spk] = 1800.0
    print(f"  ELO outlier cap: {_elo_outlier_ct} pitcher(s) capped at 1800 (were above)")

# BUG FIX (2026-08-05, real incident: Drew Anderson's spot start (DET @ SEA)
# showed ELO 1601 "ELITE" tier from exactly 2 career starts -- FIP 1.39 and
# 1.6, neither reaching 5 IP -- because this loop applies the same K_SP=18
# (tripled for 2026 outings via _k_mult) to a reliever's rare starts as it
# does to an established starter's full workload, with no sample-size
# regression anywhere above. SP_ELO_OVERRIDES right below is a floor only
# (guards against rating a thin sample too LOW), so it does nothing when a
# small sample happens to look great instead -- exactly Anderson's case.
# Regress every thin-sample pitcher's ELO toward the 1500 default in
# proportion to how few starts are on record (reusing this same cell's
# own _QUAL_MIN_GAMES=8 convention from the opp-quality batting section
# below as "enough starts to trust"), so 1-2 outlier outings can nudge a
# rating but can't alone make a reliever read as an elite starter.
_MIN_STARTS_FOR_FULL_ELO_TRUST = 8
_thin_start_ct = 0
for _spk, _tr in _sp_trk.items():
    _n_starts = _tr.get("starts", 0)
    if 0 < _n_starts < _MIN_STARTS_FOR_FULL_ELO_TRUST and _spk in mlb_pitcher_elo:
        _trust_w = _n_starts / _MIN_STARTS_FOR_FULL_ELO_TRUST
        _raw_elo = mlb_pitcher_elo[_spk]
        _regressed = 1500.0 + (_raw_elo - 1500.0) * _trust_w
        if abs(_regressed - _raw_elo) >= 1.0:
            mlb_pitcher_elo[_spk] = _regressed
            _thin_start_ct += 1
if _thin_start_ct:
    print(f"  Thin-starts-sample ELO regression: {_thin_start_ct} pitcher(s) with "
          f"< {_MIN_STARTS_FOR_FULL_ELO_TRUST} tracked starts pulled toward the 1500 "
          f"default (proportional to sample size)")

# Manual SP ELO overrides â€” pitchers not yet in historical data (call-ups, spot starters).
# ELO reference: ~1650 = solid #2/#3 starter, ~1600 = average, ~1500 = unknown/default.
# Any pitcher showing [SP DATA: DEFAULT ELO] on the card should be added here.
SP_ELO_OVERRIDES = {
    "Drew Anderson":      1510.0,   # limited MLB data â€” slight bump above pure default
    "Bubba Chandler":     1520.0,   # PIT prospect, limited sample
    "Quinn Mathews":      1530.0,   # STL rookie, solid strikeout rate
    "Bowden Francis":     1540.0,   # TOR, some MLB starts
    "Hayden Wesneski":    1530.0,   # CHC, limited data
    "Cade Povich":        1510.0,   # BAL, early career
    "Daniel Espino":      1520.0,   # CLE prospect
    "Ryne Nelson":        1520.0,   # ARI, limited sample
    "Gavin Williams":     1530.0,   # CLE, coming off injury
    "Kyle Harrison":      1530.0,   # SF, young starter
    "Trey Yesavage":      1580.0,   # TOR 2026 rookie, 2.25 ERA in first starts
    "Adam Macko":         1540.0,   # TOR 2026 debut, solid minor league track record
    "Brandon Young":      1520.0,   # BAL, limited MLB sample
}
_override_count = 0
for _sp_name, _sp_floor in SP_ELO_OVERRIDES.items():
    if _sp_name not in mlb_pitcher_elo or mlb_pitcher_elo[_sp_name] < _sp_floor:
        mlb_pitcher_elo[_sp_name] = _sp_floor
        _override_count += 1
if _override_count:
    print(f"SP ELO manual overrides applied: {_override_count} pitchers")
mlb_sp_tracker   = _sp_trk
_SP_SPLIT_CUR_MIN_IP = 25.0  # current-season split trusted above this many IP/side
_SP_SPLIT_ALL_MIN_IP = 40.0  # combined-seasons fallback trusted above this many IP/side
mlb_sp_split_era = {}
for _spn, _tk in _sp_trk.items():
    _hip_c, _rip_c = _tk.get("home_ip_cur", 0.0), _tk.get("road_ip_cur", 0.0)
    _hip_a, _rip_a = _tk.get("home_ip", 0.0), _tk.get("road_ip", 0.0)
    if _hip_c >= _SP_SPLIT_CUR_MIN_IP and _rip_c >= _SP_SPLIT_CUR_MIN_IP:
        _he, _re = 9 * _tk["home_er_cur"] / _hip_c, 9 * _tk["road_er_cur"] / _rip_c
        _hip, _rip, _src = _hip_c, _rip_c, "2026"
    elif _hip_a >= _SP_SPLIT_ALL_MIN_IP and _rip_a >= _SP_SPLIT_ALL_MIN_IP:
        _he, _re = 9 * _tk["home_er"] / _hip_a, 9 * _tk["road_er"] / _rip_a
        _hip, _rip, _src = _hip_a, _rip_a, "all-time"
    else:
        continue  # not enough IP on one side either way -- no entry, gate stays closed
    # 2026-09-10 (user audit, item 1): "8+ starts" sample-size gate (user's exact
    # spec) alongside the existing IP floor -- home_starts/road_starts are combined
    # across whichever season window happened to be used above (current vs
    # all-time), since a start-count sample-size check is a coarser, season-window-
    # independent signal than the IP floor itself. season_era is the IP-weighted
    # blend of BOTH sides from the SAME source window selected above (current-season
    # or all-time) -- used downstream (Cell 88) to scale a runs-allowed-per-start
    # figure by the real home/road ratio instead of substituting a different-scale
    # number outright.
    if _src == "2026":
        _season_era = 9 * (_tk["home_er_cur"] + _tk["road_er_cur"]) / max(_hip_c + _rip_c, 0.1)
        _hhr9 = 9 * _tk.get("home_hr", 0) / _hip_c if _hip_c > 0 else None
        _rhr9 = 9 * _tk.get("road_hr", 0) / _rip_c if _rip_c > 0 else None
    else:
        _season_era = 9 * (_tk["home_er"] + _tk["road_er"]) / max(_hip_a + _rip_a, 0.1)
        _hhr9 = 9 * _tk.get("home_hr", 0) / _hip_a if _hip_a > 0 else None
        _rhr9 = 9 * _tk.get("road_hr", 0) / _rip_a if _rip_a > 0 else None
    mlb_sp_split_era[_spn] = {
        "home_era": _he, "home_ip": round(_hip, 1),
        "road_era": _re, "road_ip": round(_rip, 1),
        "source": _src, "season_era": round(_season_era, 2),
        "home_starts": _tk.get("home_starts", 0), "road_starts": _tk.get("road_starts", 0),
        "home_hr9": round(_hhr9, 2) if _hhr9 is not None else None,
        "road_hr9": round(_rhr9, 2) if _rhr9 is not None else None,
    }

# 2026-season ERA cap: pitchers with ERA > 6.5 in 5+ starts this season get ELO capped at 1500
# Catches veterans with stale high ELO whose current-season form has collapsed (e.g. Rogers 2026)
_season_yr = "2026"
_sp_season = {}  # {name: {ip, er, starts}}
for _, _sg in _elo_games_df.iterrows():
    if str(_sg.get("game_date", ""))[:4] != _season_yr: continue
    for _side in ("home", "away"):
        _sn = str(_sg.get(f"{_side}_sp", "") or "")
        _ip = float(_sg.get(f"{_side}_sp_ip", 0) or 0)
        _er = float(_sg.get(f"{_side}_sp_er", 0) or 0)
        if not _sn or _ip < 1.0: continue
        if _sn not in _sp_season: _sp_season[_sn] = {"ip": 0.0, "er": 0, "starts": 0}
        _sp_season[_sn]["ip"]     += _ip
        _sp_season[_sn]["er"]     += _er
        _sp_season[_sn]["starts"] += 1
_era_cap_count = 0
for _sn, _ss in _sp_season.items():
    if _ss["starts"] < 5 or _ss["ip"] < 1.0: continue
    _season_era = 9.0 * _ss["er"] / _ss["ip"]
    if _season_era > 6.5 and _sn in mlb_pitcher_elo and mlb_pitcher_elo[_sn] > 1500:
        _ss_starts = _ss["starts"]
        print(f"  [ERA CAP] {_sn}: {_season_yr} ERA {_season_era:.2f} ({_ss_starts} starts) â€” ELO {mlb_pitcher_elo[_sn]:.0f} capped to 1500")
        mlb_pitcher_elo[_sn] = 1500.0
        _era_cap_count += 1
if _era_cap_count:
    print(f"ERA cap applied to {_era_cap_count} pitcher(s)")

# MLB-SP-ELO-EST-1 (2026-10-01 user audit, backlog item #7 -- Nola rated 1309
# off a run of bad results while his expected wOBA (.337 vs .353 actual) says
# he's been unlucky). The rating above is a start-by-start update on each
# start's FIP -- it tracks results, so an unlucky stretch drags it down. For
# any starter with a real FanGraphs season line (>= _EST_ELO_MIN_IP), the
# rating is now REPLACED by an estimator-implied one:
#     est = 0.2*FIP + 0.4*xFIP + 0.4*SIERA   (weights renormalized if one is missing)
# mapped onto this model's own ELO scale by a least-squares line of the
# current sequential ELO on season FIP across established starters -- so the
# scale comes from the ratings themselves, not an invented constant. Runs
# AFTER the ERA cap on purpose: that cap punishes ERA (luck included), which is
# exactly what this replaces for starters with enough innings.
_EST_ELO_MIN_IP = 40.0
_EST_W = {"fip": 0.2, "xfip": 0.4, "siera": 0.4}
_fip_lk_est = globals().get("mlb_pitcher_fip_lookup", {}) or {}
def _est_norm(s):
    import unicodedata as _eu, re as _er
    s = _eu.normalize("NFD", str(s)); s = "".join(c for c in s if _eu.category(c) != "Mn")
    return _er.sub(r"[^a-z ]", "", s.lower()).strip()
def _est_era(v):
    # shared with Cell 87's recency step (defined in Cell 19): SIERA slot falls
    # back to Savant xERA when SIERA is missing (FanGraphs 403 since 2026-09-27)
    return _mlb_sp_est_era(v)[0] if "_mlb_sp_est_era" in globals() else None
_fit_x, _fit_y = [], []
for _spk, _tr in _sp_trk.items():
    _fv = _fip_lk_est.get(_est_norm(_spk))
    if not _fv or _spk not in mlb_pitcher_elo or _tr.get("starts", 0) < _MIN_STARTS_FOR_FULL_ELO_TRUST:
        continue
    if (_fv.get("ip_season") or 0) >= _EST_ELO_MIN_IP and _fv.get("fip") == _fv.get("fip") and _fv.get("fip") is not None:
        _fit_x.append(float(_fv["fip"])); _fit_y.append(float(mlb_pitcher_elo[_spk]))
if len(_fit_x) >= 30:
    _b_est, _a_est = np.polyfit(_fit_x, _fit_y, 1)
    print(f"[SP ELO EST] scale fit: ELO = {_a_est:.0f} {_b_est:+.1f} x FIP  (n={len(_fit_x)} established starters)")
    _est_ct, _est_log = 0, []
    for _spk in list(mlb_pitcher_elo):
        _fv = _fip_lk_est.get(_est_norm(_spk))
        if not _fv or (_fv.get("ip_season") or 0) < _EST_ELO_MIN_IP:
            continue
        _e = _est_era(_fv)
        if _e is None:
            continue
        _new = float(np.clip(_a_est + _b_est * _e, min(_fit_y), 1800.0))   # floor = lowest established rating; ceiling = existing 1800 cap
        _old = float(mlb_pitcher_elo[_spk])
        mlb_pitcher_elo[_spk] = _new
        _est_ct += 1
        _est_log.append((abs(_new - _old), _spk, _old, _new, _e))
    globals()["mlb_sp_elo_est_applied"] = {n: (o, nw, e) for _, n, o, nw, e in _est_log}
    print(f"[SP ELO EST] {_est_ct} starter rating(s) now estimator-based (FIP/xFIP/SIERA 20/40/40, >= {_EST_ELO_MIN_IP:.0f} IP)")
    for _d, _n, _o, _nw, _e in sorted(_est_log, reverse=True)[:8]:
        print(f"    {_n}: {_o:.0f} -> {_nw:.0f}  (estimator ERA {_e:.2f})")
else:
    print(f"[SP ELO EST] skipped -- only {len(_fit_x)} established starters with a FanGraphs FIP to fit the scale (need 30)")

print("\n" + "="*65)
print("TEAM ELO RANKINGS  (off | def | net)")
print("="*65)
print(mlb_team_elo_df.to_string(index=False))

print("\n" + "="*65)
print("TOP 20 PITCHER ELO  (FIP-based, opponent-adjusted)")
print("  vs_hi_fip = FIP vs top-25% offenses")
print("  vs_lo_fip = FIP vs bottom-25% offenses")
print("="*65)
print(mlb_sp_elo_df.head(20).to_string(index=False))

print("\n" + "="*65)
print("BOTTOM 10 PITCHER ELO")
print("="*65)
print(mlb_sp_elo_df.tail(10).to_string(index=False))

# expose for downstream use
mlb_exp_runs_fn = _exp_runs

# sample pitcher ELO keys to verify name format
print("Sample pitcher ELO keys (top 5 by ELO):")
for _pn, _pe in sorted(_sp_elo_map.items(), key=lambda x: x[1], reverse=True)[:5]:
    print(f"  {repr(_pn)}: {_pe:.1f}")

# â”€â”€ Opponent-quality-adjusted batting â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# For each team, compute avg runs scored when facing a starter whose FIP
# in that specific outing was <= 3.20 (elite) or >= 4.50 (weak).
# FIP is box-score-derived per game so there is no leakage.
# Used in Cell 78 to adjust predicted offense when facing an elite arm.
_ELITE_FIP_THRESH = 3.20
_WEAK_FIP_THRESH  = 4.50
_QUAL_MIN_GAMES   = 8    # need at least 8 qualifying starts to trust average
_QUAL_WINDOW      = 30   # trailing starts considered

_tm_runs_vs_elite = {t: [] for t in _all_tm}
_tm_runs_vs_weak  = {t: [] for t in _all_tm}

for _, _gq in _elo_games_df.iterrows():
    _aw_q = str(_gq.get("away_team", "") or "")
    _hm_q = str(_gq.get("home_team", "") or "")
    if not _aw_q or not _hm_q:
        continue
    _ar_q  = float(_gq.get("away_runs", 0) or 0)
    _hr_q  = float(_gq.get("home_runs",  0) or 0)
    for _side_q, _sp_key, _batting_team, _runs in [
        ("home", "home_sp", _aw_q, _ar_q),
        ("away", "away_sp", _hm_q, _hr_q),
    ]:
        _sp_q  = str(_gq.get(_sp_key, "") or "")
        if not _sp_q:
            continue
        _ip_q  = float(_gq.get(f"{_side_q}_sp_ip",  0) or 0)
        if _ip_q < _MIN_IP:
            continue
        _xhr_q = int(_gq.get(f"{_side_q}_sp_hr",  0) or 0)
        _bb_q  = int(_gq.get(f"{_side_q}_sp_bb",  0) or 0)
        _k_q   = int(_gq.get(f"{_side_q}_sp_k",   0) or 0)
        _hbp_q = int(_gq.get(f"{_side_q}_sp_hbp", 0) or 0)
        _fip_q = _fip(_xhr_q, _bb_q, _hbp_q, _k_q, _ip_q)
        if _batting_team in _tm_runs_vs_elite:
            if _fip_q <= _ELITE_FIP_THRESH:
                _tm_runs_vs_elite[_batting_team].append(_runs)
            elif _fip_q >= _WEAK_FIP_THRESH:
                _tm_runs_vs_weak[_batting_team].append(_runs)

mlb_team_off_vs_elite = {}
mlb_team_off_vs_weak  = {}
for _t in _all_tm:
    _eg = _tm_runs_vs_elite[_t][-_QUAL_WINDOW:]
    _wg = _tm_runs_vs_weak[_t][-_QUAL_WINDOW:]
    if len(_eg) >= _QUAL_MIN_GAMES:
        mlb_team_off_vs_elite[_t] = round(float(np.mean(_eg)), 2)
    if len(_wg) >= _QUAL_MIN_GAMES:
        mlb_team_off_vs_weak[_t]  = round(float(np.mean(_wg)), 2)

print(f"\nOpp-quality batting built for {len(mlb_team_off_vs_elite)} teams (vs elite SP, FIP<={_ELITE_FIP_THRESH}).")
print(f"Sample (sorted by runs vs elite):")
for _t, _v in sorted(mlb_team_off_vs_elite.items(), key=lambda x: x[1], reverse=True)[:8]:
    _raw = mlb_team_elo_df[mlb_team_elo_df["team"] == _t]["off_elo"].values
    _wv = mlb_team_off_vs_weak.get(_t, float("nan"))
    print(f"  {_t:<28s} vs_elite={_v:.2f}  vs_weak={_wv:.2f}")


  Bullpen exposure: folded bulk-relief appearances for 288 pitcher(s) not otherwise recorded as a starter
  ELO outlier cap: 39 pitcher(s) capped at 1800 (were above)
  Thin-starts-sample ELO regression: 148 pitcher(s) with < 8 tracked starts pulled toward the 1500 default (proportional to sample size)
SP ELO manual overrides applied: 9 pitchers


  [ERA CAP] Hunter Greene: 2026 ERA 6.83 (5 starts) â€” ELO 1640 capped to 1500
  [ERA CAP] Brady Basso: 2026 ERA 7.56 (7 starts) â€” ELO 1615 capped to 1500
ERA cap applied to 2 pitcher(s)
[SP ELO EST] scale fit: ELO = 2608 -266.5 x FIP  (n=195 established starters)
[SP ELO EST] 255 starter rating(s) now estimator-based (FIP/xFIP/SIERA 20/40/40, >= 40 IP)
    Grant Holman: 1500 -> 576  (estimator ERA 17.50)
    Johan Oviedo: 1459 -> 576  (estimator ERA 9.17)
    Yariel Rodríguez: 1500 -> 741  (estimator ERA 7.01)
    Carson Spiers: 1501 -> 766  (estimator ERA 6.91)
    Tomoyuki Sugano: 576 -> 1299  (estimator ERA 4.91)
    Jonathan Cannon: 1394 -> 696  (estimator ERA 7.17)
    Valente Bellozo: 1480 -> 807  (estimator ERA 6.76)
    Ben Casparius: 1502 -> 873  (estimator ERA 6.51)

TEAM ELO RANKINGS  (off | def | net)
                 team  off_elo  def_elo  net_elo
     San Diego Padres   1523.1   1644.3   1583.7
      Cincinnati Reds   1496.5   1602.5   1549.5
            Athletics   


Opp-quality batting built for 30 teams (vs elite SP, FIP<=3.2).
Sample (sorted by runs vs elite):
  Washington Nationals         vs_elite=4.77  vs_weak=5.40
  Arizona Diamondbacks         vs_elite=4.00  vs_weak=6.33
  San Diego Padres             vs_elite=3.97  vs_weak=5.57
  Colorado Rockies             vs_elite=3.97  vs_weak=6.03
  Los Angeles Dodgers          vs_elite=3.80  vs_weak=5.63
  Chicago Cubs                 vs_elite=3.70  vs_weak=6.53
  Los Angeles Angels           vs_elite=3.63  vs_weak=5.80
  Miami Marlins                vs_elite=3.60  vs_weak=4.97


In [84]:
# =========================================================
# CELL Ã¢â‚¬â€ MLB MODEL CALIBRATION
# Buckets walk-forward predictions by win probability and
# shows actual accuracy per bucket.
# Run after Cell 14 (walk-forward backtest).
# This tells you what MIN_MODEL_PROB threshold to trust.
# =========================================================
import pandas as pd
import numpy as np

print("=" * 65)
print("MLB MODEL CALIBRATION Ã¢â‚¬â€ WIN PROB ACCURACY BY BUCKET")
print("=" * 65)

if "mlb_wf_preds" not in globals() or not isinstance(mlb_wf_preds, pd.DataFrame):
    print("mlb_wf_preds not found. Run Cell 14 first.")
else:
    _df = mlb_wf_preds.copy()
    _df["p_home_win"] = pd.to_numeric(_df["p_home_win"], errors="coerce")
    _df = _df.dropna(subset=["p_home_win", "home_win"])

    # Build both perspectives: home bet (p >= 0.50) and away bet (p < 0.50)
    # For away bets: flip so p_bet = 1 - p_home, actual_win = 1 - home_win
    _home_rows = _df[_df["p_home_win"] >= 0.50].copy()
    _home_rows["p_bet"]      = _home_rows["p_home_win"]
    _home_rows["actual_win"] = _home_rows["home_win"].astype(int)

    _away_rows = _df[_df["p_home_win"] < 0.50].copy()
    _away_rows["p_bet"]      = 1.0 - _away_rows["p_home_win"]
    _away_rows["actual_win"] = (1 - _away_rows["home_win"].astype(int))

    _all = pd.concat([_home_rows, _away_rows], ignore_index=True)

    print(f"Total predictions: {len(_all)}  (home: {len(_home_rows)}, away: {len(_away_rows)})")
    print(f"Overall accuracy : {_all['actual_win'].mean():.1%}")
    print()

    # Buckets
    _bins   = [0.50, 0.52, 0.54, 0.56, 0.58, 0.60, 0.65, 1.01]
    _labels = ["50-52%", "52-54%", "54-56%", "56-58%", "58-60%", "60-65%", "65%+"]

    _all["bucket"] = pd.cut(_all["p_bet"], bins=_bins, labels=_labels, right=False)
    _calib = (
        _all.groupby("bucket", observed=True)
        .agg(
            games       =("actual_win", "count"),
            actual_winrate=("actual_win", "mean"),
            avg_pred_prob =("p_bet",      "mean"),
        )
        .reset_index()
    )

    print(f"{'Bucket':<12} {'Games':>6} {'Pred%':>7} {'Actual%':>8}  {'Signal':>8}")
    print("-" * 50)
    for _, row in _calib.iterrows():
        if row["games"] == 0:
            continue
        diff = row["actual_winrate"] - row["avg_pred_prob"]
        signal = "OVER" if diff > 0.02 else ("UNDER" if diff < -0.02 else "calibrated")
        print(f"{row['bucket']:<12} {int(row['games']):>6} {row['avg_pred_prob']:>6.1%} "
              f"{row['actual_winrate']:>8.1%}  {signal:>12}")

    print()
    print("Recommendation:")
    _high = _calib[_calib["bucket"].isin(["58-60%", "60-65%", "65%+"])]
    _high_acc = (_high["actual_winrate"] * _high["games"]).sum() / _high["games"].sum() if _high["games"].sum() > 0 else 0
    _high_n   = int(_high["games"].sum())
    print(f"  At >=58% predicted prob: {_high_acc:.1%} actual accuracy ({_high_n} games)")

    _med = _calib[_calib["bucket"].isin(["56-58%", "58-60%"])]
    _med_acc = (_med["actual_winrate"] * _med["games"]).sum() / _med["games"].sum() if _med["games"].sum() > 0 else 0
    _med_n   = int(_med["games"].sum())
    print(f"  At >=56% predicted prob: {_med_acc:.1%} actual accuracy ({_med_n} games)")

    # Store calibration for reference
    globals()["mlb_calibration"] = _calib.copy()
    print("\nSaved: mlb_calibration")


MLB MODEL CALIBRATION Ã¢â‚¬â€ WIN PROB ACCURACY BY BUCKET
Total predictions: 5326  (home: 3714, away: 1612)
Overall accuracy : 53.5%

Bucket        Games   Pred%  Actual%    Signal
--------------------------------------------------
50-52%         1894  51.0%    49.4%    calibrated
52-54%         1547  53.0%    54.8%    calibrated
54-56%         1027  54.9%    55.9%    calibrated
56-58%          509  56.8%    56.6%    calibrated
58-60%          230  58.9%    56.5%         UNDER
60-65%          117  61.6%    62.4%    calibrated
65%+              2  66.0%   100.0%          OVER

Recommendation:
  At >=58% predicted prob: 58.7% actual accuracy (349 games)
  At >=56% predicted prob: 56.6% actual accuracy (739 games)

Saved: mlb_calibration


In [85]:
# %%
# =========================================================
# CELL - MLB TOTALS CALIBRATION (Cell-84 equivalent for run totals)
# Buckets projected totals against real historical actual totals,
# checking whether the totals projection is systematically biased
# high or low. Built 2026-07-23 -- there was no totals-equivalent
# of the win-model calibration check (Cell 84) anywhere in this
# notebook, meaning the O/U gate's 18% floor / 25% ceiling had no
# feedback loop telling you whether the projections behind those
# edges were trustworthy in the first place.
#
# 2026-07-28 rewrite -- root cause found while investigating why this
# calibration had zero 2026 games in it: this cell used to read
# mlb_model_table_v1.parquet, which has NO builder cell anywhere in this
# notebook (confirmed via a full-notebook grep -- only ever read, never
# written). It's an orphaned one-off file, frozen since 2026-03-29, that
# nothing in the current pipeline could regenerate. Worse: that same
# orphaned file turned out to be the PRIMARY data source for the `mlb`
# variable used throughout most of this notebook (Cell 2's loader tries
# mlb_model_table_v1.parquet FIRST, before any fresher source), so the
# staleness wasn't isolated to this one cell -- it was silently upstream
# of a large part of the pipeline.
#
# Rather than trying to fix that whole chain (would mean re-plumbing Cell
# 2's load priority and every table downstream of it), this cell now
# builds its own totals-calibration dataset directly from
# mlb_elo_cache_2025.json -- the SAME real, continuously-refreshed game
# log already used by the SP-ELO backtest earlier this project (confirmed
# reliable, currently covers 2025-03-27 through the live current date,
# 4,000+ games, real away_runs/home_runs per game). This sidesteps the
# orphaned-file problem entirely by using a data source already proven to
# stay current, computing the same trailing-10-game runs-for/against
# proxy the old version used, chronologically with no leakage.
# =========================================================
import pandas as pd
import numpy as np
import os
import json
from collections import defaultdict, deque

print("=" * 65)
print("MLB TOTALS CALIBRATION -- PROJECTED VS ACTUAL BY BUCKET")
print("=" * 65)

_ELO_CACHE_PATH = os.path.join(os.getcwd(), "mlb_elo_cache_2025.json")

if not os.path.exists(_ELO_CACHE_PATH):
    print(f"mlb_elo_cache_2025.json not found at {_ELO_CACHE_PATH} -- skipping.")
else:
    with open(_ELO_CACHE_PATH, encoding="utf-8") as _tc_f:
        _tc_gc = json.load(_tc_f)

    _tc_df = pd.DataFrame([v for v in _tc_gc.values() if v.get("game_date")])
    _tc_df = _tc_df.dropna(subset=["away_runs", "home_runs", "away_team", "home_team"]).copy()
    _tc_df = _tc_df.sort_values("game_date").reset_index(drop=True)

    _TC_PARK_FACTOR = {
        "cubs": 1.04, "whitesox": 0.96, "redsox": 1.01, "yankees": 0.99, "mets": 0.97,
        "dodgers": 0.96, "angels": 0.99, "sfgiants": 0.90, "rangers": 1.05, "mariners": 0.97,
        "astros": 1.01, "athletics": 0.96, "rays": 0.94, "orioles": 1.04, "bluejays": 1.00,
        "guardians": 0.98, "twins": 1.01, "tigers": 0.97, "royals": 0.99, "brewers": 0.97,
        "cardinals": 0.99, "pirates": 0.96, "reds": 1.04, "braves": 1.01, "marlins": 0.94,
        "phillies": 1.03, "nationals": 0.99, "dbacks": 1.05, "rockies": 1.32, "padres": 0.92,
    }

    def _tc_norm_team(s):
        s = str(s).strip().lower()
        w = s.replace("the ", "").strip().split()
        return w[-1] if w else s

    # Chronological trailing-10-game runs-for/runs-against per team, computed
    # BEFORE each game is added to its own history (no leakage) -- same
    # "10-game rolling average" concept the old parquet-based version used,
    # just computed here directly from the raw game log instead of reading a
    # pre-built column.
    _TC_WINDOW = 10
    _TC_MIN_SAMPLE = 5  # require at least this many prior games before trusting the average
    _tc_hist = defaultdict(lambda: deque(maxlen=_TC_WINDOW))

    _tc_rows = []
    for _, _g in _tc_df.iterrows():
        _aw, _hm = _g["away_team"], _g["home_team"]
        _ar, _hr = float(_g["away_runs"]), float(_g["home_runs"])
        _a_hist, _h_hist = _tc_hist[_aw], _tc_hist[_hm]
        if len(_a_hist) >= _TC_MIN_SAMPLE and len(_h_hist) >= _TC_MIN_SAMPLE:
            _a_for = sum(x[0] for x in _a_hist) / len(_a_hist)
            _a_against = sum(x[1] for x in _a_hist) / len(_a_hist)
            _h_for = sum(x[0] for x in _h_hist) / len(_h_hist)
            _h_against = sum(x[1] for x in _h_hist) / len(_h_hist)
            _pf = _TC_PARK_FACTOR.get(_tc_norm_team(_hm), 1.00)
            _proj_away = (_a_for + _h_against) / 2
            _proj_home = (_h_for + _a_against) / 2
            _proj_total = (_proj_away + _proj_home) * _pf
            _tc_rows.append({
                "date": _g["game_date"], "proj_total": _proj_total,
                "total_runs": _ar + _hr, "bias": _proj_total - (_ar + _hr),
            })
        _tc_hist[_aw].append((_ar, _hr))
        _tc_hist[_hm].append((_hr, _ar))

    _tc_df = pd.DataFrame(_tc_rows)

    print(f"Usable historical games (10-game rolling avg available): {len(_tc_df)}")
    print(f"  Date range: {_tc_df['date'].min()} to {_tc_df['date'].max()}")
    print(f"  2026 games included: {int((_tc_df['date'] >= '2026-01-01').sum())}")
    print(f"Overall mean bias (proj - actual) : {_tc_df['bias'].mean():+.3f} runs")
    print(f"Overall median bias               : {_tc_df['bias'].median():+.3f} runs")
    print(f"Mean absolute error                : {_tc_df['bias'].abs().mean():.3f} runs")
    print(f"Mean projected total               : {_tc_df['proj_total'].mean():.2f}")
    print(f"Mean actual total                  : {_tc_df['total_runs'].mean():.2f}")
    print()

    _tc_bins = [0, 7, 7.5, 8, 8.5, 9, 9.5, 10, 100]
    _tc_labels = ["<7", "7-7.5", "7.5-8", "8-8.5", "8.5-9", "9-9.5", "9.5-10", "10+"]
    _tc_df["proj_bucket"] = pd.cut(_tc_df["proj_total"], bins=_tc_bins, labels=_tc_labels)

    _tc_calib = (
        _tc_df.groupby("proj_bucket", observed=True)
        .agg(
            n=("bias", "size"),
            mean_proj=("proj_total", "mean"),
            mean_actual=("total_runs", "mean"),
            mean_bias=("bias", "mean"),
        )
        .reset_index()
    )

    print(f"{'Proj bucket':<12} {'n':>6} {'Mean proj':>10} {'Mean actual':>12} {'Bias':>8}  Signal")
    print("-" * 65)
    for _, row in _tc_calib.iterrows():
        if row["n"] == 0:
            continue
        _signal = ("UNDER-PROJECTS" if row["mean_bias"] < -0.5
                    else ("OVER-PROJECTS" if row["mean_bias"] > 0.5 else "calibrated"))
        print(f"{row['proj_bucket']:<12} {int(row['n']):>6} {row['mean_proj']:>10.2f} "
              f"{row['mean_actual']:>12.2f} {row['mean_bias']:>+8.2f}  {_signal}")

    print()
    print("Recommendation:")
    _low = _tc_calib[_tc_calib["proj_bucket"].isin(["<7", "7-7.5", "7.5-8"])]
    if _low["n"].sum() > 0:
        _low_bias = (_low["mean_bias"] * _low["n"]).sum() / _low["n"].sum()
        print(f"  Projections under 8.0: mean bias {_low_bias:+.2f}R across {int(_low['n'].sum())} games")
        if _low_bias < -0.75:
            print("  -> Historically, LOW projections have run meaningfully UNDER what actually")
            print("     happens -- a projected total in this range should NOT be trusted at face")
            print("     value for an Under play; treat edges here as inflated until the live")
            print("     formula (SP/bullpen/weather-specific) is checked the same way.")
    _high = _tc_calib[_tc_calib["proj_bucket"] == "10+"]
    if _high["n"].sum() > 0:
        _high_bias = (_high["mean_bias"] * _high["n"]).sum() / _high["n"].sum()
        print(f"  Projections 10+: mean bias {_high_bias:+.2f}R across {int(_high['n'].sum())} games")
        if _high_bias > 0.75:
            print("  -> Historically, HIGH projections have run meaningfully OVER what actually")
            print("     happens -- same caution applies to Over plays built on very high totals.")

    globals()["mlb_totals_calibration"] = _tc_calib.copy()
    print("\nSaved: mlb_totals_calibration")


MLB TOTALS CALIBRATION -- PROJECTED VS ACTUAL BY BUCKET


Usable historical games (10-game rolling avg available): 4779
  Date range: 2025-04-01 to 2026-09-27
  2026 games included: 2430
Overall mean bias (proj - actual) : +0.023 runs
Overall median bias               : +0.556 runs
Mean absolute error                : 3.678 runs
Mean projected total               : 8.93
Mean actual total                  : 8.90

Proj bucket       n  Mean proj  Mean actual     Bias  Signal
-----------------------------------------------------------------
<7              297       6.58         8.46    -1.88  UNDER-PROJECTS
7-7.5           372       7.27         8.34    -1.07  UNDER-PROJECTS
7.5-8           571       7.75         8.48    -0.73  UNDER-PROJECTS
8-8.5           704       8.26         8.47    -0.20  calibrated
8.5-9           784       8.76         8.80    -0.05  calibrated
9-9.5           607       9.24         9.33    -0.09  calibrated
9.5-10          521       9.74         9.13    +0.60  OVER-PROJECTS
10+             923      11.07         9.55  

In [86]:
# =========================================================
# CELL Ã¢â‚¬â€ PRE-POST CHECKLIST
# Run this BEFORE posting the card.
# Checks:
#   1. Starter changes (MLB API probable pitchers vs model)
#   2. Injury / scratch news (ESPN MLB news feed, last 24h)
#   3. IL transactions (MLB API, last 48h)
# Output: GO / CAUTION / SKIP per play
# =========================================================
import requests
import pandas as pd
import numpy as np
import re
from datetime import datetime, timedelta, timezone

print("=" * 65)
print("PRE-POST CHECKLIST")
print("=" * 65)

# Shift by 1 day when run after 8 PM so odds are fetched for tomorrow
_espn_base   = datetime.now(timezone.utc)
_espn_offset = timedelta(days=1) if _espn_base.hour >= 20 else timedelta(0)
TODAY     = (_espn_base + _espn_offset).strftime("%Y-%m-%d")
YESTERDAY = (_espn_base + _espn_offset - timedelta(days=1)).strftime("%Y-%m-%d")
TWO_AGO   = (_espn_base + _espn_offset - timedelta(days=2)).strftime("%Y-%m-%d")

# Ã¢â€â‚¬Ã¢â€â‚¬ helpers Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬
# 2026-08-31 BUG FIX (confirmed live via curl): every ESPN call from this
# network is edge-blocked by Akamai with a 403 "Access Denied" page
# (Server: AkamaiGHost -- ESPN's own app never sees the request), not a
# rate limit. This helper is shared by both ESPN and non-ESPN (MLB Stats
# API) URLs, so the relay retry only fires for espn.com hosts -- a 403
# from any other source still returns None exactly as before.
def _get(url, timeout=8):
    try:
        r = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=timeout)
        if r.status_code == 200:
            return r.json()
        if r.status_code == 403 and "espn.com" in url:
            rr = requests.get(f"https://r.jina.ai/{url}", timeout=timeout + 20)
            if rr.status_code == 200:
                body = rr.text
                idx = body.find("Markdown Content:")
                json_text = body[idx + len("Markdown Content:"):].strip() if idx >= 0 else body
                import json as _json_c85
                return _json_c85.loads(json_text)
    except Exception:
        pass
    return None

def _norm(s):
    return re.sub(r"\s+", " ", str(s).strip().lower())

def _name_match(a, b):
    """Loose name match: last name sufficient."""
    a, b = _norm(a), _norm(b)
    if a == b:
        return True
    a_last = a.split()[-1] if a else ""
    b_last = b.split()[-1] if b else ""
    return a_last and b_last and a_last == b_last

# Ã¢â€â‚¬Ã¢â€â‚¬ locate today's card Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬
_card = None
for _name in ["best_mlb_board", "top_market", "mlb_card_scored"]:
    if _name in globals() and isinstance(globals()[_name], pd.DataFrame) and len(globals()[_name]) > 0:
        _raw = globals()[_name].copy()
        if _name == "mlb_card_scored":
            _pass = pd.Series([False]*len(_raw))
            if "passes_edge_rule" in _raw.columns: _pass |= (_raw["passes_edge_rule"] == 1)
            if "model_only_pick" in _raw.columns: _pass |= (_raw["model_only_pick"] == 1)
            _raw = _raw[_pass].copy() if _pass.any() else pd.DataFrame()
            _dc = [c for c in ["away_team","home_team"] if c in _raw.columns]
            if _dc and len(_raw): _raw = _raw.drop_duplicates(subset=_dc, keep="first")
        if len(_raw) == 0: continue
        _card = _raw
        print(f"Card source: {_name}  ({len(_card)} plays)")
        break

if _card is None:
    print("No card found. Run Cell 74 first.")
else:
    # Get team and game info from card
    _away_col = next((c for c in ["away_team","away"] if c in _card.columns), None)
    _home_col = next((c for c in ["home_team","home"] if c in _card.columns), None)
    _gid_col  = next((c for c in ["game_id","gamepk","game_pk"] if c in _card.columns), None)
    _ap_col   = next((c for c in ["away_pitcher_final_name","away_pitcher_name","away_starter"] if c in _card.columns), None)
    _hp_col   = next((c for c in ["home_pitcher_final_name","home_pitcher_name","home_starter"] if c in _card.columns), None)

    _play_teams = set()
    _play_games = []
    for _, _r in _card.iterrows():
        if _away_col: _play_teams.add(_norm(str(_r.get(_away_col, ""))))
        if _home_col: _play_teams.add(_norm(str(_r.get(_home_col, ""))))
        _play_games.append({
            "game_id":   _r.get(_gid_col)  if _gid_col  else None,
            "away_team": _r.get(_away_col) if _away_col else "",
            "home_team": _r.get(_home_col) if _home_col else "",
            "model_away_sp": _r.get(_ap_col) if _ap_col else None,
            "model_home_sp": _r.get(_hp_col) if _hp_col else None,
            "bet_team":  _r.get("bet_team", ""),
            "edge":      _r.get("edge", 0),
        })

    # Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬
    # CHECK 1: Starter changes via MLB Stats API
    # Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬
    print("\n" + "Ã¢â€â‚¬" * 55)
    print("CHECK 1: Starters (MLB API probable pitchers)")
    print("Ã¢â€â‚¬" * 55)

    _sched_url = (
        f"https://statsapi.mlb.com/api/v1/schedule"
        f"?sportId=1&date={TODAY}&hydrate=probablePitchers,linescore"
    )
    _sched = _get(_sched_url)
    _api_starters = {}  # game_id -> {away_sp, home_sp}

    if _sched:
        for _date_obj in _sched.get("dates", []):
            for _game in _date_obj.get("games", []):
                _gid = _game.get("gamePk")
                _teams = _game.get("teams", {})
                _asp = _teams.get("away",  {}).get("probablePitcher", {}).get("fullName")
                _hsp = _teams.get("home",  {}).get("probablePitcher", {}).get("fullName")
                _api_starters[_gid] = {"away_sp": _asp, "home_sp": _hsp}

    _starter_flags = []
    for _g in _play_games:
        _gid = _g["game_id"]
        _api = _api_starters.get(int(_gid)) if _gid else None
        _flag = False
        _notes = []

        if _api:
            _a_api = _api.get("away_sp")
            _h_api = _api.get("home_sp")
            _a_model = _g["model_away_sp"]
            _h_model = _g["model_home_sp"]

            if _a_api:
                print(f"  {_g['away_team']} SP: {_a_api}", end="")
                if _a_model and not _name_match(_a_api, str(_a_model)):
                    print(f"  Ã¢Å¡Â   MODEL HAD: {_a_model}  Ã¢â€ Â CHANGED")
                    _notes.append(f"Away SP changed: {_a_model} Ã¢â€ â€™ {_a_api}")
                    _flag = True
                else:
                    print("  Ã¢Å“â€œ")
            if _h_api:
                print(f"  {_g['home_team']} SP: {_h_api}", end="")
                if _h_model and not _name_match(_h_api, str(_h_model)):
                    print(f"  Ã¢Å¡Â   MODEL HAD: {_h_model}  Ã¢â€ Â CHANGED")
                    _notes.append(f"Home SP changed: {_h_model} Ã¢â€ â€™ {_h_api}")
                    _flag = True
                else:
                    print("  Ã¢Å“â€œ")
        else:
            print(f"  {_g['away_team']} @ {_g['home_team']}: no API data")

        _starter_flags.append({"game_id": _gid, "sp_changed": _flag, "sp_notes": _notes})
    if not _api_starters:
        print("  (MLB schedule API returned no games Ã¢â‚¬â€ starter check skipped)")

    # -- CHECK 1B: ESPN probable pitchers (independent second source) --
    # Cross-references against ESPN's own probables feed, not just a second
    # read of MLB Stats API. Flags disagreement between the two sources --
    # this catches a rotation change MLB's API hasn't reflected yet even
    # when CHECK 1 (also MLB-API-sourced) sees no change.
    print("\n" + "-" * 55)
    print("CHECK 1B: ESPN probable pitchers (independent source)")
    print("-" * 55)

    _espn_sb = _get("https://site.api.espn.com/apis/site/v2/sports/baseball/mlb/scoreboard")
    _espn_probables = {}  # (away_norm, home_norm) -> {away_sp, home_sp}

    def _espn_team_key(nm):
        return _norm(str(nm))

    if _espn_sb:
        for _ev in _espn_sb.get("events", []):
            _comps = _ev.get("competitions", []) or []
            if not _comps:
                continue
            _comp = _comps[0]
            _cts = _comp.get("competitors", []) or []
            _a_blk = next((c for c in _cts if str(c.get("homeAway","")).lower() == "away"), None)
            _h_blk = next((c for c in _cts if str(c.get("homeAway","")).lower() == "home"), None)
            if not _a_blk or not _h_blk:
                continue
            def _ext_sp(blk):
                _probs = blk.get("probables") or []
                if _probs:
                    _ath = (_probs[0] or {}).get("athlete", {}) or {}
                    return _ath.get("displayName")
                return None
            _a_nm = ((_a_blk.get("team") or {}).get("displayName", ""))
            _h_nm = ((_h_blk.get("team") or {}).get("displayName", ""))
            _espn_probables[(_espn_team_key(_a_nm), _espn_team_key(_h_nm))] = {
                "away_sp": _ext_sp(_a_blk),
                "home_sp": _ext_sp(_h_blk),
            }

    _espn_flags = []
    for _i, _g in enumerate(_play_games):
        _gkey = (_espn_team_key(_g["away_team"]), _espn_team_key(_g["home_team"]))
        _espn_g = _espn_probables.get(_gkey)
        _flag_espn = False
        _notes_espn = []
        if _espn_g:
            _a_espn = _espn_g.get("away_sp")
            _h_espn = _espn_g.get("home_sp")
            _a_model = _g["model_away_sp"]
            _h_model = _g["model_home_sp"]
            if _a_espn:
                print(f"  {_g['away_team']} SP (ESPN): {_a_espn}", end="")
                if _a_model and not _name_match(_a_espn, str(_a_model)):
                    print(f"  [WARN] MODEL HAD: {_a_model}  -> DISAGREES")
                    _notes_espn.append(f"ESPN vs model away SP disagree: {_a_model} vs {_a_espn}")
                    _flag_espn = True
                else:
                    print("  [OK]")
            if _h_espn:
                print(f"  {_g['home_team']} SP (ESPN): {_h_espn}", end="")
                if _h_model and not _name_match(_h_espn, str(_h_model)):
                    print(f"  [WARN] MODEL HAD: {_h_model}  -> DISAGREES")
                    _notes_espn.append(f"ESPN vs model home SP disagree: {_h_model} vs {_h_espn}")
                    _flag_espn = True
                else:
                    print("  [OK]")
        else:
            print(f"  {_g['away_team']} @ {_g['home_team']}: no ESPN probable data")
        _espn_flags.append({"game_id": _g["game_id"], "sp_changed": _flag_espn, "sp_notes": _notes_espn})
    if not _espn_probables:
        print("  (ESPN scoreboard returned no probables -- second-source check skipped)")

    # Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬
    # CHECK 2: ESPN MLB news Ã¢â‚¬â€ last 24h
    # Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬
    print("\n" + "Ã¢â€â‚¬" * 55)
    print("CHECK 2: ESPN MLB news (last 24h)")
    print("Ã¢â€â‚¬" * 55)

    _ALERT_WORDS = [
        "scratch", "scratched", "placed on", "il ", " il,", "injured",
        "day-to-day", "dtd", "out ", "won't start", "will not start",
        "disabled list", "concussion", "surgery", "fracture", "strain", "sprain",
    ]

    _news_url = "https://site.api.espn.com/apis/site/v2/sports/baseball/mlb/news?limit=50"
    _news_data = _get(_news_url)
    _news_flags = []

    if _news_data:
        _articles = _news_data.get("articles", [])
        _cutoff = datetime.now(timezone.utc) - timedelta(hours=24)

        for _art in _articles:
            _pub = _art.get("published", "")
            try:
                _pub_dt = datetime.fromisoformat(_pub.replace("Z", "+00:00"))
                if _pub_dt < _cutoff:
                    continue
            except Exception:
                pass

            _headline = _art.get("headline", "")
            _desc     = _art.get("description", "")
            _text     = (_headline + " " + _desc).lower()

            # Check if article mentions any team in our card
            _team_hit = any(t in _text for t in _play_teams if len(t) > 4)
            if not _team_hit:
                continue

            # Check for alert keywords
            _kw_hit = any(kw in _text for kw in _ALERT_WORDS)
            if _kw_hit:
                _news_flags.append({
                    "headline": _headline,
                    "desc": _desc[:120],
                    "pub": _pub,
                })

    if _news_flags:
        print(f"  Ã¢Å¡Â   {len(_news_flags)} relevant alert(s) found:")
        for _nf in _news_flags:
            print(f"\n  [{_nf['pub'][:16]}] {_nf['headline']}")
            if _nf["desc"]:
                print(f"  {_nf['desc']}")
    else:
        print("  Ã¢Å“â€œ  No injury/scratch news found for today's plays.")

    # Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬
    # CHECK 3: MLB IL transactions (last 48h)
    # Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬
    print("\n" + "Ã¢â€â‚¬" * 55)
    print("CHECK 3: MLB IL transactions (last 48h)")
    print("Ã¢â€â‚¬" * 55)

    _trans_url = (
        f"https://statsapi.mlb.com/api/v1/transactions"
        f"?sportId=1&startDate={TWO_AGO}&endDate={TODAY}"
        f"&limit=200"
    )
    _trans_data = _get(_trans_url)
    _il_flags = []

    if _trans_data:
        for _tx in _trans_data.get("transactions", []):
            # 2026-07-29: same field-name bugs as the season-long SP-staleness
            # gate (cell 88) -- verified live against the real API. "injured
            # list"/"IL" text lives in `description`, never in the generic
            # `typeDesc` bucket ("Status Change", "Recalled", etc.), and the
            # player name is under `person`, not the nonexistent `player` key.
            # This CHECK 3 near-term scratch detector was therefore never
            # matching a real transaction either.
            _type = _tx.get("description", "").lower()
            if "injured list" not in _type and "il" not in _type:
                continue
            _player = _tx.get("person", {}).get("fullName", "")
            _team   = _norm(_tx.get("toTeam", {}).get("name", "") or _tx.get("fromTeam", {}).get("name", ""))
            _date   = _tx.get("date", "")

            # Only flag if team is in our plays
            if any(t in _team for t in _play_teams if len(t) > 4) or any(t in _norm(_player) for t in _play_teams if len(t) > 4):
                _il_flags.append({"player": _player, "team": _team, "type": _type, "date": _date})

    if _il_flags:
        print(f"  Ã¢Å¡Â   {len(_il_flags)} IL move(s) involving today's teams:")
        for _il in _il_flags:
            print(f"  {_il['date']} | {_il['player']} ({_il['team']}) Ã¢â‚¬â€ {_il['type']}")
    else:
        print("  Ã¢Å“â€œ  No IL moves found for today's teams.")

    # Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬
    # FINAL: GO / CAUTION / SKIP per play
    # Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬
    print("\n" + "=" * 65)
    print("VERDICT")
    print("=" * 65)


    for _i, _g in enumerate(_play_games):
        _sp_flag = _starter_flags[_i]["sp_changed"] if _i < len(_starter_flags) else False
        _sp_notes = _starter_flags[_i]["sp_notes"] if _i < len(_starter_flags) else []
        _espn_flag = _espn_flags[_i]["sp_changed"] if _i < len(_espn_flags) else False
        _espn_notes = _espn_flags[_i]["sp_notes"] if _i < len(_espn_flags) else []
        if _espn_flag:
            _sp_flag = True
            _sp_notes = _sp_notes + _espn_notes

        # Per-game news/IL check Ã¢â‚¬â€ only flag the specific game's teams
        _g_teams = {_norm(str(_g["away_team"])), _norm(str(_g["home_team"]))}
        _g_has_news = any(
            any(t in (_nf["headline"] + " " + _nf["desc"]).lower() for t in _g_teams if len(t) > 3)
            for _nf in _news_flags
        )
        _g_has_il = any(
            any(t in _norm(_il.get("team","")) or t in _norm(_il.get("player","")) for t in _g_teams if len(t) > 3)
            for _il in _il_flags
        )
        if _sp_flag:
            _verdict = "SKIP Ã¢â‚¬â€ starter changed, model edge invalidated"
        elif _g_has_news or _g_has_il:
            _verdict = "CAUTION Ã¢â‚¬â€ relevant news for this game"
        else:
            _verdict = "GO"

        _sym = "Ã¢Å“â€¦" if _verdict == "GO" else ("Ã¢ÂÅ’" if "SKIP" in _verdict else "Ã¢Å¡Â  ")
        print(f"\n{_sym}  {_g['away_team']} @ {_g['home_team']}")
        print(f"   Bet: {_g['bet_team']} ML  |  Edge: {_g['edge']:.1%}")
        print(f"   {_verdict}")
        for _n in _sp_notes:
            print(f"   Ã¢â€ â€™ {_n}")


PRE-POST CHECKLIST
No card found. Run Cell 74 first.


In [87]:
# %%
# =========================================================
# CELL -- PLATOON + wOBA SPLITS + LINEUP QUALITY GATE
# 1. Fetches team batting OPS vs LHP / RHP from MLB Stats API
# 2. Detects SP handedness from pitcher IDs already fetched
# 3. Fetches posted lineups and checks for missing key players
# Outputs:
#   mlb_woba_splits    : {team_norm: {'vs_lhp': float, 'vs_rhp': float}}
#   mlb_platoon_adj    : {team_norm: float}  (+/- runs vs average platoon)
#   mlb_lineup_flags   : {team_norm: str}    (warning string or "")
#   mlb_sp_hand        : {pitcher_name_norm: 'L' or 'R'}
# =========================================================

import requests
import pandas as pd
import numpy as np
import unicodedata, re, datetime

print("=" * 70)
print("PLATOON + wOBA SPLITS + LINEUP QUALITY GATE")
print("=" * 70)

_MLB_API = "https://statsapi.mlb.com/api/v1"
_SEASON  = 2026

def _norm(s):
    if not s: return ""
    s = unicodedata.normalize("NFD", str(s))
    s = "".join(c for c in s if unicodedata.category(c) != "Mn")
    return re.sub(r"[^a-z0-9 ]", "", s.lower()).strip()

# ---------------------------------------------------------
# 1) Team batting OPS vs LHP and RHP
# ---------------------------------------------------------
mlb_woba_splits = {}   # team_norm -> {vs_lhp: float, vs_rhp: float}
_LEAGUE_OPS = 0.720    # 2026 MLB average OPS as fallback

for _sitcode, _side in [("vl", "vs_lhp"), ("vr", "vs_rhp")]:
    try:
        _url = (f"{_MLB_API}/teams/stats"
                f"?stats=statSplits&sitCodes={_sitcode}&group=hitting"
                f"&season={_SEASON}&sportId=1")
        _r = requests.get(_url, timeout=12)
        _splits = _r.json().get("stats", [{}])[0].get("splits", [])
        for _sp in _splits:
            _tn = _norm(_sp.get("team", {}).get("name", ""))
            _st = _sp.get("stat", {})
            _ops = _st.get("ops")
            if _tn and _ops:
                if _tn not in mlb_woba_splits:
                    mlb_woba_splits[_tn] = {}
                mlb_woba_splits[_tn][_side] = float(_ops)
    except Exception as _e:
        print(f"  WARN: wOBA split fetch ({_sitcode}) failed: {_e}")

# Compute league average OPS vs each hand
_avg_vs_lhp = np.mean([v["vs_lhp"] for v in mlb_woba_splits.values() if "vs_lhp" in v]) if mlb_woba_splits else _LEAGUE_OPS
_avg_vs_rhp = np.mean([v["vs_rhp"] for v in mlb_woba_splits.values() if "vs_rhp" in v]) if mlb_woba_splits else _LEAGUE_OPS
print(f"  wOBA splits loaded: {len(mlb_woba_splits)} teams | avg vs LHP: {_avg_vs_lhp:.3f} | avg vs RHP: {_avg_vs_rhp:.3f}")

# ---------------------------------------------------------
# 2) SP handedness from pitcher IDs (via Cell 61 data)
# ---------------------------------------------------------
mlb_sp_hand = {}   # pitcher_name_norm -> 'L' or 'R'

_sp_ids = {}
# 2026-09-10 (user audit, found while debugging item 2/BvP below): this used
# to break on the FIRST candidate that was merely a real DataFrame, regardless
# of whether it actually had pitcher_id data. Live-confirmed root cause:
# mlb_today_starters is a real, but PRE-enrichment frame (schedule-stage --
# game_id/away_team/home_team/venue/status, no pitcher_id column at all) that
# exists in globals well before cell 79 merges real starter/pitcher_id data
# into board/projection_board -- being first in this priority list, the old
# code grabbed it, found no id column, silently produced nothing, and never
# tried the next two candidates that actually have the data (cell 79's own
# "FINAL DAILY STARTER ATTACH SUMMARY" confirms board has 5/5 real
# away_pitcher_id/home_pitcher_id rows the same run this printed "0
# pitchers"). Now requires the pitcher_id column to actually exist (and have
# at least one non-null value) before accepting a candidate, falling through
# to the next one otherwise -- same real-data-required standard every other
# split/gate fix in this notebook already applies.
for _src_name in ["mlb_today_starters", "projection_board", "board"]:
    _src = globals().get(_src_name)
    if not isinstance(_src, pd.DataFrame):
        continue
    if not any(f"{_s}_pitcher_id" in _src.columns and _src[f"{_s}_pitcher_id"].notna().any() for _s in ("away", "home")):
        continue
    for _side in ["away", "home"]:
        _id_col  = f"{_side}_pitcher_id"
        _nm_col  = f"{_side}_pitcher_name"
        if _id_col in _src.columns and _nm_col in _src.columns:
            for _, _row in _src.iterrows():
                _pid = _row.get(_id_col)
                _pnm = _row.get(_nm_col)
                if _pid and _pnm and str(_pid) not in ("nan", "None"):
                    _sp_ids[str(int(float(_pid)))] = str(_pnm)
    break

_hand_fetched = 0
for _pid, _pname in _sp_ids.items():
    try:
        _pr = requests.get(f"{_MLB_API}/people/{_pid}", timeout=6)
        _hand = _pr.json().get("people", [{}])[0].get("pitchHand", {}).get("code", "R")
        mlb_sp_hand[_norm(_pname)] = _hand
        _hand_fetched += 1
    except Exception:
        mlb_sp_hand[_norm(_pname)] = "R"   # default RHP

print(f"  SP handedness fetched: {_hand_fetched} pitchers | {mlb_sp_hand}")

# ---------------------------------------------------------
# 3) Platoon adjustment: OPS vs SP hand vs league average
#    adj = (team_ops_vs_hand - league_avg_vs_hand) / league_avg_vs_hand
#    Run impact = adj * avg_runs_per_game (4.5)
#    Capped at +/- 0.5 runs
# ---------------------------------------------------------
mlb_platoon_adj = {}   # team_norm -> float (run delta)
_AVG_RUNS = 4.50
_PLATOON_CAP = 0.50

def _platoon_run_delta(team_norm, sp_hand):
    _splits = mlb_woba_splits.get(team_norm, {})
    _side_key = "vs_lhp" if sp_hand == "L" else "vs_rhp"
    _avg_key  = _avg_vs_lhp if sp_hand == "L" else _avg_vs_rhp
    _team_ops = _splits.get(_side_key)
    if _team_ops is None:
        return 0.0
    _delta = (_team_ops - _avg_key) / max(_avg_key, 0.001) * _AVG_RUNS
    return float(np.clip(_delta, -_PLATOON_CAP, _PLATOON_CAP))

# Will be applied per game in Cell 84 using board pitcher names
print("  Platoon adjustment function ready.")

# ---------------------------------------------------------
# 4) Lineup quality gate — fetch posted lineups
#    Check if lineup quality (mean batter OPS) is significantly
#    below team season average -> flag KEY PLAYER OUT warning
# ---------------------------------------------------------
mlb_lineup_flags = {}   # team_norm -> warning string
# REPLACEMENT-QUALITY INJURY PRICING (2026-09-19, user audit): a separate
# marker set, distinct from mlb_lineup_flags above -- membership here means
# "a real confirmed lineup resolved for this team tonight," independent of
# whether that lineup turned out worse, the same, or better than the season
# average. Cell -- FINAL MLB CARD SELECTOR reads this to decide whether to
# trust THIS section's real replacement-aware OPS gap instead of guessing at
# an injury penalty from each absent player's own OPS with no idea who
# actually replaces them.
mlb_lineup_confirmed_teams = set()   # team_norm -> real confirmed-lineup data exists

# PLATOON VS ACTUAL LINEUP (2026-09-19, user audit): the [PLATOON] flag
# printed later (Cell -- FINAL MLB CARD SELECTOR) ranks teams by SEASON-
# LONG vs-LHP/vs-RHP OPS (mlb_woba_splits, section 1 above) -- built from
# every game a team played, healthy or not. A team can rank top-8 on that
# season number while tonight's actual confirmed lineup is missing exactly
# the players who drove that ranking (real case: BOS ranks #6 vs LHP for
# the season, but tonight's posted lineup is missing Contreras/Yoshida/
# Rafaela/Mead to injury) -- the season rank alone can't tell you whether
# the edge survives contact with tonight's real bench-and-injury lineup.
# Reuses the same confirmed-lineup batter loop below (already fetching
# each batter's own season OPS for the lineup-quality gate) to also pull
# each batter's OWN OPS split vs tonight's specific opposing hand -- same
# statSplits endpoint as section 1, just at the player level instead of
# team level, and only for the 9 (or fewer) players actually starting
# tonight.
mlb_lineup_platoon_ops = {}   # team_norm -> {ops_vs_opp_hand, opp_hand, n}

# ---------------------------------------------------------
# BATTER VS PITCHER (BvP) HISTORY (2026-09-10, user audit, item 2):
# "the single most useful data point... doesn't appear anywhere in the model
# output." Real case validated live against the MLB Stats API before building
# this (statsapi.mlb.com/api/v1/people/{batterId}/stats?stats=vsPlayer&
# opposingPlayerId={pitcherId}&group=hitting -- same base URL/session pattern
# already used everywhere else in this notebook, no new auth): Arozarena vs
# deGrom summed to 1-16 (.063) across the returned per-season splits, Raleigh
# 2-13 (.154), Corey Seager vs Gilbert 11-33 (.333) -- all three matched the
# user's own manual research exactly (a 4th, Pederson vs Gilbert, came out
# 6-20 vs the user's 6-19, off by one AB -- likely a minor scoring-rule edge
# case, not a methodology error). Hooks into the confirmed-lineup fetch right
# below (section 4) since that's the only place in this notebook that already
# has real batter IDs for TONIGHT's specific game -- so, same limitation as
# that lineup-quality gate, this only produces output once a game's lineup is
# actually posted (visible as "LINEUP NOT CONFIRMED" elsewhere on the card
# until then, same honest gap, not silently guessing at a lineup).
# ---------------------------------------------------------
mlb_bvp_flags = {}   # team_norm -> list of {name, avg, k_pct, pa, ab, hits, so, vs_pitcher}
_BVP_MIN_PA = 8       # user-specified floor

_team_pitcher_id = {}   # team_norm -> today's starting pitcher's MLB Stats API person id
# 2026-09-10: same source-selection bug as the SP-handedness loop above (see
# its comment) -- mlb_today_starters is a real DataFrame but pre-dates the
# pitcher_id merge, so it can't just be accepted on isinstance() alone.
for _bvp_src_name in ["mlb_today_starters", "projection_board", "board"]:
    _bvp_src = globals().get(_bvp_src_name)
    if not isinstance(_bvp_src, pd.DataFrame):
        continue
    if not any(f"{_s}_pitcher_id" in _bvp_src.columns and _bvp_src[f"{_s}_pitcher_id"].notna().any() for _s in ("away", "home")):
        continue
    for _bvp_side in ["away", "home"]:
        _bvp_id_col = f"{_bvp_side}_pitcher_id"
        _bvp_tm_col = next((c for c in [f"{_bvp_side}_team", f"{_bvp_side}_team_name", _bvp_side]
                             if c in _bvp_src.columns), None)
        if _bvp_id_col in _bvp_src.columns and _bvp_tm_col:
            for _, _bvp_row in _bvp_src.iterrows():
                _bvp_pid = _bvp_row.get(_bvp_id_col)
                _bvp_tnm = _bvp_row.get(_bvp_tm_col)
                if _bvp_pid and _bvp_tnm and str(_bvp_pid) not in ("nan", "None"):
                    try:
                        _team_pitcher_id[_norm(str(_bvp_tnm))] = str(int(float(_bvp_pid)))
                    except (TypeError, ValueError):
                        pass
    break
print(f"  BvP: {len(_team_pitcher_id)} team(s) with a resolved starting-pitcher id")

def _bvp_fetch(batter_id, pitcher_id):
    """Sums across every season split the vsPlayer endpoint returns (career
    total vs this specific pitcher) -- verified live this matches the
    conventional 'BvP' figure (see module-level note above)."""
    try:
        _r = requests.get(
            f"{_MLB_API}/people/{batter_id}/stats",
            params={"stats": "vsPlayer", "opposingPlayerId": pitcher_id, "group": "hitting"},
            headers=_headers if "_headers" in globals() else {"User-Agent": "Mozilla/5.0"},
            timeout=6)
        _stats_block = _r.json().get("stats", [{}])[0]
        # BUG FIX (MLB-BvP-2, 2026-09-24 user audit): live-verified this
        # endpoint has a real fallback quirk -- when the requested batter has
        # NO real career history vs this pitcher, MLB Stats API does not
        # return an empty split list. It silently returns a breakdown of
        # OTHER batters' career history vs the same pitcher instead,
        # completely ignoring the batter_id in the URL (the top-level
        # "type" field is NOT a reliable signal here -- confirmed live it
        # reads "vsPlayerTotal" in BOTH the real and the fallback case).
        # Confirmed live: querying this for two different real batters
        # (Antonacci, Teel) with no history vs the same pitcher returned
        # byte-identical 11-split responses whose own "batter" field named
        # a THIRD player (Tristan Peters) -- not either requested batter.
        # The old code summed these splits blindly, so every batter with no
        # real matchup history against a given pitcher got the exact same
        # fabricated 'BvP' line printed under their own name on the card.
        # The one reliable signal is each split's own "batter" id -- a
        # genuine response's splits are always about the requested batter
        # (verified live against Judge vs Cole), the fallback's never are.
        _splits = [_s for _s in _stats_block.get("splits", [])
                   if str((_s.get("batter") or {}).get("id", "")) == str(batter_id)]
        if not _splits:
            return None
        _ab = sum(int(_s.get("stat", {}).get("atBats", 0) or 0) for _s in _splits)
        _h  = sum(int(_s.get("stat", {}).get("hits", 0) or 0) for _s in _splits)
        _so = sum(int(_s.get("stat", {}).get("strikeOuts", 0) or 0) for _s in _splits)
        _pa = sum(int(_s.get("stat", {}).get("plateAppearances", 0) or 0) for _s in _splits)
        if _pa < _BVP_MIN_PA:
            return None
        return {"ab": _ab, "hits": _h, "so": _so, "pa": _pa,
                "avg": (_h / _ab) if _ab > 0 else 0.0, "k_pct": (_so / _pa) if _pa > 0 else 0.0}
    except Exception:
        return None

_today_str = globals().get("CARD_DATE") or datetime.date.today().strftime("%Y-%m-%d")

try:
    _lu_url = f"{_MLB_API}/schedule?sportId=1&date={_today_str}&hydrate=lineups,team"
    _lu_r   = requests.get(_lu_url, timeout=12)
    _lu_games = _lu_r.json().get("dates", [{}])[0].get("games", [])

    # Fetch season team OPS for comparison baseline
    _team_ops_season = {}
    try:
        _tops_r = requests.get(
            f"{_MLB_API}/teams/stats?stats=season&group=hitting&season={_SEASON}&sportId=1",
            timeout=10)
        for _sp in _tops_r.json().get("stats", [{}])[0].get("splits", []):
            _tn2 = _norm(_sp.get("team", {}).get("name", ""))
            _ops2 = _sp.get("stat", {}).get("ops")
            if _tn2 and _ops2:
                _team_ops_season[_tn2] = float(_ops2)
    except Exception:
        pass

    _lineup_games_found = 0
    for _lg in _lu_games:
        for _side in ["away", "home"]:
            _team_name = _lg.get("teams", {}).get(_side, {}).get("team", {}).get("name", "")
            _tn = _norm(_team_name)
            _batters = _lg.get("lineups", {}).get(f"{_side}Players", [])
            if not _batters:
                continue
            _lineup_games_found += 1
            _opp_side = "home" if _side == "away" else "away"
            _opp_team_name = _lg.get("teams", {}).get(_opp_side, {}).get("team", {}).get("name", "")
            _opp_pid = _team_pitcher_id.get(_norm(_opp_team_name))
            # Opposing SP's hand, reused from section 2's already-fetched
            # mlb_sp_hand/_sp_ids (no new API call) -- lets the platoon check
            # below ask each of tonight's confirmed batters for their OWN OPS
            # split against the SAME hand the [PLATOON] flag cares about.
            _opp_sp_name_for_hand = _sp_ids.get(_opp_pid, "") if _opp_pid else ""
            _opp_hand = mlb_sp_hand.get(_norm(_opp_sp_name_for_hand), "R") if _opp_sp_name_for_hand else "R"
            _opp_sitcode = "vl" if _opp_hand == "L" else "vr"
            # Fetch OPS for each batter (+ BvP vs tonight's opposing starter, item 2)
            _batter_ops = []
            _batter_platoon_ops = []
            _bvp_hits = []
            for _btr in _batters[:9]:
                _bid = _btr.get("id")
                if not _bid:
                    continue
                try:
                    _bsr = requests.get(
                        f"{_MLB_API}/people/{_bid}/stats?stats=season&group=hitting&season={_SEASON}",
                        timeout=5)
                    _bops = _bsr.json().get("stats", [{}])[0].get("splits", [{}])
                    _bops = _bops[0].get("stat", {}).get("ops", 0) if _bops else 0
                    _batter_ops.append(float(_bops or 0))
                except Exception:
                    pass
                # PLATOON VS ACTUAL LINEUP (2026-09-19, user audit): same statSplits
                # call as section 1's team-wide vs_lhp/vs_rhp, just for this ONE
                # confirmed batter -- lets the [PLATOON] flag check whether tonight's
                # real lineup (not the season-long team average) actually supports
                # the structural edge it prints.
                try:
                    _bpr = requests.get(
                        f"{_MLB_API}/people/{_bid}/stats?stats=statSplits&sitCodes={_opp_sitcode}&group=hitting&season={_SEASON}",
                        timeout=5)
                    _bp_splits = _bpr.json().get("stats", [{}])[0].get("splits", [{}])
                    _bp_ops = _bp_splits[0].get("stat", {}).get("ops") if _bp_splits else None
                    if _bp_ops:
                        _batter_platoon_ops.append(float(_bp_ops))
                except Exception:
                    pass
                if _opp_pid:
                    _bvp = _bvp_fetch(_bid, _opp_pid)
                    if _bvp:
                        _bvp_hits.append({"name": _btr.get("fullName", "?"), **_bvp})

            if _bvp_hits:
                mlb_bvp_flags[_tn] = sorted(_bvp_hits, key=lambda x: x["pa"], reverse=True)

            # Needs 5+ of the 9 confirmed starters to have a real vs-hand split --
            # looser than the season-OPS gate's 6 (platoon splits are a thinner
            # per-player sample, especially early season) but still real data,
            # never a guess. Independent of the season-OPS gate below -- a lineup
            # can clear this one and still fail (or pass) that one.
            if len(_batter_platoon_ops) >= 5:
                mlb_lineup_platoon_ops[_tn] = {
                    "ops_vs_opp_hand": float(np.mean(_batter_platoon_ops)),
                    "opp_hand": _opp_hand,
                    "n": len(_batter_platoon_ops),
                }

            if len(_batter_ops) < 6:
                continue  # not enough data

            _lineup_ops  = np.mean(_batter_ops)
            _season_ops  = _team_ops_season.get(_tn, _LEAGUE_OPS)
            _ops_gap     = _lineup_ops - _season_ops
            mlb_lineup_confirmed_teams.add(_tn)

            # REPLACEMENT-QUALITY INJURY PRICING (2026-09-19, user audit --
            # "The model currently treats every injury as the full OPS-derived
            # run penalty regardless of what the replacement looks like
            # ... systematically overstates injury tailwinds because it assumes
            # replacement by a league-average player rather than the actual
            # available substitute.") _ops_gap above is ALREADY the real,
            # replacement-aware number -- it's the CONFIRMED lineup's own actual
            # OPS (whoever is really batting tonight, replacement included)
            # against the team's healthy-season baseline. Previously this only
            # got used past a -0.050 trigger, and even then applied a flat 0.40R
            # regardless of how large the real gap was; the flat number is gone
            # now -- scaled to the actual observed gap so a great backup and a
            # replacement-level one don't get priced the same. Slope (1.75R per
            # 1.0 OPS point) matches the implicit scale already baked into the
            # absent-player OPS-bucket table below (Elite .900+ = 0.35R, a 0.20
            # OPS band above the Average floor -- 0.35/0.20 = 1.75), so this isn't
            # a new made-up constant, just the existing one applied to the real
            # gap instead of each absent player's own OPS.
            _RUNS_PER_OPS_POINT = 1.75
            _IS_LINEUP_MAX_PENALTY = 1.10  # same ceiling as the absent-player method's cap
            _lineup_penalty = min(_IS_LINEUP_MAX_PENALTY, max(0.0, -_ops_gap) * _RUNS_PER_OPS_POINT)
            if _lineup_penalty > 0:
                mlb_lineup_flags[_tn] = {"msg": f"[LINEUP DOWNGRADE: confirmed lineup OPS "
                    f"{_ops_gap:+.3f} vs season avg (replacement-aware, real confirmed batters) "
                    f"-- -{_lineup_penalty:.2f}R]", "penalty": round(_lineup_penalty, 2)}

    print(f"  Lineup gate: {_lineup_games_found} lineups posted | {len(mlb_lineup_flags)} warning(s) flagged")
    for _tn, _flag in mlb_lineup_flags.items():
        print(f"    {_tn}: {_flag['msg'] if isinstance(_flag, dict) else _flag}")
    _bvp_hitter_ct = sum(len(v) for v in mlb_bvp_flags.values())
    print(f"  BvP: {_bvp_hitter_ct} hitter(s) with {_BVP_MIN_PA}+ career PA vs tonight's confirmed opposing starter")

except Exception as _le:
    print(f"  WARN: lineup fetch failed ({_le}) — gate skipped")

print("  Platoon + lineup gate ready.")


PLATOON + wOBA SPLITS + LINEUP QUALITY GATE


  wOBA splits loaded: 30 teams | avg vs LHP: 0.712 | avg vs RHP: 0.722
  SP handedness fetched: 2 pitchers | {'sean burke': 'R', 'gavin williams': 'R'}
  Platoon adjustment function ready.
  BvP: 2 team(s) with a resolved starting-pitcher id


  Lineup gate: 0 lineups posted | 0 warning(s) flagged
  BvP: 0 hitter(s) with 8+ career PA vs tonight's confirmed opposing starter
  Platoon + lineup gate ready.


In [88]:
globals()['mlb_ps_bulk_arm'] = {}
globals()['mlb_ps_role_unrated'] = {}
# %%
# =========================================================
# CELL â€” FINAL MLB CARD SELECTOR
# Predicted scores via park factor + live weather + team/starter averages
# =========================================================

import pandas as pd
import numpy as np
import requests
import math  # BUG FIX (2026-08-26): this cell uses math.log/math.isnan etc.
             # starting at line ~3100, but never imported math itself -- it
             # only worked because the (now-removed) MLB Player Props Framework
             # cell happened to run earlier in the same kernel and imported math
             # as a side effect. Same "Notebook Cell Import Isolation" bug pattern
             # already fixed once in WNBAMODEL.ipynb -- don't trust an earlier
             # cell's import, this cell needs its own.

print("=" * 80)
print("FINAL MLB CARD SELECTOR")
print("=" * 80)

# Session-opening bullpen-data check -- this used to only appear as an inline
# warning buried ~1300 lines into this cell's output (right before the grade-
# capping logic that actually uses it), easy to miss above a wall of ELO/SP/
# score-projection prints. This is a session-wide constraint (grade ceiling
# A- on every play today) that belongs in front of anything else, not mixed
# into the middle of setup output. The detailed version still prints later,
# right where the grade cap is actually applied -- this is a loud early
# heads-up, not a replacement for it.
if not ('mlb_bp_risk_score' in globals() and bool(mlb_bp_risk_score)):
    print()
    print("#" * 80)
    print("#  BULLPEN RISK SCORE NOT LOADED -- run CELL BULLPEN before this session")
    print("#  Every play today will be capped at grade A- regardless of criteria met.")
    print("#" * 80)
    print()

# ---------------------------------------------------------
# 0) locate usable scored board
# ---------------------------------------------------------
board = None
board_source = None

for _name in ["scored_mlb_board", "mlb_scored_board", "projection_board", "board", "mlb_board"]:
    if _name in globals() and isinstance(globals()[_name], pd.DataFrame) and len(globals()[_name]) > 0:
        board = globals()[_name].copy()
        board_source = _name
        break

if board is None:
    raise ValueError("No usable MLB board found. Run scoring cell first.")


# HOISTED (audit fix, 2026-08-04): _sp_norm_era is also defined later
# in this cell (inside the pitcher-ELO recency-adjustment block), which
# is too late for the two call sites above it in execution order (the
# xFIP-blend loop and the dual-elite-SP totals suppressor both gate on
# `'_sp_norm_era' in dir()` and were silently no-ops on a first run for
# the same definition-order reason _sp_fetch_recent was). Redefining a
# pure normalization function twice is harmless (unlike _sp_form_cache,
# there's no state to reset), so the original definition below is left
# in place unchanged -- this copy just makes it available earlier too.
import unicodedata as _ud_era, re as _re_era
def _sp_norm_era(s):
    if not s: return ''
    s = _ud_era.normalize('NFD', str(s))
    s = ''.join(c for c in s if _ud_era.category(c) != 'Mn')
    return _re_era.sub(r'[^a-z ]', '', s.lower()).strip()

# HOISTED (audit fix, 2026-08-04) from later in this cell -- see the
# __MLB_SP_RECENT_FORM__ pointer comment below for why. Defining these
# here, before any call site, is what actually fixes first-run ERA/WHIP
# gaps; a fuzzy name-matching fix alone (added 2026-08-03) could not
# have helped because the cache was never populated in the first place.
# __MLB_SP_RECENT_FORM__
# ── SP RECENT FORM FETCH (last 3 starts ERA) ────────────────────────────
import unicodedata as _sp_ud
_sp_form_cache = {}

def _sp_norm(s):
    return _sp_ud.normalize("NFKD", str(s)).encode("ascii","ignore").decode().lower().strip()

# BUG FIX (2026-08-15, MLB SP staleness audit): MLB's inningsPitched field
# uses baseball notation, not decimal -- "5.1" means 5 full innings + 1 out
# (5 and 1/3 innings = 5.333), "5.2" means 5 and 2/3 (5.667), NOT 5.1/5.2 as
# literal tenths. A bare float() cast on this field (as every sum below used
# to do) silently undercounts every partial-inning start, and the error
# compounds across a season's worth of starts -- confirmed live on Grayson
# Rodriguez (2026-08-14 card showed ERA 7.49 / IP 52.9 vs. the real 7.20 /
# ~55 IP, which is the exact direction and rough magnitude this bug
# predicts: understated IP denominator -> overstated ERA). This was flagged
# as an SP-data-staleness issue, but the live fetch above is not stale --
# season=2026 gameLog is pulled fresh every run; the numbers were simply
# computed wrong. Cell 82 already has the correct version of this exact
# helper (`_parse_ip`) for a different data path -- duplicated here (not
# imported from there) so this function stays self-contained if this cell
# is ever run standalone, matching this cell's own established pattern of
# hoisting/duplicating small pure helpers rather than relying on cell order.
def _sp_parse_ip(v):
    try:
        _s = str(v); _p = _s.split(".")
        return int(_p[0]) + (int(_p[1]) if len(_p) > 1 else 0) / 3.0
    except Exception:
        return 0.0

def _sp_fetch_recent(name):
    """Return {era_l3, gs_l3, hot, cold} for a pitcher's last 3 starts, or None."""
    if not name or str(name).strip() in ("", "nan"): return None
    _nk = _sp_norm(name)
    if _nk in _sp_form_cache: return _sp_form_cache[_nk]
    try:
        # Step 1: find player ID
        _q = str(name).replace(" ", "%20")
        _sr = requests.get(
            f"https://statsapi.mlb.com/api/v1/people/search?names={_q}&sportId=1",
            timeout=6
        )
        _people = _sr.json().get("people", []) if _sr.status_code == 200 else []
        # prefer pitcher by position code "1"
        _pid = next(
            (p["id"] for p in _people if p.get("primaryPosition",{}).get("code") == "1"),
            _people[0]["id"] if _people else None
        )
        if not _pid:
            _sp_form_cache[_nk] = None; return None
        # Step 2: game log for 2026
        _lr = requests.get(
            f"https://statsapi.mlb.com/api/v1/people/{_pid}/stats"
            f"?stats=gameLog&group=pitching&season=2026&gameType=R",
            timeout=6
        )
        _splits = _lr.json().get("stats",[{}])[0].get("splits",[]) if _lr.status_code == 200 else []
        _starts = [s for s in _splits if int(s.get("stat",{}).get("gamesStarted",0) or 0) == 1]
        # BUG FIX (2026-09-06, same fix as the sister fetcher in Cell 20 --
        # see that cell's comment for the real Kikuchi case this was found
        # from. Duplicated rather than shared since Cell 20's version is a
        # nested local function -- kept in sync same as the rest of this
        # sister-function pair, e.g. the SP COLD-STREAK SHAPE capture below.
        def _sp_recent_after_gap_87(starts_chrono, date_key):
            if len(starts_chrono) < 2:
                return starts_chrono
            import datetime as _l3gap_dt87
            cut = 0
            for idx in range(1, len(starts_chrono)):
                try:
                    d0 = _l3gap_dt87.date.fromisoformat(str(starts_chrono[idx - 1].get(date_key, ""))[:10])
                    d1 = _l3gap_dt87.date.fromisoformat(str(starts_chrono[idx].get(date_key, ""))[:10])
                    if (d1 - d0).days > 20:
                        cut = idx
                except Exception:
                    pass
            return starts_chrono[cut:]
        _recent = _sp_recent_after_gap_87(_starts, "date")
        _l3 = _recent[-3:]
        _l5  = _recent[-5:]
        _l10 = _recent[-10:]
        if not _l3:
            _sp_form_cache[_nk] = None; return None
        _ip  = sum(_sp_parse_ip(s["stat"].get("inningsPitched", 0)) for s in _l3)
        _er  = sum(int(s["stat"].get("earnedRuns",0) or 0) for s in _l3)
        _hr  = sum(int(s["stat"].get("homeRuns",0) or 0) for s in _l3)
        # SP COLD-STREAK SHAPE (2026-08-31, user audit -- George Kirby): same
        # capture as the sister fetcher in Cell 20 (_get_l3_stats) -- see that
        # cell's comment for why. Kept in sync since _wf_lookup (Final Card
        # WHY loop) can resolve a pitcher through either source.
        _l3_starts_er = [int(s["stat"].get("earnedRuns", 0) or 0) for s in _l3]
        _l3_starts_ip = [round(_sp_parse_ip(s["stat"].get("inningsPitched", 0)), 1) for s in _l3]
        _era = round(_er / _ip * 9, 2) if _ip > 0 else None
        _hr9 = round(_hr / _ip * 9, 2) if _ip > 0 else None
        # WHIP (audit fix, 2026-08-04): whip_l3 was read by _sp_stats() in
        # the pitcher-note display but never actually computed anywhere in
        # this function -- the key was always absent from _res, so every
        # WHIP lookup silently came back None and only ERA ever displayed.
        # (walks + hits) / IP, same L3 window as era_l3 above.
        _h_l3  = sum(float(s["stat"].get("hits",0) or 0) for s in _l3)
        _bb_l3 = sum(float(s["stat"].get("baseOnBalls",0) or 0) for s in _l3)
        _whip_l3 = round((_h_l3 + _bb_l3) / _ip, 2) if _ip > 0 else None
        # Season ERA from all 2026 starts
        _ip_s = sum(_sp_parse_ip(s["stat"].get("inningsPitched", 0)) for s in _starts)
        _er_s = sum(int(s["stat"].get("earnedRuns",0) or 0) for s in _starts)
        _era_s = round(_er_s / _ip_s * 9, 2) if _ip_s > 0 else None
        _h_s  = sum(float(s["stat"].get("hits",0) or 0) for s in _starts)
        _bb_s = sum(float(s["stat"].get("baseOnBalls",0) or 0) for s in _starts)
        _whip_s = round((_h_s + _bb_s) / _ip_s, 2) if _ip_s > 0 else None
        # BUG FIX (2026-08-05, real incident: Drew Anderson's spot start (DET
        # @ SEA) showed season ERA 1.41 on the live card): `_starts` above is
        # filtered to gamesStarted==1 only, so a full-time reliever making a
        # rare spot start has this entire season_era computed from just his
        # 1-2 career starts (Anderson: 2 starts, 1 ER / 6.4 IP = exactly
        # 1.41), with zero signal from his real, far larger relief-appearance
        # record (season ERA 4.24 per ESPN/RotoWire across ~38 outings).
        # `_recency_adj()` below treats this "direct API data" as an override
        # that beats mlb_pitcher_fip_lookup's own era_season (which already
        # includes relief outings via Cell 20's >=1.0 IP filter and reads a
        # much more representative number) -- nulling era_season/whip_season
        # here below a minimum starts sample lets that better fallback win
        # instead of this thin-sample override.
        _MIN_STARTS_FOR_SEASON_ERA = 3
        if len(_starts) < _MIN_STARTS_FOR_SEASON_ERA:
            _era_s = None
            _whip_s = None
            # L3 draws from the same starts-only list (_l3 = _starts[-3:]) --
            # for a pitcher with only 1-2 career starts, L3 IS his season
            # sample, so it leaks the identical unrepresentative number.
            _era = None
            _hr9 = None
            _whip_l3 = None
        # Single most-recent start ERA (2026-08-01) -- distinct from era_l3
        # (3-start rolling average) above, which can dilute a single outlier
        # start (a big spike or an unusually dominant/suppressed outing) out
        # of view. Used by the SP last-start outlier flag in the main card
        # loop below.
        _last_gl = _starts[-1] if _starts else None
        _ip_last = float(_last_gl["stat"].get("inningsPitched", 0) or 0) if _last_gl else 0.0
        _er_last = int(_last_gl["stat"].get("earnedRuns", 0) or 0) if _last_gl else 0
        _era_last = round(_er_last / _ip_last * 9, 2) if _ip_last > 0 else None
        # L5/L10 windows (2026-08-25, user audit -- Will Warren): L3 alone
        # can't tell a real sustained slump (1-5, 5.77 ERA over his last 13
        # appearances) from a normal 3-start blip -- extends the same rolling-
        # window pattern L3 already uses so a longer, more significant
        # deterioration doesn't hide behind a 3-start read.
        _ip_l5  = sum(_sp_parse_ip(s["stat"].get("inningsPitched", 0)) for s in _l5)
        _er_l5  = sum(int(s["stat"].get("earnedRuns", 0) or 0) for s in _l5)
        _era_l5 = round(_er_l5 / _ip_l5 * 9, 2) if _ip_l5 > 0 and len(_l5) >= 5 else None
        _ip_l10  = sum(_sp_parse_ip(s["stat"].get("inningsPitched", 0)) for s in _l10)
        _er_l10  = sum(int(s["stat"].get("earnedRuns", 0) or 0) for s in _l10)
        _era_l10 = round(_er_l10 / _ip_l10 * 9, 2) if _ip_l10 > 0 and len(_l10) >= 5 else None
        _res = {
            "era_l3":     _era,
            "era_season": _era_s,
            "era_last":   _era_last,
            "whip_l3":    _whip_l3,
            "whip_season": _whip_s,
            "ip_last":    round(_ip_last, 1),
            "gs_l3":      len(_l3),
            "era_l5":     _era_l5,
            "gs_l5":      len(_l5),
            "ip_l5":      round(_ip_l5, 1),
            "era_l10":    _era_l10,
            "gs_l10":     len(_l10),
            "ip_l10":     round(_ip_l10, 1),
            "gs_season":  len(_starts),
            "ip_l3":      round(_ip, 1),
            "hr_l3":      _hr,
            "hr9_l3":     _hr9,
            "l3_starts_er": _l3_starts_er,
            "l3_starts_ip": _l3_starts_ip,
            "hot":        _era is not None and _era < 3.00,
            "cold":       _era is not None and _era > 5.50,
        }
        _sp_form_cache[_nk] = _res; return _res
    except Exception:
        _sp_form_cache[_nk] = None; return None

print("  SP recent form fetcher ready")
# ── END SP FORM FETCH ──────────────────────────────────────────────────


# FIX 10: Series sweep alert — fetch series game number from MLB schedule API
# Warns when the pick team is down 0-2 in a 3-game series going into game 3
_series_context = {}
_series_context_by_gid = {}  # str(gamePk) -> same dict, doubleheader/makeup-safe (MLB-DH-1, 2026-09-24 user audit)
_dh_7inn_set = set()  # home team name substrings for scheduledInnings==7 games  # {(away_team_norm, home_team_norm): {seriesGameNumber, seriesStatus}}
try:
    import requests as _srq, datetime as _sdt2
    _mlb_hr2 = _sdt2.datetime.now().hour
    if _mlb_hr2 >= 22:
        _today_s = (_sdt2.date.today() + _sdt2.timedelta(days=1)).strftime("%Y-%m-%d")
    else:
        _today_s = _sdt2.date.today().strftime("%Y-%m-%d")
    _sr = _srq.get(
        f"https://statsapi.mlb.com/api/v1/schedule?sportId=1&date={_today_s}",
        timeout=8
    )
    for _sg in _sr.json().get("dates", [{}])[0].get("games", []):
        _sgn = _sg.get("seriesGameNumber", 1)
        _sgs = _sg.get("gamesInSeries", 3)
        _away_norm = str(_sg.get("teams",{}).get("away",{}).get("team",{}).get("name","")).lower()
        _home_norm = str(_sg.get("teams",{}).get("home",{}).get("team",{}).get("name","")).lower()
        _prob_away_sp = _sg.get("teams",{}).get("away",{}).get("probablePitcher",{}).get("fullName","")
        _prob_home_sp = _sg.get("teams",{}).get("home",{}).get("probablePitcher",{}).get("fullName","")
        _sg_entry = {
            "game_num": _sgn, "games_in_series": _sgs,
            "prob_away_sp": _prob_away_sp, "prob_home_sp": _prob_home_sp,
        }
        _series_context[(_away_norm, _home_norm)] = _sg_entry
        # BUG FIX (MLB-DH-1, 2026-09-24 user audit): the dict above is keyed only
        # by team-pair, so a doubleheader or postponement-makeup (two real games,
        # same two teams, same day) silently overwrites G1's entry with G2's --
        # real incident: TOR@BAL DH, both games ended up labeled "Game 3 of 3"
        # because whichever game the API listed last clobbered the other's series
        # slot. Same class of bug already fixed once for the SP-starter merge
        # (see "doubleheader-safe" merge below, Guardians@Reds 2026-07-28
        # incident) -- applying the same gamePk-keyed fix here so each game in a
        # DH/makeup keeps its own real seriesGameNumber instead of sharing one.
        _sg_gid = _sg.get("gamePk")
        if _sg_gid is not None:
            _series_context_by_gid[str(_sg_gid)] = _sg_entry
        if _sg.get("scheduledInnings", 9) == 7:
            _dh_7inn_set.add(_home_norm)
    try:
        import datetime as _dt
        _yday = (_dt.date.today() - _dt.timedelta(days=1)).strftime("%Y-%m-%d")
        _ysr = requests.get(f"https://statsapi.mlb.com/api/v1/schedule?sportId=1&date={_yday}", timeout=6)
        if _ysr.status_code == 200:
            for _yg in _ysr.json().get("dates",[{}])[0].get("games",[]):
                _ya = str(_yg.get("teams",{}).get("away",{}).get("team",{}).get("name","")).lower()
                _yh = str(_yg.get("teams",{}).get("home",{}).get("team",{}).get("name","")).lower()
                _ys = _yg.get("status",{}).get("abstractGameState","")
                if _ys == "Final":
                    _yar = _yg.get("teams",{}).get("away",{}).get("score",0)
                    _yhr = _yg.get("teams",{}).get("home",{}).get("score",0)
                    for _k in list(_series_context.keys()):
                        _ka,_kh = _k
                        if any(w in _ka for w in _ya.split() if len(w)>3) and any(w in _kh for w in _yh.split() if len(w)>3):
                            _series_context[_k]["g1_away_score"] = _yar
                            _series_context[_k]["g1_home_score"] = _yhr
    except: pass
    print(f"Series context loaded for {len(_series_context)} games")
except Exception as _se:
    print(f"Series context fetch skipped: {_se}")

print(f"Board source : {board_source}  ({len(board)} rows)")

# Merge starter RA from projection_board/mlb_board (built by Cell 72)
# scored_mlb_board is created before Cell 72 runs so it lacks starter RA columns
_SP_RA_COLS = ["away_starter_runs_allowed_10", "home_starter_runs_allowed_10"]
if not any(c in board.columns for c in _SP_RA_COLS):
    for _sp_src_name in ["projection_board", "mlb_board"]:
        _sp_src = globals().get(_sp_src_name)
        if (isinstance(_sp_src, pd.DataFrame)
                and any(c in _sp_src.columns for c in _SP_RA_COLS)
                and "away_team" in _sp_src.columns and "home_team" in _sp_src.columns):
            _sp_merge = [c for c in _SP_RA_COLS + [
                "away_pitcher_name", "home_pitcher_name",
                "away_pitcher_id",   "home_pitcher_id",
            ] if c in _sp_src.columns]
            _key_cols = [c for c in ["away_team", "home_team"] if c in board.columns]
            if _key_cols:
                # dedup projection_board to 1 row per game before merge
                _sp_right = (
                    _sp_src[_key_cols + [c for c in _sp_merge if c not in _key_cols]]
                    .drop_duplicates(subset=_key_cols, keep="last")  # keep latest row — avoids stale series SPs
                )
                board = board.merge(_sp_right, on=_key_cols, how="left", suffixes=("", "_sp"))
                # prefer freshly merged pitcher names over blanks
                for _nc in ["away_pitcher_name", "home_pitcher_name"]:
                    if _nc + "_sp" in board.columns:
                        board[_nc] = board[_nc].where(
                            board[_nc].notna() & (board[_nc] != ""),
                            board[_nc + "_sp"]
                        )
                        board.drop(columns=[_nc + "_sp"], inplace=True)
                n_sp = int(board["away_starter_runs_allowed_10"].notna().sum()) if "away_starter_runs_allowed_10" in board.columns else 0
                print(f"Starter RA merged from {_sp_src_name}: {n_sp}/{len(board)} games have SP data")
                break
else:
    print("Starter RA columns already present in board")

# ---------------------------------------------------------
# Always override pitcher names -- prefer the LIVE, CONFIRMED starter feed
# (Cell 41/42: mlb_today_games_with_actual_starters, pulled straight from
# MLB's game-feed boxscore) over the scheduled/probable-only feed.
# 2026-07-12: this block used to hardcode mlb_today_games_with_starters
# (Cell 40's raw SCHEDULED probable-pitcher feed) as its source -- which
# completely bypassed Cell 41's live-confirmed-starter fetch regardless of
# how correct that cell's own internal priority was. A scratched starter
# (deGrom, Gallen both confirmed scratched 2026-07-12) has an unchanged,
# present "probable" entry -- so this override was re-instating the wrong
# pitcher every single run, no matter what Cells 41/42 had already fetched
# correctly. Falls back to the old scheduled-only source if the live-feed
# cell wasn't run this session, so this doesn't hard-crash on out-of-order
# execution.
# ---------------------------------------------------------
if "mlb_today_games_with_actual_starters" in globals() and isinstance(mlb_today_games_with_actual_starters, pd.DataFrame):
    # 2026-07-28 fix: this merged on team names only, after deduping `_prob` by
    # ["away_team","home_team"] -- which silently collapses a real doubleheader's
    # two distinct starter rows (Cell 41 fetches per real gamePk, correctly
    # differentiated at that stage) down to ONE row, so both DH games get the
    # exact same starter merged onto `board`. Real incident 2026-07-28: Guardians
    # @ Reds (real rain-postponed-makeup DH, confirmed live -- Cecconi/Chase Burns
    # G1, Gavin Williams G2) showed byte-identical model probability and
    # total-runs projection for both games in the Daily Market Snapshot, traced
    # to this exact merge throwing away the per-game distinction Cell 41 already
    # had. `mlb_today_games_with_actual_starters` carries a real `game_id` per
    # row (fetched from each game's own live boxscore, see Cell 41) -- use it
    # when `board` also has a gamePk-equivalent column, so each DH game keeps
    # its own starter. Falls back to the old team-name-only merge when board has
    # no such column at all (unchanged behavior, just can't disambiguate a DH).
    _sp_board_gid_col = next((c for c in ["game_id","gamepk","game_pk","pk"] if c in board.columns), None)
    if _sp_board_gid_col and "game_id" in mlb_today_games_with_actual_starters.columns:
        _prob = mlb_today_games_with_actual_starters[["game_id","away_pitcher_final_name","home_pitcher_final_name"]].copy()
        _prob = _prob.dropna(subset=["away_pitcher_final_name","home_pitcher_final_name"], how="all")
        _prob = _prob.drop_duplicates(subset=["game_id"], keep="last")
        _prob = _prob.rename(columns={"away_pitcher_final_name": "away_pitcher_name",
                                       "home_pitcher_final_name": "home_pitcher_name"})
        # Cast both merge keys to string -- board's gamePk column and Cell 41's
        # `game_id` aren't guaranteed the same dtype (int vs object), and a
        # silent dtype mismatch here would zero out every match rather than
        # error, degrading to worse than the bug this fixes.
        _prob["_sp_gid_str"] = _prob["game_id"].astype(str)
        board["_sp_gid_str"] = board[_sp_board_gid_col].astype(str)
        board = board.drop(columns=["away_pitcher_name","home_pitcher_name"], errors="ignore")
        board = board.merge(_prob.drop(columns=["game_id"]), on="_sp_gid_str", how="left").drop(columns=["_sp_gid_str"])
        _n_sp_overridden = board["away_pitcher_name"].notna().sum()
        import datetime as _dt_spov
        _sp_override_ts = _dt_spov.datetime.now().strftime("%I:%M%p")
        print(f"SP names overridden from live-confirmed starter feed (Cell 41/42), matched by game_id (doubleheader-safe): {_n_sp_overridden}/{len(board)} games  [as of {_sp_override_ts}]")
    else:
        _prob = mlb_today_games_with_actual_starters[["away_team","home_team",
                                                       "away_pitcher_final_name","home_pitcher_final_name"]].copy()
        _prob = _prob.dropna(subset=["away_pitcher_final_name","home_pitcher_final_name"], how="all")
        _prob = _prob.drop_duplicates(subset=["away_team","home_team"], keep="last")
        _prob = _prob.rename(columns={"away_pitcher_final_name": "away_pitcher_name",
                                       "home_pitcher_final_name": "home_pitcher_name"})
        board = board.drop(columns=["away_pitcher_name","home_pitcher_name"], errors="ignore")
        board = board.merge(_prob, on=["away_team","home_team"], how="left")
        _n_sp_overridden = board["away_pitcher_name"].notna().sum()
        import datetime as _dt_spov
        _sp_override_ts = _dt_spov.datetime.now().strftime("%I:%M%p")
        print(f"SP names overridden from live-confirmed starter feed (Cell 41/42), matched by team name -- no gamePk column on board, can't disambiguate a doubleheader: {_n_sp_overridden}/{len(board)} games  [as of {_sp_override_ts}]")
elif "mlb_today_games_with_starters" in globals() and isinstance(mlb_today_games_with_starters, pd.DataFrame):
    print("  [SP WARN] mlb_today_games_with_actual_starters not found -- run Cell 41/42 for live-confirmed starters. Falling back to scheduled probables only (may be stale on a scratch).")
    _prob = mlb_today_games_with_starters[["away_team","home_team",
                                            "away_probable_pitcher_name","home_probable_pitcher_name"]].copy()
    _prob = _prob.dropna(subset=["away_probable_pitcher_name","home_probable_pitcher_name"], how="all")
    _prob = _prob.drop_duplicates(subset=["away_team","home_team"], keep="last")
    _prob = _prob.rename(columns={"away_probable_pitcher_name": "away_pitcher_name",
                                   "home_probable_pitcher_name": "home_pitcher_name"})
    board = board.drop(columns=["away_pitcher_name","home_pitcher_name"], errors="ignore")
    board = board.merge(_prob, on=["away_team","home_team"], how="left")
    _n_sp_overridden = board["away_pitcher_name"].notna().sum()
    import datetime as _dt_spov
    _sp_override_ts = _dt_spov.datetime.now().strftime("%I:%M%p")
    print(f"SP names overridden from scheduled probables (fallback): {_n_sp_overridden}/{len(board)} games  [as of {_sp_override_ts} -- re-run Cell 41/42 for live confirmation]")

# Validate SP assignments (moved 2026-07-13): previously this duplicate-
# pitcher check only ran inside the scheduled-probables FALLBACK branch
# above (elif), so a merge bug on the preferred LIVE-CONFIRMED starter feed
# -- the path used almost every run -- had no safety net at all. Runs here
# unconditionally instead, against whichever branch populated the columns.
if "away_pitcher_name" in board.columns and "home_pitcher_name" in board.columns:
    _sp_team_map = {}
    for _, _bv in board.iterrows():
        for _bside, _bpcol, _btcol in [("away","away_pitcher_name",away_col),("home","home_pitcher_name",home_col)]:
            _bpn = str(_bv.get(_bpcol) or "").strip()
            _btn = str(_bv.get(_btcol) or "").strip()
            if _bpn and _bpn not in ("nan","None",""):
                if _bpn in _sp_team_map and _sp_team_map[_bpn] != _btn:
                    print(f"  [SP WARN] {_bpn} appears for {_sp_team_map[_bpn]} AND {_btn} -- verify assignment")
                _sp_team_map[_bpn] = _btn

# ---------------------------------------------------------
# Upgrade win probabilities: use compressed calibrated model (Cell 51 output)
# if available — it is better than the generic model used by Cell 25
# ---------------------------------------------------------
if "mlb_today_win_preds_new" in globals() and isinstance(mlb_today_win_preds_new, pd.DataFrame):
    _wn = mlb_today_win_preds_new.copy()
    _wn_cols = [c for c in ["away_team", "home_team", "p_home_win_new", "p_away_win_new"] if c in _wn.columns]
    if len(_wn_cols) == 4 and "away_team" in board.columns and "home_team" in board.columns:
        _wn = _wn[_wn_cols].drop_duplicates(subset=["away_team", "home_team"])
        board = board.merge(_wn, on=["away_team", "home_team"], how="left", suffixes=("", "_7f"))
        _has_new = board["p_home_win_new"].notna()
        board.loc[_has_new, "pred_home_win_prob"] = board.loc[_has_new, "p_home_win_new"]
        board.loc[_has_new, "pred_away_win_prob"] = board.loc[_has_new, "p_away_win_new"]
        board.drop(columns=["p_home_win_new", "p_away_win_new"], inplace=True, errors="ignore")
        _nfeat = len(compressed_features) if "compressed_features" in globals() else "?"
        print(f"{_nfeat}-feature model probs merged: {int(_has_new.sum())}/{len(board)} games upgraded")
    else:
        print("compressed model output found but columns missing -- using Cell 25 probs")
else:
    print("mlb_today_win_preds_new not found -- using Cell 25 model probs (re-run Cell 51 for best predictions)")

# ---------------------------------------------------------
# team name normalizer
# ---------------------------------------------------------
def _norm_team(s):
    if not s:
        return ""
    s = str(s).strip().lower()
    _map = {
        "chicago white sox":   "whitesox",
        "boston red sox":      "redsox",
        "new york yankees":    "yankees",
        "new york mets":       "mets",
        "los angeles dodgers": "dodgers",
        "los angeles angels":  "angels",
        "san francisco giants":"sfgiants",
        "texas rangers":       "rangers",
        "seattle mariners":    "mariners",
        "houston astros":      "astros",
        "oakland athletics":   "athletics",
        "athletics":           "athletics",
        "tampa bay rays":      "rays",
        "baltimore orioles":   "orioles",
        "toronto blue jays":   "bluejays",
        "cleveland guardians": "guardians",
        "minnesota twins":     "twins",
        "detroit tigers":      "tigers",
        "kansas city royals":  "royals",
        "chicago cubs":        "cubs",
        "milwaukee brewers":   "brewers",
        "st. louis cardinals": "cardinals",
        "pittsburgh pirates":  "pirates",
        "cincinnati reds":     "reds",
        "atlanta braves":      "braves",
        "miami marlins":       "marlins",
        "philadelphia phillies":"phillies",
        "washington nationals":"nationals",
        "arizona diamondbacks":"dbacks",
        "colorado rockies":    "rockies",
        "san diego padres":    "padres",
    }
    if s in _map:
        return _map[s]
    words = s.replace("the ", "").strip().split()
    return words[-1] if words else s

# _pg_abbr: alias for _norm_team -- converts display name to _PARK_FACTOR key
_pg_abbr = _norm_team

# ---------------------------------------------------------
# 1) merge ESPN odds onto board
# ---------------------------------------------------------
if "mlb_espn_odds" in globals() and isinstance(mlb_espn_odds, pd.DataFrame) and len(mlb_espn_odds) > 0:
    _odds = mlb_espn_odds.copy()
    # Always rebuild keys using this cell's _norm_team (Cell 41 may use different normalization)
    _odds["_hkey"] = _odds["home_team"].apply(_norm_team)
    _odds["_akey"] = _odds["away_team"].apply(_norm_team)
    # Doubleheader-safe match: ESPN returns one event per game, so a real DH shows
    # up as two _odds rows sharing the same team-key pair. Blindly deduping to
    # "first" silently applies game 1's line to game 2 as well. Only collapse
    # pairs that are genuinely single rows; for duplicated pairs, match each
    # board row to its nearest _odds row by start time instead.
    if "event_date" in _odds.columns:
        _odds["_event_ts"] = pd.to_datetime(_odds["event_date"], errors="coerce", utc=True)
    else:
        _odds["_event_ts"] = pd.NaT
    _dh_pairs = _odds.duplicated(subset=["_hkey", "_akey"], keep=False)
    if "home_team" in board.columns and "away_team" in board.columns:
        board["_hkey"] = board["home_team"].apply(_norm_team)
        board["_akey"] = board["away_team"].apply(_norm_team)
        _board_ts = (pd.to_datetime(board["game_datetime_utc"], errors="coerce", utc=True)
                     if "game_datetime_utc" in board.columns else pd.Series(pd.NaT, index=board.index))
        _mcols = [c for c in ["_hkey", "_akey", "home_market_prob", "away_market_prob", "has_odds",
                               "home_ml", "away_ml",
                               "over_under", "over_ml", "under_ml"] if c in _odds.columns]

        if _dh_pairs.any():
            _match_idx = []
            for _bi in board.index:
                _bh, _ba = board.at[_bi, "_hkey"], board.at[_bi, "_akey"]
                _rows = _odds[(_odds["_hkey"] == _bh) & (_odds["_akey"] == _ba)]
                if len(_rows) == 0:
                    _match_idx.append(None)
                elif len(_rows) == 1:
                    _match_idx.append(_rows.index[0])
                else:
                    _bt = _board_ts.at[_bi] if _bi in _board_ts.index else pd.NaT
                    if pd.isna(_bt) or _rows["_event_ts"].isna().all():
                        _match_idx.append(_rows.index[0])
                    else:
                        _match_idx.append((_rows["_event_ts"] - _bt).abs().idxmin())
            for _c in _mcols:
                if _c in ("_hkey", "_akey"):
                    continue
                board[_c] = [(_odds.at[_i, _c] if _i is not None else np.nan) for _i in _match_idx]
        else:
            _odds_dedup = _odds.drop_duplicates(subset=["_hkey", "_akey"], keep="first")
            board = board.merge(_odds_dedup[_mcols], on=["_hkey", "_akey"], how="left")

        board.drop(columns=["_hkey", "_akey"], inplace=True, errors="ignore")
        if "home_market_prob" not in board.columns:
            board["home_market_prob"] = np.nan
            board["away_market_prob"] = np.nan
        if "has_odds" not in board.columns:
            board["has_odds"] = board["home_market_prob"].notna()
        n_matched = int(board["home_market_prob"].notna().sum())
        n_ou = int(board["over_under"].notna().sum()) if "over_under" in board.columns else 0
        n_dh = int(_dh_pairs.sum())
        if n_dh:
            print(f"  Doubleheader-safe odds match used for {n_dh} ESPN odds row(s) (nearest start time)")
        print(f"ESPN odds merged: {n_matched}/{len(board)} games matched  |  O/U available: {n_ou}")
    else:
        board["has_odds"] = False
        print("WARNING: board missing home_team/away_team -- skipping odds merge")
else:
    board["has_odds"] = False
    print("mlb_espn_odds not in memory -- model-only mode")

# ---------------------------------------------------------
# 1b) REAL O/U totals (DraftKings/FanDuel/BetMGM/Pinnacle via The Odds API) --
#     overrides ESPN's ESTIMATED over_under/over_ml/under_ml wherever a real
#     book line exists. ESPN odds are the house's own estimate, not a
#     tradeable market price -- confirmed the totals market is live on this
#     same API key (already used below for props/F5) for the full slate.
# ---------------------------------------------------------
_mlb_real_ou = {}  # (away_norm, home_norm) -> {"line","over_ml","under_ml","book"}
try:
    import requests as _ou_req
    _OU_SHARP_PREF = ["pinnacle", "betmgm", "draftkings", "fanduel"]
    _ou_r = _ou_req.get(
        "https://api.the-odds-api.com/v4/sports/baseball_mlb/odds/",
        params={"apiKey": os.environ.get("ODDS_API_KEY_FALLBACK", ""), "regions": "us",
                "markets": "totals", "oddsFormat": "american",
                "bookmakers": ",".join(_OU_SHARP_PREF)},
        timeout=12)
    if _ou_r.status_code == 200:
        _ou_events = _ou_r.json()
        for _ou_ev in _ou_events:
            _ou_hk = _norm_team(_ou_ev.get("home_team", ""))
            _ou_ak = _norm_team(_ou_ev.get("away_team", ""))
            _ou_books = {bk.get("key"): bk for bk in _ou_ev.get("bookmakers", [])}
            for _ou_bk_name in _OU_SHARP_PREF:
                _ou_bk = _ou_books.get(_ou_bk_name)
                if not _ou_bk: continue
                _ou_mk = next((m for m in _ou_bk.get("markets", []) if m.get("key") == "totals"), None)
                if not _ou_mk: continue
                _ou_outs = _ou_mk.get("outcomes", [])
                _ou_over  = next((o for o in _ou_outs if str(o.get("name","")).lower() == "over"), None)
                _ou_under = next((o for o in _ou_outs if str(o.get("name","")).lower() == "under"), None)
                if _ou_over and _ou_under:
                    _mlb_real_ou[(_ou_ak, _ou_hk)] = {
                        "line": float(_ou_over.get("point")),
                        "over_ml": int(_ou_over.get("price")),
                        "under_ml": int(_ou_under.get("price")),
                        "book": _ou_bk_name,
                    }
                    break  # first sharp-pref book found for this game
        print("  [ODDS API: connected]")
        print(f"  [O/U REAL] Loaded real totals for {len(_mlb_real_ou)}/{len(_ou_events)} games from Odds API")
    elif _ou_r.status_code == 429:
        print(f"  [ODDS API: rate-limited] status {_ou_r.status_code} -- staying on ESPN estimate")
    else:
        print(f"  [ODDS API: failed] status {_ou_r.status_code} -- staying on ESPN estimate")
        print(f"  [O/U REAL] Odds API totals fetch returned {_ou_r.status_code} -- staying on ESPN estimate")
except Exception as _oue:
    _mlb_real_ou = {}
    print(f"  [ODDS API: failed] {_oue} -- staying on ESPN estimate")
    print(f"  [O/U REAL] Fetch failed: {_oue} -- staying on ESPN estimate")

if "home_team" in board.columns and "away_team" in board.columns:
    if "over_under" not in board.columns: board["over_under"] = np.nan
    if "over_ml" not in board.columns: board["over_ml"] = np.nan
    if "under_ml" not in board.columns: board["under_ml"] = np.nan
    board["ou_book"] = "espn_est"
    board["over_under_espn_est"] = board["over_under"]  # preserve pre-override estimate for mismatch check
    board["line_mismatch_flag"] = False
    _LINE_MISMATCH_THRESH = 0.5
    _n_line_mismatch = 0
    if _mlb_real_ou:
        for _bidx, _brow in board.iterrows():
            _bkey = (_norm_team(str(_brow.get("away_team", ""))), _norm_team(str(_brow.get("home_team", ""))))
            _bou = _mlb_real_ou.get(_bkey)
            if _bou:
                _espn_est_line = _brow.get("over_under_espn_est")
                if pd.notna(_espn_est_line) and abs(float(_bou["line"]) - float(_espn_est_line)) > _LINE_MISMATCH_THRESH:
                    _a_nm = _brow.get("away_team", "?"); _h_nm = _brow.get("home_team", "?")
                    print(f"  [LINE MISMATCH] {_a_nm} @ {_h_nm}: ESPN est {_espn_est_line:.1f} vs real book {_bou['line']:.1f} ({_bou['book']}) -- diverge >{_LINE_MISMATCH_THRESH}R, suppressing play until manually confirmed")
                    board.at[_bidx, "line_mismatch_flag"] = True
                    _n_line_mismatch += 1
                board.at[_bidx, "over_under"] = _bou["line"]
                board.at[_bidx, "over_ml"] = _bou["over_ml"]
                board.at[_bidx, "under_ml"] = _bou["under_ml"]
                board.at[_bidx, "ou_book"] = _bou["book"]
    _n_real_ou = int((board["ou_book"] != "espn_est").sum())
    print(f"  [O/U REAL] {_n_real_ou}/{len(board)} games now using real book O/U (vs ESPN estimate)")
    if _n_line_mismatch:
        print(f"  [LINE MISMATCH] {_n_line_mismatch} game(s) flagged -- O/U play suppressed on those games this card")

# ---------------------------------------------------------
# 2) park run factors + coordinates (for weather)
#    Factor = how many runs this park produces vs league avg (1.00)
#    Source: multi-year historical park factor data
# ---------------------------------------------------------
_PARK_FACTOR = {
    "cubs":       1.04,   # Wrigley Field (wind-dependent, avg above neutral)
    "whitesox":   0.96,   # Guaranteed Rate Field (pitcher-friendly)
    "redsox":     1.01,   # Fenway Park
    "yankees":    0.99,   # Yankee Stadium
    "mets":       0.97,   # Citi Field (marine air)
    "dodgers":    0.96,   # Dodger Stadium (pitcher-friendly)
    "angels":     0.99,   # Angel Stadium
    "sfgiants":   0.90,   # Oracle Park (marine layer, very pitcher-friendly)
    "rangers":    1.05,   # Globe Life Field (indoor, hot city)
    "mariners":   0.97,   # T-Mobile Park
    "astros":     1.01,   # Minute Maid Park (dome, short left field)
    "athletics":  0.96,   # Oakland Coliseum
    "rays":       0.94,   # Tropicana Field (dome, pitcher-friendly turf)
    "orioles":    1.04,   # Camden Yards
    "bluejays":   1.00,   # Rogers Centre (dome, turf)
    "guardians":  0.98,   # Progressive Field
    "twins":      1.01,   # Target Field
    "tigers":     0.97,   # Comerica Park
    "royals":     0.99,   # Kauffman Stadium
    "brewers":    0.97,   # American Family Field (dome)
    "cardinals":  0.99,   # Busch Stadium
    "pirates":    0.96,   # PNC Park
    "reds":       1.04,   # Great American Ball Park
    "braves":     1.01,   # Truist Park
    "marlins":    0.94,   # loanDepot park (dome, very pitcher-friendly)
    "phillies":   1.03,   # Citizens Bank Park
    "nationals":  0.99,   # Nationals Park
    "dbacks":     1.05,   # Chase Field (altitude 1,088ft + dome)
    "rockies":    1.32,   # Coors Field (5,280ft altitude â€” massive effect)
    "padres":     0.92,   # Petco Park (marine layer, large park)
}

# Park-factor tier dicts derived from _PARK_FACTOR (moved here -- was
# referencing _PARK_FACTOR before its definition further down this cell,
# a NameError on every run):
#   _PG_HIGH: extreme hitter parks (PF >= 1.10) -- F6/RL OVER auto-favored here
#   _PG_LOW:  all other parks -- downstream gates use SP ERA check instead
_PG_HIGH = {k: v for k, v in _PARK_FACTOR.items() if v >= 1.10}
_PG_LOW  = {k: v for k, v in _PARK_FACTOR.items() if v <  1.10}

# Dome teams: skip weather temp adjustment
_DOME_TEAMS = {"astros", "rays", "bluejays", "brewers", "marlins", "dbacks", "rangers"}

# Ballpark GPS for live weather fetch (outdoor parks only)
_PARK_COORDS = {
    "cubs":       (41.9484, -87.6553),
    "whitesox":   (41.8300, -87.6338),
    "redsox":     (42.3467, -71.0972),
    "yankees":    (40.8296, -73.9262),
    "mets":       (40.7571, -73.8458),
    "dodgers":    (34.0739, -118.2400),
    "angels":     (33.8003, -117.8827),
    "sfgiants":   (37.7786, -122.3893),
    "mariners":   (47.5914, -122.3325),
    "athletics":  (38.5799, -121.5125),  # 2026-08-14: Sutter Health Park, Sacramento (real 2025+ home; was stale Oakland Coliseum coords)
    "orioles":    (39.2840, -76.6217),
    "guardians":  (41.4962, -81.6852),
    "twins":      (44.9817, -93.2776),
    "tigers":     (42.3390, -83.0485),
    "royals":     (39.0517, -94.4803),
    "cardinals":  (38.6226, -90.1928),
    "pirates":    (40.4469, -80.0057),
    "reds":       (39.0979, -84.5081),
    "braves":     (33.8907, -84.4677),
    "phillies":   (39.9061, -75.1665),
    "nationals":  (38.8730, -77.0074),
    "rockies":    (39.7559, -104.9942),
    "padres":     (32.7073, -117.1573),
}

# ---------------------------------------------------------
# 3) live weather fetch (Open-Meteo, free, no API key)
# ---------------------------------------------------------
_wx_cache = {}

def _fetch_weather(lat, lon, game_dt_utc=None):
    import datetime as _wdt
    _target_hour = None
    if game_dt_utc:
        try:
            _gdt = _wdt.datetime.fromisoformat(str(game_dt_utc).replace('Z','').split('+')[0])
            _target_hour = _gdt.hour
        except Exception:
            pass
    key = (round(lat, 3), round(lon, 3), _target_hour)
    if key in _wx_cache:
        return _wx_cache[key]
    try:
        if _target_hour is not None:
            url = (
                f"https://api.open-meteo.com/v1/forecast"
                f"?latitude={lat}&longitude={lon}"
                # 2026-08-13 (user audit): added precipitation_probability --
                # the model previously fetched temp/wind only, so a real
                # postponement/rain-delay risk (which scrambles bullpen usage
                # and directly undermines an SP-edge-driven play) was invisible.
                f"&hourly=temperature_2m,windspeed_10m,winddirection_10m,precipitation_probability"
                f"&temperature_unit=fahrenheit&windspeed_unit=mph"
                f"&timezone=UTC&forecast_days=2"
            )
            r = requests.get(url, timeout=6)
            if r.status_code == 200:
                import datetime as _wdt2
                hrly  = r.json().get('hourly', {})
                times = hrly.get('time', [])
                temps = hrly.get('temperature_2m', [])
                winds = hrly.get('windspeed_10m', [])
                dirs  = hrly.get('winddirection_10m', [])
                precs = hrly.get('precipitation_probability', [])
                _today = _wdt2.date.today()
                _best = 0
                for _hi, _ht in enumerate(times):
                    try:
                        _hd = _wdt2.datetime.fromisoformat(_ht)
                        if _hd.date() == _today and _hd.hour == _target_hour:
                            _best = _hi
                            break
                    except Exception:
                        pass
                result = {
                    'temp_f':     float(temps[_best]) if temps else 70.0,
                    'wind_mph':   float(winds[_best]) if winds else 0.0,
                    'wind_dir':   float(dirs[_best])  if dirs  else -1.0,
                    'precip_pct': float(precs[_best]) if precs else -1.0,
                }
                _wx_cache[key] = result
                return result
        url = (
            f"https://api.open-meteo.com/v1/forecast"
            f"?latitude={lat}&longitude={lon}"
            f"&current=temperature_2m,windspeed_10m,winddirection_10m,precipitation_probability"
            f"&temperature_unit=fahrenheit&windspeed_unit=mph"
            f"&timezone=auto&forecast_days=1"
        )
        r = requests.get(url, timeout=6)
        if r.status_code == 200:
            curr = r.json().get("current", {})
            result = {
                "temp_f":     float(curr.get("temperature_2m", 70)),
                "wind_mph":   float(curr.get("windspeed_10m",  0)),
                "wind_dir":   float(curr.get("winddirection_10m", -1)),
                "precip_pct": float(curr.get("precipitation_probability", -1)),
            }
            _wx_cache[key] = result
            return result
    except Exception:
        pass
    return {"temp_f": 70.0, "wind_mph": 0.0, "wind_dir": -1.0, "precip_pct": -1.0}

def _temp_factor(temp_f):
    """Each 10Â°F below 65 â†’ -3% runs. Above 80Â°F â†’ +1.5% per 10Â°F. Clipped Â±15%."""
    delta = temp_f - 65.0
    if delta < 0:
        factor = 1.0 + delta * 0.003      # cold â†’ fewer runs
    else:
        factor = 1.0 + delta * 0.0015     # hot â†’ slightly more runs
    return float(np.clip(factor, 0.85, 1.15))

# Outfield compass bearing per park (degrees from north to center field)
# Wind blowing toward this bearing = blowing OUT = more runs
_PARK_OUTFIELD_DIR = {
    'chicago cubs': 30,     'boston red sox': 280,  'san francisco giants': 270,
    'colorado rockies': 350,'chicago white sox': 10, 'new york yankees': 310,
    'new york mets': 340,   'philadelphia phillies': 240, 'pittsburgh pirates': 35,
    'cincinnati reds': 50,  'cleveland guardians': 190, 'detroit tigers': 340,
    'minnesota twins': 10,  'kansas city royals': 30, 'texas rangers': 350,
    'los angeles angels': 330, 'los angeles dodgers': 340, 'san diego padres': 300,
    'atlanta braves': 40,   'st. louis cardinals': 30, 'baltimore orioles': 340,
    'washington nationals': 210, 'oakland athletics': 330,
}

def _wind_factor(wind_mph, wind_dir, park_key):
    """Wind out = more runs (+0.4%/mph above 5). Wind in = fewer (-0.3%/mph above 5)."""
    if wind_mph < 5 or wind_dir < 0 or park_key not in _PARK_OUTFIELD_DIR:
        return 1.0
    outfield_dir = _PARK_OUTFIELD_DIR[park_key]
    angle = abs((wind_dir - outfield_dir + 180) % 360 - 180)
    excess = wind_mph - 5
    if angle < 60:        # blowing out
        return float(np.clip(1.0 + excess * 0.004, 1.0, 1.20))
    elif angle > 120:     # blowing in
        return float(np.clip(1.0 - excess * 0.003, 0.80, 1.0))
    return 1.0            # crosswind


# ---------------------------------------------------------
# 4) predicted score computation
# ---------------------------------------------------------
def pick_col(df, options):
    lower_map = {str(c).strip().lower(): c for c in df.columns}
    for opt in options:
        if opt.lower() in lower_map:
            return lower_map[opt.lower()]
    return None

def safe_num(s):
    return pd.to_numeric(s, errors="coerce")

away_col = pick_col(board, ["away_team", "away_team_name", "away"])
home_col = pick_col(board, ["home_team", "home_team_name", "home"])

# run average columns
_a_off  = pick_col(board, ["away_runs_for_avg_road_10", "away_runs_for_avg_10", "away_runs_for_avg_7", "away_runs_for_avg"])  # FIX 2: prefer road split
_h_off  = pick_col(board, ["home_runs_for_avg_home_10", "home_runs_for_avg_10", "home_runs_for_avg_7", "home_runs_for_avg"])  # FIX 2: prefer home split
_a_def  = pick_col(board, ["away_runs_against_avg_10", "away_runs_against_avg_7", "away_runs_against_avg"])
_h_def  = pick_col(board, ["home_runs_against_avg_10", "home_runs_against_avg_7", "home_runs_against_avg"])

# starter RA columns (preferred over team RA for accuracy)
_a_sp_ra = pick_col(board, [
    "away_starter_runs_allowed_avg_10", "away_starter_ra_avg_10",
    "away_starter_runs_allowed_avg_7",  "away_starter_ra_avg",
    "away_starter_runs_allowed_10",
])
_h_sp_ra = pick_col(board, [
    "home_starter_runs_allowed_avg_10", "home_starter_ra_avg_10",
    "home_starter_runs_allowed_avg_7",  "home_starter_ra_avg",
    "home_starter_runs_allowed_10",
])

_can_score = (_a_off is not None and _h_off is not None
              and (_a_def is not None or _a_sp_ra is not None)
              and (_h_def is not None or _h_sp_ra is not None))

print(f"\nPredicted score inputs:")
print(f"  away offense  : {_a_off}")
print(f"  home offense  : {_h_off}")
print(f"  away SP RA    : {_a_sp_ra}  (fallback: {_a_def})")
print(f"  home SP RA    : {_h_sp_ra}  (fallback: {_h_def})")
print(f"  Can compute   : {_can_score}")

board["pred_away_runs"] = np.nan
board["pred_home_runs"] = np.nan
board["pred_total_runs"] = np.nan
board["park_factor_used"] = np.nan
board["game_temp_f"] = np.nan
board["is_neutral_site"] = False
board["venue_name_actual"] = ""
board["rain_risk_flag"] = False
board["precip_pct"] = np.nan

# Fix 7: Series momentum -- fetch last 2 game results for each series matchup
# Team lost 2 consecutive series games -> -0.15R projected runs penalty
# Team won 2 consecutive series games -> +0.10R boost
_series_momentum = {}  # {(away_norm, home_norm): {team_lower: {"streak": int}}}
try:
    import requests as _sm7_req, datetime as _sm7_dt
    _sm7_results = {}  # {(away_norm, home_norm): [(winner_norm, date_str), ...]}
    for _sm7_days_ago in [1, 2]:
        _sm7_d = (_sm7_dt.date.today() - _sm7_dt.timedelta(days=_sm7_days_ago)).strftime("%Y-%m-%d")
        _sm7_r = _sm7_req.get(
            f"https://statsapi.mlb.com/api/v1/schedule?sportId=1&date={_sm7_d}&hydrate=linescore",
            timeout=6
        )
        if _sm7_r.status_code != 200: continue
        for _sm7_g in _sm7_r.json().get("dates", [{}])[0].get("games", []):
            if _sm7_g.get("status", {}).get("codedGameState") not in ("F", "O"): continue
            _sm7_a = str(_sm7_g.get("teams",{}).get("away",{}).get("team",{}).get("name","")).lower()
            _sm7_h = str(_sm7_g.get("teams",{}).get("home",{}).get("team",{}).get("name","")).lower()
            _sm7_as = int(_sm7_g.get("teams",{}).get("away",{}).get("score",0) or 0)
            _sm7_hs = int(_sm7_g.get("teams",{}).get("home",{}).get("score",0) or 0)
            _sm7_winner = _sm7_h if _sm7_hs > _sm7_as else (_sm7_a if _sm7_as > _sm7_hs else None)
            _sm7_key = (_sm7_a, _sm7_h)
            _sm7_results.setdefault(_sm7_key, []).append((_sm7_winner, _sm7_d))
    # Map results to current series matchups
    for (_sm7_ca, _sm7_ch), _sm7_ctx in _series_context.items():
        _sm7_recent = []
        for (_sm7_pa, _sm7_ph), _sm7_rlist in _sm7_results.items():
            _pa_match = any(w in _sm7_ca for w in _sm7_pa.split() if len(w)>3) or any(w in _sm7_pa for w in _sm7_ca.split() if len(w)>3)
            _ph_match = any(w in _sm7_ch for w in _sm7_ph.split() if len(w)>3) or any(w in _sm7_ph for w in _sm7_ch.split() if len(w)>3)
            if _pa_match and _ph_match:
                _sm7_recent = sorted(_sm7_rlist, key=lambda x: x[1], reverse=True)[:2]
                break
        _sm7_streak = {_sm7_ca: {"streak": 0}, _sm7_ch: {"streak": 0}}
        for _sm7_w, _ in _sm7_recent:
            if _sm7_w is None: continue
            for _t in [_sm7_ca, _sm7_ch]:
                if any(w in _sm7_w for w in _t.split() if len(w)>3):
                    _sm7_streak[_t]["streak"] += 1
                else:
                    _sm7_streak[_t]["streak"] -= 1
        _series_momentum[(_sm7_ca, _sm7_ch)] = _sm7_streak
    if _series_momentum:
        print(f"  Series momentum loaded for {len(_series_momentum)} matchup(s)")
except Exception as _sm7_e:
    pass  # non-blocking

# ── BULLPEN RA LOOKUP ──────────────────────────────────────────────────────
# Blend: 65% starter RA + 35% bullpen RA for effective pitching environment
_bpen_lookup_stale = {}   # norm(team) -> bullpen_ra_avg_10, frozen (mlb_model_table_v1, ends 2025-11-01)
if 'mlb' in globals() and isinstance(mlb, pd.DataFrame):
    for _side, _tcol, _bcol in [('away','away_team','away_bullpen_ra_avg_10'),
                                  ('home','home_team','home_bullpen_ra_avg_10')]:
        if _tcol in mlb.columns and _bcol in mlb.columns:
            _bpen_sub = mlb[[_tcol, _bcol, 'date']].dropna(subset=[_bcol]).copy()
            _bpen_sub['date'] = pd.to_datetime(_bpen_sub['date'], errors='coerce')
            _bpen_latest = (_bpen_sub.sort_values('date')
                            .groupby(_tcol)[_bcol].last())
            for _tn, _bv in _bpen_latest.items():
                _bpen_lookup_stale[_norm_team(str(_tn))] = float(_bv)

# 2026-08-04: _bpen_lookup used to be built ONLY from the block above, which
# traces back through `mlb` to the orphaned mlb_model_table_v1.parquet (no
# builder cell anywhere in this notebook, real data ends 2025-11-01) -- the
# 35%-weighted bullpen half of every totals projection was silently using
# each team's bullpen as of ~Oct/Nov 2025 all season. Cell 18 ("REFRESH TEAM
# STATS FROM CURRENT 2026 SEASON") already fetches live results and now
# computes a real bullpen-only rolling estimate (bullpen_ra_avg_10_live) --
# preferring that here, falling back to the frozen value only for a team
# the live ESPN fetch didn't cover this run.
_bpen_lookup_fresh = {}
if 'mlb_team_snapshot_clean' in globals() and isinstance(mlb_team_snapshot_clean, pd.DataFrame) \
        and 'bullpen_ra_avg_10_live' in mlb_team_snapshot_clean.columns:
    for _, _snap_row in mlb_team_snapshot_clean.dropna(subset=['bullpen_ra_avg_10_live']).iterrows():
        _bpen_lookup_fresh[_norm_team(str(_snap_row['team']))] = float(_snap_row['bullpen_ra_avg_10_live'])

_bpen_lookup = {**_bpen_lookup_stale, **_bpen_lookup_fresh}
_bpen_lookup = {k: v for k, v in _bpen_lookup.items() if k in _PARK_FACTOR}
# MLB-PEN-1 (2026-10-04 user request -- "in the postseason, rate each bullpen using only the
# top 5 relievers by leverage index from the last 60 days, weighted by expected innings").
# The Stats API has no leverage index -> leverage proxied by high-leverage appearances
# (saves + holds + blown saves, IP tiebreak) over the last 60 days; top 5 -> innings-weighted
# runs/9, shrunk toward that team's whole-pen rate over the same window (k=30 IP). Uses the
# ACTIVE roster (a traded reliever's byDateRange line is player-level, not team-split).
# Postseason teams only; regular-season games keep the whole-pen _bpen_lookup.
# Validated with scripts/mlb_pen_top5_refit.py: re-fitting the postseason run shift with this
# bullpen moved it -0.41 -> -0.38R (n=188 PS games 2021-25) -- it absorbs only ~0.03R.
# Oct 3 DS teams, whole pen -> top-5: SD 3.74->3.00, MIL 3.33->2.93, NYY 4.11->3.49,
# TB 4.51->4.11, CLE 3.99->3.26, CWS 3.83->4.20.
def _mlb_top5_pen(team_id, end_date, days=60, k_ip=30.0):
    import datetime as _p5dt, requests as _p5r
    _start = (end_date - _p5dt.timedelta(days=days)).isoformat()
    _ro = _p5r.get(f"https://statsapi.mlb.com/api/v1/teams/{int(team_id)}/roster",
                   params={"rosterType": "active", "season": end_date.year}, timeout=20).json()
    _pids = [p["person"]["id"] for p in _ro.get("roster", []) if p.get("position", {}).get("abbreviation") == "P"]
    if not _pids:
        return None
    _pp = _p5r.get("https://statsapi.mlb.com/api/v1/people", timeout=30, params={
        "personIds": ",".join(map(str, _pids)),
        "hydrate": f"stats(group=pitching,type=byDateRange,startDate={_start},endDate={end_date.isoformat()},gameType=R)"}).json()
    _rel = []
    for _p in _pp.get("people", []):
        try:
            _st = _p["stats"][0]["splits"][0]["stat"]
        except Exception:
            continue
        _w, _, _f = str(_st.get("inningsPitched", "0")).partition(".")
        _ip = float(_w or 0) + (float(_f) / 3 if _f else 0)
        _gs, _gp = int(_st.get("gamesStarted", 0)), int(_st.get("gamesPlayed", 0))
        if _ip < 3 or _gp == 0 or _gs > 0.2 * _gp:
            continue
        _lev = int(_st.get("saves", 0)) + int(_st.get("holds", 0)) + int(_st.get("blownSaves", 0))
        _rel.append((_lev, _ip, float(_st.get("runs", _st.get("earnedRuns", 0))), _p.get("fullName")))
    if len(_rel) < 5:
        return None
    _whole = sum(r[2] for r in _rel) * 9 / sum(r[1] for r in _rel)
    _top = sorted(_rel, key=lambda r: (-r[0], -r[1]))[:5]
    _ip5 = sum(r[1] for r in _top); _ra5 = sum(r[2] for r in _top) * 9 / _ip5
    return (_ip5 * _ra5 + k_ip * _whole) / (_ip5 + k_ip), _whole, _top

try:
    import datetime as _p5dt2, requests as _p5r2
    _p5_today = _p5dt2.date.today()
    _p5_sched = _p5r2.get("https://statsapi.mlb.com/api/v1/schedule", params={"sportId": 1, "date": _p5_today.isoformat()}, timeout=20).json()
    _p5_teams = {}
    for _d in _p5_sched.get("dates", []):
        for _gm in _d.get("games", []):
            if _gm.get("gameType") in ("F", "D", "L", "W"):
                for _sd in ("away", "home"):
                    _p5_teams[_gm["teams"][_sd]["team"]["name"]] = _gm["teams"][_sd]["team"]["id"]
    for _tn, _tid in _p5_teams.items():
        _v = _mlb_top5_pen(_tid, _p5_today - _p5dt2.timedelta(days=1))
        _k = _norm_team(_tn)
        if _v and _k in _bpen_lookup:
            print(f"  [MLB-PEN-1] {_tn}: postseason bullpen RA {_bpen_lookup[_k]:.2f} -> {_v[0]:.2f} (top-5 by leverage, last 60d: "
                  + ", ".join(f"{r[3]} {r[0]}lev/{r[1]:.0f}IP" for r in _v[2]) + f"; whole pen {_v[1]:.2f})")
            # MLB-PEN-1 SCALE FIX (2026-10-04): _bpen_lookup is bullpen runs PER GAME (Cell 18:
            # 0.35 x team runs-allowed EWM, ~1.2-1.5); _v[0] is runs PER 9 (~2.8-3.8). Writing it here
            # (as the first MLB-PEN-1 version did) raised the old formula's pitching term by
            # 0.35 x (RA9 - per-game) ~ +0.5R per side in every postseason game. The top-5 rate now
            # feeds the component totals model (MLB-TOT-COMP-1), which works in runs/9.
            globals().setdefault("mlb_ps_top5_pen_r9", {})[_k] = _v[0]
        elif _k in _bpen_lookup:
            print(f"  [MLB-PEN-1] {_tn}: < 5 qualifying relievers in the window -- whole-pen rate kept")
except Exception as _p5e:
    print(f"  [MLB-PEN-1] postseason bullpen override skipped ({_p5e})")
_bpen_fresh_ct = sum(1 for k in _bpen_lookup if k in _bpen_lookup_fresh)
print(f"Bullpen RA loaded for {len(_bpen_lookup)} teams ({_bpen_fresh_ct} live 2026, {len(_bpen_lookup) - _bpen_fresh_ct} stale-fallback)")

# ── NEUTRAL SITE / ACTUAL VENUE DETECTION ───────────────────────────────
# 2026-08-13 (user audit, Field of Dreams game): the loop below used to apply
# the HOME team's own park factor + home park's weather to every game,
# assuming every "home" game is played at that team's actual home park.
# False for MLB's handful of real neutral-site games a season (Field of
# Dreams, London Series, Tokyo Series, Little League Classic, Mexico/
# Caribbean Series) -- confirmed live: MLB Stats API's schedule venue for
# tonight's Phillies @ Twins game is venue id 5445 "Field of Dreams"
# (Dyersville, IA), not Target Field. hydrate=venue(location) on the
# schedule call returns both the real venue name and its real lat/lon in
# one request, so this also fixes the weather pull (was silently using
# Target Field's coordinates for a game 100+ miles away in a cornfield).
_TEAM_HOME_PARK_NAME = {
    "cubs": "wrigley field", "whitesox": "guaranteed rate field", "redsox": "fenway park",
    "yankees": "yankee stadium", "mets": "citi field", "dodgers": "dodger stadium",
    "angels": "angel stadium", "sfgiants": "oracle park", "rangers": "globe life field",
    # 2026-08-14 (user audit, real bug -- found live on tonight's real Rangers @
    # Athletics game): both of these were stale stadium-rename entries that
    # would have false-flagged a completely normal home game as a fake
    # "neutral site" the moment this specific matchup finally reached this
    # cell -- confirmed live via MLB Stats API: Houston's park was renamed
    # Daikin Park (from Minute Maid Park) in 2025, and the Athletics' real
    # 2025+ home park is Sutter Health Park in Sacramento (Oakland Coliseum
    # is no longer their home at all, part of the same relocation/rename this
    # session already found breaking multiple other cells).
    "mariners": "t-mobile park", "astros": "daikin park", "athletics": "sutter health park",
    "rays": "tropicana field", "orioles": "camden yards", "bluejays": "rogers centre",
    "guardians": "progressive field", "twins": "target field", "tigers": "comerica park",
    "royals": "kauffman stadium", "brewers": "american family field", "cardinals": "busch stadium",
    "pirates": "pnc park", "reds": "great american ball park", "braves": "truist park",
    "marlins": "loandepot park", "phillies": "citizens bank park", "nationals": "nationals park",
    "dbacks": "chase field", "rockies": "coors field", "padres": "petco park",
}

_nv_gid_col = pick_col(board, ["game_id", "gamepk", "game_pk", "pk"])
_venue_lookup = {}  # gamePk (int) -> {"venue_name":.., "lat":.., "lon":..}
try:
    _nv_dates = set()
    if date_col and date_col in board.columns:
        _nv_dates = set(pd.to_datetime(board[date_col], errors="coerce").dt.strftime("%Y-%m-%d").dropna().tolist())
    if not _nv_dates:
        import datetime as _nv_dt
        _nv_dates = {_nv_dt.date.today().strftime("%Y-%m-%d")}
    for _nv_d in sorted(_nv_dates):
        _nv_r = requests.get(
            f"https://statsapi.mlb.com/api/v1/schedule?sportId=1&date={_nv_d}&hydrate=venue(location)",
            headers={"User-Agent": "Mozilla/5.0"}, timeout=10
        )
        if _nv_r.status_code != 200:
            continue
        for _nv_dobj in _nv_r.json().get("dates", []):
            for _nv_g in _nv_dobj.get("games", []):
                _nv_gid = _nv_g.get("gamePk")
                _nv_venue = _nv_g.get("venue", {}) or {}
                _nv_loc = (_nv_venue.get("location") or {}).get("defaultCoordinates") or {}
                _venue_lookup[_nv_gid] = {
                    "venue_name": _nv_venue.get("name", ""),
                    "lat": _nv_loc.get("latitude"),
                    "lon": _nv_loc.get("longitude"),
                }
except Exception as _nv_e:
    print(f"  [WARN] Venue/neutral-site fetch failed: {_nv_e}")

print(f"Venue data loaded for {len(_venue_lookup)} game(s) today")

if _can_score:

    _have_elo = "mlb_pitcher_elo" in globals()  # init before loop
    # NEW (MLB-OPENER-1, 2026-09-24 user audit): detect a same-day opener/
    # bulk-arm pairing from real recent boxscore evidence and swap the bulk
    # arm in as the effective starter for rating purposes. Real, verified
    # case (checked live against MLB Stats API boxscores before building
    # this): Daniel Espino started for Cleveland on 2026-09-18 and threw
    # only 1.0 IP before Joey Cantillo relieved him and threw 6.0 IP -- the
    # actual workhorse of that game. Espino is the announced starter again
    # tonight but has almost no real MLB track record (thin-sample -- his
    # own rating is close to meaningless), and nothing in this notebook
    # looked at what ACTUALLY happened the last time he "started." Detects
    # this from real boxscore data, not a guess: pulls the announced
    # starter's single most recent credited-start appearance; if he threw
    # under 3.0 IP there AND a specific reliever threw 3.0+ IP immediately
    # after him, that reliever becomes the real starter for pricing --
    # away_pitcher_name/home_pitcher_name are overwritten with his name so
    # every downstream consumer (FIP/ELO lookup, role flags, SP Matchup
    # display, tier label) automatically prices HIM instead of the
    # announced opener. The originally-announced name is kept in a separate
    # *_pitcher_announced_name column so the card can still show who is
    # technically listed as tonight's starter. Only triggered for pitchers
    # already flagged thin-sample (<5 MLB starts, matching this notebook's
    # existing UNRATED threshold) or season-classified as a true opener --
    # bounded so this doesn't fire (and doesn't cost extra API calls) for a
    # normal established starter.
    _MLB_OPENER_BULK_MAX_OWN_IP = 3.0     # his own IP in that game, to count as an opener-style usage
    _MLB_OPENER_BULK_MIN_RELIEF_IP = 3.0  # the reliever's IP right after him, to count as the real bulk arm
    _mlb_ob_team_id_cache = {}
    def _mlb_ob_team_id(team_name):
        if not _mlb_ob_team_id_cache:
            try:
                _r = requests.get(f"{_MLB_API}/teams?sportId=1", timeout=10).json()
                for _t in _r.get("teams", []):
                    if _t.get("name"):
                        _mlb_ob_team_id_cache[str(_t["name"]).lower()] = _t.get("id")
            except Exception:
                pass
        return _mlb_ob_team_id_cache.get(str(team_name).lower())

    _mlb_ob_partner_cache = {}  # pitcher_name -> (bulk_name, bulk_ip, own_ip) or None
    def _mlb_ob_find_bulk_partner(pitcher_name, team_id):
        if not pitcher_name or not team_id:
            return None
        _key = str(pitcher_name).strip()
        if _key in _mlb_ob_partner_cache:
            return _mlb_ob_partner_cache[_key]
        _result = None
        try:
            import datetime as _ob_dt
            _today_d = _ob_dt.date.today()
            _start_d = (_today_d - _ob_dt.timedelta(days=30)).strftime("%Y-%m-%d")
            _end_d = (_today_d - _ob_dt.timedelta(days=1)).strftime("%Y-%m-%d")
            _sched = requests.get(
                f"{_MLB_API}/schedule?sportId=1&teamId={int(team_id)}&startDate={_start_d}&endDate={_end_d}",
                timeout=10
            ).json()
            _game_pks = []
            for _d in _sched.get("dates", []):
                for _g in _d.get("games", []):
                    if _g.get("status", {}).get("codedGameState") == "F":
                        _game_pks.append(_g.get("gamePk"))
            _game_pks.sort(reverse=True)
            for _gpk in _game_pks:
                _found_start_here = False
                try:
                    _box = requests.get(f"{_MLB_API}/game/{_gpk}/boxscore", timeout=10).json()
                except Exception:
                    continue
                for _side in ("away", "home"):
                    _tb = _box.get("teams", {}).get(_side, {})
                    if _tb.get("team", {}).get("id") != int(team_id):
                        continue
                    _players = _tb.get("players", {})
                    _order = _tb.get("pitchers", [])
                    _seq = []
                    for _pid in _order:
                        _p = _players.get(f"ID{_pid}", {})
                        _nm = _p.get("person", {}).get("fullName", "")
                        _st = _p.get("stats", {}).get("pitching", {})
                        try:
                            _ip = float(_st.get("inningsPitched", "0") or 0)
                        except (TypeError, ValueError):
                            _ip = 0.0
                        _gs = _st.get("gamesStarted", 0)
                        _seq.append((_nm, _ip, _gs))
                    if _seq and str(_seq[0][0]).strip().lower() == _key.lower() and _seq[0][2] == 1:
                        _found_start_here = True
                        if _seq[0][1] < _MLB_OPENER_BULK_MAX_OWN_IP and len(_seq) > 1:
                            _bulk = max(_seq[1:], key=lambda t: t[1])
                            if _bulk[1] >= _MLB_OPENER_BULK_MIN_RELIEF_IP:
                                _result = (_bulk[0], _bulk[1], _seq[0][1])
                if _found_start_here:
                    break  # examined his single most recent credited start -- stop regardless of outcome
        except Exception:
            pass
        _mlb_ob_partner_cache[_key] = _result
        return _result

    _ob_swap_ct = 0
    for _ob_col, _ob_team_col in (("away_pitcher_name", away_col), ("home_pitcher_name", home_col)):
        if _ob_col not in board.columns or _ob_team_col not in board.columns:
            continue
        _ob_announced_col = _ob_col.replace("_pitcher_name", "_pitcher_announced_name")
        for _obi in board.index:
            _ob_pname = board.at[_obi, _ob_col]
            if pd.isna(_ob_pname) or not str(_ob_pname).strip():
                continue
            _ob_pname = str(_ob_pname).strip()
            _ob_trk = mlb_sp_tracker.get(_ob_pname) if "mlb_sp_tracker" in globals() else None
            _ob_starts = _ob_trk.get("starts") if _ob_trk else None
            _ob_fev = mlb_pitcher_fip_lookup.get(_sp_norm_era(_ob_pname), {}) if "mlb_pitcher_fip_lookup" in globals() else {}
            _ob_is_thin = _ob_starts is not None and _ob_starts < 5
            _ob_is_opener_flag = bool(_ob_fev.get("is_opener"))
            if not (_ob_is_thin or _ob_is_opener_flag):
                continue
            _ob_team_name = board.at[_obi, _ob_team_col]
            _ob_team_id = _mlb_ob_team_id(_ob_team_name)
            _ob_partner = _mlb_ob_find_bulk_partner(_ob_pname, _ob_team_id)
            if _ob_partner is None:
                continue
            _ob_bulk_name, _ob_bulk_ip, _ob_own_ip = _ob_partner
            if _ob_announced_col not in board.columns:
                board[_ob_announced_col] = np.nan
            board.at[_obi, _ob_announced_col] = _ob_pname
            board.at[_obi, _ob_col] = _ob_bulk_name
            _ob_swap_ct += 1
            print(f"  [OPENER -> BULK] {_ob_team_name}: {_ob_pname} announced, but his most recent start "
                  f"was {_ob_own_ip:.1f} IP before {_ob_bulk_name} relieved for {_ob_bulk_ip:.1f} IP -- "
                  f"pricing {_ob_bulk_name} as tonight's effective starter instead")
    if _ob_swap_ct:
        print(f"  Opener/bulk-arm detection: {_ob_swap_ct} starter(s) swapped for their real bulk-arm partner")

    for _idx, _row in board.iterrows():
        _hteam_key = _norm_team(str(_row.get(home_col, "")))

        # neutral-site check: does today's actual venue match this team's
        # real home park? (substring match against a known-park-name dict,
        # not just "any venue field present" -- a mismatch here means the
        # game literally isn't being played where the model assumes.)
        _gid_raw = _row.get(_nv_gid_col) if _nv_gid_col else None
        try:
            _gid_int = int(_gid_raw) if _gid_raw is not None and str(_gid_raw) != "nan" else None
        except Exception:
            _gid_int = None
        _venue_info = _venue_lookup.get(_gid_int, {})
        _venue_name_actual = str(_venue_info.get("venue_name", "") or "")
        _expected_park = _TEAM_HOME_PARK_NAME.get(_hteam_key, "")
        _is_neutral = bool(_venue_name_actual) and bool(_expected_park) and \
            (_expected_park not in _venue_name_actual.lower()) and \
            (_venue_name_actual.lower() not in _expected_park)
        board.at[_idx, "is_neutral_site"] = _is_neutral
        board.at[_idx, "venue_name_actual"] = _venue_name_actual
        if _is_neutral:
            print(f"  [NEUTRAL SITE] {_row.get(away_col,'?')} @ {_row.get(home_col,'?')} -- actually at {_venue_name_actual}, not {_expected_park.title()}. Home park factor/weather/home-adv suppressed.")

        # park factor -- defaults to neutral (1.00) for a confirmed neutral-site game
        _pf = 1.00 if _is_neutral else _PARK_FACTOR.get(_hteam_key, 1.00)
        board.at[_idx, "park_factor_used"] = _pf

        # temperature factor -- uses the ACTUAL venue's coordinates when the
        # game is at a neutral site (fetched above), home park coords otherwise
        _tf = 1.00
        _wx = {}
        _wx_info = "dome/indoor"
        _wx_lat = _wx_lon = None
        if _is_neutral and _venue_info.get("lat") is not None:
            _wx_lat, _wx_lon = _venue_info["lat"], _venue_info["lon"]
        elif _hteam_key not in _DOME_TEAMS and _hteam_key in _PARK_COORDS:
            _wx_lat, _wx_lon = _PARK_COORDS[_hteam_key]
        if _wx_lat is not None:
            _wx = _fetch_weather(_wx_lat, _wx_lon, game_dt_utc=_row.get(date_col))
            _tf = _temp_factor(_wx["temp_f"])
            _wx_info = f"{_wx['temp_f']:.0f}F wind {_wx['wind_mph']:.0f}mph"
            board.at[_idx, "game_temp_f"] = _wx["temp_f"]
            board.at[_idx, "wind_mph"]    = _wx["wind_mph"]
            board.at[_idx, "wind_dir"]    = _wx.get("wind_dir", -1)
            board.at[_idx, "precip_pct"]  = _wx.get("precip_pct", -1)
            # 2026-08-13 (user audit): explicit postponement-risk flag -- a
            # rain delay/postponement scrambles bullpen usage and directly
            # undermines an SP-edge-driven play, so this needs to be loud,
            # not buried in a raw temp/wind number.
            _precip_val = _wx.get("precip_pct", -1)
            if _precip_val is not None and _precip_val >= 50:
                board.at[_idx, "rain_risk_flag"] = True
                print(f"  [RAIN RISK] {_row.get(away_col,'?')} @ {_row.get(home_col,'?')} -- {_precip_val:.0f}% precip probability at first pitch, postponement/delay risk")
            else:
                board.at[_idx, "rain_risk_flag"] = False

        _wf = _wind_factor(
            _wx.get('wind_mph', 0) if isinstance(_wx, dict) else 0,
            _wx.get('wind_dir', -1) if isinstance(_wx, dict) else -1,
            _hteam_key
        ) if _hteam_key not in _DOME_TEAMS else 1.0
        _env = _pf * _tf * _wf

        # pitching: 65% starter RA + 35% bullpen RA (blended effective environment)
        # Real gap found 2026-07-23: is_opener was computed (CELL 20) but never
        # read anywhere -- a starter who won't go a normal length had no effect
        # on the SP/bullpen run-environment blend at all. Wired up here, plus a
        # new is_bulk_role flag (also CELL 20) for starters who go longer than a
        # true opener but still get pulled early or have a reliever-level K rate --
        # both push more of the blend weight onto the bullpen than the default
        # 65/35 SP/bullpen split assumes.

        # POSTSEASON SP ROLE DETECTION (MLB-PS-SPROLE-1, 2026-10-01 user audit --
        # PHI @ ATL NLWC G3: Atlanta announced Ray Kerr, a reliever (9 G / 1 GS /
        # 18.1 IP), as an opener ahead of a bulk arm, and Philadelphia had Wheeler
        # (28 GS, unused in the series) available to piggyback Nola -- yet the
        # model priced it as a normal 65/35 starter game. Extends fix #3 (missing
        # SP -> bullpen-game prior or UNRATED) to postseason roles, using the
        # Stats API's own probable + seriesStatus feed (one schedule call per day,
        # one season-stats call per probable, one roster call per team, cached):
        #   TBD (no probable posted)      -> bullpen-game prior (90/10), same as
        #                                     a confirmed bullpen game
        #   OPENER (probable is a reliever: GS/G < 0.5, or < 3.0 IP per start)
        #                                  -> bullpen-game prior (90/10) -- the
        #                                     bulk arm is unknown, so the team's
        #                                     bullpen rating is the honest number
        #   STARTER in an elimination game -> projected SP innings discounted:
        #                                     50/50 SP/bullpen (vs 65/35), same
        #                                     PROVISIONAL magnitude as the
        #                                     workload discount -- no backtest
        #   PIGGYBACK RISK (elimination game AND an unused rotation starter --
        #     GS>=8, active roster, didn't pitch yesterday -- is available)
        #                                  -> UNRATED: Gate 7d hard-blocks
        #                                     official ML/RL/TOT on the game; who
        #                                     covers innings 5-9 is unknowable
        #                                     pregame and moves runs either way.
        _PS_GAME_TYPES = {"F", "D", "L", "W"}
        # BACKTESTED 2026-10-02 (scripts/mlb_elim_starter_share.py, Stats API boxscores):
        # starters record 50.0% of team outs in postseason elimination games 2021-26
        # (n=142; 52.8% over 2015-26, n=309) vs 56.9% in non-elimination postseason
        # games -- the 0.50 guess matches the modern era. NOTE the same data puts the
        # NON-elimination postseason share at ~57-58%, below the 65/35 default
        # _eff_pit uses for every game -- not changed here (applies to all games).
        _PS_ELIM_SP_SHARE = 0.50
        # MLB-PS-SHARE-1 (2026-10-06, user decision -- "non-elimination postseason starters cover 57-58% of the
        # game, while the model assumes 65%... I'd switch to the data number"): same backtest, 56.9% (2021-26).
        _PS_NONELIM_SP_SHARE = 0.57
        _PS_PIGGYBACK_MIN_GS = 8
        _PS_OPENER_MAX_GS_SHARE = 0.5
        _PS_OPENER_MAX_IP_PER_START = 3.0
        _ps_cache = globals().setdefault("_MLB_PS_ROLE_CACHE", {})

        def _ps_ip(ip_str):
            _w, _, _f = str(ip_str or "0").partition(".")
            try:
                return float(_w or 0) + (float(_f) / 3.0 if _f else 0.0)
            except ValueError:
                return 0.0

        def _ps_api(url):
            if url not in _ps_cache:
                try:
                    _r = requests.get(url, timeout=15)
                    _r.raise_for_status()
                    _ps_cache[url] = _r.json()
                except Exception as _ps_e:
                    print(f"  [MLB-PS-SPROLE-1] fetch failed ({url.split('?')[0]}): {_ps_e}")
                    _ps_cache[url] = None
            return _ps_cache[url]

        def _ps_pitcher_role(pid):
            if not pid:
                return "TBD", None
            import datetime as _psr_dt
            _d = _ps_api(f"https://statsapi.mlb.com/api/v1/people/{int(pid)}/stats?stats=season&group=pitching"
                         f"&season={_psr_dt.date.today().year}&gameType=R")
            try:
                _st = _d["stats"][0]["splits"][0]["stat"]
            except Exception:
                return "STARTER", None   # no stats found -- don't invent a role; Gate 7b covers a blank rating
            _g, _gs, _ip = int(_st.get("gamesPlayed", 0) or 0), int(_st.get("gamesStarted", 0) or 0), _ps_ip(_st.get("inningsPitched"))
            _desc = f"{_g} G / {_gs} GS / {_st.get('inningsPitched')} IP"
            if _g and (_gs / _g < _PS_OPENER_MAX_GS_SHARE or (_gs and _ip / _gs < _PS_OPENER_MAX_IP_PER_START)):
                return "OPENER", _desc
            return "STARTER", _desc

        def _ps_piggyback_pool(team_id, probable_id, game_date):
            """Active-roster rotation arms (GS >= 8) other than today's probable
            who did NOT pitch yesterday -- the arms a manager can actually use
            in relief in a must-win game. Ordered most-likely-first: fully
            rested (no appearance in the last 3 days) before short rest, then
            by games started. MLB-PS-PIGGYPRICE-1 (2026-10-01): entries are
            dicts (name/id/gs/rested) so the top arm can be PRICED, not just
            named."""
            import datetime as _psp_dt
            _yr = game_date.year
            _ro = _ps_api(f"https://statsapi.mlb.com/api/v1/teams/{int(team_id)}/roster?rosterType=active&season={_yr}"
                          f"&hydrate=person(stats(type=season,group=pitching,season={_yr},gameType=R))")
            if not _ro:
                return []
            _used = {}   # pitcher id -> days ago (1..3) of most recent appearance
            for _ago in (3, 2, 1):
                _dt = (game_date - _psp_dt.timedelta(days=_ago)).isoformat()
                _ys = _ps_api(f"https://statsapi.mlb.com/api/v1/schedule?sportId=1&teamId={int(team_id)}&date={_dt}")
                for _yd in (_ys or {}).get("dates", []):
                    for _yg in _yd.get("games", []):
                        _bx = _ps_api(f"https://statsapi.mlb.com/api/v1/game/{_yg['gamePk']}/boxscore")
                        for _sd in ("home", "away"):
                            _bt = ((_bx or {}).get("teams") or {}).get(_sd, {})
                            if _bt.get("team", {}).get("id") == int(team_id):
                                for _pp in _bt.get("pitchers", []):
                                    _used[_pp] = _ago
            _pool = []
            for _p in _ro.get("roster", []):
                if _p.get("position", {}).get("abbreviation") != "P":
                    continue
                _pid = _p.get("person", {}).get("id")
                if _pid in (probable_id, None) or _used.get(_pid) == 1:
                    continue
                _spl = ((_p["person"].get("stats") or [{}])[0].get("splits") or [{}])
                _gs = int((_spl[0].get("stat") or {}).get("gamesStarted", 0) or 0)
                if _gs >= _PS_PIGGYBACK_MIN_GS:
                    _pool.append({"name": _p["person"].get("fullName"), "id": _pid, "gs": _gs,
                                  "rested": _pid not in _used})
            _pool.sort(key=lambda d: (not d["rested"], -d["gs"]))
            return _pool

        def _ps_news_role(team_id, probable_name, game_date):
            """MLB-PS-OPENER-NEWS-1 (2026-10-05 user audit -- "The detector missed Kay opening with Burke in bulk
            because Kay started 31 of 33 games this season. Pull the announced role from pregame news or manager
            quotes"). The GS-share rule can't see a regular starter used as an opener, so the announced plan is
            read from news headlines (Google News RSS, last 3 days): "turn to opener Anthony Kay", "Kay to open",
            "Burke set for bulk role behind Kay". Returns {"opener": True, "bulk": {...} or None, "source": headline}
            or None. Series/round openers ("NLDS opener") never match: the patterns need the pitcher's own name
            next to the opener/open wording. A named bulk arm is resolved against the team's active roster."""
            import re as _nr_re, html as _nr_html, requests as _nr_req, urllib.parse as _nr_up
            if not probable_name:
                return None
            _ck = ("news", int(team_id), str(probable_name), game_date.isoformat())
            if _ck in _ps_cache:
                return _ps_cache[_ck]
            _full = str(probable_name).strip(); _last = _full.split()[-1]
            _q = f'"{_full}" (opener OR bulk OR "open the game" OR piggyback) when:3d'
            _titles = []
            try:
                _rx = _nr_req.get("https://news.google.com/rss/search?q=" + _nr_up.quote(_q) + "&hl=en-US&gl=US&ceid=US:en",
                                  headers={"User-Agent": "Mozilla/5.0"}, timeout=15).text
                _titles = [_nr_html.unescape(_m) for _m in _nr_re.findall(r"<item>.*?<title>(.*?)</title>", _rx, _nr_re.S)][:20]
            except Exception:
                _titles = []
            _fe, _le = _nr_re.escape(_full), _nr_re.escape(_last)
            _op_pats = [rf"\bopener\s+(?:LHP\s+|RHP\s+)?{_fe}", rf"\b{_fe}\s+(?:to|will)\s+open\b", rf"\b{_le}\b[^|]{{0,60}}\bas (?:the |an )?opener\b",
                        rf"\bopen(?:s|ing)? (?:for|with)\s+(?:\w+\s+)?{_le}\b", rf"\bbulk\b[^|]{{0,30}}\bbehind\s+(?:\w+\s+)?{_le}\b"]
            _bulk_pats = [r"([A-Z][\w'.-]+(?:\s[A-Z][\w'.-]+)?)\s+(?:set for|to pitch in|will pitch in|in line for|expected to (?:pitch|work) in|slated for)\s+(?:a\s+)?bulk",
                          r"bulk (?:role|relief|innings)[^|]{0,15}\b(?:for|from|by)\s+([A-Z][\w'.-]+(?:\s[A-Z][\w'.-]+)?)",
                          rf"{_le}\s*,\s*([A-Z][\w'.-]+(?:\s[A-Z][\w'.-]+)?)\s+in\b"]
            _hit = next((t for t in _titles if any(_nr_re.search(pt, t, _nr_re.I) for pt in _op_pats)), None)
            if not _hit:
                _ps_cache[_ck] = None
                return None
            _bulk = None
            _ro = _ps_api(f"https://statsapi.mlb.com/api/v1/teams/{int(team_id)}/roster?rosterType=active&season={game_date.year}")
            _arms = [p_["person"] for p_ in (_ro or {}).get("roster", []) if p_.get("position", {}).get("abbreviation") == "P"]
            for _t in _titles:
                for _bp in _bulk_pats:
                    for _bm in _nr_re.finditer(_bp, _t):
                        _cand = _bm.group(1).strip()
                        _cl = _cand.split()[-1].lower()
                        _match = [a_ for a_ in _arms if a_.get("fullName", "").split()[-1].lower() == _cl
                                  and a_.get("fullName", "").lower() != _full.lower()]
                        if len(_match) == 1:
                            _bulk = {"name": _match[0]["fullName"], "id": _match[0]["id"], "source": _t}
                            break
                    if _bulk:
                        break
                if _bulk:
                    break
            _res = {"opener": True, "bulk": _bulk, "source": _hit}
            _ps_cache[_ck] = _res
            print(f"    [ANNOUNCED ROLE] {_full}: opener per news -- \"{_hit[:110]}\""
                  + (f"; bulk arm {_bulk['name']} -- \"{_bulk['source'][:90]}\"" if _bulk else "; bulk arm not named"))
            return _res

        def _ps_game_ctx(away_name, home_name):
            import datetime as _psg_dt, math as _psg_math
            _today = _psg_dt.date.today()
            _key = ("ctx", str(away_name).lower().strip(), str(home_name).lower().strip(), _today.isoformat())
            if _key in _ps_cache:
                return _ps_cache[_key]
            _sched = _ps_api(f"https://statsapi.mlb.com/api/v1/schedule?sportId=1&date={_today.isoformat()}"
                             f"&hydrate=probablePitcher,seriesStatus")
            _ctx = None
            for _sd in (_sched or {}).get("dates", []):
                for _g in _sd.get("games", []):
                    if _g.get("gameType") not in _PS_GAME_TYPES:
                        continue
                    _ta, _th = _g["teams"]["away"], _g["teams"]["home"]
                    if (_ta["team"]["name"].lower() != str(away_name).lower().strip()
                            or _th["team"]["name"].lower() != str(home_name).lower().strip()):
                        continue
                    _ss = _g.get("seriesStatus") or {}
                    _tot = int(_ss.get("totalGames") or _g.get("gamesInSeries") or 0)
                    _need = _psg_math.ceil(_tot / 2) if _tot else None
                    # real series tally from completed games (seriesStatus wins/losses
                    # carry no team identity) -- same approach as _mlb_series_record
                    _wins = {_ta["team"]["id"]: 0, _th["team"]["id"]: 0}
                    _hist = _ps_api(f"https://statsapi.mlb.com/api/v1/schedule?sportId=1&teamId={_th['team']['id']}"
                                    f"&startDate={_today - _psg_dt.timedelta(days=14)}&endDate={_today - _psg_dt.timedelta(days=1)}"
                                    f"&gameType=F,D,L,W")
                    for _hd in (_hist or {}).get("dates", []):
                        for _hg in _hd.get("games", []):
                            if _hg.get("status", {}).get("detailedState") != "Final":
                                continue
                            _ids = {_hg["teams"]["home"]["team"]["id"], _hg["teams"]["away"]["team"]["id"]}
                            if _ids != set(_wins):
                                continue
                            for _hs in ("home", "away"):
                                if _hg["teams"][_hs].get("isWinner"):
                                    _wins[_hg["teams"][_hs]["team"]["id"]] += 1
                    _aw, _hw = _wins[_ta["team"]["id"]], _wins[_th["team"]["id"]]
                    _elim = bool(_need and (_aw == _need - 1 or _hw == _need - 1))
                    _sides = {}
                    for _sn, _tm in (("away", _ta), ("home", _th)):
                        _pp = _tm.get("probablePitcher") or {}
                        _role, _desc = _ps_pitcher_role(_pp.get("id"))
                        _news = _ps_news_role(_tm["team"]["id"], _pp.get("fullName"), _today) if _role == "STARTER" else None
                        if _news:
                            _role = "OPENER"
                            _desc = f"{_desc}; announced opener per news: {_news['source'][:90]}"
                        # MLB-PS-OPENER-UNRATED-1 (2026-10-03): also built behind an announced
                        # opener in ANY postseason game -- the bulk arm is the same unknowable
                        # quantity as an elimination-game piggyback. Pricing below still only
                        # uses the pool in elimination games (unchanged); this feeds the label.
                        _pool = _ps_piggyback_pool(_tm["team"]["id"], _pp.get("id"), _today) if (_elim or _role == "OPENER") else []
                        _bulk_named = None
                        if _news and _news.get("bulk"):
                            _bn = _news["bulk"]
                            _bulk_named = {"name": _bn["name"], "id": _bn["id"], "gs": next((x["gs"] for x in _pool if x["id"] == _bn["id"]), 0),
                                           "rested": True, "source": _bn["source"]}
                            _pool = [_bulk_named] + [x for x in _pool if x["id"] != _bn["id"]]
                            globals().setdefault("mlb_announced_roles", {})[_tm["team"]["name"].lower()] = {
                                "opener": _pp.get("fullName"), "bulk": _bn["name"], "bulk_id": _bn["id"], "source": _bn["source"]}
                        _sides[_sn] = {"team": _tm["team"]["name"], "probable": _pp.get("fullName"),
                                       "role": _role, "role_desc": _desc, "piggyback_pool": _pool, "bulk_named": _bulk_named}
                    _ctx = {"game_pk": _g.get("gamePk"), "game_type": _g.get("gameType"),
                            "series": f"{_ta['team']['name']} {_aw}-{_hw} {_th['team']['name']}",
                            "away_wins": _aw, "home_wins": _hw, "games_in_series": _tot,
                            "elimination": _elim, "away": _sides["away"], "home": _sides["home"]}
            _ps_cache[_key] = _ctx
            return _ctx

        def _sp_role_flags(pname):
            if not pname or 'mlb_pitcher_fip_lookup' not in globals():
                return False, False, False
            import unicodedata as _rf_ud, re as _rf_re
            def _rf_norm(s):
                s = _rf_ud.normalize('NFD', str(s))
                s = ''.join(c for c in s if _rf_ud.category(c) != 'Mn')
                return _rf_re.sub(r'[^a-z ]', '', s.lower()).strip()
            _ev = mlb_pitcher_fip_lookup.get(_rf_norm(pname), {})
            return (bool(_ev.get('is_opener')), bool(_ev.get('is_bulk_role')),
                    bool(_ev.get('is_true_bullpen_game')))

        # BULLPEN-GAME BLEND (2026-09-11, user request): a confirmed true
        # bullpen game (see Cell 20's is_true_bullpen_game) gets weighted
        # even more heavily toward the bullpen than a confirmed opener --
        # an opener at least sometimes has SOME real-starter track record;
        # this pitcher has none at all, so his own SP-quality number carries
        # almost no information about how this game will actually play out.

        # SP WORKLOAD -> PROJECTED-INNINGS DISCOUNT (2026-09-26 user audit --
        # Detmers, 179+ IP already past his career-high single season, pulled
        # after 22 batters the same night this flag existed only as a
        # "[SP WORKLOAD]... informational only, no edge/tier change applied"
        # note -- the short leash that note predicted is exactly what exposed
        # the bullpen). Real, verified data: Detmers' 2026 season (185.0 IP
        # final) already exceeded his prior single-season career high (2023,
        # 148.2 IP) AND his combined 2024+2025 innings (87.1+63.2, an injury-
        # interrupted two years) by a wide margin. Follows the same
        # edge-reduction-as-a-visible-shift philosophy as MLB-EDGE-BLEND-1
        # (2026-09-25): this doesn't cap anything -- it moves real weight
        # from the starter's own run-environment number to the team's
        # bullpen rating in the SAME blend every other role flag here
        # (opener/bulk/bullpen-game) already uses, so a worn-down ace's
        # short-leash risk shows up as a real shift in projected runs, not
        # a separate cap layered on top. Gated to September+ (real
        # workload-cliff fatigue risk is a late-season phenomenon, not an
        # April one) and to seasons with at least _SP_WORKLOAD_MIN_PRIOR_IP
        # of real prior workload (excludes rookies/debut seasons with no
        # real career-high to compare against). PROVISIONAL magnitude
        # (0.50/0.50, between the normal 65/35 and bulk-role's 45/55) --
        # no backtest exists yet on how much a workload-cliff start should
        # actually discount; revisit after live grading.
        _SP_WORKLOAD_DISCOUNT_MONTH_MIN = 9
        _SP_WORKLOAD_MIN_PRIOR_IP = 20.0
        _sp_workload_cache = globals().setdefault("_SP_WORKLOAD_CAREER_HIGH_CACHE", {})
        def _sp_workload_exceeds_career_high(pname, ip_season_now):
            import datetime as _wl_dt
            if not pname or ip_season_now is None or pd.isna(ip_season_now):
                return False, None, None
            if _wl_dt.date.today().month < _SP_WORKLOAD_DISCOUNT_MONTH_MIN:
                return False, None, None
            _wl_key = str(pname).strip().lower()
            if _wl_key in _sp_workload_cache:
                _prior_high = _sp_workload_cache[_wl_key]
            else:
                _prior_high = None
                try:
                    _pid = _search_player_id(pname) if "_search_player_id" in globals() else None
                    if _pid:
                        _yr_data = requests.get(
                            f"{_MLB_API}/people/{_pid}/stats",
                            params={"stats": "yearByYear", "group": "pitching", "gameType": "R", "sportId": 1},
                            headers=_headers, timeout=10
                        ).json()
                        _cur_yr = _wl_dt.date.today().year
                        _ips = []
                        for _s in (_yr_data.get("stats", [{}])[0].get("splits", []) if _yr_data.get("stats") else []):
                            if int(_s.get("season", 0) or 0) >= _cur_yr:
                                continue  # excludes the CURRENT season -- this is a PRIOR career-high check
                            _ip_str = _s.get("stat", {}).get("inningsPitched")
                            if _ip_str is None:
                                continue
                            _whole, _, _frac = str(_ip_str).partition(".")
                            try:
                                _ip_val = float(_whole or 0) + (float(_frac or 0) / 3.0 if _frac else 0.0)
                            except ValueError:
                                continue
                            if _ip_val >= _SP_WORKLOAD_MIN_PRIOR_IP:
                                _ips.append(_ip_val)
                        _prior_high = max(_ips) if _ips else None
                except Exception:
                    _prior_high = None
                _sp_workload_cache[_wl_key] = _prior_high
            if _prior_high is None:
                return False, None, None
            _ip_now = float(ip_season_now)
            return (_ip_now > _prior_high), _ip_now, _prior_high
        globals()["_sp_workload_exceeds_career_high"] = _sp_workload_exceeds_career_high

        def _eff_pit(sp_ra, bpen_ra, fallback, is_opener=False, is_bulk=False, is_bullpen_game=False, is_workload_worn=False, is_elim_leash=False, is_postseason=False):
            if pd.notna(sp_ra) and pd.notna(bpen_ra):
                if is_bullpen_game:
                    return 0.10 * sp_ra + 0.90 * bpen_ra
                if is_opener:
                    return 0.25 * sp_ra + 0.75 * bpen_ra
                if is_bulk:
                    return 0.45 * sp_ra + 0.55 * bpen_ra
                if is_workload_worn:
                    return 0.50 * sp_ra + 0.50 * bpen_ra
                if is_elim_leash:
                    return _PS_ELIM_SP_SHARE * sp_ra + (1 - _PS_ELIM_SP_SHARE) * bpen_ra
                if is_postseason:   # MLB-PS-SHARE-1
                    return _PS_NONELIM_SP_SHARE * sp_ra + (1 - _PS_NONELIM_SP_SHARE) * bpen_ra
                return 0.65 * sp_ra + 0.35 * bpen_ra
            return sp_ra if pd.notna(sp_ra) else fallback
        _a_sp_val = safe_num(_row.get(_a_sp_ra)) if _a_sp_ra else np.nan
        _h_sp_val = safe_num(_row.get(_h_sp_ra)) if _h_sp_ra else np.nan
        _a_fb = safe_num(_row.get(_a_def)) if _a_def else np.nan
        _h_fb = safe_num(_row.get(_h_def)) if _h_def else np.nan
        _a_is_opener, _a_is_bulk, _a_is_bg = _sp_role_flags(_row.get("away_pitcher_name", ""))
        _h_is_opener, _h_is_bulk, _h_is_bg = _sp_role_flags(_row.get("home_pitcher_name", ""))
        # MLB-PS-SPROLE-1 (see helper block above): postseason role overrides
        _a_elim_leash = _h_elim_leash = False
        _ps_ctx = _ps_game_ctx(_row.get(away_col, ""), _row.get(home_col, ""))
        board.at[_idx, "ps_sp_role_unrated"] = False
        board.at[_idx, "ps_sp_role_note"] = ""
        if _ps_ctx:
            _ps_notes, _ps_unrated_bits = [], []
            for _ps_sn in ("away", "home"):
                _ps_s = _ps_ctx[_ps_sn]
                _ps_is_a = _ps_sn == "away"
                if _ps_s["role"] in ("TBD", "OPENER"):
                    if _ps_is_a:
                        _a_is_bg = True
                        if pd.isna(_a_sp_val):
                            _a_sp_val = _a_fb
                    else:
                        _h_is_bg = True
                        if pd.isna(_h_sp_val):
                            _h_sp_val = _h_fb
                    _ps_why = ("no probable posted" if _ps_s["role"] == "TBD"
                               else (f"announced probable {_ps_s['probable']} OPENS per news; bulk arm {_ps_s['bulk_named']['name']} graded as the SP"
                                     if _ps_s.get("bulk_named") else
                                     f"announced probable {_ps_s['probable']} is a reliever ({_ps_s['role_desc']}) -- opener + unknown bulk arm"))
                    _ps_notes.append(f"{_ps_s['team']}: {_ps_why} -> bullpen-game prior (90/10 bullpen/SP)")
                elif _ps_ctx["elimination"]:
                    if _ps_is_a:
                        _a_elim_leash = True
                    else:
                        _h_elim_leash = True
                    _ps_notes.append(f"{_ps_s['team']}: {_ps_s['probable']} starts an elimination game -> projected SP "
                                     f"innings discounted ({_PS_ELIM_SP_SHARE:.0%}/{1-_PS_ELIM_SP_SHARE:.0%} SP/bullpen vs 65/35)")
                # MLB-PS-OPENER-UNRATED-1 (2026-10-03 user audit -- "Braves ML printed 'SP TBD'.
                # Dylan Dodd had been announced as an opener. Gate 7d (UNRATED piggyback) should
                # have fired."). Gate 7d was scoped to elimination games only (10-01 spec), so an
                # announced opener in G1 of a DS got the 90/10 bullpen-game prior and stayed
                # eligible for OFFICIAL plays. Real case: Dodd (48 G / 0 GS in 2026) went 3.0 IP,
                # Smith-Shawver 2.2 IP of bulk behind him -- who throws innings 4-6 was exactly
                # the unknowable part. An announced opener now makes the side UNRATED in any
                # postseason game. (The 9AM card itself had no ATL probable posted at all --
                # Stats API showed none at 09:13 -- so that run printed SP TBD and the
                # starter-unresolved filter already hard-blocked the play; this covers the run
                # where the opener IS posted.)
                if _ps_s["role"] == "OPENER" and _ps_s.get("bulk_named"):
                    # MLB-PS-BULK-PRICE-1 (2026-10-06 user audit -- "Gate 7d should grade the bulk arm instead of
                    # blocking ... Keep the block only when the bulk arm is unknown"). Opener blocks with a known
                    # plan went 3-0 (Oct 4 ATL@LAD U8; Oct 5 CWS ML +130 and O6.5 -- Burke went 5.1 IP, 1 R).
                    # A NAMED bulk arm is priced as the SP (ML: his rating; totals: his innings + a short opener
                    # term for the first ~1.5 IP, MLB-TOT-COMP-1) and the side is NOT UNRATED.
                    _ps_notes.append(f"{_ps_s['team']}: opener {_ps_s['probable']} + named bulk arm {_ps_s['bulk_named']['name']} "
                                     f"-> priced with the bulk arm as SP (Gate 7d not applied)")
                elif _ps_s["role"] == "OPENER":
                    _ps_unrated_bits.append(
                        f"{_ps_s['team']} announced opener {_ps_s['probable']} ({_ps_s['role_desc']}) -- bulk arm unknown"
                        + (" (candidates: " + ", ".join(
                            f"{_pa['name']} ({_pa['gs']} GS{'' if _pa['rested'] else ', short rest'})"
                            for _pa in _ps_s["piggyback_pool"][:3]) + ")" if _ps_s["piggyback_pool"] else ""))
                elif _ps_ctx["elimination"] and _ps_s["piggyback_pool"]:
                    _ps_unrated_bits.append(f"{_ps_s['team']} can piggyback " + ", ".join(
                        f"{_pa['name']} ({_pa['gs']} GS{'' if _pa['rested'] else ', short rest'})"
                        for _pa in _ps_s["piggyback_pool"][:3]))
            _ps_hdr = (f"[POSTSEASON SP ROLE] {_ps_ctx['series']} (G{_ps_ctx['away_wins'] + _ps_ctx['home_wins'] + 1} of "
                       f"{_ps_ctx['games_in_series']}{', ELIMINATION' if _ps_ctx['elimination'] else ''})")
            print(f"  {_ps_hdr}")
            for _ps_n in _ps_notes:
                print(f"    {_ps_n}")
            if _ps_unrated_bits:
                _ps_unr_txt = (("OPENER/PIGGYBACK RISK -- " if any("announced opener" in _b for _b in _ps_unrated_bits)
                                else "PIGGYBACK RISK -- ") + "; ".join(_ps_unrated_bits) +
                               " -- who covers the middle/late innings is unknowable pregame, SP role UNRATED")
                print(f"    !! {_ps_unr_txt}")
                board.at[_idx, "ps_sp_role_unrated"] = True
                globals().setdefault("mlb_ps_role_unrated", {})[
                    (str(_row.get(away_col, "")).lower().strip(), str(_row.get(home_col, "")).lower().strip())] = _ps_unr_txt
            board.at[_idx, "ps_sp_role_note"] = " | ".join(_ps_notes)
        # Persisted so the win-probability SP nudge (_pit_adj, further below) can
        # also discount for a short-outing starter -- see that block for why.
        board.at[_idx, "away_sp_is_opener"] = bool(_a_is_opener)
        board.at[_idx, "home_sp_is_opener"] = bool(_h_is_opener)
        board.at[_idx, "away_sp_is_bulk"] = bool(_a_is_bulk)
        board.at[_idx, "home_sp_is_bulk"] = bool(_h_is_bulk)
        board.at[_idx, "away_sp_is_bullpen_game"] = bool(_a_is_bg)
        board.at[_idx, "home_sp_is_bullpen_game"] = bool(_h_is_bg)
        if _a_is_bg:
            print(f"  [BULLPEN GAME] {_row.get('away_pitcher_name','')}: no real starter "
                  f"track record this season -- effective SP quality treated almost entirely "
                  f"as bullpen strength (90/10 bullpen/SP blend, vs 75/25 for a confirmed opener)")
        elif _a_is_opener or _a_is_bulk:
            print(f"  [SP ROLE] {_row.get('away_pitcher_name','')}: "
                  f"{'opener' if _a_is_opener else 'bulk/short-role'} -- "
                  f"shifting run environment weight toward bullpen")
        if _h_is_bg:
            print(f"  [BULLPEN GAME] {_row.get('home_pitcher_name','')}: no real starter "
                  f"track record this season -- effective SP quality treated almost entirely "
                  f"as bullpen strength (90/10 bullpen/SP blend, vs 75/25 for a confirmed opener)")
        elif _h_is_opener or _h_is_bulk:
            print(f"  [SP ROLE] {_row.get('home_pitcher_name','')}: "
                  f"{'opener' if _h_is_opener else 'bulk/short-role'} -- "
                  f"shifting run environment weight toward bullpen")
        # FIX A: Weak SP run uplift — when pitcher ELO < 1420, boost SP RA 18%
        # Catches Painter-type starters (5.40 ERA) where model underpredicted runs badly
        if _have_elo and "mlb_pitcher_elo" in globals():
            def _sp_elo_lookup(pname):
                _pk = str(pname).lower().strip() if pname else ""
                return next((v for k,v in mlb_pitcher_elo.items() if _pk and _pk.split()[-1] in k), 1500)
            _a_sp_elo = _sp_elo_lookup(_row.get("away_pitcher_name",""))
            _h_sp_elo = _sp_elo_lookup(_row.get("home_pitcher_name",""))
            _BAD_SP_ELO = 1420
            _BAD_SP_MULT = 1.18
            if _a_sp_elo < _BAD_SP_ELO and pd.notna(_a_sp_val):
                _a_sp_val = _a_sp_val * _BAD_SP_MULT
            if _h_sp_elo < _BAD_SP_ELO and pd.notna(_h_sp_val):
                _h_sp_val = _h_sp_val * _BAD_SP_MULT

        # SP HOME/ROAD SPLIT -> RUN PROJECTION (2026-09-10, user audit, item 1):
        # this run-projection formula (_a_sp_val/_h_sp_val -> _away_pit/_home_pit
        # -> pred_away_runs/pred_home_runs) previously had NO home/road-split
        # preference at all -- confirmed and self-documented in this cell's own
        # M25 fix comment (see Cell 90's "[SP SPLIT]" bullet) and independently
        # verified live: deGrom 2.46 ERA/80.3 IP home vs 5.86 ERA/63.0 IP road, a
        # 3.4-run gap over comfortably more than 8 starts a side, and his
        # projected runs-allowed for tonight's ROAD start was still built off his
        # flat season-blended figure. Real case validated the user's exact rule
        # ("gap > 1.5R over 8+ starts -> use road ERA for away starts, not season
        # ERA") -- applied here as a multiplicative ratio (road_era/season_era or
        # home_era/season_era) rather than substituting ERA directly, since
        # _a_sp_val/_h_sp_val are a runs-ALLOWED-PER-START average, not a per-9-
        # innings rate, and a bare ERA substitution would silently change scale.
        # mlb_sp_split_era/_SP_SPLIT_MIN_GAP/_SP_SPLIT_MIN_STARTS are the exact
        # same dict/thresholds _sp_split_elo_adj (further below) already uses for
        # the win-probability side of this same signal -- one source of truth,
        # can't drift apart between the two pathways.
        def _sp_split_ra_ratio(pname, is_home_start):
            if not pname or 'mlb_sp_split_era' not in globals():
                return 1.0, None
            _sv = mlb_sp_split_era.get(str(pname).strip())
            if _sv is None:
                _nk = _pname_norm_sp(pname) if '_pname_norm_sp' in globals() else str(pname).lower().strip()
                _sv = next((v for k, v in mlb_sp_split_era.items()
                            if (_pname_norm_sp(k) if '_pname_norm_sp' in globals() else str(k).lower().strip()) == _nk), None)
            if not _sv:
                return 1.0, None
            _he, _re = _sv.get('home_era'), _sv.get('road_era')
            _seas = _sv.get('season_era')
            if _he is None or _re is None or not _seas:
                return 1.0, None
            if abs(_re - _he) < 1.50:
                return 1.0, None
            _starts = _sv.get('home_starts', 0) if is_home_start else _sv.get('road_starts', 0)
            if _starts < 8:
                return 1.0, None
            _this_era = _he if is_home_start else _re
            _ratio = _this_era / _seas if _seas > 0 else 1.0
            return _ratio, _sv

        _a_split_ratio, _a_split_sv = _sp_split_ra_ratio(_row.get("away_pitcher_name", ""), False)
        _h_split_ratio, _h_split_sv = _sp_split_ra_ratio(_row.get("home_pitcher_name", ""), True)
        if _a_split_ratio != 1.0 and pd.notna(_a_sp_val):
            print(f"  [SP SPLIT -> RUN PROJ] {_row.get('away_pitcher_name','')}: road ERA {_a_split_sv['road_era']:.2f} "
                  f"vs season ERA {_a_split_sv['season_era']:.2f} ({_a_split_sv['road_starts']} road starts) -- "
                  f"runs-allowed scaled {_a_split_ratio:.2f}x for tonight's road start")
            _a_sp_val = _a_sp_val * _a_split_ratio
            board.at[_idx, "away_sp_split_ra_ratio"] = round(_a_split_ratio, 3)
        if _h_split_ratio != 1.0 and pd.notna(_h_sp_val):
            print(f"  [SP SPLIT -> RUN PROJ] {_row.get('home_pitcher_name','')}: home ERA {_h_split_sv['home_era']:.2f} "
                  f"vs season ERA {_h_split_sv['season_era']:.2f} ({_h_split_sv['home_starts']} home starts) -- "
                  f"runs-allowed scaled {_h_split_ratio:.2f}x for tonight's home start")
            _h_sp_val = _h_sp_val * _h_split_ratio
            board.at[_idx, "home_sp_split_ra_ratio"] = round(_h_split_ratio, 3)

        def _wl_ip_season(pname):
            import unicodedata as _wlip_ud, re as _wlip_re
            def _wlip_norm(s):
                s = _wlip_ud.normalize("NFD", str(s))
                s = "".join(c for c in s if _wlip_ud.category(c) != "Mn")
                return _wlip_re.sub(r"[^a-z ]", "", s.lower()).strip()
            if not pname or "mlb_pitcher_fip_lookup" not in globals():
                return None
            return (mlb_pitcher_fip_lookup.get(_wlip_norm(pname), {}) or {}).get("ip_season")
        _a_workload_worn, _a_wl_ip, _a_wl_high = _sp_workload_exceeds_career_high(
            _row.get("away_pitcher_name", ""), _wl_ip_season(_row.get("away_pitcher_name", "")))
        _h_workload_worn, _h_wl_ip, _h_wl_high = _sp_workload_exceeds_career_high(
            _row.get("home_pitcher_name", ""), _wl_ip_season(_row.get("home_pitcher_name", "")))
        if _a_workload_worn and not (_a_is_bulk or _a_is_opener or _a_is_bg):
            print(f"  [SP WORKLOAD DISCOUNT] {_row.get('away_pitcher_name','')}: {_a_wl_ip:.1f} IP this season "
                  f"already past his prior career-high single season ({_a_wl_high:.1f} IP) -- shifting run-"
                  f"environment weight toward the bullpen (50/50, vs the normal 65/35 SP/bullpen split)")
        if _h_workload_worn and not (_h_is_bulk or _h_is_opener or _h_is_bg):
            print(f"  [SP WORKLOAD DISCOUNT] {_row.get('home_pitcher_name','')}: {_h_wl_ip:.1f} IP this season "
                  f"already past his prior career-high single season ({_h_wl_high:.1f} IP) -- shifting run-"
                  f"environment weight toward the bullpen (50/50, vs the normal 65/35 SP/bullpen split)")
        _away_pit = _eff_pit(_a_sp_val, _bpen_lookup.get(_norm_team(str(_row.get(away_col,''))), np.nan), _a_fb,
                              is_opener=_a_is_opener, is_bulk=_a_is_bulk, is_bullpen_game=_a_is_bg,
                              is_workload_worn=_a_workload_worn, is_elim_leash=_a_elim_leash,
                              is_postseason=bool(_ps_ctx))
        _home_pit = _eff_pit(_h_sp_val, _bpen_lookup.get(_hteam_key, np.nan), _h_fb,
                              is_opener=_h_is_opener, is_bulk=_h_is_bulk, is_bullpen_game=_h_is_bg,
                              is_workload_worn=_h_workload_worn, is_elim_leash=_h_elim_leash,
                              is_postseason=bool(_ps_ctx))
        # MLB-PS-PIGGYPRICE-1 (2026-10-01 user audit): when a piggyback is
        # detected, price the most-likely named bulk arm for the remaining
        # innings instead of the generic bullpen. Shares are PROVISIONAL (no
        # postseason backtest): announced opener -> 10% opener / 55% bulk arm /
        # 35% bullpen; real starter in an elimination game -> 50% starter / 30%
        # piggyback arm / 20% bullpen. The arm's number is his own
        # starter_runs_allowed_avg_10 from starter_snapshot (same per-start
        # scale as sp_ra), so no rescaling is needed.
        _ps_bulk_note = {}
        # MLB-PS-PIGGYPRICE-1 (2026-10-01 user audit): price the most-likely
        # named bulk arm for the innings it covers instead of treating them as
        # whatever the side's current pitching number assumes. Done as a
        # scale-free ratio, because this board's per-start RA columns are
        # empty (starter_snapshot not built) and the side's pitching number is
        # a team-level runs figure -- mixing in a raw ERA would mix scales:
        #     pit_new = pit_old * [(1 - s) + s * arm_est_era / team_era]
        # arm_est_era = the same FIP/xFIP/(SIERA|xERA) estimator as the SP
        # rating (so a lucky ERA like Holmes' 3.44 vs 5.05 xERA isn't trusted);
        # team_era = the club's season ERA (Stats API). Share s PROVISIONAL:
        # 0.55 behind an announced opener, 0.30 behind a real starter.
        if _ps_ctx and (_ps_ctx["elimination"] or any(_ps_ctx[_x].get("bulk_named") for _x in ("away", "home"))):
            for _ps_sn in ("away", "home"):
                _ps_s = _ps_ctx[_ps_sn]
                if not _ps_s["piggyback_pool"] or _ps_s["role"] == "TBD":
                    continue
                if not _ps_ctx["elimination"] and not _ps_s.get("bulk_named"):
                    continue   # MLB-PS-OPENER-NEWS-1: outside elimination games only a NAMED bulk arm is priced
                _arm = _ps_s["piggyback_pool"][0]
                _arm_v = (globals().get("mlb_pitcher_est_by_id", {}) or {}).get(_arm["id"])
                _arm_est, _arm_lab = _mlb_sp_est_era(_arm_v) if ("_mlb_sp_est_era" in globals() and _arm_v) else (None, None)
                _tm_id = None
                _tst = _ps_api(f"https://statsapi.mlb.com/api/v1/teams/stats?stats=season&group=pitching"
                               f"&season={__import__('datetime').date.today().year}&sportIds=1&gameType=R")
                for _ts in ((_tst or {}).get("stats") or [{}])[0].get("splits", []):
                    if _ts.get("team", {}).get("name", "").lower() == str(_ps_s["team"]).lower():
                        _tm_id = float(_ts["stat"].get("era"))
                if _arm_est is None or not _tm_id:
                    print(f"    [PIGGYBACK PRICED] {_ps_s['team']}: {_arm['name']} -- no estimator/team ERA, left unpriced")
                    continue
                _share = 0.55 if _ps_s["role"] == "OPENER" else 0.30
                _mult = (1 - _share) + _share * _arm_est / _tm_id
                _old_pit = _away_pit if _ps_sn == "away" else _home_pit
                if pd.isna(_old_pit):
                    continue
                _new_pit = _old_pit * _mult
                if _ps_sn == "away":
                    _away_pit = _new_pit
                else:
                    _home_pit = _new_pit
                _ps_bulk_note[_ps_sn] = _arm["name"]
                print(f"    [PIGGYBACK PRICED] {_ps_s['team']}: {_arm['name']} ({_arm['gs']} GS, estimator ERA "
                      f"{_arm_est:.2f} [{_arm_lab}] vs team ERA {_tm_id:.2f}) covers {_share:.0%} of the game -- "
                      f"pitching runs {_old_pit:.2f} -> {_new_pit:.2f} (x{_mult:.3f})")
        if _ps_ctx:
            globals().setdefault("mlb_ps_bulk_arm", {})[
                (str(_row.get(away_col, "")).lower().strip(), str(_row.get(home_col, "")).lower().strip())] = {
                _k: _ps_ctx[_k]["piggyback_pool"][0]["name"] if _ps_ctx[_k]["piggyback_pool"] else None for _k in ("away", "home")} | {
                f"{_k}_role": _ps_ctx[_k]["role"] for _k in ("away", "home")}
        board.at[_idx, "away_sp_workload_worn"] = bool(_a_workload_worn)
        board.at[_idx, "home_sp_workload_worn"] = bool(_h_workload_worn)

        _a_off_val = safe_num(_row.get(_a_off))
        _h_off_val = safe_num(_row.get(_h_off))

        # Extreme RA/G uplift: when opposing staff RA >= 7.0, boost 10%
        # Captures model underprojection vs historically weak pitching staff
        _EXTREME_RA   = 7.0
        _EXTREME_MULT = 1.10
        _home_pit_adj = (_home_pit * _EXTREME_MULT
                         if pd.notna(_home_pit) and _home_pit >= _EXTREME_RA else _home_pit)
        _away_pit_adj = (_away_pit * _EXTREME_MULT
                         if pd.notna(_away_pit) and _away_pit >= _EXTREME_RA else _away_pit)
        # away predicted runs = (away offense + home pitching) / 2 * environment
        if pd.notna(_a_off_val) and pd.notna(_home_pit):
            board.at[_idx, "pred_away_runs"] = round((_a_off_val + _home_pit_adj) / 2 * _env, 1)

        # home predicted runs = (home offense + away pitching) / 2 * environment
        if pd.notna(_h_off_val) and pd.notna(_away_pit):
            board.at[_idx, "pred_home_runs"] = round((_h_off_val + _away_pit_adj) / 2 * _env, 1)


    # WHIP/walk-rate adjustment to projected runs
    # High L3 WHIP means more baserunners � offense facing that SP scores more
    _WHIP_MILD_THRESH = 1.45
    _WHIP_HIGH_THRESH = 1.60
    _WHIP_ADJ_MILD    = 0.20
    _WHIP_ADJ_HIGH    = 0.40
    _whip_adj_ct = 0
    if 'mlb_pitcher_fip_lookup' in globals():
        import unicodedata as _w_ud, re as _w_re
        def _whip_norm(s):
            s = _w_ud.normalize('NFD', str(s).lower())
            s = ''.join(c for c in s if _w_ud.category(c) != 'Mn')
            return _w_re.sub(r'[^a-z ]', '', s).strip()
        for _widx in board.index:
            for _pncol, _runcol in [
                ("home_pitcher_name", "pred_away_runs"),
                ("away_pitcher_name", "pred_home_runs"),
            ]:
                _wpn = board.at[_widx, _pncol] if _pncol in board.columns else None
                if not _wpn or str(_wpn) in ('nan', ''): continue
                _wev = mlb_pitcher_fip_lookup.get(_whip_norm(str(_wpn)), {})
                _wh = _wev.get('whip_l3')
                if _wh is None or pd.isna(float(_wh)): continue
                _wh = float(_wh)
                if _wh <= _WHIP_MILD_THRESH: continue
                _wadj = _WHIP_ADJ_HIGH if _wh > _WHIP_HIGH_THRESH else _WHIP_ADJ_MILD
                if pd.notna(board.at[_widx, _runcol]):
                    board.at[_widx, _runcol] = round(board.at[_widx, _runcol] + _wadj, 1)
                    _whip_adj_ct += 1
    if _whip_adj_ct: print(f"  WHIP adj: {_whip_adj_ct} run projection(s) bumped (L3 WHIP >1.45)")


    # ---------------------------------------------------------
    # PLATOON ADJUSTMENT: OPS vs SP hand vs league average
    # ---------------------------------------------------------
    import unicodedata as _pl_ud, re as _pl_re
    def _pl_norm(s):
        if not s: return ""
        s = _pl_ud.normalize("NFD", str(s))
        s = "".join(c for c in s if _pl_ud.category(c) != "Mn")
        return _pl_re.sub(r"[^a-z0-9 ]", "", s.lower()).strip()

    _platoon_ct = 0
    if 'mlb_woba_splits' in globals() and 'mlb_sp_hand' in globals():
        for _pidx in board.index:
            # Away team bats against home SP
            _h_sp_name = board.at[_pidx, "home_pitcher_name"] if "home_pitcher_name" in board.columns else None
            _h_hand = mlb_sp_hand.get(_pl_norm(str(_h_sp_name or "")), "R")
            _aw_tn  = _pl_norm(str(board.at[_pidx, away_col] if away_col in board.columns else ""))
            _aw_delta = 0.0
            if _aw_tn:
                _aw_splits = mlb_woba_splits.get(_aw_tn, {})
                _side_key  = "vs_lhp" if _h_hand == "L" else "vs_rhp"
                _avg_ref   = globals().get("_avg_vs_lhp" if _h_hand == "L" else "_avg_vs_rhp", 0.720)
                _team_ops  = _aw_splits.get(_side_key)
                if _team_ops:
                    _aw_delta = float(np.clip((_team_ops - _avg_ref) / max(_avg_ref, 0.001) * 4.5, -0.5, 0.5))

            # Home team bats against away SP
            _a_sp_name = board.at[_pidx, "away_pitcher_name"] if "away_pitcher_name" in board.columns else None
            _a_hand = mlb_sp_hand.get(_pl_norm(str(_a_sp_name or "")), "R")
            _hm_tn  = _pl_norm(str(board.at[_pidx, home_col] if home_col in board.columns else ""))
            _hm_delta = 0.0
            if _hm_tn:
                _hm_splits = mlb_woba_splits.get(_hm_tn, {})
                _side_key2 = "vs_lhp" if _a_hand == "L" else "vs_rhp"
                _avg_ref2  = globals().get("_avg_vs_lhp" if _a_hand == "L" else "_avg_vs_rhp", 0.720)
                _team_ops2 = _hm_splits.get(_side_key2)
                if _team_ops2:
                    _hm_delta = float(np.clip((_team_ops2 - _avg_ref2) / max(_avg_ref2, 0.001) * 4.5, -0.5, 0.5))

            # Apply to projected runs
            if _aw_delta != 0.0 and pd.notna(board.at[_pidx, "pred_away_runs"]):
                board.at[_pidx, "pred_away_runs"] = round(board.at[_pidx, "pred_away_runs"] + _aw_delta, 1)
                _platoon_ct += 1
            if _hm_delta != 0.0 and pd.notna(board.at[_pidx, "pred_home_runs"]):
                board.at[_pidx, "pred_home_runs"] = round(board.at[_pidx, "pred_home_runs"] + _hm_delta, 1)
                _platoon_ct += 1
    if _platoon_ct:
        print(f"  Platoon adj: {_platoon_ct} run projection(s) adjusted for L/R matchup")

    # ---------------------------------------------------------
    # FIX A: LHP TOP-5 wOBA SPLIT BOOST
    # When opposing team ranks top-5 in wOBA vs LHP AND facing a LHP starter,
    # apply an additional +0.15R platoon-specific boost.
    # The general platoon delta is proportional; this catches elite LHP-punishing lineups.
    # ---------------------------------------------------------
    if 'mlb_woba_splits' in globals() and 'mlb_sp_hand' in globals() and mlb_woba_splits:
        _lhp_teams = [(tn, v.get("vs_lhp", 0)) for tn, v in mlb_woba_splits.items() if v.get("vs_lhp")]
        _lhp_teams_sorted = sorted(_lhp_teams, key=lambda x: -x[1])
        _top5_lhp = {tn for tn, _ in _lhp_teams_sorted[:5]}
        _lhp_boost_ct = 0
        for _lbidx in board.index:
            # Away team bats against home SP
            _lbh_sp = board.at[_lbidx, "home_pitcher_name"] if "home_pitcher_name" in board.columns else None
            _lbh_hand = mlb_sp_hand.get(_pl_norm(str(_lbh_sp or "")), "R")
            _lbaw_tn = _pl_norm(str(board.at[_lbidx, away_col] if away_col in board.columns else ""))
            # Tiered stack boost: count RHBs concentrated in lineup spots 1-5
            # 3-4 RHBs in spots 1-5: +0.12R (moderate stack), 5 RHBs: +0.18R (full stack)
            if _lbh_hand == "L" and pd.notna(board.at[_lbidx, "pred_away_runs"]):
                _lineup_key = str(board.at[_lbidx, away_col] if away_col in board.columns else "").lower()
                _lineup_data = globals().get("mlb_lineup_flags", {}).get(_lineup_key, {})
                _rhb_top5 = int(_lineup_data.get("rhb_top5", 0)) if isinstance(_lineup_data, dict) else 0
                # If no lineup breakdown available, fall back to team-level wOBA split
                if _rhb_top5 == 0 and _lbaw_tn in _top5_lhp:
                    _stack_boost = 0.15  # flat boost: team-level signal only
                    _stack_label = "top-5 wOBA vs LHP (team-level)"
                elif _rhb_top5 >= 5:
                    _stack_boost = 0.20  # full RHB stack in top 5
                    _stack_label = f"{_rhb_top5} RHBs in spots 1-5 (full stack)"
                elif _rhb_top5 >= 3:
                    _stack_boost = 0.12  # moderate stack
                    _stack_label = f"{_rhb_top5} RHBs in spots 1-5 (moderate stack)"
                else:
                    _stack_boost = 0.0
                    _stack_label = ""
                if _stack_boost > 0:
                    board.at[_lbidx, "pred_away_runs"] = round(board.at[_lbidx, "pred_away_runs"] + _stack_boost, 1)
                    _lhp_boost_ct += 1
                    print(f"  [LHP STACK BOOST] {_lbaw_tn} vs {_lbh_sp} (L) — +{_stack_boost:.2f}R ({_stack_label})")
            # Home team bats against away SP
            _lba_sp = board.at[_lbidx, "away_pitcher_name"] if "away_pitcher_name" in board.columns else None
            _lba_hand = mlb_sp_hand.get(_pl_norm(str(_lba_sp or "")), "R")
            _lbhm_tn = _pl_norm(str(board.at[_lbidx, home_col] if home_col in board.columns else ""))
            if _lba_hand == "L" and pd.notna(board.at[_lbidx, "pred_home_runs"]):
                _lineup_key2 = str(board.at[_lbidx, home_col] if home_col in board.columns else "").lower()
                _lineup_data2 = globals().get("mlb_lineup_flags", {}).get(_lineup_key2, {})
                _rhb_top5_h = int(_lineup_data2.get("rhb_top5", 0)) if isinstance(_lineup_data2, dict) else 0
                if _rhb_top5_h == 0 and _lbhm_tn in _top5_lhp:
                    _stack_boost2 = 0.15; _stack_label2 = "top-5 wOBA vs LHP (team-level)"
                elif _rhb_top5_h >= 5:
                    _stack_boost2 = 0.20; _stack_label2 = f"{_rhb_top5_h} RHBs in spots 1-5 (full stack)"
                elif _rhb_top5_h >= 3:
                    _stack_boost2 = 0.12; _stack_label2 = f"{_rhb_top5_h} RHBs in spots 1-5 (moderate stack)"
                else:
                    _stack_boost2 = 0.0; _stack_label2 = ""
                if _stack_boost2 > 0:
                    board.at[_lbidx, "pred_home_runs"] = round(board.at[_lbidx, "pred_home_runs"] + _stack_boost2, 1)
                    _lhp_boost_ct += 1
                    print(f"  [LHP STACK BOOST] {_lbhm_tn} vs {_lba_sp} (L) — +{_stack_boost2:.2f}R ({_stack_label2})")
    # ---------------------------------------------------------
    # FIX B: 60/40 SEASON ERA + L30 xFIP BLEND
    # When a starter's xFIP diverges from season ERA by >=0.75R,
    # adjust the opponent's run projection by 40% of the gap × (6/9 IP ratio).
    # Prevents hot/cold ERA from over- or under-stating true RA.
    # ---------------------------------------------------------
    _xfip_blend_ct = 0
    if 'mlb_pitcher_fip_lookup' in globals() and '_sp_fetch_recent' in dir() and '_sp_norm_era' in dir():
        _IP_PER_START = 6.0
        # Fix 1: 50% season ERA / 30% L5 ERA / 20% xFIP (was 60/40)
        # L5 captures mid-season collapses like Arrighetti (3.13 season, 6.95 June)
        _ERA_BLEND_W  = 0.20  # xFIP weight reduced from 0.40
        _ERA_BLEND_L5_W = 0.30  # new L5 ERA weight
        _ERA_SEASON_W = 0.50   # season ERA weight reduced from 0.60
        _ERA_BLEND_MIN_GAP = 0.75
        for _xbidx in board.index:
            for _xb_sp_col, _xb_run_col in [
                ("home_pitcher_name", "pred_away_runs"),  # home SP → adjust away batter runs
                ("away_pitcher_name", "pred_home_runs"),  # away SP → adjust home batter runs
            ]:
                _xb_spnm = str(board.at[_xbidx, _xb_sp_col] if _xb_sp_col in board.columns else "") or ""
                if not _xb_spnm or _xb_spnm in ("nan", ""): continue
                _xb_fip = mlb_pitcher_fip_lookup.get(_sp_norm_era(_xb_spnm), {})
                _xb_xfip = _xb_fip.get("xfip")
                if not _xb_xfip or (isinstance(_xb_xfip, float) and __import__("math").isnan(_xb_xfip)): continue
                _xb_xfip = float(_xb_xfip)
                _xb_spf = _sp_fetch_recent(_xb_spnm)
                if not _xb_spf or _xb_spf.get("era_season") is None: continue
                _xb_era_s = float(_xb_spf["era_season"])
                if _xb_era_s <= 0 or _xb_era_s > 12: continue
                # gap = xFIP - season_ERA; positive means ERA is running better than true talent
                # MLB-2: minimum 2 starts before L5/L3 ERA is meaningful
                _xb_gs_l3 = float(_xb_spf.get("gs_l3") or 0)
                # Fix 1b: incorporate L5 ERA into blend (only if 2+ starts in sample)
                # 2026-08-29 (user audit M2): this term is named _xb_l5 and its own
                # comment says "L5 ERA", but it read era_l3 -- era_l5/gs_l5 were added
                # to _sp_fetch_recent on 2026-08-25 and nothing ever consumed them, so
                # the whole blend/collapse path below still ran on a 3-start window.
                # Prefers the real 5-start number when the sample exists; falls back to
                # L3 (the previous behaviour) when it doesn't, so nothing that worked
                # before stops working.
                _xb_era_l5 = _xb_spf.get("era_l5")
                _xb_gs_l5 = float(_xb_spf.get("gs_l5") or 0)
                if _xb_era_l5 is not None and _xb_gs_l5 >= 5:
                    _xb_l5 = float(_xb_era_l5)
                    _xb_l5_win = "L5"
                elif _xb_gs_l3 >= 2:
                    _xb_l5 = float(_xb_spf.get("era_l3") or _xb_era_s)
                    _xb_l5_win = "L3"
                else:
                    _xb_l5 = _xb_era_s
                    _xb_l5_win = "season"
                # Blended target ERA = 50% season + 30% L5 + 20% xFIP
                _xb_blended = (_ERA_SEASON_W * _xb_era_s + _ERA_BLEND_L5_W * _xb_l5 +
                               _ERA_BLEND_W * _xb_xfip)
                _xb_gap = _xb_blended - _xb_era_s  # how much blended differs from pure season
                if abs(_xb_gap) < _ERA_BLEND_MIN_GAP: continue
                # Fix 1c: hard flag when recent ERA exceeds season ERA by 2.5+
                # 2026-08-29 (user audit M2): season ERA is a poor baseline for a
                # pitcher whose season number is ALREADY bloated -- his recent window
                # can be catastrophically worse than his own body of work while the
                # gap vs season stays under 2.5. Adds a second, additive trigger that
                # reads the recent window against the pitcher's own 10-start baseline
                # (era_l10, built 2026-08-25, previously only consumed by the absolute
                # >=5.00 sustained-cold shave). Strict superset -- the original
                # season-baseline trigger is unchanged.
                _xb_era_l10 = _xb_spf.get("era_l10")
                _xb_gs_l10 = float(_xb_spf.get("gs_l10") or 0)
                # Keep the ORIGINAL L3-vs-season trigger alongside the (more
                # stable) L5 read, so widening the window can only ever add
                # collapse detections, never remove one that used to fire.
                _xb_l3_raw = _xb_spf.get("era_l3")
                _xb_recent_worst = _xb_l5
                if _xb_l3_raw is not None and _xb_gs_l3 >= 2:
                    _xb_recent_worst = max(_xb_l5, float(_xb_l3_raw))
                _xb_collapse_vs_season = (_xb_recent_worst - _xb_era_s) >= 2.5
                _xb_collapse_vs_l10 = (
                    _xb_era_l10 is not None and _xb_gs_l10 >= 5
                    and (_xb_recent_worst - float(_xb_era_l10)) >= 2.5
                )
                _xb_collapse = _xb_collapse_vs_season or _xb_collapse_vs_l10
                if _xb_collapse:
                    # MLB-1: scale collapse penalty by opposing lineup quality
                    _xb_opp_col = "home_team" if _xb_sp_col == "away_pitcher_name" else "away_team"
                    _xb_opp_team = _norm_team(str(board.at[_xbidx, _xb_opp_col] if _xb_opp_col in board.columns else ""))
                    _xb_opp_woba = None
                    if "mlb_woba_splits" in globals() and mlb_woba_splits:
                        _xb_opp_woba = mlb_woba_splits.get(_xb_opp_team, {}).get("vs_rhp", None)
                    # Classify lineup: top/mid/bottom 10 by wOBA (~.330/.310/.290 thresholds)
                    _xb_lineup_tier = "mid"
                    if _xb_opp_woba:
                        if _xb_opp_woba >= 0.330: _xb_lineup_tier = "top"
                        elif _xb_opp_woba <= 0.295: _xb_lineup_tier = "bottom"
                    _xb_collapse_scale = {"top": 1.0, "mid": 0.75, "bottom": 0.50}.get(_xb_lineup_tier, 0.75)
                    _xb_clps_base = "season" if _xb_collapse_vs_season else "L10"
                    _xb_clps_ref = _xb_era_s if _xb_collapse_vs_season else float(_xb_era_l10)
                    _xb_clps_win = _xb_l5_win if _xb_recent_worst == _xb_l5 else "L3"
                    print(f"  !! [SP COLLAPSE] {_xb_spnm}: {_xb_clps_base} ERA {_xb_clps_ref:.2f} vs {_xb_clps_win} {_xb_recent_worst:.2f} — {_xb_recent_worst-_xb_clps_ref:.1f}R collapse vs {_xb_lineup_tier}-tier lineup (adj ×{_xb_collapse_scale:.2f})")
                else:
                    _xb_collapse_scale = 1.0
                _run_adj_cap = (1.5 * _xb_collapse_scale) if _xb_collapse else 1.0
                _xb_run_adj = round(_xb_gap * _IP_PER_START / 9.0, 2)
                _xb_run_adj = float(np.clip(_xb_run_adj, -_run_adj_cap, _run_adj_cap))
                if pd.notna(board.at[_xbidx, _xb_run_col]):
                    board.at[_xbidx, _xb_run_col] = round(board.at[_xbidx, _xb_run_col] + _xb_run_adj, 1)
                    _xfip_blend_ct += 1
                    _dir = "+" if _xb_run_adj > 0 else ""
                    print(f"  [xFIP BLEND] {_xb_spnm}: ERA {_xb_era_s:.2f} / xFIP {_xb_xfip:.2f} (gap {_xb_gap:+.2f}) — {_dir}{_xb_run_adj:.2f}R to opponent")
    if _xfip_blend_ct: print(f"  xFIP blend: {_xfip_blend_ct} run projection(s) adjusted")

    # ---------------------------------------------------------
    # TEAM IL + SP HEALTH CHECK
    # ---------------------------------------------------------
    _MLB_ESPN_SLUGS = {
        "yankees":"nyy","red sox":"bos","blue jays":"tor","orioles":"bal","rays":"tb",
        "white sox":"chw","guardians":"cle","tigers":"det","royals":"kc","twins":"min",
        "athletics":"oak","astros":"hou","angels":"laa","mariners":"sea","rangers":"tex",
        "braves":"atl","marlins":"mia","mets":"nym","phillies":"phi","nationals":"wsh",
        "cubs":"chc","reds":"cin","brewers":"mil","pirates":"pit","cardinals":"stl",
        "diamondbacks":"ari","rockies":"col","dodgers":"la","padres":"sd","giants":"sf",
    }

    def _mlb_slug(team_name):
        _tn = str(team_name).lower()
        for k, v in _MLB_ESPN_SLUGS.items():
            if k in _tn: return v
        return None

    # BUG FIX (found live 2026-07-23): the per-team injuries sub-resource
    # (.../teams/{slug}/injuries) returns an empty {} for every MLB team --
    # confirmed live against tor, nyy, and the numeric ESPN team id, all
    # empty. _fetch_espn_injuries has been silently returning [] on every
    # call since this was written, which means the INJURY STACK DETECTOR
    # below (and the per-game injury display) never actually had real data
    # to work with -- e.g. TOR's Jonatan Clase (LF) and Jesus Sanchez (RF),
    # both real 10-Day-IL entries right now, never got picked up. The
    # league-wide endpoint below returns all 30 teams' injuries (with
    # embedded athlete name/position) in a single working call.
    _ESPN_INJURY_CACHE = None
    _ESPN_INJURY_FETCH_OK = None

    # 2026-08-31 BUG FIX (confirmed live via curl): every ESPN call from
    # this network is edge-blocked by Akamai with a 403 "Access Denied"
    # page (Server: AkamaiGHost -- ESPN's own app never sees the
    # request), not a rate limit -- reproduced on a single isolated call
    # with no burst context, and this exact endpoint has separately been
    # confirmed 403'ing standalone (see the news-cache comment below).
    # Shared by the injury/news/standings ESPN calls in this cell --
    # direct call tried first, only a 403 retries once through
    # r.jina.ai's free reader-proxy relay (different egress IP, verified
    # live to return a real 200 against this exact endpoint kind).
    # Returns the same status_code/.json() shape callers already check.
    class _EspnRelayResponse:
        def __init__(self, status_code, json_data):
            self.status_code = status_code
            self._json_data = json_data
        def json(self):
            # BUG FIX (found live 2026-08-31, real crash elsewhere in this
            # notebook): raise instead of silently returning None on total
            # failure -- defense in depth, even though this cell's callers
            # already check status_code == 200 before calling .json().
            if self._json_data is None:
                raise ValueError("ESPN fetch failed (both direct call and relay fallback)")
            return self._json_data

    def _espn_get(url, headers=None, timeout=10):
        _headers = headers or {"User-Agent": "Mozilla/5.0"}
        try:
            r = requests.get(url, headers=_headers, timeout=timeout)
            if r.status_code != 403:
                return r
        except Exception:
            pass
        try:
            rr = requests.get(f"https://r.jina.ai/{url}", timeout=timeout + 20)
            rr.raise_for_status()
            body = rr.text
            idx = body.find("Markdown Content:")
            json_text = body[idx + len("Markdown Content:"):].strip() if idx >= 0 else body
            import json as _json
            return _EspnRelayResponse(200, _json.loads(json_text))
        except Exception:
            return _EspnRelayResponse(403, None)

    def _load_espn_injury_cache():
        global _ESPN_INJURY_CACHE, _ESPN_INJURY_FETCH_OK
        if _ESPN_INJURY_CACHE is not None:
            return _ESPN_INJURY_CACHE
        _cache = {}
        try:
            _resp = _espn_get(
                "https://site.api.espn.com/apis/site/v2/sports/baseball/mlb/injuries",
                headers={"User-Agent": "Mozilla/5.0"}, timeout=10
            )
            if _resp.status_code == 200:
                for _t in _resp.json().get("injuries", []):
                    _tnm = str(_t.get("displayName", "")).lower()
                    _items = []
                    for _it in _t.get("injuries", []):
                        _ath = _it.get("athlete", {})
                        _nm  = _ath.get("displayName", "")
                        _st  = _it.get("status", "")
                        _pos = _ath.get("position", {}).get("abbreviation", "")
                        if _nm: _items.append({"name": _nm, "status": _st, "pos": _pos})
                    _cache[_tnm] = _items
                _lic_total = sum(len(v) for v in _cache.values())
                print(f"  ESPN injuries loaded: {len(_cache)} teams, {_lic_total} listed injuries")
                _ESPN_INJURY_FETCH_OK = True
            else:
                print(f"  [WARN] League-wide ESPN injuries fetch returned HTTP {_resp.status_code} -- "
                      f"treating as no injury data available, NOT as a clean slate")
                _ESPN_INJURY_FETCH_OK = False
        except Exception as _lic_e:
            print(f"  [WARN] League-wide ESPN injuries fetch failed: {_lic_e}")
            _ESPN_INJURY_FETCH_OK = False
        _ESPN_INJURY_CACHE = _cache
        return _cache

    _MLB_STATSAPI_IL_CODES = {"D7", "D10", "D15", "D60"}
    _MLB_ROSTER_STATUS_CACHE = {}

    def _mlb_statsapi_roster_injuries(team_name):
        # Fallback source when ESPN's injuries endpoint is unavailable --
        # official IL status straight from the same MLB Stats API roster
        # call this cell already trusts for OPS lookups (_mlb_roster_for_team),
        # just reading the status field that fetch normally discards. Only
        # covers actual IL stints (D7/D10/D15/D60), not day-to-day/game-time
        # decisions -- those never appear on a 40Man roster status at all,
        # same real gap ESPN's news-feed supplement above already exists for.
        _tid_map = _mlb_team_id_lookup()
        _tn = str(team_name).lower()
        _tid = next((v for k, v in _tid_map.items() if v and (k in _tn or _tn in k)), None)
        if not _tid:
            return []
        if _tid in _MLB_ROSTER_STATUS_CACHE:
            return _MLB_ROSTER_STATUS_CACHE[_tid]
        _items = []
        try:
            _rr = requests.get(f"{_MLB_API}/teams/{_tid}/roster?rosterType=40Man", timeout=10).json()
            for _p in _rr.get("roster", []):
                _code = _p.get("status", {}).get("code", "")
                if _code in _MLB_STATSAPI_IL_CODES:
                    _items.append({
                        "name": _p.get("person", {}).get("fullName", ""),
                        "status": _p.get("status", {}).get("description", ""),
                        "pos": _p.get("position", {}).get("abbreviation", ""),
                    })
        except Exception:
            pass
        _MLB_ROSTER_STATUS_CACHE[_tid] = _items
        return _items

    def _fetch_espn_injuries(team_name):
        _cache = _load_espn_injury_cache()
        if not _ESPN_INJURY_FETCH_OK:
            return _mlb_statsapi_roster_injuries(team_name)
        _tn = str(team_name).lower()
        for _k, _v in _cache.items():
            if _tn in _k or _k in _tn:
                return _v
        # fall back to short-name matching via the existing slug map
        for _short in _MLB_ESPN_SLUGS:
            if _short in _tn:
                for _k, _v in _cache.items():
                    if _short in _k:
                        return _v
        return []

    # 2026-08-13 (user audit, Buxton/Harper): the league-wide injuries endpoint
    # above only lists official IL entries -- it correctly returns [] for a
    # player who's day-to-day, a game-time decision, or was JUST activated,
    # since none of those are IL moves. That's not a fetch failure, it's a
    # real data-source gap: a lineup-relevant player (Buxton activated for
    # tonight's game; confirmed live via ESPN's news feed, headline "Twins
    # activate Byron Buxton for Field of Dreams game vs. Phillies") never
    # shows up anywhere on the card. Adding a second, independent source --
    # ESPN's MLB news feed, last 48h, scanned for lineup-status keywords --
    # as a supplement whenever the IL list comes back empty (or always, since
    # an IL entry and a same-day news item aren't mutually exclusive).
    _ESPN_NEWS_CACHE = None
    # CROSS-MODEL FIX 12/18 (2026-08-27, user audit): the injuries endpoint
    # right above already tracks _ESPN_INJURY_FETCH_OK and falls back to
    # MLB Stats API IL data on failure -- but a 403/failure on THIS endpoint
    # (day-to-day/game-time-decision/injury-scare news, no Stats-API
    # equivalent exists for that) used to just print a WARN and silently
    # continue as if "no news" meant "clean bill of health". Tracked the
    # same way so the final card can tell the difference and flag degraded
    # reliability instead of a false all-clear.
    _ESPN_NEWS_FETCH_OK = None
    _LINEUP_NEWS_WORDS = [
        "day-to-day", "dtd", "game-time decision", "questionable", "scratched",
        "scratch", "activated", "activate", "reinstated", "will not start",
        "won't start", "out tonight", "placed on", "injured list",
    ]

    def _load_espn_news_cache():
        global _ESPN_NEWS_CACHE, _ESPN_NEWS_FETCH_OK
        if _ESPN_NEWS_CACHE is not None:
            return _ESPN_NEWS_CACHE
        _articles = []
        try:
            _nresp = _espn_get(
                "https://site.api.espn.com/apis/site/v2/sports/baseball/mlb/news?limit=50",
                headers={"User-Agent": "Mozilla/5.0"}, timeout=10
            )
            if _nresp.status_code == 200:
                _ESPN_NEWS_FETCH_OK = True
                # 2026-08-24: date filtering moved OUT of this shared fetch -- it now
                # returns all up-to-50 raw articles (each still carries "pub"), and
                # each consumer (_fetch_lineup_news: 48h, _fetch_sp_injury_news: 10d)
                # applies its own cutoff. Originally this function baked in a fixed
                # 48h cutoff, which was fine when it had one consumer -- adding a
                # second consumer needing a wider window would have meant a second,
                # fully duplicate HTTP call to the same endpoint (doubling traffic to
                # an endpoint that turned out to already be rate-limiting/blocking
                # this IP under normal single-call volume -- confirmed via a direct
                # standalone test hitting a 403 minutes after a normal notebook run).
                for _art in _nresp.json().get("articles", []):
                    _pub = _art.get("published", "")
                    _articles.append({
                        "headline": _art.get("headline", ""),
                        "desc": _art.get("description", ""),
                        "pub": _pub,
                    })
            else:
                print(f"  [WARN] ESPN news fetch returned status {_nresp.status_code} -- treating as no news available")
                _ESPN_NEWS_FETCH_OK = False
        except Exception as _nc_e:
            print(f"  [WARN] ESPN news fetch failed: {_nc_e}")
            _ESPN_NEWS_FETCH_OK = False
        _ESPN_NEWS_CACHE = _articles
        return _articles

    def _fetch_lineup_news(team_name):
        """Recent (48h) lineup-status headlines for a team, independent of the IL feed."""
        _tn = str(team_name).lower()
        _hits = []
        import datetime as _fln_dt
        _fln_cutoff = _fln_dt.datetime.now(_fln_dt.timezone.utc) - _fln_dt.timedelta(hours=48)
        for _art in _load_espn_news_cache():
            try:
                _fln_pub_dt = _fln_dt.datetime.fromisoformat(str(_art.get("pub", "")).replace("Z", "+00:00"))
                if _fln_pub_dt < _fln_cutoff:
                    continue
            except Exception:
                pass
            _text = (_art["headline"] + " " + _art["desc"]).lower()
            if _tn not in _text and not any(w in _text for w in _tn.split() if len(w) > 4):
                continue
            if any(kw in _text for kw in _LINEUP_NEWS_WORDS):
                _hits.append(_art["headline"])
        return _hits
    # 2026-08-24 (user audit, Kevin Gausman): _fetch_lineup_news above is
    # TEAM-level and only checks the last 48h -- fine for the hitter-lineup
    # case it was built for, but too short for "pitcher had an injury scare
    # 5 days before a confirmed start, threw a bullpen session, cleared" --
    # by raceday that news is already outside a 48h window, and nothing
    # checks a SPECIFIC pitcher's own name against news text at all. Reuses
    # the SAME shared _load_espn_news_cache() fetch as _fetch_lineup_news
    # (see the 2026-08-24 note on that function -- a second, separate HTTP
    # call to the identical endpoint was tried first and dropped after it
    # coincided with the endpoint 403-blocking this IP under doubled
    # traffic) with its own wider 10-day cutoff applied at read time.
    _SP_INJURY_WORDS = _LINEUP_NEWS_WORDS + [
        "injury", "injured", "sore", "soreness", "discomfort", "bullpen session",
        "cleared to", "setback", "mri", "x-ray", "surgery", "procedure",
    ]

    def _fetch_sp_injury_news(pitcher_name):
        """Recent (10-day) headlines mentioning this pitcher by name, matching injury/status keywords."""
        if not pitcher_name:
            return []
        _pn_parts = str(pitcher_name).lower().split()
        _pn_last = _pn_parts[-1] if _pn_parts else ""
        if len(_pn_last) <= 3:
            return []
        import datetime as _spn_dt
        _spn_cutoff = _spn_dt.datetime.now(_spn_dt.timezone.utc) - _spn_dt.timedelta(days=10)
        _hits = []
        for _art in _load_espn_news_cache():
            try:
                _spn_pub_dt = _spn_dt.datetime.fromisoformat(str(_art.get("pub", "")).replace("Z", "+00:00"))
                if _spn_pub_dt < _spn_cutoff:
                    continue
            except Exception:
                pass
            _text = (_art["headline"] + " " + _art["desc"]).lower()
            if _pn_last not in _text:
                continue
            if any(kw in _text for kw in _SP_INJURY_WORDS):
                _hits.append(_art["headline"])
        return _hits

    globals()["_fetch_sp_injury_news"] = _fetch_sp_injury_news


    # ---------------------------------------------------------
    # 2026-07-30: injury-stack WAR-weighting. No WAR (season or career) exists
    # anywhere in this notebook -- verified before building this. Season OPS
    # does exist and already flows through the posted-lineup OPS-gap gate
    # above, fetched per player ID via the MLB Stats API. ESPN's injury feed
    # only gives a name, not an ID, so this resolves name -> team roster ->
    # player ID -> season OPS, reusing the same stats endpoint/season as that
    # gate. Lineup slot was considered too (batting order matters more than
    # WAR alone -- Witt hitting 2nd disrupts more than a bench OF), but the
    # posted lineup that would supply it doesn't exist yet when this detector
    # needs to fire (that's the whole reason this gate exists as an overnight
    # fallback) -- structurally unavailable, not a build gap.
    # ---------------------------------------------------------
    import unicodedata as _is_ud, re as _is_re
    def _is_norm_name(s):
        s = _is_ud.normalize("NFD", str(s))
        s = "".join(c for c in s if _is_ud.category(c) != "Mn")
        return _is_re.sub(r"[^a-z ]", "", s.lower()).strip()

    # ── FIX A: SP IL RETURN DETECTOR ─────────────────────────────────────────
    # Fetch recent MLB transactions — flag pitchers activated from IL in last 21d
    _il_activations = {}  # {player_name_lower: (date_str, description, days_since)}
    try:
        import datetime as _ild
        _il_today = _ild.date.today()
        _il_start = (_il_today - _ild.timedelta(days=21)).strftime("%Y-%m-%d")
        _il_end   = (_il_today + _ild.timedelta(days=1)).strftime("%Y-%m-%d")
        _il_resp  = requests.get(
            f"https://statsapi.mlb.com/api/v1/transactions?startDate={_il_start}&endDate={_il_end}&sportId=1",
            timeout=8
        )
        if _il_resp.status_code == 200:
            _all_txns = _il_resp.json().get("transactions", [])
            for _tr in _all_txns:
                _tr_type = (_tr.get("typeDesc") or _tr.get("type", {}).get("description", "")).lower()
                # MLB Stats API uses 'recalled' for IL activations (not 'activated'/'reinstated')
                # 'recalled' also covers minors recalls, but only today's SP names will fire the warning
                _is_il_return = (
                    "recalled" in _tr_type
                    or ("activated" in _tr_type or "reinstated" in _tr_type)
                )
                if _is_il_return:
                    _tr_name = str(_tr.get("person", {}).get("fullName", "")).lower().strip()
                    _tr_date = str(_tr.get("date", ""))[:10]
                    if _tr_name and _tr_date:
                        try:
                            _days = (_il_today - _ild.date.fromisoformat(_tr_date)).days
                        except Exception:
                            _days = 99
                        if _tr_name not in _il_activations or _days < _il_activations[_tr_name][2]:
                            _il_activations[_tr_name] = (_tr_date, _tr_type, _days)
            print(f"  IL activations loaded: {len(_il_activations)} players activated/reinstated from IL in last 21d")
            if len(_il_activations) == 0 and _all_txns:
                _sample_types = sorted(set(
                    (_tr.get("typeDesc") or _tr.get("type",{}).get("description","")).lower()
                    for _tr in _all_txns[:50]
                ))[:8]
                print(f"  IL debug: {len(_all_txns)} total txns; types seen: {_sample_types}")
        else:
            print(f"  IL activations: API returned {_il_resp.status_code}")
    except Exception as _ile:
        print(f"  IL activations fetch skipped: {_ile}")
    # END IL RETURN DETECTOR ───────────────────────────────────────────────

    # ── FIX B: IL PLACEMENT-DATE LOOKUP (MLB-INJ-RECENCY-1, 2026-09-25
    # user audit) ─────────────────────────────────────────────────────────
    # "Seattle got -0.96R, but Raley has been out since August, so his
    # absence is already in the team's run baseline." The injury-stack
    # pricing below used to charge every OUT player's full OPS-derived
    # penalty regardless of how long he'd been out -- a team's own
    # rolling/season run-scoring numbers already reflect a long-standing
    # absence, so pricing it AGAIN double-counts the same real effect.
    # Reuses _all_txns (already fetched above for the IL-RETURN check,
    # same 21-day window, no new API call) to find each OUT player's own
    # "placed on the ID/IL" transaction and how many days ago it happened.
    _il_placements = {}  # {player_name_lower: days_since_placed}
    try:
        for _tr in _all_txns:
            _tr_type = (_tr.get("typeDesc") or _tr.get("type", {}).get("description", "")).lower()
            if "injured list" not in _tr_type and " il" not in f" {_tr_type}" and "placed on" not in _tr_type:
                continue
            _tr_name = str(_tr.get("person", {}).get("fullName", "")).lower().strip()
            _tr_date = str(_tr.get("date", ""))[:10]
            if not _tr_name or not _tr_date:
                continue
            try:
                _days = (_il_today - _ild.date.fromisoformat(_tr_date)).days
            except Exception:
                continue
            if _tr_name not in _il_placements or _days < _il_placements[_tr_name]:
                _il_placements[_tr_name] = _days
        print(f"  IL placements loaded: {len(_il_placements)} player(s) placed on IL in last 21d")
    except Exception as _ilp_e:
        print(f"  IL placement lookup skipped: {_ilp_e}")
    _ABSENCE_RECENCY_MAX_DAYS = 14  # only price absences newer than this -- older ones are already reflected in team form

    _MLB_TEAM_ID_CACHE = None
    def _mlb_team_id_lookup():
        global _MLB_TEAM_ID_CACHE
        if _MLB_TEAM_ID_CACHE is not None:
            return _MLB_TEAM_ID_CACHE
        _m = {}
        try:
            _r = requests.get(f"{_MLB_API}/teams?sportId=1", timeout=10).json()
            for _t in _r.get("teams", []):
                if _t.get("name"): _m[_t["name"].lower()] = _t.get("id")
                if _t.get("teamName"): _m[_t["teamName"].lower()] = _t.get("id")
        except Exception:
            pass
        _MLB_TEAM_ID_CACHE = _m
        return _m

    _MLB_ROSTER_CACHE = {}
    def _mlb_roster_for_team(team_name):
        _tid_map = _mlb_team_id_lookup()
        _tn = str(team_name).lower()
        _tid = next((v for k, v in _tid_map.items() if v and (k in _tn or _tn in k)), None)
        if not _tid:
            return {}
        if _tid in _MLB_ROSTER_CACHE:
            return _MLB_ROSTER_CACHE[_tid]
        _roster = {}
        try:
            _r = requests.get(f"{_MLB_API}/teams/{_tid}/roster?rosterType=40Man", timeout=10).json()  # 40Man, not the default active-26 view -- IL players are excluded from the default and this needs exactly those
            for _p in _r.get("roster", []):
                _nm = _p.get("person", {}).get("fullName", "")
                _pid = _p.get("person", {}).get("id")
                if _nm and _pid:
                    _roster[_is_norm_name(_nm)] = _pid
        except Exception:
            pass
        _MLB_ROSTER_CACHE[_tid] = _roster
        return _roster

    _MLB_PLAYER_OPS_CACHE = {}
    def _mlb_player_season_ops(player_id):
        # BUG FIX (MLB-INJ-SMALLSAMPLE-1, 2026-09-25 user audit): "Will
        # Wilson's 1.133 OPS came from a handful of at-bats; his career
        # line is closer to .192" -- now also pulls plateAppearances from
        # the SAME response (one more field, no new API call) so the
        # regression-to-mean formula below can shrink a thin-PA read
        # toward league average instead of trusting it at face value.
        if player_id in _MLB_PLAYER_OPS_CACHE:
            return _MLB_PLAYER_OPS_CACHE[player_id]
        _ops, _pa = None, None
        try:
            _r = requests.get(
                f"{_MLB_API}/people/{player_id}/stats?stats=season&group=hitting&season={_SEASON}",
                timeout=6).json()
            _splits = _r.get("stats", [{}])[0].get("splits", [])
            if _splits:
                _stat = _splits[0].get("stat", {})
                _v = _stat.get("ops")
                if _v is not None:
                    _ops = float(_v)
                _pav = _stat.get("plateAppearances")
                if _pav is not None:
                    _pa = int(_pav)
        except Exception:
            pass
        _MLB_PLAYER_OPS_CACHE[player_id] = (_ops, _pa)
        return _ops, _pa

    # BUG FIX (MLB-INJ-RECENCY-1 / MLB-INJ-SMALLSAMPLE-1, 2026-09-25 user
    # audit): replaces the OPS tier-bucket table with the continuous
    # "Option A" scaling that comment already flagged as the eventual
    # target. Reuses _RUNS_PER_OPS_POINT=1.75 -- the SAME real, already-
    # established slope the lineup-confirmed OPS-gap gate elsewhere in
    # this cell uses (matches the implicit scale the old bucket table was
    # built from: 0.35R / a 0.20 OPS band = 1.75), not a new guessed
    # number. Three real changes: (1) regresses a thin-PA OPS toward
    # league average (200-PA shrinkage, the exact formula requested) so a
    # few hot/cold at-bats can't read as a real talent level; (2) prices
    # the loss as player-minus-REPLACEMENT-level, not player-minus-zero --
    # a below-average bench bat missing still costs something above a
    # true replacement, but far less than the old flat 0.08 floor implied
    # for every single below-average player regardless of how far below;
    # (3) _REPLACEMENT_OPS is a standard sabermetric convention
    # (~league-average minus 0.08, roughly a freely-available call-up's
    # level), NOT backtested against this specific model -- flagged
    # PROVISIONAL like every other unbacktested constant in this cell.
    _RUNS_PER_OPS_POINT = 1.75  # matches the lineup-confirmed OPS-gap gate elsewhere in this cell
    _REPLACEMENT_OPS = 0.640    # provisional -- standard sabermetric replacement-level convention (~_LEAGUE_OPS-0.08), not backtested here
    _OPS_REGRESSION_PA = 200    # shrinkage strength -- user-specified
    _INJ_LEAGUE_OPS = 0.720     # matches _LEAGUE_OPS defined earlier in this notebook (Cell -- PLATOON + wOBA SPLITS)

    def _ops_regressed(ops, pa):
        if ops is None:
            return None
        if pa is None or pa < 0:
            pa = 0
        return (ops * pa + _INJ_LEAGUE_OPS * _OPS_REGRESSION_PA) / (pa + _OPS_REGRESSION_PA)

    def _ops_to_run_deduction(ops, pa=None):
        if ops is None:
            return None
        _ops_adj = _ops_regressed(ops, pa)
        return round(max(0.0, _ops_adj - _REPLACEMENT_OPS) * _RUNS_PER_OPS_POINT, 3)

    def _is_status_weight(status):
        # DTD might still play -- half weight. Out/IL-10/IL-15/IL-60/Suspended
        # are all unambiguous absences -- full weight, including anything
        # unrecognized (safer to over- than under-count an unfamiliar status).
        # BUG FIX (2026-09-18, user audit -- Contreras priced as a full -0.35R
        # OUT despite being day-to-day, not IL, after a Thursday game-time
        # injury): checked for the space-separated "day to day", but ESPN's
        # actual injuries feed -- and every other DTD check elsewhere in this
        # notebook (e.g. the news-scan marker lists) -- writes it hyphenated,
        # "Day-To-Day". The space variant never matched a real status string,
        # so this discount silently never fired. Matches the same variant set
        # used elsewhere in this notebook.
        _s = str(status).lower()
        return 0.5 if ("day-to-day" in _s or "day to day" in _s or "dtd" in _s) else 1.0

    _KEY_POS_STACK = {"SS", "C", "CF", "1B", "RF", "LF", "3B", "DH"}
    _inj_stack_ct = 0
    if 'mlb_lineup_flags' not in globals():
        mlb_lineup_flags = {}
    try:
        for _is_side_col in [away_col, home_col]:
            if _is_side_col not in board.columns:
                continue
            for _isidx in board.index:
                _is_val = board.at[_isidx, _is_side_col]
                if pd.isna(_is_val):
                    continue
                _is_team = str(_is_val)
                _is_tn = _pl_norm(_is_team)
                # REPLACEMENT-QUALITY INJURY PRICING (2026-09-19, user audit):
                # used to skip only when mlb_lineup_flags already had an entry --
                # i.e. only when Cell (PLATOON + wOBA SPLITS) happened to cross ITS
                # OWN -0.050 gap threshold. That left every team whose confirmed
                # lineup DIDN'T cross that threshold (including a gap of 0, or even
                # positive -- real replacements holding up fine) to fall through to
                # THIS method instead, which has no idea who replaced anyone and
                # just sums each absent player's own OPS -- the exact "assumes
                # replacement by a league-average player rather than the actual
                # substitute" gap. mlb_lineup_confirmed_teams means real batter-
                # level data resolved for this team tonight, regardless of what it
                # concluded -- trust that whenever it exists.
                if _is_tn in globals().get("mlb_lineup_confirmed_teams", set()):
                    continue
                _is_inj = _fetch_espn_injuries(_is_team)
                _is_key_out_raw = [i for i in _is_inj if str(i.get("pos", "")).upper() in _KEY_POS_STACK]
                # BUG FIX (MLB-IL-RETURN-1, 2026-09-24 user audit): the injuries
                # feed above lists official IL/OUT entries, but a same-day (or
                # very recent) activation/reinstatement doesn't always clear that
                # feed in time -- real incident: Ohtani priced as a full OUT
                # (-0.35R, Elite-OPS bucket) despite having been activated off IL
                # the same day. Cross-check against _il_activations (MLB Stats API
                # transactions log, moved earlier in this cell so it's available
                # here) -- a player with a recalled/activated/reinstated
                # transaction within the last _IL_RETURN_FRESH_DAYS days is treated
                # as active, not out, even if the injuries feed hasn't caught up.
                # Deliberately a short window (not the full 21d _il_activations
                # covers) -- a player could activate and get hurt again days later,
                # which the injuries feed IS tracking correctly by then.
                _IL_RETURN_FRESH_DAYS = 2
                _is_key_out = []
                for _is_out_p in _is_key_out_raw:
                    _is_act = _il_activations.get(str(_is_out_p.get("name", "")).lower().strip())
                    if _is_act is not None and _is_act[2] <= _IL_RETURN_FRESH_DAYS:
                        print(f"  [IL RETURN] {_is_out_p.get('name','?')} ({_is_team}) -- "
                              f"activated/reinstated {_is_act[2]}d ago ({_is_act[0]}), excluding from "
                              f"injury-stack pricing (injuries feed likely stale)")
                        continue
                    _is_key_out.append(_is_out_p)
                if len(_is_key_out) >= 2:
                    _IS_MAX_PENALTY = 1.10
                    _is_roster = _mlb_roster_for_team(_is_team)
                    _is_total = 0.0
                    # FIX 2 (2026-09-22, user audit): separate confirmed-only
                    # (weight-1.0) subtotal alongside the existing blended total
                    # above -- lets downstream consumers (Cell 6e66db20's [NET
                    # INJURY DELTA] bullet) show a version of this number that isn't
                    # partly built on 50%-weight DTD guesses, without touching the
                    # existing blended figure anything else already reads.
                    _is_total_confirmed = 0.0
                    _is_detail = []
                    for _p in _is_key_out[:6]:
                        # BUG FIX (MLB-INJ-RECENCY-1, 2026-09-25 user audit):
                        # a genuine IL/OUT status (not DTD -- day-to-day
                        # absences are inherently recent and rarely carry
                        # a formal IL transaction at all) placed on the IL
                        # more than _ABSENCE_RECENCY_MAX_DAYS ago is already
                        # reflected in the team's own rolling/season run-
                        # scoring numbers -- pricing it again double-counts
                        # the same real effect. Not found in the 21-day
                        # transaction window at all is treated the same way
                        # (definitely older than the cutoff, or unresolvable --
                        # either way, safer to assume already-absorbed than
                        # to keep charging an absence indefinitely).
                        _p_status = str(_p.get("status", ""))
                        _p_is_dtd = _is_status_weight(_p_status) < 1.0
                        if not _p_is_dtd:
                            _p_days_out = _il_placements.get(str(_p.get("name", "")).lower().strip())
                            if _p_days_out is None or _p_days_out > _ABSENCE_RECENCY_MAX_DAYS:
                                print(f"  [INJURY STALE] {_p.get('name','?')} ({_is_team}) -- "
                                      f"{f'placed on IL {_p_days_out}d ago' if _p_days_out is not None else 'no IL placement found in the last 21d'}, "
                                      f"beyond the {_ABSENCE_RECENCY_MAX_DAYS}d recency window -- already reflected in team form, excluding from injury-stack pricing")
                                continue
                        _pid = _is_roster.get(_is_norm_name(_p.get("name", "")))
                        _ops, _pa = _mlb_player_season_ops(_pid) if _pid else (None, None)
                        _base_ded = _ops_to_run_deduction(_ops, _pa)
                        if _base_ded is None:
                            _base_ded = round(max(0.0, _INJ_LEAGUE_OPS - _REPLACEMENT_OPS) * _RUNS_PER_OPS_POINT, 3)  # unresolved name/no OPS data -- league-average-level default, not zero or Elite
                            _tier = "AVG-fallback"
                            _why = "name lookup failed" if not _pid else "no OPS data for resolved player"
                            print(f"  [INJURY WARN: {_p.get('name','?')} ({_is_team}) -- {_why}, flat league-average-level deduction applied]")
                        else:
                            _pa_tag = f", {_pa} PA" if _pa is not None else ""
                            _tier = f"{_ops:.3f} OPS{_pa_tag}"
                        _wt = _is_status_weight(_p.get("status", ""))
                        _ded = round(_base_ded * _wt, 3)
                        _is_total += _ded
                        if _wt >= 1.0:
                            _is_total_confirmed += _ded
                        _dtd_tag = " DTD 50%" if _wt < 1.0 else ""
                        _is_detail.append(f"{_p.get('name','?')} ({_tier}{_dtd_tag} -{_ded:.2f}R)")
                    _is_penalty = min(round(_is_total, 3), _IS_MAX_PENALTY)
                    _is_penalty_confirmed = min(round(_is_total_confirmed, 3), _IS_MAX_PENALTY)
                    _is_names = ", ".join(_is_detail)
                    mlb_lineup_flags[_is_tn] = {
                        "msg": f"[LINEUP DOWNGRADE: {len(_is_key_out)} key players out ({_is_names}) -- injury stack, -{_is_penalty:.2f}R]",
                        "penalty": round(_is_penalty, 2),
                        "penalty_confirmed": round(_is_penalty_confirmed, 2),
                    }
                    _inj_stack_ct += 1
    except Exception as _is_ex:
        print(f"  [WARN] Injury stack detector failed: {_is_ex}")
    if _inj_stack_ct:
        print(f"  Injury stack detector: {_inj_stack_ct} team(s) flagged (2+ key absences)")

    # ---------------------------------------------------------
    # LINEUP QUALITY FLAGS: print warnings for depleted lineups
    # ---------------------------------------------------------
    if 'mlb_lineup_flags' in globals() and mlb_lineup_flags:
        print(f"  Lineup flags: {len(mlb_lineup_flags)} team(s) with downgraded lineup")
        for _lft, _lfv in mlb_lineup_flags.items():
            _lf_msg = _lfv['msg'] if isinstance(_lfv, dict) else _lfv
            _lf_penalty = _lfv.get('penalty', 0.40) if isinstance(_lfv, dict) else 0.40
            print(f"    {_lft}: {_lf_msg}")
            # Also reduce that team's run projection
            for _lfidx in board.index:
                _aw_match = _pl_norm(str(board.at[_lfidx, away_col] if away_col in board.columns else "")) == _lft
                _hm_match = _pl_norm(str(board.at[_lfidx, home_col] if home_col in board.columns else "")) == _lft
                if _aw_match and pd.notna(board.at[_lfidx, "pred_away_runs"]):
                    board.at[_lfidx, "pred_away_runs"] = round(board.at[_lfidx, "pred_away_runs"] - _lf_penalty, 1)
                if _hm_match and pd.notna(board.at[_lfidx, "pred_home_runs"]):
                    board.at[_lfidx, "pred_home_runs"] = round(board.at[_lfidx, "pred_home_runs"] - _lf_penalty, 1)

    # ---------------------------------------------------------
    # TIMESTAMPED INJURY SNAPSHOT + CROSS-RUN DRIFT FLAG (2026-08-31, user
    # audit -- Red Sox/Mariners): mlb_lineup_flags is rebuilt from a live
    # ESPN fetch every run with no memory of what an earlier run this same
    # day saw. Real case: first run had NET INJURY DELTA -0.18R favoring
    # Boston; a later run flipped to -0.17R favoring Seattle once ESPN
    # added Contreras -- a real, legitimate update, but nothing on the card
    # told the user the injury picture had changed mid-day and the play's
    # profile flipped sides. Snapshots each run's per-team penalty,
    # timestamped, to tracking/mlb_injury_snapshot.json and diffs against
    # the same day's previous snapshot so a materially different picture
    # (or a flag that appeared/cleared between runs) is visible, not silent.
    # ---------------------------------------------------------
    import json as _inj_json
    from pathlib import Path as _InjPath
    import datetime as _inj_dt

    _INJ_SNAPSHOT_PATH = _InjPath(r"c:\Users\<user>\OneDrive\Documents\KAI Labs\sports\MLB\tracking\mlb_injury_snapshot.json")
    _inj_today_str = globals().get("CARD_DATE") or _inj_dt.date.today().strftime("%Y-%m-%d")
    _inj_now_iso = _inj_dt.datetime.now(_inj_dt.timezone.utc).strftime("%H:%M UTC")
    _INJ_DRIFT_THRESHOLD = 0.10  # R

    _inj_prev_snapshot = {}
    try:
        if _INJ_SNAPSHOT_PATH.exists():
            _inj_prev_all = _inj_json.loads(_INJ_SNAPSHOT_PATH.read_text(encoding="utf-8"))
            _inj_prev_snapshot = _inj_prev_all.get(_inj_today_str, {})
    except Exception as _inj_e:
        print(f"  [WARN] injury snapshot load failed: {_inj_e}")
        _inj_prev_snapshot = {}

    mlb_injury_drift = {}  # team_norm -> drift note string, consumed by Cell 89's WHY loop
    for _idt_tn, _idt_v in mlb_lineup_flags.items():
        _idt_new_pen = float(_idt_v.get("penalty", 0) or 0) if isinstance(_idt_v, dict) else 0.40
        _idt_prev = _inj_prev_snapshot.get(_idt_tn)
        if _idt_prev is not None:
            _idt_prev_pen = float(_idt_prev.get("penalty", 0) or 0)
            _idt_delta = _idt_new_pen - _idt_prev_pen
            if abs(_idt_delta) >= _INJ_DRIFT_THRESHOLD:
                mlb_injury_drift[_idt_tn] = (
                    f"!! [INJURY DATA CHANGED SINCE LAST RUN TODAY] {_idt_tn}: penalty was "
                    f"-{_idt_prev_pen:.2f}R as of {_idt_prev.get('ts','?')} -> now -{_idt_new_pen:.2f}R "
                    f"({_idt_delta:+.2f}R) -- injury picture shifted mid-day, confirm before trusting this edge")
    for _idt_tn_prev, _idt_prev in _inj_prev_snapshot.items():
        if _idt_tn_prev not in mlb_lineup_flags:
            _idt_prev_pen = float(_idt_prev.get("penalty", 0) or 0)
            if _idt_prev_pen >= _INJ_DRIFT_THRESHOLD:
                mlb_injury_drift[_idt_tn_prev] = (
                    f"!! [INJURY DATA CHANGED SINCE LAST RUN TODAY] {_idt_tn_prev}: had a "
                    f"-{_idt_prev_pen:.2f}R flag as of {_idt_prev.get('ts','?')}, clear this run -- "
                    f"confirm the earlier injury report wasn't real before trusting this edge")
    if mlb_injury_drift:
        print(f"  [INJURY DRIFT] {len(mlb_injury_drift)} team(s) show a materially different "
              f"injury picture than an earlier run today")

    try:
        _inj_all = {}
        if _INJ_SNAPSHOT_PATH.exists():
            try:
                _inj_all = _inj_json.loads(_INJ_SNAPSHOT_PATH.read_text(encoding="utf-8"))
            except Exception:
                _inj_all = {}
        _inj_all[_inj_today_str] = {
            _tn: {"penalty": round(float(v.get("penalty", 0) if isinstance(v, dict) else 0.40), 3),
                  "msg": (v.get("msg", "") if isinstance(v, dict) else str(v)),
                  "ts": _inj_now_iso}
            for _tn, v in mlb_lineup_flags.items()
        }
        _inj_all = dict(sorted(_inj_all.items())[-3:])  # keep last 3 days, avoid unbounded growth
        _INJ_SNAPSHOT_PATH.parent.mkdir(parents=True, exist_ok=True)
        _INJ_SNAPSHOT_PATH.write_text(_inj_json.dumps(_inj_all, indent=2), encoding="utf-8")
    except Exception as _inj_e:
        print(f"  [WARN] injury snapshot save failed: {_inj_e}")

    # ---------------------------------------------------------
    # ---------------------------------------------------------
    # BULLPEN QUALITY: individual reliever tiers + weak link
    # Uses mlb_bp_risk_score and mlb_bp_weak_link from Cell 33
    # ---------------------------------------------------------
    _bp_adj_ct = 0
    if 'mlb_bp_risk_score' in globals() and mlb_bp_risk_score:
        for _bpidx in board.index:
            for _pitch_side, _bat_run_col in [
                (away_col if away_col in board.columns else None, 'pred_home_runs'),
                (home_col if home_col in board.columns else None, 'pred_away_runs'),
            ]:
                if not _pitch_side: continue
                _bp_tn     = _pl_norm(str(board.at[_bpidx, _pitch_side]))
                _risk      = mlb_bp_risk_score.get(_bp_tn, 0)
                _wl        = mlb_bp_weak_link.get(_bp_tn, {})
                _exit_risk = globals().get('mlb_sp_early_exit', {}).get(_bp_tn, False)
                _fatigue   = globals().get('mlb_bp_fatigue_flag', {}).get(_bp_tn, {})
                _gassed    = _fatigue.get('gassed', [])
                if _risk == 0 and not _exit_risk and not _gassed: continue
                _penalty = 0.0
                # SEVERITY-SCALED WEAK-LINK PENALTY (2026-09-14, user request --
                # real Rangers/Diamondbacks case: the off-roster filter above
                # correctly dropped 3 ghost names (Curvelo 9.58 / Ross 9.26 /
                # Martin 9.00 ERA, all confirmed off the 40-man), but the actual
                # run penalty didn't move -- this was a flat +0.35R for ANY worst
                # arm >= 5.0 ERA, so a 5.94 ERA arm and a 9.58 ERA arm drew the
                # exact identical penalty. Every other run-adjustment gate in this
                # notebook (HR/9, sustained cold-streak, FIP reliability, xERA
                # cold-streak, etc.) scales continuously above its floor --
                # this one never did. _wl['era'] already holds the worst arm's
                # exact ERA (used for years just to gate the flat bonus above 5.0
                # and to print the number) -- scaling off it directly costs
                # nothing new to fetch. Floor case (era==5.0) reproduces the old
                # flat 0.35 exactly, so this is a pure refinement, not a re-tune
                # of the baseline.
                _BP_WEAK_PENALTY_BASE    = 0.35   # unchanged from the old flat penalty -- floor case
                _BP_WEAK_PENALTY_PER_ERA = 0.03   # additional penalty per ERA point above the 5.0 floor -- provisional, revisit after live grading
                _BP_WEAK_PENALTY_MAX     = 0.55   # cap -- same magnitude family as _BP_WORKLOAD_MAX_R/_BP_ERA_MAX_R elsewhere in this cell
                if _risk >= 1 and _wl.get('era', 0) >= 5.0:
                    _penalty += min(_BP_WEAK_PENALTY_MAX,
                                     _BP_WEAK_PENALTY_BASE + _BP_WEAK_PENALTY_PER_ERA * (_wl['era'] - 5.0))
                if _risk >= 2:
                    _penalty += 0.20
                if _exit_risk:
                    _penalty += 0.20
                if _gassed:
                    _penalty += 0.25
                if _wl.get('count', 0) >= 2:
                    # BUG FIX (2026-09-02, found live -- real Orioles loss):
                    # this only ever penalized for the single worst arm --
                    # Baltimore had 2 relievers over the weak bar (Akin 5.68,
                    # Wolfram 5.15) but got the same flat penalty a team with
                    # just 1 weak arm would. Modest -- doesn't scale further
                    # past 2, matches the size of every other add-on here.
                    _penalty += 0.15

                # BULLPEN STRENGTH OFFSET (2026-09-02, user audit -- real
                # Orioles loss: COL flagged risk=2 off its weakest arm while
                # Romano/Bernardino, its actual high-leverage relievers, had
                # been excellent for weeks -- the risk penalty above only
                # ever looks at weakness, never at whether the top of the
                # pen can cover for it). Modest, capped -- this doesn't
                # erase a real weak-link penalty, just tempers it when the
                # team has 2+ genuinely hot arms right now.
                _bp_strength_str = ""
                _hl_rec = globals().get("mlb_bp_hl_recent", {}).get(_bp_tn)
                if _penalty > 0 and _hl_rec and len(_hl_rec.get("names", [])) >= 2:
                    _bp_offset = min(0.15, _penalty)
                    _penalty = round(_penalty - _bp_offset, 2)
                    _hl_rec_names = ", ".join(f"{n} {e:.2f}" for n, e in _hl_rec["names"])
                    _bp_strength_str = f" | offset -{_bp_offset:.2f}R: hot arms ({_hl_rec_names}) over trailing 30d"

                if _penalty > 0 and pd.notna(board.at[_bpidx, _bat_run_col]):
                    board.at[_bpidx, _bat_run_col] = round(
                        board.at[_bpidx, _bat_run_col] + _penalty, 1)
                    _bp_adj_ct += 1
                    if _wl.get('count', 0) >= 2:
                        _wl_all_str = ", ".join(f"{a['name']} {a['era']:.2f}" for a in _wl.get('all', []))
                        _wl_str = f" (weak links: {_wl_all_str})"
                    else:
                        _wl_str = f" (weak link: {_wl.get('name','')} {_wl.get('era',0):.2f} ERA)" if _wl.get('name') else ''
                    _gassed_str = f" (gassed: {', '.join(_gassed)})" if _gassed else ''
                    print(f"  [BULLPEN RISK] {_bp_tn}: risk={_risk}{_wl_str}{_gassed_str} -> +{_penalty:.1f}R to opponent{_bp_strength_str}")
    elif 'mlb_bullpen_depth' in globals() and mlb_bullpen_depth:
        for _bpidx in board.index:
            for _bp_side, _run_col in [
                (away_col if away_col in board.columns else None, 'pred_home_runs'),
                (home_col if home_col in board.columns else None, 'pred_away_runs'),
            ]:
                if not _bp_side: continue
                _bp_tn = _pl_norm(str(board.at[_bpidx, _bp_side]))
                _depth = mlb_bullpen_depth.get(_bp_tn, 1.0)
                if _depth < 1.0 and pd.notna(board.at[_bpidx, _run_col]):
                    _penalty = (1.0 - _depth) * 0.35 * board.at[_bpidx, _run_col]
                    board.at[_bpidx, _run_col] = round(board.at[_bpidx, _run_col] + _penalty, 1)
                    _bp_adj_ct += 1
    if _bp_adj_ct:
        print(f"  Bullpen adj: {_bp_adj_ct} run projection(s) updated")

    # ---------------------------------------------------------
    # BULLPEN WORKLOAD -> RUN TOTAL (2026-08-29, user audit M1)
    # The risk/gassed block above is season-aggregate quality plus a top-3-arm
    # appearance flag; neither knows how much the pen as a whole has actually
    # thrown. mlb_bp_workload_l3 (CELL BULLPEN) measures real last-3-day
    # reliever batters-faced + back-to-back arms against the league median for
    # the same window and prices the excess as runs. Applied to the OPPONENT's
    # projected runs -- which is what carries it into pred_total_runs below and
    # therefore into the O/U edge, the market this signal was missing from
    # entirely. Runs as its own block (not inside the risk/depth if/elif) so it
    # still applies on a run where mlb_bp_risk_score never loaded.
    # ---------------------------------------------------------
    _bpw_adj_ct = 0
    _bpw_tbl = globals().get("mlb_bp_workload_l3", {}) or {}
    globals()["_mlb_bp_workload_applied"] = {}
    if _bpw_tbl:
        for _bwidx in board.index:
            for _bw_side, _bw_run_col in [
                (away_col if away_col in board.columns else None, 'pred_home_runs'),
                (home_col if home_col in board.columns else None, 'pred_away_runs'),
            ]:
                if not _bw_side: continue
                _bw_tn = _pl_norm(str(board.at[_bwidx, _bw_side]))
                _bw_rec = _bpw_tbl.get(_bw_tn)
                if not _bw_rec: continue
                _bw_r = float(_bw_rec.get("run_adj", 0.0) or 0.0)
                if _bw_r <= 0 or pd.isna(board.at[_bwidx, _bw_run_col]): continue
                board.at[_bwidx, _bw_run_col] = round(float(board.at[_bwidx, _bw_run_col]) + _bw_r, 2)
                _bpw_adj_ct += 1
                globals()["_mlb_bp_workload_applied"][_bw_tn] = _bw_rec
                print(f"  [BULLPEN WORKLOAD] {_bw_tn}: {_bw_rec.get('bf',0)} BF / {_bw_rec.get('ip',0)} IP / "
                      f"{_bw_rec.get('b2b_arms',0)} back-to-back arm(s) over last 3 days "
                      f"(league median {float(_bw_rec.get('bf_median',0)):.0f} BF) -> "
                      f"+{_bw_r:.2f}R to the opponent's projection")
    if _bpw_adj_ct:
        print(f"  Bullpen workload adj: {_bpw_adj_ct} run projection(s) updated (feeds pred_total_runs / O/U)")
    elif not _bpw_tbl:
        print("  [BULLPEN WORKLOAD] mlb_bp_workload_l3 empty -- totals carry no L3 bullpen-fatigue term this run")

    # ---------------------------------------------------------
    # BULLPEN L3 ERA -> RUN PROJECTION (2026-08-31, user audit -- Red Sox
    # pen, 9 ER Saturday): applies era_run_adj (Cell 34) the same way the
    # workload block above applies run_adj -- to the OPPONENT's projected
    # runs, feeding both pred_total_runs/O-U and, via the score-blended ML
    # probability further down this cell, the moneyline itself. Kept as its
    # own counter/global (not merged into _bpw_adj_ct/_mlb_bp_workload_applied)
    # since this is a distinct signal (real runs allowed, not workload
    # volume) and the WHY section prints them as separate lines.
    # ---------------------------------------------------------
    _bpe_adj_ct = 0
    globals()["_mlb_bp_era_applied"] = {}
    if _bpw_tbl:
        for _beidx in board.index:
            for _be_side, _be_run_col in [
                (away_col if away_col in board.columns else None, 'pred_home_runs'),
                (home_col if home_col in board.columns else None, 'pred_away_runs'),
            ]:
                if not _be_side: continue
                _be_tn = _pl_norm(str(board.at[_beidx, _be_side]))
                _be_rec = _bpw_tbl.get(_be_tn)
                if not _be_rec: continue
                _be_r = float(_be_rec.get("era_run_adj", 0.0) or 0.0)
                if _be_r <= 0 or pd.isna(board.at[_beidx, _be_run_col]): continue
                board.at[_beidx, _be_run_col] = round(float(board.at[_beidx, _be_run_col]) + _be_r, 2)
                _bpe_adj_ct += 1
                globals()["_mlb_bp_era_applied"][_be_tn] = _be_rec
                print(f"  [BULLPEN L3 ERA] {_be_tn}: {_be_rec.get('era_l3',0):.2f} ERA "
                      f"({_be_rec.get('er',0)} ER / {_be_rec.get('ip',0)} IP) over last 3 days "
                      f"(league avg {_be_rec.get('era_l3_league_avg',0):.2f}) -> "
                      f"+{_be_r:.2f}R to the opponent's projection (damped)")
    if _bpe_adj_ct:
        print(f"  Bullpen L3 ERA adj: {_bpe_adj_ct} run projection(s) updated (feeds pred_total_runs / O/U / ML score-blend)")
    elif not _bpw_tbl:
        print("  [BULLPEN L3 ERA] mlb_bp_workload_l3 empty -- no L3 bullpen-ERA term on projections this run")

    # ---------------------------------------------------------
    # BULLPEN DEPTH -> RUN PROJECTION (2026-09-02, user audit): applies
    # mlb_bp_depth's run_adj (Cell 34) the same way the other bullpen
    # signals above apply theirs -- to the OPPONENT's projected runs. A
    # distinct signal from BULLPEN RISK (single weak-link arm) and BULLPEN
    # L3 ERA (recent-days runs allowed) -- this one reads season-long
    # workload-weighted depth (how much of the bullpen's real innings come
    # from merely-shaky, not necessarily "weak", arms).
    # ---------------------------------------------------------
    _bpd_adj_ct = 0
    if 'mlb_bp_depth' in globals() and mlb_bp_depth:
        for _bdidx in board.index:
            for _bd_side, _bd_run_col in [
                (away_col if away_col in board.columns else None, 'pred_home_runs'),
                (home_col if home_col in board.columns else None, 'pred_away_runs'),
            ]:
                if not _bd_side: continue
                _bd_tn = _pl_norm(str(board.at[_bdidx, _bd_side]))
                _bd_rec = mlb_bp_depth.get(_bd_tn)
                if not _bd_rec: continue
                _bd_r = float(_bd_rec.get("run_adj", 0.0) or 0.0)
                if _bd_r <= 0 or pd.isna(board.at[_bdidx, _bd_run_col]): continue
                board.at[_bdidx, _bd_run_col] = round(float(board.at[_bdidx, _bd_run_col]) + _bd_r, 2)
                _bpd_adj_ct += 1
    if _bpd_adj_ct:
        print(f"  Bullpen depth adj: {_bpd_adj_ct} run projection(s) updated (feeds pred_total_runs / O/U / ML score-blend)")

    # ---------------------------------------------------------
    # Fix 7 (wired): Series momentum -- applies the adjustment the
    # per-pick narrative below used to only print as text. Verified the
    # old block computed _f7_streak and printed "-0.15R penalty"/"+0.10R
    # boost" but never wrote to board or edge anywhere -- the model never
    # actually saw this signal. Magnitude increased slightly per request.
    # ---------------------------------------------------------
    _SM_COLD_PENALTY = 0.20   # was 0.15 (never applied) -- bumped per request
    _SM_HOT_BOOST    = 0.15   # was 0.10 (never applied) -- bumped per request
    # Item 2 fix (2026-07-09): don't let recent win/loss series momentum
    # override a confirmed elite starter -- see comment block above this edit.
    def _sm_elite_sp_on_mound(away_nm, home_nm):
        if "_sp_form_cache" not in globals() or not _sp_form_cache: return False
        import unicodedata as _sme_u, re as _sme_r
        def _sme_n(s): return _sme_r.sub(r"[^a-z ]","",_sme_u.normalize("NFD",str(s)).encode("ascii","ignore").decode().lower()).strip()
        for _nm in (away_nm, home_nm):
            # 2026-08-12 BUG FIX (live AttributeError: 'NoneType' object has
            # no attribute 'get'): same root cause already fixed at other
            # _sp_form_cache call sites on 2026-08-04 (cells around lines
            # 49491/49561/50287) -- this cache explicitly stores None (not
            # a missing key) for a pitcher whose lookup failed, so the
            # two-arg `.get(key, {})` never falls back to {} in that case
            # (the default only applies when the key is absent, not when
            # it's present and None). This call site was missed when the
            # other sites got the `.get(key) or {}` fix. Same pattern here.
            _c = _sp_form_cache.get(_sme_n(_nm)) or {}
            _l3 = _c.get("era_l3")
            if _l3 is not None:
                try:
                    if float(_l3) < 3.00: return True
                except (ValueError, TypeError): pass
        return False
    _sm_adj_ct = 0
    if '_series_momentum' in globals() and _series_momentum:
        for _smidx in board.index:
            _sm_away = str(board.at[_smidx, away_col] if away_col in board.columns else "").lower()
            _sm_home = str(board.at[_smidx, home_col] if home_col in board.columns else "").lower()
            _sm_ctx = (_series_momentum.get((_sm_away, _sm_home)) or _series_momentum.get((_sm_home, _sm_away)))
            if not _sm_ctx:
                continue
            # Item 2 fix (2026-07-09): skip the momentum nudge entirely when either
            # starter today is a confirmed elite arm -- team-level win/loss noise
            # from the prior series games shouldn't be allowed to override that.
            _sm_asp = next((str(board.at[_smidx, c]) for c in ["away_pitcher_final_name","away_pitcher_name","away_starter"] if c in board.columns and board.at[_smidx, c]), "")
            _sm_hsp = next((str(board.at[_smidx, c]) for c in ["home_pitcher_final_name","home_pitcher_name","home_starter"] if c in board.columns and board.at[_smidx, c]), "")
            if _sm_elite_sp_on_mound(_sm_asp, _sm_hsp):
                print(f"  [SERIES MOMENTUM SKIPPED] {_sm_away} @ {_sm_home}: elite confirmed SP on mound -- momentum nudge not applied")
                continue
            for _sm_team, _sm_col in [(_sm_away, "pred_away_runs"), (_sm_home, "pred_home_runs")]:
                if _sm_col not in board.columns or pd.isna(board.at[_smidx, _sm_col]):
                    continue
                _sm_streak = next((v.get("streak", 0) for k, v in _sm_ctx.items()
                                    if any(w in k for w in _sm_team.split() if len(w) > 3)), 0)
                if _sm_streak <= -2:
                    board.at[_smidx, _sm_col] = round(board.at[_smidx, _sm_col] - _SM_COLD_PENALTY, 2)
                    _sm_adj_ct += 1
                elif _sm_streak >= 2:
                    board.at[_smidx, _sm_col] = round(board.at[_smidx, _sm_col] + _SM_HOT_BOOST, 2)
                    _sm_adj_ct += 1
    if _sm_adj_ct:
        print(f"  Series momentum: {_sm_adj_ct} run projection(s) adjusted")

    # MLB-6: bullpen uncertainty weighted by projected margin
    # Full +1 flag for margins < 1.5R (coin flip — relief depth critical)
    # Half weight for margins > 2.5R (blowout — starter matters more)
    _bp_data_available = ('mlb_bp_risk_score' in globals() and bool(mlb_bp_risk_score))
    if not _bp_data_available:
        print("=" * 58)
        print("  [BULLPEN RISK SCORE MISSING] mlb_bp_risk_score not loaded")
        print("     Run the Bullpen ERA/Fatigue cell (CELL BULLPEN) for full-strength")
        print("     grading -- proceeding anyway, grade ceiling: A- on every play today")
        print("=" * 58)

    # FIX 1 -- HOME FIELD ADVANTAGE (+0.15R to home predicted runs)
    # Home teams outscore visitors ~0.15R/game: batting last, park familiarity, no travel.
    # Applied after SP/bullpen/platoon adjustments so it propagates through score-blend.
    # 2026-08-13 (user audit): excluded neutral-site games -- "batting last,
    # park familiarity, no travel" doesn't hold when neither team is actually
    # home (e.g. Field of Dreams), so the designated home team shouldn't get
    # a fabricated advantage there.
    _HOME_ADV_RUNS = 0.15
    _neutral_mask = board["is_neutral_site"] == True if "is_neutral_site" in board.columns else pd.Series(False, index=board.index)
    _home_adv_mask = board["pred_home_runs"].notna() & (~_neutral_mask)
    if _home_adv_mask.any():
        board.loc[_home_adv_mask, "pred_home_runs"] = (board.loc[_home_adv_mask, "pred_home_runs"] + _HOME_ADV_RUNS).round(1)
        _neutral_ct = int(_neutral_mask.sum())
        print(f"  Home advantage applied: +{_HOME_ADV_RUNS}R to pred_home_runs ({_home_adv_mask.sum()} game(s)"
              + (f", {_neutral_ct} neutral-site game(s) excluded)" if _neutral_ct else ")"))

    # ---------------------------------------------------------
# FIX C: HOME HOT STREAK BOOST
    # If home team has won >=75% of its LAST 15 HOME games, apply +0.12R boost
    # to home team's projected runs.
    # Rebuilt to use a genuine rolling last-15-home-games window from `recent_games`
    # (60-day ESPN result log) instead of season-cumulative _team_records home W-L --
    # the prior version answered "hot all season" not "hot right now."
    # 2026-07-12: added a fade for EXTREME streaks (real, current consecutive
    # wins, not just >=75% over a 15-game window). Ported from the same
    # philosophy already used in WORLDCUP2026MODEL.ipynb's "[OVER TREND ADJ]"
    # (e.g. "Norway Over 86% last 7 -- UNDER adj") -- an extreme streak is
    # partly variance/due for regression, not purely a persisting quality
    # signal, so past a real threshold the adjustment should pull back toward
    # the mean instead of continuing to chase the streak. Surfaced by the
    # Tigers ML loss (Detroit carried a 6-game win streak into that game,
    # this boost plausibly added false confidence on top of everything else)
    # -- the fade's magnitude is a judgment call sized to roughly mirror the
    # existing boost, not empirically tuned; revisit once real cards
    # accumulate with this active.
    # ---------------------------------------------------------
    _home_streak_ct = 0
    _home_streak_fade_ct = 0
    if 'recent_games' in globals() and isinstance(recent_games, pd.DataFrame) and not recent_games.empty:
        _HOME_STREAK_THRESH  = 0.75
        _HOME_STREAK_BOOST   = 0.12
        _HOME_STREAK_WINDOW  = 15
        _HOME_STREAK_EXTREME = 6      # real consecutive wins at home
        _HOME_STREAK_FADE    = -0.10  # regression-to-mean discount once extreme
        _hs_rg = recent_games.sort_values("date")
        _hs_rg_home_lower = _hs_rg["home_team"].str.lower()
        for _hsidx in board.index:
            _hs_tn = str(board.at[_hsidx, home_col] if home_col in board.columns else "").lower()
            _hs_words = [w for w in _hs_tn.split() if len(w) > 3]
            if not _hs_words: continue
            _hs_mask = _hs_rg_home_lower.apply(lambda k: any(w in k for w in _hs_words))
            _hs_last15 = _hs_rg[_hs_mask].tail(_HOME_STREAK_WINDOW)
            if len(_hs_last15) < _HOME_STREAK_WINDOW: continue
            _hs_pct = _hs_last15["home_win"].mean()
            _hs_cur_streak = 0
            for _hs_v in reversed(_hs_last15["home_win"].tolist()):
                if _hs_v == 1:
                    _hs_cur_streak += 1
                else:
                    break
            if _hs_cur_streak >= _HOME_STREAK_EXTREME and pd.notna(board.at[_hsidx, "pred_home_runs"]):
                board.at[_hsidx, "pred_home_runs"] = round(board.at[_hsidx, "pred_home_runs"] + _HOME_STREAK_FADE, 1)
                _home_streak_fade_ct += 1
                print(f"  [HOME STREAK FADE] {board.at[_hsidx, home_col]}: {_hs_cur_streak}-game actual home win streak -- extreme, regression-due; {_HOME_STREAK_FADE}R discount applied instead of a boost")
            elif _hs_pct >= _HOME_STREAK_THRESH and pd.notna(board.at[_hsidx, "pred_home_runs"]):
                board.at[_hsidx, "pred_home_runs"] = round(board.at[_hsidx, "pred_home_runs"] + _HOME_STREAK_BOOST, 1)
                _home_streak_ct += 1
                _hs_w = int(_hs_last15["home_win"].sum())
                print(f"  [HOME HOT STREAK] {board.at[_hsidx, home_col]}: {_hs_w}-{_HOME_STREAK_WINDOW - _hs_w} last {_HOME_STREAK_WINDOW} home ({_hs_pct:.0%}) -- +{_HOME_STREAK_BOOST}R boost")
    if _home_streak_ct: print(f"  Home hot streak: {_home_streak_ct} game(s) adjusted")
    if _home_streak_fade_ct: print(f"  Home streak fade: {_home_streak_fade_ct} game(s) adjusted (extreme streak, regression discount)")

    # Fix 2: when both SPs have elite recent form (L3 ERA < 2.50), suppress totals by 0.8x
    # Buehler L3 1.89 + partner L3 < 2.50 should have flagged a suppressed total
    if "_sp_fetch_recent" in dir() and "_sp_norm_era" in dir():
        import pandas as _fix2pd
        for _f2idx in board.index:
            _f2_asp = str(board.at[_f2idx,"away_pitcher_name"] or "")
            _f2_hsp = str(board.at[_f2idx,"home_pitcher_name"] or "")
            _f2_aspf = _sp_fetch_recent(_f2_asp) or {}
            _f2_hspf = _sp_fetch_recent(_f2_hsp) or {}
            # MLB-2: require 2+ starts before treating L3 as meaningful
            _f2_ags = float(_f2_aspf.get("gs_l3") or 0)
            _f2_hgs = float(_f2_hspf.get("gs_l3") or 0)
            _f2_al3 = float(_f2_aspf.get("era_l3") or 9.9) if _f2_ags >= 2 else 9.9
            _f2_hl3 = float(_f2_hspf.get("era_l3") or 9.9) if _f2_hgs >= 2 else 9.9
            if _f2_al3 < 2.50 and _f2_hl3 < 2.50:
                for _rc in ["pred_total_runs","pred_away_runs","pred_home_runs"]:
                    if _rc in board.columns and _fix2pd.notna(board.at[_f2idx,_rc]):
                        board.at[_f2idx,_rc] = round(board.at[_f2idx,_rc]*0.80, 1)
                print(f"  [DUAL ELITE SP] {_f2_asp} L3 {_f2_al3:.2f} + {_f2_hsp} L3 {_f2_hl3:.2f} — both < 2.50, totals ×0.80")
    board["pred_total_runs"] = (
        board["pred_away_runs"].fillna(0) + board["pred_home_runs"].fillna(0)
    ).where(board["pred_away_runs"].notna() & board["pred_home_runs"].notna(), np.nan)
    # Fix 3: Home/away O/U trend gate — suppress runs when team has strong Under trend
    # Padres home Under 17/24 (71%) — this should suppress Petco runs projections
    try:
        _f3_mlb_ids = globals().get("_mlb_team_ids", {})
        for _f3idx in board.index:
            _f3_hm = str(board.at[_f3idx, home_col] if home_col in board.columns else "")
            _f3_aw = str(board.at[_f3idx, away_col] if away_col in board.columns else "")
            # Check home team Under rate (home Under > 65% in last 15)
            _f3_htid = next((v for k,v in _f3_mlb_ids.items() if any(w in k for w in _f3_hm.lower().split() if len(w)>3)), None)
            if _f3_htid and "_get_home_ou_trend" in dir():
                # MLB-3: use L15 specifically, not season-long diluted record
                _f3_ht = _get_home_ou_trend(_f3_htid, n=15) if "n" in _get_home_ou_trend.__code__.co_varnames else _get_home_ou_trend(_f3_htid)
                _f3_tot = _f3_ht.get("total",0)
                if _f3_tot >= 12:
                    _f3_un_pct = _f3_ht.get("under",0) / _f3_tot
                    if _f3_un_pct >= 0.65:
                        _f3_supp = round(0.5 * (_f3_un_pct - 0.60) / 0.05, 2)  # 0.5R per 5% above 60%
                        _f3_supp = min(0.8, _f3_supp)  # cap at 0.8R
                        for _f3_col in ["pred_home_runs","pred_away_runs"]:
                            if _f3_col in board.columns and pd.notna(board.at[_f3idx,_f3_col]):
                                board.at[_f3idx,_f3_col] = round(board.at[_f3idx,_f3_col] - _f3_supp/2, 1)
                        print(f"  [HOME UNDER TREND] {_f3_hm}: {_f3_ht['under']}-{_f3_ht['over']} home U/O ({_f3_un_pct:.0%}) — -{_f3_supp:.1f}R suppression")
    except Exception: pass
    # MLB-9: Apply series fatigue boost to opponent projected runs
    if "_series_fatigue" in globals() and _series_fatigue:
        def _norm_sf(n): return str(n).lower().strip()
        def _sf_boost(team_name):
            n = _norm_sf(team_name)
            return next((v for k,v in _series_fatigue.items() if k in n or n in k), 0.0)
        board["away_fatigue_R"] = board[away_col].apply(_sf_boost)
        board["home_fatigue_R"] = board[home_col].apply(_sf_boost)
        # Opponent fatigue = away team fatigue adds to HOME projected runs (opp pitching tired)
        board["pred_home_runs"] = (board["pred_home_runs"] + board["away_fatigue_R"]).round(1)
        board["pred_away_runs"] = (board["pred_away_runs"] + board["home_fatigue_R"]).round(1)
        board["pred_total_runs"] = (board["pred_home_runs"] + board["pred_away_runs"]).round(1)
    # M1/M12: apply reliever back-to-back-day run_adj the same way as series fatigue
    # (opponent's specific tired reliever(s) -> boosts the OTHER team's projected runs)
    if "mlb_bp_b2b" in globals() and mlb_bp_b2b:
        def _norm_b2b(n): return str(n).lower().strip()
        def _b2b_boost(team_name):
            n = _norm_b2b(team_name)
            return next((v["run_adj"] for k, v in mlb_bp_b2b.items() if k in n or n in k), 0.0)
        board["away_b2b_R"] = board[away_col].apply(_b2b_boost)
        board["home_b2b_R"] = board[home_col].apply(_b2b_boost)
        board["pred_home_runs"] = (board["pred_home_runs"] + board["away_b2b_R"]).round(1)
        board["pred_away_runs"] = (board["pred_away_runs"] + board["home_b2b_R"]).round(1)
        board["pred_total_runs"] = (board["pred_home_runs"] + board["pred_away_runs"]).round(1)
    # â”€â”€ ELO adjustment â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    # Blend base prediction with ELO-based expected runs.
    # ELO source: mlb_team_off_elo, mlb_team_def_elo, mlb_pitcher_elo
    #
    # 2026-08-02 BUG FIX (found live, real card 2026-08-01): _ELO_W was 0.35 with
    # no backtest behind it. Walk-forward tested against the real mlb_elo_cache_2025
    # game log (1,659 games, 2026 season, leakage-safe -- only pre-game info used per
    # prediction) by replaying this exact blend formula outside the notebook:
    #   naive (always predict league-avg 9.0 total)........... MAE 3.677
    #   base-only prediction (0% ELO)........................  MAE 3.745
    #   CURRENT blend (65% base / 35% ELO, i.e. old _ELO_W)...  MAE 3.861
    #   ELO-only (100% ELO)....................................MAE 4.885
    #   sweep: ELO weight 0.10 -> MAE 3.736 (best), 0.20 -> 3.764, 0.35 -> 3.861
    # The old 0.35 weight was making total-runs predictions WORSE than not blending
    # in ELO at all -- every game on the card was carrying a total-runs projection
    # actively hurt by this term. 0.10 matched the sweep's best point and is a small
    # enough weight that the ELO term acts as a light nudge, not a dominant input,
    # which also addresses the root cause (pitcher ELO has ~10x the spread of team
    # ELO) -- so even a modest blend weight let SP ELO swing the number far more
    # than team offense/defense should.
    # CAVEAT: the backtest's "base-only" side is a simplified reimplementation
    # (rolling team runs-for/against + starter rolling ERA) -- it does not include
    # this cell's real weather/park/bullpen-blend/platoon/series-momentum refinements,
    # so the exact MAE numbers won't match a live run. The relative comparison (does
    # adding ELO help or hurt, and by how much) should be far more robust to that gap
    # than the absolute numbers are. Re-validate against the real base pipeline if the
    # accuracy of this exact weight matters more precisely than "much lower than 0.35".
    _ELO_W = 0.10
    _ELO_BASE = 1500.0
    # inline copy so this cell works even if Cell 75 was not run
    def _elo_exp_runs(off_e, def_e, sp_e, _lg=4.50):
        return min(8.5, max(0.5, _lg * (10 ** ((off_e - (0.4*def_e + 0.6*sp_e)) / 400.0))))
    mlb_exp_runs_fn = _elo_exp_runs
    # Fix: this used a bare exact-string .get() against mlb_team_off_elo/
    # mlb_team_def_elo, which are keyed by the box-score's ORIGINAL-CASE team
    # name. Any drift between the board's team-name source and the box-score
    # name (case, "D-backs" vs "Diamondbacks", etc.) silently fell back to the
    # 1500 default with zero visibility. Reuses the same case-insensitive
    # exact-then-mascot-fallback resolution already proven in _mlb_elo_avail()
    # below, inlined here since that helper is defined later in this cell.
    _MLB_TEAM_ALIASES = {
        "d-backs": "diamondbacks", "dbacks": "diamondbacks", "d backs": "diamondbacks",
    }
    def _elo_team_key(team_name, elo_dict):
        _tn = str(team_name).lower()
        _k = next((k for k in elo_dict if k.lower() == _tn), None)
        if _k is None:
            _mascot = _tn.split()[-1] if _tn.split() else ""
            if len(_mascot) > 3:
                _k = next((k for k in elo_dict if k.lower().endswith(_mascot)), None)
        if _k is None and _tn in _MLB_TEAM_ALIASES:
            _alias_mascot = _MLB_TEAM_ALIASES[_tn]
            _k = next((k for k in elo_dict if k.lower().endswith(_alias_mascot)), None)
        if _k is None:
            # last-word alias check too -- e.g. "Arizona D-backs" -> mascot "d-backs"
            _mascot2 = _tn.split()[-1] if _tn.split() else ""
            if _mascot2 in _MLB_TEAM_ALIASES:
                _k = next((k for k in elo_dict if k.lower().endswith(_MLB_TEAM_ALIASES[_mascot2])), None)
        return _k
    try:
        _have_elo = isinstance(mlb_team_off_elo, dict) and len(mlb_team_off_elo) > 0
    except NameError:
        _have_elo = False
    if not _have_elo:
        print("  [ELO UNAVAIL] ELO data missing — confidence discount applied to all plays")
        globals()["_elo_unavailable_flag"] = True
    if _have_elo:
        for _idx2, _row2 in board.iterrows():
            _aw2 = str(_row2.get(away_col, ""))
            _hm2 = str(_row2.get(home_col, ""))
            _asp2 = next((str(_row2.get(c) or "").strip() for c in ["away_pitcher_final_name","away_pitcher_name","away_starter"] if _row2.get(c)), "")
            _hsp2 = next((str(_row2.get(c) or "").strip() for c in ["home_pitcher_final_name","home_pitcher_name","home_starter"] if _row2.get(c)), "")
            _ao2 = mlb_team_off_elo.get(_elo_team_key(_aw2, mlb_team_off_elo), _ELO_BASE)
            _ho2 = mlb_team_off_elo.get(_elo_team_key(_hm2, mlb_team_off_elo), _ELO_BASE)
            _hd2 = mlb_team_def_elo.get(_elo_team_key(_hm2, mlb_team_def_elo), _ELO_BASE)
            _ad2 = mlb_team_def_elo.get(_elo_team_key(_aw2, mlb_team_def_elo), _ELO_BASE)
            _aspe2 = mlb_pitcher_elo.get(_asp2, _ELO_BASE)
            _hspe2 = mlb_pitcher_elo.get(_hsp2, _ELO_BASE)
            _elo_ar = mlb_exp_runs_fn(_ao2, _hd2, _hspe2)
            _elo_hr = mlb_exp_runs_fn(_ho2, _ad2, _aspe2)
            _bar = board.at[_idx2, "pred_away_runs"]
            _bhr = board.at[_idx2, "pred_home_runs"]
            if pd.notna(_bar):
                board.at[_idx2, "pred_away_runs"] = round(
                    (1 - _ELO_W) * _bar + _ELO_W * _elo_ar, 1
                )
            if pd.notna(_bhr):
                board.at[_idx2, "pred_home_runs"] = round(
                    (1 - _ELO_W) * _bhr + _ELO_W * _elo_hr, 1
                )
            # Opponent-quality-adjusted offense
            # When facing an elite SP (ELO>=1570), blend toward the team's
            # historical avg runs vs elite pitching (FIP<=3.20 starts, L-30).
            # Prevents inflated raw averages from overstating offense vs aces.
            _ELITE_SP_THRESH = 1570
            _VS_ELITE_W      = 0.30
            try:
                _bar3 = board.at[_idx2, "pred_away_runs"]
                if pd.notna(_bar3) and _hspe2 >= _ELITE_SP_THRESH:
                    _vs_e_aw = mlb_team_off_vs_elite.get(_aw2)
                    if _vs_e_aw is not None:
                        board.at[_idx2, "pred_away_runs"] = round(
                            (1 - _VS_ELITE_W) * _bar3 + _VS_ELITE_W * _vs_e_aw, 1
                        )
                _bhr3 = board.at[_idx2, "pred_home_runs"]
                if pd.notna(_bhr3) and _aspe2 >= _ELITE_SP_THRESH:
                    _vs_e_hm = mlb_team_off_vs_elite.get(_hm2)
                    if _vs_e_hm is not None:
                        board.at[_idx2, "pred_home_runs"] = round(
                            (1 - _VS_ELITE_W) * _bhr3 + _VS_ELITE_W * _vs_e_hm, 1
                        )
            except NameError:
                print("  [WARN] mlb_team_off_vs_elite missing -- re-run Cell 77")
            except Exception as _qe:
                print(f"  [WARN] vs-elite adj failed: {_qe}")
        board["pred_total_runs"] = (
            board["pred_away_runs"].fillna(0) + board["pred_home_runs"].fillna(0)
        ).where(board["pred_away_runs"].notna() & board["pred_home_runs"].notna(), np.nan)
        board["pred_total_runs"] = board["pred_total_runs"].round(1)
        print(f"  ELO adjustment applied (weight={_ELO_W})")
    else:
        # Fix: this used to be the `else` on the 7-inning DH check below, so it
        # printed on any non-doubleheader day regardless of whether ELO actually
        # loaded -- completely disconnected from the real _have_elo status.
        print("  ELO data not available -- using base predictions")
    # 7-inning DH: scale run projections by 7/9 for double-header game 2
    if "_dh_7inn_set" in globals() and _dh_7inn_set and "pred_total_runs" in board.columns:
        _7scale = 7.0 / 9.0
        for _7idx in board.index:
            _7hfull = str(board.at[_7idx, home_col] if home_col in board.columns else "").lower()
            _7match = any(
                any(w in _7hfull for w in _hraw.split() if len(w) > 3)
                for _hraw in _dh_7inn_set
            )
            if _7match:
                for _7col in ["pred_away_runs", "pred_home_runs", "pred_total_runs"]:
                    if _7col in board.columns and pd.notna(board.at[_7idx, _7col]):
                        board.at[_7idx, _7col] = round(board.at[_7idx, _7col] * _7scale, 1)
                print(f"  7-inn DH: scaled projections x0.78 for {board.at[_7idx, home_col]}")

    n_scored = int(board["pred_total_runs"].notna().sum())
    print(f"  Predicted scores computed for {n_scored}/{len(board)} games")
    if n_scored > 0:
        _show = [c for c in [away_col, home_col, "pred_away_runs", "pred_home_runs",
                              "pred_total_runs", "park_factor_used", "game_temp_f"]
                 if c in board.columns]
        _show_df = board[_show].copy().reset_index(drop=True)
        _dh_mask = _show_df.duplicated(subset=[away_col, home_col], keep=False)
        if _dh_mask.any():
            _show_df["note"] = ""
            _cnt = _show_df[_dh_mask].groupby([away_col, home_col]).cumcount().add(1)
            _show_df.loc[_dh_mask, "note"] = _cnt.astype(str).radd("DH-G")
        _show_df = _show_df.round(2)
        # Fix: this table prints game_temp_f as a raw pandas value -- dome/retractable
        # roof games show literal "NaN" instead of an explicit flag. The other spots
        # (RL section, O/U display) got [DOME/WEATHER N/A] earlier; this table was missed.
        if "game_temp_f" in _show_df.columns:
            _show_df["game_temp_f"] = _show_df["game_temp_f"].apply(
                lambda v: "DOME" if pd.isna(v) else v)
        print(_show_df.to_string(index=False))
else:
    print("  WARN: run average columns not found -- scores will not be shown")

# ---------------------------------------------------------
# MANUAL STARTER OVERRIDES
# If ESPN pulled wrong starters, fill in this dict and re-run.
# Key = substring of away_team name. Values: away_sp, home_sp.
# Example: STARTER_OVERRIDES = {"Rangers": {"away_sp": "MacKenzie Gore", "home_sp": "Patrick Detmers"}}
# ---------------------------------------------------------
STARTER_OVERRIDES = {}

# MANUAL O/U OPENING LINE TRACKER
# Fill during morning research when you spot line movement.
# Key = away team substring (case-insensitive). Value = opening O/U line.
# Model flags AGAINST when line moved >= 0.5 runs the wrong way.
# Example: OPENING_LINE_OVERRIDES = {"Rangers": 8.0}  # current line moved to 9.0
OPENING_LINE_OVERRIDES = {}

# MANUAL INJURY NOTES — fill in when ESPN API is down or data is stale
# Key = substring of team name. Value = list of "Player (status)" strings.
# Example: MANUAL_INJURY_NOTES = {"Rangers": ["Corey Seager (IL-10)", "Max Scherzer (IL-60)"]}
MANUAL_INJURY_NOTES = {}
if STARTER_OVERRIDES and "away_team" in board.columns:
    for _ov_key, _ov_vals in STARTER_OVERRIDES.items():
        _ov_mask = board["away_team"].str.contains(_ov_key, case=False, na=False)
        if not _ov_mask.any():
            _ov_mask = board["home_team"].str.contains(_ov_key, case=False, na=False)
        if _ov_mask.any():
            if "away_sp" in _ov_vals:
                for _oc in ["away_pitcher_final_name","away_pitcher_name","away_starter"]:
                    if _oc in board.columns: board.loc[_ov_mask, _oc] = _ov_vals["away_sp"]
            if "home_sp" in _ov_vals:
                for _oc in ["home_pitcher_final_name","home_pitcher_name","home_starter"]:
                    if _oc in board.columns: board.loc[_ov_mask, _oc] = _ov_vals["home_sp"]
            print(f"  STARTER OVERRIDE applied: {_ov_key} -- {_ov_vals}")
        else:
            print(f"  STARTER OVERRIDE: '{_ov_key}' not found in board")

# ---------------------------------------------------------
# 5) resolve remaining columns
# ---------------------------------------------------------
date_col    = pick_col(board, ["date", "game_date", "game_datetime", "game_datetime_utc", "start_time"])
game_id_col = pick_col(board, ["game_id", "gamepk", "game_pk", "pk"])

# ---------------------------------------------------------
# [STALE SLATE CHECK] warn if board's game dates don't match CARD_DATE --
# catches a leftover board from a prior kernel session being silently
# reused as if it were today's card (root-caused an Aug 2026 4-day-stale
# card incident: cell 26 re-fetches correctly, but nothing here ever
# checked the board it grabbed actually matched CARD_DATE)
# ---------------------------------------------------------
if date_col and "CARD_DATE" in globals():
    _slate_dates_et = (
        pd.to_datetime(board[date_col], errors="coerce", utc=True) - pd.Timedelta(hours=4)
    ).dt.normalize().dt.tz_localize(None)
    _card_date_ts = pd.Timestamp(CARD_DATE)
    _stale_rows = _slate_dates_et.notna() & (_slate_dates_et != _card_date_ts)
    if _stale_rows.any():
        print("=" * 80)
        print(f"[STALE SLATE WARNING] board_source='{board_source}' has {_stale_rows.sum()}/{len(board)} "
              f"game(s) NOT matching CARD_DATE={CARD_DATE}.")
        print(f"  Board date(s) found: {sorted(set(_slate_dates_et.dropna().dt.date))}")
        print("  This board is likely leftover from a previous session -- re-run from Cell 11 (schedule pull) onward before trusting this card.")
        print("=" * 80)

# [DAY GAMES TODAY] header -- games with first pitch before 2PM ET
if date_col:
    _dg_games = []
    for _dg_idx in board.index:
        _dg_dt = pd.to_datetime(board.at[_dg_idx, date_col], errors="coerce", utc=True)
        if pd.isna(_dg_dt):
            continue
        _dg_hour_et = (_dg_dt.hour - 4) % 24  # matches the sharp-money ET conversion elsewhere in this cell
        if _dg_hour_et < 14:
            _dg_a = board.at[_dg_idx, away_col] if away_col in board.columns else "?"
            _dg_h = board.at[_dg_idx, home_col] if home_col in board.columns else "?"
            _dg_hr12 = _dg_hour_et % 12 or 12
            _dg_ampm = "AM" if _dg_hour_et < 12 else "PM"
            _dg_games.append(f"{_dg_a} @ {_dg_h} ({_dg_hr12}{_dg_ampm} ET)")
    if _dg_games:
        print("=" * 58)
        print(f"  [DAY GAMES TODAY: {len(_dg_games)} game(s)]")
        for _dg_g in _dg_games:
            print(f"    {_dg_g}")
        print("=" * 58)

pred_home_col = pick_col(board, [
    "pred_home_win_prob", "p_home_win_calibrated", "p_home_win_final",
    "p_home_win_model", "p_home_win",
])
pred_away_col = pick_col(board, ["pred_away_win_prob", "p_away_win"])

home_price_col = pick_col(board, [
    "home_price", "home_entry_price", "home_yes_price", "price_home", "yes_price_home",
])
away_price_col = pick_col(board, [
    "away_price", "away_entry_price", "away_yes_price", "price_away", "yes_price_away",
])
edge_home_col = pick_col(board, ["edge_home", "home_edge"])
edge_away_col = pick_col(board, ["edge_away", "away_edge"])

print("\nResolved columns:")
for _lbl, _val in [
    ("away_col", away_col), ("home_col", home_col), ("date_col", date_col),
    ("pred_home_col", pred_home_col), ("pred_away_col", pred_away_col),
]:
    print(f"  {_lbl:<18}: {_val}")

if away_col is None or home_col is None:
    raise ValueError("Could not resolve away/home team columns.")
if pred_home_col is None and pred_away_col is None:
    raise ValueError("Could not resolve model win probability columns.")

# BUG FIX (2026-09-02, found live investigating a real Orioles loss): Cell
# H2H already computes a real, Bayesian-shrunk current-season head-to-head
# win rate and blends it into pred_home_win_prob -- but that write happens
# on scored_mlb_board, which "SCORE DAILY MLB BOARD WITH FINAL WIN MODEL"
# (a later cell) rebuilds from the raw `board` global instead of
# scored_mlb_board, silently discarding it. The blend has likely never
# actually influenced a card decision. Reapplying it here from
# mlb_h2h_lookup (a standalone team-pair-keyed global Cell H2H also
# populates, which survives that rebuild -- same pattern mlb_bp_b2b /
# mlb_sp_former_team / _series_context already use) so it's real again,
# not just a printed note. Placed right before the clip/renormalize step
# below so it's subject to the same safety rails as every other probability
# input, instead of being a late, unclipped tweak.
if "mlb_h2h_lookup" in globals() and mlb_h2h_lookup:
    def _h2h_blend_row(_hbr):
        _hbr_aw = str(_hbr.get(away_col, "")).lower()
        _hbr_hm = str(_hbr.get(home_col, "")).lower()
        _hbr_entry = mlb_h2h_lookup.get((_hbr_aw, _hbr_hm)) or mlb_h2h_lookup.get((_hbr_hm, _hbr_aw))
        if not _hbr_entry or not _hbr_entry.get("h2h_weight"):
            return None
        return _hbr_entry

    _h2h_applied_ct = 0
    for _h2h_bidx in board.index:
        _h2h_res = _h2h_blend_row(board.loc[_h2h_bidx])
        if _h2h_res is None:
            continue
        _h2h_bw, _h2h_brate = _h2h_res["h2h_weight"], _h2h_res["h2h_bayesian_rate"]
        if pred_home_col is not None and pd.notna(board.at[_h2h_bidx, pred_home_col]):
            _h2h_orig = float(board.at[_h2h_bidx, pred_home_col])
            _h2h_new = (1 - _h2h_bw) * _h2h_orig + _h2h_bw * _h2h_brate
            board.at[_h2h_bidx, pred_home_col] = _h2h_new
            if pred_away_col is not None:
                board.at[_h2h_bidx, pred_away_col] = 1.0 - board.at[_h2h_bidx, pred_home_col]
            _h2h_applied_ct += 1
            # transparency (2026-09-09, user audit): store the actual pp
            # change so the card can show how much the H2H blend moved the
            # pick, not just the raw run-diff context that was there before.
            _h2h_res["h2h_pp_delta_applied"] = (_h2h_new - _h2h_orig) * 100.0
    if _h2h_applied_ct:
        print(f"  [H2H BLEND] applied current-season H2H adjustment to {_h2h_applied_ct} game(s)")

# ---------------------------------------------------------
# 6) model prob setup
# ---------------------------------------------------------
if pred_home_col is not None:
    board[pred_home_col] = safe_num(board[pred_home_col])
if pred_away_col is None and pred_home_col is not None:
    board["pred_away_win_prob"] = 1.0 - board[pred_home_col]
    pred_away_col = "pred_away_win_prob"
else:
    board[pred_away_col] = safe_num(board[pred_away_col])
if pred_home_col is None and pred_away_col is not None:
    board["pred_home_win_prob"] = 1.0 - board[pred_away_col]
    pred_home_col = "pred_home_win_prob"

_CLIP_MAX, _CLIP_MIN = 0.70, 0.30
board[pred_home_col] = safe_num(board[pred_home_col]).clip(_CLIP_MIN, _CLIP_MAX)
board[pred_away_col] = safe_num(board[pred_away_col]).clip(_CLIP_MIN, _CLIP_MAX)
_total = board[pred_home_col] + board[pred_away_col]
board[pred_home_col] = board[pred_home_col] / _total
board[pred_away_col] = board[pred_away_col] / _total

# Blend model prob with score-derived prob to prevent contradictions
# (e.g. model backing home team while projection shows away winning)
_SCORE_BLEND = 0.55  # weight on projected-score probability (raised: score proj has pitcher data)
_MLB_RUN_STD = 3.0   # typical game-level uncertainty in runs (tightened from 3.5)
if "pred_home_runs" in board.columns and "pred_away_runs" in board.columns:
    from scipy import stats as _sc_stats
    _margin = safe_num(board["pred_home_runs"]) - safe_num(board["pred_away_runs"])
    _has_proj = _margin.notna()
    _score_home_p = _margin.apply(
        lambda m: float(_sc_stats.norm.cdf(m / _MLB_RUN_STD)) if (m == m) else 0.5
    )
    board.loc[_has_proj, pred_home_col] = (
        (1 - _SCORE_BLEND) * board.loc[_has_proj, pred_home_col]
        + _SCORE_BLEND * _score_home_p[_has_proj]
    )
    board.loc[_has_proj, pred_away_col] = 1.0 - board.loc[_has_proj, pred_home_col]
    board['proj_margin_proxy'] = _margin  # home runs - away runs; used by card builders

# Pitcher ELO adjustment: add systematic pitcher quality signal to board probs
# Each 100 ELO pts difference -> 2.5% win-prob shift, capped at +/-5%
_pit_home_cols = ['home_pitcher_final_name', 'home_pitcher_name', 'home_starter']
_pit_away_cols = ['away_pitcher_final_name', 'away_pitcher_name', 'away_starter']
if 'mlb_pitcher_elo' in globals() and isinstance(mlb_pitcher_elo, dict):
    # Home/road split adjustment -- the SP ELO above is a single season-blended
    # rating with no venue awareness, so a starter with an extreme split (e.g. a
    # 4+ run home/road ERA gap) reads as one average pitcher regardless of where
    # he's throwing today. Gated on sample size AND split magnitude so it only
    # fires on real, repeated signal rather than small-sample noise.
    # 2026-07-30: switched from FIP to ERA (see cell 83 tracker for why -- FIP
    # was structurally blind to Woo's real 3.47-run road ERA collapse). Gated
    # on IP (not start count) since ERA stability depends on innings, not
    # starts; the IP floor itself lives in cell 83's mlb_sp_split_era builder
    # (25 IP/side current-season, 40 IP/side all-time fallback), so an entry
    # existing at all already means the sample is trusted.
    # 2026-09-10 (user audit, item 1): raised 1.00 -> 1.50 to match the user's exact
    # spec ("when a SP's road ERA diverges from home ERA by more than 1.5R over 8+
    # starts") -- also added the starts>=8 sample-size floor this function was
    # previously missing entirely (it only ever checked mlb_sp_split_era's own IP
    # floor, 25/40 IP-per-side, which is a real but coarser proxy for "enough
    # starts" than counting starts directly).
    _SP_SPLIT_MIN_GAP    = 1.50   # ERA runs between home/road before it's trusted (ERA is noisier than FIP)
    _SP_SPLIT_MIN_STARTS = 8      # starts on the relevant (home or road) side before the split is trusted
    # 2026-07-30: raised 60.0 -> 275.0. Verified live against Woo (2026: 2.20
    # home / 6.52 road ERA, 57+ IP each side) -- the uncapped delta was -172.8,
    # which the old 60-pt cap clipped to a token -60, leaving him at 1740 ELO
    # (still elite tier) despite a real 4.32-run collapse. Safe to raise because
    # the *ultimate* win-prob impact is independently hard-capped downstream at
    # +/-5% regardless of the raw ELO gap (see _pit_adj.clip(-0.05, 0.05) below)
    # -- this cap only controls how much of a real, IP-floor-gated ERA gap the
    # split adjustment is allowed to express before that clip takes over.
    _SP_SPLIT_MAX_ADJ    = 275.0
    _SP_SPLIT_ELO_SCALE  = 40.0   # ELO pts per 1.00 ERA run of split gap
    # 2026-08-01: both pitcher-name lookups below (mlb_pitcher_elo, mlb_sp_split_era)
    # used the raw name string with zero normalization and a SILENT fallback on any
    # mismatch -- unlike the ERA-recency lookup a few lines down (_sp_norm_era +
    # an explicit "[SP ELO STALE]" print when it can't find a match). A pitcher
    # whose name is formatted even slightly differently between data sources
    # (extra whitespace, a suffix, an accented/international name) got silently
    # mis-rated with zero visibility that it happened -- same bug shape already
    # found and fixed in the Tennis model this week (Jodar/key-normalization).
    # Exact match still tried first (no behavior change for the common case);
    # normalized fallback + explicit warning only engage on an actual miss.
    import unicodedata as _ud_sp, re as _re_sp
    def _pname_norm_sp(s):
        if not s: return ''
        s = _ud_sp.normalize('NFD', str(s))
        s = ''.join(c for c in s if _ud_sp.category(c) != 'Mn')
        return _re_sp.sub(r'[^a-z ]', '', s.lower()).strip()
    _split_resolved = {}  # caller-supplied name -> resolved mlb_sp_split_era entry (any match path)
    def _sp_split_elo_adj(pitcher_name, is_home):
        if 'mlb_sp_split_era' not in globals():
            return 0.0
        _pn = str(pitcher_name).strip()
        _sv = mlb_sp_split_era.get(_pn)
        if _sv is None:
            _nk = _pname_norm_sp(_pn)
            _sv = next((v for k, v in mlb_sp_split_era.items() if _pname_norm_sp(k) == _nk), None)
        if not _sv:
            return 0.0
        _split_resolved[_pn] = _sv
        _he, _re = _sv.get('home_era'), _sv.get('road_era')
        if _he is None or _re is None:
            return 0.0
        if abs(_re - _he) < _SP_SPLIT_MIN_GAP:
            return 0.0
        _relevant_starts = _sv.get('home_starts', 0) if is_home else _sv.get('road_starts', 0)
        if _relevant_starts < _SP_SPLIT_MIN_STARTS:
            return 0.0
        _this_era  = _he if is_home else _re
        _other_era = _re if is_home else _he
        _delta = (_other_era - _this_era) * _SP_SPLIT_ELO_SCALE
        return max(-_SP_SPLIT_MAX_ADJ, min(_SP_SPLIT_MAX_ADJ, _delta))
    def _get_sp_elo(row, cols, is_home):
        for c in cols:
            v = row.get(c)
            if v and str(v) != 'nan' and str(v).strip():
                _nm = str(v).strip()
                if _nm in mlb_pitcher_elo:
                    _base = float(mlb_pitcher_elo[_nm])
                else:
                    _nk = _pname_norm_sp(_nm)
                    _norm_match = next((v for k, v in mlb_pitcher_elo.items() if _pname_norm_sp(k) == _nk), None)
                    if _norm_match is not None:
                        _base = float(_norm_match)
                    else:
                        print(f"  [SP BASE ELO STALE] {_nm}: no match in mlb_pitcher_elo (exact or normalized) -- defaulting to 1460 (conservative BELOW)")
                        _base = 1460.0
                _sadj = _sp_split_elo_adj(_nm, is_home)
                if _sadj:
                    _sv = _split_resolved.get(_nm)
                    if _sv:
                        print(f"  [SP SPLIT] {_nm}: {'home' if is_home else 'road'} ERA-based ELO adj {_sadj:+.0f} "
                              f"(home {_sv['home_era']:.2f} ERA/{_sv['home_ip']:.1f} IP "
                              f"vs road {_sv['road_era']:.2f} ERA/{_sv['road_ip']:.1f} IP, {_sv['source']})")
                return _base + _sadj
        return 1460.0  # unknown pitcher = conservative BELOW tier
    _h_sp_elos = board.apply(lambda r: _get_sp_elo(r, _pit_home_cols, True), axis=1)
    _a_sp_elos = board.apply(lambda r: _get_sp_elo(r, _pit_away_cols, False), axis=1)
    # MLB-PS-PIGGYPRICE-1: an announced opener (or TBD) in a postseason game
    # is not "a starter" for the SP-vs-SP rating nudge -- its default/unrated
    # number (Kerr: ~1500 off 1 start) inflated the gap against the other
    # side's real starter (+4.8pp for ATL on 2026-10-01). Bullpen-game prior:
    # use the named bulk arm's rating when one was priced, otherwise set the
    # side equal to the opposing starter (zero nudge -- no starter information;
    # the bullpen is already in the run projection).
    _ps_arm_map = globals().get("mlb_ps_bulk_arm", {}) or {}
    for _pe_i in board.index:
        _pe_key = (str(board.at[_pe_i, away_col]).lower().strip(), str(board.at[_pe_i, home_col]).lower().strip())
        _pe = _ps_arm_map.get(_pe_key)
        if not _pe:
            continue
        for _pe_side, _pe_ser, _pe_opp in (("away", _a_sp_elos, _h_sp_elos), ("home", _h_sp_elos, _a_sp_elos)):
            if _pe.get(f"{_pe_side}_role") not in ("OPENER", "TBD"):
                continue
            _arm_nm = _pe.get(_pe_side) if _pe.get(f"{_pe_side}_role") == "OPENER" else None
            _arm_elo = None
            if _arm_nm:
                _arm_elo = mlb_pitcher_elo.get(_arm_nm)
                if _arm_elo is None:
                    _arm_elo = next((v for k, v in mlb_pitcher_elo.items() if _pname_norm_sp(k) == _pname_norm_sp(_arm_nm)), None)
            _old_e = float(_pe_ser.at[_pe_i])
            _new_e = float(_arm_elo) if _arm_elo is not None else float(_pe_opp.at[_pe_i])
            _pe_ser.at[_pe_i] = _new_e
            print(f"  [OPENER SP RATING] {board.at[_pe_i, away_col if _pe_side == 'away' else home_col]}: "
                  f"{_old_e:.0f} -> {_new_e:.0f} ({'bulk arm ' + _arm_nm if _arm_elo is not None else 'no bulk-arm rating -- matched to the opposing starter, zero SP nudge'})")
    def _sp_name_resolved(row, cols):
        for c in cols:
            v = row.get(c)
            if v and str(v) != 'nan' and str(v).strip():
                return True
        return False
    _h_sp_name_resolved = board.apply(lambda r: _sp_name_resolved(r, _pit_home_cols), axis=1)
    _a_sp_name_resolved = board.apply(lambda r: _sp_name_resolved(r, _pit_away_cols), axis=1)

    # Current-season ERA recency adjustment (updated: L3 blend + small sample regression)
    # Blends L3 ERA (60%) + season RA avg (40%) when >= 12 L3 IP available.
    # Applies a graduated regression to league-mean ERA below _SMALL_IP_THRESH,
    # tapering to 0 extra regression right at the threshold (see _reg_w below).
    # Real gap found 2026-07-23: this was 30.0 IP, well under where ERA
    # actually stabilizes (~60-70 IP is the standard sabermetric range) --
    # a pitcher like Legumina at 41.1 IP read as "enough sample" here and got
    # zero extra regression toward the mean, even though a 3.92 ERA at that
    # sample size is still meaningfully more volatile than xFIP/SIERA. Raised
    # to sit in the middle of that range.
    _ERA_SCALE       = 150.0
    _LEAGUE_ERA      = 4.50
    _SEASON_W_LOW    = 0.30
    _SEASON_W_HIGH   = 0.70
    _DIV_THRESH      = 1.00
    _SMALL_IP_THRESH = 65.0
    _SMALL_SAMPLE_W  = 0.40

    import unicodedata as _ud_era, re as _re_era
    def _sp_norm_era(s):
        if not s: return ''
        s = _ud_era.normalize('NFD', str(s))
        s = ''.join(c for c in s if _ud_era.category(c) != 'Mn')
        return _re_era.sub(r'[^a-z ]', '', s.lower()).strip()

    def _era_implied_elo(era_val):
        if era_val is None or pd.isna(float(era_val)) or float(era_val) <= 0:
            return None
        return 1500.0 + (_LEAGUE_ERA - float(era_val)) * _ERA_SCALE

    def _recency_adj(hist_elo, ra10_val, pitcher_name=None):
        # Don't seed season_era from team-level ra10_val for named pitchers:
        # if the lookup misses them (name mismatch / IL returnee) the team RA
        # can silently inflate or deflate their rating.
        season_era = (float(ra10_val) * 9.0 / 10.0
                      if ra10_val and float(ra10_val or 0) > 0 and not pitcher_name else None)
        l3_era = l3_ip = ip_season = l9_era = None; _xera = None
        if pitcher_name and 'mlb_pitcher_fip_lookup' in globals():
            _ek = _sp_norm_era(pitcher_name)
            _ev = mlb_pitcher_fip_lookup.get(_ek, {})
            _el3 = _ev.get('era_l3'); _il3 = _ev.get('ip_l3'); _ips = _ev.get('ip_season')
            _seas = _ev.get('era_season')
            if _el3 is not None and not pd.isna(float(_el3)): l3_era = float(_el3)
            if _il3 is not None and not pd.isna(float(_il3)): l3_ip  = float(_il3)
            if _ips is not None and not pd.isna(float(_ips)): ip_season = float(_ips)
            # era_l9 (Cell 20, already fetched) was never read here -- L3
            # alone is a 3-start window and can't tell a real sustained run
            # (e.g. Singer's 8-start hot streak, 2.76 ERA since June) apart
            # from a 3-start hot blip inside a longer slump.
            _el9 = _ev.get('era_l9')
            if _el9 is not None and not pd.isna(float(_el9)): l9_era = float(_el9)
            if _ev.get('xera') is not None:
                try:
                    _xera_c = float(_ev.get('xera'))
                    if not pd.isna(_xera_c): _xera = _xera_c
                except (ValueError, TypeError): pass
            # Use pitcher-specific season ERA (overrides team-level ra10_val baseline)
            # Prevents IL returnees from inheriting team's recent clean rotation numbers
            if _seas is not None and not pd.isna(float(_seas)) and float(_seas) > 0:
                season_era = float(_seas)
        # Fallback: _sp_form_cache is populated from prior Cell 86 run and available at ELO adj time
        # On first run it's empty; on subsequent runs it has last session's direct API data
        if pitcher_name and '_sp_form_cache' in globals() and _sp_form_cache:
            _sc_k = _sp_norm_era(pitcher_name)
            _sc = _sp_form_cache.get(_sc_k)
            if _sc:
                if l3_era is None and _sc.get('era_l3') is not None:
                    try:
                        _sc_l3 = float(_sc['era_l3'])
                        if not pd.isna(_sc_l3): l3_era = _sc_l3
                    except: pass
                if l3_ip is None and _sc.get('ip_l3') is not None:
                    try:
                        _sc_ip = float(_sc['ip_l3'])
                        if not pd.isna(_sc_ip): l3_ip = _sc_ip
                    except: pass
                if _sc.get('era_season') is not None:
                    try:
                        _sc_seas = float(_sc['era_season'])
                        if not pd.isna(_sc_seas) and _sc_seas > 0:
                            season_era = _sc_seas  # direct API data overrides team-level estimate
                    except: pass
        # MLB-SP-ELO-EST-1 (2026-10-01): with an estimator rating in Cell 82,
        # blending back toward ACTUAL season ERA here would re-import exactly
        # the luck the estimator removed (Nola: hist 1205 -> 1309 via a 4.67
        # ERA vs a 4.18 xERA). When the estimator exists, it is the season
        # anchor; actual ERA stays only as the fallback.
        if pitcher_name and "_mlb_sp_est_era" in globals() and 'mlb_pitcher_fip_lookup' in globals():
            _est_v, _est_lab = _mlb_sp_est_era(mlb_pitcher_fip_lookup.get(_sp_norm_era(pitcher_name), {}))
            if _est_v is not None:
                if season_era is not None and abs(season_era - _est_v) >= 0.25:
                    print(f"  [SP EST ANCHOR] {pitcher_name}: season anchor ERA {season_era:.2f} -> estimator {_est_v:.2f} ({_est_lab} 20/40/40)")
                season_era = round(_est_v, 3)
        # Item 3 fix (2026-07-09, reordered 2026-07-10): flag + re-anchor season ERA
        # when it diverges from xERA by 1.5+ runs -- surface ERA outliers (bad-luck
        # sequencing, home/road split noise) shouldn't bleed into the grade at face
        # value. Runs after the _sp_form_cache fallback so it also catches pitchers
        # whose season_era only resolved via that fallback (not mlb_pitcher_fip_lookup).
        if _xera is not None and season_era is not None and abs(season_era - _xera) >= 1.5:
            _fip_disp = None
            try:
                _fip_v = _ev.get('fip') if pitcher_name and 'mlb_pitcher_fip_lookup' in globals() else None
                if _fip_v is not None and not pd.isna(float(_fip_v)): _fip_disp = float(_fip_v)
            except (ValueError, TypeError): pass
            _fip_str = f", FIP {_fip_disp:.2f}" if _fip_disp is not None else ""
            print(f"  [ERA OUTLIER] {pitcher_name}: season ERA {season_era:.2f} vs xERA {_xera:.2f}{_fip_str} -- diverge 1.5+, treating ERA as outlier and anchoring to xERA")
            season_era = round(0.20 * season_era + 0.80 * _xera, 3)
        if season_era is None and l3_era is None:
            if pitcher_name:
                print(f"  [SP ELO STALE] {pitcher_name}: no current-season ERA/FIP match -- ELO left unblended at {hist_elo:.0f} (verify name matches mlb_pitcher_fip_lookup/_sp_form_cache)")
            return hist_elo
        if l3_era is not None and l3_ip is not None and l3_ip >= 12.0:
            # FIX 1 -- aggressive hot/cold weighting
            # Cold (L3 ERA > 5.50): historical ELO is misleading � weight L3 hard
            # Hot  (L3 ERA < 3.00 and season much worse): recent form is real signal
            # xERA blend: L3 ERA alone overcorrects on small-sample spikes
            # (one bad, unlucky start can push a 3-start ERA to 6+). Blend in
            # season xERA (Statcast quality-of-contact based, not results-
            # based) at 50% so a single bad-luck outing doesn't get read as
            # true talent decline. Falls back to raw l3_era if xERA missing.
            _l3_era_for_calc = l3_era
            if _xera is not None:
                _l3_era_for_calc = round(0.50 * l3_era + 0.50 * _xera, 3)
            _season_ref  = season_era if season_era else _l3_era_for_calc
            _is_cold_l3  = _l3_era_for_calc > 5.50
            _is_hot_l3   = _l3_era_for_calc < 3.00 and (_season_ref - _l3_era_for_calc) > 1.00
            if _is_cold_l3:
                # Dampen cold-streak weight for small IP samples (<15 IP = ~2-3 starts)
                _l3_w = 0.65 if l3_ip < 15.0 else 0.85
            elif _is_hot_l3:
                # Dampen when season ERA is 1.5+ higher (regression risk case)
                _regression_risk = (_season_ref - _l3_era_for_calc) > 1.50
                _l3_w = 0.65 if _regression_risk else 0.80
                # L9 confirms the hot streak is real over a longer window (not
                # just a 3-start blip) -- trust recent form more when it does.
                if l9_era is not None and l9_era < 3.50:
                    _l3_w = min(_l3_w + 0.15, 0.90)
            else:
                _l3_w = 0.60
            cur_era = _l3_w * _l3_era_for_calc + (1.0 - _l3_w) * _season_ref
        else:
            _is_cold_l3 = False
            _is_hot_l3  = False
            cur_era = season_era if season_era is not None else l3_era
        if cur_era is None:
            return hist_elo
        _sample_ip = ip_season if ip_season is not None else l3_ip
        if _sample_ip is not None and _sample_ip < _SMALL_IP_THRESH:
            _reg_w = _SMALL_SAMPLE_W * (1.0 - _sample_ip / _SMALL_IP_THRESH)
            # Very small samples (<20 IP, ~4 GS): force heavy regression -- ERA not stable yet
            if _sample_ip < 20:
                _reg_w = max(_reg_w, 0.70)  # ~4 GS: don't let hot tag override stability gate
            elif _is_hot_l3:
                _reg_w *= 0.35   # hot pitcher + enough sample: trust recent form
            cur_era = cur_era * (1.0 - _reg_w) + _LEAGUE_ERA * _reg_w
        era_elo = _era_implied_elo(cur_era)
        if era_elo is None:
            return hist_elo
        elo_implied_era = _LEAGUE_ERA - (hist_elo - 1500.0) / _ERA_SCALE
        divergence = abs(cur_era - elo_implied_era)
        # Scale ELO blend weight by season IP -- small samples get dampened adjustment
        if _is_cold_l3:
            if _sample_ip is not None and _sample_ip >= 60:
                w = 0.92   # large sample (10+ starts): trust cold trend fully
            elif _sample_ip is not None and _sample_ip >= 36:
                w = 0.80   # medium sample (6-9 starts)
            else:
                w = 0.65   # small sample: cap to avoid tanking ELO on a few bad starts
        else:
            w = _SEASON_W_HIGH if divergence > _DIV_THRESH else _SEASON_W_LOW
        # 2026-07-26 fix: for a <20 IP sample (~4 GS or fewer), a LARGE divergence
        # between the tiny-sample ERA and the historical prior used to mean MORE
        # weight (up to 0.70-0.92) went to the noisy side -- backwards. A big gap
        # on a tiny sample is the highest-risk case, not a reason for more trust.
        # Real cases: Ronel Blanco (1 prior MLB start, back from Tommy John) and
        # Kohl Drake (1 prior MLB start, 6.92 ERA at AAA) both blended up into
        # ELITE-tier ELO off essentially one start and lost as model favorites
        # 2026-07-26. Cap the blend weight so a <20 IP sample can nudge the
        # rating but never dominate it -- the historical prior still carries
        # the pitcher's real track record (or the conservative default for one
        # who has none).
        if _sample_ip is not None and _sample_ip < 20:
            w = min(w, 0.35)
        _adj = hist_elo * (1.0 - w) + era_elo * w
        # Backstop: a <63 IP sample should never cross into ELITE/STRONG tier
        # (_sp_tier below reads >=1560 as ELITE, >=1520 as STRONG) no matter how
        # good cur_era looks post-regression -- a few-start sample hasn't earned
        # that label. Old cap (1600, only when cur_era<3.0) sat ABOVE the 1560
        # ELITE cutoff it was meant to guard, so it never actually blocked the
        # label; this replaces it with a cap below that cutoff, applied broadly.
        if _sample_ip is not None and _sample_ip < (_SMALL_IP_THRESH - 2):
            _adj = min(_adj, 1555.0)
        # 2026-07-27 fix: 1555 still clears the 1520 STRONG cutoff (_sp_tier below),
        # so a pitcher with almost no MLB track record at all -- e.g. Eddy Yean
        # (4 relief outings, ~6 IP total, this game his literal MLB starting debut,
        # 2.35 ERA) vs Nationals/Blue Jays 2026-07-27 -- could still be labeled
        # STRONG off essentially zero real information. Same blind spot as the
        # ELITE-tier bug this backstop was built for, one tier lower. Anyone under
        # the same 20 IP floor already used above to cap the blend weight gets a
        # tighter ceiling that holds them at AVG at most -- a handful of good
        # innings isn't enough to call a pitcher's true talent level "STRONG."
        if _sample_ip is not None and _sample_ip < 20:
            _adj = min(_adj, 1515.0)
        return _adj

    _h_ra10   = board.get("home_starter_runs_allowed_10", pd.Series([None]*len(board), index=board.index))
    _a_ra10   = board.get("away_starter_runs_allowed_10", pd.Series([None]*len(board), index=board.index))
    _h_pnames = board.get("home_pitcher_name", pd.Series([None]*len(board), index=board.index))
    _a_pnames = board.get("away_pitcher_name", pd.Series([None]*len(board), index=board.index))
    # Pre-fetch sp_form_cache for any pitcher not yet cached (handles first-run / Value Watch promotions)
    # _sp_fetch_recent is defined later in the cell, so only available on subsequent runs in same session
    if '_sp_fetch_recent' in dir() and '_sp_form_cache' in globals() and '_sp_norm_era' in dir():
        for _pre_pn in list(_h_pnames) + list(_a_pnames):
            if _pre_pn and str(_pre_pn).strip() not in ('', 'nan', 'None'):
                _pre_key = _sp_norm_era(str(_pre_pn))
                if _pre_key not in _sp_form_cache:
                    _pre_d = _sp_fetch_recent(str(_pre_pn))
                    if _pre_d:
                        _sp_form_cache[_pre_key] = _pre_d
    _h_sp_elos_adj = pd.Series(
        [_recency_adj(float(h), ha, pn) for h, ha, pn in zip(_h_sp_elos, _h_ra10, _h_pnames)],
        index=board.index)
    _a_sp_elos_adj = pd.Series(
        [_recency_adj(float(a), aa, pn) for a, aa, pn in zip(_a_sp_elos, _a_ra10, _a_pnames)],
        index=board.index)
    _era_adj_ct = int(
        ((_h_sp_elos_adj - _h_sp_elos).abs() > 10).sum() +
        ((_a_sp_elos_adj - _a_sp_elos).abs() > 10).sum())
    if _era_adj_ct:
        print(f"  ERA recency adj: {_era_adj_ct} pitcher(s) shifted >10 ELO pts")
        for _idx in board.index:
            for _side, _hist, _adj, _ra10, _pn in [
                ("home", _h_sp_elos[_idx], _h_sp_elos_adj[_idx],
                 _h_ra10.iloc[_idx] if hasattr(_h_ra10,"iloc") else None,
                 _h_pnames.iloc[_idx] if hasattr(_h_pnames,"iloc") else None),
                ("away", _a_sp_elos[_idx], _a_sp_elos_adj[_idx],
                 _a_ra10.iloc[_idx] if hasattr(_a_ra10,"iloc") else None,
                 _a_pnames.iloc[_idx] if hasattr(_a_pnames,"iloc") else None),
            ]:
                if abs(_adj - _hist) > 10:
                    # Show ERA: prefer sp_form_cache (direct MLB API) over Fangraphs
                    _cur_era = float(_ra10)*9/10 if _ra10 and float(_ra10 or 0)>0 else None
                    if '_sp_form_cache' in globals() and _pn:
                        _ce_sc = _sp_form_cache.get(_sp_norm_era(str(_pn)))
                        if _ce_sc and _ce_sc.get('era_season'):
                            try:
                                _ce_f = float(_ce_sc['era_season'])
                                if not pd.isna(_ce_f) and _ce_f > 0: _cur_era = _ce_f
                            except: pass
                    if _cur_era is None and _pn and 'mlb_pitcher_fip_lookup' in globals():
                        _ce_ev = mlb_pitcher_fip_lookup.get(_sp_norm_era(str(_pn)), {})
                        _ce_seas = _ce_ev.get('era_season')
                        if _ce_seas is not None:
                            try:
                                _ce_f2 = float(_ce_seas)
                                if not pd.isna(_ce_f2) and _ce_f2 > 0: _cur_era = _ce_f2
                            except: pass
                    _ss_flag = ''
                    if _pn and 'mlb_pitcher_fip_lookup' in globals():
                        _ips2 = mlb_pitcher_fip_lookup.get(_sp_norm_era(str(_pn)), {}).get('ip_season')
                        if _ips2 and not pd.isna(float(_ips2)) and float(_ips2) < _SMALL_IP_THRESH:
                            _ss_flag = f" [LOW SAMPLE: {float(_ips2):.0f} IP]"
                    _pn_disp = str(_pn).strip() if _pn else "??"
                    # Also show L3 ERA from cache so hot/cold adjustments are interpretable
                    _l3_era_disp = None
                    if _pn and '_sp_form_cache' in globals():
                        _dbg_sc = _sp_form_cache.get(_sp_norm_era(str(_pn)))
                        if _dbg_sc and _dbg_sc.get('era_l3') is not None:
                            try:
                                _dbg_l3 = float(_dbg_sc['era_l3'])
                                if not pd.isna(_dbg_l3) and _dbg_l3 >= 0: _l3_era_disp = _dbg_l3
                            except: pass
                    _xera_disp = None
                    if _pn and 'mlb_pitcher_fip_lookup' in globals():
                        _dbg_xera = mlb_pitcher_fip_lookup.get(_sp_norm_era(str(_pn)), {}).get('xera')
                        if _dbg_xera is not None:
                            try:
                                _dbg_xera_f = float(_dbg_xera)
                                if not pd.isna(_dbg_xera_f): _xera_disp = _dbg_xera_f
                            except (ValueError, TypeError): pass
                    print(f"    {_side} {_pn_disp}: hist ELO {_hist:.0f} -> adj {_adj:.0f}"
                          + (f" (season ERA {_cur_era:.2f})" if _cur_era else "")
                          + (f" [L3 ERA {_l3_era_disp:.2f}]" if _l3_era_disp else "")
                          + (f" [xERA {_xera_disp:.2f}]" if _xera_disp else "")
                          + _ss_flag)
    _h_sp_elos = _h_sp_elos_adj
    _a_sp_elos = _a_sp_elos_adj
    # Build name->adjusted ELO cache for use in stale warn + RL debug
    _sp_elo_adj_cache = {}
    for _cac_pn, _cac_adj in zip(
        list(_h_pnames) + list(_a_pnames),
        list(_h_sp_elos_adj) + list(_a_sp_elos_adj)
    ):
        if _cac_pn and str(_cac_pn).strip() not in ('', 'nan', 'None'):
            _sp_elo_adj_cache[str(_cac_pn).strip().lower()] = float(_cac_adj)

    _pit_adj = ((_h_sp_elos - _a_sp_elos) / 100.0 * 0.025).clip(-0.05, 0.05)
    # 2026-07-27 fix: this SP-quality nudge assumed both starters pitch deep into
    # the game -- but the run-projection side of this same cell (_eff_pit, ~line
    # 829) already knows better, shifting weight toward the bullpen whenever a
    # starter's OWN season-long avg IP/start marks him an opener (<3.0 IP/start)
    # or bulk-role arm (3.0-5.0 IP/start, or reliever-level K/9). That real,
    # pregame-knowable signal never reached the win-probability side at all -- a
    # game where a starter characteristically doesn't go deep still got the full,
    # undiscounted SP-ELO nudge as if his quality would govern most of the game.
    # Real case: Nationals ML loss 07-27 -- Scherzer (rusty 41yo return start) and
    # Yean (fresh call-up, first MLB start) both pulled in the 2nd/3rd inning, a
    # bullpen game start to finish, with nothing in the pregame ML edge reflecting
    # that risk. Reuses the role flags written per-row above: a true opener on
    # either side cuts the nudge to 35% (his own quality barely determines how the
    # game is actually pitched), a bulk-role arm to 65% (mirrors the 45/55 vs
    # default 65/35 split already used in the run projection); the more-discounting
    # side wins if both starters qualify. Does not touch _pit_adj's sign or cap,
    # only shrinks its magnitude for these specific games.
    _sp_short_mult = pd.Series(1.0, index=board.index)
    _role_cols = ["away_sp_is_opener", "home_sp_is_opener", "away_sp_is_bulk", "home_sp_is_bulk"]
    if all(c in board.columns for c in _role_cols):
        for _sdi in board.index:
            _any_opener = bool(board.at[_sdi, "away_sp_is_opener"]) or bool(board.at[_sdi, "home_sp_is_opener"])
            _any_bulk = bool(board.at[_sdi, "away_sp_is_bulk"]) or bool(board.at[_sdi, "home_sp_is_bulk"])
            if _any_opener:
                _sp_short_mult.at[_sdi] = 0.35
            elif _any_bulk:
                _sp_short_mult.at[_sdi] = 0.65
        _sp_short_ct = int((_sp_short_mult < 1.0).sum())
        if _sp_short_ct:
            print(f"  [SP SHORT-ROLE] {_sp_short_ct} game(s) had their SP-quality ML nudge discounted -- a starter in the game averages under 5 IP/start this season")
    _pit_adj = _pit_adj * _sp_short_mult
    board[pred_home_col] = (board[pred_home_col] + _pit_adj).clip(0.28, 0.72)
    board[pred_away_col] = 1.0 - board[pred_home_col]

    # NEW (MLB-BULK-1, 2026-09-24 user audit): _sp_short_mult above only ever
    # SHRINKS the SP-ELO win-prob nudge toward zero for an opener/bulk-role/
    # bullpen-game start -- it has no way to independently penalize win
    # probability when that team's OWN bullpen (who now covers more of the
    # game) is actually bad. Real case: a 100-loss Rockies team fielding a
    # 3-5 IP bulk-role starter in front of MLB's worst bullpen still got
    # projected at ~66% to win -- the run-projection side already prices
    # this correctly (_eff_pit blends more bullpen RA into a bulk/opener
    # start, further above), but that blended read never fed back into win
    # probability at all. Reuses _bpen_lookup (10-game rolling bullpen RA,
    # already fetched for the run projection -- no new data) and the same
    # "how bad is bad" bar as Gate 10's bullpen-collapse-risk edge shave
    # further below (5.00), just checked here against the steadier 10-game
    # window instead of a noisy 3-day one, and applied to WHICHEVER side
    # actually has the short-leash exposure -- not only whichever side
    # someone happens to be picking, since this is about the underlying
    # probability being wrong, not just one candidate play's edge.
    _BULLPEN_EXPOSURE_ERA = 5.00      # same bar as Gate 10's bullpen-collapse-risk shave, further below
    _BULLPEN_EXPOSURE_PENALTY = 0.04  # win-prob points -- same small-nudge family as the record-credibility weight just below (max 4pp)
    _bp_exp_cols = ["away_sp_is_opener", "home_sp_is_opener", "away_sp_is_bulk", "home_sp_is_bulk",
                    "away_sp_is_bullpen_game", "home_sp_is_bullpen_game"]
    if _bpen_lookup and all(c in board.columns for c in _bp_exp_cols):
        _bp_exp_ct = 0
        for _bpi in board.index:
            _bp_aw_exposed = (bool(board.at[_bpi, "away_sp_is_opener"]) or bool(board.at[_bpi, "away_sp_is_bulk"])
                               or bool(board.at[_bpi, "away_sp_is_bullpen_game"]))
            _bp_hm_exposed = (bool(board.at[_bpi, "home_sp_is_opener"]) or bool(board.at[_bpi, "home_sp_is_bulk"])
                               or bool(board.at[_bpi, "home_sp_is_bullpen_game"]))
            if not _bp_aw_exposed and not _bp_hm_exposed:
                continue
            _bp_aw_team = board.at[_bpi, away_col] if away_col in board.columns else None
            _bp_hm_team = board.at[_bpi, home_col] if home_col in board.columns else None
            _bp_aw_ra = _bpen_lookup.get(_norm_team(str(_bp_aw_team))) if (_bp_aw_exposed and _bp_aw_team) else None
            _bp_hm_ra = _bpen_lookup.get(_norm_team(str(_bp_hm_team))) if (_bp_hm_exposed and _bp_hm_team) else None
            _bp_delta = 0.0
            if _bp_aw_ra is not None and _bp_aw_ra >= _BULLPEN_EXPOSURE_ERA:
                _bp_delta += _BULLPEN_EXPOSURE_PENALTY  # away's bad pen exposed -> nudge toward home
                print(f"  [BULLPEN EXPOSURE] {_bp_aw_team}: short-leash starter (opener/bulk/bullpen-game) in front of a "
                      f"{_bp_aw_ra:.2f} 10-game bullpen RA -- win prob nudged -{_BULLPEN_EXPOSURE_PENALTY:.0%}")
            if _bp_hm_ra is not None and _bp_hm_ra >= _BULLPEN_EXPOSURE_ERA:
                _bp_delta -= _BULLPEN_EXPOSURE_PENALTY  # home's bad pen exposed -> nudge toward away
                print(f"  [BULLPEN EXPOSURE] {_bp_hm_team}: short-leash starter (opener/bulk/bullpen-game) in front of a "
                      f"{_bp_hm_ra:.2f} 10-game bullpen RA -- win prob nudged -{_BULLPEN_EXPOSURE_PENALTY:.0%}")
            if _bp_delta != 0.0:
                board.at[_bpi, pred_home_col] = float(np.clip(board.at[_bpi, pred_home_col] + _bp_delta, 0.28, 0.72))
                _bp_exp_ct += 1
        if _bp_exp_ct:
            board[pred_away_col] = 1.0 - board[pred_home_col]
            print(f"  Bullpen exposure win-prob adjustment: {_bp_exp_ct} game(s) adjusted "
                  f"(short-leash starter + bullpen RA >= {_BULLPEN_EXPOSURE_ERA:.2f} over the last 10 games)")

    # ── Team-record credibility weight ────────────────────────────────────
    # Mechanical version of the informational "TEAM RECORD GAP" flag further
    # down this cell -- that one only prints a warning after the model prob
    # is already fixed. This applies a real, small suppression to the losing
    # side's probability when the season-record gap is large, mirroring the
    # same win_pct lookup (built separately here, earlier, since the full
    # _team_records fetch further down in this cell isn't available yet at
    # this point in execution). Deliberately mild (max 4pp) -- a 44-61 team
    # can still be the right side on a given day's SP matchup; this nudges
    # the model, it doesn't override it.
    _trw_records = {}
    _trw_road = {}
    # M-NEW-5 (2026-09-07, user audit): playoffPercent captured from this
    # SAME standings response (no second API call) -- ESPN's own computed
    # playoff odds, not a from-scratch games-back/games-remaining derivation,
    # since ESPN's number already accounts for remaining schedule strength
    # and tiebreakers.
    _trw_playoff_pct = {}
    # MLB-MOTIV-CLINCH-2 (2026-09-24 user audit): magic numbers + playoff
    # seed, same standings response, no new API call -- see _motgap_tier below.
    _trw_magic_div = {}
    _trw_magic_wc = {}
    _trw_seed = {}
    try:
        _trw_resp = _espn_get(
            "https://site.api.espn.com/apis/v2/sports/baseball/mlb/standings",
            timeout=8
        )
        if _trw_resp.status_code == 200:
            for _trw_league in _trw_resp.json().get("children", []):
                for _trw_entry in _trw_league.get("standings", {}).get("entries", []):
                    _trw_name = _trw_entry.get("team", {}).get("displayName", "").lower()
                    for _trw_stat in _trw_entry.get("stats", []):
                        if _trw_stat.get("name") == "winPercent":
                            _trw_records[_trw_name] = float(_trw_stat.get("value", 0.5) or 0.5)
                        elif _trw_stat.get("name") == "playoffPercent":
                            _pp_raw_val = _trw_stat.get("value")
                            # BUG FIX (2026-09-08, found live -- Mets/Giants both showed 50.0%
                            # playoff odds tonight when their real ESPN value is 0.0%): the old
                            # `x or 50.0` pattern treats a legitimate 0.0 as falsy and silently
                            # substitutes the default, same class of bug as MLB's own HR9/edge
                            # zero-vs-missing confusions elsewhere in this notebook -- only fall
                            # back to 50.0 when the key is truly ABSENT, not when it is present
                            # and genuinely zero.
                            _trw_playoff_pct[_trw_name] = float(_pp_raw_val) if _pp_raw_val is not None else 50.0
                        elif _trw_stat.get("name") == "magicNumberDivision":
                            _mnd_raw = _trw_stat.get("value")
                            if _mnd_raw is not None:
                                _trw_magic_div[_trw_name] = float(_mnd_raw)
                        elif _trw_stat.get("name") == "magicNumberWildcard":
                            _mnw_raw = _trw_stat.get("value")
                            if _mnw_raw is not None:
                                _trw_magic_wc[_trw_name] = float(_mnw_raw)
                        elif _trw_stat.get("name") == "playoffSeed":
                            _sd_raw = _trw_stat.get("value")
                            if _sd_raw is not None:
                                _trw_seed[_trw_name] = float(_sd_raw)
                        elif _trw_stat.get("name") == "Road":
                            # Road-record ML edge discount (audit fix, 2026-08-04): reuses
                            # this same standings response (no second API call) -- a team's
                            # ROAD record is a different signal than its overall win% (real
                            # case: Detroit 24-31 on the road this season while comfortably
                            # above .500 overall) and nothing here ever read it before. Only
                            # stored at 10+ road decisions to avoid small-sample noise.
                            try:
                                _rw, _rl = str(_trw_stat.get("displayValue", "")).split("-")
                                _rw, _rl = int(_rw), int(_rl)
                                if _rw + _rl >= 10:
                                    _trw_road[_trw_name] = _rw / (_rw + _rl)
                            except Exception:
                                pass
        print(f"  [TEAM RECORD WEIGHT] win% loaded for {len(_trw_records)} teams, road split for {len(_trw_road)} teams, playoff odds for {len(_trw_playoff_pct)} teams")
    except Exception as _trw_ex:
        print(f"  [TEAM RECORD WEIGHT] fetch failed ({_trw_ex}) -- credibility weight skipped this run")

    def _trw_lookup_generic(_d, _team_name):
        _key = str(_team_name).lower().strip()
        if _key in _d:
            return _d[_key]
        _words = _key.split()
        if not _words:
            return None
        _last = _words[-1]
        if len(_last) <= 3:
            return None
        for k, v in _d.items():
            _kw = k.split()
            if _kw and _kw[-1] == _last:
                return v
        return None

    # M-NEW-5 (2026-09-07, user audit -- "Minnesota fighting for the last AL
    # wild card, Detroit eliminated, model backs Detroit -- this should have
    # auto-flagged and never appeared anywhere on the card"): no team-level
    # playoff-motivation signal existed anywhere in this MLB notebook --
    # SOCCER has had exactly this kind of thing for weeks (_soc_motivation)
    # but MLB never did. NOTE verified live before building: tonight's real
    # Twins/Tigers game does NOT actually show this gap -- Minnesota's real
    # playoff odds are 3.4% (17 games back), essentially as eliminated as
    # Detroit's 0.1%, not "fighting for survival" -- so this fires on
    # whatever games genuinely show the gap, tonight's included only if the
    # numbers line up. mlb_motivation_gap_flags is a global dict (same
    # pattern as mlb_lineup_flags) so Cell 6e66db20's WHY loop can print it
    # on the actual card, not just this cell's internal build log.
    mlb_motivation_gap_flags = {}
    mlb_motivation_gap_tiers = {}  # team_norm -> (tier_name, delta_runs), MLB-MOTIV-PARITY-1
    # TIERED PLAYOFF-STAKES MOTIVATION SIGNAL (2026-09-11, user request): the
    # old version only ever fired on a stark eliminated-vs-contender MISMATCH
    # between the two teams in a game, and even then applied one flat -0.15R
    # penalty regardless of how thoroughly eliminated a team was. Real gap
    # the user flagged: two teams can both be "in the race" (TB 87-59
    # leading the AL East vs HOU 75-72 leading the AL West) but carry very
    # different urgency -- the old binary check correctly did nothing there,
    # but had no way to say anything about the asymmetry either. Replaced
    # with a per-TEAM stakes tier applied independently to that side's own
    # projected runs (not a target/opponent pairing) -- a team's own
    # motivation state should move ITS OWN offense, not just whichever side
    # happens to be facing a team that's still fighting. One real behavior
    # change from the old version: two mutually-eliminated teams playing
    # each other now both take the ELIMINATED penalty (old code required an
    # in-race opponent to fire at all) -- real rest-day/call-up behavior
    # tracks a team's own stakes, not its opponent's.
    _MOTGAP_ELIMINATED_MAX = 5.0    # playoff% at/below this = ELIMINATED
    _MOTGAP_FRINGE_MAX     = 30.0   # playoff% at/below this (above ELIMINATED) = FRINGE
    _MOTGAP_IN_RACE_MAX    = 80.0   # playoff% at/below this (above FRINGE) = IN RACE, neutral
    _MOTGAP_CLINCH_MAX     = 99.0   # playoff% at/below this (above IN RACE) = CLINCHING PRESSURE
    _MOTGAP_ELIMINATED_PENALTY = 0.15  # runs -- same magnitude as the old flat penalty
    _MOTGAP_FRINGE_PENALTY     = 0.05  # runs
    _MOTGAP_CLINCH_BONUS       = 0.05  # runs -- CLINCHING PRESSURE (80-99%) only
    # BUG FIX (MLB-MOTIV-CLINCH-1, 2026-09-24 user audit): >99% playoff odds
    # (a team that's effectively clinched -- locked into a playoff spot with
    # nothing left to play for) fell through to a bare 0.0/None no-op, same
    # as a genuinely neutral IN RACE team. Real incident: Atlanta had clinched
    # and got no motivation tier at all, even though a clinched team with
    # nothing at stake -- rested regulars, reduced bullpen urgency -- is a real
    # signal much closer to ELIMINATED than to IN RACE. Magnitude mirrors
    # FRINGE's 0.05R rather than ELIMINATED's 0.15R: a clinched team still has
    # real reasons to play hard (seeding, streaks, individual stats) that a
    # truly eliminated team doesn't, so this is deliberately the smaller
    # penalty -- unvalidated/carried-forward like every other tier here, not
    # backtested, same honesty caveat as the rest of this signal.
    _MOTGAP_CLINCHED_PENALTY   = 0.05  # runs -- CLINCHED/LOCKED IN (>99%), nothing at stake

    # BUG FIX (MLB-MOTIV-CLINCH-2, 2026-09-24 user audit): this used to treat
    # a raw playoffPercent > 99% as "clinched" -- ESPN's playoffPercent is a
    # probabilistic ESTIMATE, not an actual clinch determination. Real
    # incidents, all confirmed live against this same standings response:
    # Chicago White Sox sat at 99.4% playoff odds with magicNumberWildcard=8
    # (genuinely NOT clinched, 8 wins/opponent-losses still needed); Cleveland
    # showed playoffPercent ">99.9%" with a real magic number still positive
    # too -- both got mislabeled CLINCHED/nothing-at-stake while still
    # mathematically alive for what they were chasing. Milwaukee is the
    # opposite miss: magicNumberDivision=-7 (genuinely, mathematically
    # clinched) but sitting at playoffSeed=1 -- the #1 seed race (real
    # stakes: bye/home-field) isn't decided just because the division is.
    # Real clinch status now comes from ESPN's own magicNumberDivision/
    # magicNumberWildcard (<=0 means mathematically locked, the standard
    # meaning of a magic number reaching zero or below) instead of a
    # probability threshold; a team that has clinched but currently holds
    # the #1 seed is treated as still having real stakes (that seed race is
    # its own live competition), not "nothing at stake."
    _MOTGAP_SEED_RACE_MAX = 1  # playoffSeed <= this still has real (seeding) stakes even once clinched
    def _motgap_tier(pct, magic_div=None, magic_wc=None, seed=None):
        if pct <= _MOTGAP_ELIMINATED_MAX:
            return -_MOTGAP_ELIMINATED_PENALTY, "ELIMINATED"
        # BUG FIX (MLB-MOTIV-CLINCH-3, 2026-09-25 user audit): the OR here
        # meant clinching EITHER path flipped a team to "clinched" even
        # when the OTHER path was still a live, meaningful race. Real
        # incident: "CLINCHED -- NOTHING AT STAKE" fired on a Yankees team
        # 4 games back in its own division with 4 games left -- they had
        # already clinched A playoff spot via the wildcard (magic_wc<=0),
        # but the division race (magic_div still ~4, very much alive --
        # real stakes: division winners skip the wildcard round, better
        # seeding/home-field) was nowhere near decided. Winning the
        # division is always a strictly better-or-equal outcome than a
        # wildcard berth, so it's the race that actually determines
        # whether a team has anything left to play for -- checked first
        # whenever it's available, wildcard only used as a fallback when
        # division data itself is missing (preserves the original fix's
        # Milwaukee case: division magic<=0 with no separate wildcard
        # magic number populated once the division itself is clinched).
        if magic_div is not None:
            _is_clinched = magic_div <= 0
        elif magic_wc is not None:
            _is_clinched = magic_wc <= 0
        else:
            _is_clinched = False
        if not _is_clinched:
            # not mathematically locked in yet, regardless of how high playoffPercent reads
            if pct <= _MOTGAP_FRINGE_MAX:
                return -_MOTGAP_FRINGE_PENALTY, "FRINGE"
            if pct <= _MOTGAP_IN_RACE_MAX:
                return 0.0, None
            return _MOTGAP_CLINCH_BONUS, "CLINCHING PRESSURE"
        # genuinely clinched (magic number <= 0) -- still check for a live seed race
        if seed is not None and seed <= _MOTGAP_SEED_RACE_MAX:
            # BUG FIX (MLB-MAGIC-1, 2026-09-24 user audit): "CLINCHING
            # PRESSURE" (the tier just above, for a team still pushing to
            # clinch AT ALL) and this tier read as the same label with a
            # suffix tacked on, when they mean opposite things -- this team
            # HAS already clinched (magic number <= 0); "CLINCHING" implies
            # she hasn't yet. Real case: Milwaukee printed "division magic #
            # -7" (unambiguously already clinched) right next to a
            # "CLINCHING PRESSURE" tier name, reading as a contradiction.
            return _MOTGAP_CLINCH_BONUS, "CLINCHED -- SEEDING RACE"
        return -_MOTGAP_CLINCHED_PENALTY, "CLINCHED -- NOTHING AT STAKE"

    # MLB-POSTSEASON-1 (2026-09-29, user audit -- "set motivation to maximum on both
    # sides. Turn off the clinched and eliminated tiers entirely. Add an ELIMINATION GAME
    # flag... add series state (e.g. 1-0) as context on the card"). The ELIMINATED/FRINGE/
    # CLINCHED tier system right above is entirely a REGULAR-SEASON-race concept
    # (playoffPercent, magic numbers) -- none of it means anything once a team is actually
    # IN the postseason (every postseason team by definition still has something at
    # stake). Built from mlb_today_games' game_type/series fields (Cell 11, live-verified
    # 2026-09-29 against today's real Wild Card slate) so postseason teams skip the
    # regular-season tier math entirely and get a neutral MAX MOTIVATION note plus real
    # series-state/elimination-game context instead.
    mlb_postseason_teams_today = set()
    mlb_series_state_flags = {}
    _pssn_lookup = {}   # _pl_norm(team_name) -> mlb_today_games row (dict)
    if "mlb_today_games" in globals() and isinstance(mlb_today_games, pd.DataFrame) and "game_type" in mlb_today_games.columns:
        import math as _pssn_math
        for _, _pg in mlb_today_games.iterrows():
            _pg_type = _pg.get("game_type")
            if not _pg_type or _pg_type == "R":
                continue
            for _pg_side, _pg_team, _pg_opp, _pg_tid, _pg_oid in [
                ("home", _pg.get("home_team"), _pg.get("away_team"), _pg.get("home_team_id"), _pg.get("away_team_id")),
                ("away", _pg.get("away_team"), _pg.get("home_team"), _pg.get("away_team_id"), _pg.get("home_team_id")),
            ]:
                if not _pg_team:
                    continue
                _pg_key = _pl_norm(_pg_team)
                mlb_postseason_teams_today.add(_pg_key)
                _pssn_lookup[_pg_key] = {
                    "team": _pg_team, "opp": _pg_opp, "team_id": _pg_tid, "opp_id": _pg_oid,
                    "series_game_number": _pg.get("series_game_number"),
                    "games_in_series": _pg.get("games_in_series"),
                    "series_description": _pg.get("series_description"),
                }

    def _mlb_series_record(team_id, opp_id, game_date_str, series_game_number):
        """Real series win/loss tally for `team_id` vs `opp_id` heading into today's game
        -- (team_wins, opp_wins) from actual completed games earlier in this same series.
        Trivially (0, 0) on Game 1 (no API call needed). Best-effort: any fetch failure
        returns (None, None) rather than guessing."""
        if series_game_number is not None and int(series_game_number) <= 1:
            return 0, 0
        try:
            # own import, not borrowed from an earlier cell -- see this cell's own
            # "Notebook Cell Import Isolation" note at the top (math/datetime bugs
            # already bit this exact cell once before).
            import datetime as _pssn_dt
            _sr_end = _pssn_dt.date.today() - _pssn_dt.timedelta(days=1)
            _sr_start = _pssn_dt.date.today() - _pssn_dt.timedelta(days=14)
            _sr_url = (f"https://statsapi.mlb.com/api/v1/schedule?sportId=1&teamId={int(team_id)}"
                       f"&startDate={_sr_start}&endDate={_sr_end}&gameType=F,D,L,W")
            _sr_r = requests.get(_sr_url, timeout=15)
            _sr_r.raise_for_status()
            _sr_data = _sr_r.json()
            _team_wins, _opp_wins = 0, 0
            for _sr_d in _sr_data.get("dates", []):
                for _sr_g in _sr_d.get("games", []):
                    _sr_status = _sr_g.get("status", {}).get("detailedState")
                    if _sr_status != "Final":
                        continue
                    _sr_home = _sr_g.get("teams", {}).get("home", {})
                    _sr_away = _sr_g.get("teams", {}).get("away", {})
                    _sr_home_id = _sr_home.get("team", {}).get("id")
                    _sr_away_id = _sr_away.get("team", {}).get("id")
                    if {_sr_home_id, _sr_away_id} != {int(team_id), int(opp_id)}:
                        continue  # a different series/opponent within the lookback window
                    _sr_winner_id = _sr_home_id if _sr_home.get("isWinner") else (_sr_away_id if _sr_away.get("isWinner") else None)
                    if _sr_winner_id == int(team_id):
                        _team_wins += 1
                    elif _sr_winner_id == int(opp_id):
                        _opp_wins += 1
            return _team_wins, _opp_wins
        except Exception as _sr_e:
            print(f"  [MLB-POSTSEASON-1] series-record fetch failed for team_id={team_id}: {_sr_e}")
            return None, None

    if mlb_postseason_teams_today:
        print(f"  [POSTSEASON] {len(mlb_postseason_teams_today)} team(s) in a real postseason game today -- "
              f"regular-season motivation tiers (ELIMINATED/FRINGE/CLINCHED) do not apply, MAX MOTIVATION assumed")
        for _pk, _pinfo in _pssn_lookup.items():
            _pteam_wins, _popp_wins = _mlb_series_record(
                _pinfo["team_id"], _pinfo["opp_id"], None, _pinfo["series_game_number"])
            _games_in_series = _pinfo.get("games_in_series")
            _elim_note = ""
            if _pteam_wins is not None and _games_in_series:
                _wins_needed = _pssn_math.ceil(int(_games_in_series) / 2)
                _facing_elimination = _popp_wins == _wins_needed - 1
                _series_str = f"{_pteam_wins}-{_popp_wins}"
                if _facing_elimination:
                    _elim_note = f" -- !! ELIMINATION GAME (series {_series_str}, needs a win to stay alive)"
                else:
                    _elim_note = f" -- series {_series_str}"
            globals().setdefault("mlb_postseason_series_record", {})[_pk] = {
                "team": _pinfo["team"], "wins": _pteam_wins, "opp_wins": _popp_wins,
                "games_in_series": _games_in_series, "game_number": _pinfo.get("series_game_number"),
                "elimination": bool(_elim_note and "ELIMINATION" in _elim_note)}
            mlb_series_state_flags[_pk] = (
                f"[POSTSEASON: MAX MOTIVATION] {_pinfo['team']} vs {_pinfo['opp']} "
                f"({_pinfo.get('series_description') or 'postseason'}, "
                f"game {_pinfo.get('series_game_number')} of {_games_in_series}){_elim_note}"
            )
            print(f"  {mlb_series_state_flags[_pk]}")
    globals()["mlb_postseason_teams_today"] = mlb_postseason_teams_today
    globals()["mlb_series_state_flags"] = mlb_series_state_flags

    if _trw_playoff_pct:
        _motgap_ct = 0
        for _mgi in board.index:
            _mg_aw = board.at[_mgi, away_col] if away_col in board.columns else None
            _mg_hm = board.at[_mgi, home_col] if home_col in board.columns else None
            for _mg_team, _mg_col in ((_mg_aw, "pred_away_runs"), (_mg_hm, "pred_home_runs")):
                if not _mg_team:
                    continue
                if _pl_norm(_mg_team) in mlb_postseason_teams_today:
                    continue  # MLB-POSTSEASON-1: regular-season stakes tiers don't apply, see above
                _mg_pct = _trw_lookup_generic(_trw_playoff_pct, _mg_team)
                if _mg_pct is None:
                    continue
                _mg_magic_div = _trw_lookup_generic(_trw_magic_div, _mg_team)
                _mg_magic_wc = _trw_lookup_generic(_trw_magic_wc, _mg_team)
                _mg_seed = _trw_lookup_generic(_trw_seed, _mg_team)
                _mg_delta, _mg_tier = _motgap_tier(_mg_pct, _mg_magic_div, _mg_magic_wc, _mg_seed)
                if _mg_tier is None or pd.isna(board.at[_mgi, _mg_col]):
                    continue
                board.at[_mgi, _mg_col] = round(float(board.at[_mgi, _mg_col]) + _mg_delta, 2)
                _motgap_ct += 1
                _mg_sign = "+" if _mg_delta >= 0 else "-"
                _mg_magic_str = ""
                if _mg_magic_div is not None and (_mg_magic_wc is None or _mg_magic_div <= _mg_magic_wc):
                    _mg_magic_str = f", division magic # {_mg_magic_div:.0f}"
                elif _mg_magic_wc is not None:
                    _mg_magic_str = f", wildcard magic # {_mg_magic_wc:.0f}"
                _mg_seed_str = f", seed {_mg_seed:.0f}" if _mg_seed is not None else ""
                mlb_motivation_gap_flags[_pl_norm(_mg_team)] = (
                    f"[MOTIVATION TIER: {_mg_tier}] {_mg_team} ({_mg_pct:.1f}% playoff odds{_mg_magic_str}{_mg_seed_str}) -- "
                    f"projected runs {_mg_sign}{abs(_mg_delta):.2f}R"
                )
                print(f"  {mlb_motivation_gap_flags[_pl_norm(_mg_team)]}")
                mlb_motivation_gap_tiers[_pl_norm(_mg_team)] = (_mg_tier, _mg_delta)
        if _motgap_ct:
            board["pred_total_runs"] = (board["pred_away_runs"].fillna(0) + board["pred_home_runs"].fillna(0)).round(2)
            print(f"  Motivation stakes tiers: {_motgap_ct} team(s) adjusted")
    globals()["mlb_motivation_gap_flags"] = mlb_motivation_gap_flags
    globals()["mlb_motivation_gap_tiers"] = mlb_motivation_gap_tiers

    # NEW (MLB-MOTIV-RACE-1, 2026-09-24 user audit): every tier above is
    # driven purely by ESPN's playoffPercent -- a team can sit in the
    # neutral IN RACE band (30-80% playoff odds) while still being in a
    # genuinely live, down-to-the-wire DIVISION race that playoff% alone
    # doesn't single out. Real case: Cleveland, one game up in the AL
    # Central, got no flag anywhere -- a real division title (plus the
    # bye/home-field that comes with it) is on the line, at least as much
    # urgency as the CLINCHING PRESSURE tier above already credits.
    # Computed off the SAME standings fetch above (_trw_resp, no second API
    # call) -- ESPN's MLB standings groups entries by division (children ==
    # the 6 MLB divisions), so the top-2 teams in each group and their real
    # win-loss gap can be read directly, rather than trusting a
    # "gamesBehind" stat whose exact reference point isn't verified here.
    # Deliberately kept as its own separate signal, not folded into the
    # playoff%-tier dict above -- a division leader can be BOTH in
    # CLINCHING PRESSURE (by playoff%) AND in a live division race (by
    # standings gap) at the same time, and stacking a second small bonus is
    # more honest than overwriting one real signal with another.
    # Unbacktested, same caveat as every tier above.
    mlb_division_race_flags = {}
    _DIVISION_RACE_MAX_GAP = 2.0    # games -- "live race" bar for the top 2 teams in a division
    _DIVISION_RACE_BONUS = 0.05     # runs -- same magnitude as CLINCHING PRESSURE above
    # BUG FIX (MLB-MOTIV-RACE-1 v2, 2026-09-24, live-run audit): the first
    # version of this check assumed _trw_resp's "children" were the 6 real
    # MLB divisions -- confirmed WRONG on the first live nbconvert run, which
    # flagged the Dodgers (NL West) and Brewers (NL Central) as if they were
    # racing each other. Verified live: this endpoint's "children" are only
    # the 2 LEAGUES (American League / National League, 15 teams each), not
    # divisions -- no division field exists anywhere on the entry/team
    # object either. Real MLB divisions are a fixed, unchanging set of 30
    # team assignments, so a static map (keyed by nickname, matched via
    # endswith -- multi-word nicknames like "white sox"/"red sox" so two
    # teams ending the same way can't collide) is safer here than trusting
    # any further assumption about this endpoint's grouping.
    _MLB_DIVISION_NICKS = {
        "AL East":    ["yankees", "red sox", "blue jays", "orioles", "rays"],
        "AL Central": ["guardians", "twins", "white sox", "tigers", "royals"],
        "AL West":    ["astros", "mariners", "rangers", "angels", "athletics"],
        "NL East":    ["braves", "phillies", "mets", "marlins", "nationals"],
        "NL Central": ["brewers", "cubs", "cardinals", "reds", "pirates"],
        "NL West":    ["dodgers", "padres", "giants", "diamondbacks", "rockies"],
    }
    _MLB_NICK_TO_DIVISION = {_nick: _div for _div, _nicks in _MLB_DIVISION_NICKS.items() for _nick in _nicks}
    def _dr_division_for(display_name_lower):
        for _nick, _div in _MLB_NICK_TO_DIVISION.items():
            if display_name_lower.endswith(_nick):
                return _div
        return None
    try:
        _dr_resp = globals().get("_trw_resp")
        if _dr_resp is not None and getattr(_dr_resp, "status_code", None) == 200:
            _dr_ct = 0
            _dr_by_division = {}
            for _dr_league in _dr_resp.json().get("children", []):
                for _dr_entry in _dr_league.get("standings", {}).get("entries", []):
                    _dr_name = _dr_entry.get("team", {}).get("displayName", "").lower()
                    _dr_div = _dr_division_for(_dr_name) if _dr_name else None
                    if _dr_div is None:
                        continue
                    _dr_w = _dr_l = None
                    for _dr_stat in _dr_entry.get("stats", []):
                        if _dr_stat.get("name") == "wins":
                            _dr_w = float(_dr_stat.get("value", 0) or 0)
                        elif _dr_stat.get("name") == "losses":
                            _dr_l = float(_dr_stat.get("value", 0) or 0)
                    if _dr_w is not None and _dr_l is not None:
                        _dr_by_division.setdefault(_dr_div, []).append((_dr_name, _dr_w, _dr_l))
            for _dr_div, _dr_entries in _dr_by_division.items():
                if len(_dr_entries) < 2:
                    continue
                _dr_entries.sort(key=lambda t: (t[1] - t[2]), reverse=True)
                _dr_first, _dr_second = _dr_entries[0], _dr_entries[1]
                _dr_gap = ((_dr_first[1] - _dr_second[1]) + (_dr_second[2] - _dr_first[2])) / 2.0
                if _dr_gap <= _DIVISION_RACE_MAX_GAP:
                    mlb_division_race_flags[_dr_first[0]] = round(_dr_gap, 1)
                    mlb_division_race_flags[_dr_second[0]] = round(_dr_gap, 1)
                    _dr_ct += 2
                    print(f"  [DIVISION RACE] {_dr_div}: {_dr_first[0].title()} leads {_dr_second[0].title()} by {_dr_gap:.1f} game(s)")
            if _dr_ct:
                print(f"  [DIVISION RACE] {_dr_ct} team(s) in a live division race (top-2 gap <= {_DIVISION_RACE_MAX_GAP:.1f} games)")
    except Exception as _dr_ex:
        print(f"  [DIVISION RACE] check skipped ({_dr_ex})")
    globals()["mlb_division_race_flags"] = mlb_division_race_flags

    if mlb_division_race_flags:
        _dr_apply_ct = 0
        for _dri in board.index:
            _dr_aw = board.at[_dri, away_col] if away_col in board.columns else None
            _dr_hm = board.at[_dri, home_col] if home_col in board.columns else None
            for _dr_team, _dr_col in ((_dr_aw, "pred_away_runs"), (_dr_hm, "pred_home_runs")):
                if not _dr_team:
                    continue
                _dr_gap_val = _trw_lookup_generic(mlb_division_race_flags, _dr_team)
                if _dr_gap_val is None or pd.isna(board.at[_dri, _dr_col]):
                    continue
                board.at[_dri, _dr_col] = round(float(board.at[_dri, _dr_col]) + _DIVISION_RACE_BONUS, 2)
                _dr_apply_ct += 1
                print(f"  [MOTIVATION TIER: LIVE DIVISION RACE] {_dr_team} ({_dr_gap_val:.1f} games separating the top 2) "
                      f"-- projected runs +{_DIVISION_RACE_BONUS:.2f}R")
        if _dr_apply_ct:
            board["pred_total_runs"] = (board["pred_away_runs"].fillna(0) + board["pred_home_runs"].fillna(0)).round(2)
            print(f"  Division race bonus: {_dr_apply_ct} team(s) adjusted")

    if _trw_records:
        _TRW_GAP_MIN  = 0.15   # win% points
        _TRW_SUPPRESS = 0.04   # max 4pp nudge toward the higher-record team
        def _trw_lookup(team_name):
            return _trw_lookup_generic(_trw_records, team_name)
        _trw_adj_ct = 0
        for _trwi in board.index:
            _trw_aw = board.at[_trwi, away_col] if away_col in board.columns else None
            _trw_hm = board.at[_trwi, home_col] if home_col in board.columns else None
            _trw_aw_pct = _trw_lookup(_trw_aw) if _trw_aw else None
            _trw_hm_pct = _trw_lookup(_trw_hm) if _trw_hm else None
            if _trw_aw_pct is None or _trw_hm_pct is None:
                continue
            _trw_gap = _trw_hm_pct - _trw_aw_pct  # positive = home team's record is better
            if abs(_trw_gap) < _TRW_GAP_MIN:
                continue
            # Scale linearly between the 15pp floor and a 35pp gap (roughly a 100-
            # win-team-vs-60-win-team extreme), capped at _TRW_SUPPRESS either way.
            _trw_nudge = np.clip((abs(_trw_gap) - _TRW_GAP_MIN) / 0.20, 0, 1) * _TRW_SUPPRESS
            _trw_nudge = _trw_nudge if _trw_gap > 0 else -_trw_nudge  # sign toward home if home's record is better
            if pd.notna(board.at[_trwi, pred_home_col]):
                board.at[_trwi, pred_home_col] = float(np.clip(
                    board.at[_trwi, pred_home_col] + _trw_nudge, 0.28, 0.72))
                _trw_adj_ct += 1
        if _trw_adj_ct:
            board[pred_away_col] = 1.0 - board[pred_home_col]
            print(f"  Team-record credibility weight: {_trw_adj_ct} game(s) nudged "
                  f"(>={_TRW_GAP_MIN:.0%} season win% gap, max {_TRW_SUPPRESS:.0%})")

    # ── Road-record ML edge discount (audit fix, 2026-08-04) ───────────────
    # Narrows -- never disqualifies outright, this is a supplemental signal,
    # not the primary one -- the AWAY team's win probability when their road
    # record specifically is meaningfully worse than a coinflip, even if
    # their overall record (nudged above) looks fine.
    if _trw_road and away_col in board.columns:
        _ROAD_PCT_WEAK = 0.450   # below this, the away team's road record is a real drag
        _ROAD_SUPPRESS = 0.03    # max 3pp nudge -- narrower than the overall-record weight (4pp)
        def _road_lookup(team_name):
            return _trw_lookup_generic(_trw_road, team_name)
        _road_adj_ct = 0
        for _rri in board.index:
            _rr_aw = board.at[_rri, away_col] if away_col in board.columns else None
            _rr_pct = _road_lookup(_rr_aw) if _rr_aw else None
            if _rr_pct is None or _rr_pct >= _ROAD_PCT_WEAK:
                continue
            _rr_nudge = np.clip((_ROAD_PCT_WEAK - _rr_pct) / 0.15, 0, 1) * _ROAD_SUPPRESS
            if pd.notna(board.at[_rri, pred_home_col]):
                board.at[_rri, pred_home_col] = float(np.clip(
                    board.at[_rri, pred_home_col] + _rr_nudge, 0.28, 0.72))
                _road_adj_ct += 1
        if _road_adj_ct:
            board[pred_away_col] = 1.0 - board[pred_home_col]
            print(f"  Road-record ML edge discount: {_road_adj_ct} game(s) narrowed "
                  f"(away team road win% < {_ROAD_PCT_WEAK:.0%}, max {_ROAD_SUPPRESS:.0%} nudge toward home)")

    # FIX 3 (2026-09-06, user audit -- Nationals/Dodgers -0.90R injury delta
    # that never touched win probability at all, only informational display).
    # M27 (2026-09-05) deliberately declined a hard downgrade/gate on this
    # signal for weak evidence (n=3, one real counterexample -- Pirates ML
    # was -0.80R against the pick on injury delta and still won). This is
    # intentionally NOT that: a small, capped nudge to pred_home_win_prob,
    # same mechanism and same order of magnitude as the team-record-
    # credibility and road-record nudges just above (max 4pp) -- moves in
    # the right direction instead of doing nothing, without trusting n=3
    # enough to swing win probability by a full linear R-to-win% conversion
    # (which would imply something closer to 6-8pp on tonight's real game).
    if 'mlb_lineup_flags' in globals() and mlb_lineup_flags:
        _INJ_NUDGE_PER_RUN = 0.045   # pp of win prob per 1.0R net injury delta, before the cap
        _INJ_NUDGE_CAP = 0.04        # same 4pp ceiling as the team-record credibility weight above
        _inj_nudge_ct = 0
        for _ini in board.index:
            _ini_aw = board.at[_ini, away_col] if away_col in board.columns else None
            _ini_hm = board.at[_ini, home_col] if home_col in board.columns else None
            if not _ini_aw or not _ini_hm:
                continue
            _ini_aw_pen = float((mlb_lineup_flags.get(_pl_norm(str(_ini_aw))) or {}).get('penalty', 0) or 0)
            _ini_hm_pen = float((mlb_lineup_flags.get(_pl_norm(str(_ini_hm))) or {}).get('penalty', 0) or 0)
            if not _ini_aw_pen and not _ini_hm_pen:
                continue
            _ini_net = _ini_aw_pen - _ini_hm_pen   # positive = away hurts more -> favors home
            if _ini_net == 0:
                continue
            _ini_nudge = float(np.clip(_ini_net * _INJ_NUDGE_PER_RUN, -_INJ_NUDGE_CAP, _INJ_NUDGE_CAP))
            if pd.notna(board.at[_ini, pred_home_col]):
                board.at[_ini, pred_home_col] = float(np.clip(
                    board.at[_ini, pred_home_col] + _ini_nudge, 0.28, 0.72))
                _inj_nudge_ct += 1
        if _inj_nudge_ct:
            board[pred_away_col] = 1.0 - board[pred_home_col]
            print(f"  Injury-delta ML nudge: {_inj_nudge_ct} game(s) nudged toward the healthier side (max {_INJ_NUDGE_CAP:.0%})")

    # SP-quality-to-ML transparency flag -- added 2026-07-26. Investigated whether
    # the ML win model discounts enough for a big starting-pitcher quality gap
    # (the Cease/Skenes-type postmortem question). Found: it's NOT a total gap --
    # `_pit_adj` above already nudges pred_home_col by the SP ELO differential --
    # but the underlying 7-feature win classifier (mlb_win_model_compressed_best.
    # joblib) was trained on mlb_model_table_v1.parquet, which has ZERO pitcher
    # columns at all, so `_pit_adj` (capped +-5pp, 0.025 per 100 ELO) is the ONLY
    # SP-quality signal the ML side ever sees, and it's a hand-picked coefficient.
    #
    # EMPIRICALLY BACKTESTED 2026-07-27: replayed the exact chronological SP-ELO
    # computation from Cell 83 against mlb_elo_cache_2025.json (~4000 real games,
    # 2025-03-27 to present), recording each game's PRE-GAME SP ELO gap (no
    # look-ahead) against the real final-score outcome. Result: the current
    # coefficient is broadly reasonable, NOT systematically underweighted as
    # suspected -- actual SP-favored-team win rate by |gap| bucket: 50-100=+2.1pp,
    # 100-150=-0.3pp, 150-200=+2.8pp, 200-300=+2.1pp, 300+=+5.6pp (vs coinflip).
    # The current formula implies +1.9/+3.1/+4.4/+5.0/+5.0pp at those same
    # buckets -- i.e. the live formula is if anything slightly GENEROUS in the
    # 100-300 range relative to the real historical relationship, and only
    # possibly undershoots at the 300+ tail (+5.6 real vs +5.0 capped, well
    # within noise at n=520). Do not raise this coefficient based on intuition
    # about how large an ELO gap "should" matter -- the real effect size of pure
    # SP quality on game outcome is empirically much smaller than a naive read of
    # the ELO points would suggest (most of a game's variance is offense/bullpen/
    # in-game, not the two starters). Simple bucket comparison, not a full
    # regression controlling for team ELO -- a cleaner test would model gap
    # alongside team strength, but this is a real, data-backed check, not a
    # guess, and it does not support raising the coefficient.
    _SP_GAP_FLAG = 100  # ELO points -- a genuinely large starter-quality mismatch
    _sp_gap_ct = 0
    for _sgi in board.index:
        _sg = abs(float(_h_sp_elos.get(_sgi, 1500)) - float(_a_sp_elos.get(_sgi, 1500)))
        if _sg >= _SP_GAP_FLAG:
            _sg_adj_pp = float(_pit_adj.get(_sgi, 0.0)) * 100
            _sg_aw = board.at[_sgi, away_col] if away_col in board.columns else "?"
            _sg_hm = board.at[_sgi, home_col] if home_col in board.columns else "?"
            print(f"  [SP GAP] {_sg_aw} @ {_sg_hm}: {_sg:.0f}pt starter ELO gap -> only "
                  f"{_sg_adj_pp:+.1f}pp ML adjustment (uncalibrated coefficient, verify manually)")
            _sp_gap_ct += 1
    if _sp_gap_ct:
        print(f'  {_sp_gap_ct} game(s) with a 100+pt starter ELO gap -- see [SP GAP] above')
    _h_default = (_h_sp_elos == 1460.0)  # unknown-pitcher fallback (name not in ELO dict)
    _a_default = (_a_sp_elos == 1460.0)
    board['sp_elo_weak'] = (_h_default | _a_default)
    board['sp_unresolved'] = (~_h_sp_name_resolved) | (~_a_sp_name_resolved)
    board['home_sp_elo'] = _h_sp_elos
    board['away_sp_elo'] = _a_sp_elos
    board['home_sp_elo_adj'] = _h_sp_elos_adj
    board['away_sp_elo_adj'] = _a_sp_elos_adj
    # Post-hoc SP quality run suppressor — applies when team ELO blend was skipped.
    # Guards against L10 offense overpowering an elite SP matchup in projections.
    _need_sp_supp = ('mlb_team_off_elo' not in globals()
                     or not isinstance(mlb_team_off_elo, dict)
                     or not mlb_team_off_elo)
    if _need_sp_supp:
        _SP_SUPP_T1 = 1570; _SP_SUPP_T2 = 1620
        _SP_SUPP_F1 = 0.85; _SP_SUPP_F2 = 0.75  # 15% / 25% reduction
        _sp_supp_ct = 0
        for _ss_i in board.index:
            _h_elo_ss = float(_h_sp_elos_adj.get(_ss_i, 1500))
            _a_elo_ss = float(_a_sp_elos_adj.get(_ss_i, 1500))
            _ar_ss = board.at[_ss_i, 'pred_away_runs']
            _hr_ss = board.at[_ss_i, 'pred_home_runs']
            # Elite home SP -> suppress away team runs
            if pd.notna(_ar_ss) and _h_elo_ss >= _SP_SUPP_T1:
                _ff = _SP_SUPP_F2 if _h_elo_ss >= _SP_SUPP_T2 else _SP_SUPP_F1
                board.at[_ss_i, 'pred_away_runs'] = round(float(_ar_ss) * _ff, 1)
                _sp_supp_ct += 1
            # Elite away SP -> suppress home team runs
            if pd.notna(_hr_ss) and _a_elo_ss >= _SP_SUPP_T1:
                _ff = _SP_SUPP_F2 if _a_elo_ss >= _SP_SUPP_T2 else _SP_SUPP_F1
                board.at[_ss_i, 'pred_home_runs'] = round(float(_hr_ss) * _ff, 1)
                _sp_supp_ct += 1
        board['pred_total_runs'] = (
            board['pred_away_runs'].fillna(0) + board['pred_home_runs'].fillna(0)
        ).round(1)
        if _sp_supp_ct:
            print(f'  SP quality suppressor: {_sp_supp_ct} run projection(s) reduced for elite SP matchups')

    # Opposing-SP recent-form suppressor (2026-07-13) -- independent of the
    # elite-ELO suppressor above, which only fires when the team ELO blend
    # failed entirely (essentially never in normal operation, since ELO almost
    # always loads). ELO is a slow-moving season-long rating and can lag a
    # pitcher's actual recent form; this catches a starter who has been
    # dominant in his last 2 starts specifically, not just a high aggregate
    # rating. Runs unconditionally, every day. Milder reduction (10%) than the
    # elite-ELO suppressor (15/25%) since this is a narrower, less-tested
    # signal on a smaller sample.
    _OPP_FORM_ERA_MAX = 3.50
    _OPP_FORM_LAST2_ER_MAX = 1.0
    _OPP_FORM_SUPP_F = 0.90
    _opp_form_supp_ct = 0
    if 'mlb_pitcher_fip_lookup' in globals():
        for _of_i in board.index:
            for _of_pcol, _of_runcol in [("home_pitcher_name", "pred_away_runs"),
                                          ("away_pitcher_name", "pred_home_runs")]:
                _of_nm = board.at[_of_i, _of_pcol] if _of_pcol in board.columns else None
                if not _of_nm or str(_of_nm) in ('nan', 'None', ''):
                    continue
                _of_ev = mlb_pitcher_fip_lookup.get(_norm_name(str(_of_nm)), {})
                _of_era = _of_ev.get('era_l3')
                _of_l2er = _of_ev.get('last2_starts_er')
                _of_l2ip = _of_ev.get('last2_starts_ip')
                if _of_era is None or pd.isna(_of_era):
                    continue
                if _of_l2er is None or pd.isna(_of_l2er) or _of_l2ip is None or pd.isna(_of_l2ip) or _of_l2ip < 4.0:
                    continue  # need a real 2-start sample, not a token relief inning
                if float(_of_era) <= _OPP_FORM_ERA_MAX and float(_of_l2er) <= _OPP_FORM_LAST2_ER_MAX:
                    if pd.notna(board.at[_of_i, _of_runcol]):
                        board.at[_of_i, _of_runcol] = round(float(board.at[_of_i, _of_runcol]) * _OPP_FORM_SUPP_F, 1)
                        _opp_form_supp_ct += 1
        if _opp_form_supp_ct:
            board['pred_total_runs'] = (
                board['pred_away_runs'].fillna(0) + board['pred_home_runs'].fillna(0)
            ).round(1)
            print(f'  Opposing SP recent-form suppressor: {_opp_form_supp_ct} run projection(s) reduced '
                  f'(opp L3 ERA <= {_OPP_FORM_ERA_MAX:.2f} and <= {_OPP_FORM_LAST2_ER_MAX:.0f}R combined last 2 starts)')

    # SP collapse / cold-streak run booster (2026-07-29) -- mirrors the hot
    # suppressor above but for the opposite case, which had no equivalent.
    # Real gap: Cantillo's last 2 starts (9.35 ERA combined, 43% hard-hit)
    # fed last2_starts_er/ip (Cell 20, already fetched) but nothing ever read
    # them for a COLD signal -- only the hot suppressor consumed this field.
    # Boosts the opposing team's projected runs when a starter's last 2
    # outings were a real implosion, independent of season/L3 averages that
    # would dilute exactly this signal (same reasoning as the last-start
    # walk-implosion gate above).
    _COLD_LAST2_ERA_MIN = 8.00
    _COLD_BOOST_F = 1.10
    _cold_boost_ct = 0
    if 'mlb_pitcher_fip_lookup' in globals():
        for _cf_i in board.index:
            for _cf_pcol, _cf_runcol in [("home_pitcher_name", "pred_away_runs"),
                                          ("away_pitcher_name", "pred_home_runs")]:
                _cf_nm = board.at[_cf_i, _cf_pcol] if _cf_pcol in board.columns else None
                if not _cf_nm or str(_cf_nm) in ('nan', 'None', ''):
                    continue
                _cf_ev = mlb_pitcher_fip_lookup.get(_norm_name(str(_cf_nm)), {})
                _cf_l2er = _cf_ev.get('last2_starts_er')
                _cf_l2ip = _cf_ev.get('last2_starts_ip')
                _cf_gs = _cf_ev.get('gs_season')
                if _cf_l2er is None or pd.isna(_cf_l2er) or _cf_l2ip is None or pd.isna(_cf_l2ip) or _cf_l2ip < 4.0:
                    continue  # need a real 2-start sample, not a token relief inning
                # 2026-07-29: last2_starts_ip is starts[-2:] (cell 20) -- if a pitcher
                # has thrown only 1 start all season, that's the SAME single start
                # counted once, and a 4+ IP disaster outing could still clear the
                # ip>=4.0 floor above alone. Require a genuine 2-start sample so one
                # rough outing can't tank the projection on its own.
                if _cf_gs is None or pd.isna(_cf_gs) or _cf_gs < 2:
                    continue
                _cf_l2_era = 9.0 * float(_cf_l2er) / float(_cf_l2ip)
                if _cf_l2_era >= _COLD_LAST2_ERA_MIN and pd.notna(board.at[_cf_i, _cf_runcol]):
                    board.at[_cf_i, _cf_runcol] = round(float(board.at[_cf_i, _cf_runcol]) * _COLD_BOOST_F, 1)
                    _cold_boost_ct += 1
        if _cold_boost_ct:
            board['pred_total_runs'] = (
                board['pred_away_runs'].fillna(0) + board['pred_home_runs'].fillna(0)
            ).round(1)
            print(f'  SP collapse run booster: {_cold_boost_ct} run projection(s) increased '
                  f'(last 2 starts combined ERA >= {_COLD_LAST2_ERA_MIN:.2f})')

    _nonzero = int((_pit_adj.abs() > 0.001).sum())
    _weak_count = int(board['sp_elo_weak'].sum())
    print(f'Pitcher ELO adj applied: {_nonzero}/{len(board)} games adjusted')
    if _weak_count > 0:
        print(f"  !! {_weak_count} game(s) have DEFAULT ELO (1460 fallback) ? SP name not in ELO dict:")
        for _gi in board.index[board["sp_elo_weak"]]:
            _aw_d = board.at[_gi, away_col] if away_col in board.columns else "?"
            _hw_d = board.at[_gi, home_col] if home_col in board.columns else "?"
            _asp_d = float(board.at[_gi, "away_sp_elo"] or 1500)
            _hsp_d = float(board.at[_gi, "home_sp_elo"] or 1500)
            print(f"    {_aw_d} @ {_hw_d}: away_sp_elo={_asp_d:.0f}  home_sp_elo={_hsp_d:.0f}")
    _unresolved_count = int(board['sp_unresolved'].sum())
    if _unresolved_count > 0:
        print(f"  !! {_unresolved_count} game(s) have NO starter name resolved for one side -- PENDING LINEUP:")
        for _ui in board.index[board["sp_unresolved"]]:
            _uaw = board.at[_ui, away_col] if away_col in board.columns else "?"
            _uhw = board.at[_ui, home_col] if home_col in board.columns else "?"
            print(f"    {_uaw} @ {_uhw}: starter unresolved for pick/opponent -- edge filter will hard-block this play")

if date_col is not None:
    board[date_col] = pd.to_datetime(board[date_col], errors="coerce")
else:
    board["date"] = pd.NaT
    date_col = "date"

# ---------------------------------------------------------
# 7) build two candidate rows per game (home + away)
# ---------------------------------------------------------
def _build_cands(brd, side, pred_col, mkt_col, price_col, e_col):
    c = brd.copy()
    c["bet_team"]         = c[home_col if side == "home" else away_col]
    c["bet_side"]         = side
    c["p_bet_win_model"]  = safe_num(c[pred_col])
    c["game_has_odds"]    = c["has_odds"].astype(bool) if "has_odds" in c.columns else False

    if mkt_col in c.columns and c[mkt_col].notna().any():
        c["market_implied_prob"] = np.where(c["game_has_odds"], safe_num(c[mkt_col]), np.nan)
    elif price_col is not None:
        c["market_implied_prob"] = np.where(c["game_has_odds"], safe_num(c[price_col]), np.nan)
    elif e_col is not None:
        c["market_implied_prob"] = safe_num(c[pred_col]) - safe_num(c[e_col])
        c["game_has_odds"] = True
    else:
        c["market_implied_prob"] = np.nan
    return c

home_cand = _build_cands(board, "home", pred_home_col, "home_market_prob", home_price_col, edge_home_col)
away_cand = _build_cands(board, "away", pred_away_col, "away_market_prob", away_price_col, edge_away_col)
cand = pd.concat([home_cand, away_cand], ignore_index=True)

# ---------------------------------------------------------
# 8) edge calculation
# ---------------------------------------------------------
_mask = cand["p_bet_win_model"].between(0, 1, inclusive="both")
cand = cand[_mask].copy()
if len(cand) == 0:
    raise ValueError("No valid side candidates after probability cleanup.")

# No-vig: normalize market implied probs per game to remove bookmaker margin
_gk_cols = [c for c in [date_col, away_col, home_col] if c in cand.columns]
if _gk_cols:
    cand["_gk"] = cand[_gk_cols].astype(str).agg("-".join, axis=1)
    _sum_imp = cand.groupby("_gk")["market_implied_prob"].transform("sum")
    cand["market_novig_prob"] = np.where(
        _sum_imp.between(1.0, 1.20) & cand["market_implied_prob"].notna(),
        cand["market_implied_prob"] / _sum_imp,
        cand["market_implied_prob"],
    )
else:
    cand["market_novig_prob"] = cand["market_implied_prob"]

_mkt_for_edge = np.where(
    cand["game_has_odds"],
    cand["market_novig_prob"].fillna(0.50),
    0.50,
)

# BUG FIX (MLB-EDGE-BLEND-1, 2026-09-25 user audit): edges were running
# far too hot -- real cases: a 61-98 Angels team read 71.6% model win
# prob (outside models had the game near a coinflip), Nationals 73.9%
# similarly. A real MLB edge rarely exceeds 5-8pp; a 26-29% "edge" is a
# strong signal of an input/model error, not a real market mispricing.
# Backtested the blend weight w against this notebook's own real graded
# ML ledger (tracking/mlb_official_plays_graded.csv, n=56 graded picks
# with both model_prob and market_prob, 2026-07-27 to 2026-09-25) via a
# log-loss grid search: raw model log-loss 0.737 vs raw market log-loss
# 0.693 -- the market ALONE already beats this model's own probabilities
# on this sample. Log-loss falls monotonically as w decreases from 1.0
# and bottoms in a flat minimum around w=0.10-0.15 (0.6918) before
# rising again toward pure-market. w=0.15 sits inside that minimum --
# LOWER than a 0.35-0.5 starting point would be, because the real data
# says lean on the market even more heavily than that. Only blends when
# a real market price exists (game_has_odds) -- the 0.50 placeholder
# used for a no-odds game isn't a real market signal to blend toward.
# PROVISIONAL: n=56 is a real backtest, not a guess, but still a modest
# sample on an unusually flat log-loss surface -- refit as more picks
# grade under this blended methodology.
_MODEL_MARKET_BLEND_W = 0.15   # THRESHOLD SCALE only from 2026-10-04 (~40 knockdowns/bars are expressed in it)
# SHADOW-1 (2026-10-05 user request -- "All models back to full force for shadow backtesting"):
# model_blend_weights.json "active" = "shadow" -> model-only prices (shadow_weights, w=1.0) and every
# bar/knockdown back on its original raw scale (BASE_EDGE_MIN 0.12 etc.) -- the user chose raw bars
# because these ~40 constants were rescaled x0.15 for the blend on 09-26. "live" restores both.
_MLB_SHADOW_MODE = False
try:
    import json as _sh_json
    _sh_full = _sh_json.load(open(r"C:\Users\<user>\OneDrive\Documents\KAI Labs\sports\model_blend_weights.json"))
    _MLB_SHADOW_MODE = _sh_full.get("active", "live") in ("shadow", "full")
except Exception as _she:
    print(f"[SHADOW] weights file unreadable ({_she}) -- live mode")
if _MLB_SHADOW_MODE:
    _MODEL_MARKET_BLEND_W = 1.0
    print("[FULL POWER] MLB: model-only probabilities, raw-scale bars/knockdowns (BASE_EDGE_MIN 12%) -- FULL POWER, live")
# STACKED WEIGHT (2026-10-04 user correction -- "fit each weight as a stacked regression on every
# game in the slate, not just the bets, out of sample; run MLB through the same diagnostic").
# scripts/stacked_weights.py on 4,980 full-slate games (2024-25: the win model's walk-forward
# out-of-sample home-win prob vs ESPN pregame closing moneylines): constrained log-odds weight
# (market held at 1) w = 0.10 (SE 0.083), out-of-sample log-loss gain ~0.0000 -- MARGINAL:
# the model adds almost nothing beyond the close. Caveat: that's the BASE walk-forward model,
# not the full live stack (Elo blend, SP/injury/bullpen layers) -- no point-in-time history of
# the full stack exists. Applied in log-odds space:  logit p = logit(mkt) + w*(logit(model) - logit(mkt))
_MLB_STACK_W = 0.15
try:
    import json as _sw_json
    _sw_full = _sw_json.load(open(r"C:\Users\<user>\OneDrive\Documents\KAI Labs\sports\model_blend_weights.json"))
    _sw = (_sw_full.get("live_weights") or {}).get("MLB") or {}
    if _MLB_SHADOW_MODE:
        _MLB_STACK_W = float((_sw_full.get("shadow_weights") or {}).get("MLB", 1.0))
        print(f"[FULL POWER WEIGHT] MLB: w={_MLB_STACK_W:.2f} (model only) | live weight {_sw.get('w')} kept for when betting resumes")
    else:
        _MLB_STACK_W = 0.0 if _sw.get("enabled", True) is False else float(_sw["w"])
        print(f"[LIVE WEIGHT] MLB: w={_MLB_STACK_W:.2f} (n={_sw.get('n')} full-slate games, OOS gain {_sw.get('oos_gain_w')}, set {_sw.get('set_on')})")
except Exception as _swe:
    print(f"[STACKED WEIGHT] MLB: weights file unreadable ({_swe}) -- fallback w={_MLB_STACK_W}")
def _mlb_logit_blend(pm, pk, w):
    pm = np.clip(np.asarray(pm, float), 1e-6, 1 - 1e-6); pk = np.clip(np.asarray(pk, float), 1e-6, 1 - 1e-6)
    z = np.log(pk / (1 - pk)) + w * (np.log(pm / (1 - pm)) - np.log(pk / (1 - pk)))
    return 1 / (1 + np.exp(-z))
cand["p_bet_win_model_raw"] = cand["p_bet_win_model"]
cand["p_bet_win_model"] = np.where(
    cand["game_has_odds"],
    _mlb_logit_blend(cand["p_bet_win_model"], pd.Series(_mkt_for_edge, index=cand.index), _MLB_STACK_W),
    cand["p_bet_win_model"],
)
cand["edge"] = cand["p_bet_win_model"] - _mkt_for_edge

cand["team_a"]    = cand[[away_col, home_col]].astype(str).min(axis=1)
cand["team_b"]    = cand[[away_col, home_col]].astype(str).max(axis=1)
cand["series_key"]= cand["team_a"] + " vs " + cand["team_b"]

sort_cols = [date_col, "series_key"]
if game_id_col and game_id_col in cand.columns:
    sort_cols.append(game_id_col)
cand = cand.sort_values(sort_cols).reset_index(drop=True)

# ---------------------------------------------------------
# 9) series-repeat penalty
# ---------------------------------------------------------
cand["prev_date_same_series_side"] = cand.groupby(["series_key", "bet_team"])[date_col].shift(1)
cand["days_since_prev"] = (
    pd.to_datetime(cand[date_col]) - pd.to_datetime(cand["prev_date_same_series_side"])
).dt.days
cand["is_repeat"]     = (cand["days_since_prev"].fillna(99) <= 3).astype(int)
cand["repeat_count"]  = cand.groupby(["series_key", "bet_team"]).cumcount()
cand["repeat_penalty"]= 0.0
cand.loc[cand["is_repeat"] == 1, "repeat_penalty"] = 0.020
cand.loc[cand["repeat_count"] >= 2, "repeat_penalty"] += 0.015

# ---------------------------------------------------------
# 10) final score + thresholds
#     Starter/bullpen bonus removed: those features are already
#     inside the model probability, so adding a bonus double-counts them.
# ---------------------------------------------------------
cand["final_score"] = cand["edge"] - cand["repeat_penalty"]

# BUG FIX (MLB-EDGE-BLEND-1, 2026-09-25 user audit): re-derived on the
# blended edge scale -- edge_blended = w*edge_raw whenever a real market
# price exists (see the blend above), so multiplying the OLD raw-edge
# floor by the same w=0.15 preserves this exact threshold's original
# selectivity under the new scale instead of guessing at a new number.
BASE_EDGE_MIN   = round(0.12 * _MODEL_MARKET_BLEND_W, 4)  # was 0.12 on the raw-edge scale
# from 0.13 previously for high-conviction-only; user now wants a 12% floor.
# A qualifying play in the 12-14.9% range sizes at 0.5u via _unit_label below
# (the same tier the Pitcher Advantage Exception's 12%-edge promotions
# already use), not the old 1.0u+ minimum implied by the 15% bar.
REPEAT_EDGE_MIN = BASE_EDGE_MIN  # matches base (now literally equal, not just nominally)
# Calibrated on 2025 out-of-sample holdout (n=2,614 games):
#   >= 58%: 62.1% actual accuracy, +18.6% ROI at -110 (214 plays/season)
#   >= 55%: 57.5% actual accuracy, profitable at -110 with 3%+ edge
MIN_MODEL_PROB  = 0.58  # raised from 0.57 (2026-07-09 punch list) -- eliminate coinflip plays dressed as edges

# Logit-space symmetric edge threshold -- ported 2026-07-12 from
# WORLDCUP2026MODEL.ipynb's 2026-07-11 fix (see [[wc-knockout-fixes]] /
# [[feedback_odds_scaled_edge_threshold]]). The step-bracket discount above
# approximated a fix for the same underlying problem a flat percentage-point
# edge threshold has: a favorite priced at 85% only has 15 points of room to
# diverge upward before hitting 100%, while the complementary underdog (15%)
# has 85 points of room -- so the identical real model/market disagreement
# produces a much bigger raw edge on the underdog side, structurally,
# regardless of whether the signal is actually stronger. Gating on
# logit-space edge (ln(p/(1-p)), unbounded, symmetric) fixes the asymmetry
# directly instead of discounting around it bracket by bracket.
# Verified 2026-07-12: this does NOT retroactively explain that night's two
# losing ML plays (Tigers, Blue Jays) as threshold artifacts -- both remain
# well above the logit-space bar (edge_logit ~0.80 and ~0.76 vs a ~0.62
# requirement), so those were real disagreements that simply didn't hit, not
# a percentage-point-scale illusion. This closes the same known gap for
# future/less-clear-cut cases, same as the WC fix did.
def _mlb_logit(p):
    p = min(max(p, 1e-6), 1 - 1e-6)
    return math.log(p / (1 - p))
cand["_base_edge_min"] = np.where(cand["is_repeat"]==1, REPEAT_EDGE_MIN, BASE_EDGE_MIN)
cand["_required_edge_logit"] = cand["_base_edge_min"].apply(lambda b: _mlb_logit(0.5 + b) - _mlb_logit(0.5))
cand["_edge_logit"] = cand.apply(
    lambda r: (_mlb_logit(r["p_bet_win_model"]) - _mlb_logit(r["market_implied_prob"]))
              if pd.notna(r["market_implied_prob"]) else np.nan,
    axis=1,
)

_MAX_ODDS_IMPLIED_FLOOR = 0.2857  # 2026-07-10: hard stop at +250 (was +300/0.25)
cand["_max_odds_blocked"] = cand["market_implied_prob"].notna() & (cand["market_implied_prob"] < _MAX_ODDS_IMPLIED_FLOOR)

cand["passes_edge_rule"] = 0
cand.loc[
    (cand["game_has_odds"])
    & (cand["market_implied_prob"].notna())   # require confirmed market price (not 50% baseline)
    & (cand["is_repeat"]==0)
    & (cand["_edge_logit"] >= cand["_required_edge_logit"])
    & (cand["p_bet_win_model"] >= MIN_MODEL_PROB)
    & (~cand["_max_odds_blocked"]),
    "passes_edge_rule"
] = 1
cand.loc[
    (cand["game_has_odds"]) & (cand["is_repeat"]==1)
    & (cand["_edge_logit"] >= cand["_required_edge_logit"])
    & (cand["p_bet_win_model"] >= MIN_MODEL_PROB)
    & (~cand["_max_odds_blocked"]),
    "passes_edge_rule"
] = 1
# MLB-EDGE-RULE-DETAIL-1 (2026-10-01 user audit): the below-bar card text
# used to reconstruct an approximate reason ("needs ~X% edge") instead of
# stating what passes_edge_rule actually checked. Written here, at the rule
# itself, for every row that fails it: each failed clause with its real
# numbers and the miss. p_req is the model prob this row needed at its own
# vig-included market price for the logit edge to clear the bar.
def _mlb_edge_rule_detail(r):
    if r["passes_edge_rule"] == 1:
        return None
    if not bool(r.get("game_has_odds")) or pd.isna(r.get("market_implied_prob")):
        return "passes_edge_rule failed: no confirmed market price (game_has_odds/market_implied_prob missing)"
    _p, _mkt = float(r["p_bet_win_model"]), float(r["market_implied_prob"])
    _el, _req = float(r["_edge_logit"]), float(r["_required_edge_logit"])
    _fails = []
    if _el < _req:
        _p_req = 1.0 / (1.0 + math.exp(-(_mlb_logit(_mkt) + _req)))
        _fails.append(f"logit edge {_el:+.3f} < required {_req:+.3f} ({float(r['_base_edge_min']):.1%} floor"
                      f"{', repeat-game' if r.get('is_repeat') == 1 else ''}) -- at mkt {_mkt:.1%} (vig-incl) needs "
                      f"model >= {_p_req:.1%}, has {_p:.1%}, missed by {_p_req - _p:.1%}")
    if _p < MIN_MODEL_PROB:
        _fails.append(f"model prob {_p:.1%} < MIN_MODEL_PROB {MIN_MODEL_PROB:.0%}, missed by {MIN_MODEL_PROB - _p:.1%}")
    if bool(r.get("_max_odds_blocked")):
        _fails.append(f"mkt {_mkt:.1%} < max-odds floor {_MAX_ODDS_IMPLIED_FLOOR:.1%} (longer than ~+250)")
    return "passes_edge_rule failed: " + "; ".join(_fails) if _fails else None
cand["_edge_rule_detail"] = cand.apply(_mlb_edge_rule_detail, axis=1)

_max_odds_blocked_ct = int((cand["_max_odds_blocked"] & cand["game_has_odds"] & (cand["_edge_logit"] >= cand["_required_edge_logit"]) & (cand["p_bet_win_model"] >= MIN_MODEL_PROB)).sum())
if _max_odds_blocked_ct:
    print(f"[MAX ODDS CAP] {_max_odds_blocked_ct} play(s) blocked -- market implied prob <{_MAX_ODDS_IMPLIED_FLOOR:.0%} (longer than ~+250), regardless of edge")
_logit_scaled_ct = int(((cand["edge"] < cand["_base_edge_min"]) & (cand["passes_edge_rule"] == 1)).sum())
if _logit_scaled_ct:
    print(f"[LOGIT-SPACE EDGE] {_logit_scaled_ct} play(s) qualified on symmetric logit-space edge despite a smaller raw percentage-point edge than the base threshold")

# ---------------------------------------------------------
# 10b) pitching mismatch gate + underdog threshold
# ---------------------------------------------------------
def _sp_elo_gate(nm):
    import unicodedata
    def _nn(s): return unicodedata.normalize('NFKD', str(s)).encode('ascii','ignore').decode().lower().strip()
    try:
        _d = mlb_pitcher_elo
    except NameError:
        return 1500.0
    if not isinstance(_d, dict): return 1500.0
    v = _d.get(str(nm))
    if v is not None: return float(v)
    _nk = _nn(nm)
    for k, ev in _d.items():
        if _nn(k) == _nk: return float(ev)
    return 1500.0

_home_pit_cols = ['home_pitcher_final_name', 'home_pitcher_name', 'home_starter']
_away_pit_cols = ['away_pitcher_final_name', 'away_pitcher_name', 'away_starter']

def _pick_sp_name(row):
    is_home = str(row.get('bet_team', '')).strip() == str(row.get(home_col, '')).strip()
    cols = _home_pit_cols if is_home else _away_pit_cols
    return next((str(row.get(c) or '').strip() for c in cols if row.get(c) and str(row.get(c)) != 'nan'), '')

def _opp_sp_name(row):
    is_home = str(row.get('bet_team', '')).strip() == str(row.get(home_col, '')).strip()
    cols = _away_pit_cols if is_home else _home_pit_cols
    return next((str(row.get(c) or '').strip() for c in cols if row.get(c) and str(row.get(c)) != 'nan'), '')

# Known TJS / major surgery returns � flag first-season-back pitchers
_TJS_RETURN_PITCHERS = {
    "gerrit cole":       "TJS (1st full season back 2026 -- workload/inning limits possible)",
    "tyler glasnow":     "TJS (1st full season back 2026 -- monitor pitch count)",
    "blake snell":       "TJS (1st full season back 2026 -- high variance early starts)",
    "carlos rodon":      "TJS (1st full season back 2026 -- rehab arm, extra K-rate variance)",
}
# 2026-07-29: this dict was defined but never read anywhere in this cell --
# same "computed but never wired in" pattern as the BB% gate and the
# series-momentum bug earlier in this project. Now surfaced in the grading
# loop below (see [TJS WATCH]).

# SP RATING STALENESS (60-day IL) -- hist_elo (mlb_pitcher_elo, used
# throughout the SP-rating blend above) is a rolling ELO built off a
# pitcher's start history -- if he misses months on a 60-day IL (e.g.
# Tommy John), his last pre-injury starts still anchor it, and the recency
# blend only lightly nudges that anchor when the return sample is thin
# (weight capped at 0.35 under 20 IP, rating capped at 1555 under 63 IP --
# both still clear the 1520 STRONG cutoff). Real case: Wesneski rated
# STRONG/1530 off a rating that predated his TJ recovery. Reuses the same
# MLB transactions endpoint Cell 86's pre-post checklist already queries for
# near-term scratches, just with a season-to-date window instead of 48h, so
# it can see a 60-day IL stint from months back, not just the last 2 days.
_sp_60day_il_players = {}   # norm name -> most recent reinstatement date
try:
    import datetime as _il_dt
    _il_start = f"{_il_dt.date.today().year}-01-01"
    _il_end = _il_dt.date.today().isoformat()
    # 2026-07-29 live-run bug: verified directly against the real MLB Stats API
    # that this was silently dead on arrival, in 3 independent ways -- (a)
    # "60-day"/"activated"/"reinstated" text lives in the `description` field,
    # never in `typeDesc` (which is a generic bucket -- "Status Change",
    # "Recalled", "Optioned", etc. -- and never contains that text, so the old
    # filter rejected every single row); (b) the player name lives under the
    # `person` key, not `player` (which doesn't exist in this schema at all);
    # (c) `limit=1000` silently truncates to the EARLIEST ~1000 rows of the
    # date range, not the most recent -- for a season-to-date window that's
    # only the first few weeks of the year. All three confirmed against
    # Hayden Wesneski's real 2026-07-29 transaction ("Houston Astros activated
    # RHP Hayden Wesneski from the 60-day injured list") -- the exact real
    # case, on the exact real card, that this gate exists to catch, and it
    # was invisible under the old code for all three reasons at once.
    _il_resp = requests.get(
        f"https://statsapi.mlb.com/api/v1/transactions?sportId=1&startDate={_il_start}&endDate={_il_end}",
        timeout=30
    )
    if _il_resp.status_code == 200:
        for _il_tx in _il_resp.json().get("transactions", []):
            _il_desc = str(_il_tx.get("description", "")).lower()
            if "60-day" not in _il_desc and "60 day" not in _il_desc:
                continue
            if "reinstated" not in _il_desc and "activated" not in _il_desc:
                continue  # only the RETURN matters here, not the placement
            _il_pnm = _il_tx.get("person", {}).get("fullName", "")
            if not _il_pnm:
                continue
            _il_key = _sp_norm_era(_il_pnm)
            _il_date = _il_tx.get("date", "")
            if _il_key not in _sp_60day_il_players or _il_date > _sp_60day_il_players[_il_key]:
                _sp_60day_il_players[_il_key] = _il_date
    print(f"  [SP STALENESS] {len(_sp_60day_il_players)} pitcher(s) returned from 60-day IL this season")
except Exception as _il_e:
    print(f"  WARN: 60-day IL lookup failed ({_il_e}) -- SP rating staleness check skipped")

_SP_STALE_MIN_IP = 30.0  # real IP thrown ON/AFTER the reinstatement date needed to trust the blended rating over the pre-injury anchor

_sp_ip_since_return_cache = {}
def _sp_ip_since_return(spnm, return_date):
    """Sum innings pitched strictly on/after the 60-day IL reinstatement date.
    mlb_pitcher_fip_lookup's ip_season is whole-season cumulative (cell 20) --
    it would include any starts thrown BEFORE the injury, silently clearing
    this gate on a pitcher with a real pre-injury workload even with zero
    starts since returning. Re-fetches the game log (sportId=1 -- majors
    only, so rehab/minor-league starts can't count) and filters by date."""
    _k = _sp_norm_era(spnm)
    if _k in _sp_ip_since_return_cache:
        return _sp_ip_since_return_cache[_k]
    _ip_since = 0.0
    try:
        _pid = _search_player_id(spnm)
        if _pid:
            _r = requests.get(
                f'{_MLB_API}/people/{_pid}/stats',
                params={'stats': 'gameLog', 'group': 'pitching', 'season': 2026,
                        'gameType': 'R', 'sportId': 1},
                headers=_headers, timeout=10
            )
            _splits = _r.json().get('stats', [{}])[0].get('splits', [])
            for _s in _splits:
                _gdate = _s.get('date', '')
                if _gdate and _gdate >= return_date:
                    _ip_since += float(_s.get('stat', {}).get('inningsPitched', 0) or 0)
    except Exception:
        pass
    _sp_ip_since_return_cache[_k] = round(_ip_since, 1)
    return _sp_ip_since_return_cache[_k]

_sp_starts_since_return_cache = {}
def _sp_starts_since_return(spnm, return_date):
    """M16 (2026-09-04, user audit -- McClanahan Sept 3 loss): counts real
    MLB starts (not IP) strictly on/after the IL reinstatement date -- a
    bulk/short-role arm can log meaningful IP quickly on a short leash while
    still having almost no real starts logged, which is the actual variance
    concern (bullpen decides innings 4-6 regardless of how he's throwing).
    Same fetch pattern/cache convention as _sp_ip_since_return above."""
    _k = _sp_norm_era(spnm)
    if _k in _sp_starts_since_return_cache:
        return _sp_starts_since_return_cache[_k]
    _starts = 0
    try:
        _pid = _search_player_id(spnm)
        if _pid:
            _r = requests.get(
                f'{_MLB_API}/people/{_pid}/stats',
                params={'stats': 'gameLog', 'group': 'pitching', 'season': 2026,
                        'gameType': 'R', 'sportId': 1},
                headers=_headers, timeout=10
            )
            _splits = _r.json().get('stats', [{}])[0].get('splits', [])
            for _s in _splits:
                _gdate = _s.get('date', '')
                if _gdate and _gdate >= return_date and int(_s.get('stat', {}).get('gamesStarted', 0) or 0) >= 1:
                    _starts += 1
    except Exception:
        pass
    _sp_starts_since_return_cache[_k] = _starts
    return _sp_starts_since_return_cache[_k]

_sp_post_il_trend_cache = {}
def _sp_post_il_trend(spnm, return_date):
    """M31 (2026-09-06, user audit -- Kikuchi Sept 5 shutout of the Pirates
    pick): M16 already tracks HOW MANY real starts a post-IL pitcher has
    logged, but nothing looks at the SHAPE of those starts -- a flat L3
    average (or a starts-since-return count) treats "5 ER then 2 ER" the
    same as "2 ER then 5 ER", even though the first is a real, visible trend
    a reader would want flagged. Confirmed live: Kikuchi's own first two
    starts back from a ~4-month absence were 5 ER/5.0 IP then 2 ER/7.0 IP --
    strictly improving -- before he threw a scoreless start against
    Pittsburgh. Purely informational, no grade/unit change (small sample,
    2-3 starts at most, not a backtested predictor) -- just surfaces a real
    trend the L3 average hides. Real per-pitcher game-log fetch, not the
    existing 30/90-day blanket scan (_sp_recent_il_return_players) -- this
    only runs for a pitcher already confirmed as a recent IL-return case."""
    _k = _sp_norm_era(spnm)
    if _k in _sp_post_il_trend_cache:
        return _sp_post_il_trend_cache[_k]
    _result = None
    try:
        _pid = _search_player_id(spnm)
        if _pid:
            _r = requests.get(
                f'{_MLB_API}/people/{_pid}/stats',
                params={'stats': 'gameLog', 'group': 'pitching', 'season': 2026,
                        'gameType': 'R', 'sportId': 1},
                headers=_headers, timeout=10
            )
            _splits = _r.json().get('stats', [{}])[0].get('splits', [])
            _return_starts = []
            for _s in sorted(_splits, key=lambda x: x.get('date', '')):
                _gdate = _s.get('date', '')
                if _gdate and _gdate >= return_date and int(_s.get('stat', {}).get('gamesStarted', 0) or 0) >= 1:
                    _ip = float(_s.get('stat', {}).get('inningsPitched', 0) or 0)
                    _er = float(_s.get('stat', {}).get('earnedRuns', 0) or 0)
                    _era = round(_er / _ip * 9, 2) if _ip > 0 else None
                    _return_starts.append({"date": _gdate, "ip": _ip, "er": _er, "era": _era})
            _eras = [s["era"] for s in _return_starts if s["era"] is not None]
            if len(_eras) >= 2:
                _improving = all(_eras[i] < _eras[i - 1] for i in range(1, len(_eras)))
                _declining = all(_eras[i] > _eras[i - 1] for i in range(1, len(_eras)))
                if _improving or _declining:
                    _result = {"starts": _return_starts, "trend": "improving" if _improving else "declining"}
    except Exception:
        pass
    _sp_post_il_trend_cache[_k] = _result
    return _sp_post_il_trend_cache[_k]

_sp_early_exit_history_cache = {}

# M-NEW-3 (2026-09-07, user audit -- Troy Melton ELITE tier despite a 7.84
# ERA in 2 starts against tonight's actual opponent, Twins, vs a 2.20
# season ERA): SP ELO/tier is built purely on season-aggregate stats,
# never cross-referenced against how this pitcher performs against THIS
# SPECIFIC opponent's batter pool. Reuses M15's own gameLog+opponent-name
# fetch pattern (_sp_early_exit_history above) -- same API call shape,
# just aggregating ERA across matching starts instead of scanning for an
# early-exit/IL pattern.
_sp_vs_opp_era_cache = {}
_M_NEW3_MIN_STARTS = 2   # need at least this many starts vs this opponent to trust the number
_M_NEW3_ERA_GAP = 1.0    # runs -- lowered from 1.5 (2026-09-11, user request -- Abbott vs
# Milwaukee case: real gap sat between 1.0-1.5R and stayed silent under the old threshold)

def _sp_vs_opponent_era(spnm, opp_team_name):
    """(era_vs_opp, n_starts) from this season's starts against opp_team_name,
    or (None, 0) if fewer than _M_NEW3_MIN_STARTS exist."""
    if not spnm or not opp_team_name:
        return None, 0
    _ck = (_sp_norm_era(spnm), opp_team_name)
    if _ck in _sp_vs_opp_era_cache:
        return _sp_vs_opp_era_cache[_ck]
    _result = (None, 0)
    try:
        _pid = _search_player_id(spnm)
        if _pid:
            _gl = requests.get(
                f'{_MLB_API}/people/{_pid}/stats',
                params={'stats': 'gameLog', 'group': 'pitching', 'season': 2026,
                        'gameType': 'R', 'sportId': 1},
                headers=_headers, timeout=10
            ).json()
            _splits = _gl.get('stats', [{}])[0].get('splits', [])
            _opp_splits = [
                s for s in _splits
                if (s.get('opponent') or {}).get('name', '') == opp_team_name
                and int(s.get('stat', {}).get('gamesStarted', 0) or 0) == 1
            ]
            if len(_opp_splits) >= _M_NEW3_MIN_STARTS:
                _er = sum(float(s.get('stat', {}).get('earnedRuns', 0) or 0) for s in _opp_splits)
                _ip = sum(_l3_parse_ip(s.get('stat', {}).get('inningsPitched', 0)) for s in _opp_splits)
                if _ip > 0:
                    _result = (round(_er / _ip * 9, 2), len(_opp_splits))
    except Exception:
        pass
    _sp_vs_opp_era_cache[_ck] = _result
    return _result

globals()["_sp_vs_opponent_era"] = _sp_vs_opponent_era

_sp_career_vs_opp_era_cache = {}
_SP_CAREER_H2H_MIN_APP = 5        # need at least this many career appearances vs opponent to trust the number
_SP_CAREER_H2H_LOOKBACK_SEASONS = 10  # regular seasons, current season back -- covers any active MLB career
_SP_CAREER_H2H_CUR_SEASON = 2026
_SP_CAREER_H2H_RISK_GAP = 1.5     # runs -- career ERA vs this opponent this many runs worse than season ERA

# CAREER H2H (2026-09-14, user audit -- Detmers vs Seattle: 5.24 career
# ERA in 13 appearances across 2021-2026, fully invisible on the card
# because [SP VS OPPONENT] (_sp_vs_opponent_era, M-NEW-3 above) only ever
# queries the CURRENT season and Detmers had exactly 1 start vs SEA in
# 2026 -- one below that flag's own 2-start minimum, so a real multi-year
# pattern never printed anywhere despite this season's H2H result (0.00
# ERA in 6.2 IP) looking clean. Aggregates ALL appearances (starts and
# relief -- matches how the user's own 13-appearance count was arrived
# at; a distinct question from M-NEW-3's starts-only in-season quality
# read) across a bounded season lookback rather than one hardcoded year,
# same gameLog endpoint/shape M-NEW-3 already trusts.
def _sp_career_vs_opponent_era(spnm, opp_team_name):
    """(era_vs_opp, n_appearances) aggregated across all appearances against
    opp_team_name over the last _SP_CAREER_H2H_LOOKBACK_SEASONS regular
    seasons, or (None, 0) if fewer than _SP_CAREER_H2H_MIN_APP exist.
    """
    if not spnm or not opp_team_name:
        return None, 0
    _ck = (_sp_norm_era(spnm), opp_team_name)
    if _ck in _sp_career_vs_opp_era_cache:
        return _sp_career_vs_opp_era_cache[_ck]
    _result = (None, 0)
    try:
        _pid = _search_player_id(spnm)
        if _pid:
            _total_er = 0.0
            _total_ip = 0.0
            _n = 0
            for _yr in range(_SP_CAREER_H2H_CUR_SEASON,
                              _SP_CAREER_H2H_CUR_SEASON - _SP_CAREER_H2H_LOOKBACK_SEASONS, -1):
                _gl = requests.get(
                    f'{_MLB_API}/people/{_pid}/stats',
                    params={'stats': 'gameLog', 'group': 'pitching', 'season': _yr,
                            'gameType': 'R', 'sportId': 1},
                    headers=_headers, timeout=10
                ).json()
                # BUG FIX (2026-09-14, found live-verifying this against tonight's real
                # slate -- Detmers vs Seattle never printed [CAREER H2H] despite a real
                # 5.24 ERA / 13 appearances): the MLB Stats API returns {"stats": []} for
                # any season before a pitcher's debut -- a present-but-EMPTY list, not a
                # missing key, so .get('stats', [{}])[0] skips its own default and indexes
                # [0] straight into the empty list, raising IndexError. The bare except
                # below then swallowed it -- but only after the loop had already died
                # mid-iteration, discarding every season's ER/IP already summed (2021-2026
                # all had real Detmers-vs-SEA starts and would have aggregated correctly;
                # the loop just never survived long enough to reach the return statement).
                # _sp_vs_opponent_era above never hits this since its single hardcoded
                # current season always has real stats for an active pitcher -- this gap
                # is unique to a multi-season loop that can walk past a pitcher's debut.
                _stats_arr = _gl.get('stats') or [{}]
                _splits = (_stats_arr[0] or {}).get('splits', [])
                _opp_splits = [
                    s for s in _splits
                    if (s.get('opponent') or {}).get('name', '') == opp_team_name
                ]
                for s in _opp_splits:
                    _ip = _l3_parse_ip(s.get('stat', {}).get('inningsPitched', 0))
                    _er = float(s.get('stat', {}).get('earnedRuns', 0) or 0)
                    _total_ip += _ip
                    _total_er += _er
                    _n += 1
            if _n >= _SP_CAREER_H2H_MIN_APP and _total_ip > 0:
                _result = (round(_total_er / _total_ip * 9, 2), _n)
    except Exception:
        pass
    _sp_career_vs_opp_era_cache[_ck] = _result
    return _result

globals()["_sp_career_vs_opponent_era"] = _sp_career_vs_opponent_era
_M15_EARLY_EXIT_IP_MAX  = 4.0   # innings -- below this counts as a short outing worth checking
_M15_IL_FOLLOWUP_DAYS   = 3     # a real IL placement within this many days after the start = the same injury event

def _sp_early_exit_history(spnm, opp_team_name, pick_is_home):
    """M15 (2026-09-04, user audit -- McClanahan Sept 3 loss): McClanahan
    exited a 2026-07-30 start against this exact opponent after just 3.0 IP,
    went on the 15-day IL the very next day (Left mid-back tightness), and
    tonight -- facing the same opponent again -- got hit hard. The existing
    POST-IL RETURN flag only reads "how much IP/how many days since a known
    return date" -- nothing checked whether THIS SPECIFIC opponent or park
    was the one tied to the original injury. Confirmed live via real MLB
    transactions before building this (not assumed): the July 30 start truly
    was a short outing immediately followed by a real IL placement.
    Real per-pitcher game-log + season-long transaction check -- not the
    existing 30-day-lookback _sp_recent_il_return_players scan (that's a
    blanket multi-pitcher scan for a different purpose), so this can catch
    an injury-exit pattern from anywhere this season, not just the last 30
    days. "Same park" is inferred from isHome rather than a separate venue
    fetch: two starts at the pitcher's own home park are definitionally the
    same park regardless of opponent; a start at an opponent's park is
    already covered by the same-opponent check."""
    if not spnm or not opp_team_name:
        return None
    _ck = (_sp_norm_era(spnm), opp_team_name, bool(pick_is_home))
    if _ck in _sp_early_exit_history_cache:
        return _sp_early_exit_history_cache[_ck]
    _result = None
    try:
        _pid = _search_player_id(spnm)
        if not _pid:
            _sp_early_exit_history_cache[_ck] = None
            return None
        _gl = requests.get(
            f'{_MLB_API}/people/{_pid}/stats',
            params={'stats': 'gameLog', 'group': 'pitching', 'season': 2026,
                    'gameType': 'R', 'sportId': 1},
            headers=_headers, timeout=10
        ).json()
        _splits = _gl.get('stats', [{}])[0].get('splits', [])
        _tx = requests.get(
            f'{_MLB_API}/transactions',
            params={'playerId': _pid, 'startDate': '2026-01-01',
                    'endDate': _il_dt.date.today().isoformat()},
            timeout=10
        ).json()
        _il_dates = []
        for _t in _tx.get('transactions', []):
            _desc = str(_t.get('description', '')).lower()
            if 'injured list' in _desc and 'placed' in _desc:
                try:
                    _il_dates.append(_il_dt.date.fromisoformat(str(_t.get('date', ''))[:10]))
                except Exception:
                    pass
        for _s in _splits:
            _ip = float(_s.get('stat', {}).get('inningsPitched', 0) or 0)
            if _ip >= _M15_EARLY_EXIT_IP_MAX:
                continue
            _s_opp = (_s.get('opponent') or {}).get('name', '')
            _same_opp = _s_opp == opp_team_name
            _same_park = bool(_s.get('isHome')) and bool(pick_is_home)
            if not (_same_opp or _same_park):
                continue
            try:
                _start_date = _il_dt.date.fromisoformat(str(_s.get('date', ''))[:10])
            except Exception:
                continue
            for _ild2 in _il_dates:
                _gap = (_ild2 - _start_date).days
                if 0 <= _gap <= _M15_IL_FOLLOWUP_DAYS:
                    _result = {"date": str(_s.get('date', '')), "ip": _ip, "opponent": _s_opp,
                               "il_date": _ild2.isoformat(), "same_opponent": _same_opp,
                               "same_park": _same_park}
                    break
            if _result:
                break
    except Exception:
        _result = None
    _sp_early_exit_history_cache[_ck] = _result
    return _result

# M21 (2026-09-04, user audit): lineups typically post ~2h before first
# pitch. A card built earlier than that has no confirmed batting order yet
# -- if a team also has a real key-player-out flag (mlb_lineup_flags, set
# earlier in this notebook), the replacement hitter for that spot is
# unknown and isn't reflected in tonight's offensive projection. Purely
# informational (no grade/unit change, unlike M15/M16) -- prompts a
# re-check closer to game time rather than blocking anything. Verified
# live before building: the rest of the M18-M22 batch this same audit
# proposed turned out to rest on factually wrong premises for that
# night's specific game (two Cubs hitters described as the "opposing"
# offense facing their own teammate, a stale/superseded offense stat when
# the model already blends real L10 offense into the run projection, a
# home/road ERA split that was backwards in the real data) -- this was
# the one claim (Xavier Edwards on the 10-day IL) that checked out
# against the real transaction/roster data, so it's the only one built.
_LINEUP_CONFIRM_HOURS = 2.0

def _lineup_confirm_notes(game_dt_utc, aw_team, hm_team):
    _notes = []
    _gdt = pd.to_datetime(game_dt_utc, errors="coerce", utc=True)
    if pd.isna(_gdt):
        return _notes
    _hrs = (_gdt - pd.Timestamp.now(tz="UTC")).total_seconds() / 3600.0
    if _hrs < _LINEUP_CONFIRM_HOURS:
        return _notes
    if "mlb_lineup_flags" not in globals() or not mlb_lineup_flags or "_pl_norm" not in globals():
        return _notes
    for _t in (aw_team, hm_team):
        _flag = mlb_lineup_flags.get(_pl_norm(_t))
        if _flag and _flag.get("msg"):
            _notes.append(
                f"!! LINEUP NOT CONFIRMED: {_hrs:.1f}h until first pitch, {_t}'s starting lineup "
                f"not yet posted -- has a real injury absence (see [INJURY] above) whose replacement "
                f"isn't reflected in tonight's projection yet, re-check the confirmed lineup before placing"
            )
    return _notes

cand['_pick_sp_elo'] = cand.apply(lambda r: _sp_elo_gate(_pick_sp_name(r)), axis=1)
cand['_opp_sp_elo']  = cand.apply(lambda r: _sp_elo_gate(_opp_sp_name(r)), axis=1)
cand['_sp_elo_gap']  = cand['_opp_sp_elo'] - cand['_pick_sp_elo']

# ================================================================
# GATE HARD-STOP -> EDGE KNOCKDOWN CONVERSION (2026-08-30, user request)
# Gates 1/2/3/4/5/6/8/9 below used to zero passes_edge_rule outright the
# moment their condition fired, regardless of how big the underlying edge
# was (Gate 3's walk-rate hard stop was even deliberately upgraded FROM a
# shave-with-exception TO a hard stop on 2026-07-29 after a real loss --
# user confirmed today they want that reverted too, same knockdown as the
# rest). Now every one of these subtracts a flat 8pp from cand['edge'] and
# only THEN re-checks the normal _base_edge_min floor -- identical shape to
# the post-IL/HR9/cold-streak/thin-sample shave gates already in this cell,
# just applied to the gates that used to be all-or-nothing. A big enough
# real edge now survives with a haircut instead of being wiped out.
# Gate 7 (SP unresolved -- no starter name found at all) and Gate 7b (SP
# lookup failed -- name found but zero rating/stats match) are DELIBERATELY
# NOT converted: those aren't a risk signal on a real number, they're "we
# have no real number here at all" -- there's nothing valid to apply an 8pp
# discount to. Left as hard stops.
# ================================================================
# WEIGHTED (2026-08-30, user request -- "these gates each have different
# weights... change the edge by how much that certain gate matters"): a flat
# 8pp for all 8 gates treated a soft context flag (record gap) identically to
# an acute, real-loss-validated blowup risk (walk-rate). Each gate below now
# has its own knockdown, sized off what's actually documented about it in
# this cell's own gate comments -- an acute/recent/evidence-backed risk on
# the PICK's own starter knocks harder than a general market-mismatch or
# season-context flag. All 8 are still PROVISIONAL/subjective (no backtest
# behind the differentiation itself, same as the flat 8pp it replaces) --
# re-tune individually once real graded results exist per gate.
# M-NEW-1 (2026-09-10, user audit): every gate/shave below could silently
# disqualify a play with nothing on the card explaining which one fired --
# the card's own fallback text pointed at "this cell's Gate 1-10 block-count
# summary above", which (a) is a different cell than the one actually
# rendering the card (cell 89, not this one) and (b) only ever printed
# AGGREGATE counts ("shaved edge -X% on N play(s)") with no game/team
# identification even if you found the right cell -- there was no way to
# trace a specific disqualified play (e.g. Rangers ML vs Mariners, +21.6%
# real edge) back to a specific gate from anywhere in the notebook's output.
# Single per-row reason column, written once per row (first gate to actually
# drop a play below the qualifying floor wins the reason -- each gate's own
# mask already requires passes_edge_rule==1 at check time, so a row already
# disqualified by an earlier gate is structurally excluded from every later
# gate's mask and can't overwrite an earlier reason). Cell 89 reads this
# column directly instead of guessing/pointing at a summary that isn't there.
cand['_gate_fail_reason'] = None

_GATE1_MISMATCH_KNOCKDOWN    = round(0.09 * _MODEL_MARKET_BLEND_W, 4)  # was 0.09 on the raw-edge scale  # BUG FIX (MLB-EDGE-BLEND-1, 2026-09-25 user audit): rescaled to the blended edge scale (edge_blended = w*edge_raw when a real market price exists) so this mechanism's original relative selectivity survives the new, much-smaller edge scale instead of unintentionally becoming a near-automatic disqualifier.
_GATE2_UNDERDOG_KNOCKDOWN    = round(0.06 * _MODEL_MARKET_BLEND_W, 4)  # BUG FIX (MLB-EDGE-BLEND-1, 2026-09-25): rescaled to the blended edge scale, was 0.06 -- already only fires on a marginal/near-coinflip underdog -- softest of the 8
_GATE3_WALKRATE_KNOCKDOWN    = round(0.10 * _MODEL_MARKET_BLEND_W, 4)  # BUG FIX (MLB-EDGE-BLEND-1, 2026-09-25): rescaled to the blended edge scale, was 0.10 -- pick SP's OWN season command risk -- real loss on record (Yesavage, 7BB/1.2IP)
_GATE4_LASTSTART_KNOCKDOWN   = round(0.11 * _MODEL_MARKET_BLEND_W, 4)  # BUG FIX (MLB-EDGE-BLEND-1, 2026-09-25): rescaled to the blended edge scale, was 0.11 -- pick SP's most recent outing specifically -- most acute/recent signal of the 8
_GATE5_EDGECEILING_KNOCKDOWN = round(0.08 * _MODEL_MARKET_BLEND_W, 4)  # BUG FIX (MLB-EDGE-BLEND-1, 2026-09-25): rescaled to the blended edge scale, was 0.08 -- sanity check on the EDGE NUMBER itself (likely data error), not a team/player risk per se
_GATE6_SPSTALE_KNOCKDOWN     = round(0.09 * _MODEL_MARKET_BLEND_W, 4)  # BUG FIX (MLB-EDGE-BLEND-1, 2026-09-25): rescaled to the blended edge scale, was 0.09 -- input-quality uncertainty -- rating may still reflect pre-injury form
_GATE8_ELOERA_KNOCKDOWN      = round(0.08 * _MODEL_MARKET_BLEND_W, 4)  # BUG FIX (MLB-EDGE-BLEND-1, 2026-09-25): rescaled to the blended edge scale, was 0.08 -- input-quality uncertainty -- ELO/ERA disagreement on a thin sample
_GATE9_RECORDGAP_KNOCKDOWN   = round(0.05 * _MODEL_MARKET_BLEND_W, 4)  # BUG FIX (MLB-EDGE-BLEND-1, 2026-09-25): rescaled to the blended edge scale, was 0.05 -- softest -- a single SP matchup can legitimately outweigh a season record gap for one game

# Gate 1: opp SP is 80+ ELO points better â€” require 55% model prob
_mismatch_mask = (
    (cand['passes_edge_rule'] == 1)
    & (cand['_sp_elo_gap'] >= 80)
    & (cand['p_bet_win_model'] < 0.55)
)
cand.loc[_mismatch_mask, 'edge'] = cand.loc[_mismatch_mask, 'edge'] - _GATE1_MISMATCH_KNOCKDOWN
_mismatch_now_fails = _mismatch_mask & (cand['edge'] < cand['_base_edge_min'])
cand.loc[_mismatch_now_fails, 'passes_edge_rule'] = 0
cand.loc[_mismatch_now_fails, '_gate_fail_reason'] = cand.loc[_mismatch_now_fails, '_gate_fail_reason'].fillna(
    cand.loc[_mismatch_now_fails].apply(lambda r: f"Gate 1 -- SP MISMATCH: opp SP {r['_sp_elo_gap']:.0f} ELO better, model prob {r['p_bet_win_model']:.1%} < 55% floor (edge shaved -{_GATE1_MISMATCH_KNOCKDOWN:.0%}, now below required floor)", axis=1))
print(f'Pitching mismatch gate : shaved edge -{_GATE1_MISMATCH_KNOCKDOWN:.0%} on {int(_mismatch_mask.sum())} play(s), newly blocked {int(_mismatch_now_fails.sum())} (opp SP 80+ ELO higher, model <55%)')

# Gate 2: underdog with low confidence â€” require 10% edge
_underdog_mask = (
    (cand['passes_edge_rule'] == 1)
    & (cand['market_implied_prob'].fillna(0.5) < 0.45)
    & (cand['p_bet_win_model'] < 0.55)
    & (cand['edge'] < 0.10)
)
cand.loc[_underdog_mask, 'edge'] = cand.loc[_underdog_mask, 'edge'] - _GATE2_UNDERDOG_KNOCKDOWN
_underdog_now_fails = _underdog_mask & (cand['edge'] < cand['_base_edge_min'])
cand.loc[_underdog_now_fails, 'passes_edge_rule'] = 0
cand.loc[_underdog_now_fails, '_gate_fail_reason'] = cand.loc[_underdog_now_fails, '_gate_fail_reason'].fillna(
    "Gate 2 -- UNDERDOG THRESHOLD: underdog with model prob < 55% and edge < 10% before this shave "
    f"(edge shaved -{_GATE2_UNDERDOG_KNOCKDOWN:.0%}, now below required floor)")
print(f'Underdog threshold gate: shaved edge -{_GATE2_UNDERDOG_KNOCKDOWN:.0%} on {int(_underdog_mask.sum())} play(s), newly blocked {int(_underdog_now_fails.sum())} (underdog, model <55%, edge <10%)')

# Gate 3: elevated walk-rate starter -- require a sharper edge.
# Season BB% is a real red flag ERA/FIP/WHIP alone can miss -- a
# command-unstable starter can carry a fine season ERA while still being a
# real inning-1 blowup risk. Surfaced 2026-07-11: Blue Jays ML lost after
# Yesavage walked 7 in the first two innings (franchise record), 4 ER in
# 1.2 IP -- the ERA thesis collapsed before the WHIP/FIP gates ever got a
# chance to matter. mlb_pitcher_fip_lookup already scrapes 'bb_pct_pit'
# (season BB%, FanGraphs) in the FIP/K-BB% cell above -- it was just never
# read anywhere in this cell (same "computed but never wired in" pattern as
# the QA-form goals-against and series-momentum bugs fixed earlier this
# project). Season-level only for now, no L3/recent-form walk rate exists
# yet (unlike era_l3/whip_l3) -- would need extending the cell-20
# rolling-window builder with walks-per-start data to add one.
_BB_PCT_ELEVATED  = 0.105  # ~BB/9 4.0 equivalent (BB% * ~38 PA/9IP ~= BB/9)

def _pick_sp_bb_pct(row):
    _nm = _pick_sp_name(row)
    if not _nm:
        return np.nan
    _fev = globals().get('mlb_pitcher_fip_lookup', {}).get(_norm_name(_nm))
    return float(_fev.get('bb_pct_pit')) if _fev and pd.notna(_fev.get('bb_pct_pit')) else np.nan

cand['_pick_sp_bb_pct'] = cand.apply(_pick_sp_bb_pct, axis=1)
# 2026-07-29: hard stop, no sharp-edge exception -- previously a big enough
# edge (>=20%) could still clear this gate (e.g. Cantillo, 11.2% BB% over 22
# starts, graded/played through on edge size alone). An elevated walk rate is
# a data-quality/blowup risk on the STARTER himself, not something a large
# raw edge on the market should be allowed to override.
_walk_risk_mask = (
    (cand['passes_edge_rule'] == 1)
    & (cand['_pick_sp_bb_pct'] >= _BB_PCT_ELEVATED)
)
cand.loc[_walk_risk_mask, 'edge'] = cand.loc[_walk_risk_mask, 'edge'] - _GATE3_WALKRATE_KNOCKDOWN
_walk_risk_now_fails = _walk_risk_mask & (cand['edge'] < cand['_base_edge_min'])
cand.loc[_walk_risk_now_fails, 'passes_edge_rule'] = 0
cand.loc[_walk_risk_now_fails, '_gate_fail_reason'] = cand.loc[_walk_risk_now_fails, '_gate_fail_reason'].fillna(
    cand.loc[_walk_risk_now_fails].apply(lambda r: f"Gate 3 -- WALK-RATE RISK: pick SP season BB% {r['_pick_sp_bb_pct']:.1%} >= {_BB_PCT_ELEVATED:.1%} elevated-command-risk floor (edge shaved -{_GATE3_WALKRATE_KNOCKDOWN:.0%}, now below required floor)", axis=1))
print(f'Walk-rate variance gate : shaved edge -{_GATE3_WALKRATE_KNOCKDOWN:.0%} on {int(_walk_risk_mask.sum())} play(s), newly blocked {int(_walk_risk_now_fails.sum())} (pick SP season BB% >= {_BB_PCT_ELEVATED:.1%} -- reverted to a knockdown 2026-08-30 per user request, was a hard stop since 2026-07-29)')

# Gate 4: last-start walk-rate implosion -- separate from the season BB%
# gate above. A single disaster outing (e.g. 7 BB in 1.2 IP) gets diluted
# into a 3-start or season average and can easily sit under the Gate 3
# threshold even the very next start. This reads the single most recent
# start directly (last_start_bb/last_start_ip, cell 20) rather than any
# rolling average -- catches the exact "pitched two days ago and imploded"
# case Gate 3 structurally cannot see.
_LAST_START_BB_MIN = 4     # walks
_LAST_START_IP_MAX = 2.0   # innings -- implosion-shaped outing, not just a bad start
_LAST_START_SHARP_EDGE = round(0.20 * _MODEL_MARKET_BLEND_W, 4)  # was 0.20 on the raw-edge scale  # BUG FIX (MLB-EDGE-BLEND-1, 2026-09-25 user audit): rescaled to the blended edge scale (edge_blended = w*edge_raw when a real market price exists) so this mechanism's original relative selectivity survives the new, much-smaller edge scale instead of unintentionally becoming a near-automatic disqualifier.

def _pick_sp_last_start_flag(row):
    _nm = _pick_sp_name(row)
    if not _nm:
        return False
    _fev = globals().get('mlb_pitcher_fip_lookup', {}).get(_norm_name(_nm))
    if not _fev:
        return False
    _bb = _fev.get('last_start_bb')
    _ip = _fev.get('last_start_ip')
    if _bb is None or _ip is None or pd.isna(_bb) or pd.isna(_ip):
        return False
    return bool(_bb >= _LAST_START_BB_MIN and _ip < _LAST_START_IP_MAX)

cand['_pick_sp_last_start_implosion'] = cand.apply(_pick_sp_last_start_flag, axis=1)
_last_start_risk_mask = (
    (cand['passes_edge_rule'] == 1)
    & (cand['_pick_sp_last_start_implosion'])
    & (cand['edge'] < _LAST_START_SHARP_EDGE)
)
cand.loc[_last_start_risk_mask, 'edge'] = cand.loc[_last_start_risk_mask, 'edge'] - _GATE4_LASTSTART_KNOCKDOWN
_last_start_now_fails = _last_start_risk_mask & (cand['edge'] < cand['_base_edge_min'])
cand.loc[_last_start_now_fails, 'passes_edge_rule'] = 0
cand.loc[_last_start_now_fails, '_gate_fail_reason'] = cand.loc[_last_start_now_fails, '_gate_fail_reason'].fillna(
    f"Gate 4 -- LAST-START IMPLOSION: pick SP walked {_LAST_START_BB_MIN}+ in under {_LAST_START_IP_MAX} IP last start, "
    f"edge was below the {_LAST_START_SHARP_EDGE:.0%} bar this flag requires (edge shaved -{_GATE4_LASTSTART_KNOCKDOWN:.0%}, now below required floor)")
print(f'Last-start implosion gate: shaved edge -{_GATE4_LASTSTART_KNOCKDOWN:.0%} on {int(_last_start_risk_mask.sum())} play(s), newly blocked {int(_last_start_now_fails.sum())} (pick SP walked {_LAST_START_BB_MIN}+ in under {_LAST_START_IP_MAX} IP last start, edge < {_LAST_START_SHARP_EDGE:.0%})')

# Gate 5: edge ceiling -- a raw edge this large on a real market almost
# always means a data problem (stale line, stale ELO/team-quality input, a
# scratched/mismatched starter -- see the SP-override fix above), not a
# genuine 25%+ market mispricing. Flags for manual calibration review
# instead of silently printing as an official play.
_EDGE_CEILING = 0.25
_edge_ceiling_mask = (
    (cand['passes_edge_rule'] == 1)
    & (cand['edge'] >= _EDGE_CEILING)
)
cand.loc[_edge_ceiling_mask, 'edge'] = cand.loc[_edge_ceiling_mask, 'edge'] - _GATE5_EDGECEILING_KNOCKDOWN
_edge_ceiling_now_fails = _edge_ceiling_mask & (cand['edge'] < cand['_base_edge_min'])
cand.loc[_edge_ceiling_now_fails, 'passes_edge_rule'] = 0
cand.loc[_edge_ceiling_now_fails, '_gate_fail_reason'] = cand.loc[_edge_ceiling_now_fails, '_gate_fail_reason'].fillna(
    f"Gate 5 -- EDGE CEILING: raw edge >= {_EDGE_CEILING:.0%} almost always means a data problem (stale line, stale "
    f"rating, mismatched starter), not a genuine mispricing (edge shaved -{_GATE5_EDGECEILING_KNOCKDOWN:.0%}, now below required floor)")
print(f'Edge ceiling gate       : shaved edge -{_GATE5_EDGECEILING_KNOCKDOWN:.0%} on {int(_edge_ceiling_mask.sum())} play(s), newly blocked {int(_edge_ceiling_now_fails.sum())} (edge >= {_EDGE_CEILING:.0%} -- calibration-review flag, no longer an automatic block)')

# Gate 6: SP rating staleness (60-day IL return) -- a pitcher's hist_elo
# blend can still carry his pre-injury form if the current-season sample
# since returning is thin (see the moved block above). Applied at the same
# cand/passes_edge_rule level as every other gate here, specifically so
# downstream consumers (cell 89's Final Card, the RL section's ML
# double-exposure check) see the exclusion too -- a loop-local `continue`
# later in this cell only affected this cell's own print statements.
def _cand_sp_stale(spnm):
    if not spnm: return None
    _k = _sp_norm_era(spnm)
    if _k not in _sp_60day_il_players:
        return None
    _return_date = _sp_60day_il_players[_k]
    _ip = _sp_ip_since_return(spnm, _return_date)
    if _ip >= _SP_STALE_MIN_IP:
        return None  # real post-return sample -- rating has caught up
    return _return_date

cand['_pick_sp_stale'] = cand.apply(lambda r: _cand_sp_stale(_pick_sp_name(r)), axis=1)
cand['_opp_sp_stale']  = cand.apply(lambda r: _cand_sp_stale(_opp_sp_name(r)), axis=1)
_sp_stale_mask = (
    (cand['passes_edge_rule'] == 1)
    & (cand['_pick_sp_stale'].notna() | cand['_opp_sp_stale'].notna())
)
cand.loc[_sp_stale_mask, 'edge'] = cand.loc[_sp_stale_mask, 'edge'] - _GATE6_SPSTALE_KNOCKDOWN
_sp_stale_now_fails = _sp_stale_mask & (cand['edge'] < cand['_base_edge_min'])
cand.loc[_sp_stale_now_fails, 'passes_edge_rule'] = 0
cand.loc[_sp_stale_now_fails, '_gate_fail_reason'] = cand.loc[_sp_stale_now_fails, '_gate_fail_reason'].fillna(
    f"Gate 6 -- SP RATING STALENESS: pick or opp SP returned from 60-day IL, IP since return < {_SP_STALE_MIN_IP:.0f} "
    f"-- rating may still reflect pre-injury form (edge shaved -{_GATE6_SPSTALE_KNOCKDOWN:.0%}, now below required floor)")
print(f'SP rating staleness gate : shaved edge -{_GATE6_SPSTALE_KNOCKDOWN:.0%} on {int(_sp_stale_mask.sum())} play(s), newly blocked {int(_sp_stale_now_fails.sum())} (pick or opp SP returned from 60-day IL, IP since return < {_SP_STALE_MIN_IP:.0f})')

# 2026-08-24 (user audit): post-IL-return discount, broader than Gate 6 above.
# Gate 6's IL-transaction filter hard-requires "60-day"/"60 day" in the
# description, so a normal 10/15-day IL stint (e.g. Jeffrey Springs, hip;
# Zebby Matthews, foot laceration) never appears in _sp_60day_il_players and
# gets zero adjustment today. This detector catches ANY IL length over the
# last 30 days and shaves cand['edge'] proportionally for a thin post-return
# IP sample -- additive only, does not touch Gate 6's existing 60-day block.
# BUG FIX (2026-09-04, user audit -- Braves Sept 1 loss): this window was
# 30 days, which only fits a short IL stint. Real case found live: AJ
# Smith-Shawver (60-day IL for RIGHT ELBOW RECONSTRUCTION -- Tommy John --
# March 25 through his 2026-07-29 activation) started Sept 1, 34 days after
# that activation date -- 4 days past this scan's old 30-day cutoff, so he'd
# already fallen out of _sp_recent_il_return_players entirely despite having
# made only 3 real MLB starts since returning (activated 7/29, immediately
# optioned back down, recalled for good 8/18) and being barely a month
# removed from a season-defining surgery. He got hit hard (4 IP, 5 ER, his
# worst start of the season) with zero post-IL caution applied anywhere.
# Widened to 90 days so a major injury (TJ, other surgery) stays tracked
# through a realistic recovery runway -- safe to widen because entry into
# this dict alone doesn't trigger anything: _post_il_info's edge-shave and
# M16's unit cap both separately gate on real accumulated IP/starts since
# return (_sp_ip_since_return / _sp_starts_since_return), so a pitcher who's
# actually built up a real sample still clears normally either way.
_sp_recent_il_return_players = {}   # norm name -> most recent reinstatement date
try:
    _ril_start = (_il_dt.date.today() - _il_dt.timedelta(days=90)).isoformat()
    _ril_end = _il_dt.date.today().isoformat()
    _ril_resp = requests.get(
        f"https://statsapi.mlb.com/api/v1/transactions?sportId=1&startDate={_ril_start}&endDate={_ril_end}",
        timeout=30
    )
    if _ril_resp.status_code == 200:
        for _ril_tx in _ril_resp.json().get("transactions", []):
            _ril_desc = str(_ril_tx.get("description", "")).lower()
            if "injured list" not in _ril_desc:
                continue
            if "reinstated" not in _ril_desc and "activated" not in _ril_desc:
                continue
            _ril_pnm = _ril_tx.get("person", {}).get("fullName", "")
            if not _ril_pnm:
                continue
            _ril_key = _sp_norm_era(_ril_pnm)
            _ril_date = _ril_tx.get("date", "")
            if _ril_key not in _sp_recent_il_return_players or _ril_date > _sp_recent_il_return_players[_ril_key]:
                _sp_recent_il_return_players[_ril_key] = _ril_date
    print(f"  [SP POST-IL] {len(_sp_recent_il_return_players)} pitcher(s) returned from IL (any length) in the last 90 days")
except Exception as _ril_e:
    print(f"  WARN: recent-IL-return lookup failed ({_ril_e}) -- post-IL discount skipped")

_POST_IL_EDGE_SHAVE_MAX = round(0.05 * _MODEL_MARKET_BLEND_W, 4)   # was 0.05 on the raw-edge scale  # BUG FIX (MLB-EDGE-BLEND-1, 2026-09-25 user audit): rescaled to the blended edge scale (edge_blended = w*edge_raw when a real market price exists) so this mechanism's original relative selectivity survives the new, much-smaller edge scale instead of unintentionally becoming a near-automatic disqualifier.
globals()["_sp_recent_il_return_players"] = _sp_recent_il_return_players

def _post_il_info(spnm):
    if not spnm:
        return None
    _k = _sp_norm_era(spnm)
    if _k not in _sp_recent_il_return_players:
        return None
    _return_date = _sp_recent_il_return_players[_k]
    _ip = _sp_ip_since_return(spnm, _return_date)
    if _ip >= _SP_STALE_MIN_IP:
        return None
    _frac = 1.0 - (_ip / _SP_STALE_MIN_IP)
    try:
        _days = (_il_dt.date.today() - _il_dt.date.fromisoformat(str(_return_date)[:10])).days
    except Exception:
        _days = None
    return {"return_date": _return_date, "ip_since": _ip, "days_since": _days,
            "edge_shave": round(_POST_IL_EDGE_SHAVE_MAX * _frac, 4)}

globals()["_post_il_info"] = _post_il_info

# M32 (2026-09-15, user audit -- Pallante post-IL thin sample): the edge
# shave above and M16's bulk-role cap below both discount for THIN SAMPLE
# SIZE, but neither actually checks whether the pitcher's stuff has come
# back -- a real drop in fastball velocity is the most direct signal of
# incomplete recovery from injury and no existing gate can see it (FIP/
# xFIP/SIERA are FanGraphs SEASON rate stats; the only other Statcast this
# notebook touches is batted-ball/contact-quality, never pitch velocity).
# Mandatory, not just informational: if the check can't be resolved at all
# (Statcast API failure/lag, too few pitches in either window), the play
# is forced to 0.5u rather than sized normally on an unverified assumption
# that he's fully recovered -- same shape as M16's bulk-role cap just below.
_VELO_DROP_THRESHOLD = 1.0   # mph -- below this is normal start-to-start/weather noise, not a real flag
_VELO_MIN_PITCHES    = 10    # per window, minimum to trust an average
_VELO_PRE_WINDOW_DAYS = 90   # mirrors the 90-day IL-return lookback above

_sp_velo_check_cache = {}
def _sp_post_il_velocity_check(spnm, return_date):
    """Compare pre-IL vs post-IL fastball (FF/SI/FC) release_speed via
    Statcast. Pre-IL window = starts before return_date -- he can't have
    pitched WHILE on the IL, so every start before the return date is by
    definition a pre-injury outing; no separate placement-date lookup
    needed. Returns a dict with status 'verified' (pre_velo/post_velo/
    drop/flagged) or 'unverified' (reason) -- the caller treats unverified
    as a forced 0.5u cap, not a silent pass-through."""
    _k = (_sp_norm_era(spnm), str(return_date))
    if _k in _sp_velo_check_cache:
        return _sp_velo_check_cache[_k]
    _result = {"status": "unverified", "reason": "check not run"}
    try:
        import pybaseball as _pb
        try:
            _pb.cache.enable()
        except Exception:
            pass
        _pid = _search_player_id(spnm)
        if not _pid:
            _result = {"status": "unverified", "reason": "no player id found"}
        else:
            _ret_d = _il_dt.date.fromisoformat(str(return_date)[:10])
            _pre_start = (_ret_d - _il_dt.timedelta(days=_VELO_PRE_WINDOW_DAYS)).isoformat()
            _pre_end   = (_ret_d - _il_dt.timedelta(days=1)).isoformat()
            _post_start = _ret_d.isoformat()
            _post_end   = _il_dt.date.today().isoformat()
            _pre_df  = _pb.statcast_pitcher(_pre_start, _pre_end, _pid)
            _post_df = _pb.statcast_pitcher(_post_start, _post_end, _pid)
            _fb_types = {"FF", "SI", "FC"}
            _pre_fb  = _pre_df[_pre_df["pitch_type"].isin(_fb_types)]  if _pre_df  is not None and not _pre_df.empty  else None
            _post_fb = _post_df[_post_df["pitch_type"].isin(_fb_types)] if _post_df is not None and not _post_df.empty else None
            _n_pre  = 0 if _pre_fb  is None else len(_pre_fb)
            _n_post = 0 if _post_fb is None else len(_post_fb)
            if _n_pre < _VELO_MIN_PITCHES or _n_post < _VELO_MIN_PITCHES:
                _result = {"status": "unverified",
                           "reason": f"insufficient Statcast fastballs (pre={_n_pre}, post={_n_post}, need {_VELO_MIN_PITCHES}+ each)"}
            else:
                _pre_v  = float(_pre_fb["release_speed"].mean())
                _post_v = float(_post_fb["release_speed"].mean())
                _drop = _pre_v - _post_v
                _result = {"status": "verified", "pre_velo": round(_pre_v, 1), "post_velo": round(_post_v, 1),
                           "drop": round(_drop, 1), "flagged": bool(_drop >= _VELO_DROP_THRESHOLD)}
    except Exception as _e:
        _result = {"status": "unverified", "reason": f"Statcast lookup failed ({_e})"}
    _sp_velo_check_cache[_k] = _result
    return _result

globals()["_sp_post_il_velocity_check"] = _sp_post_il_velocity_check

cand['_pick_post_il'] = cand.apply(lambda r: _post_il_info(_pick_sp_name(r)), axis=1)
cand['_opp_post_il']  = cand.apply(lambda r: _post_il_info(_opp_sp_name(r)), axis=1)

def _post_il_edge_delta(row):
    _d = 0.0
    _p = row.get('_pick_post_il')
    _o = row.get('_opp_post_il')
    if isinstance(_p, dict):
        _d += _p.get('edge_shave', 0.0)
    if isinstance(_o, dict):
        _d -= _o.get('edge_shave', 0.0) * 0.5
    return _d

_post_il_delta = cand.apply(_post_il_edge_delta, axis=1)
_post_il_active_mask = (_post_il_delta != 0.0) & (cand['passes_edge_rule'] == 1)
cand['edge'] = cand['edge'] - _post_il_delta
_post_il_now_fails = (cand['passes_edge_rule'] == 1) & (cand['edge'] < cand['_base_edge_min'])
cand.loc[_post_il_now_fails, 'passes_edge_rule'] = 0
cand.loc[_post_il_now_fails, '_gate_fail_reason'] = cand.loc[_post_il_now_fails, '_gate_fail_reason'].fillna(
    "POST-IL EDGE SHAVE: pick or opp SP has a thin post-IL-return sample (any IL length, last 90 days) -- rating not yet earned back")
print(f'Post-IL edge shave        : adjusted {int(_post_il_active_mask.sum())} play(s), newly blocked {int(_post_il_now_fails.sum())} (pick/opp SP thin post-IL sample, any IL length)')

# 2026-08-24 (user audit): explicit HR/9 collapse flag. HR is already baked
# into the FIP-based SP ELO (13x coefficient) but only in aggregate/blended
# form -- hr9_l3 (rolling HR/9 over the last 3 starts) is computed by
# _sp_fetch_recent above and was never read anywhere downstream. This adds
# an explicit rate-based gate on top of the existing ELO blend.
_HR9_FLAG_THRESHOLD = 1.8   # provisional -- revisit after live grading (a real HR/9 rate, NOT an edge-scale constant -- not rescaled)

# 2026-09-10 (user audit, item 4): this was flat/season hr9_l3 (last-3-starts,
# no venue awareness at all) -- real case, deGrom (Rangers): 10 HR allowed in 41
# road IP in 2026 (2.20 HR/9), a real T-Mobile-specific risk (dome removes wind
# as a suppressor, fly balls stay in play) that a season/L3-blended rate can't
# see. Prefers the venue-specific HR9 split from mlb_sp_split_era (same dict/
# same 8-start floor _sp_split_ra_ratio and _sp_split_elo_adj already use for
# this exact pitcher) when the road or home side has enough sample; falls back
# to the existing flat hr9_l3 otherwise -- same fallback shape as every other
# split-aware fix in this cell (never crashes/blocks on missing split data, just
# reads the less-precise number that was already here).
def _hr9_edge_shave(spnm, is_home=None):
    if not spnm:
        return 0.0, None
    _hr9 = None
    if is_home is not None and 'mlb_sp_split_era' in globals():
        _sv = mlb_sp_split_era.get(str(spnm).strip())
        if _sv:
            _starts = _sv.get('home_starts', 0) if is_home else _sv.get('road_starts', 0)
            _venue_hr9 = _sv.get('home_hr9') if is_home else _sv.get('road_hr9')
            if _starts >= 8 and _venue_hr9 is not None:
                _hr9 = _venue_hr9
    if _hr9 is None:
        _f = _sp_fetch_recent(spnm) if '_sp_fetch_recent' in globals() else None
        _hr9 = (_f or {}).get('hr9_l3')
    if _hr9 is None or _hr9 < _HR9_FLAG_THRESHOLD:
        return 0.0, None
    # BUG FIX (MLB-EDGE-BLEND-1): cap + per-unit rate rescaled -- same
    # reasoning as every other edge-scale constant in this cell.
    _shave = min(0.05 * _MODEL_MARKET_BLEND_W, 0.02 * _MODEL_MARKET_BLEND_W * (_hr9 - _HR9_FLAG_THRESHOLD))
    return round(_shave, 4), _hr9

globals()["_hr9_edge_shave"] = _hr9_edge_shave

def _pick_sp_is_home(row):
    return str(row.get('bet_team', '')).strip() == str(row.get(home_col, '')).strip()

def _opp_sp_is_home(row):
    return not _pick_sp_is_home(row)
# 2026-09-10 (user audit -- Diamondbacks -1.5, 2026-09-09 real loss):
# every SP-quality shave from here through Gate 8 below (HR9, sustained
# cold streak, FIP reliability, short-role, opponent mismatch, bulk-role
# FIP divergence, thin MLB sample, ELO-ERA divergence) only ever touches
# cand['edge'] -- the ML candidate table. The RUN LINE section further
# down builds its own probability straight from pred_home_runs/
# pred_away_runs and never sees any of this. Real case: Zac Gallen carried
# a documented sustained cold streak + elevated HR/9 that shaved his ML
# edge from qualifying down to a non-qualifying +10.2% -- correctly
# blocked -- but the exact same start, evaluated for -1.5, got zero
# discount and cleared at +15.7% as an OFFICIAL-BAR EDGE. Arizona lost
# outright. Snapshotting cand['edge'] here (before the shave chain) and
# diffing it against the post-Gate-8 value below captures the NET effect
# of every shave in this block in one place, without re-deriving any of
# their individual formulas a second time -- so this can't silently drift
# out of sync if a shave in between is retuned or a new one is added.
cand['_edge_pre_sp_quality'] = cand['edge'].copy()
cand['_pick_hr9_shave'], cand['_pick_hr9_val'] = zip(*cand.apply(lambda r: _hr9_edge_shave(_pick_sp_name(r), _pick_sp_is_home(r)), axis=1))
cand['_opp_hr9_shave'],  cand['_opp_hr9_val']  = zip(*cand.apply(lambda r: _hr9_edge_shave(_opp_sp_name(r), _opp_sp_is_home(r)), axis=1))
_hr9_delta = cand['_pick_hr9_shave'] - (cand['_opp_hr9_shave'] * 0.5)
_hr9_active_mask = (_hr9_delta != 0.0) & (cand['passes_edge_rule'] == 1)
cand['edge'] = cand['edge'] - _hr9_delta
_hr9_now_fails = (cand['passes_edge_rule'] == 1) & (cand['edge'] < cand['_base_edge_min'])
cand.loc[_hr9_now_fails, 'passes_edge_rule'] = 0
cand.loc[_hr9_now_fails, '_gate_fail_reason'] = cand.loc[_hr9_now_fails, '_gate_fail_reason'].fillna(
    f"HR/9 COLLAPSE-RISK SHAVE: pick or opp SP >= {_HR9_FLAG_THRESHOLD} HR/9 over last 3 starts")
print(f'HR/9 collapse-risk shave   : adjusted {int(_hr9_active_mask.sum())} play(s), newly blocked {int(_hr9_now_fails.sum())} (pick or opp SP >= {_HR9_FLAG_THRESHOLD} HR/9 L3)')

# SP HEALTH UNCONFIRMED shave (2026-09-21, user audit -- Twins @ Giants:
# Zebby Matthews graded WEAK(<1440)/1311 and was the sole factor driving the
# Giants' +18% ML edge ("SP ELO edge (Tidwell 1408 vs Matthews 1240)") -- but
# ESPN had Matthews listed Day-to-Day, which invalidates the SP quality gap
# the edge is built on. The only SP health check that existed (_sp_il_flags
# below, built off the MLB Stats API transactions log) only ever sees actual
# IL placements -- day-to-day/game-time-decision status is never an IL move,
# so it was structurally invisible there, and that check was only ever wired
# into a display-only function (_show_injuries, defined later in this cell)
# that is never actually called. _fetch_espn_injuries (defined above) already
# returns every player's real ESPN status regardless of position -- this just
# cross-references the actual starting pitcher's name against it. Checked on
# BOTH sides: the opposing pitcher's health is usually THE reason for the
# edge in the first place (as here), so an unconfirmed opposing starter
# invalidates the thesis just as much as an unconfirmed pick-side one -- full
# weight both ways, unlike the HR9/L10 gates above where an opponent issue is
# a tailwind (bad news for the opponent), not a risk.
_SP_HEALTH_UNCONFIRMED_WORDS = ("day-to-day", "day to day", "dtd", "questionable", "game-time decision")
_SP_HEALTH_SHAVE = round(0.10 * _MODEL_MARKET_BLEND_W, 4)  # was 0.10 on the raw-edge scale -- bigger than the SP-quality shaves above since this questions
# whether the SP matchup driving the edge even happens, not just one input into it

def _sp_health_shave(spnm, team_name):
    if not spnm or not team_name:
        return 0.0, None
    try:
        _inj_list = _fetch_espn_injuries(team_name)
    except Exception:
        return 0.0, None
    _spk = _sp_norm_era(spnm)
    for _it in (_inj_list or []):
        if _sp_norm_era(_it.get('name', '')) != _spk:
            continue
        _st = str(_it.get('status', '')).lower()
        if any(_w in _st for _w in _SP_HEALTH_UNCONFIRMED_WORDS):
            return _SP_HEALTH_SHAVE, _it.get('status')
        return 0.0, None
    return 0.0, None

globals()["_sp_health_shave"] = _sp_health_shave

def _sp_health_pick_team(row):
    return row.get(home_col, '') if _pick_sp_is_home(row) else row.get(away_col, '')

def _sp_health_opp_team(row):
    return row.get(away_col, '') if _pick_sp_is_home(row) else row.get(home_col, '')

cand['_pick_sp_health_shave'], cand['_pick_sp_health_val'] = zip(*cand.apply(lambda r: _sp_health_shave(_pick_sp_name(r), _sp_health_pick_team(r)), axis=1))
cand['_opp_sp_health_shave'],  cand['_opp_sp_health_val']  = zip(*cand.apply(lambda r: _sp_health_shave(_opp_sp_name(r), _sp_health_opp_team(r)), axis=1))
_sp_health_delta = cand['_pick_sp_health_shave'] + cand['_opp_sp_health_shave']
_sp_health_active_mask = (_sp_health_delta != 0.0) & (cand['passes_edge_rule'] == 1)
cand['edge'] = cand['edge'] - _sp_health_delta
_sp_health_now_fails = (cand['passes_edge_rule'] == 1) & (cand['edge'] < cand['_base_edge_min'])
cand.loc[_sp_health_now_fails, 'passes_edge_rule'] = 0
cand.loc[_sp_health_now_fails, '_gate_fail_reason'] = cand.loc[_sp_health_now_fails, '_gate_fail_reason'].fillna(
    "SP HEALTH UNCONFIRMED: pick or opp starter listed Day-to-Day/Questionable per ESPN -- "
    "the SP-quality gap driving this edge isn't confirmed to actually take the mound")
print(f'SP health-unconfirmed shave: adjusted {int(_sp_health_active_mask.sum())} play(s), newly blocked {int(_sp_health_now_fails.sum())} (pick or opp SP Day-to-Day/Questionable per ESPN)')

# Sustained cold-streak edge shave (2026-08-25, user audit -- Will Warren):
# the card's [SP COLD STREAK] flag only ever looked at L3 ERA, which reads
# a 3-start blip and a genuine multi-start collapse identically -- Warren
# went 1-5, 5.77 ERA over his last 13 appearances, a much longer and more
# significant decline than L3 alone can distinguish from noise. Uses the
# new era_l10 window (_sp_fetch_recent above) so a real sustained slump
# gets its own explicit shave, additive with (stacks on top of) the HR/9
# shave above rather than replacing it -- both can fire on the same start.
_SUSTAINED_COLD_L10_THRESHOLD = 5.00   # provisional -- revisit after live grading
def _sustained_cold_shave(spnm):
    if not spnm:
        return 0.0, None
    _f = _sp_fetch_recent(spnm) if '_sp_fetch_recent' in globals() else None
    _l10 = (_f or {}).get('era_l10')
    _gs10 = (_f or {}).get('gs_l10') or 0
    if _l10 is None or _gs10 < 5 or _l10 < _SUSTAINED_COLD_L10_THRESHOLD:
        return 0.0, None
    #   # BUG FIX (MLB-EDGE-BLEND-1, 2026-09-25): rescaled to the blended edge scale
    _shave = min(0.06 * _MODEL_MARKET_BLEND_W, 0.015 * _MODEL_MARKET_BLEND_W * (_l10 - _SUSTAINED_COLD_L10_THRESHOLD))
    return round(_shave, 4), _l10

globals()["_sustained_cold_shave"] = _sustained_cold_shave
cand['_pick_l10_shave'], cand['_pick_l10_val'] = zip(*cand.apply(lambda r: _sustained_cold_shave(_pick_sp_name(r)), axis=1))
cand['_opp_l10_shave'],  cand['_opp_l10_val']  = zip(*cand.apply(lambda r: _sustained_cold_shave(_opp_sp_name(r)), axis=1))
_l10_delta = cand['_pick_l10_shave'] - (cand['_opp_l10_shave'] * 0.5)
_l10_active_mask = (_l10_delta != 0.0) & (cand['passes_edge_rule'] == 1)
cand['edge'] = cand['edge'] - _l10_delta
_l10_now_fails = (cand['passes_edge_rule'] == 1) & (cand['edge'] < cand['_base_edge_min'])
cand.loc[_l10_now_fails, 'passes_edge_rule'] = 0
cand.loc[_l10_now_fails, '_gate_fail_reason'] = cand.loc[_l10_now_fails, '_gate_fail_reason'].fillna(
    f"SUSTAINED COLD-STREAK SHAVE: pick or opp SP L10 ERA >= {_SUSTAINED_COLD_L10_THRESHOLD}")
print(f'Sustained cold-streak shave: adjusted {int(_l10_active_mask.sum())} play(s), newly blocked {int(_l10_now_fails.sum())} (pick or opp SP L10 ERA >= {_SUSTAINED_COLD_L10_THRESHOLD})')

# XERA-CORROBORATED COLD-STREAK SHAVE (2026-09-13, user audit -- Arizona's
# featured pick, +15.7% edge, LOST 2-6: Brandon Pfaadt (ELITE/1618) allowed
# 6 ER on 10 hits over 6.2 IP. The card's own [SP COLD STREAK] bullet
# (Cell 6e66db20's WHY loop) had ALREADY said 'L3 ERA 4.26 vs season 3.00
# -- supported by xERA -- likely a real form change' -- the model's own
# highest-confidence read that this decline was real, not BABIP noise.
# That verdict (abs(era_l3 - xera) < 1.0, i.e. xERA corroborates the
# elevated L3 ERA instead of explaining it away as sequencing luck) was
# PURELY COSMETIC -- it fed zero edge/tier impact anywhere. The only real
# cold-streak shave that existed (_sustained_cold_shave, just above) only
# looks at the CRUDE magnitude of L10 ERA >= 5.00, completely blind to
# whether xERA backs it up -- Pfaadt's L10 never crossed that bar, so
# nothing fired despite the model's own text already flagging this as
# real. This is the missing piece: same xera/era_l3/era_season fields
# already sitting in mlb_pitcher_fip_lookup (Cell 19 FanGraphs fetch), and
# the exact same abs(era_l3 - xera) < 1.0 corroboration test the display
# bullet already computes -- just never wired to anything numeric before.
_XERA_COLD_L3_GAP = 0.75      # runs -- L3 ERA above season ERA before this starts firing
_XERA_COLD_MIN_IP = 9.0       # min L3 innings before trusting a short-window signal
_XERA_COLD_CORROBORATION = 1.0  # matches the display verdict's own threshold exactly
_XERA_COLD_SHAVE_MAX = round(0.05 * _MODEL_MARKET_BLEND_W, 4)  # BUG FIX (MLB-EDGE-BLEND-1, 2026-09-25): rescaled to the blended edge scale, was 0.05

def _xera_cold_shave(spnm):
    if not spnm or 'mlb_pitcher_fip_lookup' not in globals():
        return 0.0, None
    _fev = mlb_pitcher_fip_lookup.get(_sp_norm_era(spnm), {})
    _l3, _seas, _xera, _l3ip = _fev.get('era_l3'), _fev.get('era_season'), _fev.get('xera'), _fev.get('ip_l3')
    if _l3 is None or _seas is None or _xera is None or _l3ip is None:
        return 0.0, None
    try:
        _l3, _seas, _xera, _l3ip = float(_l3), float(_seas), float(_xera), float(_l3ip)
    except (TypeError, ValueError):
        return 0.0, None
    if pd.isna(_l3) or pd.isna(_seas) or pd.isna(_xera) or pd.isna(_l3ip):
        return 0.0, None
    if _l3ip < _XERA_COLD_MIN_IP:
        return 0.0, None
    _gap = _l3 - _seas
    if _gap <= _XERA_COLD_L3_GAP:
        return 0.0, None
    if abs(_l3 - _xera) >= _XERA_COLD_CORROBORATION:
        return 0.0, None  # xERA reads this as sequencing luck, not a true change -- matches display verdict
    #   # BUG FIX (MLB-EDGE-BLEND-1, 2026-09-25): rescaled to the blended edge scale
    _shave = min(_XERA_COLD_SHAVE_MAX, 0.02 * _MODEL_MARKET_BLEND_W * (_gap - _XERA_COLD_L3_GAP))
    return round(_shave, 4), round(_gap, 2)

globals()['_xera_cold_shave'] = _xera_cold_shave
cand['_pick_xeracold_shave'], cand['_pick_xeracold_val'] = zip(*cand.apply(lambda r: _xera_cold_shave(_pick_sp_name(r)), axis=1))
cand['_opp_xeracold_shave'],  cand['_opp_xeracold_val']  = zip(*cand.apply(lambda r: _xera_cold_shave(_opp_sp_name(r)), axis=1))
_xeracold_delta = cand['_pick_xeracold_shave'] - (cand['_opp_xeracold_shave'] * 0.5)
_xeracold_active_mask = (_xeracold_delta != 0.0) & (cand['passes_edge_rule'] == 1)
cand['edge'] = cand['edge'] - _xeracold_delta
_xeracold_now_fails = (cand['passes_edge_rule'] == 1) & (cand['edge'] < cand['_base_edge_min'])
cand.loc[_xeracold_now_fails, 'passes_edge_rule'] = 0
cand.loc[_xeracold_now_fails, '_gate_fail_reason'] = cand.loc[_xeracold_now_fails, '_gate_fail_reason'].fillna(
    f"XERA-CORROBORATED COLD-STREAK SHAVE: pick or opp SP L3 ERA >{_XERA_COLD_L3_GAP}R above season ERA, confirmed real by xERA")
print(f'xERA-corroborated cold shave  : adjusted {int(_xeracold_active_mask.sum())} play(s), newly blocked {int(_xeracold_now_fails.sum())} (pick or opp SP L3 ERA >{_XERA_COLD_L3_GAP}R above season, xERA-confirmed real)')

# TEAM OFFENSIVE COLD-STREAK ML DISCOUNT (2026-09-10, user audit): the
# existing [TEAM FORM] flag (a later display-only cell, standings win/loss
# record) is explicitly informational only -- no run-projection or grade
# impact. Real case flagged live: White Sox 4-6 L10 with a real scoring
# dropoff got zero edge treatment anywhere. L10 runs-for/game is already on
# board (home_runs_for_avg_10 / away_runs_for_avg_10); season runs-for/game
# comes from the same MLB Stats API standings response the downstream
# [TEAM FORM] flag already calls (runsScored/gamesPlayed both already in
# that JSON) -- own fetch+cache here since this gate runs earlier in the
# notebook and can't rely on that later cell's cache being populated yet.
_OFFENSE_COLD_L10_GAP = 0.80         # runs below season R/G before this starts firing
_OFFENSE_COLD_SHAVE_BASE = round(0.02 * _MODEL_MARKET_BLEND_W, 4)      # shave right at the threshold  # BUG FIX (MLB-EDGE-BLEND-1, 2026-09-25): rescaled to the blended edge scale, was 0.02
_OFFENSE_COLD_SHAVE_MAX = round(0.03 * _MODEL_MARKET_BLEND_W, 4)       # cap  # BUG FIX (MLB-EDGE-BLEND-1, 2026-09-25): rescaled to the blended edge scale, was 0.03
_OFFENSE_COLD_SHAVE_PER_RUN = round(0.01 * _MODEL_MARKET_BLEND_W, 4)   # additional shave per run beyond the threshold  # BUG FIX (MLB-EDGE-BLEND-1, 2026-09-25): rescaled to the blended edge scale, was 0.01
_OFFENSE_COLD_MIN_GP = 20            # min season games played before trusting the season R/G average

_season_rpg_cache = globals().get('_mlb_season_rpg_cache')
if _season_rpg_cache is None:
    _season_rpg_cache = {}
    try:
        import requests as _srp_requests, datetime as _srp_dt
        _srp_r = _srp_requests.get(
            f'https://statsapi.mlb.com/api/v1/standings?leagueId=103,104&season={_srp_dt.date.today().year}&standingsTypes=regularSeason',
            timeout=10)
        for _srp_rec in _srp_r.json().get('records', []):
            for _srp_tr in _srp_rec.get('teamRecords', []):
                _srp_tn = _pl_norm(_srp_tr.get('team', {}).get('name', ''))
                _srp_gp = _srp_tr.get('gamesPlayed') or 0
                _srp_rs = _srp_tr.get('runsScored')
                if _srp_tn and _srp_gp >= _OFFENSE_COLD_MIN_GP and _srp_rs is not None:
                    _season_rpg_cache[_srp_tn] = float(_srp_rs) / float(_srp_gp)
    except Exception as _srp_e:
        print(f'  WARN: season R/G fetch failed ({_srp_e})')
    globals()['_mlb_season_rpg_cache'] = _season_rpg_cache

def _offense_cold_shave(row):
    _is_home = str(row.get('bet_team', '')).strip() == str(row.get(home_col, '')).strip()
    _l10_col = 'home_runs_for_avg_10' if _is_home else 'away_runs_for_avg_10'
    _l10 = row.get(_l10_col)
    if _l10 is None or (isinstance(_l10, float) and pd.isna(_l10)):
        return 0.0, None
    _is_home_pick = str(row.get('bet_team', '')).strip() == str(row.get(home_col, '')).strip()
    _pick_tm_name = row.get(home_col, '') if _is_home_pick else row.get(away_col, '')
    _tn = _pl_norm(str(_pick_tm_name))
    _szn_rpg = _season_rpg_cache.get(_tn)
    if _szn_rpg is None:
        return 0.0, None
    _gap = float(_szn_rpg) - float(_l10)
    if _gap < _OFFENSE_COLD_L10_GAP:
        return 0.0, None
    _shave = min(_OFFENSE_COLD_SHAVE_MAX,
                 _OFFENSE_COLD_SHAVE_BASE + _OFFENSE_COLD_SHAVE_PER_RUN * (_gap - _OFFENSE_COLD_L10_GAP))
    return round(_shave, 4), round(_gap, 2)

globals()['_offense_cold_shave'] = _offense_cold_shave
cand['_pick_offcold_shave'], cand['_pick_offcold_val'] = zip(*cand.apply(_offense_cold_shave, axis=1))
_offcold_mask = (cand['_pick_offcold_shave'] > 0) & (cand['passes_edge_rule'] == 1)
cand['edge'] = cand['edge'] - cand['_pick_offcold_shave']
_offcold_now_fails = (cand['passes_edge_rule'] == 1) & (cand['edge'] < cand['_base_edge_min'])
cand.loc[_offcold_now_fails, 'passes_edge_rule'] = 0
cand.loc[_offcold_now_fails, '_gate_fail_reason'] = cand.loc[_offcold_now_fails, '_gate_fail_reason'].fillna(
    f'OFFENSIVE COLD-STREAK SHAVE: L10 R/G at least {_OFFENSE_COLD_L10_GAP} runs below season average')
print(f'Offensive cold-streak shave  : adjusted {int(_offcold_mask.sum())} play(s), newly blocked {int(_offcold_now_fails.sum())} (pick team L10 R/G >= {_OFFENSE_COLD_L10_GAP} below season avg)')

# H2H CONSECUTIVE-MEETINGS SUPPRESSOR (2026-09-10, user request): reads
# the multi-season streak computed in Cell H2H (mlb_h2h_streak_lookup) --
# if the picked team is on the LOSING end of a real same-matchup win
# streak, apply a small edge discount. Deliberately soft (1-2%, not a
# hard gate) -- MLB rosters turn over enough across the multi-season
# window this streak can span that it is real signal, but a modest one.
_H2H_STREAK_SHAVE_BASE = round(0.01 * _MODEL_MARKET_BLEND_W, 4)        # shave right at the streak-min threshold  # BUG FIX (MLB-EDGE-BLEND-1, 2026-09-25): rescaled to the blended edge scale, was 0.01
_H2H_STREAK_SHAVE_MAX = round(0.02 * _MODEL_MARKET_BLEND_W, 4)         # cap  # BUG FIX (MLB-EDGE-BLEND-1, 2026-09-25): rescaled to the blended edge scale, was 0.02
_H2H_STREAK_SHAVE_PER_GAME = round(0.0025 * _MODEL_MARKET_BLEND_W, 5)  # additional shave per game beyond the threshold  # BUG FIX (MLB-EDGE-BLEND-1, 2026-09-25): rescaled to the blended edge scale, was 0.0025

def _h2h_streak_shave(row):
    _streak_lookup = globals().get('mlb_h2h_streak_lookup')
    if not _streak_lookup:
        return 0.0, None
    _away_nm = str(row.get(away_col, '')).lower()
    _home_nm = str(row.get(home_col, '')).lower()
    _sv = _streak_lookup.get((_away_nm, _home_nm))
    if not _sv:
        return 0.0, None
    _is_home_pick = str(row.get('bet_team', '')).strip() == str(row.get(home_col, '')).strip()
    _pick_team_id = row.get('home_team_id') if _is_home_pick else row.get('away_team_id')
    if _pick_team_id is None or (isinstance(_pick_team_id, float) and pd.isna(_pick_team_id)):
        return 0.0, None
    if int(_pick_team_id) != _sv.get('streak_loser_id'):
        return 0.0, None
    _n = _sv.get('streak_len', 0)
    _shave = min(_H2H_STREAK_SHAVE_MAX,
                 _H2H_STREAK_SHAVE_BASE + _H2H_STREAK_SHAVE_PER_GAME * (_n - _H2H_STREAK_MIN))
    return round(_shave, 4), _n

globals()['_h2h_streak_shave'] = _h2h_streak_shave
cand['_pick_h2hstreak_shave'], cand['_pick_h2hstreak_val'] = zip(*cand.apply(_h2h_streak_shave, axis=1))
_h2hstreak_mask = (cand['_pick_h2hstreak_shave'] > 0) & (cand['passes_edge_rule'] == 1)
cand['edge'] = cand['edge'] - cand['_pick_h2hstreak_shave']
_h2hstreak_now_fails = (cand['passes_edge_rule'] == 1) & (cand['edge'] < cand['_base_edge_min'])
cand.loc[_h2hstreak_now_fails, 'passes_edge_rule'] = 0
cand.loc[_h2hstreak_now_fails, '_gate_fail_reason'] = cand.loc[_h2hstreak_now_fails, '_gate_fail_reason'].fillna(
    f'H2H STREAK SHAVE: picked team on the losing end of a {_H2H_STREAK_MIN}+ game same-matchup streak')
print(f'H2H streak shave             : adjusted {int(_h2hstreak_mask.sum())} play(s), newly blocked {int(_h2hstreak_now_fails.sum())} (pick on losing end of a {_H2H_STREAK_MIN}+ game matchup streak)')

# BULLPEN-GAME OPPONENT EDGE BONUS (2026-09-11, user request -- Rays/
# Astros, Ullola): the _eff_pit reweight (Cell 87, run-projection build)
# already shifts the RUN PROJECTION for a true-bullpen-game start toward
# the bullpen, but the user's own research found the card's edge (+9.4%)
# still understated the real advantage of facing a 'starter' with
# literally no real-starter track record -- a distinct, structural edge
# (bullpen-day roster/usage inefficiency, thinner middle innings, higher
# variance) beyond whatever the run-projection blend alone captures.
# Additive bonus, not a shave -- applied to whichever side is picked
# AGAINST a confirmed true bullpen game. Deliberately does not touch
# passes_edge_rule (unlike every shave gate above, which re-checks and can
# newly BLOCK a play) -- flipping a play from fail to pass here can't tell
# apart 'failed on edge size' from 'failed on an unrelated data-quality
# gate' (the exact class of bug M-NEW-1 was built to fix, see Cell 89's
# gate-fail-reason comment), so this only adjusts the displayed edge and
# downstream reasoning, not the qualification decision itself.
_BULLPEN_GAME_EDGE_BONUS = 0.04  # edge bonus, facing a confirmed true bullpen game

def _bullpen_game_opp_bonus(row):
    _opp_nm = _opp_sp_name(row) if '_opp_sp_name' in globals() else None
    if not _opp_nm or 'mlb_pitcher_fip_lookup' not in globals():
        return 0.0, False
    _ev = mlb_pitcher_fip_lookup.get(_sp_norm_era(_opp_nm), {})
    if bool(_ev.get('is_true_bullpen_game')):
        return _BULLPEN_GAME_EDGE_BONUS, True
    return 0.0, False

globals()['_bullpen_game_opp_bonus'] = _bullpen_game_opp_bonus
cand['_opp_bg_bonus'], cand['_opp_bg_flag'] = zip(*cand.apply(_bullpen_game_opp_bonus, axis=1))
_bg_bonus_mask = cand['_opp_bg_flag']
cand['edge'] = cand['edge'] + cand['_opp_bg_bonus']
print(f'Bullpen-game opponent edge bonus : applied to {int(_bg_bonus_mask.sum())} play(s) facing a confirmed true bullpen game (+{_BULLPEN_GAME_EDGE_BONUS:.0%} edge)')

# FIP reliability shave (2026-09-02, user audit -- Randy Dobnak, Royals ML
# loss): mirrors the HR9/sustained-cold shave pattern above. The SP ELO is
# deliberately FIP-based (Cell 82), but nothing ever checked FIP itself
# against the more predictive xFIP/SIERA already sitting in the same
# mlb_pitcher_fip_lookup row (Cell 19's FanGraphs fetch). Real case: Dobnak
# showed FIP 3.74 (drove his STRONG/1555 tier and this play's edge) while
# his own xFIP (4.24) and SIERA (4.67) read him as meaningfully worse --
# he gave up 6 ER on 2 HR in the actual start. Started as an informational-
# only flag (Cell 89) the same session; user asked for a real discount once
# the pattern was confirmed. Threshold set lower than the informational
# flag's original 0.75R (FIP-vs-SIERA gaps run smaller than L3-ERA-vs-xERA
# ones) so this actually bites at the magnitude the real case showed.
_FIP_RELIABILITY_GAP = 0.50        # runs -- provisional, revisit after live grading
_FIP_RELIABILITY_SHAVE_MAX = round(0.05 * _MODEL_MARKET_BLEND_W, 4)  # BUG FIX (MLB-EDGE-BLEND-1, 2026-09-25): rescaled to the blended edge scale, was 0.05
def _fip_reliability_shave(spnm):
    if not spnm:
        return 0.0, None
    _ev = mlb_pitcher_fip_lookup.get(_sp_norm_era(spnm), {}) if 'mlb_pitcher_fip_lookup' in globals() else {}
    try:
        _fip = float(_ev.get('fip')) if _ev.get('fip') is not None else None
        _xfip = float(_ev.get('xfip')) if _ev.get('xfip') is not None else None
        _siera = float(_ev.get('siera')) if _ev.get('siera') is not None else None
    except (TypeError, ValueError):
        return 0.0, None
    if _fip is None:
        return 0.0, None
    _gaps = [v - _fip for v in (_xfip, _siera) if v is not None]
    _worst_gap = max(_gaps) if _gaps else 0.0
    if _worst_gap < _FIP_RELIABILITY_GAP:
        return 0.0, None
    #   # BUG FIX (MLB-EDGE-BLEND-1, 2026-09-25): rescaled to the blended edge scale
    _shave = min(_FIP_RELIABILITY_SHAVE_MAX, 0.05 * _MODEL_MARKET_BLEND_W * (_worst_gap - _FIP_RELIABILITY_GAP))
    return round(_shave, 4), round(_worst_gap, 2)

globals()["_fip_reliability_shave"] = _fip_reliability_shave
cand['_pick_fiprel_shave'], cand['_pick_fiprel_val'] = zip(*cand.apply(lambda r: _fip_reliability_shave(_pick_sp_name(r)), axis=1))
cand['_opp_fiprel_shave'],  cand['_opp_fiprel_val']  = zip(*cand.apply(lambda r: _fip_reliability_shave(_opp_sp_name(r)), axis=1))
_fiprel_delta = cand['_pick_fiprel_shave'] - (cand['_opp_fiprel_shave'] * 0.5)
_fiprel_active_mask = (_fiprel_delta != 0.0) & (cand['passes_edge_rule'] == 1)
cand['edge'] = cand['edge'] - _fiprel_delta
_fiprel_now_fails = (cand['passes_edge_rule'] == 1) & (cand['edge'] < cand['_base_edge_min'])
cand.loc[_fiprel_now_fails, 'passes_edge_rule'] = 0
cand.loc[_fiprel_now_fails, '_gate_fail_reason'] = cand.loc[_fiprel_now_fails, '_gate_fail_reason'].fillna(
    f"FIP RELIABILITY SHAVE: pick or opp SP FIP reads >= {_FIP_RELIABILITY_GAP}R better than xFIP/SIERA -- FIP not a trustworthy number for this pitcher")
print(f'FIP reliability shave      : adjusted {int(_fiprel_active_mask.sum())} play(s), newly blocked {int(_fiprel_now_fails.sum())} (pick or opp SP FIP >= {_FIP_RELIABILITY_GAP}R better than xFIP/SIERA)')

# ERA reliability shave (MLB-FIP-2, 2026-09-24 user audit): the check above
# only ever compares FIP against xFIP/SIERA -- it says nothing about ERA
# itself reading well below BOTH peripherals, which is a related but
# distinct luck signal (strand rate / sequencing, not just a FIP-formula
# quirk). Real case: Gray was labeled ELITE on a 2.82 ERA while his FIP
# (3.50) and SIERA (3.89) sit close enough to each other (0.39R gap) to
# clear the FIP-vs-SIERA check above -- but his ERA sits 0.68R below FIP
# and 1.07R below SIERA. His shiny ERA, not FIP, is what's actually driving
# the ELITE label and this pick's edge, and neither peripheral agrees with
# it. Same shape, same gap/cap magnitudes as the FIP-vs-xFIP/SIERA check
# above, just comparing ERA to FIP/SIERA instead.
_ERA_RELIABILITY_GAP = 0.50        # runs -- same bar as the FIP-vs-xFIP/SIERA check above
_ERA_RELIABILITY_SHAVE_MAX = round(0.05 * _MODEL_MARKET_BLEND_W, 4)  # BUG FIX (MLB-EDGE-BLEND-1, 2026-09-25): rescaled to the blended edge scale, was 0.05
def _era_reliability_shave(spnm):
    if not spnm:
        return 0.0, None
    _ev = mlb_pitcher_fip_lookup.get(_sp_norm_era(spnm), {}) if 'mlb_pitcher_fip_lookup' in globals() else {}
    try:
        _era = float(_ev.get('era_season')) if _ev.get('era_season') is not None else None
        _fip = float(_ev.get('fip')) if _ev.get('fip') is not None else None
        _siera = float(_ev.get('siera')) if _ev.get('siera') is not None else None
    except (TypeError, ValueError):
        return 0.0, None
    if _era is None:
        return 0.0, None
    _gaps = [v - _era for v in (_fip, _siera) if v is not None]
    _worst_gap = max(_gaps) if _gaps else 0.0
    if _worst_gap < _ERA_RELIABILITY_GAP:
        return 0.0, None
    #   # BUG FIX (MLB-EDGE-BLEND-1, 2026-09-25): rescaled to the blended edge scale
    _shave = min(_ERA_RELIABILITY_SHAVE_MAX, 0.05 * _MODEL_MARKET_BLEND_W * (_worst_gap - _ERA_RELIABILITY_GAP))
    return round(_shave, 4), round(_worst_gap, 2)

globals()["_era_reliability_shave"] = _era_reliability_shave
cand['_pick_erarel_shave'], cand['_pick_erarel_val'] = zip(*cand.apply(lambda r: _era_reliability_shave(_pick_sp_name(r)), axis=1))
cand['_opp_erarel_shave'],  cand['_opp_erarel_val']  = zip(*cand.apply(lambda r: _era_reliability_shave(_opp_sp_name(r)), axis=1))
_erarel_delta = cand['_pick_erarel_shave'] - (cand['_opp_erarel_shave'] * 0.5)
_erarel_active_mask = (_erarel_delta != 0.0) & (cand['passes_edge_rule'] == 1)
cand['edge'] = cand['edge'] - _erarel_delta
_erarel_now_fails = (cand['passes_edge_rule'] == 1) & (cand['edge'] < cand['_base_edge_min'])
cand.loc[_erarel_now_fails, 'passes_edge_rule'] = 0
cand.loc[_erarel_now_fails, '_gate_fail_reason'] = cand.loc[_erarel_now_fails, '_gate_fail_reason'].fillna(
    f"ERA RELIABILITY SHAVE: pick or opp SP ERA reads >= {_ERA_RELIABILITY_GAP}R better than FIP/SIERA -- ERA not a trustworthy number for this pitcher")
print(f'ERA reliability shave       : adjusted {int(_erarel_active_mask.sum())} play(s), newly blocked {int(_erarel_now_fails.sum())} (pick or opp SP ERA >= {_ERA_RELIABILITY_GAP}R better than FIP/SIERA)')

# SP EDGE THESIS INVALIDATION (2026-09-13, user audit -- Quantrill ELITE vs E-Rod STRONG,
# ARI ML +16.3%): the shave above discounts the edge at the margin but still lets the SP
# tier gap stand as the pick's primary thesis -- a pick built on "Quantrill ELITE vs E-Rod
# STRONG" doesn't actually change character just because its number got knocked down a few
# points. User's diagnosis: when the PICK's own SP's FIP reads >= _LARGE_FIP_GAP_THRESHOLD
# runs better than his real peripherals (E-Rod: FIP 3.76 vs xFIP 4.41/SIERA 4.64, a 0.88R
# gap), the tier gap the pick leans on "isn't real" -- at that point the play has to clear a
# HIGHER bar on whatever's left (bullpen, lineup, motivation) rather than leaning on an SP
# edge that was never trustworthy to begin with. Deliberately keyed on the PICK's own SP
# only (_pick_fiprel_val), not the opponent's -- an unreliable OPPONENT SP number doesn't
# undermine the pick's own thesis the same way an unreliable PICK SP number does.
_LARGE_FIP_GAP_THRESHOLD = 0.75    # runs -- matches the same-named threshold in the tier-label section above
_LARGE_FIP_GAP_PICK_EXTRA_SHAVE = round(0.06 * _MODEL_MARKET_BLEND_W, 4)  # was 0.06 on the raw-edge scale  # BUG FIX (MLB-EDGE-BLEND-1, 2026-09-25 user audit): rescaled to the blended edge scale (edge_blended = w*edge_raw when a real market price exists) so this mechanism's original relative selectivity survives the new, much-smaller edge scale instead of unintentionally becoming a near-automatic disqualifier.
_large_fip_gap_pick_mask = (cand['_pick_fiprel_val'].fillna(0) >= _LARGE_FIP_GAP_THRESHOLD) & (cand['passes_edge_rule'] == 1)
# Snapshot BEFORE this gate's own blocking below zeroes out passes_edge_rule for whatever it
# just caught -- the gate ledger's plays_evaluated is meant to answer "how big was the pool
# this gate looked at," not "what's left after it acted" (live-verified this distinction
# matters: a run where this gate caught the only qualifying play printed plays_evaluated=0,
# which reads as "the gate had nothing to check" when it actually had exactly one play and
# blocked it).
_plays_evaluated_snapshot = int((cand['passes_edge_rule'] == 1).sum())
cand.loc[_large_fip_gap_pick_mask, 'edge'] -= _LARGE_FIP_GAP_PICK_EXTRA_SHAVE
_large_fip_gap_now_fails = _large_fip_gap_pick_mask & (cand['edge'] < cand['_base_edge_min'])
cand.loc[_large_fip_gap_now_fails, 'passes_edge_rule'] = 0
cand.loc[_large_fip_gap_now_fails, '_gate_fail_reason'] = cand.loc[_large_fip_gap_now_fails, '_gate_fail_reason'].fillna(
    f"SP EDGE THESIS INVALIDATED: pick SP's FIP reads >= {_LARGE_FIP_GAP_THRESHOLD}R better than xFIP/SIERA -- "
    f"the SP tier gap this pick leaned on isn't real, play must clear a higher bar on secondary factors alone")
print(f'SP thesis invalidated shave : adjusted {int(_large_fip_gap_pick_mask.sum())} play(s), newly blocked {int(_large_fip_gap_now_fails.sum())} (pick SP FIP >= {_LARGE_FIP_GAP_THRESHOLD}R better than xFIP/SIERA)')
# GATE LEDGER (2026-09-13, user request -- "make it a structured log line rather than print
# output so it's queryable later... timestamp, plays evaluated, adjusted count, blocked
# count, pitcher names that fired. That becomes your gate ledger."). Permanent, quiet audit
# trail specifically for this gate -- exactly what's needed on hand when a future play gets
# blocked and the reason has to be explained without re-deriving it from scratch. First
# confirmed real-world activation (2026-09-13): Hayden Wesneski (Tampa Bay Rays), gap 0.90R,
# blocked the Rays ML play -- see this file going forward for every activation after.
_gate_log_path = TRACK_DIR / "mlb_gate_activations.csv"
_gate_log_row = pd.DataFrame([{
    # BUG FIX (live-verified: AttributeError, 'module datetime has no attribute now') --
    # `datetime` gets shadowed to the bare module (not the class) by a later `import
    # datetime` elsewhere in this notebook's shared global namespace. pd.Timestamp.now() is
    # unambiguous regardless of what `datetime` currently refers to.
    "run_timestamp": pd.Timestamp.now().isoformat(),
    "gate_name": "sp_edge_thesis_invalidation",
    "plays_evaluated": _plays_evaluated_snapshot,
    "adjusted_count": int(_large_fip_gap_pick_mask.sum()),
    "blocked_count": int(_large_fip_gap_now_fails.sum()),
    "pitchers_fired": "; ".join(
        f"{_pick_sp_name(cand.loc[_di])} (gap={cand.at[_di, '_pick_fiprel_val']:.2f}R)"
        for _di in cand.index[_large_fip_gap_pick_mask]
    ),
}])
_gate_log_row.to_csv(_gate_log_path, mode="a", header=not _gate_log_path.exists(), index=False)
print(f"[gate ledger] {_gate_log_path.name}: {int(_large_fip_gap_pick_mask.sum())} adjusted, "
      f"{int(_large_fip_gap_now_fails.sum())} blocked this run")

# FIX 2 SUPPORT (2026-09-06, user audit -- Alvarez ELITE tier despite FIP/
# SIERA divergence): shared single source of truth for "should this
# pitcher's printed ELO tier be downgraded" -- reuses _fip_reliability_shave
# itself (same FIP-vs-xFIP/SIERA gap, same >=0.50R bar) rather than
# re-deriving the gap a second time, so the tier label used by the SP
# Matchup line and the edge discount computed above can't drift apart.
# Registered globally (like _fip_reliability_shave/_sp_thin_starts above)
# because two separate print sections need it: the main OFFICIAL card's
# _pitcher_note() and the sub-threshold Value Watch section further down.
def _sp_tier_downgrade_gap(nm):
    # MLB-FIP-2 (2026-09-24 user audit): also check the new ERA-vs-FIP/
    # SIERA reliability shave, not just the original FIP-vs-xFIP/SIERA one
    # -- a pitcher can pass the FIP-vs-SIERA check (they read close to each
    # other) while still having an ERA that reads well below BOTH, which is
    # exactly the Gray case this ticket flagged (labeled ELITE off a 2.82
    # ERA, unsupported by a 3.50 FIP / 3.89 SIERA). Whichever check finds
    # the bigger gap wins, so the printed downgrade reflects the more
    # severe divergence.
    if not nm:
        return None
    _gaps_found = []
    for _fn_name in ("_fip_reliability_shave", "_era_reliability_shave"):
        _shave_fn = globals().get(_fn_name)
        if not _shave_fn:
            continue
        _shave, _gap = _shave_fn(nm)
        # BUG FIX (2026-09-25, found live -- crashed the whole run with
        # "ValueError: cannot convert float NaN to integer"): `_gap is
        # not None` doesn't catch a real float NaN (xFIP present, SIERA
        # missing, or vice versa, produces exactly this) -- NaN passed
        # through to max(_gaps_found), then into round() at the call
        # site, which raises on NaN specifically (unlike most float ops,
        # which just silently propagate NaN). Pre-existing bug, not
        # introduced by tonight's edge-blend changes -- just never hit a
        # real NaN-producing pitcher until now.
        if _shave and _shave > 0 and _gap is not None and _gap == _gap:
            _gaps_found.append(_gap)
    return max(_gaps_found) if _gaps_found else None

globals()["_sp_tier_downgrade_gap"] = _sp_tier_downgrade_gap

# FIX 1 (2026-09-06, user audit -- Nationals/Dodgers: Alvarez, a bulk/
# short-role arm averaging 3-5 IP/start all season, got his full-season
# ELITE(1633) SP ELO tier treated as if he were making a traditional
# start). Two things already discount for a short-leash starter upstream
# -- _eff_pit's run-environment blend (~line 700) and _sp_short_mult's
# win-probability nudge (~line 2800) -- but both are small, capped
# adjustments layered on top of the trained win model's own prediction,
# which still takes the raw, undiscounted season ELO as an input feature.
# Neither one reaches the GRADED edge% actually shown on the card. This
# shaves cand['edge'] itself, mirroring the HR9/sustained-cold/FIP-
# reliability pattern immediately above. Symmetric like the thin-sample
# gate (not directional like HR9/cold-streak) -- a short-leash starter
# makes the SP-vs-SP comparison less trustworthy regardless of which side
# he's on, so both sides ADD rather than one offsetting the other. A true
# opener (<3.0 IP/start -- essentially a piggyback plan from pitch one)
# is shaved harder than a bulk-role arm (3.0-5.0 IP/start) since his
# season ELO describes even less of what actually happens in the game.
# Both magnitudes are PROVISIONAL, sized into the same range as the other
# acute-risk shaves in this cell -- revisit after live grading.
_OPENER_EDGE_SHAVE = round(0.08 * _MODEL_MARKET_BLEND_W, 4)  # was 0.08 on the raw-edge scale  # BUG FIX (MLB-EDGE-BLEND-1, 2026-09-25 user audit): rescaled to the blended edge scale (edge_blended = w*edge_raw when a real market price exists) so this mechanism's original relative selectivity survives the new, much-smaller edge scale instead of unintentionally becoming a near-automatic disqualifier.
_BULK_ROLE_EDGE_SHAVE = round(0.05 * _MODEL_MARKET_BLEND_W, 4)  # was 0.05 on the raw-edge scale, rescaled with _OPENER_EDGE_SHAVE above
# NEW (MLB-ROLE-1, 2026-09-24 user audit): milder, separate shave for a
# genuine full-time starter who's had short outings lately WITHOUT a real
# opener/bulk usage pattern behind it (mlb_pitcher_fip_lookup's new
# is_short_outing_starter flag, Cell 20) -- meaningfully smaller than the
# bulk-role shave since this is a form/performance signal, not a
# structural workload one; his own FIP/ERA-based shaves already price the
# performance side of this, so this only covers the residual "his season
# ELO assumes deeper outings than he's actually been throwing" gap.
_SHORT_OUTING_EDGE_SHAVE = round(0.02 * _MODEL_MARKET_BLEND_W, 4)  # was 0.02 on the raw-edge scale, rescaled with _OPENER_EDGE_SHAVE above

def _sp_role_shave(spnm):
    if not spnm or 'mlb_pitcher_fip_lookup' not in globals():
        return 0.0, None
    _ev = mlb_pitcher_fip_lookup.get(_sp_norm_era(spnm), {})
    if bool(_ev.get('is_opener')):
        return _OPENER_EDGE_SHAVE, 'opener'
    if bool(_ev.get('is_bulk_role')):
        return _BULK_ROLE_EDGE_SHAVE, 'bulk'
    if bool(_ev.get('is_short_outing_starter')):
        return _SHORT_OUTING_EDGE_SHAVE, 'short_outing'
    return 0.0, None

globals()["_sp_role_shave"] = _sp_role_shave
cand['_pick_role_shave'], cand['_pick_role_kind'] = zip(*cand.apply(lambda r: _sp_role_shave(_pick_sp_name(r)), axis=1))
cand['_opp_role_shave'],  cand['_opp_role_kind']  = zip(*cand.apply(lambda r: _sp_role_shave(_opp_sp_name(r)), axis=1))
_role_delta = cand['_pick_role_shave'] + (cand['_opp_role_shave'] * 0.5)
_role_active_mask = (_role_delta != 0.0) & (cand['passes_edge_rule'] == 1)
cand['edge'] = cand['edge'] - _role_delta
_role_now_fails = (cand['passes_edge_rule'] == 1) & (cand['edge'] < cand['_base_edge_min'])
cand.loc[_role_now_fails, 'passes_edge_rule'] = 0
cand.loc[_role_now_fails, '_gate_fail_reason'] = cand.loc[_role_now_fails, '_gate_fail_reason'].fillna(
    "SP SHORT-ROLE RELIABILITY SHAVE: pick or opp SP is a confirmed opener/bulk-role arm -- season ELO overstates a traditional start")
print(f'SP short-role reliability shave: adjusted {int(_role_active_mask.sum())} play(s), newly blocked {int(_role_now_fails.sum())} (pick or opp SP is a confirmed opener/bulk-role arm -- season ELO overstates a traditional start)')

# M-NEW-3 EDGE SHAVE (2026-09-07, user audit -- "edge increased when it
# should have decreased" after Melton's tier downgraded ELITE -> STRONG):
# the tier-label downgrade above (in _sp_label) was built as a pure display
# change, deliberately mirroring Fix 2's scope -- but Fix 2 had real teeth
# because it reused an ALREADY-EXISTING edge shave (_fip_reliability_shave).
# M-NEW-3 never had an equivalent -- the downgrade was a warning label with
# zero effect on edge/win probability, which is exactly why the printed
# edge number could move independently of (and in the opposite direction
# from) the tier downgrade. Mirrors the HR9/FIP-reliability shave pattern:
# directional, not symmetric (the PICK's own SP struggling vs this specific
# opponent is bad for the pick; the OPPONENT's SP struggling vs the pick's
# team is good for the pick, so it reduces the opponent term's contribution
# rather than adding to it).
_SP_OPP_MISMATCH_SHAVE_MAX = round(0.05 * _MODEL_MARKET_BLEND_W, 4)  # was 0.05 on the raw-edge scale  # BUG FIX (MLB-EDGE-BLEND-1, 2026-09-25 user audit): rescaled to the blended edge scale (edge_blended = w*edge_raw when a real market price exists) so this mechanism's original relative selectivity survives the new, much-smaller edge scale instead of unintentionally becoming a near-automatic disqualifier.

def _sp_opp_mismatch_shave(spnm, opp_team):
    if not spnm or not opp_team or "_sp_vs_opponent_era" not in globals():
        return 0.0, None
    _era_vs_opp, _n_starts = _sp_vs_opponent_era(spnm, opp_team)
    if _era_vs_opp is None:
        return 0.0, None
    _fev = globals().get('mlb_pitcher_fip_lookup', {}).get(_sp_norm_era(spnm), {}) if 'mlb_pitcher_fip_lookup' in globals() else {}
    _season_era_raw = _fev.get('era_season')
    if _season_era_raw is None or pd.isna(_season_era_raw):
        return 0.0, None
    _season_era = float(_season_era_raw)
    _gap = _era_vs_opp - _season_era  # positive = pitches worse against this specific opponent
    if _gap <= _M_NEW3_ERA_GAP:
        return 0.0, None
    # BUG FIX (MLB-EDGE-BLEND-1): per-unit rate rescaled alongside the cap above.
    _shave = min(_SP_OPP_MISMATCH_SHAVE_MAX, 0.02 * _MODEL_MARKET_BLEND_W * (_gap - _M_NEW3_ERA_GAP))
    return round(_shave, 4), round(_gap, 2)

globals()["_sp_opp_mismatch_shave"] = _sp_opp_mismatch_shave

def _pick_sp_opp_team(row):
    _is_home = str(row.get('bet_team', '')).strip() == str(row.get(home_col, '')).strip()
    return row.get(away_col, '') if _is_home else row.get(home_col, '')

def _opp_sp_opp_team(row):
    return row.get('bet_team', '')

cand['_pick_opp_mismatch_shave'], cand['_pick_opp_mismatch_val'] = zip(*cand.apply(
    lambda r: _sp_opp_mismatch_shave(_pick_sp_name(r), _pick_sp_opp_team(r)), axis=1))
cand['_opp_opp_mismatch_shave'], cand['_opp_opp_mismatch_val'] = zip(*cand.apply(
    lambda r: _sp_opp_mismatch_shave(_opp_sp_name(r), _opp_sp_opp_team(r)), axis=1))
_opp_mismatch_delta = cand['_pick_opp_mismatch_shave'] - (cand['_opp_opp_mismatch_shave'] * 0.5)
_opp_mismatch_active_mask = (_opp_mismatch_delta != 0.0) & (cand['passes_edge_rule'] == 1)
cand['edge'] = cand['edge'] - _opp_mismatch_delta
_opp_mismatch_now_fails = (cand['passes_edge_rule'] == 1) & (cand['edge'] < cand['_base_edge_min'])
cand.loc[_opp_mismatch_now_fails, 'passes_edge_rule'] = 0
cand.loc[_opp_mismatch_now_fails, '_gate_fail_reason'] = cand.loc[_opp_mismatch_now_fails, '_gate_fail_reason'].fillna(
    f"SP OPPONENT-MISMATCH EDGE SHAVE: pick or opp SP ERA vs THIS specific opponent diverges >{_M_NEW3_ERA_GAP}R from season ERA")
print(f'SP opponent-mismatch edge shave: adjusted {int(_opp_mismatch_active_mask.sum())} play(s), newly blocked {int(_opp_mismatch_now_fails.sum())} (pick or opp SP ERA vs THIS opponent diverges >{_M_NEW3_ERA_GAP}R from season ERA)')

# POST-TRADE SPLIT EDGE SHAVE (2026-09-11, user request -- Dustin May: pre-
# trade FIP 2.88 (top-8 MLB) drove a 1538 ELO, but his post-trade ERA with
# Milwaukee (14 ER/19.2 IP, ~6.43) told a completely different story that
# never reached the card). Same lesson already learned live for M-NEW-3
# (see that gate's own comment above, 2026-09-09): a tier-label downgrade
# with no matching edge shave lets the printed edge number move
# independently of the tier -- so this one ships with both, from the
# start, same directional pattern as the opponent-mismatch shave just
# above (the PICK's own SP struggling with his new team is bad for the
# pick; the OPPONENT's SP struggling with theirs is good for the pick, at
# half weight).
_POST_TRADE_ERA_GAP = 1.0      # runs -- post-trade ERA worse than pre-trade/season baseline
_POST_TRADE_SHAVE_MAX = round(0.05 * _MODEL_MARKET_BLEND_W, 4)  # BUG FIX (MLB-EDGE-BLEND-1, 2026-09-25): rescaled to the blended edge scale, was 0.05

def _post_trade_shave(spnm):
    if not spnm or 'mlb_pitcher_fip_lookup' not in globals():
        return 0.0, None
    _fev = mlb_pitcher_fip_lookup.get(_sp_norm_era(spnm), {})
    if not bool(_fev.get('is_recently_traded')):
        return 0.0, None
    _post_era = _fev.get('post_trade_era')
    _pre_era = _fev.get('pre_trade_era')
    _season_era_raw = _fev.get('era_season')
    _compare_era = _pre_era if (_pre_era is not None and pd.notna(_pre_era)) else (
        float(_season_era_raw) if _season_era_raw is not None and pd.notna(_season_era_raw) else None)
    if _post_era is None or pd.isna(_post_era) or _compare_era is None:
        return 0.0, None
    _gap = float(_post_era) - float(_compare_era)
    if _gap <= _POST_TRADE_ERA_GAP:
        return 0.0, None
    #   # BUG FIX (MLB-EDGE-BLEND-1, 2026-09-25): rescaled to the blended edge scale
    _shave = min(_POST_TRADE_SHAVE_MAX, 0.02 * _MODEL_MARKET_BLEND_W * (_gap - _POST_TRADE_ERA_GAP))
    return round(_shave, 4), round(_gap, 2)

globals()['_post_trade_shave'] = _post_trade_shave
cand['_pick_posttrade_shave'], cand['_pick_posttrade_val'] = zip(*cand.apply(lambda r: _post_trade_shave(_pick_sp_name(r)), axis=1))
cand['_opp_posttrade_shave'], cand['_opp_posttrade_val'] = zip(*cand.apply(lambda r: _post_trade_shave(_opp_sp_name(r)), axis=1))
_posttrade_delta = cand['_pick_posttrade_shave'] - (cand['_opp_posttrade_shave'] * 0.5)
_posttrade_active_mask = (_posttrade_delta != 0.0) & (cand['passes_edge_rule'] == 1)
cand['edge'] = cand['edge'] - _posttrade_delta
_posttrade_now_fails = (cand['passes_edge_rule'] == 1) & (cand['edge'] < cand['_base_edge_min'])
cand.loc[_posttrade_now_fails, 'passes_edge_rule'] = 0
cand.loc[_posttrade_now_fails, '_gate_fail_reason'] = cand.loc[_posttrade_now_fails, '_gate_fail_reason'].fillna(
    f"POST-TRADE SPLIT SHAVE: pick or opp SP post-trade ERA diverges >{_POST_TRADE_ERA_GAP}R from pre-trade/season baseline")
print(f'Post-trade split shave        : adjusted {int(_posttrade_active_mask.sum())} play(s), newly blocked {int(_posttrade_now_fails.sum())} (pick or opp SP post-trade ERA diverges >{_POST_TRADE_ERA_GAP}R from baseline)')

# M-FIX-1/M-FIX-2 SHARED CONSTANTS (2026-09-08, user audit -- Drew Anderson
# STRONG tier on 35.3 IP): defined once here, reused both by this edge
# shave (M-FIX-2) and by the tier-label floor further down in this cell
# (M-FIX-1, inside _pitcher_note/_sp_label) so the two can't drift apart.
_SMALL_SAMPLE_IP_FLOOR = 40.0   # season IP -- below this, a tier above AVG isn't earned regardless of the raw number
_BULK_FIP_ERA_GAP = 0.75        # runs -- FIP reading this much worse than ERA on a short sample is a regression flag
_BULK_FIP_IP_MAX = 40.0         # same floor as M-FIX-1 -- both fixes target the identical "still building a real record" window
globals()["_SMALL_SAMPLE_IP_FLOOR"] = _SMALL_SAMPLE_IP_FLOOR
globals()["_BULK_FIP_ERA_GAP"] = _BULK_FIP_ERA_GAP
globals()["_BULK_FIP_IP_MAX"] = _BULK_FIP_IP_MAX

# M-FIX-2 (2026-09-08, user audit): bulk/short-role arm whose FIP reads
# meaningfully worse than his own realized ERA, on a season sample still
# under 40 IP -- a good ERA this early, on a short-leash arm, is exactly
# the profile most likely to be a sequencing/strand-rate fluke rather than
# real skill, not yet the more general FIP-vs-xFIP/SIERA reliability check
# above (that one asks "do the predictive peripherals agree with FIP" --
# this asks "does the pitcher's OWN results record agree with FIP", a
# different question, gated tighter (bulk-role + sub-40 IP only) since it's
# a stronger, more specific claim). Mirrors the same directional shave
# pattern as every other SP-quality gate in this cell.
_BULK_FIP_DIVERGENCE_SHAVE_MAX = round(0.05 * _MODEL_MARKET_BLEND_W, 4)  # BUG FIX (MLB-EDGE-BLEND-1, 2026-09-25): rescaled to the blended edge scale, was 0.05

def _sp_bulk_fip_divergence_shave(spnm):
    if not spnm or "mlb_pitcher_fip_lookup" not in globals():
        return 0.0, None
    _ev = mlb_pitcher_fip_lookup.get(_sp_norm_era(spnm), {})
    if not bool(_ev.get("is_bulk_role")):
        return 0.0, None
    _ip = _ev.get("ip_season")
    if _ip is None or pd.isna(_ip) or float(_ip) >= _BULK_FIP_IP_MAX:
        return 0.0, None
    _fip = _ev.get("fip")
    _era = _ev.get("era_season")
    if _fip is None or _era is None or pd.isna(_fip) or pd.isna(_era):
        return 0.0, None
    _gap = float(_fip) - float(_era)
    if _gap <= _BULK_FIP_ERA_GAP:
        return 0.0, None
    #   # BUG FIX (MLB-EDGE-BLEND-1, 2026-09-25): rescaled to the blended edge scale
    _shave = min(_BULK_FIP_DIVERGENCE_SHAVE_MAX, 0.02 * _MODEL_MARKET_BLEND_W * (_gap - _BULK_FIP_ERA_GAP))
    return round(_shave, 4), round(_gap, 2)

globals()["_sp_bulk_fip_divergence_shave"] = _sp_bulk_fip_divergence_shave

cand["_pick_bulkfip_shave"], cand["_pick_bulkfip_val"] = zip(*cand.apply(
    lambda r: _sp_bulk_fip_divergence_shave(_pick_sp_name(r)), axis=1))
cand["_opp_bulkfip_shave"], cand["_opp_bulkfip_val"] = zip(*cand.apply(
    lambda r: _sp_bulk_fip_divergence_shave(_opp_sp_name(r)), axis=1))
_bulkfip_delta = cand["_pick_bulkfip_shave"] - (cand["_opp_bulkfip_shave"] * 0.5)
_bulkfip_active_mask = (_bulkfip_delta != 0.0) & (cand["passes_edge_rule"] == 1)
cand["edge"] = cand["edge"] - _bulkfip_delta
_bulkfip_now_fails = (cand["passes_edge_rule"] == 1) & (cand["edge"] < cand["_base_edge_min"])
cand.loc[_bulkfip_now_fails, "passes_edge_rule"] = 0
cand.loc[_bulkfip_now_fails, '_gate_fail_reason'] = cand.loc[_bulkfip_now_fails, '_gate_fail_reason'].fillna(
    f"BULK-ROLE FIP DIVERGENCE SHAVE: bulk-role SP, FIP exceeds ERA by >{_BULK_FIP_ERA_GAP}R on a sub-{_BULK_FIP_IP_MAX:.0f} IP sample")
print(f'Bulk-role FIP divergence shave: adjusted {int(_bulkfip_active_mask.sum())} play(s), newly blocked {int(_bulkfip_now_fails.sum())} (bulk-role SP, FIP exceeds ERA by >{_BULK_FIP_ERA_GAP}R, sub-{_BULK_FIP_IP_MAX:.0f} IP)')

# Rookie/thin-MLB-sample uncertainty gate (2026-08-25, user audit -- Ethan
# Pecko): a pitcher with almost no real MLB track record (Pecko: 1 career
# start, coming off thoracic outlet surgery, 55% strikes in his debut) can
# still land in the AVG tier off the small-sample ELO regression in Cell 83
# -- that regression softens the ELO NUMBER but nothing downstream flagged
# the edge itself as resting on essentially zero real MLB information.
# Mirrors the post-IL/HR9 shave pattern above so it stacks with both.
_THIN_SAMPLE_MIN_STARTS = 5   # user-specified floor: fewer than ~5 MLB starts is UNRATED-level uncertainty
_LOW_STAB_MIN_IP = 40.0   # MLB-PROSPCALL-1 (2026-09-14): FIP/xFIP/SIERA (FanGraphs season rate stats, denominator ip_season) need roughly this many IP to stabilize -- single source of truth for both the card's LOW-STABILITY PERIPHERALS flag and the FIP-vs-Statcast display swap
_THIN_SAMPLE_EDGE_SHAVE_MAX = round(0.05 * _MODEL_MARKET_BLEND_W, 4)  # was 0.05 on the raw-edge scale  # BUG FIX (MLB-EDGE-BLEND-1, 2026-09-25 user audit): rescaled to the blended edge scale (edge_blended = w*edge_raw when a real market price exists) so this mechanism's original relative selectivity survives the new, much-smaller edge scale instead of unintentionally becoming a near-automatic disqualifier.
def _thin_sample_shave(spnm):
    if not spnm:
        return 0.0, None
    _tr = mlb_sp_tracker.get(spnm) if 'mlb_sp_tracker' in globals() else None
    _n_starts = (_tr or {}).get('starts', 0)
    if _n_starts >= _THIN_SAMPLE_MIN_STARTS:
        return 0.0, None
    _frac = 1.0 - (_n_starts / _THIN_SAMPLE_MIN_STARTS)
    return round(_THIN_SAMPLE_EDGE_SHAVE_MAX * _frac, 4), _n_starts

# 2026-08-29 (user audit M4): the shave above discounts the edge but leaves the
# play fully graded and OFFICIAL-eligible, and the card still printed a confident
# ELO tier for the pitcher. The audit asks for UNRATED, not a graded AVG. This
# helper is the single source of truth for "is this starter unrated"; the SP tier
# label and the grade/size cap in the Grade block below both read it, so the
# discount, the label and the OFFICIAL decision can't drift apart.
def _sp_thin_starts(spnm):
    """Career MLB starts if below the thin-sample floor, else None."""
    if not spnm:
        return None
    _tr = mlb_sp_tracker.get(spnm) if 'mlb_sp_tracker' in globals() else None
    if _tr is None:
        return None
    _n = _tr.get('starts')
    if _n is None:
        return None
    try:
        _n = int(_n)
    except (TypeError, ValueError):
        return None
    return _n if _n < _THIN_SAMPLE_MIN_STARTS else None

globals()["_sp_thin_starts"] = _sp_thin_starts
globals()["_thin_sample_shave"] = _thin_sample_shave
cand['_pick_thin_shave'], cand['_pick_thin_starts'] = zip(*cand.apply(lambda r: _thin_sample_shave(_pick_sp_name(r)), axis=1))
cand['_opp_thin_shave'],  cand['_opp_thin_starts']  = zip(*cand.apply(lambda r: _thin_sample_shave(_opp_sp_name(r)), axis=1))
# SIGN FIX (same session, live bug found verifying against tonight's real
# Astros @ Yankees card): thin-MLB-sample is a SYMMETRIC uncertainty signal,
# not a directional quality signal like HR9/post-IL -- not knowing how good
# Pecko really is should discount confidence in the game either way, not
# favor the side facing him. The post-il/hr9 pattern this was copied from
# subtracts the opponent-side term (so an opponent issue INCREASES edge --
# correct when the signal means "opponent is worse"). Copied verbatim here,
# it meant Pecko's own uncertainty was BOOSTING the Yankees' edge instead of
# discounting it -- exactly backwards from the fix's purpose. Both sides now
# add (always reduce edge), regardless of which team's SP is thin-sampled.
_thin_delta = cand['_pick_thin_shave'] + (cand['_opp_thin_shave'] * 0.5)
_thin_active_mask = (_thin_delta != 0.0) & (cand['passes_edge_rule'] == 1)
cand['edge'] = cand['edge'] - _thin_delta
_thin_now_fails = (cand['passes_edge_rule'] == 1) & (cand['edge'] < cand['_base_edge_min'])
cand.loc[_thin_now_fails, 'passes_edge_rule'] = 0
cand.loc[_thin_now_fails, '_gate_fail_reason'] = cand.loc[_thin_now_fails, '_gate_fail_reason'].fillna(
    f"THIN-MLB-SAMPLE EDGE SHAVE: pick or opp SP has < {_THIN_SAMPLE_MIN_STARTS} career MLB starts -- essentially zero real MLB information")
print(f'Thin-MLB-sample edge shave : adjusted {int(_thin_active_mask.sum())} play(s), newly blocked {int(_thin_now_fails.sum())} (pick or opp SP < {_THIN_SAMPLE_MIN_STARTS} career MLB starts)')

# SUB-40-IP HARD GATE (2026-09-20, user post-mortem -- Kade Anderson, 25.3 IP):
# the thin-sample shave above is keyed on START COUNT (_THIN_SAMPLE_MIN_STARTS=5), not
# innings -- a pitcher can clear 5 MLB starts (Anderson has) while still sitting on a
# tiny total-IP sample, so that gate returns ZERO shave even though the card's own
# LOW-STABILITY PERIPHERALS flag (_LOW_STAB_MIN_IP=40.0, same threshold already used for
# the FIP/xFIP/SIERA display swap elsewhere) is telling the reader his peripherals aren't
# stabilized. The display warning and the real edge math were disconnected -- a play could
# lean on a thin-IP starter's weakness as a driver while carrying a scary-looking flag with
# zero actual effect on the number. Below _LOW_STAB_MIN_IP, this starter's SP-tier read is
# unstable enough to treat his contribution to edge as near-replacement-level -- a hard
# shave sized to consume a typical SP-tier edge outright, not a soft scale-down like the
# thin-starts gate above. If the edge can't survive without him, the play should not
# survive either -- same mechanism as every other shave in this cell (subtract from edge,
# recheck the real floor, block if it no longer clears).
_LOW_STAB_HARD_SHAVE = round(0.12 * _MODEL_MARKET_BLEND_W, 4)  # was 0.12 on the raw-edge scale  # BUG FIX (MLB-EDGE-BLEND-1, 2026-09-25 user audit): rescaled to the blended edge scale (edge_blended = w*edge_raw when a real market price exists) so this mechanism's original relative selectivity survives the new, much-smaller edge scale instead of unintentionally becoming a near-automatic disqualifier.
def _low_stab_hard_shave(spnm):
    # BUG FIX (2026-09-20, live-verified on re-run: Kade Anderson at 32.3 IP,
    # well under the 40 IP floor, still got 0 shave): the original lookup used
    # _sp_norm_era() for the dict key, which doesn't match how mlb_pitcher_fip_lookup
    # is actually keyed/normalized elsewhere -- the proven LOW-STABILITY PERIPHERALS
    # print (which correctly shows Anderson's 32.3 IP every run) uses _mlb_name_norm()
    # with a fallback full-scan, plus a SECOND fallback to mlb_sp_tracker's
    # ip_starts sum when no FanGraphs ip_season record exists at all. Mirrors that
    # exact working lookup instead of guessing at a different one.
    if not spnm:
        return 0.0, None
    _nk = _mlb_name_norm(spnm) if '_mlb_name_norm' in globals() else str(spnm).lower()
    _ev = None
    if 'mlb_pitcher_fip_lookup' in globals():
        _ev = mlb_pitcher_fip_lookup.get(_nk)
        if _ev is None and '_mlb_name_norm' in globals():
            _ev = next((v for k, v in mlb_pitcher_fip_lookup.items() if _mlb_name_norm(k) == _nk), None)
    _ip = (_ev or {}).get('ip_season')
    if _ip is None or pd.isna(_ip):
        _tr = mlb_sp_tracker.get(spnm) if 'mlb_sp_tracker' in globals() else None
        _ip = sum((_tr or {}).get('ip_starts', []) or [])
    if not _ip or float(_ip) >= _LOW_STAB_MIN_IP:
        return 0.0, None
    return _LOW_STAB_HARD_SHAVE, float(_ip)

globals()['_low_stab_hard_shave'] = _low_stab_hard_shave
cand['_pick_lowstab_shave'], cand['_pick_lowstab_ip'] = zip(*cand.apply(lambda r: _low_stab_hard_shave(_pick_sp_name(r)), axis=1))
cand['_opp_lowstab_shave'],  cand['_opp_lowstab_ip']  = zip(*cand.apply(lambda r: _low_stab_hard_shave(_opp_sp_name(r)), axis=1))
_lowstab_delta = cand['_pick_lowstab_shave'] + (cand['_opp_lowstab_shave'] * 0.5)
_lowstab_active_mask = (_lowstab_delta != 0.0) & (cand['passes_edge_rule'] == 1)
cand['edge'] = cand['edge'] - _lowstab_delta
_lowstab_now_fails = (cand['passes_edge_rule'] == 1) & (cand['edge'] < cand['_base_edge_min'])
cand.loc[_lowstab_now_fails, 'passes_edge_rule'] = 0
cand.loc[_lowstab_now_fails, '_gate_fail_reason'] = cand.loc[_lowstab_now_fails, '_gate_fail_reason'].fillna(
    f"SUB-{_LOW_STAB_MIN_IP:.0f}-IP HARD GATE: pick or opp SP has too few MLB innings this season for a stable SP-tier read -- treated as near-replacement-level, not a graded quantity")
print(f'Sub-{_LOW_STAB_MIN_IP:.0f}-IP hard gate: adjusted {int(_lowstab_active_mask.sum())} play(s), newly blocked {int(_lowstab_now_fails.sum())} (pick or opp SP under {_LOW_STAB_MIN_IP:.0f} IP this season)')

# COORS SP-QUALITY UNVERIFIED GATE (2026-09-20, user post-mortem FIX 4): the
# [ALTITUDE] note (2026-08-04) already discloses that a Coors home game's
# run-environment factor (1.32x) is correctly applied to run projections/total,
# but is explicitly informational-only for the SP MATCHUP read specifically --
# a Rockies starter's own SEASON ERA/FIP is built from a home-park-heavy sample
# (half his starts all year at altitude), systematically inflated relative to a
# same-quality pitcher elsewhere, and nothing in this pipeline park-adjusts SP
# rate stats to correct for that. That's a real, distinct risk from the run-
# environment factor (which is already correct) -- the SP-quality READ itself
# can't be trusted at Coors until a park-adjusted ERA/FIP exists anywhere in this
# pipeline, which it does not yet. Treated as unverifiable and shaved out of the
# edge, same mechanism as every other data-quality shave in this cell -- the run
# projection (and its 1.32x factor) stands untouched, only the SP-quality
# contribution to edge is zeroed.
_COORS_SP_UNVERIFIED_SHAVE = round(0.06 * _MODEL_MARKET_BLEND_W, 4)   # was 0.06 on the raw-edge scale  # BUG FIX (MLB-EDGE-BLEND-1, 2026-09-25 user audit): rescaled to the blended edge scale (edge_blended = w*edge_raw when a real market price exists) so this mechanism's original relative selectivity survives the new, much-smaller edge scale instead of unintentionally becoming a near-automatic disqualifier.
def _cand_is_coors_game(row):
    _hm = str(row.get(home_col, '')).lower()
    return 'rockies' in _hm or 'colorado' in _hm

cand['_coors_game'] = cand.apply(_cand_is_coors_game, axis=1)
cand['_pick_sp_is_home_flag'] = cand.apply(_pick_sp_is_home, axis=1)
cand['_pick_coors_sp_shave'] = np.where(cand['_coors_game'] & cand['_pick_sp_is_home_flag'], _COORS_SP_UNVERIFIED_SHAVE, 0.0)
cand['_opp_coors_sp_shave']  = np.where(cand['_coors_game'] & ~cand['_pick_sp_is_home_flag'], _COORS_SP_UNVERIFIED_SHAVE, 0.0)
_coors_sp_delta = cand['_pick_coors_sp_shave'] + (cand['_opp_coors_sp_shave'] * 0.5)
_coors_sp_active_mask = (_coors_sp_delta != 0.0) & (cand['passes_edge_rule'] == 1)
cand['edge'] = cand['edge'] - _coors_sp_delta
_coors_sp_now_fails = (cand['passes_edge_rule'] == 1) & (cand['edge'] < cand['_base_edge_min'])
cand.loc[_coors_sp_now_fails, 'passes_edge_rule'] = 0
cand.loc[_coors_sp_now_fails, '_gate_fail_reason'] = cand.loc[_coors_sp_now_fails, '_gate_fail_reason'].fillna(
    "COORS SP-QUALITY UNVERIFIED: Rockies starter's own season ERA/FIP is home-Coors-heavy and not park-adjusted anywhere in this pipeline -- SP-quality read treated as unverifiable, not a graded edge driver")
print(f'Coors SP-quality unverified shave: adjusted {int(_coors_sp_active_mask.sum())} play(s), newly blocked {int(_coors_sp_now_fails.sum())} (Rockies starter involved in a Coors home game, no park-adjusted ERA available)')


# Gate 7: SP unresolved (2026-07-31) -- neither a live-confirmed nor a
# scheduled-probable starter name was found for one or both teams (see the
# Cell 41/42 SP-override block above). Distinct from Gate 6/sp_elo_weak:
# this fires when there is NO starter name at all, not just a name whose
# ELO isn't in mlb_pitcher_elo. Previously this fell through silently as a
# full candidate (BAL/PHI incident, 2026-07-30) -- now hard-blocked as
# PENDING LINEUP until a starter resolves.
_sp_unresolved_mask = (
    (cand['passes_edge_rule'] == 1)
    & (cand['sp_unresolved'] == True)
)
cand.loc[_sp_unresolved_mask, 'passes_edge_rule'] = 0
cand.loc[_sp_unresolved_mask, '_gate_fail_reason'] = cand.loc[_sp_unresolved_mask, '_gate_fail_reason'].fillna(
    "Gate 7 -- SP UNRESOLVED: no starter name found for pick or opponent -- PENDING LINEUP, hard-blocked (no real number to discount)")
print(f'SP unresolved gate       : blocked {int(_sp_unresolved_mask.sum())} play(s) (no starter name found for pick or opponent -- PENDING LINEUP)')

# Gate 7b: SP lookup failed (2026-08-07, MLB Priority 2 audit) -- a
# starter NAME was found (so Gate 7/sp_unresolved above doesn't fire),
# but that name matched nothing in mlb_pitcher_elo AND nothing in
# mlb_pitcher_fip_lookup/_sp_form_cache, so _get_sp_elo/_recency_adj
# silently fell all the way through to the bare 1460 unknown-pitcher
# default with zero real signal behind it (sp_elo_weak, computed
# earlier in this cell -- previously only ever counted in a console
# print, never gated). Distinct failure mode from Gate 7: there the
# problem is "we don't know who's starting," here it's "we know who's
# starting but have no rating/stats for them" -- both should hold the
# play rather than let a play run on a fabricated-neutral rating that
# looks identical to a real average starter.
_sp_lookup_failed_mask = (
    (cand['passes_edge_rule'] == 1)
    & (cand['sp_elo_weak'] == True)
)
cand.loc[_sp_lookup_failed_mask, 'passes_edge_rule'] = 0
cand.loc[_sp_lookup_failed_mask, '_gate_fail_reason'] = cand.loc[_sp_lookup_failed_mask, '_gate_fail_reason'].fillna(
    "Gate 7b -- SP LOOKUP FAILED: starter name found but no ELO/stats match anywhere -- fell through to a fabricated-neutral 1460 default, hard-blocked (no real number to discount)")
print(f'SP lookup-failed gate    : blocked {int(_sp_lookup_failed_mask.sum())} play(s) (starter name found but no ELO/stats match -- DEFAULT 1460 rating, cannot verify projection)')


# ── MLB-TOT-COMP-1: COMPONENT TOTALS PROJECTION + DECOMPOSITION ─────────────────
# 2026-10-04 user audit -- "MLB totals driver decomposition. The below-bar Over printed
# 'calibrated proj 8.4 vs line 7.0' as its driver, which is just the edge restated. Break every
# totals projection into its parts: each starter's expected runs allowed over projected innings,
# bullpen innings x bullpen run rate, park factor, offense vs league, and the postseason
# calibration adjustment" + "the raw projection was 9.5 runs with two sub-2.50 ERA starters, so the
# model isn't pricing SP quality into totals strongly enough on its own".
# What was found: the old totals number cannot be decomposed that way because the starter's own
# run prevention is not in it. pred_total_runs = (offense L10 + pitching)/2 x env + ~20 additive
# nudges, and "pitching" = 0.65 x starter_runs_allowed_10 + 0.35 x bullpen -- but
# starter_runs_allowed_10 (Cells 71/72) is the TEAM's whole-game runs allowed in that pitcher's last
# 10 starts (runs_allowed = merged home_runs/away_runs), bullpen and all. Live 2026-10-04: Logan
# Henderson (2.47 ERA, 0.84 WHIP) entered as 6.5 runs, Ray Kerr as 11.0, Snell 3.4, King 5.2 ->
# SD/MIL raw 9.5. And "bullpen" = 0.35 x team runs-allowed EWM (Cell 18), not reliever data.
# Rebuilt on 2025-26 (scripts/mlb_totals_component_model.py, point-in-time, train 2025 / test 2026):
#   old formula core OOS RMSE 4.549 -- WORSE than a flat league mean (4.499); component 4.467;
#   closing total 4.438. Neither carries information beyond the close (beyond-close coef -0.20,
#   SE 0.17) -> the totals edge vs the market is still not a demonstrated signal.
# Per side (team T bats vs pitching team P), every term in runs vs a league-average side:
#   c0                                     league base (4.41 R/side)
#   c_sp  x (SP R9 - 4.45) x IPexp / 9     starter: est runs/9 (0.5 ERA + 0.5 FIP, starts only,
#                                          shrunk 30 IP, x1.08 R/ER) over his expected innings
#   c_pen x (pen R9 - 4.45) x (9-IPexp)/9  bullpen: last-60d reliever runs/9 over the rest
#   c_off x (off ratio - 1) x 4.45         offense: park-neutral R/G vs league
#   c_pf  x (park factor - 1) x 4.45       park
#   elite adj                              facing an SP with season ERA < 3.00 (>= 40 IP): -0.19R
#                                          (n=1,180 sides 2025-26, SE 0.09 -- the fitted terms still
#                                          over-projected these by that much)
# + postseason run environment (MLB-TC-PS-1, -0.38R, PROVISIONAL), then the existing 35% market
# shrink toward the line. The coefficients are the train-fit values, NOT 1.0: each term is shrunk
# because its inputs are noisy (c_sp 0.66, c_pen 0.54, c_off 0.57, c_pf 0.69).
# IPexp (MLB-TUNEUP-1 extension, same audit -- "Henderson had 11 days' rest and has been innings-
# managed"): 0.86 + 0.648 x last-8-start mean (shrunk 3 starts to 5.3) + 0.188 x last-3 mean,
# minus a layoff haircut fitted on 7,376 starts: 10-13 days -0.19 IP, 14-29 -0.56, 30+ -0.66.
# An innings-managed arm's short recent starts flow straight in through the last-8/last-3 means.
# The bucket calibration table (Cell 84) was fitted to the OLD raw projection and is NOT applied
# to this one -- its own calibration is the fitted coefficients.
import json as _tc1_json, datetime as _tc1_dt, math as _tc1_math
import requests as _tc1_req
from pathlib import Path as _tc1_Path
_TC1_PATH = _tc1_Path(r"c:\Users\<user>\OneDrive\Documents\KAI Labs\sports\MLB\tracking\mlb_totals_component_params.json")
_TC1_TBD_PATH = _tc1_Path(r"c:\Users\<user>\OneDrive\Documents\KAI Labs\sports\MLB\tracking\mlb_tbd_pending.json")
mlb_totals_decomp = {}
mlb_sp_tbd_unrated = {}
_TC1 = None
try:
    _TC1 = _tc1_json.loads(_TC1_PATH.read_text())
except Exception as _tc1_e:
    print(f"  [TOTALS DECOMP] params file unreadable ({_tc1_e}) -- falling back to the old totals projection this run")
_TC1_PS_TYPES = {"F", "D", "L", "W"}


def _tc1_get(url, params=None):
    try:
        _r = _tc1_req.get(url, params=params, timeout=20)
        return _r.json() if _r.status_code == 200 else None
    except Exception:
        return None


def _tc1_ip(s):
    _w, _, _f = str(s or "0").partition(".")
    try:
        return float(_w or 0) + (float(_f) / 3 if _f else 0.0)
    except ValueError:
        return 0.0


def _tc1_sp_profile(pid, today, lg_era, cfip, k_ip=30.0, lg_sp_ip=5.3):
    """Point-in-time starter profile from the Stats API game log (starts only), same recipe as the backtest."""
    _yr = today.year
    _logs = []
    for _gt in ("R", "F,D,L,W"):
        _j = _tc1_get(f"https://statsapi.mlb.com/api/v1/people/{pid}/stats",
                      {"stats": "gameLog", "group": "pitching", "season": _yr, "gameType": _gt})
        for _s in ((_j or {}).get("stats") or [{}])[0].get("splits", []):
            if _s.get("date", "") < today.isoformat():
                _logs.append(_s)
    _logs.sort(key=lambda s: s.get("date", ""))
    _starts = [s for s in _logs if int(s["stat"].get("gamesStarted", 0) or 0) == 1]
    _apps = len(_logs)
    ip = er = hr = bb = k = 0.0
    for _s in _starts:
        _st = _s["stat"]
        ip += _tc1_ip(_st.get("inningsPitched")); er += float(_st.get("earnedRuns", 0) or 0)
        hr += float(_st.get("homeRuns", 0) or 0); k += float(_st.get("strikeOuts", 0) or 0)
        bb += float(_st.get("baseOnBalls", 0) or 0) + float(_st.get("hitByPitch", 0) or 0)
    _sea_ip, _sea_er = ip, er
    _prev = _tc1_get(f"https://statsapi.mlb.com/api/v1/people/{pid}/stats", {"stats": "season", "group": "pitching", "season": _yr - 1})
    for _s in ((_prev or {}).get("stats") or [{}])[0].get("splits", [])[:1]:
        _st = _s["stat"]; _w = _TC1.get("PREV_W", 0.5)
        ip += _w * _tc1_ip(_st.get("inningsPitched")); er += _w * float(_st.get("earnedRuns", 0) or 0)
        hr += _w * float(_st.get("homeRuns", 0) or 0); k += _w * float(_st.get("strikeOuts", 0) or 0)
        bb += _w * (float(_st.get("baseOnBalls", 0) or 0) + float(_st.get("hitByPitch", 0) or 0))
    if ip > 0:
        _est = 0.5 * (9 * er / ip) + 0.5 * ((13 * hr + 3 * bb - 2 * k) / ip + cfip)
    else:
        _est = lg_era
    _r9 = (ip * _est + k_ip * lg_era) / (ip + k_ip) * _TC1.get("r_per_er", 1.08)
    _l8 = [_tc1_ip(s["stat"].get("inningsPitched")) for s in _starts if s.get("gameType") == "R"][-8:]
    _l3 = _l8[-3:]
    _ipexp = (sum(_l8) + 3.0 * lg_sp_ip) / (len(_l8) + 3.0)
    _rest = (today - _tc1_dt.date.fromisoformat(_logs[-1]["date"])).days if _logs else None
    _opener = _apps >= 5 and len(_starts) < 0.5 * _apps     # mostly a reliever this season
    _ipx = (_TC1["ip_a"] + _TC1["ip_b"] * _ipexp + _TC1["ip_c"] * (sum(_l3) / len(_l3) if len(_l3) == 3 else _ipexp))
    _rest_d = 0.0
    for _lo, _hi, _d in _TC1["rest_ip"]:
        if _rest is not None and _lo <= _rest < _hi:
            _rest_d = _d
    _ipx = min(max(_ipx + _rest_d, 1.0), 7.5)
    if _opener:
        _ipx = 1.5
    return {"r9": _r9, "ipx": _ipx, "ip_season": _sea_ip, "era_season": (9 * _sea_er / _sea_ip) if _sea_ip > 0 else None,
            "l3": _l3, "l8_mean": (sum(_l8) / len(_l8)) if _l8 else None, "rest": _rest, "rest_d": _rest_d,
            "opener": _opener, "starts": len(_starts), "apps": _apps}


def _tc1_pen_avail(team_id, today):
    """MLB-PEN-AVAIL-1 (2026-10-04 user audit -- "Milwaukee used five relievers in Game 1, all at
    10+ pitches. The top-5 reliever bullpen should take availability from prior-day pitch counts").
    Reliever availability from the team's games in the last 2 days (box-score pitch counts):
    >= 30 pitches yesterday -> out (0); 15-29 yesterday AND pitched the day before -> out (0);
    10-29 yesterday, or pitched both of the last two days -> limited (0.5); else 1.0.
    PROVISIONAL -- standard bullpen-management thresholds, not fitted (no backtest)."""
    _out = {}
    _sch = _tc1_get("https://statsapi.mlb.com/api/v1/schedule", {"sportId": 1, "teamId": team_id,
                    "startDate": (today - _tc1_dt.timedelta(days=2)).isoformat(),
                    "endDate": (today - _tc1_dt.timedelta(days=1)).isoformat()})
    _by_day = {}
    for _d in (_sch or {}).get("dates", []):
        for _g in _d.get("games", []):
            if _g.get("status", {}).get("abstractGameState") != "Final":
                continue
            _bx = _tc1_get(f"https://statsapi.mlb.com/api/v1/game/{_g['gamePk']}/boxscore")
            if not _bx:
                continue
            _side = "home" if _bx["teams"]["home"]["team"]["id"] == team_id else "away"
            _t = _bx["teams"][_side]
            _pitchers = _t.get("pitchers", [])
            for _i, _pid in enumerate(_pitchers):
                if _i == 0:
                    continue                      # the starter
                _pl = _t["players"].get(f"ID{_pid}", {})
                _np = int(((_pl.get("stats") or {}).get("pitching") or {}).get("numberOfPitches", 0) or 0)
                _by_day.setdefault(_d["date"], {})[_pid] = (_np, _pl.get("person", {}).get("fullName"))
    _y = (today - _tc1_dt.timedelta(days=1)).isoformat(); _y2 = (today - _tc1_dt.timedelta(days=2)).isoformat()
    for _pid in set(_by_day.get(_y, {})) | set(_by_day.get(_y2, {})):
        _n1 = _by_day.get(_y, {}).get(_pid, (0, None))
        _n2 = _by_day.get(_y2, {}).get(_pid, (0, None))
        _nm = _n1[1] or _n2[1]
        if _n1[0] >= 30 or (_n1[0] >= 15 and _n2[0] > 0):
            _out[_pid] = (0.0, _nm, _n1[0], _n2[0])
        elif _n1[0] >= 10 or (_n1[0] > 0 and _n2[0] > 0):
            _out[_pid] = (0.5, _nm, _n1[0], _n2[0])
    return _out


def _tc1_pen(team_id, today, top5, days=60, k_ip=30.0):
    """(runs/9, detail). Regular season: whole reliever corps, last 60 days (same as the backtest).
    Postseason (top5=True): MLB-PEN-1 top-5 by leverage, re-picked from AVAILABLE arms, innings x availability."""
    _end = today - _tc1_dt.timedelta(days=1)
    _start = (_end - _tc1_dt.timedelta(days=days)).isoformat()
    _ro = _tc1_get(f"https://statsapi.mlb.com/api/v1/teams/{int(team_id)}/roster", {"rosterType": "active", "season": today.year})
    _pids = [p["person"]["id"] for p in (_ro or {}).get("roster", []) if p.get("position", {}).get("abbreviation") == "P"]
    if not _pids:
        return None, "no active roster"
    _pp = _tc1_get("https://statsapi.mlb.com/api/v1/people", {"personIds": ",".join(map(str, _pids)),
                   "hydrate": f"stats(group=pitching,type=byDateRange,startDate={_start},endDate={_end.isoformat()},gameType=R)"})
    _rel = []
    for _p in (_pp or {}).get("people", []):
        try:
            _st = _p["stats"][0]["splits"][0]["stat"]
        except Exception:
            continue
        _ip = _tc1_ip(_st.get("inningsPitched"))
        _gs, _gp = int(_st.get("gamesStarted", 0)), int(_st.get("gamesPlayed", 0))
        if _ip < 3 or _gp == 0 or _gs > 0.2 * _gp:
            continue
        _lev = int(_st.get("saves", 0)) + int(_st.get("holds", 0)) + int(_st.get("blownSaves", 0))
        _rel.append({"id": _p["id"], "name": _p.get("fullName"), "lev": _lev, "ip": _ip,
                     "r": float(_st.get("runs", _st.get("earnedRuns", 0)) or 0)})
    if len(_rel) < 5:
        return None, "< 5 qualifying relievers"
    _whole = sum(r["r"] for r in _rel) * 9 / sum(r["ip"] for r in _rel)
    if not top5:
        return _whole, f"whole pen {len(_rel)} arms, last {days}d"
    _av = _tc1_pen_avail(team_id, today)
    for r in _rel:
        r["w"] = _av.get(r["id"], (1.0,))[0]
    _ranked = sorted(_rel, key=lambda r: (-r["lev"], -r["ip"]))
    _top_all = _ranked[:5]
    _top = [r for r in _ranked if r["w"] > 0][:5]
    _wip = sum(r["ip"] * r["w"] for r in _top)
    _ra = sum(r["r"] / r["ip"] * 9 * r["ip"] * r["w"] for r in _top) / _wip if _wip else _whole
    _shr = (_wip * _ra + k_ip * _whole) / (_wip + k_ip)
    _txt = "top-5 by leverage (available): " + ", ".join(
        f"{r['name']}{' [LIMITED]' if r['w'] < 1 else ''}" for r in _top)
    _outs = [f"{r['name']} ({_av[r['id']][2]}p yday/{_av[r['id']][3]}p day before)" for r in _top_all if r["w"] == 0]
    _lims = [f"{r['name']} ({_av[r['id']][2]}p yday/{_av[r['id']][3]}p day before)" for r in _top if r["w"] < 1]
    if _outs:
        _txt += " | UNAVAILABLE: " + ", ".join(_outs)
    if _lims:
        _txt += " | LIMITED (half weight): " + ", ".join(_lims)
    return _shr, _txt


if _TC1:
    _tc1_today = _tc1_dt.date.today()
    _tc1_sched = _tc1_get("https://statsapi.mlb.com/api/v1/schedule",
                          {"sportId": 1, "date": _tc1_today.isoformat(), "hydrate": "probablePitcher"})
    _tc1_hit = _tc1_get("https://statsapi.mlb.com/api/v1/teams/stats",
                        {"stats": "season", "group": "hitting", "season": _tc1_today.year, "sportIds": 1, "gameType": "R"})
    _tc1_pit = _tc1_get("https://statsapi.mlb.com/api/v1/teams/stats",
                        {"stats": "season", "group": "pitching", "season": _tc1_today.year, "sportIds": 1, "gameType": "R"})
    _tc1_rpg = {}
    for _s in ((_tc1_hit or {}).get("stats") or [{}])[0].get("splits", []):
        _g = int(_s["stat"].get("gamesPlayed", 0) or 0)
        if _g:
            _tc1_rpg[_s["team"]["name"].lower()] = (float(_s["stat"].get("runs", 0)), _g)
    _tc1_lg_rpg = (sum(v[0] for v in _tc1_rpg.values()) / sum(v[1] for v in _tc1_rpg.values())) if _tc1_rpg else 4.45
    _tsum = {"ip": 0.0, "er": 0.0, "hr": 0.0, "bb": 0.0, "k": 0.0}
    for _s in ((_tc1_pit or {}).get("stats") or [{}])[0].get("splits", []):
        _st = _s["stat"]
        _tsum["ip"] += _tc1_ip(_st.get("inningsPitched")); _tsum["er"] += float(_st.get("earnedRuns", 0) or 0)
        _tsum["hr"] += float(_st.get("homeRuns", 0) or 0); _tsum["k"] += float(_st.get("strikeOuts", 0) or 0)
        _tsum["bb"] += float(_st.get("baseOnBalls", 0) or 0) + float(_st.get("hitByPitch", 0) or 0)
    if _tsum["ip"] > 0:
        _tc1_lg_era = 9 * _tsum["er"] / _tsum["ip"]
        _tc1_cfip = _tc1_lg_era - (13 * _tsum["hr"] + 3 * _tsum["bb"] - 2 * _tsum["k"]) / _tsum["ip"]
    else:
        _tc1_lg_era, _tc1_cfip = 4.15, 3.1
    print(f"\n[TOTALS DECOMP] component totals model (MLB-TOT-COMP-1) -- league {_tc1_lg_rpg:.2f} R/G, ERA {_tc1_lg_era:.2f}, cFIP {_tc1_cfip:.2f}")
    _tc1_pen_cache = {}
    _LG = _TC1["lg_r9"]
    for _d in (_tc1_sched or {}).get("dates", []):
        for _g in _d.get("games", []):
            if _g.get("status", {}).get("abstractGameState") != "Preview":
                continue
            _ta, _th = _g["teams"]["away"], _g["teams"]["home"]
            _an, _hn = _ta["team"]["name"], _th["team"]["name"]
            _is_ps = _g.get("gameType") in _TC1_PS_TYPES
            _pf_h = float(_PARK_FACTOR.get(_norm_team(_hn), 1.0)) if "_PARK_FACTOR" in globals() else 1.0
            _prof, _tbd = {}, []
            for _sn, _tm in (("away", _ta), ("home", _th)):
                _pp = _tm.get("probablePitcher") or {}
                _ann = (globals().get("mlb_announced_roles") or {}).get(_tm["team"]["name"].lower())
                if _ann and _ann.get("bulk_id"):
                    # MLB-PS-OPENER-NEWS-1: announced opener + named bulk arm -> the BULK ARM is graded as the SP
                    # (his own innings expectation); the opener's inning or two go to the bullpen term
                    _prof[_sn] = _tc1_sp_profile(_ann["bulk_id"], _tc1_today, _tc1_lg_era, _tc1_cfip, k_ip=_TC1.get("K_SP_IP", 30.0))
                    _prof[_sn]["ipx"] = min(_prof[_sn]["ipx"], 7.5)
                    _prof[_sn]["opener"] = False
                    # MLB-PS-BULK-PRICE-1: short opener term -- the opener's own rate for ~1.5 IP, bulk arm after him
                    _op_prof = _tc1_sp_profile(_pp["id"], _tc1_today, _tc1_lg_era, _tc1_cfip, k_ip=_TC1.get("K_SP_IP", 30.0)) if _pp.get("id") else None
                    _prof[_sn]["op_ipx"] = 1.5
                    _prof[_sn]["op_r9"] = _op_prof["r9"] if _op_prof else None
                    _prof[_sn]["ipx"] = min(_prof[_sn]["ipx"], 9.0 - 1.5 - 0.5)
                    _prof[_sn]["name"] = f"{_ann['bulk']} (bulk; {_ann['opener']} opens)"
                elif _pp.get("id"):
                    _prof[_sn] = _tc1_sp_profile(_pp["id"], _tc1_today, _tc1_lg_era, _tc1_cfip, k_ip=_TC1.get("K_SP_IP", 30.0))
                    _prof[_sn]["name"] = _pp.get("fullName")
                else:
                    _prof[_sn] = None
                    _tbd.append(_sn)
                _tid = _tm["team"]["id"]
                if (_tid, _is_ps) not in _tc1_pen_cache:
                    _tc1_pen_cache[(_tid, _is_ps)] = _tc1_pen(_tid, _tc1_today, _is_ps)
            _sides, _parts = {}, []
            _total = 0.0
            for _bat, _pit, _bt, _pt in (("away", "home", _ta, _th), ("home", "away", _th, _ta)):
                _sp = _prof[_pit]
                _pen_r9, _pen_txt = _tc1_pen_cache[(_pt["team"]["id"], _is_ps)]
                if _pen_r9 is None:
                    _pen_r9 = 4.2
                    _pen_txt = f"league-average fallback ({_pen_txt})"
                if _sp is None:            # TBD -> bullpen-game prior for the number, side UNRATED below
                    _sp_r9, _ipx = _pen_r9, 0.0
                else:
                    _sp_r9, _ipx = _sp["r9"], _sp["ipx"]
                _rpg, _gp = _tc1_rpg.get(_bt["team"]["name"].lower(), (_tc1_lg_rpg, 1))
                _pf_own = float(_PARK_FACTOR.get(_norm_team(_bt["team"]["name"]), 1.0)) if "_PARK_FACTOR" in globals() else 1.0
                _off = (_rpg / _gp) / ((_pf_own + 3) / 4) / _tc1_lg_rpg
                _op_ip = (_sp or {}).get("op_ipx") or 0.0
                _op_r9 = (_sp or {}).get("op_r9")
                _op_ip = _op_ip if _op_r9 is not None else 0.0
                _c = {
                    "sp": _TC1["c_sp"] * ((_sp_r9 - _LG) * _ipx + ((_op_r9 - _LG) * _op_ip if _op_ip else 0.0)) / 9,
                    "pen": _TC1["c_pen"] * (_pen_r9 - _LG) * (9 - _ipx - _op_ip) / 9,
                    "off": _TC1["c_off"] * (_off - 1) * _LG,
                    "pf": _TC1["c_pf"] * (_pf_h - 1) * _LG,
                    "home": _TC1["c_home"] * (1.0 if _bat == "home" else 0.0),
                    "elite": (_TC1["elite_adj"] if (_sp and _sp["era_season"] is not None and _sp["era_season"] < _TC1["elite_era"]
                                                    and _sp["ip_season"] >= _TC1["elite_min_ip"]) else 0.0),
                }
                _runs = _TC1["c0"] + sum(_c.values())
                _total += _runs
                _sides[_bat] = {"runs": _runs, "c": _c, "sp": _sp, "sp_r9": _sp_r9, "ipx": _ipx, "pen_r9": _pen_r9,
                                "pen_txt": _pen_txt, "off": _off, "team": _bt["team"]["name"], "opp": _pt["team"]["name"]}
            _ps_d = _TC1.get("ps_run_env_delta", -0.38) if _is_ps else 0.0
            _total += _ps_d
            # ---- decomposition lines (each term in runs vs a league-average game) ----
            _base = 2 * _TC1["c0"]
            _parts.append(("league base", _base, f"2 x {_TC1['c0']:.2f}"))
            for _bat, _pit in (("away", "home"), ("home", "away")):
                _s = _sides[_bat]; _sp = _s["sp"]
                _who = _sp["name"] if _sp else "TBD"
                _sp_desc = (f"{_who} ({_s['opp']}) {_s['sp_r9']:.2f} R9 x {_s['ipx']:.1f} IP"
                            + (f" [L3 {', '.join(f'{x:.1f}' for x in _sp['l3'])}; {_sp['rest']}d rest"
                               + (f", layoff {_sp['rest_d']:+.2f} IP" if _sp["rest_d"] else "") + "]" if _sp and _sp["l3"] else "")
                            + (" [OPENER -- 1.5 IP, rest at bullpen rate]" if _sp and _sp["opener"] else "")
                            + (" [NO PROBABLE -- bullpen-game prior]" if not _sp else ""))
                _parts.append((f"SP {_who}", _s["c"]["sp"], _sp_desc))
                _opi = ((_sp or {}).get("op_ipx") or 0.0) if (_sp or {}).get("op_r9") is not None else 0.0
                if _opi:
                    _parts[-1] = (_parts[-1][0], _parts[-1][1], _parts[-1][2] + f" + opener {_opi:.1f} IP at {_sp['op_r9']:.2f} R9")
                _parts.append((f"{_s['opp']} pen", _s["c"]["pen"], f"{_s['pen_r9']:.2f} R9 x {9 - _s['ipx'] - _opi:.1f} IP ({_s['pen_txt']})"))
                _parts.append((f"{_s['team']} offense", _s["c"]["off"], f"{_s['off']:.2f}x league R/G (park-neutral)"))
                if _s["c"]["elite"]:
                    _parts.append((f"elite-SP adj ({_who})", _s["c"]["elite"],
                                   f"season ERA {_sp['era_season']:.2f} < {_TC1['elite_era']:.2f} -- fitted terms still over-project these by {abs(_TC1['elite_adj']):.2f}R"))
            _parts.append(("park", _sides["away"]["c"]["pf"] + _sides["home"]["c"]["pf"], f"PF {_pf_h:.2f}"))
            if abs(_TC1["c_home"]) >= 0.01:
                _parts.append(("home field", _TC1["c_home"], ""))
            if _ps_d:
                _parts.append(("postseason run env", _ps_d, "MLB-TC-PS-1, PROVISIONAL (SE 0.30)"))
            _key = (_an.lower().strip(), _hn.lower().strip())
            _elite_n = sum(1 for _sn in ("away", "home") if _sides[_sn]["c"]["elite"])
            mlb_totals_decomp[_key] = {"total": _total, "parts": _parts, "sides": _sides, "is_ps": _is_ps, "tbd": list(_tbd),
                                       "elite_n": _elite_n, "game_pk": _g["gamePk"], "game_time": _g.get("gameDate")}
            if _tbd:
                _tbd_txt = ("SP TBD -- no probable posted for " + " and ".join(
                    (_an if s == "away" else _hn) for s in _tbd) + " when this card was priced; side UNRATED on every market "
                    "until a starter is announced (MLB-TBD-RERUN-1 re-runs the card when it is)")
                mlb_sp_tbd_unrated[_key] = _tbd_txt
            print(f"  {_an} @ {_hn}: component total {_total:.2f}" + (" [POSTSEASON]" if _is_ps else "")
                  + (f"  !! {mlb_sp_tbd_unrated[_key]}" if _tbd else ""))
            for _lab, _v, _why in _parts:
                print(f"      {_v:+6.2f}  {_lab:<34} {_why}")
    # pending-starter file for scripts/mlb_starter_watch.py (rewritten every run)
    try:
        _TC1_TBD_PATH.write_text(_tc1_json.dumps({
            "card_date": _tc1_today.isoformat(), "written": _tc1_dt.datetime.now().isoformat(timespec="seconds"),
            "games": [{"gamePk": v["game_pk"], "matchup": f"{k[0].title()} @ {k[1].title()}", "sides": v["tbd"],
                       "game_time": v["game_time"]} for k, v in mlb_totals_decomp.items() if v["tbd"]]}, indent=2))
    except Exception as _tc1_we:
        print(f"  [MLB-TBD-RERUN-1] could not write pending file ({_tc1_we})")
globals()["mlb_totals_decomp"] = mlb_totals_decomp
globals()["mlb_sp_tbd_unrated"] = mlb_sp_tbd_unrated

# Gate 7d: POSTSEASON SP ROLE UNRATED (MLB-PS-SPROLE-1, 2026-10-01) -- an
# elimination game where either side can piggyback a rotation starter behind
# today's probable. Same hard-stop class as Gate 7/7b ("no real number"), not
# a shave: the innings split is unknowable pregame. Keyed off the real
# per-matchup map the run-projection loop wrote (board columns don't reliably
# survive into cand -- see the _emit_why cross-reference note in Cell 89).
_ps_unr_map = globals().get("mlb_ps_role_unrated", {}) or {}
_ps_unr_note = cand.apply(lambda r: _ps_unr_map.get((str(r.get(away_col, "")).lower().strip(),
                                                      str(r.get(home_col, "")).lower().strip())), axis=1)
_ps_unrated_mask = (cand['passes_edge_rule'] == 1) & _ps_unr_note.notna()
cand.loc[_ps_unrated_mask, 'passes_edge_rule'] = 0
cand.loc[_ps_unrated_mask, '_gate_fail_reason'] = cand.loc[_ps_unrated_mask, '_gate_fail_reason'].fillna(
    "Gate 7d -- POSTSEASON SP ROLE UNRATED: " + _ps_unr_note[_ps_unrated_mask].astype(str))
print(f'Postseason SP-role gate  : blocked {int(_ps_unrated_mask.sum())} play(s) (announced opener, or elimination game with piggyback-capable rotation arms -- SP role UNRATED)')

# Gate 7f: SP TBD -> UNRATED (MLB-TBD-RERUN-1, 2026-10-04 user audit -- "The ATL@LAD Over was priced
# at 9 AM with the Braves starter TBD ... missing SP -> bullpen-game prior or UNRATED, never a full
# edge"). A side with NO probable posted on the Stats API at pricing time is UNRATED on every
# market (ML/RL here, O/U in the totals selector), in any game -- not just the postseason. The
# card is re-run by scripts/mlb_starter_watch.py once the starter is announced (pending games are
# written to tracking/mlb_tbd_pending.json by the component-totals block above).
_tbd_map = globals().get("mlb_sp_tbd_unrated", {}) or {}
_tbd_note = cand.apply(lambda r: _tbd_map.get((str(r.get(away_col, "")).lower().strip(),
                                                str(r.get(home_col, "")).lower().strip())), axis=1)
_tbd_mask = (cand['passes_edge_rule'] == 1) & _tbd_note.notna()
cand.loc[_tbd_mask, 'passes_edge_rule'] = 0
cand.loc[_tbd_mask, '_gate_fail_reason'] = cand.loc[_tbd_mask, '_gate_fail_reason'].fillna(
    "Gate 7f -- " + _tbd_note[_tbd_mask].astype(str))
print(f'SP TBD gate              : blocked {int(_tbd_mask.sum())} play(s) (no probable posted at pricing time -- UNRATED until announced)')

# Gate 7e: DIRECTION CONFLICT (raw) -- MLB-DIRECTION-1 (2026-10-01 user audit,
# mirrors NFL-FIX-DIRECTION-1 / its 2026-09-29 raw-probability version). Real
# case: model had Atlanta at 69.9% pre-blend while the market had Philadelphia
# -118. MIN_EDGE-style checks measure how MUCH the model disagrees, not that it
# picks a different winner. Checked on the PRE-blend probability (the blend
# pulls toward the market and can hide a flip) against the no-vig market. Same
# consequence as NFL: the whole game is capped at PARTIAL -- no official play
# on any market for it (ML/RL here, O/U in the totals selector).
cand["_direction_conflict_raw"] = (
    cand["p_bet_win_model_raw"].notna() & cand["market_novig_prob"].notna()
    & (cand["p_bet_win_model_raw"] > 0.50) & (cand["market_novig_prob"] < 0.50))
mlb_direction_conflict_games = {}
for _, _dcr in cand[cand["_direction_conflict_raw"]].iterrows():
    _dk = (str(_dcr.get(away_col, "")).lower().strip(), str(_dcr.get(home_col, "")).lower().strip())
    mlb_direction_conflict_games[_dk] = (
        f"DIRECTION CONFLICT (raw): model has {_dcr.get('bet_team')} at {float(_dcr['p_bet_win_model_raw']):.1%} "
        f"pre-blend, market has them at {float(_dcr['market_novig_prob']):.1%} no-vig -- model and market "
        f"disagree on the winner")
    print(f"  !! {_dcr.get(away_col)} @ {_dcr.get(home_col)}: {mlb_direction_conflict_games[_dk]}")
globals()["mlb_direction_conflict_games"] = mlb_direction_conflict_games
_dc_note = cand.apply(lambda r: mlb_direction_conflict_games.get(
    (str(r.get(away_col, "")).lower().strip(), str(r.get(home_col, "")).lower().strip())), axis=1)
_dc_mask = (cand["passes_edge_rule"] == 1) & _dc_note.notna()
cand.loc[_dc_mask, "passes_edge_rule"] = 0
cand.loc[_dc_mask, "_gate_fail_reason"] = cand.loc[_dc_mask, "_gate_fail_reason"].fillna(
    "Gate 7e -- " + _dc_note[_dc_mask].astype(str) + " -- integrity PARTIAL, no official play on this game")
print(f'Direction-conflict gate  : flagged {len(mlb_direction_conflict_games)} game(s), blocked {int(_dc_mask.sum())} play(s)')

# Gate 8: SP ELO-ERA divergence (audit fix, 2026-08-04) -- the "!! ELO-ERA
# DIVERGENCE" bullet elsewhere in this cell / Cell 89's Final Card only ever
# printed a warning; it never touched passes_edge_rule, so a play could
# still clear OFFICIAL with the divergence unresolved. Real case: Melton
# rated ELO 1515 vs a 1.78 ERA implying ~1908 (393pt gap) -- the gap exists
# because a thin-IP sample got regressed toward league average by the
# _recency_adj small-sample logic above (the right call), but nothing
# downstream forced the play's edge to reflect that distrust. Hard-blocks
# (no edge override, same pattern as the walk-rate gate) whenever either
# side's rated ELO and season-ERA-implied ELO disagree by 120+ points (same
# bar the display bullet already uses) AND that pitcher's season IP is still
# under _SP_STALE_MIN_IP -- the same "not enough IP to trust this number
# yet" bar Gate 6 uses for post-IL returns. Once IP accumulates, the gate
# clears itself even with the same raw ERA.
_ELO_ERA_GAP_FLAG = 120.0

def _cand_elo_era_gap(spnm):
    if not spnm or "_era_implied_elo" not in dir():
        return None
    _elo = _sp_elo_gate(spnm)
    if _elo is None or _elo == 1500.0:
        return None
    _fev = globals().get('mlb_pitcher_fip_lookup', {}).get(_norm_name(spnm), {})
    _era = _fev.get('era_season')
    if _era is None or pd.isna(float(_era)) or not (0 < float(_era) <= 12):
        return None
    _implied = _era_implied_elo(float(_era))
    if _implied is None:
        return None
    _gap = abs(float(_elo) - _implied)
    if _gap < _ELO_ERA_GAP_FLAG:
        return None
    _ip = _fev.get('ip_season')
    _ip_val = float(_ip) if _ip is not None and not pd.isna(float(_ip)) else 0.0
    if _ip_val >= _SP_STALE_MIN_IP:
        return None  # enough IP to trust the ELO/ERA reconciliation either way
    return _gap

cand['_pick_sp_elo_era_gap'] = cand.apply(lambda r: _cand_elo_era_gap(_pick_sp_name(r)), axis=1)
cand['_opp_sp_elo_era_gap']  = cand.apply(lambda r: _cand_elo_era_gap(_opp_sp_name(r)), axis=1)
cand['_elo_era_divergent'] = cand['_pick_sp_elo_era_gap'].notna() | cand['_opp_sp_elo_era_gap'].notna()
_elo_era_mask = (cand['passes_edge_rule'] == 1) & cand['_elo_era_divergent']
cand.loc[_elo_era_mask, 'edge'] = cand.loc[_elo_era_mask, 'edge'] - _GATE8_ELOERA_KNOCKDOWN

# Net SP-quality discount actually applied above (HR9 through Gate 8),
# per candidate bet_team -- see the snapshot comment above. Looked up by
# the RUN LINE section further down so a collapsing/thin/mismatched
# starter discounts the RL probability the same way it already discounts
# the ML edge, instead of only ever showing up on one of the two markets.
cand['_sp_quality_total_shave'] = cand['_edge_pre_sp_quality'] - cand['edge']
_sp_quality_shave_by_team = dict(zip(
    cand['bet_team'].astype(str).str.lower(), cand['_sp_quality_total_shave']
)) if 'bet_team' in cand.columns else {}
globals()['_sp_quality_shave_by_team'] = _sp_quality_shave_by_team
_elo_era_now_fails = _elo_era_mask & (cand['edge'] < cand['_base_edge_min'])
cand.loc[_elo_era_now_fails, 'passes_edge_rule'] = 0
cand.loc[_elo_era_now_fails, '_gate_fail_reason'] = cand.loc[_elo_era_now_fails, '_gate_fail_reason'].fillna(
    f"Gate 8 -- ELO-ERA DIVERGENCE: pick or opp SP's rated ELO and season-ERA-implied ELO disagree {_ELO_ERA_GAP_FLAG:.0f}+ points on a sub-{_SP_STALE_MIN_IP:.0f} IP sample "
    f"(edge shaved -{_GATE8_ELOERA_KNOCKDOWN:.0%}, now below required floor)")
print(f'ELO-ERA divergence gate  : shaved edge -{_GATE8_ELOERA_KNOCKDOWN:.0%} on {int(_elo_era_mask.sum())} play(s), newly blocked {int(_elo_era_now_fails.sum())} (pick or opp SP ELO/ERA disagree {_ELO_ERA_GAP_FLAG:.0f}+pts on a sub-{_SP_STALE_MIN_IP:.0f} IP sample)')

# Gate 9: extreme season-record gap (audit fix, 2026-08-04) -- the
# informational [RECORD GAP]/[TEAM RECORD GAP] flags elsewhere in this cell
# only ever printed after the pick was already fixed, and the team-record
# credibility weight above only nudges pred_home_col by a capped 4pp --
# nowhere near enough to move a play off OFFICIAL when the gap is this
# extreme. Real case: Colorado Rockies ML cleared OFFICIAL against Tampa Bay
# despite a 19pp win% gap (TB .589 vs COL .398) -- the model was effectively
# betting a sub-.400 team to beat a .589 team off a single day's SP matchup
# alone, with no additional evidence required. Hard-blocks (no edge
# override) whenever the picked team's own season win% is BELOW
# _RECORD_GAP_DOG_MAX and the opponent's is ABOVE _RECORD_GAP_FAV_MIN --
# picking a real bottom-tier team against a real top-tier one needs stronger
# proof than "the model's own gates already cleared it," which is exactly
# what let Colorado through.
_RECORD_GAP_DOG_MAX = 0.430
_RECORD_GAP_FAV_MIN = 0.560

def _cand_record_gap_extreme(row):
    if "_trw_records" not in globals() or not _trw_records:
        return False
    def _rg_lookup(team_name):
        if not team_name:
            return None
        # BUG FIX (MLB-RECORD-1, 2026-09-24 user audit): this used to do its
        # own inline "any word >3 chars matches a substring" lookup, which
        # readily confuses same-city teams that share a city word -- Los
        # Angeles Dodgers vs Los Angeles Angels both contain "angeles", New
        # York Yankees vs New York Mets both contain "york", Chicago Cubs vs
        # Chicago White Sox both contain "chicago" -- whichever team's dict
        # entry iterates first wins, silently attaching the WRONG team's
        # win% to this one. _trw_lookup_generic (built earlier in this same
        # cell for the playoff-odds/road-record lookups) already matches on
        # the team's actual last-word nickname instead ("dodgers" vs
        # "angels"), which can't collide on a shared city name -- reusing it
        # here instead of keeping a second, unsafe copy of the same idea.
        return _trw_lookup_generic(_trw_records, team_name)
    _pick_team = row.get("bet_team", "")
    _is_home = str(_pick_team).strip() == str(row.get(home_col, "")).strip()
    _opp_team = row.get(home_col, "") if not _is_home else row.get(away_col, "")
    _pick_pct = _rg_lookup(_pick_team)
    _opp_pct = _rg_lookup(_opp_team)
    if _pick_pct is None or _opp_pct is None:
        return False
    return _pick_pct < _RECORD_GAP_DOG_MAX and _opp_pct > _RECORD_GAP_FAV_MIN

cand['_record_gap_extreme'] = cand.apply(_cand_record_gap_extreme, axis=1)
_record_gap_mask = (cand['passes_edge_rule'] == 1) & cand['_record_gap_extreme']
cand.loc[_record_gap_mask, 'edge'] = cand.loc[_record_gap_mask, 'edge'] - _GATE9_RECORDGAP_KNOCKDOWN
_record_gap_now_fails = _record_gap_mask & (cand['edge'] < cand['_base_edge_min'])
cand.loc[_record_gap_now_fails, 'passes_edge_rule'] = 0
cand.loc[_record_gap_now_fails, '_gate_fail_reason'] = cand.loc[_record_gap_now_fails, '_gate_fail_reason'].fillna(
    f"Gate 9 -- RECORD GAP: pick team season win% < {_RECORD_GAP_DOG_MAX:.0%} while opponent's is > {_RECORD_GAP_FAV_MIN:.0%} "
    f"-- a real bottom-tier-vs-top-tier gap this extreme needs more than a single day's SP matchup as proof (edge shaved -{_GATE9_RECORDGAP_KNOCKDOWN:.0%}, now below required floor)")
print(f'Record gap gate          : shaved edge -{_GATE9_RECORDGAP_KNOCKDOWN:.0%} on {int(_record_gap_mask.sum())} play(s), newly blocked {int(_record_gap_now_fails.sum())} (pick season win% < {_RECORD_GAP_DOG_MAX:.0%}, opponent win% > {_RECORD_GAP_FAV_MIN:.0%})')

# Gate 10: Model-side bullpen collapse risk (FIX 4, 2026-09-06, user audit --
# Nationals ML: Alvarez himself a bulk/short-role arm (3-5 IP/start) backed
# by a 9.00 L3 bullpen ERA -- the exact "short leash into a shaky pen"
# combination that hands innings 4-9 of the PICK's OWN game to a bad
# bullpen, with nothing before this treating that as a grade-relevant risk.
# [BULLPEN RISK]/[BULLPEN L3 ERA] elsewhere in this cell only ever price
# the OPPONENT's bad bullpen as a tailwind for the pick -- never the pick's
# own bad bullpen as a headwind on the pick's own edge. Fires only on the
# picked team's own SP + own bullpen, deliberately not the opponent's (that
# side is already captured as a tailwind by the existing bullpen blocks --
# this isn't meant to double-count it).
_COLLAPSE_BP_ERA_THRESHOLD = 5.00
_GATE10_COLLAPSE_KNOCKDOWN = round(0.07 * _MODEL_MARKET_BLEND_W, 4)  # BUG FIX (MLB-EDGE-BLEND-1, 2026-09-25): rescaled to the blended edge scale, was 0.07 -- acute, real-mechanism risk -- between Gate4 and the softer context gates

def _pick_team_name(row):
    _is_home = str(row.get('bet_team', '')).strip() == str(row.get(home_col, '')).strip()
    return row.get(home_col, '') if _is_home else row.get(away_col, '')

def _cand_collapse_risk(row):
    _spnm = _pick_sp_name(row)
    if not _spnm or 'mlb_pitcher_fip_lookup' not in globals():
        return False
    _ev = mlb_pitcher_fip_lookup.get(_sp_norm_era(_spnm), {})
    _short_leash = bool(_ev.get('is_opener')) or bool(_ev.get('is_bulk_role'))
    if not _short_leash or 'mlb_bp_workload_l3' not in globals() or not mlb_bp_workload_l3:
        return False
    _tn = _pl_norm(str(_pick_team_name(row)))
    _bw = mlb_bp_workload_l3.get(_tn, {})
    _era_l3 = _bw.get('era_l3')
    return _era_l3 is not None and float(_era_l3) >= _COLLAPSE_BP_ERA_THRESHOLD

cand['_collapse_risk'] = cand.apply(_cand_collapse_risk, axis=1)
_collapse_mask = (cand['passes_edge_rule'] == 1) & cand['_collapse_risk']
cand.loc[_collapse_mask, 'edge'] = cand.loc[_collapse_mask, 'edge'] - _GATE10_COLLAPSE_KNOCKDOWN
_collapse_now_fails = _collapse_mask & (cand['edge'] < cand['_base_edge_min'])
cand.loc[_collapse_now_fails, 'passes_edge_rule'] = 0
cand.loc[_collapse_now_fails, '_gate_fail_reason'] = cand.loc[_collapse_now_fails, '_gate_fail_reason'].fillna(
    f"Gate 10 -- BULLPEN COLLAPSE RISK: pick SP is short-leash (opener/bulk-role) backed by a pick-team bullpen L3 ERA >= {_COLLAPSE_BP_ERA_THRESHOLD:.2f} "
    f"-- hands innings 4-9 of the pick's OWN game to a bad pen (edge shaved -{_GATE10_COLLAPSE_KNOCKDOWN:.0%}, now below required floor)")
print(f'Bullpen collapse risk gate: shaved edge -{_GATE10_COLLAPSE_KNOCKDOWN:.0%} on {int(_collapse_mask.sum())} play(s), newly blocked {int(_collapse_now_fails.sum())} (pick SP short-leash + pick bullpen L3 ERA >= {_COLLAPSE_BP_ERA_THRESHOLD:.2f})')

# Gate 11: Lineup unconfirmed + injury delta is a primary edge driver (FIX 5,
# 2026-09-06, user audit -- Nationals/Dodgers: the Dodgers' -0.90R injury
# stack was the single biggest driver of tonight's edge, and the Dodgers'
# lineup wasn't confirmed yet (13h to first pitch) -- the M21 fix
# (_lineup_confirm_notes above) only ever prints a re-check reminder, it
# never withheld the play even when the whole edge rests on an absence
# that isn't locked in as the actual lineup yet. Hard-blocks (no edge
# override -- like Gate 7/7b, this is "we don't actually know this yet,"
# not a probabilistic risk to discount) whenever EITHER team in the
# matchup has an unconfirmed lineup AND that team's own injury penalty is
# >= the threshold below. Distinct from Gate 7/7b (which are about the
# STARTER being unknown) -- this is about the LINEUP/injury replacement
# being unknown.
_LINEUP_GATE_MIN_PENALTY = 0.40  # runs -- matches "primary edge driver" framing in the fix request

def _cand_lineup_unconfirmed_hard(row):
    _fn = globals().get("_lineup_confirm_notes")
    if not _fn or 'mlb_lineup_flags' not in globals() or not mlb_lineup_flags:
        return False
    _aw = row.get(away_col, ''); _hm = row.get(home_col, '')
    _notes = _fn(row.get("game_datetime_utc"), _aw, _hm)
    if not _notes:
        return False
    for _t in (_aw, _hm):
        _flag = mlb_lineup_flags.get(_pl_norm(str(_t))) or {}
        if float(_flag.get('penalty', 0) or 0) >= _LINEUP_GATE_MIN_PENALTY:
            return True
    return False

cand['_lineup_unconfirmed_hard'] = cand.apply(_cand_lineup_unconfirmed_hard, axis=1)
_lineup_gate_mask = (cand['passes_edge_rule'] == 1) & cand['_lineup_unconfirmed_hard']

# GATE 11 CLARITY (2026-09-19, user audit -- Angels ML +20.7%, Gate 11
# blocked it for an unconfirmed Twins injury absence): the gate message said
# only THAT an unconfirmed injury existed, never whether it was actually the
# thing behind the displayed edge -- a reader has no way to tell "this edge
# assumes the injury holds" apart from "this edge doesn't even reflect the
# injury yet, Gate 11 is blocking on the team's raw stack size alone." Real
# case that prompted checking this: the Angels/Twins net injury delta was
# -0.17R, BELOW the 0.20R threshold the FINAL CARD's own [NET INJURY DELTA
# PRICED IN] line requires before pricing it into the edge at all -- so that
# +20.7% was NOT actually built on the unconfirmed injury; Gate 11 fired on
# Minnesota's raw >= 0.40R stack size regardless. Reuses the exact same 0.20R
# threshold and mlb_lineup_flags penalties this gate already reads -- no new
# data, just answering the question its own message left open.
_GATE11_PRICE_IN_THRESHOLD = 0.20  # matches [NET INJURY DELTA PRICED IN]'s own bar

def _cand_injury_priced_note(row):
    _aw = row.get(away_col, ''); _hm = row.get(home_col, '')
    _aw_pen = float((mlb_lineup_flags.get(_pl_norm(str(_aw))) or {}).get('penalty', 0) or 0)
    _hm_pen = float((mlb_lineup_flags.get(_pl_norm(str(_hm))) or {}).get('penalty', 0) or 0)
    _net = _aw_pen - _hm_pen   # positive = away hurts more -> favors home
    _is_home_pick = str(row.get('bet_team', '')).strip() == str(_hm).strip()
    _net_favors_pick = (_net > 0) if _is_home_pick else (_net < 0)
    if abs(_net) >= _GATE11_PRICE_IN_THRESHOLD and _net_favors_pick:
        return f"this edge DOES currently price in a net {abs(_net):.2f}R injury tailwind favoring the pick"
    return (f"this edge does NOT currently price in a net injury tailwind for the pick (net "
            f"{abs(_net):.2f}R, below the {_GATE11_PRICE_IN_THRESHOLD:.2f}R pricing threshold or not "
            f"favoring the pick) -- Gate 11 is blocking on a team's raw unconfirmed stack size, not "
            f"a confirmed driver of THIS specific edge")

cand.loc[_lineup_gate_mask, 'passes_edge_rule'] = 0
_gate11_reasons = cand.loc[_lineup_gate_mask].apply(
    lambda r: (f"Gate 11 -- LINEUP UNCONFIRMED: lineup not yet posted + a >= {_LINEUP_GATE_MIN_PENALTY:.2f}R "
               f"injury absence not yet locked in as that team's actual lineup -- the edge's own primary "
               f"driver isn't confirmed yet, hard-blocked ({_cand_injury_priced_note(r)})"),
    axis=1)
cand.loc[_lineup_gate_mask, '_gate_fail_reason'] = cand.loc[_lineup_gate_mask, '_gate_fail_reason'].fillna(_gate11_reasons)
print(f"Lineup-unconfirmed hard gate: blocked {int(_lineup_gate_mask.sum())} play(s) (lineup not yet posted + a >= {_LINEUP_GATE_MIN_PENALTY:.2f}R injury absence not yet locked in as that team's actual lineup)")

# PITCHER MISMATCH EXCEPTION
# When pick-team SP is 150+ ELO better than opponent SP AND edge >= 12%,
# promote to official play at 0.5u even though edge < BASE_EDGE_MIN (15%).
# Captures Skubal-vs-Weathers type structural advantages the threshold would otherwise miss.
# Requires: non-default ELO on pick SP (not 1500 exactly), game has market odds.
_PITCHER_ADV_ELO  = 150   # pick SP must be this many ELO points better than opp SP
# BUG FIX (MLB-EDGE-BLEND-1, 2026-09-25 user audit): rescaled to the new
# blended edge scale (edge_blended = w*edge_raw when a real market price
# exists), same reasoning as BASE_EDGE_MIN above -- preserves this
# exception's original selectivity rather than a guessed new cutoff.
_PITCHER_ADV_EDGE = round(0.12 * globals().get("_MODEL_MARKET_BLEND_W", 1.0), 4)  # was 0.12 on the raw-edge scale
# BUG FIX (audit, 2026-08-04): this exception promoted passes_edge_rule to 1
# checking only SP ELO gap / edge / walk-rate / staleness -- unlike EVERY
# other path that can set passes_edge_rule=1 (the main gate at ~line 2798,
# and the High-Confidence Favorite path just below, which requires
# p_bet_win_model >= HIGH_CONF_PROB_MIN), it never checked the coinflip
# floor at all. Real incident: Nationals ML tagged [OFFICIAL PLAY] at 54.8%
# model prob -- well under MIN_MODEL_PROB (0.58), the exact threshold that
# comment block exists to "eliminate coinflip plays dressed as edges." A
# real 150+pt SP mismatch is a legitimate reason to accept a smaller RAW
# EDGE than the base rule requires; it was never a legitimate reason to
# accept a near-coinflip WIN PROBABILITY. Cell 89's OFFICIAL tag / 1-play-
# per-sport ranking / Final Card all read passes_edge_rule directly with no
# second independent prob check, so this was the sole point of failure.
_pitcher_adv_mask = (
    (cand['passes_edge_rule'] == 0)          # not already passing
    & (cand['_sp_elo_gap'] <= -_PITCHER_ADV_ELO)  # pick SP 150+ better (gap is opp-pick, so negative)
    & (cand['_pick_sp_elo'] != 1500.0)       # pick SP not at default ELO
    # BUG FIX (MLB-EXC-1, 2026-09-24 user audit): _opp_sp_elo defaults to
    # 1500.0 (_sp_elo_gate's fallback) whenever the opponent starter isn't
    # resolved yet -- an unannounced opponent SP was silently read as a real
    # 150+ ELO mismatch instead of missing data. Real incident: NYM@TEX with
    # no Rangers starter announced fired this exception purely because
    # 1500 (default) - NYM's real SP ELO happened to clear the -150 gap,
    # not because of any genuine pitching mismatch. Mirrors the existing
    # pick-SP-not-default guard one line up, applied to the opponent side.
    & (cand['_opp_sp_elo'] != 1500.0)        # opp SP must also be resolved, not a default-ELO placeholder
    & (cand['edge'] >= _PITCHER_ADV_EDGE)    # edge threshold
    & (cand['p_bet_win_model'] >= MIN_MODEL_PROB)  # 2026-08-04: coinflip floor, same as every other qualifying path
    & (cand['game_has_odds'])                # market odds must exist
    & (~cand['_pick_sp_bb_pct'].fillna(0).ge(_BB_PCT_ELEVATED))  # 2026-07-29: don't let this path re-promote a walk-rate hard stop
    & (cand['_pick_sp_stale'].isna())  # 2026-07-29: don't let this path re-promote a stale-rating hard stop either
    & (~cand['_elo_era_divergent'])    # 2026-08-04: don't let this path re-promote the ELO-ERA divergence hard stop
    & (~cand['_record_gap_extreme'])   # 2026-08-04: don't let this path re-promote the record-gap hard stop
)
cand.loc[_pitcher_adv_mask, 'passes_edge_rule'] = 1
cand['_pitcher_adv_exception'] = _pitcher_adv_mask
_n_adv = int(_pitcher_adv_mask.sum())
if _n_adv:
    print(f'Pitcher advantage exception: {_n_adv} play(s) promoted (pick SP 150+ ELO better, edge >={_PITCHER_ADV_EDGE:.0%}) — sized 0.5u')

# HIGH-CONFIDENCE FAVORITE PATH -- added 2026-07-25
# Full-card postmortem on 2026-07-25 (14 games) found zero official plays that
# day, yet several ML lines showed real positive edge on a genuine favorite
# (model prob well above 50%, market already favoring the same side) that
# still couldn't clear BASE_EDGE_MIN. User feedback: a smaller edge is fine
# when the market already favors the side, as long as the model's own win
# probability is genuinely high -- rather than only rewarding big model/market
# divergence, which is disproportionately produced by near-coinflip-model
# underdog plays (e.g. that day's single biggest edge, COL @ MIL: model only
# 47.8%, +15.1% edge on raw size alone, lost outright). This is a second,
# independent qualifying path alongside the edge-driven BASE_EDGE_MIN rule
# above -- a genuinely confident favorite can qualify on a much smaller edge
# than a long-shot needs to justify the same play.
HIGH_CONF_PROB_MIN = 0.62   # meaningfully above the base MIN_MODEL_PROB (0.58) floor -- a real high-probability favorite, not just clearing the coinflip bar
HIGH_CONF_EDGE_MIN = 0.04   # still requires a real (non-noise) edge, just not the full BASE_EDGE_MIN (15%) bar
_high_conf_mask = (
    (cand['passes_edge_rule'] == 0)                      # not already passing another path
    & (cand['is_repeat'] == 0)                           # keep this path conservative for its first rollout
    & (cand['game_has_odds'])
    & (cand['market_novig_prob'].fillna(0) >= 0.50)      # market already favors this side ("higher juice")
    & (cand['p_bet_win_model'] >= HIGH_CONF_PROB_MIN)
    & (cand['edge'] >= HIGH_CONF_EDGE_MIN)
    & (cand['edge'] < _EDGE_CEILING)                     # same data-sanity ceiling as the edge-driven path
    & (~cand['_max_odds_blocked'])
    & (~cand['_pick_sp_bb_pct'].fillna(0).ge(_BB_PCT_ELEVATED))       # exclude elevated season walk-rate starters
    & (~cand['_pick_sp_last_start_implosion'].fillna(False))         # exclude last-start implosion risk
    & (cand['_pick_sp_stale'].isna())                                # 2026-07-29: this path re-promotes Gate 6's
    # block right back to passes_edge_rule=1 whenever the stale pitcher's own inflated
    # rating clears HIGH_CONF_PROB_MIN on model prob alone (confirmed live: Wesneski's
    # 72% model prob cleared this path even after Gate 6 zeroed the row) -- same
    # exclusion the Pitcher Advantage Exception above already got.
    & (~cand['_elo_era_divergent'])    # 2026-08-04: same ELO-ERA divergence exclusion as the Pitcher Advantage Exception
    & (~cand['_record_gap_extreme'])   # 2026-08-04: same record-gap exclusion as the Pitcher Advantage Exception
)
cand.loc[_high_conf_mask, 'passes_edge_rule'] = 1
cand['_high_conf_favorite'] = _high_conf_mask
_n_high_conf = int(_high_conf_mask.sum())
if _n_high_conf:
    print(f'High-confidence favorite path: {_n_high_conf} play(s) promoted (model >= {HIGH_CONF_PROB_MIN:.0%}, market favorite, edge >= {HIGH_CONF_EDGE_MIN:.0%})')

# FIX 3 -- AWAY-AT-DOME/HIGH-PARK-FACTOR WARNING
# Flags (does not block) plays where pick team is AWAY at a dome or PF >= 1.04.
# Dome teams have park familiarity edge; high-PF parks inflate season run averages.
_HIGH_PF_WARN = 1.04
cand["_away_dome_flag"] = False
if away_col in cand.columns and home_col in cand.columns:
    def _away_dome_check(r):
        if str(r.get("bet_side", "")).lower() != "away":
            return False
        _hk = _norm_team(str(r.get(home_col, "")))
        return _PARK_FACTOR.get(_hk, 1.00) >= _HIGH_PF_WARN or _hk in _DOME_TEAMS
    cand["_away_dome_flag"] = cand.apply(_away_dome_check, axis=1)
    _n_dome = int(cand[cand["passes_edge_rule"] == 1]["_away_dome_flag"].sum())
    if _n_dome > 0:
        print(f"  [ROAD WARNING] {_n_dome} play(s): pick is AWAY at dome/PF>={_HIGH_PF_WARN} -- verify road record before betting")
        # Dome road penalty feeds directly into sizing — not just a warning
        cand.loc[cand["_away_dome_flag"] & (cand["passes_edge_rule"] == 1), "_dome_road_flag"] = True
    # Doubleheader flag: block plays when G1/G2 have same SP data (identical outputs)
    # 2026-07-28 fix: this computed _dh_sp_a/_dh_sp_h (each row's starters) but never
    # actually compared them -- the "already seen this matchup" check alone is true
    # for every real doubleheader's second row regardless of whether the starters
    # are actually different, so this fired and blocked EVERY DH game's plays
    # unconditionally, not just the ones with a genuine same-starter data problem.
    # Real incident 2026-07-28: Guardians @ Reds (real rain-makeup DH, confirmed
    # live -- Cecconi/Chase Burns G1, Gavin Williams G2) showed byte-identical model
    # probability (54.1%) and total-runs projection (6.2) for both games in the
    # Daily Market Snapshot -- direct evidence the starter feed attached the same
    # (likely G1's) pitcher data to both board rows, a real upstream bug this gate
    # was clearly meant to catch and didn't. Now actually tracks each matchup's
    # first-seen starter pair and only flags/blocks when a later game in the same
    # pair has the identical pair -- a correctly-differentiated DH (real distinct
    # starters) is no longer blocked just for existing.
    _dh_games = cand[cand.get("note","").str.contains("DH-", na=False)].copy() if "note" in cand.columns else pd.DataFrame()
    _dh_seen_sp = {}    # _dh_key -> (away_sp, home_sp) of the first game seen for that matchup
    _dh_dupe_keys = set()  # _dh_key where a later game repeats the exact same SP pair
    if not _dh_games.empty:
        for _dh_idx, _dh_row in _dh_games.iterrows():
            _dh_sp_a = str(_dh_row.get("away_pitcher_name","") or "")
            _dh_sp_h = str(_dh_row.get("home_pitcher_name","") or "")
            _dh_key = (_dh_row.get(away_col,""), _dh_row.get(home_col,""))
            _dh_sp_pair = (_dh_sp_a, _dh_sp_h)
            if _dh_key not in _dh_seen_sp:
                _dh_seen_sp[_dh_key] = _dh_sp_pair
            elif _dh_seen_sp[_dh_key] == _dh_sp_pair and (_dh_sp_a or _dh_sp_h):
                _dh_dupe_keys.add(_dh_key)
        if _dh_dupe_keys:
            print(f"  [DH WARNING] {len(_dh_dupe_keys)} doubleheader game(s) show IDENTICAL starters across G1/G2 -- starter feed likely isn't differentiating games, verify manually")
            if "note" in cand.columns:
                _dh_mask = (
                    cand["note"].str.contains("DH-", na=False)
                    & cand.apply(lambda r: (r.get(away_col,""), r.get(home_col,"")) in _dh_dupe_keys, axis=1)
                    & (cand["passes_edge_rule"] == 1)
                )
                cand.loc[_dh_mask, "passes_edge_rule"] = 0  # block until confirmed
                _n_dh_blocked = int(_dh_mask.sum())
                if _n_dh_blocked > 0:
                    print(f"  [DH BLOCKED] {_n_dh_blocked} DH play(s) suppressed -- same starter assigned to both games, fix starter feed then re-run")

# 2026-07-27: removed the FIX 2 public-narrative-underdog flag -- it only
# ever pointed at "check sharp money %", and sharp money is gone (see below).


# Model-only threshold: 65% when pick team SP is WEAK (<1450 ELO), else 58%
_mo_home_sp = cand.get("home_sp_elo_adj", 1500) if isinstance(cand.get("home_sp_elo_adj", 1500), (int,float)) else cand["home_sp_elo_adj"].fillna(1500)
_mo_away_sp = cand.get("away_sp_elo_adj", 1500) if isinstance(cand.get("away_sp_elo_adj", 1500), (int,float)) else cand["away_sp_elo_adj"].fillna(1500)
import pandas as _pd_mo
_mo_home_elo = cand["home_sp_elo_adj"].fillna(1500) if "home_sp_elo_adj" in cand.columns else _pd_mo.Series(1500, index=cand.index)
_mo_away_elo = cand["away_sp_elo_adj"].fillna(1500) if "away_sp_elo_adj" in cand.columns else _pd_mo.Series(1500, index=cand.index)
_mo_bet_side = cand["bet_side"] if "bet_side" in cand.columns else _pd_mo.Series("home", index=cand.index)
_mo_pick_sp_elo = _mo_home_elo.where(_mo_bet_side == "home", _mo_away_elo)
_mo_threshold  = _mo_pick_sp_elo.apply(lambda e: 0.65 if e < 1450 else 0.58)
cand["model_only_pick"] = (
    (cand["p_bet_win_model"] >= _mo_threshold)
    & (cand["passes_edge_rule"] == 0)
    & (~cand["game_has_odds"])  # only fire when ESPN has no market data
).astype(int)

# Tag play_type on all candidates
cand["play_type"] = "ML"  # default; OU plays get their own type when appended
qualified_market = cand[cand["passes_edge_rule"] == 1].copy()
qualified_model  = cand[(cand["model_only_pick"]==1) & (cand["passes_edge_rule"]==0)].copy()

# __MLB_VALUE_WATCH__
# Picks just below threshold: 0.09 <= edge < BASE_EDGE_MIN, model >= 0.55
_watch_min_edge = 0.09
qualified_watch = cand[
    (cand["game_has_odds"])
    & (cand["market_implied_prob"].notna())
    & (cand["edge"] >= _watch_min_edge)
    & (cand["edge"] < BASE_EDGE_MIN)
    & (cand["p_bet_win_model"] >= 0.55)
    & (cand["passes_edge_rule"] == 0)
].copy()
# Dedupe to one play per game (best edge)
if len(qualified_watch) > 0:
    _watch_dedup_cols = [c for c in [home_col, away_col] if c in qualified_watch.columns]
    qualified_watch = qualified_watch.sort_values("edge", ascending=False).drop_duplicates(subset=_watch_dedup_cols)

print(f"\nMarket plays qualified : {len(qualified_market)}")
print(f"Model-only picks       : {len(qualified_model)}")

game_dedupe = [c for c in [game_id_col, date_col, away_col, home_col]
               if c is not None and c in cand.columns]

def _dedupe(df):
    if len(df) == 0: return df
    return (df.sort_values(["final_score","edge","p_bet_win_model"], ascending=False)
              .drop_duplicates(subset=game_dedupe, keep="first")
              .reset_index(drop=True))

qualified_market = _dedupe(qualified_market)
qualified_model  = _dedupe(qualified_model)

# ---------------------------------------------------------
# 12) card output
# ---------------------------------------------------------
MAX_MARKET_PLAYS = 5
MAX_MODEL_PLAYS  = 3

def _pitcher_note(r):
    for ap in ["away_pitcher_final_name","away_pitcher_name","away_starter"]:
        for hp in ["home_pitcher_final_name","home_pitcher_name","home_starter"]:
            if ap in r.index and hp in r.index and pd.notna(r.get(ap)) and pd.notna(r.get(hp)):
                def _sp_elo_lookup(nm):
                    import unicodedata
                    def _nn(s): return unicodedata.normalize("NFKD",str(s)).encode("ascii","ignore").decode().lower().strip()
                    try:
                        _d = mlb_pitcher_elo
                    except NameError:
                        return 1500.0
                    if not isinstance(_d, dict): return 1500.0
                    v = _d.get(str(nm))
                    if v is not None: return v
                    _nk = _nn(nm)
                    for k, ev in _d.items():
                        if _nn(k) == _nk: return ev
                    return 1500.0
                # Use ERA-adjusted ELO from board if available, else raw lookup
                _ap_elo_raw = _sp_elo_lookup(r[ap])
                _hp_elo_raw = _sp_elo_lookup(r[hp])
                _ap_elo = float(r.get("away_sp_elo_adj") or _ap_elo_raw) if r.get("away_sp_elo_adj") and not pd.isna(r.get("away_sp_elo_adj")) else _ap_elo_raw
                _hp_elo = float(r.get("home_sp_elo_adj") or _hp_elo_raw) if r.get("home_sp_elo_adj") and not pd.isna(r.get("home_sp_elo_adj")) else _hp_elo_raw
                def _sp_tier(e):
                    if e>=1560: return "ELITE(>=1560)"
                    if e>=1520: return "STRONG(1520-1559)"
                    if e>=1480: return "AVG(1480-1519)"
                    if e>=1440: return "BELOW(1440-1479)"
                    return "WEAK(<1440)"
                def _sp_label(e, nm=None, opp_team=None, ip=None, fip=None, era=None):
                    # 2026-08-29 (user audit M4): a starter with fewer than
                    # _THIN_SAMPLE_MIN_STARTS career MLB starts has no rate worth
                    # tiering. Cell 83's small-sample regression softens his ELO
                    # NUMBER, which meant a 1-start debutant printed here as a
                    # confident "AVG(1480-1519)" -- an unearned label on essentially
                    # zero MLB information. Labelled UNRATED instead; the ELO number
                    # itself is still printed next to it, so nothing is hidden.
                    _spl_thin = globals().get("_sp_thin_starts")
                    if nm and _spl_thin and _spl_thin(nm) is not None:
                        return f"UNRATED(<{globals().get('_THIN_SAMPLE_MIN_STARTS',5)} MLB starts)"
                    if e == 1500.0:
                        return "DEFAULT"
                    _sp_base_tier = _sp_tier(e)
                    _sp_tier_order = ["WEAK(<1440)", "BELOW(1440-1479)", "AVG(1480-1519)", "STRONG(1520-1559)", "ELITE(>=1560)"]
                    _sp_downgrade_steps = 0
                    _sp_downgrade_reasons = []
                    _sp_dg_fn = globals().get("_sp_tier_downgrade_gap")
                    _sp_dg_gap = _sp_dg_fn(nm) if (_sp_dg_fn and nm) else None
                    # BUG FIX (2026-09-13, user audit -- Quantrill ELITE vs E-Rod STRONG,
                    # ARI ML +16.3%): a flat "-1 tier, regardless of gap size" treated a
                    # pitcher whose FIP is 0.89R better than his own xFIP/SIERA the same as
                    # one sitting right at the 0.50R trigger. E-Rod's real case: FIP 3.76
                    # (drives ELITE/1655) vs xFIP 4.41/SIERA 4.64 -- that's not "one tier of
                    # doubt," it's "his tier-driving number reads almost a full run better
                    # than what his own peripherals say he actually is." A -1 step still let
                    # STRONG anchor the pick as a real, if discounted, SP edge; the tier gap
                    # deserves to actually collapse when the underlying number is this far off,
                    # not just get a haircut. _LARGE_FIP_GAP_THRESHOLD mirrors the size the
                    # user flagged as "not a small-gap situation anymore."
                    # REFINEMENT (2026-09-13, user audit): a hard 0.75R cliff (2 steps at
                    # 0.75, 1 step at 0.74) meant a 0.01R data wobble could flip a pitcher's
                    # printed tier by a full extra band -- and it capped every gap >= 0.75R at
                    # exactly 2 steps forever, so a hypothetical 1.50R gap got treated the same
                    # as a 0.75R one. Continuous ramp instead: 1 step at the 0.50R trigger,
                    # scaling up by 1 more step per additional 0.50R (same width as the base
                    # trigger gap itself), so severity keeps scaling for genuinely extreme gaps
                    # without a sharp cliff at one specific number. E-Rod's real case (0.88R)
                    # still lands on 2 steps (1 + (0.88-0.50)/0.50 = 1.76 -> rounds to 2),
                    # matching the intended magnitude -- verified this doesn't overshoot to 3
                    # before shipping it.
                    _LARGE_FIP_GAP_THRESHOLD = 0.75   # kept for the separate thesis-invalidation gate below (same cutoff, different consequence)
                    if _sp_dg_gap is not None:
                        _sp_downgrade_steps += max(1, round(1 + (_sp_dg_gap - _FIP_RELIABILITY_GAP) / 0.50))
                        _sp_downgrade_reasons.append(f"FIP/SIERA gap {_sp_dg_gap:.2f}R")
                    # M-NEW-3 (2026-09-07, user audit): stacks with the
                    # FIP/SIERA downgrade above rather than replacing it --
                    # each represents independent, real distrust in the raw
                    # ELO tier, same "only as many steps as detected issues"
                    # pattern the FIP/SIERA check already established.
                    _sp_voe_fn = globals().get("_sp_vs_opponent_era")
                    if _sp_voe_fn and nm and opp_team:
                        _sp_opp_era, _sp_opp_n = _sp_voe_fn(nm, opp_team)
                        if _sp_opp_era is not None:
                            _sp_fev = globals().get('mlb_pitcher_fip_lookup', {}).get(_sp_norm_era(nm), {}) if 'mlb_pitcher_fip_lookup' in globals() else {}
                            _sp_season_era_raw = _sp_fev.get('era_season')
                            _sp_season_era = (float(_sp_season_era_raw)
                                               if _sp_season_era_raw is not None and not pd.isna(_sp_season_era_raw)
                                               else None)
                            # BUG FIX (MLB-OPPMISMATCH-1, 2026-09-24 user audit): abs()
                            # fired this downgrade in BOTH directions -- a pitcher who has
                            # historically DOMINATED this specific opponent (opp ERA far
                            # BELOW season ERA) got the exact same downgrade as one who
                            # has struggled against them (opp ERA far ABOVE season ERA).
                            # Real incident: Yamamoto downgraded from ELITE for a much
                            # BETTER-than-season ERA vs San Diego; separately verified in
                            # this same notebook's saved output, Bryan Woo downgraded from
                            # ELITE for a 1.35 ERA vs the Angels against a 3.92 season ERA.
                            # The whole point of this check is "the season rating overstates
                            # this pitcher's effectiveness against THIS opponent" -- that
                            # only makes sense when he's WORSE against them, not better.
                            if _sp_season_era is not None and (_sp_opp_era - _sp_season_era) > _M_NEW3_ERA_GAP:
                                _sp_downgrade_steps += 1
                                _sp_downgrade_reasons.append(
                                    f"SP OPPONENT MISMATCH: {_sp_opp_era:.2f} ERA vs {opp_team} in "
                                    f"{_sp_opp_n} start(s) this season vs {_sp_season_era:.2f} season ERA")
                    # POST-TRADE SPLIT (2026-09-11, user request -- Dustin May):
                    # season-aggregate ERA/FIP (used everywhere above, including
                    # the base ELO itself) blends pre- and post-trade starts
                    # together -- for a pitcher traded within the recency window
                    # below, that blend can hide a real form change with the new
                    # team entirely. Same independent-reason pattern as the two
                    # checks above; the matching edge shave lives earlier in this
                    # cell (_post_trade_shave), same M-NEW-3 lesson about not
                    # shipping a tier-only downgrade with nothing behind it.
                    _sp_pt_fev = globals().get('mlb_pitcher_fip_lookup', {}).get(_sp_norm_era(nm), {}) if (nm and 'mlb_pitcher_fip_lookup' in globals()) else {}
                    if nm and bool(_sp_pt_fev.get('is_recently_traded')):
                        _sp_pt_post_era = _sp_pt_fev.get('post_trade_era')
                        _sp_pt_pre_era = _sp_pt_fev.get('pre_trade_era')
                        _sp_pt_compare_era = (_sp_pt_pre_era if (_sp_pt_pre_era is not None and pd.notna(_sp_pt_pre_era))
                                               else era)
                        if (_sp_pt_post_era is not None and pd.notna(_sp_pt_post_era)
                                and _sp_pt_compare_era is not None and pd.notna(_sp_pt_compare_era)
                                and (_sp_pt_post_era - float(_sp_pt_compare_era)) > _POST_TRADE_ERA_GAP):
                            _sp_downgrade_steps += 1
                            _sp_downgrade_reasons.append(
                                f"POST-TRADE SPLIT: {_sp_pt_post_era:.2f} ERA in "
                                f"{int(_sp_pt_fev.get('post_trade_starts') or 0)} start(s) with "
                                f"{_sp_pt_fev.get('new_team_name') or 'new team'} "
                                f"({int(_sp_pt_fev.get('days_since_trade') or 0)}d ago) vs "
                                f"{float(_sp_pt_compare_era):.2f} prior")
                    # XERA-CORROBORATED COLD-STREAK (2026-09-13, user audit --
                    # Arizona's featured pick, +15.7% edge, LOST 2-6: Pfaadt
                    # (ELITE/1618) allowed 6 ER on 10 hits over 6.2 IP. The
                    # matching edge shave lives earlier in this cell
                    # (_xera_cold_shave, see its own comment for the full
                    # case) -- same independent-reason pattern as the checks
                    # above, and the same M-NEW-3 lesson about not shipping a
                    # tier-only downgrade with nothing behind it. Reuses
                    # _sp_pt_fev (already fetched just above) and the shave's
                    # own thresholds via globals() so tier and edge can't drift apart.
                    _sp_xc_l3 = _sp_pt_fev.get('era_l3')
                    _sp_xc_xera = _sp_pt_fev.get('xera')
                    _sp_xc_l3ip = _sp_pt_fev.get('ip_l3')
                    if (nm and _sp_xc_l3 is not None and pd.notna(_sp_xc_l3)
                            and era is not None and pd.notna(era)
                            and _sp_xc_xera is not None and pd.notna(_sp_xc_xera)
                            and _sp_xc_l3ip is not None and pd.notna(_sp_xc_l3ip)
                            and float(_sp_xc_l3ip) >= globals().get('_XERA_COLD_MIN_IP', 9.0)):
                        _sp_xc_gap = float(_sp_xc_l3) - float(era)
                        if (_sp_xc_gap > globals().get('_XERA_COLD_L3_GAP', 0.75)
                                and abs(float(_sp_xc_l3) - float(_sp_xc_xera)) < globals().get('_XERA_COLD_CORROBORATION', 1.0)):
                            _sp_downgrade_steps += 1
                            _sp_downgrade_reasons.append(
                                f"XERA-CORROBORATED COLD STREAK: L3 ERA {float(_sp_xc_l3):.2f} vs "
                                f"season {float(era):.2f} (xERA {float(_sp_xc_xera):.2f} confirms real, not luck)")
                    # M-FIX-2 (2026-09-08, user audit): bulk/short-role arm on a
                    # sub-40-IP season whose FIP exceeds his own ERA by more
                    # than the threshold -- his run-prevention record likely
                    # isn't earned yet. Distinct question from the FIP/SIERA
                    # check above (peripherals vs FIP); this is ERA vs FIP
                    # directly. Uses the shared constants also read by the
                    # edge-shave gate earlier in this cell, so the label and
                    # the edge discount can't disagree on the threshold.
                    if nm and fip is not None and era is not None and ip is not None and ip < globals().get("_BULK_FIP_IP_MAX", 40.0):
                        _sp_ev2 = globals().get('mlb_pitcher_fip_lookup', {}).get(_sp_norm_era(nm), {}) if 'mlb_pitcher_fip_lookup' in globals() else {}
                        if bool(_sp_ev2.get('is_bulk_role')):
                            _sp_bulk_gap = float(fip) - float(era)
                            if _sp_bulk_gap > globals().get("_BULK_FIP_ERA_GAP", 0.75):
                                _sp_downgrade_steps += 1
                                _sp_downgrade_reasons.append(
                                    f"FIP DIVERGENCE: FIP {fip:.2f} exceeds ERA {era:.2f} by {_sp_bulk_gap:.2f}R "
                                    f"(bulk-role, {ip:.1f} IP) -- ERA likely unsustainable")
                    _sp_result_tier = _sp_base_tier
                    if _sp_downgrade_steps and _sp_base_tier in _sp_tier_order:
                        _sp_dg_idx = max(0, _sp_tier_order.index(_sp_base_tier) - _sp_downgrade_steps)
                        _sp_result_tier = _sp_tier_order[_sp_dg_idx]
                    # M-FIX-1 (2026-09-08, user audit -- Drew Anderson STRONG
                    # tier on 35.3 IP): under 40 IP in starts, a season ERA is
                    # too thin a sample to trust a tier above AVG regardless of
                    # what the raw number says. Applied AFTER the downgrades
                    # above as a hard floor -- a small-sample pitcher who ALSO
                    # has a FIP/SIERA or opponent-mismatch issue still shows
                    # those reasons; this just guarantees the final tier can't
                    # read higher than AVG on top of them.
                    if ip is not None and ip < globals().get("_SMALL_SAMPLE_IP_FLOOR", 40.0) and _sp_result_tier in _sp_tier_order:
                        if _sp_tier_order.index(_sp_result_tier) > _sp_tier_order.index("AVG(1480-1519)"):
                            _sp_result_tier = "AVG(1480-1519)"
                            _sp_downgrade_reasons.append(
                                f"SMALL SAMPLE FLOOR: {ip:.1f} IP this season (need {globals().get('_SMALL_SAMPLE_IP_FLOOR', 40.0):.0f}+ to trust a tier above AVG)")
                    if _sp_result_tier != _sp_base_tier:
                        return f"{_sp_result_tier} [downgraded from {_sp_base_tier.split('(')[0]}: {'; '.join(_sp_downgrade_reasons)}]"
                    return _sp_base_tier
                _sp_warn = " [SP DATA: DEFAULT ELO]" if (_ap_elo == 1500.0 or _hp_elo == 1500.0) else ""
                # 2026-07-12: flag when this matchup came from the live-confirmed
                # starter feed (away_pitcher_name/home_pitcher_name -- the columns
                # the cell 88 SP-override block writes from Cell 41/42's live game
                # feed, not the scheduled probable). Real incident: external
                # research assumed CHC's Sunday starter was Assad (a stale/public
                # source), the model correctly had Boyd via the live feed, and the
                # mismatch wasted research time with no way to tell from the card
                # output which source was authoritative. This makes that explicit.
                _sp_src_tag = "  [confirmed via Cell 41/42 live feed -- external sources may lag]" if (ap == "away_pitcher_name" and hp == "home_pitcher_name") else ""
                # Real gap found 2026-07-23: this panel showed ELO tier only --
                # ERA/WHIP/xFIP/SIERA were computed elsewhere in the notebook but
                # never surfaced here, so a small-sample ERA (e.g. 3.92 over 41 IP)
                # had no xFIP/SIERA context shown alongside it to judge whether that
                # ERA is earned or lucky.
                #
                # AUDIT FIX (2026-08-03): ERA/WHIP used to only appear on a second,
                # easy-to-miss line below the ELO tier -- a "WEAK ELO 1212" headline
                # with the ERA that actually explains it buried underneath instead of
                # alongside it. ERA/WHIP now get pulled directly into the SAME
                # bracket as the ELO tier so the two numbers can't be read apart.
                # Also added a normalized-name fuzzy fallback to the cache lookup
                # (matching the pattern _sp_elo_lookup above already uses) so a
                # name-format mismatch doesn't silently drop ERA/WHIP for a pitcher
                # who does have data.
                def _sp_stats(nm):
                    import unicodedata as _pn_ud, re as _pn_re
                    def _pn_norm(s):
                        if not s: return ''
                        s = _pn_ud.normalize('NFD', str(s))
                        s = ''.join(c for c in s if _pn_ud.category(c) != 'Mn')
                        return _pn_re.sub(r'[^a-z ]', '', s.lower()).strip()
                    _key = _pn_norm(nm)
                    _era = _whip = _ip = None
                    _era_l3 = _gs_l3 = None
                    _era_l10 = _gs_l10 = None
                    _era_source = None  # 'starts_only' or 'season_all' -- 2026-08-07 label fix
                    if '_sp_form_cache' in globals():
                        _fc = _sp_form_cache.get(_key)
                        if not _fc:
                            for _ck, _cv in _sp_form_cache.items():
                                if _cv and _pn_norm(_ck) == _key:
                                    _fc = _cv
                                    break
                        _fc = _fc or {}
                        _era = _fc.get('era_season')
                        _whip = _fc.get('whip_season')
                        # 2026-08-08 fix: this line always displayed era_season
                        # (full-season, starts-only ERA) with no visibility into
                        # era_l3 (last-3-starts ERA), even though _sp_form_cache
                        # computes both -- a pitcher whose season and recent form
                        # agree (e.g. Burns, 2.35 both ways) looks identical on
                        # this card to one where they've meaningfully diverged
                        # (e.g. Alvarez, 3.94 season vs a materially different
                        # L3 number), even though that divergence is exactly the
                        # kind of signal that should raise/lower confidence in
                        # the SP-based edge. Captured here; surfaced below only
                        # when the gap is real (gs_l3>=2, matching the sample
                        # gate already used elsewhere in this cell) and large
                        # enough to matter, not on every pitcher.
                        _era_l3 = _fc.get('era_l3')
                        _gs_l3 = _fc.get('gs_l3')
                        # 2026-08-29 (user audit M2): L10 was computed but never
                        # displayed, so a 3-start blip and a 10-start slide read
                        # identically on the card's SP line.
                        _era_l10 = _fc.get('era_l10')
                        _gs_l10 = _fc.get('gs_l10')
                        if _era is not None:
                            _era_source = 'starts_only'
                    _fip = _xfip = _siera = _ip_fg = None
                    if 'mlb_pitcher_fip_lookup' in globals():
                        _fg = mlb_pitcher_fip_lookup.get(_key, {})
                        _fip = _fg.get('fip')
                        _xfip = _fg.get('xfip')
                        _siera = _fg.get('siera')
                        _ip_fg = _fg.get('ip_season')
                        # SURFACE FIX (2026-08-23, user audit): barrel%/hard-hit%
                        # were already fetched into mlb_pitcher_fip_lookup (see the
                        # Statcast pull earlier in this notebook) but never made it
                        # onto the card -- pulling them the same way fip/xfip/siera
                        # already are, right above.
                        _hh = _fg.get('hard_hit_pct')
                        _brl = _fg.get('barrel_pct')
                        _xwoba = _fg.get('xwoba')  # MLB-PROSPCALL-1 (2026-09-14)
                        # BUG FIX (2026-08-05): _sp_form_cache's era_season is
                        # intentionally None for a thin-starts-sample pitcher
                        # (see _sp_fetch_recent above) -- fall back to Cell
                        # 20's relief-inclusive era_season/era_l3 here so the
                        # card still shows a real number for a pitcher who has
                        # actually pitched all season, just mostly in relief,
                        # instead of a blank ERA.
                        if _era is None:
                            _era = _fg.get('era_season', _fg.get('era_l3'))
                            if _era is not None:
                                _era_source = 'season_all'
                    # BUG FIX (MLB-SPDATA-1, 2026-09-25 user audit): re-verified
                    # live AGAIN (2nd time -- see the 2026-09-24 investigation for
                    # the first) that the starts-only ERA this card shows matches
                    # the MLB Stats API game log exactly. The recurring "doesn't
                    # match what I see elsewhere" reports are real -- the card's
                    # number and a public site's OVERALL (incl. relief) number are
                    # both correct, just different, real splits of the same
                    # pitcher -- but expecting a reader to know that distinction
                    # from a one-word "(starts only)" tag alone isn't working.
                    # Captures the overall number here (already fetched into _fg
                    # for the season_all-fallback branch above -- no new API call)
                    # so the SP stat line can show both explicitly when they
                    # diverge, closing the loop without the reader needing to look
                    # anything up externally at all.
                    _era_overall_all = None
                    if '_fg' in dir() and _fg:
                        _era_overall_all = _fg.get('era_season')
                    _ip = _ip if _ip is not None else _ip_fg
                    return {"era": _era, "whip": _whip, "fip": _fip, "xfip": _xfip, "siera": _siera, "ip": _ip,
                            "hard_hit_pct": _hh, "barrel_pct": _brl, "xwoba": _xwoba,
                            "era_source": _era_source, "era_overall_all": _era_overall_all, "era_l3": _era_l3, "gs_l3": _gs_l3,
                            "era_l10": _era_l10, "gs_l10": _gs_l10}
                def _fmt(v, nd=2):
                    try:
                        _f = float(v)
                        return None if pd.isna(_f) else f"{_f:.{nd}f}"
                    except (TypeError, ValueError):
                        return None
                _stats_a = _sp_stats(r[ap])
                _stats_h = _sp_stats(r[hp])
                # 2026-08-08: minimum gap before the L3 number is worth printing
                # -- a 0.1-0.2 ERA wobble over 3 starts is noise, not signal.
                _ERA_L3_DIVERGENCE_MIN = 0.75
                def _inline_era_whip(_st):
                    _e = _fmt(_st.get("era")); _w = _fmt(_st.get("whip"))
                    _l3_note = ""
                    _era_v = _st.get("era"); _era_l3_v = _st.get("era_l3"); _gs_l3_v = _st.get("gs_l3") or 0
                    if (_era_v is not None and _era_l3_v is not None and _gs_l3_v >= 2
                            and abs(float(_era_l3_v) - float(_era_v)) >= _ERA_L3_DIVERGENCE_MIN):
                        _l3_note = f" [L3: {_era_l3_v:.2f}]"
                    # 2026-08-29 (user audit M2): show the L10 window alongside L3
                    # whenever the two disagree -- that gap is what separates a
                    # 3-start blip from a real, sustained slide.
                    _era_l10_v = _st.get("era_l10"); _gs_l10_v = _st.get("gs_l10") or 0
                    if (_era_l10_v is not None and _era_l3_v is not None and _gs_l10_v >= 5
                            and abs(float(_era_l10_v) - float(_era_l3_v)) >= _ERA_L3_DIVERGENCE_MIN):
                        _l3_note += f" [L10: {_era_l10_v:.2f}]"
                    if _e and _w: return f", ERA {_e}/WHIP {_w}{_l3_note}"
                    if _e: return f", ERA {_e}{_l3_note}"
                    if _w: return f", WHIP {_w}"
                    return ""
                _sp_health_a_shave, _sp_health_a_status = _sp_health_shave(r[ap], r.get(away_col))
                _sp_health_h_shave, _sp_health_h_status = _sp_health_shave(r[hp], r.get(home_col))
                _sp_health_tag = ""
                if _sp_health_a_status:
                    _sp_health_tag += f"  !! SP HEALTH UNCONFIRMED ({r[ap]}: {_sp_health_a_status})"
                if _sp_health_h_status:
                    _sp_health_tag += f"  !! SP HEALTH UNCONFIRMED ({r[hp]}: {_sp_health_h_status})"
                # FIX 1 (2026-09-22, user audit): cross-check each starter's IL status
                # against the two IL data sources this cell already computes --
                # _mlb_statsapi_roster_injuries() (current 40-man roster status,
                # defined above) for an ACTIVE D7/D10/D15/D60 stint, and _sp_il_flags
                # (21-day MLB transactions window, built further down this cell but
                # already a global by the time this function actually runs, since
                # _pitcher_note() is only ever called from the loop after it) for a
                # RECENT placement/activation. Only meaningful when this matchup came
                # from the live-confirmed starter feed (_sp_src_tag above) -- that's
                # the case where "he's on the mound tonight" and "IL data says he's
                # out" can actually disagree.
                _sp_confirmed_live = (ap == "away_pitcher_name" and hp == "home_pitcher_name")
                def _sp_il_status_note(spnm, team_name):
                    if not spnm or not team_name or not _sp_confirmed_live:
                        return ""
                    try:
                        _il_roster = _mlb_statsapi_roster_injuries(team_name) or []
                    except Exception:
                        _il_roster = []
                    _spk = _sp_norm_era(spnm)
                    _active_il_desc = None
                    for _it in _il_roster:
                        if _sp_norm_era(_it.get('name', '')) == _spk:
                            _active_il_desc = _it.get('status', '') or "IL"
                            break
                    if _active_il_desc:
                        return (f"\n  [SP IL STATUS CONFLICT] {spnm}: MLB Stats API roster status shows "
                                f"active IL ({_active_il_desc}), but the Cell 41/42 live feed confirms him "
                                f"as today's starter -- trusting the live start feed since he's actually on the mound.")
                    _sp_last = str(spnm).lower().split()[-1] if str(spnm).strip() else ""
                    if len(_sp_last) > 3:
                        for _ilk, _ilv in (globals().get('_sp_il_flags', {}) or {}).items():
                            if _sp_last in _ilk:
                                return (f"\n  [SP IL STATUS: CONFIRMED ACTIVE per Cell 41/42] {spnm} ({_ilv}) "
                                        f"-- currently confirmed starting today.")
                    return ""
                _line3 = _sp_il_status_note(r[ap], r.get(away_col)) + _sp_il_status_note(r[hp], r.get(home_col))
                _line1 = (f"  SP Matchup : {r[ap]} [{_sp_label(_ap_elo, r[ap], opp_team=r.get(home_col), ip=_stats_a.get('ip'), fip=_stats_a.get('fip'), era=_stats_a.get('era'))}/{_ap_elo:.0f}{_inline_era_whip(_stats_a)}] vs "
                        f"{r[hp]} [{_sp_label(_hp_elo, r[hp], opp_team=r.get(away_col), ip=_stats_h.get('ip'), fip=_stats_h.get('fip'), era=_stats_h.get('era'))}/{_hp_elo:.0f}{_inline_era_whip(_stats_h)}]{_sp_warn}{_sp_src_tag}{_sp_health_tag}")
                def _sp_stat_line(_st):
                    _parts = []
                    _e = _fmt(_st.get("era")); _w = _fmt(_st.get("whip")); _f = _fmt(_st.get("fip")); _x = _fmt(_st.get("xfip")); _s = _fmt(_st.get("siera")); _i = _fmt(_st.get("ip"), 1)
                    _hh = _fmt(_st.get("hard_hit_pct"), 1); _brl = _fmt(_st.get("barrel_pct"), 1); _xw = _fmt(_st.get("xwoba"), 3)
                    if _e:
                        _era_src = _st.get("era_source")
                        _e_tag = (" (starts only)" if _era_src == "starts_only"
                                  else " (season, incl. relief)" if _era_src == "season_all" else "")
                        # BUG FIX (MLB-SPDATA-1, 2026-09-25 user audit): show the
                        # overall (incl. relief) number right alongside when it
                        # meaningfully diverges from the starts-only one shown --
                        # a >=0.3 ERA gap is the threshold this cell already uses
                        # elsewhere (_ERA_L3_DIVERGENCE_MIN-style) to call a split
                        # "real, not noise."
                        _e_all = _st.get("era_overall_all")
                        _e_all_note = ""
                        if (_era_src == "starts_only" and _e_all is not None
                                and abs(float(_e_all) - float(_st.get("era") or 0)) >= 0.30):
                            _e_all_note = f" [season incl. relief: {float(_e_all):.2f}]"
                        _parts.append(f"ERA {_e}{_e_tag}{_e_all_note}")
                    if _w: _parts.append(f"WHIP {_w}")
                    # MLB-PROSPCALL-1 (2026-09-14, user audit -- Kade Anderson, FIP 5.20/
                    # SIERA 4.81 off 21 IP): FIP/xFIP/SIERA are FanGraphs SEASON rate
                    # stats (their own real denominator is ip_season, not the multi-year
                    # career total the separate LOW-STABILITY PERIPHERALS !! flag below
                    # uses) -- under _LOW_STAB_MIN_IP they're not stabilized and printing
                    # them plainly reads as more precise than they are. Swaps them for a
                    # one-line caveat and leads with xwOBA/HH%/Barrel% instead --
                    # Statcast contact-quality rates stabilize faster at this sample size
                    # (confirmed live: Anderson has no Statcast row in the Savant feed
                    # this pipeline pulls at all yet, so that gap is surfaced explicitly
                    # too, rather than silently just not showing the extra fields).
                    # Falls back to the plain FIP/xFIP/SIERA print when ip_season is
                    # unknown -- can't judge stability without it either way.
                    _ip_val = _st.get("ip")
                    _low_stab = _ip_val is not None and pd.notna(_ip_val) and float(_ip_val) < globals().get("_LOW_STAB_MIN_IP", 40.0)
                    if _low_stab:
                        if _f or _x or _s:
                            _parts.append("FIP/xFIP/SIERA unstable at this IP (see LOW-STABILITY PERIPHERALS below)")
                        if _xw: _parts.append(f"xwOBA {_xw}")
                        if _hh: _parts.append(f"HH% {_hh}")
                        if _brl: _parts.append(f"Barrel% {_brl}")
                        if not (_xw or _hh or _brl):
                            _parts.append("Statcast contact-quality data not yet available for this pitcher")
                    else:
                        # FIP added 2026-08-13 (user audit): ERA vs FIP is exactly the gap
                        # the market prices off of (defense/luck-independent run value) --
                        # xFIP/SIERA were on the card but the plain FIP number itself, the
                        # one that most directly explains "market disagrees with this ERA",
                        # was not.
                        if _f: _parts.append(f"FIP {_f}")
                        if _x: _parts.append(f"xFIP {_x}")
                        if _s: _parts.append(f"SIERA {_s}")
                        # HH%/Barrel% added 2026-08-23 (user audit): already fetched into
                        # mlb_pitcher_fip_lookup but never printed -- a pitcher whose ERA/
                        # FIP look fine can still be running into hard, well-struck contact
                        # that has not caught up to his results yet (or vice versa), and
                        # that is exactly what these two Statcast quality-of-contact
                        # numbers are for.
                        if _hh: _parts.append(f"HH% {_hh}")
                        if _brl: _parts.append(f"Barrel% {_brl}")
                    if _i: _parts.append(f"IP {_i}")
                    return " | ".join(_parts) if _parts else None
                _stat_a = _sp_stat_line(_stats_a)
                _stat_h = _sp_stat_line(_stats_h)
                _line2 = ""
                if _stat_a or _stat_h:
                    _line2 = "\n"
                    if _stat_a: _line2 += f"    {r[ap]}: {_stat_a}"
                    if _stat_a and _stat_h: _line2 += "\n"
                    if _stat_h: _line2 += f"    {r[hp]}: {_stat_h}"
                return _line1 + _line2 + _line3
    # BUG FIX (2026-08-10, user audit): this used to `return None` here,
    # silently, whenever no [away, home] pitcher-name pair resolved through
    # any of the three fallback column tiers (live-feed / probable / raw
    # schedule). The FINAL CARD + WHY loop in Cell 6e66db20 calls this and
    # only prints when the result is truthy (`if _f_pn: print(_f_pn)`), so a
    # game with zero SP data loaded showed nothing at all -- no line, no
    # warning -- instead of an explicit flag that the ML/total edge for that
    # game isn't backed by real starter data.
    #
    # BUG FIX (2026-08-18, audit found live via the new Stage 5 card review):
    # the message above always said "neither starter resolved" regardless of
    # why this fallback was reached -- but the double loop above only ever
    # tries MATCHED (ap, hp) column-tier PAIRS (e.g. both live-feed, or both
    # probable), requiring both sides to resolve from the SAME tier
    # simultaneously. A real case: Athletics @ Royals, 2026-08-18 -- the
    # Athletics starter (Jack Perkins) resolved fine, with real ERA/role
    # data printed elsewhere on the same card via a different, independent
    # lookup ([SP ROLE]/[SP COLD STREAK] in Cell 6e66db20's _emit_why) -- but
    # the Royals starter never resolved in ANY tier, so no (ap, hp) pair ever
    # matched and this fallback fired with a banner claiming BOTH starters
    # were missing when only one actually was. Checks each side independently
    # (any non-null name across all 3 tiers, ignoring pairing) so the message
    # accurately says which side is actually missing instead of overstating
    # the gap to "neither" every time.
    _any_ap = next((r.get(c) for c in ["away_pitcher_final_name", "away_pitcher_name", "away_starter"]
                     if pd.notna(r.get(c))), None)
    _any_hp = next((r.get(c) for c in ["home_pitcher_final_name", "home_pitcher_name", "home_starter"]
                     if pd.notna(r.get(c))), None)
    if _any_ap and not _any_hp:
        return (f"  !! NO SP DATA -- {r.get(home_col, 'home')} starter did not resolve "
                f"across the live-feed/probable/schedule fallback chain ({r.get(away_col, 'away')} "
                f"starter {_any_ap} did resolve, but no pairing shared a common source tier) -- "
                f"any SP-driven ML or total edge here is only half-confirmed by real starter data")
    if _any_hp and not _any_ap:
        return (f"  !! NO SP DATA -- {r.get(away_col, 'away')} starter did not resolve "
                f"across the live-feed/probable/schedule fallback chain ({r.get(home_col, 'home')} "
                f"starter {_any_hp} did resolve, but no pairing shared a common source tier) -- "
                f"any SP-driven ML or total edge here is only half-confirmed by real starter data")
    return ("  !! NO SP DATA -- neither starter resolved for this game across "
            "the live-feed/probable/schedule fallback chain -- any SP-driven "
            "ML or total edge here is not confirmed by real starter data")

def _score_line(r):
    _ar = r.get("pred_away_runs", np.nan)
    _hr = r.get("pred_home_runs", np.nan)
    if pd.notna(_ar) and pd.notna(_hr) and _ar > 0 and _hr > 0:
        _tot = _ar + _hr
        _pf  = r.get("park_factor_used", np.nan)
        _tmp = r.get("game_temp_f", np.nan)
        _env_note = ""
        _hkey2 = _norm_team(str(r.get(home_col, "")))
        if pd.notna(_tmp):
            _env_note = f" | {_tmp:.0f}F"
            # 2026-08-13 (user audit): surface wind (was fetched, used internally
            # for _wind_factor, but never shown) and precip probability/rain-risk
            # (wasn't fetched at all) on the card itself.
            _wind_v = r.get("wind_mph")
            if pd.notna(_wind_v):
                _env_note += f" wind {_wind_v:.0f}mph"
            _precip_v = r.get("precip_pct")
            if pd.notna(_precip_v) and _precip_v >= 0:
                _env_note += f" precip {_precip_v:.0f}%"
            if bool(r.get("rain_risk_flag")):
                _env_note += " [RAIN RISK -- postponement/delay possible]"
        elif _hkey2 in _DOME_TEAMS and not bool(r.get("is_neutral_site")):
            _env_note = " | dome/indoor"
        elif pd.notna(_pf):
            _env_note = f" | park x{_pf:.2f}"
        if bool(r.get("is_neutral_site")):
            _venue_nm = r.get("venue_name_actual", "")
            _env_note += f" | NEUTRAL SITE ({_venue_nm})" if _venue_nm else " | NEUTRAL SITE"
        _cmp = (globals().get("mlb_totals_decomp") or {}).get((str(r.get(away_col, "")).lower().strip(), str(r.get(home_col, "")).lower().strip()))
        _cmp_note = (f" [old run formula -- feeds RL/score display only; totals use the component model: {_cmp['total']:.1f}, see TOTALS DECOMP]"
                     if _cmp else "")
        return f"  Proj Score : {r[away_col]} {_ar:.1f}  --  {r[home_col]} {_hr:.1f}  (O/U {_tot:.1f}){_env_note}{_cmp_note}"
    return None

def _unit_label(edge, model_prob=None):
    if model_prob is not None and float(model_prob) < 0.60:
        return "0.5u"  # below confidence floor -- will be filtered to Watch
    # 2026-07-27: removed the sharp-money hard gate. It checked a variable
    # (mlb_sharp_pct) that no producer cell ever set -- the Action Network
    # public-betting-% fetch (old "Cell 85") only ever populated mlb_bet_pcts,
    # a different name -- so this check was permanently False and every play,
    # every day, regardless of real edge quality, got capped at 0.5u. Sizing
    # now runs on edge alone.
    # BUG FIX (MLB-EDGE-BLEND-1, 2026-09-25 user audit): these tiers were
    # calibrated on the OLD raw (unblended) edge scale -- edge_blended =
    # w*edge_raw whenever a real market price exists (see the blend at
    # the top of this cell), so scaling each cutoff by the same
    # _MODEL_MARKET_BLEND_W preserves their original relative selectivity
    # under the new scale instead of guessing at new numbers. Left as
    # explicit multiplications (not pre-computed constants) so the old
    # raw-scale cutoffs stay visible for reference.
    _w = globals().get("_MODEL_MARKET_BLEND_W", 1.0)
    if edge >= 0.22 * _w: return "2.0u"
    if edge >= 0.17 * _w: return "1.5u"
    if edge >= 0.15 * _w: return "1.0u"
    # 2026-08-10: this used to say "filtered to Watch" -- true for a
    # candidate that never passed passes_edge_rule (the actual watch-list
    # section below filters ON this string), but NOT true for a real
    # official top_market row (BASE_EDGE_MIN=0.12 now qualifies edges down
    # to 12%, same tier the Pitcher Advantage Exception already used) --
    # those are real, sized-down official plays, not watch-only.
    return "0.5u"

def _mlb_elo_avail(bet_team):
    """Whether this team's ELO is real, not the 1500.0 default.
    Fix: the old inline version compared a LOWERCASED search term against the
    mlb_team_off_elo dict's ORIGINAL-CASE keys ("Chicago Cubs") -- `"chicago" in
    "Chicago Cubs"` is False in Python (case-sensitive `in`), so this fuzzy match
    silently failed for every team, not just Cubs/Rockies; those were just the two
    plays on the card that day. Lowercase both sides of the comparison now.
    Also the single shared source of truth for this check -- the grade section and
    the Value Watch section were computing (or not computing) this independently,
    which is how a play could show B+ [ELO UNAVAILABLE] in one section and
    "[below 15% threshold]" in the other for the exact same root cause.
    """
    if globals().get("_elo_unavailable_flag"):
        return False
    if "mlb_team_off_elo" not in globals():
        return True
    _bt = str(bet_team).lower()
    _elo_dict = globals()["mlb_team_off_elo"]
    # Fix: matching on ANY word >3 chars is ambiguous for every city that hosts two
    # teams -- "chicago" matches both Cubs and White Sox, "los angeles" matches both
    # Dodgers and Angels, "new york" matches both Mets and Yankees. An exact match
    # first, then a mascot-only (last word) fallback, since the mascot is always
    # unambiguous even when the city isn't.
    _k = next((k for k in _elo_dict if k.lower() == _bt), "")
    if not _k:
        _mascot = _bt.split()[-1] if _bt.split() else ""
        if len(_mascot) > 3:
            _k = next((k for k in _elo_dict if k.lower().endswith(_mascot)), "")
    if not _k:
        # Alias tier for known short/nickname mismatches (e.g. ESPN's "D-backs"
        # vs the box-score's "Diamondbacks" -- neither is a suffix of the other).
        _alias = _MLB_TEAM_ALIASES.get(_bt) or _MLB_TEAM_ALIASES.get(_bt.split()[-1] if _bt.split() else "")
        if _alias:
            _k = next((k for k in _elo_dict if k.lower().endswith(_alias)), "")
    _val = _elo_dict.get(_k, 1500.0)
    if abs(float(_val) - 1500.0) <= 10:
        # 2026-07-27 fix: near-1500 alone isn't proof of missing data --
        # _mlb_elo_diag() below already reasons about game count for this exact
        # case ("near-1500 is a real, settled rating" once the log has enough
        # games) but that conclusion never fed back into this gate, so real
        # plays (Cardinals, Pirates, 2026-07-26) got an [ELO UNAVAILABLE] grade
        # cap the diagnostic text printed right next to them said didn't apply.
        _ELO_GAMECOUNT_FLOOR = 15
        _gc = globals().get("mlb_team_elo_gamecount", {}).get(_k)
        if _gc is not None and _gc >= _ELO_GAMECOUNT_FLOOR:
            return True  # settled rating, not a data gap
        return False
    return True

def _mlb_elo_diag(bet_team):
    """Human-readable reason _mlb_elo_avail() returned False -- distinguishes
    'never found this team in mlb_team_off_elo at all' (real data gap, check the
    Cell 75 ELO game log / cache) from 'found it, but the computed ELO genuinely
    landed within 10 of 1500' (not a bug, just an average team) -- both currently
    produce an identical-looking [ELO UNAVAILABLE], which is why it wasn't
    possible to tell the two apart without inspecting mlb_team_off_elo by hand.
    """
    if globals().get("_elo_unavailable_flag"):
        return "whole ELO feed unavailable"
    if "mlb_team_off_elo" not in globals():
        return "mlb_team_off_elo not loaded"
    _bt = str(bet_team).lower()
    _elo_dict = globals()["mlb_team_off_elo"]
    _k = next((k for k in _elo_dict if k.lower() == _bt), "")
    if not _k:
        _mascot = _bt.split()[-1] if _bt.split() else ""
        if len(_mascot) > 3:
            _k = next((k for k in _elo_dict if k.lower().endswith(_mascot)), "")
    if not _k:
        _alias = _MLB_TEAM_ALIASES.get(_bt) or _MLB_TEAM_ALIASES.get(_bt.split()[-1] if _bt.split() else "")
        if _alias:
            _k = next((k for k in _elo_dict if k.lower().endswith(_alias)), "")
    if not _k:
        # Near-miss diagnostic: any key sharing a word with the search term, so
        # the real stored key format is visible instead of a silent 1500 default.
        _bt_words = [w for w in _bt.split() if len(w) > 3]
        _near = [k for k in _elo_dict if any(w in k.lower() for w in _bt_words)]
        _near_str = f" -- near-miss keys: {_near[:5]}" if _near else " -- no near-miss keys found either"
        return f"no key found for '{bet_team}' in mlb_team_off_elo -- likely missing from the ELO game log, not a matching bug{_near_str}"
    _val = float(_elo_dict.get(_k, 1500.0))
    # 2026-07-12: this used to hedge "genuinely near-average, or the log has too
    # few games" as a single message covering both cases -- no way to tell which
    # applied without inspecting mlb_team_off_elo by hand. mlb_team_elo_gamecount
    # (cell 83) now gives a real answer: below the sample-size floor means the
    # ELO random walk hasn't had enough games to move meaningfully away from the
    # 1500 starting point regardless of true team quality; at or above it means
    # the walk had a real chance to move and landed near 1500 anyway, which is
    # actual signal, not a data gap.
    _ELO_GAMECOUNT_FLOOR = 15
    _gc = globals().get("mlb_team_elo_gamecount", {}).get(_k)
    if _gc is not None and _gc < _ELO_GAMECOUNT_FLOOR:
        _gc_reason = f"only {_gc} games in the ELO log for this team (below the {_ELO_GAMECOUNT_FLOOR}-game floor) -- likely a real data gap, not a settled rating"
    elif _gc is not None:
        _gc_reason = f"{_gc} games in the ELO log -- near-1500 here is a real, settled rating (genuinely average team), not insufficient data"
    else:
        _gc_reason = "game count unavailable -- genuinely near-average, or the log has too few games for this team"
    return f"matched key '{_k}', ELO={_val:.1f} -- within 10 of the 1500.0 default ({_gc_reason})"

# ---------------------------------------------------------
# Why-bullet builder
# ---------------------------------------------------------
def _safe_f(v):
    try:
        f = float(v)
        return None if (f != f) else f
    except Exception:
        return None

# MANUAL BULLPEN-EXPOSURE WATCHLIST (temporary bridge, 2026-08-04;
# SUPERSEDED 2026-08-08 -- the live consumer, Cell 89's FINAL CARD + WHY
# loop, no longer reads this at all, replaced by a real three-state read
# of mlb_sp_bullpen_exposure itself (CONFIRMED-STARTER / TBD / LOOKUP-
# FAILED) that needs no name list. Left defined here only because the
# dead _why_bullets() copy of this logic a few lines below still
# references it -- harmless, but don't add new entries; they won't do
# anything on the real card anymore.) Original note: the
# automatic mlb_sp_bullpen_exposure flag (Cell 83) needs 3+ tracked
# starts before it can fire. The ELO cache backfill for a swingman's
# PAST bulk-relief outings was deliberately left alone rather than
# force-rebuilt (full-season refetch cost not worth it for one
# pitcher -- 2026-08-04 decision), so a real bulk/piggyback arm can
# stay under-sampled for a while even after showing the pattern live.
# Add a pitcher's LAST NAME here (lowercase) ONLY once you've manually
# confirmed his bulk-relief usage for TODAY's specific game -- this is
# deliberately scoped to today's slate (matched against the live
# resolved pick_sp for each game), not a standing roster note, because
# a common surname isn't safe to key off for the whole season without
# a team/date to disambiguate (see the Eddy Yean/Andrew Alvarez mixup
# already logged earlier in this cell -- same failure mode). Remove an
# entry once mlb_sp_bullpen_exposure shows 3+ real tracked starts for
# him; the automatic flag takes over from there and this manual one
# becomes redundant.
MANUAL_BULLPEN_EXPOSURE_WATCHLIST = {
    "alvarez",  # 2026-08-04: confirmed mix of starts + bulk relief; tracked sample still <3 starts
}
# MANUAL_BULLPEN_EXPOSURE_WATCHLIST is also consumed directly by Cell 89's
# FINAL CARD + WHY loop (the real bullpen-exposure/ELO-ERA-divergence
# checks live there now -- see note on _why_bullets below), so keep this
# dict defined here regardless of _why_bullets' own status.

# DEAD CODE WARNING (found + left as-is, 2026-08-04): _why_bullets() is
# defined but never called anywhere in this cell or Cell 89 -- grepped
# both cells for "_why_bullets(" and the only match is this def line
# itself. The real "FINAL CARD -- OFFICIAL PLAYS + WHY" output Cell 89
# prints reuses _pitcher_note()/_score_line() plus its own separate inline
# injury/bullpen-risk/team-form logic, NOT this function. The bullpen-
# exposure check, the ELO-vs-ERA divergence check, and the manual
# watchlist fallback that were originally added to this function (2026-
# 08-04) were consequently invisible on every real card -- they've since
# been ported into Cell 89's actual loop, where they do run. Left this
# function's body untouched rather than deleting it (some other future
# session may have wired it in for a reason not visible from here, or may
# intend to actually call it) -- but treat anything added ONLY here as
# dead until something calls _why_bullets(), which nothing currently does.
def _why_bullets(r):
    bullets = []
    pick_is_home = (str(r.get("bet_side", "")).lower() == "home")
    pick_short = str(r.get(home_col if pick_is_home else away_col, "")).split()[-1]
    fade_short = str(r.get(away_col if pick_is_home else home_col, "")).split()[-1]

    # Pitching
    pick_sp_col = "home_starter_runs_allowed_avg_10" if pick_is_home else "away_starter_runs_allowed_avg_10"
    fade_sp_col = "away_starter_runs_allowed_avg_10" if pick_is_home else "home_starter_runs_allowed_avg_10"
    _pick_sp_cols = ["home_pitcher_final_name","home_pitcher_name","home_starter"] if pick_is_home else ["away_pitcher_final_name","away_pitcher_name","away_starter"]
    _fade_sp_cols = ["away_pitcher_final_name","away_pitcher_name","away_starter"] if pick_is_home else ["home_pitcher_final_name","home_pitcher_name","home_starter"]
    pick_sp = next((str(r.get(c) or "").strip() for c in _pick_sp_cols if r.get(c) and str(r.get(c)) != "nan"), "")
    fade_sp = next((str(r.get(c) or "").strip() for c in _fade_sp_cols if r.get(c) and str(r.get(c)) != "nan"), "")
    pick_ra = _safe_f(r.get(pick_sp_col))
    fade_ra = _safe_f(r.get(fade_sp_col))
    if pick_ra is not None and fade_ra is not None:
        _pl = f"{pick_sp} ({pick_ra:.1f} R/G L10)" if pick_sp and pick_sp != "nan" else f"{pick_ra:.1f} R/G L10"
        _fl = f"{fade_sp} ({fade_ra:.1f} R/G L10)" if fade_sp and fade_sp != "nan" else f"{fade_ra:.1f} R/G L10"
        bullets.append(f"Pitching: {_pl} vs {_fl}")
    elif pick_sp and pick_sp != "nan" and fade_sp and fade_sp != "nan":
        bullets.append(f"Pitching: {pick_sp} vs {fade_sp}")

    # Offense — prefer projected runs against today's SP over raw L10 avg
    # Raw L10 is misleading when facing an elite SP (e.g. Pirates 5.2 L10 vs Miller 1.31 ERA)
    pick_proj = _safe_f(r.get("pred_home_runs" if pick_is_home else "pred_away_runs"))
    fade_proj = _safe_f(r.get("pred_away_runs" if pick_is_home else "pred_home_runs"))
    pick_off  = _safe_f(r.get("home_runs_for_avg_10" if pick_is_home else "away_runs_for_avg_10"))
    fade_off  = _safe_f(r.get("away_runs_for_avg_10" if pick_is_home else "home_runs_for_avg_10"))
    if pick_proj is not None and fade_proj is not None:
        # Use projected runs (SP-adjusted) as primary, show L10 avg in parentheses
        _proj_caution = "  [CAUTION: pick projected below fade]" if pick_proj < fade_proj - 0.3 else ""
        bullets.append(f"Offense (projected vs today's SP): {pick_short} {pick_proj:.1f}R vs {fade_short} {fade_proj:.1f}R{_proj_caution}")
        if pick_off is not None and fade_off is not None:
            bullets.append(f"  (L10 raw: {pick_short} {pick_off:.1f} vs {fade_short} {fade_off:.1f} — context only)")
    elif pick_off is not None and fade_off is not None:
        _off_caution = "  [CAUTION: pick team scoring less recently]" if pick_off < fade_off - 0.5 else ""
        bullets.append(f"Offense: {pick_short} avg {pick_off:.1f} R/G vs {fade_short} avg {fade_off:.1f} R/G (L10){_off_caution}")
    pick_def = _safe_f(r.get("home_runs_against_avg_10" if pick_is_home else "away_runs_against_avg_10"))
    fade_def = _safe_f(r.get("away_runs_against_avg_10" if pick_is_home else "home_runs_against_avg_10"))
    if pick_def is not None and fade_def is not None:
        _def_tag = "  [CAUTION: pick staff allowing more runs]" if pick_def > fade_def + 0.5 else ""
        bullets.append(f"Staff RA/G: {pick_short} {pick_def:.1f} vs {fade_short} {fade_def:.1f} (L10){_def_tag}")

    # Form
    pick_mg = _safe_f(r.get("home_margin_avg_10" if pick_is_home else "away_margin_avg_10"))
    if pick_mg is not None:
        _bet_full_l10 = str(r.get("bet_team", "")).lower()
        _l10_hit = next(
            (v for k, v in _l10_records.items()
             if any(w in k for w in _bet_full_l10.split() if len(w) > 3)),
            None
        )
        _l10_str = f" | L10: {_l10_hit['w']}-{_l10_hit['l']}" if _l10_hit else ""
        _cold_tag = "  *** COLD STREAK" if _l10_hit and _l10_hit["w"] <= 3 else ""
        _form_caution = "  [CAUTION: cold — pitching-driven pick]" if pick_mg < 0 else ""
        # Home/road split display
        _fb_prec = next((v for k,v in _team_records.items() if any(w in k for w in pick_short.lower().split() if len(w)>3)), None)
        if _fb_prec and pick_is_home and (_fb_prec["home_w"]+_fb_prec["home_l"]) > 0:
            _hpct = _fb_prec["home_w"]/(_fb_prec["home_w"]+_fb_prec["home_l"])
            _l10_str += f" | Home: {_fb_prec['home_w']}-{_fb_prec['home_l']} ({_hpct:.0%})"
        elif _fb_prec and not pick_is_home and (_fb_prec["road_w"]+_fb_prec["road_l"]) > 0:
            _rpct = _fb_prec["road_w"]/(_fb_prec["road_w"]+_fb_prec["road_l"])
            _l10_str += f" | Road: {_fb_prec['road_w']}-{_fb_prec['road_l']} ({_rpct:.0%})"
        _streak_tag = ""
        if _fb_prec and _fb_prec.get("streak",""):
            _streak_tag = f" | Streak: {_fb_prec['streak']}"
        bullets.append(f"Form: {pick_short} at {pick_mg:+.1f} run margin avg (L10){_l10_str}{_streak_tag}{_form_caution}{_cold_tag}")

    # SP unresolved -- PENDING LINEUP (2026-07-31): no starter name was
    # found at all for one side (harder-block case than DEFAULT ELO below,
    # which fires when a name resolved but wasn't in the ELO dict).
    if r.get("sp_unresolved"):
        bullets.append("!! PENDING LINEUP -- SP UNCONFIRMED for one or both teams, play blocked until starter resolves")

    # SP lookup failed (2026-08-07, MLB Priority 2 audit): starter name
    # was found but matched no ELO/stats data anywhere (see Gate 7b
    # above) -- distinct from PENDING LINEUP (name missing entirely).
    if r.get("sp_elo_weak"):
        bullets.append("!! SP LOOKUP FAILED -- starter name found but no ELO/stats match (DEFAULT 1460 rating), play blocked until data resolves")

    # Opp SP DEFAULT ELO caution
    _opp_sp_name_cols = (["away_pitcher_final_name","away_pitcher_name","away_starter"] if pick_is_home
                         else ["home_pitcher_final_name","home_pitcher_name","home_starter"])
    _opp_sp_nm = next((str(r.get(c) or "").strip() for c in _opp_sp_name_cols
                       if r.get(c) and str(r.get(c)) != "nan"), "")
    if _opp_sp_nm:
        try:
            _opp_elo_col = "away_sp_elo_adj" if pick_is_home else "home_sp_elo_adj"
            _opp_elo_raw = r.get(_opp_elo_col)
            _opp_elo_v = float(_opp_elo_raw) if (_opp_elo_raw is not None and str(_opp_elo_raw) not in ("", "nan")) else 1500.0
            if _opp_elo_v == 1460.0:
                bullets.append(f"!! Opp SP ({_opp_sp_nm}) DEFAULT ELO � edge may be inflated, verify starter")
            elif _opp_elo_v < 1440:
                _cur_edge = _safe_f(r.get("edge"))
                if _cur_edge is not None and _cur_edge >= 0.18:
                    bullets.append(f"!! LARGE EDGE + WEAK OPP SP ({_opp_sp_nm}, ELO {_opp_elo_v:.0f}) � verify L3 ERA data; stale lookup can inflate edge")
        except Exception:
            pass

    # SP ELO-vs-ERA DIVERGENCE (audit fix, 2026-08-04): the ELO tier label
    # is FIP-based and opponent-adjusted, which CAN legitimately diverge
    # from raw ERA -- but a large, unexplained gap between what a
    # pitcher's real season ERA implies (via the same _era_implied_elo()
    # formula the ERA-recency blend above already uses) and his actual
    # rated ELO means the tier label alone isn't a safe stand-in for ERA
    # in the "why" panel. Real case found live 2026-08-03: Quantrill
    # showed WEAK(<1440) ELO despite a 3.79 season ERA -- ERA-implied ELO
    # for 3.79 is ~1607, a 165+pt gap from the FIP/ERA disagreement alone.
    # Checks both sides (pick's own SP and the opponent's), fully self-
    # contained (doesn't depend on the Opp SP DEFAULT ELO block above
    # having successfully resolved anything).
    try:
        if "_era_implied_elo" in dir() and "_sp_form_cache" in globals():
            _ed_GAP_FLAG = 120.0
            def _ed_norm(s):
                import unicodedata as _ed_ud, re as _ed_re
                s = _ed_ud.normalize("NFD", str(s))
                s = "".join(ch for ch in s if _ed_ud.category(ch) != "Mn")
                return _ed_re.sub(r"[^a-z ]", "", s.lower()).strip()
            def _ed_era_for(nm):
                _k = _ed_norm(nm)
                _fc = _sp_form_cache.get(_k)
                if not _fc:
                    for _ck, _cv in _sp_form_cache.items():
                        if _cv and _ed_norm(_ck) == _k:
                            _fc = _cv
                            break
                if not _fc or _fc.get("era_season") is None:
                    return None
                _e = float(_fc["era_season"])
                return _e if 0 < _e <= 12 else None
            def _ed_check(sp_name, elo_val, side_label):
                if not sp_name or sp_name == "nan" or elo_val is None or elo_val == 1500.0:
                    return
                _era = _ed_era_for(sp_name)
                if _era is None:
                    return
                _implied = _era_implied_elo(_era)
                _gap = float(elo_val) - _implied
                if abs(_gap) >= _ed_GAP_FLAG:
                    _dir = "higher" if _gap > 0 else "lower"
                    bullets.append(
                        f"!! ELO-ERA DIVERGENCE ({side_label}): {sp_name} ELO ({elo_val:.0f}) reads {_dir} than "
                        f"his {_era:.2f} ERA implies (~{_implied:.0f}) -- {abs(_gap):.0f}pt gap; verify tier before trusting SP edge"
                    )
            _ed_pick_elo_col = "home_sp_elo_adj" if pick_is_home else "away_sp_elo_adj"
            _ed_pick_elo_raw = r.get(_ed_pick_elo_col)
            _ed_pick_elo = float(_ed_pick_elo_raw) if (_ed_pick_elo_raw is not None and str(_ed_pick_elo_raw) not in ("", "nan")) else None
            _ed_check(pick_sp, _ed_pick_elo, "pick SP")
            if _opp_sp_nm:
                _ed_opp_elo_col = "away_sp_elo_adj" if pick_is_home else "home_sp_elo_adj"
                _ed_opp_elo_raw = r.get(_ed_opp_elo_col)
                _ed_opp_elo = float(_ed_opp_elo_raw) if (_ed_opp_elo_raw is not None and str(_ed_opp_elo_raw) not in ("", "nan")) else None
                _ed_check(_opp_sp_nm, _ed_opp_elo, "opp SP")
    except Exception:
        pass

    # SP BULLPEN EXPOSURE (audit fix, moderate priority): a starter who
    # rarely reaches 5 IP gets treated by the model like a full 6+ IP
    # starter for edge purposes, which overstates confidence in the SP
    # matchup -- most of the game is really a bullpen game once he is
    # pulled early. Flagged off real per-start IP history (Cell 83's
    # mlb_sp_tracker / mlb_sp_bullpen_exposure), not innings limits
    # guessed from a role label.
    if pick_sp and pick_sp != "nan" and "mlb_sp_bullpen_exposure" in globals():
        import unicodedata as _be_ud, re as _be_re
        def _be_norm(s):
            if not s: return ''
            s = _be_ud.normalize('NFD', str(s))
            s = ''.join(c for c in s if _be_ud.category(c) != 'Mn')
            return _be_re.sub(r'[^a-z ]', '', s.lower()).strip()
        _be_key = _be_norm(pick_sp)
        _be_rec = mlb_sp_bullpen_exposure.get(pick_sp)
        if _be_rec is None:
            for _be_nm, _be_v in mlb_sp_bullpen_exposure.items():
                if _be_norm(_be_nm) == _be_key:
                    _be_rec = _be_v
                    break
        if _be_rec and _be_rec["starts"] >= 3 and _be_rec["five_ip_pct"] < 0.20:
            bullets.append(
                f"!! BULLPEN EXPOSURE: {pick_sp} has reached 5 IP in only "
                f"{_be_rec['five_ip_starts']}/{_be_rec['starts']} starts "
                f"({_be_rec['five_ip_pct']:.0%}) -- treat SP edge as partial-game only"
            )
        elif pick_sp.strip():
            _be_last = pick_sp.strip().split()[-1].lower()
            if _be_last in MANUAL_BULLPEN_EXPOSURE_WATCHLIST:
                _be_sample_note = (
                    f" ({_be_rec['starts']} tracked start(s) so far)" if _be_rec else " (no tracked starts yet)"
                )
                bullets.append(
                    f"!! BULLPEN EXPOSURE (manual flag): {pick_sp} has shown a mix of starts + "
                    f"bulk relief this season{_be_sample_note} -- automatic tracking sample still "
                    f"too thin to confirm; treat SP edge as partial-game only until confirmed"
                )

    # Model conviction
    prob = _safe_f(r.get("p_bet_win_model"))
    edge = _safe_f(r.get("edge"))
    # Derive tier from bet team probability (strength_tier col = home team only)
    if prob is not None:
        _p = float(prob)
        if _p >= 0.60:   tier = "A"
        elif _p >= 0.58: tier = "B"
        elif _p >= 0.55: tier = "C"
        else:            tier = "D"
        tier_str = f" | Tier {tier}"
        edge_str = f" | Edge {edge:.1%}" if edge is not None else ""
        bullets.append(f"Model: {prob:.1%} win prob{edge_str}{tier_str}")

    return bullets[:8]


# ── F5 ML ODDS FETCH ─────────────────────────────────────────────────────
# Fetches first-5-innings moneyline from Odds API.
# F5 ML captures starter-driven edge without absorbing bullpen variance.
_f5_odds = {}  # {matchup_norm: {"home_ml": int, "away_ml": int, "home_nv": float, "away_nv": float}}
_f5_market_live = False  # True only when real market F5 odds loaded
# MLB-F5-FETCH-1 (2026-10-06, R-F5-1 support -- "Research couldn't price the F5 because the card had no F5 line"):
# the old fetch asked the bulk /odds endpoint for "h2h_first_5_innings" -- wrong key AND period markets are only
# served per event -- so it 422'd every run and fell back to "synthetic". Now: /events, then /events/{id}/odds
# with markets=h2h_1st_5_innings (2-way; a 5-inning tie is a push). Pinnacle when posted, else the median
# no-vig across books (book list kept).
try:
    import requests as _f5_req, statistics as _f5_st
    _f5_k = os.environ.get("ODDS_API_KEY_FALLBACK", "") or os.environ.get("ODDS_API_KEY", "")
    _f5_evs = _f5_req.get("https://api.the-odds-api.com/v4/sports/baseball_mlb/events", params={"apiKey": _f5_k}, timeout=15)
    _f5_evs = _f5_evs.json() if _f5_evs.ok else []
    _f5_today = pd.Timestamp.now(tz="UTC")
    def _f5ai(ml):
        m = float(ml); return 100 / (m + 100) if m > 0 else abs(m) / (abs(m) + 100)
    for _f5ev in _f5_evs:
        _ct = pd.Timestamp(_f5ev.get("commence_time"))
        if not (_f5_today - pd.Timedelta(hours=4) <= _ct <= _f5_today + pd.Timedelta(hours=36)):
            continue
        _r = _f5_req.get(f"https://api.the-odds-api.com/v4/sports/baseball_mlb/events/{_f5ev['id']}/odds",
                         params={"apiKey": _f5_k, "regions": "us,eu", "markets": "h2h_1st_5_innings", "oddsFormat": "american"},
                         timeout=15)
        if not _r.ok:
            continue
        _f5_home, _f5_away = str(_f5ev["home_team"]).lower(), str(_f5ev["away_team"]).lower()
        _books = {}
        for _bm in _r.json().get("bookmakers", []):
            for _mk in _bm.get("markets", []):
                if _mk.get("key") != "h2h_1st_5_innings":
                    continue
                _o = {str(o["name"]).lower(): int(o["price"]) for o in _mk.get("outcomes", []) if o.get("price") is not None}
                if _f5_home in _o and _f5_away in _o:
                    _rh, _ra = _f5ai(_o[_f5_home]), _f5ai(_o[_f5_away])
                    _books[_bm["key"]] = (_o[_f5_home], _o[_f5_away], _rh / (_rh + _ra))
        if not _books:
            continue
        if "pinnacle" in _books:
            _bk = "pinnacle"; _hml, _aml, _hnv = _books["pinnacle"]
        else:
            _hnv = _f5_st.median(v[2] for v in _books.values())
            _bk = min(_books, key=lambda b: abs(_books[b][2] - _hnv))
            _hml, _aml, _ = _books[_bk]
        _f5_odds[(_f5_away, _f5_home)] = {"home_ml": _hml, "away_ml": _aml, "home_nv": round(_hnv, 4),
                                          "away_nv": round(1 - _hnv, 4), "book": _bk, "n_books": len(_books)}
    _f5_market_live = len(_f5_odds) > 0
    print(f"  [F5 ML] real first-5 odds for {len(_f5_odds)} game(s) (per-event h2h_1st_5_innings)")
except Exception as _f5e:
    print(f"  [F5 ML] Fetch failed: {_f5e}")

# ── BULLPEN HOLD% + BLOWN SAVE RATE ───────────────────────────────────
# Holds, blown saves, saves per team (used to flag teams with poor 7th-8th
# inning bridges). MLB Stats API is now the PRIMARY source -- FanGraphs'
# reliever leaderboard is behind a confirmed permanent Cloudflare managed
# challenge (verified live, including with cloudscraper), not a UA-string
# block, so attempting it first was pure wasted latency every single run.
# FanGraphs is kept as a secondary attempt only if the MLB Stats API call
# itself fails outright (network/outage), not because its data is worse.
_bp_holds  = {}  # {team_norm: hold_rate}  holds per 9 opportunities
_bp_blowns = {}  # {team_norm: blown_save_pct}
import unicodedata as _fg_ud, re as _fg_re
def _fg_tn(s):
    return _fg_re.sub(r"[^a-z ]","",_fg_ud.normalize("NFD",str(s)).encode("ascii","ignore").decode().lower()).strip()
try:
    import requests as _mst_req
    _mst_teams = _mst_req.get(
        "https://statsapi.mlb.com/api/v1/teams?sportId=1&season=2026", timeout=15
    ).json().get("teams", [])
    for _mst_t in _mst_teams:
        _mst_tid = _mst_t.get("id")
        _mst_tkey = _fg_tn(_mst_t.get("name", ""))
        if not _mst_tid or not _mst_tkey:
            continue
        try:
            _mst_stats = _mst_req.get(
                f"https://statsapi.mlb.com/api/v1/teams/{_mst_tid}/stats"
                f"?stats=season&group=pitching&season=2026", timeout=10
            ).json()
            _mst_splits = _mst_stats.get("stats", [{}])[0].get("splits", [])
            if not _mst_splits:
                continue
            _mst_stat = _mst_splits[0].get("stat", {})
            _mst_hld = float(_mst_stat.get("holds", 0) or 0)
            _mst_bs  = float(_mst_stat.get("blownSaves", 0) or 0)
            _mst_sv  = float(_mst_stat.get("saves", 0) or 0)
            _mst_svopp = max(1.0, _mst_sv + _mst_bs)
            _bp_holds[_mst_tkey]  = round(_mst_hld, 1)
            _bp_blowns[_mst_tkey] = round(_mst_bs / _mst_svopp, 3)
        except Exception:
            continue
    if not _bp_holds:
        raise RuntimeError("MLB Stats API returned no bullpen data")
    print(f"  [BULLPEN] MLB Stats API loaded Hold/BS data for {len(_bp_holds)} teams")
except Exception as _mste:
    print(f"  [BULLPEN] MLB Stats API unavailable ({_mste}) -- trying FanGraphs as secondary")
    _bp_holds = {}; _bp_blowns = {}
    try:
        import requests as _fg_req
        _fg_url = ("https://www.fangraphs.com/api/leaders/major-league/data"
                   "?pos=all&stats=rel&lg=all&qual=0&season=2026&season1=2026"
                   "&month=0&hand=&team=0&pageitems=500&pagenum=1&ind=0&rost=0"
                   "&players=0&type=8&postseason=&sortdir=desc&sortstat=HLD")
        import random as _fg_rnd
        _fg_ua_pool = [
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36",
            "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
            "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36",
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:125.0) Gecko/20100101 Firefox/125.0",
        ]
        _fg_headers = {
            "User-Agent": _fg_rnd.choice(_fg_ua_pool),
            "Referer": "https://www.fangraphs.com/leaders/major-league",
            "Accept": "application/json, text/plain, */*",
            "Accept-Language": "en-US,en;q=0.9",
        }
        _fg_r = _fg_req.get(_fg_url, headers=_fg_headers, timeout=15)
        if _fg_r.status_code == 200:
            _fg_data = _fg_r.json().get("data",[])
            _team_hld  = {}; _team_bs = {}; _team_sv = {}
            for _fg_row in _fg_data:
                _tkey = _fg_tn(str(_fg_row.get("Team","")))
                if not _tkey: continue
                _hld = float(_fg_row.get("HLD",0) or 0)
                _bs  = float(_fg_row.get("BS",0) or 0)
                _sv  = float(_fg_row.get("SV",0) or 0)
                _team_hld.setdefault(_tkey,[]).append(_hld)
                _team_bs.setdefault(_tkey,[]).append(_bs)
                _team_sv.setdefault(_tkey,[]).append(_sv)
            for _t in set(list(_team_hld.keys())+list(_team_bs.keys())):
                _tot_hld = sum(_team_hld.get(_t,[0]))
                _tot_bs  = sum(_team_bs.get(_t,[0]))
                _tot_sv  = sum(_team_sv.get(_t,[0]))
                _sv_opp  = max(1, _tot_sv + _tot_bs)
                _bp_holds[_t]  = round(_tot_hld, 1)
                _bp_blowns[_t] = round(_tot_bs / _sv_opp, 3)
            print(f"  [BULLPEN] FanGraphs secondary loaded Hold/BS data for {len(_bp_holds)} teams")
        else:
            if _fg_r.status_code == 403:
                print("  [BULLPEN] FanGraphs 403 (Cloudflare challenge, not a UA block -- confirmed) -- bullpen data unavailable")
            else:
                print(f"  [BULLPEN] FanGraphs returned {_fg_r.status_code} -- bullpen data unavailable")
    except Exception as _fge:
        print(f"  [BULLPEN] FanGraphs secondary also failed: {_fge}")

print("\n" + "=" * 80)
print("EKO BETS -- MLB CARD")
print("=" * 80)

# Pull SP IL status from MLB transactions (21-day window)
_sp_il_flags = {}  # pitcher_name_lower -> note
try:
    import datetime as _dt
    _il_today = _dt.date.today().strftime("%Y-%m-%d")
    _il_from  = (_dt.date.today() - _dt.timedelta(days=21)).strftime("%Y-%m-%d")
    _il_url   = (f"https://statsapi.mlb.com/api/v1/transactions"
                 f"?sportId=1&startDate={_il_from}&endDate={_il_today}&limit=500")
    _il_r = requests.get(_il_url, headers={"User-Agent":"Mozilla/5.0"}, timeout=8)
    if _il_r.status_code == 200:
        for _tx in _il_r.json().get("transactions", []):
            _txtype = _tx.get("typeDesc", "").lower()
            _txplyr = _tx.get("player", {}).get("fullName", "")
            _txdate = _tx.get("date", "")[:10]
            if not _txplyr or "injured list" not in _txtype: continue
            _pk = _txplyr.lower()
            if "placed" in _txtype or "transferred" in _txtype:
                _sp_il_flags[_pk] = f"placed on IL {_txdate}"
            elif ("activated" in _txtype or "reinstated" in _txtype) and _pk not in _sp_il_flags:
                _sp_il_flags[_pk] = f"recently activated from IL {_txdate} -- may not be 100%"
except Exception:
    _sp_il_flags = {}

def _show_injuries(r):
    """Show IL/injury info + SP health flag for both teams in a pick."""
    # Positions that most impact game outcome
    _KEY_POS = {"SS", "C", "CF", "SP", "1B", "RF", "LF", "3B"}
    _a_team = str(r.get(away_col, ""))
    _h_team = str(r.get(home_col, ""))
    _a_inj = _fetch_espn_injuries(_a_team)
    _h_inj = _fetch_espn_injuries(_h_team)

    def _fmt_inj_list(inj_list, team_name):
        if not inj_list: return
        parts = []
        has_key = False
        for i in inj_list[:8]:
            _pos = str(i.get("pos", "")).upper()
            _key_tag = " [KEY ABSENCE]" if _pos in _KEY_POS else ""
            if _key_tag: has_key = True
            parts.append(f"{i['name']} ({i['status']}){_key_tag}")
        _prefix = "  !! Injuries" if has_key else "  Injuries"
        print(f"{_prefix} ({team_name}): {chr(124).join(parts)}")

    # Check manual overrides first, then ESPN API
    _manual = globals().get("MANUAL_INJURY_NOTES", {})
    for _mk, _mv in _manual.items():
        if _mk.lower() in _a_team.lower():
            print(f"  !! Injuries ({_a_team}) [MANUAL]: {chr(124).join(_mv)}")
            _a_inj = []  # suppress API results if manual provided
        if _mk.lower() in _h_team.lower():
            print(f"  !! Injuries ({_h_team}) [MANUAL]: {chr(124).join(_mv)}")
            _h_inj = []

    _fmt_inj_list(_a_inj, _a_team)
    _fmt_inj_list(_h_inj, _h_team)

    # Lineup-status news supplement (see _fetch_lineup_news def above) -- catches
    # day-to-day/activated/questionable/GTD players the IL-only feed above can't,
    # since none of those statuses are IL moves.
    for _news_team in (_a_team, _h_team):
        _news_hits = _fetch_lineup_news(_news_team)
        if _news_hits:
            print(f"  Lineup news ({_news_team}): {chr(124).join(_news_hits[:3])}")

    # SP health: check both starters against IL transaction log
    _sp_side_cols = [
        (["away_pitcher_final_name","away_pitcher_name","away_starter"], _a_team),
        (["home_pitcher_final_name","home_pitcher_name","home_starter"], _h_team),
    ]
    for _sp_cols, _side_lbl in _sp_side_cols:
        _sp_nm = next((str(r.get(c) or "").strip() for c in _sp_cols
                       if r.get(c) and str(r.get(c)) != "nan"), "").lower()
        if not _sp_nm: continue
        _sp_last = _sp_nm.split()[-1]
        for _ilk, _ilv in _sp_il_flags.items():
            if len(_sp_last) > 3 and _sp_last in _ilk:
                print(f"  !! SP HEALTH FLAG ({_side_lbl}): {_sp_nm.title()} -- {_ilv}")
                break

# __MLB_BP_PRECOMPUTE__
# ── MLB BULLPEN PRE-COMPUTATION (run once before card loop) ──────────────
import datetime as _bp_dt, urllib.request as _bp_ur, json as _bp_js

def _bp_get_json(url):
    try:
        _req = _bp_ur.Request(url, headers={"User-Agent": "Mozilla/5.0"})
        with _bp_ur.urlopen(_req, timeout=8) as _r:
            return _bp_js.loads(_r.read())
    except Exception:
        return {}

_bp_yday_str = (_bp_dt.date.today() - _bp_dt.timedelta(days=1)).strftime("%Y-%m-%d")
_bp_2day_str = (_bp_dt.date.today() - _bp_dt.timedelta(days=2)).strftime("%Y-%m-%d")

def _bp_relief_ids(date_str):
    """Return {team_abbr: set_of_reliever_ids} for all games on date_str."""
    from concurrent.futures import ThreadPoolExecutor, as_completed
    # Step 1: get gamePks for the date
    _sched = _bp_get_json(
        f"https://statsapi.mlb.com/api/v1/schedule?sportId=1&date={date_str}"
    )
    _dates_list = _sched.get("dates", [])
    _pks = [_g["gamePk"] for _d in _dates_list for _g in _d.get("games", [])]
    if not _pks:
        return {}
    # Step 2: fetch each boxscore in parallel
    def _fetch_bs(pk):
        return _bp_get_json(f"https://statsapi.mlb.com/api/v1/game/{pk}/boxscore")
    _out = {}
    with ThreadPoolExecutor(max_workers=8) as _ex:
        _futs = {_ex.submit(_fetch_bs, pk): pk for pk in _pks}
        for _fut in as_completed(_futs):
            _bs = _fut.result()
            _bt = _bs.get("teams", {})
            for _side in ("home", "away"):
                _tm = _bt.get(_side, {})
                _ab = _tm.get("team", {}).get("abbreviation", "")
                if not _ab:
                    continue
                _ids = set()
                for _pd in _tm.get("players", {}).values():
                    _st = _pd.get("stats", {}).get("pitching", {})
                    _ip = float(_st.get("inningsPitched", 0) or 0)
                    _gs = int(_st.get("gamesStarted", 0) or 0)
                    if _ip >= 0.1 and _gs == 0:
                        _ids.add(_pd.get("person", {}).get("id"))
                _out[_ab] = _ids
    return _out

print("  Fetching bullpen usage (yesterday + 2 days ago)...")
_rl_yday = _bp_relief_ids(_bp_yday_str)
_rl_2day = _bp_relief_ids(_bp_2day_str)
print(f"  Bullpen data: {len(_rl_yday)} teams yesterday, {len(_rl_2day)} teams 2 days ago")
# MLB-9: Series fatigue -- opp team used 4+ relievers yesterday → +0.3R boost
_series_fatigue = {}  # {team_norm: fatigue_R_boost}
if "_rl_yday" in globals() and _rl_yday:
    for _sf_team, _sf_pids in _rl_yday.items():
        if len(_sf_pids) >= 5:  # raised from 4+ (too common in MLB) to 5+
            _series_fatigue[_sf_team.lower()] = 0.2  # reduced from 0.3R
    if _series_fatigue:
        print(f"  [SERIES FATIGUE] {len(_series_fatigue)} team(s) used 5+ relievers yesterday: {list(_series_fatigue.keys())}")

# RELIEVER BACK-TO-BACK-DAY USAGE (2026-09-02, user audit M1/M12 -- Rockies
# 09-02 card, Romano/Bernardino): every existing bullpen signal (RISK,
# DEPTH, L3 ERA) is ERA/outcome-based -- none of them ask "did this SPECIFIC
# reliever actually pitch on both of the last two days". _rl_yday/_rl_2day
# just above already fetch exactly that (team_abbr -> set of reliever
# person IDs who appeared each day) for the series-fatigue count -- the
# per-pitcher identity was already being thrown away after just counting
# len(). Intersecting the two days' ID sets directly answers "who pitched
# both days" with no new data fetch, just reusing what's already pulled.
mlb_bp_b2b = {}  # team_abbr_lower -> [{"id":.., "name":..}, ...]
_B2B_RUN_ADJ = 0.15  # per pitcher, same conservative-cap family as the other bullpen signals
_B2B_MAX_R = 0.30
if "_rl_yday" in globals() and "_rl_2day" in globals() and _rl_yday and _rl_2day:
    _b2b_all_ids = set()
    _b2b_by_team = {}
    for _b2b_team in set(_rl_yday.keys()) & set(_rl_2day.keys()):
        _b2b_ids = _rl_yday[_b2b_team] & _rl_2day[_b2b_team]
        _b2b_ids.discard(None)
        if _b2b_ids:
            _b2b_by_team[_b2b_team] = _b2b_ids
            _b2b_all_ids.update(_b2b_ids)
    if _b2b_all_ids:
        try:
            _b2b_names_r = requests.get(
                f"https://statsapi.mlb.com/api/v1/people?personIds={','.join(str(i) for i in _b2b_all_ids)}",
                timeout=10)
            _b2b_names = {p["id"]: p.get("fullName", f"Player {p['id']}")
                          for p in _b2b_names_r.json().get("people", [])} if _b2b_names_r.status_code == 200 else {}
        except Exception:
            _b2b_names = {}
        for _b2b_team, _b2b_ids in _b2b_by_team.items():
            _b2b_pitchers = [{"id": _pid, "name": _b2b_names.get(_pid, f"Player {_pid}")} for _pid in _b2b_ids]
            _b2b_run_adj = round(min(_B2B_RUN_ADJ * len(_b2b_pitchers), _B2B_MAX_R), 3)
            mlb_bp_b2b[_b2b_team.lower()] = {"pitchers": _b2b_pitchers, "run_adj": _b2b_run_adj}
        print(f"  [BULLPEN B2B USAGE] {len(mlb_bp_b2b)} team(s) had a reliever pitch on both of the last 2 days: "
              + ", ".join(f"{t.upper()} ({', '.join(p['name'] for p in v['pitchers'])})" for t, v in mlb_bp_b2b.items()))
    else:
        print("  [BULLPEN B2B USAGE] no team had the same reliever appear on both of the last 2 days")

# M10 (2026-09-02 user audit -- Sugano @ BAL, pitched for Orioles in 2025
# 10-10 4.64 ERA): former-team matchups are a known motivational signal
# (pitchers often elevate against ex-clubs) with zero flag anywhere on the
# card. Batches ALL of today's probable starters into one MLB people call
# (hydrate=stats yearByYear) rather than one request per pitcher.
mlb_sp_former_team = {}  # normalized pitcher name -> [{"team_name":.., "season":.., "w":, "l":, "era":}, ...]
if "mlb_today_games_with_starters" in globals() and isinstance(mlb_today_games_with_starters, pd.DataFrame):
    _ft_pids = set()
    for _ft_col in ["away_probable_pitcher_id", "home_probable_pitcher_id"]:
        if _ft_col in mlb_today_games_with_starters.columns:
            _ft_pids.update(int(x) for x in mlb_today_games_with_starters[_ft_col].dropna().unique())
    if _ft_pids:
        try:
            _ft_r = requests.get(
                f"https://statsapi.mlb.com/api/v1/people?personIds={','.join(str(i) for i in _ft_pids)}"
                f"&hydrate=stats(group=[pitching],type=[yearByYear])",
                timeout=15)
            _ft_people = _ft_r.json().get("people", []) if _ft_r.status_code == 200 else []
        except Exception:
            _ft_people = []
        _ft_this_year = _bp_dt.date.today().year if "_bp_dt" in globals() else __import__("datetime").date.today().year
        for _ft_p in _ft_people:
            _ft_nk = _mlb_name_norm(_ft_p.get("fullName", "")) if "_mlb_name_norm" in globals() else str(_ft_p.get("fullName", "")).lower().strip()
            _ft_hist = []
            for _ft_stat_block in _ft_p.get("stats", []):
                for _ft_split in _ft_stat_block.get("splits", []):
                    _ft_season = _ft_split.get("season")
                    if not _ft_season or int(_ft_season) >= _ft_this_year:
                        continue  # only PRIOR seasons -- not this year's team
                    _ft_team = _ft_split.get("team", {}) or {}
                    _ft_stat = _ft_split.get("stat", {}) or {}
                    if not _ft_team.get("name"):
                        continue
                    _ft_hist.append({
                        "team_name": _ft_team.get("name"),
                        "season": int(_ft_season),
                        "w": _ft_stat.get("wins"),
                        "l": _ft_stat.get("losses"),
                        "era": _ft_stat.get("era"),
                    })
            if _ft_hist and _ft_nk:
                mlb_sp_former_team[_ft_nk] = _ft_hist
        print(f"  [FORMER TEAM] pitching history resolved for {len(mlb_sp_former_team)}/{len(_ft_pids)} probable starters")

# ── END BULLPEN PRE-COMPUTATION ──────────────────────────────────────────

# __MLB_SP_RECENT_FORM__ (audit fix, 2026-08-04): _sp_form_cache /
# _sp_norm / _sp_fetch_recent used to be DEFINED HERE, but every call site
# that uses them (the xFIP-blend loop, the dual-elite-SP totals
# suppressor, the SP-form pre-fetch loop, and _pitcher_note's inline
# ERA/WHIP display) sits EARLIER in this same cell, each behind an
# `if '_sp_fetch_recent' in dir()` guard the notebook's own prior comment
# already flagged as first-run-dead ("only available on subsequent runs
# in same session"). On a normal top-to-bottom run of this cell,
# `_sp_fetch_recent` did not exist yet at any of those call sites, so
# `_sp_form_cache` stayed permanently empty and every pitcher's ERA/WHIP
# silently came back missing -- not a name-matching failure (confirmed
# live 2026-08-03/04: this is why Nola showed ELO with no ERA/WHIP even
# after the fuzzy-name fallback fix). The real fix is definition order,
# not matching: _sp_form_cache / _sp_norm / _sp_fetch_recent are now
# defined near the TOP of this cell (right after the board is located)
# so every downstream call site -- including this one's former position
# -- can actually reach them on a first run. Do not re-add a second
# `_sp_form_cache = {}` here; that would reset the cache the early call
# sites already populated.

# __MLB_SP_H2H_VS_OPPONENT__
# ── SP CAREER H2H vs TODAYS OPPONENT (collapse-scenario escalation) ────
# 2026-07-24 fix: the model had zero SP-vs-specific-opponent weighting --
# only season-wide ERA/FIP/xFIP plus the recent-form (L3) collapse check
# below. A pitchers documented inability to get through one particular
# lineup can sit in plain sight in a pregame preview and never move the
# grade (real case: a 0-1, 5.73 ERA line in 2 career starts vs the exact
# opponent -- noted, not weighted). A 2-start sample is noise on its own,
# so this does NOT kill plays off it. It only escalates the existing
# collapse gate from moderate to high when the H2H sample also clears a
# real severity bar: >=2 starts vs this opponent AND ERA >=5.00.
_sp_h2h_opp_cache = {}
_sp_pid_cache = {}

def _sp_get_pid(name):
    _nk = _sp_norm(name)
    if _nk in _sp_pid_cache: return _sp_pid_cache[_nk]
    try:
        _q = str(name).replace(" ", "%20")
        _sr = requests.get(f"https://statsapi.mlb.com/api/v1/people/search?names={_q}&sportId=1", timeout=6)
        _people = _sr.json().get("people", []) if _sr.status_code == 200 else []
        _pid = next((p["id"] for p in _people if p.get("primaryPosition",{}).get("code") == "1"),
                     _people[0]["id"] if _people else None)
        _sp_pid_cache[_nk] = _pid
        return _pid
    except Exception:
        _sp_pid_cache[_nk] = None
        return None

def _sp_fetch_vs_opponent(name, opp_team_id, lookback_seasons=6):
    """Return {starts, era, ip} for a pitchers starts vs a specific opponent
    team, aggregated over the last `lookback_seasons` seasons (a career-ish
    window -- vs-opponent samples build up slowly, unlike recent form)."""
    if not name or not opp_team_id: return None
    _nk = f"{_sp_norm(name)}|{opp_team_id}"
    if _nk in _sp_h2h_opp_cache: return _sp_h2h_opp_cache[_nk]
    _pid = _sp_get_pid(name)
    if not _pid:
        _sp_h2h_opp_cache[_nk] = None; return None
    import datetime as _sph_dtmod
    _cur_year = _sph_dtmod.datetime.now().year
    _ip_tot = 0.0; _er_tot = 0; _n_starts = 0
    try:
        for _yr in range(_cur_year, _cur_year - lookback_seasons, -1):
            _lr = requests.get(
                f"https://statsapi.mlb.com/api/v1/people/{_pid}/stats"
                f"?stats=gameLog&group=pitching&season={_yr}&gameType=R",
                timeout=6
            )
            _splits = _lr.json().get("stats",[{}])[0].get("splits",[]) if _lr.status_code == 200 else []
            for _s in _splits:
                if int(_s.get("stat",{}).get("gamesStarted",0) or 0) != 1: continue
                if int((_s.get("opponent") or {}).get("id", -1) or -1) != int(opp_team_id): continue
                _ip_tot += float(_s.get("stat",{}).get("inningsPitched",0) or 0)
                _er_tot += int(_s.get("stat",{}).get("earnedRuns",0) or 0)
                _n_starts += 1
        _era = round(_er_tot / _ip_tot * 9, 2) if _ip_tot > 0 else None
        _res = {"starts": _n_starts, "era": _era, "ip": round(_ip_tot, 1)}
        _sp_h2h_opp_cache[_nk] = _res
        return _res
    except Exception:
        _sp_h2h_opp_cache[_nk] = None
        return None

print("  SP vs-opponent H2H fetcher ready")
# ── END SP H2H vs OPPONENT ──────────────────────────────────────────────────

# ── FETCH L10 WIN/LOSS RECORDS FROM ESPN STANDINGS ─────────────────────
# API: /apis/v2/ endpoint; L10 in stats array, name="Last Ten Games"
# __MLB_RECORD_GATE__
_l10_records  = {}   # {team_lower: {w, l}}
_team_records = {}   # {team_lower: {wins, losses, win_pct, home_w, home_l, road_w, road_l}}
try:
    _l10_resp = requests.get(
        "https://site.api.espn.com/apis/v2/sports/baseball/mlb/standings",
        timeout=8
    )
    if _l10_resp.status_code == 200:
        _l10_data = _l10_resp.json()
        for _l10_league in _l10_data.get("children", []):
            for _l10_entry in _l10_league.get("standings", {}).get("entries", []):
                _l10_tname = _l10_entry.get("team", {}).get("displayName", "").lower()
                _rec = {"wins":0,"losses":0,"win_pct":0.5,"home_w":0,"home_l":0,"road_w":0,"road_l":0,"runs_for":0,"runs_against":0,"pyth_pct":None,"streak":""}
                for _stat in _l10_entry.get("stats", []):
                    _sn = _stat.get("name", "")
                    _dv = _stat.get("displayValue", "")
                    _sv = _stat.get("value", 0)
                    if _sn == "Last Ten Games" and "-" in _dv:
                        try:
                            _pts = _dv.split("-")
                            _l10_records[_l10_tname] = {"w": int(_pts[0]), "l": int(_pts[1])}
                        except: pass
                    elif _sn == "wins":   _rec["wins"]   = int(_sv or 0)
                    elif _sn == "losses": _rec["losses"] = int(_sv or 0)
                    elif _sn == "winPercent": _rec["win_pct"] = float(_sv or 0.5)
                    elif _sn == "Home":  # "W-L" format
                        try: _hw,_hl=str(_dv).split("-"); _rec["home_w"]=int(_hw); _rec["home_l"]=int(_hl)
                        except: pass
                    elif _sn == "Road":
                        try: _rw,_rl=str(_dv).split("-"); _rec["road_w"]=int(_rw); _rec["road_l"]=int(_rl)
                        except: pass
                    elif _sn in ("pointsFor","runsScored","runs"):
                        try: _rec["runs_for"] = float(_sv or 0)
                        except: pass
                    elif _sn in ("pointsAgainst","runsAllowed"):
                        try: _rec["runs_against"] = float(_sv or 0)
                        except: pass
                    elif _sn == "streak":
                        try: _rec["streak"] = str(_dv).strip()
                        except: pass
                if _rec.get("runs_for",0)>0 and _rec.get("runs_against",0)>0:
                    _rf,_ra = _rec["runs_for"],_rec["runs_against"]
                    _rec["pyth_pct"] = round(_rf**1.83/(_rf**1.83+_ra**1.83),3)
                _team_records[_l10_tname] = _rec
    print(f"  L10 records loaded for {len(_l10_records)} teams")
    if _team_records:
        print(f"  Season records loaded for {len(_team_records)} teams (incl home/road splits)")
except Exception as _l10_ex:
    print(f"  L10 records unavailable ({_l10_ex}) -- streak gate disabled")

# __MLB_SPOT_STARTER__
# Spot starter detection — fetch probable pitcher GS from MLB Stats API
_spot_starters = {}  # {game_pk: {home_sp_gs, away_sp_gs, home_sp_name, away_sp_name}}
def _get_pitcher_gs(pitcher_id):
    """Return (name, games_started) for a pitcher this season."""
    try:
        _purl = f"https://statsapi.mlb.com/api/v1/people/{pitcher_id}/stats?stats=season&group=pitching&season=2026"
        _pr = requests.get(_purl, timeout=6)
        if _pr.status_code != 200: return None, 0
        _pd = _pr.json()
        for _pg in _pd.get("stats", []):
            for _ps in _pg.get("splits", []):
                _gs = int(_ps.get("stat", {}).get("gamesStarted", 0))
                _nm = _pd.get("people", [{}])[0].get("fullName", "")
                return _nm, _gs
    except: pass
    return None, 0

def _fetch_sp_gs_by_name(name):
    """Look up a pitcher's season GS by name using MLB People search."""
    if not name or name in ("None", "", "TBD"): return 99
    _nm_key = str(name).strip().lower()
    if _nm_key in _spot_starters: return _spot_starters[_nm_key]
    gs = 99
    try:
        import urllib.parse as _up
        _search = requests.get(
            f"https://statsapi.mlb.com/api/v1/people/search?names={_up.quote(name)}&sportId=1",
            timeout=6
        )
        _people = _search.json().get("people", [])
        # prefer pitchers
        _pid = None
        for _p in _people:
            if _p.get("primaryPosition", {}).get("abbreviation", "") in ("P", "SP", "RP"):
                _pid = _p["id"]; break
        if _pid is None and _people:
            _pid = _people[0]["id"]
        if _pid:
            _sstat = requests.get(
                f"https://statsapi.mlb.com/api/v1/people/{_pid}/stats?stats=season&group=pitching&season=2026",
                timeout=6
            ).json()
            for _sg in _sstat.get("stats", []):
                for _sp in _sg.get("splits", []):
                    gs = int(_sp.get("stat", {}).get("gamesStarted", 99) or 99)
                    break
    except: pass
    _spot_starters[_nm_key] = gs
    return gs
# ── END SPOT STARTER SETUP ───────────────────────────────────────────────

# __MLB_PITCHER_REST__
# Pitcher rest/fatigue — fetch last start date and pitch count
_sp_rest_cache = {}  # {name_lower: {days_rest, pitches, ip}}

def _fetch_sp_rest(name):
    """Return {days_rest, pitches, ip} for pitcher's most recent start."""
    if not name or str(name).strip() in ("", "None", "TBD"): return {}
    _nk = str(name).strip().lower()
    if _nk in _sp_rest_cache: return _sp_rest_cache[_nk]
    result = {}
    try:
        import urllib.parse as _up, datetime as _dt
        # Look up player ID (reuse people search)
        _sr = requests.get(
            f"https://statsapi.mlb.com/api/v1/people/search?names={_up.quote(name)}&sportId=1",
            timeout=6
        )
        _people = _sr.json().get("people", [])
        _pid = None
        for _p in _people:
            if _p.get("primaryPosition", {}).get("abbreviation", "") in ("P", "SP", "RP"):
                _pid = _p["id"]; break
        if _pid is None and _people: _pid = _people[0]["id"]
        if not _pid:
            _sp_rest_cache[_nk] = result; return result
        # Fetch game log
        _gl = requests.get(
            f"https://statsapi.mlb.com/api/v1/people/{_pid}/stats?stats=gameLog&group=pitching&season=2026",
            timeout=6
        ).json()
        _splits = []
        for _sg in _gl.get("stats", []):
            _splits.extend(_sg.get("splits", []))
        _starts = [s for s in _splits if int(s.get("stat", {}).get("gamesStarted", 0)) > 0]
        if not _starts:
            _sp_rest_cache[_nk] = result; return result
        _last = _starts[-1]
        _last_date = _dt.date.fromisoformat(_last["date"])
        _today = _dt.date.today()
        result["days_rest"] = (_today - _last_date).days
        result["pitches"]   = int(_last.get("stat", {}).get("numberOfPitches", 0) or 0)
        result["ip"]        = str(_last.get("stat", {}).get("inningsPitched", "0"))
        result["last_date"] = str(_last_date)
    except: pass
    _sp_rest_cache[_nk] = result
    return result
# ── END PITCHER REST SETUP ───────────────────────────────────────────────

# FIX (MLB-TUNEUP-1, 2026-09-24 user audit): a starter's season ELO/FIP
# rating assumes a normal outing -- it says nothing about a team giving
# its ace a deliberately short, low-stakes final turn once the team has
# already clinched (protecting him for the playoffs), or otherwise pushing
# his rest well past a normal 4-5 day turn for the same reason. Real case:
# Chris Sale got exactly this kind of managed, extended-rest start and the
# model had no way to discount for it -- _fetch_sp_rest() (days_rest) was
# already built above but never actually called anywhere until this fix,
# and the CLINCHED tier this reuses (mlb_motivation_gap_tiers) only exists
# as of the MLB-MOTIV-CLINCH-1 fix earlier in this same audit session.
# Deliberately scoped narrow: only fires when the pitcher's OWN team has
# nothing left to play for AND he's on rest clearly beyond a normal turn --
# a well-rested SP on a team still fighting for a spot is not this signal.
# Mirrors the SP SHORT-ROLE shave's symmetric pick+opp*0.5 pattern above --
# a shortened/uncertain start makes the SP-vs-SP comparison less
# trustworthy regardless of which side he's on. PROVISIONAL magnitude,
# unbacktested (no historical "tuneup start" data source exists to
# calibrate against yet) -- revisit after live grading, same honesty
# caveat as the short-role shave magnitudes above.
_TUNEUP_MIN_REST_DAYS = 7   # normal rotation turn is 4-5 days -- 7+ means at least one turn skipped
_TUNEUP_EDGE_SHAVE = round(0.05 * _MODEL_MARKET_BLEND_W, 4)   # was 0.05 on the raw-edge scale, rescaled with _BULK_ROLE_EDGE_SHAVE

def _tuneup_pick_team(row):
    _is_home = str(row.get('bet_team', '')).strip() == str(row.get(home_col, '')).strip()
    return row.get(home_col, '') if _is_home else row.get(away_col, '')

def _tuneup_opp_team(row):
    _is_home = str(row.get('bet_team', '')).strip() == str(row.get(home_col, '')).strip()
    return row.get(away_col, '') if _is_home else row.get(home_col, '')

def _sp_tuneup_shave(spnm, team_name):
    if not spnm or not team_name:
        return 0.0, None
    _tier_entry = (globals().get("mlb_motivation_gap_tiers") or {}).get(_pl_norm(str(team_name)))
    if not _tier_entry or _tier_entry[0] != "CLINCHED -- NOTHING AT STAKE":
        return 0.0, None
    _rest_info = _fetch_sp_rest(spnm)
    _days_rest = _rest_info.get("days_rest")
    if _days_rest is None or _days_rest < _TUNEUP_MIN_REST_DAYS:
        return 0.0, None
    return _TUNEUP_EDGE_SHAVE, f"{_days_rest}d rest, team clinched"

globals()["_sp_tuneup_shave"] = _sp_tuneup_shave
cand['_pick_tuneup_shave'], cand['_pick_tuneup_kind'] = zip(*cand.apply(
    lambda r: _sp_tuneup_shave(_pick_sp_name(r), _tuneup_pick_team(r)), axis=1))
cand['_opp_tuneup_shave'], cand['_opp_tuneup_kind'] = zip(*cand.apply(
    lambda r: _sp_tuneup_shave(_opp_sp_name(r), _tuneup_opp_team(r)), axis=1))
_tuneup_delta = cand['_pick_tuneup_shave'] + (cand['_opp_tuneup_shave'] * 0.5)
_tuneup_active_mask = (_tuneup_delta != 0.0) & (cand['passes_edge_rule'] == 1)
cand['edge'] = cand['edge'] - _tuneup_delta
_tuneup_now_fails = (cand['passes_edge_rule'] == 1) & (cand['edge'] < cand['_base_edge_min'])
cand.loc[_tuneup_now_fails, 'passes_edge_rule'] = 0
cand.loc[_tuneup_now_fails, '_gate_fail_reason'] = cand.loc[_tuneup_now_fails, '_gate_fail_reason'].fillna(
    "SP PRE-PLAYOFF TUNEUP/EXTENDED REST SHAVE: pick or opp SP's team has clinched and he's on 7+ days rest -- likely a managed, shortened start, season ELO overstates it")
print(f'SP pre-playoff tuneup/extended-rest shave: adjusted {int(_tuneup_active_mask.sum())} play(s), newly blocked {int(_tuneup_now_fails.sum())} (pick or opp SP on 7+ days rest for a team that has already clinched)')

# __MLB_OU_HOME_TREND__
# Home O/U tendency — fetch last 15 home games and calculate OVER/UNDER rate
_ou_home_cache = {}  # {team_id: {over, under, push, total}}

def _get_home_ou_trend(team_id, season=2026, last_n=15, line=None):
    """Return (over_count, under_count, total) for home team last N home games."""
    _cache_key = (team_id, line)
    if _cache_key in _ou_home_cache: return _ou_home_cache[_cache_key]
    result = {"over": 0, "under": 0, "push": 0, "total": 0}
    try:
        _sched_url = (
            f"https://statsapi.mlb.com/api/v1/schedule?teamId={team_id}"
            f"&season={season}&gameType=R&sportId=1"
        )
        _sr = requests.get(_sched_url, timeout=8)
        if _sr.status_code != 200:
            _ou_home_cache[_cache_key] = result; return result
        _sd = _sr.json()
        _home_games = []
        for _date in _sd.get("dates", []):
            for _game in _date.get("games", []):
                _sc = _game.get("status", {}).get("statusCode", "")
                if _sc not in ("F", "FO", "FR"): continue  # only final games
                _teams = _game.get("teams", {})
                _home_team = _teams.get("home", {})
                if _home_team.get("team", {}).get("id") != team_id: continue
                _h_score = _home_team.get("score")
                _a_score = _teams.get("away", {}).get("score")
                if _h_score is None or _a_score is None: continue
                _home_games.append({"total": int(_h_score) + int(_a_score), "date": _date.get("date","")})
        _recent = _home_games[-last_n:]
        for _g in _recent:
            _tot = _g["total"]
            _ref = line if line else 9.0
            if _tot > _ref + 0.4:
                result["over"] += 1
            elif _tot < _ref - 0.4:
                result["under"] += 1
            else:
                result["push"] += 1
        result["total"] = len(_recent)
    except Exception as _oue:
        pass
    _ou_home_cache[_cache_key] = result
    return result

def _get_road_ou_trend(team_id, season=2026, last_n=12, line=None):
    """Away team road O/U trend for last N road games."""
    _cache_key = (team_id, line, "road")
    if _cache_key in _ou_home_cache: return _ou_home_cache[_cache_key]
    result = {"over": 0, "under": 0, "push": 0, "total": 0}
    try:
        _sr2 = requests.get(
            f"https://statsapi.mlb.com/api/v1/schedule?teamId={team_id}"
            f"&season={season}&gameType=R&sportId=1", timeout=8)
        if _sr2.status_code != 200:
            _ou_home_cache[_cache_key] = result; return result
        _road_gs = []
        for _d2 in _sr2.json().get("dates", []):
            for _g2 in _d2.get("games", []):
                if _g2.get("status", {}).get("statusCode", "") not in ("F","FO","FR"): continue
                _t2 = _g2.get("teams", {})
                if _t2.get("away", {}).get("team", {}).get("id") != team_id: continue
                _hs2 = _t2.get("home", {}).get("score")
                _as2 = _t2.get("away", {}).get("score")
                if _hs2 is None or _as2 is None: continue
                _road_gs.append(int(_hs2) + int(_as2))
        _ref2 = line if line else 9.0
        for _tot2 in _road_gs[-last_n:]:
            if _tot2 > _ref2 + 0.4: result["over"] += 1
            elif _tot2 < _ref2 - 0.4: result["under"] += 1
            else: result["push"] += 1
        result["total"] = len(_road_gs[-last_n:])
    except Exception: pass
    _ou_home_cache[_cache_key] = result
    return result

_ou_season_cache = {}  # {team_id: {over, under, push, total}} — full season combined
def _get_team_ou_season(team_id, season=2026):
    """Full-season O/U record (home+away) by calling both trend funcs with last_n=100."""
    if team_id in _ou_season_cache: return _ou_season_cache[team_id]
    _h = _get_home_ou_trend(team_id, season=season, last_n=100)
    _r = _get_road_ou_trend(team_id, season=season, last_n=100)
    result = {
        "over":  _h["over"]  + _r["over"],
        "under": _h["under"] + _r["under"],
        "push":  _h["push"]  + _r["push"],
        "total": _h["total"] + _r["total"],
    }
    _ou_season_cache[team_id] = result
    return result

# Also build team name → team_id lookup from schedule API
_mlb_team_ids = {}  # {display_name_lower: team_id}
try:
    _tid_resp = requests.get(
        "https://statsapi.mlb.com/api/v1/teams?sportId=1&season=2026",
        timeout=6
    )
    if _tid_resp.status_code == 200:
        for _t in _tid_resp.json().get("teams", []):
            _mlb_team_ids[_t.get("name","").lower()] = _t.get("id")
            _mlb_team_ids[_t.get("teamName","").lower()] = _t.get("id")
        print(f"  MLB team IDs loaded for {len(_mlb_team_ids)} entries")
except Exception as _tide:
    print(f"  MLB team ID lookup unavailable ({_tide})")
# ── END O/U HOME TREND SETUP ─────────────────────────────────────────────
# ── END L10 FETCH ────────────────────────────────────────────────────────


top_market = qualified_market.head(MAX_MARKET_PLAYS)
top_model  = qualified_model.head(MAX_MODEL_PLAYS)
rank = 1
# ELO data status warning — show at top of card if the feed never loaded.
# _elo_unavailable_flag was being set but never read anywhere -- this is the first
# consumer of it, and it also drives the per-play grade cap below.
if globals().get("_elo_unavailable_flag"):
    print("=" * 58)
    print("  !! ELO DATA NOT LOADED")
    print("     mlb_team_off_elo / mlb_team_def_elo unavailable -- every")
    print("     prediction on this card is running on base stats only.")
    print("     ** All grades below are PROVISIONAL until ELO is restored **")
    print("=" * 58)


# Fix 7: Bullpen data status flag -- FanGraphs 403 = grade ceiling A-
# Fix (WC-48 session, MLB side): this checked _bpen_lookup (the historical RA
# fallback, which is a different signal that is almost always populated) --
# so the grade-ceiling warning below never fired even on a 100% FanGraphs
# block. Check the actual HLD/BS data (_bp_holds), now sourced from FanGraphs
# or the MLB Stats API fallback above.
_bullpen_data_live = bool(globals().get("_bp_holds"))
if not _bullpen_data_live:
    print("=" * 58)
    print("  [BULLPEN DATA DEAD] FanGraphs blocked or not loaded")
    print("     BP=Y grade criteria unverified -- grade ceiling: A-")
    print("=" * 58)

# SP form fetch is defined later in this cell and always available -- no halt needed
# _sp_cache_active = True always; grades and collapse detection work via lazy fetch
_sp_cache_active = True  # _sp_fetch_recent defined in this cell, fetches live per pitcher
# SP form banner removed -- fetch function always available in Cell 88



# ── Edge history tracker (persists between re-runs) ─────────────────────────
if "_mlb_edge_history" not in globals():
    globals()["_mlb_edge_history"] = {}

# ── Official-play ledger writer (real gap found 2026-07-23) ─────────────────
import os as _olp_os, csv as _olp_csv
_OLP_LEDGER_PATH = r"c:\Users\<user>\OneDrive\Documents\KAI Labs\sports\MLB\tracking\mlb_official_plays_ledger.csv"
_OLP_FIELDS = [
    "card_date", "game_id", "matchup", "market", "bet", "grade",
    "model_prob", "market_prob", "edge", "size", "net_injury_delta",
]

# BUG FIX (2026-08-07, user request -- tackle the deferred injury-vs-pick
# backtest item): the [INJURY VS PICK CONFLICT] warning (cell 6e66db20) has
# been informational-only since 2026-08-02 because one same-night
# counterexample wasn't enough evidence to justify a hard gate (Giants ML
# lost with a big against-delta; Pirates ML had an even bigger against-delta
# and won). A REAL backtest needs injury delta logged alongside outcomes --
# checked both existing tracking files (mlb_official_plays_ledger.csv,
# mlb_card_history.csv) and neither captures it, so no backtest is possible
# yet regardless of gate design. Added net_injury_delta here (signed: positive
# = injuries favor the picked team, negative = favor the opponent, same
# convention/magnitude as the [NET INJURY DELTA] bullet already computes from
# mlb_lineup_flags) so a future session can actually correlate this against
# real graded outcomes once enough rows accumulate, instead of guessing at a
# gate threshold off one data point. Existing 70-row ledger migrated to the
# new column (backfilled blank for old rows) in the same session this was
# added, so the file stays one consistent schema rather than forking.
def _log_official_play(card_date, game_id, matchup, market, bet, grade,
                        model_prob, market_prob, edge, size, net_injury_delta=None):
    _row = {
        "card_date": card_date, "game_id": game_id, "matchup": matchup,
        "market": market, "bet": bet, "grade": grade,
        "model_prob": model_prob, "market_prob": market_prob,
        "edge": edge, "size": size, "net_injury_delta": net_injury_delta,
    }
    _is_new = not _olp_os.path.exists(_OLP_LEDGER_PATH)
    try:
        # SAME-DAY REPLACE (2026-08-31, user audit -- "show me what won or
        # lost" led to comparing the tracked record against the real ledger):
        # the old dedup guard (2026-08-11) only caught an EXACT repeat of the
        # same pick (card_date + game_id + market + bet) -- it did nothing
        # when a same-day pipeline rerun's top edge shifted to a DIFFERENT
        # game (this notebook ran 5x on 2026-08-31 alone, per the ESPN 403
        # waves that day), so each rerun's "OFFICIAL" pick appended as a NEW
        # row instead of replacing the earlier one. Found live: 13 of the
        # last 32 tracked days carried 2+ different games logged OFFICIAL the
        # same date (up to 10 on one day, 4 on 2026-08-31 itself) -- every
        # extra row was never actually shown as the card's real official
        # play, yet still counted toward the win/loss record printed at the
        # top of this notebook's output. This function is called from
        # exactly one place (Cell 89, gated on _f_tag == "OFFICIAL"), and the
        # codebase's own 1-play/sport/day rule means at most one row should
        # ever exist per card_date -- so this now drops any existing row(s)
        # for THIS card_date (every other date's rows untouched) before
        # writing the new one, instead of just deduping an identical repeat.
        # A later rerun's pick is the more current one (fresher odds/injury
        # data going into it), so replace, not block.
        _kept_rows = []
        _rewrite_ok = True
        if not _is_new:
            try:
                with open(_OLP_LEDGER_PATH, "r", newline="", encoding="utf-8") as _olp_rf:
                    for _olp_existing in _olp_csv.DictReader(_olp_rf):
                        if str(_olp_existing.get("card_date")) != str(card_date):
                            _kept_rows.append(_olp_existing)
            except Exception:
                _rewrite_ok = False  # existing file unreadable -- fall through and just append below
        if not _is_new and _rewrite_ok:
            with open(_OLP_LEDGER_PATH, "w", newline="", encoding="utf-8") as _olp_f:
                _w = _olp_csv.DictWriter(_olp_f, fieldnames=_OLP_FIELDS)
                _w.writeheader()
                for _kr in _kept_rows:
                    _w.writerow({k: _kr.get(k, "") for k in _OLP_FIELDS})
                _w.writerow(_row)
            return
        with open(_OLP_LEDGER_PATH, "a", newline="", encoding="utf-8") as _olp_f:
            _w = _olp_csv.DictWriter(_olp_f, fieldnames=_OLP_FIELDS)
            if _is_new:
                _w.writeheader()
            _w.writerow(_row)
    except Exception as _olp_e:
        print(f"  [WARN] Official-play ledger write failed: {_olp_e}")

# GATE SAVE TRACKING (2026-09-20, user post-mortem FIX 7 -- Cubs ML correctly
# withheld by NO SP DATA, won 9-1 anyway): a gate that withholds a real,
# would-otherwise-qualify play has no record anywhere once it's withheld -- no
# way to ever check whether the gate is actually saving units or costing them.
# Separate file, separate row schema (gate_reason instead of grade/size) from
# the real official-play ledger above -- these are NOT real logged bets and must
# never be countable toward the OFFICIAL ML RECORD. Simple append + exact-repeat
# dedupe (no same-day-replace -- unlike the 1-play/sport/day official ledger,
# multiple different games can legitimately hit the same gate on the same day).
_GSV_LEDGER_PATH = r"c:\Users\<user>\OneDrive\Documents\KAI Labs\sports\MLB\tracking\mlb_gate_saves_ledger.csv"
_GSV_FIELDS = ["card_date", "game_id", "matchup", "market", "bet", "gate_reason",
               "model_prob", "market_prob", "edge"]
def _log_gate_save_candidate(card_date, game_id, matchup, market, bet, gate_reason,
                              model_prob, market_prob, edge):
    _row = {"card_date": card_date, "game_id": game_id, "matchup": matchup,
            "market": market, "bet": bet, "gate_reason": gate_reason,
            "model_prob": model_prob, "market_prob": market_prob, "edge": edge}
    try:
        _existing = []
        if _olp_os.path.exists(_GSV_LEDGER_PATH):
            with open(_GSV_LEDGER_PATH, "r", newline="", encoding="utf-8") as _gsv_rf:
                _existing = list(_olp_csv.DictReader(_gsv_rf))
        _dupe = any(str(e.get("card_date")) == str(card_date) and str(e.get("game_id")) == str(game_id)
                    and str(e.get("market")) == str(market) and str(e.get("bet")) == str(bet) for e in _existing)
        if _dupe:
            return
        _is_new = not _existing
        with open(_GSV_LEDGER_PATH, "a", newline="", encoding="utf-8") as _gsv_f:
            _w = _olp_csv.DictWriter(_gsv_f, fieldnames=_GSV_FIELDS)
            if _is_new:
                _w.writeheader()
            _w.writerow(_row)
    except Exception as _gsv_e:
        print(f"  [WARN] Gate-save ledger write failed: {_gsv_e}")

_olp_today = pd.Timestamp.now().strftime("%Y-%m-%d")

# ── Independent SP source cross-check (ESPN scoreboard) ──────────────────────
# The existing SP validation gate below compares two MLB-sourced fields against
# each other (_series_context's probable-pitcher feed vs the row's merged
# starter name from Cell 41/42's live game-feed) -- but both ultimately trace
# back to MLB's own StatsAPI, so they agree with each other even when MLB's
# API itself is stale/wrong. Confirmed live 2026-07-27: MLB StatsAPI's schedule
# endpoint AND its live game-feed BOTH said "Eddy Yean" for a Nationals start
# that dedicated beat reporting (Bleacher Nation, SI) said was Andrew Alvarez --
# an internal MLB-vs-MLB check can't catch MLB being wrong about itself. This
# fetches ESPN's scoreboard as a genuinely independent second source (already
# used elsewhere in this notebook for odds) once per session, not per game.
_espn_probable_cache = {}
try:
    # BUG FIX (2026-08-22, found live: 2nd occurrence -- Guardians/Rockies and
    # Cubs/Mariners both flagged SP CONFLICT UNRESOLVED and got withheld from
    # OFFICIAL; real-world reporting (Bleacher Nation, Heavy.com) confirmed our
    # own model's starters -- Tanner Bibee/Gabriel Hughes, David Peterson --
    # were correct both times, meaning "ESPN" was wrong both times. Same root
    # cause already suspected on 2026-08-18/2026-08-20: this request never
    # passed a `dates` param, so it relied entirely on ESPN's own default
    # notion of "today" -- every other ESPN scoreboard call in this codebase
    # (WNBA, NFL, etc.) pins an explicit date for exactly this reason. Without
    # it, a CDN-cached or off-by-one-day response reads as a genuine
    # disagreement instead of what it likely is -- stale/wrong-day data.
    _epc_date_param = str(CARD_DATE).replace("-", "") if "CARD_DATE" in globals() else None
    _epc_params = {"dates": _epc_date_param} if _epc_date_param else {}
    _epc_resp = requests.get(
        "https://site.api.espn.com/apis/site/v2/sports/baseball/mlb/scoreboard",
        params=_epc_params,
        timeout=8
    )
    if _epc_resp.status_code == 200:
        for _epc_ev in _epc_resp.json().get("events", []):
            _epc_comp = (_epc_ev.get("competitions") or [{}])[0]
            for _epc_c in _epc_comp.get("competitors", []):
                _epc_team = str(_epc_c.get("team", {}).get("displayName", "")).lower()
                for _epc_p in _epc_c.get("probables", []):
                    _epc_name = _epc_p.get("athlete", {}).get("fullName")
                    if _epc_team and _epc_name:
                        _espn_probable_cache[_epc_team] = _epc_name
    else:
        print(f"  [SP CROSS-CHECK] ESPN scoreboard returned {_epc_resp.status_code} -- cross-check disabled this run")
    print(f"  [SP CROSS-CHECK] ESPN probable pitchers loaded for {len(_espn_probable_cache)} teams "
          f"(dates={_epc_date_param})")
except Exception as _epc_ex:
    print(f"  [SP CROSS-CHECK] ESPN probable-pitcher fetch failed ({_epc_ex}) -- cross-check disabled this run")

# ── Main ML Play Output Loop ─────────────────────────────────────────────────
# BUG FIX (2026-09-03, found live -- real crash: "NameError: name
# '_display_units' is not defined" during the 9am scheduled run, mid-print
# of the RL-sanity-fail-cascade block below for Tampa Bay Rays -- that block
# reads _display_units unconditionally, but this only ever created it INSIDE
# `if len(top_market) > 0:`. On a run where top_market is empty at this point,
# the name never exists, and the cascade block crashes the whole cell instead
# of printing its re-review warning -- which is exactly what hid a real
# unresolved RL-vs-ML price conflict (60.9% vs 54.0%) on today's actual
# OFFICIAL pick. Hoisted out of the conditional so it always exists.
_display_units = {}
if len(top_market) > 0:
    print(f"\n{'-'*50}")
    print("  MARKET PLAYS  (ESPN odds confirmed)")
    print(f"{'-'*50}")
    _market_played_ct = 0  # real plays only -- excludes genuine "No bet" rows
    rank = 1
    # BUG FIX (2026-08-27, user audit -- Nationals ML + Giants ML both lost
    # 2026-08-26 after shipping OFFICIAL with no trace of the grade cap/watch-
    # only sizing this loop itself had already computed for them): this dict
    # records the final grade + unit this loop lands on for every pick, so
    # Cell bd676982's OFFICIAL determination can withhold OFFICIAL status the
    # same way it already does for an SP conflict, instead of re-deriving
    # "official" from passes_edge_rule alone with no memory of what this loop
    # found.
    globals()["mlb_grade_downgrades"] = {}

    for _, r in top_market.iterrows():
        pn   = _pitcher_note(r)
        sl   = _score_line(r)
        unit = _unit_label(r["edge"], r.get("p_bet_win_model"))
        # Fix: this header used to only print inside the full-play branch
        # below (after the 0.5u hard-cap early-return) -- any hard-capped row
        # printed Grade/Size with no matchup name at all. Always show it.
        print(f"\n#{rank}  {r.get(away_col,'')} @ {r.get(home_col,'')}")
        # 2026-07-12: pn/sl (SP Matchup, Proj Score) had the identical bug --
        # only printed in the full-play branch below, so any hard-capped row
        # (e.g. the 0.5u sizing tiers) never showed them at all. Moved here
        # so they always print, same fix pattern as the header line above.
        if pn: print(pn)
        if sl: print(sl)

        # ── Fix 1: Juice cap -150 ────────────────────────────────────────────
        _f1_bet_p = float(r.get("market_novig_prob") or r.get("market_implied_prob") or 0.5)
        _f1_edge = float(r.get("edge", 0) or 0)
        if _f1_bet_p > 0.5 and "No bet" not in unit:
            _f1_am = round(-_f1_bet_p / (1 - _f1_bet_p) * 100)
            if _f1_am < -150:
                # 2026-07-27: dropped the sharp-confirmation clause here (same
                # dead mlb_sharp_pct variable as the unit_label gate) -- this
                # escape hatch now runs on juice + edge alone.
                _f1_ok = (_f1_am >= -165 and _f1_edge >= 0.15)
                if not _f1_ok:
                    print(f"  !! JUICE CAP: market {_f1_am} exceeds -150 -- WATCH ONLY")
                    if not unit.startswith("0.5u"):
                        unit = f"0.5u [JUICE {_f1_am}]"

        # ── SP validation gate (item 1, 2026-07-09): moved BEFORE grade/edge so a
        # stale starter can no longer produce a wrong edge, wrong grade, or wrong
        # panel print. Confirmed live: ARI/SD card loaded Cease/Pfaadt while the
        # actual confirmed starters were King/Cabrera -- the old check (below,
        # formerly "Fix 2") ran AFTER grade/edge were already computed and only
        # printed a warning, so the wrong numbers had already been shown. This now
        # hard-blocks the whole row (no grade, no edge, no panel) on any mismatch
        # against the live MLB schedule API probable-pitcher feed (_series_context,
        # fetched fresh at the top of this cell -- independent of Cell 61).
        _spc_away = str(r.get(away_col,"")).lower(); _spc_home = str(r.get(home_col,"")).lower()
        _spc_ctx = next((v for (ka,kh),v in _series_context.items()
                         if any(w in ka for w in _spc_away.split() if len(w)>3)
                         and any(w in kh for w in _spc_home.split() if len(w)>3)), None)
        _sp_hard_blocked = False
        _spc_board_away = next((str(r.get(c,"") or "").strip() for c in ["away_pitcher_name","away_pitcher_final_name","away_starter","away_pitcher"] if r.get(c)), "")
        _spc_board_home = next((str(r.get(c,"") or "").strip() for c in ["home_pitcher_name","home_pitcher_final_name","home_starter","home_pitcher"] if r.get(c)), "")
        def _spc_match(a, b):
            return (not a or not b or a.lower() in b.lower() or b.lower() in a.lower()
                    or a.lower().split()[-1] in b.lower() or b.lower().split()[-1] in a.lower())
        if _spc_ctx:
            _spc_prob_away = str(_spc_ctx.get("prob_away_sp","")).strip()
            _spc_prob_home = str(_spc_ctx.get("prob_home_sp","")).strip()
            _spc_away_changed = _spc_prob_away and _spc_board_away and not _spc_match(_spc_prob_away, _spc_board_away)
            _spc_home_changed = _spc_prob_home and _spc_board_home and not _spc_match(_spc_prob_home, _spc_board_home)
            if _spc_away_changed or _spc_home_changed:
                _sp_hard_blocked = True
                if _spc_away_changed:
                    print(f"  [SP MISMATCH -- BLOCKED] {r.get(away_col,'')}: projection uses {_spc_board_away} | confirmed today: {_spc_prob_away}")
                if _spc_home_changed:
                    print(f"  [SP MISMATCH -- BLOCKED] {r.get(home_col,'')}: projection uses {_spc_board_home} | confirmed today: {_spc_prob_home}")
                print("  -- starter data does not match today's confirmed probable -- grading blocked for this game, re-run the card")
                # BUG FIX (2026-08-18, user audit found live): this block/print
                # only ever affected THIS loop's own grading pass -- the real
                # FINAL CARD SELECTOR (cell 6e66db20, builds the actual
                # OFFICIAL PLAY tags the card ships with) never read this flag
                # at all, so a starter conflict serious enough to say "grading
                # blocked until confirmed" right here could still reach the
                # real card as a fully-graded OFFICIAL play with zero
                # indication anything was ever in question. Real case: today's
                # Cardinals @ Reds ML shipped as OFFICIAL (+18.3% edge) while
                # this exact conflict was firing on the same game one cell
                # earlier -- turned out to be a false alarm (ESPN's probable-
                # pitcher cache was stale, likely from the ESPN 403 outage
                # affecting the whole pipeline that day; Kyle Leahy was
                # confirmed correct), but nothing caught it structurally, only
                # luck. Writes to a shared global keyed by (away, home) team
                # names so cell 6e66db20 can surface it inline instead of it
                # only ever living in an earlier cell's own output (which the
                # digest extractor's last-2-cells rule wouldn't even capture).
                globals().setdefault("_mlb_sp_source_conflicts", {})[(str(r.get(away_col,"")), str(r.get(home_col,"")))] = (
                    f"projection vs today's confirmed probable mismatch "
                    f"({_spc_board_away or '?'} / {_spc_board_home or '?'} vs confirmed "
                    f"{_spc_prob_away or '?'} / {_spc_prob_home or '?'})"
                )

        # ESPN cross-check (independent of the MLB-vs-MLB comparison above) --
        # see the cache-build comment before the main loop for why this exists.
        _espn_away_prob = _espn_probable_cache.get(_spc_away, "")
        _espn_home_prob = _espn_probable_cache.get(_spc_home, "")
        _espn_away_conflict = bool(_spc_board_away and _espn_away_prob and not _spc_match(_spc_board_away, _espn_away_prob))
        _espn_home_conflict = bool(_spc_board_home and _espn_home_prob and not _spc_match(_spc_board_home, _espn_home_prob))
        if _espn_away_conflict or _espn_home_conflict:
            _sp_hard_blocked = True
            if _espn_away_conflict:
                print(f"  [SP SOURCE CONFLICT -- PENDING] {r.get(away_col,'')}: model uses {_spc_board_away} | ESPN says {_espn_away_prob} -- starter unresolved")
            if _espn_home_conflict:
                print(f"  [SP SOURCE CONFLICT -- PENDING] {r.get(home_col,'')}: model uses {_spc_board_home} | ESPN says {_espn_home_prob} -- starter unresolved")
            print("  -- two independent sources disagree on the starter -- grading blocked until confirmed, re-run closer to first pitch")
            # BUG FIX (2026-08-18, user audit found live): same disconnect as
            # the SP MISMATCH block above -- see that comment for the real
            # case this was found from. Written to the same shared global so
            # cell 6e66db20 only needs to check one place for either conflict
            # type.
            globals().setdefault("_mlb_sp_source_conflicts", {})[(str(r.get(away_col,"")), str(r.get(home_col,"")))] = (
                f"model vs ESPN disagree on starter "
                f"(model: {_spc_board_away or '?'} / {_spc_board_home or '?'}, "
                f"ESPN: {_espn_away_prob or '?'} / {_espn_home_prob or '?'})"
            )
        if _sp_hard_blocked:
            continue

        # ── Fix 4+5: Grade computation ────────────────────────────────────────
        _g_edge = float(r.get("edge", 0) or 0)
        _g_bp   = "_rl_yday" in globals() and bool(globals().get("_rl_yday"))
        _g_jp   = float(r.get("market_novig_prob") or r.get("market_implied_prob") or 0.5) <= 0.583
        _g_bet_side = str(r.get("bet_side", r.get("bet_type", ""))).lower()
        _g_pick_sp  = next((str(r.get(c,"") or "").strip() for c in ((["away_pitcher_final_name","away_pitcher_name","away_starter"] if "away" in _g_bet_side else ["home_pitcher_final_name","home_pitcher_name","home_starter"])) if r.get(c)), "")
        _g_opp_sp   = next((str(r.get(c,"") or "").strip() for c in ((["home_pitcher_final_name","home_pitcher_name","home_starter"] if "away" in _g_bet_side else ["away_pitcher_final_name","away_pitcher_name","away_starter"])) if r.get(c)), "")

        # SP rating staleness hard block -- runs before grade so a stale
        # rating can't produce a wrong edge/grade either (same placement as
        # the SP-mismatch hard block above).
        def _g_sp_stale(spnm):
            if not spnm: return None
            _k = _sp_norm_era(spnm)
            if _k not in _sp_60day_il_players:
                return None
            _return_date = _sp_60day_il_players[_k]
            _ip = _sp_ip_since_return(spnm, _return_date)
            if _ip >= _SP_STALE_MIN_IP:
                return None  # real post-return sample -- rating has caught up
            return _return_date
        _g_pick_stale_date = _g_sp_stale(_g_pick_sp)
        _g_opp_stale_date = _g_sp_stale(_g_opp_sp)
        if _g_pick_stale_date or _g_opp_stale_date:
            _g_stale_who = _g_pick_sp if _g_pick_stale_date else _g_opp_sp
            _g_stale_date = _g_pick_stale_date or _g_opp_stale_date
            print(f"  [SP RATING UNVERIFIED] {_g_stale_who} returned from 60-day IL ({_g_stale_date}) -- "
                  f"current-season sample too thin to trust the blended rating, may still reflect pre-injury form")
            print("  -- grading blocked until this pitcher has a real post-return sample")
            continue
        for _tjs_nm in (_g_pick_sp, _g_opp_sp):
            _tjs_note = _TJS_RETURN_PITCHERS.get(str(_tjs_nm).lower())
            if _tjs_note:
                print(f"  [TJS WATCH] {_tjs_nm}: {_tjs_note}")

        def _g_is_collapse(spnm):
            if not spnm or "_sp_form_cache" not in globals(): return False
            import unicodedata as _guc2, re as _gre2
            def _gn2(s): return _gre2.sub(r"[^a-z ]","",_guc2.normalize("NFD",s).encode("ascii","ignore").decode().lower()).strip()
            cf = _sp_form_cache.get(_gn2(spnm)) or {}
            l3 = cf.get("era_l3"); sea = cf.get("era_season")
            # 2026-08-29 (user audit M2): L3-vs-season only catches a collapse
            # sharp enough to show up inside 3 starts. A slower decline that has
            # already run 10 starts deep reads as "normal" here because it has
            # by then dragged the season ERA up with it. era_l10 (added
            # 2026-08-25) gives a window long enough to see that and short
            # enough to still be current -- added as an OR branch so this gate
            # only ever fires MORE often than before, never less.
            l10 = cf.get("era_l10"); gs10 = cf.get("gs_l10") or 0
            try:
                if l3 is not None and sea is not None:
                    if float(l3) > float(sea) * 1.25 and float(l3) - float(sea) >= 1.5:
                        return True
                if l10 is not None and float(gs10) >= 5:
                    if l3 is not None and float(l3) > float(l10) * 1.25 and float(l3) - float(l10) >= 1.5:
                        return True
                    if sea is not None and float(l10) > float(sea) * 1.20 and float(l10) - float(sea) >= 1.0:
                        return True
            except (TypeError, ValueError):
                return False
            return False
        def _g_is_h2h_collapse(spnm, opp_team_id):
            if not spnm or not opp_team_id: return False
            _rec = _sp_fetch_vs_opponent(spnm, opp_team_id)
            if not _rec: return False
            return _rec["starts"] >= 2 and _rec["era"] is not None and _rec["era"] >= 5.00
        _g_bet_is_away = "away" in _g_bet_side
        _g_bet_team_nm = str(r.get(away_col,"") if _g_bet_is_away else r.get(home_col,""))
        _g_opp_team_nm = str(r.get(home_col,"") if _g_bet_is_away else r.get(away_col,""))
        _g_bet_team_id = _mlb_team_ids.get(_g_bet_team_nm.lower())
        _g_opp_team_id = _mlb_team_ids.get(_g_opp_team_nm.lower())
        _g_pick_recent_clps = _g_is_collapse(_g_pick_sp)
        _g_opp_recent_clps  = _g_is_collapse(_g_opp_sp)
        _g_pick_h2h_clps = _g_is_h2h_collapse(_g_pick_sp, _g_opp_team_id)
        _g_opp_h2h_clps  = _g_is_h2h_collapse(_g_opp_sp, _g_bet_team_id)
        _g_pick_clps = _g_pick_recent_clps or _g_pick_h2h_clps
        _g_opp_clps  = _g_opp_recent_clps or _g_opp_h2h_clps
        _g_no_sp_clps = not _g_pick_clps and not _g_opp_clps
        # Fix: shared _mlb_elo_avail() helper (see its docstring) -- starts from the
        # whole-feed flag so every play is capped/flagged when ELO never loaded at
        # all, and fixes the case-sensitivity bug that broke the per-team fuzzy match.
        # 2026-07-27: ELO-unavailable upgraded from a grade cap to a hard block.
        # A missing/default team ELO isn't a minor quality ding -- it's a core
        # model input gone, on par with the SP-mismatch hard block right above
        # this. Letting it through at a discounted letter grade meant a play
        # with no real ELO signal behind it could still size and post like a
        # normal pick.
        _g_elo_avail = _mlb_elo_avail(r.get("bet_team",""))
        if not _g_elo_avail:
            _elo_unavail_team = r.get("bet_team","?")
            print(f"  [ELO UNAVAILABLE -- BLOCKED] {_elo_unavail_team} team ELO at default -- core model input missing")
            print(f"    -> {_mlb_elo_diag(_elo_unavail_team)}")
            print("  -- no real ELO signal for this team -- grading blocked for this play")
            continue
        # 2026-07-27: sharp criterion removed (mlb_sharp_pct was never set by
        # any producer cell, so this always failed) -- 5-criteria grading is
        # now 4-criteria, re-tiered one step tighter per missing criterion.
        _g_crit = [_g_edge >= 0.15, _g_no_sp_clps, _g_bp, _g_jp]
        _g_met  = sum(_g_crit)
        _grade  = {4: "A+", 3: "A", 2: "A-", 1: "B+"}.get(_g_met, "B")
        if _g_opp_clps and _grade in ("A+", "A"):
            _grade = "A-"
            _g_opp_clps_why = " & ".join(t for t,c in (("recent form",_g_opp_recent_clps),("H2H vs opp",_g_opp_h2h_clps)) if c)
            print(f"  [SP COLLAPSE OPP] {_g_opp_sp} ({_g_opp_clps_why}) -- grade capped at A-")
        if _g_pick_clps and _grade not in ("B", "C"):
            _grade = "B+"
            if "No bet" not in unit and not unit.startswith("0.5u"):
                unit = f"0.5u [OWN SP COLLAPSE]"
            _g_pick_clps_why = " & ".join(t for t,c in (("recent form",_g_pick_recent_clps),("H2H vs opp",_g_pick_h2h_clps)) if c)
            print(f"  [SP COLLAPSE PICK] {_g_pick_sp} ({_g_pick_clps_why}) -- grade capped at B+, size 0.5u")

        # UNRATED starter enforcement (2026-08-29, user audit M4). Until now a
        # sub-5-start starter only cost the play a flat edge shave (see
        # _thin_sample_shave above) -- it still shipped as a fully-graded,
        # OFFICIAL-eligible pick off a pitcher with no real MLB rate behind him.
        # Same cap/route as SP COLLAPSE above, which means it lands in
        # mlb_grade_downgrades and Cell 6e66db20 withholds OFFICIAL, rather than
        # adding a new parallel gate. Own-SP unrated caps grade+size; an unrated
        # OPPOSING starter is an uncertainty on the game, not a reason to trust
        # the pick more, so it caps the grade only.
        _g_thin_fn = globals().get("_sp_thin_starts")
        if _g_thin_fn:
            _g_pick_thin = _g_thin_fn(_g_pick_sp)
            _g_opp_thin = _g_thin_fn(_g_opp_sp)
            _g_thin_floor = globals().get("_THIN_SAMPLE_MIN_STARTS", 5)
            if _g_pick_thin is not None:
                if _grade not in ("B", "C"):
                    _grade = "B+"
                if "No bet" not in unit and not unit.startswith("0.5u"):
                    unit = "0.5u [OWN SP UNRATED]"
                print(f"  [SP UNRATED PICK] {_g_pick_sp}: {_g_pick_thin} career MLB start(s) "
                      f"(< {_g_thin_floor}) -- no ratable MLB sample, tier shown as UNRATED; "
                      f"grade capped at B+, size 0.5u, withheld from OFFICIAL")
            elif _g_opp_thin is not None and _grade in ("A+", "A"):
                _grade = "A-"
                print(f"  [SP UNRATED OPP] {_g_opp_sp}: {_g_opp_thin} career MLB start(s) "
                      f"(< {_g_thin_floor}) -- the edge is priced against an unrated arm, grade capped at A-")

        # M15/M16 (2026-09-04, user audit -- McClanahan Sept 3 loss, both
        # fixes share the same pick-side-SP/opponent context so they're
        # applied together here). Same cap/route as SP COLLAPSE/UNRATED
        # above -- caps grade+size rather than a new parallel gate.
        _m1516_pick_is_home = str(r.get(home_col, "")) == str(r.get("bet_team", ""))
        _m1516_opp_team = str(r.get(away_col, "")) if _m1516_pick_is_home else str(r.get(home_col, ""))

        # M15: same-opponent/park early-exit-then-IL history
        _m15_hist_fn = globals().get("_sp_early_exit_history")
        if _g_pick_sp and _m15_hist_fn:
            _m15_hist = _m15_hist_fn(_g_pick_sp, _m1516_opp_team, _m1516_pick_is_home)
            if _m15_hist:
                # BUG FIX (MLB-INJ-GRADECAP-1, 2026-09-24 user audit): this used
                # to hard-pin the letter grade to B+ regardless of what the play's
                # own criteria actually earned -- a qualitative floor, not a real
                # number. Conflicts with the standing edge-reduction-over-hard-caps
                # preference (see the 2026-08-30 conversion of Gates 1-9 from hard
                # stops to flat edge knockdowns, same cell). Converted to match:
                # shave the actual edge used for the 4-criteria grade, then let the
                # SAME grading formula used above re-derive the letter from that
                # reduced number, instead of jumping straight to a fixed B+.
                _M15_INJURY_HISTORY_EDGE_SHAVE = round(0.07 * globals().get("_MODEL_MARKET_BLEND_W", 1.0), 4)  # was 0.07 on the raw-edge scale  # BUG FIX (MLB-EDGE-BLEND-1, 2026-09-25 user audit): rescaled to the blended edge scale (edge_blended = w*edge_raw when a real market price exists) so this mechanism's original relative selectivity survives the new, much-smaller edge scale instead of unintentionally becoming a near-automatic disqualifier.
                _g_edge_pre_m15 = _g_edge
                _g_edge = max(0.0, _g_edge - _M15_INJURY_HISTORY_EDGE_SHAVE)
                _g_crit = [_g_edge >= 0.15, _g_no_sp_clps, _g_bp, _g_jp]
                _g_met = sum(_g_crit)
                _grade = {4: "A+", 3: "A", 2: "A-", 1: "B+"}.get(_g_met, "B")
                _m15_park_str = "at this same park" if _m15_hist["same_park"] and not _m15_hist["same_opponent"] else f"vs {_m1516_opp_team}"
                print(f"  !! [SP INJURY HISTORY] {_g_pick_sp}: exited early ({_m15_hist['ip']:.1f} IP) "
                      f"{_m15_park_str} on {_m15_hist['date']}, placed on the IL {_m15_hist['il_date']} -- "
                      f"real injury-recurrence risk, not just general uncertainty; edge discounted "
                      f"-{_M15_INJURY_HISTORY_EDGE_SHAVE:.0%} ({_g_edge_pre_m15:.1%} -> {_g_edge:.1%}), grade now {_grade}")

        # M16: post-IL pitcher still in a bulk/short-role leash -- game
        # outcome depends heavily on the bullpen in innings 4-6 regardless
        # of how the starter throws, real variance beyond the edge discount
        # already applied for thin post-return IP.
        if _g_pick_sp and "_sp_recent_il_return_players" in globals() and _sp_recent_il_return_players:
            _m16_k = _sp_norm_era(_g_pick_sp)
            if _m16_k in _sp_recent_il_return_players:
                _m16_is_bulk = False
                try:
                    _m16_is_bulk = bool(_sp_role_flags(_g_pick_sp)[1])
                except Exception:
                    pass
                if _m16_is_bulk:
                    _m16_starts = _sp_starts_since_return(_g_pick_sp, _sp_recent_il_return_players[_m16_k])
                    if _m16_starts < 5:
                        if "No bet" not in unit and not unit.startswith("0.5u"):
                            unit = "0.5u [POST-IL BULK ROLE]"
                        print(f"  !! [SP POST-IL BULK ROLE] {_g_pick_sp}: bulk/short-role (3-5 IP leash) with "
                              f"only {_m16_starts} start(s) logged since returning from IL -- the bullpen "
                              f"decides a big chunk of this game regardless of how he pitches; size capped at 0.5u")

        # FIX 5 (2026-09-20, user post-mortem -- Sugano L10 ERA 6.75 vs Seattle, held
        # them to 1 run): a cold streak is a real, sustained signal on average, but a
        # pitcher on a bad L10 still throws individual good starts some real fraction of
        # the time -- the edge from a cold-streak read is probabilistic, not deterministic,
        # and neither the existing edge-shave (_sustained_cold_shave/_xera_cold_shave
        # above) nor the grade itself ever surfaced that variance to the reader, or
        # reflected it in sizing. When cold streak contributed a material shave to this
        # specific play (>=0.02, the same magnitude this notebook already treats as a
        # real, non-trivial shave elsewhere -- see _OFFENSE_COLD_SHAVE_BASE), cap size at
        # 0.75u and say so explicitly.
        _g_cold_total = (float(r.get("_pick_l10_shave", 0.0) or 0.0) + float(r.get("_opp_l10_shave", 0.0) or 0.0) * 0.5 +
                         float(r.get("_pick_xeracold_shave", 0.0) or 0.0) + float(r.get("_opp_xeracold_shave", 0.0) or 0.0) * 0.5)
        if _g_cold_total >= 0.02:
            if "No bet" not in unit and not unit.startswith(("0.5u", "0.75u")):
                unit = "0.75u [COLD STREAK DRIVER]"
            print(f"  [COLD STREAK VARIANCE] cold-streak signal contributed {_g_cold_total:.1%} of this "
                  f"play's edge shave -- cold-streak plays have roughly a 25-30% rate of individually "
                  f"good starts regardless of the trend; size capped at 0.75u for this reason alone")

        # M31: post-IL start-over-start trend, checked for BOTH the pick's
        # own SP and the opponent's -- informational only, doesn't touch
        # grade/unit either direction.
        if "_sp_recent_il_return_players" in globals() and _sp_recent_il_return_players:
            for _m31_sp, _m31_role in [(_g_pick_sp, "own"), (_g_opp_sp, "opponent")]:
                if not _m31_sp:
                    continue
                _m31_k = _sp_norm_era(_m31_sp)
                if _m31_k not in _sp_recent_il_return_players:
                    continue
                _m31_trend = _sp_post_il_trend(_m31_sp, _sp_recent_il_return_players[_m31_k])
                if _m31_trend:
                    _m31_line = " -> ".join(f"{s['er']:.0f} ER/{s['ip']:.1f} IP" for s in _m31_trend["starts"])
                    print(f"  !! [SP POST-IL TREND] {_m31_sp} ({_m31_role} SP): {_m31_trend['trend']} every "
                          f"start since returning from IL ({_m31_line}) -- small sample, not backtested, "
                          f"but the flat L3 average above doesn't show this trend")


        # M32: mandatory Statcast velocity check for the PICK's own SP,
        # for the same recent-IL-return population M16/M31 already scope
        # to. A verified drop caps size the same way M16's bulk-role leash
        # does; an UNVERIFIED check (Statcast unavailable) caps size too --
        # deliberately not a silent pass, since "we couldn't check" is not
        # the same claim as "he's fine."
        if _g_pick_sp and "_sp_recent_il_return_players" in globals() and _sp_recent_il_return_players:
            _m32_k = _sp_norm_era(_g_pick_sp)
            if _m32_k in _sp_recent_il_return_players:
                _m32_vfn = globals().get("_sp_post_il_velocity_check")
                _m32_chk = _m32_vfn(_g_pick_sp, _sp_recent_il_return_players[_m32_k]) if _m32_vfn else {"status": "unverified", "reason": "check unavailable"}
                if _m32_chk.get("status") == "verified":
                    if _m32_chk.get("flagged"):
                        if "No bet" not in unit and not unit.startswith("0.5u"):
                            unit = "0.5u [POST-IL VELOCITY DROP]"
                        print(f"  !! [SP POST-IL VELOCITY] {_g_pick_sp}: fastball velocity down {_m32_chk['drop']:.1f} mph "
                              f"since returning from IL ({_m32_chk['pre_velo']:.1f} -> {_m32_chk['post_velo']:.1f} mph) -- "
                              f"real sign of incomplete recovery, size capped at 0.5u")
                    else:
                        print(f"  [SP POST-IL VELOCITY] {_g_pick_sp}: fastball velocity confirmed stable since "
                              f"returning from IL ({_m32_chk['pre_velo']:.1f} -> {_m32_chk['post_velo']:.1f} mph)")
                else:
                    if "No bet" not in unit and not unit.startswith("0.5u"):
                        unit = "0.5u [POST-IL VELOCITY UNVERIFIED]"
                    print(f"  !! [SP POST-IL VELOCITY] {_g_pick_sp}: could not verify recovery via Statcast "
                          f"({_m32_chk.get('reason')}) -- size capped at 0.5u pending confirmation")


        # SP last-start outlier -- informational regression flag (2026-08-01).
        # Distinct from the SP-collapse gate above (L3-average vs season ERA,
        # which caps grade/size): this looks at the single most recent start
        # in isolation, since a one-off disaster or dominant start can still
        # sit under the L3-collapse threshold once blended with two normal
        # starts. Same magnitude convention as that gate (1.25x AND >=1.5 run
        # gap vs season ERA) -- informational only, no edge/grade/size impact,
        # since one start is too small a sample to act on directly.
        def _g_last_start_outlier(spnm):
            if not spnm or "_sp_form_cache" not in globals(): return None
            import unicodedata as _guc3, re as _gre3
            def _gn3(s): return _gre3.sub(r"[^a-z ]","",_guc3.normalize("NFD",s).encode("ascii","ignore").decode().lower()).strip()
            cf = _sp_form_cache.get(_gn3(spnm)) or {}
            _le, _se = cf.get("era_last"), cf.get("era_season")
            if _le is None or _se is None: return None
            try:
                _le, _se = float(_le), float(_se)
            except Exception:
                return None
            if _se <= 0: return None
            if _le > _se * 1.25 and _le - _se >= 1.5:
                return ("spike", _le, _se)
            if _le < _se / 1.25 and _se - _le >= 1.5:
                return ("suppressed", _le, _se)
            return None
        for _lso_nm, _lso_role in ((_g_pick_sp, "PICK"), (_g_opp_sp, "OPP")):
            _lso = _g_last_start_outlier(_lso_nm)
            if _lso:
                _lso_dir, _lso_le, _lso_se = _lso
                _lso_word = "spiked" if _lso_dir == "spike" else "was unusually suppressed"
                print(f"  [SP LAST-START OUTLIER -- {_lso_role}] {_lso_nm}: last start ERA {_lso_le:.2f} {_lso_word} vs season {_lso_se:.2f} -- likely regresses toward season mean, informational only")
        if not _bullpen_data_live and _grade in ("A+", "A"):
            _grade = "A- [BP DATA DEAD]"
        if not _bp_data_available and _grade in ("A+", "A"):
            _grade = "A- [BP RISK SCORE MISSING]"
        _g_labels = f"edge={'Y' if _g_crit[0] else 'N'} SP={'Y' if _g_crit[1] else 'N'} BP={'Y' if _g_crit[2] else 'N'} juice={'Y' if _g_crit[3] else 'N'}"
        # 2026-07-13: confidence gradient within A- -- the grade alone doesn't
        # distinguish a play that JUST cleared the 15% edge floor from one that
        # cleared it by 7+ points. Real case: TB ML at +14.9% edge (thin) lost
        # against a legitimately strong opposing arm -- a thin-edge A- against a
        # good SP is a different risk profile than a wide-edge A- against a weak
        # one, and the grade alone couldn't communicate that distinction.
        _g_edge_tier = ""
        if _grade.startswith("A-"):
            if _g_edge >= 0.22: _g_edge_tier = "  (strong edge)"
            elif _g_edge >= 0.18: _g_edge_tier = "  (solid edge)"
            elif _g_edge >= 0.15: _g_edge_tier = "  (thin edge)"
        print(f"  Grade     : {_grade}  ({_g_met}/{len(_g_crit)}: {_g_labels}){_g_edge_tier}")

        # Run-environment confidence flag (2026-07-29) -- O/U model-vs-market
        # divergence was informational-only (the O/U edge-ceiling watch-list
        # above, Cell 89's market snapshot) and never touched the ML side's
        # grade. Real cases: Astros/Rangers (model 7.8 vs market 10.0) and
        # Guardians (model 7.1 vs market 9.0) -- a gap that large means the
        # same team/SP/bullpen/park inputs the ML pick is built from likely
        # have a real problem, not just an O/U miss. Caps the grade the same
        # way SP-collapse does above; no size change.
        _RUN_ENV_GAP = 1.5
        _g_ou_line = r.get("over_under")
        _g_pred_tot = r.get("pred_total_runs")
        if pd.notna(_g_ou_line) and pd.notna(_g_pred_tot):
            _g_run_gap = abs(float(_g_pred_tot) - float(_g_ou_line))
            if _g_run_gap >= _RUN_ENV_GAP:
                if _grade in ("A+", "A"):
                    _grade = "A-"
                    print(f"  [RUN ENVIRONMENT] model total {float(_g_pred_tot):.1f} vs market {float(_g_ou_line):.1f} "
                          f"({_g_run_gap:.1f}R gap) -- model's own game-environment read is suspect, grade capped at A-")
                else:
                    print(f"  [RUN ENVIRONMENT] model total {float(_g_pred_tot):.1f} vs market {float(_g_ou_line):.1f} "
                          f"({_g_run_gap:.1f}R gap) -- verify game-environment inputs before playing")

        # ── Team record gap (informational only) ────────────────────────────
        # Flags when the pick's season record is notably worse than the
        # opponent's. Doesn't touch grade/size/edge -- an SP-matchup-driven
        # edge can legitimately outweigh a record gap for one game -- this
        # just surfaces context the card didn't show anywhere else. Built
        # 2026-07-27 after Milwaukee (66-39) vs San Francisco (44-61) needed
        # a manual web search to catch; the model's own pick was the Giants.
        # BUG FIX (MLB-RECORD-1, 2026-09-24 user audit): same same-city
        # substring-collision risk as Gate 9's lookup above (LAD/LAA, NYY/
        # NYM, Chicago Cubs/White Sox) -- reuse the safe last-word-nickname
        # matcher (_trw_lookup_generic) instead of this inline "any word
        # >3 chars" substring check.
        _trg_pick = _trw_lookup_generic(_team_records, _g_bet_team_nm)
        _trg_opp = _trw_lookup_generic(_team_records, _g_opp_team_nm)
        if _trg_pick and _trg_opp:
            _trg_pick_pct = _trg_pick.get("win_pct", 0.5)
            _trg_opp_pct = _trg_opp.get("win_pct", 0.5)
            if _trg_opp_pct - _trg_pick_pct >= 0.10:
                _trg_pick_rec = f"{_trg_pick.get('wins', 0)}-{_trg_pick.get('losses', 0)}"
                _trg_opp_rec = f"{_trg_opp.get('wins', 0)}-{_trg_opp.get('losses', 0)}"
                print(f"  !! TEAM RECORD GAP: {_g_bet_team_nm} ({_trg_pick_rec}, {_trg_pick_pct:.1%}) vs "
                      f"{_g_opp_team_nm} ({_trg_opp_rec}, {_trg_opp_pct:.1%}) -- season record favors "
                      f"the opponent; edge here is driven by today's SP matchup, verify that's enough "
                      f"to overcome the record gap")

        # ── Fix 7: Series momentum -- now applied earlier in card build
        # (see _SM_COLD_PENALTY/_SM_HOT_BOOST block above); this stays as
        # the per-pick narrative note only, not a second application.
        _f7_away = str(r.get(away_col,"")).lower(); _f7_home = str(r.get(home_col,"")).lower()
        _f7_ctx = (_series_momentum.get((_f7_away, _f7_home)) or _series_momentum.get((_f7_home, _f7_away)))
        if _f7_ctx:
            _f7_pick = str(r.get("bet_team","")).lower()
            _f7_streak = next((v.get("streak",0) for k,v in _f7_ctx.items() if any(w in k for w in _f7_pick.split() if len(w)>3)), 0)
            if _f7_streak <= -2:
                print(f"  [SERIES MOMENTUM] {r.get('bet_team','')} lost last 2 series games -- -{_SM_COLD_PENALTY:.2f}R applied earlier")
            elif _f7_streak >= 2:
                print(f"  [SERIES MOMENTUM] {r.get('bet_team','')} won last 2 series games -- +{_SM_HOT_BOOST:.2f}R applied earlier")

            # Series sweep attempt flag (informational only) -- leading team can
            # look ahead/relax, trailing team often has a bounce-back push. Not
            # an edge/grade adjustment -- see note above this block for why.
            _sw_ctx = (_series_context.get((_f7_away, _f7_home)) or _series_context.get((_f7_home, _f7_away)))
            if _sw_ctx:
                _sw_game_num = _sw_ctx.get("game_num", 1)
                _sw_games_in_series = _sw_ctx.get("games_in_series", 1)
                if _sw_game_num == _sw_games_in_series and _sw_games_in_series > 1 and _f7_streak >= 2:
                    print(f"  [SWEEP ATTEMPT] {r.get('bet_team','')} has won the series so far -- game {_sw_game_num}/{_sw_games_in_series} sweep spot, a known variance risk (informational only, no edge/grade adjustment)")

        # Pre-All-Star-break flag (informational only) -- manually curated per
        # season; update _ASB_LAST_GAME_DATE every year. Same single-data-point
        # caveat as the sweep flag above -- context only, no edge/grade change.
        _ASB_LAST_GAME_DATE = "2026-07-12"  # 2026: last game day before ASB festivities (Philadelphia, Jul 13-14)
        try:
            _asb_gdt = r.get(date_col) if date_col else None
            if _asb_gdt is not None and _asb_gdt is not pd.NaT:
                _asb_dt = pd.to_datetime(_asb_gdt, errors="coerce", utc=True)
                if _asb_dt is not pd.NaT and _asb_dt.strftime("%Y-%m-%d") == _ASB_LAST_GAME_DATE:
                    print(f"  [PRE-ASB] Last game before the All-Star break -- verify standings context manually (comfortable leaders can see a letdown, trailing teams a bounce-back push); informational only, no edge/grade adjustment")
        except Exception:
            pass

        # ── Day-game deadline flag ────────────────────────────────────
        _day_flag = ""
        try:
            import pandas as _pd_dg
            _gdt = r.get(date_col) if date_col else None
            if _gdt is not None and _gdt is not _pd_dg.NaT:
                _gdt_p = _pd_dg.to_datetime(_gdt, errors="coerce", utc=True)
                if _gdt_p is not _pd_dg.NaT:
                    _g_hour_et = (_gdt_p.hour - 4) % 24
                    if _g_hour_et < 16:
                        _gdt_min = _gdt_p.minute
                        _g_hour_12 = _g_hour_et % 12 or 12
                        _g_ampm = "AM" if _g_hour_et < 12 else "PM"
                        _day_flag = " [DAY GAME]"
                        print(f"  [DAY GAME] First pitch {_g_hour_12}:{_gdt_min:02d} {_g_ampm} ET")
        except Exception: pass


        # ── Fix 8: Edge shift ─────────────────────────────────────────────────
        _e8_key = f"{str(r.get(away_col,'')).lower()}_{str(r.get(home_col,'')).lower()}_{str(r.get('bet_team','')).lower()}"
        _e8_prior = globals().get("_mlb_edge_history",{}).get(_e8_key)
        if _e8_prior is not None:
            _e8_shift = round(float(r["edge"]) - _e8_prior, 3)
            if abs(_e8_shift) >= 0.03:
                print(f"  [EDGE SHIFT] {r.get('bet_team','')} ML: {_e8_prior:+.1%} -> {r['edge']:+.1%} ({_e8_shift:+.1%}) -- check for SP/lineup change")
        globals()["_mlb_edge_history"][_e8_key] = float(r["edge"])

        # ── F5 OVER gate ─────────────────────────────────────────────────────
        if not _f5_market_live and str(r.get("bet_type","")).upper() == "OVER":
            print("  [F5 UNVERIFIED] OVER play -- no real F5 odds; model total direction unconfirmed")
            if not unit.startswith("0.5u") and "No bet" not in unit:
                unit = "0.5u [F5 UNVERIFIED]"

        # BUG FIX (2026-08-27): record this pick's final grade/size before the
        # branch split below -- both _grade and unit are fully finalized by this
        # point (every cap above, including F5 UNVERIFIED, has already applied).
        # See the dict-init comment above for why this needs to exist at all.
        # REMOVED (2026-08-30, user request -- "remove the final card selector"):
        # this used to withhold OFFICIAL status from any play graded below A/A+
        # or sized at 0.5u (Cell 6e66db20 reads mlb_grade_downgrades to do the
        # withholding) -- e.g. today's Milwaukee Brewers ML at +11.2% edge got
        # graded B+ and was pulled from OFFICIAL despite otherwise qualifying.
        # The letter grade above still computes and prints for informational
        # value (SP collapse / bullpen-data / run-environment context is still
        # useful to read), it just no longer gates OFFICIAL status -- that's
        # now governed purely by passes_edge_rule (the Gate 1-10 system above).
        pass
        mkt = f"{r['market_implied_prob']:.1%}" if pd.notna(r.get("market_implied_prob")) else "--"
        # RAW MARKET ODDS ON THE CARD (2026-09-11, user request): edge%/prob%
        # already printed here, but the underlying price never was -- a reader
        # sees '+9.9% edge' with no way to tell that's actually a -191 favorite
        # until they go look the game up themselves. home_ml/away_ml are
        # already fetched (feed the market_implied_prob above) and ride along
        # on every cand row already -- just never printed.
        _r_ml = r.get('home_ml') if str(r.get('bet_side','')).lower() == 'home' else r.get('away_ml')
        _r_ml_str = f"{int(_r_ml):+d}" if pd.notna(_r_ml) else "--"
        if unit.startswith("0.5u") and "No bet" not in unit:
            print(f"  Bet       : {r['bet_team']} ML ({str(r.get('bet_side','')).upper()})")
            print(f"  Model     : {r['p_bet_win_model']:.1%}  |  Market: {_r_ml_str} ({mkt})  |  Edge: {r['edge']:.1%}")
            # Fix: "[WATCH ONLY -- below 1.0u minimum]" didn't say WHICH cap actually
            # produced 0.5u -- ELO and bullpen caps funnel through the same `unit`
            # string, so build the real reason stack instead of guessing.
            # ELO-unavail no longer reaches here -- it's a hard block above now.
            _size_reasons = []
            if not _bullpen_data_live: _size_reasons.append("BP data dead")
            if "[" in unit:
                _unit_reason = unit.split("[", 1)[1].rstrip("]")
                if _unit_reason not in _size_reasons:
                    _size_reasons.append(_unit_reason)
            _size_reason_str = f"  [HARD CAP: {' | '.join(_size_reasons)}]" if _size_reasons else "  [WATCH ONLY -- below 1.0u minimum]"
            print(f"  Size      : {unit}{_size_reason_str}")
            print("-" * 60)
            _market_played_ct += 1
            rank += 1
            continue
        # Duplicate header removed entirely (was: print(f"\n#{rank}  {away} @ {home}
        # {_day_flag}")). This game's header already printed once, unconditionally,
        # at the top of the loop (~line 3801) -- a 2026-07-12 fix moved it there
        # specifically so every branch (0.5u-capped and full-size) shows it exactly
        # once. This second copy, plus "_day_flag", were never cleaned up after
        # that move: [DAY GAME] itself already prints as its own standalone line
        # (~line 4002) regardless of branch, so _day_flag's only remaining job was
        # to double it onto this redundant header. The undefined "_sp_change_flag"
        # NameError (fixed last round) was masking this duplicate-print bug the
        # whole time -- every full-size play crashed here before it could ever
        # print twice, so the duplication was invisible until the crash was fixed.
        print(f"  Bet       : {r['bet_team']} ML ({str(r.get('bet_side','')).upper()})")
        print(f"  Model     : {r['p_bet_win_model']:.1%}  |  Market: {_r_ml_str} ({mkt})  |  Edge: {r['edge']:.1%}")
        # Grade already printed once, unconditionally, at line ~3939 (right after
        # SP Matchup/Proj Score, same fix-pattern comment as the header above) --
        # this was a second copy of the same line, missing only the edge-tier
        # suffix the first one has.
        print(f"  Size      : {unit}  |  Final Score: {r['final_score']:.1%}")
        print("-" * 60)
        _display_units[r.get("bet_team","")] = unit
        if not str(unit).startswith("No bet"):
            _market_played_ct += 1
            # Same mlb_lineup_flags penalty math the [NET INJURY DELTA] bullet
            # (cell 6e66db20) already computes -- reused here, not
            # reimplemented, so this can't drift from what the card displays.
            # Official-play ledger logging removed from this section (2026-08-11):
            # this "top_market" qualification loop is a separate, older selection
            # system that disagrees with the real card (see the 2026-08-07 fix note
            # in Cell 6e66db20 -- that cell's _snap_ranked/_log_official_play call is
            # the actual source of truth, matching FINAL CARD -- OFFICIAL PLAYS + WHY).
            # Logging both here and there wrote two disagreeing "official" pick sets
            # into the same ledger file.
        rank += 1

# ── Nearest Misses ────────────────────────────────────────────────────────────
_nm = top_market[top_market["passes_edge_rule"] != 1].copy() if "passes_edge_rule" in top_market.columns else pd.DataFrame()
if len(_nm) == 0 and len(top_model) > 0:
    _nm = top_model.copy()
if len(_nm) > 0 or len(top_market) == 0:
    if len(top_market) == 0:
        print("NO PLAY -- No market plays and no strong model-only picks today.")
    print("Nearest misses (top edges that did not qualify):")
    print(f"  {'Game':<38} {'Side':<6} {'Model%':>7} {'Edge':>7}  Blocked by")
    print(f"  {'-'*78}")
    _cand_all = []
    for _c_src in [globals().get("cand"), globals().get("top_market")]:
        if isinstance(_c_src, pd.DataFrame):
            _cand_all.append(_c_src)
    if _cand_all:
        import pandas as _cpd
        # BUG FIX (found live, same session): drop_duplicates() with no
        # subset hashes every column -- _pick_post_il/_opp_post_il hold raw
        # dicts (see _post_il_info above), which pandas can't hash, so this
        # crashed the whole MLB run every time it hit a real no-play day
        # (pre-existing, just never exercised until today's slate happened
        # to qualify zero plays). Excluded from dedup only -- kept in the
        # concatenated frame for anything else that reads them below.
        _cand_combo = _cpd.concat(_cand_all)
        _dedup_drop = [c for c in ['_pick_post_il', '_opp_post_il'] if c in _cand_combo.columns]
        _cand_combo = _cand_combo.drop_duplicates(subset=[c for c in _cand_combo.columns if c not in _dedup_drop])
        for _, _nr in _cand_combo.sort_values("edge",ascending=False).head(8).iterrows():
            _ne = float(_nr.get("edge",0) or 0); _np = float(_nr.get("p_bet_win_model",0) or 0)
            if _ne < 0.07: continue
            _ng = f"{_nr.get(away_col,'')} @ {_nr.get(home_col,'')}"[:38]
            _ns = str(_nr.get("bet_side",""))[:5].upper()
            _nb = f"edge {_ne:.1%}<15%" if _ne < 0.15 else (f"prob {_np:.1%}<57%" if _np < 0.57 else "other")
            if _nr.get("passes_edge_rule",0) != 1:
                print(f"  {_ng:<38} {_ns:<6} {_np:>7.1%} {_ne:>7.1%}  {_nb}")

print()


# ── RUN LINE (-1.5) EVALUATION ───────────────────────────────────────────────
_rl_espn_cache = {}

def _ml_to_prob(odds_str):
    try:
        o = int(str(odds_str).replace("+","").strip())
        return 100/(o+100) if o > 0 else abs(o)/(abs(o)+100)
    except: return 0.5

def _poisson_win_by2(mu_win, mu_lose):
    import math
    def pmf(k, mu): return (mu**min(k,20) * math.exp(-mu)) / math.factorial(min(k,20))
    def cdf(k, mu): return sum(pmf(i,mu) for i in range(k+1))
    total = 0.0
    for y in range(25):
        py = pmf(y, mu_lose)
        if py < 1e-9: break
        total += py * (1 - cdf(y+1, mu_win))
    return min(max(total, 0.0), 1.0)

def _fetch_rl_odds(home_lower):
    if home_lower in _rl_espn_cache: return _rl_espn_cache[home_lower]
    result = (None, None)
    try:
        _sb = requests.get("https://site.api.espn.com/apis/site/v2/sports/baseball/mlb/scoreboard", timeout=6).json()
        for _ev in _sb.get("events", []):
            _comp = _ev.get("competitions", [{}])[0]
            _hname = next((t.get("team",{}).get("displayName","").lower() for t in _comp.get("competitors",[]) if t.get("homeAway")=="home"), "")
            if not any(w in _hname for w in home_lower.split() if len(w)>3): continue
            _sm = requests.get(f"https://site.api.espn.com/apis/site/v2/sports/baseball/mlb/summary?event={_ev['id']}", timeout=6).json()
            _pc = (_sm.get("pickcenter") or [{}])[0]
            _ps = _pc.get("pointSpread", {})
            _h = (_ps.get("home",{}).get("close",{}).get("odds") or _ps.get("home",{}).get("open",{}).get("odds"))
            _a = (_ps.get("away",{}).get("close",{}).get("odds") or _ps.get("away",{}).get("open",{}).get("odds"))
            result = (str(_h) if _h else None, str(_a) if _a else None)
            break
    except: pass
    _rl_espn_cache[home_lower] = result
    return result

_rl_candidates = []
_rl_eval = {}
RL_MIN_EDGE = 0.08
_MAX_ODDS_IMPLIED_FLOOR_RL = 0.2857  # 2026-07-10: hard stop at +250 (was +300/0.25), mirrors the ML gate
RL_MIN_MODEL = 0.52
# BUG FIX (2026-09-02, user audit -- "RL essentially never qualifies"):
# this pre-filter required the raw MEAN projected run differential to be
# 2.0+ runs before a game was even evaluated for RL -- but _poisson_win_by2
# right below already computes the real, variance-aware probability of
# winning by 2+ from that same mean differential, and RL_MIN_MODEL (0.52)
# already gates on THAT number. A 2.0-run mean gap is a genuinely rare
# margin in MLB (most real favorites this whole session projected in the
# 0.3-1.5R range) -- requiring it here, on top of the real Poisson-based
# gate that follows, double-counts the same signal and was excluding
# legitimate win-by-2 opportunities where a modest mean edge (~1.0-1.5R)
# still carries a real win-by-2 probability once variance is accounted
# for properly. Lowered to 1.0 -- still a real pre-filter against trivial
# coin-flip games, just no longer redundant with the better check below.
# Provisional, revisit after this specific change has graded history.
RL_MIN_PRED_DIF = 1.0

_rl_bp_yday_str = (pd.Timestamp.today() - pd.Timedelta(days=1)).strftime("%Y-%m-%d")
_rl_2day_str    = (pd.Timestamp.today() - pd.Timedelta(days=2)).strftime("%Y-%m-%d")

# 2026-07-26 fix: RL had a bullpen *fatigue/usage* check but nothing reading
# closer *reliability*. Real case: Pirates -1.5 lost 2026-07-26 -- Pittsburgh
# led 7-1 by the 3rd, closer Mason Montgomery gave up a 2-run HR in the 9th
# (8-7), game still won but the cover died. Montgomery had blown a save two
# days earlier and the team was tied for 3rd-most blown saves in MLB -- both
# findable pregame. This only discounts the RL probability -- ML is untouched,
# since ML doesn't care when in the game runs score. Best-effort: wrapped in
# try/except like every other live fetch in this cell, so an API shape miss
# just no-ops (flag never fires) rather than breaking the RL gate.
_rl_closer_cache = {}

def _fetch_closer_recent_risk(team_name):
    """Return (risky, detail) for a team's primary closer's last 10 relief
    outings. risky=True on 2+ blown saves or >5.00 ERA in that span."""
    _ck = str(team_name).lower().strip()
    if _ck in _rl_closer_cache: return _rl_closer_cache[_ck]
    result = (False, None)
    try:
        _tid = _mlb_team_ids.get(_ck) if "_mlb_team_ids" in globals() else None
        if not _tid:
            _rl_closer_cache[_ck] = result; return result
        _lr = requests.get(
            "https://statsapi.mlb.com/api/v1/stats/leaders"
            "?leaderCategories=saves&sportId=1&season=2026&limit=200",
            timeout=6
        )
        _leaders = (_lr.json().get("leagueLeaders", [{}])[0].get("leaders", [])
                    if _lr.status_code == 200 else [])
        _closer_pid = None
        for _ld in _leaders:
            if int((_ld.get("team") or {}).get("id", -1) or -1) == int(_tid):
                _closer_pid = (_ld.get("person") or {}).get("id")
                break
        if not _closer_pid:
            _rl_closer_cache[_ck] = result; return result
        _gr = requests.get(
            f"https://statsapi.mlb.com/api/v1/people/{_closer_pid}/stats"
            "?stats=gameLog&group=pitching&season=2026&gameType=R",
            timeout=6
        )
        _splits = _gr.json().get("stats", [{}])[0].get("splits", []) if _gr.status_code == 200 else []
        _relief = [s for s in _splits if int(s.get("stat", {}).get("gamesStarted", 0) or 0) == 0]
        _l10 = _relief[-10:]
        if not _l10:
            _rl_closer_cache[_ck] = result; return result
        _bs = sum(int(s.get("stat", {}).get("blownSaves", 0) or 0) for s in _l10)
        _ip = sum(float(s.get("stat", {}).get("inningsPitched", 0) or 0) for s in _l10)
        _er = sum(int(s.get("stat", {}).get("earnedRuns", 0) or 0) for s in _l10)
        _era_recent = round(_er / _ip * 9, 2) if _ip > 0 else None
        if _bs >= 2 or (_era_recent is not None and _era_recent > 5.00):
            _era_part = f", {_era_recent} ERA" if _era_recent is not None else ""
            result = (True, f"{_bs} blown save(s) in last {len(_l10)} relief outings{_era_part}")
    except Exception:
        pass
    _rl_closer_cache[_ck] = result
    return result

# Official-ML-team set (2026-08-01 fix) -- a team with a real OFFICIAL ML
# play (cand.passes_edge_rule == 1) must always get an RL evaluation/status
# recorded below, even when the projected margin is under RL_MIN_PRED_DIF.
# Real case: SF Giants carried the OFFICIAL ML play today, but the RL
# section never even ran for them -- the margin gate a few lines down used
# to `continue` before _rl_team was even determined, so _rl_eval never got
# an entry and the Daily Market Snapshot printed "not evaluated" instead of
# a real status. That silently hid the closer-risk/sharp-money-relevant RL
# picture that's useful context for grading the ML play. The calibrated
# margin gate just below (`_rl_cal_margin < 2.0`) still applies and can
# still correctly block a genuinely-too-small margin from becoming its own
# qualifying RL play -- this only stops the *earlier*, silent skip that hid
# these teams from _rl_eval entirely.
_official_ml_teams = set(
    cand.loc[cand["passes_edge_rule"] == 1, "bet_team"].astype(str).str.lower()
) if "passes_edge_rule" in cand.columns and "bet_team" in cand.columns else set()

for _ri, _rr in board.iterrows():
    if not _rr.get("has_odds"): continue
    _rh_name = str(_rr.get(home_col,"")).lower()
    _ph = float(_rr.get("pred_home_runs") or 0)
    _pa = float(_rr.get("pred_away_runs") or 0)
    _diff = _ph - _pa
    _rl_official_involved = (
        str(_rr.get(home_col, "")).lower() in _official_ml_teams
        or str(_rr.get(away_col, "")).lower() in _official_ml_teams
    )
    if abs(_diff) < RL_MIN_PRED_DIF and not _rl_official_involved: continue
    if _diff > 0:
        _rl_team = _rr.get(home_col,""); _rl_side = "home"; _mu_w, _mu_l = _ph, _pa
    else:
        _rl_team = _rr.get(away_col,""); _rl_side = "away"; _mu_w, _mu_l = _pa, _ph
    # Park factor gate
    _rl_pf = float(_rr.get("park_factor_used") or 1.00)
    if _rl_pf >= 1.25 and _rl_side == "away": continue
    if _rl_pf >= 1.15 and abs(_diff) < 2.0: continue
    # Calibrated margin gate
    _rl_cal_margin = abs(_mu_w - _mu_l)
    if _rl_cal_margin < 2.0:
        _rl_eval[_rl_team.lower()] = {"status":"margin blocked","rl_odds":None,"reason":f"cal {_rl_cal_margin:.1f}R"}
        continue
    # Walk-rate variance gate (2026-07-29) -- the RL loop iterates board.iterrows()
    # directly and never touches cand/passes_edge_rule, so a starter hard-blocked
    # by the ML section's elevated-BB% gate (Gate 3, _BB_PCT_ELEVATED) could still
    # surface here on the run line (e.g. Cantillo blocked at ML but qualifying via
    # RL). Same hard-stop, no edge override, applied to whichever team's starter
    # is on the RL side being evaluated.
    _rl_pick_sp = next((str(_rr.get(c) or '').strip()
                        for c in (_home_pit_cols if _rl_side == "home" else _away_pit_cols)
                        if _rr.get(c) and str(_rr.get(c)) != 'nan'), '')
    _rl_pick_sp_bb = np.nan
    if _rl_pick_sp and 'mlb_pitcher_fip_lookup' in globals():
        _rl_fev = mlb_pitcher_fip_lookup.get(_norm_name(_rl_pick_sp))
        if _rl_fev and pd.notna(_rl_fev.get('bb_pct_pit')):
            _rl_pick_sp_bb = float(_rl_fev.get('bb_pct_pit'))
    if pd.notna(_rl_pick_sp_bb) and _rl_pick_sp_bb >= _BB_PCT_ELEVATED:
        print(f"  [RL WALK-RATE GATE] {_rl_team} -1.5 blocked -- {_rl_pick_sp} season BB% "
              f"{_rl_pick_sp_bb:.1%} >= {_BB_PCT_ELEVATED:.1%} (same hard-stop as ML Gate 3)")
        _rl_eval[_rl_team.lower()] = {"status":"walk-rate blocked","rl_odds":None,
            "reason":f"SP BB% {_rl_pick_sp_bb:.1%}"}
        continue
    # SP rating staleness gate (2026-07-29) -- same reasoning as the walk-rate gate
    # just above: this loop never touches cand/passes_edge_rule, so Gate 6's
    # DataFrame-level block (cell 88, ML section) doesn't cover the RL side at all.
    # Without this, a stale-rating starter blocked at ML could still surface here --
    # and once Gate 6 correctly excludes the ML play, the RL section's "double
    # exposure" skip (which only fires when an ML play already exists on this team)
    # no longer protects this case either, since there's no longer an ML play to
    # collide with.
    _rl_pick_sp_stale = _cand_sp_stale(_rl_pick_sp) if _rl_pick_sp else None
    if _rl_pick_sp_stale:
        print(f"  [RL SP STALENESS] {_rl_team} -1.5 blocked -- {_rl_pick_sp} returned from "
              f"60-day IL ({_rl_pick_sp_stale}), current-season sample too thin to trust the rating")
        _rl_eval[_rl_team.lower()] = {"status":"stale rating blocked","rl_odds":None,
            "reason":f"SP returned from 60-day IL {_rl_pick_sp_stale}"}
        continue
    # Bullpen check
    _pg_bet_ab_rl = _pg_abbr(str(_rl_team))
    _rl_bp_avail = bool("_rl_yday" in globals() and _rl_yday and _pg_bet_ab_rl in _rl_yday)
    if not _rl_bp_avail:
        # Fix: _rl_yday is a separate, more fragile live-usage feed than the historical
        # RA fallback (_bpen_lookup) that already covers FanGraphs 403s for predictions.
        # Don't hard-block the RL gate on live-usage alone -- fall back to historical RA
        # when it's loaded, and only hard-block when both sources are missing.
        _rl_hist_bp_avail = bool(
            "_bpen_lookup" in globals() and
            _bpen_lookup.get(_norm_team(str(_rl_team))) is not None
        )
        if _rl_hist_bp_avail:
            print(f"  [RL WATCH] {_rl_team} -1.5 -- live bullpen usage unavailable, using historical RA fallback")
        else:
            print(f"  [RL BP REQUIRED] {_rl_team} -1.5 blocked -- bullpen data unavailable")
            _rl_eval[_rl_team.lower()] = {"status":"bp required", "reason":"bullpen data unavailable"}
            continue
    _rl_model_prob = _poisson_win_by2(_mu_w, _mu_l)
    _rl_closer_risky, _rl_closer_detail = _fetch_closer_recent_risk(_rl_team)
    if _rl_closer_risky:
        print(f"  [RL CLOSER RISK] {_rl_team} -1.5 -- closer red flag: {_rl_closer_detail} -- discounting cover probability")
        _rl_model_prob = max(0.0, _rl_model_prob - 0.05)
    # 2026-09-10 (user audit -- Diamondbacks -1.5, 2026-09-09 real loss):
    # propagate the same net SP-quality discount already applied to this
    # team's ML edge (HR9/cold-streak/FIP-reliability/role/mismatch/bulk-
    # FIP/thin-sample/ELO-ERA -- see the snapshot near cand['edge'] above)
    # onto the RL probability. Without this, a starter too shaky to clear
    # the ML bar could still clear -1.5 with zero discount, which is
    # exactly what happened here: Gallen's cold-streak + HR/9 flags
    # correctly blocked the ML play (+10.2%, below the ~12% bar) but the
    # RL evaluation never saw either flag and cleared clean at +15.7%.
    _rl_sp_quality_shave = globals().get('_sp_quality_shave_by_team', {}).get(str(_rl_team).lower(), 0.0)
    if _rl_sp_quality_shave:
        _rl_model_prob_pre_shave = _rl_model_prob
        _rl_model_prob = max(0.0, _rl_model_prob - _rl_sp_quality_shave)
        print(f"  [RL SP-QUALITY SHAVE] {_rl_team} -1.5: {_rl_model_prob_pre_shave:.1%} -> "
              f"{_rl_model_prob:.1%} ({_rl_sp_quality_shave:+.1%}) -- same SP-quality discount "
              f"already applied to this team's ML edge")
    if _rl_model_prob < RL_MIN_MODEL: continue
    _rl_eval[_rl_team.lower()] = {"status":"pending","model_prob":_rl_model_prob,"pred_diff":abs(_mu_w-_mu_l),"rl_odds":None,"edge":None}
    _h_rl, _a_rl = _fetch_rl_odds(_rh_name)
    _rl_odds = _h_rl if _rl_side == "home" else _a_rl
    if not _rl_odds:
        _rl_eval[_rl_team.lower()].update({"status":"no odds"})
        continue
    # 2026-07-27 fix: RL odds sanity check. Covering -1.5 (win by 2+) is always
    # a harder outcome than just winning, so the RL price for the model's
    # favored team can never imply a HIGHER win probability than that same
    # team's own moneyline market price -- if it does, the fetched odds are for
    # the wrong side. Real incident 2026-07-26: Cardinals ML market ~55.1% but
    # the fetched "Cardinals -1.5" price (-178) implied ~64%; Astros ML market
    # ~49.0% but the fetched "Astros -1.5" price (-203) implied ~67% -- both
    # backwards, and both got blocked by the juice gate as if they were real
    # heavy-favorite prices instead of thrown out as bad data. Root cause not
    # confirmed (ESPN pickcenter home/away swap vs. a scraper-side mismatch),
    # but the harder-than-ML relationship always holds for genuine RL pricing,
    # so this is a safe, general guard rather than a guess at the internal cause.
    _rl_ml_mkt_check = float(_rr.get("home_market_prob" if _rl_side=="home" else "away_market_prob") or 0)
    _rl_odds_implied_check = _ml_to_prob(_rl_odds)
    if _rl_ml_mkt_check > 0 and _rl_odds_implied_check > _rl_ml_mkt_check:
        print(f"  [RL ODDS SANITY] {_rl_team} -1.5 @ {_rl_odds} implies {_rl_odds_implied_check:.1%}, "
              f"higher than its own ML market prob ({_rl_ml_mkt_check:.1%}) -- covering by 2+ can't be "
              f"more likely than just winning, so this price is almost certainly the wrong side; discarding")
        _rl_eval[_rl_team.lower()].update({"status":"odds sanity fail",
            "reason":f"RL implied {_rl_odds_implied_check:.1%} > ML mkt {_rl_ml_mkt_check:.1%}"})
        continue
    # Fix 2: RL juice gate
    try: _rl_am_v = int(str(_rl_odds).replace("+",""))
    except: _rl_am_v = -110
    _rl_juice_tier = "PLAY"
    if _rl_am_v <= -150:
        print(f"  [RL JUICE TIER] {_rl_team} -1.5 BLOCKED -- {_rl_odds} worse than -150")
        # Fix: same class of bug as the double-exposure gate -- no "reason" key meant
        # the RL BLOCKED summary printed "juice blocked -- " with nothing after it.
        _rl_eval[_rl_team.lower()].update({"status":"juice blocked","rl_odds":_rl_odds,
            "reason":f"{_rl_odds} worse than -150"})
        continue
    elif _rl_am_v <= -130:
        print(f"  [RL JUICE TIER] {_rl_team} -1.5 WATCH ONLY -- {_rl_odds} in -130 to -150 range")
        _rl_juice_tier = "WATCH"
    elif _rl_am_v <= -110:
        print(f"  [RL JUICE TIER] {_rl_team} -1.5 CAUTION -- {_rl_odds} in -110 to -130 range")
        _rl_juice_tier = "CAUTION"
    else:
        print(f"  [RL JUICE TIER] {_rl_team} -1.5 PLAY eligible -- {_rl_odds} better than -110")
    _rl_mkt_prob = _ml_to_prob(_rl_odds)
    _rl_edge = _rl_model_prob - _rl_mkt_prob
    # Logit-space symmetric edge -- same 2026-07-12 port as the ML section
    # above (see there for full rationale). Replaces the old step-bracket
    # discount, which structurally favored underdog-side edges over
    # favorite-side edges of the same real strength.
    _rl_required_edge_logit = _mlb_logit(0.5 + RL_MIN_EDGE) - _mlb_logit(0.5)
    _rl_edge_logit = _mlb_logit(_rl_model_prob) - _mlb_logit(_rl_mkt_prob)
    if _rl_edge_logit >= _rl_required_edge_logit and _rl_edge < RL_MIN_EDGE:
        print(f"  [LOGIT-SPACE EDGE] {_rl_team} -1.5 @ {_rl_odds}: {_rl_edge:.1%} raw edge qualifies on symmetric logit-space edge ({_rl_edge_logit:+.2f} vs {_rl_required_edge_logit:+.2f} required, {_rl_mkt_prob:.0%} implied)")
    if _rl_mkt_prob < _MAX_ODDS_IMPLIED_FLOOR_RL:
        print(f"  [MAX ODDS CAP] {_rl_team} -1.5 @ {_rl_odds} blocked -- implied {_rl_mkt_prob:.0%} below {_MAX_ODDS_IMPLIED_FLOOR_RL:.0%} floor (longer than ~+250), regardless of edge")
        _rl_eval[_rl_team.lower()].update({"status":"max odds","edge":_rl_edge,"rl_odds":_rl_odds,
            "reason":f"implied {_rl_mkt_prob:.0%} below {_MAX_ODDS_IMPLIED_FLOOR_RL:.0%} max-odds floor"})
        continue
    # HIGH-CONFIDENCE FAVORITE PATH -- added 2026-07-26, mirrors the ML path
    # (2026-07-25) and the same fix in Soccer/UFC. A genuinely confident favorite
    # (model >= 62%, market already agrees) can qualify on a much smaller edge
    # than the standard RL_MIN_EDGE (8%) bar -- user explicitly OK with a -200-type
    # favorite at a ~7% edge, rather than requiring the full logit-space bar
    # regardless of how strongly the model favors the pick.
    _RL_HIGH_CONF_PROB_MIN = 0.62
    _RL_HIGH_CONF_EDGE_MIN = 0.04
    _rl_high_conf = (
        _rl_model_prob >= _RL_HIGH_CONF_PROB_MIN
        and _rl_mkt_prob >= 0.50
        and _rl_edge >= _RL_HIGH_CONF_EDGE_MIN
        and _rl_edge < 0.25
    )
    if _rl_edge_logit < _rl_required_edge_logit and not _rl_high_conf:
        _rl_eval[_rl_team.lower()].update({"status":"no edge","edge":_rl_edge,"rl_odds":_rl_odds,
            "reason":f"{_rl_edge:.1%} raw ({_rl_edge_logit:+.2f} logit) below {RL_MIN_EDGE:.0%} min"})
        continue
    if _rl_high_conf and _rl_edge_logit < _rl_required_edge_logit:
        print(f"  [High-confidence favorite path] {_rl_team} -1.5 qualified on model "
              f"{_rl_model_prob:.1%} / edge {_rl_edge:.1%} (below standard {RL_MIN_EDGE:.0%} edge bar)")
    # ML edge floor
    _rl_ml_model = float(_rr.get("pred_home_win_prob" if _rl_side=="home" else "pred_away_win_prob") or 0)
    _rl_ml_mkt   = float(_rr.get("home_market_prob" if _rl_side=="home" else "away_market_prob") or 0)
    _rl_ml_edge  = _rl_ml_model - _rl_ml_mkt
    if _rl_ml_edge < 0.06:
        print(f"  [RL GATE] {_rl_team} -1.5 blocked -- ML edge {_rl_ml_edge:.1%} below 6% floor")
        _rl_eval[_rl_team.lower()].update({"status":"blocked","reason":f"ML edge {_rl_ml_edge:.1%}","rl_odds":_rl_odds,"edge":_rl_edge})
        continue
    # Edge ceiling -- same reasoning as the ML section's Gate 5: a genuine
    # 25%+ edge on a real market is rare; more often a data problem.
    _RL_EDGE_CEILING = 0.25
    if _rl_edge >= _RL_EDGE_CEILING:
        print(f"  [EDGE CEILING] {_rl_team} -1.5 blocked -- edge {_rl_edge:.1%} >= {_RL_EDGE_CEILING:.0%}, flag for calibration review, not auto-play")
        _rl_eval[_rl_team.lower()].update({"status":"edge ceiling","reason":f"edge {_rl_edge:.1%} >= {_RL_EDGE_CEILING:.0%} ceiling","rl_odds":_rl_odds,"edge":_rl_edge})
        continue
    # RL/ML internal-consistency sanity check -- the run-line edge should
    # almost always track the moneyline edge for the same team/game (both
    # come from the same underlying win-probability model), not run
    # dramatically ahead of it. RL edge 1.75x+ the ML edge (with a real ML
    # edge to compare against, not a near-zero one) signals the two
    # calculations have drifted apart -- stale input on one side, mismatched
    # game state, etc. -- rather than a real, larger RL-specific signal.
    # 1.75x, not a flat 2.0x, so this actually catches the real case that
    # prompted it (TOR RL +35.0% vs ML +17.9% = 1.96x).
    if _rl_ml_edge >= 0.05 and _rl_edge > 1.75 * _rl_ml_edge:
        print(f"  [RL/ML SANITY CHECK] {_rl_team} -1.5 blocked -- RL edge {_rl_edge:.1%} is {_rl_edge/_rl_ml_edge:.1f}x the ML edge ({_rl_ml_edge:.1%}); internally inconsistent, needs review before grading")
        _rl_eval[_rl_team.lower()].update({"status":"rl/ml mismatch","reason":f"RL edge {_rl_edge:.1%} vs ML edge {_rl_ml_edge:.1%}","rl_odds":_rl_odds,"edge":_rl_edge})
        continue
    # Double-exposure gate
    _ml_play_teams = {str(r.get("bet_team","")).lower() for _, r in top_market.iterrows()}
    if _rl_team.lower() in _ml_play_teams:
        # Fix: the RL BLOCKED summary at the bottom prints status + reason for this
        # team -- it was showing "double exposure -- " with no reason text, reading
        # as a different problem than this line's "ML play already on this team".
        # Same wording in both places now.
        print(f"  [RL GATE] {_rl_team} -1.5 skipped -- ML play already on this team")
        _rl_eval[_rl_team.lower()].update({"status":"double exposure","rl_odds":_rl_odds,"edge":_rl_edge,
            "reason":"ML play already on this team"})
        continue
    _rl_eval[_rl_team.lower()].update({"status":"qualified","rl_odds":_rl_odds,"edge":_rl_edge})
    _rl_su = 1.0 if (_rl_edge >= 0.10 and _rl_model_prob >= 0.55) else 0.5
    if _rl_juice_tier == "CAUTION" and _rl_su > 0.5: _rl_su = 0.5
    if _rl_juice_tier == "WATCH" and _rl_su > 0.35: _rl_su = 0.35
    _rl_sd = int(_rl_su * 30)
    _rl_candidates.append({
        "matchup": f"{_rr.get(away_col,'')} @ {_rr.get(home_col,'')}",
        "bet_team": f"{_rl_team} -1.5",
        "model_prob": round(_rl_model_prob, 4),
        "mkt_prob": round(_rl_mkt_prob, 4),
        "edge": round(_rl_edge, 4),
        "size_u": _rl_su, "size_$": _rl_sd,
        "odds_str": _rl_odds,
        "pred_diff": round(abs(_mu_w - _mu_l), 1),
        "game_temp_f": _rr.get("game_temp_f"),
        away_col: _rr.get(away_col), home_col: _rr.get(home_col),
    })

# ── O/U EVALUATION ────────────────────────────────────────────────────────────
import scipy.stats as _mlb_snorm_mod
_MARKET_SHRINK = 0.35
# SHADOW-1: MLB totals weight = 1 - shrink toward the line (live 0.65 = the old 35% shrink; the
# fitted w~0 from mlb_totals_component_model.py is NOT applied, per user 2026-10-05). Shadow w=1.0 ->
# no shrink, pure component projection vs the line; OU_MIN_EDGE 18% unchanged.
try:
    import json as _mt_json
    _mt_full = _mt_json.load(open(r"C:\Users\<user>\OneDrive\Documents\KAI Labs\sports\model_blend_weights.json"))
    _mt_w = (_mt_full.get("shadow_weights" if globals().get("_MLB_SHADOW_MODE") else "live_weights") or {}).get("MLB_TOTAL", 0.65)
    _MARKET_SHRINK = round(1.0 - float(_mt_w.get("w", 0.65) if isinstance(_mt_w, dict) else _mt_w), 4)
    print(f"[{'SHADOW' if globals().get('_MLB_SHADOW_MODE') else 'LIVE'} WEIGHT] MLB totals: w={1 - _MARKET_SHRINK:.2f} -> shrink toward the line {_MARKET_SHRINK:.0%}")
except Exception as _mte:
    print(f"[TOTALS WEIGHT] unreadable ({_mte}) -- shrink stays {_MARKET_SHRINK:.0%}")
_MLB_JUICE = 110/210
MLB_TOTAL_STD = 4.57  # 2026-07-24 fix: was 1.5, ~3x too tight vs the empirical std of actual MLB game totals (4.57R, computed from mlb_model_table_v1.parquet, 8348 games). A too-tight std here silently converts ordinary model-vs-market gaps into wildly overconfident O/U edges (e.g. a 2R gap read as ~41% edge instead of ~18%).
_mlb_snorm = _mlb_snorm_mod.norm
_ou_plays_out = []

# TOTALS CALIBRATION BIAS CORRECTION -- wired in 2026-07-25
# `mlb_totals_calibration` (Cell 85, built 2026-07-23) proved a real per-bucket
# bias in the raw projection (e.g. <8.0-total games historically run ~1.1-1.7R
# HIGHER than projected) but was computed and never read anywhere else in this
# notebook -- the O/U gate's edges never benefited from it. This closes that gap:
# corrects _pred_t toward the empirical actual-vs-projected relationship for its
# own bucket, before the market-shrink/CDF step, so the edge itself reflects the
# correction instead of it sitting in a diagnostic-only print.
#
# ROOT CAUSE (investigated 2026-07-27, flagged for next audit): this bias is very
# likely NOT a coding bug in the raw projection pipeline -- it's a genuine
# statistical property of run-scoring. Checked the real historical game log
# (mlb_elo_cache_2025.json, ~4000 games): per-team runs/game has skew ~1.08-1.09
# (mean 4.4-4.5 vs median 4.0), i.e. a real right-skewed distribution with a long
# high-scoring tail. The raw formula (pred_runs = (offense_avg + opp_pitcher_RA)/2
# * env, see ~line 883 above) is a point-estimate blend that implicitly treats
# each side's runs as roughly symmetric around that estimate -- for a genuinely
# skewed distribution, a symmetric point estimate structurally undershoots the
# TRUE mean, and by more the further the estimate sits below the population
# center (exactly the <7/<7.5 pattern seen here: the biggest correction is where
# the point estimate is furthest into the skewed distribution's body). This
# calibration table is a legitimate, defensible way to correct for a real
# distributional mismatch -- not a patch masking a fixable input error. A true
# "fix upstream" would mean replacing the point-estimate blend with a proper
# skewed-count model (e.g. Negative Binomial / Gamma-Poisson runs model), which
# is a real model-architecture change, not a quick coefficient tweak -- scope
# that as its own project if you want the raw output itself unbiased rather than
# corrected after the fact.
_TC_BINS = [0, 7, 7.5, 8, 8.5, 9, 9.5, 10, 100]
_TC_LABELS = ["<7", "7-7.5", "7.5-8", "8-8.5", "8.5-9", "9-9.5", "9.5-10", "10+"]
_TC_MIN_N = 100            # ignore a bucket's bias if the sample behind it is too thin to trust
_TC_MAX_CORRECTION = 1.5  # cap so a noisy/extreme bucket can't swing the projection implausibly far
_tc_bias_map = {}
# POSTSEASON RUN ENVIRONMENT (MLB-TC-PS-1, 2026-10-03 user audit -- "check whether
# calibration should apply at all in the postseason, and cap it at the measured bias").
# What the bucket table encodes is regression to the REGULAR-SEASON mean (the live raw
# projection's own graded slope is ~0.08: actual ~ 8.41 + 0.08*raw, n=109, Sep 2026 --
# low projections really do run high and high ones low, so the correction stays). But
# the postseason mean is lower: Stats API finals 2021-25, postseason minus regular-season
# runs/game = -0.50, -1.25, -1.04, -0.36, -0.88 (pooled -0.81R, n=208 PS games, ~2.7 SE).
# So in a postseason game the calibrated projection is re-centered by that gap instead of
# being pulled toward a regular-season scoring level that doesn't exist in October.
# CWS/CLE ALDS G1 (raw 6.7 -> 7.8 calibrated, final 3-0) -- see the revision below for the size.
# The requested cap at the Oct-2 "~0.5R" was NOT applied: that figure was the all-games
# MEAN bias (raw -0.38, calibrated -0.51, SE ~0.4); per-bucket the live residuals are
# larger than this table's corrections (<=7 proj: actual 3.3R above raw, SE 1.18).
# REVISED same night (user review -- "the -0.81R shift may double-count pitching: postseason
# games score less partly because the starters are better, and the raw model already sees
# those starters; use the postseason RESIDUAL"). Re-measured on Stats API 2021-25 with a
# starter-aware projection (lg R/G x opponent offense ratio x (0.6 x starter ERA ratio + 0.4 x
# staff ERA ratio), RS season stats): PS residual -0.32R (n=188) vs RS residual +0.09R
# (n=1,836) -> postseason-specific shift -0.40R (SE 0.30). Starters/offense explain ~61% of
# the raw -1.03R gap in that sample. -0.40 is the point estimate (not significant at 95%,
# ~1.3 SE) -- PROVISIONAL; CWS/CLE would have calibrated to 7.4, not 7.0.
# MLB-PEN-1 re-fit (2026-10-04): with the top-5-by-leverage postseason bullpen in the
# starter-aware projection the residual shift is -0.38R (was -0.41 with the whole staff).
_PS_RUN_ENV_DELTA = -0.38
_PS_GAME_TYPES_TC = {"F", "D", "L", "W"}
_tc_ps_matchups = set()
try:
    import requests as _tc_req, datetime as _tc_dt
    _tc_sched = _tc_req.get("https://statsapi.mlb.com/api/v1/schedule",
                            params={"sportId": 1, "date": _tc_dt.date.today().isoformat()}, timeout=15).json()
    for _tcd in _tc_sched.get("dates", []):
        for _tcg in _tcd.get("games", []):
            if _tcg.get("gameType") in _PS_GAME_TYPES_TC:
                _tc_ps_matchups.add((_tcg["teams"]["away"]["team"]["name"].lower(), _tcg["teams"]["home"]["team"]["name"].lower()))
except Exception as _tc_pe:
    print(f"  [TOTALS CALIB] postseason schedule lookup failed ({_tc_pe}) -- no postseason re-centering this run")
if "mlb_totals_calibration" in globals():
    try:
        for _, _tcr in globals()["mlb_totals_calibration"].iterrows():
            if int(_tcr["n"]) >= _TC_MIN_N:
                _tc_bias_map[str(_tcr["proj_bucket"])] = float(
                    np.clip(_tcr["mean_bias"], -_TC_MAX_CORRECTION, _TC_MAX_CORRECTION)
                )
    except Exception as _tc_e:
        print(f"  [TOTALS CALIB] could not read mlb_totals_calibration ({_tc_e}) -- no bias correction applied this run")
else:
    print("  [TOTALS CALIB] mlb_totals_calibration not in globals (run Cell 85 first) -- O/U projections uncorrected for known bucket bias")

# 2026-08-02 BUG FIX (found live, real card 2026-08-01): OU_MIN_EDGE was never
# actually assigned anywhere in this notebook -- every read of it (the qualifying-bar
# check further down, and the daily-snapshot cell) was a silent `globals().get(
# "OU_MIN_EDGE", 0.18)` fallback. Unlike MIN_MODEL_PROB (documented 2,614-game
# backtest: >=58% -> 62.1% accuracy / +18.6% ROI) or RL_MIN_EDGE (deliberately set
# to 0.08 with dated rationale), this number had zero calibration behind it -- it
# just happened to always default to 18%. Keeping the same 0.18 value (a real 2026-
# 08-01 card of 8 sub-bar O/U plays went 6-2-1, including both the best and the
# single worst miss of the day at the two highest edges -- too small and too mixed
# a sample to justify moving the number), but making it an explicit, intentional
# assignment instead of an accidental gap. Should get a real walk-forward backtest
# (like MIN_MODEL_PROB got) before this value is trusted either way.
OU_MIN_EDGE = 0.18

_ou_candidates_ranked = []  # every non-implausible candidate; speculative fallback picks top N from this
_ou_edge_ceiling_watch = []  # 25%+ edge O/U picks -- blocked from auto-play, surfaced for manual review

if "over_under" in board.columns and "pred_total_runs" in board.columns:
    _ou_board = board[[c for c in [away_col, home_col, "over_under", "pred_total_runs",
                                    "pred_away_runs", "pred_home_runs", "park_factor_used",
                                    "game_temp_f", "has_odds", "away_sp_elo_adj", "home_sp_elo_adj",
                                    "over_ml", "under_ml", "ou_book", "line_mismatch_flag",
                                    "away_pitcher_final_name", "away_pitcher_name", "away_starter",
                                    "home_pitcher_final_name", "home_pitcher_name", "home_starter"]
                        if c in board.columns]].copy()
    _ou_board = _ou_board[_ou_board["over_under"].notna() & _ou_board["pred_total_runs"].notna()]
    _ou_board = _ou_board[_ou_board["has_odds"].fillna(False)]
    _ou_added = []

    _thin_gap_games = []  # collapsed to one summary line at the bottom (2026-07-09 punch list)
    for _, _gr in _ou_board.iterrows():
        _ou_line = float(_gr["over_under"])
        _pred_t  = float(_gr["pred_total_runs"])
        _pred_t_old = _pred_t
        _tc1_key = (str(_gr.get(away_col, "")).lower().strip(), str(_gr.get(home_col, "")).lower().strip())
        _tc1_d = (globals().get("mlb_totals_decomp") or {}).get(_tc1_key)
        if _tc1_d:
            _pred_t = float(_tc1_d["total"])      # MLB-TOT-COMP-1: component total (PS shift already inside)
        if _ou_line < 4 or _pred_t <= 0: continue
        if bool(_gr.get("line_mismatch_flag", False)):
            continue  # [LINE MISMATCH] already printed at load time -- suppress this game's O/U play
        _hm = str(_gr.get(home_col,"")); _am = str(_gr.get(away_col,""))
        _matchup = f"{_am} @ {_hm}"
        _pf = _gr.get("park_factor_used", 1.0) or 1.0
        _eff_shrink = (_MARKET_SHRINK * 0.65) if float(_pf) < 0.95 else _MARKET_SHRINK
        _tc_bucket = pd.cut([_pred_t], bins=_TC_BINS, labels=_TC_LABELS)[0]
        _tc_bias = _tc_bias_map.get(str(_tc_bucket), 0.0) if not _tc1_d else 0.0   # bucket table fits the OLD projection only
        _pred_t_debiased = _pred_t - _tc_bias  # subtract bias (proj - actual) to correct toward historical actual
        _tc_is_ps = (_am.lower().strip(), _hm.lower().strip()) in _tc_ps_matchups
        if _tc_is_ps and not _tc1_d:
            _pred_t_debiased += _PS_RUN_ENV_DELTA   # MLB-TC-PS-1: re-center on the postseason scoring level
        if _tc1_d:
            print(f"  [TOTALS DECOMP] {_am} @ {_hm}: component total {_pred_t:.2f} (old formula {_pred_t_old:.2f}) -- "
                  f"no bucket calibration (fitted coefficients are its calibration)"
                  + (f", postseason {_TC1.get('ps_run_env_delta', -0.38):+.2f}R included" if _tc1_d.get("is_ps") else ""))
        elif abs(_tc_bias) >= 0.3 or _tc_is_ps:
            print(f"  [TOTALS CALIB] {_am} @ {_hm}: proj {_pred_t:.2f} -> {_pred_t_debiased:.2f} (bucket {_tc_bucket} historical bias {_tc_bias:+.2f}R"
                  + (f", postseason run-environment {_PS_RUN_ENV_DELTA:+.2f}R" if _tc_is_ps else "") + ")")
        _pred_t_cal = _pred_t_debiased * (1 - _eff_shrink) + _ou_line * _eff_shrink
        _p_ov = float(1 - _mlb_snorm.cdf((_ou_line - _pred_t_cal) / MLB_TOTAL_STD))
        # ELITE-SP OVER HAIRCUT (MLB-ELITE-OVER-1, 2026-10-04 user audit -- "This is a hard block, which
        # goes against your no-hard-caps rule ... Replace the block with an edge reduction shown on the
        # card"). The old gate blocked every Over unless BOTH starters' L3 ERA > 4.00 or PF >= 1.10 (no
        # backtest behind either number) -- in practice nearly every Over. Replaced by:
        #   (1) the projection itself now prices starters (MLB-TOT-COMP-1) incl. a -0.19R per-side
        #       elite-SP correction, and
        #   (2) a residual Over edge haircut of 3.0pp per elite starter (season ERA < 3.00, >= 40 IP),
        #       5.0pp if both: 2025-26 Overs vs the CLOSE hit 44.6% with exactly one elite starter (n=702)
        #       vs 48.1% overall -- 3.5pp, minus the ~1pp the projection correction already moves.
        #       Both-elite n=43 (48.8%) is too small to fit separately -- capped, PROVISIONAL.
        # Shown on the card as [ELITE-SP OVER HAIRCUT ...]; the Over is never blocked by it.
        _f6_over_blocked = False
        _elite_haircut, _elite_note = 0.0, ""
        if _tc1_d and _tc1_d.get("elite_n"):
            _el_names = [f"{_tc1_d['sides'][_s]['sp']['name']} (ERA {_tc1_d['sides'][_s]['sp']['era_season']:.2f})"
                         for _s in ("away", "home") if _tc1_d["sides"][_s]["c"]["elite"]]
            _elite_haircut = 0.03 if _tc1_d["elite_n"] == 1 else 0.05
            _elite_note = (f"ELITE-SP OVER HAIRCUT -{_elite_haircut * 100:.1f}pp: {', '.join(_el_names)} -- 2025-26 Overs vs the close "
                           f"hit 44.6% with an elite starter vs 48.1% overall (PROVISIONAL)")
        _p_un = 1.0 - _p_ov
        _ov_ml = float(_gr.get("over_ml", float("nan"))); _un_ml = float(_gr.get("under_ml", float("nan")))
        def _ai(ml):
            try: o = float(ml); return 100/(o+100) if o>0 else abs(o)/(abs(o)+100)
            except: return float("nan")
        _raw_ov = _ai(_ov_ml); _raw_un = _ai(_un_ml)
        if not (_raw_ov != _raw_ov) and not (_raw_un != _raw_un):
            _s = _raw_ov + _raw_un
            _mkt_ov = _raw_ov / _s; _mkt_un = _raw_un / _s
            _price_str = f"{_ov_ml:+.0f}/{_un_ml:+.0f}"
        else:
            _mkt_ov = _MLB_JUICE; _mkt_un = _MLB_JUICE; _price_str = "-110 (est)"
        # Hoisted so the speculative best-candidate tracking below can exclude
        # synthetic-priced games too (see Fix below) -- an edge computed against an
        # ESPN-estimated -110 is meaningless and shouldn't win "best available".
        _ou_is_synthetic = (
            _price_str in ("-110 (est)", "-110/-110")
            or _gr.get("ou_book") == "espn_est"
        )
        if _elite_haircut:
            _p_ov = _p_ov - _elite_haircut
        _ov_edge = round(_p_ov - _mkt_ov, 4); _un_edge = round(_p_un - _mkt_un, 4)
        _game_ou_min = OU_MIN_EDGE if "OU_MIN_EDGE" in globals() else 0.18
        _gap_chk = abs(_pred_t_cal - float(_ou_line))
        _hitter_park = float(_pf) >= 1.03 if _pf else False
        _tmp = _gr.get("game_temp_f")
        _hot_weather_under = False
        try:
            import math as _mh
            _hot_weather_under = ("under" in "under") and _tmp is not None and not _mh.isnan(float(_tmp)) and float(_tmp) >= 88
        except: pass
        # Fix: synthetic odds and thin-gap were two independent problems getting
        # tangled together -- a synthetic game with a coincidentally-thin gap would
        # print [THIN GAP] and never reach the synthetic handling at all, muddying
        # which gate actually did the suppressing. Catch synthetic odds upstream (one
        # print per game) so the per-side loop below never has to reason about it --
        # it just skips both sides immediately.
        # Note: this notebook has no _research_queue list (that's a WC-notebook
        # concept) -- the print below is the only record, matching how every other
        # O/U gate here (THIN GAP, OVER GATE) already works.
        if _ou_is_synthetic:
            print(f"  [SYNTHETIC O/U] {_matchup} {_ou_line} @ {_price_str} -- ESPN est odds only, not a real market price; not evaluated against thin-gap or edge gates")
        # Fix: thin-gap (calibrated proj within X of the line) used to only block
        # UNDER -- an OVER with the same razor-thin gap (one hit either way flips it)
        # was sailing through ungated. Skipped entirely for synthetic games (above).
        # 2026-07-27 fix: the old flat 1.0R cutoff was never tied to MLB_TOTAL_STD
        # (4.57) -- a 1.0R gap already implies roughly an 8-9pt raw edge, bigger than
        # the 8% RL_MIN_EDGE bar this same notebook uses to qualify run-line plays,
        # so this gate was killing games with real signal, not just noise. Real case
        # 2026-07-26: 10 of 14 games got thin-gap suppressed that day; scoring the
        # model's lean against actual results after the fact, 6 would have won, 3
        # would have lost, 1 was a genuine coin flip -- including big misses
        # (Braves/Orioles finished 5 runs under an 8.5 line; Astros/White Sox
        # finished 15 over a 9.5 line). Scaled to MLB_TOTAL_STD instead of a flat
        # number so it can't silently drift out of calibration again the way it did
        # relative to the std after the 1.5->4.57 fix on 2026-07-24.
        _THIN_GAP_STD_FRAC = 0.11  # ~0.5R at the current 4.57 std
        _thin_gap = (not _ou_is_synthetic) and _gap_chk < (_THIN_GAP_STD_FRAC * MLB_TOTAL_STD)
        if _thin_gap:
            _thin_gap_games.append(f"{_matchup} {_ou_line}")
        for _side, _p_side, _edg, _nv, _os in [("over",_p_ov,_ov_edge,_mkt_ov,f"{_ov_ml:+.0f}" if _ov_ml==_ov_ml else "-110"),
                                                  ("under",_p_un,_un_edge,_mkt_un,f"{_un_ml:+.0f}" if _un_ml==_un_ml else "-110")]:
            if _ou_is_synthetic:
                continue  # handled upstream -- one print/research-queue entry per game, not per side
            _is_under_blocked = (_side=="under" and (_thin_gap or (_gap_chk < 1.5 and _hitter_park) or _hot_weather_under))
            _is_over_blocked = (_side=="over" and (_f6_over_blocked or _thin_gap))
            _IMPLAUSIBLE_OU_PROB = 0.90
            _is_implausible = _p_side >= _IMPLAUSIBLE_OU_PROB
            if _is_implausible:
                print(f"  [SANITY GATE] {_side.upper()} {_ou_line} blocked -- model {_p_side:.1%} probability implausible for an MLB total (>=90%); likely a bad run-projection input for this game, not a real edge -- verify SP/offense data before trusting anything here")
            # 2026-07-27 fix: this used to track only the single highest-edge
            # candidate across all games -- real case 2026-07-26, Blue Jays/Red Sox
            # UNDER had a genuine ~15.5% non-thin-gap edge and hit, but got zero
            # visibility because Royals/Tigers edged it out by under a point for the
            # one fallback slot. Now keeps every non-implausible candidate (synthetic
            # already skipped via the `continue` above); the display section below
            # picks the top few instead of a single winner-take-all fallback.
            # MLB-PS-SPROLE-1: postseason SP role UNRATED blocks both sides
            _ps_unr_ou = (globals().get("mlb_ps_role_unrated", {}) or {}).get(
                (str(_am).lower().strip(), str(_hm).lower().strip()))
            if _ps_unr_ou:
                _is_over_blocked = _is_over_blocked or _side == "over"
                _is_under_blocked = _is_under_blocked or _side == "under"
            # MLB-TBD-RERUN-1: no probable posted -> UNRATED both sides
            _tbd_ou = (globals().get("mlb_sp_tbd_unrated", {}) or {}).get(
                (str(_am).lower().strip(), str(_hm).lower().strip()))
            if _tbd_ou:
                _is_over_blocked = _is_over_blocked or _side == "over"
                _is_under_blocked = _is_under_blocked or _side == "under"
            # MLB-TOT-DRIVER-1 (2026-10-04 user audit -- "Soccer already requires a directional driver,
            # so MLB should match"): a totals lean needs at least one game-specific component (a
            # starter, a bullpen, an offense, park, elite adj) moving the projection >= 0.25R in the
            # bet's direction vs a league-average game. The league base, postseason shift and market
            # shrink are not drivers. No driver -> gate (below-bar), never an official play.
            _drv_ok, _drv_top = True, None
            if _tc1_d:
                _drv_parts = [(_l, _v) for _l, _v, _w in _tc1_d["parts"] if _l not in ("league base", "postseason run env", "home field")]
                _drv_dir = [(_l, _v) for _l, _v in _drv_parts if (_v >= 0.25 if _side == "over" else _v <= -0.25)]
                _drv_ok = bool(_drv_dir)
                _drv_top = max(_drv_dir, key=lambda x: abs(x[1])) if _drv_dir else None
                if not _drv_ok:
                    _is_over_blocked = _is_over_blocked or _side == "over"
                    _is_under_blocked = _is_under_blocked or _side == "under"
            # MLB-EDGE-RULE-DETAIL-1: name the gate(s) that fired, in the selector's own terms
            _ou_gate_names = []
            if _side == "under":
                if _thin_gap: _ou_gate_names.append(f"thin gap (calibrated proj {_pred_t_cal:.1f} only {_gap_chk:.2f}R from the line, needs >= {_THIN_GAP_STD_FRAC * MLB_TOTAL_STD:.2f}R)")
                if _gap_chk < 1.5 and _hitter_park: _ou_gate_names.append(f"hitter-park Under block (park factor {float(_pf):.2f} >= 1.03 with gap {_gap_chk:.2f}R < 1.5R)")
                if _hot_weather_under: _ou_gate_names.append(f"hot-weather Under block ({float(_tmp):.0f}F >= 88F)")
            else:
                if _thin_gap: _ou_gate_names.append(f"thin gap (calibrated proj {_pred_t_cal:.1f} only {_gap_chk:.2f}R from the line, needs >= {_THIN_GAP_STD_FRAC * MLB_TOTAL_STD:.2f}R)")
            if _ps_unr_ou: _ou_gate_names.append("Gate 7d postseason SP role UNRATED -- " + _ps_unr_ou)
            if _tbd_ou: _ou_gate_names.append("Gate 7f " + _tbd_ou)
            if not _drv_ok:
                _big = max(_drv_parts, key=lambda x: abs(x[1])) if _drv_parts else None
                _ou_gate_names.append(f"no directional driver (no starter/bullpen/offense/park term >= 0.25R toward the {_side.title()}"
                                      + (f"; largest is {_big[0]} {_big[1]:+.2f}R" if _big else "") + ")")
            _dc_ou = (globals().get("mlb_direction_conflict_games", {}) or {}).get(
                (str(_am).lower().strip(), str(_hm).lower().strip()))
            if _dc_ou:
                _is_over_blocked = _is_over_blocked or _side == "over"
                _is_under_blocked = _is_under_blocked or _side == "under"
                _ou_gate_names.append("Gate 7e " + _dc_ou)
            if not _is_implausible:
                _lbl_b = f"OVER {_ou_line}" if _side=="over" else f"UNDER {_ou_line}"
                _ou_candidates_ranked.append({away_col:_am, home_col:_hm, "matchup":_matchup, "bet_team":_lbl_b,
                    "bet_side":_side, "p_bet_win_model":round(_p_side,4), "market_implied_prob":round(_nv,4),
                    "ou_price_str":_price_str, "ou_line":_ou_line, "pred_total_runs":round(_pred_t,2),
                    "pred_total_cal":round(_pred_t_cal,2), "edge":round(_edg,4),
                    "park_factor_used":_pf, "game_temp_f":_tmp, "bet_type":_side.upper(),
                    "over_under":_ou_line, "has_odds":True, "passes_edge_rule":1,
                    "final_score":round(_edg,4), "speculative":True,
                    "_over_gate_fired": _is_over_blocked if _side=="over" else False,
                    # 2026-08-29 (user audit M3): the UNDER-side data-quality gates
                    # (thin gap / hitter park / hot weather) were never recorded on the
                    # candidate row -- only the OVER gate was -- so the sub-official
                    # tier below could promote an UNDER that the real gate had already
                    # blocked. Recorded here so the LEAN tier can respect both sides.
                    "_under_gate_fired": _is_under_blocked if _side=="under" else False,
                    "_gate_fired": (_is_over_blocked if _side=="over" else _is_under_blocked),
                    "_ou_gate_names": list(_ou_gate_names),
                    "pred_total_old": round(_pred_t_old, 2), "decomp_key": _tc1_key if _tc1_d else None,
                    "_ou_adj_notes": ([_elite_note] if (_elite_note and _side == "over") else []),
                    "_ou_driver_top": _drv_top, "ou_book": _gr.get("ou_book"),
                    "_ou_min_edge": _game_ou_min,
                    "_ou_edge_logit": _mlb_logit(_p_side) - _mlb_logit(_nv),
                    "_ou_req_logit": _mlb_logit(0.5 + _game_ou_min) - _mlb_logit(0.5)})
            _OU_EDGE_CEILING = 0.25
            # Logit-space symmetric edge -- same 2026-07-12 port as MLB ML/RL (see
            # there for full rationale): a raw percentage-point edge means more the
            # further the market probability sits from a coinflip (.60->.70 is a
            # bigger real edge than .50->.60 despite an identical 10pp gap), so a
            # candidate below the flat _game_ou_min bar can still qualify if it
            # clears the equivalent logit-space bar.
            _ou_required_edge_logit = _mlb_logit(0.5 + _game_ou_min) - _mlb_logit(0.5)
            _ou_edge_logit = _mlb_logit(_p_side) - _mlb_logit(_nv)
            _ou_qualifies_logit = _ou_edge_logit >= _ou_required_edge_logit
            if _edg < _game_ou_min and _ou_qualifies_logit:
                print(f"  [LOGIT-SPACE EDGE] {_side.upper()} {_ou_line}: {_edg:+.1%} raw edge qualifies on symmetric logit-space edge ({_ou_edge_logit:+.2f} vs {_ou_required_edge_logit:+.2f} required, {_nv:.0%} implied)")
            if (_edg >= _game_ou_min or _ou_qualifies_logit) and not _is_under_blocked and not _is_over_blocked and not _is_implausible and f"{_side.upper()} {_ou_line}" not in _ou_added:
                _lbl = f"OVER {_ou_line}" if _side=="over" else f"UNDER {_ou_line}"
                # Synthetic odds are caught upstream (top of this loop) now -- this
                # branch only ever sees real-priced games.
                _su = 1.0 if (_edg >= 0.08 and _p_side >= 0.35) else 0.5
                if _edg >= _OU_EDGE_CEILING:
                    # Edge ceiling -- same reasoning as ML/RL Gate 5: real O/U lines
                    # here come from the Odds API (sharp books), so a 25%+ gap is at
                    # least as suspicious as on ML/RL, more likely a stale/moved line
                    # than genuine value. Blocked from auto-play but NOT dropped --
                    # surfaced in a dedicated watch section so a real line-move
                    # (shrinking the true gap) can still be caught manually instead
                    # of silently vanishing the way the ML/RL ceiling does.
                    print(f"  [EDGE CEILING] {_lbl} blocked -- edge {_edg:+.1%} >= {_OU_EDGE_CEILING:.0%}, flag for calibration review -- see O/U EDGE CEILING WATCH")
                    _ou_edge_ceiling_watch.append({away_col:_am, home_col:_hm, "matchup":_matchup, "bet_team":_lbl,
                        "bet_side":_side, "p_bet_win_model":round(_p_side,4), "market_implied_prob":round(_nv,4),
                        "ou_price_str":_price_str, "ou_line":_ou_line, "pred_total_runs":round(_pred_t,2),
                        "pred_total_cal":round(_pred_t_cal,2), "edge":round(_edg,4)})
                    _ou_added.append(f"{_side.upper()} {_ou_line}")
                    continue
                print(f"  !! GOALS {_lbl} [PLAY] | {_edg:+.1%} edge | {_su}u @ {_os}")
                _ou_plays_out.append({away_col:_am, home_col:_hm, "matchup":_matchup, "bet_team":_lbl,
                    "bet_side":_side, "p_bet_win_model":round(_p_side,4), "market_implied_prob":round(_nv,4),
                    "ou_price_str":_price_str, "ou_line":_ou_line, "pred_total_runs":round(_pred_t,2),
                    "pred_total_cal":round(_pred_t_cal,2), "edge":round(_edg,4),
                    "size_u":_su, "size_$":int(_su*30), "odds_str":_os,
                    "park_factor_used":_pf, "game_temp_f":_tmp, "bet_type":_side.upper(),
                    "over_under":_ou_line, "has_odds":True, "passes_edge_rule":1, "final_score":round(_edg,4)})
                _ou_added.append(f"{_side.upper()} {_ou_line}")
    if _thin_gap_games:
        print(f"  [THIN GAP] {len(_thin_gap_games)} game(s) suppressed both sides (proj within {_THIN_GAP_STD_FRAC*MLB_TOTAL_STD:.2f}R of line): " + "; ".join(_thin_gap_games))
    # ── TWO-TIER TOTALS GATE (2026-08-29, user audit M3) ─────────────────────
    # What was here before was a FALLBACK, not a tier: a 10%-floor pick was only
    # ever surfaced on days when NOTHING cleared OU_MIN_EDGE, and it inherited
    # every candidate that had already been blocked by the under-side gates
    # (thin gap / hitter park / hot weather) because only the OVER gate was
    # recorded on the candidate row. That is exactly the audit's complaint --
    # totals either go silent or emit an ungated below-bar edge.
    #
    # OU_MIN_EDGE stays at 0.18 (see its own 2026-08-02 note: the value is
    # deliberate and explicitly wants a walk-forward backtest before it moves --
    # this does NOT move it). Instead the sub-official band becomes a real,
    # always-evaluated LEAN tier that (a) runs whether or not an OFFICIAL play
    # exists, (b) respects BOTH sides' data-quality gates, and (c) keeps
    # speculative=True so every downstream official/ledger check (Cell 6e66db20's
    # `official` flag, the ledger writer) continues to exclude it exactly as it
    # already does. Strict superset of the old behaviour: nothing that used to
    # qualify stops qualifying.
    _OU_SPEC_MIN_EDGE = 0.10   # LEAN-tier floor -- below OU_MIN_EDGE, above noise
    _OU_SPEC_MAX_N = 3         # cap how many LEAN plays can show on one card
    # These two are assigned inside the per-game/per-side loops above, so they
    # are undefined on a slate where every game was skipped -- read defensively.
    _ou_min_disp = globals().get("_game_ou_min", OU_MIN_EDGE)
    _ou_ceiling = globals().get("_OU_EDGE_CEILING", 0.25)
    _ou_official_ct = len(_ou_plays_out)
    _ou_taken = {(p.get("matchup"), p.get("bet_team")) for p in _ou_plays_out}
    _ou_lean_eligible = [
        c for c in _ou_candidates_ranked
        if c["edge"] >= _OU_SPEC_MIN_EDGE
        and not c.get("_gate_fired")
        and (c.get("matchup"), c.get("bet_team")) not in _ou_taken
        and c["edge"] < _ou_ceiling
    ]
    _ou_lean_eligible.sort(key=lambda c: c["edge"], reverse=True)
    _ou_lean_picked = _ou_lean_eligible[:_OU_SPEC_MAX_N]
    for _lp in _ou_lean_picked:
        _lp["ou_tier"] = "LEAN"
    for _op_t in _ou_plays_out:
        _op_t["ou_tier"] = "OFFICIAL"
    _ou_gate_blocked_ct = sum(1 for c in _ou_candidates_ranked
                              if c["edge"] >= _OU_SPEC_MIN_EDGE and c.get("_gate_fired"))
    print(f"  [O/U TIERS] OFFICIAL (>= {_ou_min_disp:.0%}): {_ou_official_ct}  |  "
          f"LEAN ({_OU_SPEC_MIN_EDGE:.0%}-{_ou_min_disp:.0%}): {len(_ou_lean_picked)}  |  "
          f"gate-blocked above the LEAN floor: {_ou_gate_blocked_ct}")
    if _ou_lean_picked:
        print(f"  [O/U LEAN TIER -- {len(_ou_lean_picked)} play(s) between {_OU_SPEC_MIN_EDGE:.0%} and "
              f"{_ou_min_disp:.0%}; gate-cleared but sub-official, never ledger-logged]")
        _ou_plays_out.extend(_ou_lean_picked)
    elif not _ou_candidates_ranked:
        print("  [O/U] No real-priced O/U candidate today -- every line was synthetic, thin-gap, or gate-blocked")
    elif _ou_gate_blocked_ct:
        print(f"  [O/U] {_ou_gate_blocked_ct} candidate(s) cleared the {_OU_SPEC_MIN_EDGE:.0%} LEAN floor on edge "
              f"but were blocked by a data-quality gate (thin-gap / hitter-park / hot-weather / elite-SP)")
    elif not _ou_official_ct:
        print(f"  [O/U] No candidate cleared the {_OU_SPEC_MIN_EDGE:.0%} LEAN floor today")

# ── WATCH/VALUE SECTION BUILD ────────────────────────────────────────────────
_watch_plays = []
for _, _wr in top_market.iterrows():
    _w_edge = float(_wr.get("edge",0) or 0)
    _w_prob = float(_wr.get("p_bet_win_model",0) or 0)
    _w_unit = _unit_label(_w_edge, _wr.get("p_bet_win_model"))
    if "WATCH" in _w_unit or _w_unit.startswith("0.5u"):
        _mkt = f"{_wr['market_implied_prob']:.1%}" if pd.notna(_wr.get("market_implied_prob")) else "--"
        _watch_plays.append({**_wr.to_dict(), "_w_unit":_w_unit, "_mkt":_mkt})

# ── DISPLAY: RUN LINE ────────────────────────────────────────────────────────
print()
print("=" * 80)
print("EKO BETS -- RUN LINE (-1.5) PLAYS")
print("=" * 80)
if _rl_candidates:
    for _rc in _rl_candidates:
        print()
        print(f"  {_rc['matchup']}")
        print(f"  Bet       : {_rc['bet_team']}")
        print(f"  Model RL  : {_rc['model_prob']:.1%} (Poisson) | Market: {_rc['mkt_prob']:.1%} | Edge: {_rc['edge']:.1%}")
        # Fix: NaN weather (dome/retractable roof games) used to leave this blank --
        # print an explicit tag instead of silently omitting it.
        _tmp_s = f" | {_rc['game_temp_f']:.0f}F" if _rc.get("game_temp_f") and str(_rc["game_temp_f"])!="nan" else " | [DOME/WEATHER N/A]"
        print(f"  Proj      : {_rc[away_col]} @ {_rc[home_col]} -- {_rc['pred_diff']:.1f}R margin{_tmp_s}")
        _ju = _rc.get("size_u",0.5); _jd = int(_ju*30)
        print(f"  Size      : {_ju}u (${_jd})")
        print("-" * 60)
else:
    print("  No qualifying run line plays today.")
    for _team, _ev in _rl_eval.items():
        if _ev.get("status") not in ("pending","no odds",None):
            print(f"  [RL BLOCKED] {_team}: {_ev.get('status','?')} -- {_ev.get('reason','')}")

# ── RL sanity-fail cascade: flag any ML play whose game also had an RL price
# sanity failure. Covering by 2+ implying a HIGHER win probability than just
# winning outright is not a run-line-specific quirk -- it means that team's
# market pricing has a real, unresolved inconsistency somewhere, which is a
# reason to double-check the ML price on the same game too, not just drop the
# RL market and move on. This runs after the RL section (later in the cell
# than the ML play loop above) since the sanity check itself only resolves
# here -- can't be shown inline with the original ML print.
_rl_sanity_fail_teams = [t for t, ev in _rl_eval.items() if ev.get("status") == "odds sanity fail"]
if _rl_sanity_fail_teams:
    print()
    print("=" * 80)
    print("!! ML PLAYS NEEDING RE-REVIEW -- RL price sanity failure detected same game")
    print("=" * 80)
    for _team in _rl_sanity_fail_teams:
        _matched_ml = next((bt for bt in _display_units
                             if bt.lower() == _team or _team in bt.lower() or bt.lower() in _team), None)
        if _matched_ml:
            print(f"  {_matched_ml} ML ({_display_units[_matched_ml]}): {_team} run-line price failed a "
                  f"sanity check ({_rl_eval[_team].get('reason','')}) -- this team's market pricing has an "
                  f"unresolved inconsistency somewhere; re-verify the ML price/line before trusting this edge")

# ── DISPLAY: VALUE WATCH ─────────────────────────────────────────────────────
print()
print("=" * 80)
print("EKO BETS -- VALUE WATCH  (edge below main threshold -- informational)")
print("=" * 80)
_watch_printed = False
for _, _wr in top_market.sort_values("edge",ascending=False).iterrows():
    _w_edge = float(_wr.get("edge",0) or 0)
    _w_prob = float(_wr.get("p_bet_win_model",0) or 0)
    _w_unit = _unit_label(_w_edge, _wr.get("p_bet_win_model"))
    # BUG FIX (MLB-EDGE-BLEND-1, 2026-09-25 user audit): rescaled with the
    # other ML edge thresholds in this cell (edge_blended = w*edge_raw).
    if _w_edge < 0.12 * globals().get("_MODEL_MARKET_BLEND_W", 1.0): continue  # within 3pp of the base threshold, raw-edge terms
    if not (_w_unit.startswith("0.5u") or "WATCH" in _w_unit): continue
    _w_confirming = _mlb_elo_avail(_wr.get("bet_team", ""))
    if not _w_confirming: continue  # no confirming signal -- drop silently
    _watch_printed = True
    _mkt = f"{_wr['market_implied_prob']:.1%}" if pd.notna(_wr.get("market_implied_prob")) else "--"
    print()
    print(f"  {_wr.get(away_col,'')} @ {_wr.get(home_col,'')}")
    print(f"  WATCH     : {_wr['bet_team']} ML ({str(_wr.get('bet_side','')).upper()})")
    # Fix: this used to hardcode "[below 15% threshold]" even when the edge cleared
    # 15% and the REAL reason it landed here was ELO unavailability capping it --
    # same root cause as the grade section's [ELO UNAVAILABLE], just reported
    # inconsistently. Use the actual reason.
    if not _mlb_elo_avail(_wr.get("bet_team","")):
        _w_reason = "[ELO UNAVAILABLE -- grade capped at B+]"
    elif _w_edge < 0.15:
        _w_reason = "[below 15% threshold]"
    else:
        _w_reason = "[watch -- see Suggested for reason]"
    print(f"  Model     : {_w_prob:.1%} | Market: {_mkt} | Edge: {_w_edge:.1%} {_w_reason}")
    print(f"  Suggested : {_w_unit} -- sub-threshold watch (verify SP + injury before acting)")
    _sp_away = _wr.get("away_sp_elo_adj"); _sp_home = _wr.get("home_sp_elo_adj")
    if _sp_away and _sp_home:
        # 2026-08-01: this had its own threshold set (1650/1580/1500/1430),
        # different from the one _pitcher_note() uses for the actual official
        # card (1560/1520/1480/1440, ~60 lines earlier in this cell) -- the
        # exact same numeric rating could print as a different tier label
        # depending on which section of the output showed it. Synced to match
        # the official card's thresholds, the one this whole model actually
        # bets against.
        def _sp_tier(e):
            e = float(e) if e else 1500
            if e>=1560: return f"ELITE(>=1560)/{e:.0f}"
            elif e>=1520: return f"STRONG(1520-1559)/{e:.0f}"
            elif e>=1480: return f"AVG(1480-1519)/{e:.0f}"
            elif e>=1440: return f"BELOW(1440-1479)/{e:.0f}"
            else: return f"WEAK(<1440)/{e:.0f}"
        _aw_nm = _wr.get("away_pitcher_name","?"); _hw_nm = _wr.get("home_pitcher_name","?")
        _sptd_fn = globals().get("_sp_tier_downgrade_gap")
        _aw_sptd = _sptd_fn(_aw_nm) if _sptd_fn else None
        _hw_sptd = _sptd_fn(_hw_nm) if _sptd_fn else None
        _aw_sptd_sfx = f" [FIP/SIERA gap {_aw_sptd:.2f}R]" if _aw_sptd is not None else ""
        _hw_sptd_sfx = f" [FIP/SIERA gap {_hw_sptd:.2f}R]" if _hw_sptd is not None else ""
        print(f"    SP Matchup : {_aw_nm} [{_sp_tier(_sp_away)}{_aw_sptd_sfx}] vs {_hw_nm} [{_sp_tier(_sp_home)}{_hw_sptd_sfx}]")
    print("-" * 60)
if not _watch_printed:
    print("  No watch plays today.")

# ── DISPLAY: O/U RUN TOTAL ───────────────────────────────────────────────────
if _ou_plays_out:
    print()
    print(f"O/U Run Total plays: {len(_ou_plays_out)}  (total exposure: {sum(p.get('size_u',0.5) for p in _ou_plays_out):.1f}u)")
    print()
    print("=" * 80)
    print("EKO BETS -- O/U RUN TOTAL PLAYS")
    print("=" * 80)
    for _op in _ou_plays_out:
        print()
        print(f"  {_op['matchup']}")
        print(f"  Bet       : {_op['bet_team']}")
        _ou_mkt = f"{_op['market_implied_prob']:.1%}" if _op.get("market_implied_prob") else "--"
        _ou_os = _op.get("ou_price_str","--")
        print(f"  Model     : {_op['p_bet_win_model']:.1%} | Market: {_ou_mkt} ({_ou_os}) | Edge: {_op['edge']:+.1%}")
        print(f"  Proj Total: {_op['pred_total_runs']:.1f} raw -> {_op['pred_total_cal']:.1f} calibrated  (line {_op['ou_line']})")
        _gap = abs(_op['pred_total_cal'] - float(_op['ou_line']))
        if _gap < 2.0: print(f"  !! Line close to projection (gap {_gap:.1f}R) -- verify line hasn't moved")
        _size_lbl = _op.get("size_label","0.5u speculative") if _op.get("speculative") else f"{_op.get('size_u',0.5)}u"
        if _op.get("speculative"):
            _size_lbl = f"LEAN tier -- below the {OU_MIN_EDGE:.0%} official O/U bar, not a ledger play"
        print(f"  Tier      : {_op.get('ou_tier','OFFICIAL')}")
        print(f"  Size      : {_size_lbl}")
        _pf2 = _op.get("park_factor_used",1.0)
        if _pf2 and abs(float(_pf2)-1.0) >= 0.05: print(f"  Why this play:\n    -> Park: {float(_pf2):.2f}x ({'pitcher' if float(_pf2)<1 else 'hitter'}-friendly)")
        _tmp2 = _op.get("game_temp_f")
        # Fix: NaN weather (dome/retractable roof games) used to leave this blank.
        if _tmp2 and str(_tmp2)!="nan": print(f"    -> Weather: {float(_tmp2):.0f}F (already in projection)")
        else: print("    -> Weather: [DOME/WEATHER N/A]")
        # 2026-08-29 (user audit M1): the bullpen-fatigue term now inside this
        # projection was invisible on the totals output -- surfaced here so an
        # Over/Under driven by a gassed pen reads as such.
        _bpw_seen = globals().get("_mlb_bp_workload_applied", {}) or {}
        for _bpw_t in (_op.get(away_col), _op.get(home_col)):
            _bpw_k = _pl_norm(str(_bpw_t or ""))
            _bpw_d = _bpw_seen.get(_bpw_k)
            if _bpw_d:
                print(f"    -> Bullpen fatigue: {_bpw_t} pen faced {_bpw_d.get('bf',0)} batters in "
                      f"{_bpw_d.get('ip',0)} IP over the last 3 days ({_bpw_d.get('b2b_arms',0)} arm(s) "
                      f"back-to-back, league median {float(_bpw_d.get('bf_median',0)):.0f} BF) -- "
                      f"+{float(_bpw_d.get('run_adj',0)):.2f}R already in this total")
        print("-" * 60)
        # Official-play ledger logging removed from this section (2026-08-11):
        # same reason as the ML logger above -- this O/U display loop shows every
        # candidate including sub-threshold "speculative" plays, not just the real
        # official pick, and draws from an older selection pass that can disagree
        # with the real card (Cell 6e66db20's _snap_ranked). That cell is the only
        # place that should write to mlb_official_plays_ledger.csv.

# ── DISPLAY: O/U EDGE CEILING WATCH ─────────────────────────────────────────
if _ou_edge_ceiling_watch:
    print()
    print("=" * 80)
    print("EKO BETS -- O/U EDGE CEILING WATCH  (edge >= 25% -- verify line before acting)")
    print("=" * 80)
    for _ocw in _ou_edge_ceiling_watch:
        print()
        print(f"  {_ocw['matchup']}")
        print(f"  Flagged   : {_ocw['bet_team']}  (NOT auto-played -- calibration review)")
        _ocw_mkt = f"{_ocw['market_implied_prob']:.1%}" if _ocw.get("market_implied_prob") else "--"
        _ocw_os = _ocw.get("ou_price_str","--")
        print(f"  Model     : {_ocw['p_bet_win_model']:.1%} | Market: {_ocw_mkt} ({_ocw_os}) | Edge: {_ocw['edge']:+.1%}")
        print(f"  Proj Total: {_ocw['pred_total_runs']:.1f} raw -> {_ocw['pred_total_cal']:.1f} calibrated  (line {_ocw['ou_line']})")
        print(f"  -> Verify the line hasn't moved since this ran before trusting the {_ocw['edge']:+.1%} edge -- if the real current gap is smaller, this may still be worth taking manually.")
        print("-" * 60)

# ── SUMMARY ───────────────────────────────────────────────────────────────────
print()
_market_played_ct = globals().get("_market_played_ct", 0)
_model_ct = len(qualified_model) if "qualified_model" in globals() else 0
_rl_ct = len(_rl_candidates) if "_rl_candidates" in globals() else 0
_ou_ct = len(_ou_plays_out) if "_ou_plays_out" in globals() else 0
_grand_total_ct = _market_played_ct + _model_ct + _rl_ct + _ou_ct
print(f"Total plays: {_grand_total_ct}  ({_market_played_ct} market, {_model_ct} model-only, {_rl_ct} run line, {_ou_ct} O/U)")
print(f"  [F5 SYNTHETIC] No real F5 market -- synthetic SP-ELO estimate only; verify F5 direction before OVER plays")

# ── AUTO SAVE ─────────────────────────────────────────────────────────────────
_save_names = ["mlb_card_scored", "best_mlb_board", "best_mlb_play"]
try:
    globals()["mlb_card_scored"] = top_market.copy()
    globals()["best_mlb_board"] = board.copy()
    if _rl_candidates:
        globals()["best_mlb_play"] = pd.DataFrame(_rl_candidates)
    elif _ou_plays_out:
        globals()["best_mlb_play"] = pd.DataFrame(_ou_plays_out)
    print(f"Saved: {', '.join(_save_names)}")
except Exception as _sv_e:
    print(f"Save partial: {_sv_e}")


FINAL MLB CARD SELECTOR
  SP recent form fetcher ready


Series context loaded for 1 games
Board source : scored_mlb_board  (1 rows)
Starter RA merged from projection_board: 1/1 games have SP data
SP names overridden from live-confirmed starter feed (Cell 41/42), matched by game_id (doubleheader-safe): 1/1 games  [as of 09:14AM]
11-feature model probs merged: 1/1 games upgraded
ESPN odds merged: 1/1 games matched  |  O/U available: 1
  [ODDS API: connected]
  [O/U REAL] Loaded real totals for 2/2 games from Odds API
  [O/U REAL] 1/1 games now using real book O/U (vs ESPN estimate)

Predicted score inputs:
  away offense  : away_runs_for_avg_10
  home offense  : home_runs_for_avg_10
  away SP RA    : away_starter_runs_allowed_10  (fallback: away_runs_against_avg_10)
  home SP RA    : home_starter_runs_allowed_10  (fallback: home_runs_against_avg_10)
  Can compute   : True


  [MLB-PEN-1] Chicago White Sox: postseason bullpen RA 1.46 -> 3.42 (top-5 by leverage, last 60d: Grant Taylor 11lev/25IP, Bryan Hudson 6lev/18IP, Jordan Hicks 3lev/14IP, Hagen Smith 2lev/26IP, Noah Schultz 1lev/9IP; whole pen 3.34)


  [MLB-PEN-1] Cleveland Guardians: postseason bullpen RA 1.24 -> 1.74 (top-5 by leverage, last 60d: Hunter Gaddis 14lev/21IP, Cade Smith 12lev/21IP, Erik Sabrowski 9lev/16IP, Tim Herrin 2lev/17IP, Matt Festa 0lev/18IP; whole pen 1.74)
Bullpen RA loaded for 30 teams (30 live 2026, 0 stale-fallback)
Venue data loaded for 1 game(s) today


  [POSTSEASON SP ROLE] Chicago White Sox 2-2 Cleveland Guardians (G5 of 5, ELIMINATION)
    Chicago White Sox: Sean Burke starts an elimination game -> projected SP innings discounted (50%/50% SP/bullpen vs 65/35)
    Cleveland Guardians: Gavin Williams starts an elimination game -> projected SP innings discounted (50%/50% SP/bullpen vs 65/35)
    !! PIGGYBACK RISK -- Chicago White Sox can piggyback Anthony Kay (31 GS, short rest), Davis Martin (28 GS, short rest), Erick Fedde (15 GS, short rest); Cleveland Guardians can piggyback Tanner Bibee (33 GS, short rest), Parker Messick (32 GS, short rest), Foster Griffin (31 GS, short rest) -- who covers the middle/late innings is unknowable pregame, SP role UNRATED
  [SP SPLIT -> RUN PROJ] Gavin Williams: home ERA 2.57 vs season ERA 3.41 (33 home starts) -- runs-allowed scaled 0.75x for tonight's home start


...

  Bullpen data: 0 teams yesterday, 2 teams 2 days ago
  [FORMER TEAM] pitching history resolved for 2/2 probable starters
  SP vs-opponent H2H fetcher ready


  L10 records loaded for 30 teams
  Season records loaded for 30 teams (incl home/road splits)
SP pre-playoff tuneup/extended-rest shave: adjusted 0 play(s), newly blocked 1 (pick or opp SP on 7+ days rest for a team that has already clinched)
  MLB team IDs loaded for 59 entries


  [SP CROSS-CHECK] ESPN probable pitchers loaded for 2 teams (dates=20261010)

--------------------------------------------------
  MARKET PLAYS  (ESPN odds confirmed)
--------------------------------------------------

#1  Chicago White Sox @ Cleveland Guardians
  SP Matchup : Sean Burke [STRONG(1520-1559)/1555, ERA 3.73/WHIP 1.26 [L3: 6.39] [L10: 4.04]] vs Gavin Williams [ELITE(>=1560)/1728, ERA 3.76/WHIP 1.11]  [confirmed via Cell 41/42 live feed -- external sources may lag]
    Sean Burke: ERA 3.73 (starts only) | WHIP 1.26 | FIP 3.62 | xFIP 4.00 | SIERA 3.79 | HH% 38.8 | Barrel% 8.3 | IP 142.3
    Gavin Williams: ERA 3.76 (starts only) | WHIP 1.11 | FIP 3.47 | xFIP 2.85 | SIERA 2.95 | HH% 46.6 | Barrel% 11.4 | IP 184.3
  Proj Score : Chicago White Sox 3.1  --  Cleveland Guardians 4.0  (O/U 7.2) | 59F wind 4mph precip 0% [old run formula -- feeds RL/score display only; totals use the component model: 7.3, see TOTALS DECOMP]


  [SP LAST-START OUTLIER -- PICK] Gavin Williams: last start ERA 11.61 spiked vs season 3.76 -- likely regresses toward season mean, informational only
  [SP LAST-START OUTLIER -- OPP] Sean Burke: last start ERA 1.76 was unusually suppressed vs season 3.73 -- likely regresses toward season mean, informational only
  Grade     : B+  (1/4: edge=N SP=N BP=N juice=Y)
  Bet       : Cleveland Guardians ML (HOME)
  Model     : 63.0%  |  Market: -142 (56.1%)  |  Edge: 7.1%
  Size      : 0.5u  [WATCH ONLY -- below 1.0u minimum]
------------------------------------------------------------

[SHADOW WEIGHT] MLB totals: w=1.00 -> shrink toward the line 0%
  [TOTALS DECOMP] Chicago White Sox @ Cleveland Guardians: component total 7.27 (old formula 7.20) -- no bucket calibration (fitted coefficients are its calibration), postseason -0.38R included
  [THIN GAP] 1 game(s) suppressed both sides (proj within 0.50R of line): Chicago White Sox @ Cleveland Guardians 7.0
  [O/U TIERS] OFFICIAL (>= 18%): 0  |

Saved: mlb_card_scored, best_mlb_board, best_mlb_play


In [89]:
# %%
# =========================================================
# CELL -- GRADE OFFICIAL PLAYS LEDGER (ML + O/U + RL)
# Reads tracking/mlb_official_plays_ledger.csv, written by Cell
# 6e66db20's _snap_ranked loop -- the one true "official play"
# source (see the 2026-08-07/2026-08-11 fix notes near
# _log_official_play; the old Cell bd676982 loggers that used to
# also write here, from a disagreeing older selection pass, were
# removed 2026-08-11). Fetches actual final scores from MLB
# StatsAPI and grades each row as WIN/LOSS/PENDING by parsing its
# market string: "ML", "O/U {line}", or "{team} -1.5".
# Writes tracking/mlb_official_plays_graded.csv -- full rebuild
# each run (ledger is small, refetching is cheap), so this is
# always a clean re-grade rather than an incremental append.
# Built 2026-08-11, not yet live-verified against a real ledger
# with finished games -- confirm the market-string parsing holds
# once real graded rows exist.
# 2026-08-15 update (MLB ML calibration audit): now live-verified against
# 59 real graded rows -- found the ungradable-row count wasn't the only
# gap, 37 of the 59 were exact duplicates of 11 already-graded games (a
# pre-08-11 dual-writer artifact never purged from the ledger). Added a
# dedup guard here plus a loud ML-record banner so a future duplicate
# regression is self-healing and the real record can't hide in the
# grading header text again.
# =========================================================

import pandas as pd
import numpy as np
import requests
import re
from pathlib import Path

print("=" * 70)
print("GRADE OFFICIAL PLAYS LEDGER")
print("=" * 70)

_GOP_LEDGER_PATH = Path(r"c:\Users\<user>\OneDrive\Documents\KAI Labs\sports\MLB\tracking\mlb_official_plays_ledger.csv")
_GOP_GRADED_PATH = Path(r"c:\Users\<user>\OneDrive\Documents\KAI Labs\sports\MLB\tracking\mlb_official_plays_graded.csv")

if not _GOP_LEDGER_PATH.exists():
    print("No ledger file found -- nothing to grade.")
else:
    _gop_df = pd.read_csv(_GOP_LEDGER_PATH)
    print(f"Ledger rows: {len(_gop_df)}")

    # Only rows with a usable game_id can be graded via the live-feed API.
    _gop_df["game_id"] = pd.to_numeric(_gop_df["game_id"], errors="coerce")
    _gop_gradable = _gop_df[_gop_df["game_id"].notna()].copy()
    _gop_ungradable = len(_gop_df) - len(_gop_gradable)
    if _gop_ungradable:
        print(f"  {_gop_ungradable} row(s) have no game_id (pre-fix ledger rows) -- cannot grade, left out")

    # DEDUP GUARD (2026-08-15, MLB ML calibration audit): a pre-2026-08-11
    # ledger-writer duplication bug (two independent loggers appending the
    # same real pick under slightly different "bet" text, e.g. "Detroit
    # Tigers" vs "Detroit Tigers ML") left 37 duplicate rows across 11 real
    # games in the ledger -- the "24-34 (41.4%)" record flagged 2026-08-14
    # as the #1 cross-sport priority was actually 22 distinct games counted
    # as 58 rows; the true record was 10-11 (47.6%). The append-side bug was
    # already fixed 2026-08-11 (removed the redundant writer) and existing
    # debris was purged from the ledger the same session this comment was
    # added, but this defends the READ side too -- if any writer path ever
    # regresses and reintroduces duplicate rows, grading self-heals instead
    # of silently re-corrupting the headline record.
    _gop_before_dedup = len(_gop_gradable)
    _gop_gradable = _gop_gradable.drop_duplicates(subset=["card_date", "game_id", "market"], keep="first")
    _gop_dupes_dropped = _gop_before_dedup - len(_gop_gradable)
    if _gop_dupes_dropped:
        print(f"  [DEDUP] {_gop_dupes_dropped} duplicate ledger row(s) collapsed to distinct (card_date, game_id, market)")
    _gop_session = requests.Session()
    _gop_session.headers.update({"User-Agent": "Mozilla/5.0"})
    _gop_result_cache = {}

    def _gop_fetch_result(game_id):
        gid = int(game_id)
        if gid in _gop_result_cache:
            return _gop_result_cache[gid]
        try:
            _r = _gop_session.get(
                f"https://statsapi.mlb.com/api/v1.1/game/{gid}/feed/live", timeout=15
            )
            _r.raise_for_status()
            _d = _r.json()
            _game_data = _d.get("gameData", {}) or {}
            _live = _d.get("liveData", {}) or {}
            _ls = (_live.get("linescore", {}) or {}).get("teams", {}) or {}
            _away_runs = pd.to_numeric((_ls.get("away") or {}).get("runs"), errors="coerce")
            _home_runs = pd.to_numeric((_ls.get("home") or {}).get("runs"), errors="coerce")
            _teams = _game_data.get("teams", {}) or {}
            _away_nm = (_teams.get("away") or {}).get("name")
            _home_nm = (_teams.get("home") or {}).get("name")
            _state = (_game_data.get("status") or {}).get("abstractGameState")
            _res = {
                "away_team": _away_nm, "home_team": _home_nm,
                "away_runs": _away_runs, "home_runs": _home_runs,
                "state": _state,
            }
        except Exception as _e:
            _res = {"away_team": None, "home_team": None, "away_runs": np.nan,
                     "home_runs": np.nan, "state": f"ERROR: {_e}"}
        _gop_result_cache[gid] = _res
        return _res

    def _gop_norm(s):
        return str(s).strip().lower()

    def _gop_grade_row(row):
        _res = _gop_fetch_result(row["game_id"])
        if _res["state"] != "Final" or pd.isna(_res["away_runs"]) or pd.isna(_res["home_runs"]):
            return pd.Series({
                "actual_away_runs": _res["away_runs"], "actual_home_runs": _res["home_runs"],
                "actual_total": np.nan, "pick_won": np.nan, "result_label": "PENDING",
            })
        _away_r, _home_r = float(_res["away_runs"]), float(_res["home_runs"])
        _market = str(row.get("market", ""))
        _bet = str(row.get("bet", ""))
        _won = None
        if _market == "ML":
            # BUG FIX (2026-08-11, found live on first real run -- 3-52 record,
            # obviously wrong): the deprecated Cell bd676982 ML logger (removed
            # this session) wrote bet as "{team} ML" with the suffix attached;
            # the current Cell 6e66db20 logger writes just the team name. Every
            # old-format row never matched the API's plain team name and graded
            # as an automatic LOSS regardless of the real result -- 49/56 rows
            # were old-format, which is why the record looked like 3-52 instead
            # of the real outcomes. Strip a trailing " ML" (either format) before
            # comparing so historical rows grade correctly too.
            _bet_team = re.sub(r"\s+ML$", "", _bet, flags=re.IGNORECASE).strip()
            _winner = _res["away_team"] if _away_r > _home_r else (_res["home_team"] if _home_r > _away_r else None)
            _won = (_gop_norm(_bet_team) == _gop_norm(_winner)) if _winner else None
        elif _market.startswith("O/U "):
            try:
                _line = float(_market.replace("O/U ", "").strip())
                _total = _away_r + _home_r
                if _gop_norm(_bet) == "over":
                    _won = _total > _line
                elif _gop_norm(_bet) == "under":
                    _won = _total < _line
            except ValueError:
                _won = None
        elif _market.endswith(" -1.5"):
            _team = _market[: -len(" -1.5")].strip()
            if _gop_norm(_team) == _gop_norm(_res["away_team"]):
                _margin = _away_r - _home_r
            elif _gop_norm(_team) == _gop_norm(_res["home_team"]):
                _margin = _home_r - _away_r
            else:
                _margin = None
            _won = (_margin is not None) and (_margin >= 2)
        _label = "PENDING" if _won is None else ("WIN" if _won else "LOSS")
        return pd.Series({
            "actual_away_runs": _away_r, "actual_home_runs": _home_r,
            "actual_total": _away_r + _home_r, "pick_won": _won, "result_label": _label,
        })

    if len(_gop_gradable):
        _graded_cols = _gop_gradable.apply(_gop_grade_row, axis=1)
        _gop_gradable = pd.concat([_gop_gradable, _graded_cols], axis=1)

        _gop_gradable.to_csv(_GOP_GRADED_PATH, index=False)
        print(f"Saved graded ledger to: {_GOP_GRADED_PATH}")

        _finished = _gop_gradable[_gop_gradable["result_label"] != "PENDING"]
        print(f"\nGraded: {len(_finished)} / {len(_gop_gradable)} gradable rows finished")
        if len(_finished):
            for _mkt_kind, _label in [("ML", "Moneyline"), ("O/U", "Totals"), ("RL", "Run Line")]:
                if _mkt_kind == "ML":
                    _sub = _finished[_finished["market"] == "ML"]
                elif _mkt_kind == "O/U":
                    _sub = _finished[_finished["market"].astype(str).str.startswith("O/U ")]
                else:
                    _sub = _finished[_finished["market"].astype(str).str.endswith(" -1.5")]
                if len(_sub):
                    _w = int((_sub["pick_won"] == True).sum())
                    _l = int((_sub["pick_won"] == False).sum())
                    _wr = _w / (_w + _l) if (_w + _l) else float("nan")
                    print(f"  {_label:<10}: {_w}-{_l}  ({_wr:.1%})" if (_w + _l) else f"  {_label:<10}: no graded rows")
                    if _mkt_kind == "ML":
                        print()
                        print("*" * 70)
                        print(f"*  OFFICIAL ML RECORD: {_w}-{_l}  ({_wr:.1%})  over {_w + _l} distinct games  *")
                        print("*" * 70)
                        print()
        mlb_official_plays_graded = _gop_gradable.copy()
    else:
        print("No gradable rows (all missing game_id).")

# =========================================================
# GRADE GATE SAVE CANDIDATES (2026-09-20, user post-mortem FIX 7 -- Cubs ML
# correctly withheld by NO SP DATA, won 9-1 anyway): a gate that withholds a
# real, would-otherwise-qualify play had no way to ever prove it's actually
# helping -- this grades every logged gate-save candidate (see
# _log_gate_save_candidate, called from _emit_why when the NO SP DATA gate
# fires) against real results, reusing the exact same fetch/grade helpers as
# the official ledger just above (_gop_fetch_result/_gop_grade_row -- same
# bet-type rules apply, these are withheld picks of the identical shape, not
# a different kind of bet). Printed under its own header, explicitly separate
# from the OFFICIAL ML RECORD above -- these are NOT real logged bets and must
# never be countable toward win/loss the way that record is.
# =========================================================
print()
print("=" * 70)
print("GRADE GATE SAVE CANDIDATES (informational -- NOT counted in the official record)")
print("=" * 70)

_GSV_LEDGER_PATH_READ = Path(r"c:\Users\<user>\OneDrive\Documents\KAI Labs\sports\MLB\tracking\mlb_gate_saves_ledger.csv")
if not _GSV_LEDGER_PATH_READ.exists():
    print("No gate-save ledger file found -- nothing to grade.")
elif "_gop_grade_row" not in globals() or "_gop_fetch_result" not in globals():
    print("Official-ledger grading helpers unavailable this run -- skipping (needs _gop_grade_row/_gop_fetch_result).")
else:
    _gsv_df = pd.read_csv(_GSV_LEDGER_PATH_READ)
    print(f"Gate-save ledger rows: {len(_gsv_df)}")
    _gsv_df["game_id"] = pd.to_numeric(_gsv_df["game_id"], errors="coerce")
    _gsv_gradable = _gsv_df[_gsv_df["game_id"].notna()].copy()
    _gsv_ungradable = len(_gsv_df) - len(_gsv_gradable)
    if _gsv_ungradable:
        print(f"  {_gsv_ungradable} row(s) have no game_id -- cannot grade, left out")
    if not _gsv_gradable.empty:
        _gsv_gradable = _gsv_gradable.drop_duplicates(subset=["card_date", "game_id", "market"], keep="first")
        _gsv_results = _gsv_gradable.apply(_gop_grade_row, axis=1)
        _gsv_gradable = pd.concat([_gsv_gradable, _gsv_results], axis=1)
        _gsv_graded = _gsv_gradable[_gsv_gradable["pick_won"].notna()]
        print(f"Graded: {len(_gsv_graded)} / {len(_gsv_gradable)} gradable rows finished")
        if len(_gsv_graded):
            for _reason, _grp in _gsv_graded.groupby("gate_reason"):
                _w = int(_grp["pick_won"].sum())
                _n = len(_grp)
                print(f"  {_reason:<20} {_w}-{_n-_w}  ({_w/_n:.1%}) over {_n} withheld play(s) that would otherwise have qualified")
        _gsv_gradable.to_csv(Path(r"c:\Users\<user>\OneDrive\Documents\KAI Labs\sports\MLB\tracking\mlb_gate_saves_graded.csv"), index=False)

# =========================================================
# GRADE DAILY PLAY DISAGREEMENT TRACKER -- MLB-DAILYPLAY-1 informational
# graduation tracking (2026-09-14, user spec: "What The Daily Play Should
# Actually Look Like"). Reads tracking/mlb_daily_play_disagreements.csv
# (written by Cell 6e66db20's DAILY PLAY CANDIDATE ANALYSIS section),
# fetches real results for any PENDING row whose game(s) have finished, and
# fills in which pick (raw-edge vs 4-factor) actually won plus the margin.
# GAME_SELECTION rows can have the raw-edge and 4-factor picks on TWO
# DIFFERENT games (that's the whole point of a game-selection disagreement)
# -- graded by fetching each pick's own game separately, not assumed to
# share one game_id the way this cell's own ledger-grading logic above can.
# Extends the same MLB Stats API feed/live pattern _gop_fetch_result already
# uses, but also pulls per-inning linescore data (F5 grading needs innings
# 1-5 specifically, not just the final score).
# =========================================================
print("\n" + "=" * 70)
print("GRADE DAILY PLAY DISAGREEMENT TRACKER")
print("=" * 70)

_DPG_PATH = Path(r"c:\Users\<user>\OneDrive\Documents\KAI Labs\sports\MLB\tracking\mlb_daily_play_disagreements.csv")
if not _DPG_PATH.exists():
    print("No disagreement tracker file yet -- nothing to grade.")
else:
    _dpg_df = pd.read_csv(_DPG_PATH)
    _dpg_pending = _dpg_df[_dpg_df["result"] == "PENDING"].copy()
    print(f"Tracker rows: {len(_dpg_df)}  |  pending: {len(_dpg_pending)}")

    _dpg_result_cache = {}

    def _dpg_fetch_full(game_id):
        gid = int(game_id)
        if gid in _dpg_result_cache:
            return _dpg_result_cache[gid]
        try:
            _r = requests.get(f"https://statsapi.mlb.com/api/v1.1/game/{gid}/feed/live", timeout=15)
            _r.raise_for_status()
            _d = _r.json()
            _gd = _d.get("gameData", {}) or {}
            _live = _d.get("liveData", {}) or {}
            _ls = _live.get("linescore", {}) or {}
            _teams_ls = _ls.get("teams", {}) or {}
            _innings = _ls.get("innings", []) or []
            _away_f5 = sum(pd.to_numeric((inn.get("away") or {}).get("runs"), errors="coerce") or 0
                           for inn in _innings[:5])
            _home_f5 = sum(pd.to_numeric((inn.get("home") or {}).get("runs"), errors="coerce") or 0
                           for inn in _innings[:5])
            _teams = _gd.get("teams", {}) or {}
            _res = {
                "away_team": (_teams.get("away") or {}).get("name"),
                "home_team": (_teams.get("home") or {}).get("name"),
                "away_runs": pd.to_numeric((_teams_ls.get("away") or {}).get("runs"), errors="coerce"),
                "home_runs": pd.to_numeric((_teams_ls.get("home") or {}).get("runs"), errors="coerce"),
                "away_f5_runs": _away_f5 if len(_innings) >= 5 else np.nan,
                "home_f5_runs": _home_f5 if len(_innings) >= 5 else np.nan,
                "state": (_gd.get("status") or {}).get("abstractGameState"),
            }
        except Exception as _e:
            _res = {"away_team": None, "home_team": None, "away_runs": np.nan, "home_runs": np.nan,
                     "away_f5_runs": np.nan, "home_f5_runs": np.nan, "state": f"ERROR: {_e}"}
        _dpg_result_cache[gid] = _res
        return _res

    def _dpg_norm(s):
        return str(s).strip().lower()

    def _dpg_side_won(side, market, res):
        """True/False/None for whether `side` won under `market` ('F5 ML' =
        first-5-innings, anything else = full game), given a fetched result
        dict. None on a tie (F5 ties are real -- game continues -- can't
        grade as a binary winner) or missing data."""
        if market == "F5 ML":
            _ar, _hr = res.get("away_f5_runs"), res.get("home_f5_runs")
        else:
            _ar, _hr = res.get("away_runs"), res.get("home_runs")
        if pd.isna(_ar) or pd.isna(_hr):
            return None
        _ar, _hr = float(_ar), float(_hr)
        if _ar == _hr:
            return None
        _winner = res.get("away_team") if _ar > _hr else res.get("home_team")
        return _dpg_norm(side) == _dpg_norm(_winner)

    _dpg_graded_ct = 0
    for _dpgi, _dpgrow in _dpg_pending.iterrows():
        if pd.isna(_dpgrow.get("raw_edge_game_id")) or pd.isna(_dpgrow.get("four_factor_game_id")):
            continue
        _raw_res = _dpg_fetch_full(_dpgrow["raw_edge_game_id"])
        _ff_res = (_raw_res if _dpgrow["raw_edge_game_id"] == _dpgrow["four_factor_game_id"]
                   else _dpg_fetch_full(_dpgrow["four_factor_game_id"]))
        if _raw_res["state"] != "Final" or _ff_res["state"] != "Final":
            continue
        _raw_won = _dpg_side_won(_dpgrow["raw_edge_side"], _dpgrow["raw_edge_market"], _raw_res)
        _ff_won = _dpg_side_won(_dpgrow["four_factor_side"], _dpgrow["four_factor_market"], _ff_res)
        if _raw_won is None or _ff_won is None:
            continue
        if _raw_won and not _ff_won:
            _label = "RAW_EDGE_WON"
        elif _ff_won and not _raw_won:
            _label = "FOUR_FACTOR_WON"
        elif _raw_won and _ff_won:
            _label = "BOTH_WON"
        else:
            _label = "BOTH_LOST"
        _raw_ar = _raw_res["away_f5_runs"] if _dpgrow["raw_edge_market"] == "F5 ML" else _raw_res["away_runs"]
        _raw_hr = _raw_res["home_f5_runs"] if _dpgrow["raw_edge_market"] == "F5 ML" else _raw_res["home_runs"]
        _ff_ar = _ff_res["away_f5_runs"] if _dpgrow["four_factor_market"] == "F5 ML" else _ff_res["away_runs"]
        _ff_hr = _ff_res["home_f5_runs"] if _dpgrow["four_factor_market"] == "F5 ML" else _ff_res["home_runs"]
        _margin = (f"raw-edge margin {abs(float(_raw_ar) - float(_raw_hr)):.0f}R "
                   f"({_dpgrow['raw_edge_market']}) | 4-factor margin {abs(float(_ff_ar) - float(_ff_hr)):.0f}R "
                   f"({_dpgrow['four_factor_market']})")
        _dpg_df.loc[_dpgi, "result"] = _label
        _dpg_df.loc[_dpgi, "margin"] = _margin
        _dpg_graded_ct += 1
        print(f"  Graded: {_dpgrow['raw_edge_matchup']} vs {_dpgrow['four_factor_matchup']} "
              f"({_dpgrow['disagreement_type']}) -- {_label}")

    if _dpg_graded_ct:
        _dpg_df.to_csv(_DPG_PATH, index=False)
        print(f"Graded {_dpg_graded_ct} row(s), rewrote {_DPG_PATH.name}")
    else:
        print("No newly-finished games to grade this run.")

    _dpg_graded_all = _dpg_df[_dpg_df["result"] != "PENDING"]
    _dpg_total_evaluated = len(_dpg_graded_all)
    _dpg_game_sel = _dpg_graded_all[_dpg_graded_all["disagreement_type"] == "GAME_SELECTION"]
    _dpg_agreement = _dpg_graded_all[_dpg_graded_all["disagreement_type"] == "AGREEMENT"]

    # MLB-DAILYPLAY-1 (2026-09-15, user review): print total/disagreement/
    # agreement counts and the win rate as clearly separated numbers, plus an
    # automatic sample-size caution -- a 4-2 record on 6 disagreement nights
    # is 67%, clears the 60% graduation bar on paper, and is still too thin a
    # sample to make a real-money architecture decision on. The caution
    # prints whenever disagreement nights < 10 regardless of how good the
    # rate looks -- it does not wait for play 20 to say so.
    print(f"\nGAME-SELECTION graduation tracker (user's own 20-play window, 60% threshold):")
    print(f"  Total days evaluated:      {_dpg_total_evaluated:>3}")
    if _dpg_total_evaluated:
        print(f"  Disagreement nights:       {len(_dpg_game_sel):>3}  ({len(_dpg_game_sel) / _dpg_total_evaluated:.0%})")
    else:
        print(f"  Disagreement nights:       {len(_dpg_game_sel):>3}")
    if len(_dpg_game_sel):
        _dpg_ff_win_ct = (_dpg_game_sel["result"] == "FOUR_FACTOR_WON").sum()
        _dpg_raw_win_ct = (_dpg_game_sel["result"] == "RAW_EDGE_WON").sum()
        _dpg_decisive = _dpg_ff_win_ct + _dpg_raw_win_ct
        if _dpg_decisive:
            print(f"    4-factor won:            {_dpg_ff_win_ct:>3}  ({_dpg_ff_win_ct / _dpg_decisive:.1%}) "
                  f"<- graduation threshold: 60%")
            print(f"    Raw-edge won:            {_dpg_raw_win_ct:>3}")
        if len(_dpg_game_sel) < 10:
            print(f"    Sample note: {len(_dpg_game_sel)} disagreement night(s) -- too few to trust, "
                  f"extend the window regardless of the win rate above")
        elif len(_dpg_game_sel) < 20:
            print(f"    Sample note: {len(_dpg_game_sel)} disagreement night(s) -- borderline, "
                  f"consider extending window")
    print(f"  Agreement nights:          {len(_dpg_agreement):>3}")

    if len(_dpg_game_sel) >= 20:
        if _dpg_decisive and _dpg_ff_win_ct / _dpg_decisive >= 0.60:
            print("  !! GRADUATION THRESHOLD MET (>=60% on decisive disagreement nights) -- "
                  "per the user's own criterion, this methodology is ready to become the real selector")
        else:
            print("  Graduation threshold NOT met -- per the user's own rule, extend the informational "
                  "window another 20 plays and revisit the factor list rather than flipping on a coin flip")

    # MLB-F5-AUDIT-1 (2026-10-06 user audit -- "Since the F5 fetch never worked, [market-selection results] shouldn't
    # count in postmortem stats until they're re-checked against real F5 prices"): real F5 odds start 2026-10-06
    # (MLB-F5-FETCH-1); earlier MARKET_SELECTION rows (0 existed at the audit) are excluded from this tally.
    _dpg_mkt_sel = _dpg_graded_all[(_dpg_graded_all["disagreement_type"] == "MARKET_SELECTION")
                                   & (_dpg_graded_all["card_date"].astype(str) >= "2026-10-06")]
    if len(_dpg_mkt_sel):
        _dpg_f5_win_ct = (_dpg_mkt_sel["result"] == "FOUR_FACTOR_WON").sum()
        _dpg_full_win_ct = (_dpg_mkt_sel["result"] == "RAW_EDGE_WON").sum()
        print(f"\nMARKET-SELECTION (F5 vs full-game, same underlying pick): {len(_dpg_mkt_sel)} graded -- "
              f"F5 won {_dpg_f5_win_ct}, full-game won {_dpg_full_win_ct} (tracked separately per user "
              "request, may graduate on its own timeline)")


GRADE OFFICIAL PLAYS LEDGER
Ledger rows: 89
  24 row(s) have no game_id (pre-fix ledger rows) -- cannot grade, left out


Saved graded ledger to: c:\Users\<user>\OneDrive\Documents\KAI Labs\sports\MLB\tracking\mlb_official_plays_graded.csv

Graded: 65 / 65 gradable rows finished
  Moneyline : 32-31  (50.8%)

**********************************************************************
*  OFFICIAL ML RECORD: 32-31  (50.8%)  over 63 distinct games  *
**********************************************************************

  Run Line  : 1-1  (50.0%)

GRADE GATE SAVE CANDIDATES (informational -- NOT counted in the official record)
Gate-save ledger rows: 3
  1 row(s) have no game_id -- cannot grade, left out


Graded: 2 / 2 gradable rows finished
  NO SP DATA           1-1  (50.0%) over 2 withheld play(s) that would otherwise have qualified

GRADE DAILY PLAY DISAGREEMENT TRACKER
Tracker rows: 28  |  pending: 0
No newly-finished games to grade this run.

GAME-SELECTION graduation tracker (user's own 20-play window, 60% threshold):
  Total days evaluated:       28
  Disagreement nights:         7  (25%)
    4-factor won:              3  (60.0%) <- graduation threshold: 60%
    Raw-edge won:              2
    Sample note: 7 disagreement night(s) -- too few to trust, extend the window regardless of the win rate above
  Agreement nights:           21


In [90]:
# %%
# =========================================================
# CELL -- DAILY MARKET SNAPSHOT (MODEL VS MARKET, EDGE GAMES ONLY)
# Side-by-side view for every game where the model actually found a
# positive edge on at least one market (ML/TOT/RL). Reuses the final,
# fully-adjusted numbers from the card selector above (cand, board,
# _mlb_snorm, _poisson_win_by2, _fetch_rl_odds) so this never disagrees
# with the real card -- it's a clean read of the same data.
#
# 2026-07-25 rewrite: a full-card postmortem that day found every single
# game printed a "positive edge" line (some as small as +0.1%, i.e. pure
# rounding noise) and the "BIGGEST EDGES TODAY" ranking included all of them
# regardless of whether they cleared the real official-play bar (passes_edge_
# rule / OU_MIN_EDGE / RL_MIN_EDGE from the card selector above). That day
# had ZERO plays that actually cleared those bars, but the report read like
# a menu of 10 rankable picks -- the official-play signal was buried in
# noise. Fixes: (1) a minimum-edge floor below which a line is labeled noise
# and dropped from ranking, (2) explicit non-qualifying tags so a big raw
# edge on a sub-bar model prob doesn't read as a real signal, (3) the
# "biggest edges" ranking now only ever ranks plays that actually clear the
# real bar, with a loud banner when none do, and a separate informational
# list for everything else.
#
# 2026-08-14 rewrite (user request): this used to print every game on the
# slate (ML/TOT/RL lines) even when nothing on that game cleared even the
# noise floor -- on a normal 10-15 game day that was mostly dead weight to
# scroll past to find the handful of games that mattered. Now buffers each
# game's lines and only flushes them if the game produced at least one
# _snap_rows entry (i.e. some market had a real, non-noise edge). Nothing
# about the edge math changed, only what gets printed. Same session: the
# FINAL CARD + WHY section below now also runs for every edge found (not
# just the official play), and two new SP flags (home/road split, recent-
# form credibility vs xERA) were added after a manual QA of the Angels ML
# play found both were already driving the model's numbers but never
# surfaced on the card itself.
# =========================================================
import pandas as pd
import math

_SNAP_MIN_EDGE = 0.02  # below this, a "positive edge" is rounding/noise, not a signal
_SNAP_MIN_MODEL_PROB = globals().get("MIN_MODEL_PROB", 0.58)
_SNAP_OU_BAR = globals().get("OU_MIN_EDGE", 0.18)
_SNAP_RL_BAR = globals().get("RL_MIN_EDGE", 0.08)
# 2026-08-10 (user audit): the real ML edge gate in Cell bd676982 (passes_edge_
# rule) requires BOTH p_bet_win_model >= MIN_MODEL_PROB AND a logit-space edge
# at or above BASE_EDGE_MIN (non-repeat games) / REPEAT_EDGE_MIN (repeat
# games) -- reuse those exact constants here instead of a second hardcoded
# copy, so a "too small to qualify" tag can show the real number it missed by.
_SNAP_BASE_EDGE_MIN = globals().get("BASE_EDGE_MIN", 0.12)  # 2026-08-10: fallback matches lowered floor
_SNAP_REPEAT_EDGE_MIN = globals().get("REPEAT_EDGE_MIN", 0.12)

# MLB-EDGE-RULE-DETAIL-1 / BELOW-BAR DRIVER (2026-10-01 user audit): every
# below-bar entry states (a) the real rule it failed with the miss, and (b) a
# one-line primary driver -- what is actually producing the model's lean --
# so the research pipeline doesn't start every entry at PARTIAL. Kept on the
# SAME line as the entry (pick_of_the_day.py copies/caps these one line each).
def _mlb_ou_rule_tag(c):
    _gates = c.get("_ou_gate_names") or []
    if _gates:
        return f"  [blocked by O/U gate: {'; '.join(_gates)}]"
    _edge, _bar = float(c.get("edge", 0) or 0), float(c.get("_ou_min_edge", _SNAP_OU_BAR) or _SNAP_OU_BAR)
    _el, _req = c.get("_ou_edge_logit"), c.get("_ou_req_logit")
    _mkt, _p = float(c.get("market_implied_prob", 0) or 0), float(c.get("p_bet_win_model", 0) or 0)
    if _el is None or _req is None or not (0 < _mkt < 1):
        return f"  [below official O/U bar of {_bar:.0%}, informational only -- have {_edge:+.1%}]"
    _p_req = 1.0 / (1.0 + math.exp(-(_mlb_logit(_mkt) + float(_req))))
    return (f"  [O/U rule (raw edge >= {_bar:.0%} OR logit edge >= {float(_req):+.3f}) failed both: raw {_edge:+.1%} < "
            f"{_bar:.0%}; logit {float(_el):+.3f} < {float(_req):+.3f} -- at mkt {_mkt:.1%} needs model >= "
            f"{_p_req:.1%}, has {_p:.1%}, missed by {_p_req - _p:.1%}]")

def _mlb_ou_decomp(c):
    _k = c.get("decomp_key")
    return (globals().get("mlb_totals_decomp") or {}).get(tuple(_k)) if _k else None

def _mlb_ou_driver(c):
    # MLB-TOT-COMP-1 (2026-10-04 user audit -- "'calibrated proj 8.4 vs line 7.0' as its driver ... is
    # just the edge restated"): the driver now names the component(s) that actually move the
    # projection off a league-average game, toward the bet. Full term-by-term breakdown prints as
    # [TOTALS DECOMP] in the FINAL CARD block for this game.
    _line = float(c.get("ou_line"))
    _raw, _cal = float(c.get("pred_total_runs")), float(c.get("pred_total_cal"))
    _d = _mlb_ou_decomp(c)
    if not _d:
        _txt = (f"Driver: calibrated proj {_cal:.1f} vs line {_line:.1f} ({_cal - _line:+.1f}R); "
                f"raw run model {_raw:.1f} ({_raw - _line:+.1f}R) [no component breakdown this run]")
        return _txt
    _side = c.get("bet_side")
    _terms = [(l, v) for l, v, w in _d["parts"] if l not in ("league base", "home field")]
    _toward = sorted([t for t in _terms if (t[1] > 0 if _side == "over" else t[1] < 0)], key=lambda t: -abs(t[1]))
    _against = sorted([t for t in _terms if (t[1] < 0 if _side == "over" else t[1] > 0)], key=lambda t: -abs(t[1]))
    _txt = (f"Driver: component proj {_raw:.1f} (league-average game {_d['parts'][0][1]:.1f}), after {globals().get('_MARKET_SHRINK', 0.35):.0%} market shrink "
            f"{_cal:.1f} vs line {_line:.1f} ({_cal - _line:+.1f}R); toward the {str(_side).title()}: "
            + (", ".join(f"{l} {v:+.2f}" for l, v in _toward[:3]) or "nothing")
            + (" | against: " + ", ".join(f"{l} {v:+.2f}" for l, v in _against[:2]) if _against else "")
            + f" | old formula {float(c.get('pred_total_old', _raw)):.1f}")
    for _n in c.get("_ou_adj_notes") or []:
        _txt += f" | [{_n}]"
    return _txt

def _mlb_f5_line(r):
    """R-F5-1 support (2026-10-06 user audit -- "Print the F5 ML whenever the pick-side SP carries a thin-sample,
    post-IL or short-leash flag"). Model F5 from the component totals terms (MLB-TOT-COMP-1) over 5 innings:
    the SP term covers min(IPexp, 5), the opposing pen the rest of the 5, scale terms x 5/9; independent Poisson,
    tie = push. Returns None when no trigger fires."""
    try:
        _aw, _hm = str(r.get(away_col, "") or ""), str(r.get(home_col, "") or "")
        _pick_home = str(r.get("bet_team", "")).strip() == _hm.strip()
        _dec = (globals().get("mlb_totals_decomp") or {}).get((_aw.lower().strip(), _hm.lower().strip()))
        _trig = []
        if isinstance(r.get("_pick_post_il"), dict):
            _trig.append("post-IL")
        if float(r.get("_pick_thin_shave") or 0) > 0:
            _trig.append("thin sample")
        _pick_sp_side = None
        if _dec:
            _pick_sp_side = _dec["sides"]["away" if _pick_home else "home"]   # batting side facing the pick SP
            if _pick_sp_side["ipx"] < 5.0:
                _trig.append(f"short leash ({_pick_sp_side['ipx']:.1f} IP expected)")
        if not _trig:
            return None
        _mkt = (globals().get("_f5_odds") or {}).get((_aw.lower(), _hm.lower()))
        _mtxt = (f"mkt {(_mkt['home_nv'] if _pick_home else _mkt['away_nv']):.1%} "
                 f"({int(_mkt['home_ml'] if _pick_home else _mkt['away_ml']):+d} {_mkt['book']})") if _mkt else "no F5 market posted"
        if not _dec or "_TC1" not in globals() or not _TC1:
            return f"F5 ML ({', '.join(_trig)}): {_mtxt}; model F5 unavailable (no component decomposition for this game)"
        from math import exp, factorial
        _lg = _TC1["lg_r9"]
        def _f5_runs(sd):
            c, ip = sd["c"], min(float(sd["ipx"]), 5.0)
            v = (5 / 9) * (_TC1["c0"] + c["off"] + c["pf"] + c["home"])
            v += _TC1["c_sp"] * (sd["sp_r9"] - _lg) * ip / 9 + _TC1["c_pen"] * (sd["pen_r9"] - _lg) * (5 - ip) / 9
            v += c["elite"] * (ip / sd["ipx"] if sd["ipx"] else 0.0)
            v += (_TC1.get("ps_run_env_delta", -0.38) if _dec.get("is_ps") else 0.0) * 5 / 18
            return max(v, 0.3)
        _lh, _la = _f5_runs(_dec["sides"]["home"]), _f5_runs(_dec["sides"]["away"])
        _pm = lambda k, l: exp(-l) * l ** k / factorial(k)
        _ph = [_pm(k, _lh) for k in range(16)]; _pa = [_pm(k, _la) for k in range(16)]
        _hw = sum(_ph[i] * _pa[j] for i in range(16) for j in range(16) if i > j)
        _tie = sum(_ph[i] * _pa[i] for i in range(16))
        _p_home = _hw / (1 - _tie)
        _pp = _p_home if _pick_home else 1 - _p_home
        _edge = (f" -> edge {_pp - (_mkt['home_nv'] if _pick_home else _mkt['away_nv']):+.1%}") if _mkt else ""
        return (f"F5 ML ({', '.join(_trig)}): model {_pp:.1%} for {r.get('bet_team')} "
                f"(F5 proj {_la:.2f}-{_lh:.2f}, tie {_tie:.0%} = push) vs {_mtxt}{_edge}")
    except Exception as _f5le:
        return f"F5 ML: could not price ({_f5le})"


def _mlb_ml_driver(r):
    _pk, _op = _pick_sp_name(r) if "_pick_sp_name" in globals() else "", _opp_sp_name(r) if "_opp_sp_name" in globals() else ""
    _gap = r.get("_sp_elo_gap")
    _parts = []
    # MLB-PS-OPENER-NEWS-1: an announced opener with a named bulk arm -> the bulk arm is the SP in the driver
    _ann_r = globals().get("mlb_announced_roles") or {}
    _is_home_pk0 = str(r.get('bet_team', '')).strip() == str(r.get(home_col, '')).strip()
    _pk_team = str(r.get(home_col if _is_home_pk0 else away_col, "")).lower()
    _op_team = str(r.get(away_col if _is_home_pk0 else home_col, "")).lower()
    if _pk_team in _ann_r and _ann_r[_pk_team].get("bulk"):
        _pk = f"{_ann_r[_pk_team]['bulk']} (bulk; {_ann_r[_pk_team]['opener']} opens)"
    if _op_team in _ann_r and _ann_r[_op_team].get("bulk"):
        _op = f"{_ann_r[_op_team]['bulk']} (bulk; {_ann_r[_op_team]['opener']} opens)"
    if _pk or _op:
        _sp = f"SP {_pk or 'TBD'} vs {_op or 'TBD'}"
        # MLB-DRIVER-OPENER-1 (2026-10-03 user audit -- "Braves ML printed 'SP TBD' with a
        # -300 ELO placeholder"): _sp_elo_gap is the GATE-scale lookup, where a missing name
        # falls back to the 1460 default (Skubal ~1760 - 1460 = the "-300"). The price never
        # used that: an opener/TBD side in a postseason game is set equal to the opposing
        # starter (zero SP nudge, bullpen-game prior -- [OPENER SP RATING] in Cell 87). The
        # driver now says that instead of printing a gap that isn't in the number.
        _is_home_pk = str(r.get('bet_team', '')).strip() == str(r.get(home_col, '')).strip()
        _ps_e = (globals().get("mlb_ps_bulk_arm") or {}).get(
            (str(r.get(away_col, "")).lower().strip(), str(r.get(home_col, "")).lower().strip())) or {}
        _pk_role = _ps_e.get("home_role" if _is_home_pk else "away_role")
        _op_role = _ps_e.get("away_role" if _is_home_pk else "home_role")
        if (not _pk) or (not _op) or _pk_role in ("OPENER", "TBD") or _op_role in ("OPENER", "TBD"):
            _who = [f"{'pick' if _x == 'pk' else 'opponent'} side {'opener' if _rl == 'OPENER' else 'TBD'}"
                    for _x, _rl, _nm in (("pk", _pk_role, _pk), ("op", _op_role, _op))
                    if (_rl in ("OPENER", "TBD") or not _nm)
                    and not (((_pk_team if _x == 'pk' else _op_team) in _ann_r) and _ann_r[_pk_team if _x == 'pk' else _op_team].get('bulk'))]
            if _who:
                _sp += f" ({'; '.join(_who)}: bullpen-game prior, zero SP nudge in the price -- no starter ELO gap)"
            else:
                _sp += " (announced opener: the named bulk arm's rating is used for that side)"
        elif pd.notna(_gap):
            _sp += f" ({-float(_gap):+.0f} ELO for the pick's starter)"
        _parts.append(_sp)
    # MLB-CAREER-H2H-2 (2026-10-06 user audit -- "May's 1.23 ERA against San Diego never appeared on the card.
    # MLB-CAREER-H2H-1 only checks the pick-side SP against the opponent. It should run both ways and print both
    # lines"): the game block printed both pitchers, but the pick row (what reaches the pick card / below-bar
    # list) carried only the ELO gap. Both directions now ride on the pick row: pick SP vs the opponent AND the
    # opponent's SP vs the pick team, season + career.
    try:
        _pk_raw = _pick_sp_name(r) if "_pick_sp_name" in globals() else ""
        _op_raw = _opp_sp_name(r) if "_opp_sp_name" in globals() else ""
        _pk_tm = str(r.get(home_col if _is_home_pk0 else away_col, "") or "")
        _op_tm = str(r.get(away_col if _is_home_pk0 else home_col, "") or "")
        _h2h_bits = []
        for _lbl, _nm, _vs in (("pick SP", _pk_raw, _op_tm), ("opp SP", _op_raw, _pk_tm)):
            if not _nm or not _vs:
                continue
            _b = []
            if "_sp_vs_opponent_era" in globals():
                _se, _sn = _sp_vs_opponent_era(_nm, _vs)
                if _se is not None:
                    _b.append(f"{_se:.2f} ERA in {_sn} GS this season")
            if "_sp_career_vs_opponent_era" in globals():
                _ce, _cn = _sp_career_vs_opponent_era(_nm, _vs)
                if _ce is not None:
                    _b.append(f"{_ce:.2f} career ({_cn} app)")
            if _b:
                _h2h_bits.append(f"{_lbl} {_nm} vs {_vs}: " + ", ".join(_b))
        if _h2h_bits:
            _parts.append("SP H2H -- " + " | ".join(_h2h_bits))
    except Exception as _h2h_e:
        _parts.append(f"SP H2H unavailable ({_h2h_e})")
    _f5l = _mlb_f5_line(r)
    if _f5l:
        _parts.append(_f5l)
    _raw = r.get("p_bet_win_model_raw")
    if pd.notna(_raw) and pd.notna(r.get("market_implied_prob")):
        _parts.append(f"pre-blend model {float(_raw):.1%} vs mkt {float(r.get('market_implied_prob')):.1%}")
    return "Driver: " + "; ".join(_parts) if _parts else "Driver: none recorded on this row"

import unicodedata as _mnn_ud, re as _mnn_re
def _mlb_name_norm(s):
    if not s:
        return ""
    s = _mnn_ud.normalize("NFD", str(s))
    s = "".join(ch for ch in s if _mnn_ud.category(ch) != "Mn")
    return _mnn_re.sub(r"[^a-z ]", "", s.lower()).strip()

print("=" * 78)
print("MLB DAILY MARKET SNAPSHOT -- MODEL VS MARKET (edge games only)")
# 2026-08-08 (found live: a card got graded against the wrong day's real
# results because nothing in this output said which date it was actually
# for -- the SP feed's own stale-slate protection, added the same session,
# only warns in the console log, not in the card text itself).
if "CARD_DATE" in globals():
    _card_date_note = "  [TOMORROW's slate -- pipeline ran after 8 PM ET, today's already-decided games are excluded]" if globals().get("_card_delta") else "  [today's slate]"
    print(f"Card date: {CARD_DATE}{_card_date_note}")
# ODDS SNAPSHOT TIMESTAMP (2026-09-19, user audit -- Rays +26.1% edge:
# the card never says whether that was calculated against a sharp
# current line or a stale early one). mlb_odds_fetch_time (Cell -- PULL
# TODAY'S MLB MONEYLINE ODDS FROM ESPN) captures the single moment this
# run pulled ESPN's odds -- every mkt%/edge% below traces back to that
# one snapshot, so print it once right under the card date.
if globals().get("mlb_odds_fetch_time"):
    print(f"Odds pulled: {mlb_odds_fetch_time} (local) -- every mkt%/edge% below is against this single snapshot, not a live line")
print("=" * 78)

# SLATE TOTAL OUTLIER (2026-09-02, user audit -- real Orioles loss:
# TheSpread flagged the BAL@COL O/U of 11 sitting a full run and a half
# above every other game that day -- the market pricing in max Coors
# inflation on both offenses was itself a signal (first 2 games of that
# series went 3 and 6 total runs combined, well under a line that high),
# and nothing in this notebook compares a game's total to the rest of the
# day's slate at all. Real games only -- same source _s_ou_cands below
# reads from, so this can't disagree with what's actually on the card.
_SLATE_OU_OUTLIER_MIN_GAP = 1.5  # runs above the slate median to flag
# BUG FIX (2026-09-02, live-caught testing this same feature): a plain
# `is not None` filter lets a NaN ou_line through -- np.median of a list
# containing NaN returns NaN, and every downstream ">=" comparison against
# NaN is silently False, disabling this check for the whole slate with no
# visible error. pd.notna() catches NaN as well as None.
_slate_ou_lines = [float(c.get("ou_line")) for c in globals().get("_ou_candidates_ranked", [])
                   if pd.notna(c.get("ou_line"))]
_slate_ou_median = float(np.median(_slate_ou_lines)) if len(_slate_ou_lines) >= 3 else None

_snap_rows = []
_snap_games_scanned = 0
_snap_games_shown = 0

for _si, _sr in board.iterrows():
    _snap_games_scanned += 1
    _s_away = str(_sr.get(away_col, "?"))
    _s_home = str(_sr.get(home_col, "?"))
    _matchup = f"{_s_away} @ {_s_home}"
    _game_lines = []
    _pr = _game_lines.append
    _rows_before = len(_snap_rows)
    # BUG FIX (2026-08-18, user audit found live): cell bd676982's SP MISMATCH
    # / SP SOURCE CONFLICT checks used to be invisible here -- a starter
    # conflict serious enough that cell bd676982's own loop calls it "grading
    # blocked until confirmed" could still reach this card as a fully-graded
    # OFFICIAL play with no indication anything was in question (real case:
    # Cardinals @ Reds ML shipped OFFICIAL the same day this exact conflict
    # was firing one cell earlier -- turned out to be a false alarm, ESPN's
    # cache was stale, but nothing structural caught it). Surfaces whatever
    # cell bd676982 found, if anything, right on this game's snapshot entry.
    _sp_conflict_note = globals().get("_mlb_sp_source_conflicts", {}).get((_s_away, _s_home))
    # BUG FIX (2026-08-19, Stage 5 card review caught this live): the
    # bd676982 SP-conflict loop explicitly says "grading blocked until
    # confirmed" and does a continue -- but that only skipped ITS OWN
    # letter-grade computation, never reached the passes_edge_rule/OFFICIAL
    # flag this cell reads, which was set earlier and independently. Result:
    # a conflict serious enough for that loop to call "blocked until
    # confirmed" still shipped as a fully-tagged OFFICIAL play with nothing
    # but a warning banner (real case: today, all 4 MLB games flagged
    # unresolved conflicts and all 4 still shipped OFFICIAL). Withholding
    # OFFICIAL status here actually enforces what the block was always
    # supposed to mean, for every market (ML/O-U/RL), not just a banner
    # next to a play still tagged as if nothing were in question.
    _snap_sp_conflict = bool(_sp_conflict_note)
    _snap_no_sp_data = "NO SP DATA" in _pitcher_note(_sr)
    # BUG FIX (2026-08-27, user audit -- Nationals ML +13.0%/Giants ML +9.7%,
    # both 2026-08-26 OFFICIAL plays, both lost): Cell bd676982's Grade/size
    # loop had already capped these to A- [BP RISK SCORE MISSING] / B+ [OWN SP
    # COLLAPSE] and sized them 0.5u WATCH-ONLY -- none of that reached this
    # cell, which re-derives "official" from passes_edge_rule alone. Same
    # withholding pattern as the SP-conflict check right above.
    _gd_note = globals().get("mlb_grade_downgrades", {}).get((_s_away, _s_home))
    _snap_grade_downgraded = bool(_gd_note)
    if _gd_note:
        _pr(f"  !! RAW CARD DOWNGRADE: Final Card Selector graded this {_gd_note.get('grade')}, "
            f"sized {_gd_note.get('unit')} -- verify before trusting any OFFICIAL tag below")
    if _sp_conflict_note:
        _pr(f"  !! SP CONFLICT UNRESOLVED: {_sp_conflict_note} -- verify starter before trusting any SP-driven edge here")

    # ---- Moneyline (pulled from cand: final model prob + no-vig market prob) ----
    if game_id_col and game_id_col in cand.columns and game_id_col in board.columns and pd.notna(_sr.get(game_id_col)):
        _s_cand = cand[(cand[away_col].astype(str) == _s_away) & (cand[home_col].astype(str) == _s_home)
                       & (cand[game_id_col] == _sr.get(game_id_col))]
    else:
        _s_cand = cand[(cand[away_col].astype(str) == _s_away) & (cand[home_col].astype(str) == _s_home)]
    if len(_s_cand):
        _s_ml_printed = False
        for _, _cr in _s_cand.iterrows():
            _c_model = _cr.get("p_bet_win_model")
            _c_mkt = _cr.get("market_novig_prob")
            _c_has_odds = bool(_cr.get("game_has_odds"))
            if pd.notna(_c_model) and _c_has_odds and pd.notna(_c_mkt):
                _c_edge = float(_c_model) - float(_c_mkt)
                if _c_edge <= 0:
                    continue  # negative/mirror side -- only the positive-edge side is shown

                # MLB-INJ-CONF-1 (2026-09-14, user audit, escalated to 'fix the
                # architecture' after an earlier ranking-only patch today): confirmed
                # via the model's own feature list (mlb_win_model_compressed_best_
                # meta.json -- 11 rolling team-scoring/rest-day diffs, zero market or
                # injury inputs) that excluding injuries from p_bet_win_model is a
                # genuine gap, not a deliberate 'market already prices it in' design.
                # Prices the net injury delta directly into _c_model/_c_edge HERE --
                # the single point this loop computes them from -- so display,
                # ranking (_snap_ranked sorts on this same edge), and the ledger
                # (_log_official_play reads this same row) all see the honest,
                # injury-aware number instead of a footnote about it. Symmetric both
                # directions now (an earlier pass today only handled the FAVORS-the-
                # pick case, as a ranking tiebreaker) -- same margin<->prob formula,
                # same 0.20R materiality bar the INJURY VS PICK CONFLICT check below
                # already used, so the two can never disagree on what counts as
                # material. Does NOT reach back into passes_edge_rule / the ~20
                # upstream cand gates -- a play that already failed an earlier gate
                # purely on raw edge size stays failed here; fixing that needs the
                # adjustment injected into the gate cascade itself (Cell 87), a
                # bigger change than this display/ranking cell should make alone.
                _c_model_raw, _c_edge_raw = float(_c_model), _c_edge
                _c_inj_note = None
                if "mlb_lineup_flags" in globals() and "_pl_norm" in globals():
                    _c_aw_pen = float((mlb_lineup_flags.get(_pl_norm(_s_away)) or {}).get("penalty", 0) or 0)
                    _c_hm_pen = float((mlb_lineup_flags.get(_pl_norm(_s_home)) or {}).get("penalty", 0) or 0)
                    _c_net_mag = abs(_c_aw_pen - _c_hm_pen)
                    _c_net_favors = _s_home if _c_aw_pen > _c_hm_pen else (_s_away if _c_hm_pen > _c_aw_pen else None)
                    if _c_net_favors and _c_net_mag >= 0.20:
                        _c_pick_norm = _pl_norm(str(_cr.get("bet_team")))
                        _c_favors_norm = _pl_norm(_c_net_favors)
                        try:
                            from scipy.stats import norm as _c_inj_norm
                            _c_mp = float(min(max(_c_model_raw, 1e-4), 1 - 1e-4))
                            _c_implied_margin = float(_c_inj_norm.ppf(_c_mp)) * _MLB_RUN_STD
                            _c_signed_delta = _c_net_mag if _c_pick_norm == _c_favors_norm else -_c_net_mag
                            _c_adj_margin = _c_implied_margin + _c_signed_delta
                            _c_adj_prob = float(_c_inj_norm.cdf(_c_adj_margin / _MLB_RUN_STD))
                            _c_model = _c_adj_prob
                            _c_edge = _c_adj_prob - float(_c_mkt)
                            _c_dir_word = "tailwind" if _c_pick_norm == _c_favors_norm else "headwind"
                            _c_inj_note = (f"  [NET INJURY DELTA PRICED IN] win prob {_c_model_raw:.1%} -> "
                                           f"{_c_model:.1%} ({_c_net_mag:.2f}R {_c_dir_word}), edge "
                                           f"{_c_edge_raw:+.1%} -> {_c_edge:+.1%}")
                        except Exception:
                            _c_model, _c_edge = _c_model_raw, _c_edge_raw
                if _c_edge <= 0:
                    continue  # a real injury headwind can legitimately erase what was a raw positive edge
                _c_official = (_cr.get("passes_edge_rule") == 1) and not _snap_sp_conflict and not _snap_no_sp_data and not _snap_grade_downgraded
                _c_noise = _c_edge < _SNAP_MIN_EDGE
                _c_tag = ""
                if _c_official:
                    _c_tag = "  [OFFICIAL PLAY]"
                elif _cr.get("passes_edge_rule") == 1 and _snap_sp_conflict:
                    _c_tag = "  [SP CONFLICT -- WITHHELD FROM OFFICIAL, would otherwise qualify]"
                elif _cr.get("passes_edge_rule") == 1 and _snap_no_sp_data:
                    _c_tag = "  [NO SP DATA -- WITHHELD FROM OFFICIAL, would otherwise qualify]"
                elif _cr.get("passes_edge_rule") == 1 and _snap_grade_downgraded:
                    _c_tag = (f"  [RAW CARD GRADE {_gd_note.get('grade')} / SIZE {_gd_note.get('unit')} -- "
                              f"WITHHELD FROM OFFICIAL, would otherwise qualify]")
                elif _c_noise:
                    _c_tag = f"  [noise -- below {_SNAP_MIN_EDGE:.0%} floor]"
                elif isinstance(_cr.get("_edge_rule_detail"), str) and _cr.get("_edge_rule_detail"):
                    _c_tag = f"  [{_cr.get('_edge_rule_detail')}]"
                elif float(_c_model) < _SNAP_MIN_MODEL_PROB:
                    _c_tag = f"  [below model-prob floor of {_SNAP_MIN_MODEL_PROB:.0%}, not a qualifying play]"
                    if 0.45 <= float(_c_model) <= 0.55:
                        _c_tag += "  [near coin-flip]"
                else:
                    _c_req_floor = _SNAP_REPEAT_EDGE_MIN if bool(_cr.get("is_repeat")) else _SNAP_BASE_EDGE_MIN
                    _c_elog = _cr.get("_edge_logit")
                    # Convert the real gating quantity (logit-space edge vs raw,
                    # vig-included market price) back to a 50%-baseline-equivalent
                    # percentage -- directly comparable to _c_req_floor, unlike
                    # _c_edge above (which is model vs NO-VIG market price, a
                    # different, more generous number whenever real vig exists).
                    if pd.notna(_c_elog):
                        _c_gate_edge = (1.0 / (1.0 + math.exp(-float(_c_elog)))) - 0.5
                    else:
                        _c_gate_edge = float(_c_edge)
                    # BUG FIX (2026-08-26, user-reported card review): this branch used to
                    # unconditionally print "edge too small to qualify" for ANY play that
                    # reached here with passes_edge_rule==0 -- true whenever the edge really
                    # was below _c_req_floor, but WRONG whenever one of this cell's other
                    # ~10 gates (edge ceiling, underdog threshold, walk-rate, last-start
                    # implosion, SP staleness, ELO/ERA divergence, record-gap, etc.) was the
                    # actual reason and the real gating edge cleared the floor easily. Real
                    # case: Athletics showed "edge too small to qualify -- needs ~12%" with a
                    # real gating edge of +24.6% -- comfortably clears 12%, but the play was
                    # actually blocked by the 25% edge-ceiling calibration-review gate (Gate
                    # 5 above), which has no display branch here. Rather than build a full
                    # reason-string that tracks all ~15 gate sites (a much bigger change to
                    # the actual gating logic, not just this display cell), this checks
                    # whether the edge actually clears the floor before printing the "too
                    # small" text -- when it does, an honest "disqualified by another gate"
                    # message prints instead of a wrong one.
                    if _c_gate_edge >= _c_req_floor:
                        # M-NEW-1 (2026-09-10, user audit -- Rangers ML +21.6% real gating
                        # edge killed by an unidentified gate): cand['_gate_fail_reason'] is
                        # now written by every one of this cell's ~20 gate/shave sites at the
                        # exact moment a row crosses below the qualifying floor (see that
                        # column's own build comment earlier in this cell for the full
                        # rationale) -- reads the REAL reason directly instead of pointing at
                        # a summary block that (a) lives in a different, earlier cell and (b)
                        # only ever had aggregate counts with no game/team identification.
                        _c_real_reason = _cr.get('_gate_fail_reason')
                        if isinstance(_c_real_reason, str) and _c_real_reason:
                            _c_tag = (f"  [disqualified by a data-quality/risk gate, NOT edge size -- real "
                                      f"gating edge {_c_gate_edge:+.1%} already clears the ~{_c_req_floor:.0%} "
                                      f"floor; {_c_real_reason}]")
                        else:
                            _c_tag = (f"  [disqualified by a data-quality/risk gate, NOT edge size -- real "
                                      f"gating edge {_c_gate_edge:+.1%} already clears the ~{_c_req_floor:.0%} "
                                      f"floor; reason not captured -- verify manually before treating as a genuine edge]")
                    else:
                        _c_tag = (f"  [edge too small to qualify -- needs ~{_c_req_floor:.0%} edge "
                                  f"(logit-space floor{', repeat-game rate' if bool(_cr.get('is_repeat')) else ''}), "
                                  f"real gating edge is {_c_gate_edge:+.1%}]")
                # 2026-08-25 (user audit): document active edge-discount
                # gates on the record even when the game doesn't crack any
                # top-N display cutoff -- this per-game snapshot line prints
                # unconditionally, so it's the one place a discount like the
                # rookie/thin-sample or sustained-cold-streak shave can always
                # be traced back to, regardless of official/below-bar status.
                _c_gate_notes = []
                _c_pick_starts = _cr.get('_pick_thin_starts')
                if float(_cr.get('_pick_thin_shave') or 0) > 0:
                    _c_gate_notes.append(f"SP debut adjustment: {_c_pick_starts if _c_pick_starts is not None else '?'} career MLB start(s), insufficient sample, edge discounted {float(_cr.get('_pick_thin_shave')):.1%}")
                _c_opp_starts = _cr.get('_opp_thin_starts')
                if float(_cr.get('_opp_thin_shave') or 0) > 0:
                    _c_gate_notes.append(f"opp SP2 debut adjustment: {_c_opp_starts if _c_opp_starts is not None else '?'} career MLB start(s), insufficient sample, edge discounted {float(_cr.get('_opp_thin_shave')):.1%}")
                if float(_cr.get('_pick_l10_shave') or 0) > 0:
                    _c_gate_notes.append(f"SP sustained cold streak (L10 ERA {float(_cr.get('_pick_l10_val') or 0):.2f}), edge discounted {float(_cr.get('_pick_l10_shave')):.1%}")
                if float(_cr.get('_opp_l10_shave') or 0) > 0:
                    _c_gate_notes.append(f"opp SP sustained cold streak (L10 ERA {float(_cr.get('_opp_l10_val') or 0):.2f}), edge boosted {float(_cr.get('_opp_l10_shave')) * 0.5:.1%}")
                if float(_cr.get('_pick_hr9_shave') or 0) > 0:
                    _c_gate_notes.append(f"SP HR/9 elevated ({float(_cr.get('_pick_hr9_val') or 0):.2f} L3), edge discounted {float(_cr.get('_pick_hr9_shave')):.1%}")
                if isinstance(_cr.get('_pick_post_il'), dict):
                    _c_gate_notes.append(f"SP thin post-IL sample, edge discounted {_cr.get('_pick_post_il', {}).get('edge_shave', 0):.1%}")
                if float(_cr.get('_pick_sp_health_shave') or 0) > 0:
                    _c_gate_notes.append(f"SP HEALTH UNCONFIRMED (pick's own SP {_cr.get('_pick_sp_health_val') or ''}), edge discounted {float(_cr.get('_pick_sp_health_shave')):.1%}")
                if float(_cr.get('_opp_sp_health_shave') or 0) > 0:
                    _c_gate_notes.append(f"SP HEALTH UNCONFIRMED (opp SP {_cr.get('_opp_sp_health_val') or ''}), edge discounted {float(_cr.get('_opp_sp_health_shave')):.1%}")
                if _c_gate_notes:
                    _c_tag += "  [" + "; ".join(_c_gate_notes) + "]"
                if not _c_official and not _c_noise:
                    _c_tag += "  | " + _mlb_ml_driver(_cr)
                _c_dc = (globals().get("mlb_direction_conflict_games", {}) or {}).get((_s_away.lower().strip(), _s_home.lower().strip()))
                if _c_dc:
                    _c_tag = f"  [!! {_c_dc}]" + _c_tag
                # RAW MARKET ODDS ON THE CARD (2026-09-19, user audit -- Rays
                # +26.1% edge with no way to tell which line that was calculated
                # against): home_ml/away_ml already ride along on every cand row
                # (merged from mlb_espn_odds earlier in this cell) -- just never
                # printed here. RL's own snapshot row already does this (mkt %
                # followed by the raw price in parens); matching that here.
                _c_pick_ml = _cr.get('home_ml') if str(_cr.get('bet_side', '')).lower() == 'home' else _cr.get('away_ml')
                _c_pick_ml_str = f"{int(_c_pick_ml):+d}" if pd.notna(_c_pick_ml) else "n/a"
                # BUG FIX (2026-09-27 user audit, priority 1): "print pre-
                # blend and post-blend edge on every play" -- lets a reader
                # run the exact counterfactual MLB-EDGE-BLEND-1 was built to
                # support (would this have fired as an official play under
                # the old math?) directly off the card, instead of having to
                # re-derive it. p_bet_win_model_raw (Cell 87) is the model's
                # own probability BEFORE the w=0.15 market blend -- same
                # market prob (_c_mkt) both ways, so the only thing that
                # differs is the blend itself. Same principle as every shave
                # print elsewhere on this card: the reduction is transparent
                # math, not a hidden cap.
                _c_model_preblend = _cr.get("p_bet_win_model_raw")
                _c_blend_note = ""
                if pd.notna(_c_model_preblend) and pd.notna(_c_mkt):
                    _c_edge_preblend = float(_c_model_preblend) - float(_c_mkt)
                    _c_blend_note = f"  [pre-blend model {float(_c_model_preblend):.1%}/edge {_c_edge_preblend:+.1%} -> post-blend edge {_c_edge:+.1%}]"
                _pr(f"  ML   {str(_cr.get('bet_team')):<22} model {float(_c_model):5.1%}  mkt {float(_c_mkt):5.1%} ({_c_pick_ml_str})  edge {_c_edge:+5.1%}{_c_blend_note}{_c_tag}")
                # (injury delta already priced into _c_model/_c_edge above --
                # _c_inj_note carries the before/after string for display below.)
                if not _c_noise:
                    _snap_rows.append({"matchup": _matchup, "market": "ML", "side": str(_cr.get("bet_team")),
                                        "away_team": _s_away, "home_team": _s_home,
                                        "edge": _c_edge, "official": bool(_c_official), "board_idx": _si,
                                        "game_id": (_sr.get(game_id_col) if game_id_col else None),
                                        "model_prob": float(_c_model), "market_prob": float(_c_mkt),
                                        "edge_preblend": (float(_c_model_preblend) - float(_c_mkt)) if pd.notna(_c_model_preblend) else None,
                                        "reason": _c_tag.strip(), "injury_adj_note": _c_inj_note,
                                        "price": _c_pick_ml_str,
                                        "book": (globals().get("mlb_ml_book_by_home") or {}).get(str(_s_home).lower().strip(), "ESPN feed")})
                _s_ml_printed = True
            elif pd.notna(_c_model):
                _pr(f"  ML   {str(_cr.get('bet_team')):<22} model {float(_c_model):5.1%}  mkt n/a (no odds loaded)")
                _s_ml_printed = True
        if not _s_ml_printed:
            _pr("  ML   no positive-edge side")
    else:
        _pr("  ML   no candidate rows found")

    # ---- Total (Over/Under runs) ----
    # BUG FIX (2026-08-10, user audit): this used to independently recompute
    # p_ov/p_un from board's raw pred_total_runs with a simple shrink-toward-
    # line formula -- the exact "bypasses every gate the real selector
    # applies" problem the RL section below was fixed for on 2026-07-27 (this
    # cell's own header comment claims "a clean read of the same data...so
    # this never disagrees with the real card", which was true for RL but,
    # until now, not for totals). The real O/U selector (Cell bd676982,
    # _ou_candidates_ranked / _ou_plays_out) applies totals-calibration
    # debiasing, thin-gap, hitter-park/hot-weather Under blocks, an elite-SP
    # Over gate, and an implausible-probability sanity gate that this
    # recompute skipped entirely -- so the displayed number and OFFICIAL tag
    # could genuinely disagree with the real card. Now reads the real
    # selector's own per-candidate records directly instead of recomputing.
    _s_ou_cands = [c for c in globals().get("_ou_candidates_ranked", [])
                   if str(c.get(away_col, "")) == _s_away and str(c.get(home_col, "")) == _s_home]
    if _s_ou_cands:
        _s_ou_line = float(_s_ou_cands[0].get("ou_line"))
        _s_pred_total = _s_ou_cands[0].get("pred_total_runs")
        _s_pred_cal = _s_ou_cands[0].get("pred_total_cal")
        _pr(f"  TOT  line {_s_ou_line:.1f}  model proj {float(_s_pred_total):.1f}  (calibrated {float(_s_pred_cal):.1f})")
        if _slate_ou_median is not None and (_s_ou_line - _slate_ou_median) >= _SLATE_OU_OUTLIER_MIN_GAP:
            _pr(f"  [SLATE OUTLIER] this total sits {_s_ou_line - _slate_ou_median:+.1f} runs above today's "
                f"{len(_slate_ou_lines)}-game slate median ({_slate_ou_median:.1f}) -- the market pricing in "
                f"an unusually high-scoring environment is itself worth weighing before trusting an Over/high-total read")
        # _ou_candidates_ranked only ever contains real-priced games -- synthetic/
        # ESPN-est lines are `continue`d in the real selector before ever being
        # appended, so no separate "is this real" check is needed here.
        _s_official_labels = {
            c.get("bet_team") for c in globals().get("_ou_plays_out", [])
            if str(c.get(away_col, "")) == _s_away and str(c.get(home_col, "")) == _s_home
        }
        # BUG FIX (2026-08-17, live-verified against the 2026-08-17 card -- real
        # case: Dodgers @ Rockies UNDER 10.5, +11.9% raw edge): _ou_plays_out gets
        # populated two structurally different ways -- (a) a game genuinely clears
        # OU_MIN_EDGE (18%) or the logit-space equivalent, or (b) NOTHING at all
        # cleared the bar today, so the "BEST AVAILABLE O/U" speculative fallback
        # (_OU_SPEC_MIN_EDGE = 0.10, see the O/U evaluation cell) promotes the
        # single closest near-miss into the same list, explicitly printing
        # "[BEST AVAILABLE O/U -- N near-miss speculative pick(s), below 18%
        # threshold]" right there -- but that entry already carries a
        # "speculative": True key (set when it was first built as an
        # _ou_candidates_ranked row) that this snapshot/final-card layer never
        # checked, so a fallback pick that explicitly did NOT clear the bar was
        # displaying as "[OFFICIAL-BAR EDGE]" / "[OFFICIAL]" on the actual card --
        # indistinguishable from a real bar-clear. Confirmed live: today's O/U
        # cell output literally printed "[BEST AVAILABLE O/U -- 1 near-miss
        # speculative pick(s), below 18% threshold]" for this exact game.
        _s_speculative_labels = {
            c.get("bet_team") for c in globals().get("_ou_plays_out", [])
            if c.get("speculative") and str(c.get(away_col, "")) == _s_away and str(c.get(home_col, "")) == _s_home
        }
        _s_ceiling_labels = {
            c.get("bet_team") for c in globals().get("_ou_edge_ceiling_watch", [])
            if str(c.get(away_col, "")) == _s_away and str(c.get(home_col, "")) == _s_home
        }
        _s_ou_shown = False
        for _c in _s_ou_cands:
            _c_edge = float(_c.get("edge", 0) or 0)
            if _c_edge <= 0:
                continue  # mirror side -- only the positive-edge side is shown
            _c_label = _c.get("bet_team", "")
            # SUPPRESS SPECULATIVE FALLBACK PLAYS (2026-09-19, user audit):
            # a play that exists ONLY because nothing cleared the real O/U bar
            # today (the 2026-08-29 "always-evaluated LEAN tier") used to still
            # print here (and, via _snap_rows below, in OTHER EDGES BELOW THE
            # BAR too) with a [SPECULATIVE FALLBACK] caveat -- user's call: even a
            # clearly-labeled fallback creates selection pressure to act on it:
            # the correct output when nothing real cleared the bar is "no
            # qualifying O/U play," not a caveated one. `continue` here (skips
            # both this print AND the _snap_rows.append below it) rather than
            # only swapping the tag text, so it's gone from every section that
            # reads _snap_rows, not just this one. Does not touch _ou_plays_out/
            # the LEAN tier's own EKO BETS O/U section -- that stays exactly as
            # before (speculative=True, excluded from grading), this only stops
            # it from ALSO surfacing on the card as if it were a real edge.
            if _c_label in _s_official_labels and _c_label in _s_speculative_labels:
                continue
            _s_ou_shown = True
            _c_side_disp = "Over " if _c.get("bet_side") == "over" else "Under"
            _c_model = float(_c.get("p_bet_win_model", 0) or 0)
            _c_mkt = float(_c.get("market_implied_prob", 0) or 0)
            if _c_edge < _SNAP_MIN_EDGE:
                _c_tag = f"  [noise -- below {_SNAP_MIN_EDGE:.0%} floor]"
            elif _c_label in _s_official_labels:
                _c_tag = "  [OFFICIAL-BAR EDGE]"
            elif _c_label in _s_ceiling_labels:
                _c_tag = f"  [edge {_c_edge:+.1%} >= 25% ceiling -- blocked for calibration review, not auto-played]"
            else:
                _c_tag = _mlb_ou_rule_tag(_c)
            if _c_label not in _s_official_labels and _c_edge >= _SNAP_MIN_EDGE:
                try:
                    _c_tag += "  | " + _mlb_ou_driver(_c)
                except (TypeError, ValueError):
                    pass
            # RAW MARKET ODDS ON THE CARD (2026-09-19, user audit): ou_price_str
            # (the actual over/under juice, or "-110 (est)" when ESPN never gave
            # a real market price) is already built into every _ou_candidates_ranked
            # row for the EKO BETS O/U section -- just never made it onto this
            # snapshot line, same gap as the ML row above.
            _c_ou_price = _c.get("ou_price_str", "--")
            # NOTE (2026-09-27 user audit): MLB-EDGE-BLEND-1's market blend
            # is ML-only (no graded O/U history existed to fit a blend
            # weight against) -- this edge is NOT blended, pre==post here.
            # Flagged explicitly rather than silently omitted, so "print
            # pre/post on every play" is honestly answered for this market
            # too instead of just not appearing.
            _pr(f"       {_c_side_disp} model {_c_model:5.1%}  mkt {_c_mkt:5.1%} ({_c_ou_price})  edge {_c_edge:+5.1%}  [not blended -- ML-only]{_c_tag}")
            if _c_edge >= _SNAP_MIN_EDGE:
                _snap_rows.append({"matchup": _matchup, "market": f"O/U {_s_ou_line:.1f}", "side": _c_side_disp.strip(),
                                    "away_team": _s_away, "home_team": _s_home,
                                    "game_id": (_sr.get(game_id_col) if game_id_col else None),
                                    "edge": _c_edge,
                                    "official": (_c_label in _s_official_labels) and (_c_label not in _s_speculative_labels) and not _snap_sp_conflict and not _snap_no_sp_data,
                                    "board_idx": _si,
                                    "model_prob": _c_model, "market_prob": _c_mkt,
                                    "price": (_c.get("odds_str") or (_c.get("ou_price_str", "").split("/")[0 if _c.get("bet_side") == "over" else -1]
                                              if _c.get("ou_price_str") else None)),
                                    "book": _c.get("ou_book") or _sr.get("ou_book"),
                                    "reason": _c_tag.strip()})
        if not _s_ou_shown:
            _pr("       no positive-edge side")
    else:
        # No real record from the selector -- no real market line (synthetic/
        # ESPN-est), line_mismatch_flag suppressed it, ou_line<4, or the O/U
        # selector block never ran at all. Falls back to a clearly-labeled
        # rough estimate rather than going silent -- this is NOT the same
        # calculation as the real card and can't be trusted the same way.
        _s_ou_line = _sr.get("over_under")
        _s_pred_total = _sr.get("pred_total_runs")
        if pd.notna(_s_ou_line) and pd.notna(_s_pred_total) and float(_s_ou_line) >= 4:
            if not bool(_sr.get("has_odds")):
                _s_ou_reason = "no market odds loaded for this game at all"
            elif str(_sr.get("ou_book", "espn_est")) == "espn_est":
                _s_ou_reason = "ESPN-estimated total, not a real sportsbook line"
            elif bool(_sr.get("line_mismatch_flag", False)):
                _s_ou_reason = "line mismatch flagged at load time -- see [LINE MISMATCH] print above"
            else:
                _s_ou_reason = "gate-skipped upstream by the real O/U selector for a reason not captured by these checks"
            _pr(f"  TOT  line {float(_s_ou_line):.1f}  model proj {float(_s_pred_total):.1f}  "
                f"[ROUGH ESTIMATE -- {_s_ou_reason}; this is NOT the real card's number]")
        else:
            _pr("  TOT  no total line / no model total available")

    # ---- Run line (spread, -1.5/+1.5) ----
    # 2026-07-27 rewrite: this used to recompute RL probability/edge from
    # scratch via _poisson_win_by2 + _fetch_rl_odds, bypassing every gate the
    # real RL selector above applies (RL_MIN_PRED_DIF margin gate, park factor,
    # bullpen availability, closer-risk discount, juice cap, odds sanity
    # check). Real incident 2026-07-26: Pittsburgh Pirates -1.5 (Cubs @
    # Pirates) printed here as an "[OFFICIAL-BAR EDGE]" play at 49.5% model /
    # +12.4% edge, but the real selector's RL_MIN_PRED_DIF=2.0 gate means it
    # never even considered this game (projected margin was only 1.5R) -- the
    # two sections disagreed on whether a real qualifying play existed. Now
    # reads _rl_eval directly (the real selector's own per-team result dict)
    # instead of recomputing, so this genuinely can't disagree with the real
    # card anymore, per this cell's own stated purpose above.
    _s_ph = float(_sr.get("pred_home_runs") or 0)
    _s_pa = float(_sr.get("pred_away_runs") or 0)
    if _s_ph > 0 and _s_pa > 0:
        _s_rl_team = _s_home if _s_ph >= _s_pa else _s_away
        _s_rl_rec = globals().get("_rl_eval", {}).get(_s_rl_team.lower())
        if not _s_rl_rec:
            _s_min_dif = globals().get("RL_MIN_PRED_DIF", 2.0)
            _pr(f"  RL   {_s_rl_team} -1.5  not evaluated (projected margin below {_s_min_dif:.1f}R minimum)")
        elif _s_rl_rec.get("status") == "qualified":
            _s_rl_model = _s_rl_rec.get("model_prob")
            _s_rl_edge = _s_rl_rec.get("edge") or 0.0
            _s_rl_odds = _s_rl_rec.get("rl_odds")
            _s_rl_mkt = _ml_to_prob(_s_rl_odds) if _s_rl_odds else float("nan")
            if _s_rl_edge < _SNAP_MIN_EDGE:
                _rl_tag = f"  [noise -- below {_SNAP_MIN_EDGE:.0%} floor]"
            elif _s_rl_edge >= _SNAP_RL_BAR:
                _rl_tag = "  [OFFICIAL-BAR EDGE]"
            else:
                _rl_tag = "  [below official RL bar, informational only]"
            # NOTE (2026-09-27 user audit): same as the O/U line above --
            # RL has no market blend applied (ML-only), pre==post here.
            _pr(f"  RL   {_s_rl_team} -1.5  model {_s_rl_model:5.1%}  mkt {_s_rl_mkt:5.1%} ({_s_rl_odds})  edge {_s_rl_edge:+5.1%}  [not blended -- ML-only]{_rl_tag}")
            if _s_rl_edge >= _SNAP_MIN_EDGE:
                _snap_rows.append({"matchup": _matchup, "market": f"{_s_rl_team} -1.5", "side": _s_rl_team,
                                    "away_team": _s_away, "home_team": _s_home,
                                    "game_id": (_sr.get(game_id_col) if game_id_col else None),
                                    "edge": _s_rl_edge, "official": (_s_rl_edge >= _SNAP_RL_BAR) and not _snap_sp_conflict and not _snap_no_sp_data, "board_idx": _si,
                                    "model_prob": _s_rl_model, "market_prob": _s_rl_mkt,
                                    "price": _s_rl_odds, "book": "ESPN pickcenter",
                                    "reason": _rl_tag.strip()})
        else:
            _s_reason = _s_rl_rec.get("reason", _s_rl_rec.get("status", "not qualified"))
            _pr(f"  RL   {_s_rl_team} -1.5  not qualified -- {_s_reason}")
    else:
        _pr("  RL   no run projection available")

    # MLB-PS-SPROLE-ALL-1 (2026-10-07 user request -- "Piggyback / SP role UNRATED flag. Apply it to every
    # market on the game, not just the total"). Gate 7d (postseason SP role: announced opener / elimination-
    # game piggyback) and Gate 7f (SP TBD) were applied per market: the O/U selector printed them, but the ML
    # gate (Cell 87) only touched rows still passing the edge rule -- 10-06 MIL @ SD: the total showed "Gate 7d
    # ... PIGGYBACK RISK", the Padres ML below it showed only its logit-bar miss, and RL never checked at all.
    # The innings split is unknowable for EVERY market on the game, so every snapshot row on it now carries the
    # flag and none can be OFFICIAL.
    _g_key = (_s_away.lower().strip(), _s_home.lower().strip())
    _g_unr = [(_gl_lab, _gl_txt) for _gl_lab, _gl_txt in (
        ("Gate 7d postseason SP role UNRATED", (globals().get("mlb_ps_role_unrated") or {}).get(_g_key)),
        ("Gate 7f", (globals().get("mlb_sp_tbd_unrated") or {}).get(_g_key))) if _gl_txt]
    if _g_unr:
        for _ur in _snap_rows[_rows_before:]:
            _ur["official"] = False
            _ur_add = [f"{_gl_lab} -- {_gl_txt}" for _gl_lab, _gl_txt in _g_unr if " ".join(_gl_lab.split()[:2]) not in str(_ur.get("reason") or "")]
            if _ur_add:
                _ur["reason"] = ("[" + "; ".join(_ur_add) + "]  " + str(_ur.get("reason") or "")).strip()
        _pr("  !! SP ROLE UNRATED -- applies to EVERY market on this game (ML / RL / total), none can be OFFICIAL: "
            + "; ".join(f"{_gl_lab} -- {_gl_txt}" for _gl_lab, _gl_txt in _g_unr))

    if len(_snap_rows) > _rows_before:
        _snap_games_shown += 1
        print(f"\n{_matchup}")
        for _gl in _game_lines:
            print(_gl)

print(f"\n({_snap_games_shown}/{_snap_games_scanned} games shown above -- the rest had no market clear even the "
      f"{_SNAP_MIN_EDGE:.0%} noise floor on ML/TOT/RL, omitted from this snapshot)")

print("\n" + "=" * 78)
print("OFFICIAL QUALIFYING PLAYS TODAY")
print("=" * 78)
# ---------------------------------------------------------------------------
# MLB-TOT-LOG-1 / MLB-SERIES-TOT-1 / MLB-PS-TOT-TRACK-1 (2026-10-04 user audit)
#  * every positive-edge totals lean on the card is logged (tracking/mlb_totals_leans.csv; one row
#    per date + matchup + side, the latest pregame run wins), graded from Stats API finals;
#  * "Extend MLB-SERIES-1 to totals. It flags repeat team ML plays but not a repeat Over in the same
#    series": a lean on the SAME side of the same matchup within the last 4 days -> [SERIES REPEAT]
#    with the ML repeat's own penalty (REPEAT edge -2.0pp, -3.5pp from the 2nd repeat), shown on the
#    card, never a block;
#  * "The -0.40R postseason recenter was first tested today, so track its results in elite-SP games
#    separately": postseason rows carry the shift applied and the elite-starter count; the summary
#    below splits actual-vs-projection with and without the shift by elite / non-elite.
# ---------------------------------------------------------------------------
import csv as _tl_csv, datetime as _tl_dt
_tl_out = []   # printed at the END of this cell, so it never lands inside the OFFICIAL section
from pathlib import Path as _tl_P
_TL_PATH = _tl_P(r"c:\Users\<user>\OneDrive\Documents\KAI Labs\sports\MLB\tracking\mlb_totals_leans.csv")
_TL_FIELDS = ["card_date", "run_time", "matchup", "away", "home", "side", "line", "price", "book", "edge", "model_prob",
              "proj_component", "proj_old", "proj_cal", "is_ps", "ps_shift", "elite_n", "official", "gates", "source",
              "actual_total", "result"]
_tl_rows = []
try:
    if _TL_PATH.exists():
        with open(_TL_PATH, newline="", encoding="utf-8") as _tlf:
            _tl_rows = list(_tl_csv.DictReader(_tlf))
except Exception as _tle:
    _tl_out.append(f"  [TOTALS LOG] could not read {_TL_PATH.name} ({_tle})")
_tl_today = str(globals().get("CARD_DATE", _tl_dt.date.today().isoformat()))

# MLB-SERIES-GRADE-1 (2026-10-08 user audit -- "It showed yesterday's CLE/CWS U8.0 as pending after it had lost 9-3.
# The lookup should read graded results"): the repeat check below read the log as written by the PREVIOUS run, and
# finished games were only graded at the end of this cell -- after the check. The 10-07 Under 8.0 (12 runs, L) was
# graded by the 10-08 run itself, a few lines too late. Grading now runs first (and again before the write, for any
# game that went final in between).
def _tl_grade(rows):
    import requests as _tl_req
    _fin = {}
    for _d in sorted({r["card_date"] for r in rows if not r.get("result")}):
        _js = _tl_req.get("https://statsapi.mlb.com/api/v1/schedule", params={"sportId": 1, "date": _d}, timeout=15).json()
        for _dd_ in _js.get("dates", []):
            for _g in _dd_.get("games", []):
                if _g["status"].get("detailedState") == "Final":
                    _t = _g["teams"]
                    _fin[(_d, _t["away"]["team"]["name"].lower(), _t["home"]["team"]["name"].lower())] = _t["away"]["score"] + _t["home"]["score"]
    for r in rows:
        if r.get("result"):
            continue
        _a = _fin.get((r["card_date"], r["away"].lower(), r["home"].lower()))
        if _a is None:
            continue
        r["actual_total"] = _a
        if r.get("line") not in ("", None):
            _ln = float(r["line"])
            r["result"] = "PUSH" if _a == _ln else ("W" if (_a > _ln) == (r["side"].lower() == "over") else "L")
        # no recorded line (seeded rows): actual stored, result left as recorded at seeding
try:
    _tl_grade(_tl_rows)
except Exception as _tle:
    _tl_out.append(f"  [TOTALS LOG] pre-check grading failed ({_tle}) -- series-repeat history may show 'pending'")
# series repeat check against PRIOR days only
_tl_rep = {}
for _sr_i, _s_row in enumerate(_snap_rows):
    if not str(_s_row.get("market", "")).startswith("O/U"):
        continue
    _pair = frozenset((str(_s_row["away_team"]).lower(), str(_s_row["home_team"]).lower()))
    _prev = [r for r in _tl_rows if r.get("card_date", "") < _tl_today and r.get("side", "").lower() == str(_s_row["side"]).lower()
             and frozenset((r.get("away", "").lower(), r.get("home", "").lower())) == _pair
             and (_tl_dt.date.fromisoformat(_tl_today) - _tl_dt.date.fromisoformat(r["card_date"])).days <= 4]
    if _prev:
        _pen = 0.020 + (0.015 if len(_prev) >= 2 else 0.0)
        _hist = "; ".join(f"{r['card_date']} {r['side']} {r.get('line') or '?'} -> "
                          + (f"{r.get('actual_total')} runs, {r.get('result')}" if r.get("result") else "pending") for r in _prev)
        _s_row["edge"] = float(_s_row["edge"]) - _pen
        _s_row["reason"] = (f"[SERIES REPEAT -{_pen * 100:.1f}pp: same side in this series within 4 days -- {_hist}]  "
                            + str(_s_row.get("reason", "")))
        _tl_rep[(_s_row["matchup"], _s_row["side"])] = _hist
        _tl_out.append(f"  [SERIES REPEAT] {_s_row['matchup']} {_s_row['side']}: {_hist} -- edge -{_pen * 100:.1f}pp (MLB-SERIES-TOT-1)")
        if _s_row.get("official") and float(_s_row["edge"]) < float(globals().get("OU_MIN_EDGE", 0.18)):
            _s_row["official"] = False
            _s_row["reason"] = "[demoted: below the O/U bar after the series-repeat penalty]  " + _s_row["reason"]
# upsert today's leans
_tl_cands = {(c.get("matchup"), c.get("bet_side")): c for c in globals().get("_ou_candidates_ranked", [])}
for _s_row in _snap_rows:
    if not str(_s_row.get("market", "")).startswith("O/U"):
        continue
    _c = _tl_cands.get((_s_row["matchup"], str(_s_row["side"]).lower()), {})
    _dk = (str(_s_row["away_team"]).lower().strip(), str(_s_row["home_team"]).lower().strip())
    _dd = (globals().get("mlb_totals_decomp") or {}).get(_dk) or {}
    _new = {"card_date": _tl_today, "run_time": _tl_dt.datetime.now().isoformat(timespec="minutes"), "matchup": _s_row["matchup"],
            "away": _s_row["away_team"], "home": _s_row["home_team"], "side": str(_s_row["side"]).title(),
            "line": _c.get("ou_line"), "price": _s_row.get("price"), "book": _s_row.get("book"),
            "edge": round(float(_s_row["edge"]), 4), "model_prob": round(float(_s_row.get("model_prob") or 0), 4),
            "proj_component": round(float(_dd["total"]), 2) if _dd else "", "proj_old": _c.get("pred_total_old", ""),
            "proj_cal": _c.get("pred_total_cal", ""), "is_ps": bool(_dd.get("is_ps")) if _dd else "",
            "ps_shift": (globals().get("_TC1") or {}).get("ps_run_env_delta", -0.38) if _dd.get("is_ps") else 0.0,
            "elite_n": _dd.get("elite_n", ""), "official": bool(_s_row.get("official")),
            "gates": "; ".join(_c.get("_ou_gate_names") or []), "source": "card", "actual_total": "", "result": ""}
    _tl_rows = [r for r in _tl_rows if not (r.get("card_date") == _tl_today and r.get("matchup") == _new["matchup"]
                                             and r.get("side", "").lower() == _new["side"].lower())]
    _tl_rows.append(_new)
# grade finished games (MLB-SERIES-GRADE-1: same function the pre-check pass used)
try:
    _tl_grade(_tl_rows)
    with open(_TL_PATH, "w", newline="", encoding="utf-8") as _tlf:
        _w = _tl_csv.DictWriter(_tlf, fieldnames=_TL_FIELDS, extrasaction="ignore")
        _w.writeheader()
        _w.writerows(_tl_rows)
except Exception as _tle:
    _tl_out.append(f"  [TOTALS LOG] grading/write failed ({_tle})")
# postseason recenter tracker summary
_ps_g = [r for r in _tl_rows if str(r.get("is_ps")) == "True" and r.get("actual_total") not in ("", None)
         and r.get("proj_component") not in ("", None)]
_tl_out.append("\n" + "-" * 78)
_tl_out.append("POSTSEASON TOTALS TRACKER (MLB-PS-TOT-TRACK-1) -- component projection incl. the PS shift vs actual runs")
if not _ps_g:
    _tl_out.append("  no graded postseason component-projection rows yet (started 2026-10-04)")
else:
    for _lab, _sel in (("elite-SP games (>= 1 starter season ERA < 3.00)", lambda r: str(r.get("elite_n")) not in ("", "0")),
                       ("non-elite games", lambda r: str(r.get("elite_n")) in ("0",))):
        _xs = {}
        for r in _ps_g:
            if _sel(r):
                _xs[(r["card_date"], r["matchup"])] = r
        _xs = list(_xs.values())
        if not _xs:
            _tl_out.append(f"  {_lab}: n=0")
            continue
        _res = [float(r["actual_total"]) - float(r["proj_component"]) for r in _xs]
        _res_ns = [float(r["actual_total"]) - (float(r["proj_component"]) - float(r.get("ps_shift") or 0)) for r in _xs]
        _ov = [r for r in _xs if r["side"] == "Over" and r.get("result") in ("W", "L")]
        _tl_out.append(f"  {_lab}: n={len(_xs)} games | actual - proj {sum(_res) / len(_res):+.2f}R with the PS shift, "
              f"{sum(_res_ns) / len(_res_ns):+.2f}R without | Over leans {sum(1 for r in _ov if r['result'] == 'W')}-"
              f"{sum(1 for r in _ov if r['result'] == 'L')}")
_seed = [r for r in _tl_rows if r.get("source") == "seed"]
if _seed:
    _tl_out.append("  (seeded, pre-tracker: " + "; ".join(f"{r['card_date']} {r['matchup']} {r['side']} -> {r.get('actual_total')} runs {r.get('result')}" for r in _seed) + ")")

# SHADOW-1 (2026-10-05): every play + below-bar row -> tracking/shadow_log.csv (model prob, no-vig
# market prob, card-time line/price/book; scripts/shadow_close.py adds the close + result later).
# Tier mirrors the card: the top bar-clearing row is the day's OFFICIAL play (1/sport/day), the
# other bar-clearing rows are QUALIFYING POOL, everything else BELOW BAR.
try:
    import sys as _shs, importlib as _shi
    if r"C:\Users\<user>\OneDrive\Documents\KAI Labs\scripts" not in _shs.path:
        _shs.path.insert(0, r"C:\Users\<user>\OneDrive\Documents\KAI Labs\scripts")
    import shadow_log as _shadow_log
    _shi.reload(_shadow_log)
    _sh_off = sorted([r for r in _snap_rows if r.get("official")], key=lambda r: -float(r["edge"]))
    _sh_top = id(_sh_off[0]) if _sh_off else None
    _shadow_rows = []
    for _r in _snap_rows:
        _mk = str(_r.get("market", ""))
        _gt = None
        try:
            _gt = str(board.loc[_r.get("board_idx"), "game_datetime_utc"]) if "game_datetime_utc" in board.columns else None
        except Exception:
            pass
        _nv = _r.get("market_prob")
        if _mk.startswith("O/U"):
            _mkt, _ln, _sd = "TOTAL", float(_mk.split()[-1]), str(_r["side"]).title()
        elif _mk.endswith("-1.5"):
            _mkt, _ln, _sd = "RL", -1.5, str(_r["side"])
            try:
                _rlh, _rla = _fetch_rl_odds(str(_r["home_team"]).lower())
                _pa, _pb = _shadow_log.novig_two(_rlh, _rla)
                if _pa is not None:
                    _nv = _pa if str(_r["side"]) == str(_r["home_team"]) else _pb
            except Exception:
                pass
        else:
            _mkt, _ln, _sd = "ML", None, str(_r["side"])
        _shadow_rows.append({
            "card_date": str(globals().get("CARD_DATE", "")), "game_id": _r.get("game_id"), "game_time": _gt,
            "matchup": _r["matchup"], "away": _r["away_team"], "home": _r["home_team"], "market": _mkt, "side": _sd,
            "line": _ln, "price": _r.get("price"), "book": _r.get("book"), "model_prob": _r.get("model_prob"),
            "market_novig": _nv, "edge": _r["edge"], "official": id(_r) == _sh_top,
            "tier": "OFFICIAL" if id(_r) == _sh_top else ("QUALIFYING POOL" if _r.get("official") else "BELOW BAR"),
            "rule": _r.get("reason", "")})
    _sh_res = _shadow_log.log_rows("MLB", _shadow_rows, verbose=False)
    globals().setdefault("_tl_out", []).append(f"  [SHADOW LOG] MLB: {_sh_res[0]} new, {_sh_res[1]} updated, "
                                               f"{_sh_res[2]} skipped (game already started) -> tracking/shadow_log.csv")
    # SHADOW-2 (2026-10-05 user request -- "For every game on every card (all sports, all markets
    # priced), log one row per game per market ... fixed side (home team for ML/spread, Over for
    # totals)"): full-slate rows for the model-vs-market Brier. ML = cand's HOME row for every game
    # with a real price; TOTAL = the Over candidate of every real-priced total; RL = every game the
    # run-line selector priced (its model_prob is for the projected winner at -1.5 -> home side =
    # p or 1-p, home line -1.5 or +1.5), de-vigged from both run-line prices.
    _sh_full = []
    _sh_cd = str(globals().get("CARD_DATE", ""))
    _sh_gt = {}
    try:
        for _bi, _br in board.iterrows():
            _sh_gt[(str(_br.get(away_col)), str(_br.get(home_col)))] = str(_br.get("game_datetime_utc")) if "game_datetime_utc" in board.columns else None
    except Exception:
        pass
    for _, _cr in cand.iterrows():
        if str(_cr.get("bet_side", "")).lower() != "home" or not bool(_cr.get("game_has_odds", True)):
            continue
        _a, _h = str(_cr.get(away_col)), str(_cr.get(home_col))
        if not (_cr.get("market_novig_prob") == _cr.get("market_novig_prob")) or _cr.get("market_novig_prob") is None:
            continue
        _sh_full.append({"card_date": _sh_cd, "game_id": _cr.get(game_id_col) if game_id_col else None,
                         "game_time": _sh_gt.get((_a, _h)), "matchup": f"{_a} @ {_h}", "away": _a, "home": _h,
                         "market": "ML", "side": _h, "line": None, "price": _cr.get("home_ml"),
                         "book": (globals().get("mlb_ml_book_by_home") or {}).get(_h.lower().strip(), "ESPN feed"),
                         "model_prob": _cr.get("p_bet_win_model"), "market_novig": _cr.get("market_novig_prob"),
                         "edge": _cr.get("edge")})
    for _oc in globals().get("_ou_candidates_ranked", []):
        if _oc.get("bet_side") != "over":
            continue
        _a, _h = str(_oc.get(away_col)), str(_oc.get(home_col))
        _px = str(_oc.get("ou_price_str", "")).split("/")[0] if _oc.get("ou_price_str") else None
        _sh_full.append({"card_date": _sh_cd, "game_time": _sh_gt.get((_a, _h)), "matchup": f"{_a} @ {_h}", "away": _a, "home": _h,
                         "market": "TOTAL", "side": "Over", "line": _oc.get("ou_line"), "price": _px, "book": _oc.get("ou_book"),
                         "model_prob": _oc.get("p_bet_win_model"), "market_novig": _oc.get("market_implied_prob"),
                         "edge": _oc.get("edge"), "proj_total": _oc.get("pred_total_runs")})
    try:
        for _bi, _br in board.iterrows():
            _a, _h = str(_br.get(away_col)), str(_br.get(home_col))
            for _tm in (_h, _a):
                _rv = (globals().get("_rl_eval") or {}).get(_tm.lower())
                if not _rv or _rv.get("model_prob") is None:
                    continue
                _rlh, _rla = _fetch_rl_odds(_h.lower())
                _nh, _na = _shadow_log.novig_two(_rlh, _rla)
                if _nh is None:
                    continue
                _ph = float(_rv["model_prob"]) if _tm == _h else 1.0 - float(_rv["model_prob"])
                _sh_full.append({"card_date": _sh_cd, "game_time": _sh_gt.get((_a, _h)), "matchup": f"{_a} @ {_h}",
                                 "away": _a, "home": _h, "market": "RL", "side": _h, "line": -1.5 if _tm == _h else 1.5,
                                 "price": _rlh, "book": "ESPN pickcenter", "model_prob": _ph, "market_novig": _nh,
                                 "edge": _ph - _nh})
                break
    except Exception as _shrl:
        print(f"  [SHADOW FULL-SLATE] MLB RL rows skipped ({_shrl})")
    _sh_fres = _shadow_log.log_full("MLB", _sh_full, verbose=False)
    globals().setdefault("_tl_out", []).append(f"  [SHADOW FULL-SLATE] MLB: {_sh_fres[0]} new, {_sh_fres[1]} updated, "
                                               f"{_sh_fres[2]} skipped (game already started) -> tracking/shadow_fullslate.csv")
except Exception as _she:
    print(f"  [SHADOW LOG] MLB logging failed ({_she})")
_snap_df = pd.DataFrame(_snap_rows) if _snap_rows else pd.DataFrame(columns=["matchup", "market", "side", "edge", "official"])
_snap_official = _snap_df[_snap_df.get("official", False) == True] if len(_snap_df) else _snap_df
# 2026-08-01 fix: card rule is 1 play/sport/day (see project memory -- manual
# overlay process), but multiple games can independently clear the official
# bar the same day (real case: Giants/Pirates/Braves all cleared it
# 2026-08-01) and this used to print every one of them tagged as if each were
# its own standalone OFFICIAL play, with no ranking and no card-rule
# reminder. Rank by edge (reused below by FINAL CARD -- OFFICIAL PLAYS + WHY
# so both sections agree) and only tag the top play OFFICIAL; the rest are a
# labeled qualifying pool that still requires manual single-play selection.
# MLB-INJ-CONF-1 (2026-09-14): sorts on plain edge -- the ML append site above
# now prices any real net injury delta directly into that edge, so this needs
# no separate injury-aware sort key any more; a real tailwind/headwind can
# still change which play wins the day's single slot, it just does so through
# the same edge column every other market already sorts on.
# FULL-POWER-1 (2026-10-06, user chose "lift withholding gates"): active='full' -> every play that cleared
# the official bar is OFFICIAL; the 1-play/sport/day rule no longer relabels the rest "QUALIFYING POOL".
try:
    _MLB_FP = (lambda: __import__("json").load(open(r"C:\Users\<user>\OneDrive\Documents\KAI Labs\sports\model_blend_weights.json", encoding="utf-8")).get("active") == "full")()
except Exception:
    _MLB_FP = False
_snap_ranked = (_snap_official.sort_values("edge", ascending=False).reset_index(drop=True)
                if len(_snap_official) else _snap_official)
if len(_snap_ranked):
    if len(_snap_ranked) > 1 and _MLB_FP:
        print(f"  [FULL POWER] {len(_snap_ranked)} plays cleared the official bar today -- all OFFICIAL (1-play/sport rule lifted)")
    elif len(_snap_ranked) > 1:
        print(f"  NOTE: {len(_snap_ranked)} plays cleared the official bar today, but this card's")
        print("  rule caps at 1 play/sport/day (manual overlay process). Only the top-edge")
        print("  play below is tagged OFFICIAL -- the rest are a qualifying pool for manual")
        print("  review, not simultaneous plays.")
    for _oi, _row in _snap_ranked.iterrows():
        _oi_tag = "OFFICIAL" if (_oi == 0 or _MLB_FP) else "QUALIFYING POOL -- subject to 1-play/sport card rule"
        print(f"  {_row['edge']:+6.1%}  {_row['market']:<12} {_row['side']:<22} {_row['matchup']}   [{_oi_tag}]")
        if _row.get("injury_adj_note"):
            print(_row["injury_adj_note"])
else:
    print("  NONE -- no game/market cleared the model's official-play bar today.")
    print("  Recommended action: skip today, no bets. (A big raw edge shown above")
    print("  without this tag is not a substitute for a qualifying play.)")

# =========================================================
# DAILY PLAY CANDIDATE ANALYSIS -- MLB-DAILYPLAY-1 (2026-09-14, user spec:
# "What The Daily Play Should Actually Look Like"). INFORMATIONAL ONLY, per
# explicit user decision -- does NOT touch _snap_ranked's sort key,
# _c_official, _f_tag, or _log_official_play. Runs a separate, parallel
# selection methodology over the exact same _snap_official pool the real
# card already draws its OFFICIAL pick from, and prints/logs a side-by-side
# comparison. Graduation plan (user's own numbers, 2026-09-14): after 20
# real graded plays, if this methodology beats raw-edge on disagreement
# nights by >=60%, flip it to being the real selector; if it's a wash or
# worse, extend the informational window another 20 and revisit the factor
# list below.
# =========================================================
print("\n" + "=" * 78)
print("DAILY PLAY CANDIDATE ANALYSIS (INFORMATIONAL -- 20-PLAY GRADUATION WINDOW)")
print("=" * 78)

import csv as _dp_csv
from pathlib import Path as _DP_Path

_DP_SNAPSHOT_PATH = _DP_Path(r"c:\Users\<user>\OneDrive\Documents\KAI Labs\sports\MLB\tracking\mlb_line_snapshots.csv")
_DP_DISAGREE_PATH = _DP_Path(r"c:\Users\<user>\OneDrive\Documents\KAI Labs\sports\MLB\tracking\mlb_daily_play_disagreements.csv")
_DP_SNAPSHOT_FIELDS = ["snapshot_date", "snapshot_time", "game_id", "matchup", "market", "side", "market_prob", "model_prob", "edge"]
_DP_DISAGREE_FIELDS = [
    "card_date", "disagreement_type",
    "raw_edge_side", "raw_edge_market", "raw_edge_value", "raw_edge_game_id", "raw_edge_matchup",
    "four_factor_side", "four_factor_market", "four_factor_score", "four_factor_factors",
    "four_factor_game_id", "four_factor_matchup",
    "line_movement_note", "result", "margin",
]

_dp_today = pd.Timestamp.now().strftime("%Y-%m-%d")
_dp_now_time = pd.Timestamp.now().strftime("%H:%M:%S")

# ---- Step 1: snapshot today's qualifying-pool prices. Self-tracked, not the
# Odds API's own historical/closing-line endpoint -- confirmed live
# (2026-09-14) that requires a paid plan this project doesn't have
# (HISTORICAL_UNAVAILABLE_ON_FREE_USAGE_PLAN). Accumulating our own snapshots
# across the day's runs is a free substitute for "has the line moved toward
# this pick since the model first saw it today". ----
if len(_snap_official):
    _dp_new_snapshots = []
    for _dpi, _dprow in _snap_official.iterrows():
        _dp_new_snapshots.append({
            "snapshot_date": _dp_today, "snapshot_time": _dp_now_time,
            "game_id": _dprow.get("game_id"), "matchup": _dprow.get("matchup"),
            "market": _dprow.get("market"), "side": _dprow.get("side"),
            "market_prob": _dprow.get("market_prob"), "model_prob": _dprow.get("model_prob"),
            "edge": _dprow.get("edge"),
        })
    try:
        _dp_snap_is_new = not _DP_SNAPSHOT_PATH.exists()
        with open(_DP_SNAPSHOT_PATH, "a", newline="", encoding="utf-8") as _dp_sf:
            _dp_sw = _dp_csv.DictWriter(_dp_sf, fieldnames=_DP_SNAPSHOT_FIELDS)
            if _dp_snap_is_new:
                _dp_sw.writeheader()
            for _dprow2 in _dp_new_snapshots:
                _dp_sw.writerow(_dprow2)
        print(f"  Logged {len(_dp_new_snapshots)} price snapshot(s) for today's qualifying pool")
    except Exception as _dp_snap_e:
        print(f"  [WARN] snapshot logging failed: {_dp_snap_e}")


def _dp_line_movement(game_id, market, side):
    """Compares today's EARLIEST snapshot for this (game, market, side) to
    the latest one. Returns (moved_toward_pick: bool or None, note: str) --
    None means not enough snapshots yet (first pull of the day), a real,
    common answer, not an error.

    LIMITATION (2026-09-15, user review, flagged explicitly per their
    request): this is a same-day, intraday window -- earliest pull today to
    most recent pull today -- not true opening-to-closing-line CLV, which
    would need a price captured when the play first became eligible
    (potentially the day before) compared against the price right before
    first pitch. The Odds API's real historical/closing-line endpoint
    confirmed live (2026-09-14) to require a paid plan this project doesn't
    have -- this self-tracked snapshot log is the free substitute, but it's
    reading a compressed window and should not be treated as equivalent
    resolution to true opening-to-close CLV when reviewing the 20-play
    graduation window. Directionally useful, not a precise CLV number.
    """
    if not _DP_SNAPSHOT_PATH.exists() or game_id is None:
        return None, "no snapshot history yet"
    try:
        _dp_hist = pd.read_csv(_DP_SNAPSHOT_PATH)
        _dp_hist = _dp_hist[(_dp_hist["snapshot_date"] == _dp_today) & (_dp_hist["game_id"].astype(str) == str(game_id))
                             & (_dp_hist["market"] == market) & (_dp_hist["side"] == side)]
        if len(_dp_hist) < 2:
            return None, "only 1 snapshot today -- can't measure movement yet"
        _dp_hist = _dp_hist.sort_values("snapshot_time")
        _dp_first_prob = float(_dp_hist.iloc[0]["market_prob"])
        _dp_last_prob = float(_dp_hist.iloc[-1]["market_prob"])
        _dp_moved_toward = _dp_last_prob > _dp_first_prob
        _dp_note = (f"line moved {'TOWARD' if _dp_moved_toward else 'AWAY FROM'} this side today "
                    f"({_dp_first_prob:.1%} -> {_dp_last_prob:.1%}, {len(_dp_hist)} snapshots) "
                    f"[INTRADAY PROXY, NOT TRUE OPENING-TO-CLOSE CLV -- see function docstring]")
        return _dp_moved_toward, _dp_note
    except Exception as _dp_lm_e:
        return None, f"line-movement check failed ({_dp_lm_e})"


def _dp_sp_elo_for(team_side_col, board_idx):
    """Resolves this game's starter name off `board` (same 3-tier fallback
    _pitcher_note already uses) and looks up SP ELO -- standalone so the
    daily-play scorer doesn't depend on _pitcher_note's own loop state."""
    for _c in (f"{team_side_col}_pitcher_final_name", f"{team_side_col}_pitcher_name", f"{team_side_col}_starter"):
        if _c in board.columns:
            _nm = board.at[board_idx, _c] if board_idx in board.index else None
            if pd.notna(_nm) and _nm:
                _elo = globals().get("mlb_pitcher_elo", {}).get(str(_nm))
                if _elo is not None:
                    return _elo, str(_nm)
                import unicodedata as _dp_ud

                def _dp_nn(s):
                    return _dp_ud.normalize("NFKD", str(s)).encode("ascii", "ignore").decode().lower().strip()
                for _k, _v in globals().get("mlb_pitcher_elo", {}).items():
                    if _dp_nn(_k) == _dp_nn(_nm):
                        return _v, str(_nm)
                return 1500.0, str(_nm)
    return None, None


def _dp_bilateral_bp_risk(away_team, home_team):
    """Same MLB-BULLPEN-PARITY-1 condition (risk==1 both sides, L3 ERA within
    0.5R) -- standalone here since the card's own _bp_wash is computed
    loop-local to whichever game the WHY section happens to be printing."""
    if "mlb_bp_risk_score" not in globals() or "_pl_norm" not in globals():
        return False
    _aw_risk = mlb_bp_risk_score.get(_pl_norm(away_team), 0)
    _hm_risk = mlb_bp_risk_score.get(_pl_norm(home_team), 0)
    _l3_tbl = globals().get("mlb_bp_workload_l3", {}) or {}
    _aw_l3 = _l3_tbl.get(_pl_norm(away_team), {}).get("era_l3")
    _hm_l3 = _l3_tbl.get(_pl_norm(home_team), {}).get("era_l3")
    return bool(_aw_risk == 1 and _hm_risk == 1 and _aw_l3 is not None and _hm_l3 is not None
                and abs(_aw_l3 - _hm_l3) <= 0.5)


_DP_SP_ELO_GAP_MIN = 60.0  # ELO points -- roughly 1.5 tier bands (this notebook's tiers are 40pt apart); provisional, unbacktested


def _dp_score_candidate(row):
    """Counts independently-computed signals already on this card that point
    the SAME direction as `row`'s side, plus whether an F5 market-isolation
    recommendation applies. v1 factor list, reusing existing WHY-bullet
    computations rather than deriving new ones (lower bug surface, and each
    factor already has its own separate track record on this card). NOT
    exhaustive -- expand during the informational window if it's clearly
    missing something that mattered."""
    _side = row.get("side")
    _aw, _hm = row.get("away_team"), row.get("home_team")
    _opp = _hm if _pl_norm(_side) == _pl_norm(_aw) else _aw
    _factors = []

    _side_col = "away" if _pl_norm(_side) == _pl_norm(_aw) else "home"
    _opp_col = "home" if _side_col == "away" else "away"
    _bidx = row.get("board_idx")
    _side_elo, _side_sp = _dp_sp_elo_for(_side_col, _bidx) if _bidx is not None else (None, None)
    _opp_elo, _opp_sp = _dp_sp_elo_for(_opp_col, _bidx) if _bidx is not None else (None, None)
    _sp_elo_gap = None
    if _side_elo is not None and _opp_elo is not None:
        _sp_elo_gap = _side_elo - _opp_elo
        if _sp_elo_gap >= 40:
            _factors.append(f"SP ELO edge ({_side_sp} {_side_elo:.0f} vs {_opp_sp} {_opp_elo:.0f})")

    if "mlb_lineup_flags" in globals():
        _aw_pen = float((mlb_lineup_flags.get(_pl_norm(_aw)) or {}).get("penalty", 0) or 0)
        _hm_pen = float((mlb_lineup_flags.get(_pl_norm(_hm)) or {}).get("penalty", 0) or 0)
        _net_favors = _hm if _aw_pen > _hm_pen else (_aw if _hm_pen > _aw_pen else None)
        if _net_favors and _pl_norm(_net_favors) == _pl_norm(_side) and abs(_aw_pen - _hm_pen) >= 0.20:
            _factors.append(f"net injury delta favors {_side} ({abs(_aw_pen - _hm_pen):.2f}R)")

    if "mlb_motivation_gap_tiers" in globals():
        _side_mot = mlb_motivation_gap_tiers.get(_pl_norm(_side))
        _opp_mot = mlb_motivation_gap_tiers.get(_pl_norm(_opp))
        if _side_mot and (not _opp_mot or _side_mot[1] > _opp_mot[1]):
            _factors.append(f"motivation edge ({_side_mot[0]})")

    if "mlb_bp_risk_score" in globals():
        _side_bp = mlb_bp_risk_score.get(_pl_norm(_side), 0)
        _opp_bp = mlb_bp_risk_score.get(_pl_norm(_opp), 0)
        if _opp_bp >= 1 and _opp_bp > _side_bp:
            _factors.append(f"bullpen risk edge (opp risk={_opp_bp} vs this side risk={_side_bp})")

    if "mlb_h2h_lookup" in globals() and _aw and _hm:
        _h2h = mlb_h2h_lookup.get((_aw.lower(), _hm.lower()))
        if _h2h and _h2h.get("h2h_n_games"):
            _h2h_wr = _h2h.get("h2h_home_win_rate")
            if pd.notna(_h2h_wr):
                _side_is_home = _pl_norm(_side) == _pl_norm(_hm)
                _side_wr = _h2h_wr if _side_is_home else (1 - _h2h_wr)
                if _side_wr >= 0.60:
                    _factors.append(f"2026 H2H favors {_side} ({_side_wr:.1%}, {int(_h2h['h2h_n_games'])}G)")

    if "_team_records" in globals():
        _side_rec = next((v for k, v in _team_records.items() if any(w in k for w in _side.lower().split() if len(w) > 3)), None)
        _opp_rec = next((v for k, v in _team_records.items() if any(w in k for w in _opp.lower().split() if len(w) > 3)), None)
        if _side_rec and _opp_rec and _opp_rec.get("win_pct", 0.5) - _side_rec.get("win_pct", 0.5) >= 0.10:
            _factors.append("model backs the side with the worse record -- possible public overreaction to record")

    # MLB-DAILYPLAY-1 factor-independence fix (2026-09-15, user review): SP
    # ELO edge and current-season H2H can be the SAME underlying signal
    # wearing two hats -- if this year's starter has simply dominated this
    # opponent, both the ELO gap and the H2H record are reporting one fact
    # about one pitcher, not two independent pieces of evidence. Collapses
    # them into a single merged factor whenever both fire, rather than
    # double-counting -- doesn't try to prove causation (would need
    # per-game starter attribution this pipeline doesn't have), just
    # refuses to let the two most likely non-independent signals both
    # count toward the >=2-factor bar on their own.
    _elo_factor = next((f for f in _factors if f.startswith("SP ELO edge")), None)
    _h2h_factor = next((f for f in _factors if f.startswith("2026 H2H")), None)
    if _elo_factor and _h2h_factor:
        _factors = [f for f in _factors if f not in (_elo_factor, _h2h_factor)]
        _factors.append(f"SP/team dominance vs this opponent (merged -- \"{_elo_factor}\" + "
                         f"\"{_h2h_factor}\" can share one underlying cause, counted once)")

    _f5_note = None
    if _sp_elo_gap is not None and _sp_elo_gap >= _DP_SP_ELO_GAP_MIN and _dp_bilateral_bp_risk(_aw, _hm):
        _f5_key = (_aw.lower(), _hm.lower())
        if globals().get("_f5_market_live") and _f5_key in globals().get("_f5_odds", {}):
            _f5o = _f5_odds[_f5_key]
            _f5_side_nv = _f5o["home_nv"] if _side_col == "home" else _f5o["away_nv"]
            _f5_note = (f"MARKET ISOLATION: SP ELO edge ({_sp_elo_gap:+.0f}) + bilateral bullpen risk -- "
                        f"recommend F5 ML instead of full-game ML (F5 no-vig price for {_side}: {_f5_side_nv:.1%})")
        else:
            _f5_note = (f"MARKET ISOLATION: SP ELO edge ({_sp_elo_gap:+.0f}) + bilateral bullpen risk -- "
                        f"recommend F5 ML instead of full-game ML (no real F5 market odds this pull, "
                        f"synthetic/directional only)")

    return len(_factors), _factors, _f5_note


# ---- Step 2: score every ML candidate in today's qualifying pool, pick the
# best by factor count (edge as tiebreak). v1 scorer only covers ML rows --
# TOTAL/RL don't have a clean "side" to score factors against yet. ----
if len(_snap_official) and "side" in _snap_official.columns:
    _dp_scored = []
    for _dpi2, _dprow2 in _snap_official.iterrows():
        if _dprow2.get("market") != "ML" or not _dprow2.get("away_team"):
            continue
        _cnt, _facs, _f5n = _dp_score_candidate(_dprow2)
        _dp_scored.append({**_dprow2.to_dict(), "factor_count": _cnt, "factors": _facs, "f5_note": _f5n})

    if _dp_scored:
        _dp_scored_df = pd.DataFrame(_dp_scored).sort_values(
            ["factor_count", "edge"], ascending=[False, False]).reset_index(drop=True)
        _dp_best = _dp_scored_df.iloc[0]
        _dp_raw_pick = _snap_ranked.iloc[0] if len(_snap_ranked) else None

        print(f"  4-factor candidate: {_dp_best['side']} ({_dp_best['matchup']}) -- "
              f"{_dp_best['factor_count']} factor(s) stacked: "
              f"{', '.join(_dp_best['factors']) if _dp_best['factors'] else 'none'}")
        if _dp_best["factor_count"] < 2:
            print("  !! Does not meet the >=2-factor bar -- no real 'Daily Play' candidate exists today under this methodology")
        if _dp_best.get("f5_note"):
            print(f"  {_dp_best['f5_note']}")
        _dp_moved, _dp_lm_note = _dp_line_movement(_dp_best.get("game_id"), _dp_best.get("market"), _dp_best.get("side"))
        print(f"  Line movement: {_dp_lm_note}")

        _dp_disagreement_type = None
        if _dp_raw_pick is not None:
            _dp_same_game = ((_dp_best.get("game_id") == _dp_raw_pick.get("game_id"))
                              if _dp_best.get("game_id") is not None else (_dp_best["matchup"] == _dp_raw_pick["matchup"]))
            _dp_same_side = _pl_norm(str(_dp_best["side"])) == _pl_norm(str(_dp_raw_pick["side"]))
            if not _dp_same_game or not _dp_same_side:
                _dp_disagreement_type = "GAME_SELECTION"
                print(f"  !! DAILY PLAY DISAGREEMENT: raw-edge picks {_dp_raw_pick['side']} "
                      f"({_dp_raw_pick['matchup']}, {_dp_raw_pick['edge']:+.1%}) -- 4-factor methodology "
                      f"recommends {_dp_best['side']} ({_dp_best['matchup']}, {_dp_best['edge']:+.1%}) instead -- "
                      f"informational only, no automatic override")
            elif _dp_best.get("f5_note"):
                _dp_disagreement_type = "MARKET_SELECTION"
                print(f"  [MARKET ISOLATION SUGGESTION] same pick as raw-edge ({_dp_best['side']}) but 4-factor "
                      f"methodology would play it on F5 instead of full-game ML -- tracked separately from "
                      f"game-selection disagreements per user request")
            else:
                # MLB-DAILYPLAY-1 (2026-09-15, user review): logged too, not just
                # skipped -- the graduation review needs the TOTAL evaluated-day count
                # to report the overall disagreement RATE, not just the win rate among
                # disagreement nights (a disagreements-only file can't tell you how
                # often the two methodologies actually diverge if agreement days never
                # appear in it at all).
                _dp_disagreement_type = "AGREEMENT"
                print(f"  Agreement: both methodologies pick {_dp_best['side']} on the same market")

        if _dp_disagreement_type is not None:
            try:
                _dp_row_out = {
                    "card_date": _dp_today, "disagreement_type": _dp_disagreement_type,
                    "raw_edge_side": _dp_raw_pick["side"], "raw_edge_market": _dp_raw_pick["market"],
                    "raw_edge_value": _dp_raw_pick["edge"], "raw_edge_game_id": _dp_raw_pick.get("game_id"),
                    "raw_edge_matchup": _dp_raw_pick["matchup"],
                    "four_factor_side": _dp_best["side"],
                    "four_factor_market": "F5 ML" if _dp_disagreement_type == "MARKET_SELECTION" else _dp_best["market"],
                    "four_factor_score": _dp_best["factor_count"], "four_factor_factors": "; ".join(_dp_best["factors"]),
                    "four_factor_game_id": _dp_best.get("game_id"), "four_factor_matchup": _dp_best["matchup"],
                    "line_movement_note": _dp_lm_note, "result": "PENDING", "margin": "",
                }
                _dp_is_new = not _DP_DISAGREE_PATH.exists()
                with open(_DP_DISAGREE_PATH, "a", newline="", encoding="utf-8") as _dp_df:
                    _dp_dw = _dp_csv.DictWriter(_dp_df, fieldnames=_DP_DISAGREE_FIELDS)
                    if _dp_is_new:
                        _dp_dw.writeheader()
                    _dp_dw.writerow(_dp_row_out)
                print(f"  Logged to {_DP_DISAGREE_PATH.name} for the 20-play graduation review")
            except Exception as _dp_log_e:
                print(f"  [WARN] disagreement logging failed: {_dp_log_e}")
    else:
        print("  No ML candidates in today's qualifying pool to score.")
else:
    print("  No qualifying plays today -- nothing to compare.")


print("\n" + "=" * 78)
print("OTHER EDGES BELOW THE BAR -- INFORMATIONAL ONLY, NOT RECOMMENDED PLAYS")
print("=" * 78)
_snap_other = _snap_df[_snap_df.get("official", False) != True] if len(_snap_df) else _snap_df
if len(_snap_other):
    _snap_other = _snap_other.copy()
    _snap_other["abs_edge"] = _snap_other["edge"].abs()
    _snap_other_ranked = _snap_other.sort_values("abs_edge", ascending=False).head(10).reset_index(drop=True)
    for _, _row in _snap_other_ranked.iterrows():
        _reason_disp = _row.get("reason") or ""
        # BUG FIX (2026-09-27 user audit, priority 1): pre/post-blend edge
        # needs to reach the below-bar list too, not just the per-game
        # snapshot line above -- this condensed list is what actually gets
        # scanned for "would this have qualified under the old math"
        # counterfactuals (edge_preblend is only ever set for ML rows).
        _row_preblend = _row.get("edge_preblend")
        _preblend_disp = f"  [pre-blend {_row_preblend:+.1%}]" if pd.notna(_row_preblend) else ""
        # MLB-BB-ODDS-1 (2026-10-04 user audit -- "The NFL, WNBA and MLB below-bar entries carry no price,
        # so the shadow tracker can't record odds. Print the price and book on every line.")
        _px = _row.get("price"); _bk = _row.get("book")
        _px_disp = f"  @ {_px} ({_bk or 'book n/a'})" if (_px is not None and str(_px) not in ("", "nan", "None")) else "  @ price n/a"
        print(f"  {_row['edge']:+6.1%}  {_row['market']:<12} {_row['side']:<22} {_row['matchup']}{_px_disp}{_preblend_disp}  {_reason_disp}")
else:
    _snap_other_ranked = _snap_other
    print("  (none)")

# =========================================================
# FINAL CARD -- OFFICIAL PLAY + ALL EDGES FOUND, WHY (2026-07-28, user
# requested; extended 2026-08-14, user requested)
# A single consolidated view of every edge the model found today -- the
# official play, the rest of the qualifying pool, and every below-bar
# informational edge -- plus the reasoning behind each one (SP matchup/
# tier/stats, projected score, SP short-outing role, home/road SP split,
# recent-form credibility, injuries, bullpen risk, team-record gap),
# instead of piecing it together from the main card's scattered per-pick
# sections or from transient log lines during the board build. Reuses
# _pitcher_note()/_score_line()/_pl_norm() -- the exact same helpers and
# lookups the main card above already computed -- so this can't drift
# from the real card's own reasoning; it's a clean re-read of the same
# data, same pattern as the rest of this snapshot cell.
#
# 2026-08-14: originally only ran for _snap_ranked (the official play +
# qualifying pool). User asked for the same "why" detail everywhere the
# model found an edge, not just the official play -- factored the per-row
# reasoning into _emit_why() so it can run unchanged for _snap_other_ranked
# (below-bar informational edges) too, tagged distinctly so it's never
# confused with a recommended play. The official-only side effects
# (ledger logging, bullpen-exposure downgrade-from-OFFICIAL) still gate on
# _f_tag == "OFFICIAL" specifically, so they can't fire for a below-bar row.
# =========================================================
print("\n" + "=" * 78)
print("FINAL CARD -- OFFICIAL PLAY + ALL EDGES FOUND, WHY")
if "CARD_DATE" in globals():
    _card_date_note = "  [TOMORROW's slate -- pipeline ran after 8 PM ET, today's already-decided games are excluded]" if globals().get("_card_delta") else "  [today's slate]"
    print(f"Card date: {CARD_DATE}{_card_date_note}")
# ODDS SNAPSHOT TIMESTAMP (2026-09-19, user audit -- Rays +26.1% edge:
# the card never says whether that was calculated against a sharp
# current line or a stale early one). mlb_odds_fetch_time (Cell -- PULL
# TODAY'S MLB MONEYLINE ODDS FROM ESPN) captures the single moment this
# run pulled ESPN's odds -- every mkt%/edge% below traces back to that
# one snapshot, so print it once right under the card date.
if globals().get("mlb_odds_fetch_time"):
    print(f"Odds pulled: {mlb_odds_fetch_time} (local) -- every mkt%/edge% below is against this single snapshot, not a live line")
print("=" * 78)

# BUG FIX (2026-08-07, user request -- tackle the deferred "systemic SP-ELO
# staleness" open question): after the 2026-08-07 _fed_check fix, any
# "!! ELO-ERA DIVERGENCE" bullet that still fires is now guaranteed to be a
# genuinely thin-IP pitcher (established starters self-clear per Gate 8's own
# logic, reused here) -- which already answers most of the original "is this
# broad staleness or just thin samples" question by construction. This
# counter makes that visible per-card going forward (instead of requiring a
# manual re-investigation each time it looks suspicious) so a real trend
# (e.g. consistently high counts night after night) would be easy to notice.
_fed_divergence_count = 0

def _emit_why(_frow, _f_tag):
    _f_bidx = _frow.get("board_idx")
    print(f"\n{'-' * 78}")
    print(f"  {_frow['edge']:+6.1%}  {_frow['market']:<12} {_frow['side']:<22} {_frow['matchup']}   [{_f_tag}]")
    # FIX 3 (2026-09-22, user audit): flag which qualifying path this play
    # cleared -- the standard base-floor path (logit-space ~BASE_EDGE_MIN floor,
    # Cell bd676982's initial passes_edge_rule assignment) needs no callout, but
    # the two other paths that cell can promote passes_edge_rule through --
    # the Pitcher Advantage Exception (_pitcher_adv_exception) and the
    # High-Confidence Favorite path (_high_conf_favorite) -- qualify on a much
    # smaller raw edge, and a reader scanning just the edge% could otherwise
    # mistake that for the play clearing the usual ~12% bar on its own. ML-only:
    # RL/TOTAL never route through cand/passes_edge_rule at all (see their own
    # snapshot blocks above), so there is only ever one qualifying bar for those.
    if _frow.get("market") == "ML" and "cand" in globals():
        _f_qm_aw = str(_frow.get("away_team", ""))
        _f_qm_hm = str(_frow.get("home_team", ""))
        _f_qm_gid = _frow.get("game_id")
        if game_id_col and game_id_col in cand.columns and _f_qm_gid is not None:
            _f_qcand = cand[(cand[away_col].astype(str) == _f_qm_aw) & (cand[home_col].astype(str) == _f_qm_hm)
                            & (cand[game_id_col] == _f_qm_gid) & (cand["bet_team"].astype(str) == str(_frow.get("side")))]
        else:
            _f_qcand = cand[(cand[away_col].astype(str) == _f_qm_aw) & (cand[home_col].astype(str) == _f_qm_hm)
                            & (cand["bet_team"].astype(str) == str(_frow.get("side")))]
        if len(_f_qcand):
            _f_qrow = _f_qcand.iloc[0]
            _f_f5 = _mlb_f5_line(_f_qrow)   # R-F5-1 support
            if _f_f5:
                print(f"  [{_f_f5}]")
            if bool(_f_qrow.get("_pitcher_adv_exception", False)):
                print(f"  [EDGE NOTE: qualified via Pitcher Advantage Exception path "
                      f"(pick SP >={_PITCHER_ADV_ELO} ELO better than opponent SP, edge >={_PITCHER_ADV_EDGE:.0%}) "
                      f"-- did not clear the standalone ~12% logit-space floor]")
            elif bool(_f_qrow.get("_high_conf_favorite", False)):
                print(f"  [EDGE NOTE: qualified via High-Confidence Favorite path "
                      f"(model win prob >={HIGH_CONF_PROB_MIN:.0%}, market already favors this side, edge >={HIGH_CONF_EDGE_MIN:.0%}) "
                      f"-- did not clear the standalone ~12% logit-space floor]")
    if _f_bidx is None or _f_bidx not in board.index:
        print("  (reasoning unavailable -- could not relocate this game's board row)")
        return
    _f_row = board.loc[_f_bidx]
    _f_aw = str(_f_row.get(away_col, ""))
    _f_hm = str(_f_row.get(home_col, ""))

    # BUG FIX (2026-08-18, user audit found live): see the matching comment
    # on the daily snapshot loop above for the real case this was found
    # from -- surfaced here too since this "WHY" block is specifically what
    # the user reads to decide whether to trust an OFFICIAL play, and a
    # starter conflict is exactly the kind of thing that belongs front and
    # center on that specific play's own reasoning, not just the snapshot.
    _f_conflict_note = globals().get("_mlb_sp_source_conflicts", {}).get((_f_aw, _f_hm))
    if _f_conflict_note:
        print(f"  !! SP CONFLICT UNRESOLVED: {_f_conflict_note} -- verify starter before trusting any SP-driven edge here")

    # Resolved once per row, reused by every SP-name-driven flag below
    # (short-outing role check still does its own resolution per side since
    # it also needs the opener/bulk boolean columns, not just the name).
    _f_aw_sp = next((str(_f_row.get(c, "")).strip() for c in ["away_pitcher_final_name", "away_pitcher_name", "away_starter"]
                     if _f_row.get(c) and str(_f_row.get(c)) != "nan"), "")
    _f_hm_sp = next((str(_f_row.get(c, "")).strip() for c in ["home_pitcher_final_name", "home_pitcher_name", "home_starter"]
                     if _f_row.get(c) and str(_f_row.get(c)) != "nan"), "")

    _f_pn = _pitcher_note(_f_row) if "_pitcher_note" in globals() else None
    if _f_pn:
        print(_f_pn)
    _f_sl = _score_line(_f_row) if "_score_line" in globals() else None
    if _f_sl:
        print(_f_sl)
    # MLB-TOT-COMP-1: full term-by-term totals breakdown for O/U rows -- every term in runs vs a
    # league-average game, so a gap to the line can be traced to a source. Sums to the component total.
    if str(_frow.get("market", "")).startswith("O/U"):
        _f_dk = (str(_f_row.get(away_col, "")).lower().strip(), str(_f_row.get(home_col, "")).lower().strip())
        _f_dd = (globals().get("mlb_totals_decomp") or {}).get(_f_dk)
        if _f_dd:
            print(f"  [TOTALS DECOMP] component total {_f_dd['total']:.2f} = " + " ".join(
                f"{v:+.2f} {l}" if i else f"{v:.2f} {l}" for i, (l, v, w) in enumerate(_f_dd["parts"])))
            for _l, _v, _w in _f_dd["parts"][1:]:
                if _w:
                    print(f"      {_v:+6.2f}  {_l:<32} {_w}")

    # SP short-outing role -- ties into the 2026-07-28 ML-edge discount fix
    for _f_nm_cols, _f_op_c, _f_bulk_c, _f_bg_c in [
        (["away_pitcher_final_name", "away_pitcher_name", "away_starter"], "away_sp_is_opener", "away_sp_is_bulk", "away_sp_is_bullpen_game"),
        (["home_pitcher_final_name", "home_pitcher_name", "home_starter"], "home_sp_is_opener", "home_sp_is_bulk", "home_sp_is_bullpen_game"),
    ]:
        _f_nm = next((str(_f_row.get(c, "")) for c in _f_nm_cols if _f_row.get(c)), "")
        if not _f_nm:
            continue
        if bool(_f_row.get(_f_bg_c, False)):
            _f_ann = next((v for v in (globals().get("mlb_announced_roles") or {}).values() if str(v.get("opener", "")).lower() == _f_nm.lower()), None)
            if _f_ann:
                # MLB-PS-OPENER-NEWS-1: a regular starter announced as an opener -- don't call him a bullpen arm,
                # and credit the named bulk arm, not the opener
                print(f"  [ANNOUNCED OPENER] {_f_nm} opens (news: {str(_f_ann.get('source', ''))[:90]}) -- bulk arm "
                      f"{_f_ann.get('bulk') or 'not named'} is graded as the SP"
                      + ("; priced normally, Gate 7d not applied (MLB-PS-BULK-PRICE-1)" if _f_ann.get('bulk') else "; side UNRATED (Gate 7d)"))
                continue
            print(f"  [BULLPEN GAME] {_f_nm}: no real starter track record this season -- true bullpen-game arm, not a bulk/short-role starter -- SP quality treated almost entirely as bullpen strength, opposing pick gets a separate edge bonus for this")
        elif bool(_f_row.get(_f_op_c, False)):
            print(f"  [SP ROLE] {_f_nm} is a true opener (<3.0 IP/start this season) -- SP-quality ML edge already discounted for this")
        elif bool(_f_row.get(_f_bulk_c, False)):
            print(f"  [SP ROLE] {_f_nm} is a bulk/short-role arm (3-5 IP/start this season) -- SP-quality ML edge already discounted for this")

    # SP CAREER ERA VS TODAY'S OPPONENT (2026-09-11, user request -- Abbott
    # vs Milwaukee, 3.74 ERA in 9 starts never reached the card): the fetch
    # behind this (_sp_vs_opponent_era, Cell 87 M-NEW-3) already existed, but
    # only ever surfaced conditionally -- as a tier-downgrade reason, and
    # only once the gap vs season ERA crossed _M_NEW3_ERA_GAP. A real,
    # sizeable-sample career split under that threshold had zero visibility
    # anywhere. Prints unconditionally now, same standing-field treatment as
    # [H2H]/[BvP] elsewhere on this card, whenever the >=2-start minimum
    # sample is met -- the separate downgrade-reason/edge-shave logic in
    # Cell 87 is unchanged, this only adds visibility for the reader.
    if "_sp_vs_opponent_era" in globals():
        for _vo_nm, _vo_opp in ((_f_aw_sp, _f_hm), (_f_hm_sp, _f_aw)):
            if not _vo_nm or not _vo_opp:
                continue
            _vo_era, _vo_n = _sp_vs_opponent_era(_vo_nm, _vo_opp)
            if _vo_era is None:
                continue
            _vo_nk = _mlb_name_norm(_vo_nm) if "_mlb_name_norm" in globals() else _vo_nm.lower()
            _vo_fev = mlb_pitcher_fip_lookup.get(_vo_nk) if "mlb_pitcher_fip_lookup" in globals() else None
            if _vo_fev is None and "mlb_pitcher_fip_lookup" in globals() and "_mlb_name_norm" in globals():
                _vo_fev = next((v for k, v in mlb_pitcher_fip_lookup.items() if _mlb_name_norm(k) == _vo_nk), None)
            _vo_season_era = (_vo_fev or {}).get("era_season")
            _vo_has_season = _vo_season_era is not None and pd.notna(_vo_season_era)
            _vo_season_str = f" vs {float(_vo_season_era):.2f} season ERA" if _vo_has_season else ""
            _vo_flag = ""
            if _vo_has_season and abs(_vo_era - float(_vo_season_era)) > _M_NEW3_ERA_GAP:
                _vo_flag = f"  [DIVERGES >{_M_NEW3_ERA_GAP:.1f}R from season ERA]"
            print(f"  [SP VS OPPONENT] {_vo_nm}: {_vo_era:.2f} ERA vs {_vo_opp} in {_vo_n} "
                  f"start(s) this season{_vo_season_str}{_vo_flag}")

    # MLB-CAREER-H2H-1 (2026-09-14, user audit -- Detmers vs Seattle):
    # [SP VS OPPONENT] above is season-scoped and stayed silent here since
    # Detmers had only 1 start vs SEA in 2026 -- below its own 2-start
    # minimum -- even though he carries a real 5.24 ERA vs this opponent
    # across 13 career appearances. Prints the career number alongside the
    # season one above (so the reader can judge resolved-vs-persistent for
    # themselves) and calls out a CAREER H2H RISK when the career number
    # reads meaningfully worse than the pitcher's season ERA.
    if "_sp_career_vs_opponent_era" in globals():
        for _cvo_nm, _cvo_opp in ((_f_aw_sp, _f_hm), (_f_hm_sp, _f_aw)):
            if not _cvo_nm or not _cvo_opp:
                continue
            _cvo_era, _cvo_n = _sp_career_vs_opponent_era(_cvo_nm, _cvo_opp)
            if _cvo_era is None:
                continue
            _cvo_nk = _mlb_name_norm(_cvo_nm) if "_mlb_name_norm" in globals() else _cvo_nm.lower()
            _cvo_fev = mlb_pitcher_fip_lookup.get(_cvo_nk) if "mlb_pitcher_fip_lookup" in globals() else None
            if _cvo_fev is None and "mlb_pitcher_fip_lookup" in globals() and "_mlb_name_norm" in globals():
                _cvo_fev = next((v for k, v in mlb_pitcher_fip_lookup.items() if _mlb_name_norm(k) == _cvo_nk), None)
            _cvo_season_era = (_cvo_fev or {}).get("era_season")
            _cvo_has_season = _cvo_season_era is not None and pd.notna(_cvo_season_era)
            print(f"  [CAREER H2H] {_cvo_nm}: {_cvo_era:.2f} ERA vs {_cvo_opp} in {_cvo_n} career appearance(s)")
            if _cvo_has_season and (_cvo_era - float(_cvo_season_era)) > _SP_CAREER_H2H_RISK_GAP:
                print(f"  !! CAREER H2H RISK: {_cvo_nm} carries a {_cvo_era:.2f} career ERA vs {_cvo_opp} "
                      f"({_cvo_n} appearances), {_cvo_era - float(_cvo_season_era):.2f}R worse than his "
                      f"{float(_cvo_season_era):.2f} season ERA -- verify whether this season's H2H result "
                      f"resolves or repeats the pattern before trusting the SP edge")
            elif _cvo_has_season and (float(_cvo_season_era) - _cvo_era) > _SP_CAREER_H2H_RISK_GAP:
                # MLB-CAREER-H2H-2: the mirror case -- this pitcher is much BETTER vs this opponent than his season
                print(f"  !! CAREER H2H EDGE: {_cvo_nm} carries a {_cvo_era:.2f} career ERA vs {_cvo_opp} "
                      f"({_cvo_n} appearances), {float(_cvo_season_era) - _cvo_era:.2f}R better than his "
                      f"{float(_cvo_season_era):.2f} season ERA -- counts against any play on {_cvo_opp}")

    # POST-TRADE SPLIT (2026-09-11, user request -- Dustin May: pre-trade
    # FIP 2.88 drove a 1538 ELO, post-trade ERA with Milwaukee ~6.43 told a
    # different story that never reached the card): standing field, same
    # treatment as [SP VS OPPONENT] above -- prints whenever the pitcher
    # was traded within the last 45 days (Cell 20's is_recently_traded),
    # regardless of whether the gap crosses the shave/downgrade threshold,
    # so the reader always sees the split, not just once it has already
    # been judged large enough to matter.
    if "mlb_pitcher_fip_lookup" in globals() and "_mlb_name_norm" in globals():
        for _pt_nm in (_f_aw_sp, _f_hm_sp):
            if not _pt_nm:
                continue
            _pt_nk = _mlb_name_norm(_pt_nm)
            _pt_fev = mlb_pitcher_fip_lookup.get(_pt_nk)
            if _pt_fev is None:
                _pt_fev = next((v for k, v in mlb_pitcher_fip_lookup.items() if _mlb_name_norm(k) == _pt_nk), None)
            if not _pt_fev or not bool(_pt_fev.get("is_recently_traded")):
                continue
            _pt_post_era = _pt_fev.get("post_trade_era")
            _pt_post_fip = _pt_fev.get("post_trade_fip")
            _pt_post_n = _pt_fev.get("post_trade_starts") or 0
            _pt_pre_era = _pt_fev.get("pre_trade_era")
            _pt_pre_n = _pt_fev.get("pre_trade_starts") or 0
            _pt_days = _pt_fev.get("days_since_trade")
            _pt_team = _pt_fev.get("new_team_name") or "new team"
            if _pt_post_era is not None and pd.notna(_pt_post_era):
                _pt_post_str = f"{_pt_post_era:.2f} ERA" + (f" / {_pt_post_fip:.2f} FIP" if pd.notna(_pt_post_fip) else "")
            else:
                _pt_post_str = "no starts yet"
            _pt_pre_str = (f"{_pt_pre_era:.2f} ERA in {_pt_pre_n} start(s)"
                            if _pt_pre_era is not None and pd.notna(_pt_pre_era) else "no prior starts on record")
            print(f"  [POST-TRADE SPLIT] {_pt_nm}: traded to {_pt_team} {_pt_days}d ago -- "
                  f"{_pt_post_str} in {_pt_post_n} start(s) since, vs {_pt_pre_str} before")

    # SP BULLPEN EXPOSURE (audit fix, wired in here 2026-08-04): originally
    # built into _why_bullets() in Cell 88, which turned out to be dead
    # code -- defined but never called by anything, so the check was
    # never actually visible on any card. This loop is the real "FINAL
    # CARD + WHY" output, so this is where it needs to live. A starter
    # who rarely reaches 5 IP gets treated like a full starter for edge
    # purposes -- most of the game is really a bullpen game once he's
    # pulled early. Data comes from Cell 83's mlb_sp_bullpen_exposure
    # (real per-start IP history, includes bulk-relief appearances).
    _fbe_capped = False  # set True below if confirmed thin-exposure starter
    _fbe_pick_sp = None
    _fbe_rec = None
    if _frow.get("market") == "ML" and "_pl_norm" in globals():
        _fbe_pick_home = _pl_norm(str(_frow.get("side", ""))) == _pl_norm(_f_hm)
        _fbe_pick_sp = _f_hm_sp if _fbe_pick_home else _f_aw_sp
        if _fbe_pick_sp and "mlb_sp_bullpen_exposure" in globals():
            _fbe_key = _mlb_name_norm(_fbe_pick_sp)
            _fbe_rec = mlb_sp_bullpen_exposure.get(_fbe_pick_sp)
            if _fbe_rec is None:
                for _fbe_nm, _fbe_v in mlb_sp_bullpen_exposure.items():
                    if _mlb_name_norm(_fbe_nm) == _fbe_key:
                        _fbe_rec = _fbe_v
                        break
            # 2026-08-08: replaced the hardcoded MANUAL_BULLPEN_EXPOSURE_WATCHLIST
            # name list with a real three-state read of the same mlb_sp_bullpen_
            # exposure data (Cell 83) the confirmed-starter branch below already
            # uses -- CONFIRMED-STARTER (starts>=3, trust the five_ip_pct rate),
            # TBD (some real tracked starts/bulk-relief IP exist, just too thin
            # to trust a rate off), or LOOKUP-FAILED (zero tracked data at all --
            # nothing to assert a "mix of starts + bulk relief" claim from, so
            # stay silent instead of guessing). The watchlist required a human to
            # notice a swingman (e.g. Alvarez) and hardcode their surname before
            # the card would say anything, and would print "no tracked starts
            # yet" while still asserting mixed usage for a listed name even with
            # zero supporting data. This fires for any pitcher automatically the
            # moment real data shows a thin-but-nonzero mixed sample, no name
            # list to maintain or go stale.
            if _fbe_rec and _fbe_rec["starts"] >= 3:
                if _fbe_rec["five_ip_pct"] < 0.20:
                    print(f"  !! BULLPEN EXPOSURE [confirmed-starter]: {_fbe_pick_sp} has reached 5 IP in only "
                          f"{_fbe_rec['five_ip_starts']}/{_fbe_rec['starts']} starts "
                          f"({_fbe_rec['five_ip_pct']:.0%}) -- treat SP edge as partial-game only")
                    # BULLPEN EXPOSURE GATE (2026-08-11, user audit): this used to be
                    # informational-only -- a starter who has NEVER reached 5 IP across
                    # 3+ confirmed starts (0% rate, not just a low one) means most of
                    # every game he pitches is really a bullpen game, and an ML edge
                    # computed as if he were a normal starter is measuring the wrong
                    # thing. Real case: Drew Anderson, 0/3 starts reaching 5 IP, driving
                    # a 498pt ELO-vs-ERA divergence flag. Same reasoning as the WNBA
                    # roster-shock/IL-return buffer gates added this session -- a known
                    # risk signal that only printed a warning gets real teeth. Scoped to
                    # the 0% case specifically (not the whole <20% band) since that is
                    # the unambiguous "this is not really a starter right now" case;
                    # the general <20% band stays informational, same as before.
                    if _fbe_rec["five_ip_starts"] == 0:
                        _fbe_capped = True
            elif _fbe_rec:
                print(f"  !! BULLPEN EXPOSURE [TBD]: {_fbe_pick_sp} has shown a mix of starts + bulk relief "
                      f"this season ({_fbe_rec['starts']} tracked start(s) so far, 5 IP reached in "
                      f"{_fbe_rec['five_ip_starts']}) -- automatic tracking sample still too thin to confirm; "
                      f"treat SP edge as partial-game only until confirmed")
            # else: lookup-failed (no tracked data at all) -- nothing to assert, stay silent

    # SP ELO-vs-ERA DIVERGENCE (audit fix, wired in here 2026-08-04, same
    # reason as bullpen exposure above). Reuses _era_implied_elo() (Cell
    # 88's ERA-recency blend formula) rather than inventing new
    # thresholds. Checks BOTH pitchers shown in the SP Matchup line
    # above, not just the pick's own -- the real case this was built
    # for (Quantrill, WEAK(<1440) ELO vs a 3.79 ERA) was the OPPONENT
    # SP, not the pick's.
    if "_era_implied_elo" in dir() and "_sp_form_cache" in globals() and "mlb_pitcher_elo" in globals():
        def _fed_elo_for(nm):
            if not nm: return None
            if nm in mlb_pitcher_elo: return float(mlb_pitcher_elo[nm])
            _k = _mlb_name_norm(nm)
            for _ek, _ev in mlb_pitcher_elo.items():
                if _mlb_name_norm(_ek) == _k: return float(_ev)
            return None
        def _fed_era_for(nm):
            _k = _mlb_name_norm(nm)
            _fc = _sp_form_cache.get(_k)
            if not _fc:
                for _ck, _cv in _sp_form_cache.items():
                    if _cv and _mlb_name_norm(_ck) == _k:
                        _fc = _cv
                        break
            if not _fc or _fc.get("era_season") is None:
                return None
            _e = float(_fc["era_season"])
            return _e if 0 < _e <= 12 else None
        def _fed_check(sp_name, side_label):
            if not sp_name or sp_name == "nan":
                return
            _elo = _fed_elo_for(sp_name)
            if _elo is None or _elo == 1500.0:
                return
            _era = _fed_era_for(sp_name)
            if _era is None:
                return
            _implied = _era_implied_elo(_era)
            _gap = _elo - _implied
            if abs(_gap) >= 120.0:
                # BUG FIX (2026-08-07, found grading a real card -- Detroit
                # Tigers ML, Valdez/Miller): this bullet fired unconditionally
                # on any 120+pt gap, with no IP-based self-clearing -- but Gate
                # 8 (cell bd676982, "SP ELO-ERA divergence", ~line 3199)
                # already applies the identical 120pt bar AND skips it once the
                # pitcher has >= _SP_STALE_MIN_IP innings this season, treating
                # an established starter's ELO/ERA disagreement as already
                # resolved rather than a live risk to hard-block on. This
                # bullet never had that second half of the check, so it kept
                # re-flagging a divergence Gate 8 had already decided was fine
                # to trust -- real case: Framber Valdez (6-7, 4.47 ERA, a full
                # season of starts) correctly cleared Gate 8 and reached
                # OFFICIAL, but this bullet still printed "verify tier before
                # trusting SP edge" as if the play were unresolved. Also labels
                # the ELO as "raw" -- this is deliberately the same
                # pre-adjustment mlb_pitcher_elo value Gate 8 uses, NOT the
                # away_sp_elo_adj/home_sp_elo_adj value the SP Matchup line
                # two lines above already shows, which can legitimately be a
                # different number for the same pitcher (that's the whole
                # point of the adjustment) -- unlabeled, the two lines looked
                # like a data-consistency bug rather than two intentionally
                # different numbers.
                # 2026-08-12 BUG FIX (found grading a real card -- Guardians/Tigers,
                # Rockies/Diamondbacks, Reds/White Sox): this self-clearing check (built
                # 2026-08-07 specifically so an established starter like Framber Valdez
                # wouldn't keep getting flagged) silently defaulted to 0.0 IP on ANY
                # lookup miss -- no distinction from a genuinely-confirmed 0 IP season.
                # Real card showed FIVE different pitchers (Bibee, Anderson, Sugano,
                # Bratt, Lodolo -- several clearly full-time starters well past 30 IP by
                # August) all printing the exact same "thin 0 IP sample" text, which is
                # only possible if mlb_pitcher_fip_lookup.get(...).get("ip_season") is
                # missing/failing for every pitcher checked, not that all five happen to
                # genuinely have zero innings. The bare `except: _fed_ip_val = 0.0` was
                # masking that failure as a confident-looking real number instead of
                # surfacing it -- same silent-failure pattern already fixed for SP
                # loading elsewhere in this cell (`!! NO SP DATA`). Now distinguishes
                # "confirmed thin/zero IP" from "lookup unavailable, can't confirm
                # self-clear" so the two aren't printed identically.
                _fed_ip_lookup_failed = False
                try:
                    _fed_ip_rec = globals().get("mlb_pitcher_fip_lookup", {}).get(_norm_name(sp_name), {})
                    _fed_ip = _fed_ip_rec.get("ip_season")
                    if _fed_ip is None or pd.isna(_fed_ip):
                        _fed_ip_val = 0.0
                        _fed_ip_lookup_failed = True
                    else:
                        _fed_ip_val = float(_fed_ip)
                except Exception:
                    _fed_ip_val = 0.0
                    _fed_ip_lookup_failed = True
                if _fed_ip_val < globals().get("_SP_STALE_MIN_IP", 30.0):
                    _dir = "higher" if _gap > 0 else "lower"
                    if _fed_ip_lookup_failed:
                        print(f"  !! ELO-ERA DIVERGENCE ({side_label}): {sp_name} raw ELO ({_elo:.0f}) reads {_dir} than "
                              f"his {_era:.2f} ERA implies (~{_implied:.0f}) -- {abs(_gap):.0f}pt gap, IP-sample lookup "
                              f"UNAVAILABLE (no ip_season for this pitcher in mlb_pitcher_fip_lookup) -- cannot confirm "
                              f"self-clear, verify tier before trusting SP edge")
                    else:
                        print(f"  !! ELO-ERA DIVERGENCE ({side_label}): {sp_name} raw ELO ({_elo:.0f}) reads {_dir} than "
                              f"his {_era:.2f} ERA implies (~{_implied:.0f}) -- {abs(_gap):.0f}pt gap on a thin "
                              f"{_fed_ip_val:.0f} IP sample; verify tier before trusting SP edge")
                    globals()["_fed_divergence_count"] = globals().get("_fed_divergence_count", 0) + 1
        _fed_check(_f_aw_sp, _f_aw)
        _fed_check(_f_hm_sp, _f_hm)

    # SP HOME/ROAD SPLIT reminder (2026-08-14, user request -- added after a
    # manual QA of the Angels ML play traced the pick's edge back to two
    # things that drove real movement at pick time (Cell 88's
    # _sp_split_elo_adj / _recency_adj) but never made it onto the card
    # itself, only into transient log lines during the board build. Reuses
    # _split_resolved (populated live, same run, same source Cell 88 used)
    # so this can't drift from what actually fed the pick's SP ELO. Real
    # case: Grayson Rodriguez has a 9.10 home ERA vs 4.97 road ERA -- his
    # Angels home start that day was already docked -165 ELO for it, but
    # nothing on the card said so.
    if "_split_resolved" in globals() and _split_resolved:
        for _sp_nm, _sp_is_home, _sp_side in [(_f_aw_sp, False, _f_aw), (_f_hm_sp, True, _f_hm)]:
            if not _sp_nm:
                continue
            _sv = _split_resolved.get(_sp_nm)
            if _sv is None:
                _nk = _mlb_name_norm(_sp_nm)
                _sv = next((v for k, v in _split_resolved.items() if _mlb_name_norm(k) == _nk), None)
            if not _sv:
                continue
            _sp_he, _sp_re = _sv.get("home_era"), _sv.get("road_era")
            if _sp_he is None or _sp_re is None or abs(_sp_he - _sp_re) < 1.50:
                continue  # 2026-09-10: raised 1.00 -> 1.50, matches Cell 88's _SP_SPLIT_MIN_GAP (user's exact spec)
            _this_era = _sp_he if _sp_is_home else _sp_re
            _other_era = _sp_re if _sp_is_home else _sp_he
            _venue = "home" if _sp_is_home else "road"
            _direction = "worse" if _this_era > _other_era else "better"
            # PERIPHERAL CORROBORATION CHECK (2026-08-23, user audit): this
            # bullet surfaced raw ERA split numbers (e.g. Nick Martinez 3.87
            # home / 2.14 road) as a clean qualitative read, which is what let
            # the Orioles/Rays case get oversold -- ERA is exactly the noisy,
            # sequencing-driven stat xFIP/SIERA exist to correct for, and
            # _sp_split_elo_adj (Cell 88) is itself already ERA-based, so
            # printing the raw split here doubles down on the same unregressed
            # number instead of checking it against anything. True split-level
            # xFIP/SIERA isn't tracked anywhere in this codebase (only home/
            # road ERA -- season FIP/xFIP/SIERA has no home/road breakdown), so
            # the best real corroboration available is this pitcher's own
            # season ERA-vs-peripherals gap: if his overall ERA already runs a
            # full run or more hot/cold against xFIP/SIERA, his ERA isn't a
            # trustworthy number in general, split included -- print a conflict
            # flag instead of the raw split in that case. (Verified against the
            # Martinez case that motivated this: season ERA 3.05 vs xFIP 4.52/
            # SIERA 4.70, a 1.47-1.65 run gap -- this now correctly flags
            # conflict instead of printing his split as a clean signal.)
            _sp_fip_entry = None
            if "mlb_pitcher_fip_lookup" in globals():
                _sp_nk = _mlb_name_norm(_sp_nm)
                _sp_fip_entry = mlb_pitcher_fip_lookup.get(_sp_nk)
                if _sp_fip_entry is None:
                    _sp_fip_entry = next((v for k, v in mlb_pitcher_fip_lookup.items() if _mlb_name_norm(k) == _sp_nk), None)
            _sp_season_era = _sp_fip_entry.get("era_season") if _sp_fip_entry else None
            _sp_peripheral = None
            if _sp_fip_entry:
                _sp_peripheral = next((v for v in (_sp_fip_entry.get("xfip"), _sp_fip_entry.get("siera")) if v is not None), None)
            # OPPONENT-QUALITY CONTEXT (2026-08-23, user audit): a raw
            # home/road ERA split can't tell "pitches worse away from home"
            # apart from "happened to face tougher offenses on the road this
            # season" -- pure scheduling confound, not a venue effect. This
            # notebook already computes exactly the right control for that
            # (mlb_sp_elo_df's vs_hi_fip/vs_lo_fip -- FIP specifically against
            # top-25% vs bottom-25% offenses, opponent-adjusted AND FIP-based
            # so it isn't ERA-noisy either), it just never made it onto the
            # card. Surfacing it alongside the split so the reader has the
            # more meaningful comparison, not replacing the split itself.
            _sp_vshi = _sp_vslo = None
            if "mlb_sp_elo_df" in globals() and isinstance(mlb_sp_elo_df, pd.DataFrame) and not mlb_sp_elo_df.empty:
                _sp_elo_row = mlb_sp_elo_df[mlb_sp_elo_df["pitcher"].apply(lambda p: _mlb_name_norm(p) == _sp_nk)]
                if not _sp_elo_row.empty:
                    _sp_vshi = _sp_elo_row.iloc[0].get("vs_hi_fip")
                    _sp_vslo = _sp_elo_row.iloc[0].get("vs_lo_fip")
            _sp_oppq_str = ""
            if pd.notna(_sp_vshi) and pd.notna(_sp_vslo):
                _sp_oppq_str = (f" | opponent-quality FIP: {float(_sp_vshi):.2f} vs top-25% offenses, "
                                f"{float(_sp_vslo):.2f} vs bottom-25% -- more meaningful than a home/road "
                                f"split, controls for who he actually faced")
            # M9 (2026-09-02 user audit -- Rogers @ COL): the ALTITUDE flag
            # elsewhere in this same loop separately notes Coors' park factor
            # but the two were never cross-referenced -- a pitcher's road ERA
            # split is computed across ALL his road parks, so if today's road
            # game happens to be at Coors specifically, "pitches worse on the
            # road" and "Coors inflates run environment" are likely describing
            # the same thing, not two independent signals stacking.
            _sp_coors_str = ""
            if _venue == "road" and not _sp_is_home:
                _sp_hm_norm = _pl_norm(_f_hm) if "_pl_norm" in globals() else _f_hm.lower()
                if "rockies" in _sp_hm_norm or "colorado" in _sp_hm_norm:
                    _sp_coors_str = (" -- NOTE: today's road game IS at Coors, so this road-ERA split "
                                      "is likely explained by (not independent of) the altitude/park factor above, "
                                      "not a general road-form signal")
            if _sp_season_era is not None and _sp_peripheral is not None and abs(float(_sp_season_era) - float(_sp_peripheral)) >= 1.00:
                print(f"  !! SP SPLIT CONFLICT: {_sp_nm} ({_sp_side}) -- {_venue} ERA split ({_sp_he:.2f} home vs "
                      f"{_sp_re:.2f} road) not shown as a signal -- his own season ERA ({float(_sp_season_era):.2f}) "
                      f"already diverges {abs(float(_sp_season_era) - float(_sp_peripheral)):.2f} runs from xFIP/SIERA "
                      f"({float(_sp_peripheral):.2f}), so ERA isn't a trustworthy read for him in general -- "
                      f"verify the split independently before trusting the SP edge{_sp_oppq_str}{_sp_coors_str}")
            else:
                # BUG FIX (2026-09-04, user audit -- M25): this line used to claim the
                # home/road split was "already priced into the SP ELO above". Traced the
                # actual code before touching it (real case: Zebby Matthews, 2.88 home ERA
                # / 6.46 road ERA -- confirmed live via MLB Stats API): _recency_adj (which
                # produces the displayed ELO/tier) takes hist_elo/season-ERA/L3-ERA/L9-ERA/
                # xERA/sample-size only -- no home/away parameter anywhere, and its own
                # in-code comment says home/road split noise deliberately should NOT bleed
                # into the grade. The run-projection formula's SP runs-allowed columns
                # (_a_sp_ra/_h_sp_ra) also have no home/road-split preference, unlike the
                # offense columns (_a_off/_h_off) which do. So this signal isn't priced in
                # anywhere -- the old claim was false reassurance, not just imprecise.
                # 2026-09-10 (user audit, item 1): the run-projection wiring this
                # comment used to say didn't exist now does (Cell 88's
                # _sp_split_ra_ratio, gated on the exact same 1.5R/8-start bar
                # checked here) -- this bullet now tells the truth about whether
                # THIS specific pitcher's split actually got applied tonight,
                # instead of unconditionally claiming it never is.
                _sp_split_starts = _sv.get('home_starts', 0) if _sp_is_home else _sv.get('road_starts', 0)
                _sp_split_applied = _sp_split_starts >= 8
                if _sp_split_applied:
                    _sp_split_note = (f"-- APPLIED to tonight's run projection above "
                                       f"({_sp_split_starts} {_venue} starts, ratio vs season ERA)")
                else:
                    _sp_split_note = (f"-- NOT applied (only {_sp_split_starts} {_venue} starts, "
                                       f"need 8+ before this split is trusted) -- informational only, "
                                       f"gap is large enough to weigh independently before trusting this edge")
                print(f"  [SP SPLIT] {_sp_nm} ({_sp_side}) pitches {_direction} at {_venue} today "
                      f"({_sp_he:.2f} ERA/{_sv.get('home_ip', 0):.1f} IP home vs "
                      f"{_sp_re:.2f} ERA/{_sv.get('road_ip', 0):.1f} IP road) "
                      f"{_sp_split_note}{_sp_oppq_str}{_sp_coors_str}")

    # SP CONTACT QUALITY FLAG (2026-08-23, user audit): hard_hit_pct/
    # barrel_pct now print in the SP Matchup panel (Cell 88's
    # _sp_stat_line) but nothing flagged when they're actually alarming --
    # a pitcher allowing 40%+ hard contact printing next to a clean-
    # looking FIP is exactly the kind of quality-of-contact/results
    # disconnect that shouldn't require the reader to spot it themselves
    # in a wall of numbers.
    _HH_FLAG_THRESHOLD = 40.0
    # FIX 14 (2026-08-27, user audit -- Cole's 42.3% flag): "elevated HH% can
    # precede a results-vs-peripherals reversal" gave no threshold to weigh
    # it against and no historical accuracy rate -- fabricating a "Y% of
    # cases" backtest number isn't honest when none exists in this notebook
    # (no HH%-streak-vs-subsequent-ERA study has been run here). What IS
    # real and already sitting in mlb_pitcher_fip_lookup: today's actual
    # league-wide HH% distribution across every pitcher Baseball Savant
    # returned (Cell 21) -- so the flag now anchors the number to a real,
    # same-run league average instead of a bare unexplained cutoff, and is
    # explicitly labeled unvalidated rather than implying a proven signal
    # (same honesty convention already used for WNBA's 3PT-regression WATCH
    # flag: "unvalidated signal, does not reliably predict reversion").
    _hh_league_vals = [
        float(v["hard_hit_pct"]) for v in mlb_pitcher_fip_lookup.values()
        if v.get("hard_hit_pct") is not None and not pd.isna(v.get("hard_hit_pct"))
    ] if "mlb_pitcher_fip_lookup" in globals() else []
    _hh_league_avg = (sum(_hh_league_vals) / len(_hh_league_vals)) if _hh_league_vals else None
    for _hhq_nm in (_f_aw_sp, _f_hm_sp):
        if not _hhq_nm:
            continue
        _hhq_entry = None
        if "mlb_pitcher_fip_lookup" in globals():
            _hhq_nk = _mlb_name_norm(_hhq_nm)
            _hhq_entry = mlb_pitcher_fip_lookup.get(_hhq_nk)
            if _hhq_entry is None:
                _hhq_entry = next((v for k, v in mlb_pitcher_fip_lookup.items() if _mlb_name_norm(k) == _hhq_nk), None)
        _hhq_val = _hhq_entry.get("hard_hit_pct") if _hhq_entry else None
        if _hhq_val is not None and float(_hhq_val) >= _HH_FLAG_THRESHOLD:
            _hhq_fip = _hhq_entry.get("fip")
            _hhq_fip_str = f" (FIP {float(_hhq_fip):.2f})" if _hhq_fip is not None else ""
            _hhq_league_str = (f" -- league average today is {_hh_league_avg:.1f}%, "
                                f"{float(_hhq_val) - _hh_league_avg:+.1f}pp above it"
                                if _hh_league_avg is not None else "")
            print(f"  !! SP CONTACT QUALITY: {_hhq_nm} allowing {float(_hhq_val):.1f}% hard contact{_hhq_fip_str}"
                  f"{_hhq_league_str} -- unvalidated signal, no backtested rate of subsequent ERA regression "
                  f"exists in this notebook; treat as context, not a standalone reason to fade")

    # 2026-08-24 (user audit): POST-IL RETURN flag, mirrors the edge shave
    # applied in Cell 88 for the same thin-post-return-sample SPs (any IL
    # length, not just the existing 60-day-only staleness gate above).
    for _pil_nm in (_f_aw_sp, _f_hm_sp):
        if not _pil_nm:
            continue
        _pil_fn = globals().get("_post_il_info")
        _pil_info = _pil_fn(_pil_nm) if _pil_fn else None
        if _pil_info:
            _pil_days = _pil_info.get("days_since")
            _pil_days_str = f"Day {_pil_days}" if _pil_days is not None else "day unknown"
            print(f"  !! POST-IL RETURN: {_pil_nm} ({_pil_days_str} since return, {_pil_info.get('ip_since')} IP) -- "
                  f"edge discounted for thin post-return sample")

    # M15/M16 (2026-09-04, user audit -- McClanahan Sept 3 loss): display
    # companions for the grade/size caps applied in Cell 87's grading loop
    # (that cell's own prints are an internal build log, never reach this
    # card -- same reason POST-IL RETURN above needed its own display copy).
    _m15_hist_fn = globals().get("_sp_early_exit_history")
    if _m15_hist_fn:
        for _m15_nm, _m15_opp, _m15_is_home in [(_f_aw_sp, _f_hm, False), (_f_hm_sp, _f_aw, True)]:
            if not _m15_nm:
                continue
            _m15_hist = _m15_hist_fn(_m15_nm, _m15_opp, _m15_is_home)
            if _m15_hist:
                _m15_park_str = "at this same park" if _m15_hist["same_park"] and not _m15_hist["same_opponent"] else f"vs {_m15_opp}"
                print(f"  !! [SP INJURY HISTORY] {_m15_nm}: exited early ({_m15_hist['ip']:.1f} IP) "
                      f"{_m15_park_str} on {_m15_hist['date']}, placed on the IL {_m15_hist['il_date']} -- "
                      # MLB-INJ-GRADECAP-1 (2026-09-24 user audit): Cell 87's own
                      # cap for this now shaves edge (-7%) and re-derives the
                      # letter grade from that, rather than hard-pinning B+ -- this
                      # display companion text updated to match so it doesn't
                      # describe a mechanism that no longer exists.
                      f"real injury-recurrence risk, not just general uncertainty; edge discounted -7% if this is the pick")

    if "_sp_recent_il_return_players" in globals() and _sp_recent_il_return_players:
        for _m16_nm in (_f_aw_sp, _f_hm_sp):
            if not _m16_nm:
                continue
            _m16_k = _sp_norm_era(_m16_nm) if "_sp_norm_era" in globals() else _m16_nm
            if _m16_k not in _sp_recent_il_return_players:
                continue
            _m16_is_bulk = False
            try:
                _m16_is_bulk = bool(_sp_role_flags(_m16_nm)[1]) if "_sp_role_flags" in globals() else False
            except Exception:
                pass
            if not _m16_is_bulk:
                continue
            _m16_starts = _sp_starts_since_return(_m16_nm, _sp_recent_il_return_players[_m16_k]) if "_sp_starts_since_return" in globals() else None
            if _m16_starts is not None and _m16_starts < 5:
                print(f"  !! [SP POST-IL BULK ROLE] {_m16_nm}: bulk/short-role (3-5 IP leash) with only "
                      f"{_m16_starts} start(s) logged since returning from IL -- the bullpen decides a big "
                      f"chunk of this game regardless of how he pitches; size capped at 0.5u if this is the pick")

    # M31 (2026-09-06, user audit -- Kikuchi Sept 5 shutout of the Pirates
    # pick): display companion, see the function docstring in Cell 87 for
    # full rationale -- this cell's copy is the one that actually reaches
    # the digest (Cell 87's own prints are an internal build log only).
    _m31_trend_fn = globals().get("_sp_post_il_trend")
    if _m31_trend_fn and "_sp_recent_il_return_players" in globals() and _sp_recent_il_return_players:
        for _m31_nm in (_f_aw_sp, _f_hm_sp):
            if not _m31_nm:
                continue
            _m31_k = _sp_norm_era(_m31_nm) if "_sp_norm_era" in globals() else _m31_nm
            if _m31_k not in _sp_recent_il_return_players:
                continue
            _m31_trend = _m31_trend_fn(_m31_nm, _sp_recent_il_return_players[_m31_k])
            if _m31_trend:
                _m31_line = " -> ".join(f"{s['er']:.0f} ER/{s['ip']:.1f} IP" for s in _m31_trend["starts"])
                print(f"  !! [SP POST-IL TREND] {_m31_nm}: {_m31_trend['trend']} every start since returning "
                      f"from IL ({_m31_line}) -- small sample, not backtested, but the flat L3 average "
                      f"above doesn't show this trend")

    # 2026-08-24 (user audit): HR/9 collapse-risk flag, mirrors the edge
    # shave applied in Cell 88. HR is already weighted inside the FIP-based
    # SP ELO (13x coefficient), but hr9_l3 itself was computed and never
    # surfaced anywhere -- this makes an elevated rate explicit.
    _hr9d_threshold = globals().get("_HR9_FLAG_THRESHOLD", 1.8)
    for _hr9f_nm in (_f_aw_sp, _f_hm_sp):
        if not _hr9f_nm:
            continue
        _hr9f_data = _sp_fetch_recent(_hr9f_nm) if "_sp_fetch_recent" in globals() else None
        _hr9f_val = (_hr9f_data or {}).get("hr9_l3")
        if _hr9f_val is not None and _hr9f_val >= _hr9d_threshold:
            print(f"  !! HR RATE ELEVATED: {_hr9f_nm} allowing {_hr9f_val:.2f} HR/9 (L3) -- "
                  f"collapse-elevated, edge discounted for this")
    # 2026-08-25 (user audit, Ethan Pecko): mirrors the edge shave applied
    # in Cell 88 -- a pitcher graded off fewer than 5 career MLB starts
    # (Pecko: 1 career start, coming off thoracic outlet surgery) can still
    # land in the AVG tier off Cell 83's small-sample ELO regression, which
    # softens the ELO NUMBER but was never flagged on the card itself.
    _thin_min_starts = globals().get("_THIN_SAMPLE_MIN_STARTS", 5)
    for _thin_nm in (_f_aw_sp, _f_hm_sp):
        if not _thin_nm:
            continue
        _thin_tr = mlb_sp_tracker.get(_thin_nm) if "mlb_sp_tracker" in globals() else None
        _thin_starts = (_thin_tr or {}).get("starts")
        if _thin_starts is not None and _thin_starts < _thin_min_starts:
            print(f"  !! THIN MLB SAMPLE: {_thin_nm} has {_thin_starts} career MLB start(s) (< {_thin_min_starts}) -- "
                  f"edge discounted for uncertainty, tier grade should not be read as an established rate")

    # MLB-KADE-1 (2026-09-14, user audit -- Kade Anderson, FIP 5.20/SIERA
    # 4.81 off 4 career starts / 21.0 IP): the THIN MLB SAMPLE check above
    # already discounts the edge off a <5-start count, but FIP/SIERA
    # stabilize (or don't) off innings, not starts -- a pitcher could clear
    # the 5-start bar on a run of short outings and still carry meaningless
    # peripherals, or fall short of it on a couple of deep ones and be fine.
    # This is a distinct, IP-denominated caveat about trusting the specific
    # FIP/xFIP/SIERA numbers printed above (not a second uncertainty
    # discount) -- reuses mlb_sp_tracker's own per-start IP log (Cell 82),
    # nothing new to fetch.
    # BUG FIX (2026-09-14, found live while building the FIP-vs-Statcast
    # display swap above -- mlb_pitcher_fip_lookup's own FIP/xFIP/SIERA are
    # FanGraphs SEASON rate stats (denominator ip_season), but this flag was
    # gating on mlb_sp_tracker's multi-year CAREER ip_starts sum instead --
    # the wrong denominator for judging whether THESE SPECIFIC numbers are
    # stabilized. Coincidentally identical for a rookie (career IP == season
    # IP, which is why Anderson/Tong/Mathews/Newcomb all flagged correctly
    # already), but would silently miss a real case -- a veteran back from
    # injury with a long career but a thin 2026 sample -- since his large
    # career total would clear the bar his real season sample doesn't.
    # _LOW_STAB_MIN_IP now defined once, globally, in Cell 87 (M-PROSPCALL-1)
    # so the flag and the display swap can never drift out of sync.
    for _lowstab_nm in (_f_aw_sp, _f_hm_sp):
        if not _lowstab_nm:
            continue
        _lowstab_nk = _mlb_name_norm(_lowstab_nm) if "_mlb_name_norm" in globals() else _lowstab_nm.lower()
        _lowstab_fev = mlb_pitcher_fip_lookup.get(_lowstab_nk) if "mlb_pitcher_fip_lookup" in globals() else None
        if _lowstab_fev is None and "mlb_pitcher_fip_lookup" in globals() and "_mlb_name_norm" in globals():
            _lowstab_fev = next((v for k, v in mlb_pitcher_fip_lookup.items() if _mlb_name_norm(k) == _lowstab_nk), None)
        _lowstab_ip = (_lowstab_fev or {}).get("ip_season")
        _lowstab_ip_src = "MLB IP this season"
        if _lowstab_ip is None or pd.isna(_lowstab_ip):
            # fallback: no FanGraphs season-IP record at all -- fall back to
            # the career tracker sum rather than silently skipping the flag
            _lowstab_tr = mlb_sp_tracker.get(_lowstab_nm) if "mlb_sp_tracker" in globals() else None
            _lowstab_ip = sum((_lowstab_tr or {}).get("ip_starts", []) or [])
            _lowstab_ip_src = "career MLB IP"
        _low_stab_min_ip = globals().get("_LOW_STAB_MIN_IP", 40.0)
        if _lowstab_ip and _lowstab_ip < _low_stab_min_ip:
            print(f"  !! LOW-STABILITY PERIPHERALS: {_lowstab_nm} has {_lowstab_ip:.1f} {_lowstab_ip_src} "
                  f"(< {_low_stab_min_ip:.0f}) -- FIP/xFIP/SIERA shown above are not yet stabilized at "
                  f"this sample and should be read with a wide confidence interval, not the false "
                  f"precision of the printed number; Statcast contact-quality rates (xwOBA, Hard Hit%) "
                  f"are more trustworthy peripherals at this innings level")


    # 2026-08-24 (user audit, Kevin Gausman): surfaces two SP health checks
    # that already run in Cell 88 but never reach the published card -- the
    # digest extractor only keeps the notebook's last 2 output-bearing
    # cells plus any cell containing "FINAL CARD" text, and Cell 88 (where
    # these print today) is neither, so !! SP HEALTH FLAG has been silently
    # invisible on every card since it was built. Re-derives the same data
    # here where it's actually visible, plus the new 10-day name-specific
    # news check above for a pitcher (e.g. Gausman, hand injury 5 days
    # before a confirmed start, no IL stint at all) the existing
    # IL-transaction-based check can't see.
    for _spiw_nm in (_f_aw_sp, _f_hm_sp):
        if not _spiw_nm:
            continue
        _spiw_parts = _spiw_nm.lower().split()
        _spiw_last = _spiw_parts[-1] if _spiw_parts else ""
        _sp_il_flags_g = globals().get("_sp_il_flags", {})
        for _ilk, _ilv in _sp_il_flags_g.items():
            if len(_spiw_last) > 3 and _spiw_last in _ilk:
                print(f"  !! SP HEALTH FLAG: {_spiw_nm} -- {_ilv}")
                break
        _sp_news_fn = globals().get("_fetch_sp_injury_news")
        _spiw_hits = _sp_news_fn(_spiw_nm) if _sp_news_fn else []
        if _spiw_hits:
            print(f"  !! SP INJURY WATCH: {_spiw_nm} -- recent news: {_spiw_hits[0]}")

    # SP RECENT-FORM CREDIBILITY (2026-08-14, same QA as the split flag
    # above). L3 ERA alone can't tell a real form change from a 3-start
    # hot/cold blip; the model's own recency blend (Cell 88's _recency_adj)
    # already cross-checks L3 against xERA (quality-of-contact, not
    # results-based) before trusting it, but that verdict never reached the
    # card. Real case: Grayson Rodriguez showed a hot 3.94 L3 ERA against a
    # 7.49 season ERA -- read like a real turnaround, and public coverage
    # was framing it that way -- but his xERA (6.80) sat right next to the
    # ugly season number, meaning the streak was mostly sequencing luck,
    # not a talent change. The model already discounted this correctly;
    # this surfaces why, and would also catch the opposite case (a cold
    # streak that xERA says isn't real either).
    def _wf_lookup(nm):
        if not nm:
            return None
        _k = _mlb_name_norm(nm)
        if "mlb_pitcher_fip_lookup" in globals():
            _v = mlb_pitcher_fip_lookup.get(_k)
            if _v:
                return _v
        if "_sp_form_cache" in globals() and _sp_form_cache:
            for _ck, _cv in _sp_form_cache.items():
                if _cv and _mlb_name_norm(_ck) == _k:
                    return _cv
        return None
    for _sp_nm, _sp_side in [(_f_aw_sp, _f_aw), (_f_hm_sp, _f_hm)]:
        if not _sp_nm:
            continue
        _fc = _wf_lookup(_sp_nm)
        if not _fc:
            continue

        # FIP RELIABILITY (2026-09-01, user audit -- Randy Dobnak, Royals ML
        # loss): the SP ELO/tier shown above is deliberately FIP-based (Cell
        # 82's _sp_elo_delta -- HR/BB/HBP/K only, not hits/ERA), which is
        # sound in principle, but nothing ever checked FIP itself against
        # the more predictive xFIP/SIERA also already sitting in this same
        # lookup (Cell 19's FanGraphs fetch pulls all three from one row).
        # Real case: Dobnak showed STRONG/1555 (ERA 1.93, FIP 3.74) --
        # exactly what carried the edge on this play -- while his own xFIP
        # (4.24) and SIERA (4.67) were both close to a full run worse,
        # meaning even the FIP his tier is built on was flattering him
        # relative to his skill profile. He gave up 6 ER on 2 HR in the
        # actual start. This doesn't replace the ELO (still FIP-based,
        # unchanged) -- it surfaces exactly the gap that was already sitting
        # in already-fetched data with nothing pointing at it.
        _fip_v = _fc.get("fip")
        if _fip_v is not None:
            try:
                _fip_f = float(_fip_v)
                _xfip_f = float(_fc["xfip"]) if _fc.get("xfip") is not None else None
                _siera_f = float(_fc["siera"]) if _fc.get("siera") is not None else None
            except (TypeError, ValueError):
                _fip_f = _xfip_f = _siera_f = None
            if _fip_f is not None:
                # 2026-09-02: threshold/shave now match the real edge discount
                # applied upstream (Cell 87's _fip_reliability_shave) -- this
                # print is a mirror of what already happened to the edge, same
                # pattern as the HR9/sustained-cold flags above it.
                _FIP_RELIABILITY_GAP = 0.50
                _worse = {}
                if _xfip_f is not None and _xfip_f - _fip_f >= _FIP_RELIABILITY_GAP:
                    _worse["xFIP"] = _xfip_f
                if _siera_f is not None and _siera_f - _fip_f >= _FIP_RELIABILITY_GAP:
                    _worse["SIERA"] = _siera_f
                if _worse:
                    _worse_str = " / ".join(f"{k} {v:.2f}" for k, v in _worse.items())
                    _worst_gap = max(v - _fip_f for v in _worse.values())
                    print(f"  !! [FIP RELIABILITY] {_sp_nm} ({_sp_side}): FIP {_fip_f:.2f} (drives the SP "
                          f"ELO/tier above) sits {_worst_gap:.2f}R better than {_worse_str} -- more "
                          f"predictive peripherals read this pitcher as meaningfully worse than his tier "
                          f"implies, edge discounted for this (provisional, no backtest yet)")

        # SP WORKLOAD / INNINGS-FATIGUE (2026-09-19, user audit -- Suarez
        # 137.7 IP vs Seymour 67.0 IP, both shown as the same AVG ELO tier):
        # the ELO/tier above is built purely on cumulative FIP performance --
        # it has no innings-based recency/fatigue decay, so a pitcher who was
        # excellent April-July but has quietly been declining since could still
        # carry a tier that overstates his current ability late in the season.
        # Flag-only, deliberately: there is no backtest data yet on how much
        # (if any) run-projection or edge impact real September workload should
        # have, so this does NOT touch the ELO, run projection, or edge -- same
        # discipline as every other "provisional, no backtest yet" flag on this
        # card. 120 IP is the user's own stated threshold, not a fitted number.
        _SP_WORKLOAD_IP_THRESHOLD = 120.0
        _ip_szn_v = _fc.get("ip_season")
        if _ip_szn_v is not None:
            try:
                _ip_szn_f = float(_ip_szn_v)
            except (TypeError, ValueError):
                _ip_szn_f = None
            if _ip_szn_f is not None and _ip_szn_f >= _SP_WORKLOAD_IP_THRESHOLD:
                # BUG FIX (MLB-SP-WORKLOAD-1, 2026-09-26 user audit): this flag
                # is still deliberately informational-only for the ELO TIER
                # itself (that part is unchanged -- still no backtest on how
                # much a fade should move the tier). But Cell 87's run-
                # projection blend now DOES discount a pitcher who has passed
                # his own prior career-high IP (a stricter, more specific bar
                # than this flat 120-IP threshold) -- surfacing that here too
                # so the two don't read as contradictory when both are true.
                # BUG FIX (found live twice verifying this exact fix): the
                # board-column approach (away/home_sp_workload_worn) doesn't
                # reliably survive whatever rebuild produces _snap_ranked/
                # _snap_other_ranked by the time _emit_why runs (same class of
                # gap this cell's own [H2H 2026]/[RECORD GAP] fixes worked
                # around earlier -- board columns get silently dropped a few
                # cells downstream). Sidesteps that entirely: reuses the SAME
                # real global cache Cell 87's _sp_workload_exceeds_career_high
                # already populates (keyed identically, lowercased player
                # name) instead of depending on a column making the round trip.
                _wl_cache = globals().get("_SP_WORKLOAD_CAREER_HIGH_CACHE", {})
                _wl_prior_high = _wl_cache.get(str(_sp_nm).strip().lower())
                _wl_applied = _wl_prior_high is not None and _ip_szn_f > _wl_prior_high
                _wl_note = (" -- ALSO past his own career-high IP: run-projection weight already shifted "
                            "toward the bullpen for this, see Proj Score above" if _wl_applied else
                            " -- informational only here (ELO tier unaffected, unvalidated/no backtest)")
                print(f"  !! [SP WORKLOAD] {_sp_nm} ({_sp_side}): {_ip_szn_f:.1f} IP this season -- past "
                      f"the {_SP_WORKLOAD_IP_THRESHOLD:.0f}-IP fatigue-risk threshold. Season-long ELO has "
                      f"no innings-workload decay, so a late-season fade wouldn't show up in the tier above"
                      f"{_wl_note}")

        try:
            _l3 = float(_fc["era_l3"]) if _fc.get("era_l3") is not None else None
            _l3ip = float(_fc["ip_l3"]) if _fc.get("ip_l3") is not None else None
            _seas = float(_fc["era_season"]) if _fc.get("era_season") is not None else None
            _xera = float(_fc["xera"]) if _fc.get("xera") is not None else None
        except (TypeError, ValueError):
            continue
        if _l3 is None or _seas is None or _l3ip is None or _l3ip < 12.0:
            continue  # same 12 IP L3 floor Cell 88's recency blend requires
        _gap = _seas - _l3  # positive = hot (L3 better than season), negative = cold
        if abs(_gap) < 1.0:
            continue
        _label = "HOT" if _gap > 0 else "COLD"
        # 2026-08-25 (user audit, Will Warren): L3 alone reads a 3-start blip
        # and a genuine multi-start collapse identically -- Warren went 1-5,
        # 5.77 ERA over his last 13 appearances, well beyond what L3 can see.
        # era_l10 (Cell 88 _sp_fetch_recent) independently confirms whether
        # the cold read holds over a longer window, appended alongside the
        # xERA verdict rather than replacing it -- both are real signals.
        _l10_note = ""
        if _label == "COLD":
            _l10c = _fc.get("era_l10")
            _gs10c = _fc.get("gs_l10") or 0
            # BUG FIX (same session): _fc came from _wf_lookup, which checks
            # mlb_pitcher_fip_lookup FIRST and returns on any hit -- that source
            # has no era_l10/gs_l10 at all (those only live in _sp_form_cache, via
            # Cell 88 _sp_fetch_recent), so for any pitcher already resolved from
            # mlb_pitcher_fip_lookup (most rostered starters, including Warren)
            # _l10c was silently always None and this note could never fire.
            # Falls back to a direct _sp_form_cache scan when the L10 field is
            # missing from whichever source _fc actually resolved to.
            if _l10c is None and "_sp_form_cache" in globals() and _sp_form_cache:
                _l10_key = _mlb_name_norm(_sp_nm)
                for _ck, _cv in _sp_form_cache.items():
                    if _cv and _cv.get("era_l10") is not None and _mlb_name_norm(_ck) == _l10_key:
                        _l10c = _cv.get("era_l10")
                        _gs10c = _cv.get("gs_l10") or 0
                        break
            _l10_thresh = globals().get("_SUSTAINED_COLD_L10_THRESHOLD", 5.00)
            if _l10c is not None and _gs10c >= 5 and float(_l10c) >= _l10_thresh:
                _l10_note = (f" | SUSTAINED SLUMP: L10 ERA {float(_l10c):.2f} over {_gs10c} starts confirms "
                             f"this isn't just a 3-start blip -- edge separately discounted for this")
        # SP COLD-STREAK SHAPE (2026-08-31, user audit -- George Kirby): xERA
        # alone only tests whether the L3 RESULTS matched the process
        # (sequencing/BABIP luck) -- it says nothing about the SHAPE of the
        # 3 starts themselves. Real case: Kirby's L3 ERA 6.89 came from one
        # 6-run collapse start followed by two starts back within his normal
        # range, not three consecutive bad outings -- a materially
        # different (and less concerning) risk profile than a genuine
        # sustained decline, even though the xERA gap alone reads both
        # cases identically as "not supported / sequencing luck." Per-start
        # ER/IP were always fetched to build the L3 aggregate but discarded
        # before reaching here; captured at the source (l3_starts_er/
        # l3_starts_ip, both fetchers -- see their comments) and used below
        # to tell the two shapes apart.
        _shape_note = ""
        if _label == "COLD":
            _l3_er_shape = _fc.get("l3_starts_er")
            _l3_ip_shape = _fc.get("l3_starts_ip")
            if (not _l3_er_shape or not _l3_ip_shape) and "_sp_form_cache" in globals() and _sp_form_cache:
                _shape_key = _mlb_name_norm(_sp_nm)
                for _ck, _cv in _sp_form_cache.items():
                    if _cv and _cv.get("l3_starts_er") and _mlb_name_norm(_ck) == _shape_key:
                        _l3_er_shape = _cv.get("l3_starts_er")
                        _l3_ip_shape = _cv.get("l3_starts_ip")
                        break
            if _l3_er_shape and _l3_ip_shape and len(_l3_er_shape) == 3 and len(_l3_ip_shape) == 3:
                _per_start_era = [
                    (round(_er_i / _ip_i * 9, 2) if _ip_i and _ip_i > 0 else None)
                    for _er_i, _ip_i in zip(_l3_er_shape, _l3_ip_shape)
                ]
                _blowups = [e for e in _per_start_era if e is not None and e >= 6.00]
                _normal = [e for e in _per_start_era if e is not None and e < 4.50]
                _era_str = ", ".join(f"{e:.2f}" if e is not None else "?" for e in _per_start_era)
                if len(_blowups) == 1 and len(_normal) == 2:
                    _shape_note = (f" | SHAPE: one blowup start ({_era_str} ERA by start, oldest-to-newest) "
                                    f"dragging the L3 average -- other 2 starts within normal range, not a "
                                    f"sustained decline")
                elif len(_blowups) >= 2:
                    _shape_note = (f" | SHAPE: {len(_blowups)} individually bad starts within L3 "
                                    f"({_era_str} ERA by start) -- a real trend, not one outlier")
        if _xera is not None:
            _supported = abs(_l3 - _xera) < 1.0
            _verdict = ("supported by xERA -- likely a real form change" if _supported
                        else f"xERA ({_xera:.2f}) sits closer to season ERA -- likely sequencing luck, not a true change")
            print(f"  [SP {_label} STREAK] {_sp_nm} ({_sp_side}): L3 ERA {_l3:.2f} vs season {_seas:.2f} -- {_verdict}{_l10_note}{_shape_note}")
        else:
            print(f"  [SP {_label} STREAK] {_sp_nm} ({_sp_side}): L3 ERA {_l3:.2f} vs season {_seas:.2f} -- no xERA available to verify if this is real{_l10_note}{_shape_note}")

    # Coors Field altitude flag (audit fix, 2026-08-04) -- informational.
    # The run-environment adjustment itself already exists and is real:
    # _PARK_FACTOR["rockies"] = 1.32 feeds _env = _pf * _tf * _wf, which
    # is baked into pred_home_runs/pred_away_runs at board-build time --
    # so both the total AND the score-blended ML probability for any
    # Colorado home game already account for altitude. Nothing on the
    # card ever surfaced that fact, though, and the raw SP ERA lines
    # printed just above are NOT park-adjusted -- a Coors starter's
    # home-heavy ERA reads worse than a same-quality pitcher elsewhere,
    # and an opposing pitcher's Coors starts read worse too. This is a
    # reminder to sanity-check the SP matchup line above with that in
    # mind, not a second/duplicate adjustment.
    _cf_hm_norm = _pl_norm(_f_hm) if "_pl_norm" in globals() else _f_hm.lower()
    if "rockies" in _cf_hm_norm or "colorado" in _cf_hm_norm:
        _cf_pf = _PARK_FACTOR.get("rockies", 1.32) if "_PARK_FACTOR" in globals() else 1.32
        print(f"  [ALTITUDE] Coors Field home game -- park factor {_cf_pf:.2f}x already applied to "
              f"run projections/total above, but the raw SP ERA lines are NOT park-adjusted -- "
              f"verify SP quality isn't just a Coors artifact before trusting the matchup edge")

    # CROSS-MODEL FIX 12/18 (2026-08-27, user audit): _ESPN_INJURY_FETCH_OK /
    # _ESPN_NEWS_FETCH_OK (Cell 87) already exist to pick a fallback source,
    # but neither ever reached this "WHY" section -- a 403 on either feed
    # silently degraded to "no injury data" with the card printing exactly
    # like a clean night. Surfaced explicitly here so a bettor reading this
    # play's reasoning knows the injury picture is unverified, not clean.
    _f_espn_inj_ok = globals().get("_ESPN_INJURY_FETCH_OK")
    _f_espn_news_ok = globals().get("_ESPN_NEWS_FETCH_OK")
    if _f_espn_inj_ok is False or _f_espn_news_ok is False:
        _f_degraded_src = []
        if _f_espn_inj_ok is False: _f_degraded_src.append("injuries")
        if _f_espn_news_ok is False: _f_degraded_src.append("news/day-to-day")
        print(f"  !! [DATA SOURCE FAILURE] ESPN {', '.join(_f_degraded_src)} fetch failed this run -- "
              f"injury inputs unverified, edge reliability degraded")

    # Injuries / lineup downgrades
    _f_log_net_inj = None
    if "mlb_lineup_flags" in globals() and mlb_lineup_flags and "_pl_norm" in globals():
        for _f_team in [_f_aw, _f_hm]:
            _f_flag = mlb_lineup_flags.get(_pl_norm(_f_team))
            if _f_flag:
                print(f"  [INJURY] {_f_team}: {_f_flag.get('msg', '')}")
    if "mlb_motivation_gap_flags" in globals() and mlb_motivation_gap_flags and "_pl_norm" in globals():
        # MLB-MOTIV-PARITY-1 (2026-09-14, user audit): both teams tonight
        # took the same ELIMINATED penalty and each printed its own
        # separate !! warning line -- visually reading as two red flags
        # against the play when the net effect on pred_total_runs is
        # exactly zero (the two penalties cancel). Collapse to one
        # neutral note whenever both teams share the same tier; keep the
        # normal per-team warnings whenever they differ, since that
        # asymmetry (one team fighting, one dead) is the real signal.
        _mg_aw_tier = mlb_motivation_gap_tiers.get(_pl_norm(_f_aw)) if "mlb_motivation_gap_tiers" in globals() else None
        _mg_hm_tier = mlb_motivation_gap_tiers.get(_pl_norm(_f_hm)) if "mlb_motivation_gap_tiers" in globals() else None
        if _mg_aw_tier and _mg_hm_tier and _mg_aw_tier[0] == _mg_hm_tier[0]:
            print(f"  [MOTIVATION: EQUAL -- both teams {_mg_aw_tier[0]}, net effect 0.0R]")
        else:
            for _f_team in [_f_aw, _f_hm]:
                _f_motgap = mlb_motivation_gap_flags.get(_pl_norm(_f_team))
                if _f_motgap:
                    print(f"  !! {_f_motgap}")
        for _f_team in [_f_aw, _f_hm]:
            _f_drift = globals().get("mlb_injury_drift", {}).get(_pl_norm(_f_team))
            if _f_drift:
                print(f"  {_f_drift}")

        # M21 (2026-09-04, user audit): lineup-not-confirmed caveat -- see
        # the function docstring/comment in Cell 87 for full rationale.
        # This cell's copy is the one that actually reaches the digest
        # (Cell 87's own prints are an internal build log only) -- same
        # reason the M15/M16 blocks above needed their own display copies.
        _m21_fn = globals().get("_lineup_confirm_notes")
        if _m21_fn:
            for _m21_note in _m21_fn(_f_row.get("game_datetime_utc"), _f_aw, _f_hm):
                print(f"  {_m21_note}")

        # Net injury delta (2026-08-01) -- the two per-team stacks above
        # (e.g. "Giants -0.71R, Padres -0.33R") required manual
        # subtraction to see which side the injury picture actually
        # favors. Surface it as one signed number instead.
        _f_aw_flag = mlb_lineup_flags.get(_pl_norm(_f_aw)) or {}
        _f_hm_flag = mlb_lineup_flags.get(_pl_norm(_f_hm)) or {}
        _f_aw_pen = float(_f_aw_flag.get("penalty", 0) or 0)
        _f_hm_pen = float(_f_hm_flag.get("penalty", 0) or 0)
        # FIX 2 (2026-09-22, user audit): confirmed-only (weight-1.0) mirror of
        # the blended penalty above, from the penalty_confirmed subtotal Cell
        # bd676982's injury-stack loop now also carries alongside 'penalty'.
        _f_aw_pen_confirmed = float(_f_aw_flag.get("penalty_confirmed", 0) or 0)
        _f_hm_pen_confirmed = float(_f_hm_flag.get("penalty_confirmed", 0) or 0)
        if _f_aw_pen or _f_hm_pen:
            _f_net_mag = abs(_f_aw_pen - _f_hm_pen)
            _f_net_favors = _f_hm if _f_aw_pen > _f_hm_pen else (_f_aw if _f_hm_pen > _f_aw_pen else None)
            _f_net_mag_confirmed = abs(_f_aw_pen_confirmed - _f_hm_pen_confirmed)
            _f_net_favors_confirmed = _f_hm if _f_aw_pen_confirmed > _f_hm_pen_confirmed else (_f_aw if _f_hm_pen_confirmed > _f_aw_pen_confirmed else None)
            # M27 (2026-09-05, user audit -- Marlins Sept 4 loss): this line used
            # to print the net delta with no caveat, even though a comment a few
            # lines below (2026-08-02 fix) already documents that p_bet_win_model
            # -- what actually drives ML edge/qualification -- is fixed upstream
            # before this penalty is even computed, and nothing downstream feeds
            # it back in; it only ever adjusts pred_away_runs/pred_home_runs,
            # which feeds O/U, not ML. Real case that resurfaced this: Marlins ML
            # went from a +0.30R injury tailwind to a -0.10R headwind intra-day
            # (Caissie/Stowers confirmed out) and the ML edge barely moved --
            # correctly so, since it was never wired to this number at all in
            # either direction. Not turning this into a hard downgrade/gate (the
            # 08-02 comment's own reference case -- Pirates ML, -0.80R against
            # the pick, still won -- means the evidence for gating on this is
            # still weak, now n=3 with a real counterexample), just making the
            # existing blind spot visible on the card itself instead of only in
            # a code comment nobody placing a bet ever sees.
            # MLB-INJ-CONF-1 (2026-09-14): used to say ML picks never price this
            # in -- confirmed via the model's own feature list that was a genuine
            # gap, not a deliberate choice, and it's now actually priced into
            # _c_model/_c_edge upstream (DAILY MARKET SNAPSHOT section) whenever
            # the delta is material (>=0.20R). Wording reflects that instead of
            # the old blanket disclaimer.
            _m27_ml_note = (" (priced into the win probability/edge above when >=0.20R -- "
                             "see [NET INJURY DELTA PRICED IN] below if it fired)") if _frow.get("market") == "ML" else ""
            if _f_net_favors:
                print(f"  [NET INJURY DELTA] -{_f_net_mag:.2f}R favor {_f_net_favors}{_m27_ml_note}")
            else:
                print(f"  [NET INJURY DELTA] 0.00R -- injury stacks offset{_m27_ml_note}")
            # FIX 2 (2026-09-22, user audit): confirmed-only mirror of the blended
            # line just printed -- same favor/sign/rounding conventions.
            if _f_net_favors_confirmed:
                print(f"  [NET INJURY DELTA (CONFIRMED ONLY)] -{_f_net_mag_confirmed:.2f}R favor {_f_net_favors_confirmed}")
            else:
                print(f"  [NET INJURY DELTA (CONFIRMED ONLY)] 0.00R -- injury stacks offset")
            # signed capture for ledger logging below (positive = favors the pick)
            if _frow.get("market") == "ML":
                if _f_net_favors is None:
                    _f_log_net_inj = 0.0
                else:
                    _f_log_net_inj = (_f_net_mag if _pl_norm(_f_net_favors) == _pl_norm(str(_frow.get("side", "")))
                                       else -_f_net_mag)

            # 2026-08-02 fix (found live, real card 2026-08-01): p_bet_win_model
            # -- the number that actually drives ML edge/qualification -- is fixed
            # upstream (cell 33's saved classifier scoring) before this lineup-
            # injury penalty is even computed, and nothing downstream feeds this
            # penalty back into it (it only ever adjusts pred_away_runs/pred_home_
            # runs, which feeds O/U, not ML). So an ML pick's own win probability
            # can be, and was, blind to a same-day injury picture the card prints
            # right below it. Real case: Giants ML at 64.3%/+22.5% edge, tagged
            # OFFICIAL, with a -0.38R net delta favoring the Padres -- lost.
            # NOT turned into a hard suppression gate (unlike the SP-ELO-gap
            # mismatch gate above) because the same card's Pirates ML play had an
            # even bigger net delta against the pick (-0.80R) and won -- one day
            # isn't enough evidence this signal reliably predicts outcomes, only
            # that it's real information the ML side currently can't see. Surfaced
            # as a loud warning, same informational-only pattern as every other
            # flag in this section, until it's been backtested enough to justify
            # actually gating a play on it.
            if _frow.get("market") == "ML" and _f_net_favors:
                _f_pick_norm = _pl_norm(str(_frow.get("side", "")))
                _f_favors_norm = _pl_norm(_f_net_favors)
                # MLB-INJ-CONF-1 (2026-09-14, escalated to 'fix the architecture'):
                # this used to independently re-derive an honest win prob here (FIX
                # 11, 2026-08-27) and revoke OFFICIAL status if the conflict-adjusted
                # edge failed to clear the bar -- necessary back then because
                # _frow['edge']/['model_prob'] were still the RAW, injury-blind
                # numbers at this point in the pipeline. They no longer are: the
                # DAILY MARKET SNAPSHOT section above now prices any real (>=0.20R)
                # net injury delta directly into _c_model/_c_edge before ranking,
                # tagging, or ledger-logging ever happen, so whatever reached here as
                # _frow already reflects it -- redoing the same math here would double-
                # apply the same delta. This block is now purely informational: it
                # names the direction (conflict vs tailwind) and echoes the
                # before/after note the snapshot section already computed.
                if _f_pick_norm and _f_favors_norm and _f_net_mag >= 0.20:
                    if _f_pick_norm != _f_favors_norm:
                        print(f"  !! [INJURY VS PICK CONFLICT] net injury delta favored {_f_net_favors} by "
                              f"{_f_net_mag:.2f}R against this pick -- already priced into the win "
                              f"probability/edge shown above, not an unpriced blind spot")
                    else:
                        print(f"  [INJURY TAILWIND] net injury delta favored this pick by {_f_net_mag:.2f}R "
                              f"-- already priced into the win probability/edge shown above")
                    if _frow.get("injury_adj_note"):
                        print(_frow["injury_adj_note"])

    # BUG FIX (2026-08-07, found while live-testing the injury-delta
    # logging above): _log_official_play (Cell 88, tracking/
    # mlb_official_plays_ledger.csv) was only ever called from Cell 88's
    # OWN separate "top_market"/"qualified_market" play-selection loop --
    # a different, older system from the Gate-driven cand/board pipeline
    # that actually produces THIS section (FINAL CARD -- OFFICIAL PLAYS +
    # WHY, i.e. _snap_ranked), which is the real card this whole project
    # has been reading and grading all session. Live-confirmed the two
    # disagree on a given night (2026-08-07: 0 "market" plays in the old
    # system, 2 real OFFICIAL plays here) -- meaning the actual official
    # plays traded off this card have never been logged to the ledger at
    # all. Logs the single true OFFICIAL play (not the qualifying pool --
    # per the card's own 1-play/sport/day rule, those aren't simultaneous
    # real bets) here, where the real pick is decided, with the injury
    # delta captured just above so this is genuinely backtestable once
    # enough rows accumulate. Gated on _f_tag == "OFFICIAL" specifically,
    # so extending this function to the qualifying pool and below-bar
    # informational edges (2026-08-14) can't log or downgrade those rows.
    if _f_tag == "OFFICIAL" and _fbe_capped:
        print(f"  !! BULLPEN EXPOSURE GATE: {_fbe_pick_sp} has never reached 5 IP in "
              f"{_fbe_rec['starts']} confirmed starts this season -- downgraded from "
              f"OFFICIAL, not logged as a real play (ML edge on a starter this short-outing "
              f"is measuring a bullpen game, not his own start)")
    # BUG FIX (2026-08-15, MLB ML calibration audit): model_prob/market_prob
    # were hardcoded to None here even though _c_model/_c_mkt (ML), _c_model/
    # _c_mkt (O/U), and _s_rl_model/_s_rl_mkt (RL) are computed and printed a
    # few lines above each _snap_rows.append -- they just never made it into
    # the dict, so every real official play logged since 2026-08-07 (when this
    # became the one true logger) wrote blank probabilities to the ledger.
    # Found while auditing the 2026-08-14-flagged ML record: 6 of 21 completed
    # official plays had no model_prob to check calibration against. Threaded
    # the already-computed values through _snap_rows for all three market
    # types instead.
    elif _f_tag == "OFFICIAL" and "_log_official_play" in globals():
        try:
            _log_official_play(
                card_date=pd.Timestamp.now().strftime("%Y-%m-%d"),
                game_id=_f_row.get(game_id_col) if game_id_col and game_id_col in _f_row.index else "",
                matchup=_frow.get("matchup", ""), market=_frow.get("market", ""),
                bet=str(_frow.get("side", "")), grade="",
                model_prob=_frow.get("model_prob"), market_prob=_frow.get("market_prob"), edge=_frow.get("edge"),
                size="OFFICIAL", net_injury_delta=_f_log_net_inj,
            )
        except Exception as _f_log_e:
            print(f"  [WARN] Official-play ledger write failed: {_f_log_e}")

    # GATE SAVE TRACKING call site (2026-09-20, user post-mortem FIX 7): a
    # below-bar row withheld specifically by the NO SP DATA gate gets logged here
    # too, using the exact same fields the OFFICIAL logger above uses.
    # BUG FIX (2026-09-20, live-verified: this originally checked "NO SP DATA" in
    # _pitcher_note(_frow), which is true for EVERY row with an unresolved starter
    # regardless of whether it would otherwise have qualified -- logged 14 rows in
    # one run, most with edges of 2.5-9%, nowhere near the real ~12% logit-space
    # floor, diluting the whole point of this metric (measuring genuine gate-saves,
    # not every unresolved-SP row). _frow["reason"] carries the EXACT tag text the
    # snapshot section already prints (_c_tag, e.g. "[NO SP DATA -- WITHHELD FROM
    # OFFICIAL, would otherwise qualify]") -- checking that instead means this can
    # never drift from what the card itself already decided qualifies.
    elif (_f_tag == "INFORMATIONAL -- BELOW BAR, NOT A RECOMMENDED PLAY" and "_log_gate_save_candidate" in globals()
          and "NO SP DATA" in str(_frow.get("reason", "")) and "would otherwise qualify" in str(_frow.get("reason", ""))):
        try:
            _log_gate_save_candidate(
                card_date=pd.Timestamp.now().strftime("%Y-%m-%d"),
                game_id=_f_row.get(game_id_col) if game_id_col and game_id_col in _f_row.index else "",
                matchup=_frow.get("matchup", ""), market=_frow.get("market", ""),
                bet=str(_frow.get("side", "")), gate_reason="NO SP DATA",
                model_prob=_frow.get("model_prob"), market_prob=_frow.get("market_prob"), edge=_frow.get("edge"),
            )
        except Exception as _gsv_log_e:
            print(f"  [WARN] Gate-save ledger write failed: {_gsv_log_e}")

    # Platoon matchup flag (2026-07-29) -- informational only. The card
    # selector above already applies a platoon run-projection adjustment
    # to every team (Cell 87's mlb_woba_splits) plus an extra "STACK
    # BOOST" for a top-5-only cutoff, but nothing ever surfaced either
    # signal here -- a team could carry a real structural platoon edge
    # and it wouldn't show anywhere in the reasoning. Real case: CIN
    # ranked top-6 in wOBA vs LHP facing lefty Cantillo -- just outside
    # the top-5 STACK BOOST cutoff, so it silently got zero extra
    # visibility even though the baseline adjustment did apply.
    if "mlb_woba_splits" in globals() and "mlb_sp_hand" in globals() and mlb_woba_splits and "_pl_norm" in globals():
        _plat_ranked = sorted(
            [(tn, v.get("vs_lhp")) for tn, v in mlb_woba_splits.items() if v.get("vs_lhp")],
            key=lambda x: -x[1]
        )
        _plat_rank_map = {tn: i + 1 for i, (tn, _) in enumerate(_plat_ranked)}
        for _f_bat_team, _f_opp_sp_cols in [
            (_f_aw, ["home_pitcher_final_name", "home_pitcher_name", "home_starter"]),
            (_f_hm, ["away_pitcher_final_name", "away_pitcher_name", "away_starter"]),
        ]:
            _f_opp_sp_nm = next((str(_f_row.get(c, "")) for c in _f_opp_sp_cols if _f_row.get(c)), "")
            if not _f_opp_sp_nm:
                continue
            _f_hand = mlb_sp_hand.get(_pl_norm(_f_opp_sp_nm), "R")
            if _f_hand != "L":
                continue
            _f_rank = _plat_rank_map.get(_pl_norm(_f_bat_team))
            if _f_rank is not None and _f_rank <= 8:
                # PLATOON VS ACTUAL LINEUP (2026-09-19, user audit -- BOS ranks #6
                # vs LHP for the season, but tonight's posted lineup is missing
                # Contreras/Yoshida/Rafaela/Mead to injury, and the season rank has
                # no way to reflect that): mlb_lineup_platoon_ops (Cell -- PLATOON +
                # wOBA SPLITS, section 4) recomputes this same OPS-vs-hand number
                # from tonight's actually-confirmed 9 batters, once a lineup is
                # posted. When it disagrees with the season rank, say so instead of
                # printing a "structural edge" that tonight's real lineup may not
                # actually carry.
                _f_actual = globals().get("mlb_lineup_platoon_ops", {}).get(_pl_norm(_f_bat_team))
                _f_league_avg = globals().get("_avg_vs_lhp") if _f_hand == "L" else globals().get("_avg_vs_rhp")
                if _f_actual and _f_actual.get("opp_hand") == _f_hand and _f_league_avg is not None:
                    _f_act_ops = _f_actual["ops_vs_opp_hand"]
                    if _f_act_ops >= _f_league_avg:
                        print(f"  [PLATOON] {_f_bat_team} ranks #{_f_rank} vs LHP facing lefty {_f_opp_sp_nm} -- "
                              f"structural offensive edge (confirmed lineup OPS {_f_act_ops:.3f} still clears "
                              f"league avg {_f_league_avg:.3f}, n={_f_actual['n']})")
                    else:
                        print(f"  [PLATOON CAUTION] {_f_bat_team} ranks #{_f_rank} vs LHP for the SEASON facing "
                              f"lefty {_f_opp_sp_nm}, but tonight's confirmed lineup's own OPS vs LHP "
                              f"({_f_act_ops:.3f}, n={_f_actual['n']}) is below league avg ({_f_league_avg:.3f}) -- "
                              f"the season rank likely doesn't reflect tonight's actual lineup")
                else:
                    print(f"  [PLATOON] {_f_bat_team} ranks #{_f_rank} vs LHP facing lefty {_f_opp_sp_nm} -- "
                          f"structural offensive edge (season-long; tonight's lineup not yet confirmed "
                          f"enough to verify against it)")

    # Bullpen risk
    # M-FIX-TODAY-2 (2026-09-08, user audit -- Pete Fairbanks tagged as a
    # live "weak link, risk=1" arm at 6.02 ERA while actually on the 15-day
    # IL since 2026-08-25 for right median nerve irritation, with fellow
    # reliever Anthony Bender ALSO on IL (right shin stress reaction) --
    # mlb_bp_risk_score/mlb_bp_weak_link (Cell 34) never cross-check roster
    # IL status at all, so an unavailable pitcher's season ERA counts as if
    # he could actually take the mound tonight. Verified live before
    # building: both names confirmed status.code=='D15' on the real 40-man
    # roster fetch. Uses the SAME rosterType=40Man endpoint already proven
    # elsewhere in this notebook (Cell 6e66db20's _mlb_roster_for_team) --
    # that one only pulls name->id, this needs each player's status field
    # too, so it's its own small fetch rather than a shared one.
    _MFT2_TEAM_ID_CACHE = {}
    def _mft2_team_id(team_name):
        if not _MFT2_TEAM_ID_CACHE:
            try:
                _r = requests.get(f"{_MLB_API}/teams?sportId=1", timeout=10).json()
                for _t in _r.get("teams", []):
                    if _t.get("name"): _MFT2_TEAM_ID_CACHE[_t["name"].lower()] = _t.get("id")
                    if _t.get("teamName"): _MFT2_TEAM_ID_CACHE[_t["teamName"].lower()] = _t.get("id")
            except Exception:
                pass
        _tn = str(team_name).lower()
        return next((v for k, v in _MFT2_TEAM_ID_CACHE.items() if v and (k in _tn or _tn in k)), None)

    _MFT2_IL_CACHE = {}
    def _mft2_il_pitchers(team_name):
        """Set of normalized names of pitchers currently on the MLB injured
        list (any D-code: D10/D15/D60) for this team."""
        _tid = _mft2_team_id(team_name)
        if not _tid:
            return set()
        if _tid in _MFT2_IL_CACHE:
            return _MFT2_IL_CACHE[_tid]
        _il = set()
        try:
            _r = requests.get(f"{_MLB_API}/teams/{_tid}/roster?rosterType=40Man", timeout=10).json()
            for _p in _r.get("roster", []):
                _code = str((_p.get("status") or {}).get("code", ""))
                if _code.startswith("D") and (_p.get("position") or {}).get("type") == "Pitcher":
                    _nm = _p.get("person", {}).get("fullName", "")
                    if _nm:
                        _il.add(_pl_norm(_nm))
        except Exception:
            pass
        _MFT2_IL_CACHE[_tid] = _il
        return _il

    _bp_wash = False
    _bp_wash_note = None
    if "mlb_bp_risk_score" in globals() and mlb_bp_risk_score and "_pl_norm" in globals():
        _bpw_aw_risk = mlb_bp_risk_score.get(_pl_norm(_f_aw), 0)
        _bpw_hm_risk = mlb_bp_risk_score.get(_pl_norm(_f_hm), 0)
        _bpw_l3_tbl = globals().get("mlb_bp_workload_l3", {}) or {}
        _bpw_aw_l3 = _bpw_l3_tbl.get(_pl_norm(_f_aw), {}).get("era_l3")
        _bpw_hm_l3 = _bpw_l3_tbl.get(_pl_norm(_f_hm), {}).get("era_l3")
        # MLB-BULLPEN-PARITY-1 (2026-09-14, user audit -- SEA @ LAA, risk=1
        # both sides, L3 ERA 7.00 vs 7.20, hot arms on both too): six
        # separate bullpen lines (2x RISK, 2x L3 ERA, 2x STRENGTH) read as
        # risk stacked against the play when the net is close to a wash.
        # Collapses ONLY the RISK + L3 ERA pair -- the two numbers the
        # user's own research matched almost exactly -- into one neutral
        # note when both teams carry risk==1 and their L3 ERAs sit within
        # 0.5R of each other. WORKLOAD/DEPTH/STRENGTH stay separate below;
        # those carry distinguishing arm names and IL status a reader
        # still needs even when RISK/L3 ERA happen to be a wash.
        if (_bpw_aw_risk == 1 and _bpw_hm_risk == 1
                and _bpw_aw_l3 is not None and _bpw_hm_l3 is not None
                and abs(_bpw_aw_l3 - _bpw_hm_l3) <= 0.5):
            _bp_wash = True
            _bp_wash_note = (f"  [BULLPEN: BILATERAL RISK] {_f_aw} ({_bpw_aw_l3:.2f} L3 ERA) vs "
                              f"{_f_hm} ({_bpw_hm_l3:.2f} L3 ERA) -- both risk=1, approximately "
                              f"equal, net effect ~0.0R")
    if "mlb_bp_risk_score" in globals() and mlb_bp_risk_score and "_pl_norm" in globals():
        for _f_team in [_f_aw, _f_hm]:
            _f_bp_tn = _pl_norm(_f_team)
            _f_risk = mlb_bp_risk_score.get(_f_bp_tn, 0)
            if _f_risk >= 1:
                _f_wl = mlb_bp_weak_link.get(_f_bp_tn, {}) if "mlb_bp_weak_link" in globals() else {}
                _f_wl_all = _f_wl.get('all', [])
                _f_il_pitchers = _mft2_il_pitchers(_f_team)
                _f_il_named = [a for a in _f_wl_all if _pl_norm(a.get('name', '')) in _f_il_pitchers]
                _f_still_avail = [a for a in _f_wl_all if _pl_norm(a.get('name', '')) not in _f_il_pitchers]
                if _f_wl.get('count', 0) >= 2:
                    _f_wl_all_str = ", ".join(f"{a['name']} {a['era']:.2f}" for a in _f_wl_all)
                    _f_wl_str = f" (weak links: {_f_wl_all_str})"
                else:
                    _f_wl_str = f" (weak link: {_f_wl.get('name', '')} {_f_wl.get('era', 0):.2f} ERA)" if _f_wl.get("name") else ""
                if not _bp_wash:
                    print(f"  [BULLPEN RISK] {_f_team}: risk={_f_risk}{_f_wl_str}")
                if _f_il_named:
                    _f_il_names_str = ", ".join(a['name'] for a in _f_il_named)
                    _f_still_str = (f"; still available: {', '.join(a['name'] for a in _f_still_avail)}"
                                    if _f_still_avail else
                                    "; no other weak-link arm identified above -- real bullpen risk is likely "
                                    "HIGHER than the score shown, not lower, since the named risk arm(s) can't "
                                    "actually pitch")
                    print(f"  !! [BULLPEN RISK STALE] {_f_team}: {_f_il_names_str} listed above but currently on "
                          f"the MLB injured list -- not available tonight{_f_still_str}")

    if _bp_wash and _bp_wash_note:
        print(_bp_wash_note)

    # Bullpen WORKLOAD, last 3 days (2026-08-31, user audit -- Red Sox pen,
    # 9 ER Saturday): mlb_bp_workload_l3 (Cell 34) already fetches real
    # last-3-day batters-faced/back-to-back-arm volume and feeds a run_adj
    # into pred_total_runs -- so a fatigued pen like Boston's WAS already
    # shifting this game's projection (+0.28R to the opponent), it just
    # never printed here. [BULLPEN RISK] above only covers weak-link ERA
    # quality, a different signal entirely -- a pen can have zero bad
    # arms and still be gassed from recent volume. Surfaced here so the
    # WHY section shows the same workload number already baked into the
    # projection instead of leaving it invisible.
    if "mlb_bp_workload_l3" in globals() and mlb_bp_workload_l3 and "_pl_norm" in globals():
        for _f_team in [_f_aw, _f_hm]:
            _f_bw = mlb_bp_workload_l3.get(_pl_norm(_f_team), {})
            if _f_bw.get("run_adj", 0) > 0:
                print(f"  [BULLPEN WORKLOAD] {_f_team}: {_f_bw.get('bf', 0)} BF / {_f_bw.get('ip', 0)} IP / "
                      f"{_f_bw.get('b2b_arms', 0)} back-to-back arm(s) over last 3 days "
                      f"(league median {_f_bw.get('bf_median', 0):.0f} BF) -> "
                      f"+{_f_bw['run_adj']:.2f}R to the opponent's projection")
            # BULLPEN L3 ERA (2026-08-31, user audit -- Red Sox pen, 9 ER
            # Saturday): a real runs-allowed signal, distinct from the
            # workload/volume line above -- a pen can face a normal number
            # of batters and still get shelled. era_run_adj (Cell 34) is
            # already feeding pred_home_runs/pred_away_runs (Cell 87), same
            # as workload; surfaced here so it's a live, visible model
            # input on every full-game ML play, not just external research.
            if _f_bw.get("era_run_adj", 0) > 0 and not _bp_wash:
                print(f"  [BULLPEN L3 ERA] {_f_team}: {_f_bw.get('era_l3', 0):.2f} ERA "
                      f"({_f_bw.get('er', 0)} ER / {_f_bw.get('ip', 0)} IP) over last 3 days "
                      f"(league avg {_f_bw.get('era_l3_league_avg', 0):.2f}) -> "
                      f"+{_f_bw['era_run_adj']:.2f}R to the opponent's projection (damped)")

    # BULLPEN DEPTH (2026-09-02, user audit -- Brewers/Cubs total, Astros ML,
    # both lost to relievers the single weak-link flag never named): distinct
    # from BULLPEN RISK (one worst arm) -- this reads how much of the team's
    # real relief workload comes from shaky (ERA >= 4.50), not necessarily
    # "weak", arms. A team can have zero named weak link and still be a real
    # risk if most of its bullpen innings come from replacement-level arms.
    if "mlb_bp_depth" in globals() and mlb_bp_depth and "_pl_norm" in globals():
        for _f_team in [_f_aw, _f_hm]:
            _f_bd = mlb_bp_depth.get(_pl_norm(_f_team), {})
            if _f_bd.get("run_adj", 0) > 0:
                _f_bd_names = ", ".join(f"{r['name']} ({r['era']:.2f})" for r in _f_bd.get("shaky_names", []))
                print(f"  [BULLPEN DEPTH] {_f_team}: {_f_bd.get('shaky_share', 0):.0%} of "
                      f"{_f_bd.get('total_ip', 0):.0f} relief IP from shaky arms ({_f_bd_names}) -> "
                      f"+{_f_bd['run_adj']:.2f}R to the opponent's projection")

    # BULLPEN STRENGTH (2026-09-02, user audit -- real Orioles loss: card
    # only ever showed bullpen WEAKNESS (RISK/DEPTH); Romano/Bernardino
    # being genuinely elite recently was invisible anywhere. mlb_bp_hl_era
    # (season-long high-leverage-arm ERA) was already computed in Cell 34
    # but never referenced again -- fully orphaned. Prefers the recency
    # read (mlb_bp_hl_recent, trailing 30d) when available since that's
    # what actually caught this case; falls back to the season view.
    if "_pl_norm" in globals():
        for _f_team in [_f_aw, _f_hm]:
            _f_tn2 = _pl_norm(_f_team)
            _f_hlr = globals().get("mlb_bp_hl_recent", {}).get(_f_tn2)
            if _f_hlr and _f_hlr.get("names"):
                _f_hlr_str = ", ".join(f"{n} {e:.2f}" for n, e in _f_hlr["names"])
                print(f"  [BULLPEN STRENGTH] {_f_team}: hot arm(s) over trailing 30d -- {_f_hlr_str} "
                      f"(avg {_f_hlr['era']:.2f} ERA) -- informational; also dampens the bullpen risk penalty above if present")
            else:
                _f_hln = globals().get("mlb_bp_hl_names", {}).get(_f_tn2)
                if _f_hln:
                    _f_hln_str = ", ".join(f"{n} {e:.2f}" for n, e in _f_hln)
                    print(f"  [BULLPEN STRENGTH] {_f_team}: season high-leverage arm(s) -- {_f_hln_str}")

    # BULLPEN B2B USAGE (2026-09-02, user audit M1/M12 -- Rockies 09-02 card,
    # Romano/Bernardino used Mon+Tue, caught manually since nothing in the
    # card named the specific pitcher(s)). Uses the same abbr-substring match
    # as the run-projection application in cell 87 (mlb_bp_b2b keys are team
    # abbreviations, e.g. "col", "bal" -- substrings of the full team name).
    if "mlb_bp_b2b" in globals() and mlb_bp_b2b:
        for _f_team in [_f_aw, _f_hm]:
            _f_tn = _f_team.lower()
            _f_b2b = next((v for k, v in mlb_bp_b2b.items() if k in _f_tn or _f_tn in k), None)
            if _f_b2b:
                _f_b2b_names = ", ".join(p["name"] for p in _f_b2b["pitchers"])
                print(f"  [BULLPEN B2B USAGE] {_f_team}: {_f_b2b_names} pitched both of the last 2 days -> "
                      f"+{_f_b2b['run_adj']:.2f}R to the opponent's projection")

    # M10 (2026-09-02 user audit): former-team matchup context -- real case
    # was Sugano (BAL SP in 2025, 10-10/4.64 ERA) facing his former club;
    # informational only, no run-projection/grade impact (no backtest exists
    # here on how often/how much ex-club motivation actually moves outcomes).
    if "mlb_sp_former_team" in globals() and mlb_sp_former_team:
        for _ft_nm, _ft_side, _ft_opp in [(_f_aw_sp, _f_aw, _f_hm), (_f_hm_sp, _f_hm, _f_aw)]:
            if not _ft_nm:
                continue
            _ft_nk2 = _mlb_name_norm(_ft_nm) if "_mlb_name_norm" in globals() else _ft_nm.lower().strip()
            _ft_hist2 = mlb_sp_former_team.get(_ft_nk2, [])
            _ft_opp_norm = _ft_opp.lower()
            _ft_match = next((h for h in _ft_hist2 if h.get("team_name") and
                               (h["team_name"].lower() in _ft_opp_norm or _ft_opp_norm in h["team_name"].lower())), None)
            if _ft_match:
                _ft_rec = f"{_ft_match['w']}-{_ft_match['l']}" if _ft_match.get("w") is not None else "record n/a"
                _ft_era_s = f", {_ft_match['era']} ERA" if _ft_match.get("era") else ""
                print(f"  [FORMER TEAM] {_ft_nm} ({_ft_side}) pitched for {_ft_opp} in {_ft_match['season']} "
                      f"({_ft_rec}{_ft_era_s}) -- ex-club matchup, informational only")

    # M14 (2026-09-02 user audit): current-season H2H record was already
    # computed and Bayesian-blended into win prob by Cell H2H (h2h_n_games /
    # h2h_home_win_rate / h2h_run_diff_avg columns on this same board row)
    # but was never actually printed anywhere -- real case: COL 2-0 vs BAL
    # on the RL in 2026 with both games going under never appeared on the
    # card, had to be manually sourced from CappersPicks. This is a straight
    # W-L + avg run-diff record (not a run-line/totals ATS count -- that
    # would need historical odds per game, which isn't fetched here), but
    # it's real season series context that already exists and just needed
    # to reach the card.
    # BUG FIX (2026-09-02, found live): board columns (h2h_n_games etc.)
    # get silently dropped a few cells downstream ("SCORE DAILY MLB BOARD
    # WITH FINAL WIN MODEL" rebuilds scored_mlb_board from the raw `board`
    # global, not from this one) -- reading from Cell H2H's mlb_h2h_lookup
    # global instead, same survives-the-rebuild pattern as mlb_bp_b2b /
    # mlb_sp_former_team / _series_context.
    _h2h_lu = globals().get("mlb_h2h_lookup", {})
    _h2h_entry = _h2h_lu.get((_f_aw.lower(), _f_hm.lower())) or _h2h_lu.get((_f_hm.lower(), _f_aw.lower()))
    if _h2h_entry:
        _h2h_n = _h2h_entry.get("h2h_n_games", 0)
        _h2h_wr = _h2h_entry.get("h2h_home_win_rate", float("nan"))
        _h2h_rd = _h2h_entry.get("h2h_run_diff_avg", float("nan"))
        if _h2h_n and pd.notna(_h2h_wr):
            _h2h_home_wins = int(round(_h2h_wr * _h2h_n))
            _h2h_away_wins = int(_h2h_n) - _h2h_home_wins
            _h2h_msg = f"  [H2H 2026] {_f_hm} {_h2h_home_wins}-{_h2h_away_wins} vs {_f_aw} this season ({int(_h2h_n)} G)"
            if pd.notna(_h2h_rd):
                _h2h_msg += f" | avg run diff {_h2h_rd:+.1f} in {_f_hm}'s favor"
            _h2h_pp_delta = _h2h_entry.get("h2h_pp_delta_applied")
            if _h2h_pp_delta is not None:
                _h2h_msg += (
                    f" | H2H blend: {_h2h_pp_delta:+.1f}pp to {_f_hm} win prob "
                    f"(Bayesian-shrunk win rate {_h2h_entry.get('h2h_bayesian_rate', float('nan')):.3f}, "
                    f"weight {_h2h_entry.get('h2h_weight', 0):.0%})"
                )
            print(_h2h_msg)

    # CURRENT H2H STREAK (2026-09-18, user audit -- Rays/Red Sox: 2026 record
    # printed as an even 5-5, but Boston has actually won the last 4
    # meetings straight -- the season-long record blends that recency away).
    # mlb_h2h_streak_lookup (Cell H2H) already computes this multi-season
    # consecutive-meetings streak and a matching edge shave already applies
    # when the picked team is on the losing end of it (see [SP TRUST DISCOUNT]
    # breakdown -- "H2H streak: -X%"), but the fact itself (who, how many)
    # never printed anywhere on the card -- a bare percentage with no reason
    # attached. Surfaced here the same way [H2H 2026] is, right below it.
    _h2h_streak_lu = globals().get("mlb_h2h_streak_lookup", {})
    _h2h_streak_entry = _h2h_streak_lu.get((_f_aw.lower(), _f_hm.lower()))
    if _h2h_streak_entry:
        _hs_winner = _h2h_streak_entry.get("streak_winner_name", "")
        _hs_len = _h2h_streak_entry.get("streak_len", 0)
        _hs_loser = _f_hm if _hs_winner == _f_aw else _f_aw
        print(f"  !! [H2H STREAK] {_hs_winner} has won {_hs_len} straight meetings vs "
              f"{_hs_loser} (spans multiple seasons if needed) -- edge already discounted "
              f"for any pick on {_hs_loser}, see SP TRUST DISCOUNT breakdown")

    # BvP (batter vs pitcher) (2026-09-10, user audit, item 2): mlb_bvp_flags
    # built in Cell (PLATOON + wOBA SPLITS + LINEUP QUALITY GATE) off tonight's
    # actually-confirmed lineup, real career at-bats vs the specific SP starting
    # tonight -- see that cell's build comment for the live-verified real
    # numbers (Arozarena 1-16 vs deGrom, Raleigh 2-13, Corey Seager 11-33 vs
    # Gilbert) that motivated this. Only ever has data once a game's lineup is
    # actually posted -- same honest limitation as the lineup-quality gate it
    # reuses, not silently guessing.
    _bvp_lu = globals().get("mlb_bvp_flags", {})
    if _bvp_lu and "_norm" in globals():
        for _bvp_team_name, _bvp_opp_sp in [(_f_aw, _f_hm_sp), (_f_hm, _f_aw_sp)]:
            _bvp_entries = _bvp_lu.get(_norm(_bvp_team_name))
            if not _bvp_entries:
                continue
            _bvp_parts = [f"{_e['name']} {_e['avg']:.3f} ({_e['hits']}-{_e['ab']}), {_e['k_pct']:.0%} K in {_e['pa']} PA"
                          for _e in _bvp_entries[:4]]
            print(f"  [BvP] {_bvp_team_name} hitters vs {_bvp_opp_sp or 'tonight' + chr(39) + 's starter'} "
                  f"(8+ career PA): " + " | ".join(_bvp_parts))

    # M11 (2026-09-02 user audit): rubber-match / series context -- real
    # case was COL up 1-0 in the series (won Game 2) heading into a
    # getaway-day decider, never flagged anywhere on the card and added
    # as a manual research note. Game N of the series comes from
    # _series_context (Cell 87 -- MLB Stats API's own seriesGameNumber/
    # gamesInSeries, already fetched for the SWEEP ATTEMPT flag elsewhere
    # in that cell, just never surfaced for the tied/decider case); the
    # win-loss tally comes from Cell H2H's series_home_wins/series_away_wins
    # (reuses the H2H games list, no extra API call) since _series_context
    # itself doesn't track a running record.
    _sr_gn = None
    _sr_gis = None
    _sr_ctx = globals().get("_series_context", {})
    _sr_ctx_gid = globals().get("_series_context_by_gid", {})
    # MLB-DH-1 (2026-09-24 user audit): try the gamePk-keyed lookup first --
    # team-pair-only lookup collapses a doubleheader/postponement-makeup's two
    # real games onto one shared series entry (real case: TOR@BAL DH, both
    # games printed "Game 3 of 3"). Falls back to the old team-pair lookup
    # when this row has no usable gamePk column (non-DH days unaffected).
    _sr_key_ctx = None
    _sr_gid_col = next((c for c in ["game_id", "gamepk", "game_pk", "pk"] if c in _f_row.index), None)
    if _sr_gid_col is not None and pd.notna(_f_row.get(_sr_gid_col)):
        _sr_key_ctx = _sr_ctx_gid.get(str(_f_row.get(_sr_gid_col)))
    if _sr_key_ctx is None and _sr_ctx:
        _sr_key_ctx = _sr_ctx.get((_f_aw.lower(), _f_hm.lower())) or _sr_ctx.get((_f_hm.lower(), _f_aw.lower()))
    if _sr_key_ctx:
        _sr_gn = _sr_key_ctx.get("game_num")
        _sr_gis = _sr_key_ctx.get("games_in_series")
    # only _series_context carries the real game_num (MLB's own
    # seriesGameNumber) -- if that lookup failed there's no reliable
    # source for it, so _sr_gn just stays 1 and the gate below no-ops.
    if _sr_gn is None:
        _sr_gn = 1
    _h2h_lu2 = globals().get("mlb_h2h_lookup", {})
    _h2h_entry2 = _h2h_lu2.get((_f_aw.lower(), _f_hm.lower())) or _h2h_lu2.get((_f_hm.lower(), _f_aw.lower()))
    # MLB-PS-SERIES-1 (2026-10-01, found live -- PHI @ ATL NLWC G3 printed
    # "series tied 0-0" while the real series was 1-1): series_home_wins/
    # series_away_wins come from the regular-season H2H games list, which has
    # no postseason games in it. Postseason games read the real record Cell 87
    # fetched from completed postseason games instead.
    _ps_rec = (globals().get("mlb_postseason_series_record", {}) or {}).get(_pl_norm(_f_hm)) if "_pl_norm" in globals() else None
    if _ps_rec and _ps_rec.get("wins") is not None:
        _sr_hw, _sr_aw = int(_ps_rec["wins"]), int(_ps_rec["opp_wins"])
        _sr_gn = _sr_hw + _sr_aw + 1
        _sr_gis = _ps_rec.get("games_in_series") or _sr_gis
        _sr_lead = (f"{_f_hm} leads {_sr_hw}-{_sr_aw}" if _sr_hw > _sr_aw else
                    (f"{_f_aw} leads {_sr_aw}-{_sr_hw}" if _sr_aw > _sr_hw else f"series tied {_sr_hw}-{_sr_aw}"))
        _sr_el = "ELIMINATION GAME -- " if _ps_rec.get("elimination") else ""
        _sr_of = f" of {int(_sr_gis)}" if _sr_gis else ""
        print(f"  [SERIES CONTEXT] {_sr_el}POSTSEASON Game {int(_sr_gn)}{_sr_of} | {_sr_lead}")
    elif pd.notna(_sr_gn) and _sr_gn > 1:
        _sr_hw = int(_h2h_entry2.get("series_home_wins", 0)) if _h2h_entry2 else 0
        _sr_aw = int(_h2h_entry2.get("series_away_wins", 0)) if _h2h_entry2 else 0
        _sr_tag = "RUBBER MATCH -- " if _sr_hw == _sr_aw else ""
        if _sr_hw > _sr_aw:
            _sr_lead = f"{_f_hm} leads {_sr_hw}-{_sr_aw}"
        elif _sr_aw > _sr_hw:
            _sr_lead = f"{_f_aw} leads {_sr_aw}-{_sr_hw}"
        else:
            _sr_lead = f"series tied {_sr_hw}-{_sr_aw}"
        _sr_of = f" of {int(_sr_gis)}" if _sr_gis else ""
        print(f"  [SERIES CONTEXT] {_sr_tag}Game {int(_sr_gn)}{_sr_of} of the series | {_sr_lead}")

    # Team recent-form flag (2026-07-29) -- no rolling L5/L10 offensive
    # signal existed anywhere in the card output. Real case: CLE on a
    # 5-game losing streak, batting .170 as a team, shut out twice in 4
    # games -- none of it appeared anywhere. Standings API's streak +
    # lastTen record is a lightweight, already-adjacent-pattern proxy
    # (single call, same style as Cell 87's team OPS fetch) -- cached for
    # the session, informational only, no run-projection or grade impact.
    _tf_cache = globals().get("_mlb_team_form_cache")
    if _tf_cache is None:
        _tf_cache = {}
        try:
            import requests as _tf_requests, datetime as _tf_dt
            _tf_r = _tf_requests.get(
                f"https://statsapi.mlb.com/api/v1/standings?leagueId=103,104&season={_tf_dt.date.today().year}&standingsTypes=regularSeason",
                timeout=10)
            for _tf_rec in _tf_r.json().get("records", []):
                for _tf_tr in _tf_rec.get("teamRecords", []):
                    _tf_tn = _pl_norm(_tf_tr.get("team", {}).get("name", "")) if "_pl_norm" in globals() else str(_tf_tr.get("team", {}).get("name", "")).lower()
                    _tf_l10 = next((s for s in _tf_tr.get("records", {}).get("splitRecords", []) if s.get("type") == "lastTen"), None)
                    _tf_night = next((s for s in _tf_tr.get("records", {}).get("splitRecords", []) if s.get("type") == "night"), None)
                    _tf_day = next((s for s in _tf_tr.get("records", {}).get("splitRecords", []) if s.get("type") == "day"), None)
                    _tf_cache[_tf_tn] = {
                        "streak_type": _tf_tr.get("streak", {}).get("streakType"),
                        "streak_n": _tf_tr.get("streak", {}).get("streakNumber"),
                        "l10_w": _tf_l10.get("wins") if _tf_l10 else None,
                        "l10_l": _tf_l10.get("losses") if _tf_l10 else None,
                        "night_w": _tf_night.get("wins") if _tf_night else None,
                        "night_l": _tf_night.get("losses") if _tf_night else None,
                        "day_w": _tf_day.get("wins") if _tf_day else None,
                        "day_l": _tf_day.get("losses") if _tf_day else None,
                    }
        except Exception as _tf_e:
            print(f"  WARN: team form fetch failed ({_tf_e})")
        globals()["_mlb_team_form_cache"] = _tf_cache
    for _tf_team in [_f_aw, _f_hm]:
        _tf_key = _pl_norm(_tf_team) if "_pl_norm" in globals() else str(_tf_team).lower()
        _tf_v = _tf_cache.get(_tf_key)
        if not _tf_v:
            continue
        _tf_bits = []
        if _tf_v.get("streak_type") == "losses" and (_tf_v.get("streak_n") or 0) >= 4:
            _tf_bits.append(f"L{_tf_v['streak_n']} skid")
        if _tf_v.get("l10_w") is not None and _tf_v.get("l10_w") <= 3:
            _tf_bits.append(f"{_tf_v['l10_w']}-{_tf_v['l10_l']} last 10")
        if _tf_bits:
            print(f"  [TEAM FORM] {_tf_team}: {', '.join(_tf_bits)}")

    # NIGHT/DAY SPLIT SIGNAL (2026-09-11, user request): no day/night split
    # existed anywhere in the model -- real case the user's own research
    # surfaced that the card missed entirely: TB 56-32 (.636) at night vs
    # HOU 51-47 (.520), an 11-game/.116 gap the card had no way to mention
    # since it never looked at day/night at all. Reuses the exact same
    # standings call as the TEAM FORM fetch just above (night/day
    # splitRecords ride along in the same response) -- no new API call.
    # Win% gap (not raw win-count gap) so it stays fair across teams with
    # different game counts in that split; a per-side minimum sample size
    # guards against small-sample noise early in a season. Informational
    # only, same as [TEAM FORM] above -- no run-projection/grade impact.
    _ND_MIN_GAMES = 15    # games in the split before trusting either side's %
    _ND_GAP_MIN   = 0.10  # win% gap (10 points) before this is worth flagging
    _nd_is_night = None
    if date_col:
        _nd_dt = pd.to_datetime(_f_row.get(date_col), errors="coerce", utc=True)
        if pd.notna(_nd_dt):
            _nd_hour_et = (_nd_dt.hour - 4) % 24
            _nd_is_night = _nd_hour_et >= 14  # matches the [DAY GAMES TODAY] cutoff elsewhere in this notebook
    if _nd_is_night is not None:
        _nd_key = "night" if _nd_is_night else "day"
        _nd_aw_key = _pl_norm(_f_aw) if "_pl_norm" in globals() else str(_f_aw).lower()
        _nd_hm_key = _pl_norm(_f_hm) if "_pl_norm" in globals() else str(_f_hm).lower()
        _nd_aw_v = _tf_cache.get(_nd_aw_key, {})
        _nd_hm_v = _tf_cache.get(_nd_hm_key, {})
        _nd_aw_w, _nd_aw_l = _nd_aw_v.get(f"{_nd_key}_w"), _nd_aw_v.get(f"{_nd_key}_l")
        _nd_hm_w, _nd_hm_l = _nd_hm_v.get(f"{_nd_key}_w"), _nd_hm_v.get(f"{_nd_key}_l")
        if None not in (_nd_aw_w, _nd_aw_l, _nd_hm_w, _nd_hm_l):
            _nd_aw_n, _nd_hm_n = _nd_aw_w + _nd_aw_l, _nd_hm_w + _nd_hm_l
            if _nd_aw_n >= _ND_MIN_GAMES and _nd_hm_n >= _ND_MIN_GAMES:
                _nd_aw_pct, _nd_hm_pct = _nd_aw_w / _nd_aw_n, _nd_hm_w / _nd_hm_n
                if abs(_nd_hm_pct - _nd_aw_pct) >= _ND_GAP_MIN:
                    print(f"  [{_nd_key.upper()} SPLIT] {_f_aw} {_nd_aw_w}-{_nd_aw_l} ({_nd_aw_pct:.3f}) vs "
                          f"{_f_hm} {_nd_hm_w}-{_nd_hm_l} ({_nd_hm_pct:.3f}) {_nd_key} this season")

    # Team-record credibility gap -- informational, mirrors the mechanical
    # nudge already applied earlier in the card (see _trw_records above)
    if "_trw_records" in globals() and _trw_records:
        def _f_trw_lookup(team_name):
            # BUG FIX (MLB-RECORD-1, 2026-09-24 user audit): the old inline
            # "any word >3 chars matches a substring" lookup confuses
            # same-city teams sharing a city word (LAD/LAA both contain
            # "angeles", etc.) -- real incident: LAD printed at .382 (their
            # actual win% was .618) because this matched the Angels' record
            # instead. _trw_lookup_generic (Cell 87) matches on the team's
            # actual last-word nickname instead, which can't collide on a
            # shared city name.
            return _trw_lookup_generic(_trw_records, team_name)
        _f_aw_pct = _f_trw_lookup(_f_aw)
        _f_hm_pct = _f_trw_lookup(_f_hm)
        if _f_aw_pct is not None and _f_hm_pct is not None and abs(_f_hm_pct - _f_aw_pct) >= 0.15:
            print(f"  [RECORD GAP] {_f_aw} {_f_aw_pct:.3f}  vs  {_f_hm} {_f_hm_pct:.3f}")

if len(_snap_ranked):
    for _fi, _frow in _snap_ranked.iterrows():
        _f_tag = "OFFICIAL" if (_fi == 0 or _MLB_FP) else "QUALIFYING POOL -- subject to 1-play/sport card rule"
        _emit_why(_frow, _f_tag)
if len(_snap_other_ranked):
    for _fi, _frow in _snap_other_ranked.iterrows():
        _emit_why(_frow, "INFORMATIONAL -- BELOW BAR, NOT A RECOMMENDED PLAY")
if not len(_snap_ranked) and not len(_snap_other_ranked):
    print("  NONE -- no edges found anywhere today, nothing to show reasoning for.")

if _fed_divergence_count > 0:
    print(f"\n[SP ELO-ERA DIVERGENCE SUMMARY] {_fed_divergence_count} thin-IP pitcher(s) flagged tonight "
          f"(established starters with enough IP self-clear -- see Gate 8 / _fed_check). Track this count "
          f"across nights: a sustained high count would suggest broad ELO staleness, not just thin samples.")

# MLB-PS-TOT-TRACK-1 / MLB-SERIES-TOT-1 output (buffered above)
for _tl_l in globals().get("_tl_out", []):
    print(_tl_l)


MLB DAILY MARKET SNAPSHOT -- MODEL VS MARKET (edge games only)
Card date: 2026-10-10  [today's slate]
Odds pulled: 2026-10-10 09:13:02 (local) -- every mkt%/edge% below is against this single snapshot, not a live line



Chicago White Sox @ Cleveland Guardians
  ML   Cleveland Guardians    model 63.0%  mkt 56.1% (-142)  edge +6.9%  [pre-blend model 63.0%/edge +6.9% -> post-blend edge +6.9%]  [passes_edge_rule failed: logit edge +0.287 < required +0.490 (12.0% floor) -- at mkt 56.1% (vig-incl) needs model >= 67.6%, has 63.0%, missed by 4.6%]  | Driver: SP Gavin Williams vs Sean Burke (+216 ELO for the pick's starter); SP H2H -- pick SP Gavin Williams vs Chicago White Sox: 3.98 ERA in 4 GS this season, 4.72 career (7 app) | opp SP Sean Burke vs Cleveland Guardians: 1.46 ERA in 2 GS this season, 2.56 career (7 app); pre-blend model 63.0% vs mkt 56.1%
  TOT  line 7.0  model proj 7.3  (calibrated 7.3)
       Under model 47.7%  mkt 46.8% (-125/+105)  edge +0.9%  [not blended -- ML-only]  [noise -- below 2% floor]
  RL   Cleveland Guardians -1.5  not evaluated (projected margin below 1.0R minimum)
  !! SP ROLE UNRATED -- applies to EVERY market on this game (ML / RL / total), none can be OFFICIAL: Gate 7d po

  NONE -- no game/market cleared the model's official-play bar today.
  Recommended action: skip today, no bets. (A big raw edge shown above
  without this tag is not a substitute for a qualifying play.)

DAILY PLAY CANDIDATE ANALYSIS (INFORMATIONAL -- 20-PLAY GRADUATION WINDOW)
  No qualifying plays today -- nothing to compare.

OTHER EDGES BELOW THE BAR -- INFORMATIONAL ONLY, NOT RECOMMENDED PLAYS
   +6.9%  ML           Cleveland Guardians    Chicago White Sox @ Cleveland Guardians  @ -142 (ESPN feed)  [pre-blend +6.9%]  [Gate 7d postseason SP role UNRATED -- PIGGYBACK RISK -- Chicago White Sox can piggyback Anthony Kay (31 GS, short rest), Davis Martin (28 GS, short rest), Erick Fedde (15 GS, short rest); Cleveland Guardians can piggyback Tanner Bibee (33 GS, short rest), Parker Messick (32 GS, short rest), Foster Griffin (31 GS, short rest) -- who covers the middle/late innings is unknowable pregame, SP role UNRATED]  [passes_edge_rule failed: logit edge +0.287 < required +0.490 (1

  !! [SP WORKLOAD] Sean Burke (Chicago White Sox): 142.3 IP this season -- past the 120-IP fatigue-risk threshold. Season-long ELO has no innings-workload decay, so a late-season fade wouldn't show up in the tier above -- ALSO past his own career-high IP: run-projection weight already shifted toward the bullpen for this, see Proj Score above
  [SP COLD STREAK] Sean Burke (Chicago White Sox): L3 ERA 6.39 vs season 3.73 -- xERA (3.78) sits closer to season ERA -- likely sequencing luck, not a true change | SHAPE: one blowup start (48.46, 1.50, 1.70 ERA by start, oldest-to-newest) dragging the L3 average -- other 2 starts within normal range, not a sustained decline
  !! [SP WORKLOAD] Gavin Williams (Cleveland Guardians): 184.3 IP this season -- past the 120-IP fatigue-risk threshold. Season-long ELO has no innings-workload decay, so a late-season fade wouldn't show up in the tier above -- ALSO past his own career-high IP: run-projection weight already shifted toward the bullpen for this,